# Diarios de avances de exploración
¿Cuánto aporta cada algoritmo y en qué momento conviene compartir cálculos? Cada sección ejecuta una variante en el mismo puzzle pequeño y después compara las mediciones reales con el paso anterior. Menos instrucciones es mejor.

Las comparaciones usan únicamente IDs resueltos por ambos métodos. La tabla final incluye cobertura para distinguir resultados parciales de entregas completas. Los 1.000 puntuables se miden localmente.

## Datos y funciones de lectura
Se toma la corrida completa más reciente de cada conjunto. Para regenerar los reportes desde la raíz, ejecutar `python3 -m scripts.compare_dag` (práctica) o `python3 -m scripts.scored_dag` (puntuables). Ejecutar el diario solo lee esos reportes y resuelve el ejemplo.

In [1]:
import csv
from pathlib import Path
from scripts.compare_dag import (Puzzle, validate, incremental, rings, bisection, multirings,
    bisection_reuse, multirings_reuse, postprocess, optimize_a4, nested_windows, optimize_dag)

def leer(ruta):
    with ruta.open(newline='') as archivo:
        return list(csv.DictReader(archivo))

def reporte(prefijo, marca):
    archivos = list(Path('reports').glob(f'{prefijo}_*/{marca}'))
    if not archivos:
        raise FileNotFoundError(f'Falta una corrida completa de {prefijo}. Ver comandos arriba.')
    return max(archivos, key=lambda p: p.stat().st_mtime_ns).parent

carpetas = {'Práctica': reporte('dag_practice', 'summary.csv'),
            'Puntuables': reporte('dag_scored', 'submission.csv')}
datos = {k: leer(v / 'comparison.csv') for k, v in carpetas.items()}
ejemplo = Puzzle('ejemplo', [1, 3, 10], [27, 97, 99, 107], 200)
historial = []

def medir(filas, columna, algoritmo=None, etapa=None):
    valores = {}
    for fila in filas:
        if fila['error'] or (algoritmo and fila['algorithm'] != algoritmo):
            continue
        if etapa and fila['insertion'] != etapa:
            continue
        valor = int(fila[columna])
        k = fila['id']
        valores[k] = min(valores.get(k, valor), valor)
    return valores

def comparar(nombre, columna, algoritmo=None, etapa=None):
    actual = {k: medir(filas, columna, algoritmo, etapa) for k, filas in datos.items()}
    for k, valores in actual.items():
        if historial:
            prev = historial[-1][1][k]
            comunes = prev.keys() & valores.keys()
            ahorro = sum(prev[i] - valores[i] for i in comunes)
            print(f'{k}: {len(valores)} resueltos | {len(comunes)} comunes | ahorro frente a {historial[-1][0]}: {ahorro:+,}')
        else:
            print(f'{k}: {len(valores)} resueltos | {sum(valores.values()):,} instrucciones')
    historial.append((nombre, actual))

for k, ruta in carpetas.items():
    print(k, ruta)

Práctica reports/dag_practice_gpvvc01y
Puntuables reports/dag_scored_ojhhopuu


## A1 · Incremental
Conserva un uno y suma hasta alcanzar cada objetivo en orden. Sirve como primera referencia y puede agotar el límite de instrucciones.

In [2]:
programa = incremental(ejemplo)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 107, 'covered': 4, 'valid': True}


### Medición en ambos conjuntos
Establece la referencia inicial.

In [3]:
comparar('A1 · Incremental', 'before', 'incremental', 'constructor')

Práctica: 16 resueltos | 144 instrucciones
Puntuables: 16 resueltos | 139 instrucciones


## A2 · Anillos
Avanza con múltiplos de una base y completa el resto para alcanzar el objetivo.

In [4]:
programa = rings(ejemplo)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 42, 'covered': 4, 'valid': True}


### Medición en ambos conjuntos
Compara el constructor con el de la sección anterior sobre los IDs comunes.

In [5]:
comparar('A2 · Anillos', 'before', 'rings', 'constructor')

Práctica: 159 resueltos | 16 comunes | ahorro frente a A1 · Incremental: +8
Puntuables: 778 resueltos | 16 comunes | ahorro frente a A1 · Incremental: +4


## A3 · Bisección
Parte del intervalo entre múltiplos y construye valores intermedios para acercarse al objetivo.

In [6]:
programa = bisection(ejemplo)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 42, 'covered': 4, 'valid': True}


### Medición en ambos conjuntos
Compara el constructor con el de la sección anterior sobre los IDs comunes.

In [7]:
comparar('A3 · Bisección', 'before', 'bisection', 'constructor')

Práctica: 142 resueltos | 133 comunes | ahorro frente a A2 · Anillos: -126,549
Puntuables: 775 resueltos | 684 comunes | ahorro frente a A2 · Anillos: -683,353


## Múltiples anillos
Explora distintas bases iniciales para acortar la construcción.

In [8]:
programa = multirings(ejemplo)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 33, 'covered': 4, 'valid': True}


### Medición en ambos conjuntos
Compara el constructor con el de la sección anterior sobre los IDs comunes.

In [9]:
comparar('Múltiples anillos', 'before', 'multirings', 'constructor')

Práctica: 163 resueltos | 142 comunes | ahorro frente a A3 · Bisección: +67,763
Puntuables: 853 resueltos | 775 comunes | ahorro frente a A3 · Bisección: +369,669


## Bisección con reutilización
Aprovecha extremos y valores ya presentes en registros durante la bisección.

In [10]:
programa = bisection_reuse(ejemplo)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 20, 'covered': 4, 'valid': True}


### Medición en ambos conjuntos
Compara el constructor con el de la sección anterior sobre los IDs comunes.

In [11]:
comparar('Bisección con reutilización', 'before', 'bisection_reuse', 'constructor')

Práctica: 200 resueltos | 163 comunes | ahorro frente a Múltiples anillos: +153,313
Puntuables: 1000 resueltos | 853 comunes | ahorro frente a Múltiples anillos: +829,456


## Múltiples anillos con reutilización
Combina la elección de base con el aprovechamiento de valores residentes. Este programa alimenta las tres pasadas siguientes.

In [12]:
programa = multirings_reuse(ejemplo)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 14, 'covered': 4, 'valid': True}


### Medición en ambos conjuntos
Compara el constructor con el de la sección anterior sobre los IDs comunes.

In [13]:
comparar('Múltiples anillos con reutilización', 'before', 'multirings_reuse', 'constructor')

Práctica: 200 resueltos | 200 comunes | ahorro frente a Bisección con reutilización: +14,062
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a Bisección con reutilización: +66,480


## Posprocesamiento
Comprime operaciones y elimina cálculos sin uso.

In [14]:
programa = postprocess(ejemplo, programa)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 9, 'covered': 4, 'valid': True}


### Efecto de la pasada
Mide el ahorro respecto del programa de la etapa anterior.

In [15]:
comparar('Posprocesamiento', 'before', 'multirings_reuse', 'postprocess')

Práctica: 200 resueltos | 200 comunes | ahorro frente a Múltiples anillos con reutilización: +114,905
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a Múltiples anillos con reutilización: +576,348


## A4
Busca atajos que permiten pasarse de un objetivo y corregir por resta o división exacta.

In [16]:
programa = optimize_a4(ejemplo, programa)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 9, 'covered': 4, 'valid': True}


### Efecto de la pasada
Mide el ahorro respecto del programa de la etapa anterior.

In [17]:
comparar('A4', 'before', 'multirings_reuse', 'a4')

Práctica: 200 resueltos | 200 comunes | ahorro frente a Posprocesamiento: +550
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a Posprocesamiento: +2,579


## Ventanas de ventanas
Une bloques vecinos para aprovechar valores que cruzan sus límites.

In [18]:
programa = nested_windows(ejemplo, programa)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 9, 'covered': 4, 'valid': True}


### Efecto de la pasada
Mide el ahorro respecto del programa de la etapa anterior.

In [19]:
comparar('Ventanas de ventanas', 'before', 'multirings_reuse', 'nested')

Práctica: 200 resueltos | 200 comunes | ahorro frente a A4: +437
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a A4: +2,239


## Mejor control por puzzle
Cada constructor válido recibe las tres pasadas. Se elige el programa final más corto por ID para medir el efecto de poder cambiar de constructor.

In [20]:
constructores = (incremental, rings, bisection, multirings, bisection_reuse, multirings_reuse)
pasadas = (postprocess, optimize_a4, nested_windows)

def ruta(constructor, insercion=None):
    programa = constructor(ejemplo)
    for posicion in range(4):
        if posicion == insercion:
            programa = optimize_dag(ejemplo, programa)
        if posicion < len(pasadas):
            programa = pasadas[posicion](ejemplo, programa)
    return validate(ejemplo, programa)['steps']

print(min(ruta(fn) for fn in constructores))

7


### Comparación con ventanas
Cuantifica el cambio de usar el mejor constructor por ID.

In [21]:
comparar('Mejor control sin DAG', 'control_final')

Práctica: 200 resueltos | 200 comunes | ahorro frente a Ventanas de ventanas: +9,800
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a Ventanas de ventanas: +44,318


## DAG después de constructor
Comparte valores iguales en el programa de esta etapa y ejecuta las pasadas pendientes. Se selecciona por ID el mejor de los constructores en ese orden.

In [22]:
print(min(ruta(fn, 0) for fn in constructores))

10


### Comparación del resultado final
Compara los programas terminados con la sección anterior.

In [23]:
comparar('DAG después de constructor', 'dag_final', etapa='constructor')

Práctica: 200 resueltos | 200 comunes | ahorro frente a Mejor control sin DAG: +41,649
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a Mejor control sin DAG: +194,289


## DAG después de posprocesamiento
Comparte valores iguales en el programa de esta etapa y ejecuta las pasadas pendientes. Se selecciona por ID el mejor de los constructores en ese orden.

In [24]:
print(min(ruta(fn, 1) for fn in constructores))

7


### Comparación del resultado final
Compara los programas terminados con la sección anterior.

In [25]:
comparar('DAG después de posprocesamiento', 'dag_final', etapa='postprocess')

Práctica: 200 resueltos | 200 comunes | ahorro frente a DAG después de constructor: -26,811
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a DAG después de constructor: -122,126


## DAG después de A4
Comparte valores iguales en el programa de esta etapa y ejecuta las pasadas pendientes. Se selecciona por ID el mejor de los constructores en ese orden.

In [26]:
print(min(ruta(fn, 2) for fn in constructores))

7


### Comparación del resultado final
Compara los programas terminados con la sección anterior.

In [27]:
comparar('DAG después de A4', 'dag_final', etapa='a4')

Práctica: 200 resueltos | 200 comunes | ahorro frente a DAG después de posprocesamiento: -77
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a DAG después de posprocesamiento: -499


## DAG después de ventanas
Comparte valores iguales en el programa de esta etapa y ejecuta las pasadas pendientes. Se selecciona por ID el mejor de los constructores en ese orden.

In [28]:
print(min(ruta(fn, 3) for fn in constructores))

7


### Comparación del resultado final
Compara los programas terminados con la sección anterior.

In [29]:
comparar('DAG después de ventanas', 'dag_final', etapa='nested')

Práctica: 200 resueltos | 200 comunes | ahorro frente a DAG después de A4: +0
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a DAG después de A4: +0


## Selección final por puzzle
Se conserva el mejor de los controles y las cuatro posiciones del DAG. Para el entregable también compite el programa del CSV previo en cada ID.

In [30]:
seleccion = {}
for k, filas in datos.items():
    control = medir(filas, 'control_final')
    dag = medir(filas, 'dag_final')
    seleccion[k] = {i: min(valor, dag[i]) for i, valor in control.items()}
ganadores = leer(carpetas['Puntuables'] / 'winners.csv')
entrega = {r['id']: int(r['steps']) for r in ganadores}
anterior = {r['id']: int(r['previous']) for r in ganadores}
assert len(entrega) == 1000 and len(seleccion['Práctica']) == 200
assert entrega == {i: min(anterior[i], seleccion['Puntuables'][i]) for i in entrega}
print('Entregable validado:', carpetas['Puntuables'] / 'submission.csv')

Entregable validado: reports/dag_scored_ojhhopuu/submission.csv


### Comparación con la entrega anterior
Cuenta cuántos puzzles mejoraron y cuánto bajó la suma de instrucciones.

In [31]:
print('Puzzles mejorados:', sum(entrega[i] < anterior[i] for i in entrega))
print('Instrucciones ahorradas:', sum(anterior.values()) - sum(entrega.values()))

Puzzles mejorados: 978
Instrucciones ahorradas: 250357


## Tabla de avances
Totales de instrucciones calculados desde los reportes seleccionados. Cada fila indica cuántos puzzles se resolvieron sobre 200 o 1.000; una suma parcial solo describe esos IDs. El signo — indica que falta una medición para ese conjunto.

In [32]:
def celda(valores, cantidad):
    if not valores:
        return '—'
    total = f'{sum(valores.values()):,}'.replace(',', '.')
    return f'{total} ({len(valores)}/{cantidad})'

filas = historial + [
    ('Mejor control o DAG por ID', seleccion),
    ('Entregable anterior', {'Puntuables': anterior}),
    ('Entregable validado', {'Puntuables': entrega})]
from pathlib import Path
import csv

def cargar_pasos(ruta):
    with Path(ruta).open() as archivo:
        return {r["id"]: int(r["final"]) for r in csv.DictReader(archivo)}

rutas_practica = cargar_pasos("reports/partial_practice_i0l79tbp/winners.csv")
rutas_puntuables = cargar_pasos("reports/partial_scored_2jzgqmyl/winners.csv")
con_constructor = cargar_pasos("reports/backward_practice_wyyedfh3/comparison.csv")
assert rutas_practica.keys() == con_constructor.keys()
assert all(con_constructor[k] <= pasos for k, pasos in rutas_practica.items())
constructor_puntuables = cargar_pasos("reports/backward_scored_LECwzt0R/comparison.csv")
assert constructor_puntuables.keys() == rutas_puntuables.keys()
assert all(constructor_puntuables[k] <= pasos for k, pasos in rutas_puntuables.items())
nuevas_filas = [
    ("Cartera de cuatro rutas e histórico", {"Práctica": rutas_practica, "Puntuables": rutas_puntuables}),
    ("Cartera con constructor hacia atrás", {"Práctica": con_constructor, "Puntuables": constructor_puntuables}),
]
filas.extend(nuevas_filas)
from html import escape
from IPython.display import HTML, display

encabezados = ('Algoritmo o entrega', 'Práctica · pasos (cobertura)', 'Puntuables · pasos (cobertura)')
tabla = '<table><thead><tr>' + ''.join(f'<th>{escape(t)}</th>' for t in encabezados) + '</tr></thead><tbody>'
for nombre, valores in filas:
    columnas = (nombre, celda(valores.get('Práctica'), 200), celda(valores.get('Puntuables'), 1000))
    tabla += '<tr>' + ''.join(f'<td>{escape(v)}</td>' for v in columnas) + '</tr>'
display(HTML(tabla + '</tbody></table>'))

Algoritmo o entrega,Práctica · pasos (cobertura),Puntuables · pasos (cobertura)
A1 · Incremental,144 (16/200),139 (16/1000)
A2 · Anillos,234.714 (159/200),1.166.626 (778/1000)
A3 · Bisección,332.967 (142/200),1.929.943 (775/1000)
Múltiples anillos,340.301 (163/200),1.834.813 (853/1000)
Bisección con reutilización,281.552 (200/200),1.372.573 (1000/1000)
Múltiples anillos con reutilización,267.490 (200/200),1.306.093 (1000/1000)
Posprocesamiento,152.585 (200/200),729.745 (1000/1000)
A4,152.035 (200/200),727.166 (1000/1000)
Ventanas de ventanas,151.598 (200/200),724.927 (1000/1000)
Mejor control sin DAG,141.798 (200/200),680.609 (1000/1000)


## Cuatro rutas de búsqueda parcial en práctica
A conserva el original. B elige por longitud inicial y C posprocesa las mismas opciones antes de elegir. D busca sobre los originales ya posprocesados. Cada selección desempata en favor del original. La columna de pasos toma el mínimo por ID entre los seis constructores. Los tiempos suman búsqueda y optimización por constructor, atribuyendo a cada ruta el trabajo reutilizado. Excluyen construcción, indexación de donantes y reconstrucción histórica.
La [tabla por ID](reports/partial_practice_i0l79tbp/winners.csv) registra el ganador frente al histórico. El [detalle por constructor y ruta](reports/partial_practice_i0l79tbp/comparison.csv) incluye fallos de cobertura, índice del candidato (0 para el original) y longitudes por etapa. En D la entrada corresponde al candidato formado después del posprocesamiento.

In [33]:
from pathlib import Path
import csv
from scripts.validate_submission import verify

carpeta = Path("reports/partial_practice_i0l79tbp")
def leer(nombre):
    with (carpeta / nombre).open() as archivo:
        return list(csv.DictReader(archivo))

detalle = [r for r in leer("comparison.csv") if not r["error"]]
ganadores = leer("winners.csv")
validos, total = verify(Path("practice_puzzles.csv"), carpeta / "programs.csv")
por_ruta = {}
print("Ruta   IDs   Candidatos   Pasos   Segundos acumulados")
for ruta in "ABCD":
    filas = [r for r in detalle if r["route"] == ruta]
    mejores = {}
    for r in filas:
        k, pasos = r["id"], int(r["final"])
        mejores[k] = min(mejores.get(k, pasos), pasos)
    por_ruta[ruta] = mejores
    print(ruta, len(mejores), sum(int(r["candidates"]) for r in filas),
          sum(mejores.values()), round(sum(float(r["seconds"]) for r in filas), 1))

for nueva, control in [("C", "B"), ("D", "A")]:
    delta = [por_ruta[control][k] - v for k, v in por_ruta[nueva].items()]
    print(f"{nueva} frente a {control}: {sum(d > 0 for d in delta)} mejoras,"
          f" {delta.count(0)} empates, {sum(d < 0 for d in delta)} retrocesos,"
          f" {sum(delta)} pasos ahorrados.")
    pares = {(r["id"], r["algorithm"]): int(r["final"]) for r in detalle if r["route"] == control}
    cambios = [pares[r["id"], r["algorithm"]] - int(r["final"]) for r in detalle if r["route"] == nueva]
    print("Por par ID-constructor (mejor / igual / peor):", sum(d > 0 for d in cambios),
          cambios.count(0), sum(d < 0 for d in cambios))

previo = sum(int(r["historical"]) for r in ganadores)
assert validos == len(ganadores) == 200 and previo == 98127
for r in ganadores:
    assert int(r["final"]) == min(int(r["historical"]), *(s[r["id"]] for s in por_ruta.values()))
assert total == sum(int(r["final"]) for r in ganadores)
print(f"Cartera validada: {total} pasos en {validos} puzzles. Ahorro histórico: {previo-total}.")
print("IDs mejorados:", sum(int(r["final"]) < int(r["historical"]) for r in ganadores))


Ruta   IDs   Candidatos   Pasos   Segundos acumulados
A 200 880 141798 2948.0
B 200 6988 103295 2786.1
C 200 6988 100957 22459.0
D 200 6953 128287 2974.4
C frente a B: 147 mejoras, 52 empates, 1 retrocesos, 2338 pasos ahorrados.
Por par ID-constructor (mejor / igual / peor): 602 270 8
D frente a A: 173 mejoras, 26 empates, 1 retrocesos, 13511 pasos ahorrados.
Por par ID-constructor (mejor / igual / peor): 733 146 1
Cartera validada: 92881 pasos en 200 puzzles. Ahorro histórico: 5246.
IDs mejorados: 145


## Búsqueda hacia atrás y recombinación de recetas

En la versión revisada de royalacecat, combinar los mejores programas públicos por ID ya suma 132.623 pasos. La búsqueda seleccionada para `scored_0295` lleva ese puzzle de 119 a 118 y el total a 132.622. Nuestro programa tiene 366 pasos en ese ID. La mayor diferencia ya existe al comenzar esa búsqueda. [Notebook de royalacecat, versión 372165224](https://www.kaggle.com/code/royalacecat/santa-2026-public-frontier-score-132-638?scriptVersionId=372165224)

GoodPJW v42 registra 132.636 pasos para la unión pública y 132.634 al incorporar dos programas propios. Su búsqueda local elimina una rama de dependencias y la reconstruye con valores del resto del grafo. Guarda recetas de programas empatados. CP-SAT combina esas recetas, elige un orden válido y permite que dos objetivos compartan un intermedio. El autor reporta ahorros de 45, 28 y 4 pasos en tres pasadas anteriores. Los puntajes públicos son resultados de sus autores. [Notebook de GoodPJW, v42](https://www.kaggle.com/code/goodpjw2008/santa-2026-lns-cp-sat-recombine-lb-132-634?scriptVersionId=372161473)

### Qué hace el constructor de Tony Li

Para cada objetivo prepara un diccionario de expresiones que cuestan hasta dos operaciones desde valores de partida, constantes, objetivos ya alcanzados e intermedios recientes. Si el objetivo no aparece, busca descomposiciones por cociente y resto. Por ejemplo, con 16 y 1 disponibles, construir 97 se reduce a construir 6 mediante la identidad `97 = 6 × 16 + 1`. El haz conserva varias reducciones prometedoras y repite la búsqueda sobre sus cocientes. Después emite solo las dependencias necesarias, asigna registros y puede reconstruir un valor desalojado. Konbu17 amplía el diccionario y combina varias configuraciones por ID. Su total publicado de 142.940 también incluye otros métodos. [Tony Li](https://www.kaggle.com/code/tonylica/elf-calculator-with-register-reuse?scriptVersionId=370866029) · [Konbu17](https://www.kaggle.com/code/konbu17/santa-2026-wider-dictionary-portfolio-163k?scriptVersionId=371106642)

### Experimento sobre los 200 puzzles de práctica

Se compilaron las tres celdas C++ originales de Tony Li con `g++ -std=c++17 -O3`. Cada corrida recibe los mismos objetivos ordenados y solo cambia el ancho del haz (1, 3 o 6). El valor publicado es 3. Cada programa parte de los valores iniciales del puzzle. La referencia local es la mejor cartera por ID después de las cuatro rutas (92.881 instrucciones). La celda siguiente reproduce la comparación desde los programas guardados. La elección del menor resultado por ID requiere ejecutar todas las configuraciones que se comparan. [Fuente compilada](reports/backward_practice_wyyedfh3/solver.cpp)


In [34]:
import csv
import json
from pathlib import Path
from scripts.validate_submission import verify

datos = Path("practice_puzzles.csv")
carpeta = Path("reports/backward_practice_wyyedfh3")
archivos = {
    "Cartera anterior": Path("reports/partial_practice_i0l79tbp/programs.csv"),
    "Ancho 1": carpeta / "programs_w1.csv",
    "Ancho 3": carpeta / "programs.csv",
    "Ancho 6": carpeta / "programs_w6.csv",
}
programas = {}
for nombre, ruta in archivos.items():
    cantidad, total = verify(datos, ruta)
    assert cantidad == 200
    with ruta.open() as fp:
        programas[nombre] = {r["id"]: r["program"] for r in csv.DictReader(fp)}
    print(f"{nombre:16} {total:>7} pasos  ({cantidad}/200)")

with datos.open() as fp:
    puzzles_practica = list(csv.DictReader(fp))
familias = {}
mejores_tres = 0
ganancias = empates = 0
with (carpeta / "comparison.csv").open("w", newline="") as tabla, (carpeta / "portfolio.csv").open("w", newline="") as salida:
    detalle, entrega = csv.writer(tabla), csv.writer(salida)
    detalle.writerow(["id", "familia", *archivos, "ganador", "final"])
    entrega.writerow(["id", "program"])
    for puzzle in puzzles_practica:
        k, familia = puzzle["id"], puzzle["family"]
        pasos = {n: len(json.loads(banco[k])) for n, banco in programas.items()}
        previo = pasos["Cartera anterior"]
        nuevo = min(pasos[n] for n in ("Ancho 1", "Ancho 3", "Ancho 6"))
        mejores_tres += nuevo
        ganancias += nuevo < previo
        empates += nuevo == previo
        ganador = min(pasos, key=pasos.get)
        detalle.writerow([k, familia, *pasos.values(), ganador, pasos[ganador]])
        entrega.writerow([k, programas[ganador][k]])
        suma = familias.setdefault(familia, [0, 0, 0])
        suma[0] += 1
        suma[1] += previo
        suma[2] += nuevo

print(f"Mejor de los tres anchos por ID: {mejores_tres} pasos.")
print(f"Frente al anterior: {ganancias} mejoras, {empates} empates y {200-ganancias-empates} retrocesos.")
for familia, (cuantos, anterior, atras) in sorted(familias.items()):
    print(f"{familia:15} {cuantos:3} puzzles   {anterior:6} → {atras:6}")
cantidad, total = verify(datos, carpeta / "portfolio.csv")
print(f"Cartera combinada validada: {cantidad} programas, {total} pasos.")

Cartera anterior   92881 pasos  (200/200)
Ancho 1            40474 pasos  (200/200)
Ancho 3            39135 pasos  (200/200)
Ancho 6            38648 pasos  (200/200)
Mejor de los tres anchos por ID: 38566 pasos.
Frente al anterior: 183 mejoras, 3 empates y 14 retrocesos.
independent      45 puzzles    32715 →  12973
mixed            48 puzzles    27815 →  11471
multiplicative   40 puzzles    10396 →   5244
shared           51 puzzles    21901 →   8801
tiny             16 puzzles       54 →     77
Cartera combinada validada: 200 programas, 38540 pasos.


### Aporte del constructor a la cartera de práctica
Al sumar los anchos 1, 3 y 6 como opciones por ID, la cartera pasa de 92.881 a 38.540 instrucciones (54.341 menos, 58,51 %). Mejoran 183 IDs y 17 mantienen su longitud. En 14 de esos 17 el programa anterior es estrictamente más corto; en los otros 3 empata. La mayor reducción absoluta aparece en la familia independiente (19.742 instrucciones). En los 16 puzzles diminutos, la selección conserva el total previo de 54. La medición del nuevo constructor cubre los 200 puzzles de práctica.

### Constructor hacia atrás sobre los 1.000 puntuables

| Configuración | Instrucciones |
|---|---:|
| Cartera anterior | 448.063 |
| Ancho 1 | 197.890 |
| Ancho 3 | 191.867 |
| Ancho 6 | 189.141 |
| Cartera combinada | 188.917 |

El mejor de los tres anchos mejora 965 IDs, empata en 11 y queda por detrás en 24. La selección conserva la cartera anterior en los empates. El verificador reprodujo los 1.000 programas de cada configuración y los del CSV final. [Comparación por ID](reports/backward_scored_LECwzt0R/comparison.csv) · [Entrega validada](artifacts/submission_backward_188917.csv)

## Cuatro experimentos sobre el constructor hacia atrás

Cierre validado: 36049 pasos, 2491 ahorrados. [Programas](reports/four_backward_practice/portfolio.csv) · [Ganadores](reports/four_backward_practice/comparison.csv)

Piloto completado: 110 configuraciones en 10 IDs, dos por familia. Suite del entorno: 42 pruebas aprobadas.

| Diccionario y ancho | Control original | Ampliado | Diferencia a favor del ampliado |
|---|---:|---:|---:|
|52-1|40474|37554|2920|
|52-3|39135|36657|2478|
|52-6|38648|36281|2367|
|64-1|40474|37861|2613|
|64-3|39135|37089|2046|
|64-6|38648|36422|2226|

E1 elegir antes: 38436 · E1 elegir después: 38430 · E3 individual: 38491 · E3 con CP-SAT: 38349

| Método | IDs elegidos | Aporte exclusivo |
|---|---:|---:|
|E1|10|11|
|E2|164|2118|
|E3|0|0|
|E4|12|23|

| Familia | E1 | E2 | E3 | E4 | Cartera |
|---|---:|---:|---:|---:|---:|
|independent|12955|12092|12896|12968|12086|
|mixed|11446|10701|11393|11459|10692|
|multiplicative|5213|5051|5238|5149|5030|
|shared|8759|8201|8768|8730|8189|
|tiny|57|77|54|53|52|

Programas reproducidos por método: [E1](reports/four_backward_practice/E1.csv) · [E2](reports/four_backward_practice/E2.csv) · [E3](reports/four_backward_practice/E3.csv) · [E4](reports/four_backward_practice/E4.csv)

Estados de CP-SAT: FEASIBLE 28, OPTIMAL 170, rechazado por registros 2

Referencia validada: 38.540 pasos en 200 IDs. Haz 1/3/6. E2: 52/72 y 64/72. CP-SAT 9.15.6755, semilla 42, un hilo y 30 s. Reparación local: 30 s. Grupo compartido: 14 procesos.

| Método | IDs completos | Histórico | Método | Ahorro al combinar | CPU s | Tiempo acumulado s |
|---|---:|---:|---:|---:|---:|---:|
|E1|200/200|38540|38430|114|94.9|97.8|
|E2|200/200|38540|36122|2447|3200.5|3270.9|
|E3|200/200|38540|38349|191|1102.5|1104.8|
|E4|200/200|38540|38359|181|654.8|660.7|

Comparación directa sobre 200 IDs comunes. E1: 38430 · E2: 36122 · E3: 38349 · E4: 38359

Detalle de resultados recibidos. Las configuraciones restantes están pendientes.

| ID | Método | Configuración | Entrada | Etapas (pasos) | Final validado | Estado |
|---|---|---|---:|---|---:|---|
|practice_0001|E1|1|158|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|practice_0001|E1|3|155|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|practice_0001|E1|6|155|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|practice_0001|E2|52-1|158||155|válido|
|practice_0001|E2|52-3|155||154|válido|
|practice_0001|E2|52-6|155||154|válido|
|practice_0001|E2|64-1|158||155|válido|
|practice_0001|E2|64-3|155||154|válido|
|practice_0001|E2|64-6|155||154|válido|
|practice_0001|E3|joint|155|individual 155|155|OPTIMAL|
|practice_0001|E4|repair|155||154|completo|
|practice_0002|E1|1|321|postprocess 321, optimize_a4 321, nested_windows 321|321|válido|
|practice_0002|E1|3|299|postprocess 298, optimize_a4 298, nested_windows 298|298|válido|
|practice_0002|E1|6|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|practice_0002|E2|52-1|321||280|válido|
|practice_0002|E2|52-3|299||269|válido|
|practice_0002|E2|52-6|289||276|válido|
|practice_0002|E2|64-1|321||280|válido|
|practice_0002|E2|64-3|299||267|válido|
|practice_0002|E2|64-6|289||275|válido|
|practice_0002|E3|joint|289|individual 289|288|OPTIMAL|
|practice_0002|E4|repair|289||289|completo|
|practice_0003|E1|1|328|postprocess 328, optimize_a4 328, nested_windows 328|328|válido|
|practice_0003|E1|3|314|postprocess 314, optimize_a4 314, nested_windows 314|314|válido|
|practice_0003|E1|6|308|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|practice_0003|E2|52-1|328||303|válido|
|practice_0003|E2|52-3|314||288|válido|
|practice_0003|E2|52-6|308||285|válido|
|practice_0003|E2|64-1|328||304|válido|
|practice_0003|E2|64-3|314||291|válido|
|practice_0003|E2|64-6|308||289|válido|
|practice_0003|E3|joint|308|individual 307|303|FEASIBLE|
|practice_0003|E4|repair|308||308|completo|
|practice_0004|E1|1|135|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|practice_0004|E1|3|135|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|practice_0004|E1|6|134|postprocess 134, optimize_a4 134, nested_windows 134|134|válido|
|practice_0004|E2|52-1|135||133|válido|
|practice_0004|E2|52-3|135||132|válido|
|practice_0004|E2|52-6|134||130|válido|
|practice_0004|E2|64-1|135||133|válido|
|practice_0004|E2|64-3|135||132|válido|
|practice_0004|E2|64-6|134||130|válido|
|practice_0004|E3|joint|134|individual 134|134|OPTIMAL|
|practice_0004|E4|repair|134||134|completo|
|practice_0005|E1|1|252|postprocess 252, optimize_a4 252, nested_windows 252|252|válido|
|practice_0005|E1|3|247|postprocess 247, optimize_a4 247, nested_windows 247|247|válido|
|practice_0005|E1|6|245|postprocess 245, optimize_a4 245, nested_windows 245|245|válido|
|practice_0005|E2|52-1|252||236|válido|
|practice_0005|E2|52-3|247||232|válido|
|practice_0005|E2|52-6|245||234|válido|
|practice_0005|E2|64-1|252||236|válido|
|practice_0005|E2|64-3|247||236|válido|
|practice_0005|E2|64-6|245||234|válido|
|practice_0005|E3|joint|245|individual 245|244|OPTIMAL|
|practice_0005|E4|repair|245||245|completo|
|practice_0006|E1|1|40|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|practice_0006|E1|3|40|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|practice_0006|E1|6|40|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|practice_0006|E2|52-1|40||40|válido|
|practice_0006|E2|52-3|40||40|válido|
|practice_0006|E2|52-6|40||40|válido|
|practice_0006|E2|64-1|40||40|válido|
|practice_0006|E2|64-3|40||40|válido|
|practice_0006|E2|64-6|40||40|válido|
|practice_0006|E3|joint|40|individual 40|40|OPTIMAL|
|practice_0006|E4|repair|40||38|completo|
|practice_0007|E1|1|107|postprocess 106, optimize_a4 106, nested_windows 106|106|válido|
|practice_0007|E1|3|107|postprocess 106, optimize_a4 106, nested_windows 106|106|válido|
|practice_0007|E1|6|107|postprocess 106, optimize_a4 106, nested_windows 106|106|válido|
|practice_0007|E2|52-1|107||106|válido|
|practice_0007|E2|52-3|107||106|válido|
|practice_0007|E2|52-6|107||106|válido|
|practice_0007|E2|64-1|107||106|válido|
|practice_0007|E2|64-3|107||106|válido|
|practice_0007|E2|64-6|107||106|válido|
|practice_0007|E3|joint|107|individual 107|107|OPTIMAL|
|practice_0007|E4|repair|107||105|completo|
|practice_0008|E1|1|287|postprocess 287, optimize_a4 287, nested_windows 287|287|válido|
|practice_0008|E1|3|282|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|practice_0008|E1|6|278|postprocess 278, optimize_a4 278, nested_windows 278|278|válido|
|practice_0008|E2|52-1|287||272|válido|
|practice_0008|E2|52-3|282||258|válido|
|practice_0008|E2|52-6|278||258|válido|
|practice_0008|E2|64-1|287||272|válido|
|practice_0008|E2|64-3|282||260|válido|
|practice_0008|E2|64-6|278||261|válido|
|practice_0008|E3|joint|278|individual 278|278|OPTIMAL|
|practice_0008|E4|repair|278||278|completo|
|practice_0009|E1|1|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|practice_0009|E1|3|147|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|practice_0009|E1|6|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|practice_0009|E2|52-1|148||142|válido|
|practice_0009|E2|52-3|147||139|válido|
|practice_0009|E2|52-6|146||139|válido|
|practice_0009|E2|64-1|148||141|válido|
|practice_0009|E2|64-3|147||139|válido|
|practice_0009|E2|64-6|146||139|válido|
|practice_0009|E3|joint|146|individual 146|145|OPTIMAL|
|practice_0009|E4|repair|146||144|completo|
|practice_0010|E1|1|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|practice_0010|E1|3|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|practice_0010|E1|6|150|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|practice_0010|E2|52-1|151||142|válido|
|practice_0010|E2|52-3|151||141|válido|
|practice_0010|E2|52-6|150||141|válido|
|practice_0010|E2|64-1|151||142|válido|
|practice_0010|E2|64-3|151||141|válido|
|practice_0010|E2|64-6|150||141|válido|
|practice_0010|E3|joint|150|individual 150|150|OPTIMAL|
|practice_0010|E4|repair|150||148|completo|
|practice_0011|E1|1|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|practice_0011|E1|3|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|practice_0011|E1|6|136|postprocess 136, optimize_a4 136, nested_windows 136|136|válido|
|practice_0011|E2|52-1|137||128|válido|
|practice_0011|E2|52-3|137||128|válido|
|practice_0011|E2|52-6|136||128|válido|
|practice_0011|E2|64-1|137||128|válido|
|practice_0011|E2|64-3|137||128|válido|
|practice_0011|E2|64-6|136||128|válido|
|practice_0011|E3|joint|136|individual 136|136|OPTIMAL|
|practice_0011|E4|repair|136||133|completo|
|practice_0012|E1|1|295|postprocess 294, optimize_a4 294, nested_windows 294|294|válido|
|practice_0012|E1|3|286|postprocess 286, optimize_a4 286, nested_windows 286|286|válido|
|practice_0012|E1|6|278|postprocess 278, optimize_a4 278, nested_windows 278|278|válido|
|practice_0012|E2|52-1|295||275|válido|
|practice_0012|E2|52-3|286||261|válido|
|practice_0012|E2|52-6|278||259|válido|
|practice_0012|E2|64-1|295||275|válido|
|practice_0012|E2|64-3|286||264|válido|
|practice_0012|E2|64-6|278||259|válido|
|practice_0012|E3|joint|278|individual 277|275|OPTIMAL|
|practice_0012|E4|repair|278||278|completo|
|practice_0013|E1|1|243|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|practice_0013|E1|3|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|practice_0013|E1|6|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|practice_0013|E2|52-1|243||226|válido|
|practice_0013|E2|52-3|235||223|válido|
|practice_0013|E2|52-6|238||217|válido|
|practice_0013|E2|64-1|243||231|válido|
|practice_0013|E2|64-3|235||223|válido|
|practice_0013|E2|64-6|238||215|válido|
|practice_0013|E3|joint|235|individual 235|234|OPTIMAL|
|practice_0013|E4|repair|235||235|completo|
|practice_0014|E1|1|148|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|practice_0014|E1|3|148|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|practice_0014|E1|6|148|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|practice_0014|E2|52-1|148||138|válido|
|practice_0014|E2|52-3|148||138|válido|
|practice_0014|E2|52-6|148||138|válido|
|practice_0014|E2|64-1|148||139|válido|
|practice_0014|E2|64-3|148||139|válido|
|practice_0014|E2|64-6|148||139|válido|
|practice_0014|E3|joint|148|individual 148|148|OPTIMAL|
|practice_0014|E4|repair|148||147|completo|
|practice_0015|E1|1|242|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|practice_0015|E1|3|234|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|practice_0015|E1|6|233|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|practice_0015|E2|52-1|242||233|válido|
|practice_0015|E2|52-3|234||222|válido|
|practice_0015|E2|52-6|233||205|válido|
|practice_0015|E2|64-1|242||227|válido|
|practice_0015|E2|64-3|234||221|válido|
|practice_0015|E2|64-6|233||201|válido|
|practice_0015|E3|joint|233|individual 233|232|OPTIMAL|
|practice_0015|E4|repair|233||233|completo|
|practice_0016|E1|1|268|postprocess 267, optimize_a4 267, nested_windows 267|267|válido|
|practice_0016|E1|3|255|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|practice_0016|E1|6|247|postprocess 247, optimize_a4 247, nested_windows 247|247|válido|
|practice_0016|E2|52-1|268||244|válido|
|practice_0016|E2|52-3|255||233|válido|
|practice_0016|E2|52-6|247||235|válido|
|practice_0016|E2|64-1|268||244|válido|
|practice_0016|E2|64-3|255||231|válido|
|practice_0016|E2|64-6|247||229|válido|
|practice_0016|E3|joint|247|individual 247|246|OPTIMAL|
|practice_0016|E4|repair|247||246|completo|
|practice_0017|E1|1|198|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|practice_0017|E1|3|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|practice_0017|E1|6|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|practice_0017|E2|52-1|198||176|válido|
|practice_0017|E2|52-3|192||176|válido|
|practice_0017|E2|52-6|188||176|válido|
|practice_0017|E2|64-1|198||175|válido|
|practice_0017|E2|64-3|192||175|válido|
|practice_0017|E2|64-6|188||175|válido|
|practice_0017|E3|joint|188|individual 188|187|OPTIMAL|
|practice_0017|E4|repair|188||187|completo|
|practice_0018|E1|1|397|postprocess 397, optimize_a4 397, nested_windows 397|397|válido|
|practice_0018|E1|3|381|postprocess 381, optimize_a4 381, nested_windows 381|381|válido|
|practice_0018|E1|6|377|postprocess 377, optimize_a4 377, nested_windows 377|377|válido|
|practice_0018|E2|52-1|397||373|válido|
|practice_0018|E2|52-3|381||350|válido|
|practice_0018|E2|52-6|377||342|válido|
|practice_0018|E2|64-1|397||370|válido|
|practice_0018|E2|64-3|381||353|válido|
|practice_0018|E2|64-6|377||350|válido|
|practice_0018|E3|joint|377|individual 377|375|FEASIBLE|
|practice_0018|E4|repair|377||377|completo|
|practice_0019|E1|1|229|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|practice_0019|E1|3|223|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|practice_0019|E1|6|218|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|practice_0019|E2|52-1|229||213|válido|
|practice_0019|E2|52-3|223||209|válido|
|practice_0019|E2|52-6|218||211|válido|
|practice_0019|E2|64-1|229||214|válido|
|practice_0019|E2|64-3|223||212|válido|
|practice_0019|E2|64-6|218||213|válido|
|practice_0019|E3|joint|218|individual 218|217|OPTIMAL|
|practice_0019|E4|repair|218||218|completo|
|practice_0020|E1|1|118|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|practice_0020|E1|3|118|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|practice_0020|E1|6|118|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|practice_0020|E2|52-1|118||115|válido|
|practice_0020|E2|52-3|118||114|válido|
|practice_0020|E2|52-6|118||114|válido|
|practice_0020|E2|64-1|118||113|válido|
|practice_0020|E2|64-3|118||112|válido|
|practice_0020|E2|64-6|118||112|válido|
|practice_0020|E3|joint|118|individual 118|118|OPTIMAL|
|practice_0020|E4|repair|118||113|completo|
|practice_0021|E1|1|141|postprocess 141, optimize_a4 141, nested_windows 141|141|válido|
|practice_0021|E1|3|140|postprocess 140, optimize_a4 140, nested_windows 140|140|válido|
|practice_0021|E1|6|139|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|practice_0021|E2|52-1|141||136|válido|
|practice_0021|E2|52-3|140||136|válido|
|practice_0021|E2|52-6|139||136|válido|
|practice_0021|E2|64-1|141||136|válido|
|practice_0021|E2|64-3|140||136|válido|
|practice_0021|E2|64-6|139||136|válido|
|practice_0021|E3|joint|139|individual 139|139|OPTIMAL|
|practice_0021|E4|repair|139||139|completo|
|practice_0022|E1|1|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|practice_0022|E1|3|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|practice_0022|E1|6|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|practice_0022|E2|52-1|152||143|válido|
|practice_0022|E2|52-3|152||143|válido|
|practice_0022|E2|52-6|152||143|válido|
|practice_0022|E2|64-1|152||147|válido|
|practice_0022|E2|64-3|152||147|válido|
|practice_0022|E2|64-6|152||147|válido|
|practice_0022|E3|joint|152|individual 152|152|OPTIMAL|
|practice_0022|E4|repair|152||152|completo|
|practice_0023|E1|1|218|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|practice_0023|E1|3|205|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|practice_0023|E1|6|204|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|practice_0023|E2|52-1|218||195|válido|
|practice_0023|E2|52-3|205||194|válido|
|practice_0023|E2|52-6|204||193|válido|
|practice_0023|E2|64-1|218||195|válido|
|practice_0023|E2|64-3|205||195|válido|
|practice_0023|E2|64-6|204||193|válido|
|practice_0023|E3|joint|204|individual 204|202|OPTIMAL|
|practice_0023|E4|repair|204||204|completo|
|practice_0024|E1|1|43|postprocess 42, optimize_a4 42, nested_windows 42|42|válido|
|practice_0024|E1|3|43|postprocess 42, optimize_a4 42, nested_windows 42|42|válido|
|practice_0024|E1|6|43|postprocess 42, optimize_a4 42, nested_windows 42|42|válido|
|practice_0024|E2|52-1|43||43|válido|
|practice_0024|E2|52-3|43||43|válido|
|practice_0024|E2|52-6|43||43|válido|
|practice_0024|E2|64-1|43||43|válido|
|practice_0024|E2|64-3|43||43|válido|
|practice_0024|E2|64-6|43||43|válido|
|practice_0024|E3|joint|43|individual 43|43|OPTIMAL|
|practice_0024|E4|repair|43||43|completo|
|practice_0025|E1|1|204|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|practice_0025|E1|3|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|practice_0025|E1|6|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|practice_0025|E2|52-1|204||195|válido|
|practice_0025|E2|52-3|199||190|válido|
|practice_0025|E2|52-6|199||182|válido|
|practice_0025|E2|64-1|204||196|válido|
|practice_0025|E2|64-3|199||190|válido|
|practice_0025|E2|64-6|199||182|válido|
|practice_0025|E3|joint|199|individual 199|198|OPTIMAL|
|practice_0025|E4|repair|199||199|completo|
|practice_0026|E1|1|164|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|practice_0026|E1|3|161|postprocess 161, optimize_a4 161, nested_windows 161|161|válido|
|practice_0026|E1|6|160|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|practice_0026|E2|52-1|164||159|válido|
|practice_0026|E2|52-3|161||153|válido|
|practice_0026|E2|52-6|160||153|válido|
|practice_0026|E2|64-1|164||158|válido|
|practice_0026|E2|64-3|161||154|válido|
|practice_0026|E2|64-6|160||154|válido|
|practice_0026|E3|joint|160|individual 159|158|OPTIMAL|
|practice_0026|E4|repair|160||159|completo|
|practice_0027|E1|1|228|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|practice_0027|E1|3|223|postprocess 222, optimize_a4 222, nested_windows 222|222|válido|
|practice_0027|E1|6|221|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|practice_0027|E2|52-1|228||210|válido|
|practice_0027|E2|52-3|223||206|válido|
|practice_0027|E2|52-6|221||205|válido|
|practice_0027|E2|64-1|228||214|válido|
|practice_0027|E2|64-3|223||208|válido|
|practice_0027|E2|64-6|221||206|válido|
|practice_0027|E3|joint|221|individual 220|220|OPTIMAL|
|practice_0027|E4|repair|221||221|completo|
|practice_0028|E1|1|283|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|practice_0028|E1|3|262|postprocess 262, optimize_a4 262, nested_windows 262|262|válido|
|practice_0028|E1|6|254|postprocess 254, optimize_a4 254, nested_windows 254|254|válido|
|practice_0028|E2|52-1|283||257|válido|
|practice_0028|E2|52-3|262||244|válido|
|practice_0028|E2|52-6|254||240|válido|
|practice_0028|E2|64-1|283||258|válido|
|practice_0028|E2|64-3|262||246|válido|
|practice_0028|E2|64-6|254||242|válido|
|practice_0028|E3|joint|254|individual 254|253|OPTIMAL|
|practice_0028|E4|repair|254||254|completo|
|practice_0029|E1|1|244|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|practice_0029|E1|3|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|practice_0029|E1|6|227|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|practice_0029|E2|52-1|244||226|válido|
|practice_0029|E2|52-3|233||212|válido|
|practice_0029|E2|52-6|227||214|válido|
|practice_0029|E2|64-1|244||224|válido|
|practice_0029|E2|64-3|233||212|válido|
|practice_0029|E2|64-6|227||213|válido|
|practice_0029|E3|joint|227|individual 227|226|OPTIMAL|
|practice_0029|E4|repair|227||227|completo|
|practice_0030|E1|1|144|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|practice_0030|E1|3|144|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|practice_0030|E1|6|144|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|practice_0030|E2|52-1|144||146|válido|
|practice_0030|E2|52-3|144||142|válido|
|practice_0030|E2|52-6|144||142|válido|
|practice_0030|E2|64-1|144||143|válido|
|practice_0030|E2|64-3|144||143|válido|
|practice_0030|E2|64-6|144||143|válido|
|practice_0030|E3|joint|144|individual 144|144|OPTIMAL|
|practice_0030|E4|repair|144||140|completo|
|practice_0031|E1|1|264|postprocess 262, optimize_a4 262, nested_windows 262|262|válido|
|practice_0031|E1|3|250|postprocess 250, optimize_a4 250, nested_windows 250|250|válido|
|practice_0031|E1|6|245|postprocess 244, optimize_a4 244, nested_windows 244|244|válido|
|practice_0031|E2|52-1|264||238|válido|
|practice_0031|E2|52-3|250||229|válido|
|practice_0031|E2|52-6|245||226|válido|
|practice_0031|E2|64-1|264||238|válido|
|practice_0031|E2|64-3|250||229|válido|
|practice_0031|E2|64-6|245||226|válido|
|practice_0031|E3|joint|245|individual 245|244|OPTIMAL|
|practice_0031|E4|repair|245||245|completo|
|practice_0032|E1|1|147|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|practice_0032|E1|3|147|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|practice_0032|E1|6|147|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|practice_0032|E2|52-1|147||140|válido|
|practice_0032|E2|52-3|147||140|válido|
|practice_0032|E2|52-6|147||140|válido|
|practice_0032|E2|64-1|147||140|válido|
|practice_0032|E2|64-3|147||140|válido|
|practice_0032|E2|64-6|147||140|válido|
|practice_0032|E3|joint|147|individual 147|147|OPTIMAL|
|practice_0032|E4|repair|147||146|completo|
|practice_0033|E1|1|42|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|practice_0033|E1|3|42|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|practice_0033|E1|6|42|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|practice_0033|E2|52-1|42||42|válido|
|practice_0033|E2|52-3|42||42|válido|
|practice_0033|E2|52-6|42||42|válido|
|practice_0033|E2|64-1|42||42|válido|
|practice_0033|E2|64-3|42||42|válido|
|practice_0033|E2|64-6|42||42|válido|
|practice_0033|E3|joint|42|individual 42|42|OPTIMAL|
|practice_0033|E4|repair|42||42|completo|
|practice_0034|E1|1|372|postprocess 371, optimize_a4 371, nested_windows 371|371|válido|
|practice_0034|E1|3|355|postprocess 355, optimize_a4 355, nested_windows 355|355|válido|
|practice_0034|E1|6|341|postprocess 341, optimize_a4 341, nested_windows 341|341|válido|
|practice_0034|E2|52-1|372||338|válido|
|practice_0034|E2|52-3|355||322|válido|
|practice_0034|E2|52-6|341||319|válido|
|practice_0034|E2|64-1|372||337|válido|
|practice_0034|E2|64-3|355||325|válido|
|practice_0034|E2|64-6|341||318|válido|
|practice_0034|E3|joint|341|individual 341|341|FEASIBLE|
|practice_0034|E4|repair|341||341|completo|
|practice_0035|E1|1|301|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|practice_0035|E1|3|286|postprocess 286, optimize_a4 286, nested_windows 286|286|válido|
|practice_0035|E1|6|282|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|practice_0035|E2|52-1|301||273|válido|
|practice_0035|E2|52-3|286||265|válido|
|practice_0035|E2|52-6|282||265|válido|
|practice_0035|E2|64-1|301||275|válido|
|practice_0035|E2|64-3|286||265|válido|
|practice_0035|E2|64-6|282||264|válido|
|practice_0035|E3|joint|282|individual 282|282|OPTIMAL|
|practice_0035|E4|repair|282||282|completo|
|practice_0036|E1|1|139|postprocess 139, optimize_a4 138, nested_windows 138|138|válido|
|practice_0036|E1|3|137|postprocess 137, optimize_a4 136, nested_windows 136|136|válido|
|practice_0036|E1|6|137|postprocess 137, optimize_a4 136, nested_windows 136|136|válido|
|practice_0036|E2|52-1|139||133|válido|
|practice_0036|E2|52-3|137||133|válido|
|practice_0036|E2|52-6|137||133|válido|
|practice_0036|E2|64-1|139||134|válido|
|practice_0036|E2|64-3|137||133|válido|
|practice_0036|E2|64-6|137||133|válido|
|practice_0036|E3|joint|137|individual 137|137|OPTIMAL|
|practice_0036|E4|repair|137||136|completo|
|practice_0037|E1|1|338|postprocess 335, optimize_a4 335, nested_windows 335|335|válido|
|practice_0037|E1|3|322|postprocess 322, optimize_a4 322, nested_windows 322|322|válido|
|practice_0037|E1|6|313|postprocess 312, optimize_a4 312, nested_windows 312|312|válido|
|practice_0037|E2|52-1|338||313|válido|
|practice_0037|E2|52-3|322||301|válido|
|practice_0037|E2|52-6|313||293|válido|
|practice_0037|E2|64-1|338||315|válido|
|practice_0037|E2|64-3|322||301|válido|
|practice_0037|E2|64-6|313||294|válido|
|practice_0037|E3|joint|313|individual 313|312|FEASIBLE|
|practice_0037|E4|repair|313||313|completo|
|practice_0038|E1|1|45|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|practice_0038|E1|3|45|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|practice_0038|E1|6|45|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|practice_0038|E2|52-1|45||45|válido|
|practice_0038|E2|52-3|45||45|válido|
|practice_0038|E2|52-6|45||45|válido|
|practice_0038|E2|64-1|45||45|válido|
|practice_0038|E2|64-3|45||45|válido|
|practice_0038|E2|64-6|45||45|válido|
|practice_0038|E3|joint|45|individual 45|45|OPTIMAL|
|practice_0038|E4|repair|45||44|completo|
|practice_0039|E1|1|9|postprocess 8, optimize_a4 8, nested_windows 8|8|válido|
|practice_0039|E1|3|9|postprocess 8, optimize_a4 8, nested_windows 8|8|válido|
|practice_0039|E1|6|9|postprocess 8, optimize_a4 8, nested_windows 8|8|válido|
|practice_0039|E2|52-1|9||9|válido|
|practice_0039|E2|52-3|9||9|válido|
|practice_0039|E2|52-6|9||9|válido|
|practice_0039|E2|64-1|9||9|válido|
|practice_0039|E2|64-3|9||9|válido|
|practice_0039|E2|64-6|9||9|válido|
|practice_0039|E3|joint|6|individual 6|6|OPTIMAL|
|practice_0039|E4|repair|6||6|completo|
|practice_0040|E1|1|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|practice_0040|E1|3|213|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|practice_0040|E1|6|210|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|practice_0040|E2|52-1|223||206|válido|
|practice_0040|E2|52-3|213||199|válido|
|practice_0040|E2|52-6|210||203|válido|
|practice_0040|E2|64-1|223||208|válido|
|practice_0040|E2|64-3|213||198|válido|
|practice_0040|E2|64-6|210||200|válido|
|practice_0040|E3|joint|210|individual 210|209|OPTIMAL|
|practice_0040|E4|repair|210||210|completo|
|practice_0041|E1|1|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|practice_0041|E1|3|197|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|practice_0041|E1|6|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|practice_0041|E2|52-1|206||189|válido|
|practice_0041|E2|52-3|197||192|válido|
|practice_0041|E2|52-6|197||188|válido|
|practice_0041|E2|64-1|206||190|válido|
|practice_0041|E2|64-3|197||191|válido|
|practice_0041|E2|64-6|197||196|válido|
|practice_0041|E3|joint|197|individual 196|196|OPTIMAL|
|practice_0041|E4|repair|197||197|completo|
|practice_0042|E1|1|204|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|practice_0042|E1|3|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|practice_0042|E1|6|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|practice_0042|E2|52-1|204||190|válido|
|practice_0042|E2|52-3|199||189|válido|
|practice_0042|E2|52-6|197||185|válido|
|practice_0042|E2|64-1|204||189|válido|
|practice_0042|E2|64-3|199||189|válido|
|practice_0042|E2|64-6|197||186|válido|
|practice_0042|E3|joint|197|individual 197|197|OPTIMAL|
|practice_0042|E4|repair|197||197|completo|
|practice_0043|E1|1|134|postprocess 132, optimize_a4 132, nested_windows 132|132|válido|
|practice_0043|E1|3|134|postprocess 132, optimize_a4 132, nested_windows 132|132|válido|
|practice_0043|E1|6|133|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|practice_0043|E2|52-1|134||127|válido|
|practice_0043|E2|52-3|134||127|válido|
|practice_0043|E2|52-6|133||127|válido|
|practice_0043|E2|64-1|134||127|válido|
|practice_0043|E2|64-3|134||127|válido|
|practice_0043|E2|64-6|133||127|válido|
|practice_0043|E3|joint|133|individual 133|133|OPTIMAL|
|practice_0043|E4|repair|133||130|completo|
|practice_0044|E1|1|259|postprocess 259, optimize_a4 259, nested_windows 259|259|válido|
|practice_0044|E1|3|246|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|practice_0044|E1|6|246|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|practice_0044|E2|52-1|259||241|válido|
|practice_0044|E2|52-3|246||230|válido|
|practice_0044|E2|52-6|246||233|válido|
|practice_0044|E2|64-1|259||241|válido|
|practice_0044|E2|64-3|246||232|válido|
|practice_0044|E2|64-6|246||232|válido|
|practice_0044|E3|joint|246|individual 245|243|OPTIMAL|
|practice_0044|E4|repair|246||245|completo|
|practice_0045|E1|1|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|practice_0045|E1|3|158|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|practice_0045|E1|6|156|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|practice_0045|E2|52-1|162||151|válido|
|practice_0045|E2|52-3|158||150|válido|
|practice_0045|E2|52-6|156||150|válido|
|practice_0045|E2|64-1|162||152|válido|
|practice_0045|E2|64-3|158||150|válido|
|practice_0045|E2|64-6|156||150|válido|
|practice_0045|E3|joint|156|individual 155|155|OPTIMAL|
|practice_0045|E4|repair|156||155|completo|
|practice_0046|E1|1|312|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|practice_0046|E1|3|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|practice_0046|E1|6|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|practice_0046|E2|52-1|312||280|válido|
|practice_0046|E2|52-3|303||276|válido|
|practice_0046|E2|52-6|289||271|válido|
|practice_0046|E2|64-1|312||283|válido|
|practice_0046|E2|64-3|303||276|válido|
|practice_0046|E2|64-6|289||271|válido|
|practice_0046|E3|joint|289|individual 289|289|FEASIBLE|
|practice_0046|E4|repair|289||289|completo|
|practice_0047|E1|1|5|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0047|E1|3|5|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0047|E1|6|5|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0047|E2|52-1|5||5|válido|
|practice_0047|E2|52-3|5||5|válido|
|practice_0047|E2|52-6|5||5|válido|
|practice_0047|E2|64-1|5||5|válido|
|practice_0047|E2|64-3|5||5|válido|
|practice_0047|E2|64-6|5||5|válido|
|practice_0047|E3|joint|2|individual 2|2|OPTIMAL|
|practice_0047|E4|repair|2||2|completo|
|practice_0048|E1|1|131|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|practice_0048|E1|3|131|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|practice_0048|E1|6|131|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|practice_0048|E2|52-1|131||125|válido|
|practice_0048|E2|52-3|131||125|válido|
|practice_0048|E2|52-6|131||124|válido|
|practice_0048|E2|64-1|131||127|válido|
|practice_0048|E2|64-3|131||127|válido|
|practice_0048|E2|64-6|131||127|válido|
|practice_0048|E3|joint|131|individual 131|131|OPTIMAL|
|practice_0048|E4|repair|131||126|completo|
|practice_0049|E1|1|137|postprocess 136, optimize_a4 136, nested_windows 136|136|válido|
|practice_0049|E1|3|137|postprocess 136, optimize_a4 136, nested_windows 136|136|válido|
|practice_0049|E1|6|136|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|practice_0049|E2|52-1|137||131|válido|
|practice_0049|E2|52-3|137||131|válido|
|practice_0049|E2|52-6|136||131|válido|
|practice_0049|E2|64-1|137||131|válido|
|practice_0049|E2|64-3|137||131|válido|
|practice_0049|E2|64-6|136||131|válido|
|practice_0049|E3|joint|136|individual 136|136|OPTIMAL|
|practice_0049|E4|repair|136||135|completo|
|practice_0050|E1|1|5|postprocess 3, optimize_a4 3, nested_windows 3|3|válido|
|practice_0050|E1|3|5|postprocess 3, optimize_a4 3, nested_windows 3|3|válido|
|practice_0050|E1|6|5|postprocess 3, optimize_a4 3, nested_windows 3|3|válido|
|practice_0050|E2|52-1|5||5|válido|
|practice_0050|E2|52-3|5||5|válido|
|practice_0050|E2|52-6|5||5|válido|
|practice_0050|E2|64-1|5||5|válido|
|practice_0050|E2|64-3|5||5|válido|
|practice_0050|E2|64-6|5||5|válido|
|practice_0050|E3|joint|3|individual 3|3|OPTIMAL|
|practice_0050|E4|repair|3||3|completo|
|practice_0051|E1|1|7|postprocess 7, optimize_a4 5, nested_windows 5|5|válido|
|practice_0051|E1|3|7|postprocess 7, optimize_a4 5, nested_windows 5|5|válido|
|practice_0051|E1|6|7|postprocess 7, optimize_a4 5, nested_windows 5|5|válido|
|practice_0051|E2|52-1|7||7|válido|
|practice_0051|E2|52-3|7||7|válido|
|practice_0051|E2|52-6|7||7|válido|
|practice_0051|E2|64-1|7||7|válido|
|practice_0051|E2|64-3|7||7|válido|
|practice_0051|E2|64-6|7||7|válido|
|practice_0051|E3|joint|5|individual 5|5|OPTIMAL|
|practice_0051|E4|repair|5||5|completo|
|practice_0052|E1|1|230|postprocess 230, optimize_a4 230, nested_windows 230|230|válido|
|practice_0052|E1|3|228|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|practice_0052|E1|6|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|practice_0052|E2|52-1|230||216|válido|
|practice_0052|E2|52-3|228||211|válido|
|practice_0052|E2|52-6|226||214|válido|
|practice_0052|E2|64-1|230||215|válido|
|practice_0052|E2|64-3|228||209|válido|
|practice_0052|E2|64-6|226||213|válido|
|practice_0052|E3|joint|226|individual 226|223|OPTIMAL|
|practice_0052|E4|repair|226||226|completo|
|practice_0053|E1|1|306|postprocess 306, optimize_a4 306, nested_windows 306|306|válido|
|practice_0053|E1|3|294|postprocess 293, optimize_a4 293, nested_windows 293|293|válido|
|practice_0053|E1|6|286|postprocess 285, optimize_a4 285, nested_windows 285|285|válido|
|practice_0053|E2|52-1|306||275|válido|
|practice_0053|E2|52-3|294||267|válido|
|practice_0053|E2|52-6|286||267|válido|
|practice_0053|E2|64-1|306||275|válido|
|practice_0053|E2|64-3|294||271|válido|
|practice_0053|E2|64-6|286||268|válido|
|practice_0053|E3|joint|286|individual 285|281|OPTIMAL|
|practice_0053|E4|repair|286||286|completo|
|practice_0054|E1|1|413|postprocess 411, optimize_a4 411, nested_windows 411|411|válido|
|practice_0054|E1|3|385|postprocess 385, optimize_a4 385, nested_windows 385|385|válido|
|practice_0054|E1|6|377|postprocess 377, optimize_a4 377, nested_windows 377|377|válido|
|practice_0054|E2|52-1|413||375|válido|
|practice_0054|E2|52-3|385||348|válido|
|practice_0054|E2|52-6|377||346|válido|
|practice_0054|E2|64-1|413||382|válido|
|practice_0054|E2|64-3|385||352|válido|
|practice_0054|E2|64-6|377||346|válido|
|practice_0054|E3|joint|377|individual 376|375|FEASIBLE|
|practice_0054|E4|repair|377||377|completo|
|practice_0055|E1|1|407|postprocess 404, optimize_a4 404, nested_windows 404|404|válido|
|practice_0055|E1|3|387|postprocess 387, optimize_a4 387, nested_windows 387|387|válido|
|practice_0055|E1|6|377|postprocess 377, optimize_a4 377, nested_windows 377|377|válido|
|practice_0055|E2|52-1|407||366|válido|
|practice_0055|E2|52-3|387||356|válido|
|practice_0055|E2|52-6|377||349|válido|
|practice_0055|E2|64-1|407||369|válido|
|practice_0055|E2|64-3|387||358|válido|
|practice_0055|E2|64-6|377||349|válido|
|practice_0055|E3|joint|377|individual 376|371|FEASIBLE|
|practice_0055|E4|repair|377||377|completo|
|practice_0056|E1|1|7|postprocess 7, optimize_a4 6, nested_windows 6|6|válido|
|practice_0056|E1|3|7|postprocess 7, optimize_a4 6, nested_windows 6|6|válido|
|practice_0056|E1|6|7|postprocess 7, optimize_a4 6, nested_windows 6|6|válido|
|practice_0056|E2|52-1|7||7|válido|
|practice_0056|E2|52-3|7||7|válido|
|practice_0056|E2|52-6|7||7|válido|
|practice_0056|E2|64-1|7||7|válido|
|practice_0056|E2|64-3|7||7|válido|
|practice_0056|E2|64-6|7||7|válido|
|practice_0056|E3|joint|4|individual 4|4|OPTIMAL|
|practice_0056|E4|repair|4||4|completo|
|practice_0057|E1|1|295|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|practice_0057|E1|3|286|postprocess 286, optimize_a4 286, nested_windows 286|286|válido|
|practice_0057|E1|6|285|postprocess 280, optimize_a4 280, nested_windows 280|280|válido|
|practice_0057|E2|52-1|295||269|válido|
|practice_0057|E2|52-3|286||261|válido|
|practice_0057|E2|52-6|285||256|válido|
|practice_0057|E2|64-1|295||270|válido|
|practice_0057|E2|64-3|286||262|válido|
|practice_0057|E2|64-6|285||256|válido|
|practice_0057|E3|joint|285|individual 279|277|FEASIBLE|
|practice_0057|E4|repair|285||285|completo|
|practice_0058|E1|1|310|postprocess 310, optimize_a4 310, nested_windows 310|310|válido|
|practice_0058|E1|3|294|postprocess 294, optimize_a4 294, nested_windows 294|294|válido|
|practice_0058|E1|6|292|postprocess 292, optimize_a4 292, nested_windows 292|292|válido|
|practice_0058|E2|52-1|310||291|válido|
|practice_0058|E2|52-3|294||276|válido|
|practice_0058|E2|52-6|292||267|válido|
|practice_0058|E2|64-1|310||292|válido|
|practice_0058|E2|64-3|294||276|válido|
|practice_0058|E2|64-6|292||273|válido|
|practice_0058|E3|joint|292|individual 292|291|OPTIMAL|
|practice_0058|E4|repair|292||292|completo|
|practice_0059|E1|1|157|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|practice_0059|E1|3|156|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|practice_0059|E1|6|156|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|practice_0059|E2|52-1|157||153|válido|
|practice_0059|E2|52-3|156||151|válido|
|practice_0059|E2|52-6|156||151|válido|
|practice_0059|E2|64-1|157||153|válido|
|practice_0059|E2|64-3|156||150|válido|
|practice_0059|E2|64-6|156||150|válido|
|practice_0059|E3|joint|156|individual 156|156|OPTIMAL|
|practice_0059|E4|repair|156||154|completo|
|practice_0060|E1|1|301|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|practice_0060|E1|3|290|postprocess 290, optimize_a4 290, nested_windows 290|290|válido|
|practice_0060|E1|6|290|postprocess 290, optimize_a4 290, nested_windows 290|290|válido|
|practice_0060|E2|52-1|301||277|válido|
|practice_0060|E2|52-3|290||268|válido|
|practice_0060|E2|52-6|290||265|válido|
|practice_0060|E2|64-1|301||281|válido|
|practice_0060|E2|64-3|290||268|válido|
|practice_0060|E2|64-6|290||268|válido|
|practice_0060|E3|joint|290|individual 289|286|FEASIBLE|
|practice_0060|E4|repair|290||290|completo|
|practice_0061|E1|1|328|postprocess 328, optimize_a4 328, nested_windows 328|328|válido|
|practice_0061|E1|3|316|postprocess 315, optimize_a4 315, nested_windows 315|315|válido|
|practice_0061|E1|6|297|postprocess 297, optimize_a4 297, nested_windows 297|297|válido|
|practice_0061|E2|52-1|328||298|válido|
|practice_0061|E2|52-3|316||291|válido|
|practice_0061|E2|52-6|297||282|válido|
|practice_0061|E2|64-1|328||300|válido|
|practice_0061|E2|64-3|316||287|válido|
|practice_0061|E2|64-6|297||282|válido|
|practice_0061|E3|joint|297|individual 297|297|OPTIMAL|
|practice_0061|E4|repair|297||297|completo|
|practice_0062|E1|1|358|postprocess 358, optimize_a4 358, nested_windows 358|358|válido|
|practice_0062|E1|3|349|postprocess 349, optimize_a4 349, nested_windows 349|349|válido|
|practice_0062|E1|6|336|postprocess 336, optimize_a4 336, nested_windows 336|336|válido|
|practice_0062|E2|52-1|358||342|válido|
|practice_0062|E2|52-3|349||324|válido|
|practice_0062|E2|52-6|336||313|válido|
|practice_0062|E2|64-1|358||339|válido|
|practice_0062|E2|64-3|349||325|válido|
|practice_0062|E2|64-6|336||314|válido|
|practice_0062|E3|joint|336|individual 336|334|OPTIMAL|
|practice_0062|E4|repair|336||336|completo|
|practice_0063|E1|1|135|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|practice_0063|E1|3|135|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|practice_0063|E1|6|135|postprocess 134, optimize_a4 134, nested_windows 134|134|válido|
|practice_0063|E2|52-1|135||131|válido|
|practice_0063|E2|52-3|135||131|válido|
|practice_0063|E2|52-6|135||131|válido|
|practice_0063|E2|64-1|135||131|válido|
|practice_0063|E2|64-3|135||131|válido|
|practice_0063|E2|64-6|135||131|válido|
|practice_0063|E3|joint|135|individual 135|135|OPTIMAL|
|practice_0063|E4|repair|135||132|completo|
|practice_0064|E1|1|301|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|practice_0064|E1|3|286|postprocess 285, optimize_a4 285, nested_windows 285|285|válido|
|practice_0064|E1|6|283|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|practice_0064|E2|52-1|301||274|válido|
|practice_0064|E2|52-3|286||264|válido|
|practice_0064|E2|52-6|283||255|válido|
|practice_0064|E2|64-1|301||275|válido|
|practice_0064|E2|64-3|286||263|válido|
|practice_0064|E2|64-6|283||259|válido|
|practice_0064|E3|joint|283|individual 283|280|OPTIMAL|
|practice_0064|E4|repair|283||282|completo|
|practice_0065|E1|1|273|postprocess 273, optimize_a4 273, nested_windows 273|273|válido|
|practice_0065|E1|3|262|postprocess 262, optimize_a4 262, nested_windows 262|262|válido|
|practice_0065|E1|6|257|postprocess 257, optimize_a4 257, nested_windows 257|257|válido|
|practice_0065|E2|52-1|273||249|válido|
|practice_0065|E2|52-3|262||241|válido|
|practice_0065|E2|52-6|257||236|válido|
|practice_0065|E2|64-1|273||246|válido|
|practice_0065|E2|64-3|262||244|válido|
|practice_0065|E2|64-6|257||242|válido|
|practice_0065|E3|joint|257|individual 256|256|OPTIMAL|
|practice_0065|E4|repair|257||257|completo|
|practice_0066|E1|1|112|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|practice_0066|E1|3|112|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|practice_0066|E1|6|112|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|practice_0066|E2|52-1|112||112|válido|
|practice_0066|E2|52-3|112||112|válido|
|practice_0066|E2|52-6|112||112|válido|
|practice_0066|E2|64-1|112||111|válido|
|practice_0066|E2|64-3|112||111|válido|
|practice_0066|E2|64-6|112||111|válido|
|practice_0066|E3|joint|112|individual 112|112|OPTIMAL|
|practice_0066|E4|repair|112||109|completo|
|practice_0067|E1|1|191|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|practice_0067|E1|3|189|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|practice_0067|E1|6|189|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|practice_0067|E2|52-1|191||173|válido|
|practice_0067|E2|52-3|189||173|válido|
|practice_0067|E2|52-6|189||173|válido|
|practice_0067|E2|64-1|191||174|válido|
|practice_0067|E2|64-3|189||175|válido|
|practice_0067|E2|64-6|189||175|válido|
|practice_0067|E3|joint|189|individual 189|184|OPTIMAL|
|practice_0067|E4|repair|189||186|completo|
|practice_0068|E1|1|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|practice_0068|E1|3|214|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|practice_0068|E1|6|212|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|practice_0068|E2|52-1|215||206|válido|
|practice_0068|E2|52-3|214||197|válido|
|practice_0068|E2|52-6|212||198|válido|
|practice_0068|E2|64-1|215||207|válido|
|practice_0068|E2|64-3|214||198|válido|
|practice_0068|E2|64-6|212||196|válido|
|practice_0068|E3|joint|212|individual 212|212|OPTIMAL|
|practice_0068|E4|repair|212||212|completo|
|practice_0069|E1|1|240|postprocess 240, optimize_a4 240, nested_windows 240|240|válido|
|practice_0069|E1|3|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|practice_0069|E1|6|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|practice_0069|E2|52-1|240||229|válido|
|practice_0069|E2|52-3|232||220|válido|
|practice_0069|E2|52-6|233||220|válido|
|practice_0069|E2|64-1|240||229|válido|
|practice_0069|E2|64-3|232||222|válido|
|practice_0069|E2|64-6|233||221|válido|
|practice_0069|E3|joint|232|individual 232|231|OPTIMAL|
|practice_0069|E4|repair|232||232|completo|
|practice_0070|E1|1|310|postprocess 309, optimize_a4 309, nested_windows 309|309|válido|
|practice_0070|E1|3|288|postprocess 288, optimize_a4 288, nested_windows 288|288|válido|
|practice_0070|E1|6|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|practice_0070|E2|52-1|310||287|válido|
|practice_0070|E2|52-3|288||275|válido|
|practice_0070|E2|52-6|289||271|válido|
|practice_0070|E2|64-1|310||288|válido|
|practice_0070|E2|64-3|288||276|válido|
|practice_0070|E2|64-6|289||270|válido|
|practice_0070|E3|joint|288|individual 288|287|FEASIBLE|
|practice_0070|E4|repair|288||288|completo|
|practice_0071|E1|1|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|practice_0071|E1|3|185|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|practice_0071|E1|6|185|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|practice_0071|E2|52-1|188||173|válido|
|practice_0071|E2|52-3|185||172|válido|
|practice_0071|E2|52-6|185||169|válido|
|practice_0071|E2|64-1|188||176|válido|
|practice_0071|E2|64-3|185||173|válido|
|practice_0071|E2|64-6|185||170|válido|
|practice_0071|E3|joint|185|individual 184|184|OPTIMAL|
|practice_0071|E4|repair|185||183|completo|
|practice_0072|E1|1|8|postprocess 6, optimize_a4 6, nested_windows 6|6|válido|
|practice_0072|E1|3|8|postprocess 6, optimize_a4 6, nested_windows 6|6|válido|
|practice_0072|E1|6|8|postprocess 6, optimize_a4 6, nested_windows 6|6|válido|
|practice_0072|E2|52-1|8||8|válido|
|practice_0072|E2|52-3|8||8|válido|
|practice_0072|E2|52-6|8||8|válido|
|practice_0072|E2|64-1|8||8|válido|
|practice_0072|E2|64-3|8||8|válido|
|practice_0072|E2|64-6|8||8|válido|
|practice_0072|E3|joint|6|individual 6|6|OPTIMAL|
|practice_0072|E4|repair|6||5|completo|
|practice_0073|E1|1|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|practice_0073|E1|3|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|practice_0073|E1|6|166|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|practice_0073|E2|52-1|169||160|válido|
|practice_0073|E2|52-3|166||159|válido|
|practice_0073|E2|52-6|166||159|válido|
|practice_0073|E2|64-1|169||160|válido|
|practice_0073|E2|64-3|166||159|válido|
|practice_0073|E2|64-6|166||159|válido|
|practice_0073|E3|joint|166|individual 165|165|OPTIMAL|
|practice_0073|E4|repair|166||165|completo|
|practice_0074|E1|1|260|postprocess 260, optimize_a4 260, nested_windows 260|260|válido|
|practice_0074|E1|3|243|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|practice_0074|E1|6|238|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|practice_0074|E2|52-1|260||237|válido|
|practice_0074|E2|52-3|243||225|válido|
|practice_0074|E2|52-6|238||226|válido|
|practice_0074|E2|64-1|260||240|válido|
|practice_0074|E2|64-3|243||224|válido|
|practice_0074|E2|64-6|238||220|válido|
|practice_0074|E3|joint|238|individual 238|236|OPTIMAL|
|practice_0074|E4|repair|238||238|completo|
|practice_0075|E1|1|223|postprocess 222, optimize_a4 222, nested_windows 222|222|válido|
|practice_0075|E1|3|212|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|practice_0075|E1|6|214|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|practice_0075|E2|52-1|223||207|válido|
|practice_0075|E2|52-3|212||204|válido|
|practice_0075|E2|52-6|214||199|válido|
|practice_0075|E2|64-1|223||210|válido|
|practice_0075|E2|64-3|212||205|válido|
|practice_0075|E2|64-6|214||198|válido|
|practice_0075|E3|joint|212|individual 212|210|OPTIMAL|
|practice_0075|E4|repair|212||211|completo|
|practice_0076|E1|1|377|postprocess 377, optimize_a4 377, nested_windows 377|377|válido|
|practice_0076|E1|3|364|postprocess 364, optimize_a4 364, nested_windows 364|364|válido|
|practice_0076|E1|6|361|postprocess 361, optimize_a4 361, nested_windows 361|361|válido|
|practice_0076|E2|52-1|377||349|válido|
|practice_0076|E2|52-3|364||332|válido|
|practice_0076|E2|52-6|361||337|válido|
|practice_0076|E2|64-1|377||352|válido|
|practice_0076|E2|64-3|364||363|válido|
|practice_0076|E2|64-6|361||338|válido|
|practice_0076|E3|joint|361|individual 360|355|FEASIBLE|
|practice_0076|E4|repair|361||361|completo|
|practice_0077|E1|1|119|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|practice_0077|E1|3|119|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|practice_0077|E1|6|119|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|practice_0077|E2|52-1|119||115|válido|
|practice_0077|E2|52-3|119||115|válido|
|practice_0077|E2|52-6|119||115|válido|
|practice_0077|E2|64-1|119||114|válido|
|practice_0077|E2|64-3|119||114|válido|
|practice_0077|E2|64-6|119||114|válido|
|practice_0077|E3|joint|119|individual 119|119|OPTIMAL|
|practice_0077|E4|repair|119||116|completo|
|practice_0078|E1|1|331|postprocess 330, optimize_a4 330, nested_windows 330|330|válido|
|practice_0078|E1|3|317|postprocess 317, optimize_a4 317, nested_windows 317|317|válido|
|practice_0078|E1|6|369|postprocess 355, optimize_a4 355, nested_windows 355|355|válido|
|practice_0078|E2|52-1|331||300|válido|
|practice_0078|E2|52-3|317||291|válido|
|practice_0078|E2|52-6|369||284|válido|
|practice_0078|E2|64-1|331||305|válido|
|practice_0078|E2|64-3|317||293|válido|
|practice_0078|E2|64-6|369||286|válido|
|practice_0078|E3|joint|317|individual 311|311|rechazado por registros|
|practice_0078|E4|repair|317||317|completo|
|practice_0079|E1|1|142|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|practice_0079|E1|3|141|postprocess 141, optimize_a4 141, nested_windows 141|141|válido|
|practice_0079|E1|6|141|postprocess 141, optimize_a4 141, nested_windows 141|141|válido|
|practice_0079|E2|52-1|142||138|válido|
|practice_0079|E2|52-3|141||138|válido|
|practice_0079|E2|52-6|141||138|válido|
|practice_0079|E2|64-1|142||138|válido|
|practice_0079|E2|64-3|141||137|válido|
|practice_0079|E2|64-6|141||137|válido|
|practice_0079|E3|joint|141|individual 141|141|OPTIMAL|
|practice_0079|E4|repair|141||139|completo|
|practice_0080|E1|1|125|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|practice_0080|E1|3|125|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|practice_0080|E1|6|125|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|practice_0080|E2|52-1|125||124|válido|
|practice_0080|E2|52-3|125||124|válido|
|practice_0080|E2|52-6|125||124|válido|
|practice_0080|E2|64-1|125||123|válido|
|practice_0080|E2|64-3|125||123|válido|
|practice_0080|E2|64-6|125||123|válido|
|practice_0080|E3|joint|125|individual 125|125|OPTIMAL|
|practice_0080|E4|repair|125||120|completo|
|practice_0081|E1|1|218|postprocess 217, optimize_a4 217, nested_windows 217|217|válido|
|practice_0081|E1|3|221|postprocess 217, optimize_a4 217, nested_windows 217|217|válido|
|practice_0081|E1|6|214|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|practice_0081|E2|52-1|218||203|válido|
|practice_0081|E2|52-3|221||197|válido|
|practice_0081|E2|52-6|214||203|válido|
|practice_0081|E2|64-1|218||201|válido|
|practice_0081|E2|64-3|221||196|válido|
|practice_0081|E2|64-6|214||197|válido|
|practice_0081|E3|joint|214|individual 214|213|OPTIMAL|
|practice_0081|E4|repair|214||210|completo|
|practice_0082|E1|1|268|postprocess 268, optimize_a4 268, nested_windows 268|268|válido|
|practice_0082|E1|3|256|postprocess 256, optimize_a4 256, nested_windows 256|256|válido|
|practice_0082|E1|6|253|postprocess 252, optimize_a4 252, nested_windows 252|252|válido|
|practice_0082|E2|52-1|268||240|válido|
|practice_0082|E2|52-3|256||243|válido|
|practice_0082|E2|52-6|253||243|válido|
|practice_0082|E2|64-1|268||244|válido|
|practice_0082|E2|64-3|256||243|válido|
|practice_0082|E2|64-6|253||239|válido|
|practice_0082|E3|joint|253|individual 253|250|OPTIMAL|
|practice_0082|E4|repair|253||253|completo|
|practice_0083|E1|1|157|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|practice_0083|E1|3|165|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|practice_0083|E1|6|158|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|practice_0083|E2|52-1|157||154|válido|
|practice_0083|E2|52-3|165||152|válido|
|practice_0083|E2|52-6|158||155|válido|
|practice_0083|E2|64-1|157||154|válido|
|practice_0083|E2|64-3|165||152|válido|
|practice_0083|E2|64-6|158||154|válido|
|practice_0083|E3|joint|157|individual 157|156|OPTIMAL|
|practice_0083|E4|repair|157||155|completo|
|practice_0084|E1|1|180|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|practice_0084|E1|3|173|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|practice_0084|E1|6|170|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|practice_0084|E2|52-1|180||163|válido|
|practice_0084|E2|52-3|173||163|válido|
|practice_0084|E2|52-6|170||168|válido|
|practice_0084|E2|64-1|180||164|válido|
|practice_0084|E2|64-3|173||162|válido|
|practice_0084|E2|64-6|170||169|válido|
|practice_0084|E3|joint|170|individual 170|170|OPTIMAL|
|practice_0084|E4|repair|170||170|completo|
|practice_0085|E1|1|343|postprocess 342, optimize_a4 342, nested_windows 342|342|válido|
|practice_0085|E1|3|331|postprocess 330, optimize_a4 330, nested_windows 330|330|válido|
|practice_0085|E1|6|318|postprocess 318, optimize_a4 318, nested_windows 318|318|válido|
|practice_0085|E2|52-1|343||309|válido|
|practice_0085|E2|52-3|331||313|válido|
|practice_0085|E2|52-6|318||304|válido|
|practice_0085|E2|64-1|343||354|válido|
|practice_0085|E2|64-3|331||316|válido|
|practice_0085|E2|64-6|318||304|válido|
|practice_0085|E3|joint|318|individual 317|314|FEASIBLE|
|practice_0085|E4|repair|318||318|completo|
|practice_0086|E1|1|168|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|practice_0086|E1|3|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|practice_0086|E1|6|163|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|practice_0086|E2|52-1|168||160|válido|
|practice_0086|E2|52-3|166||157|válido|
|practice_0086|E2|52-6|163||153|válido|
|practice_0086|E2|64-1|168||158|válido|
|practice_0086|E2|64-3|166||154|válido|
|practice_0086|E2|64-6|163||151|válido|
|practice_0086|E3|joint|163|individual 163|163|OPTIMAL|
|practice_0086|E4|repair|163||162|completo|
|practice_0087|E1|1|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|practice_0087|E1|3|280|postprocess 280, optimize_a4 280, nested_windows 280|280|válido|
|practice_0087|E1|6|280|postprocess 280, optimize_a4 280, nested_windows 280|280|válido|
|practice_0087|E2|52-1|303||264|válido|
|practice_0087|E2|52-3|280||260|válido|
|practice_0087|E2|52-6|280||255|válido|
|practice_0087|E2|64-1|303||267|válido|
|practice_0087|E2|64-3|280||262|válido|
|practice_0087|E2|64-6|280||256|válido|
|practice_0087|E3|joint|280|individual 278|276|OPTIMAL|
|practice_0087|E4|repair|280||280|completo|
|practice_0088|E1|1|276|postprocess 275, optimize_a4 275, nested_windows 275|275|válido|
|practice_0088|E1|3|260|postprocess 260, optimize_a4 260, nested_windows 260|260|válido|
|practice_0088|E1|6|257|postprocess 257, optimize_a4 257, nested_windows 257|257|válido|
|practice_0088|E2|52-1|276||248|válido|
|practice_0088|E2|52-3|260||234|válido|
|practice_0088|E2|52-6|257||233|válido|
|practice_0088|E2|64-1|276||250|válido|
|practice_0088|E2|64-3|260||240|válido|
|practice_0088|E2|64-6|257||238|válido|
|practice_0088|E3|joint|257|individual 257|257|OPTIMAL|
|practice_0088|E4|repair|257||257|completo|
|practice_0089|E1|1|204|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|practice_0089|E1|3|198|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|practice_0089|E1|6|191|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|practice_0089|E2|52-1|204||180|válido|
|practice_0089|E2|52-3|198||186|válido|
|practice_0089|E2|52-6|191||165|válido|
|practice_0089|E2|64-1|204||183|válido|
|practice_0089|E2|64-3|198||187|válido|
|practice_0089|E2|64-6|191||165|válido|
|practice_0089|E3|joint|191|individual 191|191|OPTIMAL|
|practice_0089|E4|repair|191||191|completo|
|practice_0090|E1|1|169|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|practice_0090|E1|3|168|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|practice_0090|E1|6|167|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|practice_0090|E2|52-1|169||159|válido|
|practice_0090|E2|52-3|168||159|válido|
|practice_0090|E2|52-6|167||155|válido|
|practice_0090|E2|64-1|169||159|válido|
|practice_0090|E2|64-3|168||160|válido|
|practice_0090|E2|64-6|167||156|válido|
|practice_0090|E3|joint|167|individual 166|166|OPTIMAL|
|practice_0090|E4|repair|167||166|completo|
|practice_0091|E1|1|259|postprocess 258, optimize_a4 258, nested_windows 258|258|válido|
|practice_0091|E1|3|246|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|practice_0091|E1|6|245|postprocess 245, optimize_a4 245, nested_windows 245|245|válido|
|practice_0091|E2|52-1|259||228|válido|
|practice_0091|E2|52-3|246||224|válido|
|practice_0091|E2|52-6|245||230|válido|
|practice_0091|E2|64-1|259||235|válido|
|practice_0091|E2|64-3|246||237|válido|
|practice_0091|E2|64-6|245||226|válido|
|practice_0091|E3|joint|245|individual 244|244|OPTIMAL|
|practice_0091|E4|repair|245||244|completo|
|practice_0092|E1|1|204|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|practice_0092|E1|3|199|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|practice_0092|E1|6|201|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|practice_0092|E2|52-1|204||189|válido|
|practice_0092|E2|52-3|199||183|válido|
|practice_0092|E2|52-6|201||183|válido|
|practice_0092|E2|64-1|204||190|válido|
|practice_0092|E2|64-3|199||183|válido|
|practice_0092|E2|64-6|201||181|válido|
|practice_0092|E3|joint|199|individual 199|199|OPTIMAL|
|practice_0092|E4|repair|199||199|completo|
|practice_0093|E1|1|266|postprocess 266, optimize_a4 266, nested_windows 266|266|válido|
|practice_0093|E1|3|260|postprocess 260, optimize_a4 260, nested_windows 260|260|válido|
|practice_0093|E1|6|251|postprocess 251, optimize_a4 251, nested_windows 251|251|válido|
|practice_0093|E2|52-1|266||252|válido|
|practice_0093|E2|52-3|260||239|válido|
|practice_0093|E2|52-6|251||238|válido|
|practice_0093|E2|64-1|266||252|válido|
|practice_0093|E2|64-3|260||239|válido|
|practice_0093|E2|64-6|251||240|válido|
|practice_0093|E3|joint|251|individual 251|251|OPTIMAL|
|practice_0093|E4|repair|251||251|completo|
|practice_0094|E1|1|150|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|practice_0094|E1|3|149|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|practice_0094|E1|6|149|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|practice_0094|E2|52-1|150||142|válido|
|practice_0094|E2|52-3|149||141|válido|
|practice_0094|E2|52-6|149||141|válido|
|practice_0094|E2|64-1|150||141|válido|
|practice_0094|E2|64-3|149||141|válido|
|practice_0094|E2|64-6|149||141|válido|
|practice_0094|E3|joint|149|individual 149|149|OPTIMAL|
|practice_0094|E4|repair|149||149|completo|
|practice_0095|E1|1|294|postprocess 294, optimize_a4 294, nested_windows 294|294|válido|
|practice_0095|E1|3|283|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|practice_0095|E1|6|282|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|practice_0095|E2|52-1|294||266|válido|
|practice_0095|E2|52-3|283||262|válido|
|practice_0095|E2|52-6|282||261|válido|
|practice_0095|E2|64-1|294||267|válido|
|practice_0095|E2|64-3|283||263|válido|
|practice_0095|E2|64-6|282||259|válido|
|practice_0095|E3|joint|282|individual 282|275|OPTIMAL|
|practice_0095|E4|repair|282||282|completo|
|practice_0096|E1|1|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|practice_0096|E1|3|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|practice_0096|E1|6|230|postprocess 230, optimize_a4 230, nested_windows 230|230|válido|
|practice_0096|E2|52-1|239||225|válido|
|practice_0096|E2|52-3|235||220|válido|
|practice_0096|E2|52-6|230||214|válido|
|practice_0096|E2|64-1|239||226|válido|
|practice_0096|E2|64-3|235||220|válido|
|practice_0096|E2|64-6|230||216|válido|
|practice_0096|E3|joint|230|individual 230|229|FEASIBLE|
|practice_0096|E4|repair|230||230|completo|
|practice_0097|E1|1|141|postprocess 141, optimize_a4 141, nested_windows 141|141|válido|
|practice_0097|E1|3|141|postprocess 141, optimize_a4 141, nested_windows 141|141|válido|
|practice_0097|E1|6|139|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|practice_0097|E2|52-1|141||133|válido|
|practice_0097|E2|52-3|141||133|válido|
|practice_0097|E2|52-6|139||133|válido|
|practice_0097|E2|64-1|141||136|válido|
|practice_0097|E2|64-3|141||136|válido|
|practice_0097|E2|64-6|139||136|válido|
|practice_0097|E3|joint|139|individual 139|139|OPTIMAL|
|practice_0097|E4|repair|139||138|completo|
|practice_0098|E1|1|2|postprocess 2, optimize_a4 1, nested_windows 1|1|válido|
|practice_0098|E1|3|2|postprocess 2, optimize_a4 1, nested_windows 1|1|válido|
|practice_0098|E1|6|2|postprocess 2, optimize_a4 1, nested_windows 1|1|válido|
|practice_0098|E2|52-1|2||2|válido|
|practice_0098|E2|52-3|2||2|válido|
|practice_0098|E2|52-6|2||2|válido|
|practice_0098|E2|64-1|2||2|válido|
|practice_0098|E2|64-3|2||2|válido|
|practice_0098|E2|64-6|2||2|válido|
|practice_0098|E3|joint|1|individual 1|1|OPTIMAL|
|practice_0098|E4|repair|1||1|completo|
|practice_0099|E1|1|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|practice_0099|E1|3|135|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|practice_0099|E1|6|135|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|practice_0099|E2|52-1|137||132|válido|
|practice_0099|E2|52-3|135||132|válido|
|practice_0099|E2|52-6|135||132|válido|
|practice_0099|E2|64-1|137||132|válido|
|practice_0099|E2|64-3|135||132|válido|
|practice_0099|E2|64-6|135||132|válido|
|practice_0099|E3|joint|135|individual 135|135|OPTIMAL|
|practice_0099|E4|repair|135||135|completo|
|practice_0100|E1|1|385|postprocess 383, optimize_a4 383, nested_windows 383|383|válido|
|practice_0100|E1|3|369|postprocess 369, optimize_a4 369, nested_windows 369|369|válido|
|practice_0100|E1|6|366|postprocess 365, optimize_a4 365, nested_windows 365|365|válido|
|practice_0100|E2|52-1|385||363|válido|
|practice_0100|E2|52-3|369||341|válido|
|practice_0100|E2|52-6|366||342|válido|
|practice_0100|E2|64-1|385||361|válido|
|practice_0100|E2|64-3|369||345|válido|
|practice_0100|E2|64-6|366||344|válido|
|practice_0100|E3|joint|366|individual 365|362|FEASIBLE|
|practice_0100|E4|repair|366||366|completo|
|practice_0101|E1|1|180|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|practice_0101|E1|3|176|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|practice_0101|E1|6|173|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|practice_0101|E2|52-1|180||168|válido|
|practice_0101|E2|52-3|176||168|válido|
|practice_0101|E2|52-6|173||168|válido|
|practice_0101|E2|64-1|180||168|válido|
|practice_0101|E2|64-3|176||167|válido|
|practice_0101|E2|64-6|173||166|válido|
|practice_0101|E3|joint|173|individual 173|173|OPTIMAL|
|practice_0101|E4|repair|173||173|completo|
|practice_0102|E1|1|348|postprocess 346, optimize_a4 346, nested_windows 346|346|válido|
|practice_0102|E1|3|338|postprocess 337, optimize_a4 337, nested_windows 337|337|válido|
|practice_0102|E1|6|327|postprocess 327, optimize_a4 327, nested_windows 327|327|válido|
|practice_0102|E2|52-1|348||324|válido|
|practice_0102|E2|52-3|338||316|válido|
|practice_0102|E2|52-6|327||307|válido|
|practice_0102|E2|64-1|348||323|válido|
|practice_0102|E2|64-3|338||315|válido|
|practice_0102|E2|64-6|327||306|válido|
|practice_0102|E3|joint|327|individual 326|326|OPTIMAL|
|practice_0102|E4|repair|327||327|completo|
|practice_0103|E1|1|6|postprocess 6, optimize_a4 5, nested_windows 5|5|válido|
|practice_0103|E1|3|6|postprocess 6, optimize_a4 5, nested_windows 5|5|válido|
|practice_0103|E1|6|6|postprocess 6, optimize_a4 5, nested_windows 5|5|válido|
|practice_0103|E2|52-1|6||6|válido|
|practice_0103|E2|52-3|6||6|válido|
|practice_0103|E2|52-6|6||6|válido|
|practice_0103|E2|64-1|6||6|válido|
|practice_0103|E2|64-3|6||6|válido|
|practice_0103|E2|64-6|6||6|válido|
|practice_0103|E3|joint|5|individual 5|5|OPTIMAL|
|practice_0103|E4|repair|5||5|completo|
|practice_0104|E1|1|251|postprocess 251, optimize_a4 251, nested_windows 251|251|válido|
|practice_0104|E1|3|243|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|practice_0104|E1|6|244|postprocess 244, optimize_a4 244, nested_windows 244|244|válido|
|practice_0104|E2|52-1|251||236|válido|
|practice_0104|E2|52-3|243||238|válido|
|practice_0104|E2|52-6|244||230|válido|
|practice_0104|E2|64-1|251||240|válido|
|practice_0104|E2|64-3|243||239|válido|
|practice_0104|E2|64-6|244||230|válido|
|practice_0104|E3|joint|243|individual 243|241|OPTIMAL|
|practice_0104|E4|repair|243||242|completo|
|practice_0105|E1|1|214|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|practice_0105|E1|3|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|practice_0105|E1|6|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|practice_0105|E2|52-1|214||196|válido|
|practice_0105|E2|52-3|206||197|válido|
|practice_0105|E2|52-6|202||195|válido|
|practice_0105|E2|64-1|214||196|válido|
|practice_0105|E2|64-3|206||196|válido|
|practice_0105|E2|64-6|202||195|válido|
|practice_0105|E3|joint|202|individual 202|202|OPTIMAL|
|practice_0105|E4|repair|202||200|completo|
|practice_0106|E1|1|200|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|practice_0106|E1|3|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|practice_0106|E1|6|191|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|practice_0106|E2|52-1|200||187|válido|
|practice_0106|E2|52-3|197||189|válido|
|practice_0106|E2|52-6|191||180|válido|
|practice_0106|E2|64-1|200||190|válido|
|practice_0106|E2|64-3|197||189|válido|
|practice_0106|E2|64-6|191||180|válido|
|practice_0106|E3|joint|191|individual 191|191|OPTIMAL|
|practice_0106|E4|repair|191||190|completo|
|practice_0107|E1|1|132|postprocess 130, optimize_a4 130, nested_windows 130|130|válido|
|practice_0107|E1|3|132|postprocess 130, optimize_a4 130, nested_windows 130|130|válido|
|practice_0107|E1|6|132|postprocess 130, optimize_a4 130, nested_windows 130|130|válido|
|practice_0107|E2|52-1|132||125|válido|
|practice_0107|E2|52-3|132||125|válido|
|practice_0107|E2|52-6|132||125|válido|
|practice_0107|E2|64-1|132||125|válido|
|practice_0107|E2|64-3|132||125|válido|
|practice_0107|E2|64-6|132||125|válido|
|practice_0107|E3|joint|132|individual 132|132|OPTIMAL|
|practice_0107|E4|repair|132||130|completo|
|practice_0108|E1|1|121|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|practice_0108|E1|3|121|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|practice_0108|E1|6|121|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|practice_0108|E2|52-1|121||119|válido|
|practice_0108|E2|52-3|121||119|válido|
|practice_0108|E2|52-6|121||119|válido|
|practice_0108|E2|64-1|121||119|válido|
|practice_0108|E2|64-3|121||119|válido|
|practice_0108|E2|64-6|121||119|válido|
|practice_0108|E3|joint|121|individual 121|121|OPTIMAL|
|practice_0108|E4|repair|121||116|completo|
|practice_0109|E1|1|153|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|practice_0109|E1|3|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|practice_0109|E1|6|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|practice_0109|E2|52-1|153||151|válido|
|practice_0109|E2|52-3|152||150|válido|
|practice_0109|E2|52-6|152||150|válido|
|practice_0109|E2|64-1|153||152|válido|
|practice_0109|E2|64-3|152||151|válido|
|practice_0109|E2|64-6|152||150|válido|
|practice_0109|E3|joint|152|individual 152|152|OPTIMAL|
|practice_0109|E4|repair|152||150|completo|
|practice_0110|E1|1|296|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|practice_0110|E1|3|283|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|practice_0110|E1|6|279|postprocess 279, optimize_a4 279, nested_windows 279|279|válido|
|practice_0110|E2|52-1|296||281|válido|
|practice_0110|E2|52-3|283||268|válido|
|practice_0110|E2|52-6|279||268|válido|
|practice_0110|E2|64-1|296||281|válido|
|practice_0110|E2|64-3|283||390|válido|
|practice_0110|E2|64-6|279||270|válido|
|practice_0110|E3|joint|279|individual 279|276|FEASIBLE|
|practice_0110|E4|repair|279||279|completo|
|practice_0111|E1|1|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|practice_0111|E1|3|193|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|practice_0111|E1|6|193|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|practice_0111|E2|52-1|197||185|válido|
|practice_0111|E2|52-3|193||181|válido|
|practice_0111|E2|52-6|193||183|válido|
|practice_0111|E2|64-1|197||187|válido|
|practice_0111|E2|64-3|193||180|válido|
|practice_0111|E2|64-6|193||180|válido|
|practice_0111|E3|joint|193|individual 193|192|OPTIMAL|
|practice_0111|E4|repair|193||193|completo|
|practice_0112|E1|1|132|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|practice_0112|E1|3|131|postprocess 130, optimize_a4 130, nested_windows 130|130|válido|
|practice_0112|E1|6|131|postprocess 130, optimize_a4 130, nested_windows 130|130|válido|
|practice_0112|E2|52-1|132||129|válido|
|practice_0112|E2|52-3|131||129|válido|
|practice_0112|E2|52-6|131||129|válido|
|practice_0112|E2|64-1|132||129|válido|
|practice_0112|E2|64-3|131||129|válido|
|practice_0112|E2|64-6|131||129|válido|
|practice_0112|E3|joint|131|individual 131|131|OPTIMAL|
|practice_0112|E4|repair|131||128|completo|
|practice_0113|E1|1|345|postprocess 345, optimize_a4 345, nested_windows 345|345|válido|
|practice_0113|E1|3|331|postprocess 331, optimize_a4 331, nested_windows 331|331|válido|
|practice_0113|E1|6|330|postprocess 330, optimize_a4 330, nested_windows 330|330|válido|
|practice_0113|E2|52-1|345||324|válido|
|practice_0113|E2|52-3|331||310|válido|
|practice_0113|E2|52-6|330||306|válido|
|practice_0113|E2|64-1|345||329|válido|
|practice_0113|E2|64-3|331||308|válido|
|practice_0113|E2|64-6|330||308|válido|
|practice_0113|E3|joint|330|individual 330|329|FEASIBLE|
|practice_0113|E4|repair|330||330|completo|
|practice_0114|E1|1|56|postprocess 55, optimize_a4 55, nested_windows 55|55|válido|
|practice_0114|E1|3|56|postprocess 55, optimize_a4 55, nested_windows 55|55|válido|
|practice_0114|E1|6|56|postprocess 55, optimize_a4 55, nested_windows 55|55|válido|
|practice_0114|E2|52-1|56||57|válido|
|practice_0114|E2|52-3|56||57|válido|
|practice_0114|E2|52-6|56||57|válido|
|practice_0114|E2|64-1|56||57|válido|
|practice_0114|E2|64-3|56||57|válido|
|practice_0114|E2|64-6|56||57|válido|
|practice_0114|E3|joint|56|individual 56|56|OPTIMAL|
|practice_0114|E4|repair|56||56|completo|
|practice_0115|E1|1|214|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|practice_0115|E1|3|201|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|practice_0115|E1|6|203|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|practice_0115|E2|52-1|214||201|válido|
|practice_0115|E2|52-3|201||192|válido|
|practice_0115|E2|52-6|203||189|válido|
|practice_0115|E2|64-1|214||201|válido|
|practice_0115|E2|64-3|201||190|válido|
|practice_0115|E2|64-6|203||190|válido|
|practice_0115|E3|joint|201|individual 201|201|OPTIMAL|
|practice_0115|E4|repair|201||201|completo|
|practice_0116|E1|1|74|postprocess 73, optimize_a4 73, nested_windows 73|73|válido|
|practice_0116|E1|3|74|postprocess 73, optimize_a4 73, nested_windows 73|73|válido|
|practice_0116|E1|6|69|postprocess 67, optimize_a4 67, nested_windows 67|67|válido|
|practice_0116|E2|52-1|74||72|válido|
|practice_0116|E2|52-3|74||72|válido|
|practice_0116|E2|52-6|69||71|válido|
|practice_0116|E2|64-1|74||72|válido|
|practice_0116|E2|64-3|74||72|válido|
|practice_0116|E2|64-6|69||71|válido|
|practice_0116|E3|joint|69|individual 69|69|OPTIMAL|
|practice_0116|E4|repair|69||69|completo|
|practice_0117|E1|1|156|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|practice_0117|E1|3|156|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|practice_0117|E1|6|156|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|practice_0117|E2|52-1|156||144|válido|
|practice_0117|E2|52-3|156||143|válido|
|practice_0117|E2|52-6|156||143|válido|
|practice_0117|E2|64-1|156||143|válido|
|practice_0117|E2|64-3|156||143|válido|
|practice_0117|E2|64-6|156||143|válido|
|practice_0117|E3|joint|156|individual 156|156|OPTIMAL|
|practice_0117|E4|repair|156||155|completo|
|practice_0118|E1|1|43|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|practice_0118|E1|3|43|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|practice_0118|E1|6|43|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|practice_0118|E2|52-1|43||43|válido|
|practice_0118|E2|52-3|43||43|válido|
|practice_0118|E2|52-6|43||43|válido|
|practice_0118|E2|64-1|43||43|válido|
|practice_0118|E2|64-3|43||43|válido|
|practice_0118|E2|64-6|43||43|válido|
|practice_0118|E3|joint|43|individual 43|43|OPTIMAL|
|practice_0118|E4|repair|43||43|completo|
|practice_0119|E1|1|311|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|practice_0119|E1|3|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|practice_0119|E1|6|289|postprocess 288, optimize_a4 288, nested_windows 288|288|válido|
|practice_0119|E2|52-1|311||281|válido|
|practice_0119|E2|52-3|289||274|válido|
|practice_0119|E2|52-6|289||267|válido|
|practice_0119|E2|64-1|311||282|válido|
|practice_0119|E2|64-3|289||273|válido|
|practice_0119|E2|64-6|289||266|válido|
|practice_0119|E3|joint|289|individual 289|285|FEASIBLE|
|practice_0119|E4|repair|289||289|completo|
|practice_0120|E1|1|176|postprocess 176, optimize_a4 176, nested_windows 176|176|válido|
|practice_0120|E1|3|173|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|practice_0120|E1|6|170|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|practice_0120|E2|52-1|176||169|válido|
|practice_0120|E2|52-3|173||167|válido|
|practice_0120|E2|52-6|170||167|válido|
|practice_0120|E2|64-1|176||170|válido|
|practice_0120|E2|64-3|173||166|válido|
|practice_0120|E2|64-6|170||166|válido|
|practice_0120|E3|joint|170|individual 170|170|OPTIMAL|
|practice_0120|E4|repair|170||166|completo|
|practice_0121|E1|1|319|postprocess 319, optimize_a4 319, nested_windows 319|319|válido|
|practice_0121|E1|3|305|postprocess 305, optimize_a4 305, nested_windows 305|305|válido|
|practice_0121|E1|6|295|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|practice_0121|E2|52-1|319||296|válido|
|practice_0121|E2|52-3|305||289|válido|
|practice_0121|E2|52-6|295||280|válido|
|practice_0121|E2|64-1|319||294|válido|
|practice_0121|E2|64-3|305||290|válido|
|practice_0121|E2|64-6|295||283|válido|
|practice_0121|E3|joint|295|individual 295|295|OPTIMAL|
|practice_0121|E4|repair|295||295|completo|
|practice_0122|E1|1|370|postprocess 370, optimize_a4 370, nested_windows 370|370|válido|
|practice_0122|E1|3|350|postprocess 350, optimize_a4 350, nested_windows 350|350|válido|
|practice_0122|E1|6|346|postprocess 345, optimize_a4 345, nested_windows 345|345|válido|
|practice_0122|E2|52-1|370||345|válido|
|practice_0122|E2|52-3|350||322|válido|
|practice_0122|E2|52-6|346||319|válido|
|practice_0122|E2|64-1|370||460|válido|
|practice_0122|E2|64-3|350||324|válido|
|practice_0122|E2|64-6|346||319|válido|
|practice_0122|E3|joint|346|individual 346|345|FEASIBLE|
|practice_0122|E4|repair|346||346|completo|
|practice_0123|E1|1|3|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0123|E1|3|3|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0123|E1|6|3|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0123|E2|52-1|3||3|válido|
|practice_0123|E2|52-3|3||3|válido|
|practice_0123|E2|52-6|3||3|válido|
|practice_0123|E2|64-1|3||3|válido|
|practice_0123|E2|64-3|3||3|válido|
|practice_0123|E2|64-6|3||3|válido|
|practice_0123|E3|joint|2|individual 2|2|OPTIMAL|
|practice_0123|E4|repair|2||2|completo|
|practice_0124|E1|1|253|postprocess 253, optimize_a4 253, nested_windows 253|253|válido|
|practice_0124|E1|3|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|practice_0124|E1|6|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|practice_0124|E2|52-1|253||234|válido|
|practice_0124|E2|52-3|239||229|válido|
|practice_0124|E2|52-6|242||226|válido|
|practice_0124|E2|64-1|253||236|válido|
|practice_0124|E2|64-3|239||230|válido|
|practice_0124|E2|64-6|242||226|válido|
|practice_0124|E3|joint|239|individual 238|237|OPTIMAL|
|practice_0124|E4|repair|239||239|completo|
|practice_0125|E1|1|4|postprocess 4, optimize_a4 3, nested_windows 3|3|válido|
|practice_0125|E1|3|4|postprocess 4, optimize_a4 3, nested_windows 3|3|válido|
|practice_0125|E1|6|4|postprocess 4, optimize_a4 3, nested_windows 3|3|válido|
|practice_0125|E2|52-1|4||4|válido|
|practice_0125|E2|52-3|4||4|válido|
|practice_0125|E2|52-6|4||4|válido|
|practice_0125|E2|64-1|4||4|válido|
|practice_0125|E2|64-3|4||4|válido|
|practice_0125|E2|64-6|4||4|válido|
|practice_0125|E3|joint|3|individual 3|3|OPTIMAL|
|practice_0125|E4|repair|3||3|completo|
|practice_0126|E1|1|181|postprocess 180, optimize_a4 180, nested_windows 180|180|válido|
|practice_0126|E1|3|176|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|practice_0126|E1|6|173|postprocess 170, optimize_a4 170, nested_windows 170|170|válido|
|practice_0126|E2|52-1|181||165|válido|
|practice_0126|E2|52-3|176||163|válido|
|practice_0126|E2|52-6|173||156|válido|
|practice_0126|E2|64-1|181||171|válido|
|practice_0126|E2|64-3|176||166|válido|
|practice_0126|E2|64-6|173||158|válido|
|practice_0126|E3|joint|173|individual 173|173|OPTIMAL|
|practice_0126|E4|repair|173||170|completo|
|practice_0127|E1|1|216|postprocess 216, optimize_a4 216, nested_windows 216|216|válido|
|practice_0127|E1|3|210|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|practice_0127|E1|6|207|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|practice_0127|E2|52-1|216||201|válido|
|practice_0127|E2|52-3|210||199|válido|
|practice_0127|E2|52-6|207||196|válido|
|practice_0127|E2|64-1|216||200|válido|
|practice_0127|E2|64-3|210||198|válido|
|practice_0127|E2|64-6|207||197|válido|
|practice_0127|E3|joint|207|individual 207|207|OPTIMAL|
|practice_0127|E4|repair|207||206|completo|
|practice_0128|E1|1|4|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0128|E1|3|4|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0128|E1|6|4|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0128|E2|52-1|4||4|válido|
|practice_0128|E2|52-3|4||4|válido|
|practice_0128|E2|52-6|4||4|válido|
|practice_0128|E2|64-1|4||4|válido|
|practice_0128|E2|64-3|4||4|válido|
|practice_0128|E2|64-6|4||4|válido|
|practice_0128|E3|joint|2|individual 2|2|OPTIMAL|
|practice_0128|E4|repair|2||2|completo|
|practice_0129|E1|1|133|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|practice_0129|E1|3|133|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|practice_0129|E1|6|131|postprocess 130, optimize_a4 130, nested_windows 130|130|válido|
|practice_0129|E2|52-1|133||125|válido|
|practice_0129|E2|52-3|133||125|válido|
|practice_0129|E2|52-6|131||125|válido|
|practice_0129|E2|64-1|133||127|válido|
|practice_0129|E2|64-3|133||127|válido|
|practice_0129|E2|64-6|131||127|válido|
|practice_0129|E3|joint|131|individual 131|131|OPTIMAL|
|practice_0129|E4|repair|131||127|completo|
|practice_0130|E1|1|119|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|practice_0130|E1|3|120|postprocess 120, optimize_a4 120, nested_windows 120|120|válido|
|practice_0130|E1|6|120|postprocess 120, optimize_a4 120, nested_windows 120|120|válido|
|practice_0130|E2|52-1|119||111|válido|
|practice_0130|E2|52-3|120||111|válido|
|practice_0130|E2|52-6|120||111|válido|
|practice_0130|E2|64-1|119||114|válido|
|practice_0130|E2|64-3|120||114|válido|
|practice_0130|E2|64-6|120||113|válido|
|practice_0130|E3|joint|119|individual 119|119|OPTIMAL|
|practice_0130|E4|repair|119||116|completo|
|practice_0131|E1|1|254|postprocess 254, optimize_a4 254, nested_windows 254|254|válido|
|practice_0131|E1|3|244|postprocess 244, optimize_a4 244, nested_windows 244|244|válido|
|practice_0131|E1|6|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|practice_0131|E2|52-1|254||234|válido|
|practice_0131|E2|52-3|244||229|válido|
|practice_0131|E2|52-6|238||228|válido|
|practice_0131|E2|64-1|254||234|válido|
|practice_0131|E2|64-3|244||229|válido|
|practice_0131|E2|64-6|238||226|válido|
|practice_0131|E3|joint|238|individual 238|238|OPTIMAL|
|practice_0131|E4|repair|238||238|completo|
|practice_0132|E1|1|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|practice_0132|E1|3|196|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|practice_0132|E1|6|193|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|practice_0132|E2|52-1|199||183|válido|
|practice_0132|E2|52-3|196||181|válido|
|practice_0132|E2|52-6|193||179|válido|
|practice_0132|E2|64-1|199||183|válido|
|practice_0132|E2|64-3|196||180|válido|
|practice_0132|E2|64-6|193||180|válido|
|practice_0132|E3|joint|193|individual 192|192|OPTIMAL|
|practice_0132|E4|repair|193||193|completo|
|practice_0133|E1|1|208|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|practice_0133|E1|3|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|practice_0133|E1|6|198|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|practice_0133|E2|52-1|208||190|válido|
|practice_0133|E2|52-3|197||188|válido|
|practice_0133|E2|52-6|198||189|válido|
|practice_0133|E2|64-1|208||190|válido|
|practice_0133|E2|64-3|197||188|válido|
|practice_0133|E2|64-6|198||189|válido|
|practice_0133|E3|joint|197|individual 197|197|OPTIMAL|
|practice_0133|E4|repair|197||195|completo|
|practice_0134|E1|1|237|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|practice_0134|E1|3|230|postprocess 230, optimize_a4 230, nested_windows 230|230|válido|
|practice_0134|E1|6|228|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|practice_0134|E2|52-1|237||225|válido|
|practice_0134|E2|52-3|230||218|válido|
|practice_0134|E2|52-6|228||215|válido|
|practice_0134|E2|64-1|237||225|válido|
|practice_0134|E2|64-3|230||221|válido|
|practice_0134|E2|64-6|228||215|válido|
|practice_0134|E3|joint|228|individual 228|225|OPTIMAL|
|practice_0134|E4|repair|228||226|completo|
|practice_0135|E1|1|401|postprocess 401, optimize_a4 401, nested_windows 401|401|válido|
|practice_0135|E1|3|386|postprocess 385, optimize_a4 385, nested_windows 385|385|válido|
|practice_0135|E1|6|373|postprocess 373, optimize_a4 373, nested_windows 373|373|válido|
|practice_0135|E2|52-1|401||366|válido|
|practice_0135|E2|52-3|386||354|válido|
|practice_0135|E2|52-6|373||344|válido|
|practice_0135|E2|64-1|401||371|válido|
|practice_0135|E2|64-3|386||359|válido|
|practice_0135|E2|64-6|373||346|válido|
|practice_0135|E3|joint|373|individual 373|373|FEASIBLE|
|practice_0135|E4|repair|373||373|completo|
|practice_0136|E1|1|56|postprocess 54, optimize_a4 54, nested_windows 54|54|válido|
|practice_0136|E1|3|56|postprocess 54, optimize_a4 54, nested_windows 54|54|válido|
|practice_0136|E1|6|56|postprocess 54, optimize_a4 54, nested_windows 54|54|válido|
|practice_0136|E2|52-1|56||56|válido|
|practice_0136|E2|52-3|56||56|válido|
|practice_0136|E2|52-6|56||56|válido|
|practice_0136|E2|64-1|56||56|válido|
|practice_0136|E2|64-3|56||56|válido|
|practice_0136|E2|64-6|56||56|válido|
|practice_0136|E3|joint|56|individual 56|56|OPTIMAL|
|practice_0136|E4|repair|56||54|completo|
|practice_0137|E1|1|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|practice_0137|E1|3|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|practice_0137|E1|6|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|practice_0137|E2|52-1|113||106|válido|
|practice_0137|E2|52-3|113||106|válido|
|practice_0137|E2|52-6|113||106|válido|
|practice_0137|E2|64-1|113||106|válido|
|practice_0137|E2|64-3|113||106|válido|
|practice_0137|E2|64-6|113||106|válido|
|practice_0137|E3|joint|113|individual 113|113|OPTIMAL|
|practice_0137|E4|repair|113||110|completo|
|practice_0138|E1|1|402|postprocess 402, optimize_a4 402, nested_windows 402|402|válido|
|practice_0138|E1|3|378|postprocess 378, optimize_a4 378, nested_windows 378|378|válido|
|practice_0138|E1|6|367|postprocess 366, optimize_a4 366, nested_windows 366|366|válido|
|practice_0138|E2|52-1|402||365|válido|
|practice_0138|E2|52-3|378||356|válido|
|practice_0138|E2|52-6|367||346|válido|
|practice_0138|E2|64-1|402||369|válido|
|practice_0138|E2|64-3|378||359|válido|
|practice_0138|E2|64-6|367||345|válido|
|practice_0138|E3|joint|367|individual 365|365|FEASIBLE|
|practice_0138|E4|repair|367||367|completo|
|practice_0139|E1|1|247|postprocess 247, optimize_a4 247, nested_windows 247|247|válido|
|practice_0139|E1|3|236|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|practice_0139|E1|6|237|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|practice_0139|E2|52-1|247||224|válido|
|practice_0139|E2|52-3|236||221|válido|
|practice_0139|E2|52-6|237||218|válido|
|practice_0139|E2|64-1|247||223|válido|
|practice_0139|E2|64-3|236||222|válido|
|practice_0139|E2|64-6|237||218|válido|
|practice_0139|E3|joint|236|individual 236|236|OPTIMAL|
|practice_0139|E4|repair|236||236|completo|
|practice_0140|E1|1|1|postprocess 1, optimize_a4 1, nested_windows 1|1|válido|
|practice_0140|E1|3|1|postprocess 1, optimize_a4 1, nested_windows 1|1|válido|
|practice_0140|E1|6|1|postprocess 1, optimize_a4 1, nested_windows 1|1|válido|
|practice_0140|E2|52-1|1||1|válido|
|practice_0140|E2|52-3|1||1|válido|
|practice_0140|E2|52-6|1||1|válido|
|practice_0140|E2|64-1|1||1|válido|
|practice_0140|E2|64-3|1||1|válido|
|practice_0140|E2|64-6|1||1|válido|
|practice_0140|E3|joint|1|individual 1|1|OPTIMAL|
|practice_0140|E4|repair|1||1|completo|
|practice_0141|E1|1|170|postprocess 170, optimize_a4 170, nested_windows 170|170|válido|
|practice_0141|E1|3|168|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|practice_0141|E1|6|167|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|practice_0141|E2|52-1|170||160|válido|
|practice_0141|E2|52-3|168||159|válido|
|practice_0141|E2|52-6|167||157|válido|
|practice_0141|E2|64-1|170||159|válido|
|practice_0141|E2|64-3|168||159|válido|
|practice_0141|E2|64-6|167||156|válido|
|practice_0141|E3|joint|167|individual 167|167|OPTIMAL|
|practice_0141|E4|repair|167||167|completo|
|practice_0142|E1|1|302|postprocess 302, optimize_a4 302, nested_windows 302|302|válido|
|practice_0142|E1|3|277|postprocess 277, optimize_a4 277, nested_windows 277|277|válido|
|practice_0142|E1|6|281|postprocess 281, optimize_a4 281, nested_windows 281|281|válido|
|practice_0142|E2|52-1|302||281|válido|
|practice_0142|E2|52-3|277||276|válido|
|practice_0142|E2|52-6|281||264|válido|
|practice_0142|E2|64-1|302||283|válido|
|practice_0142|E2|64-3|277||275|válido|
|practice_0142|E2|64-6|281||268|válido|
|practice_0142|E3|joint|277|individual 277|276|OPTIMAL|
|practice_0142|E4|repair|277||277|completo|
|practice_0143|E1|1|178|postprocess 178, optimize_a4 178, nested_windows 178|178|válido|
|practice_0143|E1|3|174|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|practice_0143|E1|6|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|practice_0143|E2|52-1|178||161|válido|
|practice_0143|E2|52-3|174||159|válido|
|practice_0143|E2|52-6|172||156|válido|
|practice_0143|E2|64-1|178||163|válido|
|practice_0143|E2|64-3|174||161|válido|
|practice_0143|E2|64-6|172||156|válido|
|practice_0143|E3|joint|172|individual 172|172|OPTIMAL|
|practice_0143|E4|repair|172||172|completo|
|practice_0144|E1|1|125|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|practice_0144|E1|3|124|postprocess 123, optimize_a4 123, nested_windows 123|123|válido|
|practice_0144|E1|6|124|postprocess 123, optimize_a4 123, nested_windows 123|123|válido|
|practice_0144|E2|52-1|125||116|válido|
|practice_0144|E2|52-3|124||116|válido|
|practice_0144|E2|52-6|124||116|válido|
|practice_0144|E2|64-1|125||116|válido|
|practice_0144|E2|64-3|124||115|válido|
|practice_0144|E2|64-6|124||115|válido|
|practice_0144|E3|joint|124|individual 124|124|OPTIMAL|
|practice_0144|E4|repair|124||117|completo|
|practice_0145|E1|1|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|practice_0145|E1|3|225|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|practice_0145|E1|6|219|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|practice_0145|E2|52-1|229||213|válido|
|practice_0145|E2|52-3|225||211|válido|
|practice_0145|E2|52-6|219||207|válido|
|practice_0145|E2|64-1|229||208|válido|
|practice_0145|E2|64-3|225||201|válido|
|practice_0145|E2|64-6|219||202|válido|
|practice_0145|E3|joint|219|individual 218|218|OPTIMAL|
|practice_0145|E4|repair|219||212|completo|
|practice_0146|E1|1|47|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|practice_0146|E1|3|47|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|practice_0146|E1|6|47|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|practice_0146|E2|52-1|47||47|válido|
|practice_0146|E2|52-3|47||47|válido|
|practice_0146|E2|52-6|47||47|válido|
|practice_0146|E2|64-1|47||47|válido|
|practice_0146|E2|64-3|47||47|válido|
|practice_0146|E2|64-6|47||47|válido|
|practice_0146|E3|joint|47|individual 47|47|OPTIMAL|
|practice_0146|E4|repair|47||46|completo|
|practice_0147|E1|1|205|postprocess 205, optimize_a4 205, nested_windows 205|205|válido|
|practice_0147|E1|3|205|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|practice_0147|E1|6|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|practice_0147|E2|52-1|205||192|válido|
|practice_0147|E2|52-3|205||188|válido|
|practice_0147|E2|52-6|197||188|válido|
|practice_0147|E2|64-1|205||193|válido|
|practice_0147|E2|64-3|205||188|válido|
|practice_0147|E2|64-6|197||189|válido|
|practice_0147|E3|joint|197|individual 197|197|OPTIMAL|
|practice_0147|E4|repair|197||194|completo|
|practice_0148|E1|1|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|practice_0148|E1|3|186|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|practice_0148|E1|6|185|postprocess 184, optimize_a4 184, nested_windows 184|184|válido|
|practice_0148|E2|52-1|192||181|válido|
|practice_0148|E2|52-3|186||177|válido|
|practice_0148|E2|52-6|185||176|válido|
|practice_0148|E2|64-1|192||180|válido|
|practice_0148|E2|64-3|186||177|válido|
|practice_0148|E2|64-6|185||177|válido|
|practice_0148|E3|joint|185|individual 185|185|OPTIMAL|
|practice_0148|E4|repair|185||182|completo|
|practice_0149|E1|1|230|postprocess 230, optimize_a4 230, nested_windows 230|230|válido|
|practice_0149|E1|3|212|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|practice_0149|E1|6|212|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|practice_0149|E2|52-1|230||211|válido|
|practice_0149|E2|52-3|212||205|válido|
|practice_0149|E2|52-6|212||203|válido|
|practice_0149|E2|64-1|230||214|válido|
|practice_0149|E2|64-3|212||209|válido|
|practice_0149|E2|64-6|212||201|válido|
|practice_0149|E3|joint|212|individual 212|209|OPTIMAL|
|practice_0149|E4|repair|212||212|completo|
|practice_0150|E1|1|228|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|practice_0150|E1|3|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|practice_0150|E1|6|217|postprocess 217, optimize_a4 217, nested_windows 217|217|válido|
|practice_0150|E2|52-1|228||209|válido|
|practice_0150|E2|52-3|219||208|válido|
|practice_0150|E2|52-6|217||203|válido|
|practice_0150|E2|64-1|228||209|válido|
|practice_0150|E2|64-3|219||208|válido|
|practice_0150|E2|64-6|217||204|válido|
|practice_0150|E3|joint|217|individual 216|216|OPTIMAL|
|practice_0150|E4|repair|217||217|completo|
|practice_0151|E1|1|379|postprocess 379, optimize_a4 379, nested_windows 379|379|válido|
|practice_0151|E1|3|365|postprocess 365, optimize_a4 365, nested_windows 365|365|válido|
|practice_0151|E1|6|355|postprocess 355, optimize_a4 355, nested_windows 355|355|válido|
|practice_0151|E2|52-1|379||344|válido|
|practice_0151|E2|52-3|365||337|válido|
|practice_0151|E2|52-6|355||333|válido|
|practice_0151|E2|64-1|379||349|válido|
|practice_0151|E2|64-3|365||343|válido|
|practice_0151|E2|64-6|355||336|válido|
|practice_0151|E3|joint|355|individual 355|355|FEASIBLE|
|practice_0151|E4|repair|355||355|completo|
|practice_0152|E1|1|187|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|practice_0152|E1|3|175|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|practice_0152|E1|6|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|practice_0152|E2|52-1|187||167|válido|
|practice_0152|E2|52-3|175||164|válido|
|practice_0152|E2|52-6|172||162|válido|
|practice_0152|E2|64-1|187||168|válido|
|practice_0152|E2|64-3|175||162|válido|
|practice_0152|E2|64-6|172||161|válido|
|practice_0152|E3|joint|172|individual 171|171|OPTIMAL|
|practice_0152|E4|repair|172||172|completo|
|practice_0153|E1|1|171|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|practice_0153|E1|3|163|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|practice_0153|E1|6|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|practice_0153|E2|52-1|171||163|válido|
|practice_0153|E2|52-3|163||153|válido|
|practice_0153|E2|52-6|162||149|válido|
|practice_0153|E2|64-1|171||165|válido|
|practice_0153|E2|64-3|163||154|válido|
|practice_0153|E2|64-6|162||153|válido|
|practice_0153|E3|joint|162|individual 162|162|OPTIMAL|
|practice_0153|E4|repair|162||161|completo|
|practice_0154|E1|1|383|postprocess 383, optimize_a4 383, nested_windows 383|383|válido|
|practice_0154|E1|3|369|postprocess 369, optimize_a4 369, nested_windows 369|369|válido|
|practice_0154|E1|6|360|postprocess 360, optimize_a4 360, nested_windows 360|360|válido|
|practice_0154|E2|52-1|383||351|válido|
|practice_0154|E2|52-3|369||339|válido|
|practice_0154|E2|52-6|360||336|válido|
|practice_0154|E2|64-1|383||354|válido|
|practice_0154|E2|64-3|369||342|válido|
|practice_0154|E2|64-6|360||340|válido|
|practice_0154|E3|joint|360|individual 360|359|OPTIMAL|
|practice_0154|E4|repair|360||360|completo|
|practice_0155|E1|1|174|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|practice_0155|E1|3|170|postprocess 170, optimize_a4 170, nested_windows 170|170|válido|
|practice_0155|E1|6|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|practice_0155|E2|52-1|174||163|válido|
|practice_0155|E2|52-3|170||165|válido|
|practice_0155|E2|52-6|166||160|válido|
|practice_0155|E2|64-1|174||165|válido|
|practice_0155|E2|64-3|170||163|válido|
|practice_0155|E2|64-6|166||160|válido|
|practice_0155|E3|joint|166|individual 166|166|OPTIMAL|
|practice_0155|E4|repair|166||164|completo|
|practice_0156|E1|1|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|practice_0156|E1|3|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|practice_0156|E1|6|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|practice_0156|E2|52-1|113||111|válido|
|practice_0156|E2|52-3|113||111|válido|
|practice_0156|E2|52-6|113||111|válido|
|practice_0156|E2|64-1|113||111|válido|
|practice_0156|E2|64-3|113||111|válido|
|practice_0156|E2|64-6|113||111|válido|
|practice_0156|E3|joint|113|individual 113|113|OPTIMAL|
|practice_0156|E4|repair|113||112|completo|
|practice_0157|E1|1|158|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|practice_0157|E1|3|152|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|practice_0157|E1|6|152|postprocess 150, optimize_a4 150, nested_windows 150|150|válido|
|practice_0157|E2|52-1|158||145|válido|
|practice_0157|E2|52-3|152||143|válido|
|practice_0157|E2|52-6|152||146|válido|
|practice_0157|E2|64-1|158||147|válido|
|practice_0157|E2|64-3|152||143|válido|
|practice_0157|E2|64-6|152||147|válido|
|practice_0157|E3|joint|152|individual 152|151|OPTIMAL|
|practice_0157|E4|repair|152||150|completo|
|practice_0158|E1|1|276|postprocess 274, optimize_a4 274, nested_windows 274|274|válido|
|practice_0158|E1|3|270|postprocess 270, optimize_a4 270, nested_windows 270|270|válido|
|practice_0158|E1|6|260|postprocess 259, optimize_a4 259, nested_windows 259|259|válido|
|practice_0158|E2|52-1|276||251|válido|
|practice_0158|E2|52-3|270||244|válido|
|practice_0158|E2|52-6|260||239|válido|
|practice_0158|E2|64-1|276||252|válido|
|practice_0158|E2|64-3|270||249|válido|
|practice_0158|E2|64-6|260||236|válido|
|practice_0158|E3|joint|260|individual 259|257|OPTIMAL|
|practice_0158|E4|repair|260||260|completo|
|practice_0159|E1|1|39|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|practice_0159|E1|3|39|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|practice_0159|E1|6|39|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|practice_0159|E2|52-1|39||39|válido|
|practice_0159|E2|52-3|39||39|válido|
|practice_0159|E2|52-6|39||39|válido|
|practice_0159|E2|64-1|39||39|válido|
|practice_0159|E2|64-3|39||39|válido|
|practice_0159|E2|64-6|39||39|válido|
|practice_0159|E3|joint|39|individual 39|39|OPTIMAL|
|practice_0159|E4|repair|39||34|completo|
|practice_0160|E1|1|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|practice_0160|E1|3|196|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|practice_0160|E1|6|197|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|practice_0160|E2|52-1|202||189|válido|
|practice_0160|E2|52-3|196||187|válido|
|practice_0160|E2|52-6|197||187|válido|
|practice_0160|E2|64-1|202||190|válido|
|practice_0160|E2|64-3|196||186|válido|
|practice_0160|E2|64-6|197||187|válido|
|practice_0160|E3|joint|196|individual 195|194|OPTIMAL|
|practice_0160|E4|repair|196||196|completo|
|practice_0161|E1|1|312|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|practice_0161|E1|3|296|postprocess 294, optimize_a4 294, nested_windows 294|294|válido|
|practice_0161|E1|6|300|postprocess 300, optimize_a4 300, nested_windows 300|300|válido|
|practice_0161|E2|52-1|312||284|válido|
|practice_0161|E2|52-3|296||274|válido|
|practice_0161|E2|52-6|300||274|válido|
|practice_0161|E2|64-1|312||285|válido|
|practice_0161|E2|64-3|296||277|válido|
|practice_0161|E2|64-6|300||274|válido|
|practice_0161|E3|joint|296|individual 296|295|FEASIBLE|
|practice_0161|E4|repair|296||296|completo|
|practice_0162|E1|1|315|postprocess 315, optimize_a4 315, nested_windows 315|315|válido|
|practice_0162|E1|3|305|postprocess 305, optimize_a4 305, nested_windows 305|305|válido|
|practice_0162|E1|6|302|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|practice_0162|E2|52-1|315||287|válido|
|practice_0162|E2|52-3|305||287|válido|
|practice_0162|E2|52-6|302||285|válido|
|practice_0162|E2|64-1|315||295|válido|
|practice_0162|E2|64-3|305||291|válido|
|practice_0162|E2|64-6|302||288|válido|
|practice_0162|E3|joint|302|individual 302|299|OPTIMAL|
|practice_0162|E4|repair|302||302|completo|
|practice_0163|E1|1|301|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|practice_0163|E1|3|281|postprocess 281, optimize_a4 281, nested_windows 281|281|válido|
|practice_0163|E1|6|276|postprocess 276, optimize_a4 276, nested_windows 276|276|válido|
|practice_0163|E2|52-1|301||277|válido|
|practice_0163|E2|52-3|281||267|válido|
|practice_0163|E2|52-6|276||263|válido|
|practice_0163|E2|64-1|301||276|válido|
|practice_0163|E2|64-3|281||264|válido|
|practice_0163|E2|64-6|276||263|válido|
|practice_0163|E3|joint|276|individual 276|276|FEASIBLE|
|practice_0163|E4|repair|276||276|completo|
|practice_0164|E1|1|305|postprocess 304, optimize_a4 304, nested_windows 304|304|válido|
|practice_0164|E1|3|288|postprocess 288, optimize_a4 288, nested_windows 288|288|válido|
|practice_0164|E1|6|276|postprocess 274, optimize_a4 274, nested_windows 274|274|válido|
|practice_0164|E2|52-1|305||280|válido|
|practice_0164|E2|52-3|288||263|válido|
|practice_0164|E2|52-6|276||256|válido|
|practice_0164|E2|64-1|305||277|válido|
|practice_0164|E2|64-3|288||270|válido|
|practice_0164|E2|64-6|276||255|válido|
|practice_0164|E3|joint|276|individual 276|276|OPTIMAL|
|practice_0164|E4|repair|276||275|completo|
|practice_0165|E1|1|273|postprocess 273, optimize_a4 273, nested_windows 273|273|válido|
|practice_0165|E1|3|260|postprocess 259, optimize_a4 259, nested_windows 259|259|válido|
|practice_0165|E1|6|256|postprocess 256, optimize_a4 256, nested_windows 256|256|válido|
|practice_0165|E2|52-1|273||248|válido|
|practice_0165|E2|52-3|260||239|válido|
|practice_0165|E2|52-6|256||244|válido|
|practice_0165|E2|64-1|273||251|válido|
|practice_0165|E2|64-3|260||243|válido|
|practice_0165|E2|64-6|256||241|válido|
|practice_0165|E3|joint|256|individual 256|256|OPTIMAL|
|practice_0165|E4|repair|256||256|completo|
|practice_0166|E1|1|54|postprocess 52, optimize_a4 52, nested_windows 52|52|válido|
|practice_0166|E1|3|54|postprocess 52, optimize_a4 52, nested_windows 52|52|válido|
|practice_0166|E1|6|54|postprocess 52, optimize_a4 52, nested_windows 52|52|válido|
|practice_0166|E2|52-1|54||54|válido|
|practice_0166|E2|52-3|54||54|válido|
|practice_0166|E2|52-6|54||54|válido|
|practice_0166|E2|64-1|54||54|válido|
|practice_0166|E2|64-3|54||54|válido|
|practice_0166|E2|64-6|54||54|válido|
|practice_0166|E3|joint|54|individual 54|54|OPTIMAL|
|practice_0166|E4|repair|54||51|completo|
|practice_0167|E1|1|339|postprocess 339, optimize_a4 339, nested_windows 339|339|válido|
|practice_0167|E1|3|329|postprocess 329, optimize_a4 329, nested_windows 329|329|válido|
|practice_0167|E1|6|326|postprocess 326, optimize_a4 326, nested_windows 326|326|válido|
|practice_0167|E2|52-1|339||310|válido|
|practice_0167|E2|52-3|329||305|válido|
|practice_0167|E2|52-6|326||300|válido|
|practice_0167|E2|64-1|339||312|válido|
|practice_0167|E2|64-3|329||306|válido|
|practice_0167|E2|64-6|326||303|válido|
|practice_0167|E3|joint|326|individual 324|320|FEASIBLE|
|practice_0167|E4|repair|326||326|completo|
|practice_0168|E1|1|133|postprocess 133, optimize_a4 133, nested_windows 133|133|válido|
|practice_0168|E1|3|133|postprocess 133, optimize_a4 133, nested_windows 133|133|válido|
|practice_0168|E1|6|132|postprocess 132, optimize_a4 132, nested_windows 132|132|válido|
|practice_0168|E2|52-1|133||131|válido|
|practice_0168|E2|52-3|133||131|válido|
|practice_0168|E2|52-6|132||130|válido|
|practice_0168|E2|64-1|133||132|válido|
|practice_0168|E2|64-3|133||132|válido|
|practice_0168|E2|64-6|132||130|válido|
|practice_0168|E3|joint|132|individual 132|132|OPTIMAL|
|practice_0168|E4|repair|132||131|completo|
|practice_0169|E1|1|6|postprocess 5, optimize_a4 4, nested_windows 4|4|válido|
|practice_0169|E1|3|6|postprocess 5, optimize_a4 4, nested_windows 4|4|válido|
|practice_0169|E1|6|6|postprocess 5, optimize_a4 4, nested_windows 4|4|válido|
|practice_0169|E2|52-1|6||6|válido|
|practice_0169|E2|52-3|6||6|válido|
|practice_0169|E2|52-6|6||6|válido|
|practice_0169|E2|64-1|6||6|válido|
|practice_0169|E2|64-3|6||6|válido|
|practice_0169|E2|64-6|6||6|válido|
|practice_0169|E3|joint|5|individual 5|5|OPTIMAL|
|practice_0169|E4|repair|5||5|completo|
|practice_0170|E1|1|41|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|practice_0170|E1|3|41|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|practice_0170|E1|6|41|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|practice_0170|E2|52-1|41||41|válido|
|practice_0170|E2|52-3|41||41|válido|
|practice_0170|E2|52-6|41||41|válido|
|practice_0170|E2|64-1|41||41|válido|
|practice_0170|E2|64-3|41||41|válido|
|practice_0170|E2|64-6|41||41|válido|
|practice_0170|E3|joint|38|individual 38|38|OPTIMAL|
|practice_0170|E4|repair|38||36|completo|
|practice_0171|E1|1|342|postprocess 340, optimize_a4 340, nested_windows 340|340|válido|
|practice_0171|E1|3|327|postprocess 326, optimize_a4 326, nested_windows 326|326|válido|
|practice_0171|E1|6|311|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|practice_0171|E2|52-1|342||302|válido|
|practice_0171|E2|52-3|327||295|válido|
|practice_0171|E2|52-6|311||292|válido|
|practice_0171|E2|64-1|342||305|válido|
|practice_0171|E2|64-3|327||297|válido|
|practice_0171|E2|64-6|311||292|válido|
|practice_0171|E3|joint|311|individual 310|309|OPTIMAL|
|practice_0171|E4|repair|311||310|completo|
|practice_0172|E1|1|2|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0172|E1|3|2|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0172|E1|6|2|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0172|E2|52-1|2||2|válido|
|practice_0172|E2|52-3|2||2|válido|
|practice_0172|E2|52-6|2||2|válido|
|practice_0172|E2|64-1|2||2|válido|
|practice_0172|E2|64-3|2||2|válido|
|practice_0172|E2|64-6|2||2|válido|
|practice_0172|E3|joint|2|individual 2|2|OPTIMAL|
|practice_0172|E4|repair|2||2|completo|
|practice_0173|E1|1|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|practice_0173|E1|3|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|practice_0173|E1|6|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|practice_0173|E2|52-1|165||153|válido|
|practice_0173|E2|52-3|159||147|válido|
|practice_0173|E2|52-6|159||147|válido|
|practice_0173|E2|64-1|165||157|válido|
|practice_0173|E2|64-3|159||154|válido|
|practice_0173|E2|64-6|159||152|válido|
|practice_0173|E3|joint|159|individual 159|159|OPTIMAL|
|practice_0173|E4|repair|159||157|completo|
|practice_0174|E1|1|2|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0174|E1|3|2|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0174|E1|6|2|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|practice_0174|E2|52-1|2||2|válido|
|practice_0174|E2|52-3|2||2|válido|
|practice_0174|E2|52-6|2||2|válido|
|practice_0174|E2|64-1|2||2|válido|
|practice_0174|E2|64-3|2||2|válido|
|practice_0174|E2|64-6|2||2|válido|
|practice_0174|E3|joint|2|individual 2|2|OPTIMAL|
|practice_0174|E4|repair|2||2|completo|
|practice_0175|E1|1|6|postprocess 5, optimize_a4 5, nested_windows 5|5|válido|
|practice_0175|E1|3|6|postprocess 5, optimize_a4 5, nested_windows 5|5|válido|
|practice_0175|E1|6|6|postprocess 5, optimize_a4 5, nested_windows 5|5|válido|
|practice_0175|E2|52-1|6||6|válido|
|practice_0175|E2|52-3|6||6|válido|
|practice_0175|E2|52-6|6||6|válido|
|practice_0175|E2|64-1|6||6|válido|
|practice_0175|E2|64-3|6||6|válido|
|practice_0175|E2|64-6|6||6|válido|
|practice_0175|E3|joint|5|individual 5|5|OPTIMAL|
|practice_0175|E4|repair|5||5|completo|
|practice_0176|E1|1|362|postprocess 362, optimize_a4 362, nested_windows 362|362|válido|
|practice_0176|E1|3|340|postprocess 339, optimize_a4 339, nested_windows 339|339|válido|
|practice_0176|E1|6|337|postprocess 337, optimize_a4 337, nested_windows 337|337|válido|
|practice_0176|E2|52-1|362||334|válido|
|practice_0176|E2|52-3|340||325|válido|
|practice_0176|E2|52-6|337||318|válido|
|practice_0176|E2|64-1|362||340|válido|
|practice_0176|E2|64-3|340||326|válido|
|practice_0176|E2|64-6|337||322|válido|
|practice_0176|E3|joint|337|individual 336|335|FEASIBLE|
|practice_0176|E4|repair|337||337|completo|
|practice_0177|E1|1|200|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|practice_0177|E1|3|197|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|practice_0177|E1|6|193|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|practice_0177|E2|52-1|200||189|válido|
|practice_0177|E2|52-3|197||179|válido|
|practice_0177|E2|52-6|193||179|válido|
|practice_0177|E2|64-1|200||188|válido|
|practice_0177|E2|64-3|197||179|válido|
|practice_0177|E2|64-6|193||179|válido|
|practice_0177|E3|joint|193|individual 193|193|OPTIMAL|
|practice_0177|E4|repair|193||192|completo|
|practice_0178|E1|1|97|postprocess 95, optimize_a4 95, nested_windows 95|95|válido|
|practice_0178|E1|3|97|postprocess 95, optimize_a4 95, nested_windows 95|95|válido|
|practice_0178|E1|6|97|postprocess 95, optimize_a4 95, nested_windows 95|95|válido|
|practice_0178|E2|52-1|97||93|válido|
|practice_0178|E2|52-3|97||93|válido|
|practice_0178|E2|52-6|97||93|válido|
|practice_0178|E2|64-1|97||93|válido|
|practice_0178|E2|64-3|97||93|válido|
|practice_0178|E2|64-6|97||93|válido|
|practice_0178|E3|joint|97|individual 97|97|OPTIMAL|
|practice_0178|E4|repair|97||95|completo|
|practice_0179|E1|1|385|postprocess 385, optimize_a4 385, nested_windows 385|385|válido|
|practice_0179|E1|3|378|postprocess 378, optimize_a4 378, nested_windows 378|378|válido|
|practice_0179|E1|6|371|postprocess 371, optimize_a4 371, nested_windows 371|371|válido|
|practice_0179|E2|52-1|385||363|válido|
|practice_0179|E2|52-3|378||350|válido|
|practice_0179|E2|52-6|371||342|válido|
|practice_0179|E2|64-1|385||367|válido|
|practice_0179|E2|64-3|378||514|válido|
|practice_0179|E2|64-6|371||430|válido|
|practice_0179|E3|joint|371|individual 370|368|OPTIMAL|
|practice_0179|E4|repair|371||371|completo|
|practice_0180|E1|1|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|practice_0180|E1|3|160|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|practice_0180|E1|6|160|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|practice_0180|E2|52-1|166||156|válido|
|practice_0180|E2|52-3|160||155|válido|
|practice_0180|E2|52-6|160||155|válido|
|practice_0180|E2|64-1|166||157|válido|
|practice_0180|E2|64-3|160||157|válido|
|practice_0180|E2|64-6|160||157|válido|
|practice_0180|E3|joint|160|individual 160|160|OPTIMAL|
|practice_0180|E4|repair|160||159|completo|
|practice_0181|E1|1|329|postprocess 328, optimize_a4 328, nested_windows 328|328|válido|
|practice_0181|E1|3|311|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|practice_0181|E1|6|300|postprocess 300, optimize_a4 300, nested_windows 300|300|válido|
|practice_0181|E2|52-1|329||297|válido|
|practice_0181|E2|52-3|311||286|válido|
|practice_0181|E2|52-6|300||281|válido|
|practice_0181|E2|64-1|329||297|válido|
|practice_0181|E2|64-3|311||285|válido|
|practice_0181|E2|64-6|300||285|válido|
|practice_0181|E3|joint|300|individual 300|300|FEASIBLE|
|practice_0181|E4|repair|300||300|completo|
|practice_0182|E1|1|253|postprocess 253, optimize_a4 253, nested_windows 253|253|válido|
|practice_0182|E1|3|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|practice_0182|E1|6|230|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|practice_0182|E2|52-1|253||226|válido|
|practice_0182|E2|52-3|242||221|válido|
|practice_0182|E2|52-6|230||220|válido|
|practice_0182|E2|64-1|253||225|válido|
|practice_0182|E2|64-3|242||221|válido|
|practice_0182|E2|64-6|230||220|válido|
|practice_0182|E3|joint|230|individual 230|230|OPTIMAL|
|practice_0182|E4|repair|230||230|completo|
|practice_0183|E1|1|332|postprocess 332, optimize_a4 332, nested_windows 332|332|válido|
|practice_0183|E1|3|321|postprocess 320, optimize_a4 320, nested_windows 320|320|válido|
|practice_0183|E1|6|311|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|practice_0183|E2|52-1|332||287|válido|
|practice_0183|E2|52-3|321||286|válido|
|practice_0183|E2|52-6|311||287|válido|
|practice_0183|E2|64-1|332||294|válido|
|practice_0183|E2|64-3|321||290|válido|
|practice_0183|E2|64-6|311||292|válido|
|practice_0183|E3|joint|311|individual 311|311|rechazado por registros|
|practice_0183|E4|repair|311||311|completo|
|practice_0184|E1|1|283|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|practice_0184|E1|3|275|postprocess 275, optimize_a4 275, nested_windows 275|275|válido|
|practice_0184|E1|6|269|postprocess 269, optimize_a4 269, nested_windows 269|269|válido|
|practice_0184|E2|52-1|283||260|válido|
|practice_0184|E2|52-3|275||255|válido|
|practice_0184|E2|52-6|269||256|válido|
|practice_0184|E2|64-1|283||261|válido|
|practice_0184|E2|64-3|275||255|válido|
|practice_0184|E2|64-6|269||254|válido|
|practice_0184|E3|joint|269|individual 269|267|OPTIMAL|
|practice_0184|E4|repair|269||269|completo|
|practice_0185|E1|1|191|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|practice_0185|E1|3|191|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|practice_0185|E1|6|189|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|practice_0185|E2|52-1|191||181|válido|
|practice_0185|E2|52-3|191||180|válido|
|practice_0185|E2|52-6|189||177|válido|
|practice_0185|E2|64-1|191||183|válido|
|practice_0185|E2|64-3|191||181|válido|
|practice_0185|E2|64-6|189||172|válido|
|practice_0185|E3|joint|189|individual 189|188|OPTIMAL|
|practice_0185|E4|repair|189||186|completo|
|practice_0186|E1|1|126|postprocess 126, optimize_a4 126, nested_windows 126|126|válido|
|practice_0186|E1|3|128|postprocess 127, optimize_a4 127, nested_windows 127|127|válido|
|practice_0186|E1|6|125|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|practice_0186|E2|52-1|126||121|válido|
|practice_0186|E2|52-3|128||120|válido|
|practice_0186|E2|52-6|125||118|válido|
|practice_0186|E2|64-1|126||121|válido|
|practice_0186|E2|64-3|128||120|válido|
|practice_0186|E2|64-6|125||118|válido|
|practice_0186|E3|joint|125|individual 125|125|OPTIMAL|
|practice_0186|E4|repair|125||125|completo|
|practice_0187|E1|1|240|postprocess 240, optimize_a4 240, nested_windows 240|240|válido|
|practice_0187|E1|3|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|practice_0187|E1|6|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|practice_0187|E2|52-1|240||225|válido|
|practice_0187|E2|52-3|232||218|válido|
|practice_0187|E2|52-6|232||218|válido|
|practice_0187|E2|64-1|240||223|válido|
|practice_0187|E2|64-3|232||219|válido|
|practice_0187|E2|64-6|232||218|válido|
|practice_0187|E3|joint|232|individual 231|230|OPTIMAL|
|practice_0187|E4|repair|232||232|completo|
|practice_0188|E1|1|246|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|practice_0188|E1|3|241|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|practice_0188|E1|6|240|postprocess 240, optimize_a4 240, nested_windows 240|240|válido|
|practice_0188|E2|52-1|246||235|válido|
|practice_0188|E2|52-3|241||226|válido|
|practice_0188|E2|52-6|240||224|válido|
|practice_0188|E2|64-1|246||231|válido|
|practice_0188|E2|64-3|241||226|válido|
|practice_0188|E2|64-6|240||227|válido|
|practice_0188|E3|joint|240|individual 240|239|OPTIMAL|
|practice_0188|E4|repair|240||240|completo|
|practice_0189|E1|1|212|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|practice_0189|E1|3|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|practice_0189|E1|6|203|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|practice_0189|E2|52-1|212||189|válido|
|practice_0189|E2|52-3|206||188|válido|
|practice_0189|E2|52-6|203||187|válido|
|practice_0189|E2|64-1|212||187|válido|
|practice_0189|E2|64-3|206||184|válido|
|practice_0189|E2|64-6|203||188|válido|
|practice_0189|E3|joint|203|individual 203|202|OPTIMAL|
|practice_0189|E4|repair|203||203|completo|
|practice_0190|E1|1|343|postprocess 343, optimize_a4 343, nested_windows 343|343|válido|
|practice_0190|E1|3|326|postprocess 326, optimize_a4 326, nested_windows 326|326|válido|
|practice_0190|E1|6|316|postprocess 316, optimize_a4 316, nested_windows 316|316|válido|
|practice_0190|E2|52-1|343||312|válido|
|practice_0190|E2|52-3|326||297|válido|
|practice_0190|E2|52-6|316||295|válido|
|practice_0190|E2|64-1|343||312|válido|
|practice_0190|E2|64-3|326||300|válido|
|practice_0190|E2|64-6|316||298|válido|
|practice_0190|E3|joint|316|individual 316|316|OPTIMAL|
|practice_0190|E4|repair|316||316|completo|
|practice_0191|E1|1|370|postprocess 370, optimize_a4 370, nested_windows 370|370|válido|
|practice_0191|E1|3|348|postprocess 348, optimize_a4 348, nested_windows 348|348|válido|
|practice_0191|E1|6|337|postprocess 336, optimize_a4 336, nested_windows 336|336|válido|
|practice_0191|E2|52-1|370||332|válido|
|practice_0191|E2|52-3|348||315|válido|
|practice_0191|E2|52-6|337||313|válido|
|practice_0191|E2|64-1|370||332|válido|
|practice_0191|E2|64-3|348||318|válido|
|practice_0191|E2|64-6|337||313|válido|
|practice_0191|E3|joint|337|individual 337|337|OPTIMAL|
|practice_0191|E4|repair|337||337|completo|
|practice_0192|E1|1|129|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|practice_0192|E1|3|127|postprocess 127, optimize_a4 127, nested_windows 127|127|válido|
|practice_0192|E1|6|127|postprocess 127, optimize_a4 127, nested_windows 127|127|válido|
|practice_0192|E2|52-1|129||126|válido|
|practice_0192|E2|52-3|127||125|válido|
|practice_0192|E2|52-6|127||125|válido|
|practice_0192|E2|64-1|129||126|válido|
|practice_0192|E2|64-3|127||126|válido|
|practice_0192|E2|64-6|127||125|válido|
|practice_0192|E3|joint|127|individual 127|127|OPTIMAL|
|practice_0192|E4|repair|127||124|completo|
|practice_0193|E1|1|138|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|practice_0193|E1|3|139|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|practice_0193|E1|6|138|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|practice_0193|E2|52-1|138||136|válido|
|practice_0193|E2|52-3|139||141|válido|
|practice_0193|E2|52-6|138||141|válido|
|practice_0193|E2|64-1|138||135|válido|
|practice_0193|E2|64-3|139||139|válido|
|practice_0193|E2|64-6|138||138|válido|
|practice_0193|E3|joint|138|individual 138|137|OPTIMAL|
|practice_0193|E4|repair|138||136|completo|
|practice_0194|E1|1|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|practice_0194|E1|3|278|postprocess 278, optimize_a4 278, nested_windows 278|278|válido|
|practice_0194|E1|6|274|postprocess 273, optimize_a4 273, nested_windows 273|273|válido|
|practice_0194|E2|52-1|289||259|válido|
|practice_0194|E2|52-3|278||255|válido|
|practice_0194|E2|52-6|274||249|válido|
|practice_0194|E2|64-1|289||263|válido|
|practice_0194|E2|64-3|278||256|válido|
|practice_0194|E2|64-6|274||251|válido|
|practice_0194|E3|joint|274|individual 274|271|FEASIBLE|
|practice_0194|E4|repair|274||274|completo|
|practice_0195|E1|1|296|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|practice_0195|E1|3|291|postprocess 291, optimize_a4 291, nested_windows 291|291|válido|
|practice_0195|E1|6|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|practice_0195|E2|52-1|296||280|válido|
|practice_0195|E2|52-3|291||269|válido|
|practice_0195|E2|52-6|289||262|válido|
|practice_0195|E2|64-1|296||284|válido|
|practice_0195|E2|64-3|291||270|válido|
|practice_0195|E2|64-6|289||263|válido|
|practice_0195|E3|joint|289|individual 289|286|FEASIBLE|
|practice_0195|E4|repair|289||289|completo|
|practice_0196|E1|1|144|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|practice_0196|E1|3|145|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|practice_0196|E1|6|143|postprocess 141, optimize_a4 141, nested_windows 141|141|válido|
|practice_0196|E2|52-1|144||140|válido|
|practice_0196|E2|52-3|145||140|válido|
|practice_0196|E2|52-6|143||137|válido|
|practice_0196|E2|64-1|144||140|válido|
|practice_0196|E2|64-3|145||140|válido|
|practice_0196|E2|64-6|143||137|válido|
|practice_0196|E3|joint|143|individual 143|142|OPTIMAL|
|practice_0196|E4|repair|143||137|completo|
|practice_0197|E1|1|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|practice_0197|E1|3|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|practice_0197|E1|6|224|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|practice_0197|E2|52-1|235||218|válido|
|practice_0197|E2|52-3|223||213|válido|
|practice_0197|E2|52-6|224||209|válido|
|practice_0197|E2|64-1|235||218|válido|
|practice_0197|E2|64-3|223||211|válido|
|practice_0197|E2|64-6|224||207|válido|
|practice_0197|E3|joint|223|individual 223|223|OPTIMAL|
|practice_0197|E4|repair|223||223|completo|
|practice_0198|E1|1|47|postprocess 46, optimize_a4 46, nested_windows 46|46|válido|
|practice_0198|E1|3|47|postprocess 46, optimize_a4 46, nested_windows 46|46|válido|
|practice_0198|E1|6|47|postprocess 46, optimize_a4 46, nested_windows 46|46|válido|
|practice_0198|E2|52-1|47||47|válido|
|practice_0198|E2|52-3|47||47|válido|
|practice_0198|E2|52-6|47||47|válido|
|practice_0198|E2|64-1|47||47|válido|
|practice_0198|E2|64-3|47||47|válido|
|practice_0198|E2|64-6|47||47|válido|
|practice_0198|E3|joint|47|individual 47|47|OPTIMAL|
|practice_0198|E4|repair|47||44|completo|
|practice_0199|E1|1|338|postprocess 337, optimize_a4 337, nested_windows 337|337|válido|
|practice_0199|E1|3|316|postprocess 316, optimize_a4 316, nested_windows 316|316|válido|
|practice_0199|E1|6|317|postprocess 315, optimize_a4 315, nested_windows 315|315|válido|
|practice_0199|E2|52-1|338||310|válido|
|practice_0199|E2|52-3|316||298|válido|
|practice_0199|E2|52-6|317||291|válido|
|practice_0199|E2|64-1|338||314|válido|
|practice_0199|E2|64-3|316||297|válido|
|practice_0199|E2|64-6|317||292|válido|
|practice_0199|E3|joint|316|individual 316|313|OPTIMAL|
|practice_0199|E4|repair|316||315|completo|
|practice_0200|E1|1|155|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|practice_0200|E1|3|153|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|practice_0200|E1|6|152|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|practice_0200|E2|52-1|155||144|válido|
|practice_0200|E2|52-3|153||145|válido|
|practice_0200|E2|52-6|152||145|válido|
|practice_0200|E2|64-1|155||141|válido|
|practice_0200|E2|64-3|153||140|válido|
|practice_0200|E2|64-6|152||139|válido|
|practice_0200|E3|joint|152|individual 152|152|OPTIMAL|
|practice_0200|E4|repair|152||152|completo|

## Cuatro experimentos sobre los 1.000 puntuables

Cierre validado: 177352 pasos, 11565 ahorrados. [Programas](reports/four_backward_scored/portfolio.csv) · [Ganadores](reports/four_backward_scored/comparison.csv)

Piloto completado: 22 configuraciones en 2 IDs (puntuables sin etiqueta de familia). Suite del entorno: 42 pruebas aprobadas.

| Diccionario y ancho | Control original | Ampliado | Diferencia a favor del ampliado |
|---|---:|---:|---:|
|52-1|197890|184113|13777|
|52-3|191867|180225|11642|
|52-6|189141|178565|10576|
|64-1|197890|184973|12917|
|64-3|191867|181202|10665|
|64-6|189141|179345|9796|

E1 elegir antes: 188505 · E1 elegir después: 188484 · E3 individual: 188677 · E3 con CP-SAT: 188118

| Método | IDs elegidos | Aporte exclusivo |
|---|---:|---:|
|E1|80|125|
|E2|828|9782|
|E3|3|4|
|E4|62|117|

| Familia | E1 | E2 | E3 | E4 | Cartera |
|---|---:|---:|---:|---:|---:|
|Sin etiqueta|188484|177730|188118|187980|177352|

Programas reproducidos por método: [E1](reports/four_backward_scored/E1.csv) · [E2](reports/four_backward_scored/E2.csv) · [E3](reports/four_backward_scored/E3.csv) · [E4](reports/four_backward_scored/E4.csv)

Estados de CP-SAT: FEASIBLE 82, OPTIMAL 910, rechazado por registros 8

| Método | IDs que mejoran el histórico | Pasos ahorrados en esos IDs |
|---|---:|---:|
|E1|304|450|
|E2|861|11256|
|E3|414|799|
|E4|402|937|

Referencia validada: 188.917 pasos en 1000 IDs. Haz 1/3/6. E2: 52/72 y 64/72. CP-SAT 9.15.6755, semilla 42, un hilo y 30 s. Reparación local: 30 s. Grupo compartido: 14 procesos.

| Método | IDs completos | Histórico | Método | Ahorro al combinar | CPU s | Tiempo acumulado s |
|---|---:|---:|---:|---:|---:|---:|
|E1|1000/1000|188917|188484|450|458.2|467.4|
|E2|1000/1000|188917|177730|11256|15886.2|16127.5|
|E3|1000/1000|188917|188118|799|3918.3|3924.4|
|E4|1000/1000|188917|187980|937|3016.4|3028.5|

Comparación directa sobre 1000 IDs comunes. E1: 188484 · E2: 177730 · E3: 188118 · E4: 187980

Detalle de resultados recibidos. Las configuraciones restantes están pendientes.

| ID | Método | Configuración | Entrada | Etapas (pasos) | Final validado | Estado |
|---|---|---|---:|---|---:|---|
|scored_0001|E1|1|318|postprocess 317, optimize_a4 317, nested_windows 317|317|válido|
|scored_0001|E1|3|310|postprocess 310, optimize_a4 310, nested_windows 310|310|válido|
|scored_0001|E1|6|304|postprocess 304, optimize_a4 304, nested_windows 304|304|válido|
|scored_0001|E2|52-1|318||298|válido|
|scored_0001|E2|52-3|310||297|válido|
|scored_0001|E2|52-6|304||289|válido|
|scored_0001|E2|64-1|318||301|válido|
|scored_0001|E2|64-3|310||297|válido|
|scored_0001|E2|64-6|304||291|válido|
|scored_0001|E3|joint|304|individual 304|302|FEASIBLE|
|scored_0001|E4|repair|304||304|completo|
|scored_0002|E1|1|348|postprocess 348, optimize_a4 348, nested_windows 348|348|válido|
|scored_0002|E1|3|385|postprocess 370, optimize_a4 370, nested_windows 370|370|válido|
|scored_0002|E1|6|322|postprocess 322, optimize_a4 322, nested_windows 322|322|válido|
|scored_0002|E2|52-1|348||316|válido|
|scored_0002|E2|52-3|385||311|válido|
|scored_0002|E2|52-6|322||304|válido|
|scored_0002|E2|64-1|348||321|válido|
|scored_0002|E2|64-3|385||313|válido|
|scored_0002|E2|64-6|322||305|válido|
|scored_0002|E3|joint|322|individual 322|321|FEASIBLE|
|scored_0002|E4|repair|322||322|completo|
|scored_0003|E1|1|310|postprocess 310, optimize_a4 310, nested_windows 310|310|válido|
|scored_0003|E1|3|296|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0003|E1|6|284|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|scored_0003|E2|52-1|310||281|válido|
|scored_0003|E2|52-3|296||271|válido|
|scored_0003|E2|52-6|284||266|válido|
|scored_0003|E2|64-1|310||279|válido|
|scored_0003|E2|64-3|296||270|válido|
|scored_0003|E2|64-6|284||265|válido|
|scored_0003|E3|joint|284|individual 284|284|OPTIMAL|
|scored_0003|E4|repair|284||284|completo|
|scored_0004|E1|1|300|postprocess 299, optimize_a4 299, nested_windows 299|299|válido|
|scored_0004|E1|3|281|postprocess 281, optimize_a4 281, nested_windows 281|281|válido|
|scored_0004|E1|6|282|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|scored_0004|E2|52-1|300||276|válido|
|scored_0004|E2|52-3|281||275|válido|
|scored_0004|E2|52-6|282||269|válido|
|scored_0004|E2|64-1|300||276|válido|
|scored_0004|E2|64-3|281||276|válido|
|scored_0004|E2|64-6|282||268|válido|
|scored_0004|E3|joint|281|individual 280|280|OPTIMAL|
|scored_0004|E4|repair|281||281|completo|
|scored_0005|E1|1|71|postprocess 69, optimize_a4 69, nested_windows 69|69|válido|
|scored_0005|E1|3|71|postprocess 69, optimize_a4 69, nested_windows 69|69|válido|
|scored_0005|E1|6|71|postprocess 69, optimize_a4 69, nested_windows 69|69|válido|
|scored_0005|E2|52-1|71||70|válido|
|scored_0005|E2|52-3|71||70|válido|
|scored_0005|E2|52-6|71||70|válido|
|scored_0005|E2|64-1|71||70|válido|
|scored_0005|E2|64-3|71||70|válido|
|scored_0005|E2|64-6|71||70|válido|
|scored_0005|E3|joint|71|individual 71|71|OPTIMAL|
|scored_0005|E4|repair|71||69|completo|
|scored_0006|E1|1|189|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0006|E1|3|184|postprocess 184, optimize_a4 184, nested_windows 184|184|válido|
|scored_0006|E1|6|183|postprocess 183, optimize_a4 183, nested_windows 183|183|válido|
|scored_0006|E2|52-1|189||176|válido|
|scored_0006|E2|52-3|184||174|válido|
|scored_0006|E2|52-6|183||175|válido|
|scored_0006|E2|64-1|189||177|válido|
|scored_0006|E2|64-3|184||178|válido|
|scored_0006|E2|64-6|183||174|válido|
|scored_0006|E3|joint|183|individual 183|182|OPTIMAL|
|scored_0006|E4|repair|183||179|completo|
|scored_0007|E1|1|263|postprocess 263, optimize_a4 263, nested_windows 263|263|válido|
|scored_0007|E1|3|244|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|scored_0007|E1|6|241|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0007|E2|52-1|263||247|válido|
|scored_0007|E2|52-3|244||225|válido|
|scored_0007|E2|52-6|241||222|válido|
|scored_0007|E2|64-1|263||252|válido|
|scored_0007|E2|64-3|244||225|válido|
|scored_0007|E2|64-6|241||224|válido|
|scored_0007|E3|joint|241|individual 241|238|OPTIMAL|
|scored_0007|E4|repair|241||241|completo|
|scored_0008|E1|1|173|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0008|E1|3|171|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|scored_0008|E1|6|171|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|scored_0008|E2|52-1|173||163|válido|
|scored_0008|E2|52-3|171||163|válido|
|scored_0008|E2|52-6|171||163|válido|
|scored_0008|E2|64-1|173||163|válido|
|scored_0008|E2|64-3|171||163|válido|
|scored_0008|E2|64-6|171||163|válido|
|scored_0008|E3|joint|171|individual 171|170|OPTIMAL|
|scored_0008|E4|repair|171||171|completo|
|scored_0009|E1|1|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0009|E1|3|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0009|E1|6|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0009|E2|52-1|50||50|válido|
|scored_0009|E2|52-3|50||50|válido|
|scored_0009|E2|52-6|50||50|válido|
|scored_0009|E2|64-1|50||50|válido|
|scored_0009|E2|64-3|50||50|válido|
|scored_0009|E2|64-6|50||50|válido|
|scored_0009|E3|joint|50|individual 50|50|OPTIMAL|
|scored_0009|E4|repair|50||50|completo|
|scored_0010|E1|1|250|postprocess 250, optimize_a4 250, nested_windows 250|250|válido|
|scored_0010|E1|3|241|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|scored_0010|E1|6|241|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|scored_0010|E2|52-1|250||234|válido|
|scored_0010|E2|52-3|241||225|válido|
|scored_0010|E2|52-6|241||224|válido|
|scored_0010|E2|64-1|250||236|válido|
|scored_0010|E2|64-3|241||226|válido|
|scored_0010|E2|64-6|241||224|válido|
|scored_0010|E3|joint|241|individual 241|240|OPTIMAL|
|scored_0010|E4|repair|241||241|completo|
|scored_0011|E1|1|211|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0011|E1|3|200|postprocess 197, optimize_a4 196, nested_windows 196|196|válido|
|scored_0011|E1|6|190|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|scored_0011|E2|52-1|211||194|válido|
|scored_0011|E2|52-3|200||189|válido|
|scored_0011|E2|52-6|190||181|válido|
|scored_0011|E2|64-1|211||189|válido|
|scored_0011|E2|64-3|200||184|válido|
|scored_0011|E2|64-6|190||175|válido|
|scored_0011|E3|joint|190|individual 189|189|OPTIMAL|
|scored_0011|E4|repair|190||189|completo|
|scored_0012|E1|1|254|postprocess 253, optimize_a4 253, nested_windows 253|253|válido|
|scored_0012|E1|3|246|postprocess 245, optimize_a4 245, nested_windows 245|245|válido|
|scored_0012|E1|6|245|postprocess 245, optimize_a4 245, nested_windows 245|245|válido|
|scored_0012|E2|52-1|254||232|válido|
|scored_0012|E2|52-3|246||230|válido|
|scored_0012|E2|52-6|245||226|válido|
|scored_0012|E2|64-1|254||231|válido|
|scored_0012|E2|64-3|246||232|válido|
|scored_0012|E2|64-6|245||226|válido|
|scored_0012|E3|joint|245|individual 245|244|OPTIMAL|
|scored_0012|E4|repair|245||245|completo|
|scored_0013|E1|1|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0013|E1|3|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0013|E1|6|237|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|scored_0013|E2|52-1|239||217|válido|
|scored_0013|E2|52-3|239||219|válido|
|scored_0013|E2|52-6|237||222|válido|
|scored_0013|E2|64-1|239||220|válido|
|scored_0013|E2|64-3|239||219|válido|
|scored_0013|E2|64-6|237||221|válido|
|scored_0013|E3|joint|237|individual 236|235|OPTIMAL|
|scored_0013|E4|repair|237||237|completo|
|scored_0014|E1|1|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0014|E1|3|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0014|E1|6|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0014|E2|52-1|41||41|válido|
|scored_0014|E2|52-3|41||41|válido|
|scored_0014|E2|52-6|41||41|válido|
|scored_0014|E2|64-1|41||41|válido|
|scored_0014|E2|64-3|41||41|válido|
|scored_0014|E2|64-6|41||41|válido|
|scored_0014|E3|joint|41|individual 41|41|OPTIMAL|
|scored_0014|E4|repair|41||41|completo|
|scored_0015|E1|1|317|postprocess 317, optimize_a4 317, nested_windows 317|317|válido|
|scored_0015|E1|3|295|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|scored_0015|E1|6|298|postprocess 298, optimize_a4 298, nested_windows 298|298|válido|
|scored_0015|E2|52-1|317||291|válido|
|scored_0015|E2|52-3|295||276|válido|
|scored_0015|E2|52-6|298||276|válido|
|scored_0015|E2|64-1|317||293|válido|
|scored_0015|E2|64-3|295||277|válido|
|scored_0015|E2|64-6|298||276|válido|
|scored_0015|E3|joint|295|individual 294|291|FEASIBLE|
|scored_0015|E4|repair|295||295|completo|
|scored_0016|E1|1|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0016|E1|3|231|postprocess 230, optimize_a4 230, nested_windows 230|230|válido|
|scored_0016|E1|6|225|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0016|E2|52-1|242||222|válido|
|scored_0016|E2|52-3|231||207|válido|
|scored_0016|E2|52-6|225||203|válido|
|scored_0016|E2|64-1|242||222|válido|
|scored_0016|E2|64-3|231||210|válido|
|scored_0016|E2|64-6|225||205|válido|
|scored_0016|E3|joint|225|individual 225|224|OPTIMAL|
|scored_0016|E4|repair|225||225|completo|
|scored_0017|E1|1|176|postprocess 176, optimize_a4 176, nested_windows 176|176|válido|
|scored_0017|E1|3|176|postprocess 176, optimize_a4 176, nested_windows 176|176|válido|
|scored_0017|E1|6|172|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|scored_0017|E2|52-1|176||168|válido|
|scored_0017|E2|52-3|176||165|válido|
|scored_0017|E2|52-6|172||161|válido|
|scored_0017|E2|64-1|176||166|válido|
|scored_0017|E2|64-3|176||165|válido|
|scored_0017|E2|64-6|172||161|válido|
|scored_0017|E3|joint|172|individual 171|171|OPTIMAL|
|scored_0017|E4|repair|172||172|completo|
|scored_0018|E1|1|175|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|scored_0018|E1|3|173|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|scored_0018|E1|6|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0018|E2|52-1|175||170|válido|
|scored_0018|E2|52-3|173||165|válido|
|scored_0018|E2|52-6|172||169|válido|
|scored_0018|E2|64-1|175||170|válido|
|scored_0018|E2|64-3|173||167|válido|
|scored_0018|E2|64-6|172||169|válido|
|scored_0018|E3|joint|172|individual 172|172|OPTIMAL|
|scored_0018|E4|repair|172||172|completo|
|scored_0019|E1|1|36|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0019|E1|3|36|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0019|E1|6|36|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0019|E2|52-1|36||36|válido|
|scored_0019|E2|52-3|36||36|válido|
|scored_0019|E2|52-6|36||36|válido|
|scored_0019|E2|64-1|36||36|válido|
|scored_0019|E2|64-3|36||36|válido|
|scored_0019|E2|64-6|36||36|válido|
|scored_0019|E3|joint|36|individual 36|36|OPTIMAL|
|scored_0019|E4|repair|36||36|completo|
|scored_0020|E1|1|358|postprocess 358, optimize_a4 358, nested_windows 358|358|válido|
|scored_0020|E1|3|330|postprocess 328, optimize_a4 328, nested_windows 328|328|válido|
|scored_0020|E1|6|324|postprocess 324, optimize_a4 324, nested_windows 324|324|válido|
|scored_0020|E2|52-1|358||315|válido|
|scored_0020|E2|52-3|330||311|válido|
|scored_0020|E2|52-6|324||303|válido|
|scored_0020|E2|64-1|358||319|válido|
|scored_0020|E2|64-3|330||309|válido|
|scored_0020|E2|64-6|324||301|válido|
|scored_0020|E3|joint|324|individual 324|324|OPTIMAL|
|scored_0020|E4|repair|324||324|completo|
|scored_0021|E1|1|55|postprocess 54, optimize_a4 54, nested_windows 54|54|válido|
|scored_0021|E1|3|55|postprocess 54, optimize_a4 54, nested_windows 54|54|válido|
|scored_0021|E1|6|55|postprocess 54, optimize_a4 54, nested_windows 54|54|válido|
|scored_0021|E2|52-1|55||55|válido|
|scored_0021|E2|52-3|55||55|válido|
|scored_0021|E2|52-6|55||55|válido|
|scored_0021|E2|64-1|55||55|válido|
|scored_0021|E2|64-3|55||55|válido|
|scored_0021|E2|64-6|55||55|válido|
|scored_0021|E3|joint|55|individual 55|55|OPTIMAL|
|scored_0021|E4|repair|55||54|completo|
|scored_0022|E1|1|170|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0022|E1|3|169|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0022|E1|6|168|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0022|E2|52-1|170||156|válido|
|scored_0022|E2|52-3|169||156|válido|
|scored_0022|E2|52-6|168||156|válido|
|scored_0022|E2|64-1|170||155|válido|
|scored_0022|E2|64-3|169||155|válido|
|scored_0022|E2|64-6|168||155|válido|
|scored_0022|E3|joint|168|individual 168|168|OPTIMAL|
|scored_0022|E4|repair|168||166|completo|
|scored_0023|E1|1|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0023|E1|3|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0023|E1|6|214|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0023|E2|52-1|220||207|válido|
|scored_0023|E2|52-3|215||204|válido|
|scored_0023|E2|52-6|214||201|válido|
|scored_0023|E2|64-1|220||209|válido|
|scored_0023|E2|64-3|215||205|válido|
|scored_0023|E2|64-6|214||201|válido|
|scored_0023|E3|joint|214|individual 214|212|OPTIMAL|
|scored_0023|E4|repair|214||214|completo|
|scored_0024|E1|1|43|postprocess 43, optimize_a4 42, nested_windows 42|42|válido|
|scored_0024|E1|3|43|postprocess 43, optimize_a4 42, nested_windows 42|42|válido|
|scored_0024|E1|6|43|postprocess 43, optimize_a4 42, nested_windows 42|42|válido|
|scored_0024|E2|52-1|43||43|válido|
|scored_0024|E2|52-3|43||43|válido|
|scored_0024|E2|52-6|43||43|válido|
|scored_0024|E2|64-1|43||43|válido|
|scored_0024|E2|64-3|43||43|válido|
|scored_0024|E2|64-6|43||43|válido|
|scored_0024|E3|joint|43|individual 43|43|OPTIMAL|
|scored_0024|E4|repair|43||43|completo|
|scored_0025|E1|1|194|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0025|E1|3|191|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|scored_0025|E1|6|187|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0025|E2|52-1|194||181|válido|
|scored_0025|E2|52-3|191||177|válido|
|scored_0025|E2|52-6|187||177|válido|
|scored_0025|E2|64-1|194||181|válido|
|scored_0025|E2|64-3|191||177|válido|
|scored_0025|E2|64-6|187||177|válido|
|scored_0025|E3|joint|187|individual 187|186|OPTIMAL|
|scored_0025|E4|repair|187||185|completo|
|scored_0026|E1|1|250|postprocess 250, optimize_a4 250, nested_windows 250|250|válido|
|scored_0026|E1|3|242|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|scored_0026|E1|6|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0026|E2|52-1|250||234|válido|
|scored_0026|E2|52-3|242||232|válido|
|scored_0026|E2|52-6|242||227|válido|
|scored_0026|E2|64-1|250||237|válido|
|scored_0026|E2|64-3|242||236|válido|
|scored_0026|E2|64-6|242||231|válido|
|scored_0026|E3|joint|242|individual 241|239|OPTIMAL|
|scored_0026|E4|repair|242||242|completo|
|scored_0027|E1|1|149|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0027|E1|3|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0027|E1|6|146|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0027|E2|52-1|149||145|válido|
|scored_0027|E2|52-3|148||140|válido|
|scored_0027|E2|52-6|146||139|válido|
|scored_0027|E2|64-1|149||143|válido|
|scored_0027|E2|64-3|148||140|válido|
|scored_0027|E2|64-6|146||138|válido|
|scored_0027|E3|joint|146|individual 146|146|OPTIMAL|
|scored_0027|E4|repair|146||140|completo|
|scored_0028|E1|1|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0028|E1|3|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0028|E1|6|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0028|E2|52-1|211||190|válido|
|scored_0028|E2|52-3|202||189|válido|
|scored_0028|E2|52-6|202||188|válido|
|scored_0028|E2|64-1|211||195|válido|
|scored_0028|E2|64-3|202||190|válido|
|scored_0028|E2|64-6|202||188|válido|
|scored_0028|E3|joint|202|individual 202|201|OPTIMAL|
|scored_0028|E4|repair|202||200|completo|
|scored_0029|E1|1|54|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0029|E1|3|54|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0029|E1|6|54|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0029|E2|52-1|54||54|válido|
|scored_0029|E2|52-3|54||54|válido|
|scored_0029|E2|52-6|54||54|válido|
|scored_0029|E2|64-1|54||54|válido|
|scored_0029|E2|64-3|54||54|válido|
|scored_0029|E2|64-6|54||54|válido|
|scored_0029|E3|joint|54|individual 54|54|OPTIMAL|
|scored_0029|E4|repair|54||53|completo|
|scored_0030|E1|1|279|postprocess 278, optimize_a4 278, nested_windows 278|278|válido|
|scored_0030|E1|3|274|postprocess 273, optimize_a4 273, nested_windows 273|273|válido|
|scored_0030|E1|6|265|postprocess 265, optimize_a4 265, nested_windows 265|265|válido|
|scored_0030|E2|52-1|279||264|válido|
|scored_0030|E2|52-3|274||257|válido|
|scored_0030|E2|52-6|265||250|válido|
|scored_0030|E2|64-1|279||265|válido|
|scored_0030|E2|64-3|274||259|válido|
|scored_0030|E2|64-6|265||253|válido|
|scored_0030|E3|joint|265|individual 265|265|OPTIMAL|
|scored_0030|E4|repair|265||265|completo|
|scored_0031|E1|1|347|postprocess 347, optimize_a4 347, nested_windows 347|347|válido|
|scored_0031|E1|3|327|postprocess 327, optimize_a4 327, nested_windows 327|327|válido|
|scored_0031|E1|6|321|postprocess 321, optimize_a4 321, nested_windows 321|321|válido|
|scored_0031|E2|52-1|347||318|válido|
|scored_0031|E2|52-3|327||308|válido|
|scored_0031|E2|52-6|321||301|válido|
|scored_0031|E2|64-1|347||323|válido|
|scored_0031|E2|64-3|327||310|válido|
|scored_0031|E2|64-6|321||302|válido|
|scored_0031|E3|joint|321|individual 320|320|FEASIBLE|
|scored_0031|E4|repair|321||321|completo|
|scored_0032|E1|1|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0032|E1|3|142|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0032|E1|6|142|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0032|E2|52-1|145||138|válido|
|scored_0032|E2|52-3|142||138|válido|
|scored_0032|E2|52-6|142||136|válido|
|scored_0032|E2|64-1|145||138|válido|
|scored_0032|E2|64-3|142||138|válido|
|scored_0032|E2|64-6|142||136|válido|
|scored_0032|E3|joint|142|individual 142|142|OPTIMAL|
|scored_0032|E4|repair|142||142|completo|
|scored_0033|E1|1|306|postprocess 306, optimize_a4 306, nested_windows 306|306|válido|
|scored_0033|E1|3|288|postprocess 288, optimize_a4 288, nested_windows 288|288|válido|
|scored_0033|E1|6|284|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|scored_0033|E2|52-1|306||277|válido|
|scored_0033|E2|52-3|288||268|válido|
|scored_0033|E2|52-6|284||263|válido|
|scored_0033|E2|64-1|306||275|válido|
|scored_0033|E2|64-3|288||269|válido|
|scored_0033|E2|64-6|284||260|válido|
|scored_0033|E3|joint|284|individual 284|283|FEASIBLE|
|scored_0033|E4|repair|284||284|completo|
|scored_0034|E1|1|231|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|scored_0034|E1|3|221|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0034|E1|6|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0034|E2|52-1|231||219|válido|
|scored_0034|E2|52-3|221||217|válido|
|scored_0034|E2|52-6|220||214|válido|
|scored_0034|E2|64-1|231||221|válido|
|scored_0034|E2|64-3|221||218|válido|
|scored_0034|E2|64-6|220||216|válido|
|scored_0034|E3|joint|220|individual 220|220|OPTIMAL|
|scored_0034|E4|repair|220||220|completo|
|scored_0035|E1|1|150|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|scored_0035|E1|3|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0035|E1|6|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0035|E2|52-1|150||139|válido|
|scored_0035|E2|52-3|148||142|válido|
|scored_0035|E2|52-6|148||142|válido|
|scored_0035|E2|64-1|150||138|válido|
|scored_0035|E2|64-3|148||140|válido|
|scored_0035|E2|64-6|148||140|válido|
|scored_0035|E3|joint|148|individual 148|148|OPTIMAL|
|scored_0035|E4|repair|148||148|completo|
|scored_0036|E1|1|343|postprocess 343, optimize_a4 343, nested_windows 343|343|válido|
|scored_0036|E1|3|329|postprocess 329, optimize_a4 329, nested_windows 329|329|válido|
|scored_0036|E1|6|313|postprocess 313, optimize_a4 313, nested_windows 313|313|válido|
|scored_0036|E2|52-1|343||310|válido|
|scored_0036|E2|52-3|329||298|válido|
|scored_0036|E2|52-6|313||295|válido|
|scored_0036|E2|64-1|343||308|válido|
|scored_0036|E2|64-3|329||300|válido|
|scored_0036|E2|64-6|313||295|válido|
|scored_0036|E3|joint|313|individual 313|313|FEASIBLE|
|scored_0036|E4|repair|313||313|completo|
|scored_0037|E1|1|278|postprocess 277, optimize_a4 277, nested_windows 277|277|válido|
|scored_0037|E1|3|265|postprocess 265, optimize_a4 265, nested_windows 265|265|válido|
|scored_0037|E1|6|262|postprocess 262, optimize_a4 262, nested_windows 262|262|válido|
|scored_0037|E2|52-1|278||259|válido|
|scored_0037|E2|52-3|265||253|válido|
|scored_0037|E2|52-6|262||251|válido|
|scored_0037|E2|64-1|278||260|válido|
|scored_0037|E2|64-3|265||255|válido|
|scored_0037|E2|64-6|262||252|válido|
|scored_0037|E3|joint|262|individual 262|260|OPTIMAL|
|scored_0037|E4|repair|262||262|completo|
|scored_0038|E1|1|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0038|E1|3|193|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0038|E1|6|193|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|scored_0038|E2|52-1|199||187|válido|
|scored_0038|E2|52-3|193||186|válido|
|scored_0038|E2|52-6|193||179|válido|
|scored_0038|E2|64-1|199||187|válido|
|scored_0038|E2|64-3|193||186|válido|
|scored_0038|E2|64-6|193||179|válido|
|scored_0038|E3|joint|193|individual 192|192|OPTIMAL|
|scored_0038|E4|repair|193||192|completo|
|scored_0039|E1|1|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0039|E1|3|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0039|E1|6|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0039|E2|52-1|143||140|válido|
|scored_0039|E2|52-3|143||140|válido|
|scored_0039|E2|52-6|143||140|válido|
|scored_0039|E2|64-1|143||138|válido|
|scored_0039|E2|64-3|143||138|válido|
|scored_0039|E2|64-6|143||138|válido|
|scored_0039|E3|joint|143|individual 143|143|OPTIMAL|
|scored_0039|E4|repair|143||142|completo|
|scored_0040|E1|1|240|postprocess 240, optimize_a4 240, nested_windows 240|240|válido|
|scored_0040|E1|3|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0040|E1|6|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0040|E2|52-1|240||223|válido|
|scored_0040|E2|52-3|239||227|válido|
|scored_0040|E2|52-6|235||217|válido|
|scored_0040|E2|64-1|240||232|válido|
|scored_0040|E2|64-3|239||226|válido|
|scored_0040|E2|64-6|235||222|válido|
|scored_0040|E3|joint|235|individual 234|234|OPTIMAL|
|scored_0040|E4|repair|235||235|completo|
|scored_0041|E1|1|200|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0041|E1|3|197|postprocess 194, optimize_a4 194, nested_windows 194|194|válido|
|scored_0041|E1|6|194|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0041|E2|52-1|200||188|válido|
|scored_0041|E2|52-3|197||182|válido|
|scored_0041|E2|52-6|194||183|válido|
|scored_0041|E2|64-1|200||187|válido|
|scored_0041|E2|64-3|197||182|válido|
|scored_0041|E2|64-6|194||183|válido|
|scored_0041|E3|joint|194|individual 194|191|OPTIMAL|
|scored_0041|E4|repair|194||194|completo|
|scored_0042|E1|1|229|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|scored_0042|E1|3|217|postprocess 217, optimize_a4 217, nested_windows 217|217|válido|
|scored_0042|E1|6|213|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0042|E2|52-1|229||205|válido|
|scored_0042|E2|52-3|217||203|válido|
|scored_0042|E2|52-6|213||202|válido|
|scored_0042|E2|64-1|229||205|válido|
|scored_0042|E2|64-3|217||203|válido|
|scored_0042|E2|64-6|213||204|válido|
|scored_0042|E3|joint|213|individual 213|213|OPTIMAL|
|scored_0042|E4|repair|213||213|completo|
|scored_0043|E1|1|297|postprocess 297, optimize_a4 297, nested_windows 297|297|válido|
|scored_0043|E1|3|288|postprocess 288, optimize_a4 288, nested_windows 288|288|válido|
|scored_0043|E1|6|285|postprocess 285, optimize_a4 285, nested_windows 285|285|válido|
|scored_0043|E2|52-1|297||280|válido|
|scored_0043|E2|52-3|288||270|válido|
|scored_0043|E2|52-6|285||268|válido|
|scored_0043|E2|64-1|297||279|válido|
|scored_0043|E2|64-3|288||270|válido|
|scored_0043|E2|64-6|285||267|válido|
|scored_0043|E3|joint|285|individual 285|284|OPTIMAL|
|scored_0043|E4|repair|285||285|completo|
|scored_0044|E1|1|108|postprocess 106, optimize_a4 106, nested_windows 106|106|válido|
|scored_0044|E1|3|108|postprocess 106, optimize_a4 106, nested_windows 106|106|válido|
|scored_0044|E1|6|108|postprocess 106, optimize_a4 106, nested_windows 106|106|válido|
|scored_0044|E2|52-1|108||105|válido|
|scored_0044|E2|52-3|108||105|válido|
|scored_0044|E2|52-6|108||105|válido|
|scored_0044|E2|64-1|108||107|válido|
|scored_0044|E2|64-3|108||107|válido|
|scored_0044|E2|64-6|108||107|válido|
|scored_0044|E3|joint|108|individual 108|108|OPTIMAL|
|scored_0044|E4|repair|108||102|completo|
|scored_0045|E1|1|222|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0045|E1|3|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0045|E1|6|214|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0045|E2|52-1|222||206|válido|
|scored_0045|E2|52-3|215||203|válido|
|scored_0045|E2|52-6|214||201|válido|
|scored_0045|E2|64-1|222||207|válido|
|scored_0045|E2|64-3|215||204|válido|
|scored_0045|E2|64-6|214||205|válido|
|scored_0045|E3|joint|214|individual 213|212|OPTIMAL|
|scored_0045|E4|repair|214||214|completo|
|scored_0046|E1|1|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0046|E1|3|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0046|E1|6|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0046|E2|52-1|41||41|válido|
|scored_0046|E2|52-3|41||41|válido|
|scored_0046|E2|52-6|41||41|válido|
|scored_0046|E2|64-1|41||41|válido|
|scored_0046|E2|64-3|41||41|válido|
|scored_0046|E2|64-6|41||41|válido|
|scored_0046|E3|joint|41|individual 41|41|OPTIMAL|
|scored_0046|E4|repair|41||41|completo|
|scored_0047|E1|1|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0047|E1|3|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0047|E1|6|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0047|E2|52-1|242||219|válido|
|scored_0047|E2|52-3|232||215|válido|
|scored_0047|E2|52-6|226||209|válido|
|scored_0047|E2|64-1|242||221|válido|
|scored_0047|E2|64-3|232||217|válido|
|scored_0047|E2|64-6|226||208|válido|
|scored_0047|E3|joint|226|individual 226|225|OPTIMAL|
|scored_0047|E4|repair|226||226|completo|
|scored_0048|E1|1|202|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0048|E1|3|200|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0048|E1|6|193|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0048|E2|52-1|202||186|válido|
|scored_0048|E2|52-3|200||188|válido|
|scored_0048|E2|52-6|193||183|válido|
|scored_0048|E2|64-1|202||189|válido|
|scored_0048|E2|64-3|200||187|válido|
|scored_0048|E2|64-6|193||183|válido|
|scored_0048|E3|joint|193|individual 192|192|OPTIMAL|
|scored_0048|E4|repair|193||193|completo|
|scored_0049|E1|1|212|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|scored_0049|E1|3|213|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0049|E1|6|205|postprocess 205, optimize_a4 205, nested_windows 205|205|válido|
|scored_0049|E2|52-1|212||198|válido|
|scored_0049|E2|52-3|213||195|válido|
|scored_0049|E2|52-6|205||194|válido|
|scored_0049|E2|64-1|212||199|válido|
|scored_0049|E2|64-3|213||196|válido|
|scored_0049|E2|64-6|205||193|válido|
|scored_0049|E3|joint|205|individual 205|205|OPTIMAL|
|scored_0049|E4|repair|205||205|completo|
|scored_0050|E1|1|44|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0050|E1|3|44|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0050|E1|6|44|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0050|E2|52-1|44||44|válido|
|scored_0050|E2|52-3|44||44|válido|
|scored_0050|E2|52-6|44||44|válido|
|scored_0050|E2|64-1|44||44|válido|
|scored_0050|E2|64-3|44||44|válido|
|scored_0050|E2|64-6|44||44|válido|
|scored_0050|E3|joint|43|individual 43|41|OPTIMAL|
|scored_0050|E4|repair|43||36|completo|
|scored_0051|E1|1|414|postprocess 413, optimize_a4 413, nested_windows 413|413|válido|
|scored_0051|E1|3|386|postprocess 386, optimize_a4 386, nested_windows 386|386|válido|
|scored_0051|E1|6|379|postprocess 379, optimize_a4 379, nested_windows 379|379|válido|
|scored_0051|E2|52-1|414||372|válido|
|scored_0051|E2|52-3|386||358|válido|
|scored_0051|E2|52-6|379||352|válido|
|scored_0051|E2|64-1|414||421|válido|
|scored_0051|E2|64-3|386||363|válido|
|scored_0051|E2|64-6|379||357|válido|
|scored_0051|E3|joint|379|individual 378|378|FEASIBLE|
|scored_0051|E4|repair|379||379|completo|
|scored_0052|E1|1|143|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0052|E1|3|144|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0052|E1|6|144|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0052|E2|52-1|143||141|válido|
|scored_0052|E2|52-3|144||139|válido|
|scored_0052|E2|52-6|144||139|válido|
|scored_0052|E2|64-1|143||139|válido|
|scored_0052|E2|64-3|144||139|válido|
|scored_0052|E2|64-6|144||139|válido|
|scored_0052|E3|joint|143|individual 143|143|OPTIMAL|
|scored_0052|E4|repair|143||142|completo|
|scored_0053|E1|1|225|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0053|E1|3|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0053|E1|6|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0053|E2|52-1|225||199|válido|
|scored_0053|E2|52-3|220||197|válido|
|scored_0053|E2|52-6|215||197|válido|
|scored_0053|E2|64-1|225||199|válido|
|scored_0053|E2|64-3|220||195|válido|
|scored_0053|E2|64-6|215||193|válido|
|scored_0053|E3|joint|215|individual 213|211|OPTIMAL|
|scored_0053|E4|repair|215||213|completo|
|scored_0054|E1|1|269|postprocess 269, optimize_a4 269, nested_windows 269|269|válido|
|scored_0054|E1|3|265|postprocess 264, optimize_a4 264, nested_windows 264|264|válido|
|scored_0054|E1|6|256|postprocess 253, optimize_a4 253, nested_windows 253|253|válido|
|scored_0054|E2|52-1|269||253|válido|
|scored_0054|E2|52-3|265||244|válido|
|scored_0054|E2|52-6|256||242|válido|
|scored_0054|E2|64-1|269||253|válido|
|scored_0054|E2|64-3|265||245|válido|
|scored_0054|E2|64-6|256||242|válido|
|scored_0054|E3|joint|256|individual 255|255|FEASIBLE|
|scored_0054|E4|repair|256||256|completo|
|scored_0055|E1|1|104|postprocess 104, optimize_a4 104, nested_windows 104|104|válido|
|scored_0055|E1|3|103|postprocess 103, optimize_a4 103, nested_windows 103|103|válido|
|scored_0055|E1|6|103|postprocess 103, optimize_a4 103, nested_windows 103|103|válido|
|scored_0055|E2|52-1|104||100|válido|
|scored_0055|E2|52-3|103||100|válido|
|scored_0055|E2|52-6|103||100|válido|
|scored_0055|E2|64-1|104||100|válido|
|scored_0055|E2|64-3|103||100|válido|
|scored_0055|E2|64-6|103||100|válido|
|scored_0055|E3|joint|103|individual 103|103|OPTIMAL|
|scored_0055|E4|repair|103||103|completo|
|scored_0056|E1|1|37|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0056|E1|3|37|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0056|E1|6|37|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0056|E2|52-1|37||37|válido|
|scored_0056|E2|52-3|37||37|válido|
|scored_0056|E2|52-6|37||37|válido|
|scored_0056|E2|64-1|37||37|válido|
|scored_0056|E2|64-3|37||37|válido|
|scored_0056|E2|64-6|37||37|válido|
|scored_0056|E3|joint|37|individual 37|37|OPTIMAL|
|scored_0056|E4|repair|37||37|completo|
|scored_0057|E1|1|305|postprocess 304, optimize_a4 304, nested_windows 304|304|válido|
|scored_0057|E1|3|288|postprocess 288, optimize_a4 288, nested_windows 288|288|válido|
|scored_0057|E1|6|288|postprocess 288, optimize_a4 288, nested_windows 288|288|válido|
|scored_0057|E2|52-1|305||281|válido|
|scored_0057|E2|52-3|288||276|válido|
|scored_0057|E2|52-6|288||270|válido|
|scored_0057|E2|64-1|305||288|válido|
|scored_0057|E2|64-3|288||279|válido|
|scored_0057|E2|64-6|288||270|válido|
|scored_0057|E3|joint|288|individual 287|285|OPTIMAL|
|scored_0057|E4|repair|288||288|completo|
|scored_0058|E1|1|375|postprocess 375, optimize_a4 375, nested_windows 375|375|válido|
|scored_0058|E1|3|355|postprocess 355, optimize_a4 355, nested_windows 355|355|válido|
|scored_0058|E1|6|349|postprocess 349, optimize_a4 349, nested_windows 349|349|válido|
|scored_0058|E2|52-1|375||351|válido|
|scored_0058|E2|52-3|355||339|válido|
|scored_0058|E2|52-6|349||333|válido|
|scored_0058|E2|64-1|375||349|válido|
|scored_0058|E2|64-3|355||344|válido|
|scored_0058|E2|64-6|349||336|válido|
|scored_0058|E3|joint|349|individual 349|349|FEASIBLE|
|scored_0058|E4|repair|349||349|completo|
|scored_0059|E1|1|198|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|scored_0059|E1|3|195|postprocess 195, optimize_a4 195, nested_windows 195|195|válido|
|scored_0059|E1|6|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0059|E2|52-1|198||185|válido|
|scored_0059|E2|52-3|195||187|válido|
|scored_0059|E2|52-6|192||186|válido|
|scored_0059|E2|64-1|198||183|válido|
|scored_0059|E2|64-3|195||189|válido|
|scored_0059|E2|64-6|192||187|válido|
|scored_0059|E3|joint|192|individual 191|191|OPTIMAL|
|scored_0059|E4|repair|192||192|completo|
|scored_0060|E1|1|370|postprocess 370, optimize_a4 370, nested_windows 370|370|válido|
|scored_0060|E1|3|355|postprocess 354, optimize_a4 354, nested_windows 354|354|válido|
|scored_0060|E1|6|342|postprocess 342, optimize_a4 342, nested_windows 342|342|válido|
|scored_0060|E2|52-1|370||330|válido|
|scored_0060|E2|52-3|355||326|válido|
|scored_0060|E2|52-6|342||317|válido|
|scored_0060|E2|64-1|370||330|válido|
|scored_0060|E2|64-3|355||326|válido|
|scored_0060|E2|64-6|342||320|válido|
|scored_0060|E3|joint|342|individual 340|339|OPTIMAL|
|scored_0060|E4|repair|342||342|completo|
|scored_0061|E1|1|4|postprocess 4, optimize_a4 4, nested_windows 4|4|válido|
|scored_0061|E1|3|4|postprocess 4, optimize_a4 4, nested_windows 4|4|válido|
|scored_0061|E1|6|4|postprocess 4, optimize_a4 4, nested_windows 4|4|válido|
|scored_0061|E2|52-1|4||4|válido|
|scored_0061|E2|52-3|4||4|válido|
|scored_0061|E2|52-6|4||4|válido|
|scored_0061|E2|64-1|4||4|válido|
|scored_0061|E2|64-3|4||4|válido|
|scored_0061|E2|64-6|4||4|válido|
|scored_0061|E3|joint|4|individual 4|4|OPTIMAL|
|scored_0061|E4|repair|4||4|completo|
|scored_0062|E1|1|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0062|E1|3|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0062|E1|6|190|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|scored_0062|E2|52-1|192||185|válido|
|scored_0062|E2|52-3|188||184|válido|
|scored_0062|E2|52-6|190||182|válido|
|scored_0062|E2|64-1|192||185|válido|
|scored_0062|E2|64-3|188||184|válido|
|scored_0062|E2|64-6|190||182|válido|
|scored_0062|E3|joint|188|individual 188|188|OPTIMAL|
|scored_0062|E4|repair|188||188|completo|
|scored_0063|E1|1|167|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0063|E1|3|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|scored_0063|E1|6|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0063|E2|52-1|167||160|válido|
|scored_0063|E2|52-3|166||160|válido|
|scored_0063|E2|52-6|165||154|válido|
|scored_0063|E2|64-1|167||160|válido|
|scored_0063|E2|64-3|166||160|válido|
|scored_0063|E2|64-6|165||157|válido|
|scored_0063|E3|joint|165|individual 164|164|OPTIMAL|
|scored_0063|E4|repair|165||165|completo|
|scored_0064|E1|1|185|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|scored_0064|E1|3|182|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0064|E1|6|183|postprocess 183, optimize_a4 183, nested_windows 183|183|válido|
|scored_0064|E2|52-1|185||170|válido|
|scored_0064|E2|52-3|182||170|válido|
|scored_0064|E2|52-6|183||168|válido|
|scored_0064|E2|64-1|185||172|válido|
|scored_0064|E2|64-3|182||172|válido|
|scored_0064|E2|64-6|183||172|válido|
|scored_0064|E3|joint|182|individual 182|182|OPTIMAL|
|scored_0064|E4|repair|182||179|completo|
|scored_0065|E1|1|50|postprocess 49, optimize_a4 49, nested_windows 49|49|válido|
|scored_0065|E1|3|50|postprocess 49, optimize_a4 49, nested_windows 49|49|válido|
|scored_0065|E1|6|50|postprocess 49, optimize_a4 49, nested_windows 49|49|válido|
|scored_0065|E2|52-1|50||50|válido|
|scored_0065|E2|52-3|50||50|válido|
|scored_0065|E2|52-6|50||50|válido|
|scored_0065|E2|64-1|50||50|válido|
|scored_0065|E2|64-3|50||50|válido|
|scored_0065|E2|64-6|50||50|válido|
|scored_0065|E3|joint|50|individual 50|50|OPTIMAL|
|scored_0065|E4|repair|50||48|completo|
|scored_0066|E1|1|55|postprocess 52, optimize_a4 52, nested_windows 52|52|válido|
|scored_0066|E1|3|55|postprocess 52, optimize_a4 52, nested_windows 52|52|válido|
|scored_0066|E1|6|55|postprocess 52, optimize_a4 52, nested_windows 52|52|válido|
|scored_0066|E2|52-1|55||55|válido|
|scored_0066|E2|52-3|55||55|válido|
|scored_0066|E2|52-6|55||55|válido|
|scored_0066|E2|64-1|55||55|válido|
|scored_0066|E2|64-3|55||55|válido|
|scored_0066|E2|64-6|55||55|válido|
|scored_0066|E3|joint|55|individual 55|55|OPTIMAL|
|scored_0066|E4|repair|55||55|completo|
|scored_0067|E1|1|377|postprocess 377, optimize_a4 377, nested_windows 377|377|válido|
|scored_0067|E1|3|346|postprocess 345, optimize_a4 345, nested_windows 345|345|válido|
|scored_0067|E1|6|343|postprocess 343, optimize_a4 343, nested_windows 343|343|válido|
|scored_0067|E2|52-1|377||346|válido|
|scored_0067|E2|52-3|346||324|válido|
|scored_0067|E2|52-6|343||321|válido|
|scored_0067|E2|64-1|377||344|válido|
|scored_0067|E2|64-3|346||326|válido|
|scored_0067|E2|64-6|343||322|válido|
|scored_0067|E3|joint|343|individual 343|341|OPTIMAL|
|scored_0067|E4|repair|343||343|completo|
|scored_0068|E1|1|225|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0068|E1|3|208|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0068|E1|6|203|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0068|E2|52-1|225||204|válido|
|scored_0068|E2|52-3|208||201|válido|
|scored_0068|E2|52-6|203||195|válido|
|scored_0068|E2|64-1|225||208|válido|
|scored_0068|E2|64-3|208||200|válido|
|scored_0068|E2|64-6|203||194|válido|
|scored_0068|E3|joint|203|individual 203|203|OPTIMAL|
|scored_0068|E4|repair|203||203|completo|
|scored_0069|E1|1|405|postprocess 405, optimize_a4 405, nested_windows 405|405|válido|
|scored_0069|E1|3|375|postprocess 375, optimize_a4 375, nested_windows 375|375|válido|
|scored_0069|E1|6|370|postprocess 370, optimize_a4 370, nested_windows 370|370|válido|
|scored_0069|E2|52-1|405||366|válido|
|scored_0069|E2|52-3|375||355|válido|
|scored_0069|E2|52-6|370||353|válido|
|scored_0069|E2|64-1|405||374|válido|
|scored_0069|E2|64-3|375||361|válido|
|scored_0069|E2|64-6|370||356|válido|
|scored_0069|E3|joint|370|individual 370|369|OPTIMAL|
|scored_0069|E4|repair|370||370|completo|
|scored_0070|E1|1|212|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0070|E1|3|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0070|E1|6|201|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0070|E2|52-1|212||191|válido|
|scored_0070|E2|52-3|207||186|válido|
|scored_0070|E2|52-6|201||184|válido|
|scored_0070|E2|64-1|212||194|válido|
|scored_0070|E2|64-3|207||185|válido|
|scored_0070|E2|64-6|201||185|válido|
|scored_0070|E3|joint|201|individual 201|201|OPTIMAL|
|scored_0070|E4|repair|201||201|completo|
|scored_0071|E1|1|148|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0071|E1|3|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0071|E1|6|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0071|E2|52-1|148||146|válido|
|scored_0071|E2|52-3|148||146|válido|
|scored_0071|E2|52-6|148||146|válido|
|scored_0071|E2|64-1|148||147|válido|
|scored_0071|E2|64-3|148||147|válido|
|scored_0071|E2|64-6|148||147|válido|
|scored_0071|E3|joint|148|individual 148|148|OPTIMAL|
|scored_0071|E4|repair|148||146|completo|
|scored_0072|E1|1|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0072|E1|3|181|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0072|E1|6|180|postprocess 180, optimize_a4 180, nested_windows 180|180|válido|
|scored_0072|E2|52-1|188||173|válido|
|scored_0072|E2|52-3|181||162|válido|
|scored_0072|E2|52-6|180||161|válido|
|scored_0072|E2|64-1|188||174|válido|
|scored_0072|E2|64-3|181||162|válido|
|scored_0072|E2|64-6|180||163|válido|
|scored_0072|E3|joint|180|individual 180|180|OPTIMAL|
|scored_0072|E4|repair|180||179|completo|
|scored_0073|E1|1|168|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0073|E1|3|163|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0073|E1|6|163|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0073|E2|52-1|168||158|válido|
|scored_0073|E2|52-3|163||157|válido|
|scored_0073|E2|52-6|163||160|válido|
|scored_0073|E2|64-1|168||159|válido|
|scored_0073|E2|64-3|163||158|válido|
|scored_0073|E2|64-6|163||156|válido|
|scored_0073|E3|joint|163|individual 162|162|OPTIMAL|
|scored_0073|E4|repair|163||161|completo|
|scored_0074|E1|1|296|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0074|E1|3|284|postprocess 284, optimize_a4 284, nested_windows 284|284|válido|
|scored_0074|E1|6|277|postprocess 277, optimize_a4 277, nested_windows 277|277|válido|
|scored_0074|E2|52-1|296||270|válido|
|scored_0074|E2|52-3|284||266|válido|
|scored_0074|E2|52-6|277||260|válido|
|scored_0074|E2|64-1|296||273|válido|
|scored_0074|E2|64-3|284||268|válido|
|scored_0074|E2|64-6|277||259|válido|
|scored_0074|E3|joint|277|individual 277|276|OPTIMAL|
|scored_0074|E4|repair|277||277|completo|
|scored_0075|E1|1|370|postprocess 370, optimize_a4 370, nested_windows 370|370|válido|
|scored_0075|E1|3|359|postprocess 359, optimize_a4 359, nested_windows 359|359|válido|
|scored_0075|E1|6|341|postprocess 340, optimize_a4 340, nested_windows 340|340|válido|
|scored_0075|E2|52-1|370||337|válido|
|scored_0075|E2|52-3|359||323|válido|
|scored_0075|E2|52-6|341||318|válido|
|scored_0075|E2|64-1|370||336|válido|
|scored_0075|E2|64-3|359||325|válido|
|scored_0075|E2|64-6|341||318|válido|
|scored_0075|E3|joint|341|individual 341|338|FEASIBLE|
|scored_0075|E4|repair|341||341|completo|
|scored_0076|E1|1|190|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|scored_0076|E1|3|187|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0076|E1|6|185|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|scored_0076|E2|52-1|190||182|válido|
|scored_0076|E2|52-3|187||180|válido|
|scored_0076|E2|52-6|185||178|válido|
|scored_0076|E2|64-1|190||181|válido|
|scored_0076|E2|64-3|187||176|válido|
|scored_0076|E2|64-6|185||176|válido|
|scored_0076|E3|joint|185|individual 185|185|OPTIMAL|
|scored_0076|E4|repair|185||185|completo|
|scored_0077|E1|1|144|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0077|E1|3|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0077|E1|6|142|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0077|E2|52-1|144||135|válido|
|scored_0077|E2|52-3|143||135|válido|
|scored_0077|E2|52-6|142||135|válido|
|scored_0077|E2|64-1|144||137|válido|
|scored_0077|E2|64-3|143||137|válido|
|scored_0077|E2|64-6|142||136|válido|
|scored_0077|E3|joint|142|individual 142|142|OPTIMAL|
|scored_0077|E4|repair|142||139|completo|
|scored_0078|E1|1|139|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|scored_0078|E1|3|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0078|E1|6|136|postprocess 136, optimize_a4 136, nested_windows 136|136|válido|
|scored_0078|E2|52-1|139||130|válido|
|scored_0078|E2|52-3|137||130|válido|
|scored_0078|E2|52-6|136||130|válido|
|scored_0078|E2|64-1|139||130|válido|
|scored_0078|E2|64-3|137||130|válido|
|scored_0078|E2|64-6|136||130|válido|
|scored_0078|E3|joint|136|individual 136|136|OPTIMAL|
|scored_0078|E4|repair|136||133|completo|
|scored_0079|E1|1|209|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0079|E1|3|209|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0079|E1|6|207|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0079|E2|52-1|209||201|válido|
|scored_0079|E2|52-3|209||199|válido|
|scored_0079|E2|52-6|207||191|válido|
|scored_0079|E2|64-1|209||202|válido|
|scored_0079|E2|64-3|209||197|válido|
|scored_0079|E2|64-6|207||193|válido|
|scored_0079|E3|joint|207|individual 207|205|OPTIMAL|
|scored_0079|E4|repair|207||204|completo|
|scored_0080|E1|1|203|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0080|E1|3|196|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|scored_0080|E1|6|193|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0080|E2|52-1|203||190|válido|
|scored_0080|E2|52-3|196||186|válido|
|scored_0080|E2|52-6|193||186|válido|
|scored_0080|E2|64-1|203||190|válido|
|scored_0080|E2|64-3|196||186|válido|
|scored_0080|E2|64-6|193||184|válido|
|scored_0080|E3|joint|193|individual 193|193|OPTIMAL|
|scored_0080|E4|repair|193||193|completo|
|scored_0081|E1|1|161|postprocess 161, optimize_a4 161, nested_windows 161|161|válido|
|scored_0081|E1|3|160|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0081|E1|6|157|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0081|E2|52-1|161||154|válido|
|scored_0081|E2|52-3|160||154|válido|
|scored_0081|E2|52-6|157||149|válido|
|scored_0081|E2|64-1|161||154|válido|
|scored_0081|E2|64-3|160||154|válido|
|scored_0081|E2|64-6|157||148|válido|
|scored_0081|E3|joint|157|individual 157|157|OPTIMAL|
|scored_0081|E4|repair|157||156|completo|
|scored_0082|E1|1|168|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0082|E1|3|164|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0082|E1|6|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|scored_0082|E2|52-1|168||164|válido|
|scored_0082|E2|52-3|164||156|válido|
|scored_0082|E2|52-6|166||158|válido|
|scored_0082|E2|64-1|168||164|válido|
|scored_0082|E2|64-3|164||156|válido|
|scored_0082|E2|64-6|166||158|válido|
|scored_0082|E3|joint|164|individual 164|156|OPTIMAL|
|scored_0082|E4|repair|164||161|completo|
|scored_0083|E1|1|242|postprocess 240, optimize_a4 240, nested_windows 240|240|válido|
|scored_0083|E1|3|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0083|E1|6|236|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0083|E2|52-1|242||228|válido|
|scored_0083|E2|52-3|235||225|válido|
|scored_0083|E2|52-6|236||219|válido|
|scored_0083|E2|64-1|242||227|válido|
|scored_0083|E2|64-3|235||225|válido|
|scored_0083|E2|64-6|236||221|válido|
|scored_0083|E3|joint|235|individual 235|234|OPTIMAL|
|scored_0083|E4|repair|235||235|completo|
|scored_0084|E1|1|52|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0084|E1|3|52|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0084|E1|6|52|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0084|E2|52-1|52||52|válido|
|scored_0084|E2|52-3|52||52|válido|
|scored_0084|E2|52-6|52||52|válido|
|scored_0084|E2|64-1|52||52|válido|
|scored_0084|E2|64-3|52||52|válido|
|scored_0084|E2|64-6|52||52|válido|
|scored_0084|E3|joint|52|individual 52|52|OPTIMAL|
|scored_0084|E4|repair|52||52|completo|
|scored_0085|E1|1|122|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0085|E1|3|122|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0085|E1|6|122|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0085|E2|52-1|122||116|válido|
|scored_0085|E2|52-3|122||116|válido|
|scored_0085|E2|52-6|122||116|válido|
|scored_0085|E2|64-1|122||116|válido|
|scored_0085|E2|64-3|122||116|válido|
|scored_0085|E2|64-6|122||116|válido|
|scored_0085|E3|joint|122|individual 122|122|OPTIMAL|
|scored_0085|E4|repair|122||119|completo|
|scored_0086|E1|1|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0086|E1|3|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0086|E1|6|136|postprocess 136, optimize_a4 136, nested_windows 136|136|válido|
|scored_0086|E2|52-1|137||132|válido|
|scored_0086|E2|52-3|137||132|válido|
|scored_0086|E2|52-6|136||132|válido|
|scored_0086|E2|64-1|137||132|válido|
|scored_0086|E2|64-3|137||132|válido|
|scored_0086|E2|64-6|136||132|válido|
|scored_0086|E3|joint|136|individual 136|136|OPTIMAL|
|scored_0086|E4|repair|136||134|completo|
|scored_0087|E1|1|356|postprocess 355, optimize_a4 355, nested_windows 355|355|válido|
|scored_0087|E1|3|329|postprocess 329, optimize_a4 329, nested_windows 329|329|válido|
|scored_0087|E1|6|325|postprocess 325, optimize_a4 325, nested_windows 325|325|válido|
|scored_0087|E2|52-1|356||319|válido|
|scored_0087|E2|52-3|329||311|válido|
|scored_0087|E2|52-6|325||306|válido|
|scored_0087|E2|64-1|356||319|válido|
|scored_0087|E2|64-3|329||317|válido|
|scored_0087|E2|64-6|325||304|válido|
|scored_0087|E3|joint|325|individual 323|323|FEASIBLE|
|scored_0087|E4|repair|325||325|completo|
|scored_0088|E1|1|302|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|scored_0088|E1|3|290|postprocess 288, optimize_a4 288, nested_windows 288|288|válido|
|scored_0088|E1|6|285|postprocess 285, optimize_a4 285, nested_windows 285|285|válido|
|scored_0088|E2|52-1|302||271|válido|
|scored_0088|E2|52-3|290||265|válido|
|scored_0088|E2|52-6|285||264|válido|
|scored_0088|E2|64-1|302||274|válido|
|scored_0088|E2|64-3|290||264|válido|
|scored_0088|E2|64-6|285||266|válido|
|scored_0088|E3|joint|285|individual 285|285|OPTIMAL|
|scored_0088|E4|repair|285||285|completo|
|scored_0089|E1|1|163|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0089|E1|3|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0089|E1|6|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0089|E2|52-1|163||156|válido|
|scored_0089|E2|52-3|159||155|válido|
|scored_0089|E2|52-6|159||154|válido|
|scored_0089|E2|64-1|163||154|válido|
|scored_0089|E2|64-3|159||154|válido|
|scored_0089|E2|64-6|159||153|válido|
|scored_0089|E3|joint|159|individual 159|159|OPTIMAL|
|scored_0089|E4|repair|159||159|completo|
|scored_0090|E1|1|271|postprocess 270, optimize_a4 270, nested_windows 270|270|válido|
|scored_0090|E1|3|259|postprocess 258, optimize_a4 258, nested_windows 258|258|válido|
|scored_0090|E1|6|254|postprocess 254, optimize_a4 254, nested_windows 254|254|válido|
|scored_0090|E2|52-1|271||248|válido|
|scored_0090|E2|52-3|259||238|válido|
|scored_0090|E2|52-6|254||238|válido|
|scored_0090|E2|64-1|271||247|válido|
|scored_0090|E2|64-3|259||242|válido|
|scored_0090|E2|64-6|254||239|válido|
|scored_0090|E3|joint|254|individual 253|253|OPTIMAL|
|scored_0090|E4|repair|254||254|completo|
|scored_0091|E1|1|42|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0091|E1|3|42|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0091|E1|6|42|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0091|E2|52-1|42||42|válido|
|scored_0091|E2|52-3|42||42|válido|
|scored_0091|E2|52-6|42||42|válido|
|scored_0091|E2|64-1|42||42|válido|
|scored_0091|E2|64-3|42||42|válido|
|scored_0091|E2|64-6|42||42|válido|
|scored_0091|E3|joint|42|individual 42|42|OPTIMAL|
|scored_0091|E4|repair|42||42|completo|
|scored_0092|E1|1|56|postprocess 55, optimize_a4 55, nested_windows 55|55|válido|
|scored_0092|E1|3|56|postprocess 55, optimize_a4 55, nested_windows 55|55|válido|
|scored_0092|E1|6|56|postprocess 55, optimize_a4 55, nested_windows 55|55|válido|
|scored_0092|E2|52-1|56||56|válido|
|scored_0092|E2|52-3|56||56|válido|
|scored_0092|E2|52-6|56||56|válido|
|scored_0092|E2|64-1|56||56|válido|
|scored_0092|E2|64-3|56||56|válido|
|scored_0092|E2|64-6|56||56|válido|
|scored_0092|E3|joint|56|individual 56|56|OPTIMAL|
|scored_0092|E4|repair|56||54|completo|
|scored_0093|E1|1|147|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0093|E1|3|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0093|E1|6|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0093|E2|52-1|147||138|válido|
|scored_0093|E2|52-3|146||138|válido|
|scored_0093|E2|52-6|145||138|válido|
|scored_0093|E2|64-1|147||139|válido|
|scored_0093|E2|64-3|146||139|válido|
|scored_0093|E2|64-6|145||139|válido|
|scored_0093|E3|joint|145|individual 145|145|OPTIMAL|
|scored_0093|E4|repair|145||142|completo|
|scored_0094|E1|1|356|postprocess 356, optimize_a4 356, nested_windows 356|356|válido|
|scored_0094|E1|3|337|postprocess 337, optimize_a4 337, nested_windows 337|337|válido|
|scored_0094|E1|6|331|postprocess 331, optimize_a4 331, nested_windows 331|331|válido|
|scored_0094|E2|52-1|356||327|válido|
|scored_0094|E2|52-3|337||315|válido|
|scored_0094|E2|52-6|331||313|válido|
|scored_0094|E2|64-1|356||331|válido|
|scored_0094|E2|64-3|337||316|válido|
|scored_0094|E2|64-6|331||316|válido|
|scored_0094|E3|joint|331|individual 331|331|OPTIMAL|
|scored_0094|E4|repair|331||331|completo|
|scored_0095|E1|1|47|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0095|E1|3|47|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0095|E1|6|47|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0095|E2|52-1|47||47|válido|
|scored_0095|E2|52-3|47||47|válido|
|scored_0095|E2|52-6|47||47|válido|
|scored_0095|E2|64-1|47||47|válido|
|scored_0095|E2|64-3|47||47|válido|
|scored_0095|E2|64-6|47||47|válido|
|scored_0095|E3|joint|47|individual 47|47|OPTIMAL|
|scored_0095|E4|repair|47||47|completo|
|scored_0096|E1|1|240|postprocess 240, optimize_a4 240, nested_windows 240|240|válido|
|scored_0096|E1|3|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0096|E1|6|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0096|E2|52-1|240||223|válido|
|scored_0096|E2|52-3|235||222|válido|
|scored_0096|E2|52-6|233||221|válido|
|scored_0096|E2|64-1|240||223|válido|
|scored_0096|E2|64-3|235||223|válido|
|scored_0096|E2|64-6|233||223|válido|
|scored_0096|E3|joint|233|individual 233|232|OPTIMAL|
|scored_0096|E4|repair|233||231|completo|
|scored_0097|E1|1|131|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|scored_0097|E1|3|131|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|scored_0097|E1|6|131|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|scored_0097|E2|52-1|131||127|válido|
|scored_0097|E2|52-3|131||127|válido|
|scored_0097|E2|52-6|131||127|válido|
|scored_0097|E2|64-1|131||125|válido|
|scored_0097|E2|64-3|131||125|válido|
|scored_0097|E2|64-6|131||125|válido|
|scored_0097|E3|joint|131|individual 131|131|OPTIMAL|
|scored_0097|E4|repair|131||129|completo|
|scored_0098|E1|1|258|postprocess 258, optimize_a4 258, nested_windows 258|258|válido|
|scored_0098|E1|3|249|postprocess 249, optimize_a4 249, nested_windows 249|249|válido|
|scored_0098|E1|6|246|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|scored_0098|E2|52-1|258||235|válido|
|scored_0098|E2|52-3|249||235|válido|
|scored_0098|E2|52-6|246||235|válido|
|scored_0098|E2|64-1|258||237|válido|
|scored_0098|E2|64-3|249||233|válido|
|scored_0098|E2|64-6|246||236|válido|
|scored_0098|E3|joint|246|individual 246|244|OPTIMAL|
|scored_0098|E4|repair|246||246|completo|
|scored_0099|E1|1|224|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0099|E1|3|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0099|E1|6|217|postprocess 216, optimize_a4 216, nested_windows 216|216|válido|
|scored_0099|E2|52-1|224||193|válido|
|scored_0099|E2|52-3|220||192|válido|
|scored_0099|E2|52-6|217||197|válido|
|scored_0099|E2|64-1|224||197|válido|
|scored_0099|E2|64-3|220||191|válido|
|scored_0099|E2|64-6|217||199|válido|
|scored_0099|E3|joint|217|individual 217|217|OPTIMAL|
|scored_0099|E4|repair|217||214|completo|
|scored_0100|E1|1|204|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0100|E1|3|199|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|scored_0100|E1|6|196|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|scored_0100|E2|52-1|204||187|válido|
|scored_0100|E2|52-3|199||189|válido|
|scored_0100|E2|52-6|196||188|válido|
|scored_0100|E2|64-1|204||186|válido|
|scored_0100|E2|64-3|199||186|válido|
|scored_0100|E2|64-6|196||185|válido|
|scored_0100|E3|joint|196|individual 194|193|OPTIMAL|
|scored_0100|E4|repair|196||196|completo|
|scored_0101|E1|1|164|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0101|E1|3|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0101|E1|6|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0101|E2|52-1|164||156|válido|
|scored_0101|E2|52-3|162||154|válido|
|scored_0101|E2|52-6|162||154|válido|
|scored_0101|E2|64-1|164||153|válido|
|scored_0101|E2|64-3|162||153|válido|
|scored_0101|E2|64-6|162||153|válido|
|scored_0101|E3|joint|162|individual 162|162|OPTIMAL|
|scored_0101|E4|repair|162||160|completo|
|scored_0102|E1|1|213|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|scored_0102|E1|3|208|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0102|E1|6|203|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0102|E2|52-1|213||200|válido|
|scored_0102|E2|52-3|208||193|válido|
|scored_0102|E2|52-6|203||190|válido|
|scored_0102|E2|64-1|213||200|válido|
|scored_0102|E2|64-3|208||194|válido|
|scored_0102|E2|64-6|203||192|válido|
|scored_0102|E3|joint|203|individual 203|202|OPTIMAL|
|scored_0102|E4|repair|203||203|completo|
|scored_0103|E1|1|61|postprocess 59, optimize_a4 59, nested_windows 59|59|válido|
|scored_0103|E1|3|61|postprocess 59, optimize_a4 59, nested_windows 59|59|válido|
|scored_0103|E1|6|61|postprocess 59, optimize_a4 59, nested_windows 59|59|válido|
|scored_0103|E2|52-1|61||61|válido|
|scored_0103|E2|52-3|61||61|válido|
|scored_0103|E2|52-6|61||61|válido|
|scored_0103|E2|64-1|61||61|válido|
|scored_0103|E2|64-3|61||61|válido|
|scored_0103|E2|64-6|61||61|válido|
|scored_0103|E3|joint|61|individual 61|61|OPTIMAL|
|scored_0103|E4|repair|61||61|completo|
|scored_0104|E1|1|329|postprocess 328, optimize_a4 328, nested_windows 328|328|válido|
|scored_0104|E1|3|312|postprocess 310, optimize_a4 310, nested_windows 310|310|válido|
|scored_0104|E1|6|306|postprocess 306, optimize_a4 306, nested_windows 306|306|válido|
|scored_0104|E2|52-1|329||301|válido|
|scored_0104|E2|52-3|312||289|válido|
|scored_0104|E2|52-6|306||286|válido|
|scored_0104|E2|64-1|329||373|válido|
|scored_0104|E2|64-3|312||295|válido|
|scored_0104|E2|64-6|306||286|válido|
|scored_0104|E3|joint|306|individual 306|306|FEASIBLE|
|scored_0104|E4|repair|306||306|completo|
|scored_0105|E1|1|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0105|E1|3|201|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0105|E1|6|201|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0105|E2|52-1|207||188|válido|
|scored_0105|E2|52-3|201||190|válido|
|scored_0105|E2|52-6|201||189|válido|
|scored_0105|E2|64-1|207||190|válido|
|scored_0105|E2|64-3|201||192|válido|
|scored_0105|E2|64-6|201||189|válido|
|scored_0105|E3|joint|201|individual 200|200|OPTIMAL|
|scored_0105|E4|repair|201||201|completo|
|scored_0106|E1|1|341|postprocess 340, optimize_a4 340, nested_windows 340|340|válido|
|scored_0106|E1|3|323|postprocess 323, optimize_a4 323, nested_windows 323|323|válido|
|scored_0106|E1|6|309|postprocess 309, optimize_a4 309, nested_windows 309|309|válido|
|scored_0106|E2|52-1|341||302|válido|
|scored_0106|E2|52-3|323||301|válido|
|scored_0106|E2|52-6|309||289|válido|
|scored_0106|E2|64-1|341||327|válido|
|scored_0106|E2|64-3|323||303|válido|
|scored_0106|E2|64-6|309||291|válido|
|scored_0106|E3|joint|309|individual 309|309|OPTIMAL|
|scored_0106|E4|repair|309||308|completo|
|scored_0107|E1|1|351|postprocess 351, optimize_a4 351, nested_windows 351|351|válido|
|scored_0107|E1|3|328|postprocess 328, optimize_a4 328, nested_windows 328|328|válido|
|scored_0107|E1|6|319|postprocess 319, optimize_a4 319, nested_windows 319|319|válido|
|scored_0107|E2|52-1|351||327|válido|
|scored_0107|E2|52-3|328||311|válido|
|scored_0107|E2|52-6|319||300|válido|
|scored_0107|E2|64-1|351||327|válido|
|scored_0107|E2|64-3|328||310|válido|
|scored_0107|E2|64-6|319||302|válido|
|scored_0107|E3|joint|319|individual 319|318|FEASIBLE|
|scored_0107|E4|repair|319||319|completo|
|scored_0108|E1|1|249|postprocess 249, optimize_a4 249, nested_windows 249|249|válido|
|scored_0108|E1|3|248|postprocess 248, optimize_a4 248, nested_windows 248|248|válido|
|scored_0108|E1|6|245|postprocess 245, optimize_a4 245, nested_windows 245|245|válido|
|scored_0108|E2|52-1|249||232|válido|
|scored_0108|E2|52-3|248||233|válido|
|scored_0108|E2|52-6|245||228|válido|
|scored_0108|E2|64-1|249||233|válido|
|scored_0108|E2|64-3|248||236|válido|
|scored_0108|E2|64-6|245||229|válido|
|scored_0108|E3|joint|245|individual 245|245|OPTIMAL|
|scored_0108|E4|repair|245||245|completo|
|scored_0109|E1|1|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0109|E1|3|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|scored_0109|E1|6|194|postprocess 194, optimize_a4 194, nested_windows 194|194|válido|
|scored_0109|E2|52-1|199||186|válido|
|scored_0109|E2|52-3|197||183|válido|
|scored_0109|E2|52-6|194||177|válido|
|scored_0109|E2|64-1|199||188|válido|
|scored_0109|E2|64-3|197||181|válido|
|scored_0109|E2|64-6|194||178|válido|
|scored_0109|E3|joint|194|individual 194|194|OPTIMAL|
|scored_0109|E4|repair|194||194|completo|
|scored_0110|E1|1|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0110|E1|3|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|scored_0110|E1|6|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0110|E2|52-1|242||226|válido|
|scored_0110|E2|52-3|238||215|válido|
|scored_0110|E2|52-6|232||213|válido|
|scored_0110|E2|64-1|242||226|válido|
|scored_0110|E2|64-3|238||215|válido|
|scored_0110|E2|64-6|232||213|válido|
|scored_0110|E3|joint|232|individual 231|231|OPTIMAL|
|scored_0110|E4|repair|232||232|completo|
|scored_0111|E1|1|310|postprocess 309, optimize_a4 309, nested_windows 309|309|válido|
|scored_0111|E1|3|298|postprocess 298, optimize_a4 298, nested_windows 298|298|válido|
|scored_0111|E1|6|287|postprocess 287, optimize_a4 287, nested_windows 287|287|válido|
|scored_0111|E2|52-1|310||279|válido|
|scored_0111|E2|52-3|298||268|válido|
|scored_0111|E2|52-6|287||266|válido|
|scored_0111|E2|64-1|310||281|válido|
|scored_0111|E2|64-3|298||269|válido|
|scored_0111|E2|64-6|287||266|válido|
|scored_0111|E3|joint|287|individual 287|287|OPTIMAL|
|scored_0111|E4|repair|287||287|completo|
|scored_0112|E1|1|298|postprocess 298, optimize_a4 298, nested_windows 298|298|válido|
|scored_0112|E1|3|275|postprocess 274, optimize_a4 274, nested_windows 274|274|válido|
|scored_0112|E1|6|272|postprocess 272, optimize_a4 272, nested_windows 272|272|válido|
|scored_0112|E2|52-1|298||280|válido|
|scored_0112|E2|52-3|275||262|válido|
|scored_0112|E2|52-6|272||264|válido|
|scored_0112|E2|64-1|298||280|válido|
|scored_0112|E2|64-3|275||265|válido|
|scored_0112|E2|64-6|272||265|válido|
|scored_0112|E3|joint|272|individual 272|272|OPTIMAL|
|scored_0112|E4|repair|272||272|completo|
|scored_0113|E1|1|40|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|scored_0113|E1|3|40|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|scored_0113|E1|6|40|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|scored_0113|E2|52-1|40||40|válido|
|scored_0113|E2|52-3|40||40|válido|
|scored_0113|E2|52-6|40||40|válido|
|scored_0113|E2|64-1|40||40|válido|
|scored_0113|E2|64-3|40||40|válido|
|scored_0113|E2|64-6|40||40|válido|
|scored_0113|E3|joint|40|individual 40|40|OPTIMAL|
|scored_0113|E4|repair|40||39|completo|
|scored_0114|E1|1|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0114|E1|3|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0114|E1|6|222|postprocess 222, optimize_a4 222, nested_windows 222|222|válido|
|scored_0114|E2|52-1|226||215|válido|
|scored_0114|E2|52-3|223||211|válido|
|scored_0114|E2|52-6|222||211|válido|
|scored_0114|E2|64-1|226||216|válido|
|scored_0114|E2|64-3|223||211|válido|
|scored_0114|E2|64-6|222||212|válido|
|scored_0114|E3|joint|222|individual 222|222|OPTIMAL|
|scored_0114|E4|repair|222||222|completo|
|scored_0115|E1|1|282|postprocess 280, optimize_a4 280, nested_windows 280|280|válido|
|scored_0115|E1|3|279|postprocess 279, optimize_a4 279, nested_windows 279|279|válido|
|scored_0115|E1|6|271|postprocess 271, optimize_a4 271, nested_windows 271|271|válido|
|scored_0115|E2|52-1|282||266|válido|
|scored_0115|E2|52-3|279||254|válido|
|scored_0115|E2|52-6|271||259|válido|
|scored_0115|E2|64-1|282||268|válido|
|scored_0115|E2|64-3|279||254|válido|
|scored_0115|E2|64-6|271||258|válido|
|scored_0115|E3|joint|271|individual 271|271|OPTIMAL|
|scored_0115|E4|repair|271||271|completo|
|scored_0116|E1|1|117|postprocess 117, optimize_a4 117, nested_windows 117|117|válido|
|scored_0116|E1|3|117|postprocess 117, optimize_a4 117, nested_windows 117|117|válido|
|scored_0116|E1|6|117|postprocess 117, optimize_a4 117, nested_windows 117|117|válido|
|scored_0116|E2|52-1|117||111|válido|
|scored_0116|E2|52-3|117||111|válido|
|scored_0116|E2|52-6|117||111|válido|
|scored_0116|E2|64-1|117||112|válido|
|scored_0116|E2|64-3|117||112|válido|
|scored_0116|E2|64-6|117||110|válido|
|scored_0116|E3|joint|117|individual 117|117|OPTIMAL|
|scored_0116|E4|repair|117||114|completo|
|scored_0117|E1|1|272|postprocess 272, optimize_a4 272, nested_windows 272|272|válido|
|scored_0117|E1|3|259|postprocess 259, optimize_a4 259, nested_windows 259|259|válido|
|scored_0117|E1|6|261|postprocess 261, optimize_a4 261, nested_windows 261|261|válido|
|scored_0117|E2|52-1|272||254|válido|
|scored_0117|E2|52-3|259||249|válido|
|scored_0117|E2|52-6|261||242|válido|
|scored_0117|E2|64-1|272||254|válido|
|scored_0117|E2|64-3|259||249|válido|
|scored_0117|E2|64-6|261||246|válido|
|scored_0117|E3|joint|259|individual 258|258|rechazado por registros|
|scored_0117|E4|repair|259||259|completo|
|scored_0118|E1|1|137|postprocess 136, optimize_a4 136, nested_windows 136|136|válido|
|scored_0118|E1|3|136|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|scored_0118|E1|6|136|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|scored_0118|E2|52-1|137||132|válido|
|scored_0118|E2|52-3|136||132|válido|
|scored_0118|E2|52-6|136||132|válido|
|scored_0118|E2|64-1|137||132|válido|
|scored_0118|E2|64-3|136||131|válido|
|scored_0118|E2|64-6|136||131|válido|
|scored_0118|E3|joint|136|individual 136|136|OPTIMAL|
|scored_0118|E4|repair|136||134|completo|
|scored_0119|E1|1|330|postprocess 330, optimize_a4 330, nested_windows 330|330|válido|
|scored_0119|E1|3|311|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|scored_0119|E1|6|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0119|E2|52-1|330||302|válido|
|scored_0119|E2|52-3|311||296|válido|
|scored_0119|E2|52-6|303||287|válido|
|scored_0119|E2|64-1|330||303|válido|
|scored_0119|E2|64-3|311||294|válido|
|scored_0119|E2|64-6|303||288|válido|
|scored_0119|E3|joint|303|individual 303|301|OPTIMAL|
|scored_0119|E4|repair|303||303|completo|
|scored_0120|E1|1|282|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|scored_0120|E1|3|268|postprocess 268, optimize_a4 268, nested_windows 268|268|válido|
|scored_0120|E1|6|271|postprocess 271, optimize_a4 271, nested_windows 271|271|válido|
|scored_0120|E2|52-1|282||262|válido|
|scored_0120|E2|52-3|268||251|válido|
|scored_0120|E2|52-6|271||247|válido|
|scored_0120|E2|64-1|282||263|válido|
|scored_0120|E2|64-3|268||256|válido|
|scored_0120|E2|64-6|271||250|válido|
|scored_0120|E3|joint|268|individual 268|267|OPTIMAL|
|scored_0120|E4|repair|268||268|completo|
|scored_0121|E1|1|66|postprocess 66, optimize_a4 66, nested_windows 66|66|válido|
|scored_0121|E1|3|66|postprocess 66, optimize_a4 66, nested_windows 66|66|válido|
|scored_0121|E1|6|66|postprocess 66, optimize_a4 66, nested_windows 66|66|válido|
|scored_0121|E2|52-1|66||67|válido|
|scored_0121|E2|52-3|66||67|válido|
|scored_0121|E2|52-6|66||67|válido|
|scored_0121|E2|64-1|66||67|válido|
|scored_0121|E2|64-3|66||67|válido|
|scored_0121|E2|64-6|66||67|válido|
|scored_0121|E3|joint|66|individual 66|66|OPTIMAL|
|scored_0121|E4|repair|66||64|completo|
|scored_0122|E1|1|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0122|E1|3|210|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0122|E1|6|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0122|E2|52-1|219||205|válido|
|scored_0122|E2|52-3|210||200|válido|
|scored_0122|E2|52-6|211||199|válido|
|scored_0122|E2|64-1|219||208|válido|
|scored_0122|E2|64-3|210||199|válido|
|scored_0122|E2|64-6|211||201|válido|
|scored_0122|E3|joint|210|individual 210|209|OPTIMAL|
|scored_0122|E4|repair|210||210|completo|
|scored_0123|E1|1|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0123|E1|3|291|postprocess 291, optimize_a4 291, nested_windows 291|291|válido|
|scored_0123|E1|6|292|postprocess 292, optimize_a4 292, nested_windows 292|292|válido|
|scored_0123|E2|52-1|303||289|válido|
|scored_0123|E2|52-3|291||273|válido|
|scored_0123|E2|52-6|292||271|válido|
|scored_0123|E2|64-1|303||290|válido|
|scored_0123|E2|64-3|291||274|válido|
|scored_0123|E2|64-6|292||269|válido|
|scored_0123|E3|joint|291|individual 289|284|FEASIBLE|
|scored_0123|E4|repair|291||291|completo|
|scored_0124|E1|1|37|postprocess 35, optimize_a4 35, nested_windows 35|35|válido|
|scored_0124|E1|3|37|postprocess 35, optimize_a4 35, nested_windows 35|35|válido|
|scored_0124|E1|6|37|postprocess 35, optimize_a4 35, nested_windows 35|35|válido|
|scored_0124|E2|52-1|37||37|válido|
|scored_0124|E2|52-3|37||37|válido|
|scored_0124|E2|52-6|37||37|válido|
|scored_0124|E2|64-1|37||37|válido|
|scored_0124|E2|64-3|37||37|válido|
|scored_0124|E2|64-6|37||37|válido|
|scored_0124|E3|joint|37|individual 37|37|OPTIMAL|
|scored_0124|E4|repair|37||37|completo|
|scored_0125|E1|1|157|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0125|E1|3|153|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0125|E1|6|152|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0125|E2|52-1|157||149|válido|
|scored_0125|E2|52-3|153||148|válido|
|scored_0125|E2|52-6|152||148|válido|
|scored_0125|E2|64-1|157||150|válido|
|scored_0125|E2|64-3|153||148|válido|
|scored_0125|E2|64-6|152||149|válido|
|scored_0125|E3|joint|152|individual 151|151|OPTIMAL|
|scored_0125|E4|repair|152||150|completo|
|scored_0126|E1|1|5|postprocess 5, optimize_a4 4, nested_windows 4|4|válido|
|scored_0126|E1|3|5|postprocess 5, optimize_a4 4, nested_windows 4|4|válido|
|scored_0126|E1|6|5|postprocess 5, optimize_a4 4, nested_windows 4|4|válido|
|scored_0126|E2|52-1|5||5|válido|
|scored_0126|E2|52-3|5||5|válido|
|scored_0126|E2|52-6|5||5|válido|
|scored_0126|E2|64-1|5||5|válido|
|scored_0126|E2|64-3|5||5|válido|
|scored_0126|E2|64-6|5||5|válido|
|scored_0126|E3|joint|4|individual 4|4|OPTIMAL|
|scored_0126|E4|repair|4||4|completo|
|scored_0127|E1|1|172|postprocess 170, optimize_a4 170, nested_windows 170|170|válido|
|scored_0127|E1|3|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0127|E1|6|164|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0127|E2|52-1|172||149|válido|
|scored_0127|E2|52-3|169||148|válido|
|scored_0127|E2|52-6|164||146|válido|
|scored_0127|E2|64-1|172||155|válido|
|scored_0127|E2|64-3|169||155|válido|
|scored_0127|E2|64-6|164||151|válido|
|scored_0127|E3|joint|164|individual 164|164|OPTIMAL|
|scored_0127|E4|repair|164||156|completo|
|scored_0128|E1|1|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0128|E1|3|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0128|E1|6|168|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0128|E2|52-1|172||161|válido|
|scored_0128|E2|52-3|169||161|válido|
|scored_0128|E2|52-6|168||163|válido|
|scored_0128|E2|64-1|172||161|válido|
|scored_0128|E2|64-3|169||161|válido|
|scored_0128|E2|64-6|168||160|válido|
|scored_0128|E3|joint|168|individual 168|168|OPTIMAL|
|scored_0128|E4|repair|168||167|completo|
|scored_0129|E1|1|382|postprocess 382, optimize_a4 382, nested_windows 382|382|válido|
|scored_0129|E1|3|369|postprocess 369, optimize_a4 369, nested_windows 369|369|válido|
|scored_0129|E1|6|366|postprocess 366, optimize_a4 366, nested_windows 366|366|válido|
|scored_0129|E2|52-1|382||356|válido|
|scored_0129|E2|52-3|369||346|válido|
|scored_0129|E2|52-6|366||340|válido|
|scored_0129|E2|64-1|382||358|válido|
|scored_0129|E2|64-3|369||349|válido|
|scored_0129|E2|64-6|366||455|válido|
|scored_0129|E3|joint|366|individual 366|365|FEASIBLE|
|scored_0129|E4|repair|366||366|completo|
|scored_0130|E1|1|39|postprocess 36, optimize_a4 35, nested_windows 35|35|válido|
|scored_0130|E1|3|39|postprocess 36, optimize_a4 35, nested_windows 35|35|válido|
|scored_0130|E1|6|39|postprocess 36, optimize_a4 35, nested_windows 35|35|válido|
|scored_0130|E2|52-1|39||39|válido|
|scored_0130|E2|52-3|39||39|válido|
|scored_0130|E2|52-6|39||39|válido|
|scored_0130|E2|64-1|39||39|válido|
|scored_0130|E2|64-3|39||39|válido|
|scored_0130|E2|64-6|39||39|válido|
|scored_0130|E3|joint|39|individual 39|39|OPTIMAL|
|scored_0130|E4|repair|39||39|completo|
|scored_0131|E1|1|125|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0131|E1|3|125|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0131|E1|6|125|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0131|E2|52-1|125||122|válido|
|scored_0131|E2|52-3|125||122|válido|
|scored_0131|E2|52-6|125||122|válido|
|scored_0131|E2|64-1|125||121|válido|
|scored_0131|E2|64-3|125||121|válido|
|scored_0131|E2|64-6|125||121|válido|
|scored_0131|E3|joint|125|individual 125|125|OPTIMAL|
|scored_0131|E4|repair|125||121|completo|
|scored_0132|E1|1|138|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0132|E1|3|138|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0132|E1|6|138|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0132|E2|52-1|138||133|válido|
|scored_0132|E2|52-3|138||133|válido|
|scored_0132|E2|52-6|138||133|válido|
|scored_0132|E2|64-1|138||133|válido|
|scored_0132|E2|64-3|138||133|válido|
|scored_0132|E2|64-6|138||133|válido|
|scored_0132|E3|joint|138|individual 138|138|OPTIMAL|
|scored_0132|E4|repair|138||138|completo|
|scored_0133|E1|1|348|postprocess 348, optimize_a4 348, nested_windows 348|348|válido|
|scored_0133|E1|3|315|postprocess 315, optimize_a4 315, nested_windows 315|315|válido|
|scored_0133|E1|6|308|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0133|E2|52-1|348||305|válido|
|scored_0133|E2|52-3|315||293|válido|
|scored_0133|E2|52-6|308||289|válido|
|scored_0133|E2|64-1|348||308|válido|
|scored_0133|E2|64-3|315||295|válido|
|scored_0133|E2|64-6|308||292|válido|
|scored_0133|E3|joint|308|individual 307|305|FEASIBLE|
|scored_0133|E4|repair|308||308|completo|
|scored_0134|E1|1|44|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0134|E1|3|44|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0134|E1|6|44|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0134|E2|52-1|44||44|válido|
|scored_0134|E2|52-3|44||44|válido|
|scored_0134|E2|52-6|44||44|válido|
|scored_0134|E2|64-1|44||44|válido|
|scored_0134|E2|64-3|44||44|válido|
|scored_0134|E2|64-6|44||44|válido|
|scored_0134|E3|joint|44|individual 44|44|OPTIMAL|
|scored_0134|E4|repair|44||44|completo|
|scored_0135|E1|1|123|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0135|E1|3|123|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0135|E1|6|123|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0135|E2|52-1|123||120|válido|
|scored_0135|E2|52-3|123||120|válido|
|scored_0135|E2|52-6|123||120|válido|
|scored_0135|E2|64-1|123||119|válido|
|scored_0135|E2|64-3|123||119|válido|
|scored_0135|E2|64-6|123||119|válido|
|scored_0135|E3|joint|123|individual 123|123|OPTIMAL|
|scored_0135|E4|repair|123||122|completo|
|scored_0136|E1|1|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0136|E1|3|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0136|E1|6|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0136|E2|52-1|192||163|válido|
|scored_0136|E2|52-3|188||167|válido|
|scored_0136|E2|52-6|188||165|válido|
|scored_0136|E2|64-1|192||160|válido|
|scored_0136|E2|64-3|188||160|válido|
|scored_0136|E2|64-6|188||157|válido|
|scored_0136|E3|joint|188|individual 187|187|OPTIMAL|
|scored_0136|E4|repair|188||187|completo|
|scored_0137|E1|1|76|postprocess 74, optimize_a4 74, nested_windows 74|74|válido|
|scored_0137|E1|3|76|postprocess 74, optimize_a4 74, nested_windows 74|74|válido|
|scored_0137|E1|6|76|postprocess 74, optimize_a4 74, nested_windows 74|74|válido|
|scored_0137|E2|52-1|76||75|válido|
|scored_0137|E2|52-3|76||75|válido|
|scored_0137|E2|52-6|76||75|válido|
|scored_0137|E2|64-1|76||75|válido|
|scored_0137|E2|64-3|76||75|válido|
|scored_0137|E2|64-6|76||75|válido|
|scored_0137|E3|joint|76|individual 76|76|OPTIMAL|
|scored_0137|E4|repair|76||72|completo|
|scored_0138|E1|1|124|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|scored_0138|E1|3|124|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|scored_0138|E1|6|122|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0138|E2|52-1|124||119|válido|
|scored_0138|E2|52-3|124||119|válido|
|scored_0138|E2|52-6|122||119|válido|
|scored_0138|E2|64-1|124||119|válido|
|scored_0138|E2|64-3|124||119|válido|
|scored_0138|E2|64-6|122||119|válido|
|scored_0138|E3|joint|122|individual 122|122|OPTIMAL|
|scored_0138|E4|repair|122||120|completo|
|scored_0139|E1|1|251|postprocess 251, optimize_a4 251, nested_windows 251|251|válido|
|scored_0139|E1|3|249|postprocess 249, optimize_a4 249, nested_windows 249|249|válido|
|scored_0139|E1|6|246|postprocess 245, optimize_a4 245, nested_windows 245|245|válido|
|scored_0139|E2|52-1|251||238|válido|
|scored_0139|E2|52-3|249||233|válido|
|scored_0139|E2|52-6|246||228|válido|
|scored_0139|E2|64-1|251||238|válido|
|scored_0139|E2|64-3|249||231|válido|
|scored_0139|E2|64-6|246||228|válido|
|scored_0139|E3|joint|246|individual 246|243|OPTIMAL|
|scored_0139|E4|repair|246||246|completo|
|scored_0140|E1|1|226|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0140|E1|3|215|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0140|E1|6|209|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0140|E2|52-1|226||209|válido|
|scored_0140|E2|52-3|215||200|válido|
|scored_0140|E2|52-6|209||199|válido|
|scored_0140|E2|64-1|226||208|válido|
|scored_0140|E2|64-3|215||201|válido|
|scored_0140|E2|64-6|209||204|válido|
|scored_0140|E3|joint|209|individual 209|209|OPTIMAL|
|scored_0140|E4|repair|209||209|completo|
|scored_0141|E1|1|174|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0141|E1|3|173|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|scored_0141|E1|6|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0141|E2|52-1|174||168|válido|
|scored_0141|E2|52-3|173||168|válido|
|scored_0141|E2|52-6|172||168|válido|
|scored_0141|E2|64-1|174||170|válido|
|scored_0141|E2|64-3|173||167|válido|
|scored_0141|E2|64-6|172||168|válido|
|scored_0141|E3|joint|172|individual 172|172|OPTIMAL|
|scored_0141|E4|repair|172||170|completo|
|scored_0142|E1|1|158|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0142|E1|3|156|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0142|E1|6|156|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0142|E2|52-1|158||148|válido|
|scored_0142|E2|52-3|156||148|válido|
|scored_0142|E2|52-6|156||148|válido|
|scored_0142|E2|64-1|158||149|válido|
|scored_0142|E2|64-3|156||149|válido|
|scored_0142|E2|64-6|156||149|válido|
|scored_0142|E3|joint|156|individual 156|156|OPTIMAL|
|scored_0142|E4|repair|156||156|completo|
|scored_0143|E1|1|236|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0143|E1|3|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0143|E1|6|221|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0143|E2|52-1|236||202|válido|
|scored_0143|E2|52-3|233||207|válido|
|scored_0143|E2|52-6|221||190|válido|
|scored_0143|E2|64-1|236||204|válido|
|scored_0143|E2|64-3|233||203|válido|
|scored_0143|E2|64-6|221||188|válido|
|scored_0143|E3|joint|221|individual 221|220|OPTIMAL|
|scored_0143|E4|repair|221||220|completo|
|scored_0144|E1|1|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0144|E1|3|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0144|E1|6|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0144|E2|52-1|151||147|válido|
|scored_0144|E2|52-3|151||145|válido|
|scored_0144|E2|52-6|148||144|válido|
|scored_0144|E2|64-1|151||142|válido|
|scored_0144|E2|64-3|151||142|válido|
|scored_0144|E2|64-6|148||142|válido|
|scored_0144|E3|joint|148|individual 148|148|OPTIMAL|
|scored_0144|E4|repair|148||141|completo|
|scored_0145|E1|1|308|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0145|E1|3|297|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0145|E1|6|287|postprocess 287, optimize_a4 287, nested_windows 287|287|válido|
|scored_0145|E2|52-1|308||280|válido|
|scored_0145|E2|52-3|297||276|válido|
|scored_0145|E2|52-6|287||269|válido|
|scored_0145|E2|64-1|308||282|válido|
|scored_0145|E2|64-3|297||276|válido|
|scored_0145|E2|64-6|287||273|válido|
|scored_0145|E3|joint|287|individual 287|287|OPTIMAL|
|scored_0145|E4|repair|287||287|completo|
|scored_0146|E1|1|364|postprocess 364, optimize_a4 364, nested_windows 364|364|válido|
|scored_0146|E1|3|350|postprocess 350, optimize_a4 350, nested_windows 350|350|válido|
|scored_0146|E1|6|344|postprocess 344, optimize_a4 344, nested_windows 344|344|válido|
|scored_0146|E2|52-1|364||338|válido|
|scored_0146|E2|52-3|350||330|válido|
|scored_0146|E2|52-6|344||329|válido|
|scored_0146|E2|64-1|364||335|válido|
|scored_0146|E2|64-3|350||328|válido|
|scored_0146|E2|64-6|344||327|válido|
|scored_0146|E3|joint|344|individual 344|343|OPTIMAL|
|scored_0146|E4|repair|344||344|completo|
|scored_0147|E1|1|123|postprocess 123, optimize_a4 123, nested_windows 123|123|válido|
|scored_0147|E1|3|122|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0147|E1|6|122|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0147|E2|52-1|123||123|válido|
|scored_0147|E2|52-3|122||123|válido|
|scored_0147|E2|52-6|122||123|válido|
|scored_0147|E2|64-1|123||122|válido|
|scored_0147|E2|64-3|122||122|válido|
|scored_0147|E2|64-6|122||122|válido|
|scored_0147|E3|joint|122|individual 122|122|OPTIMAL|
|scored_0147|E4|repair|122||119|completo|
|scored_0148|E1|1|248|postprocess 248, optimize_a4 248, nested_windows 248|248|válido|
|scored_0148|E1|3|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0148|E1|6|234|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0148|E2|52-1|248||228|válido|
|scored_0148|E2|52-3|239||222|válido|
|scored_0148|E2|52-6|234||217|válido|
|scored_0148|E2|64-1|248||230|válido|
|scored_0148|E2|64-3|239||221|válido|
|scored_0148|E2|64-6|234||221|válido|
|scored_0148|E3|joint|234|individual 234|233|OPTIMAL|
|scored_0148|E4|repair|234||234|completo|
|scored_0149|E1|1|30|postprocess 28, optimize_a4 27, nested_windows 26|26|válido|
|scored_0149|E1|3|30|postprocess 28, optimize_a4 27, nested_windows 26|26|válido|
|scored_0149|E1|6|30|postprocess 28, optimize_a4 27, nested_windows 26|26|válido|
|scored_0149|E2|52-1|30||30|válido|
|scored_0149|E2|52-3|30||30|válido|
|scored_0149|E2|52-6|30||30|válido|
|scored_0149|E2|64-1|30||30|válido|
|scored_0149|E2|64-3|30||30|válido|
|scored_0149|E2|64-6|30||30|válido|
|scored_0149|E3|joint|28|individual 27|26|OPTIMAL|
|scored_0149|E4|repair|28||28|completo|
|scored_0150|E1|1|228|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|scored_0150|E1|3|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0150|E1|6|217|postprocess 217, optimize_a4 217, nested_windows 217|217|válido|
|scored_0150|E2|52-1|228||216|válido|
|scored_0150|E2|52-3|223||208|válido|
|scored_0150|E2|52-6|217||208|válido|
|scored_0150|E2|64-1|228||217|válido|
|scored_0150|E2|64-3|223||211|válido|
|scored_0150|E2|64-6|217||205|válido|
|scored_0150|E3|joint|217|individual 217|217|OPTIMAL|
|scored_0150|E4|repair|217||217|completo|
|scored_0151|E1|1|44|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0151|E1|3|44|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0151|E1|6|44|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0151|E2|52-1|44||44|válido|
|scored_0151|E2|52-3|44||44|válido|
|scored_0151|E2|52-6|44||44|válido|
|scored_0151|E2|64-1|44||44|válido|
|scored_0151|E2|64-3|44||44|válido|
|scored_0151|E2|64-6|44||44|válido|
|scored_0151|E3|joint|44|individual 44|44|OPTIMAL|
|scored_0151|E4|repair|44||44|completo|
|scored_0152|E1|1|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0152|E1|3|157|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0152|E1|6|156|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0152|E2|52-1|159||159|válido|
|scored_0152|E2|52-3|157||158|válido|
|scored_0152|E2|52-6|156||158|válido|
|scored_0152|E2|64-1|159||158|válido|
|scored_0152|E2|64-3|157||156|válido|
|scored_0152|E2|64-6|156||156|válido|
|scored_0152|E3|joint|156|individual 155|155|OPTIMAL|
|scored_0152|E4|repair|156||154|completo|
|scored_0153|E1|1|142|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0153|E1|3|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0153|E1|6|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0153|E2|52-1|142||138|válido|
|scored_0153|E2|52-3|145||138|válido|
|scored_0153|E2|52-6|145||138|válido|
|scored_0153|E2|64-1|142||138|válido|
|scored_0153|E2|64-3|145||138|válido|
|scored_0153|E2|64-6|145||138|válido|
|scored_0153|E3|joint|142|individual 142|142|OPTIMAL|
|scored_0153|E4|repair|142||141|completo|
|scored_0154|E1|1|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0154|E1|3|225|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0154|E1|6|222|postprocess 222, optimize_a4 222, nested_windows 222|222|válido|
|scored_0154|E2|52-1|233||213|válido|
|scored_0154|E2|52-3|225||215|válido|
|scored_0154|E2|52-6|222||206|válido|
|scored_0154|E2|64-1|233||214|válido|
|scored_0154|E2|64-3|225||215|válido|
|scored_0154|E2|64-6|222||206|válido|
|scored_0154|E3|joint|222|individual 222|222|OPTIMAL|
|scored_0154|E4|repair|222||222|completo|
|scored_0155|E1|1|321|postprocess 320, optimize_a4 320, nested_windows 320|320|válido|
|scored_0155|E1|3|308|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0155|E1|6|298|postprocess 298, optimize_a4 298, nested_windows 298|298|válido|
|scored_0155|E2|52-1|321||289|válido|
|scored_0155|E2|52-3|308||286|válido|
|scored_0155|E2|52-6|298||285|válido|
|scored_0155|E2|64-1|321||291|válido|
|scored_0155|E2|64-3|308||289|válido|
|scored_0155|E2|64-6|298||282|válido|
|scored_0155|E3|joint|298|individual 298|298|FEASIBLE|
|scored_0155|E4|repair|298||298|completo|
|scored_0156|E1|1|187|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0156|E1|3|182|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0156|E1|6|182|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0156|E2|52-1|187||175|válido|
|scored_0156|E2|52-3|182||177|válido|
|scored_0156|E2|52-6|182||173|válido|
|scored_0156|E2|64-1|187||176|válido|
|scored_0156|E2|64-3|182||177|válido|
|scored_0156|E2|64-6|182||171|válido|
|scored_0156|E3|joint|182|individual 182|181|OPTIMAL|
|scored_0156|E4|repair|182||182|completo|
|scored_0157|E1|1|216|postprocess 216, optimize_a4 216, nested_windows 216|216|válido|
|scored_0157|E1|3|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0157|E1|6|208|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|scored_0157|E2|52-1|216||205|válido|
|scored_0157|E2|52-3|206||200|válido|
|scored_0157|E2|52-6|208||198|válido|
|scored_0157|E2|64-1|216||205|válido|
|scored_0157|E2|64-3|206||203|válido|
|scored_0157|E2|64-6|208||200|válido|
|scored_0157|E3|joint|206|individual 205|205|OPTIMAL|
|scored_0157|E4|repair|206||206|completo|
|scored_0158|E1|1|204|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0158|E1|3|200|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0158|E1|6|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0158|E2|52-1|204||197|válido|
|scored_0158|E2|52-3|200||194|válido|
|scored_0158|E2|52-6|199||192|válido|
|scored_0158|E2|64-1|204||197|válido|
|scored_0158|E2|64-3|200||193|válido|
|scored_0158|E2|64-6|199||192|válido|
|scored_0158|E3|joint|199|individual 199|199|OPTIMAL|
|scored_0158|E4|repair|199||199|completo|
|scored_0159|E1|1|330|postprocess 330, optimize_a4 330, nested_windows 330|330|válido|
|scored_0159|E1|3|304|postprocess 304, optimize_a4 304, nested_windows 304|304|válido|
|scored_0159|E1|6|307|postprocess 306, optimize_a4 306, nested_windows 306|306|válido|
|scored_0159|E2|52-1|330||300|válido|
|scored_0159|E2|52-3|304||288|válido|
|scored_0159|E2|52-6|307||286|válido|
|scored_0159|E2|64-1|330||301|válido|
|scored_0159|E2|64-3|304||312|válido|
|scored_0159|E2|64-6|307||285|válido|
|scored_0159|E3|joint|304|individual 304|301|OPTIMAL|
|scored_0159|E4|repair|304||304|completo|
|scored_0160|E1|1|256|postprocess 254, optimize_a4 254, nested_windows 254|254|válido|
|scored_0160|E1|3|253|postprocess 253, optimize_a4 253, nested_windows 253|253|válido|
|scored_0160|E1|6|248|postprocess 248, optimize_a4 248, nested_windows 248|248|válido|
|scored_0160|E2|52-1|256||238|válido|
|scored_0160|E2|52-3|253||231|válido|
|scored_0160|E2|52-6|248||229|válido|
|scored_0160|E2|64-1|256||237|válido|
|scored_0160|E2|64-3|253||236|válido|
|scored_0160|E2|64-6|248||229|válido|
|scored_0160|E3|joint|248|individual 248|248|OPTIMAL|
|scored_0160|E4|repair|248||248|completo|
|scored_0161|E1|1|213|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0161|E1|3|213|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0161|E1|6|209|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0161|E2|52-1|213||201|válido|
|scored_0161|E2|52-3|213||201|válido|
|scored_0161|E2|52-6|209||197|válido|
|scored_0161|E2|64-1|213||201|válido|
|scored_0161|E2|64-3|213||200|válido|
|scored_0161|E2|64-6|209||196|válido|
|scored_0161|E3|joint|209|individual 209|209|OPTIMAL|
|scored_0161|E4|repair|209||208|completo|
|scored_0162|E1|1|264|postprocess 264, optimize_a4 264, nested_windows 264|264|válido|
|scored_0162|E1|3|254|postprocess 254, optimize_a4 254, nested_windows 254|254|válido|
|scored_0162|E1|6|250|postprocess 250, optimize_a4 250, nested_windows 250|250|válido|
|scored_0162|E2|52-1|264||250|válido|
|scored_0162|E2|52-3|254||239|válido|
|scored_0162|E2|52-6|250||229|válido|
|scored_0162|E2|64-1|264||242|válido|
|scored_0162|E2|64-3|254||231|válido|
|scored_0162|E2|64-6|250||231|válido|
|scored_0162|E3|joint|250|individual 250|249|OPTIMAL|
|scored_0162|E4|repair|250||250|completo|
|scored_0163|E1|1|352|postprocess 350, optimize_a4 350, nested_windows 350|350|válido|
|scored_0163|E1|3|333|postprocess 333, optimize_a4 333, nested_windows 333|333|válido|
|scored_0163|E1|6|333|postprocess 333, optimize_a4 333, nested_windows 333|333|válido|
|scored_0163|E2|52-1|352||328|válido|
|scored_0163|E2|52-3|333||312|válido|
|scored_0163|E2|52-6|333||314|válido|
|scored_0163|E2|64-1|352||329|válido|
|scored_0163|E2|64-3|333||313|válido|
|scored_0163|E2|64-6|333||315|válido|
|scored_0163|E3|joint|333|individual 331|327|FEASIBLE|
|scored_0163|E4|repair|333||333|completo|
|scored_0164|E1|1|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0164|E1|3|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0164|E1|6|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0164|E2|52-1|169||161|válido|
|scored_0164|E2|52-3|165||161|válido|
|scored_0164|E2|52-6|165||161|válido|
|scored_0164|E2|64-1|169||161|válido|
|scored_0164|E2|64-3|165||161|válido|
|scored_0164|E2|64-6|165||161|válido|
|scored_0164|E3|joint|165|individual 165|165|OPTIMAL|
|scored_0164|E4|repair|165||164|completo|
|scored_0165|E1|1|346|postprocess 346, optimize_a4 346, nested_windows 346|346|válido|
|scored_0165|E1|3|330|postprocess 330, optimize_a4 330, nested_windows 330|330|válido|
|scored_0165|E1|6|324|postprocess 324, optimize_a4 324, nested_windows 324|324|válido|
|scored_0165|E2|52-1|346||311|válido|
|scored_0165|E2|52-3|330||304|válido|
|scored_0165|E2|52-6|324||302|válido|
|scored_0165|E2|64-1|346||314|válido|
|scored_0165|E2|64-3|330||304|válido|
|scored_0165|E2|64-6|324||303|válido|
|scored_0165|E3|joint|324|individual 323|319|OPTIMAL|
|scored_0165|E4|repair|324||324|completo|
|scored_0166|E1|1|298|postprocess 298, optimize_a4 298, nested_windows 298|298|válido|
|scored_0166|E1|3|288|postprocess 287, optimize_a4 287, nested_windows 287|287|válido|
|scored_0166|E1|6|282|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|scored_0166|E2|52-1|298||274|válido|
|scored_0166|E2|52-3|288||262|válido|
|scored_0166|E2|52-6|282||257|válido|
|scored_0166|E2|64-1|298||274|válido|
|scored_0166|E2|64-3|288||264|válido|
|scored_0166|E2|64-6|282||257|válido|
|scored_0166|E3|joint|282|individual 282|282|OPTIMAL|
|scored_0166|E4|repair|282||282|completo|
|scored_0167|E1|1|356|postprocess 356, optimize_a4 356, nested_windows 356|356|válido|
|scored_0167|E1|3|337|postprocess 337, optimize_a4 337, nested_windows 337|337|válido|
|scored_0167|E1|6|333|postprocess 332, optimize_a4 332, nested_windows 332|332|válido|
|scored_0167|E2|52-1|356||327|válido|
|scored_0167|E2|52-3|337||317|válido|
|scored_0167|E2|52-6|333||312|válido|
|scored_0167|E2|64-1|356||328|válido|
|scored_0167|E2|64-3|337||317|válido|
|scored_0167|E2|64-6|333||310|válido|
|scored_0167|E3|joint|333|individual 333|333|FEASIBLE|
|scored_0167|E4|repair|333||333|completo|
|scored_0168|E1|1|389|postprocess 388, optimize_a4 388, nested_windows 388|388|válido|
|scored_0168|E1|3|364|postprocess 363, optimize_a4 363, nested_windows 363|363|válido|
|scored_0168|E1|6|364|postprocess 364, optimize_a4 364, nested_windows 364|364|válido|
|scored_0168|E2|52-1|389||361|válido|
|scored_0168|E2|52-3|364||343|válido|
|scored_0168|E2|52-6|364||337|válido|
|scored_0168|E2|64-1|389||363|válido|
|scored_0168|E2|64-3|364||343|válido|
|scored_0168|E2|64-6|364||341|válido|
|scored_0168|E3|joint|364|individual 363|357|FEASIBLE|
|scored_0168|E4|repair|364||364|completo|
|scored_0169|E1|1|140|postprocess 140, optimize_a4 140, nested_windows 140|140|válido|
|scored_0169|E1|3|140|postprocess 140, optimize_a4 140, nested_windows 140|140|válido|
|scored_0169|E1|6|139|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|scored_0169|E2|52-1|140||131|válido|
|scored_0169|E2|52-3|140||131|válido|
|scored_0169|E2|52-6|139||132|válido|
|scored_0169|E2|64-1|140||133|válido|
|scored_0169|E2|64-3|140||133|válido|
|scored_0169|E2|64-6|139||133|válido|
|scored_0169|E3|joint|139|individual 139|139|OPTIMAL|
|scored_0169|E4|repair|139||136|completo|
|scored_0170|E1|1|320|postprocess 319, optimize_a4 319, nested_windows 319|319|válido|
|scored_0170|E1|3|309|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0170|E1|6|298|postprocess 298, optimize_a4 298, nested_windows 298|298|válido|
|scored_0170|E2|52-1|320||301|válido|
|scored_0170|E2|52-3|309||284|válido|
|scored_0170|E2|52-6|298||285|válido|
|scored_0170|E2|64-1|320||298|válido|
|scored_0170|E2|64-3|309||283|válido|
|scored_0170|E2|64-6|298||283|válido|
|scored_0170|E3|joint|298|individual 298|296|OPTIMAL|
|scored_0170|E4|repair|298||297|completo|
|scored_0171|E1|1|175|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|scored_0171|E1|3|173|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|scored_0171|E1|6|170|postprocess 170, optimize_a4 170, nested_windows 170|170|válido|
|scored_0171|E2|52-1|175||164|válido|
|scored_0171|E2|52-3|173||159|válido|
|scored_0171|E2|52-6|170||164|válido|
|scored_0171|E2|64-1|175||164|válido|
|scored_0171|E2|64-3|173||161|válido|
|scored_0171|E2|64-6|170||159|válido|
|scored_0171|E3|joint|170|individual 170|170|OPTIMAL|
|scored_0171|E4|repair|170||170|completo|
|scored_0172|E1|1|237|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0172|E1|3|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0172|E1|6|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0172|E2|52-1|237||214|válido|
|scored_0172|E2|52-3|223||206|válido|
|scored_0172|E2|52-6|219||204|válido|
|scored_0172|E2|64-1|237||214|válido|
|scored_0172|E2|64-3|223||206|válido|
|scored_0172|E2|64-6|219||206|válido|
|scored_0172|E3|joint|219|individual 219|219|OPTIMAL|
|scored_0172|E4|repair|219||219|completo|
|scored_0173|E1|1|267|postprocess 266, optimize_a4 266, nested_windows 266|266|válido|
|scored_0173|E1|3|262|postprocess 262, optimize_a4 262, nested_windows 262|262|válido|
|scored_0173|E1|6|254|postprocess 254, optimize_a4 254, nested_windows 254|254|válido|
|scored_0173|E2|52-1|267||241|válido|
|scored_0173|E2|52-3|262||235|válido|
|scored_0173|E2|52-6|254||236|válido|
|scored_0173|E2|64-1|267||242|válido|
|scored_0173|E2|64-3|262||234|válido|
|scored_0173|E2|64-6|254||236|válido|
|scored_0173|E3|joint|254|individual 254|254|OPTIMAL|
|scored_0173|E4|repair|254||254|completo|
|scored_0174|E1|1|190|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0174|E1|3|187|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0174|E1|6|183|postprocess 183, optimize_a4 183, nested_windows 183|183|válido|
|scored_0174|E2|52-1|190||180|válido|
|scored_0174|E2|52-3|187||178|válido|
|scored_0174|E2|52-6|183||175|válido|
|scored_0174|E2|64-1|190||180|válido|
|scored_0174|E2|64-3|187||178|válido|
|scored_0174|E2|64-6|183||177|válido|
|scored_0174|E3|joint|183|individual 183|183|OPTIMAL|
|scored_0174|E4|repair|183||181|completo|
|scored_0175|E1|1|136|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|scored_0175|E1|3|136|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|scored_0175|E1|6|136|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|scored_0175|E2|52-1|136||133|válido|
|scored_0175|E2|52-3|136||133|válido|
|scored_0175|E2|52-6|136||133|válido|
|scored_0175|E2|64-1|136||133|válido|
|scored_0175|E2|64-3|136||133|válido|
|scored_0175|E2|64-6|136||133|válido|
|scored_0175|E3|joint|136|individual 136|136|OPTIMAL|
|scored_0175|E4|repair|136||133|completo|
|scored_0176|E1|1|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0176|E1|3|153|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0176|E1|6|155|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0176|E2|52-1|152||151|válido|
|scored_0176|E2|52-3|153||144|válido|
|scored_0176|E2|52-6|155||144|válido|
|scored_0176|E2|64-1|152||151|válido|
|scored_0176|E2|64-3|153||144|válido|
|scored_0176|E2|64-6|155||144|válido|
|scored_0176|E3|joint|152|individual 151|150|OPTIMAL|
|scored_0176|E4|repair|152||152|completo|
|scored_0177|E1|1|189|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0177|E1|3|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0177|E1|6|187|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0177|E2|52-1|189||181|válido|
|scored_0177|E2|52-3|188||175|válido|
|scored_0177|E2|52-6|187||171|válido|
|scored_0177|E2|64-1|189||181|válido|
|scored_0177|E2|64-3|188||176|válido|
|scored_0177|E2|64-6|187||172|válido|
|scored_0177|E3|joint|187|individual 187|187|OPTIMAL|
|scored_0177|E4|repair|187||186|completo|
|scored_0178|E1|1|145|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0178|E1|3|146|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0178|E1|6|143|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0178|E2|52-1|145||138|válido|
|scored_0178|E2|52-3|146||137|válido|
|scored_0178|E2|52-6|143||136|válido|
|scored_0178|E2|64-1|145||136|válido|
|scored_0178|E2|64-3|146||136|válido|
|scored_0178|E2|64-6|143||136|válido|
|scored_0178|E3|joint|143|individual 143|143|OPTIMAL|
|scored_0178|E4|repair|143||143|completo|
|scored_0179|E1|1|294|postprocess 294, optimize_a4 294, nested_windows 294|294|válido|
|scored_0179|E1|3|292|postprocess 292, optimize_a4 292, nested_windows 292|292|válido|
|scored_0179|E1|6|280|postprocess 280, optimize_a4 280, nested_windows 280|280|válido|
|scored_0179|E2|52-1|294||270|válido|
|scored_0179|E2|52-3|292||270|válido|
|scored_0179|E2|52-6|280||263|válido|
|scored_0179|E2|64-1|294||276|válido|
|scored_0179|E2|64-3|292||270|válido|
|scored_0179|E2|64-6|280||262|válido|
|scored_0179|E3|joint|280|individual 280|278|OPTIMAL|
|scored_0179|E4|repair|280||280|completo|
|scored_0180|E1|1|114|postprocess 114, optimize_a4 113, nested_windows 113|113|válido|
|scored_0180|E1|3|114|postprocess 114, optimize_a4 113, nested_windows 113|113|válido|
|scored_0180|E1|6|114|postprocess 114, optimize_a4 113, nested_windows 113|113|válido|
|scored_0180|E2|52-1|114||113|válido|
|scored_0180|E2|52-3|114||113|válido|
|scored_0180|E2|52-6|114||113|válido|
|scored_0180|E2|64-1|114||114|válido|
|scored_0180|E2|64-3|114||114|válido|
|scored_0180|E2|64-6|114||114|válido|
|scored_0180|E3|joint|114|individual 114|114|OPTIMAL|
|scored_0180|E4|repair|114||112|completo|
|scored_0181|E1|1|40|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0181|E1|3|40|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0181|E1|6|40|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0181|E2|52-1|40||40|válido|
|scored_0181|E2|52-3|40||40|válido|
|scored_0181|E2|52-6|40||40|válido|
|scored_0181|E2|64-1|40||40|válido|
|scored_0181|E2|64-3|40||40|válido|
|scored_0181|E2|64-6|40||40|válido|
|scored_0181|E3|joint|40|individual 40|40|OPTIMAL|
|scored_0181|E4|repair|40||40|completo|
|scored_0182|E1|1|225|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0182|E1|3|218|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|scored_0182|E1|6|217|postprocess 217, optimize_a4 217, nested_windows 217|217|válido|
|scored_0182|E2|52-1|225||208|válido|
|scored_0182|E2|52-3|218||207|válido|
|scored_0182|E2|52-6|217||204|válido|
|scored_0182|E2|64-1|225||208|válido|
|scored_0182|E2|64-3|218||208|válido|
|scored_0182|E2|64-6|217||204|válido|
|scored_0182|E3|joint|217|individual 216|216|OPTIMAL|
|scored_0182|E4|repair|217||217|completo|
|scored_0183|E1|1|409|postprocess 409, optimize_a4 409, nested_windows 409|409|válido|
|scored_0183|E1|3|393|postprocess 393, optimize_a4 393, nested_windows 393|393|válido|
|scored_0183|E1|6|379|postprocess 378, optimize_a4 378, nested_windows 378|378|válido|
|scored_0183|E2|52-1|409||373|válido|
|scored_0183|E2|52-3|393||358|válido|
|scored_0183|E2|52-6|379||352|válido|
|scored_0183|E2|64-1|409||375|válido|
|scored_0183|E2|64-3|393||359|válido|
|scored_0183|E2|64-6|379||354|válido|
|scored_0183|E3|joint|379|individual 379|377|FEASIBLE|
|scored_0183|E4|repair|379||379|completo|
|scored_0184|E1|1|190|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|scored_0184|E1|3|185|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|scored_0184|E1|6|182|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0184|E2|52-1|190||181|válido|
|scored_0184|E2|52-3|185||173|válido|
|scored_0184|E2|52-6|182||175|válido|
|scored_0184|E2|64-1|190||182|válido|
|scored_0184|E2|64-3|185||175|válido|
|scored_0184|E2|64-6|182||178|válido|
|scored_0184|E3|joint|182|individual 182|180|OPTIMAL|
|scored_0184|E4|repair|182||182|completo|
|scored_0185|E1|1|246|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|scored_0185|E1|3|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|scored_0185|E1|6|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0185|E2|52-1|246||224|válido|
|scored_0185|E2|52-3|238||233|válido|
|scored_0185|E2|52-6|233||226|válido|
|scored_0185|E2|64-1|246||229|válido|
|scored_0185|E2|64-3|238||233|válido|
|scored_0185|E2|64-6|233||226|válido|
|scored_0185|E3|joint|233|individual 231|231|OPTIMAL|
|scored_0185|E4|repair|233||232|completo|
|scored_0186|E1|1|190|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|scored_0186|E1|3|187|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0186|E1|6|181|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0186|E2|52-1|190||179|válido|
|scored_0186|E2|52-3|187||174|válido|
|scored_0186|E2|52-6|181||168|válido|
|scored_0186|E2|64-1|190||182|válido|
|scored_0186|E2|64-3|187||175|válido|
|scored_0186|E2|64-6|181||170|válido|
|scored_0186|E3|joint|181|individual 181|181|OPTIMAL|
|scored_0186|E4|repair|181||179|completo|
|scored_0187|E1|1|123|postprocess 123, optimize_a4 123, nested_windows 123|123|válido|
|scored_0187|E1|3|123|postprocess 123, optimize_a4 123, nested_windows 123|123|válido|
|scored_0187|E1|6|124|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|scored_0187|E2|52-1|123||122|válido|
|scored_0187|E2|52-3|123||122|válido|
|scored_0187|E2|52-6|124||122|válido|
|scored_0187|E2|64-1|123||121|válido|
|scored_0187|E2|64-3|123||121|válido|
|scored_0187|E2|64-6|124||121|válido|
|scored_0187|E3|joint|123|individual 123|123|OPTIMAL|
|scored_0187|E4|repair|123||121|completo|
|scored_0188|E1|1|127|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0188|E1|3|127|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0188|E1|6|126|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|scored_0188|E2|52-1|127||124|válido|
|scored_0188|E2|52-3|127||124|válido|
|scored_0188|E2|52-6|126||124|válido|
|scored_0188|E2|64-1|127||125|válido|
|scored_0188|E2|64-3|127||125|válido|
|scored_0188|E2|64-6|126||125|válido|
|scored_0188|E3|joint|126|individual 126|126|OPTIMAL|
|scored_0188|E4|repair|126||124|completo|
|scored_0189|E1|1|225|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0189|E1|3|205|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0189|E1|6|205|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0189|E2|52-1|225||205|válido|
|scored_0189|E2|52-3|205||201|válido|
|scored_0189|E2|52-6|205||197|válido|
|scored_0189|E2|64-1|225||205|válido|
|scored_0189|E2|64-3|205||204|válido|
|scored_0189|E2|64-6|205||197|válido|
|scored_0189|E3|joint|205|individual 205|205|OPTIMAL|
|scored_0189|E4|repair|205||205|completo|
|scored_0190|E1|1|171|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|scored_0190|E1|3|170|postprocess 170, optimize_a4 170, nested_windows 170|170|válido|
|scored_0190|E1|6|166|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0190|E2|52-1|171||164|válido|
|scored_0190|E2|52-3|170||164|válido|
|scored_0190|E2|52-6|166||156|válido|
|scored_0190|E2|64-1|171||162|válido|
|scored_0190|E2|64-3|170||162|válido|
|scored_0190|E2|64-6|166||154|válido|
|scored_0190|E3|joint|166|individual 166|165|OPTIMAL|
|scored_0190|E4|repair|166||166|completo|
|scored_0191|E1|1|181|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|scored_0191|E1|3|171|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|scored_0191|E1|6|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0191|E2|52-1|181||176|válido|
|scored_0191|E2|52-3|171||165|válido|
|scored_0191|E2|52-6|169||161|válido|
|scored_0191|E2|64-1|181||178|válido|
|scored_0191|E2|64-3|171||165|válido|
|scored_0191|E2|64-6|169||160|válido|
|scored_0191|E3|joint|169|individual 169|167|OPTIMAL|
|scored_0191|E4|repair|169||168|completo|
|scored_0192|E1|1|205|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|scored_0192|E1|3|202|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0192|E1|6|200|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|scored_0192|E2|52-1|205||193|válido|
|scored_0192|E2|52-3|202||188|válido|
|scored_0192|E2|52-6|200||182|válido|
|scored_0192|E2|64-1|205||191|válido|
|scored_0192|E2|64-3|202||188|válido|
|scored_0192|E2|64-6|200||189|válido|
|scored_0192|E3|joint|200|individual 200|198|OPTIMAL|
|scored_0192|E4|repair|200||198|completo|
|scored_0193|E1|1|162|postprocess 161, optimize_a4 161, nested_windows 161|161|válido|
|scored_0193|E1|3|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0193|E1|6|158|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0193|E2|52-1|162||155|válido|
|scored_0193|E2|52-3|159||155|válido|
|scored_0193|E2|52-6|158||153|válido|
|scored_0193|E2|64-1|162||154|válido|
|scored_0193|E2|64-3|159||154|válido|
|scored_0193|E2|64-6|158||154|válido|
|scored_0193|E3|joint|158|individual 158|158|OPTIMAL|
|scored_0193|E4|repair|158||157|completo|
|scored_0194|E1|1|210|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0194|E1|3|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0194|E1|6|204|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|scored_0194|E2|52-1|210||196|válido|
|scored_0194|E2|52-3|206||189|válido|
|scored_0194|E2|52-6|204||190|válido|
|scored_0194|E2|64-1|210||194|válido|
|scored_0194|E2|64-3|206||188|válido|
|scored_0194|E2|64-6|204||190|válido|
|scored_0194|E3|joint|204|individual 203|202|OPTIMAL|
|scored_0194|E4|repair|204||204|completo|
|scored_0195|E1|1|172|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|scored_0195|E1|3|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0195|E1|6|167|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0195|E2|52-1|172||156|válido|
|scored_0195|E2|52-3|169||156|válido|
|scored_0195|E2|52-6|167||156|válido|
|scored_0195|E2|64-1|172||157|válido|
|scored_0195|E2|64-3|169||155|válido|
|scored_0195|E2|64-6|167||155|válido|
|scored_0195|E3|joint|167|individual 167|167|OPTIMAL|
|scored_0195|E4|repair|167||166|completo|
|scored_0196|E1|1|41|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0196|E1|3|41|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0196|E1|6|41|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0196|E2|52-1|41||41|válido|
|scored_0196|E2|52-3|41||41|válido|
|scored_0196|E2|52-6|41||41|válido|
|scored_0196|E2|64-1|41||41|válido|
|scored_0196|E2|64-3|41||41|válido|
|scored_0196|E2|64-6|41||41|válido|
|scored_0196|E3|joint|41|individual 41|41|OPTIMAL|
|scored_0196|E4|repair|41||40|completo|
|scored_0197|E1|1|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0197|E1|3|162|postprocess 161, optimize_a4 161, nested_windows 161|161|válido|
|scored_0197|E1|6|160|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0197|E2|52-1|165||158|válido|
|scored_0197|E2|52-3|162||158|válido|
|scored_0197|E2|52-6|160||155|válido|
|scored_0197|E2|64-1|165||157|válido|
|scored_0197|E2|64-3|162||157|válido|
|scored_0197|E2|64-6|160||155|válido|
|scored_0197|E3|joint|160|individual 160|160|OPTIMAL|
|scored_0197|E4|repair|160||159|completo|
|scored_0198|E1|1|305|postprocess 305, optimize_a4 305, nested_windows 305|305|válido|
|scored_0198|E1|3|292|postprocess 291, optimize_a4 291, nested_windows 291|291|válido|
|scored_0198|E1|6|290|postprocess 290, optimize_a4 290, nested_windows 290|290|válido|
|scored_0198|E2|52-1|305||281|válido|
|scored_0198|E2|52-3|292||277|válido|
|scored_0198|E2|52-6|290||271|válido|
|scored_0198|E2|64-1|305||283|válido|
|scored_0198|E2|64-3|292||276|válido|
|scored_0198|E2|64-6|290||269|válido|
|scored_0198|E3|joint|290|individual 288|287|OPTIMAL|
|scored_0198|E4|repair|290||290|completo|
|scored_0199|E1|1|125|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0199|E1|3|125|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0199|E1|6|124|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|scored_0199|E2|52-1|125||117|válido|
|scored_0199|E2|52-3|125||117|válido|
|scored_0199|E2|52-6|124||117|válido|
|scored_0199|E2|64-1|125||117|válido|
|scored_0199|E2|64-3|125||117|válido|
|scored_0199|E2|64-6|124||117|válido|
|scored_0199|E3|joint|124|individual 124|124|OPTIMAL|
|scored_0199|E4|repair|124||124|completo|
|scored_0200|E1|1|201|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0200|E1|3|194|postprocess 194, optimize_a4 194, nested_windows 194|194|válido|
|scored_0200|E1|6|185|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|scored_0200|E2|52-1|201||179|válido|
|scored_0200|E2|52-3|194||178|válido|
|scored_0200|E2|52-6|185||173|válido|
|scored_0200|E2|64-1|201||177|válido|
|scored_0200|E2|64-3|194||175|válido|
|scored_0200|E2|64-6|185||173|válido|
|scored_0200|E3|joint|185|individual 185|185|OPTIMAL|
|scored_0200|E4|repair|185||185|completo|
|scored_0201|E1|1|208|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|scored_0201|E1|3|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0201|E1|6|203|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0201|E2|52-1|208||198|válido|
|scored_0201|E2|52-3|207||197|válido|
|scored_0201|E2|52-6|203||195|válido|
|scored_0201|E2|64-1|208||200|válido|
|scored_0201|E2|64-3|207||195|válido|
|scored_0201|E2|64-6|203||194|válido|
|scored_0201|E3|joint|203|individual 203|200|OPTIMAL|
|scored_0201|E4|repair|203||203|completo|
|scored_0202|E1|1|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0202|E1|3|164|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0202|E1|6|161|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0202|E2|52-1|165||152|válido|
|scored_0202|E2|52-3|164||149|válido|
|scored_0202|E2|52-6|161||149|válido|
|scored_0202|E2|64-1|165||151|válido|
|scored_0202|E2|64-3|164||149|válido|
|scored_0202|E2|64-6|161||148|válido|
|scored_0202|E3|joint|161|individual 161|161|OPTIMAL|
|scored_0202|E4|repair|161||159|completo|
|scored_0203|E1|1|315|postprocess 315, optimize_a4 315, nested_windows 315|315|válido|
|scored_0203|E1|3|304|postprocess 304, optimize_a4 304, nested_windows 304|304|válido|
|scored_0203|E1|6|305|postprocess 305, optimize_a4 305, nested_windows 305|305|válido|
|scored_0203|E2|52-1|315||290|válido|
|scored_0203|E2|52-3|304||277|válido|
|scored_0203|E2|52-6|305||275|válido|
|scored_0203|E2|64-1|315||293|válido|
|scored_0203|E2|64-3|304||275|válido|
|scored_0203|E2|64-6|305||277|válido|
|scored_0203|E3|joint|304|individual 303|302|OPTIMAL|
|scored_0203|E4|repair|304||303|completo|
|scored_0204|E1|1|308|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0204|E1|3|304|postprocess 304, optimize_a4 304, nested_windows 304|304|válido|
|scored_0204|E1|6|290|postprocess 290, optimize_a4 290, nested_windows 290|290|válido|
|scored_0204|E2|52-1|308||283|válido|
|scored_0204|E2|52-3|304||276|válido|
|scored_0204|E2|52-6|290||273|válido|
|scored_0204|E2|64-1|308||289|válido|
|scored_0204|E2|64-3|304||276|válido|
|scored_0204|E2|64-6|290||271|válido|
|scored_0204|E3|joint|290|individual 289|288|OPTIMAL|
|scored_0204|E4|repair|290||290|completo|
|scored_0205|E1|1|181|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0205|E1|3|179|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|scored_0205|E1|6|178|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0205|E2|52-1|181||175|válido|
|scored_0205|E2|52-3|179||174|válido|
|scored_0205|E2|52-6|178||174|válido|
|scored_0205|E2|64-1|181||172|válido|
|scored_0205|E2|64-3|179||166|válido|
|scored_0205|E2|64-6|178||168|válido|
|scored_0205|E3|joint|178|individual 177|176|OPTIMAL|
|scored_0205|E4|repair|178||178|completo|
|scored_0206|E1|1|216|postprocess 216, optimize_a4 216, nested_windows 216|216|válido|
|scored_0206|E1|3|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0206|E1|6|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0206|E2|52-1|216||205|válido|
|scored_0206|E2|52-3|211||200|válido|
|scored_0206|E2|52-6|207||199|válido|
|scored_0206|E2|64-1|216||205|válido|
|scored_0206|E2|64-3|211||198|válido|
|scored_0206|E2|64-6|207||198|válido|
|scored_0206|E3|joint|207|individual 207|206|OPTIMAL|
|scored_0206|E4|repair|207||207|completo|
|scored_0207|E1|1|320|postprocess 319, optimize_a4 319, nested_windows 319|319|válido|
|scored_0207|E1|3|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0207|E1|6|301|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|scored_0207|E2|52-1|320||290|válido|
|scored_0207|E2|52-3|303||280|válido|
|scored_0207|E2|52-6|301||276|válido|
|scored_0207|E2|64-1|320||293|válido|
|scored_0207|E2|64-3|303||280|válido|
|scored_0207|E2|64-6|301||278|válido|
|scored_0207|E3|joint|301|individual 300|299|OPTIMAL|
|scored_0207|E4|repair|301||301|completo|
|scored_0208|E1|1|326|postprocess 326, optimize_a4 326, nested_windows 326|326|válido|
|scored_0208|E1|3|312|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|scored_0208|E1|6|299|postprocess 299, optimize_a4 299, nested_windows 299|299|válido|
|scored_0208|E2|52-1|326||295|válido|
|scored_0208|E2|52-3|312||288|válido|
|scored_0208|E2|52-6|299||285|válido|
|scored_0208|E2|64-1|326||300|válido|
|scored_0208|E2|64-3|312||289|válido|
|scored_0208|E2|64-6|299||286|válido|
|scored_0208|E3|joint|299|individual 297|297|OPTIMAL|
|scored_0208|E4|repair|299||299|completo|
|scored_0209|E1|1|312|postprocess 312, optimize_a4 312, nested_windows 312|312|válido|
|scored_0209|E1|3|300|postprocess 300, optimize_a4 300, nested_windows 300|300|válido|
|scored_0209|E1|6|297|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0209|E2|52-1|312||288|válido|
|scored_0209|E2|52-3|300||281|válido|
|scored_0209|E2|52-6|297||281|válido|
|scored_0209|E2|64-1|312||290|válido|
|scored_0209|E2|64-3|300||332|válido|
|scored_0209|E2|64-6|297||285|válido|
|scored_0209|E3|joint|297|individual 297|296|OPTIMAL|
|scored_0209|E4|repair|297||297|completo|
|scored_0210|E1|1|123|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0210|E1|3|123|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0210|E1|6|123|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0210|E2|52-1|123||123|válido|
|scored_0210|E2|52-3|123||123|válido|
|scored_0210|E2|52-6|123||123|válido|
|scored_0210|E2|64-1|123||122|válido|
|scored_0210|E2|64-3|123||122|válido|
|scored_0210|E2|64-6|123||122|válido|
|scored_0210|E3|joint|123|individual 123|123|OPTIMAL|
|scored_0210|E4|repair|123||121|completo|
|scored_0211|E1|1|54|postprocess 52, optimize_a4 52, nested_windows 52|52|válido|
|scored_0211|E1|3|54|postprocess 52, optimize_a4 52, nested_windows 52|52|válido|
|scored_0211|E1|6|54|postprocess 52, optimize_a4 52, nested_windows 52|52|válido|
|scored_0211|E2|52-1|54||54|válido|
|scored_0211|E2|52-3|54||54|válido|
|scored_0211|E2|52-6|54||54|válido|
|scored_0211|E2|64-1|54||54|válido|
|scored_0211|E2|64-3|54||54|válido|
|scored_0211|E2|64-6|54||54|válido|
|scored_0211|E3|joint|54|individual 54|54|OPTIMAL|
|scored_0211|E4|repair|54||53|completo|
|scored_0212|E1|1|30|postprocess 28, optimize_a4 28, nested_windows 28|28|válido|
|scored_0212|E1|3|30|postprocess 28, optimize_a4 28, nested_windows 28|28|válido|
|scored_0212|E1|6|30|postprocess 28, optimize_a4 28, nested_windows 28|28|válido|
|scored_0212|E2|52-1|30||30|válido|
|scored_0212|E2|52-3|30||30|válido|
|scored_0212|E2|52-6|30||30|válido|
|scored_0212|E2|64-1|30||30|válido|
|scored_0212|E2|64-3|30||30|válido|
|scored_0212|E2|64-6|30||30|válido|
|scored_0212|E3|joint|26|individual 26|26|OPTIMAL|
|scored_0212|E4|repair|26||25|completo|
|scored_0213|E1|1|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0213|E1|3|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0213|E1|6|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0213|E2|52-1|152||149|válido|
|scored_0213|E2|52-3|151||148|válido|
|scored_0213|E2|52-6|151||148|válido|
|scored_0213|E2|64-1|152||150|válido|
|scored_0213|E2|64-3|151||149|válido|
|scored_0213|E2|64-6|151||149|válido|
|scored_0213|E3|joint|151|individual 151|151|OPTIMAL|
|scored_0213|E4|repair|151||146|completo|
|scored_0214|E1|1|243|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0214|E1|3|237|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|scored_0214|E1|6|236|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0214|E2|52-1|243||227|válido|
|scored_0214|E2|52-3|237||213|válido|
|scored_0214|E2|52-6|236||215|válido|
|scored_0214|E2|64-1|243||228|válido|
|scored_0214|E2|64-3|237||216|válido|
|scored_0214|E2|64-6|236||215|válido|
|scored_0214|E3|joint|236|individual 236|233|OPTIMAL|
|scored_0214|E4|repair|236||236|completo|
|scored_0215|E1|1|225|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0215|E1|3|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0215|E1|6|215|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0215|E2|52-1|225||207|válido|
|scored_0215|E2|52-3|215||208|válido|
|scored_0215|E2|52-6|215||203|válido|
|scored_0215|E2|64-1|225||207|válido|
|scored_0215|E2|64-3|215||209|válido|
|scored_0215|E2|64-6|215||203|válido|
|scored_0215|E3|joint|215|individual 215|214|OPTIMAL|
|scored_0215|E4|repair|215||215|completo|
|scored_0216|E1|1|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0216|E1|3|205|postprocess 205, optimize_a4 205, nested_windows 205|205|válido|
|scored_0216|E1|6|202|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0216|E2|52-1|211||202|válido|
|scored_0216|E2|52-3|205||191|válido|
|scored_0216|E2|52-6|202||190|válido|
|scored_0216|E2|64-1|211||204|válido|
|scored_0216|E2|64-3|205||192|válido|
|scored_0216|E2|64-6|202||189|válido|
|scored_0216|E3|joint|202|individual 202|201|OPTIMAL|
|scored_0216|E4|repair|202||202|completo|
|scored_0217|E1|1|364|postprocess 364, optimize_a4 364, nested_windows 364|364|válido|
|scored_0217|E1|3|324|postprocess 323, optimize_a4 323, nested_windows 323|323|válido|
|scored_0217|E1|6|315|postprocess 315, optimize_a4 315, nested_windows 315|315|válido|
|scored_0217|E2|52-1|364||318|válido|
|scored_0217|E2|52-3|324||303|válido|
|scored_0217|E2|52-6|315||301|válido|
|scored_0217|E2|64-1|364||320|válido|
|scored_0217|E2|64-3|324||303|válido|
|scored_0217|E2|64-6|315||306|válido|
|scored_0217|E3|joint|315|individual 315|314|OPTIMAL|
|scored_0217|E4|repair|315||315|completo|
|scored_0218|E1|1|412|postprocess 411, optimize_a4 411, nested_windows 411|411|válido|
|scored_0218|E1|3|392|postprocess 392, optimize_a4 392, nested_windows 392|392|válido|
|scored_0218|E1|6|380|postprocess 380, optimize_a4 380, nested_windows 380|380|válido|
|scored_0218|E2|52-1|412||371|válido|
|scored_0218|E2|52-3|392||353|válido|
|scored_0218|E2|52-6|380||348|válido|
|scored_0218|E2|64-1|412||374|válido|
|scored_0218|E2|64-3|392||361|válido|
|scored_0218|E2|64-6|380||352|válido|
|scored_0218|E3|joint|380|individual 380|379|FEASIBLE|
|scored_0218|E4|repair|380||380|completo|
|scored_0219|E1|1|167|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0219|E1|3|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0219|E1|6|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0219|E2|52-1|167||156|válido|
|scored_0219|E2|52-3|165||154|válido|
|scored_0219|E2|52-6|162||154|válido|
|scored_0219|E2|64-1|167||157|válido|
|scored_0219|E2|64-3|165||151|válido|
|scored_0219|E2|64-6|162||151|válido|
|scored_0219|E3|joint|162|individual 162|162|OPTIMAL|
|scored_0219|E4|repair|162||158|completo|
|scored_0220|E1|1|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0220|E1|3|143|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0220|E1|6|143|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0220|E2|52-1|145||141|válido|
|scored_0220|E2|52-3|143||141|válido|
|scored_0220|E2|52-6|143||141|válido|
|scored_0220|E2|64-1|145||141|válido|
|scored_0220|E2|64-3|143||141|válido|
|scored_0220|E2|64-6|143||140|válido|
|scored_0220|E3|joint|143|individual 143|143|OPTIMAL|
|scored_0220|E4|repair|143||142|completo|
|scored_0221|E1|1|51|postprocess 51, optimize_a4 51, nested_windows 51|51|válido|
|scored_0221|E1|3|51|postprocess 51, optimize_a4 51, nested_windows 51|51|válido|
|scored_0221|E1|6|51|postprocess 51, optimize_a4 51, nested_windows 51|51|válido|
|scored_0221|E2|52-1|51||51|válido|
|scored_0221|E2|52-3|51||51|válido|
|scored_0221|E2|52-6|51||51|válido|
|scored_0221|E2|64-1|51||51|válido|
|scored_0221|E2|64-3|51||51|válido|
|scored_0221|E2|64-6|51||51|válido|
|scored_0221|E3|joint|51|individual 51|51|OPTIMAL|
|scored_0221|E4|repair|51||51|completo|
|scored_0222|E1|1|51|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0222|E1|3|51|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0222|E1|6|51|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0222|E2|52-1|51||51|válido|
|scored_0222|E2|52-3|51||51|válido|
|scored_0222|E2|52-6|51||51|válido|
|scored_0222|E2|64-1|51||51|válido|
|scored_0222|E2|64-3|51||51|válido|
|scored_0222|E2|64-6|51||51|válido|
|scored_0222|E3|joint|51|individual 51|51|OPTIMAL|
|scored_0222|E4|repair|51||49|completo|
|scored_0223|E1|1|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0223|E1|3|290|postprocess 290, optimize_a4 290, nested_windows 290|290|válido|
|scored_0223|E1|6|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|scored_0223|E2|52-1|303||272|válido|
|scored_0223|E2|52-3|290||276|válido|
|scored_0223|E2|52-6|289||263|válido|
|scored_0223|E2|64-1|303||279|válido|
|scored_0223|E2|64-3|290||277|válido|
|scored_0223|E2|64-6|289||269|válido|
|scored_0223|E3|joint|289|individual 289|284|OPTIMAL|
|scored_0223|E4|repair|289||289|completo|
|scored_0224|E1|1|252|postprocess 252, optimize_a4 252, nested_windows 252|252|válido|
|scored_0224|E1|3|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0224|E1|6|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0224|E2|52-1|252||232|válido|
|scored_0224|E2|52-3|242||226|válido|
|scored_0224|E2|52-6|239||225|válido|
|scored_0224|E2|64-1|252||232|válido|
|scored_0224|E2|64-3|242||227|válido|
|scored_0224|E2|64-6|239||226|válido|
|scored_0224|E3|joint|239|individual 239|239|OPTIMAL|
|scored_0224|E4|repair|239||239|completo|
|scored_0225|E1|1|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0225|E1|3|138|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0225|E1|6|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0225|E2|52-1|143||132|válido|
|scored_0225|E2|52-3|138||132|válido|
|scored_0225|E2|52-6|137||132|válido|
|scored_0225|E2|64-1|143||132|válido|
|scored_0225|E2|64-3|138||132|válido|
|scored_0225|E2|64-6|137||132|válido|
|scored_0225|E3|joint|137|individual 137|137|OPTIMAL|
|scored_0225|E4|repair|137||133|completo|
|scored_0226|E1|1|167|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0226|E1|3|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|scored_0226|E1|6|164|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0226|E2|52-1|167||159|válido|
|scored_0226|E2|52-3|166||154|válido|
|scored_0226|E2|52-6|164||154|válido|
|scored_0226|E2|64-1|167||161|válido|
|scored_0226|E2|64-3|166||154|válido|
|scored_0226|E2|64-6|164||154|válido|
|scored_0226|E3|joint|164|individual 162|161|OPTIMAL|
|scored_0226|E4|repair|164||163|completo|
|scored_0227|E1|1|212|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|scored_0227|E1|3|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0227|E1|6|205|postprocess 205, optimize_a4 205, nested_windows 205|205|válido|
|scored_0227|E2|52-1|212||196|válido|
|scored_0227|E2|52-3|206||196|válido|
|scored_0227|E2|52-6|205||189|válido|
|scored_0227|E2|64-1|212||203|válido|
|scored_0227|E2|64-3|206||196|válido|
|scored_0227|E2|64-6|205||196|válido|
|scored_0227|E3|joint|205|individual 205|205|OPTIMAL|
|scored_0227|E4|repair|205||203|completo|
|scored_0228|E1|1|167|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0228|E1|3|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0228|E1|6|164|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0228|E2|52-1|167||161|válido|
|scored_0228|E2|52-3|165||150|válido|
|scored_0228|E2|52-6|164||146|válido|
|scored_0228|E2|64-1|167||161|válido|
|scored_0228|E2|64-3|165||152|válido|
|scored_0228|E2|64-6|164||151|válido|
|scored_0228|E3|joint|164|individual 164|164|OPTIMAL|
|scored_0228|E4|repair|164||161|completo|
|scored_0229|E1|1|193|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0229|E1|3|189|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0229|E1|6|184|postprocess 184, optimize_a4 184, nested_windows 184|184|válido|
|scored_0229|E2|52-1|193||182|válido|
|scored_0229|E2|52-3|189||182|válido|
|scored_0229|E2|52-6|184||182|válido|
|scored_0229|E2|64-1|193||183|válido|
|scored_0229|E2|64-3|189||180|válido|
|scored_0229|E2|64-6|184||180|válido|
|scored_0229|E3|joint|184|individual 184|184|OPTIMAL|
|scored_0229|E4|repair|184||184|completo|
|scored_0230|E1|1|299|postprocess 299, optimize_a4 299, nested_windows 299|299|válido|
|scored_0230|E1|3|292|postprocess 292, optimize_a4 292, nested_windows 292|292|válido|
|scored_0230|E1|6|282|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|scored_0230|E2|52-1|299||277|válido|
|scored_0230|E2|52-3|292||275|válido|
|scored_0230|E2|52-6|282||266|válido|
|scored_0230|E2|64-1|299||276|válido|
|scored_0230|E2|64-3|292||272|válido|
|scored_0230|E2|64-6|282||269|válido|
|scored_0230|E3|joint|282|individual 282|281|FEASIBLE|
|scored_0230|E4|repair|282||282|completo|
|scored_0231|E1|1|248|postprocess 248, optimize_a4 248, nested_windows 248|248|válido|
|scored_0231|E1|3|236|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0231|E1|6|231|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|scored_0231|E2|52-1|248||234|válido|
|scored_0231|E2|52-3|236||226|válido|
|scored_0231|E2|52-6|231||221|válido|
|scored_0231|E2|64-1|248||234|válido|
|scored_0231|E2|64-3|236||228|válido|
|scored_0231|E2|64-6|231||226|válido|
|scored_0231|E3|joint|231|individual 231|229|OPTIMAL|
|scored_0231|E4|repair|231||231|completo|
|scored_0232|E1|1|224|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0232|E1|3|218|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|scored_0232|E1|6|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0232|E2|52-1|224||212|válido|
|scored_0232|E2|52-3|218||207|válido|
|scored_0232|E2|52-6|219||203|válido|
|scored_0232|E2|64-1|224||210|válido|
|scored_0232|E2|64-3|218||208|válido|
|scored_0232|E2|64-6|219||203|válido|
|scored_0232|E3|joint|218|individual 218|216|OPTIMAL|
|scored_0232|E4|repair|218||218|completo|
|scored_0233|E1|1|52|postprocess 49, optimize_a4 49, nested_windows 49|49|válido|
|scored_0233|E1|3|52|postprocess 49, optimize_a4 49, nested_windows 49|49|válido|
|scored_0233|E1|6|52|postprocess 49, optimize_a4 49, nested_windows 49|49|válido|
|scored_0233|E2|52-1|52||52|válido|
|scored_0233|E2|52-3|52||52|válido|
|scored_0233|E2|52-6|52||52|válido|
|scored_0233|E2|64-1|52||52|válido|
|scored_0233|E2|64-3|52||52|válido|
|scored_0233|E2|64-6|52||52|válido|
|scored_0233|E3|joint|52|individual 52|52|OPTIMAL|
|scored_0233|E4|repair|52||52|completo|
|scored_0234|E1|1|247|postprocess 247, optimize_a4 247, nested_windows 247|247|válido|
|scored_0234|E1|3|245|postprocess 245, optimize_a4 245, nested_windows 245|245|válido|
|scored_0234|E1|6|244|postprocess 244, optimize_a4 244, nested_windows 244|244|válido|
|scored_0234|E2|52-1|247||235|válido|
|scored_0234|E2|52-3|245||230|válido|
|scored_0234|E2|52-6|244||227|válido|
|scored_0234|E2|64-1|247||236|válido|
|scored_0234|E2|64-3|245||230|válido|
|scored_0234|E2|64-6|244||229|válido|
|scored_0234|E3|joint|244|individual 244|244|OPTIMAL|
|scored_0234|E4|repair|244||244|completo|
|scored_0235|E1|1|297|postprocess 297, optimize_a4 297, nested_windows 297|297|válido|
|scored_0235|E1|3|287|postprocess 286, optimize_a4 286, nested_windows 286|286|válido|
|scored_0235|E1|6|282|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|scored_0235|E2|52-1|297||273|válido|
|scored_0235|E2|52-3|287||266|válido|
|scored_0235|E2|52-6|282||264|válido|
|scored_0235|E2|64-1|297||272|válido|
|scored_0235|E2|64-3|287||267|válido|
|scored_0235|E2|64-6|282||264|válido|
|scored_0235|E3|joint|282|individual 281|280|OPTIMAL|
|scored_0235|E4|repair|282||282|completo|
|scored_0236|E1|1|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0236|E1|3|144|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0236|E1|6|144|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0236|E2|52-1|145||137|válido|
|scored_0236|E2|52-3|144||137|válido|
|scored_0236|E2|52-6|144||137|válido|
|scored_0236|E2|64-1|145||136|válido|
|scored_0236|E2|64-3|144||136|válido|
|scored_0236|E2|64-6|144||136|válido|
|scored_0236|E3|joint|144|individual 144|144|OPTIMAL|
|scored_0236|E4|repair|144||144|completo|
|scored_0237|E1|1|46|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0237|E1|3|46|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0237|E1|6|46|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0237|E2|52-1|46||46|válido|
|scored_0237|E2|52-3|46||46|válido|
|scored_0237|E2|52-6|46||46|válido|
|scored_0237|E2|64-1|46||46|válido|
|scored_0237|E2|64-3|46||46|válido|
|scored_0237|E2|64-6|46||46|válido|
|scored_0237|E3|joint|46|individual 46|46|OPTIMAL|
|scored_0237|E4|repair|46||45|completo|
|scored_0238|E1|1|195|postprocess 195, optimize_a4 195, nested_windows 195|195|válido|
|scored_0238|E1|3|195|postprocess 195, optimize_a4 195, nested_windows 195|195|válido|
|scored_0238|E1|6|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0238|E2|52-1|195||184|válido|
|scored_0238|E2|52-3|195||183|válido|
|scored_0238|E2|52-6|192||183|válido|
|scored_0238|E2|64-1|195||184|válido|
|scored_0238|E2|64-3|195||183|válido|
|scored_0238|E2|64-6|192||183|válido|
|scored_0238|E3|joint|192|individual 191|191|OPTIMAL|
|scored_0238|E4|repair|192||192|completo|
|scored_0239|E1|1|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|scored_0239|E1|3|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0239|E1|6|228|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|scored_0239|E2|52-1|238||222|válido|
|scored_0239|E2|52-3|235||219|válido|
|scored_0239|E2|52-6|228||212|válido|
|scored_0239|E2|64-1|238||222|válido|
|scored_0239|E2|64-3|235||220|válido|
|scored_0239|E2|64-6|228||212|válido|
|scored_0239|E3|joint|228|individual 228|228|OPTIMAL|
|scored_0239|E4|repair|228||228|completo|
|scored_0240|E1|1|160|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0240|E1|3|158|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0240|E1|6|155|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|scored_0240|E2|52-1|160||156|válido|
|scored_0240|E2|52-3|158||154|válido|
|scored_0240|E2|52-6|155||154|válido|
|scored_0240|E2|64-1|160||153|válido|
|scored_0240|E2|64-3|158||153|válido|
|scored_0240|E2|64-6|155||153|válido|
|scored_0240|E3|joint|155|individual 155|155|OPTIMAL|
|scored_0240|E4|repair|155||155|completo|
|scored_0241|E1|1|43|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0241|E1|3|43|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0241|E1|6|43|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0241|E2|52-1|43||43|válido|
|scored_0241|E2|52-3|43||43|válido|
|scored_0241|E2|52-6|43||43|válido|
|scored_0241|E2|64-1|43||43|válido|
|scored_0241|E2|64-3|43||43|válido|
|scored_0241|E2|64-6|43||43|válido|
|scored_0241|E3|joint|43|individual 43|43|OPTIMAL|
|scored_0241|E4|repair|43||40|completo|
|scored_0242|E1|1|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0242|E1|3|213|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0242|E1|6|214|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0242|E2|52-1|220||207|válido|
|scored_0242|E2|52-3|213||205|válido|
|scored_0242|E2|52-6|214||201|válido|
|scored_0242|E2|64-1|220||210|válido|
|scored_0242|E2|64-3|213||203|válido|
|scored_0242|E2|64-6|214||202|válido|
|scored_0242|E3|joint|213|individual 213|212|OPTIMAL|
|scored_0242|E4|repair|213||213|completo|
|scored_0243|E1|1|162|postprocess 161, optimize_a4 161, nested_windows 161|161|válido|
|scored_0243|E1|3|159|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0243|E1|6|161|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0243|E2|52-1|162||155|válido|
|scored_0243|E2|52-3|159||155|válido|
|scored_0243|E2|52-6|161||153|válido|
|scored_0243|E2|64-1|162||152|válido|
|scored_0243|E2|64-3|159||151|válido|
|scored_0243|E2|64-6|161||151|válido|
|scored_0243|E3|joint|159|individual 159|159|OPTIMAL|
|scored_0243|E4|repair|159||158|completo|
|scored_0244|E1|1|276|postprocess 276, optimize_a4 276, nested_windows 276|276|válido|
|scored_0244|E1|3|259|postprocess 259, optimize_a4 259, nested_windows 259|259|válido|
|scored_0244|E1|6|254|postprocess 254, optimize_a4 254, nested_windows 254|254|válido|
|scored_0244|E2|52-1|276||247|válido|
|scored_0244|E2|52-3|259||237|válido|
|scored_0244|E2|52-6|254||233|válido|
|scored_0244|E2|64-1|276||247|válido|
|scored_0244|E2|64-3|259||237|válido|
|scored_0244|E2|64-6|254||234|válido|
|scored_0244|E3|joint|254|individual 253|253|OPTIMAL|
|scored_0244|E4|repair|254||253|completo|
|scored_0245|E1|1|153|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0245|E1|3|150|postprocess 150, optimize_a4 150, nested_windows 150|150|válido|
|scored_0245|E1|6|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0245|E2|52-1|153||146|válido|
|scored_0245|E2|52-3|150||145|válido|
|scored_0245|E2|52-6|148||143|válido|
|scored_0245|E2|64-1|153||145|válido|
|scored_0245|E2|64-3|150||143|válido|
|scored_0245|E2|64-6|148||143|válido|
|scored_0245|E3|joint|148|individual 148|148|OPTIMAL|
|scored_0245|E4|repair|148||148|completo|
|scored_0246|E1|1|177|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0246|E1|3|177|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|scored_0246|E1|6|177|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|scored_0246|E2|52-1|177||171|válido|
|scored_0246|E2|52-3|177||167|válido|
|scored_0246|E2|52-6|177||167|válido|
|scored_0246|E2|64-1|177||169|válido|
|scored_0246|E2|64-3|177||169|válido|
|scored_0246|E2|64-6|177||168|válido|
|scored_0246|E3|joint|177|individual 177|177|OPTIMAL|
|scored_0246|E4|repair|177||177|completo|
|scored_0247|E1|1|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|scored_0247|E1|3|195|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0247|E1|6|187|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0247|E2|52-1|197||179|válido|
|scored_0247|E2|52-3|195||178|válido|
|scored_0247|E2|52-6|187||177|válido|
|scored_0247|E2|64-1|197||179|válido|
|scored_0247|E2|64-3|195||179|válido|
|scored_0247|E2|64-6|187||176|válido|
|scored_0247|E3|joint|187|individual 186|186|OPTIMAL|
|scored_0247|E4|repair|187||187|completo|
|scored_0248|E1|1|242|postprocess 240, optimize_a4 240, nested_windows 240|240|válido|
|scored_0248|E1|3|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0248|E1|6|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0248|E2|52-1|242||226|válido|
|scored_0248|E2|52-3|235||221|válido|
|scored_0248|E2|52-6|233||217|válido|
|scored_0248|E2|64-1|242||227|válido|
|scored_0248|E2|64-3|235||220|válido|
|scored_0248|E2|64-6|233||220|válido|
|scored_0248|E3|joint|233|individual 233|231|OPTIMAL|
|scored_0248|E4|repair|233||233|completo|
|scored_0249|E1|1|373|postprocess 373, optimize_a4 373, nested_windows 373|373|válido|
|scored_0249|E1|3|350|postprocess 350, optimize_a4 350, nested_windows 350|350|válido|
|scored_0249|E1|6|347|postprocess 347, optimize_a4 347, nested_windows 347|347|válido|
|scored_0249|E2|52-1|373||336|válido|
|scored_0249|E2|52-3|350||330|válido|
|scored_0249|E2|52-6|347||323|válido|
|scored_0249|E2|64-1|373||343|válido|
|scored_0249|E2|64-3|350||330|válido|
|scored_0249|E2|64-6|347||325|válido|
|scored_0249|E3|joint|347|individual 347|344|OPTIMAL|
|scored_0249|E4|repair|347||347|completo|
|scored_0250|E1|1|34|postprocess 34, optimize_a4 34, nested_windows 34|34|válido|
|scored_0250|E1|3|34|postprocess 34, optimize_a4 34, nested_windows 34|34|válido|
|scored_0250|E1|6|34|postprocess 34, optimize_a4 34, nested_windows 34|34|válido|
|scored_0250|E2|52-1|34||34|válido|
|scored_0250|E2|52-3|34||34|válido|
|scored_0250|E2|52-6|34||34|válido|
|scored_0250|E2|64-1|34||34|válido|
|scored_0250|E2|64-3|34||34|válido|
|scored_0250|E2|64-6|34||34|válido|
|scored_0250|E3|joint|34|individual 34|34|OPTIMAL|
|scored_0250|E4|repair|34||31|completo|
|scored_0251|E1|1|319|postprocess 319, optimize_a4 319, nested_windows 319|319|válido|
|scored_0251|E1|3|301|postprocess 300, optimize_a4 300, nested_windows 300|300|válido|
|scored_0251|E1|6|297|postprocess 297, optimize_a4 297, nested_windows 297|297|válido|
|scored_0251|E2|52-1|319||286|válido|
|scored_0251|E2|52-3|301||274|válido|
|scored_0251|E2|52-6|297||272|válido|
|scored_0251|E2|64-1|319||286|válido|
|scored_0251|E2|64-3|301||276|válido|
|scored_0251|E2|64-6|297||267|válido|
|scored_0251|E3|joint|297|individual 297|296|OPTIMAL|
|scored_0251|E4|repair|297||297|completo|
|scored_0252|E1|1|90|postprocess 90, optimize_a4 90, nested_windows 90|90|válido|
|scored_0252|E1|3|90|postprocess 90, optimize_a4 90, nested_windows 90|90|válido|
|scored_0252|E1|6|90|postprocess 90, optimize_a4 90, nested_windows 90|90|válido|
|scored_0252|E2|52-1|90||89|válido|
|scored_0252|E2|52-3|90||89|válido|
|scored_0252|E2|52-6|90||89|válido|
|scored_0252|E2|64-1|90||89|válido|
|scored_0252|E2|64-3|90||89|válido|
|scored_0252|E2|64-6|90||89|válido|
|scored_0252|E3|joint|90|individual 90|90|OPTIMAL|
|scored_0252|E4|repair|90||90|completo|
|scored_0253|E1|1|106|postprocess 105, optimize_a4 105, nested_windows 105|105|válido|
|scored_0253|E1|3|106|postprocess 105, optimize_a4 105, nested_windows 105|105|válido|
|scored_0253|E1|6|106|postprocess 105, optimize_a4 105, nested_windows 105|105|válido|
|scored_0253|E2|52-1|106||102|válido|
|scored_0253|E2|52-3|106||102|válido|
|scored_0253|E2|52-6|106||102|válido|
|scored_0253|E2|64-1|106||102|válido|
|scored_0253|E2|64-3|106||102|válido|
|scored_0253|E2|64-6|106||102|válido|
|scored_0253|E3|joint|106|individual 106|106|OPTIMAL|
|scored_0253|E4|repair|106||106|completo|
|scored_0254|E1|1|340|postprocess 340, optimize_a4 340, nested_windows 340|340|válido|
|scored_0254|E1|3|318|postprocess 318, optimize_a4 318, nested_windows 318|318|válido|
|scored_0254|E1|6|317|postprocess 317, optimize_a4 317, nested_windows 317|317|válido|
|scored_0254|E2|52-1|340||300|válido|
|scored_0254|E2|52-3|318||290|válido|
|scored_0254|E2|52-6|317||286|válido|
|scored_0254|E2|64-1|340||306|válido|
|scored_0254|E2|64-3|318||295|válido|
|scored_0254|E2|64-6|317||288|válido|
|scored_0254|E3|joint|317|individual 316|306|OPTIMAL|
|scored_0254|E4|repair|317||317|completo|
|scored_0255|E1|1|387|postprocess 387, optimize_a4 387, nested_windows 387|387|válido|
|scored_0255|E1|3|380|postprocess 379, optimize_a4 379, nested_windows 379|379|válido|
|scored_0255|E1|6|363|postprocess 363, optimize_a4 363, nested_windows 363|363|válido|
|scored_0255|E2|52-1|387||368|válido|
|scored_0255|E2|52-3|380||349|válido|
|scored_0255|E2|52-6|363||346|válido|
|scored_0255|E2|64-1|387||367|válido|
|scored_0255|E2|64-3|380||352|válido|
|scored_0255|E2|64-6|363||447|válido|
|scored_0255|E3|joint|363|individual 362|361|FEASIBLE|
|scored_0255|E4|repair|363||363|completo|
|scored_0256|E1|1|118|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0256|E1|3|118|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0256|E1|6|118|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0256|E2|52-1|118||113|válido|
|scored_0256|E2|52-3|118||113|válido|
|scored_0256|E2|52-6|118||113|válido|
|scored_0256|E2|64-1|118||113|válido|
|scored_0256|E2|64-3|118||113|válido|
|scored_0256|E2|64-6|118||113|válido|
|scored_0256|E3|joint|118|individual 118|118|OPTIMAL|
|scored_0256|E4|repair|118||112|completo|
|scored_0257|E1|1|270|postprocess 270, optimize_a4 270, nested_windows 270|270|válido|
|scored_0257|E1|3|260|postprocess 259, optimize_a4 259, nested_windows 259|259|válido|
|scored_0257|E1|6|258|postprocess 258, optimize_a4 258, nested_windows 258|258|válido|
|scored_0257|E2|52-1|270||252|válido|
|scored_0257|E2|52-3|260||241|válido|
|scored_0257|E2|52-6|258||242|válido|
|scored_0257|E2|64-1|270||253|válido|
|scored_0257|E2|64-3|260||241|válido|
|scored_0257|E2|64-6|258||243|válido|
|scored_0257|E3|joint|258|individual 258|255|OPTIMAL|
|scored_0257|E4|repair|258||258|completo|
|scored_0258|E1|1|402|postprocess 402, optimize_a4 402, nested_windows 402|402|válido|
|scored_0258|E1|3|371|postprocess 369, optimize_a4 369, nested_windows 369|369|válido|
|scored_0258|E1|6|366|postprocess 366, optimize_a4 366, nested_windows 366|366|válido|
|scored_0258|E2|52-1|402||350|válido|
|scored_0258|E2|52-3|371||348|válido|
|scored_0258|E2|52-6|366||342|válido|
|scored_0258|E2|64-1|402||356|válido|
|scored_0258|E2|64-3|371||348|válido|
|scored_0258|E2|64-6|366||346|válido|
|scored_0258|E3|joint|366|individual 364|361|OPTIMAL|
|scored_0258|E4|repair|366||366|completo|
|scored_0259|E1|1|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0259|E1|3|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0259|E1|6|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0259|E2|52-1|145||136|válido|
|scored_0259|E2|52-3|146||136|válido|
|scored_0259|E2|52-6|143||136|válido|
|scored_0259|E2|64-1|145||137|válido|
|scored_0259|E2|64-3|146||137|válido|
|scored_0259|E2|64-6|143||137|válido|
|scored_0259|E3|joint|143|individual 141|141|OPTIMAL|
|scored_0259|E4|repair|143||137|completo|
|scored_0260|E1|1|122|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0260|E1|3|122|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0260|E1|6|122|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0260|E2|52-1|122||119|válido|
|scored_0260|E2|52-3|122||119|válido|
|scored_0260|E2|52-6|122||119|válido|
|scored_0260|E2|64-1|122||121|válido|
|scored_0260|E2|64-3|122||121|válido|
|scored_0260|E2|64-6|122||121|válido|
|scored_0260|E3|joint|122|individual 122|122|OPTIMAL|
|scored_0260|E4|repair|122||120|completo|
|scored_0261|E1|1|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0261|E1|3|228|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0261|E1|6|225|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0261|E2|52-1|233||217|válido|
|scored_0261|E2|52-3|228||213|válido|
|scored_0261|E2|52-6|225||214|válido|
|scored_0261|E2|64-1|233||214|válido|
|scored_0261|E2|64-3|228||216|válido|
|scored_0261|E2|64-6|225||215|válido|
|scored_0261|E3|joint|225|individual 225|225|OPTIMAL|
|scored_0261|E4|repair|225||224|completo|
|scored_0262|E1|1|307|postprocess 307, optimize_a4 307, nested_windows 307|307|válido|
|scored_0262|E1|3|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0262|E1|6|295|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|scored_0262|E2|52-1|307||290|válido|
|scored_0262|E2|52-3|303||279|válido|
|scored_0262|E2|52-6|295||277|válido|
|scored_0262|E2|64-1|307||292|válido|
|scored_0262|E2|64-3|303||277|válido|
|scored_0262|E2|64-6|295||279|válido|
|scored_0262|E3|joint|295|individual 295|290|OPTIMAL|
|scored_0262|E4|repair|295||295|completo|
|scored_0263|E1|1|336|postprocess 336, optimize_a4 336, nested_windows 336|336|válido|
|scored_0263|E1|3|323|postprocess 323, optimize_a4 323, nested_windows 323|323|válido|
|scored_0263|E1|6|319|postprocess 319, optimize_a4 319, nested_windows 319|319|válido|
|scored_0263|E2|52-1|336||326|válido|
|scored_0263|E2|52-3|323||303|válido|
|scored_0263|E2|52-6|319||299|válido|
|scored_0263|E2|64-1|336||335|válido|
|scored_0263|E2|64-3|323||305|válido|
|scored_0263|E2|64-6|319||301|válido|
|scored_0263|E3|joint|319|individual 318|318|OPTIMAL|
|scored_0263|E4|repair|319||319|completo|
|scored_0264|E1|1|318|postprocess 318, optimize_a4 318, nested_windows 318|318|válido|
|scored_0264|E1|3|299|postprocess 299, optimize_a4 299, nested_windows 299|299|válido|
|scored_0264|E1|6|295|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|scored_0264|E2|52-1|318||283|válido|
|scored_0264|E2|52-3|299||279|válido|
|scored_0264|E2|52-6|295||269|válido|
|scored_0264|E2|64-1|318||291|válido|
|scored_0264|E2|64-3|299||274|válido|
|scored_0264|E2|64-6|295||271|válido|
|scored_0264|E3|joint|295|individual 295|292|OPTIMAL|
|scored_0264|E4|repair|295||295|completo|
|scored_0265|E1|1|281|postprocess 281, optimize_a4 281, nested_windows 281|281|válido|
|scored_0265|E1|3|276|postprocess 274, optimize_a4 274, nested_windows 274|274|válido|
|scored_0265|E1|6|263|postprocess 263, optimize_a4 263, nested_windows 263|263|válido|
|scored_0265|E2|52-1|281||254|válido|
|scored_0265|E2|52-3|276||247|válido|
|scored_0265|E2|52-6|263||239|válido|
|scored_0265|E2|64-1|281||256|válido|
|scored_0265|E2|64-3|276||245|válido|
|scored_0265|E2|64-6|263||234|válido|
|scored_0265|E3|joint|263|individual 263|263|OPTIMAL|
|scored_0265|E4|repair|263||263|completo|
|scored_0266|E1|1|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0266|E1|3|144|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0266|E1|6|147|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0266|E2|52-1|151||144|válido|
|scored_0266|E2|52-3|144||144|válido|
|scored_0266|E2|52-6|147||143|válido|
|scored_0266|E2|64-1|151||146|válido|
|scored_0266|E2|64-3|144||143|válido|
|scored_0266|E2|64-6|147||143|válido|
|scored_0266|E3|joint|144|individual 143|143|OPTIMAL|
|scored_0266|E4|repair|144||142|completo|
|scored_0267|E1|1|156|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0267|E1|3|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0267|E1|6|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0267|E2|52-1|156||150|válido|
|scored_0267|E2|52-3|151||149|válido|
|scored_0267|E2|52-6|151||149|válido|
|scored_0267|E2|64-1|156||149|válido|
|scored_0267|E2|64-3|151||149|válido|
|scored_0267|E2|64-6|151||149|válido|
|scored_0267|E3|joint|151|individual 150|150|OPTIMAL|
|scored_0267|E4|repair|151||149|completo|
|scored_0268|E1|1|291|postprocess 291, optimize_a4 291, nested_windows 291|291|válido|
|scored_0268|E1|3|285|postprocess 285, optimize_a4 285, nested_windows 285|285|válido|
|scored_0268|E1|6|281|postprocess 281, optimize_a4 281, nested_windows 281|281|válido|
|scored_0268|E2|52-1|291||266|válido|
|scored_0268|E2|52-3|285||263|válido|
|scored_0268|E2|52-6|281||260|válido|
|scored_0268|E2|64-1|291||268|válido|
|scored_0268|E2|64-3|285||262|válido|
|scored_0268|E2|64-6|281||261|válido|
|scored_0268|E3|joint|281|individual 280|280|OPTIMAL|
|scored_0268|E4|repair|281||281|completo|
|scored_0269|E1|1|408|postprocess 408, optimize_a4 408, nested_windows 408|408|válido|
|scored_0269|E1|3|377|postprocess 377, optimize_a4 377, nested_windows 377|377|válido|
|scored_0269|E1|6|371|postprocess 371, optimize_a4 371, nested_windows 371|371|válido|
|scored_0269|E2|52-1|408||377|válido|
|scored_0269|E2|52-3|377||351|válido|
|scored_0269|E2|52-6|371||347|válido|
|scored_0269|E2|64-1|408||383|válido|
|scored_0269|E2|64-3|377||352|válido|
|scored_0269|E2|64-6|371||356|válido|
|scored_0269|E3|joint|371|individual 371|371|FEASIBLE|
|scored_0269|E4|repair|371||371|completo|
|scored_0270|E1|1|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0270|E1|3|168|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0270|E1|6|167|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0270|E2|52-1|169||159|válido|
|scored_0270|E2|52-3|168||158|válido|
|scored_0270|E2|52-6|167||157|válido|
|scored_0270|E2|64-1|169||157|válido|
|scored_0270|E2|64-3|168||157|válido|
|scored_0270|E2|64-6|167||157|válido|
|scored_0270|E3|joint|167|individual 167|167|OPTIMAL|
|scored_0270|E4|repair|167||166|completo|
|scored_0271|E1|1|59|postprocess 58, optimize_a4 58, nested_windows 58|58|válido|
|scored_0271|E1|3|59|postprocess 58, optimize_a4 58, nested_windows 58|58|válido|
|scored_0271|E1|6|59|postprocess 58, optimize_a4 58, nested_windows 58|58|válido|
|scored_0271|E2|52-1|59||59|válido|
|scored_0271|E2|52-3|59||59|válido|
|scored_0271|E2|52-6|59||59|válido|
|scored_0271|E2|64-1|59||59|válido|
|scored_0271|E2|64-3|59||59|válido|
|scored_0271|E2|64-6|59||59|válido|
|scored_0271|E3|joint|59|individual 59|59|OPTIMAL|
|scored_0271|E4|repair|59||57|completo|
|scored_0272|E1|1|34|postprocess 32, optimize_a4 32, nested_windows 32|32|válido|
|scored_0272|E1|3|34|postprocess 32, optimize_a4 32, nested_windows 32|32|válido|
|scored_0272|E1|6|34|postprocess 32, optimize_a4 32, nested_windows 32|32|válido|
|scored_0272|E2|52-1|34||34|válido|
|scored_0272|E2|52-3|34||34|válido|
|scored_0272|E2|52-6|34||34|válido|
|scored_0272|E2|64-1|34||34|válido|
|scored_0272|E2|64-3|34||34|válido|
|scored_0272|E2|64-6|34||34|válido|
|scored_0272|E3|joint|34|individual 34|34|OPTIMAL|
|scored_0272|E4|repair|34||34|completo|
|scored_0273|E1|1|254|postprocess 254, optimize_a4 254, nested_windows 254|254|válido|
|scored_0273|E1|3|246|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|scored_0273|E1|6|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0273|E2|52-1|254||235|válido|
|scored_0273|E2|52-3|246||231|válido|
|scored_0273|E2|52-6|242||231|válido|
|scored_0273|E2|64-1|254||236|válido|
|scored_0273|E2|64-3|246||234|válido|
|scored_0273|E2|64-6|242||230|válido|
|scored_0273|E3|joint|242|individual 242|240|OPTIMAL|
|scored_0273|E4|repair|242||242|completo|
|scored_0274|E1|1|231|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|scored_0274|E1|3|230|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0274|E1|6|228|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|scored_0274|E2|52-1|231||219|válido|
|scored_0274|E2|52-3|230||217|válido|
|scored_0274|E2|52-6|228||211|válido|
|scored_0274|E2|64-1|231||220|válido|
|scored_0274|E2|64-3|230||218|válido|
|scored_0274|E2|64-6|228||208|válido|
|scored_0274|E3|joint|228|individual 228|226|OPTIMAL|
|scored_0274|E4|repair|228||228|completo|
|scored_0275|E1|1|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0275|E1|3|196|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|scored_0275|E1|6|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0275|E2|52-1|202||193|válido|
|scored_0275|E2|52-3|196||186|válido|
|scored_0275|E2|52-6|192||184|válido|
|scored_0275|E2|64-1|202||194|válido|
|scored_0275|E2|64-3|196||185|válido|
|scored_0275|E2|64-6|192||184|válido|
|scored_0275|E3|joint|192|individual 192|192|OPTIMAL|
|scored_0275|E4|repair|192||192|completo|
|scored_0276|E1|1|95|postprocess 94, optimize_a4 94, nested_windows 94|94|válido|
|scored_0276|E1|3|95|postprocess 94, optimize_a4 94, nested_windows 94|94|válido|
|scored_0276|E1|6|95|postprocess 94, optimize_a4 94, nested_windows 94|94|válido|
|scored_0276|E2|52-1|95||91|válido|
|scored_0276|E2|52-3|95||91|válido|
|scored_0276|E2|52-6|95||91|válido|
|scored_0276|E2|64-1|95||91|válido|
|scored_0276|E2|64-3|95||91|válido|
|scored_0276|E2|64-6|95||91|válido|
|scored_0276|E3|joint|95|individual 95|95|OPTIMAL|
|scored_0276|E4|repair|95||95|completo|
|scored_0277|E1|1|182|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0277|E1|3|175|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|scored_0277|E1|6|175|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|scored_0277|E2|52-1|182||175|válido|
|scored_0277|E2|52-3|175||172|válido|
|scored_0277|E2|52-6|175||171|válido|
|scored_0277|E2|64-1|182||177|válido|
|scored_0277|E2|64-3|175||172|válido|
|scored_0277|E2|64-6|175||171|válido|
|scored_0277|E3|joint|175|individual 175|175|OPTIMAL|
|scored_0277|E4|repair|175||175|completo|
|scored_0278|E1|1|107|postprocess 107, optimize_a4 107, nested_windows 107|107|válido|
|scored_0278|E1|3|107|postprocess 107, optimize_a4 107, nested_windows 107|107|válido|
|scored_0278|E1|6|107|postprocess 107, optimize_a4 107, nested_windows 107|107|válido|
|scored_0278|E2|52-1|107||104|válido|
|scored_0278|E2|52-3|107||104|válido|
|scored_0278|E2|52-6|107||104|válido|
|scored_0278|E2|64-1|107||104|válido|
|scored_0278|E2|64-3|107||104|válido|
|scored_0278|E2|64-6|107||104|válido|
|scored_0278|E3|joint|107|individual 107|107|OPTIMAL|
|scored_0278|E4|repair|107||105|completo|
|scored_0279|E1|1|366|postprocess 365, optimize_a4 365, nested_windows 365|365|válido|
|scored_0279|E1|3|350|postprocess 350, optimize_a4 350, nested_windows 350|350|válido|
|scored_0279|E1|6|344|postprocess 344, optimize_a4 344, nested_windows 344|344|válido|
|scored_0279|E2|52-1|366||340|válido|
|scored_0279|E2|52-3|350||330|válido|
|scored_0279|E2|52-6|344||320|válido|
|scored_0279|E2|64-1|366||340|válido|
|scored_0279|E2|64-3|350||330|válido|
|scored_0279|E2|64-6|344||321|válido|
|scored_0279|E3|joint|344|individual 344|341|OPTIMAL|
|scored_0279|E4|repair|344||344|completo|
|scored_0280|E1|1|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0280|E1|3|202|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0280|E1|6|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0280|E2|52-1|211||194|válido|
|scored_0280|E2|52-3|202||186|válido|
|scored_0280|E2|52-6|199||190|válido|
|scored_0280|E2|64-1|211||195|válido|
|scored_0280|E2|64-3|202||186|válido|
|scored_0280|E2|64-6|199||188|válido|
|scored_0280|E3|joint|199|individual 199|197|OPTIMAL|
|scored_0280|E4|repair|199||194|completo|
|scored_0281|E1|1|251|postprocess 251, optimize_a4 251, nested_windows 251|251|válido|
|scored_0281|E1|3|245|postprocess 245, optimize_a4 245, nested_windows 245|245|válido|
|scored_0281|E1|6|244|postprocess 244, optimize_a4 244, nested_windows 244|244|válido|
|scored_0281|E2|52-1|251||234|válido|
|scored_0281|E2|52-3|245||232|válido|
|scored_0281|E2|52-6|244||230|válido|
|scored_0281|E2|64-1|251||233|válido|
|scored_0281|E2|64-3|245||232|válido|
|scored_0281|E2|64-6|244||232|válido|
|scored_0281|E3|joint|244|individual 244|239|FEASIBLE|
|scored_0281|E4|repair|244||244|completo|
|scored_0282|E1|1|63|postprocess 62, optimize_a4 62, nested_windows 62|62|válido|
|scored_0282|E1|3|63|postprocess 62, optimize_a4 62, nested_windows 62|62|válido|
|scored_0282|E1|6|63|postprocess 62, optimize_a4 62, nested_windows 62|62|válido|
|scored_0282|E2|52-1|63||62|válido|
|scored_0282|E2|52-3|63||62|válido|
|scored_0282|E2|52-6|63||62|válido|
|scored_0282|E2|64-1|63||62|válido|
|scored_0282|E2|64-3|63||62|válido|
|scored_0282|E2|64-6|63||62|válido|
|scored_0282|E3|joint|63|individual 63|63|OPTIMAL|
|scored_0282|E4|repair|63||60|completo|
|scored_0283|E1|1|305|postprocess 305, optimize_a4 305, nested_windows 305|305|válido|
|scored_0283|E1|3|294|postprocess 294, optimize_a4 294, nested_windows 294|294|válido|
|scored_0283|E1|6|282|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|scored_0283|E2|52-1|305||279|válido|
|scored_0283|E2|52-3|294||270|válido|
|scored_0283|E2|52-6|282||273|válido|
|scored_0283|E2|64-1|305||279|válido|
|scored_0283|E2|64-3|294||271|válido|
|scored_0283|E2|64-6|282||274|válido|
|scored_0283|E3|joint|282|individual 282|282|OPTIMAL|
|scored_0283|E4|repair|282||282|completo|
|scored_0284|E1|1|200|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0284|E1|3|193|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0284|E1|6|189|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0284|E2|52-1|200||188|válido|
|scored_0284|E2|52-3|193||178|válido|
|scored_0284|E2|52-6|189||176|válido|
|scored_0284|E2|64-1|200||188|válido|
|scored_0284|E2|64-3|193||178|válido|
|scored_0284|E2|64-6|189||178|válido|
|scored_0284|E3|joint|189|individual 189|189|OPTIMAL|
|scored_0284|E4|repair|189||189|completo|
|scored_0285|E1|1|300|postprocess 300, optimize_a4 300, nested_windows 300|300|válido|
|scored_0285|E1|3|280|postprocess 279, optimize_a4 279, nested_windows 279|279|válido|
|scored_0285|E1|6|275|postprocess 275, optimize_a4 275, nested_windows 275|275|válido|
|scored_0285|E2|52-1|300||267|válido|
|scored_0285|E2|52-3|280||261|válido|
|scored_0285|E2|52-6|275||256|válido|
|scored_0285|E2|64-1|300||272|válido|
|scored_0285|E2|64-3|280||262|válido|
|scored_0285|E2|64-6|275||257|válido|
|scored_0285|E3|joint|275|individual 274|274|FEASIBLE|
|scored_0285|E4|repair|275||275|completo|
|scored_0286|E1|1|178|postprocess 178, optimize_a4 178, nested_windows 178|178|válido|
|scored_0286|E1|3|177|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0286|E1|6|177|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0286|E2|52-1|178||159|válido|
|scored_0286|E2|52-3|177||159|válido|
|scored_0286|E2|52-6|177||162|válido|
|scored_0286|E2|64-1|178||170|válido|
|scored_0286|E2|64-3|177||165|válido|
|scored_0286|E2|64-6|177||165|válido|
|scored_0286|E3|joint|177|individual 177|175|OPTIMAL|
|scored_0286|E4|repair|177||173|completo|
|scored_0287|E1|1|276|postprocess 276, optimize_a4 276, nested_windows 276|276|válido|
|scored_0287|E1|3|275|postprocess 275, optimize_a4 275, nested_windows 275|275|válido|
|scored_0287|E1|6|266|postprocess 265, optimize_a4 265, nested_windows 265|265|válido|
|scored_0287|E2|52-1|276||252|válido|
|scored_0287|E2|52-3|275||245|válido|
|scored_0287|E2|52-6|266||244|válido|
|scored_0287|E2|64-1|276||252|válido|
|scored_0287|E2|64-3|275||246|válido|
|scored_0287|E2|64-6|266||243|válido|
|scored_0287|E3|joint|266|individual 265|264|OPTIMAL|
|scored_0287|E4|repair|266||266|completo|
|scored_0288|E1|1|185|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|scored_0288|E1|3|181|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0288|E1|6|180|postprocess 180, optimize_a4 180, nested_windows 180|180|válido|
|scored_0288|E2|52-1|185||175|válido|
|scored_0288|E2|52-3|181||169|válido|
|scored_0288|E2|52-6|180||170|válido|
|scored_0288|E2|64-1|185||174|válido|
|scored_0288|E2|64-3|181||168|válido|
|scored_0288|E2|64-6|180||170|válido|
|scored_0288|E3|joint|180|individual 180|180|OPTIMAL|
|scored_0288|E4|repair|180||178|completo|
|scored_0289|E1|1|27|postprocess 25, optimize_a4 25, nested_windows 25|25|válido|
|scored_0289|E1|3|27|postprocess 25, optimize_a4 25, nested_windows 25|25|válido|
|scored_0289|E1|6|27|postprocess 25, optimize_a4 25, nested_windows 25|25|válido|
|scored_0289|E2|52-1|27||27|válido|
|scored_0289|E2|52-3|27||27|válido|
|scored_0289|E2|52-6|27||27|válido|
|scored_0289|E2|64-1|27||27|válido|
|scored_0289|E2|64-3|27||27|válido|
|scored_0289|E2|64-6|27||27|válido|
|scored_0289|E3|joint|26|individual 26|26|OPTIMAL|
|scored_0289|E4|repair|26||24|completo|
|scored_0290|E1|1|194|postprocess 194, optimize_a4 194, nested_windows 194|194|válido|
|scored_0290|E1|3|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0290|E1|6|189|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0290|E2|52-1|194||188|válido|
|scored_0290|E2|52-3|192||186|válido|
|scored_0290|E2|52-6|189||181|válido|
|scored_0290|E2|64-1|194||188|válido|
|scored_0290|E2|64-3|192||186|válido|
|scored_0290|E2|64-6|189||182|válido|
|scored_0290|E3|joint|189|individual 189|189|OPTIMAL|
|scored_0290|E4|repair|189||189|completo|
|scored_0291|E1|1|233|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0291|E1|3|225|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0291|E1|6|221|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0291|E2|52-1|233||210|válido|
|scored_0291|E2|52-3|225||206|válido|
|scored_0291|E2|52-6|221||207|válido|
|scored_0291|E2|64-1|233||211|válido|
|scored_0291|E2|64-3|225||204|válido|
|scored_0291|E2|64-6|221||204|válido|
|scored_0291|E3|joint|221|individual 221|221|OPTIMAL|
|scored_0291|E4|repair|221||221|completo|
|scored_0292|E1|1|71|postprocess 69, optimize_a4 69, nested_windows 69|69|válido|
|scored_0292|E1|3|71|postprocess 69, optimize_a4 69, nested_windows 69|69|válido|
|scored_0292|E1|6|71|postprocess 69, optimize_a4 69, nested_windows 69|69|válido|
|scored_0292|E2|52-1|71||66|válido|
|scored_0292|E2|52-3|71||66|válido|
|scored_0292|E2|52-6|71||66|válido|
|scored_0292|E2|64-1|71||66|válido|
|scored_0292|E2|64-3|71||66|válido|
|scored_0292|E2|64-6|71||66|válido|
|scored_0292|E3|joint|71|individual 71|71|OPTIMAL|
|scored_0292|E4|repair|71||71|completo|
|scored_0293|E1|1|273|postprocess 273, optimize_a4 273, nested_windows 273|273|válido|
|scored_0293|E1|3|270|postprocess 270, optimize_a4 270, nested_windows 270|270|válido|
|scored_0293|E1|6|265|postprocess 265, optimize_a4 265, nested_windows 265|265|válido|
|scored_0293|E2|52-1|273||254|válido|
|scored_0293|E2|52-3|270||245|válido|
|scored_0293|E2|52-6|265||245|válido|
|scored_0293|E2|64-1|273||255|válido|
|scored_0293|E2|64-3|270||247|válido|
|scored_0293|E2|64-6|265||248|válido|
|scored_0293|E3|joint|265|individual 265|262|OPTIMAL|
|scored_0293|E4|repair|265||265|completo|
|scored_0294|E1|1|221|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0294|E1|3|213|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0294|E1|6|210|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0294|E2|52-1|221||203|válido|
|scored_0294|E2|52-3|213||196|válido|
|scored_0294|E2|52-6|210||195|válido|
|scored_0294|E2|64-1|221||204|válido|
|scored_0294|E2|64-3|213||195|válido|
|scored_0294|E2|64-6|210||196|válido|
|scored_0294|E3|joint|210|individual 209|209|OPTIMAL|
|scored_0294|E4|repair|210||210|completo|
|scored_0295|E1|1|176|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|scored_0295|E1|3|174|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0295|E1|6|173|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|scored_0295|E2|52-1|176||168|válido|
|scored_0295|E2|52-3|174||167|válido|
|scored_0295|E2|52-6|173||163|válido|
|scored_0295|E2|64-1|176||168|válido|
|scored_0295|E2|64-3|174||167|válido|
|scored_0295|E2|64-6|173||164|válido|
|scored_0295|E3|joint|173|individual 172|172|OPTIMAL|
|scored_0295|E4|repair|173||170|completo|
|scored_0296|E1|1|56|postprocess 54, optimize_a4 54, nested_windows 54|54|válido|
|scored_0296|E1|3|56|postprocess 54, optimize_a4 54, nested_windows 54|54|válido|
|scored_0296|E1|6|56|postprocess 54, optimize_a4 54, nested_windows 54|54|válido|
|scored_0296|E2|52-1|56||56|válido|
|scored_0296|E2|52-3|56||56|válido|
|scored_0296|E2|52-6|56||56|válido|
|scored_0296|E2|64-1|56||56|válido|
|scored_0296|E2|64-3|56||56|válido|
|scored_0296|E2|64-6|56||56|válido|
|scored_0296|E3|joint|56|individual 56|56|OPTIMAL|
|scored_0296|E4|repair|56||56|completo|
|scored_0297|E1|1|168|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|scored_0297|E1|3|167|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0297|E1|6|166|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0297|E2|52-1|168||158|válido|
|scored_0297|E2|52-3|167||155|válido|
|scored_0297|E2|52-6|166||155|válido|
|scored_0297|E2|64-1|168||158|válido|
|scored_0297|E2|64-3|167||158|válido|
|scored_0297|E2|64-6|166||156|válido|
|scored_0297|E3|joint|166|individual 165|165|OPTIMAL|
|scored_0297|E4|repair|166||166|completo|
|scored_0298|E1|1|136|postprocess 136, optimize_a4 136, nested_windows 136|136|válido|
|scored_0298|E1|3|134|postprocess 134, optimize_a4 134, nested_windows 134|134|válido|
|scored_0298|E1|6|134|postprocess 134, optimize_a4 134, nested_windows 134|134|válido|
|scored_0298|E2|52-1|136||128|válido|
|scored_0298|E2|52-3|134||128|válido|
|scored_0298|E2|52-6|134||128|válido|
|scored_0298|E2|64-1|136||127|válido|
|scored_0298|E2|64-3|134||127|válido|
|scored_0298|E2|64-6|134||127|válido|
|scored_0298|E3|joint|134|individual 134|134|OPTIMAL|
|scored_0298|E4|repair|134||133|completo|
|scored_0299|E1|1|119|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|scored_0299|E1|3|120|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|scored_0299|E1|6|119|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0299|E2|52-1|119||118|válido|
|scored_0299|E2|52-3|120||118|válido|
|scored_0299|E2|52-6|119||117|válido|
|scored_0299|E2|64-1|119||118|válido|
|scored_0299|E2|64-3|120||118|válido|
|scored_0299|E2|64-6|119||117|válido|
|scored_0299|E3|joint|119|individual 119|119|OPTIMAL|
|scored_0299|E4|repair|119||118|completo|
|scored_0300|E1|1|57|postprocess 57, optimize_a4 57, nested_windows 57|57|válido|
|scored_0300|E1|3|57|postprocess 57, optimize_a4 57, nested_windows 57|57|válido|
|scored_0300|E1|6|57|postprocess 57, optimize_a4 57, nested_windows 57|57|válido|
|scored_0300|E2|52-1|57||56|válido|
|scored_0300|E2|52-3|57||56|válido|
|scored_0300|E2|52-6|57||56|válido|
|scored_0300|E2|64-1|57||56|válido|
|scored_0300|E2|64-3|57||56|válido|
|scored_0300|E2|64-6|57||56|válido|
|scored_0300|E3|joint|57|individual 57|57|OPTIMAL|
|scored_0300|E4|repair|57||56|completo|
|scored_0301|E1|1|5|postprocess 4, optimize_a4 4, nested_windows 4|4|válido|
|scored_0301|E1|3|5|postprocess 4, optimize_a4 4, nested_windows 4|4|válido|
|scored_0301|E1|6|5|postprocess 4, optimize_a4 4, nested_windows 4|4|válido|
|scored_0301|E2|52-1|5||5|válido|
|scored_0301|E2|52-3|5||5|válido|
|scored_0301|E2|52-6|5||5|válido|
|scored_0301|E2|64-1|5||5|válido|
|scored_0301|E2|64-3|5||5|válido|
|scored_0301|E2|64-6|5||5|válido|
|scored_0301|E3|joint|4|individual 4|4|OPTIMAL|
|scored_0301|E4|repair|4||4|completo|
|scored_0302|E1|1|236|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0302|E1|3|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0302|E1|6|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0302|E2|52-1|236||206|válido|
|scored_0302|E2|52-3|226||202|válido|
|scored_0302|E2|52-6|223||207|válido|
|scored_0302|E2|64-1|236||209|válido|
|scored_0302|E2|64-3|226||205|válido|
|scored_0302|E2|64-6|223||200|válido|
|scored_0302|E3|joint|223|individual 222|218|OPTIMAL|
|scored_0302|E4|repair|223||223|completo|
|scored_0303|E1|1|285|postprocess 285, optimize_a4 285, nested_windows 285|285|válido|
|scored_0303|E1|3|278|postprocess 278, optimize_a4 278, nested_windows 278|278|válido|
|scored_0303|E1|6|265|postprocess 265, optimize_a4 265, nested_windows 265|265|válido|
|scored_0303|E2|52-1|285||266|válido|
|scored_0303|E2|52-3|278||260|válido|
|scored_0303|E2|52-6|265||255|válido|
|scored_0303|E2|64-1|285||268|válido|
|scored_0303|E2|64-3|278||260|válido|
|scored_0303|E2|64-6|265||256|válido|
|scored_0303|E3|joint|265|individual 265|265|OPTIMAL|
|scored_0303|E4|repair|265||265|completo|
|scored_0304|E1|1|219|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|scored_0304|E1|3|213|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0304|E1|6|213|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0304|E2|52-1|219||199|válido|
|scored_0304|E2|52-3|213||198|válido|
|scored_0304|E2|52-6|213||197|válido|
|scored_0304|E2|64-1|219||199|válido|
|scored_0304|E2|64-3|213||198|válido|
|scored_0304|E2|64-6|213||196|válido|
|scored_0304|E3|joint|213|individual 213|212|OPTIMAL|
|scored_0304|E4|repair|213||211|completo|
|scored_0305|E1|1|228|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0305|E1|3|224|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0305|E1|6|219|postprocess 217, optimize_a4 217, nested_windows 217|217|válido|
|scored_0305|E2|52-1|228||211|válido|
|scored_0305|E2|52-3|224||209|válido|
|scored_0305|E2|52-6|219||205|válido|
|scored_0305|E2|64-1|228||210|válido|
|scored_0305|E2|64-3|224||208|válido|
|scored_0305|E2|64-6|219||208|válido|
|scored_0305|E3|joint|219|individual 218|217|OPTIMAL|
|scored_0305|E4|repair|219||218|completo|
|scored_0306|E1|1|381|postprocess 380, optimize_a4 380, nested_windows 380|380|válido|
|scored_0306|E1|3|353|postprocess 353, optimize_a4 353, nested_windows 353|353|válido|
|scored_0306|E1|6|352|postprocess 352, optimize_a4 352, nested_windows 352|352|válido|
|scored_0306|E2|52-1|381||346|válido|
|scored_0306|E2|52-3|353||332|válido|
|scored_0306|E2|52-6|352||328|válido|
|scored_0306|E2|64-1|381||353|válido|
|scored_0306|E2|64-3|353||337|válido|
|scored_0306|E2|64-6|352||331|válido|
|scored_0306|E3|joint|352|individual 351|349|FEASIBLE|
|scored_0306|E4|repair|352||352|completo|
|scored_0307|E1|1|182|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0307|E1|3|180|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|scored_0307|E1|6|174|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0307|E2|52-1|182||163|válido|
|scored_0307|E2|52-3|180||163|válido|
|scored_0307|E2|52-6|174||165|válido|
|scored_0307|E2|64-1|182||163|válido|
|scored_0307|E2|64-3|180||163|válido|
|scored_0307|E2|64-6|174||165|válido|
|scored_0307|E3|joint|174|individual 174|174|OPTIMAL|
|scored_0307|E4|repair|174||171|completo|
|scored_0308|E1|1|44|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0308|E1|3|44|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0308|E1|6|44|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0308|E2|52-1|44||44|válido|
|scored_0308|E2|52-3|44||44|válido|
|scored_0308|E2|52-6|44||44|válido|
|scored_0308|E2|64-1|44||44|válido|
|scored_0308|E2|64-3|44||44|válido|
|scored_0308|E2|64-6|44||44|válido|
|scored_0308|E3|joint|44|individual 44|44|OPTIMAL|
|scored_0308|E4|repair|44||42|completo|
|scored_0309|E1|1|182|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0309|E1|3|178|postprocess 176, optimize_a4 176, nested_windows 176|176|válido|
|scored_0309|E1|6|177|postprocess 176, optimize_a4 176, nested_windows 176|176|válido|
|scored_0309|E2|52-1|182||164|válido|
|scored_0309|E2|52-3|178||164|válido|
|scored_0309|E2|52-6|177||162|válido|
|scored_0309|E2|64-1|182||167|válido|
|scored_0309|E2|64-3|178||164|válido|
|scored_0309|E2|64-6|177||165|válido|
|scored_0309|E3|joint|177|individual 177|177|OPTIMAL|
|scored_0309|E4|repair|177||173|completo|
|scored_0310|E1|1|159|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0310|E1|3|157|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0310|E1|6|157|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0310|E2|52-1|159||149|válido|
|scored_0310|E2|52-3|157||148|válido|
|scored_0310|E2|52-6|157||146|válido|
|scored_0310|E2|64-1|159||148|válido|
|scored_0310|E2|64-3|157||146|válido|
|scored_0310|E2|64-6|157||145|válido|
|scored_0310|E3|joint|157|individual 157|157|OPTIMAL|
|scored_0310|E4|repair|157||157|completo|
|scored_0311|E1|1|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0311|E1|3|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0311|E1|6|144|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0311|E2|52-1|146||137|válido|
|scored_0311|E2|52-3|145||136|válido|
|scored_0311|E2|52-6|144||136|válido|
|scored_0311|E2|64-1|146||137|válido|
|scored_0311|E2|64-3|145||137|válido|
|scored_0311|E2|64-6|144||137|válido|
|scored_0311|E3|joint|144|individual 144|144|OPTIMAL|
|scored_0311|E4|repair|144||143|completo|
|scored_0312|E1|1|354|postprocess 354, optimize_a4 354, nested_windows 354|354|válido|
|scored_0312|E1|3|330|postprocess 330, optimize_a4 330, nested_windows 330|330|válido|
|scored_0312|E1|6|330|postprocess 329, optimize_a4 329, nested_windows 329|329|válido|
|scored_0312|E2|52-1|354||330|válido|
|scored_0312|E2|52-3|330||313|válido|
|scored_0312|E2|52-6|330||307|válido|
|scored_0312|E2|64-1|354||329|válido|
|scored_0312|E2|64-3|330||311|válido|
|scored_0312|E2|64-6|330||308|válido|
|scored_0312|E3|joint|330|individual 328|324|FEASIBLE|
|scored_0312|E4|repair|330||330|completo|
|scored_0313|E1|1|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|scored_0313|E1|3|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0313|E1|6|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0313|E2|52-1|238||222|válido|
|scored_0313|E2|52-3|229||217|válido|
|scored_0313|E2|52-6|226||212|válido|
|scored_0313|E2|64-1|238||223|válido|
|scored_0313|E2|64-3|229||217|válido|
|scored_0313|E2|64-6|226||212|válido|
|scored_0313|E3|joint|226|individual 226|225|OPTIMAL|
|scored_0313|E4|repair|226||226|completo|
|scored_0314|E1|1|279|postprocess 278, optimize_a4 278, nested_windows 278|278|válido|
|scored_0314|E1|3|264|postprocess 264, optimize_a4 264, nested_windows 264|264|válido|
|scored_0314|E1|6|252|postprocess 252, optimize_a4 252, nested_windows 252|252|válido|
|scored_0314|E2|52-1|279||252|válido|
|scored_0314|E2|52-3|264||248|válido|
|scored_0314|E2|52-6|252||243|válido|
|scored_0314|E2|64-1|279||255|válido|
|scored_0314|E2|64-3|264||247|válido|
|scored_0314|E2|64-6|252||243|válido|
|scored_0314|E3|joint|252|individual 252|252|OPTIMAL|
|scored_0314|E4|repair|252||252|completo|
|scored_0315|E1|1|316|postprocess 316, optimize_a4 316, nested_windows 316|316|válido|
|scored_0315|E1|3|299|postprocess 297, optimize_a4 297, nested_windows 297|297|válido|
|scored_0315|E1|6|297|postprocess 297, optimize_a4 297, nested_windows 297|297|válido|
|scored_0315|E2|52-1|316||297|válido|
|scored_0315|E2|52-3|299||285|válido|
|scored_0315|E2|52-6|297||283|válido|
|scored_0315|E2|64-1|316||300|válido|
|scored_0315|E2|64-3|299||288|válido|
|scored_0315|E2|64-6|297||286|válido|
|scored_0315|E3|joint|297|individual 297|296|OPTIMAL|
|scored_0315|E4|repair|297||297|completo|
|scored_0316|E1|1|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0316|E1|3|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0316|E1|6|144|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0316|E2|52-1|145||141|válido|
|scored_0316|E2|52-3|145||141|válido|
|scored_0316|E2|52-6|144||141|válido|
|scored_0316|E2|64-1|145||142|válido|
|scored_0316|E2|64-3|145||141|válido|
|scored_0316|E2|64-6|144||141|válido|
|scored_0316|E3|joint|144|individual 144|144|OPTIMAL|
|scored_0316|E4|repair|144||144|completo|
|scored_0317|E1|1|246|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|scored_0317|E1|3|237|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0317|E1|6|237|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|scored_0317|E2|52-1|246||229|válido|
|scored_0317|E2|52-3|237||226|válido|
|scored_0317|E2|52-6|237||224|válido|
|scored_0317|E2|64-1|246||231|válido|
|scored_0317|E2|64-3|237||227|válido|
|scored_0317|E2|64-6|237||226|válido|
|scored_0317|E3|joint|237|individual 237|235|OPTIMAL|
|scored_0317|E4|repair|237||237|completo|
|scored_0318|E1|1|243|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|scored_0318|E1|3|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0318|E1|6|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|scored_0318|E2|52-1|243||233|válido|
|scored_0318|E2|52-3|239||228|válido|
|scored_0318|E2|52-6|238||226|válido|
|scored_0318|E2|64-1|243||230|válido|
|scored_0318|E2|64-3|239||228|válido|
|scored_0318|E2|64-6|238||226|válido|
|scored_0318|E3|joint|238|individual 238|238|OPTIMAL|
|scored_0318|E4|repair|238||238|completo|
|scored_0319|E1|1|333|postprocess 333, optimize_a4 333, nested_windows 333|333|válido|
|scored_0319|E1|3|318|postprocess 318, optimize_a4 318, nested_windows 318|318|válido|
|scored_0319|E1|6|313|postprocess 313, optimize_a4 313, nested_windows 313|313|válido|
|scored_0319|E2|52-1|333||293|válido|
|scored_0319|E2|52-3|318||290|válido|
|scored_0319|E2|52-6|313||282|válido|
|scored_0319|E2|64-1|333||298|válido|
|scored_0319|E2|64-3|318||290|válido|
|scored_0319|E2|64-6|313||283|válido|
|scored_0319|E3|joint|313|individual 313|309|FEASIBLE|
|scored_0319|E4|repair|313||313|completo|
|scored_0320|E1|1|42|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0320|E1|3|42|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0320|E1|6|42|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0320|E2|52-1|42||42|válido|
|scored_0320|E2|52-3|42||42|válido|
|scored_0320|E2|52-6|42||42|válido|
|scored_0320|E2|64-1|42||42|válido|
|scored_0320|E2|64-3|42||42|válido|
|scored_0320|E2|64-6|42||42|válido|
|scored_0320|E3|joint|42|individual 42|42|OPTIMAL|
|scored_0320|E4|repair|42||41|completo|
|scored_0321|E1|1|400|postprocess 400, optimize_a4 400, nested_windows 400|400|válido|
|scored_0321|E1|3|373|postprocess 373, optimize_a4 373, nested_windows 373|373|válido|
|scored_0321|E1|6|364|postprocess 364, optimize_a4 364, nested_windows 364|364|válido|
|scored_0321|E2|52-1|400||358|válido|
|scored_0321|E2|52-3|373||347|válido|
|scored_0321|E2|52-6|364||340|válido|
|scored_0321|E2|64-1|400||363|válido|
|scored_0321|E2|64-3|373||350|válido|
|scored_0321|E2|64-6|364||346|válido|
|scored_0321|E3|joint|364|individual 364|360|FEASIBLE|
|scored_0321|E4|repair|364||364|completo|
|scored_0322|E1|1|249|postprocess 249, optimize_a4 249, nested_windows 249|249|válido|
|scored_0322|E1|3|241|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|scored_0322|E1|6|236|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0322|E2|52-1|249||228|válido|
|scored_0322|E2|52-3|241||221|válido|
|scored_0322|E2|52-6|236||222|válido|
|scored_0322|E2|64-1|249||228|válido|
|scored_0322|E2|64-3|241||220|válido|
|scored_0322|E2|64-6|236||222|válido|
|scored_0322|E3|joint|236|individual 236|235|FEASIBLE|
|scored_0322|E4|repair|236||236|completo|
|scored_0323|E1|1|203|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0323|E1|3|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0323|E1|6|198|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|scored_0323|E2|52-1|203||191|válido|
|scored_0323|E2|52-3|199||184|válido|
|scored_0323|E2|52-6|198||184|válido|
|scored_0323|E2|64-1|203||191|válido|
|scored_0323|E2|64-3|199||188|válido|
|scored_0323|E2|64-6|198||188|válido|
|scored_0323|E3|joint|198|individual 197|197|OPTIMAL|
|scored_0323|E4|repair|198||196|completo|
|scored_0324|E1|1|256|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|scored_0324|E1|3|245|postprocess 244, optimize_a4 244, nested_windows 244|244|válido|
|scored_0324|E1|6|245|postprocess 245, optimize_a4 245, nested_windows 245|245|válido|
|scored_0324|E2|52-1|256||234|válido|
|scored_0324|E2|52-3|245||226|válido|
|scored_0324|E2|52-6|245||224|válido|
|scored_0324|E2|64-1|256||234|válido|
|scored_0324|E2|64-3|245||226|válido|
|scored_0324|E2|64-6|245||223|válido|
|scored_0324|E3|joint|245|individual 245|241|FEASIBLE|
|scored_0324|E4|repair|245||245|completo|
|scored_0325|E1|1|168|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0325|E1|3|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0325|E1|6|163|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0325|E2|52-1|168||162|válido|
|scored_0325|E2|52-3|165||161|válido|
|scored_0325|E2|52-6|163||155|válido|
|scored_0325|E2|64-1|168||163|válido|
|scored_0325|E2|64-3|165||160|válido|
|scored_0325|E2|64-6|163||156|válido|
|scored_0325|E3|joint|163|individual 162|161|OPTIMAL|
|scored_0325|E4|repair|163||162|completo|
|scored_0326|E1|1|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0326|E1|3|195|postprocess 195, optimize_a4 195, nested_windows 195|195|válido|
|scored_0326|E1|6|193|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0326|E2|52-1|202||186|válido|
|scored_0326|E2|52-3|195||184|válido|
|scored_0326|E2|52-6|193||182|válido|
|scored_0326|E2|64-1|202||187|válido|
|scored_0326|E2|64-3|195||180|válido|
|scored_0326|E2|64-6|193||180|válido|
|scored_0326|E3|joint|193|individual 193|192|OPTIMAL|
|scored_0326|E4|repair|193||192|completo|
|scored_0327|E1|1|177|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0327|E1|3|174|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0327|E1|6|176|postprocess 176, optimize_a4 176, nested_windows 176|176|válido|
|scored_0327|E2|52-1|177||170|válido|
|scored_0327|E2|52-3|174||170|válido|
|scored_0327|E2|52-6|176||164|válido|
|scored_0327|E2|64-1|177||174|válido|
|scored_0327|E2|64-3|174||168|válido|
|scored_0327|E2|64-6|176||166|válido|
|scored_0327|E3|joint|174|individual 173|172|OPTIMAL|
|scored_0327|E4|repair|174||174|completo|
|scored_0328|E1|1|329|postprocess 329, optimize_a4 329, nested_windows 329|329|válido|
|scored_0328|E1|3|322|postprocess 322, optimize_a4 322, nested_windows 322|322|válido|
|scored_0328|E1|6|312|postprocess 312, optimize_a4 312, nested_windows 312|312|válido|
|scored_0328|E2|52-1|329||304|válido|
|scored_0328|E2|52-3|322||288|válido|
|scored_0328|E2|52-6|312||286|válido|
|scored_0328|E2|64-1|329||305|válido|
|scored_0328|E2|64-3|322||291|válido|
|scored_0328|E2|64-6|312||288|válido|
|scored_0328|E3|joint|312|individual 312|309|OPTIMAL|
|scored_0328|E4|repair|312||312|completo|
|scored_0329|E1|1|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0329|E1|3|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0329|E1|6|163|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0329|E2|52-1|169||158|válido|
|scored_0329|E2|52-3|165||158|válido|
|scored_0329|E2|52-6|163||158|válido|
|scored_0329|E2|64-1|169||158|válido|
|scored_0329|E2|64-3|165||156|válido|
|scored_0329|E2|64-6|163||156|válido|
|scored_0329|E3|joint|163|individual 163|163|OPTIMAL|
|scored_0329|E4|repair|163||163|completo|
|scored_0330|E1|1|236|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0330|E1|3|230|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|scored_0330|E1|6|224|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0330|E2|52-1|236||197|válido|
|scored_0330|E2|52-3|230||197|válido|
|scored_0330|E2|52-6|224||215|válido|
|scored_0330|E2|64-1|236||199|válido|
|scored_0330|E2|64-3|230||208|válido|
|scored_0330|E2|64-6|224||199|válido|
|scored_0330|E3|joint|224|individual 224|224|OPTIMAL|
|scored_0330|E4|repair|224||219|completo|
|scored_0331|E1|1|357|postprocess 356, optimize_a4 356, nested_windows 356|356|válido|
|scored_0331|E1|3|341|postprocess 341, optimize_a4 341, nested_windows 341|341|válido|
|scored_0331|E1|6|333|postprocess 333, optimize_a4 333, nested_windows 333|333|válido|
|scored_0331|E2|52-1|357||323|válido|
|scored_0331|E2|52-3|341||320|válido|
|scored_0331|E2|52-6|333||309|válido|
|scored_0331|E2|64-1|357||326|válido|
|scored_0331|E2|64-3|341||545|válido|
|scored_0331|E2|64-6|333||313|válido|
|scored_0331|E3|joint|333|individual 332|331|OPTIMAL|
|scored_0331|E4|repair|333||333|completo|
|scored_0332|E1|1|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0332|E1|3|203|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0332|E1|6|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|scored_0332|E2|52-1|207||192|válido|
|scored_0332|E2|52-3|203||190|válido|
|scored_0332|E2|52-6|197||181|válido|
|scored_0332|E2|64-1|207||192|válido|
|scored_0332|E2|64-3|203||191|válido|
|scored_0332|E2|64-6|197||184|válido|
|scored_0332|E3|joint|197|individual 197|197|OPTIMAL|
|scored_0332|E4|repair|197||195|completo|
|scored_0333|E1|1|304|postprocess 304, optimize_a4 304, nested_windows 304|304|válido|
|scored_0333|E1|3|287|postprocess 285, optimize_a4 285, nested_windows 285|285|válido|
|scored_0333|E1|6|281|postprocess 281, optimize_a4 281, nested_windows 281|281|válido|
|scored_0333|E2|52-1|304||285|válido|
|scored_0333|E2|52-3|287||271|válido|
|scored_0333|E2|52-6|281||267|válido|
|scored_0333|E2|64-1|304||285|válido|
|scored_0333|E2|64-3|287||272|válido|
|scored_0333|E2|64-6|281||265|válido|
|scored_0333|E3|joint|281|individual 281|280|FEASIBLE|
|scored_0333|E4|repair|281||281|completo|
|scored_0334|E1|1|347|postprocess 345, optimize_a4 345, nested_windows 345|345|válido|
|scored_0334|E1|3|321|postprocess 321, optimize_a4 321, nested_windows 321|321|válido|
|scored_0334|E1|6|318|postprocess 318, optimize_a4 318, nested_windows 318|318|válido|
|scored_0334|E2|52-1|347||301|válido|
|scored_0334|E2|52-3|321||295|válido|
|scored_0334|E2|52-6|318||285|válido|
|scored_0334|E2|64-1|347||305|válido|
|scored_0334|E2|64-3|321||295|válido|
|scored_0334|E2|64-6|318||287|válido|
|scored_0334|E3|joint|318|individual 318|316|OPTIMAL|
|scored_0334|E4|repair|318||318|completo|
|scored_0335|E1|1|270|postprocess 270, optimize_a4 270, nested_windows 270|270|válido|
|scored_0335|E1|3|257|postprocess 256, optimize_a4 256, nested_windows 256|256|válido|
|scored_0335|E1|6|252|postprocess 251, optimize_a4 251, nested_windows 251|251|válido|
|scored_0335|E2|52-1|270||246|válido|
|scored_0335|E2|52-3|257||237|válido|
|scored_0335|E2|52-6|252||233|válido|
|scored_0335|E2|64-1|270||248|válido|
|scored_0335|E2|64-3|257||232|válido|
|scored_0335|E2|64-6|252||234|válido|
|scored_0335|E3|joint|252|individual 251|249|OPTIMAL|
|scored_0335|E4|repair|252||252|completo|
|scored_0336|E1|1|212|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0336|E1|3|209|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0336|E1|6|205|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0336|E2|52-1|212||190|válido|
|scored_0336|E2|52-3|209||190|válido|
|scored_0336|E2|52-6|205||193|válido|
|scored_0336|E2|64-1|212||191|válido|
|scored_0336|E2|64-3|209||190|válido|
|scored_0336|E2|64-6|205||193|válido|
|scored_0336|E3|joint|205|individual 205|205|OPTIMAL|
|scored_0336|E4|repair|205||205|completo|
|scored_0337|E1|1|164|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0337|E1|3|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0337|E1|6|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0337|E2|52-1|164||155|válido|
|scored_0337|E2|52-3|162||155|válido|
|scored_0337|E2|52-6|162||154|válido|
|scored_0337|E2|64-1|164||151|válido|
|scored_0337|E2|64-3|162||151|válido|
|scored_0337|E2|64-6|162||150|válido|
|scored_0337|E3|joint|162|individual 162|162|OPTIMAL|
|scored_0337|E4|repair|162||162|completo|
|scored_0338|E1|1|245|postprocess 245, optimize_a4 245, nested_windows 245|245|válido|
|scored_0338|E1|3|236|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0338|E1|6|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0338|E2|52-1|245||230|válido|
|scored_0338|E2|52-3|236||222|válido|
|scored_0338|E2|52-6|235||221|válido|
|scored_0338|E2|64-1|245||227|válido|
|scored_0338|E2|64-3|236||223|válido|
|scored_0338|E2|64-6|235||221|válido|
|scored_0338|E3|joint|235|individual 235|234|OPTIMAL|
|scored_0338|E4|repair|235||235|completo|
|scored_0339|E1|1|124|postprocess 123, optimize_a4 123, nested_windows 123|123|válido|
|scored_0339|E1|3|124|postprocess 123, optimize_a4 123, nested_windows 123|123|válido|
|scored_0339|E1|6|124|postprocess 123, optimize_a4 123, nested_windows 123|123|válido|
|scored_0339|E2|52-1|124||117|válido|
|scored_0339|E2|52-3|124||117|válido|
|scored_0339|E2|52-6|124||117|válido|
|scored_0339|E2|64-1|124||113|válido|
|scored_0339|E2|64-3|124||113|válido|
|scored_0339|E2|64-6|124||113|válido|
|scored_0339|E3|joint|124|individual 124|124|OPTIMAL|
|scored_0339|E4|repair|124||115|completo|
|scored_0340|E1|1|160|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0340|E1|3|154|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0340|E1|6|154|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0340|E2|52-1|160||148|válido|
|scored_0340|E2|52-3|154||148|válido|
|scored_0340|E2|52-6|154||148|válido|
|scored_0340|E2|64-1|160||148|válido|
|scored_0340|E2|64-3|154||146|válido|
|scored_0340|E2|64-6|154||146|válido|
|scored_0340|E3|joint|154|individual 154|154|OPTIMAL|
|scored_0340|E4|repair|154||154|completo|
|scored_0341|E1|1|183|postprocess 183, optimize_a4 183, nested_windows 183|183|válido|
|scored_0341|E1|3|179|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|scored_0341|E1|6|174|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0341|E2|52-1|183||195|válido|
|scored_0341|E2|52-3|179||187|válido|
|scored_0341|E2|52-6|174||194|válido|
|scored_0341|E2|64-1|183||187|válido|
|scored_0341|E2|64-3|179||185|válido|
|scored_0341|E2|64-6|174||185|válido|
|scored_0341|E3|joint|174|individual 174|174|OPTIMAL|
|scored_0341|E4|repair|174||173|completo|
|scored_0342|E1|1|310|postprocess 309, optimize_a4 309, nested_windows 309|309|válido|
|scored_0342|E1|3|290|postprocess 290, optimize_a4 290, nested_windows 290|290|válido|
|scored_0342|E1|6|292|postprocess 292, optimize_a4 292, nested_windows 292|292|válido|
|scored_0342|E2|52-1|310||283|válido|
|scored_0342|E2|52-3|290||270|válido|
|scored_0342|E2|52-6|292||270|válido|
|scored_0342|E2|64-1|310||288|válido|
|scored_0342|E2|64-3|290||271|válido|
|scored_0342|E2|64-6|292||270|válido|
|scored_0342|E3|joint|290|individual 290|288|OPTIMAL|
|scored_0342|E4|repair|290||289|completo|
|scored_0343|E1|1|155|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|scored_0343|E1|3|153|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0343|E1|6|153|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0343|E2|52-1|155||147|válido|
|scored_0343|E2|52-3|153||146|válido|
|scored_0343|E2|52-6|153||145|válido|
|scored_0343|E2|64-1|155||146|válido|
|scored_0343|E2|64-3|153||147|válido|
|scored_0343|E2|64-6|153||146|válido|
|scored_0343|E3|joint|153|individual 152|152|OPTIMAL|
|scored_0343|E4|repair|153||153|completo|
|scored_0344|E1|1|223|postprocess 222, optimize_a4 222, nested_windows 222|222|válido|
|scored_0344|E1|3|216|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0344|E1|6|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0344|E2|52-1|223||202|válido|
|scored_0344|E2|52-3|216||206|válido|
|scored_0344|E2|52-6|211||203|válido|
|scored_0344|E2|64-1|223||202|válido|
|scored_0344|E2|64-3|216||206|válido|
|scored_0344|E2|64-6|211||205|válido|
|scored_0344|E3|joint|211|individual 210|210|OPTIMAL|
|scored_0344|E4|repair|211||210|completo|
|scored_0345|E1|1|279|postprocess 279, optimize_a4 279, nested_windows 279|279|válido|
|scored_0345|E1|3|269|postprocess 269, optimize_a4 269, nested_windows 269|269|válido|
|scored_0345|E1|6|267|postprocess 267, optimize_a4 267, nested_windows 267|267|válido|
|scored_0345|E2|52-1|279||255|válido|
|scored_0345|E2|52-3|269||257|válido|
|scored_0345|E2|52-6|267||246|válido|
|scored_0345|E2|64-1|279||257|válido|
|scored_0345|E2|64-3|269||255|válido|
|scored_0345|E2|64-6|267||245|válido|
|scored_0345|E3|joint|267|individual 267|263|OPTIMAL|
|scored_0345|E4|repair|267||267|completo|
|scored_0346|E1|1|234|postprocess 234, optimize_a4 234, nested_windows 234|234|válido|
|scored_0346|E1|3|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0346|E1|6|227|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0346|E2|52-1|234||221|válido|
|scored_0346|E2|52-3|226||213|válido|
|scored_0346|E2|52-6|227||213|válido|
|scored_0346|E2|64-1|234||222|válido|
|scored_0346|E2|64-3|226||216|válido|
|scored_0346|E2|64-6|227||214|válido|
|scored_0346|E3|joint|226|individual 226|226|OPTIMAL|
|scored_0346|E4|repair|226||225|completo|
|scored_0347|E1|1|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0347|E1|3|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0347|E1|6|155|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0347|E2|52-1|159||145|válido|
|scored_0347|E2|52-3|159||145|válido|
|scored_0347|E2|52-6|155||145|válido|
|scored_0347|E2|64-1|159||146|válido|
|scored_0347|E2|64-3|159||141|válido|
|scored_0347|E2|64-6|155||142|válido|
|scored_0347|E3|joint|155|individual 154|154|OPTIMAL|
|scored_0347|E4|repair|155||155|completo|
|scored_0348|E1|1|3|postprocess 3, optimize_a4 3, nested_windows 3|3|válido|
|scored_0348|E1|3|3|postprocess 3, optimize_a4 3, nested_windows 3|3|válido|
|scored_0348|E1|6|3|postprocess 3, optimize_a4 3, nested_windows 3|3|válido|
|scored_0348|E2|52-1|3||3|válido|
|scored_0348|E2|52-3|3||3|válido|
|scored_0348|E2|52-6|3||3|válido|
|scored_0348|E2|64-1|3||3|válido|
|scored_0348|E2|64-3|3||3|válido|
|scored_0348|E2|64-6|3||3|válido|
|scored_0348|E3|joint|3|individual 3|3|OPTIMAL|
|scored_0348|E4|repair|3||3|completo|
|scored_0349|E1|1|270|postprocess 269, optimize_a4 269, nested_windows 269|269|válido|
|scored_0349|E1|3|261|postprocess 261, optimize_a4 261, nested_windows 261|261|válido|
|scored_0349|E1|6|258|postprocess 257, optimize_a4 257, nested_windows 257|257|válido|
|scored_0349|E2|52-1|270||243|válido|
|scored_0349|E2|52-3|261||237|válido|
|scored_0349|E2|52-6|258||235|válido|
|scored_0349|E2|64-1|270||244|válido|
|scored_0349|E2|64-3|261||238|válido|
|scored_0349|E2|64-6|258||236|válido|
|scored_0349|E3|joint|258|individual 258|257|OPTIMAL|
|scored_0349|E4|repair|258||258|completo|
|scored_0350|E1|1|77|postprocess 76, optimize_a4 76, nested_windows 76|76|válido|
|scored_0350|E1|3|77|postprocess 76, optimize_a4 76, nested_windows 76|76|válido|
|scored_0350|E1|6|77|postprocess 76, optimize_a4 76, nested_windows 76|76|válido|
|scored_0350|E2|52-1|77||74|válido|
|scored_0350|E2|52-3|77||74|válido|
|scored_0350|E2|52-6|77||74|válido|
|scored_0350|E2|64-1|77||74|válido|
|scored_0350|E2|64-3|77||74|válido|
|scored_0350|E2|64-6|77||74|válido|
|scored_0350|E3|joint|77|individual 77|77|OPTIMAL|
|scored_0350|E4|repair|77||77|completo|
|scored_0351|E1|1|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0351|E1|3|216|postprocess 216, optimize_a4 216, nested_windows 216|216|válido|
|scored_0351|E1|6|214|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0351|E2|52-1|226||207|válido|
|scored_0351|E2|52-3|216||201|válido|
|scored_0351|E2|52-6|214||202|válido|
|scored_0351|E2|64-1|226||204|válido|
|scored_0351|E2|64-3|216||201|válido|
|scored_0351|E2|64-6|214||202|válido|
|scored_0351|E3|joint|214|individual 214|214|OPTIMAL|
|scored_0351|E4|repair|214||210|completo|
|scored_0352|E1|1|327|postprocess 327, optimize_a4 327, nested_windows 327|327|válido|
|scored_0352|E1|3|307|postprocess 307, optimize_a4 307, nested_windows 307|307|válido|
|scored_0352|E1|6|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0352|E2|52-1|327||296|válido|
|scored_0352|E2|52-3|307||288|válido|
|scored_0352|E2|52-6|303||288|válido|
|scored_0352|E2|64-1|327||294|válido|
|scored_0352|E2|64-3|307||290|válido|
|scored_0352|E2|64-6|303||288|válido|
|scored_0352|E3|joint|303|individual 303|299|OPTIMAL|
|scored_0352|E4|repair|303||303|completo|
|scored_0353|E1|1|200|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0353|E1|3|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0353|E1|6|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0353|E2|52-1|200||183|válido|
|scored_0353|E2|52-3|192||183|válido|
|scored_0353|E2|52-6|192||183|válido|
|scored_0353|E2|64-1|200||183|válido|
|scored_0353|E2|64-3|192||183|válido|
|scored_0353|E2|64-6|192||182|válido|
|scored_0353|E3|joint|192|individual 191|191|OPTIMAL|
|scored_0353|E4|repair|192||192|completo|
|scored_0354|E1|1|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0354|E1|3|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0354|E1|6|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0354|E2|52-1|211||201|válido|
|scored_0354|E2|52-3|202||191|válido|
|scored_0354|E2|52-6|199||189|válido|
|scored_0354|E2|64-1|211||194|válido|
|scored_0354|E2|64-3|202||184|válido|
|scored_0354|E2|64-6|199||182|válido|
|scored_0354|E3|joint|199|individual 198|198|OPTIMAL|
|scored_0354|E4|repair|199||195|completo|
|scored_0355|E1|1|158|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0355|E1|3|156|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0355|E1|6|156|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0355|E2|52-1|158||148|válido|
|scored_0355|E2|52-3|156||148|válido|
|scored_0355|E2|52-6|156||148|válido|
|scored_0355|E2|64-1|158||149|válido|
|scored_0355|E2|64-3|156||149|válido|
|scored_0355|E2|64-6|156||149|válido|
|scored_0355|E3|joint|156|individual 156|156|OPTIMAL|
|scored_0355|E4|repair|156||155|completo|
|scored_0356|E1|1|108|postprocess 108, optimize_a4 108, nested_windows 108|108|válido|
|scored_0356|E1|3|108|postprocess 108, optimize_a4 108, nested_windows 108|108|válido|
|scored_0356|E1|6|108|postprocess 108, optimize_a4 108, nested_windows 108|108|válido|
|scored_0356|E2|52-1|108||104|válido|
|scored_0356|E2|52-3|108||104|válido|
|scored_0356|E2|52-6|108||104|válido|
|scored_0356|E2|64-1|108||106|válido|
|scored_0356|E2|64-3|108||106|válido|
|scored_0356|E2|64-6|108||106|válido|
|scored_0356|E3|joint|108|individual 108|108|OPTIMAL|
|scored_0356|E4|repair|108||108|completo|
|scored_0357|E1|1|321|postprocess 321, optimize_a4 321, nested_windows 321|321|válido|
|scored_0357|E1|3|310|postprocess 310, optimize_a4 310, nested_windows 310|310|válido|
|scored_0357|E1|6|304|postprocess 304, optimize_a4 304, nested_windows 304|304|válido|
|scored_0357|E2|52-1|321||296|válido|
|scored_0357|E2|52-3|310||290|válido|
|scored_0357|E2|52-6|304||285|válido|
|scored_0357|E2|64-1|321||299|válido|
|scored_0357|E2|64-3|310||293|válido|
|scored_0357|E2|64-6|304||290|válido|
|scored_0357|E3|joint|304|individual 303|302|OPTIMAL|
|scored_0357|E4|repair|304||304|completo|
|scored_0358|E1|1|210|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0358|E1|3|203|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0358|E1|6|203|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0358|E2|52-1|210||187|válido|
|scored_0358|E2|52-3|203||182|válido|
|scored_0358|E2|52-6|203||188|válido|
|scored_0358|E2|64-1|210||186|válido|
|scored_0358|E2|64-3|203||182|válido|
|scored_0358|E2|64-6|203||184|válido|
|scored_0358|E3|joint|203|individual 203|201|OPTIMAL|
|scored_0358|E4|repair|203||200|completo|
|scored_0359|E1|1|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0359|E1|3|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0359|E1|6|149|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|scored_0359|E2|52-1|151||146|válido|
|scored_0359|E2|52-3|151||146|válido|
|scored_0359|E2|52-6|149||146|válido|
|scored_0359|E2|64-1|151||145|válido|
|scored_0359|E2|64-3|151||145|válido|
|scored_0359|E2|64-6|149||145|válido|
|scored_0359|E3|joint|149|individual 149|149|OPTIMAL|
|scored_0359|E4|repair|149||147|completo|
|scored_0360|E1|1|167|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0360|E1|3|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|scored_0360|E1|6|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|scored_0360|E2|52-1|167||162|válido|
|scored_0360|E2|52-3|166||161|válido|
|scored_0360|E2|52-6|166||161|válido|
|scored_0360|E2|64-1|167||161|válido|
|scored_0360|E2|64-3|166||161|válido|
|scored_0360|E2|64-6|166||161|válido|
|scored_0360|E3|joint|166|individual 166|166|OPTIMAL|
|scored_0360|E4|repair|166||163|completo|
|scored_0361|E1|1|244|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|scored_0361|E1|3|240|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0361|E1|6|237|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0361|E2|52-1|244||233|válido|
|scored_0361|E2|52-3|240||224|válido|
|scored_0361|E2|52-6|237||226|válido|
|scored_0361|E2|64-1|244||237|válido|
|scored_0361|E2|64-3|240||224|válido|
|scored_0361|E2|64-6|237||228|válido|
|scored_0361|E3|joint|237|individual 237|233|OPTIMAL|
|scored_0361|E4|repair|237||234|completo|
|scored_0362|E1|1|148|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0362|E1|3|146|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0362|E1|6|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0362|E2|52-1|148||147|válido|
|scored_0362|E2|52-3|146||144|válido|
|scored_0362|E2|52-6|143||144|válido|
|scored_0362|E2|64-1|148||147|válido|
|scored_0362|E2|64-3|146||144|válido|
|scored_0362|E2|64-6|143||142|válido|
|scored_0362|E3|joint|143|individual 143|143|OPTIMAL|
|scored_0362|E4|repair|143||140|completo|
|scored_0363|E1|1|222|postprocess 222, optimize_a4 222, nested_windows 222|222|válido|
|scored_0363|E1|3|223|postprocess 222, optimize_a4 222, nested_windows 222|222|válido|
|scored_0363|E1|6|217|postprocess 217, optimize_a4 217, nested_windows 217|217|válido|
|scored_0363|E2|52-1|222||208|válido|
|scored_0363|E2|52-3|223||207|válido|
|scored_0363|E2|52-6|217||205|válido|
|scored_0363|E2|64-1|222||208|válido|
|scored_0363|E2|64-3|223||205|válido|
|scored_0363|E2|64-6|217||205|válido|
|scored_0363|E3|joint|217|individual 217|215|OPTIMAL|
|scored_0363|E4|repair|217||217|completo|
|scored_0364|E1|1|253|postprocess 253, optimize_a4 253, nested_windows 253|253|válido|
|scored_0364|E1|3|240|postprocess 240, optimize_a4 240, nested_windows 240|240|válido|
|scored_0364|E1|6|233|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0364|E2|52-1|253||228|válido|
|scored_0364|E2|52-3|240||225|válido|
|scored_0364|E2|52-6|233||217|válido|
|scored_0364|E2|64-1|253||223|válido|
|scored_0364|E2|64-3|240||222|válido|
|scored_0364|E2|64-6|233||219|válido|
|scored_0364|E3|joint|233|individual 232|232|OPTIMAL|
|scored_0364|E4|repair|233||232|completo|
|scored_0365|E1|1|5|postprocess 5, optimize_a4 4, nested_windows 4|4|válido|
|scored_0365|E1|3|5|postprocess 5, optimize_a4 4, nested_windows 4|4|válido|
|scored_0365|E1|6|5|postprocess 5, optimize_a4 4, nested_windows 4|4|válido|
|scored_0365|E2|52-1|5||5|válido|
|scored_0365|E2|52-3|5||5|válido|
|scored_0365|E2|52-6|5||5|válido|
|scored_0365|E2|64-1|5||5|válido|
|scored_0365|E2|64-3|5||5|válido|
|scored_0365|E2|64-6|5||5|válido|
|scored_0365|E3|joint|4|individual 4|4|OPTIMAL|
|scored_0365|E4|repair|4||4|completo|
|scored_0366|E1|1|158|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0366|E1|3|154|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0366|E1|6|154|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0366|E2|52-1|158||157|válido|
|scored_0366|E2|52-3|154||155|válido|
|scored_0366|E2|52-6|154||151|válido|
|scored_0366|E2|64-1|158||155|válido|
|scored_0366|E2|64-3|154||155|válido|
|scored_0366|E2|64-6|154||154|válido|
|scored_0366|E3|joint|154|individual 154|154|OPTIMAL|
|scored_0366|E4|repair|154||148|completo|
|scored_0367|E1|1|158|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0367|E1|3|158|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0367|E1|6|158|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0367|E2|52-1|158||153|válido|
|scored_0367|E2|52-3|158||153|válido|
|scored_0367|E2|52-6|158||153|válido|
|scored_0367|E2|64-1|158||150|válido|
|scored_0367|E2|64-3|158||151|válido|
|scored_0367|E2|64-6|158||151|válido|
|scored_0367|E3|joint|158|individual 158|158|OPTIMAL|
|scored_0367|E4|repair|158||157|completo|
|scored_0368|E1|1|39|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0368|E1|3|39|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0368|E1|6|39|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0368|E2|52-1|39||39|válido|
|scored_0368|E2|52-3|39||39|válido|
|scored_0368|E2|52-6|39||39|válido|
|scored_0368|E2|64-1|39||39|válido|
|scored_0368|E2|64-3|39||39|válido|
|scored_0368|E2|64-6|39||39|válido|
|scored_0368|E3|joint|39|individual 39|39|OPTIMAL|
|scored_0368|E4|repair|39||39|completo|
|scored_0369|E1|1|345|postprocess 345, optimize_a4 345, nested_windows 345|345|válido|
|scored_0369|E1|3|316|postprocess 316, optimize_a4 316, nested_windows 316|316|válido|
|scored_0369|E1|6|310|postprocess 309, optimize_a4 309, nested_windows 309|309|válido|
|scored_0369|E2|52-1|345||301|válido|
|scored_0369|E2|52-3|316||304|válido|
|scored_0369|E2|52-6|310||298|válido|
|scored_0369|E2|64-1|345||302|válido|
|scored_0369|E2|64-3|316||302|válido|
|scored_0369|E2|64-6|310||294|válido|
|scored_0369|E3|joint|310|individual 310|310|OPTIMAL|
|scored_0369|E4|repair|310||310|completo|
|scored_0370|E1|1|374|postprocess 374, optimize_a4 374, nested_windows 374|374|válido|
|scored_0370|E1|3|357|postprocess 357, optimize_a4 357, nested_windows 357|357|válido|
|scored_0370|E1|6|355|postprocess 355, optimize_a4 355, nested_windows 355|355|válido|
|scored_0370|E2|52-1|374||343|válido|
|scored_0370|E2|52-3|357||328|válido|
|scored_0370|E2|52-6|355||327|válido|
|scored_0370|E2|64-1|374||473|válido|
|scored_0370|E2|64-3|357||334|válido|
|scored_0370|E2|64-6|355||334|válido|
|scored_0370|E3|joint|355|individual 355|354|OPTIMAL|
|scored_0370|E4|repair|355||355|completo|
|scored_0371|E1|1|179|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|scored_0371|E1|3|173|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|scored_0371|E1|6|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0371|E2|52-1|179||172|válido|
|scored_0371|E2|52-3|173||168|válido|
|scored_0371|E2|52-6|172||161|válido|
|scored_0371|E2|64-1|179||164|válido|
|scored_0371|E2|64-3|173||162|válido|
|scored_0371|E2|64-6|172||160|válido|
|scored_0371|E3|joint|172|individual 172|170|OPTIMAL|
|scored_0371|E4|repair|172||164|completo|
|scored_0372|E1|1|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0372|E1|3|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0372|E1|6|147|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0372|E2|52-1|148||145|válido|
|scored_0372|E2|52-3|148||144|válido|
|scored_0372|E2|52-6|147||144|válido|
|scored_0372|E2|64-1|148||141|válido|
|scored_0372|E2|64-3|148||141|válido|
|scored_0372|E2|64-6|147||141|válido|
|scored_0372|E3|joint|147|individual 147|147|OPTIMAL|
|scored_0372|E4|repair|147||147|completo|
|scored_0373|E1|1|338|postprocess 334, optimize_a4 334, nested_windows 334|334|válido|
|scored_0373|E1|3|300|postprocess 299, optimize_a4 299, nested_windows 299|299|válido|
|scored_0373|E1|6|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0373|E2|52-1|338||297|válido|
|scored_0373|E2|52-3|300||275|válido|
|scored_0373|E2|52-6|303||281|válido|
|scored_0373|E2|64-1|338||296|válido|
|scored_0373|E2|64-3|300||275|válido|
|scored_0373|E2|64-6|303||281|válido|
|scored_0373|E3|joint|300|individual 300|299|FEASIBLE|
|scored_0373|E4|repair|300||300|completo|
|scored_0374|E1|1|318|postprocess 318, optimize_a4 318, nested_windows 318|318|válido|
|scored_0374|E1|3|302|postprocess 300, optimize_a4 300, nested_windows 300|300|válido|
|scored_0374|E1|6|295|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|scored_0374|E2|52-1|318||281|válido|
|scored_0374|E2|52-3|302||277|válido|
|scored_0374|E2|52-6|295||274|válido|
|scored_0374|E2|64-1|318||324|válido|
|scored_0374|E2|64-3|302||278|válido|
|scored_0374|E2|64-6|295||280|válido|
|scored_0374|E3|joint|295|individual 295|295|OPTIMAL|
|scored_0374|E4|repair|295||295|completo|
|scored_0375|E1|1|308|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0375|E1|3|296|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0375|E1|6|287|postprocess 287, optimize_a4 287, nested_windows 287|287|válido|
|scored_0375|E2|52-1|308||280|válido|
|scored_0375|E2|52-3|296||269|válido|
|scored_0375|E2|52-6|287||263|válido|
|scored_0375|E2|64-1|308||280|válido|
|scored_0375|E2|64-3|296||273|válido|
|scored_0375|E2|64-6|287||263|válido|
|scored_0375|E3|joint|287|individual 287|287|OPTIMAL|
|scored_0375|E4|repair|287||287|completo|
|scored_0376|E1|1|37|postprocess 33, optimize_a4 33, nested_windows 33|33|válido|
|scored_0376|E1|3|37|postprocess 33, optimize_a4 33, nested_windows 33|33|válido|
|scored_0376|E1|6|37|postprocess 33, optimize_a4 33, nested_windows 33|33|válido|
|scored_0376|E2|52-1|37||37|válido|
|scored_0376|E2|52-3|37||37|válido|
|scored_0376|E2|52-6|37||37|válido|
|scored_0376|E2|64-1|37||37|válido|
|scored_0376|E2|64-3|37||37|válido|
|scored_0376|E2|64-6|37||37|válido|
|scored_0376|E3|joint|37|individual 37|37|OPTIMAL|
|scored_0376|E4|repair|37||37|completo|
|scored_0377|E1|1|111|postprocess 110, optimize_a4 110, nested_windows 110|110|válido|
|scored_0377|E1|3|108|postprocess 108, optimize_a4 108, nested_windows 108|108|válido|
|scored_0377|E1|6|108|postprocess 108, optimize_a4 108, nested_windows 108|108|válido|
|scored_0377|E2|52-1|111||107|válido|
|scored_0377|E2|52-3|108||105|válido|
|scored_0377|E2|52-6|108||105|válido|
|scored_0377|E2|64-1|111||108|válido|
|scored_0377|E2|64-3|108||105|válido|
|scored_0377|E2|64-6|108||105|válido|
|scored_0377|E3|joint|108|individual 108|108|OPTIMAL|
|scored_0377|E4|repair|108||106|completo|
|scored_0378|E1|1|112|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0378|E1|3|111|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0378|E1|6|112|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0378|E2|52-1|112||105|válido|
|scored_0378|E2|52-3|111||108|válido|
|scored_0378|E2|52-6|112||109|válido|
|scored_0378|E2|64-1|112||105|válido|
|scored_0378|E2|64-3|111||108|válido|
|scored_0378|E2|64-6|112||109|válido|
|scored_0378|E3|joint|111|individual 111|110|OPTIMAL|
|scored_0378|E4|repair|111||111|completo|
|scored_0379|E1|1|154|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0379|E1|3|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0379|E1|6|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0379|E2|52-1|154||150|válido|
|scored_0379|E2|52-3|152||149|válido|
|scored_0379|E2|52-6|152||148|válido|
|scored_0379|E2|64-1|154||150|válido|
|scored_0379|E2|64-3|152||149|válido|
|scored_0379|E2|64-6|152||149|válido|
|scored_0379|E3|joint|152|individual 152|152|OPTIMAL|
|scored_0379|E4|repair|152||152|completo|
|scored_0380|E1|1|216|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0380|E1|3|212|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|scored_0380|E1|6|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0380|E2|52-1|216||202|válido|
|scored_0380|E2|52-3|212||196|válido|
|scored_0380|E2|52-6|211||195|válido|
|scored_0380|E2|64-1|216||206|válido|
|scored_0380|E2|64-3|212||196|válido|
|scored_0380|E2|64-6|211||198|válido|
|scored_0380|E3|joint|211|individual 211|211|OPTIMAL|
|scored_0380|E4|repair|211||209|completo|
|scored_0381|E1|1|284|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|scored_0381|E1|3|270|postprocess 270, optimize_a4 270, nested_windows 270|270|válido|
|scored_0381|E1|6|267|postprocess 267, optimize_a4 267, nested_windows 267|267|válido|
|scored_0381|E2|52-1|284||262|válido|
|scored_0381|E2|52-3|270||256|válido|
|scored_0381|E2|52-6|267||245|válido|
|scored_0381|E2|64-1|284||262|válido|
|scored_0381|E2|64-3|270||257|válido|
|scored_0381|E2|64-6|267||253|válido|
|scored_0381|E3|joint|267|individual 267|267|OPTIMAL|
|scored_0381|E4|repair|267||267|completo|
|scored_0382|E1|1|203|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0382|E1|3|194|postprocess 194, optimize_a4 194, nested_windows 194|194|válido|
|scored_0382|E1|6|194|postprocess 194, optimize_a4 194, nested_windows 194|194|válido|
|scored_0382|E2|52-1|203||181|válido|
|scored_0382|E2|52-3|194||179|válido|
|scored_0382|E2|52-6|194||179|válido|
|scored_0382|E2|64-1|203||183|válido|
|scored_0382|E2|64-3|194||178|válido|
|scored_0382|E2|64-6|194||178|válido|
|scored_0382|E3|joint|194|individual 194|194|OPTIMAL|
|scored_0382|E4|repair|194||194|completo|
|scored_0383|E1|1|206|postprocess 205, optimize_a4 205, nested_windows 205|205|válido|
|scored_0383|E1|3|201|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0383|E1|6|204|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|scored_0383|E2|52-1|206||191|válido|
|scored_0383|E2|52-3|201||185|válido|
|scored_0383|E2|52-6|204||183|válido|
|scored_0383|E2|64-1|206||194|válido|
|scored_0383|E2|64-3|201||186|válido|
|scored_0383|E2|64-6|204||183|válido|
|scored_0383|E3|joint|201|individual 201|199|OPTIMAL|
|scored_0383|E4|repair|201||200|completo|
|scored_0384|E1|1|201|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0384|E1|3|194|postprocess 194, optimize_a4 194, nested_windows 194|194|válido|
|scored_0384|E1|6|189|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0384|E2|52-1|201||187|válido|
|scored_0384|E2|52-3|194||181|válido|
|scored_0384|E2|52-6|189||180|válido|
|scored_0384|E2|64-1|201||185|válido|
|scored_0384|E2|64-3|194||182|válido|
|scored_0384|E2|64-6|189||182|válido|
|scored_0384|E3|joint|189|individual 189|189|OPTIMAL|
|scored_0384|E4|repair|189||188|completo|
|scored_0385|E1|1|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0385|E1|3|204|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|scored_0385|E1|6|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0385|E2|52-1|215||199|válido|
|scored_0385|E2|52-3|204||189|válido|
|scored_0385|E2|52-6|207||189|válido|
|scored_0385|E2|64-1|215||199|válido|
|scored_0385|E2|64-3|204||190|válido|
|scored_0385|E2|64-6|207||188|válido|
|scored_0385|E3|joint|204|individual 204|202|OPTIMAL|
|scored_0385|E4|repair|204||204|completo|
|scored_0386|E1|1|270|postprocess 269, optimize_a4 269, nested_windows 269|269|válido|
|scored_0386|E1|3|263|postprocess 263, optimize_a4 263, nested_windows 263|263|válido|
|scored_0386|E1|6|258|postprocess 258, optimize_a4 258, nested_windows 258|258|válido|
|scored_0386|E2|52-1|270||250|válido|
|scored_0386|E2|52-3|263||240|válido|
|scored_0386|E2|52-6|258||244|válido|
|scored_0386|E2|64-1|270||250|válido|
|scored_0386|E2|64-3|263||241|válido|
|scored_0386|E2|64-6|258||245|válido|
|scored_0386|E3|joint|258|individual 257|257|OPTIMAL|
|scored_0386|E4|repair|258||258|completo|
|scored_0387|E1|1|39|postprocess 37, optimize_a4 37, nested_windows 37|37|válido|
|scored_0387|E1|3|39|postprocess 37, optimize_a4 37, nested_windows 37|37|válido|
|scored_0387|E1|6|39|postprocess 37, optimize_a4 37, nested_windows 37|37|válido|
|scored_0387|E2|52-1|39||39|válido|
|scored_0387|E2|52-3|39||39|válido|
|scored_0387|E2|52-6|39||39|válido|
|scored_0387|E2|64-1|39||39|válido|
|scored_0387|E2|64-3|39||39|válido|
|scored_0387|E2|64-6|39||39|válido|
|scored_0387|E3|joint|38|individual 38|38|OPTIMAL|
|scored_0387|E4|repair|38||38|completo|
|scored_0388|E1|1|33|postprocess 30, optimize_a4 30, nested_windows 30|30|válido|
|scored_0388|E1|3|33|postprocess 30, optimize_a4 30, nested_windows 30|30|válido|
|scored_0388|E1|6|33|postprocess 30, optimize_a4 30, nested_windows 30|30|válido|
|scored_0388|E2|52-1|33||33|válido|
|scored_0388|E2|52-3|33||33|válido|
|scored_0388|E2|52-6|33||33|válido|
|scored_0388|E2|64-1|33||33|válido|
|scored_0388|E2|64-3|33||33|válido|
|scored_0388|E2|64-6|33||33|válido|
|scored_0388|E3|joint|33|individual 31|30|OPTIMAL|
|scored_0388|E4|repair|33||32|completo|
|scored_0389|E1|1|236|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0389|E1|3|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0389|E1|6|234|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0389|E2|52-1|236||222|válido|
|scored_0389|E2|52-3|232||221|válido|
|scored_0389|E2|52-6|234||219|válido|
|scored_0389|E2|64-1|236||223|válido|
|scored_0389|E2|64-3|232||224|válido|
|scored_0389|E2|64-6|234||219|válido|
|scored_0389|E3|joint|232|individual 231|229|OPTIMAL|
|scored_0389|E4|repair|232||232|completo|
|scored_0390|E1|1|143|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0390|E1|3|142|postprocess 141, optimize_a4 141, nested_windows 141|141|válido|
|scored_0390|E1|6|140|postprocess 140, optimize_a4 140, nested_windows 140|140|válido|
|scored_0390|E2|52-1|143||132|válido|
|scored_0390|E2|52-3|142||131|válido|
|scored_0390|E2|52-6|140||131|válido|
|scored_0390|E2|64-1|143||132|válido|
|scored_0390|E2|64-3|142||130|válido|
|scored_0390|E2|64-6|140||130|válido|
|scored_0390|E3|joint|140|individual 140|139|OPTIMAL|
|scored_0390|E4|repair|140||138|completo|
|scored_0391|E1|1|319|postprocess 319, optimize_a4 319, nested_windows 319|319|válido|
|scored_0391|E1|3|301|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|scored_0391|E1|6|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|scored_0391|E2|52-1|319||283|válido|
|scored_0391|E2|52-3|301||274|válido|
|scored_0391|E2|52-6|289||270|válido|
|scored_0391|E2|64-1|319||285|válido|
|scored_0391|E2|64-3|301||275|válido|
|scored_0391|E2|64-6|289||270|válido|
|scored_0391|E3|joint|289|individual 289|289|rechazado por registros|
|scored_0391|E4|repair|289||289|completo|
|scored_0392|E1|1|153|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0392|E1|3|150|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|scored_0392|E1|6|151|postprocess 150, optimize_a4 150, nested_windows 150|150|válido|
|scored_0392|E2|52-1|153||149|válido|
|scored_0392|E2|52-3|150||144|válido|
|scored_0392|E2|52-6|151||144|válido|
|scored_0392|E2|64-1|153||149|válido|
|scored_0392|E2|64-3|150||148|válido|
|scored_0392|E2|64-6|151||150|válido|
|scored_0392|E3|joint|150|individual 150|150|OPTIMAL|
|scored_0392|E4|repair|150||147|completo|
|scored_0393|E1|1|35|postprocess 34, optimize_a4 34, nested_windows 34|34|válido|
|scored_0393|E1|3|35|postprocess 34, optimize_a4 34, nested_windows 34|34|válido|
|scored_0393|E1|6|35|postprocess 34, optimize_a4 34, nested_windows 34|34|válido|
|scored_0393|E2|52-1|35||35|válido|
|scored_0393|E2|52-3|35||35|válido|
|scored_0393|E2|52-6|35||35|válido|
|scored_0393|E2|64-1|35||35|válido|
|scored_0393|E2|64-3|35||35|válido|
|scored_0393|E2|64-6|35||35|válido|
|scored_0393|E3|joint|35|individual 35|35|OPTIMAL|
|scored_0393|E4|repair|35||35|completo|
|scored_0394|E1|1|183|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0394|E1|3|183|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0394|E1|6|183|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0394|E2|52-1|183||175|válido|
|scored_0394|E2|52-3|183||173|válido|
|scored_0394|E2|52-6|183||172|válido|
|scored_0394|E2|64-1|183||175|válido|
|scored_0394|E2|64-3|183||173|válido|
|scored_0394|E2|64-6|183||173|válido|
|scored_0394|E3|joint|183|individual 182|181|OPTIMAL|
|scored_0394|E4|repair|183||180|completo|
|scored_0395|E1|1|117|postprocess 117, optimize_a4 117, nested_windows 117|117|válido|
|scored_0395|E1|3|117|postprocess 117, optimize_a4 117, nested_windows 117|117|válido|
|scored_0395|E1|6|117|postprocess 117, optimize_a4 117, nested_windows 117|117|válido|
|scored_0395|E2|52-1|117||115|válido|
|scored_0395|E2|52-3|117||115|válido|
|scored_0395|E2|52-6|117||115|válido|
|scored_0395|E2|64-1|117||116|válido|
|scored_0395|E2|64-3|117||116|válido|
|scored_0395|E2|64-6|117||116|válido|
|scored_0395|E3|joint|117|individual 117|117|OPTIMAL|
|scored_0395|E4|repair|117||115|completo|
|scored_0396|E1|1|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0396|E1|3|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0396|E1|6|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0396|E2|52-1|50||50|válido|
|scored_0396|E2|52-3|50||50|válido|
|scored_0396|E2|52-6|50||50|válido|
|scored_0396|E2|64-1|50||50|válido|
|scored_0396|E2|64-3|50||50|válido|
|scored_0396|E2|64-6|50||50|válido|
|scored_0396|E3|joint|50|individual 50|50|OPTIMAL|
|scored_0396|E4|repair|50||50|completo|
|scored_0397|E1|1|273|postprocess 273, optimize_a4 273, nested_windows 273|273|válido|
|scored_0397|E1|3|261|postprocess 261, optimize_a4 261, nested_windows 261|261|válido|
|scored_0397|E1|6|257|postprocess 257, optimize_a4 257, nested_windows 257|257|válido|
|scored_0397|E2|52-1|273||247|válido|
|scored_0397|E2|52-3|261||240|válido|
|scored_0397|E2|52-6|257||240|válido|
|scored_0397|E2|64-1|273||250|válido|
|scored_0397|E2|64-3|261||240|válido|
|scored_0397|E2|64-6|257||243|válido|
|scored_0397|E3|joint|257|individual 257|257|OPTIMAL|
|scored_0397|E4|repair|257||257|completo|
|scored_0398|E1|1|288|postprocess 288, optimize_a4 288, nested_windows 288|288|válido|
|scored_0398|E1|3|277|postprocess 276, optimize_a4 276, nested_windows 276|276|válido|
|scored_0398|E1|6|269|postprocess 269, optimize_a4 269, nested_windows 269|269|válido|
|scored_0398|E2|52-1|288||265|válido|
|scored_0398|E2|52-3|277||256|válido|
|scored_0398|E2|52-6|269||255|válido|
|scored_0398|E2|64-1|288||265|válido|
|scored_0398|E2|64-3|277||256|válido|
|scored_0398|E2|64-6|269||254|válido|
|scored_0398|E3|joint|269|individual 268|268|OPTIMAL|
|scored_0398|E4|repair|269||269|completo|
|scored_0399|E1|1|33|postprocess 31, optimize_a4 31, nested_windows 31|31|válido|
|scored_0399|E1|3|33|postprocess 31, optimize_a4 31, nested_windows 31|31|válido|
|scored_0399|E1|6|33|postprocess 31, optimize_a4 31, nested_windows 31|31|válido|
|scored_0399|E2|52-1|33||33|válido|
|scored_0399|E2|52-3|33||33|válido|
|scored_0399|E2|52-6|33||33|válido|
|scored_0399|E2|64-1|33||33|válido|
|scored_0399|E2|64-3|33||33|válido|
|scored_0399|E2|64-6|33||33|válido|
|scored_0399|E3|joint|30|individual 28|28|OPTIMAL|
|scored_0399|E4|repair|30||29|completo|
|scored_0400|E1|1|314|postprocess 314, optimize_a4 314, nested_windows 314|314|válido|
|scored_0400|E1|3|307|postprocess 307, optimize_a4 307, nested_windows 307|307|válido|
|scored_0400|E1|6|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0400|E2|52-1|314||297|válido|
|scored_0400|E2|52-3|307||284|válido|
|scored_0400|E2|52-6|303||313|válido|
|scored_0400|E2|64-1|314||299|válido|
|scored_0400|E2|64-3|307||284|válido|
|scored_0400|E2|64-6|303||330|válido|
|scored_0400|E3|joint|303|individual 302|300|OPTIMAL|
|scored_0400|E4|repair|303||303|completo|
|scored_0401|E1|1|42|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|scored_0401|E1|3|42|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|scored_0401|E1|6|42|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|scored_0401|E2|52-1|42||42|válido|
|scored_0401|E2|52-3|42||42|válido|
|scored_0401|E2|52-6|42||42|válido|
|scored_0401|E2|64-1|42||42|válido|
|scored_0401|E2|64-3|42||42|válido|
|scored_0401|E2|64-6|42||42|válido|
|scored_0401|E3|joint|42|individual 42|42|OPTIMAL|
|scored_0401|E4|repair|42||41|completo|
|scored_0402|E1|1|124|postprocess 123, optimize_a4 123, nested_windows 123|123|válido|
|scored_0402|E1|3|123|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0402|E1|6|123|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0402|E2|52-1|124||120|válido|
|scored_0402|E2|52-3|123||120|válido|
|scored_0402|E2|52-6|123||120|válido|
|scored_0402|E2|64-1|124||120|válido|
|scored_0402|E2|64-3|123||120|válido|
|scored_0402|E2|64-6|123||120|válido|
|scored_0402|E3|joint|123|individual 123|123|OPTIMAL|
|scored_0402|E4|repair|123||120|completo|
|scored_0403|E1|1|334|postprocess 333, optimize_a4 333, nested_windows 333|333|válido|
|scored_0403|E1|3|329|postprocess 329, optimize_a4 329, nested_windows 329|329|válido|
|scored_0403|E1|6|315|postprocess 315, optimize_a4 315, nested_windows 315|315|válido|
|scored_0403|E2|52-1|334||303|válido|
|scored_0403|E2|52-3|329||292|válido|
|scored_0403|E2|52-6|315||295|válido|
|scored_0403|E2|64-1|334||306|válido|
|scored_0403|E2|64-3|329||297|válido|
|scored_0403|E2|64-6|315||294|válido|
|scored_0403|E3|joint|315|individual 315|314|FEASIBLE|
|scored_0403|E4|repair|315||315|completo|
|scored_0404|E1|1|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0404|E1|3|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0404|E1|6|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0404|E2|52-1|137||130|válido|
|scored_0404|E2|52-3|137||130|válido|
|scored_0404|E2|52-6|137||130|válido|
|scored_0404|E2|64-1|137||132|válido|
|scored_0404|E2|64-3|137||131|válido|
|scored_0404|E2|64-6|137||131|válido|
|scored_0404|E3|joint|137|individual 137|137|OPTIMAL|
|scored_0404|E4|repair|137||137|completo|
|scored_0405|E1|1|404|postprocess 401, optimize_a4 401, nested_windows 401|401|válido|
|scored_0405|E1|3|380|postprocess 380, optimize_a4 380, nested_windows 380|380|válido|
|scored_0405|E1|6|363|postprocess 363, optimize_a4 363, nested_windows 363|363|válido|
|scored_0405|E2|52-1|404||359|válido|
|scored_0405|E2|52-3|380||345|válido|
|scored_0405|E2|52-6|363||336|válido|
|scored_0405|E2|64-1|404||362|válido|
|scored_0405|E2|64-3|380||434|válido|
|scored_0405|E2|64-6|363||338|válido|
|scored_0405|E3|joint|363|individual 363|361|FEASIBLE|
|scored_0405|E4|repair|363||363|completo|
|scored_0406|E1|1|189|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0406|E1|3|184|postprocess 183, optimize_a4 183, nested_windows 183|183|válido|
|scored_0406|E1|6|183|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0406|E2|52-1|189||176|válido|
|scored_0406|E2|52-3|184||173|válido|
|scored_0406|E2|52-6|183||176|válido|
|scored_0406|E2|64-1|189||174|válido|
|scored_0406|E2|64-3|184||172|válido|
|scored_0406|E2|64-6|183||173|válido|
|scored_0406|E3|joint|183|individual 183|183|OPTIMAL|
|scored_0406|E4|repair|183||181|completo|
|scored_0407|E1|1|218|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|scored_0407|E1|3|209|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0407|E1|6|208|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0407|E2|52-1|218||202|válido|
|scored_0407|E2|52-3|209||199|válido|
|scored_0407|E2|52-6|208||198|válido|
|scored_0407|E2|64-1|218||201|válido|
|scored_0407|E2|64-3|209||198|válido|
|scored_0407|E2|64-6|208||199|válido|
|scored_0407|E3|joint|208|individual 208|207|OPTIMAL|
|scored_0407|E4|repair|208||208|completo|
|scored_0408|E1|1|228|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0408|E1|3|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0408|E1|6|222|postprocess 222, optimize_a4 222, nested_windows 222|222|válido|
|scored_0408|E2|52-1|228||214|válido|
|scored_0408|E2|52-3|226||208|válido|
|scored_0408|E2|52-6|222||207|válido|
|scored_0408|E2|64-1|228||213|válido|
|scored_0408|E2|64-3|226||207|válido|
|scored_0408|E2|64-6|222||207|válido|
|scored_0408|E3|joint|222|individual 222|222|OPTIMAL|
|scored_0408|E4|repair|222||221|completo|
|scored_0409|E1|1|33|postprocess 32, optimize_a4 32, nested_windows 32|32|válido|
|scored_0409|E1|3|33|postprocess 32, optimize_a4 32, nested_windows 32|32|válido|
|scored_0409|E1|6|33|postprocess 32, optimize_a4 32, nested_windows 32|32|válido|
|scored_0409|E2|52-1|33||33|válido|
|scored_0409|E2|52-3|33||33|válido|
|scored_0409|E2|52-6|33||33|válido|
|scored_0409|E2|64-1|33||33|válido|
|scored_0409|E2|64-3|33||33|válido|
|scored_0409|E2|64-6|33||33|válido|
|scored_0409|E3|joint|31|individual 30|30|OPTIMAL|
|scored_0409|E4|repair|31||31|completo|
|scored_0410|E1|1|253|postprocess 253, optimize_a4 253, nested_windows 253|253|válido|
|scored_0410|E1|3|243|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|scored_0410|E1|6|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0410|E2|52-1|253||236|válido|
|scored_0410|E2|52-3|243||228|válido|
|scored_0410|E2|52-6|242||228|válido|
|scored_0410|E2|64-1|253||236|válido|
|scored_0410|E2|64-3|243||229|válido|
|scored_0410|E2|64-6|242||225|válido|
|scored_0410|E3|joint|242|individual 242|242|OPTIMAL|
|scored_0410|E4|repair|242||242|completo|
|scored_0411|E1|1|254|postprocess 254, optimize_a4 254, nested_windows 254|254|válido|
|scored_0411|E1|3|237|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|scored_0411|E1|6|247|postprocess 247, optimize_a4 247, nested_windows 247|247|válido|
|scored_0411|E2|52-1|254||230|válido|
|scored_0411|E2|52-3|237||226|válido|
|scored_0411|E2|52-6|247||226|válido|
|scored_0411|E2|64-1|254||232|válido|
|scored_0411|E2|64-3|237||227|válido|
|scored_0411|E2|64-6|247||226|válido|
|scored_0411|E3|joint|237|individual 237|237|OPTIMAL|
|scored_0411|E4|repair|237||237|completo|
|scored_0412|E1|1|244|postprocess 244, optimize_a4 244, nested_windows 244|244|válido|
|scored_0412|E1|3|236|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0412|E1|6|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0412|E2|52-1|244||225|válido|
|scored_0412|E2|52-3|236||225|válido|
|scored_0412|E2|52-6|232||218|válido|
|scored_0412|E2|64-1|244||225|válido|
|scored_0412|E2|64-3|236||225|válido|
|scored_0412|E2|64-6|232||219|válido|
|scored_0412|E3|joint|232|individual 231|231|FEASIBLE|
|scored_0412|E4|repair|232||232|completo|
|scored_0413|E1|1|212|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0413|E1|3|209|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|scored_0413|E1|6|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0413|E2|52-1|212||200|válido|
|scored_0413|E2|52-3|209||193|válido|
|scored_0413|E2|52-6|206||193|válido|
|scored_0413|E2|64-1|212||201|válido|
|scored_0413|E2|64-3|209||194|válido|
|scored_0413|E2|64-6|206||191|válido|
|scored_0413|E3|joint|206|individual 206|206|OPTIMAL|
|scored_0413|E4|repair|206||206|completo|
|scored_0414|E1|1|177|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|scored_0414|E1|3|176|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0414|E1|6|174|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0414|E2|52-1|177||170|válido|
|scored_0414|E2|52-3|176||169|válido|
|scored_0414|E2|52-6|174||163|válido|
|scored_0414|E2|64-1|177||172|válido|
|scored_0414|E2|64-3|176||170|válido|
|scored_0414|E2|64-6|174||165|válido|
|scored_0414|E3|joint|174|individual 174|173|OPTIMAL|
|scored_0414|E4|repair|174||171|completo|
|scored_0415|E1|1|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0415|E1|3|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0415|E1|6|198|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|scored_0415|E2|52-1|206||190|válido|
|scored_0415|E2|52-3|202||190|válido|
|scored_0415|E2|52-6|198||188|válido|
|scored_0415|E2|64-1|206||190|válido|
|scored_0415|E2|64-3|202||190|válido|
|scored_0415|E2|64-6|198||189|válido|
|scored_0415|E3|joint|198|individual 198|198|OPTIMAL|
|scored_0415|E4|repair|198||198|completo|
|scored_0416|E1|1|262|postprocess 262, optimize_a4 262, nested_windows 262|262|válido|
|scored_0416|E1|3|246|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|scored_0416|E1|6|246|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|scored_0416|E2|52-1|262||231|válido|
|scored_0416|E2|52-3|246||228|válido|
|scored_0416|E2|52-6|246||227|válido|
|scored_0416|E2|64-1|262||233|válido|
|scored_0416|E2|64-3|246||227|válido|
|scored_0416|E2|64-6|246||225|válido|
|scored_0416|E3|joint|246|individual 245|241|OPTIMAL|
|scored_0416|E4|repair|246||246|completo|
|scored_0417|E1|1|237|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|scored_0417|E1|3|228|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|scored_0417|E1|6|225|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0417|E2|52-1|237||218|válido|
|scored_0417|E2|52-3|228||213|válido|
|scored_0417|E2|52-6|225||214|válido|
|scored_0417|E2|64-1|237||218|válido|
|scored_0417|E2|64-3|228||214|válido|
|scored_0417|E2|64-6|225||214|válido|
|scored_0417|E3|joint|225|individual 225|224|OPTIMAL|
|scored_0417|E4|repair|225||225|completo|
|scored_0418|E1|1|161|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0418|E1|3|157|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0418|E1|6|157|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0418|E2|52-1|161||154|válido|
|scored_0418|E2|52-3|157||153|válido|
|scored_0418|E2|52-6|157||150|válido|
|scored_0418|E2|64-1|161||153|válido|
|scored_0418|E2|64-3|157||149|válido|
|scored_0418|E2|64-6|157||149|válido|
|scored_0418|E3|joint|157|individual 156|156|OPTIMAL|
|scored_0418|E4|repair|157||155|completo|
|scored_0419|E1|1|32|postprocess 31, optimize_a4 31, nested_windows 31|31|válido|
|scored_0419|E1|3|32|postprocess 31, optimize_a4 31, nested_windows 31|31|válido|
|scored_0419|E1|6|32|postprocess 31, optimize_a4 31, nested_windows 31|31|válido|
|scored_0419|E2|52-1|32||32|válido|
|scored_0419|E2|52-3|32||32|válido|
|scored_0419|E2|52-6|32||32|válido|
|scored_0419|E2|64-1|32||32|válido|
|scored_0419|E2|64-3|32||32|válido|
|scored_0419|E2|64-6|32||32|válido|
|scored_0419|E3|joint|32|individual 32|32|OPTIMAL|
|scored_0419|E4|repair|32||32|completo|
|scored_0420|E1|1|221|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0420|E1|3|210|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0420|E1|6|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0420|E2|52-1|221||204|válido|
|scored_0420|E2|52-3|210||197|válido|
|scored_0420|E2|52-6|207||191|válido|
|scored_0420|E2|64-1|221||200|válido|
|scored_0420|E2|64-3|210||199|válido|
|scored_0420|E2|64-6|207||193|válido|
|scored_0420|E3|joint|207|individual 207|207|OPTIMAL|
|scored_0420|E4|repair|207||207|completo|
|scored_0421|E1|1|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0421|E1|3|213|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0421|E1|6|205|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|scored_0421|E2|52-1|215||203|válido|
|scored_0421|E2|52-3|213||199|válido|
|scored_0421|E2|52-6|205||196|válido|
|scored_0421|E2|64-1|215||201|válido|
|scored_0421|E2|64-3|213||200|válido|
|scored_0421|E2|64-6|205||196|válido|
|scored_0421|E3|joint|205|individual 205|204|OPTIMAL|
|scored_0421|E4|repair|205||201|completo|
|scored_0422|E1|1|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0422|E1|3|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0422|E1|6|219|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|scored_0422|E2|52-1|223||209|válido|
|scored_0422|E2|52-3|219||203|válido|
|scored_0422|E2|52-6|219||201|válido|
|scored_0422|E2|64-1|223||209|válido|
|scored_0422|E2|64-3|219||203|válido|
|scored_0422|E2|64-6|219||202|válido|
|scored_0422|E3|joint|219|individual 218|217|OPTIMAL|
|scored_0422|E4|repair|219||219|completo|
|scored_0423|E1|1|118|postprocess 117, optimize_a4 117, nested_windows 117|117|válido|
|scored_0423|E1|3|118|postprocess 117, optimize_a4 117, nested_windows 117|117|válido|
|scored_0423|E1|6|118|postprocess 117, optimize_a4 117, nested_windows 117|117|válido|
|scored_0423|E2|52-1|118||116|válido|
|scored_0423|E2|52-3|118||116|válido|
|scored_0423|E2|52-6|118||116|válido|
|scored_0423|E2|64-1|118||116|válido|
|scored_0423|E2|64-3|118||116|válido|
|scored_0423|E2|64-6|118||116|válido|
|scored_0423|E3|joint|118|individual 118|118|OPTIMAL|
|scored_0423|E4|repair|118||118|completo|
|scored_0424|E1|1|322|postprocess 322, optimize_a4 322, nested_windows 322|322|válido|
|scored_0424|E1|3|306|postprocess 306, optimize_a4 306, nested_windows 306|306|válido|
|scored_0424|E1|6|296|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0424|E2|52-1|322||289|válido|
|scored_0424|E2|52-3|306||278|válido|
|scored_0424|E2|52-6|296||275|válido|
|scored_0424|E2|64-1|322||289|válido|
|scored_0424|E2|64-3|306||278|válido|
|scored_0424|E2|64-6|296||293|válido|
|scored_0424|E3|joint|296|individual 296|296|FEASIBLE|
|scored_0424|E4|repair|296||296|completo|
|scored_0425|E1|1|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0425|E1|3|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0425|E1|6|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0425|E2|52-1|41||41|válido|
|scored_0425|E2|52-3|41||41|válido|
|scored_0425|E2|52-6|41||41|válido|
|scored_0425|E2|64-1|41||41|válido|
|scored_0425|E2|64-3|41||41|válido|
|scored_0425|E2|64-6|41||41|válido|
|scored_0425|E3|joint|41|individual 41|41|OPTIMAL|
|scored_0425|E4|repair|41||41|completo|
|scored_0426|E1|1|194|postprocess 194, optimize_a4 194, nested_windows 194|194|válido|
|scored_0426|E1|3|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0426|E1|6|184|postprocess 184, optimize_a4 184, nested_windows 184|184|válido|
|scored_0426|E2|52-1|194||178|válido|
|scored_0426|E2|52-3|188||175|válido|
|scored_0426|E2|52-6|184||170|válido|
|scored_0426|E2|64-1|194||178|válido|
|scored_0426|E2|64-3|188||173|válido|
|scored_0426|E2|64-6|184||171|válido|
|scored_0426|E3|joint|184|individual 184|184|OPTIMAL|
|scored_0426|E4|repair|184||184|completo|
|scored_0427|E1|1|209|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0427|E1|3|208|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0427|E1|6|194|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0427|E2|52-1|209||195|válido|
|scored_0427|E2|52-3|208||190|válido|
|scored_0427|E2|52-6|194||183|válido|
|scored_0427|E2|64-1|209||187|válido|
|scored_0427|E2|64-3|208||186|válido|
|scored_0427|E2|64-6|194||179|válido|
|scored_0427|E3|joint|194|individual 194|194|OPTIMAL|
|scored_0427|E4|repair|194||193|completo|
|scored_0428|E1|1|310|postprocess 305, optimize_a4 305, nested_windows 305|305|válido|
|scored_0428|E1|3|288|postprocess 288, optimize_a4 288, nested_windows 288|288|válido|
|scored_0428|E1|6|284|postprocess 284, optimize_a4 284, nested_windows 284|284|válido|
|scored_0428|E2|52-1|310||276|válido|
|scored_0428|E2|52-3|288||267|válido|
|scored_0428|E2|52-6|284||264|válido|
|scored_0428|E2|64-1|310||281|válido|
|scored_0428|E2|64-3|288||270|válido|
|scored_0428|E2|64-6|284||264|válido|
|scored_0428|E3|joint|284|individual 284|284|rechazado por registros|
|scored_0428|E4|repair|284||284|completo|
|scored_0429|E1|1|150|postprocess 150, optimize_a4 150, nested_windows 150|150|válido|
|scored_0429|E1|3|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0429|E1|6|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0429|E2|52-1|150||142|válido|
|scored_0429|E2|52-3|148||142|válido|
|scored_0429|E2|52-6|148||141|válido|
|scored_0429|E2|64-1|150||144|válido|
|scored_0429|E2|64-3|148||144|válido|
|scored_0429|E2|64-6|148||143|válido|
|scored_0429|E3|joint|148|individual 148|148|OPTIMAL|
|scored_0429|E4|repair|148||146|completo|
|scored_0430|E1|1|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|scored_0430|E1|3|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|scored_0430|E1|6|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|scored_0430|E2|52-1|113||111|válido|
|scored_0430|E2|52-3|113||111|válido|
|scored_0430|E2|52-6|113||111|válido|
|scored_0430|E2|64-1|113||111|válido|
|scored_0430|E2|64-3|113||111|válido|
|scored_0430|E2|64-6|113||111|válido|
|scored_0430|E3|joint|113|individual 113|113|OPTIMAL|
|scored_0430|E4|repair|113||110|completo|
|scored_0431|E1|1|110|postprocess 110, optimize_a4 110, nested_windows 110|110|válido|
|scored_0431|E1|3|110|postprocess 110, optimize_a4 110, nested_windows 110|110|válido|
|scored_0431|E1|6|110|postprocess 110, optimize_a4 110, nested_windows 110|110|válido|
|scored_0431|E2|52-1|110||107|válido|
|scored_0431|E2|52-3|110||107|válido|
|scored_0431|E2|52-6|110||107|válido|
|scored_0431|E2|64-1|110||107|válido|
|scored_0431|E2|64-3|110||107|válido|
|scored_0431|E2|64-6|110||107|válido|
|scored_0431|E3|joint|110|individual 110|110|OPTIMAL|
|scored_0431|E4|repair|110||108|completo|
|scored_0432|E1|1|187|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0432|E1|3|182|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0432|E1|6|179|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|scored_0432|E2|52-1|187||175|válido|
|scored_0432|E2|52-3|182||172|válido|
|scored_0432|E2|52-6|179||172|válido|
|scored_0432|E2|64-1|187||176|válido|
|scored_0432|E2|64-3|182||170|válido|
|scored_0432|E2|64-6|179||172|válido|
|scored_0432|E3|joint|179|individual 179|179|OPTIMAL|
|scored_0432|E4|repair|179||177|completo|
|scored_0433|E1|1|177|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0433|E1|3|173|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|scored_0433|E1|6|171|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|scored_0433|E2|52-1|177||166|válido|
|scored_0433|E2|52-3|173||165|válido|
|scored_0433|E2|52-6|171||164|válido|
|scored_0433|E2|64-1|177||165|válido|
|scored_0433|E2|64-3|173||164|válido|
|scored_0433|E2|64-6|171||163|válido|
|scored_0433|E3|joint|171|individual 171|171|OPTIMAL|
|scored_0433|E4|repair|171||171|completo|
|scored_0434|E1|1|141|postprocess 140, optimize_a4 140, nested_windows 140|140|válido|
|scored_0434|E1|3|140|postprocess 140, optimize_a4 140, nested_windows 140|140|válido|
|scored_0434|E1|6|140|postprocess 140, optimize_a4 140, nested_windows 140|140|válido|
|scored_0434|E2|52-1|141||137|válido|
|scored_0434|E2|52-3|140||137|válido|
|scored_0434|E2|52-6|140||137|válido|
|scored_0434|E2|64-1|141||136|válido|
|scored_0434|E2|64-3|140||136|válido|
|scored_0434|E2|64-6|140||136|válido|
|scored_0434|E3|joint|140|individual 140|140|OPTIMAL|
|scored_0434|E4|repair|140||139|completo|
|scored_0435|E1|1|205|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|scored_0435|E1|3|205|postprocess 205, optimize_a4 205, nested_windows 205|205|válido|
|scored_0435|E1|6|203|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0435|E2|52-1|205||203|válido|
|scored_0435|E2|52-3|205||194|válido|
|scored_0435|E2|52-6|203||191|válido|
|scored_0435|E2|64-1|205||202|válido|
|scored_0435|E2|64-3|205||195|válido|
|scored_0435|E2|64-6|203||193|válido|
|scored_0435|E3|joint|203|individual 202|201|OPTIMAL|
|scored_0435|E4|repair|203||202|completo|
|scored_0436|E1|1|210|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|scored_0436|E1|3|196|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|scored_0436|E1|6|193|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0436|E2|52-1|210||181|válido|
|scored_0436|E2|52-3|196||175|válido|
|scored_0436|E2|52-6|193||181|válido|
|scored_0436|E2|64-1|210||176|válido|
|scored_0436|E2|64-3|196||177|válido|
|scored_0436|E2|64-6|193||178|válido|
|scored_0436|E3|joint|193|individual 193|193|OPTIMAL|
|scored_0436|E4|repair|193||191|completo|
|scored_0437|E1|1|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0437|E1|3|216|postprocess 216, optimize_a4 216, nested_windows 216|216|válido|
|scored_0437|E1|6|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0437|E2|52-1|232||216|válido|
|scored_0437|E2|52-3|216||211|válido|
|scored_0437|E2|52-6|219||209|válido|
|scored_0437|E2|64-1|232||216|válido|
|scored_0437|E2|64-3|216||210|válido|
|scored_0437|E2|64-6|219||209|válido|
|scored_0437|E3|joint|216|individual 216|216|OPTIMAL|
|scored_0437|E4|repair|216||216|completo|
|scored_0438|E1|1|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|scored_0438|E1|3|274|postprocess 274, optimize_a4 274, nested_windows 274|274|válido|
|scored_0438|E1|6|266|postprocess 266, optimize_a4 266, nested_windows 266|266|válido|
|scored_0438|E2|52-1|289||264|válido|
|scored_0438|E2|52-3|274||253|válido|
|scored_0438|E2|52-6|266||245|válido|
|scored_0438|E2|64-1|289||264|válido|
|scored_0438|E2|64-3|274||256|válido|
|scored_0438|E2|64-6|266||247|válido|
|scored_0438|E3|joint|266|individual 266|264|OPTIMAL|
|scored_0438|E4|repair|266||266|completo|
|scored_0439|E1|1|211|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0439|E1|3|213|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0439|E1|6|211|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0439|E2|52-1|211||204|válido|
|scored_0439|E2|52-3|213||197|válido|
|scored_0439|E2|52-6|211||198|válido|
|scored_0439|E2|64-1|211||205|válido|
|scored_0439|E2|64-3|213||199|válido|
|scored_0439|E2|64-6|211||194|válido|
|scored_0439|E3|joint|211|individual 210|210|OPTIMAL|
|scored_0439|E4|repair|211||210|completo|
|scored_0440|E1|1|258|postprocess 257, optimize_a4 257, nested_windows 257|257|válido|
|scored_0440|E1|3|258|postprocess 258, optimize_a4 258, nested_windows 258|258|válido|
|scored_0440|E1|6|255|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|scored_0440|E2|52-1|258||244|válido|
|scored_0440|E2|52-3|258||239|válido|
|scored_0440|E2|52-6|255||237|válido|
|scored_0440|E2|64-1|258||244|válido|
|scored_0440|E2|64-3|258||239|válido|
|scored_0440|E2|64-6|255||238|válido|
|scored_0440|E3|joint|255|individual 255|254|OPTIMAL|
|scored_0440|E4|repair|255||255|completo|
|scored_0441|E1|1|259|postprocess 259, optimize_a4 259, nested_windows 259|259|válido|
|scored_0441|E1|3|250|postprocess 250, optimize_a4 250, nested_windows 250|250|válido|
|scored_0441|E1|6|241|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|scored_0441|E2|52-1|259||235|válido|
|scored_0441|E2|52-3|250||234|válido|
|scored_0441|E2|52-6|241||226|válido|
|scored_0441|E2|64-1|259||236|válido|
|scored_0441|E2|64-3|250||233|válido|
|scored_0441|E2|64-6|241||226|válido|
|scored_0441|E3|joint|241|individual 241|239|OPTIMAL|
|scored_0441|E4|repair|241||241|completo|
|scored_0442|E1|1|251|postprocess 251, optimize_a4 251, nested_windows 251|251|válido|
|scored_0442|E1|3|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0442|E1|6|234|postprocess 234, optimize_a4 234, nested_windows 234|234|válido|
|scored_0442|E2|52-1|251||230|válido|
|scored_0442|E2|52-3|239||220|válido|
|scored_0442|E2|52-6|234||220|válido|
|scored_0442|E2|64-1|251||231|válido|
|scored_0442|E2|64-3|239||222|válido|
|scored_0442|E2|64-6|234||219|válido|
|scored_0442|E3|joint|234|individual 234|230|OPTIMAL|
|scored_0442|E4|repair|234||233|completo|
|scored_0443|E1|1|45|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0443|E1|3|45|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0443|E1|6|45|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0443|E2|52-1|45||45|válido|
|scored_0443|E2|52-3|45||45|válido|
|scored_0443|E2|52-6|45||45|válido|
|scored_0443|E2|64-1|45||45|válido|
|scored_0443|E2|64-3|45||45|válido|
|scored_0443|E2|64-6|45||45|válido|
|scored_0443|E3|joint|45|individual 45|45|OPTIMAL|
|scored_0443|E4|repair|45||45|completo|
|scored_0444|E1|1|337|postprocess 337, optimize_a4 337, nested_windows 337|337|válido|
|scored_0444|E1|3|326|postprocess 326, optimize_a4 326, nested_windows 326|326|válido|
|scored_0444|E1|6|323|postprocess 323, optimize_a4 323, nested_windows 323|323|válido|
|scored_0444|E2|52-1|337||313|válido|
|scored_0444|E2|52-3|326||303|válido|
|scored_0444|E2|52-6|323||298|válido|
|scored_0444|E2|64-1|337||314|válido|
|scored_0444|E2|64-3|326||309|válido|
|scored_0444|E2|64-6|323||300|válido|
|scored_0444|E3|joint|323|individual 323|319|FEASIBLE|
|scored_0444|E4|repair|323||323|completo|
|scored_0445|E1|1|325|postprocess 324, optimize_a4 324, nested_windows 324|324|válido|
|scored_0445|E1|3|315|postprocess 314, optimize_a4 314, nested_windows 314|314|válido|
|scored_0445|E1|6|311|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|scored_0445|E2|52-1|325||308|válido|
|scored_0445|E2|52-3|315||289|válido|
|scored_0445|E2|52-6|311||292|válido|
|scored_0445|E2|64-1|325||307|válido|
|scored_0445|E2|64-3|315||293|válido|
|scored_0445|E2|64-6|311||292|válido|
|scored_0445|E3|joint|311|individual 311|310|OPTIMAL|
|scored_0445|E4|repair|311||311|completo|
|scored_0446|E1|1|348|postprocess 347, optimize_a4 347, nested_windows 347|347|válido|
|scored_0446|E1|3|338|postprocess 338, optimize_a4 338, nested_windows 338|338|válido|
|scored_0446|E1|6|329|postprocess 329, optimize_a4 329, nested_windows 329|329|válido|
|scored_0446|E2|52-1|348||327|válido|
|scored_0446|E2|52-3|338||314|válido|
|scored_0446|E2|52-6|329||310|válido|
|scored_0446|E2|64-1|348||327|válido|
|scored_0446|E2|64-3|338||315|válido|
|scored_0446|E2|64-6|329||310|válido|
|scored_0446|E3|joint|329|individual 327|327|FEASIBLE|
|scored_0446|E4|repair|329||329|completo|
|scored_0447|E1|1|318|postprocess 317, optimize_a4 317, nested_windows 317|317|válido|
|scored_0447|E1|3|299|postprocess 299, optimize_a4 299, nested_windows 299|299|válido|
|scored_0447|E1|6|298|postprocess 298, optimize_a4 298, nested_windows 298|298|válido|
|scored_0447|E2|52-1|318||287|válido|
|scored_0447|E2|52-3|299||274|válido|
|scored_0447|E2|52-6|298||272|válido|
|scored_0447|E2|64-1|318||290|válido|
|scored_0447|E2|64-3|299||281|válido|
|scored_0447|E2|64-6|298||277|válido|
|scored_0447|E3|joint|298|individual 298|297|OPTIMAL|
|scored_0447|E4|repair|298||298|completo|
|scored_0448|E1|1|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|scored_0448|E1|3|277|postprocess 277, optimize_a4 277, nested_windows 277|277|válido|
|scored_0448|E1|6|274|postprocess 274, optimize_a4 274, nested_windows 274|274|válido|
|scored_0448|E2|52-1|289||269|válido|
|scored_0448|E2|52-3|277||260|válido|
|scored_0448|E2|52-6|274||260|válido|
|scored_0448|E2|64-1|289||269|válido|
|scored_0448|E2|64-3|277||261|válido|
|scored_0448|E2|64-6|274||260|válido|
|scored_0448|E3|joint|274|individual 274|274|OPTIMAL|
|scored_0448|E4|repair|274||274|completo|
|scored_0449|E1|1|128|postprocess 128, optimize_a4 128, nested_windows 128|128|válido|
|scored_0449|E1|3|127|postprocess 127, optimize_a4 127, nested_windows 127|127|válido|
|scored_0449|E1|6|127|postprocess 127, optimize_a4 127, nested_windows 127|127|válido|
|scored_0449|E2|52-1|128||126|válido|
|scored_0449|E2|52-3|127||125|válido|
|scored_0449|E2|52-6|127||125|válido|
|scored_0449|E2|64-1|128||125|válido|
|scored_0449|E2|64-3|127||125|válido|
|scored_0449|E2|64-6|127||125|válido|
|scored_0449|E3|joint|127|individual 127|127|OPTIMAL|
|scored_0449|E4|repair|127||124|completo|
|scored_0450|E1|1|384|postprocess 383, optimize_a4 383, nested_windows 383|383|válido|
|scored_0450|E1|3|372|postprocess 372, optimize_a4 372, nested_windows 372|372|válido|
|scored_0450|E1|6|360|postprocess 360, optimize_a4 360, nested_windows 360|360|válido|
|scored_0450|E2|52-1|384||360|válido|
|scored_0450|E2|52-3|372||352|válido|
|scored_0450|E2|52-6|360||342|válido|
|scored_0450|E2|64-1|384||366|válido|
|scored_0450|E2|64-3|372||350|válido|
|scored_0450|E2|64-6|360||346|válido|
|scored_0450|E3|joint|360|individual 360|360|FEASIBLE|
|scored_0450|E4|repair|360||360|completo|
|scored_0451|E1|1|155|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0451|E1|3|154|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0451|E1|6|152|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0451|E2|52-1|155||149|válido|
|scored_0451|E2|52-3|154||148|válido|
|scored_0451|E2|52-6|152||148|válido|
|scored_0451|E2|64-1|155||149|válido|
|scored_0451|E2|64-3|154||148|válido|
|scored_0451|E2|64-6|152||146|válido|
|scored_0451|E3|joint|152|individual 152|152|OPTIMAL|
|scored_0451|E4|repair|152||149|completo|
|scored_0452|E1|1|294|postprocess 293, optimize_a4 293, nested_windows 293|293|válido|
|scored_0452|E1|3|277|postprocess 277, optimize_a4 277, nested_windows 277|277|válido|
|scored_0452|E1|6|269|postprocess 269, optimize_a4 269, nested_windows 269|269|válido|
|scored_0452|E2|52-1|294||271|válido|
|scored_0452|E2|52-3|277||253|válido|
|scored_0452|E2|52-6|269||254|válido|
|scored_0452|E2|64-1|294||272|válido|
|scored_0452|E2|64-3|277||253|válido|
|scored_0452|E2|64-6|269||256|válido|
|scored_0452|E3|joint|269|individual 269|269|OPTIMAL|
|scored_0452|E4|repair|269||269|completo|
|scored_0453|E1|1|200|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0453|E1|3|194|postprocess 194, optimize_a4 194, nested_windows 194|194|válido|
|scored_0453|E1|6|198|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|scored_0453|E2|52-1|200||190|válido|
|scored_0453|E2|52-3|194||185|válido|
|scored_0453|E2|52-6|198||185|válido|
|scored_0453|E2|64-1|200||186|válido|
|scored_0453|E2|64-3|194||184|válido|
|scored_0453|E2|64-6|198||184|válido|
|scored_0453|E3|joint|194|individual 194|193|OPTIMAL|
|scored_0453|E4|repair|194||192|completo|
|scored_0454|E1|1|292|postprocess 291, optimize_a4 291, nested_windows 291|291|válido|
|scored_0454|E1|3|275|postprocess 275, optimize_a4 275, nested_windows 275|275|válido|
|scored_0454|E1|6|275|postprocess 274, optimize_a4 274, nested_windows 274|274|válido|
|scored_0454|E2|52-1|292||271|válido|
|scored_0454|E2|52-3|275||261|válido|
|scored_0454|E2|52-6|275||258|válido|
|scored_0454|E2|64-1|292||272|válido|
|scored_0454|E2|64-3|275||261|válido|
|scored_0454|E2|64-6|275||258|válido|
|scored_0454|E3|joint|275|individual 275|274|OPTIMAL|
|scored_0454|E4|repair|275||275|completo|
|scored_0455|E1|1|39|postprocess 37, optimize_a4 37, nested_windows 37|37|válido|
|scored_0455|E1|3|39|postprocess 37, optimize_a4 37, nested_windows 37|37|válido|
|scored_0455|E1|6|39|postprocess 37, optimize_a4 37, nested_windows 37|37|válido|
|scored_0455|E2|52-1|39||39|válido|
|scored_0455|E2|52-3|39||39|válido|
|scored_0455|E2|52-6|39||39|válido|
|scored_0455|E2|64-1|39||39|válido|
|scored_0455|E2|64-3|39||39|válido|
|scored_0455|E2|64-6|39||39|válido|
|scored_0455|E3|joint|39|individual 39|39|OPTIMAL|
|scored_0455|E4|repair|39||39|completo|
|scored_0456|E1|1|157|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0456|E1|3|156|postprocess 156, optimize_a4 155, nested_windows 155|155|válido|
|scored_0456|E1|6|156|postprocess 156, optimize_a4 155, nested_windows 155|155|válido|
|scored_0456|E2|52-1|157||149|válido|
|scored_0456|E2|52-3|156||149|válido|
|scored_0456|E2|52-6|156||149|válido|
|scored_0456|E2|64-1|157||150|válido|
|scored_0456|E2|64-3|156||148|válido|
|scored_0456|E2|64-6|156||148|válido|
|scored_0456|E3|joint|156|individual 156|156|OPTIMAL|
|scored_0456|E4|repair|156||154|completo|
|scored_0457|E1|1|191|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|scored_0457|E1|3|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0457|E1|6|186|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0457|E2|52-1|191||181|válido|
|scored_0457|E2|52-3|188||176|válido|
|scored_0457|E2|52-6|186||175|válido|
|scored_0457|E2|64-1|191||180|válido|
|scored_0457|E2|64-3|188||176|válido|
|scored_0457|E2|64-6|186||176|válido|
|scored_0457|E3|joint|186|individual 186|185|OPTIMAL|
|scored_0457|E4|repair|186||185|completo|
|scored_0458|E1|1|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0458|E1|3|204|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|scored_0458|E1|6|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|scored_0458|E2|52-1|211||194|válido|
|scored_0458|E2|52-3|204||190|válido|
|scored_0458|E2|52-6|197||189|válido|
|scored_0458|E2|64-1|211||196|válido|
|scored_0458|E2|64-3|204||190|válido|
|scored_0458|E2|64-6|197||188|válido|
|scored_0458|E3|joint|197|individual 197|197|OPTIMAL|
|scored_0458|E4|repair|197||197|completo|
|scored_0459|E1|1|115|postprocess 114, optimize_a4 114, nested_windows 114|114|válido|
|scored_0459|E1|3|115|postprocess 114, optimize_a4 114, nested_windows 114|114|válido|
|scored_0459|E1|6|115|postprocess 114, optimize_a4 114, nested_windows 114|114|válido|
|scored_0459|E2|52-1|115||109|válido|
|scored_0459|E2|52-3|115||109|válido|
|scored_0459|E2|52-6|115||109|válido|
|scored_0459|E2|64-1|115||109|válido|
|scored_0459|E2|64-3|115||109|válido|
|scored_0459|E2|64-6|115||109|válido|
|scored_0459|E3|joint|115|individual 115|115|OPTIMAL|
|scored_0459|E4|repair|115||112|completo|
|scored_0460|E1|1|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0460|E1|3|217|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0460|E1|6|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0460|E2|52-1|220||211|válido|
|scored_0460|E2|52-3|217||205|válido|
|scored_0460|E2|52-6|211||202|válido|
|scored_0460|E2|64-1|220||212|válido|
|scored_0460|E2|64-3|217||206|válido|
|scored_0460|E2|64-6|211||204|válido|
|scored_0460|E3|joint|211|individual 211|211|OPTIMAL|
|scored_0460|E4|repair|211||211|completo|
|scored_0461|E1|1|46|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0461|E1|3|46|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0461|E1|6|46|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0461|E2|52-1|46||46|válido|
|scored_0461|E2|52-3|46||46|válido|
|scored_0461|E2|52-6|46||46|válido|
|scored_0461|E2|64-1|46||46|válido|
|scored_0461|E2|64-3|46||46|válido|
|scored_0461|E2|64-6|46||46|válido|
|scored_0461|E3|joint|46|individual 46|46|OPTIMAL|
|scored_0461|E4|repair|46||45|completo|
|scored_0462|E1|1|155|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0462|E1|3|155|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0462|E1|6|152|postprocess 150, optimize_a4 150, nested_windows 150|150|válido|
|scored_0462|E2|52-1|155||147|válido|
|scored_0462|E2|52-3|155||147|válido|
|scored_0462|E2|52-6|152||146|válido|
|scored_0462|E2|64-1|155||146|válido|
|scored_0462|E2|64-3|155||146|válido|
|scored_0462|E2|64-6|152||145|válido|
|scored_0462|E3|joint|152|individual 152|151|OPTIMAL|
|scored_0462|E4|repair|152||150|completo|
|scored_0463|E1|1|139|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0463|E1|3|139|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0463|E1|6|139|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0463|E2|52-1|139||134|válido|
|scored_0463|E2|52-3|139||134|válido|
|scored_0463|E2|52-6|139||133|válido|
|scored_0463|E2|64-1|139||135|válido|
|scored_0463|E2|64-3|139||135|válido|
|scored_0463|E2|64-6|139||135|válido|
|scored_0463|E3|joint|139|individual 139|139|OPTIMAL|
|scored_0463|E4|repair|139||135|completo|
|scored_0464|E1|1|105|postprocess 103, optimize_a4 103, nested_windows 103|103|válido|
|scored_0464|E1|3|105|postprocess 103, optimize_a4 103, nested_windows 103|103|válido|
|scored_0464|E1|6|104|postprocess 102, optimize_a4 102, nested_windows 102|102|válido|
|scored_0464|E2|52-1|105||102|válido|
|scored_0464|E2|52-3|105||102|válido|
|scored_0464|E2|52-6|104||102|válido|
|scored_0464|E2|64-1|105||102|válido|
|scored_0464|E2|64-3|105||102|válido|
|scored_0464|E2|64-6|104||102|válido|
|scored_0464|E3|joint|104|individual 104|104|OPTIMAL|
|scored_0464|E4|repair|104||102|completo|
|scored_0465|E1|1|206|postprocess 205, optimize_a4 205, nested_windows 205|205|válido|
|scored_0465|E1|3|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|scored_0465|E1|6|195|postprocess 195, optimize_a4 195, nested_windows 195|195|válido|
|scored_0465|E2|52-1|206||188|válido|
|scored_0465|E2|52-3|197||187|válido|
|scored_0465|E2|52-6|195||178|válido|
|scored_0465|E2|64-1|206||189|válido|
|scored_0465|E2|64-3|197||187|válido|
|scored_0465|E2|64-6|195||181|válido|
|scored_0465|E3|joint|195|individual 195|194|OPTIMAL|
|scored_0465|E4|repair|195||193|completo|
|scored_0466|E1|1|264|postprocess 264, optimize_a4 264, nested_windows 264|264|válido|
|scored_0466|E1|3|260|postprocess 260, optimize_a4 260, nested_windows 260|260|válido|
|scored_0466|E1|6|253|postprocess 252, optimize_a4 252, nested_windows 252|252|válido|
|scored_0466|E2|52-1|264||237|válido|
|scored_0466|E2|52-3|260||231|válido|
|scored_0466|E2|52-6|253||231|válido|
|scored_0466|E2|64-1|264||240|válido|
|scored_0466|E2|64-3|260||232|válido|
|scored_0466|E2|64-6|253||231|válido|
|scored_0466|E3|joint|253|individual 252|252|OPTIMAL|
|scored_0466|E4|repair|253||253|completo|
|scored_0467|E1|1|130|postprocess 130, optimize_a4 130, nested_windows 130|130|válido|
|scored_0467|E1|3|129|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|scored_0467|E1|6|129|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|scored_0467|E2|52-1|130||123|válido|
|scored_0467|E2|52-3|129||122|válido|
|scored_0467|E2|52-6|129||122|válido|
|scored_0467|E2|64-1|130||123|válido|
|scored_0467|E2|64-3|129||123|válido|
|scored_0467|E2|64-6|129||123|válido|
|scored_0467|E3|joint|129|individual 129|129|OPTIMAL|
|scored_0467|E4|repair|129||124|completo|
|scored_0468|E1|1|215|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0468|E1|3|209|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0468|E1|6|208|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|scored_0468|E2|52-1|215||200|válido|
|scored_0468|E2|52-3|209||194|válido|
|scored_0468|E2|52-6|208||193|válido|
|scored_0468|E2|64-1|215||199|válido|
|scored_0468|E2|64-3|209||195|válido|
|scored_0468|E2|64-6|208||193|válido|
|scored_0468|E3|joint|208|individual 208|208|OPTIMAL|
|scored_0468|E4|repair|208||208|completo|
|scored_0469|E1|1|111|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0469|E1|3|111|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0469|E1|6|111|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0469|E2|52-1|111||111|válido|
|scored_0469|E2|52-3|111||111|válido|
|scored_0469|E2|52-6|111||111|válido|
|scored_0469|E2|64-1|111||111|válido|
|scored_0469|E2|64-3|111||111|válido|
|scored_0469|E2|64-6|111||111|válido|
|scored_0469|E3|joint|111|individual 111|111|OPTIMAL|
|scored_0469|E4|repair|111||107|completo|
|scored_0470|E1|1|374|postprocess 374, optimize_a4 374, nested_windows 374|374|válido|
|scored_0470|E1|3|346|postprocess 346, optimize_a4 346, nested_windows 346|346|válido|
|scored_0470|E1|6|347|postprocess 347, optimize_a4 347, nested_windows 347|347|válido|
|scored_0470|E2|52-1|374||341|válido|
|scored_0470|E2|52-3|346||329|válido|
|scored_0470|E2|52-6|347||327|válido|
|scored_0470|E2|64-1|374||344|válido|
|scored_0470|E2|64-3|346||589|válido|
|scored_0470|E2|64-6|347||430|válido|
|scored_0470|E3|joint|346|individual 345|344|FEASIBLE|
|scored_0470|E4|repair|346||346|completo|
|scored_0471|E1|1|133|postprocess 133, optimize_a4 133, nested_windows 133|133|válido|
|scored_0471|E1|3|132|postprocess 132, optimize_a4 132, nested_windows 132|132|válido|
|scored_0471|E1|6|132|postprocess 132, optimize_a4 132, nested_windows 132|132|válido|
|scored_0471|E2|52-1|133||126|válido|
|scored_0471|E2|52-3|132||126|válido|
|scored_0471|E2|52-6|132||126|válido|
|scored_0471|E2|64-1|133||126|válido|
|scored_0471|E2|64-3|132||126|válido|
|scored_0471|E2|64-6|132||126|válido|
|scored_0471|E3|joint|132|individual 132|132|OPTIMAL|
|scored_0471|E4|repair|132||130|completo|
|scored_0472|E1|1|33|postprocess 30, optimize_a4 30, nested_windows 30|30|válido|
|scored_0472|E1|3|33|postprocess 30, optimize_a4 30, nested_windows 30|30|válido|
|scored_0472|E1|6|33|postprocess 30, optimize_a4 30, nested_windows 30|30|válido|
|scored_0472|E2|52-1|33||33|válido|
|scored_0472|E2|52-3|33||33|válido|
|scored_0472|E2|52-6|33||33|válido|
|scored_0472|E2|64-1|33||33|válido|
|scored_0472|E2|64-3|33||33|válido|
|scored_0472|E2|64-6|33||33|válido|
|scored_0472|E3|joint|33|individual 33|33|OPTIMAL|
|scored_0472|E4|repair|33||32|completo|
|scored_0473|E1|1|39|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0473|E1|3|39|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0473|E1|6|39|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0473|E2|52-1|39||39|válido|
|scored_0473|E2|52-3|39||39|válido|
|scored_0473|E2|52-6|39||39|válido|
|scored_0473|E2|64-1|39||39|válido|
|scored_0473|E2|64-3|39||39|válido|
|scored_0473|E2|64-6|39||39|válido|
|scored_0473|E3|joint|39|individual 39|39|OPTIMAL|
|scored_0473|E4|repair|39||37|completo|
|scored_0474|E1|1|111|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0474|E1|3|111|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0474|E1|6|111|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0474|E2|52-1|111||109|válido|
|scored_0474|E2|52-3|111||109|válido|
|scored_0474|E2|52-6|111||109|válido|
|scored_0474|E2|64-1|111||109|válido|
|scored_0474|E2|64-3|111||109|válido|
|scored_0474|E2|64-6|111||109|válido|
|scored_0474|E3|joint|111|individual 111|111|OPTIMAL|
|scored_0474|E4|repair|111||108|completo|
|scored_0475|E1|1|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|scored_0475|E1|3|278|postprocess 278, optimize_a4 278, nested_windows 278|278|válido|
|scored_0475|E1|6|264|postprocess 264, optimize_a4 264, nested_windows 264|264|válido|
|scored_0475|E2|52-1|289||267|válido|
|scored_0475|E2|52-3|278||261|válido|
|scored_0475|E2|52-6|264||248|válido|
|scored_0475|E2|64-1|289||267|válido|
|scored_0475|E2|64-3|278||258|válido|
|scored_0475|E2|64-6|264||248|válido|
|scored_0475|E3|joint|264|individual 264|263|OPTIMAL|
|scored_0475|E4|repair|264||264|completo|
|scored_0476|E1|1|303|postprocess 302, optimize_a4 302, nested_windows 302|302|válido|
|scored_0476|E1|3|301|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|scored_0476|E1|6|285|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|scored_0476|E2|52-1|303||276|válido|
|scored_0476|E2|52-3|301||268|válido|
|scored_0476|E2|52-6|285||279|válido|
|scored_0476|E2|64-1|303||277|válido|
|scored_0476|E2|64-3|301||271|válido|
|scored_0476|E2|64-6|285||262|válido|
|scored_0476|E3|joint|285|individual 285|284|OPTIMAL|
|scored_0476|E4|repair|285||285|completo|
|scored_0477|E1|1|235|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0477|E1|3|228|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0477|E1|6|217|postprocess 216, optimize_a4 216, nested_windows 216|216|válido|
|scored_0477|E2|52-1|235||215|válido|
|scored_0477|E2|52-3|228||206|válido|
|scored_0477|E2|52-6|217||209|válido|
|scored_0477|E2|64-1|235||212|válido|
|scored_0477|E2|64-3|228||206|válido|
|scored_0477|E2|64-6|217||211|válido|
|scored_0477|E3|joint|217|individual 217|216|OPTIMAL|
|scored_0477|E4|repair|217||217|completo|
|scored_0478|E1|1|109|postprocess 108, optimize_a4 108, nested_windows 108|108|válido|
|scored_0478|E1|3|110|postprocess 108, optimize_a4 108, nested_windows 108|108|válido|
|scored_0478|E1|6|108|postprocess 108, optimize_a4 108, nested_windows 108|108|válido|
|scored_0478|E2|52-1|109||107|válido|
|scored_0478|E2|52-3|110||102|válido|
|scored_0478|E2|52-6|108||99|válido|
|scored_0478|E2|64-1|109||107|válido|
|scored_0478|E2|64-3|110||102|válido|
|scored_0478|E2|64-6|108||99|válido|
|scored_0478|E3|joint|108|individual 108|108|OPTIMAL|
|scored_0478|E4|repair|108||108|completo|
|scored_0479|E1|1|158|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0479|E1|3|155|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|scored_0479|E1|6|155|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|scored_0479|E2|52-1|158||147|válido|
|scored_0479|E2|52-3|155||148|válido|
|scored_0479|E2|52-6|155||147|válido|
|scored_0479|E2|64-1|158||146|válido|
|scored_0479|E2|64-3|155||144|válido|
|scored_0479|E2|64-6|155||144|válido|
|scored_0479|E3|joint|155|individual 155|155|OPTIMAL|
|scored_0479|E4|repair|155||152|completo|
|scored_0480|E1|1|206|postprocess 205, optimize_a4 205, nested_windows 205|205|válido|
|scored_0480|E1|3|202|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0480|E1|6|199|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|scored_0480|E2|52-1|206||189|válido|
|scored_0480|E2|52-3|202||184|válido|
|scored_0480|E2|52-6|199||185|válido|
|scored_0480|E2|64-1|206||190|válido|
|scored_0480|E2|64-3|202||185|válido|
|scored_0480|E2|64-6|199||183|válido|
|scored_0480|E3|joint|199|individual 199|199|OPTIMAL|
|scored_0480|E4|repair|199||198|completo|
|scored_0481|E1|1|244|postprocess 244, optimize_a4 244, nested_windows 244|244|válido|
|scored_0481|E1|3|243|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|scored_0481|E1|6|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0481|E2|52-1|244||233|válido|
|scored_0481|E2|52-3|243||226|válido|
|scored_0481|E2|52-6|242||224|válido|
|scored_0481|E2|64-1|244||233|válido|
|scored_0481|E2|64-3|243||229|válido|
|scored_0481|E2|64-6|242||228|válido|
|scored_0481|E3|joint|242|individual 242|242|OPTIMAL|
|scored_0481|E4|repair|242||242|completo|
|scored_0482|E1|1|262|postprocess 261, optimize_a4 261, nested_windows 261|261|válido|
|scored_0482|E1|3|255|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|scored_0482|E1|6|247|postprocess 247, optimize_a4 247, nested_windows 247|247|válido|
|scored_0482|E2|52-1|262||239|válido|
|scored_0482|E2|52-3|255||238|válido|
|scored_0482|E2|52-6|247||235|válido|
|scored_0482|E2|64-1|262||240|válido|
|scored_0482|E2|64-3|255||238|válido|
|scored_0482|E2|64-6|247||236|válido|
|scored_0482|E3|joint|247|individual 247|246|OPTIMAL|
|scored_0482|E4|repair|247||246|completo|
|scored_0483|E1|1|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0483|E1|3|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0483|E1|6|221|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0483|E2|52-1|232||220|válido|
|scored_0483|E2|52-3|223||209|válido|
|scored_0483|E2|52-6|221||210|válido|
|scored_0483|E2|64-1|232||223|válido|
|scored_0483|E2|64-3|223||209|válido|
|scored_0483|E2|64-6|221||210|válido|
|scored_0483|E3|joint|221|individual 221|221|OPTIMAL|
|scored_0483|E4|repair|221||217|completo|
|scored_0484|E1|1|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0484|E1|3|190|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0484|E1|6|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0484|E2|52-1|192||180|válido|
|scored_0484|E2|52-3|190||176|válido|
|scored_0484|E2|52-6|188||176|válido|
|scored_0484|E2|64-1|192||180|válido|
|scored_0484|E2|64-3|190||176|válido|
|scored_0484|E2|64-6|188||176|válido|
|scored_0484|E3|joint|188|individual 188|187|OPTIMAL|
|scored_0484|E4|repair|188||188|completo|
|scored_0485|E1|1|2|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|scored_0485|E1|3|2|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|scored_0485|E1|6|2|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|scored_0485|E2|52-1|2||2|válido|
|scored_0485|E2|52-3|2||2|válido|
|scored_0485|E2|52-6|2||2|válido|
|scored_0485|E2|64-1|2||2|válido|
|scored_0485|E2|64-3|2||2|válido|
|scored_0485|E2|64-6|2||2|válido|
|scored_0485|E3|joint|2|individual 2|2|OPTIMAL|
|scored_0485|E4|repair|2||2|completo|
|scored_0486|E1|1|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0486|E1|3|147|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0486|E1|6|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0486|E2|52-1|148||144|válido|
|scored_0486|E2|52-3|147||142|válido|
|scored_0486|E2|52-6|145||144|válido|
|scored_0486|E2|64-1|148||140|válido|
|scored_0486|E2|64-3|147||140|válido|
|scored_0486|E2|64-6|145||140|válido|
|scored_0486|E3|joint|145|individual 145|144|OPTIMAL|
|scored_0486|E4|repair|145||145|completo|
|scored_0487|E1|1|44|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0487|E1|3|44|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0487|E1|6|44|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0487|E2|52-1|44||44|válido|
|scored_0487|E2|52-3|44||44|válido|
|scored_0487|E2|52-6|44||44|válido|
|scored_0487|E2|64-1|44||44|válido|
|scored_0487|E2|64-3|44||44|válido|
|scored_0487|E2|64-6|44||44|válido|
|scored_0487|E3|joint|44|individual 44|44|OPTIMAL|
|scored_0487|E4|repair|44||42|completo|
|scored_0488|E1|1|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0488|E1|3|193|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0488|E1|6|189|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0488|E2|52-1|199||180|válido|
|scored_0488|E2|52-3|193||183|válido|
|scored_0488|E2|52-6|189||179|válido|
|scored_0488|E2|64-1|199||184|válido|
|scored_0488|E2|64-3|193||179|válido|
|scored_0488|E2|64-6|189||177|válido|
|scored_0488|E3|joint|189|individual 189|189|OPTIMAL|
|scored_0488|E4|repair|189||189|completo|
|scored_0489|E1|1|195|postprocess 194, optimize_a4 194, nested_windows 194|194|válido|
|scored_0489|E1|3|191|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|scored_0489|E1|6|191|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|scored_0489|E2|52-1|195||183|válido|
|scored_0489|E2|52-3|191||182|válido|
|scored_0489|E2|52-6|191||180|válido|
|scored_0489|E2|64-1|195||180|válido|
|scored_0489|E2|64-3|191||180|válido|
|scored_0489|E2|64-6|191||179|válido|
|scored_0489|E3|joint|191|individual 191|190|OPTIMAL|
|scored_0489|E4|repair|191||190|completo|
|scored_0490|E1|1|6|postprocess 5, optimize_a4 4, nested_windows 4|4|válido|
|scored_0490|E1|3|6|postprocess 5, optimize_a4 4, nested_windows 4|4|válido|
|scored_0490|E1|6|6|postprocess 5, optimize_a4 4, nested_windows 4|4|válido|
|scored_0490|E2|52-1|6||6|válido|
|scored_0490|E2|52-3|6||6|válido|
|scored_0490|E2|52-6|6||6|válido|
|scored_0490|E2|64-1|6||6|válido|
|scored_0490|E2|64-3|6||6|válido|
|scored_0490|E2|64-6|6||6|válido|
|scored_0490|E3|joint|4|individual 4|4|OPTIMAL|
|scored_0490|E4|repair|4||4|completo|
|scored_0491|E1|1|269|postprocess 268, optimize_a4 268, nested_windows 268|268|válido|
|scored_0491|E1|3|258|postprocess 258, optimize_a4 258, nested_windows 258|258|válido|
|scored_0491|E1|6|255|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|scored_0491|E2|52-1|269||242|válido|
|scored_0491|E2|52-3|258||238|válido|
|scored_0491|E2|52-6|255||236|válido|
|scored_0491|E2|64-1|269||242|válido|
|scored_0491|E2|64-3|258||237|válido|
|scored_0491|E2|64-6|255||241|válido|
|scored_0491|E3|joint|255|individual 255|254|OPTIMAL|
|scored_0491|E4|repair|255||255|completo|
|scored_0492|E1|1|52|postprocess 51, optimize_a4 51, nested_windows 51|51|válido|
|scored_0492|E1|3|52|postprocess 51, optimize_a4 51, nested_windows 51|51|válido|
|scored_0492|E1|6|52|postprocess 51, optimize_a4 51, nested_windows 51|51|válido|
|scored_0492|E2|52-1|52||52|válido|
|scored_0492|E2|52-3|52||52|válido|
|scored_0492|E2|52-6|52||52|válido|
|scored_0492|E2|64-1|52||52|válido|
|scored_0492|E2|64-3|52||52|válido|
|scored_0492|E2|64-6|52||52|válido|
|scored_0492|E3|joint|52|individual 52|52|OPTIMAL|
|scored_0492|E4|repair|52||52|completo|
|scored_0493|E1|1|342|postprocess 341, optimize_a4 341, nested_windows 341|341|válido|
|scored_0493|E1|3|356|postprocess 340, optimize_a4 340, nested_windows 340|340|válido|
|scored_0493|E1|6|360|postprocess 354, optimize_a4 354, nested_windows 354|354|válido|
|scored_0493|E2|52-1|342||313|válido|
|scored_0493|E2|52-3|356||311|válido|
|scored_0493|E2|52-6|360||301|válido|
|scored_0493|E2|64-1|342||315|válido|
|scored_0493|E2|64-3|356||316|válido|
|scored_0493|E2|64-6|360||306|válido|
|scored_0493|E3|joint|342|individual 329|324|OPTIMAL|
|scored_0493|E4|repair|342||342|completo|
|scored_0494|E1|1|320|postprocess 317, optimize_a4 317, nested_windows 317|317|válido|
|scored_0494|E1|3|308|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0494|E1|6|292|postprocess 292, optimize_a4 292, nested_windows 292|292|válido|
|scored_0494|E2|52-1|320||281|válido|
|scored_0494|E2|52-3|308||280|válido|
|scored_0494|E2|52-6|292||278|válido|
|scored_0494|E2|64-1|320||287|válido|
|scored_0494|E2|64-3|308||282|válido|
|scored_0494|E2|64-6|292||292|válido|
|scored_0494|E3|joint|292|individual 292|291|OPTIMAL|
|scored_0494|E4|repair|292||292|completo|
|scored_0495|E1|1|153|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0495|E1|3|153|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0495|E1|6|153|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0495|E2|52-1|153||144|válido|
|scored_0495|E2|52-3|153||144|válido|
|scored_0495|E2|52-6|153||144|válido|
|scored_0495|E2|64-1|153||144|válido|
|scored_0495|E2|64-3|153||144|válido|
|scored_0495|E2|64-6|153||143|válido|
|scored_0495|E3|joint|153|individual 153|153|OPTIMAL|
|scored_0495|E4|repair|153||150|completo|
|scored_0496|E1|1|177|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0496|E1|3|174|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0496|E1|6|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0496|E2|52-1|177||171|válido|
|scored_0496|E2|52-3|174||166|válido|
|scored_0496|E2|52-6|172||166|válido|
|scored_0496|E2|64-1|177||168|válido|
|scored_0496|E2|64-3|174||160|válido|
|scored_0496|E2|64-6|172||160|válido|
|scored_0496|E3|joint|172|individual 172|172|OPTIMAL|
|scored_0496|E4|repair|172||172|completo|
|scored_0497|E1|1|250|postprocess 249, optimize_a4 249, nested_windows 249|249|válido|
|scored_0497|E1|3|231|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|scored_0497|E1|6|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0497|E2|52-1|250||229|válido|
|scored_0497|E2|52-3|231||218|válido|
|scored_0497|E2|52-6|229||218|válido|
|scored_0497|E2|64-1|250||228|válido|
|scored_0497|E2|64-3|231||216|válido|
|scored_0497|E2|64-6|229||216|válido|
|scored_0497|E3|joint|229|individual 229|227|OPTIMAL|
|scored_0497|E4|repair|229||229|completo|
|scored_0498|E1|1|211|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0498|E1|3|207|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0498|E1|6|209|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|scored_0498|E2|52-1|211||197|válido|
|scored_0498|E2|52-3|207||201|válido|
|scored_0498|E2|52-6|209||194|válido|
|scored_0498|E2|64-1|211||193|válido|
|scored_0498|E2|64-3|207||197|válido|
|scored_0498|E2|64-6|209||192|válido|
|scored_0498|E3|joint|207|individual 207|207|OPTIMAL|
|scored_0498|E4|repair|207||205|completo|
|scored_0499|E1|1|210|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0499|E1|3|208|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0499|E1|6|202|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0499|E2|52-1|210||188|válido|
|scored_0499|E2|52-3|208||182|válido|
|scored_0499|E2|52-6|202||175|válido|
|scored_0499|E2|64-1|210||191|válido|
|scored_0499|E2|64-3|208||185|válido|
|scored_0499|E2|64-6|202||176|válido|
|scored_0499|E3|joint|202|individual 202|202|OPTIMAL|
|scored_0499|E4|repair|202||201|completo|
|scored_0500|E1|1|111|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0500|E1|3|111|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0500|E1|6|111|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0500|E2|52-1|111||107|válido|
|scored_0500|E2|52-3|111||107|válido|
|scored_0500|E2|52-6|111||107|válido|
|scored_0500|E2|64-1|111||106|válido|
|scored_0500|E2|64-3|111||106|válido|
|scored_0500|E2|64-6|111||106|válido|
|scored_0500|E3|joint|111|individual 111|111|OPTIMAL|
|scored_0500|E4|repair|111||109|completo|
|scored_0501|E1|1|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|scored_0501|E1|3|185|postprocess 184, optimize_a4 184, nested_windows 184|184|válido|
|scored_0501|E1|6|183|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0501|E2|52-1|197||178|válido|
|scored_0501|E2|52-3|185||179|válido|
|scored_0501|E2|52-6|183||171|válido|
|scored_0501|E2|64-1|197||179|válido|
|scored_0501|E2|64-3|185||177|válido|
|scored_0501|E2|64-6|183||172|válido|
|scored_0501|E3|joint|183|individual 182|182|OPTIMAL|
|scored_0501|E4|repair|183||183|completo|
|scored_0502|E1|1|216|postprocess 216, optimize_a4 216, nested_windows 216|216|válido|
|scored_0502|E1|3|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0502|E1|6|213|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0502|E2|52-1|216||201|válido|
|scored_0502|E2|52-3|215||201|válido|
|scored_0502|E2|52-6|213||202|válido|
|scored_0502|E2|64-1|216||204|válido|
|scored_0502|E2|64-3|215||201|válido|
|scored_0502|E2|64-6|213||202|válido|
|scored_0502|E3|joint|213|individual 213|211|OPTIMAL|
|scored_0502|E4|repair|213||213|completo|
|scored_0503|E1|1|110|postprocess 110, optimize_a4 110, nested_windows 110|110|válido|
|scored_0503|E1|3|110|postprocess 110, optimize_a4 110, nested_windows 110|110|válido|
|scored_0503|E1|6|110|postprocess 110, optimize_a4 110, nested_windows 110|110|válido|
|scored_0503|E2|52-1|110||107|válido|
|scored_0503|E2|52-3|110||107|válido|
|scored_0503|E2|52-6|110||107|válido|
|scored_0503|E2|64-1|110||107|válido|
|scored_0503|E2|64-3|110||107|válido|
|scored_0503|E2|64-6|110||107|válido|
|scored_0503|E3|joint|110|individual 110|110|OPTIMAL|
|scored_0503|E4|repair|110||110|completo|
|scored_0504|E1|1|267|postprocess 267, optimize_a4 267, nested_windows 267|267|válido|
|scored_0504|E1|3|254|postprocess 252, optimize_a4 252, nested_windows 252|252|válido|
|scored_0504|E1|6|253|postprocess 253, optimize_a4 253, nested_windows 253|253|válido|
|scored_0504|E2|52-1|267||243|válido|
|scored_0504|E2|52-3|254||238|válido|
|scored_0504|E2|52-6|253||238|válido|
|scored_0504|E2|64-1|267||243|válido|
|scored_0504|E2|64-3|254||240|válido|
|scored_0504|E2|64-6|253||238|válido|
|scored_0504|E3|joint|253|individual 252|251|OPTIMAL|
|scored_0504|E4|repair|253||253|completo|
|scored_0505|E1|1|163|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0505|E1|3|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0505|E1|6|160|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0505|E2|52-1|163||154|válido|
|scored_0505|E2|52-3|162||152|válido|
|scored_0505|E2|52-6|160||153|válido|
|scored_0505|E2|64-1|163||153|válido|
|scored_0505|E2|64-3|162||150|válido|
|scored_0505|E2|64-6|160||150|válido|
|scored_0505|E3|joint|160|individual 160|160|OPTIMAL|
|scored_0505|E4|repair|160||159|completo|
|scored_0506|E1|1|159|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0506|E1|3|158|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0506|E1|6|158|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0506|E2|52-1|159||149|válido|
|scored_0506|E2|52-3|158||147|válido|
|scored_0506|E2|52-6|158||147|válido|
|scored_0506|E2|64-1|159||149|válido|
|scored_0506|E2|64-3|158||147|válido|
|scored_0506|E2|64-6|158||147|válido|
|scored_0506|E3|joint|158|individual 157|157|OPTIMAL|
|scored_0506|E4|repair|158||156|completo|
|scored_0507|E1|1|227|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0507|E1|3|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0507|E1|6|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0507|E2|52-1|227||213|válido|
|scored_0507|E2|52-3|220||208|válido|
|scored_0507|E2|52-6|219||205|válido|
|scored_0507|E2|64-1|227||213|válido|
|scored_0507|E2|64-3|220||207|válido|
|scored_0507|E2|64-6|219||205|válido|
|scored_0507|E3|joint|219|individual 219|219|OPTIMAL|
|scored_0507|E4|repair|219||219|completo|
|scored_0508|E1|1|297|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0508|E1|3|276|postprocess 276, optimize_a4 276, nested_windows 276|276|válido|
|scored_0508|E1|6|276|postprocess 275, optimize_a4 275, nested_windows 275|275|válido|
|scored_0508|E2|52-1|297||262|válido|
|scored_0508|E2|52-3|276||259|válido|
|scored_0508|E2|52-6|276||254|válido|
|scored_0508|E2|64-1|297||263|válido|
|scored_0508|E2|64-3|276||265|válido|
|scored_0508|E2|64-6|276||252|válido|
|scored_0508|E3|joint|276|individual 276|271|OPTIMAL|
|scored_0508|E4|repair|276||276|completo|
|scored_0509|E1|1|182|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0509|E1|3|182|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0509|E1|6|178|postprocess 178, optimize_a4 178, nested_windows 178|178|válido|
|scored_0509|E2|52-1|182||172|válido|
|scored_0509|E2|52-3|182||166|válido|
|scored_0509|E2|52-6|178||166|válido|
|scored_0509|E2|64-1|182||174|válido|
|scored_0509|E2|64-3|182||167|válido|
|scored_0509|E2|64-6|178||167|válido|
|scored_0509|E3|joint|178|individual 178|178|OPTIMAL|
|scored_0509|E4|repair|178||178|completo|
|scored_0510|E1|1|116|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0510|E1|3|116|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0510|E1|6|116|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0510|E2|52-1|116||111|válido|
|scored_0510|E2|52-3|116||111|válido|
|scored_0510|E2|52-6|116||108|válido|
|scored_0510|E2|64-1|116||111|válido|
|scored_0510|E2|64-3|116||111|válido|
|scored_0510|E2|64-6|116||108|válido|
|scored_0510|E3|joint|116|individual 116|116|OPTIMAL|
|scored_0510|E4|repair|116||112|completo|
|scored_0511|E1|1|333|postprocess 333, optimize_a4 333, nested_windows 333|333|válido|
|scored_0511|E1|3|321|postprocess 321, optimize_a4 321, nested_windows 321|321|válido|
|scored_0511|E1|6|326|postprocess 326, optimize_a4 326, nested_windows 326|326|válido|
|scored_0511|E2|52-1|333||317|válido|
|scored_0511|E2|52-3|321||307|válido|
|scored_0511|E2|52-6|326||306|válido|
|scored_0511|E2|64-1|333||314|válido|
|scored_0511|E2|64-3|321||305|válido|
|scored_0511|E2|64-6|326||304|válido|
|scored_0511|E3|joint|321|individual 321|318|OPTIMAL|
|scored_0511|E4|repair|321||321|completo|
|scored_0512|E1|1|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0512|E1|3|198|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|scored_0512|E1|6|194|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0512|E2|52-1|206||191|válido|
|scored_0512|E2|52-3|198||190|válido|
|scored_0512|E2|52-6|194||187|válido|
|scored_0512|E2|64-1|206||196|válido|
|scored_0512|E2|64-3|198||190|válido|
|scored_0512|E2|64-6|194||190|válido|
|scored_0512|E3|joint|194|individual 194|194|OPTIMAL|
|scored_0512|E4|repair|194||189|completo|
|scored_0513|E1|1|201|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0513|E1|3|195|postprocess 195, optimize_a4 195, nested_windows 195|195|válido|
|scored_0513|E1|6|195|postprocess 195, optimize_a4 195, nested_windows 195|195|válido|
|scored_0513|E2|52-1|201||190|válido|
|scored_0513|E2|52-3|195||183|válido|
|scored_0513|E2|52-6|195||183|válido|
|scored_0513|E2|64-1|201||190|válido|
|scored_0513|E2|64-3|195||183|válido|
|scored_0513|E2|64-6|195||183|válido|
|scored_0513|E3|joint|195|individual 195|195|OPTIMAL|
|scored_0513|E4|repair|195||195|completo|
|scored_0514|E1|1|306|postprocess 305, optimize_a4 305, nested_windows 305|305|válido|
|scored_0514|E1|3|290|postprocess 290, optimize_a4 290, nested_windows 290|290|válido|
|scored_0514|E1|6|281|postprocess 281, optimize_a4 281, nested_windows 281|281|válido|
|scored_0514|E2|52-1|306||278|válido|
|scored_0514|E2|52-3|290||270|válido|
|scored_0514|E2|52-6|281||267|válido|
|scored_0514|E2|64-1|306||277|válido|
|scored_0514|E2|64-3|290||272|válido|
|scored_0514|E2|64-6|281||268|válido|
|scored_0514|E3|joint|281|individual 281|280|FEASIBLE|
|scored_0514|E4|repair|281||281|completo|
|scored_0515|E1|1|39|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|scored_0515|E1|3|39|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|scored_0515|E1|6|39|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|scored_0515|E2|52-1|39||39|válido|
|scored_0515|E2|52-3|39||39|válido|
|scored_0515|E2|52-6|39||39|válido|
|scored_0515|E2|64-1|39||39|válido|
|scored_0515|E2|64-3|39||39|válido|
|scored_0515|E2|64-6|39||39|válido|
|scored_0515|E3|joint|39|individual 39|39|OPTIMAL|
|scored_0515|E4|repair|39||39|completo|
|scored_0516|E1|1|216|postprocess 216, optimize_a4 216, nested_windows 216|216|válido|
|scored_0516|E1|3|209|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0516|E1|6|208|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|scored_0516|E2|52-1|216||204|válido|
|scored_0516|E2|52-3|209||204|válido|
|scored_0516|E2|52-6|208||199|válido|
|scored_0516|E2|64-1|216||207|válido|
|scored_0516|E2|64-3|209||201|válido|
|scored_0516|E2|64-6|208||201|válido|
|scored_0516|E3|joint|208|individual 208|208|OPTIMAL|
|scored_0516|E4|repair|208||208|completo|
|scored_0517|E1|1|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0517|E1|3|209|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0517|E1|6|205|postprocess 205, optimize_a4 205, nested_windows 205|205|válido|
|scored_0517|E2|52-1|211||202|válido|
|scored_0517|E2|52-3|209||198|válido|
|scored_0517|E2|52-6|205||198|válido|
|scored_0517|E2|64-1|211||202|válido|
|scored_0517|E2|64-3|209||198|válido|
|scored_0517|E2|64-6|205||198|válido|
|scored_0517|E3|joint|205|individual 205|204|OPTIMAL|
|scored_0517|E4|repair|205||203|completo|
|scored_0518|E1|1|269|postprocess 268, optimize_a4 268, nested_windows 268|268|válido|
|scored_0518|E1|3|265|postprocess 265, optimize_a4 265, nested_windows 265|265|válido|
|scored_0518|E1|6|263|postprocess 263, optimize_a4 263, nested_windows 263|263|válido|
|scored_0518|E2|52-1|269||253|válido|
|scored_0518|E2|52-3|265||247|válido|
|scored_0518|E2|52-6|263||245|válido|
|scored_0518|E2|64-1|269||253|válido|
|scored_0518|E2|64-3|265||247|válido|
|scored_0518|E2|64-6|263||244|válido|
|scored_0518|E3|joint|263|individual 262|262|OPTIMAL|
|scored_0518|E4|repair|263||263|completo|
|scored_0519|E1|1|118|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0519|E1|3|118|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0519|E1|6|118|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0519|E2|52-1|118||115|válido|
|scored_0519|E2|52-3|118||113|válido|
|scored_0519|E2|52-6|118||113|válido|
|scored_0519|E2|64-1|118||114|válido|
|scored_0519|E2|64-3|118||113|válido|
|scored_0519|E2|64-6|118||113|válido|
|scored_0519|E3|joint|118|individual 118|118|OPTIMAL|
|scored_0519|E4|repair|118||114|completo|
|scored_0520|E1|1|236|postprocess 234, optimize_a4 234, nested_windows 234|234|válido|
|scored_0520|E1|3|231|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|scored_0520|E1|6|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0520|E2|52-1|236||222|válido|
|scored_0520|E2|52-3|231||222|válido|
|scored_0520|E2|52-6|229||219|válido|
|scored_0520|E2|64-1|236||223|válido|
|scored_0520|E2|64-3|231||222|válido|
|scored_0520|E2|64-6|229||222|válido|
|scored_0520|E3|joint|229|individual 229|229|OPTIMAL|
|scored_0520|E4|repair|229||228|completo|
|scored_0521|E1|1|219|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|scored_0521|E1|3|213|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|scored_0521|E1|6|208|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0521|E2|52-1|219||208|válido|
|scored_0521|E2|52-3|213||203|válido|
|scored_0521|E2|52-6|208||198|válido|
|scored_0521|E2|64-1|219||203|válido|
|scored_0521|E2|64-3|213||199|válido|
|scored_0521|E2|64-6|208||196|válido|
|scored_0521|E3|joint|208|individual 208|208|OPTIMAL|
|scored_0521|E4|repair|208||208|completo|
|scored_0522|E1|1|121|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0522|E1|3|121|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0522|E1|6|121|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0522|E2|52-1|121||118|válido|
|scored_0522|E2|52-3|121||118|válido|
|scored_0522|E2|52-6|121||118|válido|
|scored_0522|E2|64-1|121||118|válido|
|scored_0522|E2|64-3|121||117|válido|
|scored_0522|E2|64-6|121||117|válido|
|scored_0522|E3|joint|121|individual 121|121|OPTIMAL|
|scored_0522|E4|repair|121||119|completo|
|scored_0523|E1|1|163|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0523|E1|3|160|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0523|E1|6|159|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0523|E2|52-1|163||159|válido|
|scored_0523|E2|52-3|160||155|válido|
|scored_0523|E2|52-6|159||155|válido|
|scored_0523|E2|64-1|163||157|válido|
|scored_0523|E2|64-3|160||156|válido|
|scored_0523|E2|64-6|159||155|válido|
|scored_0523|E3|joint|159|individual 159|159|OPTIMAL|
|scored_0523|E4|repair|159||158|completo|
|scored_0524|E1|1|184|postprocess 184, optimize_a4 184, nested_windows 184|184|válido|
|scored_0524|E1|3|176|postprocess 176, optimize_a4 176, nested_windows 176|176|válido|
|scored_0524|E1|6|173|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0524|E2|52-1|184||163|válido|
|scored_0524|E2|52-3|176||163|válido|
|scored_0524|E2|52-6|173||160|válido|
|scored_0524|E2|64-1|184||163|válido|
|scored_0524|E2|64-3|176||161|válido|
|scored_0524|E2|64-6|173||160|válido|
|scored_0524|E3|joint|173|individual 173|173|OPTIMAL|
|scored_0524|E4|repair|173||170|completo|
|scored_0525|E1|1|161|postprocess 161, optimize_a4 161, nested_windows 161|161|válido|
|scored_0525|E1|3|160|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0525|E1|6|155|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|scored_0525|E2|52-1|161||152|válido|
|scored_0525|E2|52-3|160||152|válido|
|scored_0525|E2|52-6|155||152|válido|
|scored_0525|E2|64-1|161||151|válido|
|scored_0525|E2|64-3|160||151|válido|
|scored_0525|E2|64-6|155||151|válido|
|scored_0525|E3|joint|155|individual 155|155|OPTIMAL|
|scored_0525|E4|repair|155||154|completo|
|scored_0526|E1|1|300|postprocess 299, optimize_a4 299, nested_windows 299|299|válido|
|scored_0526|E1|3|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|scored_0526|E1|6|284|postprocess 284, optimize_a4 284, nested_windows 284|284|válido|
|scored_0526|E2|52-1|300||282|válido|
|scored_0526|E2|52-3|289||271|válido|
|scored_0526|E2|52-6|284||265|válido|
|scored_0526|E2|64-1|300||282|válido|
|scored_0526|E2|64-3|289||271|válido|
|scored_0526|E2|64-6|284||267|válido|
|scored_0526|E3|joint|284|individual 283|283|FEASIBLE|
|scored_0526|E4|repair|284||284|completo|
|scored_0527|E1|1|315|postprocess 314, optimize_a4 314, nested_windows 314|314|válido|
|scored_0527|E1|3|310|postprocess 310, optimize_a4 310, nested_windows 310|310|válido|
|scored_0527|E1|6|298|postprocess 298, optimize_a4 298, nested_windows 298|298|válido|
|scored_0527|E2|52-1|315||287|válido|
|scored_0527|E2|52-3|310||280|válido|
|scored_0527|E2|52-6|298||276|válido|
|scored_0527|E2|64-1|315||294|válido|
|scored_0527|E2|64-3|310||284|válido|
|scored_0527|E2|64-6|298||280|válido|
|scored_0527|E3|joint|298|individual 298|298|rechazado por registros|
|scored_0527|E4|repair|298||298|completo|
|scored_0528|E1|1|183|postprocess 183, optimize_a4 183, nested_windows 183|183|válido|
|scored_0528|E1|3|181|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0528|E1|6|180|postprocess 180, optimize_a4 180, nested_windows 180|180|válido|
|scored_0528|E2|52-1|183||175|válido|
|scored_0528|E2|52-3|181||174|válido|
|scored_0528|E2|52-6|180||170|válido|
|scored_0528|E2|64-1|183||174|válido|
|scored_0528|E2|64-3|181||171|válido|
|scored_0528|E2|64-6|180||171|válido|
|scored_0528|E3|joint|180|individual 180|180|OPTIMAL|
|scored_0528|E4|repair|180||180|completo|
|scored_0529|E1|1|204|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|scored_0529|E1|3|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0529|E1|6|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0529|E2|52-1|204||188|válido|
|scored_0529|E2|52-3|206||192|válido|
|scored_0529|E2|52-6|199||186|válido|
|scored_0529|E2|64-1|204||192|válido|
|scored_0529|E2|64-3|206||194|válido|
|scored_0529|E2|64-6|199||190|válido|
|scored_0529|E3|joint|199|individual 198|196|OPTIMAL|
|scored_0529|E4|repair|199||198|completo|
|scored_0530|E1|1|220|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0530|E1|3|214|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0530|E1|6|212|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|scored_0530|E2|52-1|220||198|válido|
|scored_0530|E2|52-3|214||199|válido|
|scored_0530|E2|52-6|212||196|válido|
|scored_0530|E2|64-1|220||200|válido|
|scored_0530|E2|64-3|214||200|válido|
|scored_0530|E2|64-6|212||194|válido|
|scored_0530|E3|joint|212|individual 212|212|OPTIMAL|
|scored_0530|E4|repair|212||212|completo|
|scored_0531|E1|1|270|postprocess 269, optimize_a4 269, nested_windows 269|269|válido|
|scored_0531|E1|3|260|postprocess 260, optimize_a4 260, nested_windows 260|260|válido|
|scored_0531|E1|6|255|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|scored_0531|E2|52-1|270||247|válido|
|scored_0531|E2|52-3|260||245|válido|
|scored_0531|E2|52-6|255||238|válido|
|scored_0531|E2|64-1|270||249|válido|
|scored_0531|E2|64-3|260||240|válido|
|scored_0531|E2|64-6|255||238|válido|
|scored_0531|E3|joint|255|individual 255|255|OPTIMAL|
|scored_0531|E4|repair|255||255|completo|
|scored_0532|E1|1|107|postprocess 107, optimize_a4 107, nested_windows 107|107|válido|
|scored_0532|E1|3|107|postprocess 107, optimize_a4 107, nested_windows 107|107|válido|
|scored_0532|E1|6|107|postprocess 107, optimize_a4 107, nested_windows 107|107|válido|
|scored_0532|E2|52-1|107||108|válido|
|scored_0532|E2|52-3|107||108|válido|
|scored_0532|E2|52-6|107||108|válido|
|scored_0532|E2|64-1|107||105|válido|
|scored_0532|E2|64-3|107||105|válido|
|scored_0532|E2|64-6|107||105|válido|
|scored_0532|E3|joint|107|individual 107|107|OPTIMAL|
|scored_0532|E4|repair|107||104|completo|
|scored_0533|E1|1|150|postprocess 150, optimize_a4 150, nested_windows 150|150|válido|
|scored_0533|E1|3|150|postprocess 150, optimize_a4 150, nested_windows 150|150|válido|
|scored_0533|E1|6|150|postprocess 150, optimize_a4 150, nested_windows 150|150|válido|
|scored_0533|E2|52-1|150||144|válido|
|scored_0533|E2|52-3|150||144|válido|
|scored_0533|E2|52-6|150||144|válido|
|scored_0533|E2|64-1|150||143|válido|
|scored_0533|E2|64-3|150||143|válido|
|scored_0533|E2|64-6|150||143|válido|
|scored_0533|E3|joint|150|individual 150|150|OPTIMAL|
|scored_0533|E4|repair|150||150|completo|
|scored_0534|E1|1|179|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|scored_0534|E1|3|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0534|E1|6|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|scored_0534|E2|52-1|179||163|válido|
|scored_0534|E2|52-3|172||164|válido|
|scored_0534|E2|52-6|166||161|válido|
|scored_0534|E2|64-1|179||163|válido|
|scored_0534|E2|64-3|172||163|válido|
|scored_0534|E2|64-6|166||161|válido|
|scored_0534|E3|joint|166|individual 166|165|OPTIMAL|
|scored_0534|E4|repair|166||165|completo|
|scored_0535|E1|1|190|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|scored_0535|E1|3|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0535|E1|6|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0535|E2|52-1|190||170|válido|
|scored_0535|E2|52-3|192||170|válido|
|scored_0535|E2|52-6|188||167|válido|
|scored_0535|E2|64-1|190||172|válido|
|scored_0535|E2|64-3|192||169|válido|
|scored_0535|E2|64-6|188||172|válido|
|scored_0535|E3|joint|188|individual 188|184|OPTIMAL|
|scored_0535|E4|repair|188||188|completo|
|scored_0536|E1|1|340|postprocess 340, optimize_a4 340, nested_windows 340|340|válido|
|scored_0536|E1|3|335|postprocess 335, optimize_a4 335, nested_windows 335|335|válido|
|scored_0536|E1|6|319|postprocess 318, optimize_a4 318, nested_windows 318|318|válido|
|scored_0536|E2|52-1|340||322|válido|
|scored_0536|E2|52-3|335||308|válido|
|scored_0536|E2|52-6|319||309|válido|
|scored_0536|E2|64-1|340||324|válido|
|scored_0536|E2|64-3|335||308|válido|
|scored_0536|E2|64-6|319||308|válido|
|scored_0536|E3|joint|319|individual 319|319|OPTIMAL|
|scored_0536|E4|repair|319||319|completo|
|scored_0537|E1|1|2|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|scored_0537|E1|3|2|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|scored_0537|E1|6|2|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|scored_0537|E2|52-1|2||2|válido|
|scored_0537|E2|52-3|2||2|válido|
|scored_0537|E2|52-6|2||2|válido|
|scored_0537|E2|64-1|2||2|válido|
|scored_0537|E2|64-3|2||2|válido|
|scored_0537|E2|64-6|2||2|válido|
|scored_0537|E3|joint|2|individual 2|2|OPTIMAL|
|scored_0537|E4|repair|2||2|completo|
|scored_0538|E1|1|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0538|E1|3|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0538|E1|6|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0538|E2|52-1|239||222|válido|
|scored_0538|E2|52-3|235||216|válido|
|scored_0538|E2|52-6|229||211|válido|
|scored_0538|E2|64-1|239||223|válido|
|scored_0538|E2|64-3|235||220|válido|
|scored_0538|E2|64-6|229||214|válido|
|scored_0538|E3|joint|229|individual 229|229|OPTIMAL|
|scored_0538|E4|repair|229||229|completo|
|scored_0539|E1|1|123|postprocess 123, optimize_a4 123, nested_windows 123|123|válido|
|scored_0539|E1|3|122|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0539|E1|6|122|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0539|E2|52-1|123||119|válido|
|scored_0539|E2|52-3|122||119|válido|
|scored_0539|E2|52-6|122||119|válido|
|scored_0539|E2|64-1|123||119|válido|
|scored_0539|E2|64-3|122||119|válido|
|scored_0539|E2|64-6|122||119|válido|
|scored_0539|E3|joint|122|individual 122|122|OPTIMAL|
|scored_0539|E4|repair|122||119|completo|
|scored_0540|E1|1|228|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|scored_0540|E1|3|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0540|E1|6|225|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0540|E2|52-1|228||219|válido|
|scored_0540|E2|52-3|226||215|válido|
|scored_0540|E2|52-6|225||214|válido|
|scored_0540|E2|64-1|228||219|válido|
|scored_0540|E2|64-3|226||215|válido|
|scored_0540|E2|64-6|225||212|válido|
|scored_0540|E3|joint|225|individual 225|224|OPTIMAL|
|scored_0540|E4|repair|225||225|completo|
|scored_0541|E1|1|71|postprocess 71, optimize_a4 71, nested_windows 71|71|válido|
|scored_0541|E1|3|71|postprocess 71, optimize_a4 71, nested_windows 71|71|válido|
|scored_0541|E1|6|70|postprocess 69, optimize_a4 69, nested_windows 69|69|válido|
|scored_0541|E2|52-1|71||69|válido|
|scored_0541|E2|52-3|71||69|válido|
|scored_0541|E2|52-6|70||69|válido|
|scored_0541|E2|64-1|71||69|válido|
|scored_0541|E2|64-3|71||69|válido|
|scored_0541|E2|64-6|70||69|válido|
|scored_0541|E3|joint|70|individual 70|70|OPTIMAL|
|scored_0541|E4|repair|70||69|completo|
|scored_0542|E1|1|147|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0542|E1|3|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0542|E1|6|140|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|scored_0542|E2|52-1|147||141|válido|
|scored_0542|E2|52-3|143||140|válido|
|scored_0542|E2|52-6|140||140|válido|
|scored_0542|E2|64-1|147||139|válido|
|scored_0542|E2|64-3|143||136|válido|
|scored_0542|E2|64-6|140||136|válido|
|scored_0542|E3|joint|140|individual 140|140|OPTIMAL|
|scored_0542|E4|repair|140||137|completo|
|scored_0543|E1|1|133|postprocess 133, optimize_a4 133, nested_windows 133|133|válido|
|scored_0543|E1|3|133|postprocess 133, optimize_a4 133, nested_windows 133|133|válido|
|scored_0543|E1|6|133|postprocess 133, optimize_a4 133, nested_windows 133|133|válido|
|scored_0543|E2|52-1|133||127|válido|
|scored_0543|E2|52-3|133||127|válido|
|scored_0543|E2|52-6|133||127|válido|
|scored_0543|E2|64-1|133||128|válido|
|scored_0543|E2|64-3|133||127|válido|
|scored_0543|E2|64-6|133||127|válido|
|scored_0543|E3|joint|133|individual 133|133|OPTIMAL|
|scored_0543|E4|repair|133||131|completo|
|scored_0544|E1|1|243|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0544|E1|3|235|postprocess 234, optimize_a4 234, nested_windows 234|234|válido|
|scored_0544|E1|6|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0544|E2|52-1|243||227|válido|
|scored_0544|E2|52-3|235||219|válido|
|scored_0544|E2|52-6|232||219|válido|
|scored_0544|E2|64-1|243||227|válido|
|scored_0544|E2|64-3|235||221|válido|
|scored_0544|E2|64-6|232||217|válido|
|scored_0544|E3|joint|232|individual 232|232|FEASIBLE|
|scored_0544|E4|repair|232||232|completo|
|scored_0545|E1|1|150|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|scored_0545|E1|3|148|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0545|E1|6|148|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0545|E2|52-1|150||144|válido|
|scored_0545|E2|52-3|148||142|válido|
|scored_0545|E2|52-6|148||141|válido|
|scored_0545|E2|64-1|150||143|válido|
|scored_0545|E2|64-3|148||142|válido|
|scored_0545|E2|64-6|148||140|válido|
|scored_0545|E3|joint|148|individual 147|147|OPTIMAL|
|scored_0545|E4|repair|148||145|completo|
|scored_0546|E1|1|206|postprocess 205, optimize_a4 205, nested_windows 205|205|válido|
|scored_0546|E1|3|200|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0546|E1|6|198|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|scored_0546|E2|52-1|206||185|válido|
|scored_0546|E2|52-3|200||187|válido|
|scored_0546|E2|52-6|198||184|válido|
|scored_0546|E2|64-1|206||184|válido|
|scored_0546|E2|64-3|200||187|válido|
|scored_0546|E2|64-6|198||184|válido|
|scored_0546|E3|joint|198|individual 197|197|OPTIMAL|
|scored_0546|E4|repair|198||198|completo|
|scored_0547|E1|1|139|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|scored_0547|E1|3|138|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0547|E1|6|138|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0547|E2|52-1|139||134|válido|
|scored_0547|E2|52-3|138||133|válido|
|scored_0547|E2|52-6|138||133|válido|
|scored_0547|E2|64-1|139||134|válido|
|scored_0547|E2|64-3|138||134|válido|
|scored_0547|E2|64-6|138||134|válido|
|scored_0547|E3|joint|138|individual 138|138|OPTIMAL|
|scored_0547|E4|repair|138||137|completo|
|scored_0548|E1|1|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0548|E1|3|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0548|E1|6|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0548|E2|52-1|50||50|válido|
|scored_0548|E2|52-3|50||50|válido|
|scored_0548|E2|52-6|50||50|válido|
|scored_0548|E2|64-1|50||50|válido|
|scored_0548|E2|64-3|50||50|válido|
|scored_0548|E2|64-6|50||50|válido|
|scored_0548|E3|joint|50|individual 50|50|OPTIMAL|
|scored_0548|E4|repair|50||50|completo|
|scored_0549|E1|1|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0549|E1|3|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0549|E1|6|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0549|E2|52-1|146||140|válido|
|scored_0549|E2|52-3|146||139|válido|
|scored_0549|E2|52-6|146||139|válido|
|scored_0549|E2|64-1|146||140|válido|
|scored_0549|E2|64-3|146||140|válido|
|scored_0549|E2|64-6|146||140|válido|
|scored_0549|E3|joint|146|individual 146|146|OPTIMAL|
|scored_0549|E4|repair|146||146|completo|
|scored_0550|E1|1|146|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0550|E1|3|143|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0550|E1|6|136|postprocess 136, optimize_a4 136, nested_windows 136|136|válido|
|scored_0550|E2|52-1|146||141|válido|
|scored_0550|E2|52-3|143||134|válido|
|scored_0550|E2|52-6|136||133|válido|
|scored_0550|E2|64-1|146||143|válido|
|scored_0550|E2|64-3|143||135|válido|
|scored_0550|E2|64-6|136||134|válido|
|scored_0550|E3|joint|136|individual 136|136|OPTIMAL|
|scored_0550|E4|repair|136||135|completo|
|scored_0551|E1|1|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|scored_0551|E1|3|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0551|E1|6|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0551|E2|52-1|238||228|válido|
|scored_0551|E2|52-3|233||220|válido|
|scored_0551|E2|52-6|233||220|válido|
|scored_0551|E2|64-1|238||230|válido|
|scored_0551|E2|64-3|233||223|válido|
|scored_0551|E2|64-6|233||222|válido|
|scored_0551|E3|joint|233|individual 232|229|OPTIMAL|
|scored_0551|E4|repair|233||233|completo|
|scored_0552|E1|1|310|postprocess 309, optimize_a4 309, nested_windows 309|309|válido|
|scored_0552|E1|3|294|postprocess 294, optimize_a4 294, nested_windows 294|294|válido|
|scored_0552|E1|6|284|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|scored_0552|E2|52-1|310||288|válido|
|scored_0552|E2|52-3|294||279|válido|
|scored_0552|E2|52-6|284||271|válido|
|scored_0552|E2|64-1|310||288|válido|
|scored_0552|E2|64-3|294||277|válido|
|scored_0552|E2|64-6|284||273|válido|
|scored_0552|E3|joint|284|individual 283|283|OPTIMAL|
|scored_0552|E4|repair|284||284|completo|
|scored_0553|E1|1|256|postprocess 256, optimize_a4 256, nested_windows 256|256|válido|
|scored_0553|E1|3|243|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|scored_0553|E1|6|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0553|E2|52-1|256||230|válido|
|scored_0553|E2|52-3|243||224|válido|
|scored_0553|E2|52-6|239||220|válido|
|scored_0553|E2|64-1|256||230|válido|
|scored_0553|E2|64-3|243||226|válido|
|scored_0553|E2|64-6|239||220|válido|
|scored_0553|E3|joint|239|individual 239|238|OPTIMAL|
|scored_0553|E4|repair|239||238|completo|
|scored_0554|E1|1|142|postprocess 141, optimize_a4 141, nested_windows 141|141|válido|
|scored_0554|E1|3|143|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0554|E1|6|143|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0554|E2|52-1|142||140|válido|
|scored_0554|E2|52-3|143||139|válido|
|scored_0554|E2|52-6|143||139|válido|
|scored_0554|E2|64-1|142||139|válido|
|scored_0554|E2|64-3|143||140|válido|
|scored_0554|E2|64-6|143||140|válido|
|scored_0554|E3|joint|142|individual 142|142|OPTIMAL|
|scored_0554|E4|repair|142||142|completo|
|scored_0555|E1|1|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0555|E1|3|171|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|scored_0555|E1|6|168|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0555|E2|52-1|169||161|válido|
|scored_0555|E2|52-3|171||157|válido|
|scored_0555|E2|52-6|168||157|válido|
|scored_0555|E2|64-1|169||160|válido|
|scored_0555|E2|64-3|171||155|válido|
|scored_0555|E2|64-6|168||155|válido|
|scored_0555|E3|joint|168|individual 168|168|OPTIMAL|
|scored_0555|E4|repair|168||168|completo|
|scored_0556|E1|1|41|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0556|E1|3|41|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0556|E1|6|41|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0556|E2|52-1|41||41|válido|
|scored_0556|E2|52-3|41||41|válido|
|scored_0556|E2|52-6|41||41|válido|
|scored_0556|E2|64-1|41||41|válido|
|scored_0556|E2|64-3|41||41|válido|
|scored_0556|E2|64-6|41||41|válido|
|scored_0556|E3|joint|41|individual 41|41|OPTIMAL|
|scored_0556|E4|repair|41||36|completo|
|scored_0557|E1|1|174|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0557|E1|3|171|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|scored_0557|E1|6|170|postprocess 170, optimize_a4 170, nested_windows 170|170|válido|
|scored_0557|E2|52-1|174||165|válido|
|scored_0557|E2|52-3|171||161|válido|
|scored_0557|E2|52-6|170||163|válido|
|scored_0557|E2|64-1|174||164|válido|
|scored_0557|E2|64-3|171||165|válido|
|scored_0557|E2|64-6|170||164|válido|
|scored_0557|E3|joint|170|individual 170|170|OPTIMAL|
|scored_0557|E4|repair|170||170|completo|
|scored_0558|E1|1|161|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0558|E1|3|160|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0558|E1|6|160|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0558|E2|52-1|161||154|válido|
|scored_0558|E2|52-3|160||152|válido|
|scored_0558|E2|52-6|160||153|válido|
|scored_0558|E2|64-1|161||152|válido|
|scored_0558|E2|64-3|160||151|válido|
|scored_0558|E2|64-6|160||149|válido|
|scored_0558|E3|joint|160|individual 160|160|OPTIMAL|
|scored_0558|E4|repair|160||160|completo|
|scored_0559|E1|1|48|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0559|E1|3|48|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0559|E1|6|48|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0559|E2|52-1|48||48|válido|
|scored_0559|E2|52-3|48||48|válido|
|scored_0559|E2|52-6|48||48|válido|
|scored_0559|E2|64-1|48||48|válido|
|scored_0559|E2|64-3|48||48|válido|
|scored_0559|E2|64-6|48||48|válido|
|scored_0559|E3|joint|48|individual 48|48|OPTIMAL|
|scored_0559|E4|repair|48||48|completo|
|scored_0560|E1|1|53|postprocess 53, optimize_a4 53, nested_windows 53|53|válido|
|scored_0560|E1|3|53|postprocess 53, optimize_a4 53, nested_windows 53|53|válido|
|scored_0560|E1|6|53|postprocess 53, optimize_a4 53, nested_windows 53|53|válido|
|scored_0560|E2|52-1|53||53|válido|
|scored_0560|E2|52-3|53||53|válido|
|scored_0560|E2|52-6|53||53|válido|
|scored_0560|E2|64-1|53||53|válido|
|scored_0560|E2|64-3|53||53|válido|
|scored_0560|E2|64-6|53||53|válido|
|scored_0560|E3|joint|53|individual 53|53|OPTIMAL|
|scored_0560|E4|repair|53||50|completo|
|scored_0561|E1|1|131|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|scored_0561|E1|3|131|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|scored_0561|E1|6|131|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|scored_0561|E2|52-1|131||124|válido|
|scored_0561|E2|52-3|131||124|válido|
|scored_0561|E2|52-6|131||124|válido|
|scored_0561|E2|64-1|131||125|válido|
|scored_0561|E2|64-3|131||125|válido|
|scored_0561|E2|64-6|131||125|válido|
|scored_0561|E3|joint|131|individual 131|131|OPTIMAL|
|scored_0561|E4|repair|131||126|completo|
|scored_0562|E1|1|117|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0562|E1|3|116|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0562|E1|6|116|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0562|E2|52-1|117||117|válido|
|scored_0562|E2|52-3|116||117|válido|
|scored_0562|E2|52-6|116||117|válido|
|scored_0562|E2|64-1|117||116|válido|
|scored_0562|E2|64-3|116||116|válido|
|scored_0562|E2|64-6|116||116|válido|
|scored_0562|E3|joint|116|individual 116|116|OPTIMAL|
|scored_0562|E4|repair|116||116|completo|
|scored_0563|E1|1|141|postprocess 141, optimize_a4 141, nested_windows 141|141|válido|
|scored_0563|E1|3|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0563|E1|6|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0563|E2|52-1|141||136|válido|
|scored_0563|E2|52-3|137||132|válido|
|scored_0563|E2|52-6|137||133|válido|
|scored_0563|E2|64-1|141||135|válido|
|scored_0563|E2|64-3|137||133|válido|
|scored_0563|E2|64-6|137||133|válido|
|scored_0563|E3|joint|137|individual 137|137|OPTIMAL|
|scored_0563|E4|repair|137||137|completo|
|scored_0564|E1|1|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0564|E1|3|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0564|E1|6|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0564|E2|52-1|41||41|válido|
|scored_0564|E2|52-3|41||41|válido|
|scored_0564|E2|52-6|41||41|válido|
|scored_0564|E2|64-1|41||41|válido|
|scored_0564|E2|64-3|41||41|válido|
|scored_0564|E2|64-6|41||41|válido|
|scored_0564|E3|joint|41|individual 41|41|OPTIMAL|
|scored_0564|E4|repair|41||41|completo|
|scored_0565|E1|1|302|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|scored_0565|E1|3|294|postprocess 293, optimize_a4 293, nested_windows 293|293|válido|
|scored_0565|E1|6|287|postprocess 287, optimize_a4 287, nested_windows 287|287|válido|
|scored_0565|E2|52-1|302||280|válido|
|scored_0565|E2|52-3|294||279|válido|
|scored_0565|E2|52-6|287||265|válido|
|scored_0565|E2|64-1|302||283|válido|
|scored_0565|E2|64-3|294||277|válido|
|scored_0565|E2|64-6|287||267|válido|
|scored_0565|E3|joint|287|individual 287|287|OPTIMAL|
|scored_0565|E4|repair|287||287|completo|
|scored_0566|E1|1|33|postprocess 30, optimize_a4 30, nested_windows 30|30|válido|
|scored_0566|E1|3|33|postprocess 30, optimize_a4 30, nested_windows 30|30|válido|
|scored_0566|E1|6|33|postprocess 30, optimize_a4 30, nested_windows 30|30|válido|
|scored_0566|E2|52-1|33||33|válido|
|scored_0566|E2|52-3|33||33|válido|
|scored_0566|E2|52-6|33||33|válido|
|scored_0566|E2|64-1|33||33|válido|
|scored_0566|E2|64-3|33||33|válido|
|scored_0566|E2|64-6|33||33|válido|
|scored_0566|E3|joint|33|individual 33|33|OPTIMAL|
|scored_0566|E4|repair|33||31|completo|
|scored_0567|E1|1|156|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0567|E1|3|153|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0567|E1|6|150|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|scored_0567|E2|52-1|156||148|válido|
|scored_0567|E2|52-3|153||147|válido|
|scored_0567|E2|52-6|150||145|válido|
|scored_0567|E2|64-1|156||141|válido|
|scored_0567|E2|64-3|153||141|válido|
|scored_0567|E2|64-6|150||140|válido|
|scored_0567|E3|joint|150|individual 150|150|OPTIMAL|
|scored_0567|E4|repair|150||148|completo|
|scored_0568|E1|1|156|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|scored_0568|E1|3|150|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|scored_0568|E1|6|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0568|E2|52-1|156||140|válido|
|scored_0568|E2|52-3|150||140|válido|
|scored_0568|E2|52-6|148||140|válido|
|scored_0568|E2|64-1|156||140|válido|
|scored_0568|E2|64-3|150||140|válido|
|scored_0568|E2|64-6|148||140|válido|
|scored_0568|E3|joint|148|individual 148|148|OPTIMAL|
|scored_0568|E4|repair|148||144|completo|
|scored_0569|E1|1|208|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|scored_0569|E1|3|202|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0569|E1|6|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0569|E2|52-1|208||197|válido|
|scored_0569|E2|52-3|202||191|válido|
|scored_0569|E2|52-6|199||190|válido|
|scored_0569|E2|64-1|208||198|válido|
|scored_0569|E2|64-3|202||190|válido|
|scored_0569|E2|64-6|199||187|válido|
|scored_0569|E3|joint|199|individual 199|199|OPTIMAL|
|scored_0569|E4|repair|199||197|completo|
|scored_0570|E1|1|119|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0570|E1|3|119|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0570|E1|6|119|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0570|E2|52-1|119||114|válido|
|scored_0570|E2|52-3|119||114|válido|
|scored_0570|E2|52-6|119||114|válido|
|scored_0570|E2|64-1|119||112|válido|
|scored_0570|E2|64-3|119||112|válido|
|scored_0570|E2|64-6|119||112|válido|
|scored_0570|E3|joint|119|individual 119|119|OPTIMAL|
|scored_0570|E4|repair|119||113|completo|
|scored_0571|E1|1|274|postprocess 273, optimize_a4 273, nested_windows 273|273|válido|
|scored_0571|E1|3|270|postprocess 270, optimize_a4 270, nested_windows 270|270|válido|
|scored_0571|E1|6|257|postprocess 257, optimize_a4 257, nested_windows 257|257|válido|
|scored_0571|E2|52-1|274||249|válido|
|scored_0571|E2|52-3|270||246|válido|
|scored_0571|E2|52-6|257||240|válido|
|scored_0571|E2|64-1|274||247|válido|
|scored_0571|E2|64-3|270||247|válido|
|scored_0571|E2|64-6|257||240|válido|
|scored_0571|E3|joint|257|individual 257|257|OPTIMAL|
|scored_0571|E4|repair|257||257|completo|
|scored_0572|E1|1|317|postprocess 316, optimize_a4 316, nested_windows 316|316|válido|
|scored_0572|E1|3|308|postprocess 306, optimize_a4 306, nested_windows 306|306|válido|
|scored_0572|E1|6|304|postprocess 304, optimize_a4 304, nested_windows 304|304|válido|
|scored_0572|E2|52-1|317||299|válido|
|scored_0572|E2|52-3|308||293|válido|
|scored_0572|E2|52-6|304||286|válido|
|scored_0572|E2|64-1|317||298|válido|
|scored_0572|E2|64-3|308||294|válido|
|scored_0572|E2|64-6|304||289|válido|
|scored_0572|E3|joint|304|individual 303|302|OPTIMAL|
|scored_0572|E4|repair|304||304|completo|
|scored_0573|E1|1|255|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|scored_0573|E1|3|248|postprocess 248, optimize_a4 248, nested_windows 248|248|válido|
|scored_0573|E1|6|237|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|scored_0573|E2|52-1|255||238|válido|
|scored_0573|E2|52-3|248||235|válido|
|scored_0573|E2|52-6|237||231|válido|
|scored_0573|E2|64-1|255||239|válido|
|scored_0573|E2|64-3|248||238|válido|
|scored_0573|E2|64-6|237||232|válido|
|scored_0573|E3|joint|237|individual 237|237|OPTIMAL|
|scored_0573|E4|repair|237||237|completo|
|scored_0574|E1|1|136|postprocess 134, optimize_a4 134, nested_windows 134|134|válido|
|scored_0574|E1|3|136|postprocess 134, optimize_a4 134, nested_windows 134|134|válido|
|scored_0574|E1|6|136|postprocess 134, optimize_a4 134, nested_windows 134|134|válido|
|scored_0574|E2|52-1|136||128|válido|
|scored_0574|E2|52-3|136||128|válido|
|scored_0574|E2|52-6|136||131|válido|
|scored_0574|E2|64-1|136||128|válido|
|scored_0574|E2|64-3|136||128|válido|
|scored_0574|E2|64-6|136||128|válido|
|scored_0574|E3|joint|136|individual 136|136|OPTIMAL|
|scored_0574|E4|repair|136||134|completo|
|scored_0575|E1|1|170|postprocess 170, optimize_a4 170, nested_windows 170|170|válido|
|scored_0575|E1|3|168|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0575|E1|6|167|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0575|E2|52-1|170||165|válido|
|scored_0575|E2|52-3|168||159|válido|
|scored_0575|E2|52-6|167||165|válido|
|scored_0575|E2|64-1|170||163|válido|
|scored_0575|E2|64-3|168||160|válido|
|scored_0575|E2|64-6|167||160|válido|
|scored_0575|E3|joint|167|individual 167|167|OPTIMAL|
|scored_0575|E4|repair|167||166|completo|
|scored_0576|E1|1|230|postprocess 230, optimize_a4 230, nested_windows 230|230|válido|
|scored_0576|E1|3|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0576|E1|6|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0576|E2|52-1|230||209|válido|
|scored_0576|E2|52-3|219||207|válido|
|scored_0576|E2|52-6|215||208|válido|
|scored_0576|E2|64-1|230||210|válido|
|scored_0576|E2|64-3|219||210|válido|
|scored_0576|E2|64-6|215||211|válido|
|scored_0576|E3|joint|215|individual 214|213|OPTIMAL|
|scored_0576|E4|repair|215||215|completo|
|scored_0577|E1|1|243|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|scored_0577|E1|3|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0577|E1|6|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0577|E2|52-1|243||217|válido|
|scored_0577|E2|52-3|233||215|válido|
|scored_0577|E2|52-6|226||211|válido|
|scored_0577|E2|64-1|243||221|válido|
|scored_0577|E2|64-3|233||215|válido|
|scored_0577|E2|64-6|226||210|válido|
|scored_0577|E3|joint|226|individual 225|225|OPTIMAL|
|scored_0577|E4|repair|226||226|completo|
|scored_0578|E1|1|171|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|scored_0578|E1|3|171|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0578|E1|6|163|postprocess 161, optimize_a4 161, nested_windows 161|161|válido|
|scored_0578|E2|52-1|171||160|válido|
|scored_0578|E2|52-3|171||160|válido|
|scored_0578|E2|52-6|163||154|válido|
|scored_0578|E2|64-1|171||166|válido|
|scored_0578|E2|64-3|171||161|válido|
|scored_0578|E2|64-6|163||159|válido|
|scored_0578|E3|joint|163|individual 163|162|OPTIMAL|
|scored_0578|E4|repair|163||163|completo|
|scored_0579|E1|1|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0579|E1|3|225|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0579|E1|6|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0579|E2|52-1|223||215|válido|
|scored_0579|E2|52-3|225||209|válido|
|scored_0579|E2|52-6|220||204|válido|
|scored_0579|E2|64-1|223||216|válido|
|scored_0579|E2|64-3|225||213|válido|
|scored_0579|E2|64-6|220||205|válido|
|scored_0579|E3|joint|220|individual 219|219|OPTIMAL|
|scored_0579|E4|repair|220||220|completo|
|scored_0580|E1|1|329|postprocess 328, optimize_a4 328, nested_windows 328|328|válido|
|scored_0580|E1|3|312|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|scored_0580|E1|6|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0580|E2|52-1|329||300|válido|
|scored_0580|E2|52-3|312||289|válido|
|scored_0580|E2|52-6|303||282|válido|
|scored_0580|E2|64-1|329||299|válido|
|scored_0580|E2|64-3|312||290|válido|
|scored_0580|E2|64-6|303||283|válido|
|scored_0580|E3|joint|303|individual 302|301|OPTIMAL|
|scored_0580|E4|repair|303||303|completo|
|scored_0581|E1|1|351|postprocess 351, optimize_a4 351, nested_windows 351|351|válido|
|scored_0581|E1|3|339|postprocess 339, optimize_a4 339, nested_windows 339|339|válido|
|scored_0581|E1|6|329|postprocess 329, optimize_a4 329, nested_windows 329|329|válido|
|scored_0581|E2|52-1|351||324|válido|
|scored_0581|E2|52-3|339||316|válido|
|scored_0581|E2|52-6|329||314|válido|
|scored_0581|E2|64-1|351||327|válido|
|scored_0581|E2|64-3|339||319|válido|
|scored_0581|E2|64-6|329||313|válido|
|scored_0581|E3|joint|329|individual 329|329|FEASIBLE|
|scored_0581|E4|repair|329||329|completo|
|scored_0582|E1|1|254|postprocess 254, optimize_a4 254, nested_windows 254|254|válido|
|scored_0582|E1|3|247|postprocess 247, optimize_a4 247, nested_windows 247|247|válido|
|scored_0582|E1|6|243|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|scored_0582|E2|52-1|254||234|válido|
|scored_0582|E2|52-3|247||228|válido|
|scored_0582|E2|52-6|243||228|válido|
|scored_0582|E2|64-1|254||238|válido|
|scored_0582|E2|64-3|247||230|válido|
|scored_0582|E2|64-6|243||226|válido|
|scored_0582|E3|joint|243|individual 242|241|OPTIMAL|
|scored_0582|E4|repair|243||243|completo|
|scored_0583|E1|1|51|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0583|E1|3|51|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0583|E1|6|51|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0583|E2|52-1|51||51|válido|
|scored_0583|E2|52-3|51||51|válido|
|scored_0583|E2|52-6|51||51|válido|
|scored_0583|E2|64-1|51||51|válido|
|scored_0583|E2|64-3|51||51|válido|
|scored_0583|E2|64-6|51||51|válido|
|scored_0583|E3|joint|51|individual 51|51|OPTIMAL|
|scored_0583|E4|repair|51||51|completo|
|scored_0584|E1|1|59|postprocess 58, optimize_a4 58, nested_windows 58|58|válido|
|scored_0584|E1|3|59|postprocess 58, optimize_a4 58, nested_windows 58|58|válido|
|scored_0584|E1|6|59|postprocess 58, optimize_a4 58, nested_windows 58|58|válido|
|scored_0584|E2|52-1|59||60|válido|
|scored_0584|E2|52-3|59||60|válido|
|scored_0584|E2|52-6|59||60|válido|
|scored_0584|E2|64-1|59||60|válido|
|scored_0584|E2|64-3|59||60|válido|
|scored_0584|E2|64-6|59||60|válido|
|scored_0584|E3|joint|59|individual 59|59|OPTIMAL|
|scored_0584|E4|repair|59||57|completo|
|scored_0585|E1|1|248|postprocess 248, optimize_a4 248, nested_windows 248|248|válido|
|scored_0585|E1|3|244|postprocess 244, optimize_a4 244, nested_windows 244|244|válido|
|scored_0585|E1|6|241|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|scored_0585|E2|52-1|248||226|válido|
|scored_0585|E2|52-3|244||229|válido|
|scored_0585|E2|52-6|241||226|válido|
|scored_0585|E2|64-1|248||230|válido|
|scored_0585|E2|64-3|244||226|válido|
|scored_0585|E2|64-6|241||228|válido|
|scored_0585|E3|joint|241|individual 241|237|FEASIBLE|
|scored_0585|E4|repair|241||241|completo|
|scored_0586|E1|1|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|scored_0586|E1|3|112|postprocess 112, optimize_a4 112, nested_windows 112|112|válido|
|scored_0586|E1|6|112|postprocess 112, optimize_a4 112, nested_windows 112|112|válido|
|scored_0586|E2|52-1|113||105|válido|
|scored_0586|E2|52-3|112||107|válido|
|scored_0586|E2|52-6|112||107|válido|
|scored_0586|E2|64-1|113||105|válido|
|scored_0586|E2|64-3|112||105|válido|
|scored_0586|E2|64-6|112||105|válido|
|scored_0586|E3|joint|112|individual 112|112|OPTIMAL|
|scored_0586|E4|repair|112||110|completo|
|scored_0587|E1|1|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0587|E1|3|196|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|scored_0587|E1|6|195|postprocess 195, optimize_a4 195, nested_windows 195|195|válido|
|scored_0587|E2|52-1|202||190|válido|
|scored_0587|E2|52-3|196||187|válido|
|scored_0587|E2|52-6|195||186|válido|
|scored_0587|E2|64-1|202||187|válido|
|scored_0587|E2|64-3|196||182|válido|
|scored_0587|E2|64-6|195||185|válido|
|scored_0587|E3|joint|195|individual 194|194|OPTIMAL|
|scored_0587|E4|repair|195||195|completo|
|scored_0588|E1|1|190|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|scored_0588|E1|3|186|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0588|E1|6|184|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0588|E2|52-1|190||176|válido|
|scored_0588|E2|52-3|186||174|válido|
|scored_0588|E2|52-6|184||172|válido|
|scored_0588|E2|64-1|190||175|válido|
|scored_0588|E2|64-3|186||172|válido|
|scored_0588|E2|64-6|184||170|válido|
|scored_0588|E3|joint|184|individual 184|183|OPTIMAL|
|scored_0588|E4|repair|184||184|completo|
|scored_0589|E1|1|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|scored_0589|E1|3|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|scored_0589|E1|6|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|scored_0589|E2|52-1|113||114|válido|
|scored_0589|E2|52-3|113||114|válido|
|scored_0589|E2|52-6|113||114|válido|
|scored_0589|E2|64-1|113||114|válido|
|scored_0589|E2|64-3|113||114|válido|
|scored_0589|E2|64-6|113||114|válido|
|scored_0589|E3|joint|113|individual 113|113|OPTIMAL|
|scored_0589|E4|repair|113||111|completo|
|scored_0590|E1|1|189|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0590|E1|3|181|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|scored_0590|E1|6|181|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0590|E2|52-1|189||180|válido|
|scored_0590|E2|52-3|181||176|válido|
|scored_0590|E2|52-6|181||173|válido|
|scored_0590|E2|64-1|189||179|válido|
|scored_0590|E2|64-3|181||174|válido|
|scored_0590|E2|64-6|181||171|válido|
|scored_0590|E3|joint|181|individual 180|180|OPTIMAL|
|scored_0590|E4|repair|181||179|completo|
|scored_0591|E1|1|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|scored_0591|E1|3|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|scored_0591|E1|6|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0591|E2|52-1|166||160|válido|
|scored_0591|E2|52-3|166||159|válido|
|scored_0591|E2|52-6|165||157|válido|
|scored_0591|E2|64-1|166||161|válido|
|scored_0591|E2|64-3|166||158|válido|
|scored_0591|E2|64-6|165||157|válido|
|scored_0591|E3|joint|165|individual 164|164|OPTIMAL|
|scored_0591|E4|repair|165||163|completo|
|scored_0592|E1|1|298|postprocess 298, optimize_a4 298, nested_windows 298|298|válido|
|scored_0592|E1|3|281|postprocess 281, optimize_a4 281, nested_windows 281|281|válido|
|scored_0592|E1|6|274|postprocess 274, optimize_a4 274, nested_windows 274|274|válido|
|scored_0592|E2|52-1|298||271|válido|
|scored_0592|E2|52-3|281||262|válido|
|scored_0592|E2|52-6|274||259|válido|
|scored_0592|E2|64-1|298||273|válido|
|scored_0592|E2|64-3|281||264|válido|
|scored_0592|E2|64-6|274||260|válido|
|scored_0592|E3|joint|274|individual 273|273|OPTIMAL|
|scored_0592|E4|repair|274||274|completo|
|scored_0593|E1|1|320|postprocess 320, optimize_a4 320, nested_windows 320|320|válido|
|scored_0593|E1|3|316|postprocess 316, optimize_a4 316, nested_windows 316|316|válido|
|scored_0593|E1|6|307|postprocess 307, optimize_a4 307, nested_windows 307|307|válido|
|scored_0593|E2|52-1|320||306|válido|
|scored_0593|E2|52-3|316||294|válido|
|scored_0593|E2|52-6|307||282|válido|
|scored_0593|E2|64-1|320||306|válido|
|scored_0593|E2|64-3|316||294|válido|
|scored_0593|E2|64-6|307||285|válido|
|scored_0593|E3|joint|307|individual 306|305|FEASIBLE|
|scored_0593|E4|repair|307||307|completo|
|scored_0594|E1|1|325|postprocess 325, optimize_a4 325, nested_windows 325|325|válido|
|scored_0594|E1|3|315|postprocess 314, optimize_a4 314, nested_windows 314|314|válido|
|scored_0594|E1|6|309|postprocess 307, optimize_a4 307, nested_windows 307|307|válido|
|scored_0594|E2|52-1|325||297|válido|
|scored_0594|E2|52-3|315||288|válido|
|scored_0594|E2|52-6|309||284|válido|
|scored_0594|E2|64-1|325||297|válido|
|scored_0594|E2|64-3|315||288|válido|
|scored_0594|E2|64-6|309||285|válido|
|scored_0594|E3|joint|309|individual 308|308|rechazado por registros|
|scored_0594|E4|repair|309||309|completo|
|scored_0595|E1|1|148|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0595|E1|3|147|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0595|E1|6|147|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0595|E2|52-1|148||146|válido|
|scored_0595|E2|52-3|147||147|válido|
|scored_0595|E2|52-6|147||145|válido|
|scored_0595|E2|64-1|148||146|válido|
|scored_0595|E2|64-3|147||147|válido|
|scored_0595|E2|64-6|147||145|válido|
|scored_0595|E3|joint|147|individual 147|147|OPTIMAL|
|scored_0595|E4|repair|147||146|completo|
|scored_0596|E1|1|169|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0596|E1|3|164|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0596|E1|6|163|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0596|E2|52-1|169||161|válido|
|scored_0596|E2|52-3|164||153|válido|
|scored_0596|E2|52-6|163||153|válido|
|scored_0596|E2|64-1|169||161|válido|
|scored_0596|E2|64-3|164||154|válido|
|scored_0596|E2|64-6|163||153|válido|
|scored_0596|E3|joint|163|individual 162|162|OPTIMAL|
|scored_0596|E4|repair|163||163|completo|
|scored_0597|E1|1|5|postprocess 4, optimize_a4 3, nested_windows 3|3|válido|
|scored_0597|E1|3|5|postprocess 4, optimize_a4 3, nested_windows 3|3|válido|
|scored_0597|E1|6|5|postprocess 4, optimize_a4 3, nested_windows 3|3|válido|
|scored_0597|E2|52-1|5||5|válido|
|scored_0597|E2|52-3|5||5|válido|
|scored_0597|E2|52-6|5||5|válido|
|scored_0597|E2|64-1|5||5|válido|
|scored_0597|E2|64-3|5||5|válido|
|scored_0597|E2|64-6|5||5|válido|
|scored_0597|E3|joint|3|individual 3|3|OPTIMAL|
|scored_0597|E4|repair|3||3|completo|
|scored_0598|E1|1|103|postprocess 103, optimize_a4 103, nested_windows 103|103|válido|
|scored_0598|E1|3|103|postprocess 103, optimize_a4 103, nested_windows 103|103|válido|
|scored_0598|E1|6|103|postprocess 103, optimize_a4 103, nested_windows 103|103|válido|
|scored_0598|E2|52-1|103||101|válido|
|scored_0598|E2|52-3|103||101|válido|
|scored_0598|E2|52-6|103||101|válido|
|scored_0598|E2|64-1|103||101|válido|
|scored_0598|E2|64-3|103||101|válido|
|scored_0598|E2|64-6|103||101|válido|
|scored_0598|E3|joint|103|individual 103|103|OPTIMAL|
|scored_0598|E4|repair|103||101|completo|
|scored_0599|E1|1|237|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0599|E1|3|230|postprocess 230, optimize_a4 230, nested_windows 230|230|válido|
|scored_0599|E1|6|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0599|E2|52-1|237||228|válido|
|scored_0599|E2|52-3|230||202|válido|
|scored_0599|E2|52-6|226||196|válido|
|scored_0599|E2|64-1|237||228|válido|
|scored_0599|E2|64-3|230||202|válido|
|scored_0599|E2|64-6|226||200|válido|
|scored_0599|E3|joint|226|individual 225|224|OPTIMAL|
|scored_0599|E4|repair|226||226|completo|
|scored_0600|E1|1|3|postprocess 3, optimize_a4 3, nested_windows 3|3|válido|
|scored_0600|E1|3|3|postprocess 3, optimize_a4 3, nested_windows 3|3|válido|
|scored_0600|E1|6|3|postprocess 3, optimize_a4 3, nested_windows 3|3|válido|
|scored_0600|E2|52-1|3||3|válido|
|scored_0600|E2|52-3|3||3|válido|
|scored_0600|E2|52-6|3||3|válido|
|scored_0600|E2|64-1|3||3|válido|
|scored_0600|E2|64-3|3||3|válido|
|scored_0600|E2|64-6|3||3|válido|
|scored_0600|E3|joint|3|individual 3|3|OPTIMAL|
|scored_0600|E4|repair|3||3|completo|
|scored_0601|E1|1|153|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0601|E1|3|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0601|E1|6|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0601|E2|52-1|153||150|válido|
|scored_0601|E2|52-3|152||150|válido|
|scored_0601|E2|52-6|152||147|válido|
|scored_0601|E2|64-1|153||148|válido|
|scored_0601|E2|64-3|152||148|válido|
|scored_0601|E2|64-6|152||148|válido|
|scored_0601|E3|joint|152|individual 152|152|OPTIMAL|
|scored_0601|E4|repair|152||152|completo|
|scored_0602|E1|1|139|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0602|E1|3|139|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0602|E1|6|137|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|scored_0602|E2|52-1|139||136|válido|
|scored_0602|E2|52-3|139||135|válido|
|scored_0602|E2|52-6|137||135|válido|
|scored_0602|E2|64-1|139||136|válido|
|scored_0602|E2|64-3|139||135|válido|
|scored_0602|E2|64-6|137||134|válido|
|scored_0602|E3|joint|137|individual 137|137|OPTIMAL|
|scored_0602|E4|repair|137||135|completo|
|scored_0603|E1|1|208|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0603|E1|3|203|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0603|E1|6|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0603|E2|52-1|208||197|válido|
|scored_0603|E2|52-3|203||192|válido|
|scored_0603|E2|52-6|202||190|válido|
|scored_0603|E2|64-1|208||194|válido|
|scored_0603|E2|64-3|203||190|válido|
|scored_0603|E2|64-6|202||189|válido|
|scored_0603|E3|joint|202|individual 201|201|OPTIMAL|
|scored_0603|E4|repair|202||200|completo|
|scored_0604|E1|1|178|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0604|E1|3|178|postprocess 178, optimize_a4 178, nested_windows 178|178|válido|
|scored_0604|E1|6|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0604|E2|52-1|178||159|válido|
|scored_0604|E2|52-3|178||159|válido|
|scored_0604|E2|52-6|172||160|válido|
|scored_0604|E2|64-1|178||159|válido|
|scored_0604|E2|64-3|178||158|válido|
|scored_0604|E2|64-6|172||161|válido|
|scored_0604|E3|joint|172|individual 172|172|OPTIMAL|
|scored_0604|E4|repair|172||166|completo|
|scored_0605|E1|1|221|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0605|E1|3|224|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0605|E1|6|221|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0605|E2|52-1|221||210|válido|
|scored_0605|E2|52-3|224||208|válido|
|scored_0605|E2|52-6|221||207|válido|
|scored_0605|E2|64-1|221||209|válido|
|scored_0605|E2|64-3|224||207|válido|
|scored_0605|E2|64-6|221||207|válido|
|scored_0605|E3|joint|221|individual 221|219|OPTIMAL|
|scored_0605|E4|repair|221||221|completo|
|scored_0606|E1|1|233|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0606|E1|3|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0606|E1|6|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0606|E2|52-1|233||219|válido|
|scored_0606|E2|52-3|226||210|válido|
|scored_0606|E2|52-6|223||207|válido|
|scored_0606|E2|64-1|233||216|válido|
|scored_0606|E2|64-3|226||208|válido|
|scored_0606|E2|64-6|223||205|válido|
|scored_0606|E3|joint|223|individual 223|221|OPTIMAL|
|scored_0606|E4|repair|223||223|completo|
|scored_0607|E1|1|168|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0607|E1|3|166|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0607|E1|6|163|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0607|E2|52-1|168||157|válido|
|scored_0607|E2|52-3|166||156|válido|
|scored_0607|E2|52-6|163||156|válido|
|scored_0607|E2|64-1|168||156|válido|
|scored_0607|E2|64-3|166||156|válido|
|scored_0607|E2|64-6|163||155|válido|
|scored_0607|E3|joint|163|individual 163|163|OPTIMAL|
|scored_0607|E4|repair|163||161|completo|
|scored_0608|E1|1|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0608|E1|3|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0608|E1|6|201|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0608|E2|52-1|207||192|válido|
|scored_0608|E2|52-3|199||193|válido|
|scored_0608|E2|52-6|201||187|válido|
|scored_0608|E2|64-1|207||189|válido|
|scored_0608|E2|64-3|199||191|válido|
|scored_0608|E2|64-6|201||188|válido|
|scored_0608|E3|joint|199|individual 198|198|OPTIMAL|
|scored_0608|E4|repair|199||197|completo|
|scored_0609|E1|1|210|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0609|E1|3|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0609|E1|6|201|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0609|E2|52-1|210||195|válido|
|scored_0609|E2|52-3|206||192|válido|
|scored_0609|E2|52-6|201||188|válido|
|scored_0609|E2|64-1|210||195|válido|
|scored_0609|E2|64-3|206||192|válido|
|scored_0609|E2|64-6|201||189|válido|
|scored_0609|E3|joint|201|individual 200|200|OPTIMAL|
|scored_0609|E4|repair|201||201|completo|
|scored_0610|E1|1|393|postprocess 393, optimize_a4 393, nested_windows 393|393|válido|
|scored_0610|E1|3|370|postprocess 370, optimize_a4 370, nested_windows 370|370|válido|
|scored_0610|E1|6|360|postprocess 360, optimize_a4 360, nested_windows 360|360|válido|
|scored_0610|E2|52-1|393||351|válido|
|scored_0610|E2|52-3|370||335|válido|
|scored_0610|E2|52-6|360||336|válido|
|scored_0610|E2|64-1|393||347|válido|
|scored_0610|E2|64-3|370||339|válido|
|scored_0610|E2|64-6|360||336|válido|
|scored_0610|E3|joint|360|individual 360|358|OPTIMAL|
|scored_0610|E4|repair|360||360|completo|
|scored_0611|E1|1|42|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0611|E1|3|42|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0611|E1|6|42|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0611|E2|52-1|42||42|válido|
|scored_0611|E2|52-3|42||42|válido|
|scored_0611|E2|52-6|42||42|válido|
|scored_0611|E2|64-1|42||42|válido|
|scored_0611|E2|64-3|42||42|válido|
|scored_0611|E2|64-6|42||42|válido|
|scored_0611|E3|joint|42|individual 42|42|OPTIMAL|
|scored_0611|E4|repair|42||41|completo|
|scored_0612|E1|1|361|postprocess 361, optimize_a4 361, nested_windows 361|361|válido|
|scored_0612|E1|3|342|postprocess 341, optimize_a4 341, nested_windows 341|341|válido|
|scored_0612|E1|6|346|postprocess 346, optimize_a4 346, nested_windows 346|346|válido|
|scored_0612|E2|52-1|361||335|válido|
|scored_0612|E2|52-3|342||324|válido|
|scored_0612|E2|52-6|346||321|válido|
|scored_0612|E2|64-1|361||339|válido|
|scored_0612|E2|64-3|342||323|válido|
|scored_0612|E2|64-6|346||325|válido|
|scored_0612|E3|joint|342|individual 341|338|OPTIMAL|
|scored_0612|E4|repair|342||342|completo|
|scored_0613|E1|1|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0613|E1|3|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0613|E1|6|137|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0613|E2|52-1|137||133|válido|
|scored_0613|E2|52-3|137||133|válido|
|scored_0613|E2|52-6|137||133|válido|
|scored_0613|E2|64-1|137||135|válido|
|scored_0613|E2|64-3|137||134|válido|
|scored_0613|E2|64-6|137||134|válido|
|scored_0613|E3|joint|137|individual 137|137|OPTIMAL|
|scored_0613|E4|repair|137||135|completo|
|scored_0614|E1|1|32|postprocess 31, optimize_a4 30, nested_windows 30|30|válido|
|scored_0614|E1|3|32|postprocess 31, optimize_a4 30, nested_windows 30|30|válido|
|scored_0614|E1|6|32|postprocess 31, optimize_a4 30, nested_windows 30|30|válido|
|scored_0614|E2|52-1|32||32|válido|
|scored_0614|E2|52-3|32||32|válido|
|scored_0614|E2|52-6|32||32|válido|
|scored_0614|E2|64-1|32||32|válido|
|scored_0614|E2|64-3|32||32|válido|
|scored_0614|E2|64-6|32||32|válido|
|scored_0614|E3|joint|30|individual 30|30|OPTIMAL|
|scored_0614|E4|repair|30||28|completo|
|scored_0615|E1|1|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0615|E1|3|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0615|E1|6|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0615|E2|52-1|146||140|válido|
|scored_0615|E2|52-3|143||140|válido|
|scored_0615|E2|52-6|143||140|válido|
|scored_0615|E2|64-1|146||140|válido|
|scored_0615|E2|64-3|143||140|válido|
|scored_0615|E2|64-6|143||141|válido|
|scored_0615|E3|joint|143|individual 143|143|OPTIMAL|
|scored_0615|E4|repair|143||141|completo|
|scored_0616|E1|1|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0616|E1|3|204|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0616|E1|6|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0616|E2|52-1|215||190|válido|
|scored_0616|E2|52-3|204||186|válido|
|scored_0616|E2|52-6|199||186|válido|
|scored_0616|E2|64-1|215||190|válido|
|scored_0616|E2|64-3|204||186|válido|
|scored_0616|E2|64-6|199||186|válido|
|scored_0616|E3|joint|199|individual 199|199|OPTIMAL|
|scored_0616|E4|repair|199||197|completo|
|scored_0617|E1|1|138|postprocess 136, optimize_a4 136, nested_windows 136|136|válido|
|scored_0617|E1|3|139|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|scored_0617|E1|6|137|postprocess 136, optimize_a4 136, nested_windows 136|136|válido|
|scored_0617|E2|52-1|138||132|válido|
|scored_0617|E2|52-3|139||131|válido|
|scored_0617|E2|52-6|137||129|válido|
|scored_0617|E2|64-1|138||131|válido|
|scored_0617|E2|64-3|139||130|válido|
|scored_0617|E2|64-6|137||129|válido|
|scored_0617|E3|joint|137|individual 137|137|OPTIMAL|
|scored_0617|E4|repair|137||137|completo|
|scored_0618|E1|1|136|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|scored_0618|E1|3|134|postprocess 134, optimize_a4 134, nested_windows 134|134|válido|
|scored_0618|E1|6|134|postprocess 134, optimize_a4 134, nested_windows 134|134|válido|
|scored_0618|E2|52-1|136||132|válido|
|scored_0618|E2|52-3|134||130|válido|
|scored_0618|E2|52-6|134||130|válido|
|scored_0618|E2|64-1|136||132|válido|
|scored_0618|E2|64-3|134||131|válido|
|scored_0618|E2|64-6|134||131|válido|
|scored_0618|E3|joint|134|individual 134|134|OPTIMAL|
|scored_0618|E4|repair|134||130|completo|
|scored_0619|E1|1|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0619|E1|3|196|postprocess 195, optimize_a4 195, nested_windows 195|195|válido|
|scored_0619|E1|6|204|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0619|E2|52-1|202||193|válido|
|scored_0619|E2|52-3|196||193|válido|
|scored_0619|E2|52-6|204||193|válido|
|scored_0619|E2|64-1|202||197|válido|
|scored_0619|E2|64-3|196||196|válido|
|scored_0619|E2|64-6|204||191|válido|
|scored_0619|E3|joint|196|individual 196|196|OPTIMAL|
|scored_0619|E4|repair|196||195|completo|
|scored_0620|E1|1|52|postprocess 51, optimize_a4 51, nested_windows 51|51|válido|
|scored_0620|E1|3|52|postprocess 51, optimize_a4 51, nested_windows 51|51|válido|
|scored_0620|E1|6|52|postprocess 51, optimize_a4 51, nested_windows 51|51|válido|
|scored_0620|E2|52-1|52||52|válido|
|scored_0620|E2|52-3|52||52|válido|
|scored_0620|E2|52-6|52||52|válido|
|scored_0620|E2|64-1|52||52|válido|
|scored_0620|E2|64-3|52||52|válido|
|scored_0620|E2|64-6|52||52|válido|
|scored_0620|E3|joint|52|individual 52|52|OPTIMAL|
|scored_0620|E4|repair|52||50|completo|
|scored_0621|E1|1|241|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|scored_0621|E1|3|230|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0621|E1|6|227|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0621|E2|52-1|241||222|válido|
|scored_0621|E2|52-3|230||214|válido|
|scored_0621|E2|52-6|227||217|válido|
|scored_0621|E2|64-1|241||221|válido|
|scored_0621|E2|64-3|230||217|válido|
|scored_0621|E2|64-6|227||219|válido|
|scored_0621|E3|joint|227|individual 227|227|OPTIMAL|
|scored_0621|E4|repair|227||227|completo|
|scored_0622|E1|1|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0622|E1|3|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0622|E1|6|221|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0622|E2|52-1|233||213|válido|
|scored_0622|E2|52-3|226||210|válido|
|scored_0622|E2|52-6|221||201|válido|
|scored_0622|E2|64-1|233||219|válido|
|scored_0622|E2|64-3|226||210|válido|
|scored_0622|E2|64-6|221||203|válido|
|scored_0622|E3|joint|221|individual 221|220|OPTIMAL|
|scored_0622|E4|repair|221||220|completo|
|scored_0623|E1|1|103|postprocess 103, optimize_a4 103, nested_windows 103|103|válido|
|scored_0623|E1|3|103|postprocess 103, optimize_a4 103, nested_windows 103|103|válido|
|scored_0623|E1|6|103|postprocess 103, optimize_a4 103, nested_windows 103|103|válido|
|scored_0623|E2|52-1|103||102|válido|
|scored_0623|E2|52-3|103||102|válido|
|scored_0623|E2|52-6|103||102|válido|
|scored_0623|E2|64-1|103||104|válido|
|scored_0623|E2|64-3|103||104|válido|
|scored_0623|E2|64-6|103||104|válido|
|scored_0623|E3|joint|103|individual 103|103|OPTIMAL|
|scored_0623|E4|repair|103||102|completo|
|scored_0624|E1|1|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0624|E1|3|158|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0624|E1|6|156|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0624|E2|52-1|159||148|válido|
|scored_0624|E2|52-3|158||145|válido|
|scored_0624|E2|52-6|156||145|válido|
|scored_0624|E2|64-1|159||149|válido|
|scored_0624|E2|64-3|158||148|válido|
|scored_0624|E2|64-6|156||146|válido|
|scored_0624|E3|joint|156|individual 156|156|OPTIMAL|
|scored_0624|E4|repair|156||154|completo|
|scored_0625|E1|1|126|postprocess 126, optimize_a4 126, nested_windows 126|126|válido|
|scored_0625|E1|3|126|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0625|E1|6|123|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0625|E2|52-1|126||124|válido|
|scored_0625|E2|52-3|126||122|válido|
|scored_0625|E2|52-6|123||122|válido|
|scored_0625|E2|64-1|126||123|válido|
|scored_0625|E2|64-3|126||123|válido|
|scored_0625|E2|64-6|123||123|válido|
|scored_0625|E3|joint|123|individual 123|123|OPTIMAL|
|scored_0625|E4|repair|123||122|completo|
|scored_0626|E1|1|247|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|scored_0626|E1|3|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0626|E1|6|228|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|scored_0626|E2|52-1|247||226|válido|
|scored_0626|E2|52-3|235||219|válido|
|scored_0626|E2|52-6|228||218|válido|
|scored_0626|E2|64-1|247||221|válido|
|scored_0626|E2|64-3|235||217|válido|
|scored_0626|E2|64-6|228||218|válido|
|scored_0626|E3|joint|228|individual 228|228|OPTIMAL|
|scored_0626|E4|repair|228||228|completo|
|scored_0627|E1|1|184|postprocess 183, optimize_a4 183, nested_windows 183|183|válido|
|scored_0627|E1|3|184|postprocess 183, optimize_a4 183, nested_windows 183|183|válido|
|scored_0627|E1|6|181|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0627|E2|52-1|184||175|válido|
|scored_0627|E2|52-3|184||169|válido|
|scored_0627|E2|52-6|181||170|válido|
|scored_0627|E2|64-1|184||176|válido|
|scored_0627|E2|64-3|184||171|válido|
|scored_0627|E2|64-6|181||172|válido|
|scored_0627|E3|joint|181|individual 180|178|OPTIMAL|
|scored_0627|E4|repair|181||178|completo|
|scored_0628|E1|1|295|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|scored_0628|E1|3|278|postprocess 278, optimize_a4 278, nested_windows 278|278|válido|
|scored_0628|E1|6|276|postprocess 276, optimize_a4 276, nested_windows 276|276|válido|
|scored_0628|E2|52-1|295||273|válido|
|scored_0628|E2|52-3|278||261|válido|
|scored_0628|E2|52-6|276||256|válido|
|scored_0628|E2|64-1|295||272|válido|
|scored_0628|E2|64-3|278||261|válido|
|scored_0628|E2|64-6|276||255|válido|
|scored_0628|E3|joint|276|individual 276|274|FEASIBLE|
|scored_0628|E4|repair|276||276|completo|
|scored_0629|E1|1|214|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0629|E1|3|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0629|E1|6|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0629|E2|52-1|214||199|válido|
|scored_0629|E2|52-3|211||198|válido|
|scored_0629|E2|52-6|211||196|válido|
|scored_0629|E2|64-1|214||201|válido|
|scored_0629|E2|64-3|211||196|válido|
|scored_0629|E2|64-6|211||197|válido|
|scored_0629|E3|joint|211|individual 211|210|OPTIMAL|
|scored_0629|E4|repair|211||211|completo|
|scored_0630|E1|1|44|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0630|E1|3|44|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0630|E1|6|44|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0630|E2|52-1|44||44|válido|
|scored_0630|E2|52-3|44||44|válido|
|scored_0630|E2|52-6|44||44|válido|
|scored_0630|E2|64-1|44||44|válido|
|scored_0630|E2|64-3|44||44|válido|
|scored_0630|E2|64-6|44||44|válido|
|scored_0630|E3|joint|44|individual 44|44|OPTIMAL|
|scored_0630|E4|repair|44||42|completo|
|scored_0631|E1|1|175|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0631|E1|3|173|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|scored_0631|E1|6|173|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|scored_0631|E2|52-1|175||167|válido|
|scored_0631|E2|52-3|173||166|válido|
|scored_0631|E2|52-6|173||165|válido|
|scored_0631|E2|64-1|175||166|válido|
|scored_0631|E2|64-3|173||165|válido|
|scored_0631|E2|64-6|173||165|válido|
|scored_0631|E3|joint|173|individual 173|172|OPTIMAL|
|scored_0631|E4|repair|173||170|completo|
|scored_0632|E1|1|272|postprocess 272, optimize_a4 272, nested_windows 272|272|válido|
|scored_0632|E1|3|260|postprocess 260, optimize_a4 260, nested_windows 260|260|válido|
|scored_0632|E1|6|256|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|scored_0632|E2|52-1|272||245|válido|
|scored_0632|E2|52-3|260||241|válido|
|scored_0632|E2|52-6|256||238|válido|
|scored_0632|E2|64-1|272||246|válido|
|scored_0632|E2|64-3|260||242|válido|
|scored_0632|E2|64-6|256||236|válido|
|scored_0632|E3|joint|256|individual 256|256|OPTIMAL|
|scored_0632|E4|repair|256||256|completo|
|scored_0633|E1|1|228|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|scored_0633|E1|3|225|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0633|E1|6|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0633|E2|52-1|228||219|válido|
|scored_0633|E2|52-3|225||208|válido|
|scored_0633|E2|52-6|223||207|válido|
|scored_0633|E2|64-1|228||214|válido|
|scored_0633|E2|64-3|225||209|válido|
|scored_0633|E2|64-6|223||207|válido|
|scored_0633|E3|joint|223|individual 223|223|OPTIMAL|
|scored_0633|E4|repair|223||223|completo|
|scored_0634|E1|1|231|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|scored_0634|E1|3|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0634|E1|6|220|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0634|E2|52-1|231||216|válido|
|scored_0634|E2|52-3|219||207|válido|
|scored_0634|E2|52-6|220||207|válido|
|scored_0634|E2|64-1|231||217|válido|
|scored_0634|E2|64-3|219||209|válido|
|scored_0634|E2|64-6|220||210|válido|
|scored_0634|E3|joint|219|individual 219|219|OPTIMAL|
|scored_0634|E4|repair|219||219|completo|
|scored_0635|E1|1|50|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0635|E1|3|50|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0635|E1|6|50|postprocess 50, optimize_a4 50, nested_windows 50|50|válido|
|scored_0635|E2|52-1|50||50|válido|
|scored_0635|E2|52-3|50||50|válido|
|scored_0635|E2|52-6|50||50|válido|
|scored_0635|E2|64-1|50||50|válido|
|scored_0635|E2|64-3|50||50|válido|
|scored_0635|E2|64-6|50||50|válido|
|scored_0635|E3|joint|50|individual 50|50|OPTIMAL|
|scored_0635|E4|repair|50||46|completo|
|scored_0636|E1|1|353|postprocess 353, optimize_a4 353, nested_windows 353|353|válido|
|scored_0636|E1|3|335|postprocess 335, optimize_a4 335, nested_windows 335|335|válido|
|scored_0636|E1|6|331|postprocess 331, optimize_a4 331, nested_windows 331|331|válido|
|scored_0636|E2|52-1|353||329|válido|
|scored_0636|E2|52-3|335||316|válido|
|scored_0636|E2|52-6|331||314|válido|
|scored_0636|E2|64-1|353||330|válido|
|scored_0636|E2|64-3|335||317|válido|
|scored_0636|E2|64-6|331||315|válido|
|scored_0636|E3|joint|331|individual 331|329|OPTIMAL|
|scored_0636|E4|repair|331||331|completo|
|scored_0637|E1|1|43|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0637|E1|3|43|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0637|E1|6|43|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0637|E2|52-1|43||43|válido|
|scored_0637|E2|52-3|43||43|válido|
|scored_0637|E2|52-6|43||43|válido|
|scored_0637|E2|64-1|43||43|válido|
|scored_0637|E2|64-3|43||43|válido|
|scored_0637|E2|64-6|43||43|válido|
|scored_0637|E3|joint|43|individual 43|43|OPTIMAL|
|scored_0637|E4|repair|43||43|completo|
|scored_0638|E1|1|380|postprocess 379, optimize_a4 379, nested_windows 379|379|válido|
|scored_0638|E1|3|373|postprocess 372, optimize_a4 372, nested_windows 372|372|válido|
|scored_0638|E1|6|363|postprocess 363, optimize_a4 363, nested_windows 363|363|válido|
|scored_0638|E2|52-1|380||351|válido|
|scored_0638|E2|52-3|373||342|válido|
|scored_0638|E2|52-6|363||341|válido|
|scored_0638|E2|64-1|380||357|válido|
|scored_0638|E2|64-3|373||347|válido|
|scored_0638|E2|64-6|363||341|válido|
|scored_0638|E3|joint|363|individual 362|362|FEASIBLE|
|scored_0638|E4|repair|363||363|completo|
|scored_0639|E1|1|164|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0639|E1|3|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0639|E1|6|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0639|E2|52-1|164||157|válido|
|scored_0639|E2|52-3|165||155|válido|
|scored_0639|E2|52-6|162||153|válido|
|scored_0639|E2|64-1|164||153|válido|
|scored_0639|E2|64-3|165||153|válido|
|scored_0639|E2|64-6|162||153|válido|
|scored_0639|E3|joint|162|individual 162|162|OPTIMAL|
|scored_0639|E4|repair|162||160|completo|
|scored_0640|E1|1|302|postprocess 302, optimize_a4 302, nested_windows 302|302|válido|
|scored_0640|E1|3|283|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|scored_0640|E1|6|285|postprocess 285, optimize_a4 285, nested_windows 285|285|válido|
|scored_0640|E2|52-1|302||276|válido|
|scored_0640|E2|52-3|283||266|válido|
|scored_0640|E2|52-6|285||260|válido|
|scored_0640|E2|64-1|302||276|válido|
|scored_0640|E2|64-3|283||265|válido|
|scored_0640|E2|64-6|285||261|válido|
|scored_0640|E3|joint|283|individual 283|281|OPTIMAL|
|scored_0640|E4|repair|283||283|completo|
|scored_0641|E1|1|150|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|scored_0641|E1|3|150|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|scored_0641|E1|6|149|postprocess 149, optimize_a4 149, nested_windows 149|149|válido|
|scored_0641|E2|52-1|150||143|válido|
|scored_0641|E2|52-3|150||142|válido|
|scored_0641|E2|52-6|149||142|válido|
|scored_0641|E2|64-1|150||142|válido|
|scored_0641|E2|64-3|150||139|válido|
|scored_0641|E2|64-6|149||139|válido|
|scored_0641|E3|joint|149|individual 149|149|OPTIMAL|
|scored_0641|E4|repair|149||149|completo|
|scored_0642|E1|1|161|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0642|E1|3|161|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0642|E1|6|161|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0642|E2|52-1|161||153|válido|
|scored_0642|E2|52-3|161||153|válido|
|scored_0642|E2|52-6|161||152|válido|
|scored_0642|E2|64-1|161||154|válido|
|scored_0642|E2|64-3|161||152|válido|
|scored_0642|E2|64-6|161||152|válido|
|scored_0642|E3|joint|161|individual 161|161|OPTIMAL|
|scored_0642|E4|repair|161||158|completo|
|scored_0643|E1|1|266|postprocess 266, optimize_a4 266, nested_windows 266|266|válido|
|scored_0643|E1|3|250|postprocess 250, optimize_a4 250, nested_windows 250|250|válido|
|scored_0643|E1|6|255|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|scored_0643|E2|52-1|266||233|válido|
|scored_0643|E2|52-3|250||238|válido|
|scored_0643|E2|52-6|255||229|válido|
|scored_0643|E2|64-1|266||241|válido|
|scored_0643|E2|64-3|250||238|válido|
|scored_0643|E2|64-6|255||229|válido|
|scored_0643|E3|joint|250|individual 250|247|OPTIMAL|
|scored_0643|E4|repair|250||244|completo|
|scored_0644|E1|1|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0644|E1|3|184|postprocess 184, optimize_a4 184, nested_windows 184|184|válido|
|scored_0644|E1|6|186|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0644|E2|52-1|192||182|válido|
|scored_0644|E2|52-3|184||176|válido|
|scored_0644|E2|52-6|186||176|válido|
|scored_0644|E2|64-1|192||182|válido|
|scored_0644|E2|64-3|184||174|válido|
|scored_0644|E2|64-6|186||174|válido|
|scored_0644|E3|joint|184|individual 184|183|OPTIMAL|
|scored_0644|E4|repair|184||184|completo|
|scored_0645|E1|1|273|postprocess 273, optimize_a4 273, nested_windows 273|273|válido|
|scored_0645|E1|3|265|postprocess 265, optimize_a4 265, nested_windows 265|265|válido|
|scored_0645|E1|6|260|postprocess 260, optimize_a4 260, nested_windows 260|260|válido|
|scored_0645|E2|52-1|273||248|válido|
|scored_0645|E2|52-3|265||246|válido|
|scored_0645|E2|52-6|260||247|válido|
|scored_0645|E2|64-1|273||249|válido|
|scored_0645|E2|64-3|265||247|válido|
|scored_0645|E2|64-6|260||246|válido|
|scored_0645|E3|joint|260|individual 260|259|OPTIMAL|
|scored_0645|E4|repair|260||260|completo|
|scored_0646|E1|1|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0646|E1|3|227|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0646|E1|6|227|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0646|E2|52-1|233||213|válido|
|scored_0646|E2|52-3|227||204|válido|
|scored_0646|E2|52-6|227||206|válido|
|scored_0646|E2|64-1|233||212|válido|
|scored_0646|E2|64-3|227||205|válido|
|scored_0646|E2|64-6|227||203|válido|
|scored_0646|E3|joint|227|individual 227|227|OPTIMAL|
|scored_0646|E4|repair|227||226|completo|
|scored_0647|E1|1|296|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0647|E1|3|284|postprocess 284, optimize_a4 284, nested_windows 284|284|válido|
|scored_0647|E1|6|277|postprocess 277, optimize_a4 277, nested_windows 277|277|válido|
|scored_0647|E2|52-1|296||273|válido|
|scored_0647|E2|52-3|284||264|válido|
|scored_0647|E2|52-6|277||259|válido|
|scored_0647|E2|64-1|296||275|válido|
|scored_0647|E2|64-3|284||267|válido|
|scored_0647|E2|64-6|277||260|válido|
|scored_0647|E3|joint|277|individual 277|277|OPTIMAL|
|scored_0647|E4|repair|277||277|completo|
|scored_0648|E1|1|127|postprocess 127, optimize_a4 127, nested_windows 127|127|válido|
|scored_0648|E1|3|126|postprocess 126, optimize_a4 126, nested_windows 126|126|válido|
|scored_0648|E1|6|126|postprocess 126, optimize_a4 126, nested_windows 126|126|válido|
|scored_0648|E2|52-1|127||127|válido|
|scored_0648|E2|52-3|126||127|válido|
|scored_0648|E2|52-6|126||127|válido|
|scored_0648|E2|64-1|127||129|válido|
|scored_0648|E2|64-3|126||128|válido|
|scored_0648|E2|64-6|126||128|válido|
|scored_0648|E3|joint|126|individual 126|126|OPTIMAL|
|scored_0648|E4|repair|126||124|completo|
|scored_0649|E1|1|154|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0649|E1|3|154|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0649|E1|6|154|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0649|E2|52-1|154||152|válido|
|scored_0649|E2|52-3|154||152|válido|
|scored_0649|E2|52-6|154||151|válido|
|scored_0649|E2|64-1|154||152|válido|
|scored_0649|E2|64-3|154||152|válido|
|scored_0649|E2|64-6|154||152|válido|
|scored_0649|E3|joint|154|individual 154|154|OPTIMAL|
|scored_0649|E4|repair|154||154|completo|
|scored_0650|E1|1|45|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0650|E1|3|45|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0650|E1|6|45|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0650|E2|52-1|45||45|válido|
|scored_0650|E2|52-3|45||45|válido|
|scored_0650|E2|52-6|45||45|válido|
|scored_0650|E2|64-1|45||45|válido|
|scored_0650|E2|64-3|45||45|válido|
|scored_0650|E2|64-6|45||45|válido|
|scored_0650|E3|joint|45|individual 45|45|OPTIMAL|
|scored_0650|E4|repair|45||44|completo|
|scored_0651|E1|1|241|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|scored_0651|E1|3|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0651|E1|6|228|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0651|E2|52-1|241||223|válido|
|scored_0651|E2|52-3|235||209|válido|
|scored_0651|E2|52-6|228||206|válido|
|scored_0651|E2|64-1|241||222|válido|
|scored_0651|E2|64-3|235||212|válido|
|scored_0651|E2|64-6|228||208|válido|
|scored_0651|E3|joint|228|individual 228|228|OPTIMAL|
|scored_0651|E4|repair|228||228|completo|
|scored_0652|E1|1|130|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|scored_0652|E1|3|130|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|scored_0652|E1|6|130|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|scored_0652|E2|52-1|130||127|válido|
|scored_0652|E2|52-3|130||127|válido|
|scored_0652|E2|52-6|130||127|válido|
|scored_0652|E2|64-1|130||127|válido|
|scored_0652|E2|64-3|130||127|válido|
|scored_0652|E2|64-6|130||127|válido|
|scored_0652|E3|joint|130|individual 130|130|OPTIMAL|
|scored_0652|E4|repair|130||127|completo|
|scored_0653|E1|1|233|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|scored_0653|E1|3|226|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0653|E1|6|228|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0653|E2|52-1|233||219|válido|
|scored_0653|E2|52-3|226||216|válido|
|scored_0653|E2|52-6|228||212|válido|
|scored_0653|E2|64-1|233||218|válido|
|scored_0653|E2|64-3|226||219|válido|
|scored_0653|E2|64-6|228||209|válido|
|scored_0653|E3|joint|226|individual 225|225|OPTIMAL|
|scored_0653|E4|repair|226||224|completo|
|scored_0654|E1|1|48|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0654|E1|3|48|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0654|E1|6|48|postprocess 44, optimize_a4 44, nested_windows 44|44|válido|
|scored_0654|E2|52-1|48||48|válido|
|scored_0654|E2|52-3|48||48|válido|
|scored_0654|E2|52-6|48||48|válido|
|scored_0654|E2|64-1|48||48|válido|
|scored_0654|E2|64-3|48||48|válido|
|scored_0654|E2|64-6|48||48|válido|
|scored_0654|E3|joint|48|individual 48|48|OPTIMAL|
|scored_0654|E4|repair|48||44|completo|
|scored_0655|E1|1|367|postprocess 366, optimize_a4 366, nested_windows 366|366|válido|
|scored_0655|E1|3|353|postprocess 353, optimize_a4 353, nested_windows 353|353|válido|
|scored_0655|E1|6|345|postprocess 345, optimize_a4 345, nested_windows 345|345|válido|
|scored_0655|E2|52-1|367||341|válido|
|scored_0655|E2|52-3|353||325|válido|
|scored_0655|E2|52-6|345||326|válido|
|scored_0655|E2|64-1|367||346|válido|
|scored_0655|E2|64-3|353||326|válido|
|scored_0655|E2|64-6|345||327|válido|
|scored_0655|E3|joint|345|individual 345|343|OPTIMAL|
|scored_0655|E4|repair|345||345|completo|
|scored_0656|E1|1|51|postprocess 48, optimize_a4 47, nested_windows 47|47|válido|
|scored_0656|E1|3|51|postprocess 48, optimize_a4 47, nested_windows 47|47|válido|
|scored_0656|E1|6|51|postprocess 48, optimize_a4 47, nested_windows 47|47|válido|
|scored_0656|E2|52-1|51||51|válido|
|scored_0656|E2|52-3|51||51|válido|
|scored_0656|E2|52-6|51||51|válido|
|scored_0656|E2|64-1|51||51|válido|
|scored_0656|E2|64-3|51||51|válido|
|scored_0656|E2|64-6|51||51|válido|
|scored_0656|E3|joint|51|individual 51|51|OPTIMAL|
|scored_0656|E4|repair|51||51|completo|
|scored_0657|E1|1|125|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0657|E1|3|125|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0657|E1|6|124|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|scored_0657|E2|52-1|125||122|válido|
|scored_0657|E2|52-3|125||122|válido|
|scored_0657|E2|52-6|124||122|válido|
|scored_0657|E2|64-1|125||122|válido|
|scored_0657|E2|64-3|125||122|válido|
|scored_0657|E2|64-6|124||122|válido|
|scored_0657|E3|joint|124|individual 124|124|OPTIMAL|
|scored_0657|E4|repair|124||120|completo|
|scored_0658|E1|1|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0658|E1|3|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0658|E1|6|227|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0658|E2|52-1|235||219|válido|
|scored_0658|E2|52-3|232||218|válido|
|scored_0658|E2|52-6|227||215|válido|
|scored_0658|E2|64-1|235||222|válido|
|scored_0658|E2|64-3|232||217|válido|
|scored_0658|E2|64-6|227||216|válido|
|scored_0658|E3|joint|227|individual 227|227|OPTIMAL|
|scored_0658|E4|repair|227||227|completo|
|scored_0659|E1|1|110|postprocess 110, optimize_a4 109, nested_windows 109|109|válido|
|scored_0659|E1|3|110|postprocess 110, optimize_a4 109, nested_windows 109|109|válido|
|scored_0659|E1|6|110|postprocess 110, optimize_a4 109, nested_windows 109|109|válido|
|scored_0659|E2|52-1|110||109|válido|
|scored_0659|E2|52-3|110||109|válido|
|scored_0659|E2|52-6|110||109|válido|
|scored_0659|E2|64-1|110||108|válido|
|scored_0659|E2|64-3|110||108|válido|
|scored_0659|E2|64-6|110||108|válido|
|scored_0659|E3|joint|110|individual 110|110|OPTIMAL|
|scored_0659|E4|repair|110||110|completo|
|scored_0660|E1|1|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|scored_0660|E1|3|230|postprocess 230, optimize_a4 230, nested_windows 230|230|válido|
|scored_0660|E1|6|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0660|E2|52-1|238||218|válido|
|scored_0660|E2|52-3|230||217|válido|
|scored_0660|E2|52-6|226||215|válido|
|scored_0660|E2|64-1|238||219|válido|
|scored_0660|E2|64-3|230||221|válido|
|scored_0660|E2|64-6|226||213|válido|
|scored_0660|E3|joint|226|individual 226|225|OPTIMAL|
|scored_0660|E4|repair|226||226|completo|
|scored_0661|E1|1|314|postprocess 313, optimize_a4 313, nested_windows 313|313|válido|
|scored_0661|E1|3|299|postprocess 299, optimize_a4 299, nested_windows 299|299|válido|
|scored_0661|E1|6|297|postprocess 297, optimize_a4 297, nested_windows 297|297|válido|
|scored_0661|E2|52-1|314||295|válido|
|scored_0661|E2|52-3|299||286|válido|
|scored_0661|E2|52-6|297||282|válido|
|scored_0661|E2|64-1|314||297|válido|
|scored_0661|E2|64-3|299||290|válido|
|scored_0661|E2|64-6|297||357|válido|
|scored_0661|E3|joint|297|individual 297|296|OPTIMAL|
|scored_0661|E4|repair|297||297|completo|
|scored_0662|E1|1|238|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|scored_0662|E1|3|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0662|E1|6|224|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0662|E2|52-1|238||217|válido|
|scored_0662|E2|52-3|229||205|válido|
|scored_0662|E2|52-6|224||207|válido|
|scored_0662|E2|64-1|238||218|válido|
|scored_0662|E2|64-3|229||209|válido|
|scored_0662|E2|64-6|224||209|válido|
|scored_0662|E3|joint|224|individual 224|224|OPTIMAL|
|scored_0662|E4|repair|224||224|completo|
|scored_0663|E1|1|257|postprocess 257, optimize_a4 257, nested_windows 257|257|válido|
|scored_0663|E1|3|252|postprocess 252, optimize_a4 252, nested_windows 252|252|válido|
|scored_0663|E1|6|244|postprocess 244, optimize_a4 244, nested_windows 244|244|válido|
|scored_0663|E2|52-1|257||246|válido|
|scored_0663|E2|52-3|252||228|válido|
|scored_0663|E2|52-6|244||225|válido|
|scored_0663|E2|64-1|257||245|válido|
|scored_0663|E2|64-3|252||231|válido|
|scored_0663|E2|64-6|244||224|válido|
|scored_0663|E3|joint|244|individual 244|244|OPTIMAL|
|scored_0663|E4|repair|244||244|completo|
|scored_0664|E1|1|296|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|scored_0664|E1|3|277|postprocess 277, optimize_a4 277, nested_windows 277|277|válido|
|scored_0664|E1|6|278|postprocess 278, optimize_a4 278, nested_windows 278|278|válido|
|scored_0664|E2|52-1|296||278|válido|
|scored_0664|E2|52-3|277||269|válido|
|scored_0664|E2|52-6|278||287|válido|
|scored_0664|E2|64-1|296||278|válido|
|scored_0664|E2|64-3|277||269|válido|
|scored_0664|E2|64-6|278||294|válido|
|scored_0664|E3|joint|277|individual 276|275|OPTIMAL|
|scored_0664|E4|repair|277||277|completo|
|scored_0665|E1|1|127|postprocess 126, optimize_a4 126, nested_windows 126|126|válido|
|scored_0665|E1|3|127|postprocess 126, optimize_a4 126, nested_windows 126|126|válido|
|scored_0665|E1|6|127|postprocess 126, optimize_a4 126, nested_windows 126|126|válido|
|scored_0665|E2|52-1|127||118|válido|
|scored_0665|E2|52-3|127||118|válido|
|scored_0665|E2|52-6|127||118|válido|
|scored_0665|E2|64-1|127||118|válido|
|scored_0665|E2|64-3|127||118|válido|
|scored_0665|E2|64-6|127||118|válido|
|scored_0665|E3|joint|127|individual 127|127|OPTIMAL|
|scored_0665|E4|repair|127||121|completo|
|scored_0666|E1|1|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0666|E1|3|201|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0666|E1|6|198|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|scored_0666|E2|52-1|207||195|válido|
|scored_0666|E2|52-3|201||188|válido|
|scored_0666|E2|52-6|198||187|válido|
|scored_0666|E2|64-1|207||195|válido|
|scored_0666|E2|64-3|201||190|válido|
|scored_0666|E2|64-6|198||186|válido|
|scored_0666|E3|joint|198|individual 197|196|OPTIMAL|
|scored_0666|E4|repair|198||198|completo|
|scored_0667|E1|1|228|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0667|E1|3|226|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0667|E1|6|224|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0667|E2|52-1|228||214|válido|
|scored_0667|E2|52-3|226||213|válido|
|scored_0667|E2|52-6|224||212|válido|
|scored_0667|E2|64-1|228||217|válido|
|scored_0667|E2|64-3|226||210|válido|
|scored_0667|E2|64-6|224||211|válido|
|scored_0667|E3|joint|224|individual 224|224|OPTIMAL|
|scored_0667|E4|repair|224||223|completo|
|scored_0668|E1|1|158|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0668|E1|3|157|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0668|E1|6|155|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|scored_0668|E2|52-1|158||149|válido|
|scored_0668|E2|52-3|157||149|válido|
|scored_0668|E2|52-6|155||149|válido|
|scored_0668|E2|64-1|158||151|válido|
|scored_0668|E2|64-3|157||151|válido|
|scored_0668|E2|64-6|155||149|válido|
|scored_0668|E3|joint|155|individual 155|155|OPTIMAL|
|scored_0668|E4|repair|155||154|completo|
|scored_0669|E1|1|186|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0669|E1|3|183|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0669|E1|6|179|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|scored_0669|E2|52-1|186||175|válido|
|scored_0669|E2|52-3|183||167|válido|
|scored_0669|E2|52-6|179||161|válido|
|scored_0669|E2|64-1|186||174|válido|
|scored_0669|E2|64-3|183||169|válido|
|scored_0669|E2|64-6|179||166|válido|
|scored_0669|E3|joint|179|individual 179|179|OPTIMAL|
|scored_0669|E4|repair|179||176|completo|
|scored_0670|E1|1|252|postprocess 252, optimize_a4 252, nested_windows 252|252|válido|
|scored_0670|E1|3|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0670|E1|6|241|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|scored_0670|E2|52-1|252||233|válido|
|scored_0670|E2|52-3|242||228|válido|
|scored_0670|E2|52-6|241||222|válido|
|scored_0670|E2|64-1|252||232|válido|
|scored_0670|E2|64-3|242||227|válido|
|scored_0670|E2|64-6|241||224|válido|
|scored_0670|E3|joint|241|individual 240|239|OPTIMAL|
|scored_0670|E4|repair|241||241|completo|
|scored_0671|E1|1|273|postprocess 272, optimize_a4 272, nested_windows 272|272|válido|
|scored_0671|E1|3|269|postprocess 269, optimize_a4 269, nested_windows 269|269|válido|
|scored_0671|E1|6|259|postprocess 259, optimize_a4 259, nested_windows 259|259|válido|
|scored_0671|E2|52-1|273||247|válido|
|scored_0671|E2|52-3|269||240|válido|
|scored_0671|E2|52-6|259||242|válido|
|scored_0671|E2|64-1|273||246|válido|
|scored_0671|E2|64-3|269||244|válido|
|scored_0671|E2|64-6|259||242|válido|
|scored_0671|E3|joint|259|individual 259|258|OPTIMAL|
|scored_0671|E4|repair|259||259|completo|
|scored_0672|E1|1|74|postprocess 74, optimize_a4 74, nested_windows 74|74|válido|
|scored_0672|E1|3|75|postprocess 72, optimize_a4 72, nested_windows 72|72|válido|
|scored_0672|E1|6|75|postprocess 72, optimize_a4 72, nested_windows 72|72|válido|
|scored_0672|E2|52-1|74||74|válido|
|scored_0672|E2|52-3|75||75|válido|
|scored_0672|E2|52-6|75||75|válido|
|scored_0672|E2|64-1|74||74|válido|
|scored_0672|E2|64-3|75||75|válido|
|scored_0672|E2|64-6|75||75|válido|
|scored_0672|E3|joint|74|individual 74|74|OPTIMAL|
|scored_0672|E4|repair|74||74|completo|
|scored_0673|E1|1|224|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0673|E1|3|224|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0673|E1|6|222|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0673|E2|52-1|224||213|válido|
|scored_0673|E2|52-3|224||212|válido|
|scored_0673|E2|52-6|222||208|válido|
|scored_0673|E2|64-1|224||213|válido|
|scored_0673|E2|64-3|224||210|válido|
|scored_0673|E2|64-6|222||207|válido|
|scored_0673|E3|joint|222|individual 222|219|OPTIMAL|
|scored_0673|E4|repair|222||222|completo|
|scored_0674|E1|1|168|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|scored_0674|E1|3|167|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0674|E1|6|166|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0674|E2|52-1|168||164|válido|
|scored_0674|E2|52-3|167||161|válido|
|scored_0674|E2|52-6|166||156|válido|
|scored_0674|E2|64-1|168||162|válido|
|scored_0674|E2|64-3|167||161|válido|
|scored_0674|E2|64-6|166||155|válido|
|scored_0674|E3|joint|166|individual 166|166|OPTIMAL|
|scored_0674|E4|repair|166||163|completo|
|scored_0675|E1|1|201|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0675|E1|3|200|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0675|E1|6|195|postprocess 195, optimize_a4 195, nested_windows 195|195|válido|
|scored_0675|E2|52-1|201||181|válido|
|scored_0675|E2|52-3|200||185|válido|
|scored_0675|E2|52-6|195||184|válido|
|scored_0675|E2|64-1|201||181|válido|
|scored_0675|E2|64-3|200||184|válido|
|scored_0675|E2|64-6|195||183|válido|
|scored_0675|E3|joint|195|individual 195|194|OPTIMAL|
|scored_0675|E4|repair|195||188|completo|
|scored_0676|E1|1|7|postprocess 6, optimize_a4 6, nested_windows 6|6|válido|
|scored_0676|E1|3|7|postprocess 6, optimize_a4 6, nested_windows 6|6|válido|
|scored_0676|E1|6|7|postprocess 6, optimize_a4 6, nested_windows 6|6|válido|
|scored_0676|E2|52-1|7||7|válido|
|scored_0676|E2|52-3|7||7|válido|
|scored_0676|E2|52-6|7||7|válido|
|scored_0676|E2|64-1|7||7|válido|
|scored_0676|E2|64-3|7||7|válido|
|scored_0676|E2|64-6|7||7|válido|
|scored_0676|E3|joint|6|individual 6|6|OPTIMAL|
|scored_0676|E4|repair|6||6|completo|
|scored_0677|E1|1|318|postprocess 318, optimize_a4 318, nested_windows 318|318|válido|
|scored_0677|E1|3|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0677|E1|6|296|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0677|E2|52-1|318||298|válido|
|scored_0677|E2|52-3|303||286|válido|
|scored_0677|E2|52-6|296||285|válido|
|scored_0677|E2|64-1|318||297|válido|
|scored_0677|E2|64-3|303||289|válido|
|scored_0677|E2|64-6|296||287|válido|
|scored_0677|E3|joint|296|individual 296|296|FEASIBLE|
|scored_0677|E4|repair|296||296|completo|
|scored_0678|E1|1|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0678|E1|3|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0678|E1|6|221|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0678|E2|52-1|229||212|válido|
|scored_0678|E2|52-3|223||207|válido|
|scored_0678|E2|52-6|221||206|válido|
|scored_0678|E2|64-1|229||213|válido|
|scored_0678|E2|64-3|223||208|válido|
|scored_0678|E2|64-6|221||207|válido|
|scored_0678|E3|joint|221|individual 221|221|OPTIMAL|
|scored_0678|E4|repair|221||221|completo|
|scored_0679|E1|1|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0679|E1|3|155|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|scored_0679|E1|6|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0679|E2|52-1|159||147|válido|
|scored_0679|E2|52-3|155||147|válido|
|scored_0679|E2|52-6|151||147|válido|
|scored_0679|E2|64-1|159||146|válido|
|scored_0679|E2|64-3|155||146|válido|
|scored_0679|E2|64-6|151||146|válido|
|scored_0679|E3|joint|151|individual 151|151|OPTIMAL|
|scored_0679|E4|repair|151||147|completo|
|scored_0680|E1|1|287|postprocess 286, optimize_a4 286, nested_windows 286|286|válido|
|scored_0680|E1|3|274|postprocess 273, optimize_a4 273, nested_windows 273|273|válido|
|scored_0680|E1|6|266|postprocess 265, optimize_a4 265, nested_windows 265|265|válido|
|scored_0680|E2|52-1|287||253|válido|
|scored_0680|E2|52-3|274||250|válido|
|scored_0680|E2|52-6|266||245|válido|
|scored_0680|E2|64-1|287||255|válido|
|scored_0680|E2|64-3|274||249|válido|
|scored_0680|E2|64-6|266||246|válido|
|scored_0680|E3|joint|266|individual 265|265|OPTIMAL|
|scored_0680|E4|repair|266||266|completo|
|scored_0681|E1|1|312|postprocess 312, optimize_a4 312, nested_windows 312|312|válido|
|scored_0681|E1|3|287|postprocess 287, optimize_a4 287, nested_windows 287|287|válido|
|scored_0681|E1|6|282|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|scored_0681|E2|52-1|312||273|válido|
|scored_0681|E2|52-3|287||255|válido|
|scored_0681|E2|52-6|282||254|válido|
|scored_0681|E2|64-1|312||273|válido|
|scored_0681|E2|64-3|287||258|válido|
|scored_0681|E2|64-6|282||255|válido|
|scored_0681|E3|joint|282|individual 282|280|FEASIBLE|
|scored_0681|E4|repair|282||282|completo|
|scored_0682|E1|1|381|postprocess 381, optimize_a4 381, nested_windows 381|381|válido|
|scored_0682|E1|3|354|postprocess 354, optimize_a4 354, nested_windows 354|354|válido|
|scored_0682|E1|6|351|postprocess 351, optimize_a4 351, nested_windows 351|351|válido|
|scored_0682|E2|52-1|381||346|válido|
|scored_0682|E2|52-3|354||334|válido|
|scored_0682|E2|52-6|351||327|válido|
|scored_0682|E2|64-1|381||354|válido|
|scored_0682|E2|64-3|354||341|válido|
|scored_0682|E2|64-6|351||329|válido|
|scored_0682|E3|joint|351|individual 351|346|OPTIMAL|
|scored_0682|E4|repair|351||351|completo|
|scored_0683|E1|1|193|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|scored_0683|E1|3|191|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0683|E1|6|191|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0683|E2|52-1|193||181|válido|
|scored_0683|E2|52-3|191||180|válido|
|scored_0683|E2|52-6|191||180|válido|
|scored_0683|E2|64-1|193||182|válido|
|scored_0683|E2|64-3|191||182|válido|
|scored_0683|E2|64-6|191||182|válido|
|scored_0683|E3|joint|191|individual 190|190|OPTIMAL|
|scored_0683|E4|repair|191||189|completo|
|scored_0684|E1|1|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0684|E1|3|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0684|E1|6|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0684|E2|52-1|143||140|válido|
|scored_0684|E2|52-3|143||140|válido|
|scored_0684|E2|52-6|143||139|válido|
|scored_0684|E2|64-1|143||139|válido|
|scored_0684|E2|64-3|143||139|válido|
|scored_0684|E2|64-6|143||139|válido|
|scored_0684|E3|joint|143|individual 143|143|OPTIMAL|
|scored_0684|E4|repair|143||141|completo|
|scored_0685|E1|1|243|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|scored_0685|E1|3|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|scored_0685|E1|6|233|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0685|E2|52-1|243||230|válido|
|scored_0685|E2|52-3|238||220|válido|
|scored_0685|E2|52-6|233||220|válido|
|scored_0685|E2|64-1|243||231|válido|
|scored_0685|E2|64-3|238||226|válido|
|scored_0685|E2|64-6|233||222|válido|
|scored_0685|E3|joint|233|individual 233|233|OPTIMAL|
|scored_0685|E4|repair|233||233|completo|
|scored_0686|E1|1|125|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0686|E1|3|125|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|scored_0686|E1|6|125|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|scored_0686|E2|52-1|125||117|válido|
|scored_0686|E2|52-3|125||117|válido|
|scored_0686|E2|52-6|125||117|válido|
|scored_0686|E2|64-1|125||119|válido|
|scored_0686|E2|64-3|125||119|válido|
|scored_0686|E2|64-6|125||119|válido|
|scored_0686|E3|joint|125|individual 125|125|OPTIMAL|
|scored_0686|E4|repair|125||123|completo|
|scored_0687|E1|1|324|postprocess 323, optimize_a4 323, nested_windows 323|323|válido|
|scored_0687|E1|3|313|postprocess 312, optimize_a4 312, nested_windows 312|312|válido|
|scored_0687|E1|6|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0687|E2|52-1|324||304|válido|
|scored_0687|E2|52-3|313||290|válido|
|scored_0687|E2|52-6|303||286|válido|
|scored_0687|E2|64-1|324||305|válido|
|scored_0687|E2|64-3|313||293|válido|
|scored_0687|E2|64-6|303||286|válido|
|scored_0687|E3|joint|303|individual 303|303|OPTIMAL|
|scored_0687|E4|repair|303||303|completo|
|scored_0688|E1|1|227|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0688|E1|3|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0688|E1|6|214|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0688|E2|52-1|227||210|válido|
|scored_0688|E2|52-3|220||202|válido|
|scored_0688|E2|52-6|214||204|válido|
|scored_0688|E2|64-1|227||207|válido|
|scored_0688|E2|64-3|220||203|válido|
|scored_0688|E2|64-6|214||206|válido|
|scored_0688|E3|joint|214|individual 214|213|OPTIMAL|
|scored_0688|E4|repair|214||214|completo|
|scored_0689|E1|1|254|postprocess 254, optimize_a4 254, nested_windows 254|254|válido|
|scored_0689|E1|3|241|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|scored_0689|E1|6|243|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|scored_0689|E2|52-1|254||233|válido|
|scored_0689|E2|52-3|241||227|válido|
|scored_0689|E2|52-6|243||225|válido|
|scored_0689|E2|64-1|254||236|válido|
|scored_0689|E2|64-3|241||228|válido|
|scored_0689|E2|64-6|243||228|válido|
|scored_0689|E3|joint|241|individual 241|240|OPTIMAL|
|scored_0689|E4|repair|241||241|completo|
|scored_0690|E1|1|381|postprocess 381, optimize_a4 381, nested_windows 381|381|válido|
|scored_0690|E1|3|365|postprocess 364, optimize_a4 364, nested_windows 364|364|válido|
|scored_0690|E1|6|350|postprocess 350, optimize_a4 350, nested_windows 350|350|válido|
|scored_0690|E2|52-1|381||351|válido|
|scored_0690|E2|52-3|365||340|válido|
|scored_0690|E2|52-6|350||331|válido|
|scored_0690|E2|64-1|381||353|válido|
|scored_0690|E2|64-3|365||343|válido|
|scored_0690|E2|64-6|350||333|válido|
|scored_0690|E3|joint|350|individual 350|350|FEASIBLE|
|scored_0690|E4|repair|350||350|completo|
|scored_0691|E1|1|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0691|E1|3|189|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0691|E1|6|186|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0691|E2|52-1|192||178|válido|
|scored_0691|E2|52-3|189||174|válido|
|scored_0691|E2|52-6|186||177|válido|
|scored_0691|E2|64-1|192||180|válido|
|scored_0691|E2|64-3|189||176|válido|
|scored_0691|E2|64-6|186||176|válido|
|scored_0691|E3|joint|186|individual 186|186|OPTIMAL|
|scored_0691|E4|repair|186||185|completo|
|scored_0692|E1|1|291|postprocess 291, optimize_a4 291, nested_windows 291|291|válido|
|scored_0692|E1|3|268|postprocess 268, optimize_a4 268, nested_windows 268|268|válido|
|scored_0692|E1|6|270|postprocess 270, optimize_a4 270, nested_windows 270|270|válido|
|scored_0692|E2|52-1|291||263|válido|
|scored_0692|E2|52-3|268||254|válido|
|scored_0692|E2|52-6|270||249|válido|
|scored_0692|E2|64-1|291||262|válido|
|scored_0692|E2|64-3|268||255|válido|
|scored_0692|E2|64-6|270||249|válido|
|scored_0692|E3|joint|268|individual 268|265|OPTIMAL|
|scored_0692|E4|repair|268||268|completo|
|scored_0693|E1|1|236|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0693|E1|3|234|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0693|E1|6|228|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|scored_0693|E2|52-1|236||213|válido|
|scored_0693|E2|52-3|234||208|válido|
|scored_0693|E2|52-6|228||209|válido|
|scored_0693|E2|64-1|236||213|válido|
|scored_0693|E2|64-3|234||211|válido|
|scored_0693|E2|64-6|228||209|válido|
|scored_0693|E3|joint|228|individual 228|228|OPTIMAL|
|scored_0693|E4|repair|228||227|completo|
|scored_0694|E1|1|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0694|E1|3|208|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0694|E1|6|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0694|E2|52-1|219||201|válido|
|scored_0694|E2|52-3|208||197|válido|
|scored_0694|E2|52-6|207||193|válido|
|scored_0694|E2|64-1|219||200|válido|
|scored_0694|E2|64-3|208||198|válido|
|scored_0694|E2|64-6|207||192|válido|
|scored_0694|E3|joint|207|individual 207|206|OPTIMAL|
|scored_0694|E4|repair|207||207|completo|
|scored_0695|E1|1|155|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|scored_0695|E1|3|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0695|E1|6|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0695|E2|52-1|155||147|válido|
|scored_0695|E2|52-3|152||145|válido|
|scored_0695|E2|52-6|151||144|válido|
|scored_0695|E2|64-1|155||145|válido|
|scored_0695|E2|64-3|152||145|válido|
|scored_0695|E2|64-6|151||142|válido|
|scored_0695|E3|joint|151|individual 151|151|OPTIMAL|
|scored_0695|E4|repair|151||150|completo|
|scored_0696|E1|1|106|postprocess 106, optimize_a4 106, nested_windows 106|106|válido|
|scored_0696|E1|3|106|postprocess 106, optimize_a4 106, nested_windows 106|106|válido|
|scored_0696|E1|6|106|postprocess 106, optimize_a4 106, nested_windows 106|106|válido|
|scored_0696|E2|52-1|106||104|válido|
|scored_0696|E2|52-3|106||104|válido|
|scored_0696|E2|52-6|106||104|válido|
|scored_0696|E2|64-1|106||104|válido|
|scored_0696|E2|64-3|106||104|válido|
|scored_0696|E2|64-6|106||104|válido|
|scored_0696|E3|joint|106|individual 106|106|OPTIMAL|
|scored_0696|E4|repair|106||101|completo|
|scored_0697|E1|1|144|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0697|E1|3|144|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0697|E1|6|143|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0697|E2|52-1|144||137|válido|
|scored_0697|E2|52-3|144||135|válido|
|scored_0697|E2|52-6|143||135|válido|
|scored_0697|E2|64-1|144||137|válido|
|scored_0697|E2|64-3|144||135|válido|
|scored_0697|E2|64-6|143||135|válido|
|scored_0697|E3|joint|143|individual 143|143|OPTIMAL|
|scored_0697|E4|repair|143||138|completo|
|scored_0698|E1|1|347|postprocess 347, optimize_a4 347, nested_windows 347|347|válido|
|scored_0698|E1|3|324|postprocess 324, optimize_a4 324, nested_windows 324|324|válido|
|scored_0698|E1|6|315|postprocess 313, optimize_a4 313, nested_windows 313|313|válido|
|scored_0698|E2|52-1|347||302|válido|
|scored_0698|E2|52-3|324||294|válido|
|scored_0698|E2|52-6|315||291|válido|
|scored_0698|E2|64-1|347||309|válido|
|scored_0698|E2|64-3|324||294|válido|
|scored_0698|E2|64-6|315||293|válido|
|scored_0698|E3|joint|315|individual 315|315|rechazado por registros|
|scored_0698|E4|repair|315||315|completo|
|scored_0699|E1|1|196|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|scored_0699|E1|3|191|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|scored_0699|E1|6|190|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|scored_0699|E2|52-1|196||177|válido|
|scored_0699|E2|52-3|191||175|válido|
|scored_0699|E2|52-6|190||176|válido|
|scored_0699|E2|64-1|196||177|válido|
|scored_0699|E2|64-3|191||175|válido|
|scored_0699|E2|64-6|190||176|válido|
|scored_0699|E3|joint|190|individual 189|189|OPTIMAL|
|scored_0699|E4|repair|190||187|completo|
|scored_0700|E1|1|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0700|E1|3|225|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0700|E1|6|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0700|E2|52-1|233||221|válido|
|scored_0700|E2|52-3|225||212|válido|
|scored_0700|E2|52-6|220||213|válido|
|scored_0700|E2|64-1|233||220|válido|
|scored_0700|E2|64-3|225||213|válido|
|scored_0700|E2|64-6|220||213|válido|
|scored_0700|E3|joint|220|individual 220|220|OPTIMAL|
|scored_0700|E4|repair|220||220|completo|
|scored_0701|E1|1|41|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0701|E1|3|41|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0701|E1|6|41|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0701|E2|52-1|41||41|válido|
|scored_0701|E2|52-3|41||41|válido|
|scored_0701|E2|52-6|41||41|válido|
|scored_0701|E2|64-1|41||41|válido|
|scored_0701|E2|64-3|41||41|válido|
|scored_0701|E2|64-6|41||41|válido|
|scored_0701|E3|joint|41|individual 41|41|OPTIMAL|
|scored_0701|E4|repair|41||41|completo|
|scored_0702|E1|1|306|postprocess 306, optimize_a4 306, nested_windows 306|306|válido|
|scored_0702|E1|3|279|postprocess 279, optimize_a4 279, nested_windows 279|279|válido|
|scored_0702|E1|6|272|postprocess 272, optimize_a4 272, nested_windows 272|272|válido|
|scored_0702|E2|52-1|306||277|válido|
|scored_0702|E2|52-3|279||267|válido|
|scored_0702|E2|52-6|272||262|válido|
|scored_0702|E2|64-1|306||277|válido|
|scored_0702|E2|64-3|279||267|válido|
|scored_0702|E2|64-6|272||263|válido|
|scored_0702|E3|joint|272|individual 272|272|OPTIMAL|
|scored_0702|E4|repair|272||272|completo|
|scored_0703|E1|1|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0703|E1|3|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0703|E1|6|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0703|E2|52-1|235||214|válido|
|scored_0703|E2|52-3|229||214|válido|
|scored_0703|E2|52-6|223||214|válido|
|scored_0703|E2|64-1|235||217|válido|
|scored_0703|E2|64-3|229||215|válido|
|scored_0703|E2|64-6|223||214|válido|
|scored_0703|E3|joint|223|individual 223|223|OPTIMAL|
|scored_0703|E4|repair|223||223|completo|
|scored_0704|E1|1|166|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0704|E1|3|165|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0704|E1|6|166|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0704|E2|52-1|166||161|válido|
|scored_0704|E2|52-3|165||156|válido|
|scored_0704|E2|52-6|166||156|válido|
|scored_0704|E2|64-1|166||158|válido|
|scored_0704|E2|64-3|165||157|válido|
|scored_0704|E2|64-6|166||157|válido|
|scored_0704|E3|joint|165|individual 165|165|OPTIMAL|
|scored_0704|E4|repair|165||162|completo|
|scored_0705|E1|1|218|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|scored_0705|E1|3|209|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0705|E1|6|208|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|scored_0705|E2|52-1|218||200|válido|
|scored_0705|E2|52-3|209||192|válido|
|scored_0705|E2|52-6|208||191|válido|
|scored_0705|E2|64-1|218||202|válido|
|scored_0705|E2|64-3|209||192|válido|
|scored_0705|E2|64-6|208||192|válido|
|scored_0705|E3|joint|208|individual 208|208|OPTIMAL|
|scored_0705|E4|repair|208||208|completo|
|scored_0706|E1|1|263|postprocess 262, optimize_a4 262, nested_windows 262|262|válido|
|scored_0706|E1|3|261|postprocess 261, optimize_a4 261, nested_windows 261|261|válido|
|scored_0706|E1|6|247|postprocess 245, optimize_a4 245, nested_windows 245|245|válido|
|scored_0706|E2|52-1|263||242|válido|
|scored_0706|E2|52-3|261||237|válido|
|scored_0706|E2|52-6|247||229|válido|
|scored_0706|E2|64-1|263||241|válido|
|scored_0706|E2|64-3|261||235|válido|
|scored_0706|E2|64-6|247||228|válido|
|scored_0706|E3|joint|247|individual 247|247|OPTIMAL|
|scored_0706|E4|repair|247||247|completo|
|scored_0707|E1|1|122|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0707|E1|3|122|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0707|E1|6|122|postprocess 122, optimize_a4 122, nested_windows 122|122|válido|
|scored_0707|E2|52-1|122||117|válido|
|scored_0707|E2|52-3|122||117|válido|
|scored_0707|E2|52-6|122||117|válido|
|scored_0707|E2|64-1|122||119|válido|
|scored_0707|E2|64-3|122||119|válido|
|scored_0707|E2|64-6|122||118|válido|
|scored_0707|E3|joint|122|individual 122|122|OPTIMAL|
|scored_0707|E4|repair|122||117|completo|
|scored_0708|E1|1|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0708|E1|3|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0708|E1|6|198|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|scored_0708|E2|52-1|202||194|válido|
|scored_0708|E2|52-3|199||186|válido|
|scored_0708|E2|52-6|198||190|válido|
|scored_0708|E2|64-1|202||195|válido|
|scored_0708|E2|64-3|199||188|válido|
|scored_0708|E2|64-6|198||188|válido|
|scored_0708|E3|joint|198|individual 198|196|OPTIMAL|
|scored_0708|E4|repair|198||198|completo|
|scored_0709|E1|1|178|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|scored_0709|E1|3|169|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0709|E1|6|169|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0709|E2|52-1|178||162|válido|
|scored_0709|E2|52-3|169||159|válido|
|scored_0709|E2|52-6|169||158|válido|
|scored_0709|E2|64-1|178||168|válido|
|scored_0709|E2|64-3|169||165|válido|
|scored_0709|E2|64-6|169||160|válido|
|scored_0709|E3|joint|169|individual 169|169|OPTIMAL|
|scored_0709|E4|repair|169||166|completo|
|scored_0710|E1|1|160|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0710|E1|3|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0710|E1|6|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0710|E2|52-1|160||152|válido|
|scored_0710|E2|52-3|159||148|válido|
|scored_0710|E2|52-6|159||148|válido|
|scored_0710|E2|64-1|160||152|válido|
|scored_0710|E2|64-3|159||148|válido|
|scored_0710|E2|64-6|159||148|válido|
|scored_0710|E3|joint|159|individual 158|157|OPTIMAL|
|scored_0710|E4|repair|159||157|completo|
|scored_0711|E1|1|31|postprocess 29, optimize_a4 29, nested_windows 29|29|válido|
|scored_0711|E1|3|31|postprocess 29, optimize_a4 29, nested_windows 29|29|válido|
|scored_0711|E1|6|31|postprocess 29, optimize_a4 29, nested_windows 29|29|válido|
|scored_0711|E2|52-1|31||31|válido|
|scored_0711|E2|52-3|31||31|válido|
|scored_0711|E2|52-6|31||31|válido|
|scored_0711|E2|64-1|31||31|válido|
|scored_0711|E2|64-3|31||31|válido|
|scored_0711|E2|64-6|31||31|válido|
|scored_0711|E3|joint|31|individual 31|31|OPTIMAL|
|scored_0711|E4|repair|31||31|completo|
|scored_0712|E1|1|269|postprocess 269, optimize_a4 269, nested_windows 269|269|válido|
|scored_0712|E1|3|258|postprocess 258, optimize_a4 258, nested_windows 258|258|válido|
|scored_0712|E1|6|255|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|scored_0712|E2|52-1|269||243|válido|
|scored_0712|E2|52-3|258||235|válido|
|scored_0712|E2|52-6|255||237|válido|
|scored_0712|E2|64-1|269||244|válido|
|scored_0712|E2|64-3|258||233|válido|
|scored_0712|E2|64-6|255||238|válido|
|scored_0712|E3|joint|255|individual 255|254|OPTIMAL|
|scored_0712|E4|repair|255||255|completo|
|scored_0713|E1|1|130|postprocess 130, optimize_a4 130, nested_windows 130|130|válido|
|scored_0713|E1|3|130|postprocess 130, optimize_a4 130, nested_windows 130|130|válido|
|scored_0713|E1|6|130|postprocess 130, optimize_a4 130, nested_windows 130|130|válido|
|scored_0713|E2|52-1|130||122|válido|
|scored_0713|E2|52-3|130||122|válido|
|scored_0713|E2|52-6|130||122|válido|
|scored_0713|E2|64-1|130||124|válido|
|scored_0713|E2|64-3|130||124|válido|
|scored_0713|E2|64-6|130||124|válido|
|scored_0713|E3|joint|130|individual 130|130|OPTIMAL|
|scored_0713|E4|repair|130||127|completo|
|scored_0714|E1|1|271|postprocess 271, optimize_a4 271, nested_windows 271|271|válido|
|scored_0714|E1|3|261|postprocess 261, optimize_a4 261, nested_windows 261|261|válido|
|scored_0714|E1|6|256|postprocess 256, optimize_a4 256, nested_windows 256|256|válido|
|scored_0714|E2|52-1|271||244|válido|
|scored_0714|E2|52-3|261||243|válido|
|scored_0714|E2|52-6|256||232|válido|
|scored_0714|E2|64-1|271||251|válido|
|scored_0714|E2|64-3|261||247|válido|
|scored_0714|E2|64-6|256||241|válido|
|scored_0714|E3|joint|256|individual 256|254|OPTIMAL|
|scored_0714|E4|repair|256||254|completo|
|scored_0715|E1|1|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0715|E1|3|214|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0715|E1|6|210|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0715|E2|52-1|219||207|válido|
|scored_0715|E2|52-3|214||197|válido|
|scored_0715|E2|52-6|210||200|válido|
|scored_0715|E2|64-1|219||206|válido|
|scored_0715|E2|64-3|214||197|válido|
|scored_0715|E2|64-6|210||199|válido|
|scored_0715|E3|joint|210|individual 210|209|OPTIMAL|
|scored_0715|E4|repair|210||210|completo|
|scored_0716|E1|1|154|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0716|E1|3|154|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0716|E1|6|154|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0716|E2|52-1|154||151|válido|
|scored_0716|E2|52-3|154||148|válido|
|scored_0716|E2|52-6|154||148|válido|
|scored_0716|E2|64-1|154||150|válido|
|scored_0716|E2|64-3|154||150|válido|
|scored_0716|E2|64-6|154||150|válido|
|scored_0716|E3|joint|154|individual 154|154|OPTIMAL|
|scored_0716|E4|repair|154||151|completo|
|scored_0717|E1|1|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0717|E1|3|164|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0717|E1|6|160|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0717|E2|52-1|165||156|válido|
|scored_0717|E2|52-3|164||155|válido|
|scored_0717|E2|52-6|160||155|válido|
|scored_0717|E2|64-1|165||153|válido|
|scored_0717|E2|64-3|164||153|válido|
|scored_0717|E2|64-6|160||153|válido|
|scored_0717|E3|joint|160|individual 160|160|OPTIMAL|
|scored_0717|E4|repair|160||156|completo|
|scored_0718|E1|1|188|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0718|E1|3|183|postprocess 183, optimize_a4 183, nested_windows 183|183|válido|
|scored_0718|E1|6|183|postprocess 183, optimize_a4 183, nested_windows 183|183|válido|
|scored_0718|E2|52-1|188||177|válido|
|scored_0718|E2|52-3|183||173|válido|
|scored_0718|E2|52-6|183||173|válido|
|scored_0718|E2|64-1|188||176|válido|
|scored_0718|E2|64-3|183||171|válido|
|scored_0718|E2|64-6|183||171|válido|
|scored_0718|E3|joint|183|individual 182|180|OPTIMAL|
|scored_0718|E4|repair|183||181|completo|
|scored_0719|E1|1|353|postprocess 353, optimize_a4 353, nested_windows 353|353|válido|
|scored_0719|E1|3|337|postprocess 337, optimize_a4 337, nested_windows 337|337|válido|
|scored_0719|E1|6|330|postprocess 330, optimize_a4 330, nested_windows 330|330|válido|
|scored_0719|E2|52-1|353||332|válido|
|scored_0719|E2|52-3|337||320|válido|
|scored_0719|E2|52-6|330||309|válido|
|scored_0719|E2|64-1|353||338|válido|
|scored_0719|E2|64-3|337||326|válido|
|scored_0719|E2|64-6|330||315|válido|
|scored_0719|E3|joint|330|individual 330|330|OPTIMAL|
|scored_0719|E4|repair|330||330|completo|
|scored_0720|E1|1|38|postprocess 37, optimize_a4 37, nested_windows 37|37|válido|
|scored_0720|E1|3|38|postprocess 37, optimize_a4 37, nested_windows 37|37|válido|
|scored_0720|E1|6|38|postprocess 37, optimize_a4 37, nested_windows 37|37|válido|
|scored_0720|E2|52-1|38||38|válido|
|scored_0720|E2|52-3|38||38|válido|
|scored_0720|E2|52-6|38||38|válido|
|scored_0720|E2|64-1|38||38|válido|
|scored_0720|E2|64-3|38||38|válido|
|scored_0720|E2|64-6|38||38|válido|
|scored_0720|E3|joint|38|individual 38|38|OPTIMAL|
|scored_0720|E4|repair|38||38|completo|
|scored_0721|E1|1|6|postprocess 6, optimize_a4 4, nested_windows 4|4|válido|
|scored_0721|E1|3|6|postprocess 6, optimize_a4 4, nested_windows 4|4|válido|
|scored_0721|E1|6|6|postprocess 6, optimize_a4 4, nested_windows 4|4|válido|
|scored_0721|E2|52-1|6||6|válido|
|scored_0721|E2|52-3|6||6|válido|
|scored_0721|E2|52-6|6||6|válido|
|scored_0721|E2|64-1|6||6|válido|
|scored_0721|E2|64-3|6||6|válido|
|scored_0721|E2|64-6|6||6|válido|
|scored_0721|E3|joint|4|individual 4|4|OPTIMAL|
|scored_0721|E4|repair|4||4|completo|
|scored_0722|E1|1|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0722|E1|3|221|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0722|E1|6|210|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0722|E2|52-1|226||207|válido|
|scored_0722|E2|52-3|221||198|válido|
|scored_0722|E2|52-6|210||198|válido|
|scored_0722|E2|64-1|226||212|válido|
|scored_0722|E2|64-3|221||204|válido|
|scored_0722|E2|64-6|210||211|válido|
|scored_0722|E3|joint|210|individual 208|208|OPTIMAL|
|scored_0722|E4|repair|210||208|completo|
|scored_0723|E1|1|132|postprocess 132, optimize_a4 132, nested_windows 132|132|válido|
|scored_0723|E1|3|131|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|scored_0723|E1|6|131|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|scored_0723|E2|52-1|132||125|válido|
|scored_0723|E2|52-3|131||125|válido|
|scored_0723|E2|52-6|131||125|válido|
|scored_0723|E2|64-1|132||125|válido|
|scored_0723|E2|64-3|131||125|válido|
|scored_0723|E2|64-6|131||125|válido|
|scored_0723|E3|joint|131|individual 131|131|OPTIMAL|
|scored_0723|E4|repair|131||129|completo|
|scored_0724|E1|1|221|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0724|E1|3|218|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|scored_0724|E1|6|218|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|scored_0724|E2|52-1|221||210|válido|
|scored_0724|E2|52-3|218||206|válido|
|scored_0724|E2|52-6|218||199|válido|
|scored_0724|E2|64-1|221||211|válido|
|scored_0724|E2|64-3|218||207|válido|
|scored_0724|E2|64-6|218||200|válido|
|scored_0724|E3|joint|218|individual 217|217|OPTIMAL|
|scored_0724|E4|repair|218||218|completo|
|scored_0725|E1|1|319|postprocess 319, optimize_a4 319, nested_windows 319|319|válido|
|scored_0725|E1|3|300|postprocess 300, optimize_a4 300, nested_windows 300|300|válido|
|scored_0725|E1|6|302|postprocess 302, optimize_a4 302, nested_windows 302|302|válido|
|scored_0725|E2|52-1|319||289|válido|
|scored_0725|E2|52-3|300||281|válido|
|scored_0725|E2|52-6|302||273|válido|
|scored_0725|E2|64-1|319||354|válido|
|scored_0725|E2|64-3|300||279|válido|
|scored_0725|E2|64-6|302||275|válido|
|scored_0725|E3|joint|300|individual 299|298|OPTIMAL|
|scored_0725|E4|repair|300||300|completo|
|scored_0726|E1|1|199|postprocess 198, optimize_a4 198, nested_windows 198|198|válido|
|scored_0726|E1|3|196|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|scored_0726|E1|6|191|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|scored_0726|E2|52-1|199||190|válido|
|scored_0726|E2|52-3|196||186|válido|
|scored_0726|E2|52-6|191||186|válido|
|scored_0726|E2|64-1|199||190|válido|
|scored_0726|E2|64-3|196||187|válido|
|scored_0726|E2|64-6|191||186|válido|
|scored_0726|E3|joint|191|individual 191|189|OPTIMAL|
|scored_0726|E4|repair|191||191|completo|
|scored_0727|E1|1|108|postprocess 107, optimize_a4 107, nested_windows 107|107|válido|
|scored_0727|E1|3|108|postprocess 107, optimize_a4 107, nested_windows 107|107|válido|
|scored_0727|E1|6|108|postprocess 107, optimize_a4 107, nested_windows 107|107|válido|
|scored_0727|E2|52-1|108||105|válido|
|scored_0727|E2|52-3|108||105|válido|
|scored_0727|E2|52-6|108||105|válido|
|scored_0727|E2|64-1|108||105|válido|
|scored_0727|E2|64-3|108||105|válido|
|scored_0727|E2|64-6|108||105|válido|
|scored_0727|E3|joint|108|individual 108|108|OPTIMAL|
|scored_0727|E4|repair|108||103|completo|
|scored_0728|E1|1|375|postprocess 375, optimize_a4 375, nested_windows 375|375|válido|
|scored_0728|E1|3|343|postprocess 343, optimize_a4 343, nested_windows 343|343|válido|
|scored_0728|E1|6|339|postprocess 339, optimize_a4 339, nested_windows 339|339|válido|
|scored_0728|E2|52-1|375||337|válido|
|scored_0728|E2|52-3|343||322|válido|
|scored_0728|E2|52-6|339||315|válido|
|scored_0728|E2|64-1|375||337|válido|
|scored_0728|E2|64-3|343||322|válido|
|scored_0728|E2|64-6|339||361|válido|
|scored_0728|E3|joint|339|individual 339|339|FEASIBLE|
|scored_0728|E4|repair|339||339|completo|
|scored_0729|E1|1|425|postprocess 425, optimize_a4 425, nested_windows 425|425|válido|
|scored_0729|E1|3|393|postprocess 393, optimize_a4 393, nested_windows 393|393|válido|
|scored_0729|E1|6|384|postprocess 383, optimize_a4 383, nested_windows 383|383|válido|
|scored_0729|E2|52-1|425||372|válido|
|scored_0729|E2|52-3|393||355|válido|
|scored_0729|E2|52-6|384||352|válido|
|scored_0729|E2|64-1|425||375|válido|
|scored_0729|E2|64-3|393||360|válido|
|scored_0729|E2|64-6|384||355|válido|
|scored_0729|E3|joint|384|individual 382|381|FEASIBLE|
|scored_0729|E4|repair|384||384|completo|
|scored_0730|E1|1|176|postprocess 176, optimize_a4 176, nested_windows 176|176|válido|
|scored_0730|E1|3|171|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|scored_0730|E1|6|167|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0730|E2|52-1|176||165|válido|
|scored_0730|E2|52-3|171||164|válido|
|scored_0730|E2|52-6|167||161|válido|
|scored_0730|E2|64-1|176||165|válido|
|scored_0730|E2|64-3|171||167|válido|
|scored_0730|E2|64-6|167||163|válido|
|scored_0730|E3|joint|167|individual 167|167|OPTIMAL|
|scored_0730|E4|repair|167||161|completo|
|scored_0731|E1|1|227|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0731|E1|3|220|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0731|E1|6|208|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|scored_0731|E2|52-1|227||199|válido|
|scored_0731|E2|52-3|220||195|válido|
|scored_0731|E2|52-6|208||195|válido|
|scored_0731|E2|64-1|227||200|válido|
|scored_0731|E2|64-3|220||199|válido|
|scored_0731|E2|64-6|208||197|válido|
|scored_0731|E3|joint|208|individual 208|208|OPTIMAL|
|scored_0731|E4|repair|208||207|completo|
|scored_0732|E1|1|163|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0732|E1|3|159|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0732|E1|6|158|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0732|E2|52-1|163||157|válido|
|scored_0732|E2|52-3|159||154|válido|
|scored_0732|E2|52-6|158||154|válido|
|scored_0732|E2|64-1|163||156|válido|
|scored_0732|E2|64-3|159||154|válido|
|scored_0732|E2|64-6|158||154|válido|
|scored_0732|E3|joint|158|individual 158|158|OPTIMAL|
|scored_0732|E4|repair|158||156|completo|
|scored_0733|E1|1|123|postprocess 123, optimize_a4 123, nested_windows 123|123|válido|
|scored_0733|E1|3|116|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0733|E1|6|116|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0733|E2|52-1|123||121|válido|
|scored_0733|E2|52-3|116||116|válido|
|scored_0733|E2|52-6|116||117|válido|
|scored_0733|E2|64-1|123||121|válido|
|scored_0733|E2|64-3|116||116|válido|
|scored_0733|E2|64-6|116||117|válido|
|scored_0733|E3|joint|116|individual 116|116|OPTIMAL|
|scored_0733|E4|repair|116||116|completo|
|scored_0734|E1|1|193|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0734|E1|3|184|postprocess 183, optimize_a4 183, nested_windows 183|183|válido|
|scored_0734|E1|6|182|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0734|E2|52-1|193||180|válido|
|scored_0734|E2|52-3|184||179|válido|
|scored_0734|E2|52-6|182||178|válido|
|scored_0734|E2|64-1|193||179|válido|
|scored_0734|E2|64-3|184||178|válido|
|scored_0734|E2|64-6|182||179|válido|
|scored_0734|E3|joint|182|individual 182|182|OPTIMAL|
|scored_0734|E4|repair|182||182|completo|
|scored_0735|E1|1|158|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0735|E1|3|157|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0735|E1|6|157|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0735|E2|52-1|158||150|válido|
|scored_0735|E2|52-3|157||148|válido|
|scored_0735|E2|52-6|157||148|válido|
|scored_0735|E2|64-1|158||147|válido|
|scored_0735|E2|64-3|157||145|válido|
|scored_0735|E2|64-6|157||145|válido|
|scored_0735|E3|joint|157|individual 157|157|OPTIMAL|
|scored_0735|E4|repair|157||155|completo|
|scored_0736|E1|1|223|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0736|E1|3|211|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0736|E1|6|208|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|scored_0736|E2|52-1|223||203|válido|
|scored_0736|E2|52-3|211||202|válido|
|scored_0736|E2|52-6|208||196|válido|
|scored_0736|E2|64-1|223||206|válido|
|scored_0736|E2|64-3|211||203|válido|
|scored_0736|E2|64-6|208||195|válido|
|scored_0736|E3|joint|208|individual 207|203|OPTIMAL|
|scored_0736|E4|repair|208||206|completo|
|scored_0737|E1|1|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0737|E1|3|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0737|E1|6|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0737|E2|52-1|151||148|válido|
|scored_0737|E2|52-3|151||148|válido|
|scored_0737|E2|52-6|148||148|válido|
|scored_0737|E2|64-1|151||147|válido|
|scored_0737|E2|64-3|151||147|válido|
|scored_0737|E2|64-6|148||147|válido|
|scored_0737|E3|joint|148|individual 148|148|OPTIMAL|
|scored_0737|E4|repair|148||144|completo|
|scored_0738|E1|1|160|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0738|E1|3|155|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0738|E1|6|152|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0738|E2|52-1|160||147|válido|
|scored_0738|E2|52-3|155||147|válido|
|scored_0738|E2|52-6|152||146|válido|
|scored_0738|E2|64-1|160||147|válido|
|scored_0738|E2|64-3|155||147|válido|
|scored_0738|E2|64-6|152||146|válido|
|scored_0738|E3|joint|152|individual 152|152|OPTIMAL|
|scored_0738|E4|repair|152||151|completo|
|scored_0739|E1|1|208|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0739|E1|3|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0739|E1|6|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0739|E2|52-1|208||194|válido|
|scored_0739|E2|52-3|202||191|válido|
|scored_0739|E2|52-6|202||191|válido|
|scored_0739|E2|64-1|208||193|válido|
|scored_0739|E2|64-3|202||192|válido|
|scored_0739|E2|64-6|202||187|válido|
|scored_0739|E3|joint|202|individual 201|201|OPTIMAL|
|scored_0739|E4|repair|202||202|completo|
|scored_0740|E1|1|180|postprocess 180, optimize_a4 180, nested_windows 180|180|válido|
|scored_0740|E1|3|178|postprocess 178, optimize_a4 178, nested_windows 178|178|válido|
|scored_0740|E1|6|176|postprocess 176, optimize_a4 176, nested_windows 176|176|válido|
|scored_0740|E2|52-1|180||169|válido|
|scored_0740|E2|52-3|178||168|válido|
|scored_0740|E2|52-6|176||167|válido|
|scored_0740|E2|64-1|180||167|válido|
|scored_0740|E2|64-3|178||166|válido|
|scored_0740|E2|64-6|176||164|válido|
|scored_0740|E3|joint|176|individual 176|176|OPTIMAL|
|scored_0740|E4|repair|176||175|completo|
|scored_0741|E1|1|186|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0741|E1|3|183|postprocess 183, optimize_a4 183, nested_windows 183|183|válido|
|scored_0741|E1|6|182|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0741|E2|52-1|186||173|válido|
|scored_0741|E2|52-3|183||168|válido|
|scored_0741|E2|52-6|182||167|válido|
|scored_0741|E2|64-1|186||178|válido|
|scored_0741|E2|64-3|183||168|válido|
|scored_0741|E2|64-6|182||168|válido|
|scored_0741|E3|joint|182|individual 182|180|OPTIMAL|
|scored_0741|E4|repair|182||182|completo|
|scored_0742|E1|1|247|postprocess 247, optimize_a4 247, nested_windows 247|247|válido|
|scored_0742|E1|3|240|postprocess 240, optimize_a4 240, nested_windows 240|240|válido|
|scored_0742|E1|6|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0742|E2|52-1|247||230|válido|
|scored_0742|E2|52-3|240||229|válido|
|scored_0742|E2|52-6|242||224|válido|
|scored_0742|E2|64-1|247||234|válido|
|scored_0742|E2|64-3|240||231|válido|
|scored_0742|E2|64-6|242||230|válido|
|scored_0742|E3|joint|240|individual 239|237|OPTIMAL|
|scored_0742|E4|repair|240||240|completo|
|scored_0743|E1|1|140|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|scored_0743|E1|3|140|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|scored_0743|E1|6|140|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|scored_0743|E2|52-1|140||138|válido|
|scored_0743|E2|52-3|140||138|válido|
|scored_0743|E2|52-6|140||134|válido|
|scored_0743|E2|64-1|140||133|válido|
|scored_0743|E2|64-3|140||133|válido|
|scored_0743|E2|64-6|140||132|válido|
|scored_0743|E3|joint|140|individual 140|140|OPTIMAL|
|scored_0743|E4|repair|140||138|completo|
|scored_0744|E1|1|397|postprocess 397, optimize_a4 397, nested_windows 397|397|válido|
|scored_0744|E1|3|378|postprocess 377, optimize_a4 377, nested_windows 377|377|válido|
|scored_0744|E1|6|370|postprocess 370, optimize_a4 370, nested_windows 370|370|válido|
|scored_0744|E2|52-1|397||365|válido|
|scored_0744|E2|52-3|378||354|válido|
|scored_0744|E2|52-6|370||343|válido|
|scored_0744|E2|64-1|397||369|válido|
|scored_0744|E2|64-3|378||353|válido|
|scored_0744|E2|64-6|370||349|válido|
|scored_0744|E3|joint|370|individual 369|367|FEASIBLE|
|scored_0744|E4|repair|370||370|completo|
|scored_0745|E1|1|247|postprocess 247, optimize_a4 247, nested_windows 247|247|válido|
|scored_0745|E1|3|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0745|E1|6|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0745|E2|52-1|247||238|válido|
|scored_0745|E2|52-3|239||223|válido|
|scored_0745|E2|52-6|235||220|válido|
|scored_0745|E2|64-1|247||239|válido|
|scored_0745|E2|64-3|239||226|válido|
|scored_0745|E2|64-6|235||222|válido|
|scored_0745|E3|joint|235|individual 235|234|FEASIBLE|
|scored_0745|E4|repair|235||232|completo|
|scored_0746|E1|1|209|postprocess 208, optimize_a4 208, nested_windows 208|208|válido|
|scored_0746|E1|3|205|postprocess 205, optimize_a4 205, nested_windows 205|205|válido|
|scored_0746|E1|6|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0746|E2|52-1|209||188|válido|
|scored_0746|E2|52-3|205||184|válido|
|scored_0746|E2|52-6|202||187|válido|
|scored_0746|E2|64-1|209||194|válido|
|scored_0746|E2|64-3|205||190|válido|
|scored_0746|E2|64-6|202||189|válido|
|scored_0746|E3|joint|202|individual 201|200|OPTIMAL|
|scored_0746|E4|repair|202||202|completo|
|scored_0747|E1|1|210|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0747|E1|3|209|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0747|E1|6|204|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|scored_0747|E2|52-1|210||195|válido|
|scored_0747|E2|52-3|209||191|válido|
|scored_0747|E2|52-6|204||191|válido|
|scored_0747|E2|64-1|210||194|válido|
|scored_0747|E2|64-3|209||190|válido|
|scored_0747|E2|64-6|204||193|válido|
|scored_0747|E3|joint|204|individual 204|204|OPTIMAL|
|scored_0747|E4|repair|204||204|completo|
|scored_0748|E1|1|327|postprocess 327, optimize_a4 327, nested_windows 327|327|válido|
|scored_0748|E1|3|314|postprocess 314, optimize_a4 314, nested_windows 314|314|válido|
|scored_0748|E1|6|306|postprocess 305, optimize_a4 305, nested_windows 305|305|válido|
|scored_0748|E2|52-1|327||301|válido|
|scored_0748|E2|52-3|314||296|válido|
|scored_0748|E2|52-6|306||288|válido|
|scored_0748|E2|64-1|327||301|válido|
|scored_0748|E2|64-3|314||299|válido|
|scored_0748|E2|64-6|306||291|válido|
|scored_0748|E3|joint|306|individual 306|306|OPTIMAL|
|scored_0748|E4|repair|306||306|completo|
|scored_0749|E1|1|124|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|scored_0749|E1|3|124|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|scored_0749|E1|6|124|postprocess 124, optimize_a4 124, nested_windows 124|124|válido|
|scored_0749|E2|52-1|124||119|válido|
|scored_0749|E2|52-3|124||119|válido|
|scored_0749|E2|52-6|124||118|válido|
|scored_0749|E2|64-1|124||118|válido|
|scored_0749|E2|64-3|124||118|válido|
|scored_0749|E2|64-6|124||118|válido|
|scored_0749|E3|joint|124|individual 124|124|OPTIMAL|
|scored_0749|E4|repair|124||122|completo|
|scored_0750|E1|1|149|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0750|E1|3|149|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0750|E1|6|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0750|E2|52-1|149||143|válido|
|scored_0750|E2|52-3|149||142|válido|
|scored_0750|E2|52-6|146||141|válido|
|scored_0750|E2|64-1|149||142|válido|
|scored_0750|E2|64-3|149||141|válido|
|scored_0750|E2|64-6|146||141|válido|
|scored_0750|E3|joint|146|individual 146|146|OPTIMAL|
|scored_0750|E4|repair|146||143|completo|
|scored_0751|E1|1|321|postprocess 321, optimize_a4 321, nested_windows 321|321|válido|
|scored_0751|E1|3|291|postprocess 291, optimize_a4 291, nested_windows 291|291|válido|
|scored_0751|E1|6|284|postprocess 284, optimize_a4 284, nested_windows 284|284|válido|
|scored_0751|E2|52-1|321||278|válido|
|scored_0751|E2|52-3|291||272|válido|
|scored_0751|E2|52-6|284||268|válido|
|scored_0751|E2|64-1|321||278|válido|
|scored_0751|E2|64-3|291||271|válido|
|scored_0751|E2|64-6|284||270|válido|
|scored_0751|E3|joint|284|individual 283|283|OPTIMAL|
|scored_0751|E4|repair|284||284|completo|
|scored_0752|E1|1|133|postprocess 133, optimize_a4 133, nested_windows 133|133|válido|
|scored_0752|E1|3|133|postprocess 133, optimize_a4 133, nested_windows 133|133|válido|
|scored_0752|E1|6|132|postprocess 132, optimize_a4 132, nested_windows 132|132|válido|
|scored_0752|E2|52-1|133||128|válido|
|scored_0752|E2|52-3|133||128|válido|
|scored_0752|E2|52-6|132||128|válido|
|scored_0752|E2|64-1|133||129|válido|
|scored_0752|E2|64-3|133||129|válido|
|scored_0752|E2|64-6|132||129|válido|
|scored_0752|E3|joint|132|individual 132|132|OPTIMAL|
|scored_0752|E4|repair|132||132|completo|
|scored_0753|E1|1|126|postprocess 126, optimize_a4 126, nested_windows 126|126|válido|
|scored_0753|E1|3|125|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0753|E1|6|125|postprocess 125, optimize_a4 125, nested_windows 125|125|válido|
|scored_0753|E2|52-1|126||122|válido|
|scored_0753|E2|52-3|125||122|válido|
|scored_0753|E2|52-6|125||122|válido|
|scored_0753|E2|64-1|126||122|válido|
|scored_0753|E2|64-3|125||122|válido|
|scored_0753|E2|64-6|125||122|válido|
|scored_0753|E3|joint|125|individual 125|125|OPTIMAL|
|scored_0753|E4|repair|125||123|completo|
|scored_0754|E1|1|236|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0754|E1|3|227|postprocess 227, optimize_a4 227, nested_windows 227|227|válido|
|scored_0754|E1|6|221|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0754|E2|52-1|236||216|válido|
|scored_0754|E2|52-3|227||212|válido|
|scored_0754|E2|52-6|221||208|válido|
|scored_0754|E2|64-1|236||219|válido|
|scored_0754|E2|64-3|227||212|válido|
|scored_0754|E2|64-6|221||211|válido|
|scored_0754|E3|joint|221|individual 220|220|OPTIMAL|
|scored_0754|E4|repair|221||220|completo|
|scored_0755|E1|1|150|postprocess 150, optimize_a4 150, nested_windows 150|150|válido|
|scored_0755|E1|3|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0755|E1|6|147|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0755|E2|52-1|150||147|válido|
|scored_0755|E2|52-3|148||147|válido|
|scored_0755|E2|52-6|147||144|válido|
|scored_0755|E2|64-1|150||147|válido|
|scored_0755|E2|64-3|148||147|válido|
|scored_0755|E2|64-6|147||147|válido|
|scored_0755|E3|joint|147|individual 147|147|OPTIMAL|
|scored_0755|E4|repair|147||142|completo|
|scored_0756|E1|1|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0756|E1|3|205|postprocess 205, optimize_a4 205, nested_windows 205|205|válido|
|scored_0756|E1|6|201|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0756|E2|52-1|211||197|válido|
|scored_0756|E2|52-3|205||191|válido|
|scored_0756|E2|52-6|201||191|válido|
|scored_0756|E2|64-1|211||197|válido|
|scored_0756|E2|64-3|205||192|válido|
|scored_0756|E2|64-6|201||192|válido|
|scored_0756|E3|joint|201|individual 201|201|OPTIMAL|
|scored_0756|E4|repair|201||201|completo|
|scored_0757|E1|1|175|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0757|E1|3|174|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|scored_0757|E1|6|170|postprocess 170, optimize_a4 170, nested_windows 170|170|válido|
|scored_0757|E2|52-1|175||163|válido|
|scored_0757|E2|52-3|174||163|válido|
|scored_0757|E2|52-6|170||163|válido|
|scored_0757|E2|64-1|175||165|válido|
|scored_0757|E2|64-3|174||164|válido|
|scored_0757|E2|64-6|170||163|válido|
|scored_0757|E3|joint|170|individual 170|170|OPTIMAL|
|scored_0757|E4|repair|170||168|completo|
|scored_0758|E1|1|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0758|E1|3|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0758|E1|6|143|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0758|E2|52-1|145||137|válido|
|scored_0758|E2|52-3|143||137|válido|
|scored_0758|E2|52-6|143||138|válido|
|scored_0758|E2|64-1|145||136|válido|
|scored_0758|E2|64-3|143||137|válido|
|scored_0758|E2|64-6|143||137|válido|
|scored_0758|E3|joint|143|individual 143|143|OPTIMAL|
|scored_0758|E4|repair|143||141|completo|
|scored_0759|E1|1|283|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|scored_0759|E1|3|276|postprocess 276, optimize_a4 276, nested_windows 276|276|válido|
|scored_0759|E1|6|268|postprocess 268, optimize_a4 268, nested_windows 268|268|válido|
|scored_0759|E2|52-1|283||258|válido|
|scored_0759|E2|52-3|276||253|válido|
|scored_0759|E2|52-6|268||244|válido|
|scored_0759|E2|64-1|283||259|válido|
|scored_0759|E2|64-3|276||255|válido|
|scored_0759|E2|64-6|268||244|válido|
|scored_0759|E3|joint|268|individual 268|268|OPTIMAL|
|scored_0759|E4|repair|268||268|completo|
|scored_0760|E1|1|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0760|E1|3|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0760|E1|6|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0760|E2|52-1|146||141|válido|
|scored_0760|E2|52-3|146||138|válido|
|scored_0760|E2|52-6|146||138|válido|
|scored_0760|E2|64-1|146||138|válido|
|scored_0760|E2|64-3|146||137|válido|
|scored_0760|E2|64-6|146||137|válido|
|scored_0760|E3|joint|146|individual 146|146|OPTIMAL|
|scored_0760|E4|repair|146||146|completo|
|scored_0761|E1|1|128|postprocess 128, optimize_a4 128, nested_windows 128|128|válido|
|scored_0761|E1|3|128|postprocess 128, optimize_a4 128, nested_windows 128|128|válido|
|scored_0761|E1|6|127|postprocess 127, optimize_a4 127, nested_windows 127|127|válido|
|scored_0761|E2|52-1|128||125|válido|
|scored_0761|E2|52-3|128||125|válido|
|scored_0761|E2|52-6|127||124|válido|
|scored_0761|E2|64-1|128||125|válido|
|scored_0761|E2|64-3|128||123|válido|
|scored_0761|E2|64-6|127||123|válido|
|scored_0761|E3|joint|127|individual 127|127|OPTIMAL|
|scored_0761|E4|repair|127||124|completo|
|scored_0762|E1|1|385|postprocess 385, optimize_a4 385, nested_windows 385|385|válido|
|scored_0762|E1|3|364|postprocess 364, optimize_a4 364, nested_windows 364|364|válido|
|scored_0762|E1|6|361|postprocess 361, optimize_a4 361, nested_windows 361|361|válido|
|scored_0762|E2|52-1|385||358|válido|
|scored_0762|E2|52-3|364||341|válido|
|scored_0762|E2|52-6|361||347|válido|
|scored_0762|E2|64-1|385||363|válido|
|scored_0762|E2|64-3|364||342|válido|
|scored_0762|E2|64-6|361||348|válido|
|scored_0762|E3|joint|361|individual 360|359|FEASIBLE|
|scored_0762|E4|repair|361||361|completo|
|scored_0763|E1|1|119|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|scored_0763|E1|3|120|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|scored_0763|E1|6|121|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0763|E2|52-1|119||116|válido|
|scored_0763|E2|52-3|120||115|válido|
|scored_0763|E2|52-6|121||115|válido|
|scored_0763|E2|64-1|119||116|válido|
|scored_0763|E2|64-3|120||115|válido|
|scored_0763|E2|64-6|121||115|válido|
|scored_0763|E3|joint|119|individual 119|119|OPTIMAL|
|scored_0763|E4|repair|119||119|completo|
|scored_0764|E1|1|291|postprocess 290, optimize_a4 290, nested_windows 290|290|válido|
|scored_0764|E1|3|270|postprocess 270, optimize_a4 270, nested_windows 270|270|válido|
|scored_0764|E1|6|266|postprocess 265, optimize_a4 265, nested_windows 265|265|válido|
|scored_0764|E2|52-1|291||253|válido|
|scored_0764|E2|52-3|270||249|válido|
|scored_0764|E2|52-6|266||249|válido|
|scored_0764|E2|64-1|291||250|válido|
|scored_0764|E2|64-3|270||251|válido|
|scored_0764|E2|64-6|266||248|válido|
|scored_0764|E3|joint|266|individual 266|266|OPTIMAL|
|scored_0764|E4|repair|266||266|completo|
|scored_0765|E1|1|255|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|scored_0765|E1|3|249|postprocess 249, optimize_a4 249, nested_windows 249|249|válido|
|scored_0765|E1|6|246|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|scored_0765|E2|52-1|255||237|válido|
|scored_0765|E2|52-3|249||228|válido|
|scored_0765|E2|52-6|246||231|válido|
|scored_0765|E2|64-1|255||237|válido|
|scored_0765|E2|64-3|249||230|válido|
|scored_0765|E2|64-6|246||234|válido|
|scored_0765|E3|joint|246|individual 246|244|OPTIMAL|
|scored_0765|E4|repair|246||245|completo|
|scored_0766|E1|1|295|postprocess 294, optimize_a4 294, nested_windows 294|294|válido|
|scored_0766|E1|3|281|postprocess 279, optimize_a4 279, nested_windows 279|279|válido|
|scored_0766|E1|6|279|postprocess 279, optimize_a4 279, nested_windows 279|279|válido|
|scored_0766|E2|52-1|295||270|válido|
|scored_0766|E2|52-3|281||263|válido|
|scored_0766|E2|52-6|279||262|válido|
|scored_0766|E2|64-1|295||271|válido|
|scored_0766|E2|64-3|281||269|válido|
|scored_0766|E2|64-6|279||263|válido|
|scored_0766|E3|joint|279|individual 279|277|OPTIMAL|
|scored_0766|E4|repair|279||279|completo|
|scored_0767|E1|1|161|postprocess 161, optimize_a4 161, nested_windows 161|161|válido|
|scored_0767|E1|3|157|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0767|E1|6|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0767|E2|52-1|161||153|válido|
|scored_0767|E2|52-3|157||150|válido|
|scored_0767|E2|52-6|152||149|válido|
|scored_0767|E2|64-1|161||153|válido|
|scored_0767|E2|64-3|157||150|válido|
|scored_0767|E2|64-6|152||149|válido|
|scored_0767|E3|joint|152|individual 152|152|OPTIMAL|
|scored_0767|E4|repair|152||151|completo|
|scored_0768|E1|1|165|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0768|E1|3|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0768|E1|6|158|postprocess 157, optimize_a4 157, nested_windows 157|157|válido|
|scored_0768|E2|52-1|165||155|válido|
|scored_0768|E2|52-3|162||156|válido|
|scored_0768|E2|52-6|158||152|válido|
|scored_0768|E2|64-1|165||156|válido|
|scored_0768|E2|64-3|162||157|válido|
|scored_0768|E2|64-6|158||153|válido|
|scored_0768|E3|joint|158|individual 158|158|OPTIMAL|
|scored_0768|E4|repair|158||158|completo|
|scored_0769|E1|1|172|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0769|E1|3|169|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0769|E1|6|168|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0769|E2|52-1|172||150|válido|
|scored_0769|E2|52-3|169||145|válido|
|scored_0769|E2|52-6|168||144|válido|
|scored_0769|E2|64-1|172||147|válido|
|scored_0769|E2|64-3|169||148|válido|
|scored_0769|E2|64-6|168||148|válido|
|scored_0769|E3|joint|168|individual 168|167|OPTIMAL|
|scored_0769|E4|repair|168||164|completo|
|scored_0770|E1|1|209|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0770|E1|3|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0770|E1|6|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|scored_0770|E2|52-1|209||191|válido|
|scored_0770|E2|52-3|202||184|válido|
|scored_0770|E2|52-6|197||191|válido|
|scored_0770|E2|64-1|209||191|válido|
|scored_0770|E2|64-3|202||188|válido|
|scored_0770|E2|64-6|197||194|válido|
|scored_0770|E3|joint|197|individual 197|196|OPTIMAL|
|scored_0770|E4|repair|197||197|completo|
|scored_0771|E1|1|358|postprocess 358, optimize_a4 358, nested_windows 358|358|válido|
|scored_0771|E1|3|348|postprocess 348, optimize_a4 348, nested_windows 348|348|válido|
|scored_0771|E1|6|340|postprocess 340, optimize_a4 340, nested_windows 340|340|válido|
|scored_0771|E2|52-1|358||339|válido|
|scored_0771|E2|52-3|348||322|válido|
|scored_0771|E2|52-6|340||321|válido|
|scored_0771|E2|64-1|358||344|válido|
|scored_0771|E2|64-3|348||326|válido|
|scored_0771|E2|64-6|340||325|válido|
|scored_0771|E3|joint|340|individual 340|340|OPTIMAL|
|scored_0771|E4|repair|340||340|completo|
|scored_0772|E1|1|159|postprocess 158, optimize_a4 158, nested_windows 158|158|válido|
|scored_0772|E1|3|155|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0772|E1|6|155|postprocess 154, optimize_a4 154, nested_windows 154|154|válido|
|scored_0772|E2|52-1|159||156|válido|
|scored_0772|E2|52-3|155||150|válido|
|scored_0772|E2|52-6|155||148|válido|
|scored_0772|E2|64-1|159||156|válido|
|scored_0772|E2|64-3|155||149|válido|
|scored_0772|E2|64-6|155||149|válido|
|scored_0772|E3|joint|155|individual 154|154|OPTIMAL|
|scored_0772|E4|repair|155||151|completo|
|scored_0773|E1|1|63|postprocess 60, optimize_a4 60, nested_windows 60|60|válido|
|scored_0773|E1|3|63|postprocess 60, optimize_a4 60, nested_windows 60|60|válido|
|scored_0773|E1|6|63|postprocess 60, optimize_a4 60, nested_windows 60|60|válido|
|scored_0773|E2|52-1|63||63|válido|
|scored_0773|E2|52-3|63||63|válido|
|scored_0773|E2|52-6|63||63|válido|
|scored_0773|E2|64-1|63||63|válido|
|scored_0773|E2|64-3|63||63|válido|
|scored_0773|E2|64-6|63||63|válido|
|scored_0773|E3|joint|63|individual 63|63|OPTIMAL|
|scored_0773|E4|repair|63||63|completo|
|scored_0774|E1|1|401|postprocess 399, optimize_a4 399, nested_windows 399|399|válido|
|scored_0774|E1|3|385|postprocess 385, optimize_a4 385, nested_windows 385|385|válido|
|scored_0774|E1|6|374|postprocess 374, optimize_a4 374, nested_windows 374|374|válido|
|scored_0774|E2|52-1|401||366|válido|
|scored_0774|E2|52-3|385||352|válido|
|scored_0774|E2|52-6|374||344|válido|
|scored_0774|E2|64-1|401||368|válido|
|scored_0774|E2|64-3|385||353|válido|
|scored_0774|E2|64-6|374||344|válido|
|scored_0774|E3|joint|374|individual 374|374|OPTIMAL|
|scored_0774|E4|repair|374||374|completo|
|scored_0775|E1|1|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0775|E1|3|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|scored_0775|E1|6|196|postprocess 196, optimize_a4 196, nested_windows 196|196|válido|
|scored_0775|E2|52-1|211||194|válido|
|scored_0775|E2|52-3|197||196|válido|
|scored_0775|E2|52-6|196||186|válido|
|scored_0775|E2|64-1|211||188|válido|
|scored_0775|E2|64-3|197||197|válido|
|scored_0775|E2|64-6|196||186|válido|
|scored_0775|E3|joint|196|individual 196|196|OPTIMAL|
|scored_0775|E4|repair|196||194|completo|
|scored_0776|E1|1|298|postprocess 298, optimize_a4 298, nested_windows 298|298|válido|
|scored_0776|E1|3|299|postprocess 299, optimize_a4 299, nested_windows 299|299|válido|
|scored_0776|E1|6|292|postprocess 292, optimize_a4 292, nested_windows 292|292|válido|
|scored_0776|E2|52-1|298||286|válido|
|scored_0776|E2|52-3|299||282|válido|
|scored_0776|E2|52-6|292||273|válido|
|scored_0776|E2|64-1|298||284|válido|
|scored_0776|E2|64-3|299||283|válido|
|scored_0776|E2|64-6|292||279|válido|
|scored_0776|E3|joint|292|individual 292|291|OPTIMAL|
|scored_0776|E4|repair|292||292|completo|
|scored_0777|E1|1|49|postprocess 49, optimize_a4 49, nested_windows 49|49|válido|
|scored_0777|E1|3|49|postprocess 49, optimize_a4 49, nested_windows 49|49|válido|
|scored_0777|E1|6|49|postprocess 49, optimize_a4 49, nested_windows 49|49|válido|
|scored_0777|E2|52-1|49||50|válido|
|scored_0777|E2|52-3|49||50|válido|
|scored_0777|E2|52-6|49||50|válido|
|scored_0777|E2|64-1|49||50|válido|
|scored_0777|E2|64-3|49||50|válido|
|scored_0777|E2|64-6|49||50|válido|
|scored_0777|E3|joint|49|individual 49|49|OPTIMAL|
|scored_0777|E4|repair|49||49|completo|
|scored_0778|E1|1|176|postprocess 176, optimize_a4 176, nested_windows 176|176|válido|
|scored_0778|E1|3|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0778|E1|6|171|postprocess 171, optimize_a4 171, nested_windows 171|171|válido|
|scored_0778|E2|52-1|176||163|válido|
|scored_0778|E2|52-3|172||163|válido|
|scored_0778|E2|52-6|171||163|válido|
|scored_0778|E2|64-1|176||167|válido|
|scored_0778|E2|64-3|172||165|válido|
|scored_0778|E2|64-6|171||165|válido|
|scored_0778|E3|joint|171|individual 171|171|OPTIMAL|
|scored_0778|E4|repair|171||170|completo|
|scored_0779|E1|1|213|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0779|E1|3|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0779|E1|6|200|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0779|E2|52-1|213||199|válido|
|scored_0779|E2|52-3|206||179|válido|
|scored_0779|E2|52-6|200||180|válido|
|scored_0779|E2|64-1|213||199|válido|
|scored_0779|E2|64-3|206||179|válido|
|scored_0779|E2|64-6|200||180|válido|
|scored_0779|E3|joint|200|individual 200|200|OPTIMAL|
|scored_0779|E4|repair|200||199|completo|
|scored_0780|E1|1|379|postprocess 377, optimize_a4 377, nested_windows 377|377|válido|
|scored_0780|E1|3|358|postprocess 358, optimize_a4 358, nested_windows 358|358|válido|
|scored_0780|E1|6|359|postprocess 358, optimize_a4 358, nested_windows 358|358|válido|
|scored_0780|E2|52-1|379||353|válido|
|scored_0780|E2|52-3|358||331|válido|
|scored_0780|E2|52-6|359||333|válido|
|scored_0780|E2|64-1|379||355|válido|
|scored_0780|E2|64-3|358||336|válido|
|scored_0780|E2|64-6|359||333|válido|
|scored_0780|E3|joint|358|individual 354|353|FEASIBLE|
|scored_0780|E4|repair|358||358|completo|
|scored_0781|E1|1|348|postprocess 348, optimize_a4 348, nested_windows 348|348|válido|
|scored_0781|E1|3|336|postprocess 336, optimize_a4 336, nested_windows 336|336|válido|
|scored_0781|E1|6|327|postprocess 327, optimize_a4 327, nested_windows 327|327|válido|
|scored_0781|E2|52-1|348||322|válido|
|scored_0781|E2|52-3|336||311|válido|
|scored_0781|E2|52-6|327||305|válido|
|scored_0781|E2|64-1|348||325|válido|
|scored_0781|E2|64-3|336||313|válido|
|scored_0781|E2|64-6|327||307|válido|
|scored_0781|E3|joint|327|individual 326|326|FEASIBLE|
|scored_0781|E4|repair|327||327|completo|
|scored_0782|E1|1|206|postprocess 206, optimize_a4 206, nested_windows 206|206|válido|
|scored_0782|E1|3|200|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0782|E1|6|194|postprocess 194, optimize_a4 194, nested_windows 194|194|válido|
|scored_0782|E2|52-1|206||190|válido|
|scored_0782|E2|52-3|200||188|válido|
|scored_0782|E2|52-6|194||189|válido|
|scored_0782|E2|64-1|206||190|válido|
|scored_0782|E2|64-3|200||188|válido|
|scored_0782|E2|64-6|194||186|válido|
|scored_0782|E3|joint|194|individual 194|194|OPTIMAL|
|scored_0782|E4|repair|194||194|completo|
|scored_0783|E1|1|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0783|E1|3|147|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0783|E1|6|148|postprocess 148, optimize_a4 148, nested_windows 148|148|válido|
|scored_0783|E2|52-1|151||140|válido|
|scored_0783|E2|52-3|147||140|válido|
|scored_0783|E2|52-6|148||139|válido|
|scored_0783|E2|64-1|151||139|válido|
|scored_0783|E2|64-3|147||139|válido|
|scored_0783|E2|64-6|148||139|válido|
|scored_0783|E3|joint|147|individual 147|147|OPTIMAL|
|scored_0783|E4|repair|147||146|completo|
|scored_0784|E1|1|322|postprocess 322, optimize_a4 322, nested_windows 322|322|válido|
|scored_0784|E1|3|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0784|E1|6|297|postprocess 297, optimize_a4 297, nested_windows 297|297|válido|
|scored_0784|E2|52-1|322||289|válido|
|scored_0784|E2|52-3|303||275|válido|
|scored_0784|E2|52-6|297||281|válido|
|scored_0784|E2|64-1|322||290|válido|
|scored_0784|E2|64-3|303||278|válido|
|scored_0784|E2|64-6|297||282|válido|
|scored_0784|E3|joint|297|individual 297|294|FEASIBLE|
|scored_0784|E4|repair|297||297|completo|
|scored_0785|E1|1|140|postprocess 140, optimize_a4 140, nested_windows 140|140|válido|
|scored_0785|E1|3|140|postprocess 140, optimize_a4 140, nested_windows 140|140|válido|
|scored_0785|E1|6|140|postprocess 140, optimize_a4 140, nested_windows 140|140|válido|
|scored_0785|E2|52-1|140||134|válido|
|scored_0785|E2|52-3|140||134|válido|
|scored_0785|E2|52-6|140||134|válido|
|scored_0785|E2|64-1|140||134|válido|
|scored_0785|E2|64-3|140||134|válido|
|scored_0785|E2|64-6|140||134|válido|
|scored_0785|E3|joint|140|individual 140|140|OPTIMAL|
|scored_0785|E4|repair|140||138|completo|
|scored_0786|E1|1|347|postprocess 347, optimize_a4 347, nested_windows 347|347|válido|
|scored_0786|E1|3|327|postprocess 327, optimize_a4 327, nested_windows 327|327|válido|
|scored_0786|E1|6|326|postprocess 326, optimize_a4 326, nested_windows 326|326|válido|
|scored_0786|E2|52-1|347||321|válido|
|scored_0786|E2|52-3|327||309|válido|
|scored_0786|E2|52-6|326||299|válido|
|scored_0786|E2|64-1|347||325|válido|
|scored_0786|E2|64-3|327||308|válido|
|scored_0786|E2|64-6|326||304|válido|
|scored_0786|E3|joint|326|individual 325|323|FEASIBLE|
|scored_0786|E4|repair|326||326|completo|
|scored_0787|E1|1|134|postprocess 134, optimize_a4 134, nested_windows 134|134|válido|
|scored_0787|E1|3|134|postprocess 134, optimize_a4 134, nested_windows 134|134|válido|
|scored_0787|E1|6|133|postprocess 133, optimize_a4 133, nested_windows 133|133|válido|
|scored_0787|E2|52-1|134||127|válido|
|scored_0787|E2|52-3|134||127|válido|
|scored_0787|E2|52-6|133||127|válido|
|scored_0787|E2|64-1|134||126|válido|
|scored_0787|E2|64-3|134||126|válido|
|scored_0787|E2|64-6|133||126|válido|
|scored_0787|E3|joint|133|individual 133|133|OPTIMAL|
|scored_0787|E4|repair|133||133|completo|
|scored_0788|E1|1|319|postprocess 319, optimize_a4 319, nested_windows 319|319|válido|
|scored_0788|E1|3|311|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|scored_0788|E1|6|300|postprocess 300, optimize_a4 300, nested_windows 300|300|válido|
|scored_0788|E2|52-1|319||294|válido|
|scored_0788|E2|52-3|311||287|válido|
|scored_0788|E2|52-6|300||286|válido|
|scored_0788|E2|64-1|319||295|válido|
|scored_0788|E2|64-3|311||291|válido|
|scored_0788|E2|64-6|300||285|válido|
|scored_0788|E3|joint|300|individual 300|300|OPTIMAL|
|scored_0788|E4|repair|300||300|completo|
|scored_0789|E1|1|250|postprocess 250, optimize_a4 250, nested_windows 250|250|válido|
|scored_0789|E1|3|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0789|E1|6|226|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0789|E2|52-1|250||226|válido|
|scored_0789|E2|52-3|239||226|válido|
|scored_0789|E2|52-6|226||219|válido|
|scored_0789|E2|64-1|250||229|válido|
|scored_0789|E2|64-3|239||225|válido|
|scored_0789|E2|64-6|226||220|válido|
|scored_0789|E3|joint|226|individual 225|225|OPTIMAL|
|scored_0789|E4|repair|226||226|completo|
|scored_0790|E1|1|336|postprocess 336, optimize_a4 336, nested_windows 336|336|válido|
|scored_0790|E1|3|370|postprocess 339, optimize_a4 339, nested_windows 339|339|válido|
|scored_0790|E1|6|301|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|scored_0790|E2|52-1|336||289|válido|
|scored_0790|E2|52-3|370||274|válido|
|scored_0790|E2|52-6|301||272|válido|
|scored_0790|E2|64-1|336||293|válido|
|scored_0790|E2|64-3|370||277|válido|
|scored_0790|E2|64-6|301||277|válido|
|scored_0790|E3|joint|301|individual 301|300|OPTIMAL|
|scored_0790|E4|repair|301||301|completo|
|scored_0791|E1|1|218|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|scored_0791|E1|3|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0791|E1|6|204|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0791|E2|52-1|218||202|válido|
|scored_0791|E2|52-3|207||199|válido|
|scored_0791|E2|52-6|204||193|válido|
|scored_0791|E2|64-1|218||203|válido|
|scored_0791|E2|64-3|207||198|válido|
|scored_0791|E2|64-6|204||192|válido|
|scored_0791|E3|joint|204|individual 204|202|OPTIMAL|
|scored_0791|E4|repair|204||203|completo|
|scored_0792|E1|1|182|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0792|E1|3|177|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0792|E1|6|174|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0792|E2|52-1|182||170|válido|
|scored_0792|E2|52-3|177||169|válido|
|scored_0792|E2|52-6|174||168|válido|
|scored_0792|E2|64-1|182||174|válido|
|scored_0792|E2|64-3|177||167|válido|
|scored_0792|E2|64-6|174||169|válido|
|scored_0792|E3|joint|174|individual 174|173|OPTIMAL|
|scored_0792|E4|repair|174||174|completo|
|scored_0793|E1|1|318|postprocess 318, optimize_a4 318, nested_windows 318|318|válido|
|scored_0793|E1|3|306|postprocess 306, optimize_a4 306, nested_windows 306|306|válido|
|scored_0793|E1|6|302|postprocess 302, optimize_a4 302, nested_windows 302|302|válido|
|scored_0793|E2|52-1|318||284|válido|
|scored_0793|E2|52-3|306||273|válido|
|scored_0793|E2|52-6|302||277|válido|
|scored_0793|E2|64-1|318||287|válido|
|scored_0793|E2|64-3|306||273|válido|
|scored_0793|E2|64-6|302||278|válido|
|scored_0793|E3|joint|302|individual 302|302|FEASIBLE|
|scored_0793|E4|repair|302||302|completo|
|scored_0794|E1|1|240|postprocess 240, optimize_a4 240, nested_windows 240|240|válido|
|scored_0794|E1|3|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0794|E1|6|223|postprocess 222, optimize_a4 222, nested_windows 222|222|válido|
|scored_0794|E2|52-1|240||227|válido|
|scored_0794|E2|52-3|229||213|válido|
|scored_0794|E2|52-6|223||210|válido|
|scored_0794|E2|64-1|240||230|válido|
|scored_0794|E2|64-3|229||213|válido|
|scored_0794|E2|64-6|223||210|válido|
|scored_0794|E3|joint|223|individual 222|222|OPTIMAL|
|scored_0794|E4|repair|223||223|completo|
|scored_0795|E1|1|248|postprocess 248, optimize_a4 248, nested_windows 248|248|válido|
|scored_0795|E1|3|241|postprocess 241, optimize_a4 241, nested_windows 241|241|válido|
|scored_0795|E1|6|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|scored_0795|E2|52-1|248||231|válido|
|scored_0795|E2|52-3|241||225|válido|
|scored_0795|E2|52-6|238||221|válido|
|scored_0795|E2|64-1|248||226|válido|
|scored_0795|E2|64-3|241||225|válido|
|scored_0795|E2|64-6|238||220|válido|
|scored_0795|E3|joint|238|individual 238|238|OPTIMAL|
|scored_0795|E4|repair|238||238|completo|
|scored_0796|E1|1|320|postprocess 319, optimize_a4 319, nested_windows 319|319|válido|
|scored_0796|E1|3|302|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|scored_0796|E1|6|297|postprocess 297, optimize_a4 297, nested_windows 297|297|válido|
|scored_0796|E2|52-1|320||293|válido|
|scored_0796|E2|52-3|302||278|válido|
|scored_0796|E2|52-6|297||278|válido|
|scored_0796|E2|64-1|320||293|válido|
|scored_0796|E2|64-3|302||279|válido|
|scored_0796|E2|64-6|297||278|válido|
|scored_0796|E3|joint|297|individual 296|295|FEASIBLE|
|scored_0796|E4|repair|297||297|completo|
|scored_0797|E1|1|90|postprocess 90, optimize_a4 90, nested_windows 90|90|válido|
|scored_0797|E1|3|94|postprocess 94, optimize_a4 94, nested_windows 94|94|válido|
|scored_0797|E1|6|91|postprocess 91, optimize_a4 91, nested_windows 91|91|válido|
|scored_0797|E2|52-1|90||91|válido|
|scored_0797|E2|52-3|94||94|válido|
|scored_0797|E2|52-6|91||90|válido|
|scored_0797|E2|64-1|90||91|válido|
|scored_0797|E2|64-3|94||94|válido|
|scored_0797|E2|64-6|91||90|válido|
|scored_0797|E3|joint|90|individual 90|90|OPTIMAL|
|scored_0797|E4|repair|90||90|completo|
|scored_0798|E1|1|4|postprocess 4, optimize_a4 4, nested_windows 4|4|válido|
|scored_0798|E1|3|4|postprocess 4, optimize_a4 4, nested_windows 4|4|válido|
|scored_0798|E1|6|4|postprocess 4, optimize_a4 4, nested_windows 4|4|válido|
|scored_0798|E2|52-1|4||4|válido|
|scored_0798|E2|52-3|4||4|válido|
|scored_0798|E2|52-6|4||4|válido|
|scored_0798|E2|64-1|4||4|válido|
|scored_0798|E2|64-3|4||4|válido|
|scored_0798|E2|64-6|4||4|válido|
|scored_0798|E3|joint|2|individual 2|2|OPTIMAL|
|scored_0798|E4|repair|2||2|completo|
|scored_0799|E1|1|329|postprocess 327, optimize_a4 327, nested_windows 327|327|válido|
|scored_0799|E1|3|315|postprocess 315, optimize_a4 315, nested_windows 315|315|válido|
|scored_0799|E1|6|311|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|scored_0799|E2|52-1|329||301|válido|
|scored_0799|E2|52-3|315||290|válido|
|scored_0799|E2|52-6|311||286|válido|
|scored_0799|E2|64-1|329||302|válido|
|scored_0799|E2|64-3|315||298|válido|
|scored_0799|E2|64-6|311||285|válido|
|scored_0799|E3|joint|311|individual 311|310|FEASIBLE|
|scored_0799|E4|repair|311||311|completo|
|scored_0800|E1|1|104|postprocess 104, optimize_a4 103, nested_windows 103|103|válido|
|scored_0800|E1|3|104|postprocess 104, optimize_a4 103, nested_windows 103|103|válido|
|scored_0800|E1|6|104|postprocess 104, optimize_a4 103, nested_windows 103|103|válido|
|scored_0800|E2|52-1|104||103|válido|
|scored_0800|E2|52-3|104||103|válido|
|scored_0800|E2|52-6|104||103|válido|
|scored_0800|E2|64-1|104||102|válido|
|scored_0800|E2|64-3|104||102|válido|
|scored_0800|E2|64-6|104||102|válido|
|scored_0800|E3|joint|104|individual 104|104|OPTIMAL|
|scored_0800|E4|repair|104||102|completo|
|scored_0801|E1|1|129|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|scored_0801|E1|3|129|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|scored_0801|E1|6|129|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|scored_0801|E2|52-1|129||127|válido|
|scored_0801|E2|52-3|129||126|válido|
|scored_0801|E2|52-6|129||126|válido|
|scored_0801|E2|64-1|129||127|válido|
|scored_0801|E2|64-3|129||126|válido|
|scored_0801|E2|64-6|129||126|válido|
|scored_0801|E3|joint|129|individual 127|127|OPTIMAL|
|scored_0801|E4|repair|129||125|completo|
|scored_0802|E1|1|264|postprocess 264, optimize_a4 264, nested_windows 264|264|válido|
|scored_0802|E1|3|249|postprocess 248, optimize_a4 248, nested_windows 248|248|válido|
|scored_0802|E1|6|249|postprocess 249, optimize_a4 249, nested_windows 249|249|válido|
|scored_0802|E2|52-1|264||243|válido|
|scored_0802|E2|52-3|249||235|válido|
|scored_0802|E2|52-6|249||232|válido|
|scored_0802|E2|64-1|264||245|válido|
|scored_0802|E2|64-3|249||235|válido|
|scored_0802|E2|64-6|249||233|válido|
|scored_0802|E3|joint|249|individual 249|247|OPTIMAL|
|scored_0802|E4|repair|249||249|completo|
|scored_0803|E1|1|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0803|E1|3|184|postprocess 184, optimize_a4 184, nested_windows 184|184|válido|
|scored_0803|E1|6|187|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0803|E2|52-1|192||178|válido|
|scored_0803|E2|52-3|184||177|válido|
|scored_0803|E2|52-6|187||177|válido|
|scored_0803|E2|64-1|192||177|válido|
|scored_0803|E2|64-3|184||174|válido|
|scored_0803|E2|64-6|187||174|válido|
|scored_0803|E3|joint|184|individual 184|184|OPTIMAL|
|scored_0803|E4|repair|184||184|completo|
|scored_0804|E1|1|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0804|E1|3|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0804|E1|6|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0804|E2|52-1|145||144|válido|
|scored_0804|E2|52-3|145||143|válido|
|scored_0804|E2|52-6|145||143|válido|
|scored_0804|E2|64-1|145||144|válido|
|scored_0804|E2|64-3|145||144|válido|
|scored_0804|E2|64-6|145||144|válido|
|scored_0804|E3|joint|145|individual 145|145|OPTIMAL|
|scored_0804|E4|repair|145||141|completo|
|scored_0805|E1|1|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0805|E1|3|219|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|scored_0805|E1|6|214|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0805|E2|52-1|226||200|válido|
|scored_0805|E2|52-3|219||200|válido|
|scored_0805|E2|52-6|214||200|válido|
|scored_0805|E2|64-1|226||200|válido|
|scored_0805|E2|64-3|219||201|válido|
|scored_0805|E2|64-6|214||201|válido|
|scored_0805|E3|joint|214|individual 213|213|OPTIMAL|
|scored_0805|E4|repair|214||214|completo|
|scored_0806|E1|1|121|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0806|E1|3|120|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|scored_0806|E1|6|120|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|scored_0806|E2|52-1|121||121|válido|
|scored_0806|E2|52-3|120||120|válido|
|scored_0806|E2|52-6|120||120|válido|
|scored_0806|E2|64-1|121||121|válido|
|scored_0806|E2|64-3|120||120|válido|
|scored_0806|E2|64-6|120||120|válido|
|scored_0806|E3|joint|120|individual 120|120|OPTIMAL|
|scored_0806|E4|repair|120||119|completo|
|scored_0807|E1|1|197|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|scored_0807|E1|3|187|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0807|E1|6|186|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0807|E2|52-1|197||184|válido|
|scored_0807|E2|52-3|187||179|válido|
|scored_0807|E2|52-6|186||177|válido|
|scored_0807|E2|64-1|197||186|válido|
|scored_0807|E2|64-3|187||184|válido|
|scored_0807|E2|64-6|186||176|válido|
|scored_0807|E3|joint|186|individual 186|184|OPTIMAL|
|scored_0807|E4|repair|186||183|completo|
|scored_0808|E1|1|363|postprocess 363, optimize_a4 363, nested_windows 363|363|válido|
|scored_0808|E1|3|339|postprocess 339, optimize_a4 339, nested_windows 339|339|válido|
|scored_0808|E1|6|339|postprocess 339, optimize_a4 339, nested_windows 339|339|válido|
|scored_0808|E2|52-1|363||332|válido|
|scored_0808|E2|52-3|339||310|válido|
|scored_0808|E2|52-6|339||307|válido|
|scored_0808|E2|64-1|363||332|válido|
|scored_0808|E2|64-3|339||313|válido|
|scored_0808|E2|64-6|339||305|válido|
|scored_0808|E3|joint|339|individual 337|336|FEASIBLE|
|scored_0808|E4|repair|339||339|completo|
|scored_0809|E1|1|185|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|scored_0809|E1|3|182|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0809|E1|6|181|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0809|E2|52-1|185||171|válido|
|scored_0809|E2|52-3|182||166|válido|
|scored_0809|E2|52-6|181||166|válido|
|scored_0809|E2|64-1|185||168|válido|
|scored_0809|E2|64-3|182||170|válido|
|scored_0809|E2|64-6|181||170|válido|
|scored_0809|E3|joint|181|individual 181|180|OPTIMAL|
|scored_0809|E4|repair|181||180|completo|
|scored_0810|E1|1|45|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0810|E1|3|45|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0810|E1|6|45|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0810|E2|52-1|45||45|válido|
|scored_0810|E2|52-3|45||45|válido|
|scored_0810|E2|52-6|45||45|válido|
|scored_0810|E2|64-1|45||45|válido|
|scored_0810|E2|64-3|45||45|válido|
|scored_0810|E2|64-6|45||45|válido|
|scored_0810|E3|joint|45|individual 45|45|OPTIMAL|
|scored_0810|E4|repair|45||41|completo|
|scored_0811|E1|1|115|postprocess 115, optimize_a4 115, nested_windows 115|115|válido|
|scored_0811|E1|3|115|postprocess 115, optimize_a4 115, nested_windows 115|115|válido|
|scored_0811|E1|6|115|postprocess 115, optimize_a4 115, nested_windows 115|115|válido|
|scored_0811|E2|52-1|115||113|válido|
|scored_0811|E2|52-3|115||113|válido|
|scored_0811|E2|52-6|115||113|válido|
|scored_0811|E2|64-1|115||113|válido|
|scored_0811|E2|64-3|115||113|válido|
|scored_0811|E2|64-6|115||113|válido|
|scored_0811|E3|joint|115|individual 115|115|OPTIMAL|
|scored_0811|E4|repair|115||114|completo|
|scored_0812|E1|1|133|postprocess 133, optimize_a4 133, nested_windows 133|133|válido|
|scored_0812|E1|3|133|postprocess 133, optimize_a4 133, nested_windows 133|133|válido|
|scored_0812|E1|6|133|postprocess 133, optimize_a4 133, nested_windows 133|133|válido|
|scored_0812|E2|52-1|133||131|válido|
|scored_0812|E2|52-3|133||131|válido|
|scored_0812|E2|52-6|133||131|válido|
|scored_0812|E2|64-1|133||131|válido|
|scored_0812|E2|64-3|133||131|válido|
|scored_0812|E2|64-6|133||131|válido|
|scored_0812|E3|joint|133|individual 133|133|OPTIMAL|
|scored_0812|E4|repair|133||133|completo|
|scored_0813|E1|1|326|postprocess 326, optimize_a4 326, nested_windows 326|326|válido|
|scored_0813|E1|3|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0813|E1|6|295|postprocess 294, optimize_a4 294, nested_windows 294|294|válido|
|scored_0813|E2|52-1|326||286|válido|
|scored_0813|E2|52-3|303||285|válido|
|scored_0813|E2|52-6|295||279|válido|
|scored_0813|E2|64-1|326||289|válido|
|scored_0813|E2|64-3|303||284|válido|
|scored_0813|E2|64-6|295||279|válido|
|scored_0813|E3|joint|295|individual 295|295|OPTIMAL|
|scored_0813|E4|repair|295||295|completo|
|scored_0814|E1|1|256|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|scored_0814|E1|3|231|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|scored_0814|E1|6|222|postprocess 222, optimize_a4 222, nested_windows 222|222|válido|
|scored_0814|E2|52-1|256||218|válido|
|scored_0814|E2|52-3|231||212|válido|
|scored_0814|E2|52-6|222||203|válido|
|scored_0814|E2|64-1|256||219|válido|
|scored_0814|E2|64-3|231||204|válido|
|scored_0814|E2|64-6|222||200|válido|
|scored_0814|E3|joint|222|individual 222|222|OPTIMAL|
|scored_0814|E4|repair|222||222|completo|
|scored_0815|E1|1|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0815|E1|3|290|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|scored_0815|E1|6|284|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|scored_0815|E2|52-1|303||272|válido|
|scored_0815|E2|52-3|290||261|válido|
|scored_0815|E2|52-6|284||262|válido|
|scored_0815|E2|64-1|303||276|válido|
|scored_0815|E2|64-3|290||264|válido|
|scored_0815|E2|64-6|284||264|válido|
|scored_0815|E3|joint|284|individual 281|280|OPTIMAL|
|scored_0815|E4|repair|284||283|completo|
|scored_0816|E1|1|141|postprocess 141, optimize_a4 141, nested_windows 141|141|válido|
|scored_0816|E1|3|140|postprocess 140, optimize_a4 140, nested_windows 140|140|válido|
|scored_0816|E1|6|139|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|scored_0816|E2|52-1|141||133|válido|
|scored_0816|E2|52-3|140||133|válido|
|scored_0816|E2|52-6|139||133|válido|
|scored_0816|E2|64-1|141||133|válido|
|scored_0816|E2|64-3|140||133|válido|
|scored_0816|E2|64-6|139||133|válido|
|scored_0816|E3|joint|139|individual 139|139|OPTIMAL|
|scored_0816|E4|repair|139||135|completo|
|scored_0817|E1|1|236|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0817|E1|3|225|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0817|E1|6|226|postprocess 226, optimize_a4 226, nested_windows 226|226|válido|
|scored_0817|E2|52-1|236||218|válido|
|scored_0817|E2|52-3|225||214|válido|
|scored_0817|E2|52-6|226||209|válido|
|scored_0817|E2|64-1|236||219|válido|
|scored_0817|E2|64-3|225||215|válido|
|scored_0817|E2|64-6|226||209|válido|
|scored_0817|E3|joint|225|individual 225|224|OPTIMAL|
|scored_0817|E4|repair|225||225|completo|
|scored_0818|E1|1|34|postprocess 33, optimize_a4 32, nested_windows 32|32|válido|
|scored_0818|E1|3|34|postprocess 33, optimize_a4 32, nested_windows 32|32|válido|
|scored_0818|E1|6|34|postprocess 33, optimize_a4 32, nested_windows 32|32|válido|
|scored_0818|E2|52-1|34||34|válido|
|scored_0818|E2|52-3|34||34|válido|
|scored_0818|E2|52-6|34||34|válido|
|scored_0818|E2|64-1|34||34|válido|
|scored_0818|E2|64-3|34||34|válido|
|scored_0818|E2|64-6|34||34|válido|
|scored_0818|E3|joint|34|individual 34|34|OPTIMAL|
|scored_0818|E4|repair|34||34|completo|
|scored_0819|E1|1|327|postprocess 327, optimize_a4 327, nested_windows 327|327|válido|
|scored_0819|E1|3|306|postprocess 306, optimize_a4 306, nested_windows 306|306|válido|
|scored_0819|E1|6|310|postprocess 310, optimize_a4 310, nested_windows 310|310|válido|
|scored_0819|E2|52-1|327||308|válido|
|scored_0819|E2|52-3|306||296|válido|
|scored_0819|E2|52-6|310||282|válido|
|scored_0819|E2|64-1|327||308|válido|
|scored_0819|E2|64-3|306||297|válido|
|scored_0819|E2|64-6|310||282|válido|
|scored_0819|E3|joint|306|individual 305|304|FEASIBLE|
|scored_0819|E4|repair|306||306|completo|
|scored_0820|E1|1|326|postprocess 326, optimize_a4 326, nested_windows 326|326|válido|
|scored_0820|E1|3|308|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0820|E1|6|304|postprocess 304, optimize_a4 304, nested_windows 304|304|válido|
|scored_0820|E2|52-1|326||287|válido|
|scored_0820|E2|52-3|308||276|válido|
|scored_0820|E2|52-6|304||276|válido|
|scored_0820|E2|64-1|326||288|válido|
|scored_0820|E2|64-3|308||276|válido|
|scored_0820|E2|64-6|304||277|válido|
|scored_0820|E3|joint|304|individual 304|303|OPTIMAL|
|scored_0820|E4|repair|304||302|completo|
|scored_0821|E1|1|133|postprocess 133, optimize_a4 132, nested_windows 132|132|válido|
|scored_0821|E1|3|133|postprocess 133, optimize_a4 132, nested_windows 132|132|válido|
|scored_0821|E1|6|133|postprocess 133, optimize_a4 132, nested_windows 132|132|válido|
|scored_0821|E2|52-1|133||127|válido|
|scored_0821|E2|52-3|133||127|válido|
|scored_0821|E2|52-6|133||127|válido|
|scored_0821|E2|64-1|133||127|válido|
|scored_0821|E2|64-3|133||127|válido|
|scored_0821|E2|64-6|133||127|válido|
|scored_0821|E3|joint|133|individual 133|133|OPTIMAL|
|scored_0821|E4|repair|133||133|completo|
|scored_0822|E1|1|48|postprocess 46, optimize_a4 46, nested_windows 46|46|válido|
|scored_0822|E1|3|48|postprocess 46, optimize_a4 46, nested_windows 46|46|válido|
|scored_0822|E1|6|48|postprocess 46, optimize_a4 46, nested_windows 46|46|válido|
|scored_0822|E2|52-1|48||48|válido|
|scored_0822|E2|52-3|48||48|válido|
|scored_0822|E2|52-6|48||48|válido|
|scored_0822|E2|64-1|48||48|válido|
|scored_0822|E2|64-3|48||48|válido|
|scored_0822|E2|64-6|48||48|válido|
|scored_0822|E3|joint|48|individual 48|48|OPTIMAL|
|scored_0822|E4|repair|48||48|completo|
|scored_0823|E1|1|232|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|scored_0823|E1|3|226|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0823|E1|6|221|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0823|E2|52-1|232||211|válido|
|scored_0823|E2|52-3|226||208|válido|
|scored_0823|E2|52-6|221||209|válido|
|scored_0823|E2|64-1|232||218|válido|
|scored_0823|E2|64-3|226||208|válido|
|scored_0823|E2|64-6|221||192|válido|
|scored_0823|E3|joint|221|individual 221|221|OPTIMAL|
|scored_0823|E4|repair|221||219|completo|
|scored_0824|E1|1|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0824|E1|3|187|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0824|E1|6|187|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0824|E2|52-1|192||178|válido|
|scored_0824|E2|52-3|187||176|válido|
|scored_0824|E2|52-6|187||178|válido|
|scored_0824|E2|64-1|192||178|válido|
|scored_0824|E2|64-3|187||177|válido|
|scored_0824|E2|64-6|187||177|válido|
|scored_0824|E3|joint|187|individual 187|187|OPTIMAL|
|scored_0824|E4|repair|187||187|completo|
|scored_0825|E1|1|155|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|scored_0825|E1|3|153|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0825|E1|6|153|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0825|E2|52-1|155||151|válido|
|scored_0825|E2|52-3|153||151|válido|
|scored_0825|E2|52-6|153||151|válido|
|scored_0825|E2|64-1|155||152|válido|
|scored_0825|E2|64-3|153||152|válido|
|scored_0825|E2|64-6|153||152|válido|
|scored_0825|E3|joint|153|individual 153|153|OPTIMAL|
|scored_0825|E4|repair|153||152|completo|
|scored_0826|E1|1|301|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|scored_0826|E1|3|294|postprocess 294, optimize_a4 294, nested_windows 294|294|válido|
|scored_0826|E1|6|284|postprocess 284, optimize_a4 284, nested_windows 284|284|válido|
|scored_0826|E2|52-1|301||270|válido|
|scored_0826|E2|52-3|294||271|válido|
|scored_0826|E2|52-6|284||259|válido|
|scored_0826|E2|64-1|301||277|válido|
|scored_0826|E2|64-3|294||268|válido|
|scored_0826|E2|64-6|284||261|válido|
|scored_0826|E3|joint|284|individual 283|283|OPTIMAL|
|scored_0826|E4|repair|284||284|completo|
|scored_0827|E1|1|281|postprocess 280, optimize_a4 280, nested_windows 280|280|válido|
|scored_0827|E1|3|265|postprocess 265, optimize_a4 265, nested_windows 265|265|válido|
|scored_0827|E1|6|260|postprocess 260, optimize_a4 260, nested_windows 260|260|válido|
|scored_0827|E2|52-1|281||249|válido|
|scored_0827|E2|52-3|265||246|válido|
|scored_0827|E2|52-6|260||241|válido|
|scored_0827|E2|64-1|281||253|válido|
|scored_0827|E2|64-3|265||245|válido|
|scored_0827|E2|64-6|260||264|válido|
|scored_0827|E3|joint|260|individual 260|256|OPTIMAL|
|scored_0827|E4|repair|260||259|completo|
|scored_0828|E1|1|45|postprocess 42, optimize_a4 42, nested_windows 42|42|válido|
|scored_0828|E1|3|45|postprocess 42, optimize_a4 42, nested_windows 42|42|válido|
|scored_0828|E1|6|45|postprocess 42, optimize_a4 42, nested_windows 42|42|válido|
|scored_0828|E2|52-1|45||45|válido|
|scored_0828|E2|52-3|45||45|válido|
|scored_0828|E2|52-6|45||45|válido|
|scored_0828|E2|64-1|45||45|válido|
|scored_0828|E2|64-3|45||45|válido|
|scored_0828|E2|64-6|45||45|válido|
|scored_0828|E3|joint|45|individual 45|45|OPTIMAL|
|scored_0828|E4|repair|45||45|completo|
|scored_0829|E1|1|237|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|scored_0829|E1|3|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0829|E1|6|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0829|E2|52-1|237||220|válido|
|scored_0829|E2|52-3|232||212|válido|
|scored_0829|E2|52-6|232||213|válido|
|scored_0829|E2|64-1|237||221|válido|
|scored_0829|E2|64-3|232||215|válido|
|scored_0829|E2|64-6|232||212|válido|
|scored_0829|E3|joint|232|individual 231|229|OPTIMAL|
|scored_0829|E4|repair|232||232|completo|
|scored_0830|E1|1|132|postprocess 132, optimize_a4 132, nested_windows 132|132|válido|
|scored_0830|E1|3|131|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|scored_0830|E1|6|131|postprocess 131, optimize_a4 131, nested_windows 131|131|válido|
|scored_0830|E2|52-1|132||130|válido|
|scored_0830|E2|52-3|131||128|válido|
|scored_0830|E2|52-6|131||127|válido|
|scored_0830|E2|64-1|132||130|válido|
|scored_0830|E2|64-3|131||127|válido|
|scored_0830|E2|64-6|131||127|válido|
|scored_0830|E3|joint|131|individual 131|131|OPTIMAL|
|scored_0830|E4|repair|131||130|completo|
|scored_0831|E1|1|121|postprocess 120, optimize_a4 120, nested_windows 120|120|válido|
|scored_0831|E1|3|121|postprocess 120, optimize_a4 120, nested_windows 120|120|válido|
|scored_0831|E1|6|121|postprocess 120, optimize_a4 120, nested_windows 120|120|válido|
|scored_0831|E2|52-1|121||119|válido|
|scored_0831|E2|52-3|121||119|válido|
|scored_0831|E2|52-6|121||119|válido|
|scored_0831|E2|64-1|121||119|válido|
|scored_0831|E2|64-3|121||119|válido|
|scored_0831|E2|64-6|121||119|válido|
|scored_0831|E3|joint|121|individual 121|121|OPTIMAL|
|scored_0831|E4|repair|121||119|completo|
|scored_0832|E1|1|250|postprocess 250, optimize_a4 250, nested_windows 250|250|válido|
|scored_0832|E1|3|250|postprocess 250, optimize_a4 250, nested_windows 250|250|válido|
|scored_0832|E1|6|246|postprocess 246, optimize_a4 246, nested_windows 246|246|válido|
|scored_0832|E2|52-1|250||231|válido|
|scored_0832|E2|52-3|250||230|válido|
|scored_0832|E2|52-6|246||230|válido|
|scored_0832|E2|64-1|250||233|válido|
|scored_0832|E2|64-3|250||230|válido|
|scored_0832|E2|64-6|246||235|válido|
|scored_0832|E3|joint|246|individual 246|243|OPTIMAL|
|scored_0832|E4|repair|246||246|completo|
|scored_0833|E1|1|170|postprocess 170, optimize_a4 170, nested_windows 170|170|válido|
|scored_0833|E1|3|167|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0833|E1|6|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0833|E2|52-1|170||162|válido|
|scored_0833|E2|52-3|167||165|válido|
|scored_0833|E2|52-6|165||157|válido|
|scored_0833|E2|64-1|170||161|válido|
|scored_0833|E2|64-3|167||160|válido|
|scored_0833|E2|64-6|165||157|válido|
|scored_0833|E3|joint|165|individual 165|165|OPTIMAL|
|scored_0833|E4|repair|165||165|completo|
|scored_0834|E1|1|300|postprocess 300, optimize_a4 300, nested_windows 300|300|válido|
|scored_0834|E1|3|294|postprocess 294, optimize_a4 294, nested_windows 294|294|válido|
|scored_0834|E1|6|283|postprocess 282, optimize_a4 282, nested_windows 282|282|válido|
|scored_0834|E2|52-1|300||276|válido|
|scored_0834|E2|52-3|294||262|válido|
|scored_0834|E2|52-6|283||267|válido|
|scored_0834|E2|64-1|300||277|válido|
|scored_0834|E2|64-3|294||262|válido|
|scored_0834|E2|64-6|283||266|válido|
|scored_0834|E3|joint|283|individual 282|282|OPTIMAL|
|scored_0834|E4|repair|283||283|completo|
|scored_0835|E1|1|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0835|E1|3|150|postprocess 150, optimize_a4 150, nested_windows 150|150|válido|
|scored_0835|E1|6|142|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0835|E2|52-1|151||144|válido|
|scored_0835|E2|52-3|150||143|válido|
|scored_0835|E2|52-6|142||142|válido|
|scored_0835|E2|64-1|151||141|válido|
|scored_0835|E2|64-3|150||141|válido|
|scored_0835|E2|64-6|142||143|válido|
|scored_0835|E3|joint|142|individual 142|142|OPTIMAL|
|scored_0835|E4|repair|142||142|completo|
|scored_0836|E1|1|257|postprocess 257, optimize_a4 257, nested_windows 257|257|válido|
|scored_0836|E1|3|251|postprocess 251, optimize_a4 251, nested_windows 251|251|válido|
|scored_0836|E1|6|249|postprocess 249, optimize_a4 249, nested_windows 249|249|válido|
|scored_0836|E2|52-1|257||242|válido|
|scored_0836|E2|52-3|251||234|válido|
|scored_0836|E2|52-6|249||232|válido|
|scored_0836|E2|64-1|257||243|válido|
|scored_0836|E2|64-3|251||235|válido|
|scored_0836|E2|64-6|249||233|válido|
|scored_0836|E3|joint|249|individual 249|249|OPTIMAL|
|scored_0836|E4|repair|249||249|completo|
|scored_0837|E1|1|311|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|scored_0837|E1|3|304|postprocess 304, optimize_a4 304, nested_windows 304|304|válido|
|scored_0837|E1|6|295|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|scored_0837|E2|52-1|311||294|válido|
|scored_0837|E2|52-3|304||288|válido|
|scored_0837|E2|52-6|295||289|válido|
|scored_0837|E2|64-1|311||294|válido|
|scored_0837|E2|64-3|304||336|válido|
|scored_0837|E2|64-6|295||289|válido|
|scored_0837|E3|joint|295|individual 295|295|rechazado por registros|
|scored_0837|E4|repair|295||295|completo|
|scored_0838|E1|1|186|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0838|E1|3|185|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|scored_0838|E1|6|181|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0838|E2|52-1|186||176|válido|
|scored_0838|E2|52-3|185||172|válido|
|scored_0838|E2|52-6|181||172|válido|
|scored_0838|E2|64-1|186||176|válido|
|scored_0838|E2|64-3|185||172|válido|
|scored_0838|E2|64-6|181||172|válido|
|scored_0838|E3|joint|181|individual 181|181|OPTIMAL|
|scored_0838|E4|repair|181||181|completo|
|scored_0839|E1|1|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0839|E1|3|231|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|scored_0839|E1|6|231|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|scored_0839|E2|52-1|235||218|válido|
|scored_0839|E2|52-3|231||215|válido|
|scored_0839|E2|52-6|231||212|válido|
|scored_0839|E2|64-1|235||218|válido|
|scored_0839|E2|64-3|231||216|válido|
|scored_0839|E2|64-6|231||210|válido|
|scored_0839|E3|joint|231|individual 231|231|OPTIMAL|
|scored_0839|E4|repair|231||231|completo|
|scored_0840|E1|1|330|postprocess 330, optimize_a4 330, nested_windows 330|330|válido|
|scored_0840|E1|3|308|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0840|E1|6|302|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|scored_0840|E2|52-1|330||300|válido|
|scored_0840|E2|52-3|308||292|válido|
|scored_0840|E2|52-6|302||283|válido|
|scored_0840|E2|64-1|330||301|válido|
|scored_0840|E2|64-3|308||290|válido|
|scored_0840|E2|64-6|302||282|válido|
|scored_0840|E3|joint|302|individual 302|301|OPTIMAL|
|scored_0840|E4|repair|302||302|completo|
|scored_0841|E1|1|228|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|scored_0841|E1|3|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0841|E1|6|209|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0841|E2|52-1|228||203|válido|
|scored_0841|E2|52-3|211||203|válido|
|scored_0841|E2|52-6|209||190|válido|
|scored_0841|E2|64-1|228||204|válido|
|scored_0841|E2|64-3|211||206|válido|
|scored_0841|E2|64-6|209||195|válido|
|scored_0841|E3|joint|209|individual 208|206|OPTIMAL|
|scored_0841|E4|repair|209||203|completo|
|scored_0842|E1|1|39|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0842|E1|3|39|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0842|E1|6|39|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0842|E2|52-1|39||39|válido|
|scored_0842|E2|52-3|39||39|válido|
|scored_0842|E2|52-6|39||39|válido|
|scored_0842|E2|64-1|39||39|válido|
|scored_0842|E2|64-3|39||39|válido|
|scored_0842|E2|64-6|39||39|válido|
|scored_0842|E3|joint|38|individual 38|37|OPTIMAL|
|scored_0842|E4|repair|38||36|completo|
|scored_0843|E1|1|193|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0843|E1|3|190|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0843|E1|6|190|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|scored_0843|E2|52-1|193||178|válido|
|scored_0843|E2|52-3|190||180|válido|
|scored_0843|E2|52-6|190||178|válido|
|scored_0843|E2|64-1|193||179|válido|
|scored_0843|E2|64-3|190||181|válido|
|scored_0843|E2|64-6|190||177|válido|
|scored_0843|E3|joint|190|individual 189|188|OPTIMAL|
|scored_0843|E4|repair|190||190|completo|
|scored_0844|E1|1|359|postprocess 359, optimize_a4 359, nested_windows 359|359|válido|
|scored_0844|E1|3|337|postprocess 337, optimize_a4 337, nested_windows 337|337|válido|
|scored_0844|E1|6|330|postprocess 329, optimize_a4 329, nested_windows 329|329|válido|
|scored_0844|E2|52-1|359||332|válido|
|scored_0844|E2|52-3|337||319|válido|
|scored_0844|E2|52-6|330||318|válido|
|scored_0844|E2|64-1|359||333|válido|
|scored_0844|E2|64-3|337||320|válido|
|scored_0844|E2|64-6|330||318|válido|
|scored_0844|E3|joint|330|individual 330|329|FEASIBLE|
|scored_0844|E4|repair|330||330|completo|
|scored_0845|E1|1|313|postprocess 312, optimize_a4 312, nested_windows 312|312|válido|
|scored_0845|E1|3|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|scored_0845|E1|6|288|postprocess 286, optimize_a4 286, nested_windows 286|286|válido|
|scored_0845|E2|52-1|313||281|válido|
|scored_0845|E2|52-3|289||278|válido|
|scored_0845|E2|52-6|288||265|válido|
|scored_0845|E2|64-1|313||284|válido|
|scored_0845|E2|64-3|289||277|válido|
|scored_0845|E2|64-6|288||269|válido|
|scored_0845|E3|joint|288|individual 288|287|FEASIBLE|
|scored_0845|E4|repair|288||288|completo|
|scored_0846|E1|1|129|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|scored_0846|E1|3|129|postprocess 129, optimize_a4 129, nested_windows 129|129|válido|
|scored_0846|E1|6|127|postprocess 127, optimize_a4 127, nested_windows 127|127|válido|
|scored_0846|E2|52-1|129||127|válido|
|scored_0846|E2|52-3|129||128|válido|
|scored_0846|E2|52-6|127||128|válido|
|scored_0846|E2|64-1|129||127|válido|
|scored_0846|E2|64-3|129||128|válido|
|scored_0846|E2|64-6|127||128|válido|
|scored_0846|E3|joint|127|individual 127|127|OPTIMAL|
|scored_0846|E4|repair|127||125|completo|
|scored_0847|E1|1|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|scored_0847|E1|3|234|postprocess 234, optimize_a4 234, nested_windows 234|234|válido|
|scored_0847|E1|6|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0847|E2|52-1|238||227|válido|
|scored_0847|E2|52-3|234||216|válido|
|scored_0847|E2|52-6|229||220|válido|
|scored_0847|E2|64-1|238||225|válido|
|scored_0847|E2|64-3|234||218|válido|
|scored_0847|E2|64-6|229||222|válido|
|scored_0847|E3|joint|229|individual 229|229|OPTIMAL|
|scored_0847|E4|repair|229||228|completo|
|scored_0848|E1|1|357|postprocess 356, optimize_a4 356, nested_windows 356|356|válido|
|scored_0848|E1|3|334|postprocess 334, optimize_a4 334, nested_windows 334|334|válido|
|scored_0848|E1|6|338|postprocess 337, optimize_a4 337, nested_windows 337|337|válido|
|scored_0848|E2|52-1|357||324|válido|
|scored_0848|E2|52-3|334||312|válido|
|scored_0848|E2|52-6|338||312|válido|
|scored_0848|E2|64-1|357||327|válido|
|scored_0848|E2|64-3|334||312|válido|
|scored_0848|E2|64-6|338||312|válido|
|scored_0848|E3|joint|334|individual 334|331|FEASIBLE|
|scored_0848|E4|repair|334||334|completo|
|scored_0849|E1|1|270|postprocess 269, optimize_a4 269, nested_windows 269|269|válido|
|scored_0849|E1|3|259|postprocess 259, optimize_a4 259, nested_windows 259|259|válido|
|scored_0849|E1|6|258|postprocess 258, optimize_a4 258, nested_windows 258|258|válido|
|scored_0849|E2|52-1|270||249|válido|
|scored_0849|E2|52-3|259||240|válido|
|scored_0849|E2|52-6|258||243|válido|
|scored_0849|E2|64-1|270||244|válido|
|scored_0849|E2|64-3|259||241|válido|
|scored_0849|E2|64-6|258||240|válido|
|scored_0849|E3|joint|258|individual 257|256|OPTIMAL|
|scored_0849|E4|repair|258||258|completo|
|scored_0850|E1|1|240|postprocess 240, optimize_a4 240, nested_windows 240|240|válido|
|scored_0850|E1|3|234|postprocess 234, optimize_a4 234, nested_windows 234|234|válido|
|scored_0850|E1|6|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0850|E2|52-1|240||222|válido|
|scored_0850|E2|52-3|234||219|válido|
|scored_0850|E2|52-6|232||218|válido|
|scored_0850|E2|64-1|240||224|válido|
|scored_0850|E2|64-3|234||219|válido|
|scored_0850|E2|64-6|232||219|válido|
|scored_0850|E3|joint|232|individual 232|229|OPTIMAL|
|scored_0850|E4|repair|232||232|completo|
|scored_0851|E1|1|44|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0851|E1|3|44|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0851|E1|6|44|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0851|E2|52-1|44||44|válido|
|scored_0851|E2|52-3|44||44|válido|
|scored_0851|E2|52-6|44||44|válido|
|scored_0851|E2|64-1|44||44|válido|
|scored_0851|E2|64-3|44||44|válido|
|scored_0851|E2|64-6|44||44|válido|
|scored_0851|E3|joint|44|individual 44|44|OPTIMAL|
|scored_0851|E4|repair|44||43|completo|
|scored_0852|E1|1|224|postprocess 223, optimize_a4 223, nested_windows 223|223|válido|
|scored_0852|E1|3|214|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0852|E1|6|210|postprocess 210, optimize_a4 210, nested_windows 210|210|válido|
|scored_0852|E2|52-1|224||207|válido|
|scored_0852|E2|52-3|214||199|válido|
|scored_0852|E2|52-6|210||199|válido|
|scored_0852|E2|64-1|224||206|válido|
|scored_0852|E2|64-3|214||198|válido|
|scored_0852|E2|64-6|210||198|válido|
|scored_0852|E3|joint|210|individual 210|210|OPTIMAL|
|scored_0852|E4|repair|210||209|completo|
|scored_0853|E1|1|236|postprocess 236, optimize_a4 236, nested_windows 236|236|válido|
|scored_0853|E1|3|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0853|E1|6|232|postprocess 232, optimize_a4 232, nested_windows 232|232|válido|
|scored_0853|E2|52-1|236||227|válido|
|scored_0853|E2|52-3|235||225|válido|
|scored_0853|E2|52-6|232||218|válido|
|scored_0853|E2|64-1|236||227|válido|
|scored_0853|E2|64-3|235||224|válido|
|scored_0853|E2|64-6|232||220|válido|
|scored_0853|E3|joint|232|individual 232|231|OPTIMAL|
|scored_0853|E4|repair|232||232|completo|
|scored_0854|E1|1|221|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0854|E1|3|216|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0854|E1|6|213|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|scored_0854|E2|52-1|221||207|válido|
|scored_0854|E2|52-3|216||203|válido|
|scored_0854|E2|52-6|213||201|válido|
|scored_0854|E2|64-1|221||206|válido|
|scored_0854|E2|64-3|216||199|válido|
|scored_0854|E2|64-6|213||197|válido|
|scored_0854|E3|joint|213|individual 213|212|OPTIMAL|
|scored_0854|E4|repair|213||213|completo|
|scored_0855|E1|1|93|postprocess 93, optimize_a4 93, nested_windows 93|93|válido|
|scored_0855|E1|3|93|postprocess 93, optimize_a4 93, nested_windows 93|93|válido|
|scored_0855|E1|6|93|postprocess 93, optimize_a4 93, nested_windows 93|93|válido|
|scored_0855|E2|52-1|93||91|válido|
|scored_0855|E2|52-3|93||91|válido|
|scored_0855|E2|52-6|93||91|válido|
|scored_0855|E2|64-1|93||91|válido|
|scored_0855|E2|64-3|93||91|válido|
|scored_0855|E2|64-6|93||91|válido|
|scored_0855|E3|joint|93|individual 93|93|OPTIMAL|
|scored_0855|E4|repair|93||91|completo|
|scored_0856|E1|1|313|postprocess 312, optimize_a4 312, nested_windows 312|312|válido|
|scored_0856|E1|3|286|postprocess 286, optimize_a4 286, nested_windows 286|286|válido|
|scored_0856|E1|6|285|postprocess 284, optimize_a4 284, nested_windows 284|284|válido|
|scored_0856|E2|52-1|313||287|válido|
|scored_0856|E2|52-3|286||272|válido|
|scored_0856|E2|52-6|285||268|válido|
|scored_0856|E2|64-1|313||286|válido|
|scored_0856|E2|64-3|286||272|válido|
|scored_0856|E2|64-6|285||267|válido|
|scored_0856|E3|joint|285|individual 285|282|FEASIBLE|
|scored_0856|E4|repair|285||285|completo|
|scored_0857|E1|1|357|postprocess 348, optimize_a4 348, nested_windows 348|348|válido|
|scored_0857|E1|3|308|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0857|E1|6|299|postprocess 299, optimize_a4 299, nested_windows 299|299|válido|
|scored_0857|E2|52-1|357||295|válido|
|scored_0857|E2|52-3|308||286|válido|
|scored_0857|E2|52-6|299||280|válido|
|scored_0857|E2|64-1|357||293|válido|
|scored_0857|E2|64-3|308||286|válido|
|scored_0857|E2|64-6|299||281|válido|
|scored_0857|E3|joint|299|individual 299|299|OPTIMAL|
|scored_0857|E4|repair|299||299|completo|
|scored_0858|E1|1|57|postprocess 56, optimize_a4 56, nested_windows 56|56|válido|
|scored_0858|E1|3|57|postprocess 56, optimize_a4 56, nested_windows 56|56|válido|
|scored_0858|E1|6|57|postprocess 56, optimize_a4 56, nested_windows 56|56|válido|
|scored_0858|E2|52-1|57||57|válido|
|scored_0858|E2|52-3|57||57|válido|
|scored_0858|E2|52-6|57||57|válido|
|scored_0858|E2|64-1|57||57|válido|
|scored_0858|E2|64-3|57||57|válido|
|scored_0858|E2|64-6|57||57|válido|
|scored_0858|E3|joint|57|individual 57|57|OPTIMAL|
|scored_0858|E4|repair|57||57|completo|
|scored_0859|E1|1|310|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0859|E1|3|288|postprocess 288, optimize_a4 288, nested_windows 288|288|válido|
|scored_0859|E1|6|279|postprocess 279, optimize_a4 279, nested_windows 279|279|válido|
|scored_0859|E2|52-1|310||288|válido|
|scored_0859|E2|52-3|288||274|válido|
|scored_0859|E2|52-6|279||267|válido|
|scored_0859|E2|64-1|310||289|válido|
|scored_0859|E2|64-3|288||280|válido|
|scored_0859|E2|64-6|279||268|válido|
|scored_0859|E3|joint|279|individual 278|278|FEASIBLE|
|scored_0859|E4|repair|279||279|completo|
|scored_0860|E1|1|194|postprocess 194, optimize_a4 194, nested_windows 194|194|válido|
|scored_0860|E1|3|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0860|E1|6|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0860|E2|52-1|194||183|válido|
|scored_0860|E2|52-3|188||181|válido|
|scored_0860|E2|52-6|188||180|válido|
|scored_0860|E2|64-1|194||184|válido|
|scored_0860|E2|64-3|188||181|válido|
|scored_0860|E2|64-6|188||181|válido|
|scored_0860|E3|joint|188|individual 188|188|OPTIMAL|
|scored_0860|E4|repair|188||187|completo|
|scored_0861|E1|1|213|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|scored_0861|E1|3|213|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0861|E1|6|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0861|E2|52-1|213||202|válido|
|scored_0861|E2|52-3|213||198|válido|
|scored_0861|E2|52-6|211||199|válido|
|scored_0861|E2|64-1|213||204|válido|
|scored_0861|E2|64-3|213||197|válido|
|scored_0861|E2|64-6|211||204|válido|
|scored_0861|E3|joint|211|individual 211|210|OPTIMAL|
|scored_0861|E4|repair|211||211|completo|
|scored_0862|E1|1|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0862|E1|3|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0862|E1|6|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0862|E2|52-1|169||165|válido|
|scored_0862|E2|52-3|169||165|válido|
|scored_0862|E2|52-6|169||165|válido|
|scored_0862|E2|64-1|169||169|válido|
|scored_0862|E2|64-3|169||168|válido|
|scored_0862|E2|64-6|169||166|válido|
|scored_0862|E3|joint|169|individual 168|168|OPTIMAL|
|scored_0862|E4|repair|169||166|completo|
|scored_0863|E1|1|45|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0863|E1|3|45|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0863|E1|6|45|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0863|E2|52-1|45||45|válido|
|scored_0863|E2|52-3|45||45|válido|
|scored_0863|E2|52-6|45||45|válido|
|scored_0863|E2|64-1|45||45|válido|
|scored_0863|E2|64-3|45||45|válido|
|scored_0863|E2|64-6|45||45|válido|
|scored_0863|E3|joint|45|individual 45|45|OPTIMAL|
|scored_0863|E4|repair|45||44|completo|
|scored_0864|E1|1|270|postprocess 270, optimize_a4 270, nested_windows 270|270|válido|
|scored_0864|E1|3|249|postprocess 249, optimize_a4 249, nested_windows 249|249|válido|
|scored_0864|E1|6|250|postprocess 249, optimize_a4 249, nested_windows 249|249|válido|
|scored_0864|E2|52-1|270||238|válido|
|scored_0864|E2|52-3|249||235|válido|
|scored_0864|E2|52-6|250||230|válido|
|scored_0864|E2|64-1|270||243|válido|
|scored_0864|E2|64-3|249||234|válido|
|scored_0864|E2|64-6|250||231|válido|
|scored_0864|E3|joint|249|individual 249|244|OPTIMAL|
|scored_0864|E4|repair|249||249|completo|
|scored_0865|E1|1|335|postprocess 333, optimize_a4 333, nested_windows 333|333|válido|
|scored_0865|E1|3|320|postprocess 320, optimize_a4 320, nested_windows 320|320|válido|
|scored_0865|E1|6|311|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|scored_0865|E2|52-1|335||306|válido|
|scored_0865|E2|52-3|320||298|válido|
|scored_0865|E2|52-6|311||294|válido|
|scored_0865|E2|64-1|335||310|válido|
|scored_0865|E2|64-3|320||299|válido|
|scored_0865|E2|64-6|311||293|válido|
|scored_0865|E3|joint|311|individual 310|309|OPTIMAL|
|scored_0865|E4|repair|311||311|completo|
|scored_0866|E1|1|44|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0866|E1|3|44|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0866|E1|6|44|postprocess 41, optimize_a4 41, nested_windows 41|41|válido|
|scored_0866|E2|52-1|44||44|válido|
|scored_0866|E2|52-3|44||44|válido|
|scored_0866|E2|52-6|44||44|válido|
|scored_0866|E2|64-1|44||44|válido|
|scored_0866|E2|64-3|44||44|válido|
|scored_0866|E2|64-6|44||44|válido|
|scored_0866|E3|joint|44|individual 44|44|OPTIMAL|
|scored_0866|E4|repair|44||43|completo|
|scored_0867|E1|1|119|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0867|E1|3|117|postprocess 115, optimize_a4 115, nested_windows 115|115|válido|
|scored_0867|E1|6|117|postprocess 115, optimize_a4 115, nested_windows 115|115|válido|
|scored_0867|E2|52-1|119||113|válido|
|scored_0867|E2|52-3|117||114|válido|
|scored_0867|E2|52-6|117||111|válido|
|scored_0867|E2|64-1|119||113|válido|
|scored_0867|E2|64-3|117||114|válido|
|scored_0867|E2|64-6|117||111|válido|
|scored_0867|E3|joint|117|individual 117|117|OPTIMAL|
|scored_0867|E4|repair|117||117|completo|
|scored_0868|E1|1|43|postprocess 42, optimize_a4 42, nested_windows 42|42|válido|
|scored_0868|E1|3|43|postprocess 42, optimize_a4 42, nested_windows 42|42|válido|
|scored_0868|E1|6|43|postprocess 42, optimize_a4 42, nested_windows 42|42|válido|
|scored_0868|E2|52-1|43||43|válido|
|scored_0868|E2|52-3|43||43|válido|
|scored_0868|E2|52-6|43||43|válido|
|scored_0868|E2|64-1|43||43|válido|
|scored_0868|E2|64-3|43||43|válido|
|scored_0868|E2|64-6|43||43|válido|
|scored_0868|E3|joint|42|individual 40|40|OPTIMAL|
|scored_0868|E4|repair|42||42|completo|
|scored_0869|E1|1|174|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0869|E1|3|165|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0869|E1|6|165|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0869|E2|52-1|174||159|válido|
|scored_0869|E2|52-3|165||158|válido|
|scored_0869|E2|52-6|165||156|válido|
|scored_0869|E2|64-1|174||159|válido|
|scored_0869|E2|64-3|165||158|válido|
|scored_0869|E2|64-6|165||157|válido|
|scored_0869|E3|joint|165|individual 165|165|OPTIMAL|
|scored_0869|E4|repair|165||163|completo|
|scored_0870|E1|1|310|postprocess 310, optimize_a4 310, nested_windows 310|310|válido|
|scored_0870|E1|3|296|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0870|E1|6|287|postprocess 287, optimize_a4 287, nested_windows 287|287|válido|
|scored_0870|E2|52-1|310||283|válido|
|scored_0870|E2|52-3|296||272|válido|
|scored_0870|E2|52-6|287||269|válido|
|scored_0870|E2|64-1|310||282|válido|
|scored_0870|E2|64-3|296||274|válido|
|scored_0870|E2|64-6|287||271|válido|
|scored_0870|E3|joint|287|individual 287|287|OPTIMAL|
|scored_0870|E4|repair|287||287|completo|
|scored_0871|E1|1|153|postprocess 153, optimize_a4 153, nested_windows 153|153|válido|
|scored_0871|E1|3|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0871|E1|6|151|postprocess 151, optimize_a4 151, nested_windows 151|151|válido|
|scored_0871|E2|52-1|153||142|válido|
|scored_0871|E2|52-3|151||142|válido|
|scored_0871|E2|52-6|151||140|válido|
|scored_0871|E2|64-1|153||142|válido|
|scored_0871|E2|64-3|151||142|válido|
|scored_0871|E2|64-6|151||140|válido|
|scored_0871|E3|joint|151|individual 151|151|OPTIMAL|
|scored_0871|E4|repair|151||149|completo|
|scored_0872|E1|1|139|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0872|E1|3|137|postprocess 135, optimize_a4 135, nested_windows 135|135|válido|
|scored_0872|E1|6|136|postprocess 134, optimize_a4 134, nested_windows 134|134|válido|
|scored_0872|E2|52-1|139||132|válido|
|scored_0872|E2|52-3|137||132|válido|
|scored_0872|E2|52-6|136||132|válido|
|scored_0872|E2|64-1|139||132|válido|
|scored_0872|E2|64-3|137||132|válido|
|scored_0872|E2|64-6|136||132|válido|
|scored_0872|E3|joint|136|individual 136|136|OPTIMAL|
|scored_0872|E4|repair|136||132|completo|
|scored_0873|E1|1|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0873|E1|3|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0873|E1|6|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0873|E2|52-1|50||50|válido|
|scored_0873|E2|52-3|50||50|válido|
|scored_0873|E2|52-6|50||50|válido|
|scored_0873|E2|64-1|50||50|válido|
|scored_0873|E2|64-3|50||50|válido|
|scored_0873|E2|64-6|50||50|válido|
|scored_0873|E3|joint|50|individual 50|50|OPTIMAL|
|scored_0873|E4|repair|50||50|completo|
|scored_0874|E1|1|47|postprocess 46, optimize_a4 46, nested_windows 46|46|válido|
|scored_0874|E1|3|47|postprocess 46, optimize_a4 46, nested_windows 46|46|válido|
|scored_0874|E1|6|47|postprocess 46, optimize_a4 46, nested_windows 46|46|válido|
|scored_0874|E2|52-1|47||47|válido|
|scored_0874|E2|52-3|47||47|válido|
|scored_0874|E2|52-6|47||47|válido|
|scored_0874|E2|64-1|47||47|válido|
|scored_0874|E2|64-3|47||47|válido|
|scored_0874|E2|64-6|47||47|válido|
|scored_0874|E3|joint|47|individual 47|47|OPTIMAL|
|scored_0874|E4|repair|47||47|completo|
|scored_0875|E1|1|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0875|E1|3|217|postprocess 217, optimize_a4 217, nested_windows 217|217|válido|
|scored_0875|E1|6|214|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0875|E2|52-1|220||206|válido|
|scored_0875|E2|52-3|217||204|válido|
|scored_0875|E2|52-6|214||197|válido|
|scored_0875|E2|64-1|220||207|válido|
|scored_0875|E2|64-3|217||204|válido|
|scored_0875|E2|64-6|214||197|válido|
|scored_0875|E3|joint|214|individual 214|213|OPTIMAL|
|scored_0875|E4|repair|214||214|completo|
|scored_0876|E1|1|289|postprocess 289, optimize_a4 289, nested_windows 289|289|válido|
|scored_0876|E1|3|272|postprocess 272, optimize_a4 272, nested_windows 272|272|válido|
|scored_0876|E1|6|269|postprocess 269, optimize_a4 269, nested_windows 269|269|válido|
|scored_0876|E2|52-1|289||257|válido|
|scored_0876|E2|52-3|272||255|válido|
|scored_0876|E2|52-6|269||245|válido|
|scored_0876|E2|64-1|289||261|válido|
|scored_0876|E2|64-3|272||256|válido|
|scored_0876|E2|64-6|269||246|válido|
|scored_0876|E3|joint|269|individual 268|267|OPTIMAL|
|scored_0876|E4|repair|269||269|completo|
|scored_0877|E1|1|79|postprocess 78, optimize_a4 78, nested_windows 78|78|válido|
|scored_0877|E1|3|79|postprocess 78, optimize_a4 78, nested_windows 78|78|válido|
|scored_0877|E1|6|79|postprocess 78, optimize_a4 78, nested_windows 78|78|válido|
|scored_0877|E2|52-1|79||77|válido|
|scored_0877|E2|52-3|79||77|válido|
|scored_0877|E2|52-6|79||77|válido|
|scored_0877|E2|64-1|79||77|válido|
|scored_0877|E2|64-3|79||77|válido|
|scored_0877|E2|64-6|79||77|válido|
|scored_0877|E3|joint|79|individual 79|79|OPTIMAL|
|scored_0877|E4|repair|79||75|completo|
|scored_0878|E1|1|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0878|E1|3|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0878|E1|6|212|postprocess 212, optimize_a4 212, nested_windows 212|212|válido|
|scored_0878|E2|52-1|219||203|válido|
|scored_0878|E2|52-3|215||194|válido|
|scored_0878|E2|52-6|212||193|válido|
|scored_0878|E2|64-1|219||203|válido|
|scored_0878|E2|64-3|215||195|válido|
|scored_0878|E2|64-6|212||195|válido|
|scored_0878|E3|joint|212|individual 212|212|OPTIMAL|
|scored_0878|E4|repair|212||212|completo|
|scored_0879|E1|1|138|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0879|E1|3|138|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0879|E1|6|138|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0879|E2|52-1|138||132|válido|
|scored_0879|E2|52-3|138||131|válido|
|scored_0879|E2|52-6|138||130|válido|
|scored_0879|E2|64-1|138||130|válido|
|scored_0879|E2|64-3|138||130|válido|
|scored_0879|E2|64-6|138||129|válido|
|scored_0879|E3|joint|138|individual 136|135|OPTIMAL|
|scored_0879|E4|repair|138||137|completo|
|scored_0880|E1|1|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0880|E1|3|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0880|E1|6|50|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0880|E2|52-1|50||50|válido|
|scored_0880|E2|52-3|50||50|válido|
|scored_0880|E2|52-6|50||50|válido|
|scored_0880|E2|64-1|50||50|válido|
|scored_0880|E2|64-3|50||50|válido|
|scored_0880|E2|64-6|50||50|válido|
|scored_0880|E3|joint|50|individual 50|50|OPTIMAL|
|scored_0880|E4|repair|50||50|completo|
|scored_0881|E1|1|114|postprocess 114, optimize_a4 114, nested_windows 114|114|válido|
|scored_0881|E1|3|114|postprocess 114, optimize_a4 114, nested_windows 114|114|válido|
|scored_0881|E1|6|113|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|scored_0881|E2|52-1|114||111|válido|
|scored_0881|E2|52-3|114||111|válido|
|scored_0881|E2|52-6|113||111|válido|
|scored_0881|E2|64-1|114||111|válido|
|scored_0881|E2|64-3|114||111|válido|
|scored_0881|E2|64-6|113||111|válido|
|scored_0881|E3|joint|113|individual 113|113|OPTIMAL|
|scored_0881|E4|repair|113||109|completo|
|scored_0882|E1|1|37|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0882|E1|3|37|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0882|E1|6|37|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0882|E2|52-1|37||37|válido|
|scored_0882|E2|52-3|37||37|válido|
|scored_0882|E2|52-6|37||37|válido|
|scored_0882|E2|64-1|37||37|válido|
|scored_0882|E2|64-3|37||37|válido|
|scored_0882|E2|64-6|37||37|válido|
|scored_0882|E3|joint|36|individual 35|35|OPTIMAL|
|scored_0882|E4|repair|36||32|completo|
|scored_0883|E1|1|10|postprocess 8, optimize_a4 7, nested_windows 7|7|válido|
|scored_0883|E1|3|10|postprocess 8, optimize_a4 7, nested_windows 7|7|válido|
|scored_0883|E1|6|10|postprocess 8, optimize_a4 7, nested_windows 7|7|válido|
|scored_0883|E2|52-1|10||10|válido|
|scored_0883|E2|52-3|10||10|válido|
|scored_0883|E2|52-6|10||10|válido|
|scored_0883|E2|64-1|10||10|válido|
|scored_0883|E2|64-3|10||10|válido|
|scored_0883|E2|64-6|10||10|válido|
|scored_0883|E3|joint|7|individual 7|7|OPTIMAL|
|scored_0883|E4|repair|7||7|completo|
|scored_0884|E1|1|144|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0884|E1|3|144|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0884|E1|6|144|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0884|E2|52-1|144||139|válido|
|scored_0884|E2|52-3|144||139|válido|
|scored_0884|E2|52-6|144||138|válido|
|scored_0884|E2|64-1|144||138|válido|
|scored_0884|E2|64-3|144||138|válido|
|scored_0884|E2|64-6|144||137|válido|
|scored_0884|E3|joint|144|individual 144|144|OPTIMAL|
|scored_0884|E4|repair|144||143|completo|
|scored_0885|E1|1|225|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0885|E1|3|218|postprocess 218, optimize_a4 218, nested_windows 218|218|válido|
|scored_0885|E1|6|211|postprocess 211, optimize_a4 211, nested_windows 211|211|válido|
|scored_0885|E2|52-1|225||207|válido|
|scored_0885|E2|52-3|218||209|válido|
|scored_0885|E2|52-6|211||202|válido|
|scored_0885|E2|64-1|225||209|válido|
|scored_0885|E2|64-3|218||206|válido|
|scored_0885|E2|64-6|211||204|válido|
|scored_0885|E3|joint|211|individual 211|211|OPTIMAL|
|scored_0885|E4|repair|211||210|completo|
|scored_0886|E1|1|214|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0886|E1|3|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0886|E1|6|203|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0886|E2|52-1|214||199|válido|
|scored_0886|E2|52-3|207||194|válido|
|scored_0886|E2|52-6|203||195|válido|
|scored_0886|E2|64-1|214||201|válido|
|scored_0886|E2|64-3|207||191|válido|
|scored_0886|E2|64-6|203||191|válido|
|scored_0886|E3|joint|203|individual 203|203|OPTIMAL|
|scored_0886|E4|repair|203||203|completo|
|scored_0887|E1|1|4|postprocess 4, optimize_a4 4, nested_windows 4|4|válido|
|scored_0887|E1|3|4|postprocess 4, optimize_a4 4, nested_windows 4|4|válido|
|scored_0887|E1|6|4|postprocess 4, optimize_a4 4, nested_windows 4|4|válido|
|scored_0887|E2|52-1|4||4|válido|
|scored_0887|E2|52-3|4||4|válido|
|scored_0887|E2|52-6|4||4|válido|
|scored_0887|E2|64-1|4||4|válido|
|scored_0887|E2|64-3|4||4|válido|
|scored_0887|E2|64-6|4||4|válido|
|scored_0887|E3|joint|4|individual 4|4|OPTIMAL|
|scored_0887|E4|repair|4||4|completo|
|scored_0888|E1|1|168|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0888|E1|3|164|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0888|E1|6|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0888|E2|52-1|168||161|válido|
|scored_0888|E2|52-3|164||161|válido|
|scored_0888|E2|52-6|162||158|válido|
|scored_0888|E2|64-1|168||161|válido|
|scored_0888|E2|64-3|164||161|válido|
|scored_0888|E2|64-6|162||156|válido|
|scored_0888|E3|joint|162|individual 162|162|OPTIMAL|
|scored_0888|E4|repair|162||159|completo|
|scored_0889|E1|1|142|postprocess 142, optimize_a4 142, nested_windows 142|142|válido|
|scored_0889|E1|3|141|postprocess 141, optimize_a4 141, nested_windows 141|141|válido|
|scored_0889|E1|6|141|postprocess 141, optimize_a4 141, nested_windows 141|141|válido|
|scored_0889|E2|52-1|142||137|válido|
|scored_0889|E2|52-3|141||137|válido|
|scored_0889|E2|52-6|141||137|válido|
|scored_0889|E2|64-1|142||136|válido|
|scored_0889|E2|64-3|141||136|válido|
|scored_0889|E2|64-6|141||136|válido|
|scored_0889|E3|joint|141|individual 141|141|OPTIMAL|
|scored_0889|E4|repair|141||139|completo|
|scored_0890|E1|1|120|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|scored_0890|E1|3|120|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|scored_0890|E1|6|120|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|scored_0890|E2|52-1|120||117|válido|
|scored_0890|E2|52-3|120||117|válido|
|scored_0890|E2|52-6|120||117|válido|
|scored_0890|E2|64-1|120||116|válido|
|scored_0890|E2|64-3|120||116|válido|
|scored_0890|E2|64-6|120||116|válido|
|scored_0890|E3|joint|120|individual 120|120|OPTIMAL|
|scored_0890|E4|repair|120||115|completo|
|scored_0891|E1|1|139|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|scored_0891|E1|3|139|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|scored_0891|E1|6|139|postprocess 139, optimize_a4 139, nested_windows 139|139|válido|
|scored_0891|E2|52-1|139||138|válido|
|scored_0891|E2|52-3|139||137|válido|
|scored_0891|E2|52-6|139||137|válido|
|scored_0891|E2|64-1|139||136|válido|
|scored_0891|E2|64-3|139||136|válido|
|scored_0891|E2|64-6|139||136|válido|
|scored_0891|E3|joint|139|individual 139|139|OPTIMAL|
|scored_0891|E4|repair|139||135|completo|
|scored_0892|E1|1|29|postprocess 27, optimize_a4 27, nested_windows 27|27|válido|
|scored_0892|E1|3|29|postprocess 27, optimize_a4 27, nested_windows 27|27|válido|
|scored_0892|E1|6|29|postprocess 27, optimize_a4 27, nested_windows 27|27|válido|
|scored_0892|E2|52-1|29||29|válido|
|scored_0892|E2|52-3|29||29|válido|
|scored_0892|E2|52-6|29||29|válido|
|scored_0892|E2|64-1|29||29|válido|
|scored_0892|E2|64-3|29||29|válido|
|scored_0892|E2|64-6|29||29|válido|
|scored_0892|E3|joint|22|individual 22|22|OPTIMAL|
|scored_0892|E4|repair|22||21|completo|
|scored_0893|E1|1|177|postprocess 176, optimize_a4 176, nested_windows 176|176|válido|
|scored_0893|E1|3|174|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|scored_0893|E1|6|169|postprocess 169, optimize_a4 169, nested_windows 169|169|válido|
|scored_0893|E2|52-1|177||163|válido|
|scored_0893|E2|52-3|174||163|válido|
|scored_0893|E2|52-6|169||163|válido|
|scored_0893|E2|64-1|177||166|válido|
|scored_0893|E2|64-3|174||163|válido|
|scored_0893|E2|64-6|169||161|válido|
|scored_0893|E3|joint|169|individual 169|169|OPTIMAL|
|scored_0893|E4|repair|169||167|completo|
|scored_0894|E1|1|173|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0894|E1|3|168|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0894|E1|6|166|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0894|E2|52-1|173||158|válido|
|scored_0894|E2|52-3|168||156|válido|
|scored_0894|E2|52-6|166||154|válido|
|scored_0894|E2|64-1|173||160|válido|
|scored_0894|E2|64-3|168||156|válido|
|scored_0894|E2|64-6|166||153|válido|
|scored_0894|E3|joint|166|individual 165|165|OPTIMAL|
|scored_0894|E4|repair|166||164|completo|
|scored_0895|E1|1|162|postprocess 161, optimize_a4 161, nested_windows 161|161|válido|
|scored_0895|E1|3|160|postprocess 159, optimize_a4 159, nested_windows 159|159|válido|
|scored_0895|E1|6|160|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0895|E2|52-1|162||152|válido|
|scored_0895|E2|52-3|160||149|válido|
|scored_0895|E2|52-6|160||147|válido|
|scored_0895|E2|64-1|162||150|válido|
|scored_0895|E2|64-3|160||147|válido|
|scored_0895|E2|64-6|160||147|válido|
|scored_0895|E3|joint|160|individual 160|160|OPTIMAL|
|scored_0895|E4|repair|160||160|completo|
|scored_0896|E1|1|251|postprocess 251, optimize_a4 251, nested_windows 251|251|válido|
|scored_0896|E1|3|234|postprocess 234, optimize_a4 234, nested_windows 234|234|válido|
|scored_0896|E1|6|233|postprocess 233, optimize_a4 233, nested_windows 233|233|válido|
|scored_0896|E2|52-1|251||223|válido|
|scored_0896|E2|52-3|234||224|válido|
|scored_0896|E2|52-6|233||216|válido|
|scored_0896|E2|64-1|251||229|válido|
|scored_0896|E2|64-3|234||225|válido|
|scored_0896|E2|64-6|233||219|válido|
|scored_0896|E3|joint|233|individual 232|230|OPTIMAL|
|scored_0896|E4|repair|233||232|completo|
|scored_0897|E1|1|51|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0897|E1|3|51|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0897|E1|6|51|postprocess 48, optimize_a4 48, nested_windows 48|48|válido|
|scored_0897|E2|52-1|51||51|válido|
|scored_0897|E2|52-3|51||51|válido|
|scored_0897|E2|52-6|51||51|válido|
|scored_0897|E2|64-1|51||51|válido|
|scored_0897|E2|64-3|51||51|válido|
|scored_0897|E2|64-6|51||51|válido|
|scored_0897|E3|joint|51|individual 51|51|OPTIMAL|
|scored_0897|E4|repair|51||51|completo|
|scored_0898|E1|1|108|postprocess 108, optimize_a4 108, nested_windows 108|108|válido|
|scored_0898|E1|3|109|postprocess 109, optimize_a4 109, nested_windows 109|109|válido|
|scored_0898|E1|6|109|postprocess 109, optimize_a4 109, nested_windows 109|109|válido|
|scored_0898|E2|52-1|108||103|válido|
|scored_0898|E2|52-3|109||103|válido|
|scored_0898|E2|52-6|109||103|válido|
|scored_0898|E2|64-1|108||103|válido|
|scored_0898|E2|64-3|109||103|válido|
|scored_0898|E2|64-6|109||103|válido|
|scored_0898|E3|joint|108|individual 108|108|OPTIMAL|
|scored_0898|E4|repair|108||106|completo|
|scored_0899|E1|1|336|postprocess 336, optimize_a4 336, nested_windows 336|336|válido|
|scored_0899|E1|3|311|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|scored_0899|E1|6|309|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0899|E2|52-1|336||307|válido|
|scored_0899|E2|52-3|311||292|válido|
|scored_0899|E2|52-6|309||289|válido|
|scored_0899|E2|64-1|336||309|válido|
|scored_0899|E2|64-3|311||291|válido|
|scored_0899|E2|64-6|309||290|válido|
|scored_0899|E3|joint|309|individual 309|304|OPTIMAL|
|scored_0899|E4|repair|309||309|completo|
|scored_0900|E1|1|235|postprocess 235, optimize_a4 235, nested_windows 235|235|válido|
|scored_0900|E1|3|226|postprocess 225, optimize_a4 225, nested_windows 225|225|válido|
|scored_0900|E1|6|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0900|E2|52-1|235||212|válido|
|scored_0900|E2|52-3|226||209|válido|
|scored_0900|E2|52-6|219||211|válido|
|scored_0900|E2|64-1|235||211|válido|
|scored_0900|E2|64-3|226||208|válido|
|scored_0900|E2|64-6|219||211|válido|
|scored_0900|E3|joint|219|individual 219|219|FEASIBLE|
|scored_0900|E4|repair|219||219|completo|
|scored_0901|E1|1|27|postprocess 24, optimize_a4 24, nested_windows 24|24|válido|
|scored_0901|E1|3|27|postprocess 24, optimize_a4 24, nested_windows 24|24|válido|
|scored_0901|E1|6|27|postprocess 24, optimize_a4 24, nested_windows 24|24|válido|
|scored_0901|E2|52-1|27||27|válido|
|scored_0901|E2|52-3|27||27|válido|
|scored_0901|E2|52-6|27||27|válido|
|scored_0901|E2|64-1|27||27|válido|
|scored_0901|E2|64-3|27||27|válido|
|scored_0901|E2|64-6|27||27|válido|
|scored_0901|E3|joint|24|individual 24|24|OPTIMAL|
|scored_0901|E4|repair|24||21|completo|
|scored_0902|E1|1|302|postprocess 302, optimize_a4 302, nested_windows 302|302|válido|
|scored_0902|E1|3|289|postprocess 287, optimize_a4 287, nested_windows 287|287|válido|
|scored_0902|E1|6|283|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|scored_0902|E2|52-1|302||278|válido|
|scored_0902|E2|52-3|289||265|válido|
|scored_0902|E2|52-6|283||262|válido|
|scored_0902|E2|64-1|302||278|válido|
|scored_0902|E2|64-3|289||265|válido|
|scored_0902|E2|64-6|283||262|válido|
|scored_0902|E3|joint|283|individual 283|283|OPTIMAL|
|scored_0902|E4|repair|283||283|completo|
|scored_0903|E1|1|216|postprocess 214, optimize_a4 214, nested_windows 214|214|válido|
|scored_0903|E1|3|209|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0903|E1|6|201|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0903|E2|52-1|216||193|válido|
|scored_0903|E2|52-3|209||190|válido|
|scored_0903|E2|52-6|201||192|válido|
|scored_0903|E2|64-1|216||191|válido|
|scored_0903|E2|64-3|209||192|válido|
|scored_0903|E2|64-6|201||193|válido|
|scored_0903|E3|joint|201|individual 201|200|OPTIMAL|
|scored_0903|E4|repair|201||201|completo|
|scored_0904|E1|1|191|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|scored_0904|E1|3|185|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|scored_0904|E1|6|178|postprocess 178, optimize_a4 178, nested_windows 178|178|válido|
|scored_0904|E2|52-1|191||175|válido|
|scored_0904|E2|52-3|185||172|válido|
|scored_0904|E2|52-6|178||169|válido|
|scored_0904|E2|64-1|191||177|válido|
|scored_0904|E2|64-3|185||174|válido|
|scored_0904|E2|64-6|178||173|válido|
|scored_0904|E3|joint|178|individual 178|178|OPTIMAL|
|scored_0904|E4|repair|178||178|completo|
|scored_0905|E1|1|341|postprocess 340, optimize_a4 340, nested_windows 340|340|válido|
|scored_0905|E1|3|319|postprocess 319, optimize_a4 319, nested_windows 319|319|válido|
|scored_0905|E1|6|317|postprocess 317, optimize_a4 317, nested_windows 317|317|válido|
|scored_0905|E2|52-1|341||307|válido|
|scored_0905|E2|52-3|319||299|válido|
|scored_0905|E2|52-6|317||296|válido|
|scored_0905|E2|64-1|341||310|válido|
|scored_0905|E2|64-3|319||300|válido|
|scored_0905|E2|64-6|317||296|válido|
|scored_0905|E3|joint|317|individual 317|314|FEASIBLE|
|scored_0905|E4|repair|317||317|completo|
|scored_0906|E1|1|279|postprocess 279, optimize_a4 279, nested_windows 279|279|válido|
|scored_0906|E1|3|263|postprocess 263, optimize_a4 263, nested_windows 263|263|válido|
|scored_0906|E1|6|263|postprocess 263, optimize_a4 263, nested_windows 263|263|válido|
|scored_0906|E2|52-1|279||263|válido|
|scored_0906|E2|52-3|263||260|válido|
|scored_0906|E2|52-6|263||258|válido|
|scored_0906|E2|64-1|279||264|válido|
|scored_0906|E2|64-3|263||258|válido|
|scored_0906|E2|64-6|263||258|válido|
|scored_0906|E3|joint|263|individual 263|261|OPTIMAL|
|scored_0906|E4|repair|263||263|completo|
|scored_0907|E1|1|155|postprocess 155, optimize_a4 155, nested_windows 155|155|válido|
|scored_0907|E1|3|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0907|E1|6|152|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0907|E2|52-1|155||149|válido|
|scored_0907|E2|52-3|152||146|válido|
|scored_0907|E2|52-6|152||145|válido|
|scored_0907|E2|64-1|155||150|válido|
|scored_0907|E2|64-3|152||149|válido|
|scored_0907|E2|64-6|152||147|válido|
|scored_0907|E3|joint|152|individual 152|152|OPTIMAL|
|scored_0907|E4|repair|152||151|completo|
|scored_0908|E1|1|44|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0908|E1|3|44|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0908|E1|6|44|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0908|E2|52-1|44||44|válido|
|scored_0908|E2|52-3|44||44|válido|
|scored_0908|E2|52-6|44||44|válido|
|scored_0908|E2|64-1|44||44|válido|
|scored_0908|E2|64-3|44||44|válido|
|scored_0908|E2|64-6|44||44|válido|
|scored_0908|E3|joint|44|individual 44|44|OPTIMAL|
|scored_0908|E4|repair|44||44|completo|
|scored_0909|E1|1|222|postprocess 222, optimize_a4 222, nested_windows 222|222|válido|
|scored_0909|E1|3|216|postprocess 216, optimize_a4 216, nested_windows 216|216|válido|
|scored_0909|E1|6|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0909|E2|52-1|222||206|válido|
|scored_0909|E2|52-3|216||200|válido|
|scored_0909|E2|52-6|219||201|válido|
|scored_0909|E2|64-1|222||206|válido|
|scored_0909|E2|64-3|216||199|válido|
|scored_0909|E2|64-6|219||201|válido|
|scored_0909|E3|joint|216|individual 216|216|OPTIMAL|
|scored_0909|E4|repair|216||216|completo|
|scored_0910|E1|1|46|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0910|E1|3|46|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0910|E1|6|46|postprocess 45, optimize_a4 45, nested_windows 45|45|válido|
|scored_0910|E2|52-1|46||46|válido|
|scored_0910|E2|52-3|46||46|válido|
|scored_0910|E2|52-6|46||46|válido|
|scored_0910|E2|64-1|46||46|válido|
|scored_0910|E2|64-3|46||46|válido|
|scored_0910|E2|64-6|46||46|válido|
|scored_0910|E3|joint|46|individual 46|46|OPTIMAL|
|scored_0910|E4|repair|46||46|completo|
|scored_0911|E1|1|112|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0911|E1|3|112|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0911|E1|6|112|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0911|E2|52-1|112||107|válido|
|scored_0911|E2|52-3|112||107|válido|
|scored_0911|E2|52-6|112||107|válido|
|scored_0911|E2|64-1|112||107|válido|
|scored_0911|E2|64-3|112||107|válido|
|scored_0911|E2|64-6|112||107|válido|
|scored_0911|E3|joint|112|individual 112|112|OPTIMAL|
|scored_0911|E4|repair|112||111|completo|
|scored_0912|E1|1|291|postprocess 291, optimize_a4 291, nested_windows 291|291|válido|
|scored_0912|E1|3|281|postprocess 281, optimize_a4 281, nested_windows 281|281|válido|
|scored_0912|E1|6|275|postprocess 275, optimize_a4 275, nested_windows 275|275|válido|
|scored_0912|E2|52-1|291||262|válido|
|scored_0912|E2|52-3|281||265|válido|
|scored_0912|E2|52-6|275||260|válido|
|scored_0912|E2|64-1|291||261|válido|
|scored_0912|E2|64-3|281||267|válido|
|scored_0912|E2|64-6|275||262|válido|
|scored_0912|E3|joint|275|individual 275|275|OPTIMAL|
|scored_0912|E4|repair|275||275|completo|
|scored_0913|E1|1|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0913|E1|3|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0913|E1|6|41|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0913|E2|52-1|41||41|válido|
|scored_0913|E2|52-3|41||41|válido|
|scored_0913|E2|52-6|41||41|válido|
|scored_0913|E2|64-1|41||41|válido|
|scored_0913|E2|64-3|41||41|válido|
|scored_0913|E2|64-6|41||41|válido|
|scored_0913|E3|joint|34|individual 34|34|OPTIMAL|
|scored_0913|E4|repair|34||34|completo|
|scored_0914|E1|1|54|postprocess 54, optimize_a4 54, nested_windows 54|54|válido|
|scored_0914|E1|3|54|postprocess 54, optimize_a4 54, nested_windows 54|54|válido|
|scored_0914|E1|6|54|postprocess 54, optimize_a4 54, nested_windows 54|54|válido|
|scored_0914|E2|52-1|54||54|válido|
|scored_0914|E2|52-3|54||54|válido|
|scored_0914|E2|52-6|54||54|válido|
|scored_0914|E2|64-1|54||54|válido|
|scored_0914|E2|64-3|54||54|válido|
|scored_0914|E2|64-6|54||54|válido|
|scored_0914|E3|joint|54|individual 54|54|OPTIMAL|
|scored_0914|E4|repair|54||50|completo|
|scored_0915|E1|1|266|postprocess 266, optimize_a4 266, nested_windows 266|266|válido|
|scored_0915|E1|3|258|postprocess 258, optimize_a4 258, nested_windows 258|258|válido|
|scored_0915|E1|6|255|postprocess 255, optimize_a4 255, nested_windows 255|255|válido|
|scored_0915|E2|52-1|266||247|válido|
|scored_0915|E2|52-3|258||239|válido|
|scored_0915|E2|52-6|255||235|válido|
|scored_0915|E2|64-1|266||246|válido|
|scored_0915|E2|64-3|258||241|válido|
|scored_0915|E2|64-6|255||238|válido|
|scored_0915|E3|joint|255|individual 253|253|OPTIMAL|
|scored_0915|E4|repair|255||255|completo|
|scored_0916|E1|1|324|postprocess 323, optimize_a4 323, nested_windows 323|323|válido|
|scored_0916|E1|3|308|postprocess 307, optimize_a4 307, nested_windows 307|307|válido|
|scored_0916|E1|6|302|postprocess 301, optimize_a4 301, nested_windows 301|301|válido|
|scored_0916|E2|52-1|324||299|válido|
|scored_0916|E2|52-3|308||290|válido|
|scored_0916|E2|52-6|302||286|válido|
|scored_0916|E2|64-1|324||299|válido|
|scored_0916|E2|64-3|308||291|válido|
|scored_0916|E2|64-6|302||284|válido|
|scored_0916|E3|joint|302|individual 301|299|FEASIBLE|
|scored_0916|E4|repair|302||302|completo|
|scored_0917|E1|1|306|postprocess 306, optimize_a4 306, nested_windows 306|306|válido|
|scored_0917|E1|3|295|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|scored_0917|E1|6|287|postprocess 287, optimize_a4 287, nested_windows 287|287|válido|
|scored_0917|E2|52-1|306||275|válido|
|scored_0917|E2|52-3|295||268|válido|
|scored_0917|E2|52-6|287||265|válido|
|scored_0917|E2|64-1|306||275|válido|
|scored_0917|E2|64-3|295||267|válido|
|scored_0917|E2|64-6|287||264|válido|
|scored_0917|E3|joint|287|individual 286|286|OPTIMAL|
|scored_0917|E4|repair|287||287|completo|
|scored_0918|E1|1|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0918|E1|3|292|postprocess 292, optimize_a4 292, nested_windows 292|292|válido|
|scored_0918|E1|6|284|postprocess 284, optimize_a4 284, nested_windows 284|284|válido|
|scored_0918|E2|52-1|303||281|válido|
|scored_0918|E2|52-3|292||273|válido|
|scored_0918|E2|52-6|284||266|válido|
|scored_0918|E2|64-1|303||281|válido|
|scored_0918|E2|64-3|292||269|válido|
|scored_0918|E2|64-6|284||265|válido|
|scored_0918|E3|joint|284|individual 284|284|FEASIBLE|
|scored_0918|E4|repair|284||284|completo|
|scored_0919|E1|1|205|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|scored_0919|E1|3|203|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0919|E1|6|199|postprocess 199, optimize_a4 199, nested_windows 199|199|válido|
|scored_0919|E2|52-1|205||198|válido|
|scored_0919|E2|52-3|203||194|válido|
|scored_0919|E2|52-6|199||193|válido|
|scored_0919|E2|64-1|205||189|válido|
|scored_0919|E2|64-3|203||188|válido|
|scored_0919|E2|64-6|199||187|válido|
|scored_0919|E3|joint|199|individual 199|198|OPTIMAL|
|scored_0919|E4|repair|199||198|completo|
|scored_0920|E1|1|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0920|E1|3|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0920|E1|6|187|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0920|E2|52-1|192||182|válido|
|scored_0920|E2|52-3|188||182|válido|
|scored_0920|E2|52-6|187||182|válido|
|scored_0920|E2|64-1|192||182|válido|
|scored_0920|E2|64-3|188||182|válido|
|scored_0920|E2|64-6|187||182|válido|
|scored_0920|E3|joint|187|individual 187|186|OPTIMAL|
|scored_0920|E4|repair|187||187|completo|
|scored_0921|E1|1|231|postprocess 230, optimize_a4 230, nested_windows 230|230|válido|
|scored_0921|E1|3|219|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0921|E1|6|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0921|E2|52-1|231||219|válido|
|scored_0921|E2|52-3|219||207|válido|
|scored_0921|E2|52-6|215||209|válido|
|scored_0921|E2|64-1|231||216|válido|
|scored_0921|E2|64-3|219||207|válido|
|scored_0921|E2|64-6|215||209|válido|
|scored_0921|E3|joint|215|individual 215|215|OPTIMAL|
|scored_0921|E4|repair|215||215|completo|
|scored_0922|E1|1|35|postprocess 34, optimize_a4 34, nested_windows 34|34|válido|
|scored_0922|E1|3|35|postprocess 34, optimize_a4 34, nested_windows 34|34|válido|
|scored_0922|E1|6|35|postprocess 34, optimize_a4 34, nested_windows 34|34|válido|
|scored_0922|E2|52-1|35||35|válido|
|scored_0922|E2|52-3|35||35|válido|
|scored_0922|E2|52-6|35||35|válido|
|scored_0922|E2|64-1|35||35|válido|
|scored_0922|E2|64-3|35||35|válido|
|scored_0922|E2|64-6|35||35|válido|
|scored_0922|E3|joint|35|individual 35|35|OPTIMAL|
|scored_0922|E4|repair|35||35|completo|
|scored_0923|E1|1|281|postprocess 280, optimize_a4 280, nested_windows 280|280|válido|
|scored_0923|E1|3|267|postprocess 267, optimize_a4 267, nested_windows 267|267|válido|
|scored_0923|E1|6|262|postprocess 262, optimize_a4 262, nested_windows 262|262|válido|
|scored_0923|E2|52-1|281||260|válido|
|scored_0923|E2|52-3|267||248|válido|
|scored_0923|E2|52-6|262||247|válido|
|scored_0923|E2|64-1|281||264|válido|
|scored_0923|E2|64-3|267||249|válido|
|scored_0923|E2|64-6|262||248|válido|
|scored_0923|E3|joint|262|individual 262|262|OPTIMAL|
|scored_0923|E4|repair|262||262|completo|
|scored_0924|E1|1|147|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0924|E1|3|147|postprocess 147, optimize_a4 147, nested_windows 147|147|válido|
|scored_0924|E1|6|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0924|E2|52-1|147||142|válido|
|scored_0924|E2|52-3|147||142|válido|
|scored_0924|E2|52-6|146||142|válido|
|scored_0924|E2|64-1|147||144|válido|
|scored_0924|E2|64-3|147||143|válido|
|scored_0924|E2|64-6|146||143|válido|
|scored_0924|E3|joint|146|individual 146|146|OPTIMAL|
|scored_0924|E4|repair|146||145|completo|
|scored_0925|E1|1|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0925|E1|3|207|postprocess 207, optimize_a4 207, nested_windows 207|207|válido|
|scored_0925|E1|6|204|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|scored_0925|E2|52-1|220||203|válido|
|scored_0925|E2|52-3|207||198|válido|
|scored_0925|E2|52-6|204||196|válido|
|scored_0925|E2|64-1|220||203|válido|
|scored_0925|E2|64-3|207||198|válido|
|scored_0925|E2|64-6|204||196|válido|
|scored_0925|E3|joint|204|individual 202|201|OPTIMAL|
|scored_0925|E4|repair|204||202|completo|
|scored_0926|E1|1|33|postprocess 32, optimize_a4 32, nested_windows 32|32|válido|
|scored_0926|E1|3|33|postprocess 32, optimize_a4 32, nested_windows 32|32|válido|
|scored_0926|E1|6|33|postprocess 32, optimize_a4 32, nested_windows 32|32|válido|
|scored_0926|E2|52-1|33||33|válido|
|scored_0926|E2|52-3|33||33|válido|
|scored_0926|E2|52-6|33||33|válido|
|scored_0926|E2|64-1|33||33|válido|
|scored_0926|E2|64-3|33||33|válido|
|scored_0926|E2|64-6|33||33|válido|
|scored_0926|E3|joint|33|individual 33|33|OPTIMAL|
|scored_0926|E4|repair|33||32|completo|
|scored_0927|E1|1|185|postprocess 184, optimize_a4 184, nested_windows 184|184|válido|
|scored_0927|E1|3|183|postprocess 182, optimize_a4 182, nested_windows 182|182|válido|
|scored_0927|E1|6|181|postprocess 180, optimize_a4 180, nested_windows 180|180|válido|
|scored_0927|E2|52-1|185||174|válido|
|scored_0927|E2|52-3|183||171|válido|
|scored_0927|E2|52-6|181||171|válido|
|scored_0927|E2|64-1|185||173|válido|
|scored_0927|E2|64-3|183||173|válido|
|scored_0927|E2|64-6|181||171|válido|
|scored_0927|E3|joint|181|individual 181|181|OPTIMAL|
|scored_0927|E4|repair|181||181|completo|
|scored_0928|E1|1|46|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0928|E1|3|46|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0928|E1|6|46|postprocess 43, optimize_a4 43, nested_windows 43|43|válido|
|scored_0928|E2|52-1|46||46|válido|
|scored_0928|E2|52-3|46||46|válido|
|scored_0928|E2|52-6|46||46|válido|
|scored_0928|E2|64-1|46||46|válido|
|scored_0928|E2|64-3|46||46|válido|
|scored_0928|E2|64-6|46||46|válido|
|scored_0928|E3|joint|46|individual 46|46|OPTIMAL|
|scored_0928|E4|repair|46||42|completo|
|scored_0929|E1|1|198|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|scored_0929|E1|3|193|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0929|E1|6|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0929|E2|52-1|198||184|válido|
|scored_0929|E2|52-3|193||172|válido|
|scored_0929|E2|52-6|188||176|válido|
|scored_0929|E2|64-1|198||182|válido|
|scored_0929|E2|64-3|193||178|válido|
|scored_0929|E2|64-6|188||175|válido|
|scored_0929|E3|joint|188|individual 188|185|OPTIMAL|
|scored_0929|E4|repair|188||187|completo|
|scored_0930|E1|1|112|postprocess 111, optimize_a4 109, nested_windows 109|109|válido|
|scored_0930|E1|3|112|postprocess 111, optimize_a4 109, nested_windows 109|109|válido|
|scored_0930|E1|6|112|postprocess 111, optimize_a4 109, nested_windows 109|109|válido|
|scored_0930|E2|52-1|112||109|válido|
|scored_0930|E2|52-3|112||108|válido|
|scored_0930|E2|52-6|112||108|válido|
|scored_0930|E2|64-1|112||110|válido|
|scored_0930|E2|64-3|112||110|válido|
|scored_0930|E2|64-6|112||110|válido|
|scored_0930|E3|joint|112|individual 112|112|OPTIMAL|
|scored_0930|E4|repair|112||106|completo|
|scored_0931|E1|1|4|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|scored_0931|E1|3|4|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|scored_0931|E1|6|4|postprocess 2, optimize_a4 2, nested_windows 2|2|válido|
|scored_0931|E2|52-1|4||4|válido|
|scored_0931|E2|52-3|4||4|válido|
|scored_0931|E2|52-6|4||4|válido|
|scored_0931|E2|64-1|4||4|válido|
|scored_0931|E2|64-3|4||4|válido|
|scored_0931|E2|64-6|4||4|válido|
|scored_0931|E3|joint|2|individual 2|2|OPTIMAL|
|scored_0931|E4|repair|2||2|completo|
|scored_0932|E1|1|335|postprocess 334, optimize_a4 334, nested_windows 334|334|válido|
|scored_0932|E1|3|320|postprocess 320, optimize_a4 320, nested_windows 320|320|válido|
|scored_0932|E1|6|308|postprocess 308, optimize_a4 308, nested_windows 308|308|válido|
|scored_0932|E2|52-1|335||307|válido|
|scored_0932|E2|52-3|320||283|válido|
|scored_0932|E2|52-6|308||288|válido|
|scored_0932|E2|64-1|335||312|válido|
|scored_0932|E2|64-3|320||284|válido|
|scored_0932|E2|64-6|308||287|válido|
|scored_0932|E3|joint|308|individual 308|308|OPTIMAL|
|scored_0932|E4|repair|308||308|completo|
|scored_0933|E1|1|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0933|E1|3|217|postprocess 217, optimize_a4 217, nested_windows 217|217|válido|
|scored_0933|E1|6|215|postprocess 215, optimize_a4 215, nested_windows 215|215|válido|
|scored_0933|E2|52-1|229||205|válido|
|scored_0933|E2|52-3|217||203|válido|
|scored_0933|E2|52-6|215||197|válido|
|scored_0933|E2|64-1|229||206|válido|
|scored_0933|E2|64-3|217||206|válido|
|scored_0933|E2|64-6|215||200|válido|
|scored_0933|E3|joint|215|individual 215|214|OPTIMAL|
|scored_0933|E4|repair|215||215|completo|
|scored_0934|E1|1|193|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0934|E1|3|191|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|scored_0934|E1|6|190|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|scored_0934|E2|52-1|193||181|válido|
|scored_0934|E2|52-3|191||183|válido|
|scored_0934|E2|52-6|190||179|válido|
|scored_0934|E2|64-1|193||182|válido|
|scored_0934|E2|64-3|191||183|válido|
|scored_0934|E2|64-6|190||177|válido|
|scored_0934|E3|joint|190|individual 189|189|OPTIMAL|
|scored_0934|E4|repair|190||189|completo|
|scored_0935|E1|1|237|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|scored_0935|E1|3|229|postprocess 228, optimize_a4 228, nested_windows 228|228|válido|
|scored_0935|E1|6|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0935|E2|52-1|237||221|válido|
|scored_0935|E2|52-3|229||213|válido|
|scored_0935|E2|52-6|229||213|válido|
|scored_0935|E2|64-1|237||222|válido|
|scored_0935|E2|64-3|229||212|válido|
|scored_0935|E2|64-6|229||214|válido|
|scored_0935|E3|joint|229|individual 228|226|OPTIMAL|
|scored_0935|E4|repair|229||229|completo|
|scored_0936|E1|1|209|postprocess 209, optimize_a4 209, nested_windows 209|209|válido|
|scored_0936|E1|3|202|postprocess 202, optimize_a4 202, nested_windows 202|202|válido|
|scored_0936|E1|6|201|postprocess 201, optimize_a4 201, nested_windows 201|201|válido|
|scored_0936|E2|52-1|209||190|válido|
|scored_0936|E2|52-3|202||189|válido|
|scored_0936|E2|52-6|201||192|válido|
|scored_0936|E2|64-1|209||192|válido|
|scored_0936|E2|64-3|202||187|válido|
|scored_0936|E2|64-6|201||192|válido|
|scored_0936|E3|joint|201|individual 201|201|OPTIMAL|
|scored_0936|E4|repair|201||201|completo|
|scored_0937|E1|1|41|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|scored_0937|E1|3|41|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|scored_0937|E1|6|41|postprocess 38, optimize_a4 38, nested_windows 38|38|válido|
|scored_0937|E2|52-1|41||41|válido|
|scored_0937|E2|52-3|41||41|válido|
|scored_0937|E2|52-6|41||41|válido|
|scored_0937|E2|64-1|41||41|válido|
|scored_0937|E2|64-3|41||41|válido|
|scored_0937|E2|64-6|41||41|válido|
|scored_0937|E3|joint|41|individual 41|41|OPTIMAL|
|scored_0937|E4|repair|41||41|completo|
|scored_0938|E1|1|180|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|scored_0938|E1|3|181|postprocess 180, optimize_a4 180, nested_windows 180|180|válido|
|scored_0938|E1|6|175|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|scored_0938|E2|52-1|180||175|válido|
|scored_0938|E2|52-3|181||170|válido|
|scored_0938|E2|52-6|175||172|válido|
|scored_0938|E2|64-1|180||167|válido|
|scored_0938|E2|64-3|181||165|válido|
|scored_0938|E2|64-6|175||165|válido|
|scored_0938|E3|joint|175|individual 173|172|OPTIMAL|
|scored_0938|E4|repair|175||174|completo|
|scored_0939|E1|1|369|postprocess 369, optimize_a4 369, nested_windows 369|369|válido|
|scored_0939|E1|3|331|postprocess 330, optimize_a4 330, nested_windows 330|330|válido|
|scored_0939|E1|6|322|postprocess 322, optimize_a4 322, nested_windows 322|322|válido|
|scored_0939|E2|52-1|369||321|válido|
|scored_0939|E2|52-3|331||309|válido|
|scored_0939|E2|52-6|322||289|válido|
|scored_0939|E2|64-1|369||324|válido|
|scored_0939|E2|64-3|331||310|válido|
|scored_0939|E2|64-6|322||295|válido|
|scored_0939|E3|joint|322|individual 320|320|rechazado por registros|
|scored_0939|E4|repair|322||322|completo|
|scored_0940|E1|1|178|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0940|E1|3|175|postprocess 174, optimize_a4 174, nested_windows 174|174|válido|
|scored_0940|E1|6|167|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0940|E2|52-1|178||164|válido|
|scored_0940|E2|52-3|175||159|válido|
|scored_0940|E2|52-6|167||157|válido|
|scored_0940|E2|64-1|178||164|válido|
|scored_0940|E2|64-3|175||160|válido|
|scored_0940|E2|64-6|167||158|válido|
|scored_0940|E3|joint|167|individual 167|167|OPTIMAL|
|scored_0940|E4|repair|167||167|completo|
|scored_0941|E1|1|334|postprocess 334, optimize_a4 334, nested_windows 334|334|válido|
|scored_0941|E1|3|323|postprocess 323, optimize_a4 323, nested_windows 323|323|válido|
|scored_0941|E1|6|315|postprocess 315, optimize_a4 315, nested_windows 315|315|válido|
|scored_0941|E2|52-1|334||307|válido|
|scored_0941|E2|52-3|323||299|válido|
|scored_0941|E2|52-6|315||290|válido|
|scored_0941|E2|64-1|334||307|válido|
|scored_0941|E2|64-3|323||302|válido|
|scored_0941|E2|64-6|315||293|válido|
|scored_0941|E3|joint|315|individual 315|314|FEASIBLE|
|scored_0941|E4|repair|315||315|completo|
|scored_0942|E1|1|200|postprocess 200, optimize_a4 200, nested_windows 200|200|válido|
|scored_0942|E1|3|193|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0942|E1|6|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0942|E2|52-1|200||190|válido|
|scored_0942|E2|52-3|193||189|válido|
|scored_0942|E2|52-6|188||176|válido|
|scored_0942|E2|64-1|200||193|válido|
|scored_0942|E2|64-3|193||189|válido|
|scored_0942|E2|64-6|188||179|válido|
|scored_0942|E3|joint|188|individual 188|188|OPTIMAL|
|scored_0942|E4|repair|188||184|completo|
|scored_0943|E1|1|118|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0943|E1|3|118|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0943|E1|6|118|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0943|E2|52-1|118||117|válido|
|scored_0943|E2|52-3|118||117|válido|
|scored_0943|E2|52-6|118||117|válido|
|scored_0943|E2|64-1|118||117|válido|
|scored_0943|E2|64-3|118||117|válido|
|scored_0943|E2|64-6|118||117|válido|
|scored_0943|E3|joint|118|individual 118|118|OPTIMAL|
|scored_0943|E4|repair|118||117|completo|
|scored_0944|E1|1|312|postprocess 312, optimize_a4 312, nested_windows 312|312|válido|
|scored_0944|E1|3|298|postprocess 298, optimize_a4 298, nested_windows 298|298|válido|
|scored_0944|E1|6|296|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0944|E2|52-1|312||288|válido|
|scored_0944|E2|52-3|298||280|válido|
|scored_0944|E2|52-6|296||280|válido|
|scored_0944|E2|64-1|312||291|válido|
|scored_0944|E2|64-3|298||281|válido|
|scored_0944|E2|64-6|296||281|válido|
|scored_0944|E3|joint|296|individual 295|294|OPTIMAL|
|scored_0944|E4|repair|296||296|completo|
|scored_0945|E1|1|43|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0945|E1|3|43|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0945|E1|6|43|postprocess 40, optimize_a4 40, nested_windows 40|40|válido|
|scored_0945|E2|52-1|43||43|válido|
|scored_0945|E2|52-3|43||43|válido|
|scored_0945|E2|52-6|43||43|válido|
|scored_0945|E2|64-1|43||43|válido|
|scored_0945|E2|64-3|43||43|válido|
|scored_0945|E2|64-6|43||43|válido|
|scored_0945|E3|joint|43|individual 43|43|OPTIMAL|
|scored_0945|E4|repair|43||37|completo|
|scored_0946|E1|1|303|postprocess 303, optimize_a4 303, nested_windows 303|303|válido|
|scored_0946|E1|3|286|postprocess 286, optimize_a4 286, nested_windows 286|286|válido|
|scored_0946|E1|6|278|postprocess 278, optimize_a4 278, nested_windows 278|278|válido|
|scored_0946|E2|52-1|303||273|válido|
|scored_0946|E2|52-3|286||267|válido|
|scored_0946|E2|52-6|278||267|válido|
|scored_0946|E2|64-1|303||276|válido|
|scored_0946|E2|64-3|286||271|válido|
|scored_0946|E2|64-6|278||269|válido|
|scored_0946|E3|joint|278|individual 278|277|OPTIMAL|
|scored_0946|E4|repair|278||278|completo|
|scored_0947|E1|1|122|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0947|E1|3|122|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0947|E1|6|122|postprocess 121, optimize_a4 121, nested_windows 121|121|válido|
|scored_0947|E2|52-1|122||118|válido|
|scored_0947|E2|52-3|122||118|válido|
|scored_0947|E2|52-6|122||118|válido|
|scored_0947|E2|64-1|122||120|válido|
|scored_0947|E2|64-3|122||120|válido|
|scored_0947|E2|64-6|122||120|válido|
|scored_0947|E3|joint|122|individual 122|122|OPTIMAL|
|scored_0947|E4|repair|122||121|completo|
|scored_0948|E1|1|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0948|E1|3|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0948|E1|6|168|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0948|E2|52-1|172||167|válido|
|scored_0948|E2|52-3|172||164|válido|
|scored_0948|E2|52-6|168||163|válido|
|scored_0948|E2|64-1|172||166|válido|
|scored_0948|E2|64-3|172||165|válido|
|scored_0948|E2|64-6|168||163|válido|
|scored_0948|E3|joint|168|individual 168|168|OPTIMAL|
|scored_0948|E4|repair|168||166|completo|
|scored_0949|E1|1|113|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0949|E1|3|113|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0949|E1|6|113|postprocess 111, optimize_a4 111, nested_windows 111|111|válido|
|scored_0949|E2|52-1|113||109|válido|
|scored_0949|E2|52-3|113||109|válido|
|scored_0949|E2|52-6|113||109|válido|
|scored_0949|E2|64-1|113||109|válido|
|scored_0949|E2|64-3|113||109|válido|
|scored_0949|E2|64-6|113||109|válido|
|scored_0949|E3|joint|113|individual 113|113|OPTIMAL|
|scored_0949|E4|repair|113||109|completo|
|scored_0950|E1|1|168|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0950|E1|3|167|postprocess 167, optimize_a4 167, nested_windows 167|167|válido|
|scored_0950|E1|6|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|scored_0950|E2|52-1|168||162|válido|
|scored_0950|E2|52-3|167||159|válido|
|scored_0950|E2|52-6|166||159|válido|
|scored_0950|E2|64-1|168||159|válido|
|scored_0950|E2|64-3|167||156|válido|
|scored_0950|E2|64-6|166||156|válido|
|scored_0950|E3|joint|166|individual 166|166|OPTIMAL|
|scored_0950|E4|repair|166||166|completo|
|scored_0951|E1|1|119|postprocess 119, optimize_a4 119, nested_windows 119|119|válido|
|scored_0951|E1|3|118|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0951|E1|6|118|postprocess 118, optimize_a4 118, nested_windows 118|118|válido|
|scored_0951|E2|52-1|119||115|válido|
|scored_0951|E2|52-3|118||115|válido|
|scored_0951|E2|52-6|118||114|válido|
|scored_0951|E2|64-1|119||117|válido|
|scored_0951|E2|64-3|118||117|válido|
|scored_0951|E2|64-6|118||115|válido|
|scored_0951|E3|joint|118|individual 118|118|OPTIMAL|
|scored_0951|E4|repair|118||118|completo|
|scored_0952|E1|1|298|postprocess 297, optimize_a4 297, nested_windows 297|297|válido|
|scored_0952|E1|3|286|postprocess 286, optimize_a4 286, nested_windows 286|286|válido|
|scored_0952|E1|6|283|postprocess 283, optimize_a4 283, nested_windows 283|283|válido|
|scored_0952|E2|52-1|298||279|válido|
|scored_0952|E2|52-3|286||267|válido|
|scored_0952|E2|52-6|283||270|válido|
|scored_0952|E2|64-1|298||279|válido|
|scored_0952|E2|64-3|286||268|válido|
|scored_0952|E2|64-6|283||271|válido|
|scored_0952|E3|joint|283|individual 283|283|OPTIMAL|
|scored_0952|E4|repair|283||283|completo|
|scored_0953|E1|1|243|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0953|E1|3|242|postprocess 242, optimize_a4 242, nested_windows 242|242|válido|
|scored_0953|E1|6|234|postprocess 234, optimize_a4 234, nested_windows 234|234|válido|
|scored_0953|E2|52-1|243||229|válido|
|scored_0953|E2|52-3|242||231|válido|
|scored_0953|E2|52-6|234||224|válido|
|scored_0953|E2|64-1|243||230|válido|
|scored_0953|E2|64-3|242||231|válido|
|scored_0953|E2|64-6|234||226|válido|
|scored_0953|E3|joint|234|individual 234|233|OPTIMAL|
|scored_0953|E4|repair|234||234|completo|
|scored_0954|E1|1|105|postprocess 105, optimize_a4 105, nested_windows 105|105|válido|
|scored_0954|E1|3|105|postprocess 105, optimize_a4 105, nested_windows 105|105|válido|
|scored_0954|E1|6|105|postprocess 105, optimize_a4 105, nested_windows 105|105|válido|
|scored_0954|E2|52-1|105||102|válido|
|scored_0954|E2|52-3|105||102|válido|
|scored_0954|E2|52-6|105||102|válido|
|scored_0954|E2|64-1|105||103|válido|
|scored_0954|E2|64-3|105||103|válido|
|scored_0954|E2|64-6|105||103|válido|
|scored_0954|E3|joint|105|individual 105|105|OPTIMAL|
|scored_0954|E4|repair|105||99|completo|
|scored_0955|E1|1|116|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0955|E1|3|116|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0955|E1|6|116|postprocess 116, optimize_a4 116, nested_windows 116|116|válido|
|scored_0955|E2|52-1|116||113|válido|
|scored_0955|E2|52-3|116||113|válido|
|scored_0955|E2|52-6|116||113|válido|
|scored_0955|E2|64-1|116||113|válido|
|scored_0955|E2|64-3|116||113|válido|
|scored_0955|E2|64-6|116||113|válido|
|scored_0955|E3|joint|116|individual 116|116|OPTIMAL|
|scored_0955|E4|repair|116||113|completo|
|scored_0956|E1|1|316|postprocess 316, optimize_a4 316, nested_windows 316|316|válido|
|scored_0956|E1|3|310|postprocess 310, optimize_a4 310, nested_windows 310|310|válido|
|scored_0956|E1|6|297|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0956|E2|52-1|316||290|válido|
|scored_0956|E2|52-3|310||279|válido|
|scored_0956|E2|52-6|297||287|válido|
|scored_0956|E2|64-1|316||292|válido|
|scored_0956|E2|64-3|310||281|válido|
|scored_0956|E2|64-6|297||288|válido|
|scored_0956|E3|joint|297|individual 296|294|FEASIBLE|
|scored_0956|E4|repair|297||297|completo|
|scored_0957|E1|1|42|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0957|E1|3|42|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0957|E1|6|42|postprocess 39, optimize_a4 39, nested_windows 39|39|válido|
|scored_0957|E2|52-1|42||42|válido|
|scored_0957|E2|52-3|42||42|válido|
|scored_0957|E2|52-6|42||42|válido|
|scored_0957|E2|64-1|42||42|válido|
|scored_0957|E2|64-3|42||42|válido|
|scored_0957|E2|64-6|42||42|válido|
|scored_0957|E3|joint|42|individual 42|42|OPTIMAL|
|scored_0957|E4|repair|42||42|completo|
|scored_0958|E1|1|41|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0958|E1|3|41|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0958|E1|6|41|postprocess 36, optimize_a4 36, nested_windows 36|36|válido|
|scored_0958|E2|52-1|41||41|válido|
|scored_0958|E2|52-3|41||41|válido|
|scored_0958|E2|52-6|41||41|válido|
|scored_0958|E2|64-1|41||41|válido|
|scored_0958|E2|64-3|41||41|válido|
|scored_0958|E2|64-6|41||41|válido|
|scored_0958|E3|joint|41|individual 41|41|OPTIMAL|
|scored_0958|E4|repair|41||41|completo|
|scored_0959|E1|1|330|postprocess 330, optimize_a4 330, nested_windows 330|330|válido|
|scored_0959|E1|3|313|postprocess 312, optimize_a4 312, nested_windows 312|312|válido|
|scored_0959|E1|6|313|postprocess 312, optimize_a4 312, nested_windows 312|312|válido|
|scored_0959|E2|52-1|330||299|válido|
|scored_0959|E2|52-3|313||297|válido|
|scored_0959|E2|52-6|313||289|válido|
|scored_0959|E2|64-1|330||300|válido|
|scored_0959|E2|64-3|313||294|válido|
|scored_0959|E2|64-6|313||290|válido|
|scored_0959|E3|joint|313|individual 312|312|OPTIMAL|
|scored_0959|E4|repair|313||313|completo|
|scored_0960|E1|1|163|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0960|E1|3|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0960|E1|6|162|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0960|E2|52-1|163||159|válido|
|scored_0960|E2|52-3|162||157|válido|
|scored_0960|E2|52-6|162||157|válido|
|scored_0960|E2|64-1|163||160|válido|
|scored_0960|E2|64-3|162||159|válido|
|scored_0960|E2|64-6|162||158|válido|
|scored_0960|E3|joint|162|individual 161|161|OPTIMAL|
|scored_0960|E4|repair|162||161|completo|
|scored_0961|E1|1|166|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0961|E1|3|164|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0961|E1|6|164|postprocess 162, optimize_a4 162, nested_windows 162|162|válido|
|scored_0961|E2|52-1|166||157|válido|
|scored_0961|E2|52-3|164||155|válido|
|scored_0961|E2|52-6|164||155|válido|
|scored_0961|E2|64-1|166||155|válido|
|scored_0961|E2|64-3|164||155|válido|
|scored_0961|E2|64-6|164||155|válido|
|scored_0961|E3|joint|164|individual 163|163|OPTIMAL|
|scored_0961|E4|repair|164||164|completo|
|scored_0962|E1|1|48|postprocess 47, optimize_a4 47, nested_windows 47|47|válido|
|scored_0962|E1|3|48|postprocess 47, optimize_a4 47, nested_windows 47|47|válido|
|scored_0962|E1|6|48|postprocess 47, optimize_a4 47, nested_windows 47|47|válido|
|scored_0962|E2|52-1|48||48|válido|
|scored_0962|E2|52-3|48||48|válido|
|scored_0962|E2|52-6|48||48|válido|
|scored_0962|E2|64-1|48||48|válido|
|scored_0962|E2|64-3|48||48|válido|
|scored_0962|E2|64-6|48||48|válido|
|scored_0962|E3|joint|48|individual 48|48|OPTIMAL|
|scored_0962|E4|repair|48||47|completo|
|scored_0963|E1|1|165|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0963|E1|3|166|postprocess 166, optimize_a4 166, nested_windows 166|166|válido|
|scored_0963|E1|6|165|postprocess 164, optimize_a4 164, nested_windows 164|164|válido|
|scored_0963|E2|52-1|165||158|válido|
|scored_0963|E2|52-3|166||158|válido|
|scored_0963|E2|52-6|165||155|válido|
|scored_0963|E2|64-1|165||160|válido|
|scored_0963|E2|64-3|166||156|válido|
|scored_0963|E2|64-6|165||156|válido|
|scored_0963|E3|joint|165|individual 164|164|OPTIMAL|
|scored_0963|E4|repair|165||164|completo|
|scored_0964|E1|1|296|postprocess 296, optimize_a4 296, nested_windows 296|296|válido|
|scored_0964|E1|3|288|postprocess 287, optimize_a4 287, nested_windows 287|287|válido|
|scored_0964|E1|6|281|postprocess 281, optimize_a4 281, nested_windows 281|281|válido|
|scored_0964|E2|52-1|296||274|válido|
|scored_0964|E2|52-3|288||258|válido|
|scored_0964|E2|52-6|281||258|válido|
|scored_0964|E2|64-1|296||274|válido|
|scored_0964|E2|64-3|288||262|válido|
|scored_0964|E2|64-6|281||262|válido|
|scored_0964|E3|joint|281|individual 281|280|FEASIBLE|
|scored_0964|E4|repair|281||281|completo|
|scored_0965|E1|1|295|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|scored_0965|E1|3|275|postprocess 274, optimize_a4 274, nested_windows 274|274|válido|
|scored_0965|E1|6|272|postprocess 272, optimize_a4 272, nested_windows 272|272|válido|
|scored_0965|E2|52-1|295||265|válido|
|scored_0965|E2|52-3|275||256|válido|
|scored_0965|E2|52-6|272||257|válido|
|scored_0965|E2|64-1|295||265|válido|
|scored_0965|E2|64-3|275||258|válido|
|scored_0965|E2|64-6|272||254|válido|
|scored_0965|E3|joint|272|individual 271|269|OPTIMAL|
|scored_0965|E4|repair|272||272|completo|
|scored_0966|E1|1|114|postprocess 114, optimize_a4 114, nested_windows 114|114|válido|
|scored_0966|E1|3|114|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|scored_0966|E1|6|114|postprocess 113, optimize_a4 113, nested_windows 113|113|válido|
|scored_0966|E2|52-1|114||110|válido|
|scored_0966|E2|52-3|114||110|válido|
|scored_0966|E2|52-6|114||110|válido|
|scored_0966|E2|64-1|114||110|válido|
|scored_0966|E2|64-3|114||110|válido|
|scored_0966|E2|64-6|114||110|válido|
|scored_0966|E3|joint|114|individual 114|114|OPTIMAL|
|scored_0966|E4|repair|114||114|completo|
|scored_0967|E1|1|146|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0967|E1|3|146|postprocess 144, optimize_a4 144, nested_windows 144|144|válido|
|scored_0967|E1|6|145|postprocess 143, optimize_a4 143, nested_windows 143|143|válido|
|scored_0967|E2|52-1|146||138|válido|
|scored_0967|E2|52-3|146||138|válido|
|scored_0967|E2|52-6|145||138|válido|
|scored_0967|E2|64-1|146||138|válido|
|scored_0967|E2|64-3|146||138|válido|
|scored_0967|E2|64-6|145||138|válido|
|scored_0967|E3|joint|145|individual 145|145|OPTIMAL|
|scored_0967|E4|repair|145||145|completo|
|scored_0968|E1|1|224|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0968|E1|3|221|postprocess 221, optimize_a4 221, nested_windows 221|221|válido|
|scored_0968|E1|6|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0968|E2|52-1|224||209|válido|
|scored_0968|E2|52-3|221||207|válido|
|scored_0968|E2|52-6|220||206|válido|
|scored_0968|E2|64-1|224||209|válido|
|scored_0968|E2|64-3|221||207|válido|
|scored_0968|E2|64-6|220||205|válido|
|scored_0968|E3|joint|220|individual 220|220|OPTIMAL|
|scored_0968|E4|repair|220||219|completo|
|scored_0969|E1|1|205|postprocess 204, optimize_a4 204, nested_windows 204|204|válido|
|scored_0969|E1|3|195|postprocess 195, optimize_a4 195, nested_windows 195|195|válido|
|scored_0969|E1|6|193|postprocess 193, optimize_a4 193, nested_windows 193|193|válido|
|scored_0969|E2|52-1|205||187|válido|
|scored_0969|E2|52-3|195||184|válido|
|scored_0969|E2|52-6|193||184|válido|
|scored_0969|E2|64-1|205||188|válido|
|scored_0969|E2|64-3|195||184|válido|
|scored_0969|E2|64-6|193||184|válido|
|scored_0969|E3|joint|193|individual 193|193|OPTIMAL|
|scored_0969|E4|repair|193||193|completo|
|scored_0970|E1|1|330|postprocess 329, optimize_a4 329, nested_windows 329|329|válido|
|scored_0970|E1|3|315|postprocess 315, optimize_a4 315, nested_windows 315|315|válido|
|scored_0970|E1|6|305|postprocess 305, optimize_a4 305, nested_windows 305|305|válido|
|scored_0970|E2|52-1|330||309|válido|
|scored_0970|E2|52-3|315||301|válido|
|scored_0970|E2|52-6|305||289|válido|
|scored_0970|E2|64-1|330||311|válido|
|scored_0970|E2|64-3|315||299|válido|
|scored_0970|E2|64-6|305||292|válido|
|scored_0970|E3|joint|305|individual 305|305|FEASIBLE|
|scored_0970|E4|repair|305||305|completo|
|scored_0971|E1|1|27|postprocess 23, optimize_a4 23, nested_windows 23|23|válido|
|scored_0971|E1|3|27|postprocess 23, optimize_a4 23, nested_windows 23|23|válido|
|scored_0971|E1|6|27|postprocess 23, optimize_a4 23, nested_windows 23|23|válido|
|scored_0971|E2|52-1|27||27|válido|
|scored_0971|E2|52-3|27||27|válido|
|scored_0971|E2|52-6|27||27|válido|
|scored_0971|E2|64-1|27||27|válido|
|scored_0971|E2|64-3|27||27|válido|
|scored_0971|E2|64-6|27||27|válido|
|scored_0971|E3|joint|27|individual 27|26|OPTIMAL|
|scored_0971|E4|repair|27||24|completo|
|scored_0972|E1|1|163|postprocess 163, optimize_a4 163, nested_windows 163|163|válido|
|scored_0972|E1|3|161|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0972|E1|6|161|postprocess 160, optimize_a4 160, nested_windows 160|160|válido|
|scored_0972|E2|52-1|163||156|válido|
|scored_0972|E2|52-3|161||155|válido|
|scored_0972|E2|52-6|161||155|válido|
|scored_0972|E2|64-1|163||157|válido|
|scored_0972|E2|64-3|161||156|válido|
|scored_0972|E2|64-6|161||155|válido|
|scored_0972|E3|joint|161|individual 161|161|OPTIMAL|
|scored_0972|E4|repair|161||159|completo|
|scored_0973|E1|1|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0973|E1|3|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0973|E1|6|146|postprocess 146, optimize_a4 146, nested_windows 146|146|válido|
|scored_0973|E2|52-1|146||136|válido|
|scored_0973|E2|52-3|146||136|válido|
|scored_0973|E2|52-6|146||136|válido|
|scored_0973|E2|64-1|146||136|válido|
|scored_0973|E2|64-3|146||136|válido|
|scored_0973|E2|64-6|146||136|válido|
|scored_0973|E3|joint|146|individual 146|146|OPTIMAL|
|scored_0973|E4|repair|146||142|completo|
|scored_0974|E1|1|325|postprocess 324, optimize_a4 324, nested_windows 324|324|válido|
|scored_0974|E1|3|319|postprocess 319, optimize_a4 319, nested_windows 319|319|válido|
|scored_0974|E1|6|310|postprocess 310, optimize_a4 310, nested_windows 310|310|válido|
|scored_0974|E2|52-1|325||302|válido|
|scored_0974|E2|52-3|319||294|válido|
|scored_0974|E2|52-6|310||286|válido|
|scored_0974|E2|64-1|325||311|válido|
|scored_0974|E2|64-3|319||296|válido|
|scored_0974|E2|64-6|310||289|válido|
|scored_0974|E3|joint|310|individual 310|309|FEASIBLE|
|scored_0974|E4|repair|310||310|completo|
|scored_0975|E1|1|171|postprocess 170, optimize_a4 170, nested_windows 170|170|válido|
|scored_0975|E1|3|167|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0975|E1|6|166|postprocess 165, optimize_a4 165, nested_windows 165|165|válido|
|scored_0975|E2|52-1|171||157|válido|
|scored_0975|E2|52-3|167||158|válido|
|scored_0975|E2|52-6|166||158|válido|
|scored_0975|E2|64-1|171||159|válido|
|scored_0975|E2|64-3|167||161|válido|
|scored_0975|E2|64-6|166||161|válido|
|scored_0975|E3|joint|166|individual 166|166|OPTIMAL|
|scored_0975|E4|repair|166||163|completo|
|scored_0976|E1|1|157|postprocess 156, optimize_a4 156, nested_windows 156|156|válido|
|scored_0976|E1|3|154|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0976|E1|6|154|postprocess 152, optimize_a4 152, nested_windows 152|152|válido|
|scored_0976|E2|52-1|157||152|válido|
|scored_0976|E2|52-3|154||151|válido|
|scored_0976|E2|52-6|154||151|válido|
|scored_0976|E2|64-1|157||152|válido|
|scored_0976|E2|64-3|154||152|válido|
|scored_0976|E2|64-6|154||152|válido|
|scored_0976|E3|joint|154|individual 153|153|OPTIMAL|
|scored_0976|E4|repair|154||154|completo|
|scored_0977|E1|1|229|postprocess 229, optimize_a4 229, nested_windows 229|229|válido|
|scored_0977|E1|3|220|postprocess 220, optimize_a4 220, nested_windows 220|220|válido|
|scored_0977|E1|6|217|postprocess 217, optimize_a4 217, nested_windows 217|217|válido|
|scored_0977|E2|52-1|229||214|válido|
|scored_0977|E2|52-3|220||210|válido|
|scored_0977|E2|52-6|217||207|válido|
|scored_0977|E2|64-1|229||216|válido|
|scored_0977|E2|64-3|220||210|válido|
|scored_0977|E2|64-6|217||208|válido|
|scored_0977|E3|joint|217|individual 217|217|OPTIMAL|
|scored_0977|E4|repair|217||217|completo|
|scored_0978|E1|1|139|postprocess 136, optimize_a4 136, nested_windows 136|136|válido|
|scored_0978|E1|3|138|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0978|E1|6|138|postprocess 137, optimize_a4 137, nested_windows 137|137|válido|
|scored_0978|E2|52-1|139||132|válido|
|scored_0978|E2|52-3|138||133|válido|
|scored_0978|E2|52-6|138||133|válido|
|scored_0978|E2|64-1|139||132|válido|
|scored_0978|E2|64-3|138||133|válido|
|scored_0978|E2|64-6|138||133|válido|
|scored_0978|E3|joint|138|individual 138|138|OPTIMAL|
|scored_0978|E4|repair|138||136|completo|
|scored_0979|E1|1|257|postprocess 257, optimize_a4 257, nested_windows 257|257|válido|
|scored_0979|E1|3|245|postprocess 245, optimize_a4 245, nested_windows 245|245|válido|
|scored_0979|E1|6|239|postprocess 239, optimize_a4 239, nested_windows 239|239|válido|
|scored_0979|E2|52-1|257||227|válido|
|scored_0979|E2|52-3|245||221|válido|
|scored_0979|E2|52-6|239||220|válido|
|scored_0979|E2|64-1|257||228|válido|
|scored_0979|E2|64-3|245||222|válido|
|scored_0979|E2|64-6|239||221|válido|
|scored_0979|E3|joint|239|individual 238|238|OPTIMAL|
|scored_0979|E4|repair|239||238|completo|
|scored_0980|E1|1|172|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0980|E1|3|171|postprocess 170, optimize_a4 170, nested_windows 170|170|válido|
|scored_0980|E1|6|168|postprocess 168, optimize_a4 168, nested_windows 168|168|válido|
|scored_0980|E2|52-1|172||161|válido|
|scored_0980|E2|52-3|171||159|válido|
|scored_0980|E2|52-6|168||158|válido|
|scored_0980|E2|64-1|172||158|válido|
|scored_0980|E2|64-3|171||157|válido|
|scored_0980|E2|64-6|168||157|válido|
|scored_0980|E3|joint|168|individual 168|168|OPTIMAL|
|scored_0980|E4|repair|168||167|completo|
|scored_0981|E1|1|36|postprocess 34, optimize_a4 34, nested_windows 34|34|válido|
|scored_0981|E1|3|36|postprocess 34, optimize_a4 34, nested_windows 34|34|válido|
|scored_0981|E1|6|36|postprocess 34, optimize_a4 34, nested_windows 34|34|válido|
|scored_0981|E2|52-1|36||36|válido|
|scored_0981|E2|52-3|36||36|válido|
|scored_0981|E2|52-6|36||36|válido|
|scored_0981|E2|64-1|36||36|válido|
|scored_0981|E2|64-3|36||36|válido|
|scored_0981|E2|64-6|36||36|válido|
|scored_0981|E3|joint|36|individual 36|36|OPTIMAL|
|scored_0981|E4|repair|36||36|completo|
|scored_0982|E1|1|395|postprocess 394, optimize_a4 394, nested_windows 394|394|válido|
|scored_0982|E1|3|371|postprocess 370, optimize_a4 370, nested_windows 370|370|válido|
|scored_0982|E1|6|368|postprocess 366, optimize_a4 366, nested_windows 366|366|válido|
|scored_0982|E2|52-1|395||355|válido|
|scored_0982|E2|52-3|371||342|válido|
|scored_0982|E2|52-6|368||342|válido|
|scored_0982|E2|64-1|395||358|válido|
|scored_0982|E2|64-3|371||344|válido|
|scored_0982|E2|64-6|368||345|válido|
|scored_0982|E3|joint|368|individual 368|363|OPTIMAL|
|scored_0982|E4|repair|368||368|completo|
|scored_0983|E1|1|191|postprocess 189, optimize_a4 189, nested_windows 189|189|válido|
|scored_0983|E1|3|186|postprocess 186, optimize_a4 186, nested_windows 186|186|válido|
|scored_0983|E1|6|185|postprocess 185, optimize_a4 185, nested_windows 185|185|válido|
|scored_0983|E2|52-1|191||177|válido|
|scored_0983|E2|52-3|186||171|válido|
|scored_0983|E2|52-6|185||174|válido|
|scored_0983|E2|64-1|191||179|válido|
|scored_0983|E2|64-3|186||173|válido|
|scored_0983|E2|64-6|185||171|válido|
|scored_0983|E3|joint|185|individual 185|184|OPTIMAL|
|scored_0983|E4|repair|185||185|completo|
|scored_0984|E1|1|181|postprocess 181, optimize_a4 181, nested_windows 181|181|válido|
|scored_0984|E1|3|182|postprocess 179, optimize_a4 179, nested_windows 179|179|válido|
|scored_0984|E1|6|177|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0984|E2|52-1|181||168|válido|
|scored_0984|E2|52-3|182||171|válido|
|scored_0984|E2|52-6|177||170|válido|
|scored_0984|E2|64-1|181||169|válido|
|scored_0984|E2|64-3|182||169|válido|
|scored_0984|E2|64-6|177||169|válido|
|scored_0984|E3|joint|177|individual 177|175|OPTIMAL|
|scored_0984|E4|repair|177||177|completo|
|scored_0985|E1|1|180|postprocess 180, optimize_a4 180, nested_windows 180|180|válido|
|scored_0985|E1|3|176|postprocess 175, optimize_a4 175, nested_windows 175|175|válido|
|scored_0985|E1|6|177|postprocess 176, optimize_a4 176, nested_windows 176|176|válido|
|scored_0985|E2|52-1|180||170|válido|
|scored_0985|E2|52-3|176||167|válido|
|scored_0985|E2|52-6|177||168|válido|
|scored_0985|E2|64-1|180||171|válido|
|scored_0985|E2|64-3|176||168|válido|
|scored_0985|E2|64-6|177||165|válido|
|scored_0985|E3|joint|176|individual 176|176|OPTIMAL|
|scored_0985|E4|repair|176||172|completo|
|scored_0986|E1|1|237|postprocess 237, optimize_a4 237, nested_windows 237|237|válido|
|scored_0986|E1|3|234|postprocess 234, optimize_a4 234, nested_windows 234|234|válido|
|scored_0986|E1|6|224|postprocess 224, optimize_a4 224, nested_windows 224|224|válido|
|scored_0986|E2|52-1|237||217|válido|
|scored_0986|E2|52-3|234||214|válido|
|scored_0986|E2|52-6|224||212|válido|
|scored_0986|E2|64-1|237||220|válido|
|scored_0986|E2|64-3|234||219|válido|
|scored_0986|E2|64-6|224||212|válido|
|scored_0986|E3|joint|224|individual 223|223|OPTIMAL|
|scored_0986|E4|repair|224||222|completo|
|scored_0987|E1|1|203|postprocess 203, optimize_a4 203, nested_windows 203|203|válido|
|scored_0987|E1|3|188|postprocess 188, optimize_a4 188, nested_windows 188|188|válido|
|scored_0987|E1|6|187|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0987|E2|52-1|203||185|válido|
|scored_0987|E2|52-3|188||182|válido|
|scored_0987|E2|52-6|187||181|válido|
|scored_0987|E2|64-1|203||185|válido|
|scored_0987|E2|64-3|188||182|válido|
|scored_0987|E2|64-6|187||177|válido|
|scored_0987|E3|joint|187|individual 187|187|OPTIMAL|
|scored_0987|E4|repair|187||185|completo|
|scored_0988|E1|1|196|postprocess 195, optimize_a4 195, nested_windows 195|195|válido|
|scored_0988|E1|3|192|postprocess 192, optimize_a4 192, nested_windows 192|192|válido|
|scored_0988|E1|6|191|postprocess 191, optimize_a4 191, nested_windows 191|191|válido|
|scored_0988|E2|52-1|196||185|válido|
|scored_0988|E2|52-3|192||187|válido|
|scored_0988|E2|52-6|191||184|válido|
|scored_0988|E2|64-1|196||181|válido|
|scored_0988|E2|64-3|192||186|válido|
|scored_0988|E2|64-6|191||187|válido|
|scored_0988|E3|joint|191|individual 190|186|OPTIMAL|
|scored_0988|E4|repair|191||188|completo|
|scored_0989|E1|1|108|postprocess 106, optimize_a4 106, nested_windows 106|106|válido|
|scored_0989|E1|3|107|postprocess 105, optimize_a4 105, nested_windows 105|105|válido|
|scored_0989|E1|6|105|postprocess 105, optimize_a4 105, nested_windows 105|105|válido|
|scored_0989|E2|52-1|108||108|válido|
|scored_0989|E2|52-3|107||101|válido|
|scored_0989|E2|52-6|105||102|válido|
|scored_0989|E2|64-1|108||108|válido|
|scored_0989|E2|64-3|107||101|válido|
|scored_0989|E2|64-6|105||102|válido|
|scored_0989|E3|joint|105|individual 105|105|OPTIMAL|
|scored_0989|E4|repair|105||105|completo|
|scored_0990|E1|1|243|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|scored_0990|E1|3|238|postprocess 238, optimize_a4 238, nested_windows 238|238|válido|
|scored_0990|E1|6|243|postprocess 243, optimize_a4 243, nested_windows 243|243|válido|
|scored_0990|E2|52-1|243||230|válido|
|scored_0990|E2|52-3|238||230|válido|
|scored_0990|E2|52-6|243||222|válido|
|scored_0990|E2|64-1|243||232|válido|
|scored_0990|E2|64-3|238||230|válido|
|scored_0990|E2|64-6|243||226|válido|
|scored_0990|E3|joint|238|individual 238|238|OPTIMAL|
|scored_0990|E4|repair|238||237|completo|
|scored_0991|E1|1|253|postprocess 252, optimize_a4 252, nested_windows 252|252|válido|
|scored_0991|E1|3|240|postprocess 240, optimize_a4 240, nested_windows 240|240|válido|
|scored_0991|E1|6|232|postprocess 231, optimize_a4 231, nested_windows 231|231|válido|
|scored_0991|E2|52-1|253||229|válido|
|scored_0991|E2|52-3|240||224|válido|
|scored_0991|E2|52-6|232||219|válido|
|scored_0991|E2|64-1|253||234|válido|
|scored_0991|E2|64-3|240||224|válido|
|scored_0991|E2|64-6|232||218|válido|
|scored_0991|E3|joint|232|individual 232|232|OPTIMAL|
|scored_0991|E4|repair|232||232|completo|
|scored_0992|E1|1|364|postprocess 364, optimize_a4 364, nested_windows 364|364|válido|
|scored_0992|E1|3|357|postprocess 356, optimize_a4 356, nested_windows 356|356|válido|
|scored_0992|E1|6|348|postprocess 348, optimize_a4 348, nested_windows 348|348|válido|
|scored_0992|E2|52-1|364||341|válido|
|scored_0992|E2|52-3|357||328|válido|
|scored_0992|E2|52-6|348||326|válido|
|scored_0992|E2|64-1|364||345|válido|
|scored_0992|E2|64-3|357||329|válido|
|scored_0992|E2|64-6|348||326|válido|
|scored_0992|E3|joint|348|individual 348|348|OPTIMAL|
|scored_0992|E4|repair|348||348|completo|
|scored_0993|E1|1|316|postprocess 315, optimize_a4 315, nested_windows 315|315|válido|
|scored_0993|E1|3|304|postprocess 304, optimize_a4 304, nested_windows 304|304|válido|
|scored_0993|E1|6|292|postprocess 292, optimize_a4 292, nested_windows 292|292|válido|
|scored_0993|E2|52-1|316||285|válido|
|scored_0993|E2|52-3|304||276|válido|
|scored_0993|E2|52-6|292||274|válido|
|scored_0993|E2|64-1|316||287|válido|
|scored_0993|E2|64-3|304||276|válido|
|scored_0993|E2|64-6|292||275|válido|
|scored_0993|E3|joint|292|individual 291|290|OPTIMAL|
|scored_0993|E4|repair|292||291|completo|
|scored_0994|E1|1|220|postprocess 219, optimize_a4 219, nested_windows 219|219|válido|
|scored_0994|E1|3|214|postprocess 213, optimize_a4 213, nested_windows 213|213|válido|
|scored_0994|E1|6|216|postprocess 216, optimize_a4 216, nested_windows 216|216|válido|
|scored_0994|E2|52-1|220||203|válido|
|scored_0994|E2|52-3|214||195|válido|
|scored_0994|E2|52-6|216||200|válido|
|scored_0994|E2|64-1|220||202|válido|
|scored_0994|E2|64-3|214||194|válido|
|scored_0994|E2|64-6|216||197|válido|
|scored_0994|E3|joint|214|individual 212|209|OPTIMAL|
|scored_0994|E4|repair|214||211|completo|
|scored_0995|E1|1|198|postprocess 197, optimize_a4 197, nested_windows 197|197|válido|
|scored_0995|E1|3|190|postprocess 190, optimize_a4 190, nested_windows 190|190|válido|
|scored_0995|E1|6|187|postprocess 187, optimize_a4 187, nested_windows 187|187|válido|
|scored_0995|E2|52-1|198||184|válido|
|scored_0995|E2|52-3|190||182|válido|
|scored_0995|E2|52-6|187||181|válido|
|scored_0995|E2|64-1|198||186|válido|
|scored_0995|E2|64-3|190||178|válido|
|scored_0995|E2|64-6|187||178|válido|
|scored_0995|E3|joint|187|individual 187|187|OPTIMAL|
|scored_0995|E4|repair|187||187|completo|
|scored_0996|E1|1|174|postprocess 172, optimize_a4 172, nested_windows 172|172|válido|
|scored_0996|E1|3|171|postprocess 169, optimize_a4 168, nested_windows 168|168|válido|
|scored_0996|E1|6|174|postprocess 173, optimize_a4 173, nested_windows 173|173|válido|
|scored_0996|E2|52-1|174||163|válido|
|scored_0996|E2|52-3|171||161|válido|
|scored_0996|E2|52-6|174||160|válido|
|scored_0996|E2|64-1|174||162|válido|
|scored_0996|E2|64-3|171||162|válido|
|scored_0996|E2|64-6|174||162|válido|
|scored_0996|E3|joint|171|individual 171|171|OPTIMAL|
|scored_0996|E4|repair|171||170|completo|
|scored_0997|E1|1|177|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0997|E1|3|177|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0997|E1|6|177|postprocess 177, optimize_a4 177, nested_windows 177|177|válido|
|scored_0997|E2|52-1|177||168|válido|
|scored_0997|E2|52-3|177||163|válido|
|scored_0997|E2|52-6|177||161|válido|
|scored_0997|E2|64-1|177||163|válido|
|scored_0997|E2|64-3|177||161|válido|
|scored_0997|E2|64-6|177||162|válido|
|scored_0997|E3|joint|177|individual 177|175|OPTIMAL|
|scored_0997|E4|repair|177||176|completo|
|scored_0998|E1|1|295|postprocess 295, optimize_a4 295, nested_windows 295|295|válido|
|scored_0998|E1|3|278|postprocess 278, optimize_a4 278, nested_windows 278|278|válido|
|scored_0998|E1|6|279|postprocess 278, optimize_a4 278, nested_windows 278|278|válido|
|scored_0998|E2|52-1|295||269|válido|
|scored_0998|E2|52-3|278||259|válido|
|scored_0998|E2|52-6|279||255|válido|
|scored_0998|E2|64-1|295||270|válido|
|scored_0998|E2|64-3|278||258|válido|
|scored_0998|E2|64-6|279||256|válido|
|scored_0998|E3|joint|278|individual 278|276|OPTIMAL|
|scored_0998|E4|repair|278||278|completo|
|scored_0999|E1|1|145|postprocess 145, optimize_a4 145, nested_windows 145|145|válido|
|scored_0999|E1|3|138|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0999|E1|6|138|postprocess 138, optimize_a4 138, nested_windows 138|138|válido|
|scored_0999|E2|52-1|145||134|válido|
|scored_0999|E2|52-3|138||133|válido|
|scored_0999|E2|52-6|138||132|válido|
|scored_0999|E2|64-1|145||134|válido|
|scored_0999|E2|64-3|138||135|válido|
|scored_0999|E2|64-6|138||133|válido|
|scored_0999|E3|joint|138|individual 138|138|OPTIMAL|
|scored_0999|E4|repair|138||134|completo|
|scored_1000|E1|1|345|postprocess 345, optimize_a4 345, nested_windows 345|345|válido|
|scored_1000|E1|3|323|postprocess 322, optimize_a4 322, nested_windows 322|322|válido|
|scored_1000|E1|6|311|postprocess 311, optimize_a4 311, nested_windows 311|311|válido|
|scored_1000|E2|52-1|345||312|válido|
|scored_1000|E2|52-3|323||297|válido|
|scored_1000|E2|52-6|311||299|válido|
|scored_1000|E2|64-1|345||313|válido|
|scored_1000|E2|64-3|323||297|válido|
|scored_1000|E2|64-6|311||300|válido|
|scored_1000|E3|joint|311|individual 311|311|OPTIMAL|
|scored_1000|E4|repair|311||311|completo|

## Greedy por objetivos compartidos en práctica

Control E4 solo: 35944 pasos. Frente al control, la cartera greedy + E4 mejora 4 IDs, empata 196 y pierde en 0. Diferencia neta: 6 pasos.

Control: 36.049 pasos. Por ronda se construye el mismo banco a partir del estado actual. Hasta 8 rondas, 128 intentos por ronda y 40 intermedios más las semillas. Tras cada aceptación se regenera el banco. Cada orden termina con la misma reparación E4 (hasta 30 s).

200 programas reproducidos. Cartera: 35938. [Comparación](reports/shared_greedy_vfo65a3a/comparison.csv)
coverage: 36043 antes de E4 y 35941 después. Pasos ahorrados por movimientos con 1, 2 y 3+ objetivos: 4, 1, 1.
saving: 36040 antes de E4 y 35938 después. Pasos ahorrados por movimientos con 1, 2 y 3+ objetivos: 8, 0, 1.

| ID | Control | Cobertura | Cobertura + E4 | Ahorro estimado | Ahorro estimado + E4 | Mejor |
|---|---:|---:|---:|---:|---:|---:|
|practice_0001|154|154|153|154|153|153|
|practice_0002|267|267|267|267|267|267|
|practice_0003|285|285|284|285|284|284|
|practice_0004|130|130|129|130|129|129|
|practice_0005|232|232|232|232|232|232|
|practice_0006|38|38|38|38|38|38|
|practice_0007|105|105|105|105|105|105|
|practice_0008|258|258|258|258|258|258|
|practice_0009|139|139|137|139|137|137|
|practice_0010|141|141|140|141|140|140|
|practice_0011|128|128|128|128|128|128|
|practice_0012|259|259|259|259|259|259|
|practice_0013|215|215|215|215|215|215|
|practice_0014|138|138|138|138|138|138|
|practice_0015|201|201|199|201|199|199|
|practice_0016|229|229|229|229|229|229|
|practice_0017|175|175|175|175|175|175|
|practice_0018|342|342|342|342|342|342|
|practice_0019|209|209|207|209|207|207|
|practice_0020|112|112|112|112|112|112|
|practice_0021|136|136|134|136|134|134|
|practice_0022|143|143|143|143|143|143|
|practice_0023|193|193|192|193|192|192|
|practice_0024|42|42|42|39|39|39|
|practice_0025|182|182|182|182|182|182|
|practice_0026|153|153|152|153|152|152|
|practice_0027|205|205|204|205|204|204|
|practice_0028|240|240|240|240|240|240|
|practice_0029|212|212|212|212|212|212|
|practice_0030|140|140|140|140|140|140|
|practice_0031|226|226|226|226|226|226|
|practice_0032|140|140|140|140|140|140|
|practice_0033|41|41|41|41|41|41|
|practice_0034|318|318|318|318|318|318|
|practice_0035|264|264|264|264|264|264|
|practice_0036|133|133|132|133|132|132|
|practice_0037|293|293|293|293|293|293|
|practice_0038|43|43|43|43|43|43|
|practice_0039|6|6|6|6|6|6|
|practice_0040|198|198|198|198|198|198|
|practice_0041|188|188|188|188|188|188|
|practice_0042|185|185|182|185|182|182|
|practice_0043|127|126|122|126|122|122|
|practice_0044|230|230|230|230|230|230|
|practice_0045|150|150|147|150|147|147|
|practice_0046|271|271|271|271|271|271|
|practice_0047|2|2|2|2|2|2|
|practice_0048|124|124|123|124|123|123|
|practice_0049|131|130|129|130|129|129|
|practice_0050|3|3|3|3|3|3|
|practice_0051|5|5|5|5|5|5|
|practice_0052|209|209|208|209|208|208|
|practice_0053|267|267|267|267|267|267|
|practice_0054|346|346|346|346|346|346|
|practice_0055|349|349|349|349|349|349|
|practice_0056|4|4|4|4|4|4|
|practice_0057|256|256|256|256|256|256|
|practice_0058|267|267|267|267|267|267|
|practice_0059|150|150|147|150|147|147|
|practice_0060|265|265|265|265|265|265|
|practice_0061|282|282|282|282|282|282|
|practice_0062|313|313|313|313|313|313|
|practice_0063|131|131|128|131|128|128|
|practice_0064|255|255|255|255|255|255|
|practice_0065|236|236|236|236|236|236|
|practice_0066|109|109|109|109|109|109|
|practice_0067|173|173|171|173|171|171|
|practice_0068|196|196|196|196|196|196|
|practice_0069|220|220|220|220|220|220|
|practice_0070|270|270|270|270|270|270|
|practice_0071|169|169|167|169|167|167|
|practice_0072|5|5|5|5|5|5|
|practice_0073|159|159|159|159|159|159|
|practice_0074|220|220|217|220|217|217|
|practice_0075|198|198|198|198|198|198|
|practice_0076|332|332|332|332|332|332|
|practice_0077|114|114|113|114|113|113|
|practice_0078|284|284|284|284|284|284|
|practice_0079|137|137|137|137|137|137|
|practice_0080|120|120|120|120|120|120|
|practice_0081|196|196|196|196|196|196|
|practice_0082|239|239|239|239|239|239|
|practice_0083|152|152|152|152|152|152|
|practice_0084|162|162|158|162|158|158|
|practice_0085|304|304|304|304|304|304|
|practice_0086|151|151|151|151|151|151|
|practice_0087|255|255|255|255|255|255|
|practice_0088|233|233|232|233|232|232|
|practice_0089|165|165|163|165|163|163|
|practice_0090|155|155|155|155|155|155|
|practice_0091|224|224|223|224|223|223|
|practice_0092|181|181|181|181|181|181|
|practice_0093|238|238|238|238|238|238|
|practice_0094|141|141|141|141|141|141|
|practice_0095|259|259|259|259|259|259|
|practice_0096|214|214|214|214|214|214|
|practice_0097|133|133|131|133|131|131|
|practice_0098|1|1|1|1|1|1|
|practice_0099|132|132|130|132|130|130|
|practice_0100|341|341|341|341|341|341|
|practice_0101|166|166|166|166|166|166|
|practice_0102|306|306|306|306|306|306|
|practice_0103|5|5|5|5|5|5|
|practice_0104|230|230|230|230|230|230|
|practice_0105|195|195|195|195|195|195|
|practice_0106|180|180|179|180|179|179|
|practice_0107|125|125|124|125|124|124|
|practice_0108|116|116|114|116|114|114|
|practice_0109|150|150|148|150|148|148|
|practice_0110|268|268|268|268|268|268|
|practice_0111|180|180|180|180|180|180|
|practice_0112|128|128|128|128|128|128|
|practice_0113|306|306|306|306|306|306|
|practice_0114|55|55|55|55|55|55|
|practice_0115|189|189|189|189|189|189|
|practice_0116|67|67|67|67|67|67|
|practice_0117|143|143|142|143|142|142|
|practice_0118|40|40|40|40|40|40|
|practice_0119|266|266|266|266|266|266|
|practice_0120|166|166|166|166|166|166|
|practice_0121|280|280|280|280|280|280|
|practice_0122|319|319|319|319|319|319|
|practice_0123|2|2|2|2|2|2|
|practice_0124|226|226|224|226|224|224|
|practice_0125|3|3|3|3|3|3|
|practice_0126|156|156|155|156|155|155|
|practice_0127|196|196|196|196|196|196|
|practice_0128|2|2|2|2|2|2|
|practice_0129|125|125|124|125|124|124|
|practice_0130|111|111|110|111|110|110|
|practice_0131|226|226|226|226|226|226|
|practice_0132|179|179|179|179|179|179|
|practice_0133|188|188|187|188|187|187|
|practice_0134|215|215|215|215|215|215|
|practice_0135|344|344|344|344|344|344|
|practice_0136|54|54|53|54|53|53|
|practice_0137|106|105|105|105|105|105|
|practice_0138|345|345|345|345|345|345|
|practice_0139|218|218|218|218|218|218|
|practice_0140|1|1|1|1|1|1|
|practice_0141|156|156|155|156|155|155|
|practice_0142|264|264|264|264|264|264|
|practice_0143|156|156|153|156|153|153|
|practice_0144|115|115|113|115|113|113|
|practice_0145|201|201|199|201|199|199|
|practice_0146|45|45|44|45|44|44|
|practice_0147|188|188|188|188|188|188|
|practice_0148|176|176|176|176|176|176|
|practice_0149|201|201|201|201|201|201|
|practice_0150|203|203|203|203|203|203|
|practice_0151|333|333|333|333|333|333|
|practice_0152|161|161|161|161|161|161|
|practice_0153|149|149|146|149|146|146|
|practice_0154|336|336|336|336|336|336|
|practice_0155|160|160|158|160|158|158|
|practice_0156|111|111|109|111|109|109|
|practice_0157|143|143|138|143|138|138|
|practice_0158|236|236|235|236|235|235|
|practice_0159|34|34|34|34|34|34|
|practice_0160|186|186|186|186|186|186|
|practice_0161|274|274|274|274|274|274|
|practice_0162|285|285|285|285|285|285|
|practice_0163|263|263|263|263|263|263|
|practice_0164|255|255|255|255|255|255|
|practice_0165|239|239|239|239|239|239|
|practice_0166|51|51|51|51|51|51|
|practice_0167|300|300|300|300|300|300|
|practice_0168|130|130|128|130|128|128|
|practice_0169|4|4|4|4|4|4|
|practice_0170|36|36|36|36|36|36|
|practice_0171|292|292|292|292|292|292|
|practice_0172|2|2|2|2|2|2|
|practice_0173|147|147|147|147|147|147|
|practice_0174|2|2|2|2|2|2|
|practice_0175|5|4|4|4|4|4|
|practice_0176|318|318|317|318|317|317|
|practice_0177|179|179|178|179|178|178|
|practice_0178|93|93|92|93|92|92|
|practice_0179|342|342|342|342|342|342|
|practice_0180|155|155|154|155|154|154|
|practice_0181|281|281|281|281|281|281|
|practice_0182|220|220|220|220|220|220|
|practice_0183|286|286|286|286|286|286|
|practice_0184|254|254|254|254|254|254|
|practice_0185|172|172|171|172|171|171|
|practice_0186|118|118|118|118|118|118|
|practice_0187|218|218|218|218|218|218|
|practice_0188|224|224|224|224|224|224|
|practice_0189|184|184|184|184|184|184|
|practice_0190|295|295|295|295|295|295|
|practice_0191|313|313|313|313|313|313|
|practice_0192|124|123|121|123|121|121|
|practice_0193|135|135|133|135|133|133|
|practice_0194|249|249|249|249|249|249|
|practice_0195|262|262|262|262|262|262|
|practice_0196|137|137|137|137|137|137|
|practice_0197|207|207|207|207|207|207|
|practice_0198|44|43|43|43|43|43|
|practice_0199|291|291|291|291|291|291|
|practice_0200|139|139|139|139|139|139|

## Caminos compartidos de hasta tres operaciones

Control: 35.938 pasos. Se busca un valor nuevo útil a dos o más objetivos. Cada eslabón combina el anterior con un ancla. Mismos límites por variante: semillas + 24 intermedios, 48 valores compartidos por ronda, hasta 4 rondas. Orden por cobertura decreciente. Cada columna muestra antes/después de E4.

200 programas reproducidos: 35935 pasos. [Comparación por ID](reports/shared_paths_4haiz295/comparison.csv)
Profundidad 0: 35938 antes de E4, 35936 después.
Profundidad 1: 35938 antes de E4, 35936 después.
Profundidad 2: 35937 antes de E4, 35935 después.
Profundidad 3: 35937 antes de E4, 35935 después.

| ID | Control | Hasta 1 | Hasta 2 | Hasta 3 | Mejor |
|---|---:|---:|---:|---:|---:|
|practice_0001|153/153|153/153|153/153|153/153|153|
|practice_0002|267/267|267/267|267/267|267/267|267|
|practice_0003|284/284|284/284|284/284|284/284|284|
|practice_0004|129/129|129/129|129/129|129/129|129|
|practice_0005|232/232|232/232|232/232|232/232|232|
|practice_0006|38/38|38/38|38/38|38/38|38|
|practice_0007|105/105|105/105|105/105|105/105|105|
|practice_0008|258/258|258/258|258/258|258/258|258|
|practice_0009|137/137|137/137|137/137|137/137|137|
|practice_0010|140/140|140/140|140/140|140/140|140|
|practice_0011|128/128|128/128|128/128|128/128|128|
|practice_0012|259/259|259/259|259/259|259/259|259|
|practice_0013|215/215|215/215|215/215|215/215|215|
|practice_0014|138/138|138/138|138/138|138/138|138|
|practice_0015|199/199|199/199|199/199|199/199|199|
|practice_0016|229/229|229/229|229/229|229/229|229|
|practice_0017|175/175|175/175|175/175|175/175|175|
|practice_0018|342/342|342/342|342/342|342/342|342|
|practice_0019|207/207|207/207|207/207|207/207|207|
|practice_0020|112/112|112/112|112/112|112/112|112|
|practice_0021|134/134|134/134|134/134|134/134|134|
|practice_0022|143/143|143/143|143/143|143/143|143|
|practice_0023|192/192|192/192|192/192|192/192|192|
|practice_0024|39/39|39/39|39/39|39/39|39|
|practice_0025|182/182|182/182|182/182|182/182|182|
|practice_0026|152/152|152/152|152/152|152/152|152|
|practice_0027|204/204|204/204|204/204|204/204|204|
|practice_0028|240/240|240/240|240/240|240/240|240|
|practice_0029|212/212|212/212|212/212|212/212|212|
|practice_0030|140/140|140/140|140/140|140/140|140|
|practice_0031|226/226|226/226|226/226|226/226|226|
|practice_0032|140/140|140/140|140/140|140/140|140|
|practice_0033|41/41|41/41|41/41|41/41|41|
|practice_0034|318/318|318/318|318/318|318/318|318|
|practice_0035|264/264|264/264|264/264|264/264|264|
|practice_0036|132/132|132/132|132/132|132/132|132|
|practice_0037|293/293|293/293|293/293|293/293|293|
|practice_0038|43/43|43/43|43/43|43/43|43|
|practice_0039|6/6|6/6|6/6|6/6|6|
|practice_0040|198/198|198/198|198/198|198/198|198|
|practice_0041|188/188|188/188|188/188|188/188|188|
|practice_0042|182/182|182/182|182/182|182/182|182|
|practice_0043|122/122|122/122|122/122|122/122|122|
|practice_0044|230/230|230/230|230/230|230/230|230|
|practice_0045|147/147|147/147|147/147|147/147|147|
|practice_0046|271/271|271/271|271/271|271/271|271|
|practice_0047|2/2|2/2|2/2|2/2|2|
|practice_0048|123/123|123/123|123/123|123/123|123|
|practice_0049|129/129|129/129|129/129|129/129|129|
|practice_0050|3/3|3/3|3/3|3/3|3|
|practice_0051|5/5|5/5|5/5|5/5|5|
|practice_0052|208/208|208/208|208/208|208/208|208|
|practice_0053|267/267|267/267|267/267|267/267|267|
|practice_0054|346/346|346/346|346/346|346/346|346|
|practice_0055|349/349|349/349|349/349|349/349|349|
|practice_0056|4/4|4/4|4/4|4/4|4|
|practice_0057|256/256|256/256|256/256|256/256|256|
|practice_0058|267/267|267/267|267/267|267/267|267|
|practice_0059|147/147|147/147|147/147|147/147|147|
|practice_0060|265/265|265/265|265/265|265/265|265|
|practice_0061|282/282|282/282|282/282|282/282|282|
|practice_0062|313/313|313/313|313/313|313/313|313|
|practice_0063|128/128|128/128|128/128|128/128|128|
|practice_0064|255/255|255/255|255/255|255/255|255|
|practice_0065|236/236|236/236|236/236|236/236|236|
|practice_0066|109/109|109/109|109/109|109/109|109|
|practice_0067|171/171|171/171|171/171|171/171|171|
|practice_0068|196/196|196/196|196/196|196/196|196|
|practice_0069|220/220|220/220|220/220|220/220|220|
|practice_0070|270/270|270/270|270/270|270/270|270|
|practice_0071|167/167|167/167|167/167|167/167|167|
|practice_0072|5/5|5/5|5/5|5/5|5|
|practice_0073|159/159|159/159|159/159|159/159|159|
|practice_0074|217/217|217/217|217/217|217/217|217|
|practice_0075|198/198|198/198|198/198|198/198|198|
|practice_0076|332/332|332/332|332/332|332/332|332|
|practice_0077|113/113|113/113|113/113|113/113|113|
|practice_0078|284/284|284/284|284/284|284/284|284|
|practice_0079|137/137|137/137|137/137|137/137|137|
|practice_0080|120/120|120/120|120/120|120/120|120|
|practice_0081|196/196|196/196|196/196|196/196|196|
|practice_0082|239/239|239/239|239/239|239/239|239|
|practice_0083|152/152|152/152|152/152|152/152|152|
|practice_0084|158/158|158/158|158/158|158/158|158|
|practice_0085|304/304|304/304|304/304|304/304|304|
|practice_0086|151/151|151/151|151/151|151/151|151|
|practice_0087|255/255|255/255|255/255|255/255|255|
|practice_0088|232/232|232/232|232/232|232/232|232|
|practice_0089|163/162|163/162|163/162|163/162|162|
|practice_0090|155/155|155/155|155/155|155/155|155|
|practice_0091|223/223|223/223|223/223|223/223|223|
|practice_0092|181/181|181/181|181/181|181/181|181|
|practice_0093|238/238|238/238|238/238|238/238|238|
|practice_0094|141/141|141/141|141/141|141/141|141|
|practice_0095|259/259|259/259|259/259|259/259|259|
|practice_0096|214/214|214/214|214/214|214/214|214|
|practice_0097|131/131|131/131|131/131|131/131|131|
|practice_0098|1/1|1/1|1/1|1/1|1|
|practice_0099|130/130|130/130|130/130|130/130|130|
|practice_0100|341/341|341/341|341/341|341/341|341|
|practice_0101|166/166|166/166|166/166|166/166|166|
|practice_0102|306/306|306/306|306/306|306/306|306|
|practice_0103|5/5|5/5|5/5|5/5|5|
|practice_0104|230/230|230/230|230/230|230/230|230|
|practice_0105|195/195|195/195|195/195|195/195|195|
|practice_0106|179/179|179/179|179/179|179/179|179|
|practice_0107|124/124|124/124|124/124|124/124|124|
|practice_0108|114/114|114/114|114/114|114/114|114|
|practice_0109|148/147|148/147|148/147|148/147|147|
|practice_0110|268/268|268/268|268/268|268/268|268|
|practice_0111|180/180|180/180|180/180|180/180|180|
|practice_0112|128/128|128/128|128/128|128/128|128|
|practice_0113|306/306|306/306|306/306|306/306|306|
|practice_0114|55/55|55/55|55/55|55/55|55|
|practice_0115|189/189|189/189|189/189|189/189|189|
|practice_0116|67/67|67/67|67/67|67/67|67|
|practice_0117|142/142|142/142|142/142|142/142|142|
|practice_0118|40/40|40/40|40/40|40/40|40|
|practice_0119|266/266|266/266|266/266|266/266|266|
|practice_0120|166/166|166/166|166/166|166/166|166|
|practice_0121|280/280|280/280|280/280|280/280|280|
|practice_0122|319/319|319/319|319/319|319/319|319|
|practice_0123|2/2|2/2|2/2|2/2|2|
|practice_0124|224/224|224/224|224/224|224/224|224|
|practice_0125|3/3|3/3|3/3|3/3|3|
|practice_0126|155/155|155/155|155/155|155/155|155|
|practice_0127|196/196|196/196|196/196|196/196|196|
|practice_0128|2/2|2/2|2/2|2/2|2|
|practice_0129|124/124|124/124|124/124|124/124|124|
|practice_0130|110/110|110/110|110/110|110/110|110|
|practice_0131|226/226|226/226|226/226|226/226|226|
|practice_0132|179/179|179/179|179/179|179/179|179|
|practice_0133|187/187|187/187|187/187|187/187|187|
|practice_0134|215/215|215/215|215/215|215/215|215|
|practice_0135|344/344|344/344|344/344|344/344|344|
|practice_0136|53/53|53/53|53/53|53/53|53|
|practice_0137|105/105|105/105|105/105|105/105|105|
|practice_0138|345/345|345/345|345/345|345/345|345|
|practice_0139|218/218|218/218|218/218|218/218|218|
|practice_0140|1/1|1/1|1/1|1/1|1|
|practice_0141|155/155|155/155|155/155|155/155|155|
|practice_0142|264/264|264/264|264/264|264/264|264|
|practice_0143|153/153|153/153|153/153|153/153|153|
|practice_0144|113/113|113/113|113/113|113/113|113|
|practice_0145|199/199|199/199|199/199|199/199|199|
|practice_0146|44/44|44/44|44/44|44/44|44|
|practice_0147|188/188|188/188|188/188|188/188|188|
|practice_0148|176/176|176/176|176/176|176/176|176|
|practice_0149|201/201|201/201|201/201|201/201|201|
|practice_0150|203/203|203/203|203/203|203/203|203|
|practice_0151|333/333|333/333|333/333|333/333|333|
|practice_0152|161/161|161/161|161/161|161/161|161|
|practice_0153|146/146|146/146|146/146|146/146|146|
|practice_0154|336/336|336/336|336/336|336/336|336|
|practice_0155|158/158|158/158|158/158|158/158|158|
|practice_0156|109/109|109/109|109/109|109/109|109|
|practice_0157|138/138|138/138|138/138|138/138|138|
|practice_0158|235/235|235/235|235/235|235/235|235|
|practice_0159|34/34|34/34|34/34|34/34|34|
|practice_0160|186/186|186/186|186/186|186/186|186|
|practice_0161|274/274|274/274|274/274|274/274|274|
|practice_0162|285/285|285/285|285/285|285/285|285|
|practice_0163|263/263|263/263|263/263|263/263|263|
|practice_0164|255/255|255/255|255/255|255/255|255|
|practice_0165|239/239|239/239|239/239|239/239|239|
|practice_0166|51/51|51/51|51/51|51/51|51|
|practice_0167|300/300|300/300|300/300|300/300|300|
|practice_0168|128/128|128/128|128/128|128/128|128|
|practice_0169|4/4|4/4|4/4|4/4|4|
|practice_0170|36/36|36/36|36/36|36/36|36|
|practice_0171|292/292|292/292|292/292|292/292|292|
|practice_0172|2/2|2/2|2/2|2/2|2|
|practice_0173|147/147|147/147|147/147|147/147|147|
|practice_0174|2/2|2/2|2/2|2/2|2|
|practice_0175|4/4|4/4|4/4|4/4|4|
|practice_0176|317/317|317/317|317/317|317/317|317|
|practice_0177|178/178|178/178|178/178|178/178|178|
|practice_0178|92/92|92/92|92/92|92/92|92|
|practice_0179|342/342|342/342|342/342|342/342|342|
|practice_0180|154/154|154/154|154/154|154/154|154|
|practice_0181|281/281|281/281|281/281|281/281|281|
|practice_0182|220/220|220/220|220/220|220/220|220|
|practice_0183|286/286|286/286|286/286|286/286|286|
|practice_0184|254/254|254/254|254/254|254/254|254|
|practice_0185|171/171|171/171|171/171|171/171|171|
|practice_0186|118/118|118/118|118/118|118/118|118|
|practice_0187|218/218|218/218|218/218|218/218|218|
|practice_0188|224/224|224/224|224/224|224/224|224|
|practice_0189|184/184|184/184|184/184|184/184|184|
|practice_0190|295/295|295/295|295/295|295/295|295|
|practice_0191|313/313|313/313|313/313|313/313|313|
|practice_0192|121/121|121/121|120/120|120/120|120|
|practice_0193|133/133|133/133|133/133|133/133|133|
|practice_0194|249/249|249/249|249/249|249/249|249|
|practice_0195|262/262|262/262|262/262|262/262|262|
|practice_0196|137/137|137/137|137/137|137/137|137|
|practice_0197|207/207|207/207|207/207|207/207|207|
|practice_0198|43/43|43/43|43/43|43/43|43|
|practice_0199|291/291|291/291|291/291|291/291|291|
|practice_0200|139/139|139/139|139/139|139/139|139|

## Diferencias de objetivos y punto de encuentro

Base: 35.935 pasos. Hasta cuatro sustituciones estrictas por puzzle; centro disponible o construido en una o dos operaciones. Solo práctica, sin E4 adicional.

200 programas validados, 35926 pasos; 8 IDs mejorados. [Movimientos por ID](reports/offsets_gt_i3wcj/comparison.csv)

| ID | Antes | Después | Ahorro |
|---|---:|---:|---:|
|practice_0001|153|153|0|
|practice_0002|267|267|0|
|practice_0003|284|284|0|
|practice_0004|129|129|0|
|practice_0005|232|232|0|
|practice_0006|38|38|0|
|practice_0007|105|105|0|
|practice_0008|258|258|0|
|practice_0009|137|137|0|
|practice_0010|140|139|1|
|practice_0011|128|128|0|
|practice_0012|259|259|0|
|practice_0013|215|215|0|
|practice_0014|138|138|0|
|practice_0015|199|199|0|
|practice_0016|229|229|0|
|practice_0017|175|175|0|
|practice_0018|342|342|0|
|practice_0019|207|207|0|
|practice_0020|112|112|0|
|practice_0021|134|133|1|
|practice_0022|143|143|0|
|practice_0023|192|192|0|
|practice_0024|39|39|0|
|practice_0025|182|182|0|
|practice_0026|152|152|0|
|practice_0027|204|204|0|
|practice_0028|240|240|0|
|practice_0029|212|212|0|
|practice_0030|140|140|0|
|practice_0031|226|226|0|
|practice_0032|140|140|0|
|practice_0033|41|41|0|
|practice_0034|318|318|0|
|practice_0035|264|264|0|
|practice_0036|132|132|0|
|practice_0037|293|293|0|
|practice_0038|43|43|0|
|practice_0039|6|6|0|
|practice_0040|198|198|0|
|practice_0041|188|188|0|
|practice_0042|182|182|0|
|practice_0043|122|122|0|
|practice_0044|230|230|0|
|practice_0045|147|147|0|
|practice_0046|271|271|0|
|practice_0047|2|2|0|
|practice_0048|123|122|1|
|practice_0049|129|129|0|
|practice_0050|3|3|0|
|practice_0051|5|5|0|
|practice_0052|208|208|0|
|practice_0053|267|267|0|
|practice_0054|346|346|0|
|practice_0055|349|349|0|
|practice_0056|4|4|0|
|practice_0057|256|256|0|
|practice_0058|267|267|0|
|practice_0059|147|147|0|
|practice_0060|265|265|0|
|practice_0061|282|282|0|
|practice_0062|313|313|0|
|practice_0063|128|128|0|
|practice_0064|255|255|0|
|practice_0065|236|236|0|
|practice_0066|109|107|2|
|practice_0067|171|171|0|
|practice_0068|196|196|0|
|practice_0069|220|220|0|
|practice_0070|270|270|0|
|practice_0071|167|167|0|
|practice_0072|5|5|0|
|practice_0073|159|159|0|
|practice_0074|217|217|0|
|practice_0075|198|198|0|
|practice_0076|332|332|0|
|practice_0077|113|113|0|
|practice_0078|284|284|0|
|practice_0079|137|137|0|
|practice_0080|120|119|1|
|practice_0081|196|196|0|
|practice_0082|239|239|0|
|practice_0083|152|151|1|
|practice_0084|158|158|0|
|practice_0085|304|304|0|
|practice_0086|151|151|0|
|practice_0087|255|255|0|
|practice_0088|232|232|0|
|practice_0089|162|162|0|
|practice_0090|155|155|0|
|practice_0091|223|223|0|
|practice_0092|181|181|0|
|practice_0093|238|238|0|
|practice_0094|141|141|0|
|practice_0095|259|259|0|
|practice_0096|214|214|0|
|practice_0097|131|131|0|
|practice_0098|1|1|0|
|practice_0099|130|130|0|
|practice_0100|341|341|0|
|practice_0101|166|166|0|
|practice_0102|306|306|0|
|practice_0103|5|5|0|
|practice_0104|230|230|0|
|practice_0105|195|195|0|
|practice_0106|179|179|0|
|practice_0107|124|124|0|
|practice_0108|114|114|0|
|practice_0109|147|147|0|
|practice_0110|268|268|0|
|practice_0111|180|180|0|
|practice_0112|128|128|0|
|practice_0113|306|306|0|
|practice_0114|55|55|0|
|practice_0115|189|189|0|
|practice_0116|67|67|0|
|practice_0117|142|142|0|
|practice_0118|40|40|0|
|practice_0119|266|266|0|
|practice_0120|166|166|0|
|practice_0121|280|280|0|
|practice_0122|319|319|0|
|practice_0123|2|2|0|
|practice_0124|224|224|0|
|practice_0125|3|3|0|
|practice_0126|155|155|0|
|practice_0127|196|196|0|
|practice_0128|2|2|0|
|practice_0129|124|124|0|
|practice_0130|110|110|0|
|practice_0131|226|226|0|
|practice_0132|179|179|0|
|practice_0133|187|187|0|
|practice_0134|215|215|0|
|practice_0135|344|344|0|
|practice_0136|53|53|0|
|practice_0137|105|105|0|
|practice_0138|345|345|0|
|practice_0139|218|218|0|
|practice_0140|1|1|0|
|practice_0141|155|155|0|
|practice_0142|264|264|0|
|practice_0143|153|153|0|
|practice_0144|113|113|0|
|practice_0145|199|199|0|
|practice_0146|44|44|0|
|practice_0147|188|188|0|
|practice_0148|176|176|0|
|practice_0149|201|201|0|
|practice_0150|203|203|0|
|practice_0151|333|333|0|
|practice_0152|161|161|0|
|practice_0153|146|146|0|
|practice_0154|336|336|0|
|practice_0155|158|158|0|
|practice_0156|109|109|0|
|practice_0157|138|138|0|
|practice_0158|235|235|0|
|practice_0159|34|34|0|
|practice_0160|186|186|0|
|practice_0161|274|274|0|
|practice_0162|285|285|0|
|practice_0163|263|263|0|
|practice_0164|255|254|1|
|practice_0165|239|239|0|
|practice_0166|51|51|0|
|practice_0167|300|300|0|
|practice_0168|128|128|0|
|practice_0169|4|4|0|
|practice_0170|36|36|0|
|practice_0171|292|292|0|
|practice_0172|2|2|0|
|practice_0173|147|147|0|
|practice_0174|2|2|0|
|practice_0175|4|4|0|
|practice_0176|317|317|0|
|practice_0177|178|178|0|
|practice_0178|92|92|0|
|practice_0179|342|342|0|
|practice_0180|154|154|0|
|practice_0181|281|281|0|
|practice_0182|220|220|0|
|practice_0183|286|286|0|
|practice_0184|254|254|0|
|practice_0185|171|171|0|
|practice_0186|118|118|0|
|practice_0187|218|218|0|
|practice_0188|224|224|0|
|practice_0189|184|184|0|
|practice_0190|295|295|0|
|practice_0191|313|313|0|
|practice_0192|120|119|1|
|practice_0193|133|133|0|
|practice_0194|249|249|0|
|practice_0195|262|262|0|
|practice_0196|137|137|0|
|practice_0197|207|207|0|
|practice_0198|43|43|0|
|practice_0199|291|291|0|
|practice_0200|139|139|0|

## Dos pasadas de posprocesamiento después de la búsqueda de puntos de encuentro

Control: 35.926 pasos. En cada etapa se conserva el candidato válido más corto.
200 programas reproducidos, 35 IDs mejorados. Totales en orden: [35926, 35883, 35882, 35882, 35882, 35882, 35882]. [Detalle](reports/repolish_rkbtk2g8/comparison.csv)

| ID | Entrada | Pos 1 | A4 1 | Ventanas 1 | Pos 2 | A4 2 | Ventanas 2 |
|---|---:|---:|---:|---:|---:|---:|---:|
|practice_0001|153|153|153|153|153|153|153|
|practice_0002|267|267|267|267|267|267|267|
|practice_0003|284|284|284|284|284|284|284|
|practice_0004|129|129|129|129|129|129|129|
|practice_0005|232|232|232|232|232|232|232|
|practice_0006|38|36|36|36|36|36|36|
|practice_0007|105|104|104|104|104|104|104|
|practice_0008|258|258|258|258|258|258|258|
|practice_0009|137|137|137|137|137|137|137|
|practice_0010|139|139|139|139|139|139|139|
|practice_0011|128|128|128|128|128|128|128|
|practice_0012|259|258|258|258|258|258|258|
|practice_0013|215|215|215|215|215|215|215|
|practice_0014|138|138|138|138|138|138|138|
|practice_0015|199|199|199|199|199|199|199|
|practice_0016|229|229|229|229|229|229|229|
|practice_0017|175|175|175|175|175|175|175|
|practice_0018|342|342|342|342|342|342|342|
|practice_0019|207|207|207|207|207|207|207|
|practice_0020|112|111|111|111|111|111|111|
|practice_0021|133|132|132|132|132|132|132|
|practice_0022|143|143|143|143|143|143|143|
|practice_0023|192|192|192|192|192|192|192|
|practice_0024|39|38|37|37|37|37|37|
|practice_0025|182|182|182|182|182|182|182|
|practice_0026|152|152|152|152|152|152|152|
|practice_0027|204|204|204|204|204|204|204|
|practice_0028|240|240|240|240|240|240|240|
|practice_0029|212|211|211|211|211|211|211|
|practice_0030|140|139|139|139|139|139|139|
|practice_0031|226|226|226|226|226|226|226|
|practice_0032|140|140|140|140|140|140|140|
|practice_0033|41|41|41|41|41|41|41|
|practice_0034|318|318|318|318|318|318|318|
|practice_0035|264|264|264|264|264|264|264|
|practice_0036|132|132|132|132|132|132|132|
|practice_0037|293|293|293|293|293|293|293|
|practice_0038|43|43|43|43|43|43|43|
|practice_0039|6|6|6|6|6|6|6|
|practice_0040|198|198|198|198|198|198|198|
|practice_0041|188|188|188|188|188|188|188|
|practice_0042|182|182|182|182|182|182|182|
|practice_0043|122|122|122|122|122|122|122|
|practice_0044|230|230|230|230|230|230|230|
|practice_0045|147|147|147|147|147|147|147|
|practice_0046|271|271|271|271|271|271|271|
|practice_0047|2|2|2|2|2|2|2|
|practice_0048|122|120|120|120|120|120|120|
|practice_0049|129|129|129|129|129|129|129|
|practice_0050|3|3|3|3|3|3|3|
|practice_0051|5|5|5|5|5|5|5|
|practice_0052|208|208|208|208|208|208|208|
|practice_0053|267|267|267|267|267|267|267|
|practice_0054|346|346|346|346|346|346|346|
|practice_0055|349|349|349|349|349|349|349|
|practice_0056|4|4|4|4|4|4|4|
|practice_0057|256|255|255|255|255|255|255|
|practice_0058|267|267|267|267|267|267|267|
|practice_0059|147|147|147|147|147|147|147|
|practice_0060|265|265|265|265|265|265|265|
|practice_0061|282|282|282|282|282|282|282|
|practice_0062|313|313|313|313|313|313|313|
|practice_0063|128|128|128|128|128|128|128|
|practice_0064|255|255|255|255|255|255|255|
|practice_0065|236|236|236|236|236|236|236|
|practice_0066|107|106|106|106|106|106|106|
|practice_0067|171|171|171|171|171|171|171|
|practice_0068|196|196|196|196|196|196|196|
|practice_0069|220|220|220|220|220|220|220|
|practice_0070|270|270|270|270|270|270|270|
|practice_0071|167|167|167|167|167|167|167|
|practice_0072|5|5|5|5|5|5|5|
|practice_0073|159|158|158|158|158|158|158|
|practice_0074|217|216|216|216|216|216|216|
|practice_0075|198|198|198|198|198|198|198|
|practice_0076|332|332|332|332|332|332|332|
|practice_0077|113|111|111|111|111|111|111|
|practice_0078|284|284|284|284|284|284|284|
|practice_0079|137|137|137|137|137|137|137|
|practice_0080|119|119|119|119|119|119|119|
|practice_0081|196|196|196|196|196|196|196|
|practice_0082|239|239|239|239|239|239|239|
|practice_0083|151|151|151|151|151|151|151|
|practice_0084|158|158|158|158|158|158|158|
|practice_0085|304|304|304|304|304|304|304|
|practice_0086|151|151|151|151|151|151|151|
|practice_0087|255|255|255|255|255|255|255|
|practice_0088|232|232|232|232|232|232|232|
|practice_0089|162|162|162|162|162|162|162|
|practice_0090|155|154|154|154|154|154|154|
|practice_0091|223|223|223|223|223|223|223|
|practice_0092|181|181|181|181|181|181|181|
|practice_0093|238|238|238|238|238|238|238|
|practice_0094|141|141|141|141|141|141|141|
|practice_0095|259|259|259|259|259|259|259|
|practice_0096|214|214|214|214|214|214|214|
|practice_0097|131|131|131|131|131|131|131|
|practice_0098|1|1|1|1|1|1|1|
|practice_0099|130|130|130|130|130|130|130|
|practice_0100|341|341|341|341|341|341|341|
|practice_0101|166|166|166|166|166|166|166|
|practice_0102|306|306|306|306|306|306|306|
|practice_0103|5|5|5|5|5|5|5|
|practice_0104|230|230|230|230|230|230|230|
|practice_0105|195|193|193|193|193|193|193|
|practice_0106|179|179|179|179|179|179|179|
|practice_0107|124|124|124|124|124|124|124|
|practice_0108|114|113|113|113|113|113|113|
|practice_0109|147|147|147|147|147|147|147|
|practice_0110|268|268|268|268|268|268|268|
|practice_0111|180|180|180|180|180|180|180|
|practice_0112|128|128|128|128|128|128|128|
|practice_0113|306|306|306|306|306|306|306|
|practice_0114|55|55|55|55|55|55|55|
|practice_0115|189|189|189|189|189|189|189|
|practice_0116|67|67|67|67|67|67|67|
|practice_0117|142|142|142|142|142|142|142|
|practice_0118|40|40|40|40|40|40|40|
|practice_0119|266|266|266|266|266|266|266|
|practice_0120|166|166|166|166|166|166|166|
|practice_0121|280|280|280|280|280|280|280|
|practice_0122|319|319|319|319|319|319|319|
|practice_0123|2|2|2|2|2|2|2|
|practice_0124|224|224|224|224|224|224|224|
|practice_0125|3|3|3|3|3|3|3|
|practice_0126|155|154|154|154|154|154|154|
|practice_0127|196|196|196|196|196|196|196|
|practice_0128|2|2|2|2|2|2|2|
|practice_0129|124|124|124|124|124|124|124|
|practice_0130|110|110|110|110|110|110|110|
|practice_0131|226|226|226|226|226|226|226|
|practice_0132|179|179|179|179|179|179|179|
|practice_0133|187|187|187|187|187|187|187|
|practice_0134|215|215|215|215|215|215|215|
|practice_0135|344|344|344|344|344|344|344|
|practice_0136|53|53|53|53|53|53|53|
|practice_0137|105|104|104|104|104|104|104|
|practice_0138|345|345|345|345|345|345|345|
|practice_0139|218|218|218|218|218|218|218|
|practice_0140|1|1|1|1|1|1|1|
|practice_0141|155|154|154|154|154|154|154|
|practice_0142|264|264|264|264|264|264|264|
|practice_0143|153|152|152|152|152|152|152|
|practice_0144|113|113|113|113|113|113|113|
|practice_0145|199|198|198|198|198|198|198|
|practice_0146|44|44|44|44|44|44|44|
|practice_0147|188|187|187|187|187|187|187|
|practice_0148|176|176|176|176|176|176|176|
|practice_0149|201|201|201|201|201|201|201|
|practice_0150|203|203|203|203|203|203|203|
|practice_0151|333|333|333|333|333|333|333|
|practice_0152|161|161|161|161|161|161|161|
|practice_0153|146|144|144|144|144|144|144|
|practice_0154|336|335|335|335|335|335|335|
|practice_0155|158|158|158|158|158|158|158|
|practice_0156|109|109|109|109|109|109|109|
|practice_0157|138|137|137|137|137|137|137|
|practice_0158|235|235|235|235|235|235|235|
|practice_0159|34|34|34|34|34|34|34|
|practice_0160|186|185|185|185|185|185|185|
|practice_0161|274|274|274|274|274|274|274|
|practice_0162|285|285|285|285|285|285|285|
|practice_0163|263|262|262|262|262|262|262|
|practice_0164|254|254|254|254|254|254|254|
|practice_0165|239|238|238|238|238|238|238|
|practice_0166|51|48|48|48|48|48|48|
|practice_0167|300|300|300|300|300|300|300|
|practice_0168|128|128|128|128|128|128|128|
|practice_0169|4|4|4|4|4|4|4|
|practice_0170|36|36|36|36|36|36|36|
|practice_0171|292|292|292|292|292|292|292|
|practice_0172|2|2|2|2|2|2|2|
|practice_0173|147|147|147|147|147|147|147|
|practice_0174|2|2|2|2|2|2|2|
|practice_0175|4|4|4|4|4|4|4|
|practice_0176|317|317|317|317|317|317|317|
|practice_0177|178|178|178|178|178|178|178|
|practice_0178|92|92|92|92|92|92|92|
|practice_0179|342|342|342|342|342|342|342|
|practice_0180|154|154|154|154|154|154|154|
|practice_0181|281|281|281|281|281|281|281|
|practice_0182|220|220|220|220|220|220|220|
|practice_0183|286|286|286|286|286|286|286|
|practice_0184|254|254|254|254|254|254|254|
|practice_0185|171|170|170|170|170|170|170|
|practice_0186|118|117|117|117|117|117|117|
|practice_0187|218|218|218|218|218|218|218|
|practice_0188|224|224|224|224|224|224|224|
|practice_0189|184|184|184|184|184|184|184|
|practice_0190|295|295|295|295|295|295|295|
|practice_0191|313|313|313|313|313|313|313|
|practice_0192|119|119|119|119|119|119|119|
|practice_0193|133|132|132|132|132|132|132|
|practice_0194|249|249|249|249|249|249|249|
|practice_0195|262|262|262|262|262|262|262|
|practice_0196|137|137|137|137|137|137|137|
|practice_0197|207|207|207|207|207|207|207|
|practice_0198|43|41|41|41|41|41|41|
|practice_0199|291|291|291|291|291|291|291|
|practice_0200|139|138|138|138|138|138|138|

## Ciclo extensivo y pulido sobre los 1.000 puntuables

1000 IDs validados; 176594 pasos; 400 IDs mejorados. [Detalle](reports/final_cycle_sppbgmwl/comparison.csv) · [Entrega](artifacts/submission_cycle_176594.csv)

Histórico: 177352 · Pulido_directo: 177152 · Greedy_E4: 176862 · Caminos_E4: 176842 · Puntos_de_encuentro: 176779 · Pulido_sin_puntos: 176663 · Pos1: 176599 · A4_1: 176596 · Ventanas1: 176596 · Pos2: 176596 · A4_2: 176596 · Ventanas2: 176596 · Final: 176594

|ID|Histórico|Pulido_directo|Greedy_E4|Caminos_E4|Puntos_de_encuentro|Pulido_sin_puntos|Pos1|A4_1|Ventanas1|Pos2|A4_2|Ventanas2|Final|
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
|scored_0001|289|289|289|289|289|289|289|289|289|289|289|289|289|
|scored_0002|304|304|304|304|304|304|304|304|304|304|304|304|304|
|scored_0003|265|265|265|265|265|265|265|265|265|265|265|265|265|
|scored_0004|268|268|268|268|268|268|268|268|268|268|268|268|268|
|scored_0005|69|69|67|67|67|67|67|67|67|67|67|67|67|
|scored_0006|174|174|174|174|174|174|174|174|174|174|174|174|174|
|scored_0007|222|222|221|221|221|221|221|221|221|221|221|221|221|
|scored_0008|163|162|158|157|157|156|156|156|156|156|156|156|156|
|scored_0009|48|48|48|48|48|48|48|48|48|48|48|48|48|
|scored_0010|224|224|223|223|223|223|223|223|223|223|223|223|223|
|scored_0011|175|175|175|175|175|175|175|175|175|175|175|175|175|
|scored_0012|226|226|226|226|226|226|226|226|226|226|226|226|226|
|scored_0013|217|217|217|217|217|217|217|217|217|217|217|217|217|
|scored_0014|39|39|39|39|39|39|39|39|39|39|39|39|39|
|scored_0015|276|275|276|276|276|275|275|275|275|275|275|275|275|
|scored_0016|203|203|202|202|202|202|202|202|202|202|202|202|202|
|scored_0017|161|161|161|161|161|161|161|161|161|161|161|161|161|
|scored_0018|165|165|165|165|165|165|165|165|165|165|165|165|165|
|scored_0019|36|36|36|36|36|36|36|36|36|36|36|36|36|
|scored_0020|301|301|301|301|301|301|301|301|301|301|301|301|301|
|scored_0021|54|54|53|53|52|52|51|51|51|51|51|51|51|
|scored_0022|155|155|155|155|155|155|155|155|155|155|155|155|155|
|scored_0023|201|201|201|201|201|201|201|201|201|201|201|201|201|
|scored_0024|42|42|40|40|40|40|40|40|40|40|40|40|40|
|scored_0025|177|177|175|175|175|175|175|175|175|175|175|175|175|
|scored_0026|227|227|227|227|227|227|227|227|227|227|227|227|227|
|scored_0027|138|138|137|137|137|136|136|136|136|136|136|136|136|
|scored_0028|188|188|184|184|184|183|183|183|183|183|183|183|183|
|scored_0029|50|50|49|49|49|48|48|48|48|48|48|48|48|
|scored_0030|250|250|249|249|249|249|249|249|249|249|249|249|249|
|scored_0031|301|301|301|301|301|301|301|301|301|301|301|301|301|
|scored_0032|136|136|132|132|132|132|132|132|132|132|132|132|132|
|scored_0033|260|260|260|260|260|260|260|260|260|260|260|260|260|
|scored_0034|214|214|214|214|214|214|214|214|214|214|214|214|214|
|scored_0035|138|137|137|137|135|137|135|135|135|135|135|135|135|
|scored_0036|295|295|295|295|295|295|295|295|295|295|295|295|295|
|scored_0037|251|251|251|251|251|251|251|251|251|251|251|251|251|
|scored_0038|179|179|179|179|179|179|179|179|179|179|179|179|179|
|scored_0039|138|138|136|136|136|136|136|136|136|136|136|136|136|
|scored_0040|217|217|217|217|217|217|217|217|217|217|217|217|217|
|scored_0041|182|182|181|181|181|181|181|181|181|181|181|181|181|
|scored_0042|202|202|202|202|202|202|202|202|202|202|202|202|202|
|scored_0043|267|267|267|267|267|267|267|267|267|267|267|267|267|
|scored_0044|102|102|102|102|102|102|102|102|102|102|102|102|102|
|scored_0045|201|201|201|201|201|201|201|201|201|201|201|201|201|
|scored_0046|39|39|39|39|39|39|39|39|39|39|39|39|39|
|scored_0047|208|208|208|208|208|208|208|208|208|208|208|208|208|
|scored_0048|183|183|183|183|183|183|183|183|183|183|183|183|183|
|scored_0049|193|193|193|193|193|193|193|193|193|193|193|193|193|
|scored_0050|36|35|35|35|34|35|34|34|34|34|34|34|34|
|scored_0051|352|352|352|352|352|352|352|352|352|352|352|352|352|
|scored_0052|139|139|139|139|138|139|137|137|137|137|137|137|137|
|scored_0053|193|193|193|193|193|193|193|193|193|193|193|193|193|
|scored_0054|242|242|242|242|242|242|242|242|242|242|242|242|242|
|scored_0055|100|100|99|99|99|99|99|99|99|99|99|99|99|
|scored_0056|36|36|36|36|36|36|36|36|36|36|36|36|36|
|scored_0057|270|270|270|270|270|270|270|270|270|270|270|270|270|
|scored_0058|333|333|333|333|333|333|333|333|333|333|333|333|333|
|scored_0059|183|183|183|183|183|183|183|183|183|183|183|183|183|
|scored_0060|317|317|316|316|316|316|316|316|316|316|316|316|316|
|scored_0061|4|4|4|4|4|4|4|4|4|4|4|4|4|
|scored_0062|182|182|182|182|182|182|182|182|182|182|182|182|182|
|scored_0063|154|154|154|154|154|154|154|154|154|154|154|154|154|
|scored_0064|168|168|167|167|167|167|167|167|167|167|167|167|167|
|scored_0065|48|47|48|48|48|47|47|47|47|47|47|47|47|
|scored_0066|52|52|52|52|52|52|52|52|52|52|52|52|52|
|scored_0067|321|321|321|321|321|321|321|321|321|321|321|321|321|
|scored_0068|194|194|194|194|194|194|194|194|194|194|194|194|194|
|scored_0069|353|353|353|353|353|353|353|353|353|353|353|353|353|
|scored_0070|184|183|184|184|184|183|183|183|183|183|183|183|183|
|scored_0071|146|146|145|145|145|145|145|145|145|145|145|145|145|
|scored_0072|161|161|161|161|161|161|161|161|161|161|161|161|161|
|scored_0073|156|155|156|156|156|155|155|155|155|155|155|155|155|
|scored_0074|259|259|259|259|259|259|259|259|259|259|259|259|259|
|scored_0075|318|318|318|318|318|318|318|318|318|318|318|318|318|
|scored_0076|176|176|174|174|174|174|174|174|174|174|174|174|174|
|scored_0077|135|135|134|134|133|134|133|133|133|133|133|133|133|
|scored_0078|130|130|129|129|129|129|129|129|129|129|129|129|129|
|scored_0079|191|191|191|191|191|191|191|191|191|191|191|191|191|
|scored_0080|184|183|184|184|184|183|183|183|183|183|183|183|183|
|scored_0081|148|148|148|148|148|148|148|148|148|148|148|148|148|
|scored_0082|156|156|154|154|154|154|154|154|154|154|154|154|154|
|scored_0083|219|219|219|219|219|219|219|219|219|219|219|219|219|
|scored_0084|50|50|50|50|50|50|50|50|50|50|50|50|50|
|scored_0085|116|116|116|116|116|116|116|116|116|116|116|116|116|
|scored_0086|132|132|132|132|132|132|132|132|132|132|132|132|132|
|scored_0087|304|304|304|304|304|304|304|304|304|304|304|304|304|
|scored_0088|264|264|264|264|264|264|264|264|264|264|264|264|264|
|scored_0089|153|153|152|152|152|152|152|152|152|152|152|152|152|
|scored_0090|238|238|238|238|238|238|238|238|238|238|238|238|238|
|scored_0091|40|40|40|40|40|40|40|40|40|40|40|40|40|
|scored_0092|54|53|54|54|54|53|53|53|53|53|53|53|53|
|scored_0093|138|137|136|136|136|135|135|135|135|135|135|135|135|
|scored_0094|313|313|313|313|313|313|313|313|313|313|313|313|313|
|scored_0095|44|44|44|44|44|44|44|44|44|44|44|44|44|
|scored_0096|221|221|220|220|220|220|220|220|220|220|220|220|220|
|scored_0097|125|125|125|125|125|125|125|125|125|125|125|125|125|
|scored_0098|233|233|233|233|233|233|233|233|233|233|233|233|233|
|scored_0099|191|191|190|190|190|190|190|190|190|190|190|190|190|
|scored_0100|185|185|185|185|185|185|185|185|185|185|185|185|185|
|scored_0101|153|153|152|152|152|152|152|152|152|152|152|152|152|
|scored_0102|190|190|190|190|190|190|190|190|190|190|190|190|190|
|scored_0103|59|59|59|59|59|59|59|59|59|59|59|59|59|
|scored_0104|286|286|286|286|286|286|286|286|286|286|286|286|286|
|scored_0105|188|188|188|188|188|188|188|188|188|188|188|188|188|
|scored_0106|289|289|288|288|288|288|288|288|288|288|288|288|288|
|scored_0107|300|300|300|300|300|300|300|300|300|300|300|300|300|
|scored_0108|228|228|228|228|228|228|228|228|228|228|228|228|228|
|scored_0109|177|177|177|177|177|177|177|177|177|177|177|177|177|
|scored_0110|213|213|213|213|213|213|213|213|213|213|213|213|213|
|scored_0111|266|266|266|266|266|266|266|266|266|266|266|266|266|
|scored_0112|262|262|262|262|262|262|262|262|262|262|262|262|262|
|scored_0113|38|38|36|35|35|34|34|34|34|34|34|34|34|
|scored_0114|211|211|211|211|211|211|211|211|211|211|211|211|211|
|scored_0115|254|254|254|254|254|254|254|254|254|254|254|254|254|
|scored_0116|110|109|110|110|110|109|109|109|109|109|109|109|109|
|scored_0117|242|242|241|241|241|241|241|241|241|241|241|241|241|
|scored_0118|131|131|131|131|131|131|131|131|131|131|131|131|131|
|scored_0119|287|287|287|287|287|287|287|287|287|287|287|287|287|
|scored_0120|247|247|247|247|247|247|247|247|247|247|247|247|247|
|scored_0121|64|64|64|64|64|64|64|64|64|64|64|64|64|
|scored_0122|199|199|198|198|198|198|198|198|198|198|198|198|198|
|scored_0123|269|269|269|269|269|269|269|269|269|269|269|269|269|
|scored_0124|35|35|35|35|35|35|35|35|35|35|35|35|35|
|scored_0125|148|148|145|145|145|145|145|145|145|145|145|145|145|
|scored_0126|4|4|4|4|4|4|4|4|4|4|4|4|4|
|scored_0127|146|144|143|142|141|140|141|141|141|141|141|141|140|
|scored_0128|160|160|160|160|160|160|160|160|160|160|160|160|160|
|scored_0129|340|340|340|340|340|340|340|340|340|340|340|340|340|
|scored_0130|35|35|33|33|33|33|33|33|33|33|33|33|33|
|scored_0131|121|121|118|118|118|118|118|118|118|118|118|118|118|
|scored_0132|133|133|133|133|133|133|133|133|133|133|133|133|133|
|scored_0133|289|289|288|288|288|288|288|288|288|288|288|288|288|
|scored_0134|43|43|43|42|42|41|41|41|41|41|41|41|41|
|scored_0135|119|119|118|118|118|118|118|118|118|118|118|118|118|
|scored_0136|157|155|156|155|155|155|155|155|155|155|155|155|155|
|scored_0137|72|70|72|72|72|70|70|70|70|70|70|70|70|
|scored_0138|119|119|118|118|118|118|118|118|118|118|118|118|118|
|scored_0139|228|228|228|228|228|228|228|228|228|228|228|228|228|
|scored_0140|199|199|198|198|198|198|198|198|198|198|198|198|198|
|scored_0141|167|167|167|167|167|167|167|167|167|167|167|167|167|
|scored_0142|148|148|148|148|148|148|148|148|148|148|148|148|148|
|scored_0143|188|187|185|185|185|184|184|184|184|184|184|184|184|
|scored_0144|141|141|141|141|141|141|141|141|141|141|141|141|141|
|scored_0145|269|269|269|269|269|269|269|269|269|269|269|269|269|
|scored_0146|327|327|327|327|327|327|327|327|327|327|327|327|327|
|scored_0147|119|119|119|119|119|119|119|119|119|119|119|119|119|
|scored_0148|217|217|217|217|217|217|217|217|217|217|217|217|217|
|scored_0149|26|26|26|26|26|26|26|26|26|26|26|26|26|
|scored_0150|205|204|205|205|205|204|204|204|204|204|204|204|204|
|scored_0151|43|43|43|43|43|43|43|43|43|43|43|43|43|
|scored_0152|154|154|154|154|154|154|154|154|154|154|154|154|154|
|scored_0153|138|137|137|137|137|137|137|137|137|137|137|137|137|
|scored_0154|206|206|206|206|206|206|206|206|206|206|206|206|206|
|scored_0155|282|281|282|282|282|281|281|281|281|281|281|281|281|
|scored_0156|171|171|171|171|171|171|171|171|171|171|171|171|171|
|scored_0157|198|198|198|198|198|198|198|198|198|198|198|198|198|
|scored_0158|192|192|190|190|190|190|190|190|190|190|190|190|190|
|scored_0159|285|285|284|284|284|284|284|284|284|284|284|284|284|
|scored_0160|229|229|229|229|229|229|229|229|229|229|229|229|229|
|scored_0161|196|196|196|196|196|196|196|196|196|196|196|196|196|
|scored_0162|229|229|229|229|229|229|229|229|229|229|229|229|229|
|scored_0163|312|312|312|312|312|312|312|312|312|312|312|312|312|
|scored_0164|161|161|161|161|161|161|161|161|161|161|161|161|161|
|scored_0165|302|302|302|302|302|302|302|302|302|302|302|302|302|
|scored_0166|257|257|257|257|257|257|257|257|257|257|257|257|257|
|scored_0167|310|310|310|310|310|310|310|310|310|310|310|310|310|
|scored_0168|337|337|337|337|337|337|337|337|337|337|337|337|337|
|scored_0169|131|131|129|129|129|129|129|129|129|129|129|129|129|
|scored_0170|283|283|283|283|283|283|283|283|283|283|283|283|283|
|scored_0171|159|159|159|159|159|159|159|159|159|159|159|159|159|
|scored_0172|204|204|204|204|204|204|204|204|204|204|204|204|204|
|scored_0173|234|234|234|234|234|234|234|234|234|234|234|234|234|
|scored_0174|175|175|175|175|175|175|175|175|175|175|175|175|175|
|scored_0175|133|133|131|131|131|131|131|131|131|131|131|131|131|
|scored_0176|144|144|144|144|144|144|144|144|144|144|144|144|144|
|scored_0177|171|171|169|169|169|169|169|169|169|169|169|169|169|
|scored_0178|136|136|135|135|134|135|134|134|134|134|134|134|134|
|scored_0179|262|262|262|262|262|262|262|262|262|262|262|262|262|
|scored_0180|112|112|112|111|109|111|108|108|108|108|108|108|108|
|scored_0181|39|39|39|39|39|39|39|39|39|39|39|39|39|
|scored_0182|204|204|204|204|204|204|204|204|204|204|204|204|204|
|scored_0183|352|352|352|352|352|352|352|352|352|352|352|352|352|
|scored_0184|173|173|172|172|172|172|172|172|172|172|172|172|172|
|scored_0185|224|224|224|224|224|224|224|224|224|224|224|224|224|
|scored_0186|168|168|167|167|166|167|166|166|166|166|166|166|166|
|scored_0187|121|121|120|120|119|120|119|119|119|119|119|119|119|
|scored_0188|124|124|123|123|123|123|123|123|123|123|123|123|123|
|scored_0189|197|197|197|197|197|197|197|197|197|197|197|197|197|
|scored_0190|154|154|154|154|154|154|154|154|154|154|154|154|154|
|scored_0191|160|158|159|159|159|159|159|159|159|159|159|159|158|
|scored_0192|182|182|181|181|181|181|181|181|181|181|181|181|181|
|scored_0193|153|152|153|153|153|152|152|152|152|152|152|152|152|
|scored_0194|188|188|188|188|188|188|188|188|188|188|188|188|188|
|scored_0195|155|154|155|155|155|154|154|154|154|154|154|154|154|
|scored_0196|40|40|39|39|39|39|39|39|39|39|39|39|39|
|scored_0197|155|155|154|154|154|154|154|154|154|154|154|154|154|
|scored_0198|269|269|269|269|269|269|269|269|269|269|269|269|269|
|scored_0199|117|117|117|117|117|117|117|117|117|117|117|117|117|
|scored_0200|173|172|169|169|169|167|167|167|167|167|167|167|167|
|scored_0201|194|194|194|194|194|194|194|194|194|194|194|194|194|
|scored_0202|148|148|146|146|146|146|146|146|146|146|146|146|146|
|scored_0203|275|275|275|275|275|275|275|275|275|275|275|275|275|
|scored_0204|271|271|271|271|271|271|271|271|271|271|271|271|271|
|scored_0205|166|165|165|165|165|164|164|164|164|164|164|164|164|
|scored_0206|198|198|198|198|198|198|198|198|198|198|198|198|198|
|scored_0207|276|276|276|276|276|276|276|276|276|276|276|276|276|
|scored_0208|285|285|285|285|285|285|285|285|285|285|285|285|285|
|scored_0209|281|280|281|281|281|280|280|280|280|280|280|280|280|
|scored_0210|121|121|121|120|119|120|119|119|119|119|119|119|119|
|scored_0211|52|52|51|51|51|51|51|51|51|51|51|51|51|
|scored_0212|25|25|25|25|25|25|25|25|25|25|25|25|25|
|scored_0213|146|146|146|146|146|146|146|146|146|146|146|146|146|
|scored_0214|213|213|213|213|213|213|213|213|213|213|213|213|213|
|scored_0215|203|203|203|203|203|203|203|203|203|203|203|203|203|
|scored_0216|189|189|189|189|189|189|189|189|189|189|189|189|189|
|scored_0217|301|301|300|300|300|300|300|300|300|300|300|300|300|
|scored_0218|348|348|348|348|348|348|348|348|348|348|348|348|348|
|scored_0219|151|151|149|149|149|149|149|149|149|149|149|149|149|
|scored_0220|140|140|140|140|140|140|140|140|140|140|140|140|140|
|scored_0221|51|51|51|51|51|51|51|51|51|51|51|51|51|
|scored_0222|49|49|49|49|49|49|49|49|49|49|49|49|49|
|scored_0223|263|263|262|262|262|262|262|262|262|262|262|262|262|
|scored_0224|225|225|225|225|225|225|225|225|225|225|225|225|225|
|scored_0225|132|132|131|131|129|131|129|129|129|129|129|129|129|
|scored_0226|154|154|154|154|154|154|154|154|154|154|154|154|154|
|scored_0227|189|189|185|185|185|185|185|185|185|185|185|185|185|
|scored_0228|146|145|144|144|144|144|144|144|144|144|144|144|144|
|scored_0229|180|179|180|180|180|179|179|179|179|179|179|179|179|
|scored_0230|266|266|266|266|266|266|266|266|266|266|266|266|266|
|scored_0231|221|221|221|221|221|221|221|221|221|221|221|221|221|
|scored_0232|203|203|203|203|203|203|203|203|203|203|203|203|203|
|scored_0233|49|49|49|49|49|49|49|49|49|49|49|49|49|
|scored_0234|227|227|227|227|227|227|227|227|227|227|227|227|227|
|scored_0235|264|264|264|264|264|264|264|264|264|264|264|264|264|
|scored_0236|136|136|135|135|135|135|135|135|135|135|135|135|135|
|scored_0237|44|44|42|42|42|42|42|42|42|42|42|42|42|
|scored_0238|183|183|183|183|183|183|183|183|183|183|183|183|183|
|scored_0239|212|212|212|212|212|212|212|212|212|212|212|212|212|
|scored_0240|153|153|152|152|152|152|152|152|152|152|152|152|152|
|scored_0241|40|39|40|40|40|39|39|39|39|39|39|39|39|
|scored_0242|201|201|201|201|201|201|201|201|201|201|201|201|201|
|scored_0243|151|151|149|148|148|148|148|148|148|148|148|148|148|
|scored_0244|233|232|233|233|233|232|232|232|232|232|232|232|232|
|scored_0245|143|142|142|142|142|142|142|142|142|142|142|142|142|
|scored_0246|167|167|167|167|167|167|167|167|167|167|167|167|167|
|scored_0247|176|176|174|173|173|173|173|173|173|173|173|173|173|
|scored_0248|217|217|217|217|217|217|217|217|217|217|217|217|217|
|scored_0249|323|322|323|323|323|322|322|322|322|322|322|322|322|
|scored_0250|31|31|31|31|31|31|31|31|31|31|31|31|31|
|scored_0251|267|267|267|267|267|267|267|267|267|267|267|267|267|
|scored_0252|89|88|89|89|89|88|88|88|88|88|88|88|88|
|scored_0253|102|98|101|101|101|97|97|97|97|97|97|97|97|
|scored_0254|286|286|286|286|286|286|286|286|286|286|286|286|286|
|scored_0255|346|346|346|346|346|346|346|346|346|346|346|346|346|
|scored_0256|112|112|112|112|112|112|112|112|112|112|112|112|112|
|scored_0257|241|241|241|241|241|241|241|241|241|241|241|241|241|
|scored_0258|342|342|342|342|342|342|342|342|342|342|342|342|342|
|scored_0259|136|134|133|133|133|133|133|133|133|133|133|133|133|
|scored_0260|119|118|118|118|113|117|112|112|112|112|112|112|112|
|scored_0261|213|213|213|213|213|213|213|213|213|213|213|213|213|
|scored_0262|277|277|277|277|277|277|277|277|277|277|277|277|277|
|scored_0263|299|299|299|299|299|299|299|299|299|299|299|299|299|
|scored_0264|269|269|269|269|269|269|269|269|269|269|269|269|269|
|scored_0265|234|234|233|233|233|233|233|233|233|233|233|233|233|
|scored_0266|142|142|142|142|142|142|142|142|142|142|142|142|142|
|scored_0267|149|148|144|144|144|143|143|143|143|143|143|143|143|
|scored_0268|260|260|260|260|260|260|260|260|260|260|260|260|260|
|scored_0269|347|347|347|347|347|347|347|347|347|347|347|347|347|
|scored_0270|157|157|154|154|152|154|152|152|152|152|152|152|152|
|scored_0271|57|56|57|57|57|56|56|56|56|56|56|56|56|
|scored_0272|32|32|32|32|32|32|32|32|32|32|32|32|32|
|scored_0273|230|230|230|230|230|230|230|230|230|230|230|230|230|
|scored_0274|208|207|208|208|208|207|207|207|207|207|207|207|207|
|scored_0275|184|184|183|183|183|183|183|183|183|183|183|183|183|
|scored_0276|91|90|91|91|91|90|90|90|90|90|90|90|90|
|scored_0277|171|171|170|170|170|170|170|170|170|170|170|170|170|
|scored_0278|104|104|102|102|101|102|101|101|101|101|101|101|101|
|scored_0279|320|320|320|320|320|320|320|320|320|320|320|320|320|
|scored_0280|186|186|184|184|184|184|184|184|184|184|184|184|184|
|scored_0281|230|230|230|230|230|230|230|230|230|230|230|230|230|
|scored_0282|60|60|60|60|60|60|60|60|60|60|60|60|60|
|scored_0283|270|269|270|270|270|269|269|269|269|269|269|269|269|
|scored_0284|176|176|176|176|176|176|176|176|176|176|176|176|176|
|scored_0285|256|256|256|256|256|256|256|256|256|256|256|256|256|
|scored_0286|159|157|157|157|157|155|155|155|155|155|155|155|155|
|scored_0287|243|243|243|243|243|243|243|243|243|243|243|243|243|
|scored_0288|168|167|168|168|168|167|167|167|167|167|167|167|167|
|scored_0289|24|24|24|24|24|24|24|24|24|24|24|24|24|
|scored_0290|181|181|181|181|181|181|181|181|181|181|181|181|181|
|scored_0291|204|204|203|203|203|203|203|203|203|203|203|203|203|
|scored_0292|66|66|66|66|66|66|66|66|66|66|66|66|66|
|scored_0293|245|245|245|245|245|245|245|245|245|245|245|245|245|
|scored_0294|195|194|195|195|195|194|194|194|194|194|194|194|194|
|scored_0295|163|163|163|163|163|163|163|163|163|163|163|163|163|
|scored_0296|54|54|54|54|54|54|54|54|54|54|54|54|54|
|scored_0297|155|154|154|154|154|153|153|153|153|153|153|153|153|
|scored_0298|127|127|126|126|126|126|126|126|126|126|126|126|126|
|scored_0299|117|116|117|117|117|116|116|116|116|116|116|116|116|
|scored_0300|56|56|55|55|55|54|54|54|54|54|54|54|54|
|scored_0301|4|4|4|4|4|4|4|4|4|4|4|4|4|
|scored_0302|200|200|200|200|200|200|200|200|200|200|200|200|200|
|scored_0303|255|255|255|255|255|255|255|255|255|255|255|255|255|
|scored_0304|196|196|196|196|196|196|196|196|196|196|196|196|196|
|scored_0305|205|204|204|204|204|204|204|204|204|204|204|204|204|
|scored_0306|328|328|328|328|328|328|328|328|328|328|328|328|328|
|scored_0307|163|163|163|163|163|163|163|163|163|163|163|163|163|
|scored_0308|42|42|42|42|42|42|42|42|42|42|42|42|42|
|scored_0309|162|162|162|162|162|162|162|162|162|162|162|162|162|
|scored_0310|145|145|145|145|145|145|145|145|145|145|145|145|145|
|scored_0311|136|135|135|135|135|134|134|134|134|134|134|134|134|
|scored_0312|307|307|307|307|307|307|307|307|307|307|307|307|307|
|scored_0313|212|212|212|212|212|212|212|212|212|212|212|212|212|
|scored_0314|243|243|243|243|243|243|243|243|243|243|243|243|243|
|scored_0315|283|283|283|283|283|283|283|283|283|283|283|283|283|
|scored_0316|141|141|141|141|141|141|141|141|141|141|141|141|141|
|scored_0317|224|224|224|224|224|224|224|224|224|224|224|224|224|
|scored_0318|226|226|226|226|226|226|226|226|226|226|226|226|226|
|scored_0319|282|282|282|282|281|282|281|281|281|281|281|281|281|
|scored_0320|40|40|40|40|40|40|40|40|40|40|40|40|40|
|scored_0321|340|340|340|340|340|340|340|340|340|340|340|340|340|
|scored_0322|220|220|219|219|219|219|219|219|219|219|219|219|219|
|scored_0323|184|184|183|183|183|183|183|183|183|183|183|183|183|
|scored_0324|223|223|223|223|223|223|223|223|223|223|223|223|223|
|scored_0325|155|154|154|154|154|154|154|154|154|154|154|154|154|
|scored_0326|180|180|176|176|176|176|176|176|176|176|176|176|176|
|scored_0327|164|164|164|164|164|164|164|164|164|164|164|164|164|
|scored_0328|286|286|286|286|286|286|286|286|286|286|286|286|286|
|scored_0329|156|156|156|156|156|156|156|156|156|156|156|156|156|
|scored_0330|197|197|194|194|194|194|194|194|194|194|194|194|194|
|scored_0331|309|309|309|309|309|309|309|309|309|309|309|309|309|
|scored_0332|181|181|181|181|181|181|181|181|181|181|181|181|181|
|scored_0333|265|264|265|265|265|264|264|264|264|264|264|264|264|
|scored_0334|285|285|285|285|285|285|285|285|285|285|285|285|285|
|scored_0335|232|232|232|232|232|232|232|232|232|232|232|232|232|
|scored_0336|190|189|190|190|190|189|189|189|189|189|189|189|189|
|scored_0337|150|150|150|150|150|150|150|150|150|150|150|150|150|
|scored_0338|221|221|221|221|221|221|221|221|221|221|221|221|221|
|scored_0339|113|113|113|113|112|113|112|112|112|112|112|112|112|
|scored_0340|146|145|145|145|145|144|144|144|144|144|144|144|144|
|scored_0341|173|173|173|173|173|173|173|173|173|173|173|173|173|
|scored_0342|270|270|269|269|269|269|269|269|269|269|269|269|269|
|scored_0343|145|143|145|145|145|143|143|143|143|143|143|143|143|
|scored_0344|202|202|202|202|202|202|202|202|202|202|202|202|202|
|scored_0345|245|245|245|245|245|245|245|245|245|245|245|245|245|
|scored_0346|213|213|213|213|213|213|213|213|213|213|213|213|213|
|scored_0347|141|141|141|141|141|141|141|141|141|141|141|141|141|
|scored_0348|3|3|3|3|3|3|3|3|3|3|3|3|3|
|scored_0349|235|235|235|235|235|235|235|235|235|235|235|235|235|
|scored_0350|74|73|74|74|74|73|73|73|73|73|73|73|73|
|scored_0351|201|201|200|200|199|200|198|198|198|198|198|198|198|
|scored_0352|288|288|288|288|288|288|288|288|288|288|288|288|288|
|scored_0353|182|181|182|182|182|181|181|181|181|181|181|181|181|
|scored_0354|182|182|180|180|180|180|180|180|180|180|180|180|180|
|scored_0355|148|148|147|147|147|147|147|147|147|147|147|147|147|
|scored_0356|104|104|102|102|102|102|102|102|102|102|102|102|102|
|scored_0357|285|285|285|285|285|285|285|285|285|285|285|285|285|
|scored_0358|182|180|181|181|181|180|180|180|180|180|180|180|180|
|scored_0359|145|144|143|143|143|142|142|142|142|142|142|142|142|
|scored_0360|161|161|160|160|160|160|160|160|160|160|160|160|160|
|scored_0361|224|224|222|222|222|222|222|222|222|222|222|222|222|
|scored_0362|140|139|140|140|140|139|139|139|139|139|139|139|139|
|scored_0363|205|205|205|205|205|205|205|205|205|205|205|205|205|
|scored_0364|217|217|217|217|217|217|217|217|217|217|217|217|217|
|scored_0365|4|4|4|4|4|4|4|4|4|4|4|4|4|
|scored_0366|148|148|145|145|145|145|145|145|145|145|145|145|145|
|scored_0367|150|150|149|149|149|149|149|149|149|149|149|149|149|
|scored_0368|39|39|39|39|35|39|34|34|34|34|34|34|34|
|scored_0369|294|294|294|294|294|294|294|294|294|294|294|294|294|
|scored_0370|327|327|327|327|327|327|327|327|327|327|327|327|327|
|scored_0371|160|159|158|158|158|157|157|157|157|157|157|157|157|
|scored_0372|141|141|141|141|141|141|141|141|141|141|141|141|141|
|scored_0373|275|275|273|273|273|273|273|273|273|273|273|273|273|
|scored_0374|274|273|274|274|274|273|273|273|273|273|273|273|273|
|scored_0375|263|263|263|263|263|263|263|263|263|263|263|263|263|
|scored_0376|33|33|33|33|33|33|33|33|33|33|33|33|33|
|scored_0377|105|105|104|104|104|104|104|104|104|104|104|104|104|
|scored_0378|105|104|105|105|105|104|104|104|104|104|104|104|104|
|scored_0379|148|148|145|145|145|145|145|145|145|145|145|145|145|
|scored_0380|195|195|195|195|195|195|195|195|195|195|195|195|195|
|scored_0381|245|244|245|245|244|244|243|243|243|243|243|243|243|
|scored_0382|178|178|178|178|178|178|178|178|178|178|178|178|178|
|scored_0383|183|183|182|182|182|182|182|182|182|182|182|182|182|
|scored_0384|180|180|180|180|180|180|180|180|180|180|180|180|180|
|scored_0385|188|187|188|188|188|187|187|187|187|187|187|187|187|
|scored_0386|240|240|240|240|240|240|240|240|240|240|240|240|240|
|scored_0387|37|37|34|32|32|32|32|32|32|32|32|32|32|
|scored_0388|30|30|30|30|30|30|30|30|30|30|30|30|30|
|scored_0389|219|219|219|219|219|219|219|219|219|219|219|219|219|
|scored_0390|130|127|128|128|128|125|125|125|125|125|125|125|125|
|scored_0391|270|270|269|269|269|269|269|269|269|269|269|269|269|
|scored_0392|144|144|143|143|143|142|142|142|142|142|142|142|142|
|scored_0393|34|34|34|34|34|34|34|34|34|34|34|34|34|
|scored_0394|172|172|171|171|171|171|171|171|171|171|171|171|171|
|scored_0395|115|114|114|114|113|113|112|112|112|112|112|112|112|
|scored_0396|48|48|48|48|48|48|48|48|48|48|48|48|48|
|scored_0397|240|240|239|239|239|239|239|239|239|239|239|239|239|
|scored_0398|254|254|254|254|254|254|254|254|254|254|254|254|254|
|scored_0399|28|28|28|28|28|28|28|28|28|28|28|28|28|
|scored_0400|284|284|284|284|284|284|284|284|284|284|284|284|284|
|scored_0401|38|38|38|38|38|38|38|38|38|38|38|38|38|
|scored_0402|120|118|119|119|119|118|118|118|118|118|118|118|118|
|scored_0403|292|292|291|291|291|291|291|291|291|291|291|291|291|
|scored_0404|130|130|130|130|130|130|130|130|130|130|130|130|130|
|scored_0405|336|336|336|336|336|336|336|336|336|336|336|336|336|
|scored_0406|172|172|172|172|172|172|172|172|172|172|172|172|172|
|scored_0407|198|198|198|198|198|198|198|198|198|198|198|198|198|
|scored_0408|207|206|207|207|207|206|206|206|206|206|206|206|206|
|scored_0409|30|30|30|30|30|30|30|30|30|30|30|30|30|
|scored_0410|225|225|225|225|225|225|225|225|225|225|225|225|225|
|scored_0411|226|225|226|226|226|225|225|225|225|225|225|225|225|
|scored_0412|218|218|218|218|218|218|218|218|218|218|218|218|218|
|scored_0413|191|190|190|190|190|189|189|189|189|189|189|189|189|
|scored_0414|163|162|161|161|161|161|161|161|161|161|161|161|161|
|scored_0415|188|187|188|188|188|187|187|187|187|187|187|187|187|
|scored_0416|225|225|224|224|224|224|224|224|224|224|224|224|224|
|scored_0417|213|213|213|213|213|213|213|213|213|213|213|213|213|
|scored_0418|149|148|149|149|149|148|148|148|148|148|148|148|148|
|scored_0419|31|31|31|31|31|31|31|31|31|31|31|31|31|
|scored_0420|191|191|190|190|190|190|190|190|190|190|190|190|190|
|scored_0421|196|196|195|195|195|195|195|195|195|195|195|195|195|
|scored_0422|201|200|201|201|201|200|200|200|200|200|200|200|200|
|scored_0423|116|116|116|116|115|116|115|115|115|115|115|115|115|
|scored_0424|275|275|275|275|275|275|275|275|275|275|275|275|275|
|scored_0425|39|39|38|38|38|36|36|36|36|36|36|36|36|
|scored_0426|170|170|170|170|170|170|170|170|170|170|170|170|170|
|scored_0427|179|179|175|175|175|175|175|175|175|175|175|175|175|
|scored_0428|264|264|264|264|264|264|264|264|264|264|264|264|264|
|scored_0429|141|141|138|138|138|138|138|138|138|138|138|138|138|
|scored_0430|110|110|110|110|110|110|110|110|110|110|110|110|110|
|scored_0431|107|106|107|107|107|106|106|106|106|106|106|106|106|
|scored_0432|170|169|169|169|169|169|169|169|169|169|169|169|169|
|scored_0433|163|163|163|163|163|163|163|163|163|163|163|163|163|
|scored_0434|136|135|136|136|136|135|135|135|135|135|135|135|135|
|scored_0435|191|191|191|191|191|191|191|191|191|191|191|191|191|
|scored_0436|175|175|170|170|170|170|170|170|170|170|170|170|170|
|scored_0437|209|209|209|209|209|209|209|209|209|209|209|209|209|
|scored_0438|245|245|245|245|245|245|245|245|245|245|245|245|245|
|scored_0439|194|194|192|192|192|192|192|192|192|192|192|192|192|
|scored_0440|237|237|237|237|237|237|237|237|237|237|237|237|237|
|scored_0441|226|226|226|226|226|226|226|226|226|226|226|226|226|
|scored_0442|219|219|219|219|219|219|219|219|219|219|219|219|219|
|scored_0443|45|45|45|45|45|45|45|45|45|45|45|45|45|
|scored_0444|298|298|298|298|298|298|298|298|298|298|298|298|298|
|scored_0445|289|289|289|289|289|289|289|289|289|289|289|289|289|
|scored_0446|310|310|310|310|310|310|310|310|310|310|310|310|310|
|scored_0447|272|272|272|272|272|272|272|272|272|272|272|272|272|
|scored_0448|260|260|260|260|260|260|260|260|260|260|260|260|260|
|scored_0449|124|123|124|124|124|123|123|123|123|123|123|123|123|
|scored_0450|342|342|342|342|342|342|342|342|342|342|342|342|342|
|scored_0451|146|146|146|146|146|146|146|146|146|146|146|146|146|
|scored_0452|253|253|253|253|253|253|253|253|253|253|253|253|253|
|scored_0453|184|184|179|179|179|177|177|177|177|177|177|177|177|
|scored_0454|258|258|258|258|258|258|258|258|258|258|258|258|258|
|scored_0455|37|37|37|37|37|37|37|37|37|37|37|37|37|
|scored_0456|148|148|148|148|148|148|148|148|148|148|148|148|148|
|scored_0457|175|174|175|175|175|174|174|174|174|174|174|174|174|
|scored_0458|188|188|188|188|188|188|188|188|188|188|188|188|188|
|scored_0459|109|108|107|107|107|105|105|105|105|105|105|105|105|
|scored_0460|202|202|202|202|202|202|202|202|202|202|202|202|202|
|scored_0461|45|45|45|45|45|45|45|45|45|45|45|45|45|
|scored_0462|145|145|144|144|144|144|144|144|144|144|144|144|144|
|scored_0463|133|132|131|131|131|131|131|131|131|131|131|131|131|
|scored_0464|102|102|100|100|99|100|99|99|99|99|99|99|99|
|scored_0465|178|177|176|175|175|175|175|175|175|175|175|175|175|
|scored_0466|231|231|231|231|231|231|231|231|231|231|231|231|231|
|scored_0467|122|122|119|119|119|119|119|119|119|119|119|119|119|
|scored_0468|193|193|193|193|193|193|193|193|193|193|193|193|193|
|scored_0469|107|107|107|107|107|107|107|107|107|107|107|107|107|
|scored_0470|327|327|327|327|327|327|327|327|327|327|327|327|327|
|scored_0471|126|125|125|125|125|124|124|124|124|124|124|124|124|
|scored_0472|30|30|30|30|30|30|30|30|30|30|30|30|30|
|scored_0473|36|36|35|35|35|35|35|35|35|35|35|35|35|
|scored_0474|108|108|106|106|106|106|106|106|106|106|106|106|106|
|scored_0475|248|248|248|248|248|248|248|248|248|248|248|248|248|
|scored_0476|262|262|262|262|262|262|262|262|262|262|262|262|262|
|scored_0477|206|206|206|206|206|206|206|206|206|206|206|206|206|
|scored_0478|99|98|99|99|99|98|98|98|98|98|98|98|98|
|scored_0479|144|144|142|142|142|142|142|142|142|142|142|142|142|
|scored_0480|183|182|183|183|183|182|182|182|182|182|182|182|182|
|scored_0481|224|224|224|224|224|224|224|224|224|224|224|224|224|
|scored_0482|235|235|234|234|234|234|234|234|234|234|234|234|234|
|scored_0483|209|209|206|205|205|205|205|205|205|205|205|205|205|
|scored_0484|176|176|176|176|176|176|176|176|176|176|176|176|176|
|scored_0485|2|2|2|2|2|2|2|2|2|2|2|2|2|
|scored_0486|140|139|140|140|140|139|140|139|139|139|139|139|139|
|scored_0487|42|41|37|37|37|37|37|37|37|37|37|37|37|
|scored_0488|177|177|176|176|176|176|176|176|176|176|176|176|176|
|scored_0489|179|178|179|179|179|178|178|178|178|178|178|178|178|
|scored_0490|4|4|4|4|4|4|4|4|4|4|4|4|4|
|scored_0491|236|236|236|236|236|236|236|236|236|236|236|236|236|
|scored_0492|51|51|51|51|51|51|51|51|51|51|51|51|51|
|scored_0493|301|301|301|301|301|301|301|301|301|301|301|301|301|
|scored_0494|278|277|278|278|278|277|277|277|277|277|277|277|277|
|scored_0495|143|143|141|141|141|141|141|141|141|141|141|141|141|
|scored_0496|160|160|160|160|160|160|160|160|160|160|160|160|160|
|scored_0497|216|216|213|213|213|213|213|213|213|213|213|213|213|
|scored_0498|192|192|192|192|192|192|192|192|192|192|192|192|192|
|scored_0499|175|174|175|175|175|174|174|174|174|174|174|174|174|
|scored_0500|106|105|101|101|101|101|101|101|101|101|101|101|101|
|scored_0501|171|171|170|170|170|170|170|170|170|170|170|170|170|
|scored_0502|201|201|200|200|200|200|200|200|200|200|200|200|200|
|scored_0503|107|107|105|105|105|105|105|105|105|105|105|105|105|
|scored_0504|238|238|238|238|238|238|238|238|238|238|238|238|238|
|scored_0505|150|150|150|150|150|150|150|150|150|150|150|150|150|
|scored_0506|147|147|147|147|147|147|147|147|147|147|147|147|147|
|scored_0507|205|205|205|205|205|205|205|205|205|205|205|205|205|
|scored_0508|252|252|252|252|252|252|252|252|252|252|252|252|252|
|scored_0509|166|166|166|166|166|166|166|166|166|166|166|166|166|
|scored_0510|108|107|107|107|107|106|106|106|106|106|106|106|106|
|scored_0511|304|304|304|304|304|304|304|304|304|304|304|304|304|
|scored_0512|187|187|187|187|187|187|187|187|187|187|187|187|187|
|scored_0513|183|183|183|183|183|183|183|183|183|183|183|183|183|
|scored_0514|267|267|267|267|267|267|267|267|267|267|267|267|267|
|scored_0515|38|38|38|38|38|38|38|38|38|38|38|38|38|
|scored_0516|199|199|199|199|199|199|199|199|199|199|199|199|199|
|scored_0517|198|198|198|198|198|198|198|198|198|198|198|198|198|
|scored_0518|244|244|244|244|244|244|244|244|244|244|244|244|244|
|scored_0519|113|112|113|113|113|112|112|112|112|112|112|112|112|
|scored_0520|219|219|219|219|219|219|219|219|219|219|219|219|219|
|scored_0521|196|195|196|196|196|195|195|195|195|195|195|195|195|
|scored_0522|117|117|115|115|113|115|113|113|113|113|113|113|113|
|scored_0523|155|154|155|155|155|154|154|154|154|154|154|154|154|
|scored_0524|160|160|156|156|156|156|156|156|156|156|156|156|156|
|scored_0525|151|151|150|150|150|150|150|150|150|150|150|150|150|
|scored_0526|265|265|265|265|265|265|265|265|265|265|265|265|265|
|scored_0527|276|276|276|276|276|276|276|276|276|276|276|276|276|
|scored_0528|170|170|170|170|170|170|170|170|170|170|170|170|170|
|scored_0529|186|186|186|186|186|186|186|186|186|186|186|186|186|
|scored_0530|194|194|194|194|194|194|194|194|194|194|194|194|194|
|scored_0531|238|238|238|238|238|238|238|238|238|238|238|238|238|
|scored_0532|104|104|104|104|102|104|102|102|102|102|102|102|102|
|scored_0533|143|143|143|143|143|143|143|143|143|143|143|143|143|
|scored_0534|161|161|161|161|161|161|161|161|161|161|161|161|161|
|scored_0535|167|167|167|167|167|167|167|167|167|167|167|167|167|
|scored_0536|308|308|308|308|308|308|308|308|308|308|308|308|308|
|scored_0537|2|2|2|2|2|2|2|2|2|2|2|2|2|
|scored_0538|211|210|211|211|211|210|210|210|210|210|210|210|210|
|scored_0539|119|119|117|117|117|117|117|117|117|117|117|117|117|
|scored_0540|212|212|212|212|212|212|212|212|212|212|212|212|212|
|scored_0541|69|69|69|69|69|69|69|69|69|69|69|69|69|
|scored_0542|136|136|136|136|136|136|136|136|136|136|136|136|136|
|scored_0543|127|127|127|127|127|127|127|127|127|127|127|127|127|
|scored_0544|217|217|217|217|217|217|217|217|217|217|217|217|217|
|scored_0545|140|139|139|139|139|138|138|138|138|138|138|138|138|
|scored_0546|184|184|184|184|184|184|184|184|184|184|184|184|184|
|scored_0547|133|133|132|132|132|132|132|132|132|132|132|132|132|
|scored_0548|48|48|48|48|48|48|48|48|48|48|48|48|48|
|scored_0549|139|139|139|139|139|139|139|139|139|139|139|139|139|
|scored_0550|133|133|133|133|133|133|133|133|133|133|133|133|133|
|scored_0551|220|220|220|220|220|220|220|220|220|220|220|220|220|
|scored_0552|271|271|271|271|271|271|271|271|271|271|271|271|271|
|scored_0553|220|219|220|220|220|219|219|219|219|219|219|219|219|
|scored_0554|139|139|139|139|139|139|139|139|139|139|139|139|139|
|scored_0555|155|155|153|153|153|153|153|153|153|153|153|153|153|
|scored_0556|36|36|36|36|36|36|36|36|36|36|36|36|36|
|scored_0557|161|161|161|161|161|161|161|161|161|161|161|161|161|
|scored_0558|149|148|148|148|148|148|148|148|148|148|148|148|148|
|scored_0559|45|45|45|45|45|45|45|45|45|45|45|45|45|
|scored_0560|50|50|49|49|49|48|48|48|48|48|48|48|48|
|scored_0561|124|123|120|120|120|120|120|120|120|120|120|120|120|
|scored_0562|116|116|116|116|116|116|116|116|116|116|116|116|116|
|scored_0563|132|132|132|132|131|132|131|131|131|131|131|131|131|
|scored_0564|39|39|39|39|38|39|36|36|36|36|36|36|36|
|scored_0565|265|264|265|265|265|264|264|264|264|264|264|264|264|
|scored_0566|30|30|30|30|30|30|30|30|30|30|30|30|30|
|scored_0567|140|139|138|138|138|137|137|137|137|137|137|137|137|
|scored_0568|140|140|140|140|140|140|140|140|140|140|140|140|140|
|scored_0569|187|187|187|187|187|187|187|187|187|187|187|187|187|
|scored_0570|112|111|111|111|110|110|110|110|110|110|110|110|110|
|scored_0571|240|240|240|240|240|240|240|240|240|240|240|240|240|
|scored_0572|286|286|286|286|286|286|286|286|286|286|286|286|286|
|scored_0573|231|231|231|231|231|231|231|231|231|231|231|231|231|
|scored_0574|128|127|127|127|127|126|126|126|126|126|126|126|126|
|scored_0575|159|159|158|158|158|158|158|158|158|158|158|158|158|
|scored_0576|207|207|207|207|207|207|207|207|207|207|207|207|207|
|scored_0577|210|210|209|209|209|209|209|209|209|209|209|209|209|
|scored_0578|154|154|152|152|152|152|152|152|152|152|152|152|152|
|scored_0579|204|203|204|204|204|203|203|203|203|203|203|203|203|
|scored_0580|282|282|282|282|282|282|282|282|282|282|282|282|282|
|scored_0581|313|312|313|313|313|312|312|312|312|312|312|312|312|
|scored_0582|226|226|226|226|226|226|226|226|226|226|226|226|226|
|scored_0583|50|50|50|50|50|50|50|50|50|50|50|50|50|
|scored_0584|57|57|57|57|57|57|57|57|57|57|57|57|57|
|scored_0585|226|226|226|226|226|226|226|226|226|226|226|226|226|
|scored_0586|105|105|105|105|105|105|105|105|105|105|105|105|105|
|scored_0587|182|182|182|182|182|182|182|182|182|182|182|182|182|
|scored_0588|170|169|169|169|169|168|168|168|168|168|168|168|168|
|scored_0589|111|111|111|111|111|111|111|111|111|111|111|111|111|
|scored_0590|171|170|171|171|171|170|170|170|170|170|170|170|170|
|scored_0591|157|156|154|154|154|153|153|153|153|153|153|153|153|
|scored_0592|259|259|259|259|259|259|259|259|259|259|259|259|259|
|scored_0593|282|282|282|282|282|282|282|282|282|282|282|282|282|
|scored_0594|284|284|284|284|284|284|284|284|284|284|284|284|284|
|scored_0595|145|145|144|144|144|144|144|144|144|144|144|144|144|
|scored_0596|153|153|152|152|152|152|152|152|152|152|152|152|152|
|scored_0597|3|3|3|3|3|3|3|3|3|3|3|3|3|
|scored_0598|101|101|100|100|99|100|99|99|99|99|99|99|99|
|scored_0599|196|196|196|196|196|196|196|196|196|196|196|196|196|
|scored_0600|3|3|3|3|3|3|3|3|3|3|3|3|3|
|scored_0601|147|147|146|146|146|146|146|146|146|146|146|146|146|
|scored_0602|134|134|132|132|132|132|132|132|132|132|132|132|132|
|scored_0603|189|189|186|186|186|186|186|186|186|186|186|186|186|
|scored_0604|158|157|158|158|158|157|157|157|157|157|157|157|157|
|scored_0605|207|207|207|207|207|207|207|207|207|207|207|207|207|
|scored_0606|205|205|205|205|205|205|205|205|205|205|205|205|205|
|scored_0607|155|155|154|154|154|154|154|154|154|154|154|154|154|
|scored_0608|187|187|187|187|187|187|187|187|187|187|187|187|187|
|scored_0609|188|188|188|188|188|188|188|188|188|188|188|188|188|
|scored_0610|335|335|335|335|335|335|335|335|335|335|335|335|335|
|scored_0611|40|40|39|39|39|39|39|39|39|39|39|39|39|
|scored_0612|321|321|321|321|321|321|321|321|321|321|321|321|321|
|scored_0613|133|132|132|132|132|131|131|131|131|131|131|131|131|
|scored_0614|28|26|28|28|28|26|27|26|26|26|26|26|26|
|scored_0615|140|140|140|140|140|140|140|140|140|140|140|140|140|
|scored_0616|186|185|185|185|185|185|185|185|185|185|185|185|185|
|scored_0617|129|127|127|127|127|125|125|125|125|125|125|125|125|
|scored_0618|130|130|129|129|129|129|129|129|129|129|129|129|129|
|scored_0619|191|191|191|191|191|191|191|191|191|191|191|191|191|
|scored_0620|50|50|50|50|50|50|50|50|50|50|50|50|50|
|scored_0621|214|214|214|214|214|214|214|214|214|214|214|214|214|
|scored_0622|201|201|201|201|201|201|201|201|201|201|201|201|201|
|scored_0623|102|102|102|102|102|102|102|102|102|102|102|102|102|
|scored_0624|145|144|143|143|143|143|143|143|143|143|143|143|143|
|scored_0625|122|122|121|121|119|121|119|119|119|119|119|119|119|
|scored_0626|217|216|212|212|212|211|211|211|211|211|211|211|211|
|scored_0627|169|168|160|159|159|159|159|159|159|159|159|159|159|
|scored_0628|255|255|255|255|255|255|255|255|255|255|255|255|255|
|scored_0629|196|195|196|196|196|195|195|195|195|195|195|195|195|
|scored_0630|41|41|39|39|39|39|39|39|39|39|39|39|39|
|scored_0631|165|165|162|162|162|162|162|162|162|162|162|162|162|
|scored_0632|236|236|236|236|236|236|236|236|236|236|236|236|236|
|scored_0633|207|207|207|207|207|207|207|207|207|207|207|207|207|
|scored_0634|207|206|206|206|206|205|205|205|205|205|205|205|205|
|scored_0635|46|46|46|46|46|46|46|46|46|46|46|46|46|
|scored_0636|314|314|314|314|314|314|314|314|314|314|314|314|314|
|scored_0637|43|43|43|43|43|43|43|43|43|43|43|43|43|
|scored_0638|341|340|341|341|341|340|340|340|340|340|340|340|340|
|scored_0639|153|153|151|151|151|151|151|151|151|151|151|151|151|
|scored_0640|260|260|260|260|260|260|260|260|260|260|260|260|260|
|scored_0641|139|138|138|138|138|138|138|138|138|138|138|138|138|
|scored_0642|152|152|151|151|151|151|151|151|151|151|151|151|151|
|scored_0643|229|227|229|229|229|227|227|227|227|227|227|227|227|
|scored_0644|174|174|174|174|174|174|174|174|174|174|174|174|174|
|scored_0645|246|246|246|246|246|246|246|246|246|246|246|246|246|
|scored_0646|203|203|202|202|202|202|202|202|202|202|202|202|202|
|scored_0647|259|259|259|259|259|259|259|259|259|259|259|259|259|
|scored_0648|124|124|124|124|124|124|124|124|124|124|124|124|124|
|scored_0649|151|151|150|150|150|150|150|150|150|150|150|150|150|
|scored_0650|44|44|44|44|44|44|44|44|44|44|44|44|44|
|scored_0651|206|205|206|206|206|205|205|205|205|205|205|205|205|
|scored_0652|127|126|122|122|121|122|121|121|121|121|121|121|121|
|scored_0653|209|209|209|209|209|209|209|209|209|209|209|209|209|
|scored_0654|44|44|43|43|43|42|42|42|42|42|42|42|42|
|scored_0655|325|325|325|325|325|325|325|325|325|325|325|325|325|
|scored_0656|47|46|47|47|47|46|46|46|46|46|46|46|46|
|scored_0657|120|120|119|119|119|119|119|119|119|119|119|119|119|
|scored_0658|215|215|215|215|215|215|215|215|215|215|215|215|215|
|scored_0659|108|107|108|108|104|107|104|104|104|104|104|104|104|
|scored_0660|213|213|209|209|209|209|209|209|209|209|209|209|209|
|scored_0661|282|282|282|282|282|282|282|282|282|282|282|282|282|
|scored_0662|205|203|205|205|205|203|203|203|203|203|203|203|203|
|scored_0663|224|224|224|224|224|224|224|224|224|224|224|224|224|
|scored_0664|269|269|269|269|269|269|269|269|269|269|269|269|269|
|scored_0665|118|117|118|118|118|117|117|117|117|117|117|117|117|
|scored_0666|186|186|186|186|186|186|186|186|186|186|186|186|186|
|scored_0667|210|208|210|210|210|208|208|208|208|208|208|208|208|
|scored_0668|149|149|148|148|148|148|148|148|148|148|148|148|148|
|scored_0669|161|160|160|160|160|159|159|159|159|159|159|159|159|
|scored_0670|222|221|222|222|222|221|221|221|221|221|221|221|221|
|scored_0671|240|240|240|240|240|240|240|240|240|240|240|240|240|
|scored_0672|72|72|72|72|72|72|72|72|72|72|72|72|72|
|scored_0673|207|207|207|207|207|207|207|207|207|207|207|207|207|
|scored_0674|155|155|155|155|155|155|155|155|155|155|155|155|155|
|scored_0675|181|181|179|179|179|179|179|179|179|179|179|179|179|
|scored_0676|6|6|6|6|6|6|6|6|6|6|6|6|6|
|scored_0677|285|285|285|285|285|285|285|285|285|285|285|285|285|
|scored_0678|206|206|206|206|206|206|206|206|206|206|206|206|206|
|scored_0679|146|146|146|146|146|146|146|146|146|146|146|146|146|
|scored_0680|245|245|244|244|244|244|244|244|244|244|244|244|244|
|scored_0681|254|254|254|254|254|254|254|254|254|254|254|254|254|
|scored_0682|327|327|327|327|327|327|327|327|327|327|327|327|327|
|scored_0683|180|180|176|176|176|176|176|176|176|176|176|176|176|
|scored_0684|139|139|138|138|138|138|138|138|138|138|138|138|138|
|scored_0685|220|220|220|220|220|220|220|220|220|220|220|220|220|
|scored_0686|117|117|115|115|115|115|115|115|115|115|115|115|115|
|scored_0687|286|286|286|286|286|286|286|286|286|286|286|286|286|
|scored_0688|202|202|202|202|202|202|202|202|202|202|202|202|202|
|scored_0689|225|225|225|225|225|225|225|225|225|225|225|225|225|
|scored_0690|331|331|331|331|331|331|331|331|331|331|331|331|331|
|scored_0691|174|174|174|174|174|174|174|174|174|174|174|174|174|
|scored_0692|249|249|249|249|249|249|249|249|249|249|249|249|249|
|scored_0693|208|208|208|208|208|208|208|208|208|208|208|208|208|
|scored_0694|192|192|192|192|192|192|192|192|192|192|192|192|192|
|scored_0695|142|142|142|142|142|142|142|142|142|142|142|142|142|
|scored_0696|101|101|101|101|101|101|101|101|101|101|101|101|101|
|scored_0697|135|135|134|134|134|134|134|134|134|134|134|134|134|
|scored_0698|291|291|291|291|291|291|291|291|291|291|291|291|291|
|scored_0699|175|175|169|169|169|169|169|169|169|169|169|169|169|
|scored_0700|212|212|212|212|212|212|212|212|212|212|212|212|212|
|scored_0701|41|41|41|41|41|41|41|41|41|41|41|41|41|
|scored_0702|262|262|262|262|262|262|262|262|262|262|262|262|262|
|scored_0703|214|214|214|214|214|214|214|214|214|214|214|214|214|
|scored_0704|156|156|156|156|156|156|156|156|156|156|156|156|156|
|scored_0705|191|191|191|191|191|191|191|191|191|191|191|191|191|
|scored_0706|228|228|228|228|228|228|228|228|228|228|228|228|228|
|scored_0707|117|116|116|115|115|115|115|115|115|115|115|115|115|
|scored_0708|186|186|186|186|186|186|186|186|186|186|186|186|186|
|scored_0709|158|157|157|157|157|157|157|157|157|157|157|157|157|
|scored_0710|148|148|148|148|148|148|148|148|148|148|148|148|148|
|scored_0711|29|29|27|27|27|26|27|26|26|26|26|26|26|
|scored_0712|233|233|233|233|233|233|233|233|233|233|233|233|233|
|scored_0713|122|121|122|122|122|121|121|121|121|121|121|121|121|
|scored_0714|232|232|232|232|232|232|232|232|232|232|232|232|232|
|scored_0715|197|197|197|197|197|197|197|197|197|197|197|197|197|
|scored_0716|148|148|147|147|147|147|147|147|147|147|147|147|147|
|scored_0717|153|153|151|151|151|151|151|151|151|151|151|151|151|
|scored_0718|171|171|169|169|169|169|169|169|169|169|169|169|169|
|scored_0719|309|309|309|309|309|309|309|309|309|309|309|309|309|
|scored_0720|37|37|37|37|37|37|37|37|37|37|37|37|37|
|scored_0721|4|4|4|4|4|4|4|4|4|4|4|4|4|
|scored_0722|198|198|198|198|198|198|198|198|198|198|198|198|198|
|scored_0723|125|124|122|122|122|122|122|122|122|122|122|122|122|
|scored_0724|199|199|199|199|199|199|199|199|199|199|199|199|199|
|scored_0725|273|273|273|273|273|273|273|273|273|273|273|273|273|
|scored_0726|186|186|186|186|186|186|186|186|186|186|186|186|186|
|scored_0727|103|102|102|102|102|101|101|101|101|101|101|101|101|
|scored_0728|315|315|315|315|315|315|315|315|315|315|315|315|315|
|scored_0729|352|352|352|352|352|352|352|352|352|352|352|352|352|
|scored_0730|161|161|158|158|158|158|158|158|158|158|158|158|158|
|scored_0731|195|195|195|195|195|195|195|195|195|195|195|195|195|
|scored_0732|154|154|154|154|154|154|154|154|154|154|154|154|154|
|scored_0733|116|116|116|116|116|116|116|116|116|116|116|116|116|
|scored_0734|178|178|178|178|178|178|178|178|178|178|178|178|178|
|scored_0735|145|145|142|142|141|142|140|140|140|140|140|140|140|
|scored_0736|195|195|195|195|195|195|195|195|195|195|195|195|195|
|scored_0737|144|144|144|144|143|144|143|143|143|143|143|143|143|
|scored_0738|146|146|146|146|146|146|146|146|146|146|146|146|146|
|scored_0739|187|187|187|187|187|187|187|187|187|187|187|187|187|
|scored_0740|164|164|163|163|163|163|163|163|163|163|163|163|163|
|scored_0741|167|166|166|166|166|166|166|166|166|166|166|166|166|
|scored_0742|224|224|224|224|224|224|224|224|224|224|224|224|224|
|scored_0743|132|131|131|131|131|130|130|130|130|130|130|130|130|
|scored_0744|343|343|343|343|343|343|343|343|343|343|343|343|343|
|scored_0745|220|220|220|220|220|220|220|220|220|220|220|220|220|
|scored_0746|184|184|183|183|183|183|183|183|183|183|183|183|183|
|scored_0747|190|190|190|190|190|190|190|190|190|190|190|190|190|
|scored_0748|288|288|288|288|288|288|288|288|288|288|288|288|288|
|scored_0749|118|118|118|118|118|118|118|118|118|118|118|118|118|
|scored_0750|141|139|139|139|139|139|139|139|139|139|139|139|139|
|scored_0751|268|268|268|268|268|268|268|268|268|268|268|268|268|
|scored_0752|128|128|126|126|126|126|126|126|126|126|126|126|126|
|scored_0753|122|120|121|121|120|120|119|119|119|119|119|119|119|
|scored_0754|208|208|208|208|208|208|208|208|208|208|208|208|208|
|scored_0755|142|142|142|142|142|142|142|142|142|142|142|142|142|
|scored_0756|191|191|191|191|191|191|191|191|191|191|191|191|191|
|scored_0757|163|163|162|162|162|161|161|161|161|161|161|161|161|
|scored_0758|136|136|136|136|136|136|136|136|136|136|136|136|136|
|scored_0759|244|244|244|244|244|244|244|244|244|244|244|244|244|
|scored_0760|137|137|136|136|136|136|136|136|136|136|136|136|136|
|scored_0761|123|123|123|123|122|123|122|122|122|122|122|122|122|
|scored_0762|341|341|341|341|341|341|341|341|341|341|341|341|341|
|scored_0763|115|115|115|115|115|115|115|115|115|115|115|115|115|
|scored_0764|248|248|248|248|248|248|248|248|248|248|248|248|248|
|scored_0765|228|228|228|228|228|228|228|228|228|228|228|228|228|
|scored_0766|262|262|262|262|262|262|262|262|262|262|262|262|262|
|scored_0767|149|149|147|147|147|147|147|147|147|147|147|147|147|
|scored_0768|152|151|151|150|150|149|149|149|149|149|149|149|149|
|scored_0769|144|144|143|143|143|143|143|143|143|143|143|143|143|
|scored_0770|184|184|182|182|182|182|182|182|182|182|182|182|182|
|scored_0771|321|321|321|321|321|321|321|321|321|321|321|321|321|
|scored_0772|148|148|145|145|145|145|145|145|145|145|145|145|145|
|scored_0773|60|60|60|60|60|60|60|60|60|60|60|60|60|
|scored_0774|344|344|344|344|344|344|344|344|344|344|344|344|344|
|scored_0775|186|186|184|184|184|184|184|184|184|184|184|184|184|
|scored_0776|273|273|273|273|273|273|273|273|273|273|273|273|273|
|scored_0777|49|49|49|49|49|49|49|49|49|49|49|49|49|
|scored_0778|163|163|163|163|163|163|163|163|163|163|163|163|163|
|scored_0779|179|179|178|178|178|178|178|178|178|178|178|178|178|
|scored_0780|331|331|331|331|331|331|331|331|331|331|331|331|331|
|scored_0781|305|305|305|305|305|305|305|305|305|305|305|305|305|
|scored_0782|186|185|186|186|186|185|185|185|185|185|185|185|185|
|scored_0783|139|139|137|137|137|137|137|137|137|137|137|137|137|
|scored_0784|275|275|275|275|275|275|275|275|275|275|275|275|275|
|scored_0785|134|134|134|134|134|134|134|134|134|134|134|134|134|
|scored_0786|299|298|299|299|299|298|298|298|298|298|298|298|298|
|scored_0787|126|126|125|125|125|125|125|125|125|125|125|125|125|
|scored_0788|285|285|285|285|285|285|285|285|285|285|285|285|285|
|scored_0789|219|219|219|219|219|219|219|219|219|219|219|219|219|
|scored_0790|272|272|272|272|272|272|272|272|272|272|272|272|272|
|scored_0791|192|192|192|192|192|192|192|192|192|192|192|192|192|
|scored_0792|167|167|167|167|167|167|167|167|167|167|167|167|167|
|scored_0793|273|272|273|273|273|272|272|272|272|272|272|272|272|
|scored_0794|210|210|210|210|210|210|210|210|210|210|210|210|210|
|scored_0795|220|220|220|220|220|220|220|220|220|220|220|220|220|
|scored_0796|278|278|278|278|278|278|278|278|278|278|278|278|278|
|scored_0797|90|90|90|90|90|90|90|90|90|90|90|90|90|
|scored_0798|2|2|2|2|2|2|2|2|2|2|2|2|2|
|scored_0799|285|285|285|285|285|285|285|285|285|285|285|285|285|
|scored_0800|102|102|101|101|101|101|101|101|101|101|101|101|101|
|scored_0801|125|125|125|125|125|125|125|125|125|125|125|125|125|
|scored_0802|232|232|232|232|232|232|232|232|232|232|232|232|232|
|scored_0803|174|174|173|173|173|173|173|173|173|173|173|173|173|
|scored_0804|141|141|141|141|141|141|141|141|141|141|141|141|141|
|scored_0805|200|200|199|199|199|199|199|199|199|199|199|199|199|
|scored_0806|119|119|118|118|118|118|118|118|118|118|118|118|118|
|scored_0807|176|176|175|175|175|175|175|175|175|175|175|175|175|
|scored_0808|305|305|305|305|305|305|305|305|305|305|305|305|305|
|scored_0809|166|166|166|166|166|166|166|166|166|166|166|166|166|
|scored_0810|41|41|41|41|41|41|41|41|41|41|41|41|41|
|scored_0811|113|112|112|112|112|110|110|110|110|110|110|110|110|
|scored_0812|131|131|131|131|131|131|131|131|131|131|131|131|131|
|scored_0813|279|279|279|279|279|279|279|279|279|279|279|279|279|
|scored_0814|200|200|198|198|198|198|198|198|198|198|198|198|198|
|scored_0815|261|261|261|261|261|261|261|261|261|261|261|261|261|
|scored_0816|133|133|131|131|131|131|131|131|131|131|131|131|131|
|scored_0817|209|209|209|209|209|209|209|209|209|209|209|209|209|
|scored_0818|32|32|31|31|31|31|31|31|31|31|31|31|31|
|scored_0819|282|282|282|282|282|282|282|282|282|282|282|282|282|
|scored_0820|276|275|276|276|276|275|275|275|275|275|275|275|275|
|scored_0821|127|126|126|126|126|125|125|125|125|125|125|125|125|
|scored_0822|46|46|46|46|46|46|46|46|46|46|46|46|46|
|scored_0823|192|192|189|189|189|189|189|189|189|189|189|189|189|
|scored_0824|176|176|176|176|176|176|176|176|176|176|176|176|176|
|scored_0825|151|151|150|150|150|150|150|150|150|150|150|150|150|
|scored_0826|259|259|259|259|259|259|259|259|259|259|259|259|259|
|scored_0827|241|241|240|240|240|240|240|240|240|240|240|240|240|
|scored_0828|42|42|42|42|42|42|42|42|42|42|42|42|42|
|scored_0829|212|212|212|212|212|212|212|212|212|212|212|212|212|
|scored_0830|127|126|126|126|126|126|126|126|126|126|126|126|126|
|scored_0831|119|119|119|119|119|119|119|119|119|119|119|119|119|
|scored_0832|230|230|230|230|230|230|230|230|230|230|230|230|230|
|scored_0833|157|157|157|157|157|157|157|157|157|157|157|157|157|
|scored_0834|262|262|262|262|262|262|262|262|262|262|262|262|262|
|scored_0835|141|141|141|141|141|141|141|141|141|141|141|141|141|
|scored_0836|232|232|232|232|232|232|232|232|232|232|232|232|232|
|scored_0837|288|288|288|288|288|288|288|288|288|288|288|288|288|
|scored_0838|172|172|172|172|172|172|172|172|172|172|172|172|172|
|scored_0839|210|210|210|210|209|210|209|209|209|209|209|209|209|
|scored_0840|282|281|282|282|282|281|281|281|281|281|281|281|281|
|scored_0841|190|189|190|190|190|189|189|189|189|189|189|189|189|
|scored_0842|36|34|32|32|32|32|32|32|32|32|32|32|32|
|scored_0843|177|176|177|177|177|176|176|176|176|176|176|176|176|
|scored_0844|318|318|318|318|318|318|318|318|318|318|318|318|318|
|scored_0845|265|265|265|265|265|265|265|265|265|265|265|265|265|
|scored_0846|125|125|125|125|125|125|125|125|125|125|125|125|125|
|scored_0847|216|216|216|216|216|216|216|216|216|216|216|216|216|
|scored_0848|312|312|312|312|312|312|312|312|312|312|312|312|312|
|scored_0849|240|240|240|240|240|240|240|240|240|240|240|240|240|
|scored_0850|218|218|218|218|218|218|218|218|218|218|218|218|218|
|scored_0851|41|41|41|41|41|41|41|41|41|41|41|41|41|
|scored_0852|198|198|198|198|198|198|198|198|198|198|198|198|198|
|scored_0853|218|217|218|218|218|217|217|217|217|217|217|217|217|
|scored_0854|197|197|196|196|196|196|196|196|196|196|196|196|196|
|scored_0855|91|90|90|89|88|89|88|88|88|88|88|88|88|
|scored_0856|267|266|267|267|267|266|266|266|266|266|266|266|266|
|scored_0857|280|280|280|280|280|280|280|280|280|280|280|280|280|
|scored_0858|56|56|56|56|56|56|56|56|56|56|56|56|56|
|scored_0859|267|267|267|267|267|267|267|267|267|267|267|267|267|
|scored_0860|180|180|180|180|180|180|180|180|180|180|180|180|180|
|scored_0861|197|197|197|197|197|197|197|197|197|197|197|197|197|
|scored_0862|165|165|162|162|162|162|162|162|162|162|162|162|162|
|scored_0863|43|43|43|43|43|43|43|43|43|43|43|43|43|
|scored_0864|230|230|230|230|230|230|230|230|230|230|230|230|230|
|scored_0865|293|293|293|293|293|293|293|293|293|293|293|293|293|
|scored_0866|41|41|41|41|41|41|41|41|41|41|41|41|41|
|scored_0867|111|111|111|111|111|111|111|111|111|111|111|111|111|
|scored_0868|40|40|39|39|39|39|39|39|39|39|39|39|39|
|scored_0869|156|156|154|154|154|154|154|154|154|154|154|154|154|
|scored_0870|269|269|269|269|269|269|269|269|269|269|269|269|269|
|scored_0871|140|140|140|140|140|140|140|140|140|140|140|140|140|
|scored_0872|132|132|131|131|130|131|130|130|130|130|130|130|130|
|scored_0873|48|48|48|48|48|48|48|48|48|48|48|48|48|
|scored_0874|46|46|46|44|43|43|42|42|42|42|42|42|42|
|scored_0875|197|197|197|197|197|197|197|197|197|197|197|197|197|
|scored_0876|245|245|245|245|245|245|245|245|245|245|245|245|245|
|scored_0877|75|74|75|75|75|74|74|74|74|74|74|74|74|
|scored_0878|193|193|193|193|193|193|193|193|193|193|193|193|193|
|scored_0879|129|129|129|129|128|129|128|128|128|128|128|128|128|
|scored_0880|48|48|48|48|48|48|48|48|48|48|48|48|48|
|scored_0881|109|109|109|109|109|109|109|109|109|109|109|109|109|
|scored_0882|32|32|32|32|31|32|31|31|31|31|31|31|31|
|scored_0883|7|7|7|7|7|7|7|7|7|7|7|7|7|
|scored_0884|137|136|136|136|136|135|135|135|135|135|135|135|135|
|scored_0885|202|202|202|202|202|202|202|202|202|202|202|202|202|
|scored_0886|191|191|191|191|191|191|191|191|191|191|191|191|191|
|scored_0887|4|4|4|4|4|4|4|4|4|4|4|4|4|
|scored_0888|156|156|155|155|155|155|155|155|155|155|155|155|155|
|scored_0889|136|136|135|135|135|135|135|135|135|135|135|135|135|
|scored_0890|115|113|115|115|115|113|113|113|113|113|113|113|113|
|scored_0891|135|135|135|135|135|135|135|135|135|135|135|135|135|
|scored_0892|21|21|21|21|21|21|21|21|21|21|21|21|21|
|scored_0893|161|161|161|161|161|161|161|161|161|161|161|161|161|
|scored_0894|153|153|153|153|153|153|153|153|153|153|153|153|153|
|scored_0895|147|147|146|146|146|146|146|146|146|146|146|146|146|
|scored_0896|216|214|215|215|215|213|213|213|213|213|213|213|213|
|scored_0897|48|48|48|47|47|47|47|47|47|47|47|47|47|
|scored_0898|103|102|103|103|103|102|102|102|102|102|102|102|102|
|scored_0899|289|289|289|289|289|289|289|289|289|289|289|289|289|
|scored_0900|208|208|208|208|208|208|208|208|208|208|208|208|208|
|scored_0901|21|21|21|21|21|21|21|21|21|21|21|21|21|
|scored_0902|262|262|262|262|262|262|262|262|262|262|262|262|262|
|scored_0903|190|190|190|190|190|190|190|190|190|190|190|190|190|
|scored_0904|169|169|169|169|169|169|169|169|169|169|169|169|169|
|scored_0905|296|296|296|296|296|296|296|296|296|296|296|296|296|
|scored_0906|258|258|258|258|258|258|258|258|258|258|258|258|258|
|scored_0907|145|145|145|145|145|145|145|145|145|145|145|145|145|
|scored_0908|43|43|43|43|43|43|43|43|43|43|43|43|43|
|scored_0909|199|199|199|199|199|199|199|199|199|199|199|199|199|
|scored_0910|45|45|45|45|45|45|45|45|45|45|45|45|45|
|scored_0911|107|105|105|105|105|103|103|103|103|103|103|103|103|
|scored_0912|260|260|260|260|260|260|260|260|260|260|260|260|260|
|scored_0913|34|34|34|34|34|34|34|34|34|34|34|34|34|
|scored_0914|50|50|50|50|50|50|50|50|50|50|50|50|50|
|scored_0915|235|235|235|235|235|235|235|235|235|235|235|235|235|
|scored_0916|284|284|284|284|284|284|284|284|284|284|284|284|284|
|scored_0917|264|264|264|264|264|264|264|264|264|264|264|264|264|
|scored_0918|265|264|265|265|265|264|264|264|264|264|264|264|264|
|scored_0919|187|187|185|185|185|185|185|185|185|185|185|185|185|
|scored_0920|182|182|182|182|182|182|182|182|182|182|182|182|182|
|scored_0921|207|207|207|207|207|207|207|207|207|207|207|207|207|
|scored_0922|34|34|34|34|34|34|34|34|34|34|34|34|34|
|scored_0923|247|247|247|247|247|247|247|247|247|247|247|247|247|
|scored_0924|142|142|141|141|141|141|141|141|141|141|141|141|141|
|scored_0925|196|196|196|196|196|196|196|196|196|196|196|196|196|
|scored_0926|32|32|31|31|31|31|31|31|31|31|31|31|31|
|scored_0927|171|170|171|171|171|170|170|170|170|170|170|170|170|
|scored_0928|42|41|42|42|42|41|41|41|41|41|41|41|41|
|scored_0929|172|172|171|171|171|171|171|171|171|171|171|171|171|
|scored_0930|106|106|106|106|106|106|106|106|106|106|106|106|106|
|scored_0931|2|2|2|2|2|2|2|2|2|2|2|2|2|
|scored_0932|283|283|283|283|283|283|283|283|283|283|283|283|283|
|scored_0933|197|196|197|197|197|196|196|196|196|196|196|196|196|
|scored_0934|177|177|177|177|177|177|177|177|177|177|177|177|177|
|scored_0935|212|212|210|210|210|210|210|210|210|210|210|210|210|
|scored_0936|187|187|187|187|187|187|187|187|187|187|187|187|187|
|scored_0937|38|38|38|38|38|38|38|38|38|38|38|38|38|
|scored_0938|165|164|163|163|163|163|163|163|163|163|163|163|163|
|scored_0939|289|289|289|289|289|289|289|289|289|289|289|289|289|
|scored_0940|157|157|155|155|155|155|155|155|155|155|155|155|155|
|scored_0941|290|290|290|290|290|290|290|290|290|290|290|290|290|
|scored_0942|176|176|174|174|174|174|174|174|174|174|174|174|174|
|scored_0943|117|117|116|116|116|116|116|116|116|116|116|116|116|
|scored_0944|280|280|280|280|280|280|280|280|280|280|280|280|280|
|scored_0945|37|36|35|35|35|34|34|34|34|34|34|34|34|
|scored_0946|267|267|267|267|267|267|267|267|267|267|267|267|267|
|scored_0947|118|116|117|117|117|115|115|115|115|115|115|115|115|
|scored_0948|163|163|163|163|163|163|163|163|163|163|163|163|163|
|scored_0949|109|106|107|107|107|106|106|106|106|106|106|106|106|
|scored_0950|156|156|154|154|154|154|154|154|154|154|154|154|154|
|scored_0951|114|114|111|111|111|111|111|111|111|111|111|111|111|
|scored_0952|267|267|267|267|267|267|267|267|267|267|267|267|267|
|scored_0953|224|224|224|224|224|224|224|224|224|224|224|224|224|
|scored_0954|99|99|99|99|99|99|99|99|99|99|99|99|99|
|scored_0955|113|113|113|113|112|113|111|111|111|111|111|111|111|
|scored_0956|279|279|279|279|279|279|279|279|279|279|279|279|279|
|scored_0957|39|39|39|39|39|39|39|39|39|39|39|39|39|
|scored_0958|36|36|36|36|36|36|36|36|36|36|36|36|36|
|scored_0959|289|289|289|289|289|289|289|289|289|289|289|289|289|
|scored_0960|157|157|157|157|155|157|155|155|155|155|155|155|155|
|scored_0961|155|155|155|155|155|155|155|155|155|155|155|155|155|
|scored_0962|47|47|44|44|44|43|43|43|43|43|43|43|43|
|scored_0963|155|154|155|155|155|154|154|154|154|154|154|154|154|
|scored_0964|258|258|258|258|258|258|258|258|258|258|258|258|258|
|scored_0965|254|254|254|254|254|254|254|254|254|254|254|254|254|
|scored_0966|110|110|109|109|109|109|109|109|109|109|109|109|109|
|scored_0967|138|138|137|137|137|137|137|137|137|137|137|137|137|
|scored_0968|205|205|205|205|205|205|205|205|205|205|205|205|205|
|scored_0969|184|184|184|184|184|184|184|184|184|184|184|184|184|
|scored_0970|289|289|289|289|289|289|289|289|289|289|289|289|289|
|scored_0971|23|23|23|23|23|23|23|23|23|23|23|23|23|
|scored_0972|155|154|155|155|155|154|154|154|154|154|154|154|154|
|scored_0973|136|136|136|136|136|136|136|136|136|136|136|136|136|
|scored_0974|286|286|286|286|286|286|286|286|286|286|286|286|286|
|scored_0975|157|157|157|157|157|157|157|157|157|157|157|157|157|
|scored_0976|151|151|150|150|149|150|149|149|149|149|149|149|149|
|scored_0977|207|207|207|207|207|207|207|207|207|207|207|207|207|
|scored_0978|132|132|132|132|132|132|132|132|132|132|132|132|132|
|scored_0979|220|220|220|220|220|220|220|220|220|220|220|220|220|
|scored_0980|157|157|157|157|157|157|157|157|157|157|157|157|157|
|scored_0981|34|34|33|33|33|33|33|33|33|33|33|33|33|
|scored_0982|342|342|342|342|342|342|342|342|342|342|342|342|342|
|scored_0983|171|171|171|171|171|171|171|171|171|171|171|171|171|
|scored_0984|168|168|168|168|168|168|168|168|168|168|168|168|168|
|scored_0985|165|165|165|165|165|165|165|165|165|165|165|165|165|
|scored_0986|212|211|212|212|212|211|211|211|211|211|211|211|211|
|scored_0987|177|176|177|177|177|176|176|176|176|176|176|176|176|
|scored_0988|181|181|181|181|181|181|181|181|181|181|181|181|181|
|scored_0989|101|100|101|101|101|100|100|100|100|100|100|100|100|
|scored_0990|222|222|222|222|222|222|222|222|222|222|222|222|222|
|scored_0991|218|218|218|218|218|218|218|218|218|218|218|218|218|
|scored_0992|326|326|326|326|326|326|326|326|326|326|326|326|326|
|scored_0993|274|274|274|274|274|274|274|274|274|274|274|274|274|
|scored_0994|194|194|189|189|189|189|189|189|189|189|189|189|189|
|scored_0995|178|177|178|178|178|177|177|177|177|177|177|177|177|
|scored_0996|160|160|160|160|160|160|160|160|160|160|160|160|160|
|scored_0997|161|161|161|161|161|161|161|161|161|161|161|161|161|
|scored_0998|255|255|255|255|255|255|255|255|255|255|255|255|255|
|scored_0999|132|132|132|132|132|132|132|132|132|132|132|132|132|
|scored_1000|297|297|297|297|297|297|297|297|297|297|297|297|297|

## Elegir intermedios por utilidad para objetivos pendientes

Control guardado 52/72, ancho 6. Utilidad cuenta objetivos distintos alcanzables con una operación o producto con resto firmado usando hojas fijas. Empates por recencia. Mixta reserva la mitad de los lugares libres para los recientes.

Totales en orden: [35882, 36281, 36278, 36281, 35880]. IDs que mejoran la cartera por variante: [1, 0]. [Detalle](reports/pending_dictionary_t1qgx25_/comparison.csv)

| ID | Cartera previa | Recencia | Utilidad | Mixta | Final |
|---|---:|---:|---:|---:|---:|
|practice_0001|153|154|154|154|153|
|practice_0002|267|276|276|276|267|
|practice_0003|284|285|285|285|284|
|practice_0004|129|130|130|130|129|
|practice_0005|232|234|234|234|232|
|practice_0006|36|40|40|40|36|
|practice_0007|104|106|106|106|104|
|practice_0008|258|258|258|258|258|
|practice_0009|137|139|139|139|137|
|practice_0010|139|141|141|141|139|
|practice_0011|128|128|128|128|128|
|practice_0012|258|259|259|259|258|
|practice_0013|215|217|217|217|215|
|practice_0014|138|138|138|138|138|
|practice_0015|199|205|205|205|199|
|practice_0016|229|235|235|235|229|
|practice_0017|175|176|176|176|175|
|practice_0018|342|342|342|342|342|
|practice_0019|207|211|211|211|207|
|practice_0020|111|114|114|114|111|
|practice_0021|132|136|136|136|132|
|practice_0022|143|143|143|143|143|
|practice_0023|192|193|193|193|192|
|practice_0024|37|43|43|43|37|
|practice_0025|182|182|182|182|182|
|practice_0026|152|153|153|153|152|
|practice_0027|204|205|205|205|204|
|practice_0028|240|240|240|240|240|
|practice_0029|211|214|214|214|211|
|practice_0030|139|142|142|142|139|
|practice_0031|226|226|226|226|226|
|practice_0032|140|140|140|140|140|
|practice_0033|41|42|42|42|41|
|practice_0034|318|319|319|319|318|
|practice_0035|264|265|265|265|264|
|practice_0036|132|133|133|133|132|
|practice_0037|293|293|293|293|293|
|practice_0038|43|45|45|45|43|
|practice_0039|6|9|9|9|6|
|practice_0040|198|203|203|203|198|
|practice_0041|188|188|188|188|188|
|practice_0042|182|185|185|185|182|
|practice_0043|122|127|127|127|122|
|practice_0044|230|233|233|233|230|
|practice_0045|147|150|150|150|147|
|practice_0046|271|271|271|271|271|
|practice_0047|2|5|5|5|2|
|practice_0048|120|124|124|124|120|
|practice_0049|129|131|131|131|129|
|practice_0050|3|5|5|5|3|
|practice_0051|5|7|7|7|5|
|practice_0052|208|214|214|214|208|
|practice_0053|267|267|267|267|267|
|practice_0054|346|346|346|346|346|
|practice_0055|349|349|349|349|349|
|practice_0056|4|7|7|7|4|
|practice_0057|255|256|256|256|255|
|practice_0058|267|267|267|267|267|
|practice_0059|147|151|151|151|147|
|practice_0060|265|265|265|265|265|
|practice_0061|282|282|282|282|282|
|practice_0062|313|313|313|313|313|
|practice_0063|128|131|131|131|128|
|practice_0064|255|255|255|255|255|
|practice_0065|236|236|236|236|236|
|practice_0066|106|112|112|112|106|
|practice_0067|171|173|173|173|171|
|practice_0068|196|198|198|198|196|
|practice_0069|220|220|220|220|220|
|practice_0070|270|271|271|271|270|
|practice_0071|167|169|169|169|167|
|practice_0072|5|8|8|8|5|
|practice_0073|158|159|159|159|158|
|practice_0074|216|226|226|226|216|
|practice_0075|198|199|199|199|198|
|practice_0076|332|337|337|337|332|
|practice_0077|111|115|115|115|111|
|practice_0078|284|284|284|284|284|
|practice_0079|137|138|138|138|137|
|practice_0080|119|124|124|124|119|
|practice_0081|196|203|203|203|196|
|practice_0082|239|243|243|243|239|
|practice_0083|151|155|155|155|151|
|practice_0084|158|168|168|168|158|
|practice_0085|304|304|304|304|304|
|practice_0086|151|153|153|153|151|
|practice_0087|255|255|255|255|255|
|practice_0088|232|233|233|233|232|
|practice_0089|162|165|165|165|162|
|practice_0090|154|155|155|155|154|
|practice_0091|223|230|230|230|223|
|practice_0092|181|183|183|183|181|
|practice_0093|238|238|238|238|238|
|practice_0094|141|141|141|141|141|
|practice_0095|259|261|261|261|259|
|practice_0096|214|214|214|214|214|
|practice_0097|131|133|133|133|131|
|practice_0098|1|2|2|2|1|
|practice_0099|130|132|132|132|130|
|practice_0100|341|342|342|342|341|
|practice_0101|166|168|168|168|166|
|practice_0102|306|307|307|307|306|
|practice_0103|5|6|6|6|5|
|practice_0104|230|230|230|230|230|
|practice_0105|193|195|195|195|193|
|practice_0106|179|180|180|180|179|
|practice_0107|124|125|125|125|124|
|practice_0108|113|119|119|119|113|
|practice_0109|147|150|150|150|147|
|practice_0110|268|268|268|268|268|
|practice_0111|180|183|183|183|180|
|practice_0112|128|129|126|129|126|
|practice_0113|306|306|306|306|306|
|practice_0114|55|57|57|57|55|
|practice_0115|189|189|189|189|189|
|practice_0116|67|71|71|71|67|
|practice_0117|142|143|143|143|142|
|practice_0118|40|43|43|43|40|
|practice_0119|266|267|267|267|266|
|practice_0120|166|167|167|167|166|
|practice_0121|280|280|280|280|280|
|practice_0122|319|319|319|319|319|
|practice_0123|2|3|3|3|2|
|practice_0124|224|226|226|226|224|
|practice_0125|3|4|4|4|3|
|practice_0126|154|156|156|156|154|
|practice_0127|196|196|196|196|196|
|practice_0128|2|4|4|4|2|
|practice_0129|124|125|125|125|124|
|practice_0130|110|111|111|111|110|
|practice_0131|226|228|228|228|226|
|practice_0132|179|179|179|179|179|
|practice_0133|187|189|189|189|187|
|practice_0134|215|215|215|215|215|
|practice_0135|344|344|344|344|344|
|practice_0136|53|56|56|56|53|
|practice_0137|104|106|106|106|104|
|practice_0138|345|346|346|346|345|
|practice_0139|218|218|218|218|218|
|practice_0140|1|1|1|1|1|
|practice_0141|154|157|157|157|154|
|practice_0142|264|264|264|264|264|
|practice_0143|152|156|156|156|152|
|practice_0144|113|116|116|116|113|
|practice_0145|198|207|207|207|198|
|practice_0146|44|47|47|47|44|
|practice_0147|187|188|188|188|187|
|practice_0148|176|176|176|176|176|
|practice_0149|201|203|203|203|201|
|practice_0150|203|203|203|203|203|
|practice_0151|333|333|333|333|333|
|practice_0152|161|162|162|162|161|
|practice_0153|144|149|149|149|144|
|practice_0154|335|336|336|336|335|
|practice_0155|158|160|160|160|158|
|practice_0156|109|111|111|111|109|
|practice_0157|137|146|146|146|137|
|practice_0158|235|239|239|239|235|
|practice_0159|34|39|39|39|34|
|practice_0160|185|187|187|187|185|
|practice_0161|274|274|274|274|274|
|practice_0162|285|285|285|285|285|
|practice_0163|262|263|263|263|262|
|practice_0164|254|256|256|256|254|
|practice_0165|238|244|244|244|238|
|practice_0166|48|54|54|54|48|
|practice_0167|300|300|300|300|300|
|practice_0168|128|130|130|130|128|
|practice_0169|4|6|6|6|4|
|practice_0170|36|41|41|41|36|
|practice_0171|292|292|292|292|292|
|practice_0172|2|2|2|2|2|
|practice_0173|147|147|147|147|147|
|practice_0174|2|2|2|2|2|
|practice_0175|4|6|6|6|4|
|practice_0176|317|318|318|318|317|
|practice_0177|178|179|179|179|178|
|practice_0178|92|93|93|93|92|
|practice_0179|342|342|342|342|342|
|practice_0180|154|155|155|155|154|
|practice_0181|281|281|281|281|281|
|practice_0182|220|220|220|220|220|
|practice_0183|286|287|287|287|286|
|practice_0184|254|256|256|256|254|
|practice_0185|170|177|177|177|170|
|practice_0186|117|118|118|118|117|
|practice_0187|218|218|218|218|218|
|practice_0188|224|224|224|224|224|
|practice_0189|184|187|187|187|184|
|practice_0190|295|295|295|295|295|
|practice_0191|313|313|313|313|313|
|practice_0192|119|125|125|125|119|
|practice_0193|132|141|141|141|132|
|practice_0194|249|249|249|249|249|
|practice_0195|262|262|262|262|262|
|practice_0196|137|137|137|137|137|
|practice_0197|207|209|209|209|207|
|practice_0198|41|47|47|47|41|
|practice_0199|291|291|291|291|291|
|practice_0200|138|145|145|145|138|

## Orden de objetivos y registros virtuales en práctica

200 programas validados: 32892 pasos. Ahorro por familia: {'multiplicative': 208, 'mixed': 1035, 'shared': 672, 'independent': 1073, 'tiny': 0}. [Comparación](reports/target_planner_practice/comparison.csv)

| Capacidad, política | Total | Aporte exclusivo |
|---|---:|---:|
|(0, 0)|36281|0|
|(0, 1)|39962|18|
|(0, 2)|41471|69|
|(48, 0)|37604|0|
|(48, 1)|36446|1|
|(48, 2)|36706|2|
|(128, 0)|37155|89|
|(128, 1)|34416|585|
|(128, 2)|35081|503|

Referencia validada: 35880. Ahorro final: 2988. Empates a favor del histórico.

| Capacidad, política | IDs mejores que la cartera | Ahorro al añadirla sola | Segundos acumulados | Reconstrucciones reales |
|---|---:|---:|---:|---:|
|(0, 0)|0|0|567.6|0|
|(0, 1)|90|589|1640.0|5262|
|(0, 2)|90|672|10716.7|7450|
|(48, 0)|10|26|247.9|0|
|(48, 1)|53|237|675.4|0|
|(48, 2)|41|116|4422.9|0|
|(128, 0)|67|700|1104.5|3552|
|(128, 1)|131|2063|3390.9|1722|
|(128, 2)|118|1750|21545.8|2958|

Totales por familia. Cada columna usa la misma lista de IDs.

| Familia | Histórico | (0, 0) | (0, 1) | (0, 2) | (48, 0) | (48, 1) | (48, 2) | (128, 0) | (128, 1) | (128, 2) | Cartera |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
|independent|12080|12112|15923|16232|12386|12282|12277|12243|11406|11835|11007|
|mixed|10678|10744|11001|12270|10908|10656|10819|11054|10008|10576|9643|
|multiplicative|4958|5065|4981|4851|5250|5149|5042|5205|4985|4858|4750|
|shared|8113|8283|7980|8041|8980|8279|8488|8573|7937|7732|7441|
|tiny|51|77|77|77|80|80|80|80|80|80|51|

Muestra del piloto: practice_0001, 128/anticipación: 105604 KiB RSS pico, 58.0 segundos.

Políticas: 0 fija, 1 menor costo inmediato, 2 anticipación de seis. Capacidad 0: diccionario 52/72. Ancho 6 en todos los casos.

| ID | Familia | Capacidad | Política | Pasos DAG | Operaciones virtuales | Reconstrucciones reales | Total validado | Segundos | Orden |
|---|---|---:|---:|---:|---:|---:|---:|---:|---|
|practice_0001|multiplicative|0|0|154|0|0|154|3.44|879299, 1976630, 2250653, 2657284, 2668286, 5243979, 13130353, 13391328, 17069416, 17277231, 20100162, 27407850, 39394517, 42828032, 52521312, 52710311, 66420364, 94876393, 126556175, 134411316, 177259590, 358356190, 2413698180, 2435461626, 13010454144, 496834172943, 1986426664556, 2930005403136, 4729300430305, 476650284652800|
|practice_0001|multiplicative|0|1|146|0|0|146|5.77|5243979, 879299, 1976630, 134411316, 2250653, 2657284, 2668286, 13130353, 52521312, 17277231, 17069416, 42828032, 13391328, 20100162, 27407850, 39394517, 177259590, 52710311, 66420364, 94876393, 126556175, 358356190, 13010454144, 2413698180, 2435461626, 496834172943, 476650284652800, 1986426664556, 2930005403136, 4729300430305|
|practice_0001|multiplicative|0|2|143|0|0|143|41.77|1976630, 2250653, 13391328, 879299, 13130353, 2657284, 126556175, 177259590, 52521312, 134411316, 2668286, 5243979, 42828032, 17069416, 17277231, 52710311, 20100162, 2413698180, 2435461626, 94876393, 13010454144, 27407850, 66420364, 39394517, 358356190, 496834172943, 2930005403136, 476650284652800, 1986426664556, 4729300430305|
|practice_0001|multiplicative|48|0|164|195|0|164|1.40|879299, 1976630, 2250653, 2657284, 2668286, 5243979, 13130353, 13391328, 17069416, 17277231, 20100162, 27407850, 39394517, 42828032, 52521312, 52710311, 66420364, 94876393, 126556175, 134411316, 177259590, 358356190, 2413698180, 2435461626, 13010454144, 496834172943, 1986426664556, 2930005403136, 4729300430305, 476650284652800|
|practice_0001|multiplicative|48|1|151|157|0|151|2.60|5243979, 879299, 1976630, 134411316, 2250653, 2657284, 2668286, 13130353, 39394517, 52521312, 13391328, 17069416, 42828032, 177259590, 17277231, 20100162, 52710311, 94876393, 27407850, 66420364, 126556175, 358356190, 2435461626, 2413698180, 496834172943, 13010454144, 2930005403136, 476650284652800, 4729300430305, 1986426664556|
|practice_0001|multiplicative|48|2|154|161|0|154|16.57|1976630, 2250653, 13391328, 134411316, 879299, 17069416, 13130353, 2657284, 17277231, 42828032, 2668286, 52521312, 177259590, 5243979, 52710311, 27407850, 39394517, 358356190, 13010454144, 20100162, 2413698180, 126556175, 2435461626, 66420364, 496834172943, 4729300430305, 94876393, 476650284652800, 2930005403136, 1986426664556|
|practice_0001|multiplicative|128|0|149|153|39|188|4.21|879299, 1976630, 2250653, 2657284, 2668286, 5243979, 13130353, 13391328, 17069416, 17277231, 20100162, 27407850, 39394517, 42828032, 52521312, 52710311, 66420364, 94876393, 126556175, 134411316, 177259590, 358356190, 2413698180, 2435461626, 13010454144, 496834172943, 1986426664556, 2930005403136, 4729300430305, 476650284652800|
|practice_0001|multiplicative|128|1|147|150|2|149|8.91|5243979, 879299, 1976630, 134411316, 2250653, 2657284, 2668286, 13130353, 39394517, 52521312, 13391328, 17069416, 20100162, 17277231, 42828032, 27407850, 52710311, 66420364, 94876393, 126556175, 177259590, 358356190, 2435461626, 2413698180, 13010454144, 496834172943, 2930005403136, 476650284652800, 1986426664556, 4729300430305|
|practice_0001|multiplicative|128|2|139|142|5|144|55.52|1976630, 2250653, 13391328, 879299, 13130353, 177259590, 2657284, 52521312, 134411316, 126556175, 17069416, 2668286, 5243979, 39394517, 20100162, 42828032, 52710311, 2435461626, 17277231, 13010454144, 27407850, 66420364, 94876393, 358356190, 2413698180, 496834172943, 4729300430305, 476650284652800, 2930005403136, 1986426664556|
|practice_0002|mixed|0|0|276|0|0|276|3.75|8516318529, 10874959773, 184874317979, 233599606480, 754319359057, 1574501644522, 2160249995729, 3142181818174, 4328234035822, 8052392858847, 10062720315576, 10861337885832, 593872362383328, 1569704580493161, 132806242634852598, 165971298662814561, 197259535980297600, 232678195419217413, 303417905867737193, 377849167282750822, 543640860216664901, 623243760274667821, 627430561331062821, 837175186283075433, 866364152184203677, 899358350072548720, 916714612128359003, 940611078565954180, 963902739619962051|
|practice_0002|mixed|0|1|258|0|65|323|15.46|8516318529, 10874959773, 184874317979, 4328234035822, 233599606480, 754319359057, 1574501644522, 2160249995729, 3142181818174, 8052392858847, 593872362383328, 10062720315576, 10861337885832, 1569704580493161, 165971298662814561, 197259535980297600, 627430561331062821, 132806242634852598, 543640860216664901, 940611078565954180, 623243760274667821, 837175186283075433, 377849167282750822, 303417905867737193, 232678195419217413, 963902739619962051, 866364152184203677, 916714612128359003, 899358350072548720|
|practice_0002|mixed|0|2|251|0|97|348|100.02|184874317979, 2160249995729, 10874959773, 4328234035822, 10062720315576, 754319359057, 8516318529, 10861337885832, 1574501644522, 3142181818174, 940611078565954180, 593872362383328, 233599606480, 377849167282750822, 1569704580493161, 8052392858847, 197259535980297600, 866364152184203677, 165971298662814561, 899358350072548720, 963902739619962051, 132806242634852598, 837175186283075433, 232678195419217413, 623243760274667821, 303417905867737193, 627430561331062821, 916714612128359003, 543640860216664901|
|practice_0002|mixed|48|0|278|383|0|278|1.63|8516318529, 10874959773, 184874317979, 233599606480, 754319359057, 1574501644522, 2160249995729, 3142181818174, 4328234035822, 8052392858847, 10062720315576, 10861337885832, 593872362383328, 1569704580493161, 132806242634852598, 165971298662814561, 197259535980297600, 232678195419217413, 303417905867737193, 377849167282750822, 543640860216664901, 623243760274667821, 627430561331062821, 837175186283075433, 866364152184203677, 899358350072548720, 916714612128359003, 940611078565954180, 963902739619962051|
|practice_0002|mixed|48|1|272|321|0|272|5.88|8516318529, 10874959773, 184874317979, 4328234035822, 754319359057, 10062720315576, 233599606480, 1574501644522, 8052392858847, 3142181818174, 593872362383328, 10861337885832, 132806242634852598, 1569704580493161, 165971298662814561, 197259535980297600, 2160249995729, 627430561331062821, 899358350072548720, 940611078565954180, 232678195419217413, 303417905867737193, 623243760274667821, 377849167282750822, 916714612128359003, 543640860216664901, 837175186283075433, 866364152184203677, 963902739619962051|
|practice_0002|mixed|48|2|272|291|0|272|38.10|8516318529, 10874959773, 1574501644522, 10062720315576, 2160249995729, 184874317979, 233599606480, 8052392858847, 543640860216664901, 10861337885832, 197259535980297600, 132806242634852598, 754319359057, 4328234035822, 3142181818174, 916714612128359003, 899358350072548720, 232678195419217413, 627430561331062821, 1569704580493161, 165971298662814561, 866364152184203677, 940611078565954180, 377849167282750822, 837175186283075433, 963902739619962051, 623243760274667821, 303417905867737193, 593872362383328|
|practice_0002|mixed|128|0|239|257|30|269|7.79|8516318529, 10874959773, 184874317979, 233599606480, 754319359057, 1574501644522, 2160249995729, 3142181818174, 4328234035822, 8052392858847, 10062720315576, 10861337885832, 593872362383328, 1569704580493161, 132806242634852598, 165971298662814561, 197259535980297600, 232678195419217413, 303417905867737193, 377849167282750822, 543640860216664901, 623243760274667821, 627430561331062821, 837175186283075433, 866364152184203677, 899358350072548720, 916714612128359003, 940611078565954180, 963902739619962051|
|practice_0002|mixed|128|1|233|240|25|258|31.20|8516318529, 10874959773, 184874317979, 4328234035822, 233599606480, 754319359057, 2160249995729, 1574501644522, 8052392858847, 3142181818174, 593872362383328, 10062720315576, 10861337885832, 1569704580493161, 837175186283075433, 165971298662814561, 197259535980297600, 623243760274667821, 132806242634852598, 916714612128359003, 627430561331062821, 303417905867737193, 940611078565954180, 232678195419217413, 377849167282750822, 543640860216664901, 866364152184203677, 899358350072548720, 963902739619962051|
|practice_0002|mixed|128|2|234|238|4|238|147.70|8516318529, 233599606480, 184874317979, 1574501644522, 8052392858847, 2160249995729, 3142181818174, 593872362383328, 10874959773, 754319359057, 10062720315576, 10861337885832, 4328234035822, 1569704580493161, 132806242634852598, 916714612128359003, 197259535980297600, 899358350072548720, 303417905867737193, 627430561331062821, 623243760274667821, 866364152184203677, 940611078565954180, 377849167282750822, 543640860216664901, 232678195419217413, 165971298662814561, 837175186283075433, 963902739619962051|
|practice_0003|mixed|0|0|285|0|0|285|4.01|481959288172, 589728097291, 709332826442, 709332827021, 815170626634, 1927837150916, 11567022948000, 69276689530434, 76607945219934, 239754495523452, 312106443101212, 479124367545120, 959337666552403, 8171550809299424, 57085679733753440, 101428606547190628, 127436980505987581, 230899215277772123, 354357624717631369, 418357624139699936, 427839737635638754, 475265238702255955, 501396063024874598, 549247750767182464, 568408321141580916, 727026897435956166, 748804093621534820, 749047838825815947, 856633860385735906, 869937406810341475, 950508132901030667|
|practice_0003|mixed|0|1|274|0|61|335|16.65|481959288172, 1927837150916, 11567022948000, 589728097291, 709332826442, 709332827021, 76607945219934, 239754495523452, 815170626634, 312106443101212, 69276689530434, 479124367545120, 959337666552403, 8171550809299424, 57085679733753440, 127436980505987581, 101428606547190628, 475265238702255955, 749047838825815947, 354357624717631369, 856633860385735906, 568408321141580916, 748804093621534820, 501396063024874598, 418357624139699936, 427839737635638754, 950508132901030667, 549247750767182464, 230899215277772123, 727026897435956166, 869937406810341475|
|practice_0003|mixed|0|2|270|0|305|575|97.36|709332827021, 481959288172, 709332826442, 76607945219934, 1927837150916, 589728097291, 312106443101212, 11567022948000, 239754495523452, 230899215277772123, 479124367545120, 8171550809299424, 959337666552403, 815170626634, 57085679733753440, 69276689530434, 749047838825815947, 549247750767182464, 950508132901030667, 856633860385735906, 418357624139699936, 127436980505987581, 869937406810341475, 101428606547190628, 354357624717631369, 727026897435956166, 568408321141580916, 748804093621534820, 427839737635638754, 475265238702255955, 501396063024874598|
|practice_0003|mixed|48|0|293|433|0|293|2.11|481959288172, 589728097291, 709332826442, 709332827021, 815170626634, 1927837150916, 11567022948000, 69276689530434, 76607945219934, 239754495523452, 312106443101212, 479124367545120, 959337666552403, 8171550809299424, 57085679733753440, 101428606547190628, 127436980505987581, 230899215277772123, 354357624717631369, 418357624139699936, 427839737635638754, 475265238702255955, 501396063024874598, 549247750767182464, 568408321141580916, 727026897435956166, 748804093621534820, 749047838825815947, 856633860385735906, 869937406810341475, 950508132901030667|
|practice_0003|mixed|48|1|288|295|0|288|6.57|481959288172, 1927837150916, 11567022948000, 709332826442, 709332827021, 589728097291, 57085679733753440, 239754495523452, 815170626634, 76607945219934, 312106443101212, 479124367545120, 959337666552403, 69276689530434, 8171550809299424, 418357624139699936, 427839737635638754, 549247750767182464, 727026897435956166, 230899215277772123, 101428606547190628, 354357624717631369, 127436980505987581, 501396063024874598, 568408321141580916, 475265238702255955, 950508132901030667, 869937406810341475, 748804093621534820, 749047838825815947, 856633860385735906|
|practice_0003|mixed|48|2|292|310|0|292|42.89|709332827021, 481959288172, 709332826442, 11567022948000, 1927837150916, 312106443101212, 589728097291, 69276689530434, 76607945219934, 8171550809299424, 479124367545120, 549247750767182464, 57085679733753440, 354357624717631369, 568408321141580916, 748804093621534820, 856633860385735906, 959337666552403, 239754495523452, 101428606547190628, 815170626634, 230899215277772123, 127436980505987581, 869937406810341475, 727026897435956166, 418357624139699936, 749047838825815947, 427839737635638754, 475265238702255955, 950508132901030667, 501396063024874598|
|practice_0003|mixed|128|0|257|261|18|275|8.74|481959288172, 589728097291, 709332826442, 709332827021, 815170626634, 1927837150916, 11567022948000, 69276689530434, 76607945219934, 239754495523452, 312106443101212, 479124367545120, 959337666552403, 8171550809299424, 57085679733753440, 101428606547190628, 127436980505987581, 230899215277772123, 354357624717631369, 418357624139699936, 427839737635638754, 475265238702255955, 501396063024874598, 549247750767182464, 568408321141580916, 727026897435956166, 748804093621534820, 749047838825815947, 856633860385735906, 869937406810341475, 950508132901030667|
|practice_0003|mixed|128|1|246|250|10|256|35.29|481959288172, 1927837150916, 11567022948000, 709332826442, 709332827021, 239754495523452, 76607945219934, 312106443101212, 589728097291, 57085679733753440, 69276689530434, 815170626634, 479124367545120, 959337666552403, 8171550809299424, 101428606547190628, 549247750767182464, 127436980505987581, 230899215277772123, 354357624717631369, 418357624139699936, 427839737635638754, 856633860385735906, 950508132901030667, 475265238702255955, 501396063024874598, 727026897435956166, 568408321141580916, 869937406810341475, 748804093621534820, 749047838825815947|
|practice_0003|mixed|128|2|245|248|3|248|143.69|709332827021, 481959288172, 709332826442, 76607945219934, 312106443101212, 589728097291, 479124367545120, 1927837150916, 239754495523452, 11567022948000, 69276689530434, 815170626634, 57085679733753440, 856633860385735906, 748804093621534820, 959337666552403, 749047838825815947, 8171550809299424, 418357624139699936, 101428606547190628, 427839737635638754, 127436980505987581, 727026897435956166, 549247750767182464, 230899215277772123, 475265238702255955, 869937406810341475, 354357624717631369, 568408321141580916, 501396063024874598, 950508132901030667|
|practice_0004|shared|0|0|130|0|0|130|2.63|1867940, 1868647, 10049962, 10050468, 10973744, 10974352, 10974650, 10975284, 32923888, 59021881, 59022099, 110552090, 118043757, 164614427, 263382624, 301474421, 321605402, 340196489, 452220690, 885327885, 1207143674, 1298480857, 11588835232, 19850225879, 54214059682, 271070298560, 272680976466, 570788212302, 2749056404250|
|practice_0004|shared|0|1|125|0|0|125|4.29|1867940, 1868647, 10050468, 10049962, 110552090, 10973744, 10974352, 10974650, 10975284, 32923888, 263382624, 59022099, 59021881, 118043757, 885327885, 1298480857, 164614427, 301474421, 321605402, 340196489, 452220690, 1207143674, 11588835232, 54214059682, 271070298560, 272680976466, 19850225879, 570788212302, 2749056404250|
|practice_0004|shared|0|2|123|0|0|123|29.53|10049962, 10974352, 10050468, 10974650, 1868647, 59022099, 59021881, 1867940, 10973744, 10975284, 118043757, 1298480857, 885327885, 110552090, 164614427, 263382624, 32923888, 1207143674, 340196489, 452220690, 54214059682, 271070298560, 301474421, 321605402, 11588835232, 19850225879, 272680976466, 570788212302, 2749056404250|
|practice_0004|shared|48|0|138|153|0|138|1.06|1867940, 1868647, 10049962, 10050468, 10973744, 10974352, 10974650, 10975284, 32923888, 59021881, 59022099, 110552090, 118043757, 164614427, 263382624, 301474421, 321605402, 340196489, 452220690, 885327885, 1207143674, 1298480857, 11588835232, 19850225879, 54214059682, 271070298560, 272680976466, 570788212302, 2749056404250|
|practice_0004|shared|48|1|132|142|0|132|2.27|1867940, 1868647, 10050468, 10049962, 110552090, 10973744, 10974352, 10974650, 10975284, 32923888, 59021881, 118043757, 59022099, 885327885, 1298480857, 164614427, 263382624, 321605402, 301474421, 452220690, 340196489, 1207143674, 11588835232, 19850225879, 271070298560, 54214059682, 2749056404250, 272680976466, 570788212302|
|practice_0004|shared|48|2|130|135|0|130|15.18|10973744, 10050468, 10974352, 10049962, 10974650, 1868647, 10975284, 32923888, 1867940, 263382624, 110552090, 1207143674, 59021881, 118043757, 321605402, 452220690, 885327885, 59022099, 1298480857, 164614427, 301474421, 272680976466, 271070298560, 54214059682, 340196489, 2749056404250, 11588835232, 19850225879, 570788212302|
|practice_0004|shared|128|0|124|127|0|124|3.84|1867940, 1868647, 10049962, 10050468, 10973744, 10974352, 10974650, 10975284, 32923888, 59021881, 59022099, 110552090, 118043757, 164614427, 263382624, 301474421, 321605402, 340196489, 452220690, 885327885, 1207143674, 1298480857, 11588835232, 19850225879, 54214059682, 271070298560, 272680976466, 570788212302, 2749056404250|
|practice_0004|shared|128|1|123|124|0|123|6.50|1867940, 1868647, 10050468, 10049962, 110552090, 10973744, 10974352, 10974650, 10975284, 32923888, 263382624, 59022099, 59021881, 118043757, 885327885, 1298480857, 321605402, 164614427, 301474421, 340196489, 452220690, 1207143674, 11588835232, 272680976466, 19850225879, 54214059682, 271070298560, 2749056404250, 570788212302|
|practice_0004|shared|128|2|122|124|0|122|44.15|10973744, 10050468, 10974352, 10049962, 10974650, 263382624, 1868647, 10975284, 1867940, 59021881, 885327885, 59022099, 118043757, 32923888, 110552090, 1298480857, 164614427, 321605402, 1207143674, 271070298560, 54214059682, 340196489, 301474421, 11588835232, 452220690, 272680976466, 570788212302, 19850225879, 2749056404250|
|practice_0005|independent|0|0|234|0|0|234|2.93|146607961507, 216294434497, 271402506729, 306550480001, 353625841148, 355807667938, 429675730099, 453072538348, 468958930589, 473190777331, 529566281785, 584010571551, 586969317716, 588049873221, 591771965592, 604752570444, 671141709269, 681355305647, 682038777914, 696073686656, 725203299914, 755790013203, 774386032165, 778633575109, 808453467749, 823991509479, 862748464039, 882182220678, 900982706167, 906691220173, 936776487205, 969296445604|
|practice_0005|independent|0|1|219|0|0|219|8.05|146607961507, 604752570444, 755790013203, 216294434497, 271402506729, 306550480001, 353625841148, 355807667938, 429675730099, 453072538348, 473190777331, 468958930589, 584010571551, 586969317716, 529566281785, 588049873221, 591771965592, 681355305647, 682038777914, 696073686656, 882182220678, 862748464039, 671141709269, 725203299914, 774386032165, 778633575109, 808453467749, 969296445604, 900982706167, 823991509479, 906691220173, 936776487205|
|practice_0005|independent|0|2|211|0|78|289|49.70|900982706167, 146607961507, 906691220173, 778633575109, 969296445604, 473190777331, 355807667938, 271402506729, 936776487205, 696073686656, 468958930589, 353625841148, 774386032165, 591771965592, 453072538348, 429675730099, 588049873221, 306550480001, 586969317716, 808453467749, 216294434497, 584010571551, 529566281785, 755790013203, 682038777914, 604752570444, 862748464039, 823991509479, 725203299914, 671141709269, 681355305647, 882182220678|
|practice_0005|independent|48|0|233|279|0|233|1.50|146607961507, 216294434497, 271402506729, 306550480001, 353625841148, 355807667938, 429675730099, 453072538348, 468958930589, 473190777331, 529566281785, 584010571551, 586969317716, 588049873221, 591771965592, 604752570444, 671141709269, 681355305647, 682038777914, 696073686656, 725203299914, 755790013203, 774386032165, 778633575109, 808453467749, 823991509479, 862748464039, 882182220678, 900982706167, 906691220173, 936776487205, 969296445604|
|practice_0005|independent|48|1|228|233|0|228|3.72|146607961507, 604752570444, 755790013203, 216294434497, 778633575109, 271402506729, 936776487205, 591771965592, 588049873221, 696073686656, 584010571551, 306550480001, 586969317716, 355807667938, 682038777914, 353625841148, 473190777331, 681355305647, 429675730099, 808453467749, 468958930589, 671141709269, 453072538348, 774386032165, 725203299914, 529566281785, 862748464039, 906691220173, 969296445604, 900982706167, 823991509479, 882182220678|
|practice_0005|independent|48|2|228|235|0|228|25.05|900982706167, 146607961507, 774386032165, 586969317716, 906691220173, 271402506729, 355807667938, 429675730099, 778633575109, 453072538348, 755790013203, 353625841148, 473190777331, 216294434497, 969296445604, 468958930589, 681355305647, 823991509479, 591771965592, 529566281785, 584010571551, 671141709269, 682038777914, 604752570444, 862748464039, 725203299914, 808453467749, 306550480001, 696073686656, 882182220678, 588049873221, 936776487205|
|practice_0005|independent|128|0|212|222|18|230|7.36|146607961507, 216294434497, 271402506729, 306550480001, 353625841148, 355807667938, 429675730099, 453072538348, 468958930589, 473190777331, 529566281785, 584010571551, 586969317716, 588049873221, 591771965592, 604752570444, 671141709269, 681355305647, 682038777914, 696073686656, 725203299914, 755790013203, 774386032165, 778633575109, 808453467749, 823991509479, 862748464039, 882182220678, 900982706167, 906691220173, 936776487205, 969296445604|
|practice_0005|independent|128|1|202|204|1|203|23.18|146607961507, 604752570444, 755790013203, 216294434497, 271402506729, 306550480001, 353625841148, 355807667938, 429675730099, 453072538348, 468958930589, 473190777331, 529566281785, 774386032165, 591771965592, 778633575109, 882182220678, 862748464039, 900982706167, 906691220173, 584010571551, 586969317716, 588049873221, 808453467749, 671141709269, 969296445604, 681355305647, 682038777914, 696073686656, 823991509479, 936776487205, 725203299914|
|practice_0005|independent|128|2|200|203|11|211|90.08|900982706167, 146607961507, 774386032165, 755790013203, 306550480001, 271402506729, 353625841148, 468958930589, 671141709269, 778633575109, 588049873221, 453072538348, 473190777331, 586969317716, 591771965592, 882182220678, 823991509479, 216294434497, 355807667938, 429675730099, 584010571551, 862748464039, 808453467749, 906691220173, 936776487205, 682038777914, 681355305647, 725203299914, 529566281785, 696073686656, 604752570444, 969296445604|
|practice_0006|shared|0|0|40|0|0|40|0.09|45440, 121766, 121851, 327180, 365508, 408321, 654368, 5562128, 22902880|
|practice_0006|shared|0|1|39|0|0|39|0.12|45440, 121851, 121766, 365508, 327180, 654368, 5562128, 22902880, 408321|
|practice_0006|shared|0|2|36|0|0|36|0.52|654368, 121851, 121766, 327180, 5562128, 365508, 22902880, 408321, 45440|
|practice_0006|shared|48|0|41|45|0|41|0.14|45440, 121766, 121851, 327180, 365508, 408321, 654368, 5562128, 22902880|
|practice_0006|shared|48|1|39|43|0|39|0.18|45440, 121851, 121766, 365508, 327180, 654368, 5562128, 22902880, 408321|
|practice_0006|shared|48|2|38|42|0|38|0.51|654368, 121851, 327180, 5562128, 121766, 365508, 22902880, 45440, 408321|
|practice_0006|shared|128|0|41|45|0|41|0.14|45440, 121766, 121851, 327180, 365508, 408321, 654368, 5562128, 22902880|
|practice_0006|shared|128|1|39|43|0|39|0.12|45440, 121851, 121766, 365508, 327180, 654368, 5562128, 22902880, 408321|
|practice_0006|shared|128|2|38|42|0|38|0.44|654368, 121851, 327180, 5562128, 121766, 365508, 22902880, 45440, 408321|
|practice_0007|multiplicative|0|0|106|0|0|106|1.95|2563, 5744, 7894, 15588, 22583, 23368, 26760, 29435, 47658, 51728, 58008, 70116, 74010, 133185, 163267, 197400, 332399, 394800, 458091, 545879, 620368, 990920, 1343162, 2680040, 14933729, 66579288, 122982420, 417300896, 934599600|
|practice_0007|multiplicative|0|1|108|0|0|108|2.62|2563, 23368, 70116, 5744, 51728, 7894, 15588, 22583, 26760, 29435, 47658, 58008, 74010, 133185, 163267, 197400, 394800, 332399, 458091, 545879, 620368, 990920, 1343162, 2680040, 14933729, 66579288, 122982420, 417300896, 934599600|
|practice_0007|multiplicative|0|2|103|0|0|103|15.99|2563, 15588, 26760, 22583, 23368, 5744, 51728, 70116, 7894, 394800, 197400, 74010, 29435, 458091, 47658, 620368, 163267, 990920, 58008, 133185, 332399, 545879, 1343162, 14933729, 2680040, 66579288, 934599600, 122982420, 417300896|
|practice_0007|multiplicative|48|0|110|116|0|110|0.93|2563, 5744, 7894, 15588, 22583, 23368, 26760, 29435, 47658, 51728, 58008, 70116, 74010, 133185, 163267, 197400, 332399, 394800, 458091, 545879, 620368, 990920, 1343162, 2680040, 14933729, 66579288, 122982420, 417300896, 934599600|
|practice_0007|multiplicative|48|1|106|112|0|106|1.30|2563, 23368, 70116, 5744, 51728, 7894, 15588, 620368, 22583, 26760, 29435, 47658, 58008, 133185, 74010, 163267, 197400, 394800, 332399, 458091, 545879, 2680040, 990920, 14933729, 1343162, 122982420, 417300896, 66579288, 934599600|
|practice_0007|multiplicative|48|2|104|110|0|104|7.87|2563, 15588, 26760, 22583, 23368, 5744, 51728, 74010, 7894, 394800, 197400, 58008, 47658, 133185, 545879, 29435, 620368, 458091, 332399, 1343162, 163267, 990920, 70116, 66579288, 2680040, 934599600, 14933729, 122982420, 417300896|
|practice_0007|multiplicative|128|0|106|107|0|106|2.73|2563, 5744, 7894, 15588, 22583, 23368, 26760, 29435, 47658, 51728, 58008, 70116, 74010, 133185, 163267, 197400, 332399, 394800, 458091, 545879, 620368, 990920, 1343162, 2680040, 14933729, 66579288, 122982420, 417300896, 934599600|
|practice_0007|multiplicative|128|1|104|106|0|104|3.88|2563, 23368, 70116, 5744, 51728, 7894, 15588, 620368, 22583, 26760, 29435, 47658, 58008, 74010, 133185, 163267, 197400, 394800, 332399, 458091, 545879, 990920, 1343162, 2680040, 14933729, 122982420, 66579288, 417300896, 934599600|
|practice_0007|multiplicative|128|2|98|100|0|98|20.06|2563, 15588, 26760, 22583, 23368, 5744, 51728, 74010, 7894, 394800, 197400, 47658, 70116, 458091, 545879, 29435, 990920, 58008, 133185, 163267, 332399, 620368, 1343162, 2680040, 66579288, 122982420, 14933729, 417300896, 934599600|
|practice_0008|independent|0|0|258|0|0|258|3.25|103505396216609, 107232565925878, 156649554755697, 204583310080031, 206330452153558, 211583078795603, 219956117018159, 262332388860984, 280627846540070, 295103209319462, 298039064235841, 336834178502701, 361106648638092, 397827070501361, 414753341785582, 418600918836199, 468825964178179, 485523243062699, 485552303839065, 503259035936703, 568823054394876, 596164792087139, 678571446880428, 731342362817470, 761927505635182, 789179975448350, 878971658110199, 904210658806598, 916885071705446|
|practice_0008|independent|0|1|249|0|11|260|11.07|103505396216609, 280627846540070, 219956117018159, 295103209319462, 206330452153558, 107232565925878, 568823054394876, 262332388860984, 204583310080031, 361106648638092, 156649554755697, 298039064235841, 904210658806598, 211583078795603, 418600918836199, 336834178502701, 485552303839065, 485523243062699, 397827070501361, 468825964178179, 916885071705446, 414753341785582, 503259035936703, 596164792087139, 678571446880428, 731342362817470, 761927505635182, 789179975448350, 878971658110199|
|practice_0008|independent|0|2|238|0|43|281|47.67|156649554755697, 280627846540070, 298039064235841, 103505396216609, 904210658806598, 204583310080031, 211583078795603, 568823054394876, 414753341785582, 916885071705446, 219956117018159, 206330452153558, 418600918836199, 361106648638092, 596164792087139, 295103209319462, 485523243062699, 262332388860984, 485552303839065, 878971658110199, 503259035936703, 107232565925878, 397827070501361, 468825964178179, 678571446880428, 761927505635182, 731342362817470, 336834178502701, 789179975448350|
|practice_0008|independent|48|0|274|290|0|274|1.62|103505396216609, 107232565925878, 156649554755697, 204583310080031, 206330452153558, 211583078795603, 219956117018159, 262332388860984, 280627846540070, 295103209319462, 298039064235841, 336834178502701, 361106648638092, 397827070501361, 414753341785582, 418600918836199, 468825964178179, 485523243062699, 485552303839065, 503259035936703, 568823054394876, 596164792087139, 678571446880428, 731342362817470, 761927505635182, 789179975448350, 878971658110199, 904210658806598, 916885071705446|
|practice_0008|independent|48|1|264|269|0|264|4.40|103505396216609, 280627846540070, 262332388860984, 211583078795603, 156649554755697, 204583310080031, 206330452153558, 361106648638092, 485552303839065, 485523243062699, 219956117018159, 336834178502701, 418600918836199, 678571446880428, 107232565925878, 761927505635182, 414753341785582, 397827070501361, 904210658806598, 916885071705446, 503259035936703, 878971658110199, 295103209319462, 298039064235841, 596164792087139, 789179975448350, 468825964178179, 568823054394876, 731342362817470|
|practice_0008|independent|48|2|262|275|0|262|22.01|156649554755697, 414753341785582, 418600918836199, 596164792087139, 916885071705446, 206330452153558, 107232565925878, 211583078795603, 298039064235841, 280627846540070, 397827070501361, 295103209319462, 361106648638092, 103505396216609, 485552303839065, 219956117018159, 336834178502701, 468825964178179, 678571446880428, 761927505635182, 262332388860984, 878971658110199, 568823054394876, 904210658806598, 485523243062699, 731342362817470, 204583310080031, 789179975448350, 503259035936703|
|practice_0008|independent|128|0|234|238|7|241|7.12|103505396216609, 107232565925878, 156649554755697, 204583310080031, 206330452153558, 211583078795603, 219956117018159, 262332388860984, 280627846540070, 295103209319462, 298039064235841, 336834178502701, 361106648638092, 397827070501361, 414753341785582, 418600918836199, 468825964178179, 485523243062699, 485552303839065, 503259035936703, 568823054394876, 596164792087139, 678571446880428, 731342362817470, 761927505635182, 789179975448350, 878971658110199, 904210658806598, 916885071705446|
|practice_0008|independent|128|1|234|240|11|245|22.00|103505396216609, 280627846540070, 219956117018159, 295103209319462, 206330452153558, 107232565925878, 568823054394876, 418600918836199, 211583078795603, 414753341785582, 262332388860984, 156649554755697, 298039064235841, 361106648638092, 485523243062699, 204583310080031, 397827070501361, 678571446880428, 485552303839065, 336834178502701, 503259035936703, 789179975448350, 468825964178179, 761927505635182, 731342362817470, 878971658110199, 596164792087139, 904210658806598, 916885071705446|
|practice_0008|independent|128|2|227|232|64|291|78.09|156649554755697, 280627846540070, 295103209319462, 204583310080031, 468825964178179, 568823054394876, 298039064235841, 211583078795603, 418600918836199, 206330452153558, 219956117018159, 262332388860984, 107232565925878, 103505396216609, 397827070501361, 485552303839065, 361106648638092, 731342362817470, 485523243062699, 596164792087139, 761927505635182, 678571446880428, 414753341785582, 904210658806598, 789179975448350, 336834178502701, 878971658110199, 916885071705446, 503259035936703|
|practice_0009|multiplicative|0|0|139|0|0|139|3.06|508312, 884167, 1731717, 1733899, 5127510, 7957431, 8671222, 14387258, 14592284, 25696175, 29216139, 34481970, 38854818, 55353546, 115367590, 146132183, 184842944, 185669507, 369181472, 463293656, 1075957317, 1897038632, 2185878132, 8536669245, 13437406688, 110845381379, 8492379393544, 10966058519416|
|practice_0009|multiplicative|0|1|140|0|0|140|5.75|508312, 884167, 7957431, 1731717, 1733899, 5127510, 14387258, 8671222, 14592284, 25696175, 29216139, 34481970, 38854818, 55353546, 115367590, 146132183, 184842944, 185669507, 369181472, 463293656, 1897038632, 1075957317, 8536669245, 2185878132, 13437406688, 110845381379, 10966058519416, 8492379393544|
|practice_0009|multiplicative|0|2|140|0|0|140|35.08|1733899, 884167, 508312, 1731717, 184842944, 7957431, 25696175, 5127510, 14387258, 38854818, 34481970, 55353546, 14592284, 115367590, 8671222, 146132183, 185669507, 29216139, 369181472, 1075957317, 1897038632, 463293656, 8536669245, 13437406688, 2185878132, 110845381379, 8492379393544, 10966058519416|
|practice_0009|multiplicative|48|0|145|165|0|145|1.24|508312, 884167, 1731717, 1733899, 5127510, 7957431, 8671222, 14387258, 14592284, 25696175, 29216139, 34481970, 38854818, 55353546, 115367590, 146132183, 184842944, 185669507, 369181472, 463293656, 1075957317, 1897038632, 2185878132, 8536669245, 13437406688, 110845381379, 8492379393544, 10966058519416|
|practice_0009|multiplicative|48|1|146|153|0|146|2.54|508312, 884167, 7957431, 1731717, 1733899, 5127510, 14387258, 8671222, 14592284, 25696175, 29216139, 34481970, 38854818, 55353546, 115367590, 146132183, 184842944, 185669507, 369181472, 463293656, 1075957317, 1897038632, 2185878132, 13437406688, 8536669245, 110845381379, 8492379393544, 10966058519416|
|practice_0009|multiplicative|48|2|143|150|0|143|15.00|1733899, 884167, 508312, 1731717, 7957431, 14387258, 184842944, 29216139, 38854818, 1075957317, 34481970, 5127510, 146132183, 8671222, 25696175, 14592284, 8536669245, 55353546, 115367590, 185669507, 2185878132, 369181472, 10966058519416, 1897038632, 463293656, 13437406688, 110845381379, 8492379393544|
|practice_0009|multiplicative|128|0|136|139|27|163|4.46|508312, 884167, 1731717, 1733899, 5127510, 7957431, 8671222, 14387258, 14592284, 25696175, 29216139, 34481970, 38854818, 55353546, 115367590, 146132183, 184842944, 185669507, 369181472, 463293656, 1075957317, 1897038632, 2185878132, 8536669245, 13437406688, 110845381379, 8492379393544, 10966058519416|
|practice_0009|multiplicative|128|1|138|140|6|144|8.49|508312, 884167, 7957431, 1731717, 1733899, 5127510, 14387258, 8671222, 14592284, 25696175, 29216139, 34481970, 38854818, 55353546, 115367590, 146132183, 184842944, 185669507, 369181472, 463293656, 1897038632, 1075957317, 2185878132, 8536669245, 13437406688, 110845381379, 10966058519416, 8492379393544|
|practice_0009|multiplicative|128|2|134|137|16|150|46.57|1733899, 884167, 508312, 1731717, 5127510, 7957431, 14387258, 8671222, 14592284, 184842944, 25696175, 146132183, 38854818, 185669507, 29216139, 34481970, 8536669245, 55353546, 115367590, 369181472, 1897038632, 1075957317, 2185878132, 463293656, 13437406688, 110845381379, 8492379393544, 10966058519416|
|practice_0010|multiplicative|0|0|141|0|0|141|3.27|95517, 95539, 222400, 271204, 316534, 384266, 436656, 561594, 978464, 1628977, 1633769, 2234341, 2234822, 3437761, 9167572, 13969645, 14299409, 26193103, 27505781, 37980430, 51835700, 80434272, 263657134, 372231888, 1254706012, 1948916550, 2511925003, 2513646408, 6085955433, 36422842895, 200116309906, 310016315399|
|practice_0010|multiplicative|0|1|137|0|0|137|5.28|978464, 95517, 95539, 222400, 316534, 436656, 271204, 384266, 1633769, 1628977, 13969645, 561594, 2234822, 2234341, 14299409, 26193103, 37980430, 3437761, 51835700, 9167572, 27505781, 80434272, 372231888, 263657134, 2511925003, 1254706012, 1948916550, 2513646408, 36422842895, 6085955433, 310016315399, 200116309906|
|practice_0010|multiplicative|0|2|142|0|0|142|33.14|95517, 222400, 271204, 316534, 436656, 1633769, 13969645, 95539, 384266, 14299409, 561594, 2234822, 1628977, 978464, 2234341, 37980430, 9167572, 26193103, 27505781, 80434272, 51835700, 3437761, 372231888, 6085955433, 263657134, 1254706012, 1948916550, 2511925003, 2513646408, 36422842895, 200116309906, 310016315399|
|practice_0010|multiplicative|48|0|151|165|0|151|1.40|95517, 95539, 222400, 271204, 316534, 384266, 436656, 561594, 978464, 1628977, 1633769, 2234341, 2234822, 3437761, 9167572, 13969645, 14299409, 26193103, 27505781, 37980430, 51835700, 80434272, 263657134, 372231888, 1254706012, 1948916550, 2511925003, 2513646408, 6085955433, 36422842895, 200116309906, 310016315399|
|practice_0010|multiplicative|48|1|148|173|0|148|2.43|978464, 95517, 95539, 222400, 316534, 436656, 271204, 384266, 1633769, 1628977, 14299409, 2234341, 2234822, 561594, 3437761, 9167572, 37980430, 13969645, 26193103, 27505781, 51835700, 80434272, 263657134, 372231888, 1254706012, 6085955433, 2513646408, 2511925003, 1948916550, 310016315399, 36422842895, 200116309906|
|practice_0010|multiplicative|48|2|145|153|0|145|14.58|95517, 222400, 271204, 316534, 436656, 1633769, 384266, 14299409, 2234822, 95539, 561594, 9167572, 27505781, 2234341, 978464, 3437761, 26193103, 13969645, 1628977, 263657134, 80434272, 37980430, 51835700, 1254706012, 2511925003, 200116309906, 372231888, 1948916550, 2513646408, 310016315399, 36422842895, 6085955433|
|practice_0010|multiplicative|128|0|137|139|5|142|4.12|95517, 95539, 222400, 271204, 316534, 384266, 436656, 561594, 978464, 1628977, 1633769, 2234341, 2234822, 3437761, 9167572, 13969645, 14299409, 26193103, 27505781, 37980430, 51835700, 80434272, 263657134, 372231888, 1254706012, 1948916550, 2511925003, 2513646408, 6085955433, 36422842895, 200116309906, 310016315399|
|practice_0010|multiplicative|128|1|134|136|0|134|6.90|978464, 95517, 95539, 222400, 316534, 436656, 271204, 384266, 1633769, 1628977, 13969645, 561594, 2234822, 2234341, 14299409, 26193103, 37980430, 27505781, 3437761, 80434272, 9167572, 51835700, 263657134, 372231888, 1254706012, 1948916550, 2511925003, 2513646408, 310016315399, 6085955433, 36422842895, 200116309906|
|practice_0010|multiplicative|128|2|131|134|4|135|41.95|95517, 222400, 271204, 316534, 436656, 1633769, 13969645, 95539, 384266, 14299409, 561594, 2234822, 1628977, 978464, 27505781, 9167572, 80434272, 263657134, 2234341, 3437761, 26193103, 37980430, 372231888, 1254706012, 51835700, 2511925003, 1948916550, 2513646408, 6085955433, 200116309906, 310016315399, 36422842895|
|practice_0011|multiplicative|0|0|128|0|0|128|2.34|68514, 99577, 103329, 149196, 186536, 318811, 344744, 357944, 372329, 826264, 930538, 1158475, 1240908, 1675972, 4017740, 6251666, 7517991, 9372636, 10341434, 20847288, 32904285, 53564769, 62565580, 144625030, 249182080, 291611124, 361847412, 938538720, 1554726747, 5409163188|
|practice_0011|multiplicative|0|1|124|0|0|124|3.76|103329, 68514, 99577, 149196, 186536, 318811, 344744, 357944, 372329, 826264, 930538, 1158475, 1240908, 1675972, 6251666, 7517991, 4017740, 9372636, 10341434, 291611124, 20847288, 32904285, 53564769, 62565580, 144625030, 249182080, 361847412, 938538720, 1554726747, 5409163188|
|practice_0011|multiplicative|0|2|120|0|0|120|21.63|103329, 99577, 68514, 149196, 318811, 826264, 930538, 186536, 344744, 372329, 357944, 1158475, 1240908, 1675972, 6251666, 10341434, 4017740, 32904285, 53564769, 9372636, 20847288, 62565580, 938538720, 361847412, 1554726747, 7517991, 144625030, 249182080, 291611124, 5409163188|
|practice_0011|multiplicative|48|0|131|176|0|131|1.14|68514, 99577, 103329, 149196, 186536, 318811, 344744, 357944, 372329, 826264, 930538, 1158475, 1240908, 1675972, 4017740, 6251666, 7517991, 9372636, 10341434, 20847288, 32904285, 53564769, 62565580, 144625030, 249182080, 291611124, 361847412, 938538720, 1554726747, 5409163188|
|practice_0011|multiplicative|48|1|128|136|0|128|1.76|103329, 68514, 99577, 149196, 344744, 357944, 186536, 318811, 372329, 826264, 930538, 1240908, 7517991, 1158475, 4017740, 1675972, 6251666, 9372636, 20847288, 10341434, 32904285, 62565580, 53564769, 144625030, 361847412, 938538720, 249182080, 291611124, 5409163188, 1554726747|
|practice_0011|multiplicative|48|2|124|131|0|124|10.93|103329, 99577, 149196, 68514, 318811, 930538, 357944, 372329, 344744, 1158475, 7517991, 1240908, 4017740, 20847288, 1675972, 6251666, 10341434, 62565580, 53564769, 826264, 186536, 144625030, 5409163188, 9372636, 361847412, 938538720, 1554726747, 32904285, 249182080, 291611124|
|practice_0011|multiplicative|128|0|124|126|0|124|4.18|68514, 99577, 103329, 149196, 186536, 318811, 344744, 357944, 372329, 826264, 930538, 1158475, 1240908, 1675972, 4017740, 6251666, 7517991, 9372636, 10341434, 20847288, 32904285, 53564769, 62565580, 144625030, 249182080, 291611124, 361847412, 938538720, 1554726747, 5409163188|
|practice_0011|multiplicative|128|1|121|123|0|121|4.85|103329, 68514, 99577, 149196, 344744, 357944, 186536, 318811, 372329, 826264, 930538, 1158475, 1240908, 1675972, 4017740, 7517991, 9372636, 32904285, 361847412, 6251666, 10341434, 20847288, 144625030, 53564769, 62565580, 249182080, 291611124, 938538720, 1554726747, 5409163188|
|practice_0011|multiplicative|128|2|123|125|0|123|29.94|103329, 99577, 149196, 68514, 318811, 930538, 186536, 344744, 357944, 4017740, 372329, 1158475, 1240908, 826264, 1675972, 6251666, 7517991, 9372636, 20847288, 53564769, 10341434, 32904285, 62565580, 144625030, 249182080, 291611124, 361847412, 938538720, 1554726747, 5409163188|
|practice_0012|mixed|0|0|259|0|0|259|4.24|34058962938, 49071111547, 77115656420, 77115656865, 90238267713, 121087300652, 375791995383, 392568892440, 395859353711, 902382686191, 1907302153106, 6497155285036, 34933392640479, 107429188796449508, 139475540331233290, 182111571591108150, 183866138809263553, 267269600419998850, 350513883943790406, 419515162735831450, 459150535226725892, 520343071974103603, 559295560669078282, 590210294551298535, 630949920135572650, 757975375864713885, 803577091367210131, 828456104928847235, 843396976570733797|
|practice_0012|mixed|0|1|246|0|42|288|15.25|77115656420, 77115656865, 34933392640479, 34058962938, 49071111547, 392568892440, 375791995383, 90238267713, 902382686191, 1907302153106, 6497155285036, 121087300652, 395859353711, 139475540331233290, 419515162735831450, 267269600419998850, 559295560669078282, 107429188796449508, 182111571591108150, 183866138809263553, 459150535226725892, 520343071974103603, 350513883943790406, 630949920135572650, 590210294551298535, 757975375864713885, 843396976570733797, 828456104928847235, 803577091367210131|
|practice_0012|mixed|0|2|246|0|51|297|97.06|77115656865, 49071111547, 90238267713, 34058962938, 77115656420, 392568892440, 902382686191, 6497155285036, 375791995383, 34933392640479, 121087300652, 1907302153106, 395859353711, 139475540331233290, 350513883943790406, 107429188796449508, 559295560669078282, 183866138809263553, 520343071974103603, 267269600419998850, 757975375864713885, 419515162735831450, 828456104928847235, 803577091367210131, 182111571591108150, 459150535226725892, 630949920135572650, 590210294551298535, 843396976570733797|
|practice_0012|mixed|48|0|272|388|0|272|1.77|34058962938, 49071111547, 77115656420, 77115656865, 90238267713, 121087300652, 375791995383, 392568892440, 395859353711, 902382686191, 1907302153106, 6497155285036, 34933392640479, 107429188796449508, 139475540331233290, 182111571591108150, 183866138809263553, 267269600419998850, 350513883943790406, 419515162735831450, 459150535226725892, 520343071974103603, 559295560669078282, 590210294551298535, 630949920135572650, 757975375864713885, 803577091367210131, 828456104928847235, 843396976570733797|
|practice_0012|mixed|48|1|254|316|0|254|5.81|77115656420, 77115656865, 34933392640479, 34058962938, 121087300652, 392568892440, 49071111547, 395859353711, 1907302153106, 90238267713, 902382686191, 6497155285036, 375791995383, 182111571591108150, 419515162735831450, 183866138809263553, 139475540331233290, 559295560669078282, 757975375864713885, 630949920135572650, 803577091367210131, 520343071974103603, 267269600419998850, 107429188796449508, 590210294551298535, 350513883943790406, 843396976570733797, 828456104928847235, 459150535226725892|
|practice_0012|mixed|48|2|261|276|0|261|36.83|77115656865, 49071111547, 90238267713, 392568892440, 77115656420, 34058962938, 121087300652, 6497155285036, 1907302153106, 375791995383, 34933392640479, 803577091367210131, 757975375864713885, 902382686191, 459150535226725892, 419515162735831450, 395859353711, 843396976570733797, 267269600419998850, 520343071974103603, 183866138809263553, 828456104928847235, 350513883943790406, 630949920135572650, 559295560669078282, 590210294551298535, 182111571591108150, 139475540331233290, 107429188796449508|
|practice_0012|mixed|128|0|230|246|20|250|8.20|34058962938, 49071111547, 77115656420, 77115656865, 90238267713, 121087300652, 375791995383, 392568892440, 395859353711, 902382686191, 1907302153106, 6497155285036, 34933392640479, 107429188796449508, 139475540331233290, 182111571591108150, 183866138809263553, 267269600419998850, 350513883943790406, 419515162735831450, 459150535226725892, 520343071974103603, 559295560669078282, 590210294551298535, 630949920135572650, 757975375864713885, 803577091367210131, 828456104928847235, 843396976570733797|
|practice_0012|mixed|128|1|228|232|12|240|26.91|77115656420, 77115656865, 34933392640479, 34058962938, 49071111547, 392568892440, 121087300652, 1907302153106, 375791995383, 90238267713, 902382686191, 6497155285036, 395859353711, 139475540331233290, 630949920135572650, 107429188796449508, 183866138809263553, 267269600419998850, 419515162735831450, 182111571591108150, 350513883943790406, 459150535226725892, 520343071974103603, 559295560669078282, 590210294551298535, 803577091367210131, 828456104928847235, 843396976570733797, 757975375864713885|
|practice_0012|mixed|128|2|220|224|17|237|189.60|77115656865, 49071111547, 90238267713, 34058962938, 77115656420, 34933392640479, 121087300652, 392568892440, 395859353711, 902382686191, 6497155285036, 375791995383, 139475540331233290, 1907302153106, 107429188796449508, 459150535226725892, 630949920135572650, 267269600419998850, 590210294551298535, 182111571591108150, 843396976570733797, 183866138809263553, 803577091367210131, 350513883943790406, 419515162735831450, 520343071974103603, 559295560669078282, 757975375864713885, 828456104928847235|
|practice_0013|independent|0|0|217|0|0|217|3.22|159619723056, 181573781924, 222250194736, 231905048317, 250879158693, 255524761237, 282967034343, 283056253923, 283744232868, 302754703671, 314400842820, 340127477416, 342168862546, 355045686357, 481727761415, 556929041413, 588230216071, 598097285693, 603406574450, 604222467857, 647875292453, 681195459362, 726486402994, 731158962823, 745520680642, 795385632250, 887265194644, 905596747371, 925776159949, 932779113462, 987993130817|
|practice_0013|independent|0|1|206|0|0|206|8.18|159619723056, 222250194736, 231905048317, 181573781924, 282967034343, 283056253923, 250879158693, 255524761237, 283744232868, 302754703671, 314400842820, 340127477416, 598097285693, 342168862546, 481727761415, 355045686357, 588230216071, 603406574450, 745520680642, 604222467857, 726486402994, 887265194644, 731158962823, 925776159949, 556929041413, 795385632250, 987993130817, 647875292453, 932779113462, 681195459362, 905596747371|
|practice_0013|independent|0|2|203|0|66|269|54.64|159619723056, 222250194736, 481727761415, 302754703671, 681195459362, 283744232868, 283056253923, 355045686357, 250879158693, 795385632250, 181573781924, 556929041413, 282967034343, 255524761237, 604222467857, 598097285693, 314400842820, 887265194644, 342168862546, 603406574450, 745520680642, 726486402994, 932779113462, 647875292453, 231905048317, 340127477416, 588230216071, 987993130817, 905596747371, 731158962823, 925776159949|
|practice_0013|independent|48|0|219|299|0|219|1.55|159619723056, 181573781924, 222250194736, 231905048317, 250879158693, 255524761237, 282967034343, 283056253923, 283744232868, 302754703671, 314400842820, 340127477416, 342168862546, 355045686357, 481727761415, 556929041413, 588230216071, 598097285693, 603406574450, 604222467857, 647875292453, 681195459362, 726486402994, 731158962823, 745520680642, 795385632250, 887265194644, 905596747371, 925776159949, 932779113462, 987993130817|
|practice_0013|independent|48|1|224|231|0|224|3.64|159619723056, 222250194736, 231905048317, 181573781924, 250879158693, 283744232868, 283056253923, 282967034343, 255524761237, 726486402994, 745520680642, 731158962823, 302754703671, 314400842820, 342168862546, 887265194644, 340127477416, 603406574450, 795385632250, 604222467857, 598097285693, 481727761415, 355045686357, 556929041413, 647875292453, 588230216071, 925776159949, 905596747371, 681195459362, 932779113462, 987993130817|
|practice_0013|independent|48|2|217|226|0|217|26.09|159619723056, 222250194736, 481727761415, 250879158693, 255524761237, 556929041413, 231905048317, 283056253923, 282967034343, 603406574450, 604222467857, 745520680642, 731158962823, 314400842820, 681195459362, 588230216071, 340127477416, 726486402994, 598097285693, 355045686357, 647875292453, 302754703671, 925776159949, 932779113462, 283744232868, 987993130817, 795385632250, 181573781924, 342168862546, 905596747371, 887265194644|
|practice_0013|independent|128|0|203|219|47|250|7.78|159619723056, 181573781924, 222250194736, 231905048317, 250879158693, 255524761237, 282967034343, 283056253923, 283744232868, 302754703671, 314400842820, 340127477416, 342168862546, 355045686357, 481727761415, 556929041413, 588230216071, 598097285693, 603406574450, 604222467857, 647875292453, 681195459362, 726486402994, 731158962823, 745520680642, 795385632250, 887265194644, 905596747371, 925776159949, 932779113462, 987993130817|
|practice_0013|independent|128|1|197|199|0|197|22.22|159619723056, 222250194736, 231905048317, 181573781924, 282967034343, 283056253923, 250879158693, 255524761237, 726486402994, 745520680642, 283744232868, 731158962823, 905596747371, 887265194644, 598097285693, 987993130817, 302754703671, 603406574450, 342168862546, 340127477416, 925776159949, 681195459362, 604222467857, 314400842820, 355045686357, 481727761415, 556929041413, 588230216071, 647875292453, 932779113462, 795385632250|
|practice_0013|independent|128|2|192|196|71|263|123.57|159619723056, 222250194736, 481727761415, 302754703671, 681195459362, 231905048317, 282967034343, 283056253923, 340127477416, 255524761237, 745520680642, 314400842820, 342168862546, 181573781924, 250879158693, 283744232868, 588230216071, 603406574450, 556929041413, 604222467857, 795385632250, 887265194644, 647875292453, 726486402994, 731158962823, 987993130817, 905596747371, 598097285693, 355045686357, 925776159949, 932779113462|
|practice_0014|multiplicative|0|0|138|0|0|138|2.52|118201, 177196, 177508, 319219, 768314, 1189774, 1276500, 1309638, 1536163, 1783595, 3568020, 3900758, 5849959, 5934472, 7915165, 9498454, 17413277, 23744420, 26596655, 54949406, 63236711, 70543081, 164691406, 310419306, 360151246, 492583834, 580165580, 655401936, 1291507952, 1318668988, 7478771560, 8139856416|
|practice_0014|multiplicative|0|1|137|0|0|137|3.25|118201, 1536163, 3900758, 177196, 177508, 319219, 768314, 1189774, 1276500, 1783595, 1309638, 3568020, 5934472, 5849959, 7915165, 9498454, 17413277, 23744420, 1318668988, 26596655, 54949406, 63236711, 70543081, 164691406, 310419306, 360151246, 492583834, 580165580, 7478771560, 655401936, 8139856416, 1291507952|
|practice_0014|multiplicative|0|2|134|0|0|134|21.41|177196, 118201, 319219, 768314, 1536163, 1276500, 1309638, 3568020, 177508, 1189774, 23744420, 7915165, 9498454, 17413277, 70543081, 1783595, 5934472, 3900758, 63236711, 360151246, 5849959, 655401936, 164691406, 26596655, 492583834, 7478771560, 54949406, 310419306, 1291507952, 580165580, 1318668988, 8139856416|
|practice_0014|multiplicative|48|0|152|177|0|152|1.09|118201, 177196, 177508, 319219, 768314, 1189774, 1276500, 1309638, 1536163, 1783595, 3568020, 3900758, 5849959, 5934472, 7915165, 9498454, 17413277, 23744420, 26596655, 54949406, 63236711, 70543081, 164691406, 310419306, 360151246, 492583834, 580165580, 655401936, 1291507952, 1318668988, 7478771560, 8139856416|
|practice_0014|multiplicative|48|1|146|153|0|146|1.41|118201, 1536163, 3900758, 177196, 177508, 319219, 1276500, 768314, 1189774, 3568020, 1309638, 1783595, 5849959, 5934472, 7915165, 9498454, 17413277, 23744420, 26596655, 54949406, 70543081, 360151246, 164691406, 63236711, 310419306, 655401936, 580165580, 1291507952, 492583834, 1318668988, 7478771560, 8139856416|
|practice_0014|multiplicative|48|2|137|147|0|137|11.25|177196, 177508, 118201, 319219, 768314, 1189774, 1276500, 1783595, 5849959, 1309638, 7915165, 3568020, 3900758, 23744420, 54949406, 1536163, 5934472, 1291507952, 17413277, 26596655, 63236711, 1318668988, 70543081, 9498454, 655401936, 580165580, 164691406, 7478771560, 310419306, 360151246, 492583834, 8139856416|
|practice_0014|multiplicative|128|0|136|139|3|139|4.35|118201, 177196, 177508, 319219, 768314, 1189774, 1276500, 1309638, 1536163, 1783595, 3568020, 3900758, 5849959, 5934472, 7915165, 9498454, 17413277, 23744420, 26596655, 54949406, 63236711, 70543081, 164691406, 310419306, 360151246, 492583834, 580165580, 655401936, 1291507952, 1318668988, 7478771560, 8139856416|
|practice_0014|multiplicative|128|1|132|137|0|132|5.37|118201, 1536163, 3900758, 177196, 177508, 319219, 1276500, 1309638, 768314, 1783595, 1189774, 7915165, 9498454, 17413277, 3568020, 5934472, 23744420, 63236711, 492583834, 5849959, 26596655, 54949406, 70543081, 164691406, 310419306, 360151246, 580165580, 655401936, 1291507952, 1318668988, 7478771560, 8139856416|
|practice_0014|multiplicative|128|2|131|133|0|131|38.26|177196, 177508, 118201, 319219, 768314, 1189774, 1276500, 1309638, 1536163, 3568020, 1783595, 3900758, 7915165, 9498454, 23744420, 5849959, 5934472, 63236711, 70543081, 310419306, 1318668988, 17413277, 164691406, 26596655, 54949406, 360151246, 492583834, 580165580, 1291507952, 7478771560, 655401936, 8139856416|
|practice_0015|shared|0|0|205|0|0|205|3.83|81121293247, 81121293758, 85327958960, 122702665592, 148686388722, 272525761460, 1396613578650, 5375661482646, 5486536001311, 7889706860031, 8175772847462, 10836650861326, 13652473605800, 24297516535998, 24297516536151, 34940130265610, 42322668173211, 42945933000362, 54865359920001, 75681372102800, 84645336351489, 218165339610939, 302725487433920, 331742285575588, 479276612251556, 969101611866023, 1873669705300589, 5390849859407089, 5431670631208858, 18787417332725250, 24146099214287910|
|practice_0015|shared|0|1|191|0|0|191|11.14|85327958960, 10836650861326, 5375661482646, 13652473605800, 42322668173211, 84645336351489, 5390849859407089, 81121293758, 81121293247, 122702665592, 218165339610939, 479276612251556, 42945933000362, 1873669705300589, 148686388722, 302725487433920, 75681372102800, 54865359920001, 272525761460, 8175772847462, 24297516535998, 24297516536151, 5486536001311, 7889706860031, 34940130265610, 18787417332725250, 1396613578650, 5431670631208858, 331742285575588, 969101611866023, 24146099214287910|
|practice_0015|shared|0|2|194|0|24|218|65.56|85327958960, 122702665592, 10836650861326, 84645336351489, 5375661482646, 81121293758, 218165339610939, 81121293247, 42322668173211, 148686388722, 1873669705300589, 302725487433920, 75681372102800, 5390849859407089, 479276612251556, 272525761460, 5486536001311, 54865359920001, 18787417332725250, 24297516536151, 24297516535998, 42945933000362, 7889706860031, 8175772847462, 5431670631208858, 34940130265610, 13652473605800, 1396613578650, 331742285575588, 969101611866023, 24146099214287910|
|practice_0015|shared|48|0|245|304|0|245|1.79|81121293247, 81121293758, 85327958960, 122702665592, 148686388722, 272525761460, 1396613578650, 5375661482646, 5486536001311, 7889706860031, 8175772847462, 10836650861326, 13652473605800, 24297516535998, 24297516536151, 34940130265610, 42322668173211, 42945933000362, 54865359920001, 75681372102800, 84645336351489, 218165339610939, 302725487433920, 331742285575588, 479276612251556, 969101611866023, 1873669705300589, 5390849859407089, 5431670631208858, 18787417332725250, 24146099214287910|
|practice_0015|shared|48|1|208|222|0|208|4.95|85327958960, 10836650861326, 5375661482646, 13652473605800, 42322668173211, 84645336351489, 5390849859407089, 81121293758, 81121293247, 272525761460, 8175772847462, 1396613578650, 122702665592, 42945933000362, 148686388722, 18787417332725250, 302725487433920, 75681372102800, 1873669705300589, 218165339610939, 24297516535998, 24297516536151, 7889706860031, 479276612251556, 331742285575588, 34940130265610, 54865359920001, 5486536001311, 5431670631208858, 969101611866023, 24146099214287910|
|practice_0015|shared|48|2|229|235|0|229|34.24|85327958960, 122702665592, 218165339610939, 5375661482646, 42945933000362, 13652473605800, 10836650861326, 24297516536151, 24297516535998, 81121293247, 81121293758, 1396613578650, 5486536001311, 84645336351489, 54865359920001, 272525761460, 42322668173211, 331742285575588, 5431670631208858, 8175772847462, 7889706860031, 479276612251556, 5390849859407089, 75681372102800, 302725487433920, 18787417332725250, 24146099214287910, 969101611866023, 148686388722, 1873669705300589, 34940130265610|
|practice_0015|shared|128|0|203|209|26|229|6.14|81121293247, 81121293758, 85327958960, 122702665592, 148686388722, 272525761460, 1396613578650, 5375661482646, 5486536001311, 7889706860031, 8175772847462, 10836650861326, 13652473605800, 24297516535998, 24297516536151, 34940130265610, 42322668173211, 42945933000362, 54865359920001, 75681372102800, 84645336351489, 218165339610939, 302725487433920, 331742285575588, 479276612251556, 969101611866023, 1873669705300589, 5390849859407089, 5431670631208858, 18787417332725250, 24146099214287910|
|practice_0015|shared|128|1|186|191|0|186|21.41|85327958960, 10836650861326, 5375661482646, 13652473605800, 42322668173211, 84645336351489, 5390849859407089, 148686388722, 302725487433920, 75681372102800, 18787417332725250, 81121293247, 81121293758, 122702665592, 1396613578650, 42945933000362, 479276612251556, 218165339610939, 1873669705300589, 272525761460, 8175772847462, 5486536001311, 54865359920001, 969101611866023, 5431670631208858, 24297516535998, 24297516536151, 7889706860031, 34940130265610, 331742285575588, 24146099214287910|
|practice_0015|shared|128|2|196|198|0|196|141.09|85327958960, 122702665592, 84645336351489, 81121293247, 148686388722, 81121293758, 8175772847462, 75681372102800, 1873669705300589, 42322668173211, 302725487433920, 13652473605800, 272525761460, 10836650861326, 42945933000362, 5486536001311, 5375661482646, 1396613578650, 54865359920001, 218165339610939, 18787417332725250, 24297516536151, 24297516535998, 5390849859407089, 5431670631208858, 479276612251556, 34940130265610, 969101611866023, 7889706860031, 331742285575588, 24146099214287910|
|practice_0016|shared|0|0|235|0|0|235|3.72|110969540114, 148535173516, 429998349249, 848202313980, 855003513910, 2330360318895, 2362764330460, 4523745701804, 4673626058503, 9637458486285, 27572482030548, 61797598119180, 246435679378920, 266175031040433, 278547389553409, 336501074961072, 371812280520960, 655668426190778, 832905892521152, 985742717496358, 2616098311960058, 5107605618198618, 5833078542827049, 20325721196104366, 62132233254724647, 94444598409739200, 111403972323154822, 346358127698668800, 568515221558565060|
|practice_0016|shared|0|1|221|0|0|221|13.40|110969540114, 2330360318895, 2362764330460, 429998349249, 371812280520960, 148535173516, 855003513910, 246435679378920, 985742717496358, 27572482030548, 848202313980, 4523745701804, 266175031040433, 4673626058503, 336501074961072, 9637458486285, 61797598119180, 346358127698668800, 278547389553409, 568515221558565060, 111403972323154822, 655668426190778, 20325721196104366, 832905892521152, 5833078542827049, 2616098311960058, 5107605618198618, 62132233254724647, 94444598409739200|
|practice_0016|shared|0|2|226|0|186|412|89.46|2330360318895, 429998349249, 855003513910, 110969540114, 848202313980, 111403972323154822, 148535173516, 266175031040433, 2362764330460, 346358127698668800, 4673626058503, 61797598119180, 246435679378920, 985742717496358, 371812280520960, 4523745701804, 655668426190778, 20325721196104366, 336501074961072, 27572482030548, 9637458486285, 5833078542827049, 2616098311960058, 832905892521152, 94444598409739200, 568515221558565060, 278547389553409, 5107605618198618, 62132233254724647|
|practice_0016|shared|48|0|255|287|0|255|1.77|110969540114, 148535173516, 429998349249, 848202313980, 855003513910, 2330360318895, 2362764330460, 4523745701804, 4673626058503, 9637458486285, 27572482030548, 61797598119180, 246435679378920, 266175031040433, 278547389553409, 336501074961072, 371812280520960, 655668426190778, 832905892521152, 985742717496358, 2616098311960058, 5107605618198618, 5833078542827049, 20325721196104366, 62132233254724647, 94444598409739200, 111403972323154822, 346358127698668800, 568515221558565060|
|practice_0016|shared|48|1|227|233|0|227|4.58|110969540114, 2330360318895, 2362764330460, 429998349249, 371812280520960, 148535173516, 855003513910, 246435679378920, 27572482030548, 985742717496358, 848202313980, 4523745701804, 9637458486285, 2616098311960058, 111403972323154822, 266175031040433, 346358127698668800, 4673626058503, 336501074961072, 61797598119180, 655668426190778, 278547389553409, 94444598409739200, 832905892521152, 5833078542827049, 20325721196104366, 5107605618198618, 62132233254724647, 568515221558565060|
|practice_0016|shared|48|2|247|256|0|247|32.47|2330360318895, 110969540114, 848202313980, 27572482030548, 148535173516, 61797598119180, 371812280520960, 4523745701804, 278547389553409, 346358127698668800, 266175031040433, 855003513910, 655668426190778, 9637458486285, 246435679378920, 2362764330460, 832905892521152, 985742717496358, 5833078542827049, 62132233254724647, 5107605618198618, 94444598409739200, 429998349249, 336501074961072, 111403972323154822, 20325721196104366, 568515221558565060, 2616098311960058, 4673626058503|
|practice_0016|shared|128|0|215|221|23|238|7.33|110969540114, 148535173516, 429998349249, 848202313980, 855003513910, 2330360318895, 2362764330460, 4523745701804, 4673626058503, 9637458486285, 27572482030548, 61797598119180, 246435679378920, 266175031040433, 278547389553409, 336501074961072, 371812280520960, 655668426190778, 832905892521152, 985742717496358, 2616098311960058, 5107605618198618, 5833078542827049, 20325721196104366, 62132233254724647, 94444598409739200, 111403972323154822, 346358127698668800, 568515221558565060|
|practice_0016|shared|128|1|204|207|5|209|23.27|110969540114, 2330360318895, 2362764330460, 429998349249, 371812280520960, 148535173516, 855003513910, 246435679378920, 985742717496358, 848202313980, 27572482030548, 61797598119180, 4523745701804, 266175031040433, 9637458486285, 346358127698668800, 4673626058503, 336501074961072, 111403972323154822, 278547389553409, 832905892521152, 655668426190778, 20325721196104366, 2616098311960058, 94444598409739200, 5833078542827049, 5107605618198618, 62132233254724647, 568515221558565060|
|practice_0016|shared|128|2|205|208|20|225|177.53|2330360318895, 429998349249, 27572482030548, 848202313980, 111403972323154822, 110969540114, 148535173516, 346358127698668800, 855003513910, 655668426190778, 20325721196104366, 94444598409739200, 2362764330460, 246435679378920, 985742717496358, 4523745701804, 9637458486285, 336501074961072, 4673626058503, 266175031040433, 2616098311960058, 371812280520960, 568515221558565060, 61797598119180, 278547389553409, 832905892521152, 5107605618198618, 62132233254724647, 5833078542827049|
|practice_0017|shared|0|0|176|0|0|176|3.64|668711074, 820650610, 950196725, 1551957537, 1551959076, 2205241192, 2724479375, 4012235298, 4420253796, 4420257297, 4655874919, 9291167284, 9311743941, 11026204851, 13231445757, 17641927592, 28505771328, 30406155105, 35362041023, 36306024677, 70291539001, 99104987776, 490298178294, 1144520061062, 1388905078716, 5702129239642, 6448391117949, 13629365315298, 1043601951470979|
|practice_0017|shared|0|1|169|0|0|169|7.38|950196725, 820650610, 668711074, 1551959076, 1551957537, 4655874919, 9311743941, 2205241192, 11026204851, 13231445757, 17641927592, 2724479375, 4420253796, 4420257297, 35362041023, 28505771328, 4012235298, 9291167284, 30406155105, 36306024677, 99104987776, 1144520061062, 490298178294, 70291539001, 1388905078716, 5702129239642, 6448391117949, 13629365315298, 1043601951470979|
|practice_0017|shared|0|2|169|0|0|169|48.19|2205241192, 1551959076, 13231445757, 950196725, 820650610, 11026204851, 4655874919, 1551957537, 17641927592, 4420253796, 9311743941, 35362041023, 2724479375, 4420257297, 28505771328, 4012235298, 30406155105, 70291539001, 9291167284, 6448391117949, 668711074, 99104987776, 36306024677, 13629365315298, 1144520061062, 1388905078716, 490298178294, 5702129239642, 1043601951470979|
|practice_0017|shared|48|0|189|234|0|189|1.36|668711074, 820650610, 950196725, 1551957537, 1551959076, 2205241192, 2724479375, 4012235298, 4420253796, 4420257297, 4655874919, 9291167284, 9311743941, 11026204851, 13231445757, 17641927592, 28505771328, 30406155105, 35362041023, 36306024677, 70291539001, 99104987776, 490298178294, 1144520061062, 1388905078716, 5702129239642, 6448391117949, 13629365315298, 1043601951470979|
|practice_0017|shared|48|1|184|192|0|184|3.41|950196725, 820650610, 668711074, 1551959076, 1551957537, 4420253796, 4420257297, 2205241192, 35362041023, 4655874919, 9311743941, 13231445757, 11026204851, 17641927592, 28505771328, 9291167284, 4012235298, 30406155105, 2724479375, 36306024677, 70291539001, 99104987776, 6448391117949, 1388905078716, 1144520061062, 490298178294, 5702129239642, 1043601951470979, 13629365315298|
|practice_0017|shared|48|2|174|185|0|174|22.99|950196725, 1551959076, 1551957537, 13231445757, 2205241192, 17641927592, 4655874919, 11026204851, 9311743941, 4420257297, 4420253796, 668711074, 820650610, 4012235298, 9291167284, 28505771328, 36306024677, 99104987776, 70291539001, 1388905078716, 1144520061062, 35362041023, 2724479375, 30406155105, 13629365315298, 5702129239642, 6448391117949, 1043601951470979, 490298178294|
|practice_0017|shared|128|0|162|168|17|179|5.44|668711074, 820650610, 950196725, 1551957537, 1551959076, 2205241192, 2724479375, 4012235298, 4420253796, 4420257297, 4655874919, 9291167284, 9311743941, 11026204851, 13231445757, 17641927592, 28505771328, 30406155105, 35362041023, 36306024677, 70291539001, 99104987776, 490298178294, 1144520061062, 1388905078716, 5702129239642, 6448391117949, 13629365315298, 1043601951470979|
|practice_0017|shared|128|1|163|166|13|176|12.08|950196725, 820650610, 668711074, 1551959076, 1551957537, 4655874919, 9311743941, 4012235298, 2724479375, 2205241192, 11026204851, 13231445757, 17641927592, 4420253796, 4420257297, 35362041023, 36306024677, 9291167284, 28505771328, 30406155105, 70291539001, 99104987776, 490298178294, 1144520061062, 1388905078716, 5702129239642, 6448391117949, 13629365315298, 1043601951470979|
|practice_0017|shared|128|2|149|156|9|158|64.09|950196725, 1551959076, 1551957537, 13231445757, 2205241192, 668711074, 9311743941, 820650610, 4420253796, 4420257297, 17641927592, 11026204851, 28505771328, 35362041023, 30406155105, 4012235298, 4655874919, 9291167284, 1144520061062, 6448391117949, 1388905078716, 2724479375, 99104987776, 13629365315298, 36306024677, 490298178294, 70291539001, 5702129239642, 1043601951470979|
|practice_0018|independent|0|0|342|0|0|342|3.23|126228171410710972, 157592711953265835, 165507328511749093, 192183915987565593, 215225737616790469, 262639944403659219, 279040815654835287, 289913349334568854, 357693944950829393, 375951886511139796, 422494409593870166, 440316733423865547, 446155069253164346, 459025614936654288, 460505151512157935, 506157417214062026, 591052217600522658, 606897752975948289, 618750321094544663, 690629664294524028, 701361938094763772, 703465969115262120, 729010110497677120, 761662087534074873, 771077168239478076, 780146223698445288, 803403215236753544, 851413708445244965, 871916309956513301, 917828578112990712, 944217031140315202, 985642966055347110|
|practice_0018|independent|0|1|320|0|309|629|16.72|729010110497677120, 126228171410710972, 944217031140315202, 459025614936654288, 215225737616790469, 165507328511749093, 157592711953265835, 279040815654835287, 262639944403659219, 289913349334568854, 460505151512157935, 703465969115262120, 422494409593870166, 701361938094763772, 192183915987565593, 506157417214062026, 357693944950829393, 446155069253164346, 780146223698445288, 440316733423865547, 618750321094544663, 761662087534074873, 375951886511139796, 690629664294524028, 606897752975948289, 803403215236753544, 591052217600522658, 985642966055347110, 771077168239478076, 917828578112990712, 871916309956513301, 851413708445244965|
|practice_0018|independent|0|2|325|0|227|552|116.10|126228171410710972, 506157417214062026, 215225737616790469, 618750321094544663, 262639944403659219, 422494409593870166, 192183915987565593, 459025614936654288, 165507328511749093, 357693944950829393, 460505151512157935, 780146223698445288, 761662087534074873, 803403215236753544, 701361938094763772, 591052217600522658, 289913349334568854, 375951886511139796, 446155069253164346, 851413708445244965, 729010110497677120, 703465969115262120, 279040815654835287, 157592711953265835, 690629664294524028, 944217031140315202, 440316733423865547, 871916309956513301, 606897752975948289, 917828578112990712, 771077168239478076, 985642966055347110|
|practice_0018|independent|48|0|358|487|0|358|2.02|126228171410710972, 157592711953265835, 165507328511749093, 192183915987565593, 215225737616790469, 262639944403659219, 279040815654835287, 289913349334568854, 357693944950829393, 375951886511139796, 422494409593870166, 440316733423865547, 446155069253164346, 459025614936654288, 460505151512157935, 506157417214062026, 591052217600522658, 606897752975948289, 618750321094544663, 690629664294524028, 701361938094763772, 703465969115262120, 729010110497677120, 761662087534074873, 771077168239478076, 780146223698445288, 803403215236753544, 851413708445244965, 871916309956513301, 917828578112990712, 944217031140315202, 985642966055347110|
|practice_0018|independent|48|1|358|465|0|358|6.38|729010110497677120, 126228171410710972, 289913349334568854, 279040815654835287, 761662087534074873, 215225737616790469, 165507328511749093, 917828578112990712, 690629664294524028, 446155069253164346, 422494409593870166, 192183915987565593, 157592711953265835, 262639944403659219, 606897752975948289, 771077168239478076, 703465969115262120, 460505151512157935, 459025614936654288, 506157417214062026, 440316733423865547, 375951886511139796, 780146223698445288, 803403215236753544, 357693944950829393, 985642966055347110, 618750321094544663, 701361938094763772, 591052217600522658, 851413708445244965, 944217031140315202, 871916309956513301|
|practice_0018|independent|48|2|348|377|0|348|43.19|165507328511749093, 606897752975948289, 460505151512157935, 192183915987565593, 157592711953265835, 771077168239478076, 422494409593870166, 279040815654835287, 985642966055347110, 803403215236753544, 126228171410710972, 851413708445244965, 917828578112990712, 459025614936654288, 618750321094544663, 703465969115262120, 357693944950829393, 729010110497677120, 262639944403659219, 871916309956513301, 690629664294524028, 440316733423865547, 215225737616790469, 761662087534074873, 780146223698445288, 446155069253164346, 289913349334568854, 506157417214062026, 591052217600522658, 944217031140315202, 375951886511139796, 701361938094763772|
|practice_0018|independent|128|0|315|342|46|361|10.73|126228171410710972, 157592711953265835, 165507328511749093, 192183915987565593, 215225737616790469, 262639944403659219, 279040815654835287, 289913349334568854, 357693944950829393, 375951886511139796, 422494409593870166, 440316733423865547, 446155069253164346, 459025614936654288, 460505151512157935, 506157417214062026, 591052217600522658, 606897752975948289, 618750321094544663, 690629664294524028, 701361938094763772, 703465969115262120, 729010110497677120, 761662087534074873, 771077168239478076, 780146223698445288, 803403215236753544, 851413708445244965, 871916309956513301, 917828578112990712, 944217031140315202, 985642966055347110|
|practice_0018|independent|128|1|303|304|141|444|44.30|729010110497677120, 126228171410710972, 165507328511749093, 262639944403659219, 192183915987565593, 357693944950829393, 215225737616790469, 157592711953265835, 761662087534074873, 780146223698445288, 803403215236753544, 422494409593870166, 440316733423865547, 459025614936654288, 771077168239478076, 289913349334568854, 279040815654835287, 375951886511139796, 446155069253164346, 460505151512157935, 606897752975948289, 690629664294524028, 506157417214062026, 591052217600522658, 944217031140315202, 618750321094544663, 701361938094763772, 703465969115262120, 851413708445244965, 871916309956513301, 917828578112990712, 985642966055347110|
|practice_0018|independent|128|2|296|299|8|304|322.82|126228171410710972, 506157417214062026, 165507328511749093, 440316733423865547, 215225737616790469, 289913349334568854, 279040815654835287, 703465969115262120, 157592711953265835, 761662087534074873, 422494409593870166, 729010110497677120, 262639944403659219, 459025614936654288, 780146223698445288, 357693944950829393, 690629664294524028, 460505151512157935, 591052217600522658, 701361938094763772, 606897752975948289, 618750321094544663, 803403215236753544, 771077168239478076, 944217031140315202, 446155069253164346, 917828578112990712, 375951886511139796, 985642966055347110, 871916309956513301, 192183915987565593, 851413708445244965|
|practice_0019|shared|0|0|211|0|0|211|3.83|8599595524, 36553053502, 36553055042, 50923917939, 50923942293, 50923948373, 155370752035, 216455061762, 216455062941, 216455074468, 466112330946, 731061068760, 1222174770312, 1754546570304, 4971215277904, 7018186274712, 35090931373440, 156743914287896, 313589823430908, 478231176524346, 900447920266962, 900484454596316, 1913701863868613, 2025185376894264, 3713948177641405, 7527068866215624, 10170930359622918, 22283689039920168, 22940184513345972, 259307946477948132, 367445673343580353, 964601844803443946|
|practice_0019|shared|0|1|201|0|0|201|10.61|8599595524, 36553053502, 731061068760, 36553055042, 1754546570304, 4971215277904, 35090931373440, 7018186274712, 900447920266962, 2025185376894264, 259307946477948132, 155370752035, 466112330946, 478231176524346, 50923917939, 50923942293, 50923948373, 1222174770312, 156743914287896, 7527068866215624, 900484454596316, 216455061762, 216455062941, 216455074468, 313589823430908, 1913701863868613, 3713948177641405, 22283689039920168, 10170930359622918, 22940184513345972, 367445673343580353, 964601844803443946|
|practice_0019|shared|0|2|196|0|0|196|86.93|36553053502, 35090931373440, 7018186274712, 36553055042, 8599595524, 50923948373, 1222174770312, 259307946477948132, 50923942293, 155370752035, 731061068760, 4971215277904, 1754546570304, 216455062941, 156743914287896, 216455061762, 216455074468, 50923917939, 2025185376894264, 964601844803443946, 466112330946, 900447920266962, 478231176524346, 900484454596316, 313589823430908, 1913701863868613, 22283689039920168, 3713948177641405, 22940184513345972, 10170930359622918, 7527068866215624, 367445673343580353|
|practice_0019|shared|48|0|228|288|0|228|1.65|8599595524, 36553053502, 36553055042, 50923917939, 50923942293, 50923948373, 155370752035, 216455061762, 216455062941, 216455074468, 466112330946, 731061068760, 1222174770312, 1754546570304, 4971215277904, 7018186274712, 35090931373440, 156743914287896, 313589823430908, 478231176524346, 900447920266962, 900484454596316, 1913701863868613, 2025185376894264, 3713948177641405, 7527068866215624, 10170930359622918, 22283689039920168, 22940184513345972, 259307946477948132, 367445673343580353, 964601844803443946|
|practice_0019|shared|48|1|194|237|0|194|4.12|8599595524, 36553053502, 731061068760, 36553055042, 1754546570304, 4971215277904, 35090931373440, 7018186274712, 900447920266962, 2025185376894264, 259307946477948132, 900484454596316, 216455074468, 216455061762, 216455062941, 50923948373, 50923942293, 50923917939, 1222174770312, 7527068866215624, 156743914287896, 155370752035, 466112330946, 478231176524346, 1913701863868613, 313589823430908, 22940184513345972, 367445673343580353, 10170930359622918, 22283689039920168, 3713948177641405, 964601844803443946|
|practice_0019|shared|48|2|215|225|0|215|34.04|36553053502, 35090931373440, 7018186274712, 259307946477948132, 36553055042, 1754546570304, 4971215277904, 50923942293, 1222174770312, 900447920266962, 156743914287896, 50923917939, 50923948373, 8599595524, 313589823430908, 2025185376894264, 731061068760, 466112330946, 7527068866215624, 1913701863868613, 216455061762, 22283689039920168, 3713948177641405, 216455062941, 155370752035, 22940184513345972, 216455074468, 964601844803443946, 900484454596316, 478231176524346, 10170930359622918, 367445673343580353|
|practice_0019|shared|128|0|180|187|2|182|6.99|8599595524, 36553053502, 36553055042, 50923917939, 50923942293, 50923948373, 155370752035, 216455061762, 216455062941, 216455074468, 466112330946, 731061068760, 1222174770312, 1754546570304, 4971215277904, 7018186274712, 35090931373440, 156743914287896, 313589823430908, 478231176524346, 900447920266962, 900484454596316, 1913701863868613, 2025185376894264, 3713948177641405, 7527068866215624, 10170930359622918, 22283689039920168, 22940184513345972, 259307946477948132, 367445673343580353, 964601844803443946|
|practice_0019|shared|128|1|177|188|74|251|24.43|8599595524, 36553053502, 731061068760, 36553055042, 1754546570304, 4971215277904, 35090931373440, 7018186274712, 259307946477948132, 50923942293, 50923948373, 50923917939, 1222174770312, 156743914287896, 900447920266962, 313589823430908, 7527068866215624, 155370752035, 466112330946, 478231176524346, 216455061762, 216455062941, 216455074468, 900484454596316, 2025185376894264, 22283689039920168, 3713948177641405, 22940184513345972, 964601844803443946, 1913701863868613, 10170930359622918, 367445673343580353|
|practice_0019|shared|128|2|176|180|0|176|127.55|36553053502, 35090931373440, 7018186274712, 36553055042, 259307946477948132, 50923942293, 50923948373, 1222174770312, 156743914287896, 7527068866215624, 731061068760, 50923917939, 1754546570304, 155370752035, 4971215277904, 216455061762, 216455062941, 216455074468, 313589823430908, 900447920266962, 478231176524346, 466112330946, 900484454596316, 8599595524, 3713948177641405, 22283689039920168, 2025185376894264, 964601844803443946, 10170930359622918, 367445673343580353, 1913701863868613, 22940184513345972|
|practice_0020|multiplicative|0|0|114|0|0|114|2.85|2033, 4434, 5805, 7862, 9267, 12407, 13000, 15452, 37690, 123853, 181564, 590410, 734725, 919963, 959812, 1005420, 1404620, 1800256, 4686110, 9358689, 15208776, 66614955, 67177778, 122732120, 819281127, 73293337600, 107664524000, 1006774940288|
|practice_0020|multiplicative|0|1|112|0|0|112|4.01|2033, 4434, 9267, 7862, 5805, 12407, 15452, 13000, 37690, 123853, 1800256, 181564, 590410, 734725, 919963, 959812, 1005420, 1404620, 4686110, 9358689, 15208776, 66614955, 67177778, 122732120, 819281127, 73293337600, 107664524000, 1006774940288|
|practice_0020|multiplicative|0|2|109|0|0|109|24.67|9267, 7862, 4434, 15452, 12407, 13000, 2033, 5805, 734725, 37690, 1404620, 1005420, 1800256, 590410, 4686110, 9358689, 123853, 181564, 959812, 919963, 66614955, 67177778, 15208776, 122732120, 819281127, 73293337600, 107664524000, 1006774940288|
|practice_0020|multiplicative|48|0|112|128|0|112|1.04|2033, 4434, 5805, 7862, 9267, 12407, 13000, 15452, 37690, 123853, 181564, 590410, 734725, 919963, 959812, 1005420, 1404620, 1800256, 4686110, 9358689, 15208776, 66614955, 67177778, 122732120, 819281127, 73293337600, 107664524000, 1006774940288|
|practice_0020|multiplicative|48|1|115|122|0|115|1.72|2033, 13000, 4434, 9267, 5805, 15452, 7862, 12407, 37690, 123853, 181564, 734725, 919963, 959812, 1005420, 1404620, 4686110, 590410, 1800256, 9358689, 15208776, 66614955, 67177778, 122732120, 819281127, 73293337600, 107664524000, 1006774940288|
|practice_0020|multiplicative|48|2|111|118|0|111|11.26|9267, 7862, 5805, 13000, 4434, 2033, 15452, 37690, 12407, 123853, 181564, 734725, 590410, 919963, 1005420, 1800256, 67177778, 959812, 15208776, 66614955, 1404620, 4686110, 9358689, 122732120, 819281127, 73293337600, 107664524000, 1006774940288|
|practice_0020|multiplicative|128|0|110|113|0|110|3.31|2033, 4434, 5805, 7862, 9267, 12407, 13000, 15452, 37690, 123853, 181564, 590410, 734725, 919963, 959812, 1005420, 1404620, 1800256, 4686110, 9358689, 15208776, 66614955, 67177778, 122732120, 819281127, 73293337600, 107664524000, 1006774940288|
|practice_0020|multiplicative|128|1|108|113|0|108|4.84|2033, 13000, 4434, 9267, 5805, 15452, 7862, 12407, 37690, 123853, 181564, 590410, 734725, 919963, 959812, 1005420, 1404620, 1800256, 4686110, 9358689, 15208776, 66614955, 67177778, 122732120, 819281127, 73293337600, 107664524000, 1006774940288|
|practice_0020|multiplicative|128|2|107|109|0|107|24.93|9267, 7862, 5805, 13000, 4434, 2033, 15452, 12407, 123853, 37690, 181564, 590410, 1800256, 4686110, 734725, 15208776, 919963, 959812, 1005420, 9358689, 122732120, 66614955, 1404620, 67177778, 819281127, 73293337600, 107664524000, 1006774940288|
|practice_0021|multiplicative|0|0|136|0|0|136|2.83|44655, 47337, 55397, 165167, 166631, 231276, 254859, 396187, 406706, 495747, 1003732, 2339961, 4017150, 21394119, 27737431, 36837018, 113369281, 122372380, 132289851, 135521944, 364020552, 515127600, 517106235, 1635643341, 1715799824, 1968778075, 3316914116, 125862922168, 2334901693216|
|practice_0021|multiplicative|0|1|131|0|0|131|4.72|55397, 165167, 44655, 47337, 166631, 231276, 396187, 254859, 495747, 406706, 1003732, 4017150, 2339961, 21394119, 27737431, 36837018, 113369281, 122372380, 132289851, 135521944, 364020552, 515127600, 517106235, 1635643341, 3316914116, 1715799824, 1968778075, 125862922168, 2334901693216|
|practice_0021|multiplicative|0|2|132|0|0|132|30.17|165167, 44655, 55397, 396187, 406706, 254859, 231276, 47337, 166631, 495747, 1003732, 2339961, 132289851, 4017150, 21394119, 364020552, 515127600, 36837018, 113369281, 135521944, 517106235, 122372380, 27737431, 3316914116, 1635643341, 1715799824, 1968778075, 125862922168, 2334901693216|
|practice_0021|multiplicative|48|0|141|165|0|141|1.12|44655, 47337, 55397, 165167, 166631, 231276, 254859, 396187, 406706, 495747, 1003732, 2339961, 4017150, 21394119, 27737431, 36837018, 113369281, 122372380, 132289851, 135521944, 364020552, 515127600, 517106235, 1635643341, 1715799824, 1968778075, 3316914116, 125862922168, 2334901693216|
|practice_0021|multiplicative|48|1|140|149|0|140|2.08|55397, 165167, 44655, 47337, 166631, 231276, 396187, 254859, 495747, 1003732, 4017150, 406706, 2339961, 21394119, 27737431, 132289851, 122372380, 113369281, 364020552, 36837018, 515127600, 1635643341, 135521944, 517106235, 1715799824, 125862922168, 1968778075, 3316914116, 2334901693216|
|practice_0021|multiplicative|48|2|133|143|0|133|13.18|165167, 44655, 55397, 396187, 406706, 231276, 1003732, 495747, 47337, 166631, 254859, 364020552, 515127600, 27737431, 21394119, 4017150, 2339961, 517106235, 113369281, 1715799824, 132289851, 122372380, 135521944, 1968778075, 3316914116, 125862922168, 36837018, 1635643341, 2334901693216|
|practice_0021|multiplicative|128|0|128|133|0|128|4.12|44655, 47337, 55397, 165167, 166631, 231276, 254859, 396187, 406706, 495747, 1003732, 2339961, 4017150, 21394119, 27737431, 36837018, 113369281, 122372380, 132289851, 135521944, 364020552, 515127600, 517106235, 1635643341, 1715799824, 1968778075, 3316914116, 125862922168, 2334901693216|
|practice_0021|multiplicative|128|1|130|132|0|130|7.01|55397, 165167, 44655, 47337, 166631, 231276, 396187, 254859, 495747, 1003732, 4017150, 406706, 2339961, 21394119, 27737431, 36837018, 113369281, 122372380, 132289851, 135521944, 364020552, 515127600, 517106235, 1635643341, 1968778075, 1715799824, 3316914116, 125862922168, 2334901693216|
|practice_0021|multiplicative|128|2|129|131|0|129|34.10|165167, 44655, 55397, 396187, 406706, 231276, 254859, 1003732, 47337, 166631, 495747, 36837018, 2339961, 21394119, 27737431, 132289851, 4017150, 113369281, 122372380, 135521944, 364020552, 515127600, 517106235, 1715799824, 1968778075, 1635643341, 3316914116, 125862922168, 2334901693216|
|practice_0022|multiplicative|0|0|143|0|0|143|3.49|95745, 417136, 429832, 703416, 998291, 1030417, 1186127, 1374223, 2429664, 2922624, 3075448, 4049767, 6472315, 6850510, 7006023, 12362748, 28325842, 34019365, 60736875, 116577526, 116610144, 1595664737, 1628769904, 1766834160, 2171663218, 6458763206, 7682028408, 21723958450, 212827794816, 4043289267221|
|practice_0022|multiplicative|0|1|136|0|0|136|5.06|95745, 417136, 2922624, 429832, 703416, 1374223, 1186127, 1030417, 2429664, 3075448, 998291, 12362748, 4049767, 6472315, 6850510, 7006023, 28325842, 34019365, 60736875, 116610144, 116577526, 1628769904, 1595664737, 1766834160, 212827794816, 2171663218, 6458763206, 7682028408, 21723958450, 4043289267221|
|practice_0022|multiplicative|0|2|135|0|0|135|34.68|417136, 95745, 703416, 2922624, 429832, 998291, 1030417, 3075448, 6850510, 2429664, 60736875, 4049767, 6472315, 34019365, 1186127, 1374223, 28325842, 7006023, 116577526, 12362748, 116610144, 1628769904, 1766834160, 21723958450, 6458763206, 7682028408, 1595664737, 2171663218, 212827794816, 4043289267221|
|practice_0022|multiplicative|48|0|150|206|0|150|1.33|95745, 417136, 429832, 703416, 998291, 1030417, 1186127, 1374223, 2429664, 2922624, 3075448, 4049767, 6472315, 6850510, 7006023, 12362748, 28325842, 34019365, 60736875, 116577526, 116610144, 1595664737, 1628769904, 1766834160, 2171663218, 6458763206, 7682028408, 21723958450, 212827794816, 4043289267221|
|practice_0022|multiplicative|48|1|146|151|0|146|2.31|95745, 417136, 2922624, 429832, 703416, 1374223, 1186127, 3075448, 998291, 12362748, 1030417, 2429664, 4049767, 6472315, 6850510, 7006023, 60736875, 116610144, 116577526, 1766834160, 34019365, 28325842, 6458763206, 212827794816, 1595664737, 2171663218, 1628769904, 7682028408, 21723958450, 4043289267221|
|practice_0022|multiplicative|48|2|145|158|0|145|14.49|417136, 95745, 1030417, 703416, 1186127, 3075448, 2922624, 998291, 1374223, 4049767, 7006023, 12362748, 6850510, 28325842, 60736875, 2429664, 429832, 7682028408, 6472315, 6458763206, 1766834160, 212827794816, 1628769904, 116610144, 21723958450, 4043289267221, 116577526, 2171663218, 34019365, 1595664737|
|practice_0022|multiplicative|128|0|136|138|17|153|4.44|95745, 417136, 429832, 703416, 998291, 1030417, 1186127, 1374223, 2429664, 2922624, 3075448, 4049767, 6472315, 6850510, 7006023, 12362748, 28325842, 34019365, 60736875, 116577526, 116610144, 1595664737, 1628769904, 1766834160, 2171663218, 6458763206, 7682028408, 21723958450, 212827794816, 4043289267221|
|practice_0022|multiplicative|128|1|135|140|0|135|7.35|95745, 417136, 2922624, 429832, 703416, 1374223, 1186127, 3075448, 998291, 1030417, 2429664, 12362748, 4049767, 6472315, 6850510, 7006023, 28325842, 34019365, 60736875, 116577526, 1766834160, 116610144, 7682028408, 212827794816, 1595664737, 1628769904, 2171663218, 6458763206, 21723958450, 4043289267221|
|practice_0022|multiplicative|128|2|137|138|13|150|41.75|417136, 95745, 1030417, 703416, 1186127, 3075448, 2922624, 429832, 6472315, 1374223, 998291, 4049767, 34019365, 2429664, 6850510, 7006023, 1628769904, 116610144, 60736875, 1766834160, 212827794816, 12362748, 28325842, 116577526, 1595664737, 2171663218, 6458763206, 4043289267221, 7682028408, 21723958450|
|practice_0023|mixed|0|0|193|0|0|193|3.22|48975294, 137091034, 643689036, 925413523, 1675362342, 2330523666, 2330543861, 3442747500, 74577409351, 125059425603, 167528145804, 215754264372, 237000187102, 379334629066, 404746616597, 488619465945, 499128726953, 563169608606, 636361293400, 683227840824, 730692145637, 819285510992, 887802260749, 923514924928, 936993615654, 956546150599, 963504769468, 7992231056065, 175583433542400|
|practice_0023|mixed|0|1|183|0|0|183|9.26|48975294, 137091034, 925413523, 643689036, 1675362342, 2330523666, 2330543861, 74577409351, 3442747500, 819285510992, 125059425603, 175583433542400, 488619465945, 237000187102, 563169608606, 167528145804, 730692145637, 215754264372, 379334629066, 404746616597, 887802260749, 499128726953, 956546150599, 923514924928, 683227840824, 963504769468, 936993615654, 636361293400, 7992231056065|
|practice_0023|mixed|0|2|181|0|0|181|54.67|137091034, 3442747500, 74577409351, 643689036, 2330543861, 2330523666, 215754264372, 499128726953, 925413523, 237000187102, 1675362342, 48975294, 167528145804, 963504769468, 125059425603, 175583433542400, 379334629066, 563169608606, 488619465945, 936993615654, 683227840824, 819285510992, 730692145637, 923514924928, 956546150599, 7992231056065, 404746616597, 636361293400, 887802260749|
|practice_0023|mixed|48|0|200|243|0|200|1.50|48975294, 137091034, 643689036, 925413523, 1675362342, 2330523666, 2330543861, 3442747500, 74577409351, 125059425603, 167528145804, 215754264372, 237000187102, 379334629066, 404746616597, 488619465945, 499128726953, 563169608606, 636361293400, 683227840824, 730692145637, 819285510992, 887802260749, 923514924928, 936993615654, 956546150599, 963504769468, 7992231056065, 175583433542400|
|practice_0023|mixed|48|1|195|203|0|195|3.63|48975294, 137091034, 925413523, 643689036, 2330543861, 1675362342, 2330523666, 167528145804, 3442747500, 74577409351, 125059425603, 175583433542400, 215754264372, 379334629066, 237000187102, 488619465945, 499128726953, 683227840824, 936993615654, 730692145637, 636361293400, 404746616597, 887802260749, 563169608606, 963504769468, 819285510992, 923514924928, 7992231056065, 956546150599|
|practice_0023|mixed|48|2|193|202|0|193|24.68|137091034, 3442747500, 2330543861, 2330523666, 74577409351, 48975294, 925413523, 1675362342, 125059425603, 167528145804, 237000187102, 215754264372, 175583433542400, 488619465945, 730692145637, 563169608606, 499128726953, 683227840824, 887802260749, 404746616597, 923514924928, 636361293400, 936993615654, 7992231056065, 819285510992, 643689036, 963504769468, 379334629066, 956546150599|
|practice_0023|mixed|128|0|183|189|51|234|6.36|48975294, 137091034, 643689036, 925413523, 1675362342, 2330523666, 2330543861, 3442747500, 74577409351, 125059425603, 167528145804, 215754264372, 237000187102, 379334629066, 404746616597, 488619465945, 499128726953, 563169608606, 636361293400, 683227840824, 730692145637, 819285510992, 887802260749, 923514924928, 936993615654, 956546150599, 963504769468, 7992231056065, 175583433542400|
|practice_0023|mixed|128|1|176|179|14|190|18.07|48975294, 137091034, 925413523, 643689036, 2330543861, 2330523666, 1675362342, 74577409351, 3442747500, 125059425603, 175583433542400, 404746616597, 167528145804, 215754264372, 237000187102, 379334629066, 488619465945, 563169608606, 730692145637, 499128726953, 636361293400, 683227840824, 923514924928, 963504769468, 819285510992, 956546150599, 887802260749, 936993615654, 7992231056065|
|practice_0023|mixed|128|2|171|175|52|223|105.19|137091034, 3442747500, 2330543861, 48975294, 2330523666, 499128726953, 125059425603, 1675362342, 175583433542400, 643689036, 74577409351, 925413523, 379334629066, 167528145804, 237000187102, 488619465945, 730692145637, 923514924928, 404746616597, 936993615654, 887802260749, 215754264372, 683227840824, 819285510992, 956546150599, 563169608606, 7992231056065, 636361293400, 963504769468|
|practice_0024|shared|0|0|43|0|0|43|0.25|1286, 1303, 10368, 57780, 102393, 102434, 113928, 1024080, 132754461, 1327207680, 42467328000|
|practice_0024|shared|0|1|44|0|0|44|0.26|1303, 1286, 10368, 57780, 102393, 102434, 113928, 1024080, 1327207680, 132754461, 42467328000|
|practice_0024|shared|0|2|38|0|0|38|1.11|1303, 102393, 102434, 1286, 1024080, 1327207680, 10368, 113928, 132754461, 57780, 42467328000|
|practice_0024|shared|48|0|45|52|0|45|0.19|1286, 1303, 10368, 57780, 102393, 102434, 113928, 1024080, 132754461, 1327207680, 42467328000|
|practice_0024|shared|48|1|43|50|0|43|0.30|1303, 1286, 10368, 57780, 102393, 102434, 113928, 1024080, 1327207680, 132754461, 42467328000|
|practice_0024|shared|48|2|38|46|0|38|1.08|1303, 102393, 102434, 1286, 1024080, 1327207680, 10368, 113928, 132754461, 57780, 42467328000|
|practice_0024|shared|128|0|44|50|0|44|0.24|1286, 1303, 10368, 57780, 102393, 102434, 113928, 1024080, 132754461, 1327207680, 42467328000|
|practice_0024|shared|128|1|43|49|0|43|0.20|1303, 1286, 10368, 57780, 102393, 102434, 113928, 1024080, 1327207680, 132754461, 42467328000|
|practice_0024|shared|128|2|38|46|0|38|1.19|1303, 102393, 102434, 1286, 1024080, 1327207680, 10368, 113928, 132754461, 57780, 42467328000|
|practice_0025|mixed|0|0|182|0|0|182|2.99|1153914, 1382822, 5475655, 37564692, 109192436, 116523195, 261558225, 611693166, 2092406770, 2591049186, 2800786338, 6056011500, 22938495804, 172373182168, 202122247208, 286342827836, 294066794344, 354599730124, 415907280395, 437881506882, 447795651250, 483432372203, 719928696338, 804518873304, 809119964511, 861629092032, 896617819808, 913529868523, 4718499741321|
|practice_0025|mixed|0|1|181|0|0|181|7.82|1153914, 1382822, 5475655, 37564692, 116523195, 261558225, 109192436, 611693166, 2092406770, 2591049186, 2800786338, 6056011500, 22938495804, 437881506882, 447795651250, 172373182168, 202122247208, 286342827836, 294066794344, 354599730124, 415907280395, 809119964511, 719928696338, 483432372203, 861629092032, 896617819808, 804518873304, 913529868523, 4718499741321|
|practice_0025|mixed|0|2|175|0|0|175|47.20|37564692, 1382822, 109192436, 611693166, 261558225, 6056011500, 1153914, 2092406770, 5475655, 116523195, 22938495804, 4718499741321, 2591049186, 861629092032, 2800786338, 294066794344, 202122247208, 804518873304, 719928696338, 172373182168, 286342827836, 354599730124, 896617819808, 437881506882, 483432372203, 447795651250, 809119964511, 415907280395, 913529868523|
|practice_0025|mixed|48|0|196|250|0|196|1.36|1153914, 1382822, 5475655, 37564692, 109192436, 116523195, 261558225, 611693166, 2092406770, 2591049186, 2800786338, 6056011500, 22938495804, 172373182168, 202122247208, 286342827836, 294066794344, 354599730124, 415907280395, 437881506882, 447795651250, 483432372203, 719928696338, 804518873304, 809119964511, 861629092032, 896617819808, 913529868523, 4718499741321|
|practice_0025|mixed|48|1|188|194|0|188|3.50|1153914, 116523195, 1382822, 5475655, 37564692, 109192436, 261558225, 2092406770, 2800786338, 6056011500, 611693166, 22938495804, 2591049186, 202122247208, 172373182168, 804518873304, 809119964511, 437881506882, 447795651250, 286342827836, 483432372203, 354599730124, 294066794344, 415907280395, 719928696338, 861629092032, 896617819808, 913529868523, 4718499741321|
|practice_0025|mixed|48|2|191|196|0|191|22.90|1153914, 116523195, 37564692, 5475655, 1382822, 2591049186, 2092406770, 109192436, 172373182168, 2800786338, 286342827836, 611693166, 294066794344, 804518873304, 6056011500, 261558225, 437881506882, 447795651250, 354599730124, 861629092032, 4718499741321, 483432372203, 913529868523, 719928696338, 22938495804, 896617819808, 415907280395, 809119964511, 202122247208|
|practice_0025|mixed|128|0|177|180|46|223|6.04|1153914, 1382822, 5475655, 37564692, 109192436, 116523195, 261558225, 611693166, 2092406770, 2591049186, 2800786338, 6056011500, 22938495804, 172373182168, 202122247208, 286342827836, 294066794344, 354599730124, 415907280395, 437881506882, 447795651250, 483432372203, 719928696338, 804518873304, 809119964511, 861629092032, 896617819808, 913529868523, 4718499741321|
|practice_0025|mixed|128|1|173|178|28|201|15.47|1153914, 116523195, 1382822, 5475655, 37564692, 109192436, 261558225, 611693166, 2800786338, 2092406770, 2591049186, 6056011500, 22938495804, 172373182168, 286342827836, 354599730124, 202122247208, 294066794344, 415907280395, 437881506882, 447795651250, 483432372203, 719928696338, 861629092032, 804518873304, 809119964511, 896617819808, 913529868523, 4718499741321|
|practice_0025|mixed|128|2|173|176|12|185|107.63|1153914, 116523195, 37564692, 109192436, 611693166, 1382822, 5475655, 261558225, 2092406770, 2800786338, 2591049186, 6056011500, 22938495804, 294066794344, 437881506882, 913529868523, 172373182168, 354599730124, 202122247208, 447795651250, 415907280395, 804518873304, 809119964511, 861629092032, 719928696338, 896617819808, 483432372203, 286342827836, 4718499741321|
|practice_0026|shared|0|0|153|0|0|153|2.66|1689974, 3825051, 4501378, 6194356, 9003798, 10188232, 22537089, 22705222, 22705402, 31010576, 31010729, 37167005, 40517556, 42354253, 49503570, 50762390, 193574787, 204412054, 574396026, 648273966, 726052627, 2118518820, 11349886647, 12837063950, 15579656830, 139342788000, 264107470572, 340258854481, 752675001935, 2169502777260|
|practice_0026|shared|0|1|154|0|0|154|5.12|1689974, 3825051, 4501378, 9003798, 6194356, 10188232, 22537089, 22705222, 22705402, 49503570, 31010576, 31010729, 37167005, 40517556, 726052627, 42354253, 50762390, 193574787, 204412054, 648273966, 2118518820, 574396026, 11349886647, 12837063950, 15579656830, 139342788000, 264107470572, 2169502777260, 340258854481, 752675001935|
|practice_0026|shared|0|2|147|0|0|147|33.76|3825051, 10188232, 22705222, 31010729, 1689974, 6194356, 9003798, 31010576, 37167005, 40517556, 4501378, 22705402, 49503570, 50762390, 22537089, 648273966, 204412054, 726052627, 193574787, 2118518820, 42354253, 574396026, 12837063950, 15579656830, 11349886647, 139342788000, 264107470572, 752675001935, 2169502777260, 340258854481|
|practice_0026|shared|48|0|161|184|0|161|1.32|1689974, 3825051, 4501378, 6194356, 9003798, 10188232, 22537089, 22705222, 22705402, 31010576, 31010729, 37167005, 40517556, 42354253, 49503570, 50762390, 193574787, 204412054, 574396026, 648273966, 726052627, 2118518820, 11349886647, 12837063950, 15579656830, 139342788000, 264107470572, 340258854481, 752675001935, 2169502777260|
|practice_0026|shared|48|1|156|172|0|156|2.55|1689974, 4501378, 9003798, 40517556, 3825051, 6194356, 10188232, 22705402, 22705222, 22537089, 31010576, 31010729, 37167005, 49503570, 50762390, 42354253, 204412054, 574396026, 193574787, 139342788000, 648273966, 2118518820, 11349886647, 12837063950, 15579656830, 726052627, 340258854481, 264107470572, 752675001935, 2169502777260|
|practice_0026|shared|48|2|157|170|0|157|17.30|3825051, 9003798, 40517556, 4501378, 10188232, 6194356, 22705402, 648273966, 31010576, 22705222, 22537089, 42354253, 49503570, 1689974, 31010729, 726052627, 574396026, 50762390, 204412054, 193574787, 37167005, 11349886647, 15579656830, 12837063950, 264107470572, 340258854481, 752675001935, 139342788000, 2169502777260, 2118518820|
|practice_0026|shared|128|0|148|152|27|175|5.11|1689974, 3825051, 4501378, 6194356, 9003798, 10188232, 22537089, 22705222, 22705402, 31010576, 31010729, 37167005, 40517556, 42354253, 49503570, 50762390, 193574787, 204412054, 574396026, 648273966, 726052627, 2118518820, 11349886647, 12837063950, 15579656830, 139342788000, 264107470572, 340258854481, 752675001935, 2169502777260|
|practice_0026|shared|128|1|146|150|18|164|9.10|1689974, 4501378, 9003798, 40517556, 3825051, 6194356, 10188232, 22537089, 22705222, 22705402, 31010576, 31010729, 37167005, 42354253, 204412054, 49503570, 50762390, 193574787, 574396026, 648273966, 726052627, 2118518820, 11349886647, 12837063950, 15579656830, 139342788000, 264107470572, 340258854481, 2169502777260, 752675001935|
|practice_0026|shared|128|2|145|149|20|165|56.52|3825051, 9003798, 40517556, 6194356, 31010576, 31010729, 1689974, 22537089, 37167005, 4501378, 22705402, 22705222, 2118518820, 648273966, 10188232, 49503570, 726052627, 42354253, 50762390, 193574787, 204412054, 574396026, 139342788000, 11349886647, 15579656830, 264107470572, 2169502777260, 12837063950, 340258854481, 752675001935|
|practice_0027|shared|0|0|205|0|0|205|3.70|177316895, 268697673, 748269834, 748270207, 789774774, 1654774423, 1654774629, 2084769143, 4078107092, 8335145061, 8797470181, 32923889456, 63126087677, 113273845960, 153624729942, 202341799328, 621484153602, 906233950864, 2134814912001, 2231690151272, 2967310602908, 4047162179244, 5255000180629, 14486511304320, 21019986505296, 79689337995848, 200793423645936, 803173841752896, 24563486972638427, 46182495911290542, 352960083410318902|
|practice_0027|shared|0|1|197|0|0|197|12.96|268697673, 177316895, 789774774, 4078107092, 748269834, 748270207, 32923889456, 14486511304320, 1654774423, 1654774629, 8335145061, 2084769143, 8797470181, 202341799328, 63126087677, 113273845960, 153624729942, 2967310602908, 2134814912001, 906233950864, 4047162179244, 803173841752896, 200793423645936, 46182495911290542, 621484153602, 2231690151272, 5255000180629, 21019986505296, 79689337995848, 24563486972638427, 352960083410318902|
|practice_0027|shared|0|2|190|0|0|190|67.97|748270207, 177316895, 748269834, 268697673, 789774774, 4078107092, 1654774629, 1654774423, 32923889456, 14486511304320, 8797470181, 2134814912001, 803173841752896, 200793423645936, 8335145061, 202341799328, 906233950864, 2084769143, 5255000180629, 46182495911290542, 153624729942, 63126087677, 621484153602, 2967310602908, 113273845960, 21019986505296, 24563486972638427, 4047162179244, 79689337995848, 352960083410318902, 2231690151272|
|practice_0027|shared|48|0|220|331|0|220|1.69|177316895, 268697673, 748269834, 748270207, 789774774, 1654774423, 1654774629, 2084769143, 4078107092, 8335145061, 8797470181, 32923889456, 63126087677, 113273845960, 153624729942, 202341799328, 621484153602, 906233950864, 2134814912001, 2231690151272, 2967310602908, 4047162179244, 5255000180629, 14486511304320, 21019986505296, 79689337995848, 200793423645936, 803173841752896, 24563486972638427, 46182495911290542, 352960083410318902|
|practice_0027|shared|48|1|216|270|0|216|4.64|268697673, 177316895, 789774774, 748270207, 748269834, 32923889456, 14486511304320, 4078107092, 1654774423, 1654774629, 2084769143, 2134814912001, 2231690151272, 8335145061, 8797470181, 202341799328, 621484153602, 906233950864, 21019986505296, 113273845960, 5255000180629, 2967310602908, 63126087677, 153624729942, 200793423645936, 4047162179244, 79689337995848, 803173841752896, 24563486972638427, 46182495911290542, 352960083410318902|
|practice_0027|shared|48|2|204|213|0|204|29.08|748270207, 177316895, 268697673, 748269834, 32923889456, 14486511304320, 1654774423, 4078107092, 63126087677, 1654774629, 8335145061, 8797470181, 803173841752896, 2084769143, 202341799328, 200793423645936, 153624729942, 2967310602908, 906233950864, 2134814912001, 621484153602, 24563486972638427, 4047162179244, 2231690151272, 113273845960, 79689337995848, 5255000180629, 789774774, 46182495911290542, 21019986505296, 352960083410318902|
|practice_0027|shared|128|0|184|188|3|187|6.78|177316895, 268697673, 748269834, 748270207, 789774774, 1654774423, 1654774629, 2084769143, 4078107092, 8335145061, 8797470181, 32923889456, 63126087677, 113273845960, 153624729942, 202341799328, 621484153602, 906233950864, 2134814912001, 2231690151272, 2967310602908, 4047162179244, 5255000180629, 14486511304320, 21019986505296, 79689337995848, 200793423645936, 803173841752896, 24563486972638427, 46182495911290542, 352960083410318902|
|practice_0027|shared|128|1|186|190|13|199|20.21|268697673, 177316895, 789774774, 748270207, 748269834, 32923889456, 14486511304320, 1654774423, 1654774629, 4078107092, 8335145061, 2084769143, 8797470181, 202341799328, 113273845960, 153624729942, 63126087677, 621484153602, 2967310602908, 2134814912001, 803173841752896, 200793423645936, 906233950864, 2231690151272, 4047162179244, 5255000180629, 21019986505296, 46182495911290542, 79689337995848, 24563486972638427, 352960083410318902|
|practice_0027|shared|128|2|181|184|9|190|125.57|748270207, 177316895, 748269834, 268697673, 1654774423, 789774774, 1654774629, 14486511304320, 8797470181, 2134814912001, 202341799328, 803173841752896, 8335145061, 32923889456, 200793423645936, 113273845960, 2084769143, 4078107092, 5255000180629, 63126087677, 906233950864, 21019986505296, 4047162179244, 2231690151272, 153624729942, 2967310602908, 46182495911290542, 621484153602, 79689337995848, 24563486972638427, 352960083410318902|
|practice_0028|mixed|0|0|240|0|0|240|4.71|344460159, 605001055, 1378002138, 1540609925, 1540623335, 2231487262, 12604470576, 12972646831, 15974661528, 22057817480, 31240671296, 3748881664500, 4400725558320, 115960669644161, 119982520640091, 200128614241405, 201400042072413, 307116080273461, 312270541074717, 411033519483338, 446735836347702, 485022838156565, 501848788125571, 587042223335438, 648012775479485, 702681065139436, 795898582244875, 892574913484800, 926948031682017, 934337293572598, 824539121807036166|
|practice_0028|mixed|0|1|233|0|0|233|14.53|344460159, 2231487262, 605001055, 1378002138, 1540609925, 1540623335, 12604470576, 22057817480, 15974661528, 31240671296, 3748881664500, 12972646831, 892574913484800, 4400725558320, 119982520640091, 312270541074717, 934337293572598, 115960669644161, 201400042072413, 702681065139436, 200128614241405, 307116080273461, 587042223335438, 411033519483338, 446735836347702, 485022838156565, 501848788125571, 926948031682017, 648012775479485, 795898582244875, 824539121807036166|
|practice_0028|mixed|0|2|228|0|0|228|103.24|1378002138, 605001055, 1540623335, 344460159, 12972646831, 1540609925, 15974661528, 22057817480, 31240671296, 3748881664500, 892574913484800, 4400725558320, 2231487262, 12604470576, 201400042072413, 119982520640091, 795898582244875, 200128614241405, 648012775479485, 307116080273461, 446735836347702, 312270541074717, 587042223335438, 501848788125571, 115960669644161, 485022838156565, 702681065139436, 411033519483338, 926948031682017, 934337293572598, 824539121807036166|
|practice_0028|mixed|48|0|245|317|0|245|1.61|344460159, 605001055, 1378002138, 1540609925, 1540623335, 2231487262, 12604470576, 12972646831, 15974661528, 22057817480, 31240671296, 3748881664500, 4400725558320, 115960669644161, 119982520640091, 200128614241405, 201400042072413, 307116080273461, 312270541074717, 411033519483338, 446735836347702, 485022838156565, 501848788125571, 587042223335438, 648012775479485, 702681065139436, 795898582244875, 892574913484800, 926948031682017, 934337293572598, 824539121807036166|
|practice_0028|mixed|48|1|234|254|0|234|4.96|344460159, 2231487262, 605001055, 1378002138, 1540609925, 1540623335, 22057817480, 12604470576, 12972646831, 15974661528, 31240671296, 892574913484800, 3748881664500, 4400725558320, 312270541074717, 411033519483338, 587042223335438, 307116080273461, 119982520640091, 201400042072413, 200128614241405, 446735836347702, 115960669644161, 648012775479485, 501848788125571, 702681065139436, 926948031682017, 485022838156565, 795898582244875, 824539121807036166, 934337293572598|
|practice_0028|mixed|48|2|244|253|0|244|33.89|1378002138, 605001055, 1540623335, 344460159, 1540609925, 12604470576, 892574913484800, 2231487262, 115960669644161, 200128614241405, 31240671296, 15974661528, 4400725558320, 702681065139436, 307116080273461, 201400042072413, 22057817480, 312270541074717, 501848788125571, 12972646831, 934337293572598, 119982520640091, 485022838156565, 795898582244875, 411033519483338, 648012775479485, 446735836347702, 3748881664500, 824539121807036166, 926948031682017, 587042223335438|
|practice_0028|mixed|128|0|218|224|44|262|7.07|344460159, 605001055, 1378002138, 1540609925, 1540623335, 2231487262, 12604470576, 12972646831, 15974661528, 22057817480, 31240671296, 3748881664500, 4400725558320, 115960669644161, 119982520640091, 200128614241405, 201400042072413, 307116080273461, 312270541074717, 411033519483338, 446735836347702, 485022838156565, 501848788125571, 587042223335438, 648012775479485, 702681065139436, 795898582244875, 892574913484800, 926948031682017, 934337293572598, 824539121807036166|
|practice_0028|mixed|128|1|214|221|18|232|23.70|344460159, 2231487262, 605001055, 1378002138, 1540609925, 1540623335, 12604470576, 22057817480, 15974661528, 31240671296, 12972646831, 3748881664500, 4400725558320, 892574913484800, 115960669644161, 119982520640091, 648012775479485, 501848788125571, 200128614241405, 201400042072413, 312270541074717, 307116080273461, 411033519483338, 702681065139436, 446735836347702, 485022838156565, 587042223335438, 795898582244875, 926948031682017, 934337293572598, 824539121807036166|
|practice_0028|mixed|128|2|212|216|26|238|177.48|1378002138, 605001055, 1540623335, 344460159, 1540609925, 22057817480, 31240671296, 3748881664500, 12972646831, 2231487262, 892574913484800, 15974661528, 12604470576, 926948031682017, 115960669644161, 307116080273461, 201400042072413, 200128614241405, 312270541074717, 4400725558320, 587042223335438, 485022838156565, 501848788125571, 648012775479485, 702681065139436, 934337293572598, 446735836347702, 411033519483338, 824539121807036166, 119982520640091, 795898582244875|
|practice_0029|mixed|0|0|214|0|0|214|3.46|162345321, 391907380, 638072594, 1098628151, 1098628189, 1098630180, 2552290052, 3684291048, 3896259303, 15380794353, 22202533080, 25082099072, 58606230930, 185021109000, 5286317400023, 102275916566139, 142350480993139, 260628702814173, 266912131052899, 480036309474984, 511107136392299, 536722234517061, 616284113342134, 655509297791521, 760511661896217, 789255655895644, 800328538217706, 865211498449188, 946830694766067, 979443322563045|
|practice_0029|mixed|0|1|205|0|0|205|10.41|162345321, 3896259303, 58606230930, 391907380, 25082099072, 638072594, 2552290052, 1098628151, 1098628189, 1098630180, 15380794353, 3684291048, 22202533080, 185021109000, 5286317400023, 102275916566139, 536722234517061, 142350480993139, 260628702814173, 266912131052899, 511107136392299, 480036309474984, 760511661896217, 616284113342134, 800328538217706, 789255655895644, 946830694766067, 655509297791521, 865211498449188, 979443322563045|
|practice_0029|mixed|0|2|199|0|11|210|61.71|1098628151, 162345321, 1098630180, 22202533080, 391907380, 1098628189, 638072594, 2552290052, 25082099072, 3684291048, 58606230930, 3896259303, 185021109000, 15380794353, 102275916566139, 5286317400023, 266912131052899, 800328538217706, 142350480993139, 616284113342134, 789255655895644, 511107136392299, 865211498449188, 946830694766067, 480036309474984, 760511661896217, 979443322563045, 536722234517061, 260628702814173, 655509297791521|
|practice_0029|mixed|48|0|217|341|0|217|1.39|162345321, 391907380, 638072594, 1098628151, 1098628189, 1098630180, 2552290052, 3684291048, 3896259303, 15380794353, 22202533080, 25082099072, 58606230930, 185021109000, 5286317400023, 102275916566139, 142350480993139, 260628702814173, 266912131052899, 480036309474984, 511107136392299, 536722234517061, 616284113342134, 655509297791521, 760511661896217, 789255655895644, 800328538217706, 865211498449188, 946830694766067, 979443322563045|
|practice_0029|mixed|48|1|210|220|0|210|3.82|162345321, 3896259303, 58606230930, 391907380, 25082099072, 638072594, 2552290052, 1098628189, 1098628151, 1098630180, 3684291048, 15380794353, 22202533080, 185021109000, 5286317400023, 102275916566139, 260628702814173, 480036309474984, 266912131052899, 800328538217706, 946830694766067, 511107136392299, 979443322563045, 142350480993139, 536722234517061, 616284113342134, 760511661896217, 655509297791521, 789255655895644, 865211498449188|
|practice_0029|mixed|48|2|210|226|0|210|27.95|1098628151, 162345321, 1098630180, 22202533080, 1098628189, 391907380, 638072594, 15380794353, 185021109000, 2552290052, 25082099072, 480036309474984, 3684291048, 5286317400023, 58606230930, 616284113342134, 3896259303, 655509297791521, 142350480993139, 536722234517061, 789255655895644, 266912131052899, 760511661896217, 865211498449188, 260628702814173, 979443322563045, 511107136392299, 800328538217706, 946830694766067, 102275916566139|
|practice_0029|mixed|128|0|193|204|11|204|6.14|162345321, 391907380, 638072594, 1098628151, 1098628189, 1098630180, 2552290052, 3684291048, 3896259303, 15380794353, 22202533080, 25082099072, 58606230930, 185021109000, 5286317400023, 102275916566139, 142350480993139, 260628702814173, 266912131052899, 480036309474984, 511107136392299, 536722234517061, 616284113342134, 655509297791521, 760511661896217, 789255655895644, 800328538217706, 865211498449188, 946830694766067, 979443322563045|
|practice_0029|mixed|128|1|192|196|1|193|18.66|162345321, 3896259303, 58606230930, 391907380, 25082099072, 638072594, 2552290052, 1098628151, 1098628189, 1098630180, 15380794353, 3684291048, 22202533080, 185021109000, 5286317400023, 102275916566139, 142350480993139, 480036309474984, 789255655895644, 536722234517061, 655509297791521, 800328538217706, 260628702814173, 760511661896217, 616284113342134, 511107136392299, 266912131052899, 865211498449188, 946830694766067, 979443322563045|
|practice_0029|mixed|128|2|187|192|8|195|118.33|1098628151, 638072594, 22202533080, 1098630180, 162345321, 391907380, 1098628189, 25082099072, 185021109000, 3684291048, 3896259303, 58606230930, 15380794353, 2552290052, 5286317400023, 102275916566139, 480036309474984, 142350480993139, 260628702814173, 511107136392299, 616284113342134, 789255655895644, 979443322563045, 760511661896217, 536722234517061, 800328538217706, 946830694766067, 266912131052899, 655509297791521, 865211498449188|
|practice_0030|multiplicative|0|0|142|0|0|142|3.52|13560, 19388, 25894, 40069, 48651, 90490, 163063, 166948, 249770, 263351, 474810, 582038, 601206, 826328, 971096, 1009675, 3401800, 4560822, 9315451, 10601096, 17811390, 37404342, 142762565, 7664824885, 41699558402, 57177776064, 111538104228, 187589986574, 360491363808, 20478626876792, 34256401454979|
|practice_0030|multiplicative|0|1|142|0|0|142|6.18|13560, 19388, 25894, 40069, 48651, 163063, 90490, 166948, 249770, 263351, 474810, 582038, 601206, 826328, 971096, 1009675, 3401800, 4560822, 9315451, 10601096, 17811390, 37404342, 142762565, 7664824885, 41699558402, 57177776064, 111538104228, 187589986574, 360491363808, 20478626876792, 34256401454979|
|practice_0030|multiplicative|0|2|134|0|0|134|45.20|13560, 40069, 48651, 163063, 25894, 474810, 263351, 19388, 166948, 971096, 90490, 249770, 826328, 582038, 1009675, 601206, 9315451, 3401800, 37404342, 4560822, 142762565, 17811390, 57177776064, 10601096, 360491363808, 7664824885, 111538104228, 41699558402, 187589986574, 20478626876792, 34256401454979|
|practice_0030|multiplicative|48|0|147|193|0|147|1.41|13560, 19388, 25894, 40069, 48651, 90490, 163063, 166948, 249770, 263351, 474810, 582038, 601206, 826328, 971096, 1009675, 3401800, 4560822, 9315451, 10601096, 17811390, 37404342, 142762565, 7664824885, 41699558402, 57177776064, 111538104228, 187589986574, 360491363808, 20478626876792, 34256401454979|
|practice_0030|multiplicative|48|1|145|154|0|145|2.15|13560, 19388, 25894, 40069, 48651, 90490, 163063, 166948, 249770, 263351, 474810, 582038, 601206, 826328, 971096, 3401800, 4560822, 1009675, 9315451, 10601096, 17811390, 142762565, 37404342, 7664824885, 57177776064, 20478626876792, 41699558402, 187589986574, 111538104228, 360491363808, 34256401454979|
|practice_0030|multiplicative|48|2|146|154|0|146|16.13|13560, 40069, 48651, 19388, 474810, 163063, 166948, 601206, 249770, 3401800, 263351, 582038, 826328, 9315451, 90490, 17811390, 4560822, 1009675, 971096, 142762565, 37404342, 25894, 10601096, 7664824885, 360491363808, 111538104228, 187589986574, 34256401454979, 20478626876792, 41699558402, 57177776064|
|practice_0030|multiplicative|128|0|138|142|4|142|4.66|13560, 19388, 25894, 40069, 48651, 90490, 163063, 166948, 249770, 263351, 474810, 582038, 601206, 826328, 971096, 1009675, 3401800, 4560822, 9315451, 10601096, 17811390, 37404342, 142762565, 7664824885, 41699558402, 57177776064, 111538104228, 187589986574, 360491363808, 20478626876792, 34256401454979|
|practice_0030|multiplicative|128|1|138|142|4|142|9.00|13560, 19388, 25894, 40069, 48651, 90490, 163063, 166948, 249770, 263351, 474810, 582038, 601206, 826328, 971096, 1009675, 3401800, 4560822, 9315451, 10601096, 17811390, 37404342, 142762565, 7664824885, 41699558402, 57177776064, 111538104228, 187589986574, 360491363808, 20478626876792, 34256401454979|
|practice_0030|multiplicative|128|2|132|136|11|143|52.54|13560, 40069, 48651, 19388, 474810, 163063, 166948, 249770, 25894, 90490, 826328, 263351, 582038, 4560822, 9315451, 601206, 971096, 1009675, 17811390, 3401800, 10601096, 57177776064, 37404342, 7664824885, 142762565, 41699558402, 111538104228, 187589986574, 360491363808, 20478626876792, 34256401454979|
|practice_0031|mixed|0|0|226|0|0|226|3.64|1415060970, 1415061568, 2578879952, 4644394666, 4644394866, 8464186884, 15290577110, 31131372338, 2258209057680, 3703271208992, 4561693850618, 7324194498822, 8165177175708, 115370898573966, 131993024409184, 344931536189072, 362515945097702, 380331522184229, 403314585344618, 461252243011010, 615568438857800, 650594097273567, 677361519685729, 683540951902888, 780349422998389, 872599576660036, 874957229727461, 904420123917944, 2660800769141918|
|practice_0031|mixed|0|1|213|0|0|213|11.54|1415060970, 1415061568, 31131372338, 2578879952, 4644394666, 4644394866, 8464186884, 15290577110, 8165177175708, 3703271208992, 7324194498822, 2258209057680, 4561693850618, 344931536189072, 115370898573966, 615568438857800, 362515945097702, 131993024409184, 380331522184229, 461252243011010, 403314585344618, 650594097273567, 2660800769141918, 904420123917944, 677361519685729, 683540951902888, 780349422998389, 872599576660036, 874957229727461|
|practice_0031|mixed|0|2|214|0|33|247|71.87|4644394666, 1415061568, 2578879952, 4644394866, 1415060970, 8165177175708, 15290577110, 2258209057680, 31131372338, 8464186884, 3703271208992, 7324194498822, 4561693850618, 461252243011010, 344931536189072, 650594097273567, 403314585344618, 615568438857800, 362515945097702, 872599576660036, 131993024409184, 380331522184229, 115370898573966, 677361519685729, 904420123917944, 2660800769141918, 874957229727461, 780349422998389, 683540951902888|
|practice_0031|mixed|48|0|230|317|0|230|1.22|1415060970, 1415061568, 2578879952, 4644394666, 4644394866, 8464186884, 15290577110, 31131372338, 2258209057680, 3703271208992, 4561693850618, 7324194498822, 8165177175708, 115370898573966, 131993024409184, 344931536189072, 362515945097702, 380331522184229, 403314585344618, 461252243011010, 615568438857800, 650594097273567, 677361519685729, 683540951902888, 780349422998389, 872599576660036, 874957229727461, 904420123917944, 2660800769141918|
|practice_0031|mixed|48|1|228|240|0|228|4.36|1415060970, 1415061568, 31131372338, 2578879952, 4644394666, 4644394866, 8464186884, 15290577110, 2258209057680, 3703271208992, 7324194498822, 4561693850618, 115370898573966, 8165177175708, 904420123917944, 403314585344618, 131993024409184, 461252243011010, 344931536189072, 362515945097702, 380331522184229, 615568438857800, 677361519685729, 650594097273567, 874957229727461, 683540951902888, 872599576660036, 2660800769141918, 780349422998389|
|practice_0031|mixed|48|2|222|234|0|222|29.19|1415061568, 4644394866, 2578879952, 1415060970, 3703271208992, 15290577110, 31131372338, 2660800769141918, 8464186884, 131993024409184, 2258209057680, 8165177175708, 7324194498822, 904420123917944, 362515945097702, 4561693850618, 115370898573966, 780349422998389, 461252243011010, 4644394666, 872599576660036, 683540951902888, 615568438857800, 874957229727461, 380331522184229, 677361519685729, 650594097273567, 344931536189072, 403314585344618|
|practice_0031|mixed|128|0|208|221|22|230|6.73|1415060970, 1415061568, 2578879952, 4644394666, 4644394866, 8464186884, 15290577110, 31131372338, 2258209057680, 3703271208992, 4561693850618, 7324194498822, 8165177175708, 115370898573966, 131993024409184, 344931536189072, 362515945097702, 380331522184229, 403314585344618, 461252243011010, 615568438857800, 650594097273567, 677361519685729, 683540951902888, 780349422998389, 872599576660036, 874957229727461, 904420123917944, 2660800769141918|
|practice_0031|mixed|128|1|199|203|20|219|19.89|1415060970, 1415061568, 31131372338, 2578879952, 4644394666, 4644394866, 8464186884, 15290577110, 8165177175708, 2258209057680, 7324194498822, 4561693850618, 3703271208992, 115370898573966, 131993024409184, 344931536189072, 904420123917944, 362515945097702, 461252243011010, 615568438857800, 677361519685729, 874957229727461, 683540951902888, 872599576660036, 403314585344618, 380331522184229, 650594097273567, 2660800769141918, 780349422998389|
|practice_0031|mixed|128|2|194|199|4|198|133.31|1415061568, 4644394666, 1415060970, 4644394866, 2578879952, 15290577110, 31131372338, 8464186884, 8165177175708, 344931536189072, 2258209057680, 4561693850618, 362515945097702, 615568438857800, 7324194498822, 3703271208992, 403314585344618, 650594097273567, 380331522184229, 872599576660036, 683540951902888, 780349422998389, 115370898573966, 131993024409184, 677361519685729, 874957229727461, 461252243011010, 904420123917944, 2660800769141918|
|practice_0032|multiplicative|0|0|140|0|0|140|3.62|64001, 77009, 78989, 95326, 131579, 158044, 185398, 191169, 320273, 648552, 716412, 1144692, 1560876, 1844266, 1921282, 2372262, 4892059, 11883790, 32017750, 61871656, 158083000, 316572207, 1009532605, 2290984981, 3163238787, 4899035400, 9494555580, 9805938519, 35787680898, 119134859979, 78467480489488|
|practice_0032|multiplicative|0|1|140|0|0|140|5.20|64001, 77009, 78989, 95326, 131579, 158044, 185398, 191169, 320273, 648552, 716412, 1144692, 1560876, 1844266, 1921282, 2372262, 4892059, 11883790, 32017750, 61871656, 158083000, 316572207, 1009532605, 3163238787, 4899035400, 2290984981, 9805938519, 9494555580, 35787680898, 119134859979, 78467480489488|
|practice_0032|multiplicative|0|2|138|0|0|138|39.75|158044, 64001, 77009, 185398, 320273, 716412, 648552, 131579, 1921282, 1844266, 95326, 78989, 191169, 11883790, 1144692, 2372262, 1009532605, 1560876, 4899035400, 9805938519, 4892059, 158083000, 32017750, 61871656, 316572207, 2290984981, 3163238787, 119134859979, 9494555580, 35787680898, 78467480489488|
|practice_0032|multiplicative|48|0|148|214|0|148|1.44|64001, 77009, 78989, 95326, 131579, 158044, 185398, 191169, 320273, 648552, 716412, 1144692, 1560876, 1844266, 1921282, 2372262, 4892059, 11883790, 32017750, 61871656, 158083000, 316572207, 1009532605, 2290984981, 3163238787, 4899035400, 9494555580, 9805938519, 35787680898, 119134859979, 78467480489488|
|practice_0032|multiplicative|48|1|150|156|0|150|2.61|64001, 77009, 78989, 95326, 131579, 158044, 185398, 191169, 648552, 320273, 1144692, 716412, 1560876, 1844266, 2372262, 11883790, 1921282, 4892059, 32017750, 158083000, 61871656, 316572207, 1009532605, 4899035400, 9494555580, 9805938519, 2290984981, 35787680898, 3163238787, 119134859979, 78467480489488|
|practice_0032|multiplicative|48|2|148|156|0|148|16.27|158044, 64001, 77009, 185398, 320273, 95326, 716412, 78989, 648552, 191169, 131579, 1921282, 11883790, 1844266, 1560876, 316572207, 2372262, 61871656, 158083000, 1144692, 9494555580, 1009532605, 9805938519, 2290984981, 3163238787, 4899035400, 4892059, 32017750, 119134859979, 35787680898, 78467480489488|
|practice_0032|multiplicative|128|0|135|140|15|150|4.27|64001, 77009, 78989, 95326, 131579, 158044, 185398, 191169, 320273, 648552, 716412, 1144692, 1560876, 1844266, 1921282, 2372262, 4892059, 11883790, 32017750, 61871656, 158083000, 316572207, 1009532605, 2290984981, 3163238787, 4899035400, 9494555580, 9805938519, 35787680898, 119134859979, 78467480489488|
|practice_0032|multiplicative|128|1|136|139|11|147|6.33|64001, 77009, 78989, 95326, 131579, 158044, 185398, 191169, 320273, 648552, 716412, 1144692, 1560876, 1844266, 1921282, 2372262, 4892059, 11883790, 32017750, 61871656, 158083000, 316572207, 1009532605, 2290984981, 4899035400, 9494555580, 3163238787, 9805938519, 35787680898, 78467480489488, 119134859979|
|practice_0032|multiplicative|128|2|132|135|4|136|45.90|158044, 64001, 77009, 185398, 320273, 716412, 648552, 131579, 1921282, 1844266, 78989, 95326, 11883790, 191169, 1144692, 1560876, 2372262, 316572207, 4892059, 158083000, 32017750, 2290984981, 3163238787, 61871656, 1009532605, 9494555580, 4899035400, 9805938519, 35787680898, 119134859979, 78467480489488|
|practice_0033|shared|0|0|42|0|0|42|0.17|3538, 3566, 3577, 5846, 6481, 21780, 46326, 50703, 69664, 114048, 213840, 648648|
|practice_0033|shared|0|1|41|0|0|41|0.14|3577, 3566, 3538, 46326, 69664, 114048, 213840, 648648, 5846, 6481, 21780, 50703|
|practice_0033|shared|0|2|39|0|0|39|0.67|3577, 3566, 114048, 3538, 46326, 69664, 6481, 5846, 21780, 213840, 648648, 50703|
|practice_0033|shared|48|0|40|48|0|40|0.17|3538, 3566, 3577, 5846, 6481, 21780, 46326, 50703, 69664, 114048, 213840, 648648|
|practice_0033|shared|48|1|41|46|0|41|0.14|3577, 3566, 3538, 46326, 69664, 114048, 213840, 648648, 5846, 6481, 21780, 50703|
|practice_0033|shared|48|2|42|46|0|42|0.69|3577, 3566, 114048, 3538, 46326, 69664, 213840, 648648, 5846, 21780, 6481, 50703|
|practice_0033|shared|128|0|40|48|0|40|0.24|3538, 3566, 3577, 5846, 6481, 21780, 46326, 50703, 69664, 114048, 213840, 648648|
|practice_0033|shared|128|1|41|46|0|41|0.13|3577, 3566, 3538, 46326, 69664, 114048, 213840, 648648, 5846, 6481, 21780, 50703|
|practice_0033|shared|128|2|42|46|0|42|0.74|3577, 3566, 114048, 3538, 46326, 69664, 213840, 648648, 5846, 21780, 6481, 50703|
|practice_0034|independent|0|0|319|0|0|319|3.63|101181788029710766, 109045945666095854, 210161988755598540, 223263612503529506, 239041802034133299, 271733010962642628, 346088341674474587, 377935901965139177, 380725222159077806, 395633917438565177, 400440798686427318, 404909130390329023, 515127540626807406, 515620110477775414, 550358109712356093, 616494342142391550, 644965308286333487, 649158742032097674, 742989197954160843, 752105172529229462, 780544829745605743, 801343371679528758, 816120026090166881, 818497040519396521, 821240920750780361, 832212558718489617, 859011983665403215, 917143299895366489, 920838418575812689|
|practice_0034|independent|0|1|303|0|71|374|13.52|550358109712356093, 920838418575812689, 239041802034133299, 346088341674474587, 210161988755598540, 380725222159077806, 644965308286333487, 109045945666095854, 752105172529229462, 101181788029710766, 400440798686427318, 377935901965139177, 223263612503529506, 395633917438565177, 649158742032097674, 515127540626807406, 515620110477775414, 271733010962642628, 616494342142391550, 404909130390329023, 742989197954160843, 816120026090166881, 917143299895366489, 780544829745605743, 801343371679528758, 832212558718489617, 818497040519396521, 859011983665403215, 821240920750780361|
|practice_0034|independent|0|2|299|0|165|464|88.67|239041802034133299, 109045945666095854, 377935901965139177, 210161988755598540, 271733010962642628, 550358109712356093, 400440798686427318, 380725222159077806, 515127540626807406, 515620110477775414, 395633917438565177, 832212558718489617, 644965308286333487, 752105172529229462, 920838418575812689, 780544829745605743, 101181788029710766, 649158742032097674, 404909130390329023, 859011983665403215, 616494342142391550, 818497040519396521, 346088341674474587, 816120026090166881, 821240920750780361, 917143299895366489, 742989197954160843, 801343371679528758, 223263612503529506|
|practice_0034|independent|48|0|318|489|0|318|1.55|101181788029710766, 109045945666095854, 210161988755598540, 223263612503529506, 239041802034133299, 271733010962642628, 346088341674474587, 377935901965139177, 380725222159077806, 395633917438565177, 400440798686427318, 404909130390329023, 515127540626807406, 515620110477775414, 550358109712356093, 616494342142391550, 644965308286333487, 649158742032097674, 742989197954160843, 752105172529229462, 780544829745605743, 801343371679528758, 816120026090166881, 818497040519396521, 821240920750780361, 832212558718489617, 859011983665403215, 917143299895366489, 920838418575812689|
|practice_0034|independent|48|1|320|488|0|320|5.22|550358109712356093, 239041802034133299, 780544829745605743, 101181788029710766, 742989197954160843, 649158742032097674, 109045945666095854, 404909130390329023, 400440798686427318, 223263612503529506, 377935901965139177, 210161988755598540, 515620110477775414, 271733010962642628, 346088341674474587, 752105172529229462, 801343371679528758, 818497040519396521, 816120026090166881, 917143299895366489, 920838418575812689, 821240920750780361, 832212558718489617, 395633917438565177, 515127540626807406, 380725222159077806, 644965308286333487, 616494342142391550, 859011983665403215|
|practice_0034|independent|48|2|319|408|0|319|35.72|239041802034133299, 210161988755598540, 346088341674474587, 550358109712356093, 780544829745605743, 223263612503529506, 616494342142391550, 271733010962642628, 377935901965139177, 109045945666095854, 400440798686427318, 801343371679528758, 920838418575812689, 821240920750780361, 101181788029710766, 515620110477775414, 380725222159077806, 752105172529229462, 818497040519396521, 816120026090166881, 515127540626807406, 859011983665403215, 395633917438565177, 644965308286333487, 832212558718489617, 742989197954160843, 917143299895366489, 404909130390329023, 649158742032097674|
|practice_0034|independent|128|0|287|303|50|337|10.71|101181788029710766, 109045945666095854, 210161988755598540, 223263612503529506, 239041802034133299, 271733010962642628, 346088341674474587, 377935901965139177, 380725222159077806, 395633917438565177, 400440798686427318, 404909130390329023, 515127540626807406, 515620110477775414, 550358109712356093, 616494342142391550, 644965308286333487, 649158742032097674, 742989197954160843, 752105172529229462, 780544829745605743, 801343371679528758, 816120026090166881, 818497040519396521, 821240920750780361, 832212558718489617, 859011983665403215, 917143299895366489, 920838418575812689|
|practice_0034|independent|128|1|281|282|0|281|41.01|550358109712356093, 239041802034133299, 780544829745605743, 101181788029710766, 271733010962642628, 109045945666095854, 917143299895366489, 400440798686427318, 380725222159077806, 395633917438565177, 377935901965139177, 404909130390329023, 515127540626807406, 818497040519396521, 210161988755598540, 816120026090166881, 515620110477775414, 821240920750780361, 223263612503529506, 346088341674474587, 616494342142391550, 649158742032097674, 644965308286333487, 742989197954160843, 752105172529229462, 920838418575812689, 801343371679528758, 832212558718489617, 859011983665403215|
|practice_0034|independent|128|2|274|277|30|304|271.64|239041802034133299, 109045945666095854, 377935901965139177, 210161988755598540, 271733010962642628, 400440798686427318, 223263612503529506, 395633917438565177, 649158742032097674, 818497040519396521, 821240920750780361, 101181788029710766, 515127540626807406, 742989197954160843, 917143299895366489, 816120026090166881, 920838418575812689, 404909130390329023, 346088341674474587, 780544829745605743, 832212558718489617, 801343371679528758, 644965308286333487, 752105172529229462, 380725222159077806, 859011983665403215, 515620110477775414, 550358109712356093, 616494342142391550|
|practice_0035|mixed|0|0|265|0|0|265|3.82|35182377452, 70037177746, 70037177758, 244041791125, 244041792088, 814808207187, 1175024697628, 1512842230393, 1817995244091, 26597877345309, 29207756770182, 175246540615872, 212705443614808, 822517288311501, 12414975233668902, 136660875672831281, 167281911170880853, 191963669025938421, 275906607027906054, 347799814509203282, 350807270275043017, 486387939851822369, 510591983971474039, 523345138893544029, 531283651743083411, 575928774596356189, 725480556410650141, 875539911402847623, 929939713566498602, 950440197149264724|
|practice_0035|mixed|0|1|249|0|18|267|14.24|244041791125, 244041792088, 35182377452, 1512842230393, 26597877345309, 814808207187, 1175024697628, 29207756770182, 175246540615872, 70037177746, 70037177758, 822517288311501, 1817995244091, 212705443614808, 12414975233668902, 523345138893544029, 191963669025938421, 486387939851822369, 275906607027906054, 510591983971474039, 136660875672831281, 167281911170880853, 950440197149264724, 531283651743083411, 347799814509203282, 350807270275043017, 575928774596356189, 725480556410650141, 929939713566498602, 875539911402847623|
|practice_0035|mixed|0|2|246|0|0|246|104.92|814808207187, 70037177758, 35182377452, 244041792088, 1175024697628, 70037177746, 175246540615872, 244041791125, 1512842230393, 212705443614808, 1817995244091, 29207756770182, 26597877345309, 347799814509203282, 822517288311501, 167281911170880853, 12414975233668902, 136660875672831281, 191963669025938421, 929939713566498602, 510591983971474039, 350807270275043017, 575928774596356189, 486387939851822369, 950440197149264724, 275906607027906054, 875539911402847623, 523345138893544029, 531283651743083411, 725480556410650141|
|practice_0035|mixed|48|0|267|335|0|267|1.75|35182377452, 70037177746, 70037177758, 244041791125, 244041792088, 814808207187, 1175024697628, 1512842230393, 1817995244091, 26597877345309, 29207756770182, 175246540615872, 212705443614808, 822517288311501, 12414975233668902, 136660875672831281, 167281911170880853, 191963669025938421, 275906607027906054, 347799814509203282, 350807270275043017, 486387939851822369, 510591983971474039, 523345138893544029, 531283651743083411, 575928774596356189, 725480556410650141, 875539911402847623, 929939713566498602, 950440197149264724|
|practice_0035|mixed|48|1|254|284|0|254|4.32|244041791125, 244041792088, 35182377452, 1512842230393, 26597877345309, 70037177746, 70037177758, 1817995244091, 212705443614808, 814808207187, 1175024697628, 29207756770182, 175246540615872, 167281911170880853, 822517288311501, 12414975233668902, 275906607027906054, 347799814509203282, 350807270275043017, 136660875672831281, 875539911402847623, 486387939851822369, 191963669025938421, 510591983971474039, 575928774596356189, 523345138893544029, 929939713566498602, 950440197149264724, 725480556410650141, 531283651743083411|
|practice_0035|mixed|48|2|279|292|0|279|37.55|70037177746, 35182377452, 70037177758, 1512842230393, 244041791125, 1175024697628, 175246540615872, 244041792088, 822517288311501, 1817995244091, 575928774596356189, 212705443614808, 12414975233668902, 486387939851822369, 29207756770182, 26597877345309, 814808207187, 725480556410650141, 167281911170880853, 875539911402847623, 531283651743083411, 275906607027906054, 191963669025938421, 350807270275043017, 136660875672831281, 950440197149264724, 510591983971474039, 523345138893544029, 929939713566498602, 347799814509203282|
|practice_0035|mixed|128|0|243|248|0|243|8.04|35182377452, 70037177746, 70037177758, 244041791125, 244041792088, 814808207187, 1175024697628, 1512842230393, 1817995244091, 26597877345309, 29207756770182, 175246540615872, 212705443614808, 822517288311501, 12414975233668902, 136660875672831281, 167281911170880853, 191963669025938421, 275906607027906054, 347799814509203282, 350807270275043017, 486387939851822369, 510591983971474039, 523345138893544029, 531283651743083411, 575928774596356189, 725480556410650141, 875539911402847623, 929939713566498602, 950440197149264724|
|practice_0035|mixed|128|1|234|236|12|246|29.42|244041791125, 244041792088, 35182377452, 1512842230393, 26597877345309, 70037177746, 70037177758, 814808207187, 1175024697628, 175246540615872, 29207756770182, 822517288311501, 1817995244091, 212705443614808, 12414975233668902, 167281911170880853, 136660875672831281, 486387939851822369, 510591983971474039, 531283651743083411, 523345138893544029, 191963669025938421, 347799814509203282, 350807270275043017, 275906607027906054, 575928774596356189, 725480556410650141, 875539911402847623, 929939713566498602, 950440197149264724|
|practice_0035|mixed|128|2|235|237|30|265|206.52|70037177746, 1175024697628, 35182377452, 29207756770182, 70037177758, 1512842230393, 244041791125, 1817995244091, 244041792088, 175246540615872, 26597877345309, 212705443614808, 814808207187, 822517288311501, 12414975233668902, 191963669025938421, 875539911402847623, 531283651743083411, 523345138893544029, 167281911170880853, 575928774596356189, 275906607027906054, 510591983971474039, 350807270275043017, 725480556410650141, 347799814509203282, 136660875672831281, 486387939851822369, 929939713566498602, 950440197149264724|
|practice_0036|shared|0|0|133|0|0|133|2.92|17426, 289947, 289960, 359132, 930124, 2029652, 2147454, 2195676, 2886400, 5960650, 7162404, 7357660, 11923484, 12175939, 15457757, 22080972, 29805461, 41732376, 122595260, 225506610, 242894986, 608177934, 629679618, 751018371, 3185918994, 6689724261, 6757532603, 20961172756, 32064406080, 438886148064|
|practice_0036|shared|0|1|131|0|0|131|4.08|17426, 2195676, 2886400, 289947, 289960, 2029652, 359132, 930124, 2147454, 5960650, 751018371, 7162404, 7357660, 608177934, 11923484, 29805461, 12175939, 15457757, 22080972, 242894986, 41732376, 122595260, 629679618, 6757532603, 6689724261, 225506610, 3185918994, 32064406080, 438886148064, 20961172756|
|practice_0036|shared|0|2|126|0|0|126|24.28|289960, 17426, 289947, 2029652, 2147454, 930124, 11923484, 2886400, 5960650, 2195676, 7162404, 22080972, 359132, 29805461, 242894986, 7357660, 608177934, 41732376, 225506610, 12175939, 122595260, 629679618, 751018371, 6689724261, 3185918994, 20961172756, 6757532603, 438886148064, 15457757, 32064406080|
|practice_0036|shared|48|0|137|148|0|137|1.03|17426, 289947, 289960, 359132, 930124, 2029652, 2147454, 2195676, 2886400, 5960650, 7162404, 7357660, 11923484, 12175939, 15457757, 22080972, 29805461, 41732376, 122595260, 225506610, 242894986, 608177934, 629679618, 751018371, 3185918994, 6689724261, 6757532603, 20961172756, 32064406080, 438886148064|
|practice_0036|shared|48|1|142|168|0|142|1.82|17426, 2195676, 2886400, 2029652, 289947, 289960, 930124, 359132, 2147454, 5960650, 11923484, 7162404, 7357660, 12175939, 22080972, 41732376, 15457757, 29805461, 122595260, 225506610, 242894986, 608177934, 629679618, 6689724261, 3185918994, 6757532603, 751018371, 32064406080, 20961172756, 438886148064|
|practice_0036|shared|48|2|136|144|0|136|11.20|289960, 17426, 289947, 2029652, 2147454, 930124, 2886400, 5960650, 2195676, 11923484, 12175939, 7162404, 29805461, 22080972, 15457757, 41732376, 7357660, 359132, 225506610, 608177934, 122595260, 32064406080, 6757532603, 751018371, 242894986, 629679618, 3185918994, 438886148064, 6689724261, 20961172756|
|practice_0036|shared|128|0|129|131|1|130|3.74|17426, 289947, 289960, 359132, 930124, 2029652, 2147454, 2195676, 2886400, 5960650, 7162404, 7357660, 11923484, 12175939, 15457757, 22080972, 29805461, 41732376, 122595260, 225506610, 242894986, 608177934, 629679618, 751018371, 3185918994, 6689724261, 6757532603, 20961172756, 32064406080, 438886148064|
|practice_0036|shared|128|1|128|130|0|128|6.14|17426, 2195676, 2886400, 2029652, 289947, 289960, 930124, 359132, 2147454, 5960650, 11923484, 29805461, 7162404, 7357660, 12175939, 15457757, 22080972, 225506610, 242894986, 41732376, 122595260, 608177934, 629679618, 751018371, 3185918994, 6689724261, 6757532603, 32064406080, 20961172756, 438886148064|
|practice_0036|shared|128|2|129|130|0|129|37.30|289960, 17426, 289947, 2029652, 2147454, 930124, 11923484, 2886400, 29805461, 7162404, 2195676, 5960650, 22080972, 359132, 242894986, 122595260, 12175939, 7357660, 15457757, 225506610, 41732376, 608177934, 629679618, 751018371, 6689724261, 3185918994, 6757532603, 20961172756, 32064406080, 438886148064|
|practice_0037|independent|0|0|293|0|0|293|3.66|102495003043300, 141525022496131, 172964021511505, 187656483268684, 221465847362582, 234015911007239, 244703437611103, 267558627397255, 294671002617233, 335459352459461, 346683852842805, 394805884825055, 426156915115293, 435513367615915, 446282957613116, 460015581371104, 487633168643086, 506835759809868, 552655901998713, 569510974201597, 602127194567663, 643975359912352, 645578042705681, 710135517856240, 723521510181670, 745395759602787, 788118642638711, 797665454552205, 807242541118465, 838943947672286, 854782465092750, 918568133070286|
|practice_0037|independent|0|1|274|0|58|332|12.14|141525022496131, 102495003043300, 643975359912352, 294671002617233, 172964021511505, 335459352459461, 435513367615915, 187656483268684, 221465847362582, 234015911007239, 267558627397255, 745395759602787, 346683852842805, 244703437611103, 506835759809868, 569510974201597, 552655901998713, 394805884825055, 426156915115293, 446282957613116, 645578042705681, 487633168643086, 854782465092750, 838943947672286, 460015581371104, 797665454552205, 788118642638711, 723521510181670, 710135517856240, 602127194567663, 807242541118465, 918568133070286|
|practice_0037|independent|0|2|269|0|191|460|75.47|267558627397255, 643975359912352, 435513367615915, 102495003043300, 645578042705681, 487633168643086, 552655901998713, 426156915115293, 602127194567663, 294671002617233, 723521510181670, 187656483268684, 172964021511505, 335459352459461, 234015911007239, 854782465092750, 221465847362582, 460015581371104, 244703437611103, 710135517856240, 446282957613116, 569510974201597, 807242541118465, 918568133070286, 506835759809868, 797665454552205, 838943947672286, 745395759602787, 346683852842805, 788118642638711, 394805884825055, 141525022496131|
|practice_0037|independent|48|0|301|361|0|301|1.66|102495003043300, 141525022496131, 172964021511505, 187656483268684, 221465847362582, 234015911007239, 244703437611103, 267558627397255, 294671002617233, 335459352459461, 346683852842805, 394805884825055, 426156915115293, 435513367615915, 446282957613116, 460015581371104, 487633168643086, 506835759809868, 552655901998713, 569510974201597, 602127194567663, 643975359912352, 645578042705681, 710135517856240, 723521510181670, 745395759602787, 788118642638711, 797665454552205, 807242541118465, 838943947672286, 854782465092750, 918568133070286|
|practice_0037|independent|48|1|296|366|0|296|5.38|141525022496131, 102495003043300, 267558627397255, 294671002617233, 460015581371104, 394805884825055, 187656483268684, 346683852842805, 172964021511505, 645578042705681, 506835759809868, 552655901998713, 643975359912352, 918568133070286, 234015911007239, 221465847362582, 244703437611103, 435513367615915, 426156915115293, 335459352459461, 446282957613116, 745395759602787, 723521510181670, 710135517856240, 807242541118465, 602127194567663, 838943947672286, 797665454552205, 788118642638711, 854782465092750, 487633168643086, 569510974201597|
|practice_0037|independent|48|2|296|312|0|296|30.68|267558627397255, 643975359912352, 187656483268684, 426156915115293, 244703437611103, 435513367615915, 854782465092750, 723521510181670, 141525022496131, 446282957613116, 102495003043300, 294671002617233, 487633168643086, 552655901998713, 234015911007239, 506835759809868, 569510974201597, 645578042705681, 710135517856240, 394805884825055, 346683852842805, 918568133070286, 797665454552205, 745395759602787, 788118642638711, 221465847362582, 172964021511505, 838943947672286, 602127194567663, 460015581371104, 335459352459461, 807242541118465|
|practice_0037|independent|128|0|267|271|2|269|8.55|102495003043300, 141525022496131, 172964021511505, 187656483268684, 221465847362582, 234015911007239, 244703437611103, 267558627397255, 294671002617233, 335459352459461, 346683852842805, 394805884825055, 426156915115293, 435513367615915, 446282957613116, 460015581371104, 487633168643086, 506835759809868, 552655901998713, 569510974201597, 602127194567663, 643975359912352, 645578042705681, 710135517856240, 723521510181670, 745395759602787, 788118642638711, 797665454552205, 807242541118465, 838943947672286, 854782465092750, 918568133070286|
|practice_0037|independent|128|1|259|262|0|259|33.68|141525022496131, 102495003043300, 267558627397255, 172964021511505, 234015911007239, 221465847362582, 187656483268684, 335459352459461, 506835759809868, 487633168643086, 244703437611103, 346683852842805, 394805884825055, 294671002617233, 552655901998713, 797665454552205, 569510974201597, 446282957613116, 426156915115293, 645578042705681, 643975359912352, 460015581371104, 602127194567663, 745395759602787, 435513367615915, 854782465092750, 710135517856240, 723521510181670, 788118642638711, 807242541118465, 838943947672286, 918568133070286|
|practice_0037|independent|128|2|248|251|48|296|188.14|267558627397255, 643975359912352, 487633168643086, 172964021511505, 141525022496131, 102495003043300, 244703437611103, 335459352459461, 221465847362582, 234015911007239, 346683852842805, 552655901998713, 569510974201597, 446282957613116, 187656483268684, 394805884825055, 645578042705681, 435513367615915, 294671002617233, 426156915115293, 602127194567663, 710135517856240, 460015581371104, 838943947672286, 745395759602787, 723521510181670, 854782465092750, 807242541118465, 506835759809868, 797665454552205, 788118642638711, 918568133070286|
|practice_0038|independent|0|0|45|0|0|45|0.13|147246, 384456, 498994, 509305, 516935, 857303, 874926, 952513|
|practice_0038|independent|0|1|45|0|0|45|0.13|147246, 384456, 498994, 509305, 516935, 857303, 874926, 952513|
|practice_0038|independent|0|2|46|0|0|46|0.54|857303, 509305, 147246, 384456, 498994, 516935, 874926, 952513|
|practice_0038|independent|48|0|45|53|0|45|0.11|147246, 384456, 498994, 509305, 516935, 857303, 874926, 952513|
|practice_0038|independent|48|1|45|52|0|45|0.12|147246, 874926, 384456, 498994, 509305, 516935, 857303, 952513|
|practice_0038|independent|48|2|47|51|0|47|0.55|857303, 147246, 509305, 384456, 498994, 516935, 874926, 952513|
|practice_0038|independent|128|0|46|53|0|46|0.16|147246, 384456, 498994, 509305, 516935, 857303, 874926, 952513|
|practice_0038|independent|128|1|46|52|0|46|0.14|147246, 874926, 384456, 498994, 509305, 516935, 857303, 952513|
|practice_0038|independent|128|2|47|51|0|47|0.55|857303, 147246, 509305, 384456, 498994, 516935, 874926, 952513|
|practice_0039|tiny|0|0|9|0|0|9|0.01|7, 9, 12, 13|
|practice_0039|tiny|0|1|9|0|0|9|0.01|7, 9, 13, 12|
|practice_0039|tiny|0|2|9|0|0|9|0.01|7, 9, 13, 12|
|practice_0039|tiny|48|0|9|13|0|9|0.01|7, 9, 13, 12|
|practice_0039|tiny|48|1|9|13|0|9|0.01|7, 9, 13, 12|
|practice_0039|tiny|48|2|9|13|0|9|0.01|7, 9, 13, 12|
|practice_0039|tiny|128|0|9|13|0|9|0.01|7, 9, 13, 12|
|practice_0039|tiny|128|1|9|13|0|9|0.01|7, 9, 13, 12|
|practice_0039|tiny|128|2|9|13|0|9|0.01|7, 9, 13, 12|
|practice_0040|mixed|0|0|203|0|0|203|3.12|19428256, 19430440, 33872434, 41224831, 87459927, 436127824, 446897475, 575831193, 2011565119, 3514671387, 3514674726, 3701146364, 5647635375, 9314985075, 248088810817, 348335089281, 356093123851, 367863797424, 368440411327, 430192057235, 689774154994, 733660654959, 736340996779, 798675373071, 819848861015, 845675198009, 880753534713, 927989384279, 934562986007, 942246779738, 977907380498, 7732326278600|
|practice_0040|mixed|0|1|195|0|0|195|9.42|19428256, 19430440, 446897475, 33872434, 575831193, 9314985075, 41224831, 87459927, 2011565119, 436127824, 3514671387, 3514674726, 3701146364, 7732326278600, 5647635375, 880753534713, 348335089281, 248088810817, 356093123851, 367863797424, 368440411327, 689774154994, 733660654959, 736340996779, 430192057235, 798675373071, 845675198009, 934562986007, 819848861015, 927989384279, 942246779738, 977907380498|
|practice_0040|mixed|0|2|187|0|0|187|57.93|33872434, 19430440, 87459927, 9314985075, 2011565119, 3514671387, 19428256, 3701146364, 446897475, 41224831, 575831193, 3514674726, 436127824, 356093123851, 368440411327, 367863797424, 5647635375, 798675373071, 430192057235, 819848861015, 880753534713, 348335089281, 733660654959, 927989384279, 942246779738, 248088810817, 845675198009, 689774154994, 736340996779, 934562986007, 7732326278600, 977907380498|
|practice_0040|mixed|48|0|204|309|0|204|1.39|19428256, 19430440, 33872434, 41224831, 87459927, 436127824, 446897475, 575831193, 2011565119, 3514671387, 3514674726, 3701146364, 5647635375, 9314985075, 248088810817, 348335089281, 356093123851, 367863797424, 368440411327, 430192057235, 689774154994, 733660654959, 736340996779, 798675373071, 819848861015, 845675198009, 880753534713, 927989384279, 934562986007, 942246779738, 977907380498, 7732326278600|
|practice_0040|mixed|48|1|197|290|0|197|3.96|19428256, 19430440, 446897475, 33872434, 575831193, 9314985075, 87459927, 2011565119, 41224831, 436127824, 248088810817, 3514671387, 3514674726, 5647635375, 3701146364, 348335089281, 356093123851, 367863797424, 368440411327, 430192057235, 736340996779, 733660654959, 689774154994, 7732326278600, 819848861015, 934562986007, 977907380498, 798675373071, 880753534713, 927989384279, 942246779738, 845675198009|
|practice_0040|mixed|48|2|197|204|0|197|25.31|33872434, 19430440, 87459927, 446897475, 575831193, 19428256, 5647635375, 41224831, 2011565119, 3514674726, 3514671387, 9314985075, 798675373071, 248088810817, 436127824, 845675198009, 977907380498, 880753534713, 3701146364, 430192057235, 368440411327, 367863797424, 689774154994, 736340996779, 819848861015, 927989384279, 934562986007, 733660654959, 356093123851, 942246779738, 348335089281, 7732326278600|
|practice_0040|mixed|128|0|188|197|15|203|6.43|19428256, 19430440, 33872434, 41224831, 87459927, 436127824, 446897475, 575831193, 2011565119, 3514671387, 3514674726, 3701146364, 5647635375, 9314985075, 248088810817, 348335089281, 356093123851, 367863797424, 368440411327, 430192057235, 689774154994, 733660654959, 736340996779, 798675373071, 819848861015, 845675198009, 880753534713, 927989384279, 934562986007, 942246779738, 977907380498, 7732326278600|
|practice_0040|mixed|128|1|183|183|0|183|16.72|19428256, 19430440, 446897475, 33872434, 575831193, 9314985075, 41224831, 87459927, 2011565119, 436127824, 3514671387, 3514674726, 3701146364, 5647635375, 880753534713, 248088810817, 348335089281, 733660654959, 367863797424, 368440411327, 736340996779, 356093123851, 430192057235, 927989384279, 942246779738, 934562986007, 977907380498, 689774154994, 798675373071, 819848861015, 845675198009, 7732326278600|
|practice_0040|mixed|128|2|174|176|0|174|111.03|33872434, 19430440, 87459927, 446897475, 19428256, 575831193, 3514671387, 3514674726, 2011565119, 41224831, 9314985075, 3701146364, 5647635375, 436127824, 367863797424, 430192057235, 368440411327, 356093123851, 798675373071, 689774154994, 736340996779, 733660654959, 348335089281, 845675198009, 934562986007, 927989384279, 880753534713, 819848861015, 248088810817, 977907380498, 942246779738, 7732326278600|
|practice_0041|mixed|0|0|188|0|0|188|2.53|6103481, 6282598, 6283047, 6695959, 7357588, 8274353, 20177152, 29432460, 75395736, 116279712, 793460590, 793555354, 1000662288, 2895407848, 217558097654, 221216340201, 278038653587, 367009981557, 442289962028, 499938625213, 509048381965, 561026817391, 575108119641, 611488555004, 644251940711, 657320508085, 822018938835, 831212921473, 879893339748, 895278558671, 954157236758|
|practice_0041|mixed|0|1|182|0|0|182|6.45|7357588, 29432460, 6695959, 6103481, 6282598, 6283047, 75395736, 8274353, 20177152, 116279712, 793460590, 793555354, 1000662288, 2895407848, 221216340201, 367009981557, 217558097654, 442289962028, 644251940711, 278038653587, 499938625213, 509048381965, 657320508085, 561026817391, 822018938835, 831212921473, 575108119641, 611488555004, 879893339748, 895278558671, 954157236758|
|practice_0041|mixed|0|2|176|0|30|206|43.20|6283047, 6695959, 8274353, 6282598, 6103481, 75395736, 29432460, 7357588, 20177152, 793460590, 1000662288, 116279712, 793555354, 2895407848, 221216340201, 499938625213, 644251940711, 611488555004, 278038653587, 442289962028, 822018938835, 367009981557, 217558097654, 831212921473, 895278558671, 657320508085, 509048381965, 575108119641, 561026817391, 879893339748, 954157236758|
|practice_0041|mixed|48|0|191|251|0|191|1.26|6103481, 6282598, 6283047, 6695959, 7357588, 8274353, 20177152, 29432460, 75395736, 116279712, 793460590, 793555354, 1000662288, 2895407848, 217558097654, 221216340201, 278038653587, 367009981557, 442289962028, 499938625213, 509048381965, 561026817391, 575108119641, 611488555004, 644251940711, 657320508085, 822018938835, 831212921473, 879893339748, 895278558671, 954157236758|
|practice_0041|mixed|48|1|190|199|0|190|3.50|7357588, 29432460, 6695959, 6103481, 6282598, 6283047, 75395736, 8274353, 20177152, 116279712, 793460590, 793555354, 1000662288, 2895407848, 442289962028, 499938625213, 575108119641, 644251940711, 367009981557, 561026817391, 217558097654, 221216340201, 278038653587, 509048381965, 611488555004, 657320508085, 831212921473, 822018938835, 895278558671, 879893339748, 954157236758|
|practice_0041|mixed|48|2|194|201|0|194|18.95|6283047, 7357588, 29432460, 6282598, 6695959, 8274353, 116279712, 75395736, 1000662288, 278038653587, 793460590, 6103481, 793555354, 575108119641, 442289962028, 367009981557, 509048381965, 2895407848, 831212921473, 879893339748, 20177152, 217558097654, 657320508085, 499938625213, 644251940711, 822018938835, 611488555004, 561026817391, 221216340201, 954157236758, 895278558671|
|practice_0041|mixed|128|0|175|180|92|267|6.16|6103481, 6282598, 6283047, 6695959, 7357588, 8274353, 20177152, 29432460, 75395736, 116279712, 793460590, 793555354, 1000662288, 2895407848, 217558097654, 221216340201, 278038653587, 367009981557, 442289962028, 499938625213, 509048381965, 561026817391, 575108119641, 611488555004, 644251940711, 657320508085, 822018938835, 831212921473, 879893339748, 895278558671, 954157236758|
|practice_0041|mixed|128|1|170|174|43|213|13.90|7357588, 29432460, 6695959, 6103481, 6282598, 6283047, 75395736, 8274353, 20177152, 116279712, 793460590, 793555354, 1000662288, 2895407848, 221216340201, 278038653587, 217558097654, 367009981557, 442289962028, 499938625213, 561026817391, 509048381965, 575108119641, 611488555004, 644251940711, 954157236758, 822018938835, 657320508085, 831212921473, 879893339748, 895278558671|
|practice_0041|mixed|128|2|165|170|0|165|92.99|6283047, 7357588, 29432460, 8274353, 6103481, 116279712, 6695959, 6282598, 75395736, 20177152, 1000662288, 793460590, 793555354, 575108119641, 278038653587, 644251940711, 2895407848, 561026817391, 657320508085, 217558097654, 221216340201, 509048381965, 611488555004, 442289962028, 879893339748, 822018938835, 499938625213, 831212921473, 895278558671, 954157236758, 367009981557|
|practice_0042|shared|0|0|185|0|0|185|3.41|1287230909, 2246822756, 2246825111, 2445515689, 2445518057, 3069088963, 3340493037, 8668913532, 11585077548, 26018219528, 31731797670, 44019292600, 46340324903, 145966281380, 935815796337, 1158507795060, 1430627032813, 1633441042911, 1633441052999, 1686519142482, 2428540485858, 4549815667800, 4709298416757, 5184010246642, 13403511004224, 26990139197160, 59979442065634, 1355063184791912, 2281432639738550|
|practice_0042|shared|0|1|170|0|0|170|7.48|8668913532, 2246822756, 2246825111, 2445518057, 2445515689, 44019292600, 11585077548, 1287230909, 46340324903, 1158507795060, 3069088963, 3340493037, 1430627032813, 935815796337, 1633441042911, 1633441052999, 2428540485858, 4549815667800, 5184010246642, 26018219528, 145966281380, 31731797670, 13403511004224, 4709298416757, 59979442065634, 1355063184791912, 2281432639738550, 1686519142482, 26990139197160|
|practice_0042|shared|0|2|170|0|0|170|65.09|2246825111, 2445518057, 1287230909, 2246822756, 2445515689, 3069088963, 3340493037, 31731797670, 11585077548, 46340324903, 145966281380, 1158507795060, 1430627032813, 8668913532, 1633441052999, 1633441042911, 44019292600, 59979442065634, 935815796337, 2428540485858, 13403511004224, 26018219528, 4709298416757, 4549815667800, 2281432639738550, 1355063184791912, 5184010246642, 1686519142482, 26990139197160|
|practice_0042|shared|48|0|200|251|0|200|1.23|1287230909, 2246822756, 2246825111, 2445515689, 2445518057, 3069088963, 3340493037, 8668913532, 11585077548, 26018219528, 31731797670, 44019292600, 46340324903, 145966281380, 935815796337, 1158507795060, 1430627032813, 1633441042911, 1633441052999, 1686519142482, 2428540485858, 4549815667800, 4709298416757, 5184010246642, 13403511004224, 26990139197160, 59979442065634, 1355063184791912, 2281432639738550|
|practice_0042|shared|48|1|187|194|0|187|3.86|8668913532, 2246822756, 2246825111, 2445518057, 2445515689, 44019292600, 11585077548, 46340324903, 1287230909, 1158507795060, 3340493037, 5184010246642, 3069088963, 145966281380, 26018219528, 1430627032813, 935815796337, 31731797670, 13403511004224, 1633441042911, 1633441052999, 1686519142482, 2428540485858, 4549815667800, 4709298416757, 59979442065634, 2281432639738550, 26990139197160, 1355063184791912|
|practice_0042|shared|48|2|181|192|0|181|23.70|2246825111, 1287230909, 935815796337, 11585077548, 1633441042911, 3069088963, 1633441052999, 3340493037, 2445518057, 2246822756, 2445515689, 26018219528, 44019292600, 8668913532, 31731797670, 4709298416757, 1430627032813, 145966281380, 5184010246642, 13403511004224, 26990139197160, 46340324903, 59979442065634, 4549815667800, 2281432639738550, 2428540485858, 1158507795060, 1355063184791912, 1686519142482|
|practice_0042|shared|128|0|169|176|23|192|5.44|1287230909, 2246822756, 2246825111, 2445515689, 2445518057, 3069088963, 3340493037, 8668913532, 11585077548, 26018219528, 31731797670, 44019292600, 46340324903, 145966281380, 935815796337, 1158507795060, 1430627032813, 1633441042911, 1633441052999, 1686519142482, 2428540485858, 4549815667800, 4709298416757, 5184010246642, 13403511004224, 26990139197160, 59979442065634, 1355063184791912, 2281432639738550|
|practice_0042|shared|128|1|172|175|26|198|17.50|8668913532, 2246822756, 2246825111, 2445518057, 2445515689, 44019292600, 46340324903, 11585077548, 1287230909, 1158507795060, 3069088963, 5184010246642, 3340493037, 26018219528, 31731797670, 145966281380, 13403511004224, 935815796337, 1430627032813, 1633441042911, 1633441052999, 4549815667800, 1686519142482, 2428540485858, 4709298416757, 26990139197160, 59979442065634, 1355063184791912, 2281432639738550|
|practice_0042|shared|128|2|156|160|0|156|88.63|2246825111, 1287230909, 935815796337, 11585077548, 2445515689, 2445518057, 1633441042911, 2246822756, 44019292600, 46340324903, 3340493037, 3069088963, 1158507795060, 1633441052999, 2428540485858, 31731797670, 145966281380, 13403511004224, 1355063184791912, 59979442065634, 8668913532, 1686519142482, 4549815667800, 1430627032813, 5184010246642, 26018219528, 4709298416757, 26990139197160, 2281432639738550|
|practice_0043|shared|0|0|127|0|0|127|2.83|1551922, 1746502, 3576614, 3576723, 3576800, 5937454, 12416063, 13791730, 15892400, 22004673, 24831737, 47677743, 49663073, 50995606, 57227888, 95355180, 113287755, 127141626, 219762000, 254280806, 549400377, 622655058, 1271406800, 2317722802, 2482515385, 9742702914, 42051674016, 1177953652200|
|practice_0043|shared|0|1|123|0|0|123|3.70|3576800, 3576723, 3576614, 57227888, 1746502, 5937454, 1551922, 12416063, 49663073, 24831737, 13791730, 15892400, 47677743, 95355180, 22004673, 50995606, 113287755, 127141626, 219762000, 1271406800, 254280806, 622655058, 549400377, 2317722802, 2482515385, 9742702914, 42051674016, 1177953652200|
|practice_0043|shared|0|2|125|0|0|125|25.74|3576614, 15892400, 1551922, 3576800, 3576723, 1746502, 5937454, 24831737, 47677743, 12416063, 49663073, 57227888, 50995606, 95355180, 1271406800, 22004673, 254280806, 13791730, 113287755, 622655058, 219762000, 549400377, 2482515385, 127141626, 2317722802, 9742702914, 1177953652200, 42051674016|
|practice_0043|shared|48|0|130|169|0|130|0.98|1551922, 1746502, 3576614, 3576723, 3576800, 5937454, 12416063, 13791730, 15892400, 22004673, 24831737, 47677743, 49663073, 50995606, 57227888, 95355180, 113287755, 127141626, 219762000, 254280806, 549400377, 622655058, 1271406800, 2317722802, 2482515385, 9742702914, 42051674016, 1177953652200|
|practice_0043|shared|48|1|137|144|0|137|1.64|3576800, 3576723, 3576614, 57227888, 1746502, 1551922, 5937454, 12416063, 13791730, 15892400, 22004673, 24831737, 49663073, 47677743, 50995606, 95355180, 113287755, 127141626, 254280806, 1271406800, 549400377, 219762000, 622655058, 2317722802, 2482515385, 42051674016, 1177953652200, 9742702914|
|practice_0043|shared|48|2|129|138|0|129|12.20|3576614, 15892400, 1551922, 3576800, 3576723, 1746502, 49663073, 47677743, 95355180, 12416063, 24831737, 5937454, 127141626, 50995606, 13791730, 113287755, 1271406800, 57227888, 2482515385, 254280806, 9742702914, 1177953652200, 22004673, 549400377, 2317722802, 219762000, 622655058, 42051674016|
|practice_0043|shared|128|0|126|130|0|126|3.65|1551922, 1746502, 3576614, 3576723, 3576800, 5937454, 12416063, 13791730, 15892400, 22004673, 24831737, 47677743, 49663073, 50995606, 57227888, 95355180, 113287755, 127141626, 219762000, 254280806, 549400377, 622655058, 1271406800, 2317722802, 2482515385, 9742702914, 42051674016, 1177953652200|
|practice_0043|shared|128|1|127|130|0|127|6.58|3576800, 3576723, 3576614, 57227888, 1746502, 1551922, 5937454, 12416063, 13791730, 15892400, 22004673, 24831737, 49663073, 47677743, 95355180, 50995606, 113287755, 127141626, 254280806, 1271406800, 219762000, 549400377, 622655058, 2482515385, 2317722802, 9742702914, 42051674016, 1177953652200|
|practice_0043|shared|128|2|122|127|0|122|36.61|3576614, 15892400, 1551922, 3576800, 3576723, 12416063, 5937454, 47677743, 1746502, 49663073, 24831737, 22004673, 13791730, 57227888, 95355180, 127141626, 50995606, 254280806, 42051674016, 113287755, 219762000, 549400377, 622655058, 2482515385, 1271406800, 2317722802, 9742702914, 1177953652200|
|practice_0044|mixed|0|0|233|0|0|233|2.79|3328909854, 4831420733, 26795330208, 29964363456, 70669379118, 89880567516, 214362662304, 2073159353418, 2396384671202, 26053048593920, 120655476597336, 165497714971609, 207425480158176, 266658265523546, 317286341281166, 508033948936893, 559502821692540, 572450429986320, 587241639097911, 638950594850897, 677016612794704, 724742000299431, 741839608390223, 814211530392690, 898424384083233, 906286658749688, 974768631252990, 1476153625646296|
|practice_0044|mixed|0|1|216|0|0|216|11.29|4831420733, 2396384671202, 3328909854, 89880567516, 29964363456, 26795330208, 214362662304, 70669379118, 2073159353418, 26053048593920, 120655476597336, 207425480158176, 559502821692540, 572450429986320, 165497714971609, 814211530392690, 266658265523546, 724742000299431, 317286341281166, 638950594850897, 898424384083233, 508033948936893, 677016612794704, 587241639097911, 974768631252990, 741839608390223, 906286658749688, 1476153625646296|
|practice_0044|mixed|0|2|214|0|155|369|75.56|29964363456, 26795330208, 2073159353418, 3328909854, 572450429986320, 4831420733, 89880567516, 214362662304, 26053048593920, 2396384671202, 207425480158176, 677016612794704, 70669379118, 165497714971609, 814211530392690, 120655476597336, 724742000299431, 508033948936893, 741839608390223, 587241639097911, 559502821692540, 638950594850897, 974768631252990, 266658265523546, 906286658749688, 898424384083233, 1476153625646296, 317286341281166|
|practice_0044|mixed|48|0|238|278|0|238|1.58|3328909854, 4831420733, 26795330208, 29964363456, 70669379118, 89880567516, 214362662304, 2073159353418, 2396384671202, 26053048593920, 120655476597336, 165497714971609, 207425480158176, 266658265523546, 317286341281166, 508033948936893, 559502821692540, 572450429986320, 587241639097911, 638950594850897, 677016612794704, 724742000299431, 741839608390223, 814211530392690, 898424384083233, 906286658749688, 974768631252990, 1476153625646296|
|practice_0044|mixed|48|1|226|233|0|226|5.01|4831420733, 2396384671202, 3328909854, 89880567516, 26795330208, 214362662304, 29964363456, 70669379118, 26053048593920, 572450429986320, 2073159353418, 207425480158176, 638950594850897, 266658265523546, 120655476597336, 165497714971609, 317286341281166, 724742000299431, 559502821692540, 741839608390223, 814211530392690, 587241639097911, 508033948936893, 677016612794704, 1476153625646296, 906286658749688, 974768631252990, 898424384083233|
|practice_0044|mixed|48|2|232|242|0|232|30.57|29964363456, 26795330208, 89880567516, 3328909854, 214362662304, 26053048593920, 4831420733, 207425480158176, 2073159353418, 2396384671202, 572450429986320, 165497714971609, 120655476597336, 814211530392690, 638950594850897, 741839608390223, 898424384083233, 587241639097911, 906286658749688, 508033948936893, 559502821692540, 1476153625646296, 724742000299431, 70669379118, 266658265523546, 317286341281166, 974768631252990, 677016612794704|
|practice_0044|mixed|128|0|211|222|33|244|7.42|3328909854, 4831420733, 26795330208, 29964363456, 70669379118, 89880567516, 214362662304, 2073159353418, 2396384671202, 26053048593920, 120655476597336, 165497714971609, 207425480158176, 266658265523546, 317286341281166, 508033948936893, 559502821692540, 572450429986320, 587241639097911, 638950594850897, 677016612794704, 724742000299431, 741839608390223, 814211530392690, 898424384083233, 906286658749688, 974768631252990, 1476153625646296|
|practice_0044|mixed|128|1|201|205|20|221|21.58|4831420733, 2396384671202, 3328909854, 89880567516, 29964363456, 26795330208, 214362662304, 70669379118, 572450429986320, 2073159353418, 26053048593920, 120655476597336, 165497714971609, 207425480158176, 677016612794704, 266658265523546, 508033948936893, 638950594850897, 587241639097911, 559502821692540, 898424384083233, 317286341281166, 724742000299431, 741839608390223, 814211530392690, 1476153625646296, 906286658749688, 974768631252990|
|practice_0044|mixed|128|2|195|205|91|286|121.35|29964363456, 26795330208, 89880567516, 4831420733, 214362662304, 2396384671202, 3328909854, 2073159353418, 572450429986320, 26053048593920, 1476153625646296, 120655476597336, 165497714971609, 559502821692540, 70669379118, 207425480158176, 814211530392690, 266658265523546, 638950594850897, 587241639097911, 724742000299431, 898424384083233, 741839608390223, 677016612794704, 317286341281166, 974768631252990, 508033948936893, 906286658749688|
|practice_0045|multiplicative|0|0|150|0|0|150|4.49|8272, 18246, 124079, 187989, 239320, 239577, 282083, 294537, 423131, 1774941, 2345339, 2367326, 3616041, 3934892, 5913112, 12690244, 19605140, 24821918, 45411084, 54615808, 55111004, 94485989, 211536598, 241541784, 1548112783, 1620259125, 2507357126, 177303518960, 736175691264, 2526684042688, 10060283143158, 2489332104889344|
|practice_0045|multiplicative|0|1|148|0|0|148|9.77|8272, 124079, 18246, 187989, 294537, 423131, 239320, 239577, 282083, 2345339, 2367326, 3934892, 5913112, 1774941, 3616041, 12690244, 19605140, 24821918, 45411084, 54615808, 55111004, 94485989, 211536598, 1548112783, 241541784, 1620259125, 2507357126, 177303518960, 736175691264, 2526684042688, 10060283143158, 2489332104889344|
|practice_0045|multiplicative|0|2|141|0|0|141|51.68|18246, 124079, 423131, 294537, 187989, 8272, 239320, 239577, 282083, 3616041, 1774941, 5913112, 2345339, 2367326, 24821918, 19605140, 3934892, 211536598, 45411084, 54615808, 12690244, 55111004, 94485989, 241541784, 1548112783, 1620259125, 2507357126, 177303518960, 10060283143158, 736175691264, 2526684042688, 2489332104889344|
|practice_0045|multiplicative|48|0|155|177|0|155|1.64|8272, 18246, 124079, 187989, 239320, 239577, 282083, 294537, 423131, 1774941, 2345339, 2367326, 3616041, 3934892, 5913112, 12690244, 19605140, 24821918, 45411084, 54615808, 55111004, 94485989, 211536598, 241541784, 1548112783, 1620259125, 2507357126, 177303518960, 736175691264, 2526684042688, 10060283143158, 2489332104889344|
|practice_0045|multiplicative|48|1|157|277|0|157|3.15|8272, 124079, 18246, 187989, 294537, 282083, 423131, 239320, 239577, 1774941, 2367326, 3934892, 5913112, 2345339, 3616041, 24821918, 45411084, 55111004, 94485989, 12690244, 54615808, 19605140, 211536598, 241541784, 1548112783, 1620259125, 2507357126, 736175691264, 2526684042688, 177303518960, 2489332104889344, 10060283143158|
|practice_0045|multiplicative|48|2|153|162|0|153|19.89|18246, 187989, 124079, 294537, 423131, 239577, 239320, 8272, 55111004, 2345339, 3934892, 12690244, 24821918, 282083, 3616041, 45411084, 54615808, 2367326, 241541784, 1774941, 211536598, 94485989, 5913112, 1548112783, 2507357126, 1620259125, 177303518960, 2526684042688, 2489332104889344, 19605140, 736175691264, 10060283143158|
|practice_0045|multiplicative|128|0|141|146|21|162|5.04|8272, 18246, 124079, 187989, 239320, 239577, 282083, 294537, 423131, 1774941, 2345339, 2367326, 3616041, 3934892, 5913112, 12690244, 19605140, 24821918, 45411084, 54615808, 55111004, 94485989, 211536598, 241541784, 1548112783, 1620259125, 2507357126, 177303518960, 736175691264, 2526684042688, 10060283143158, 2489332104889344|
|practice_0045|multiplicative|128|1|136|142|21|157|9.55|8272, 124079, 18246, 187989, 294537, 282083, 423131, 239320, 239577, 1774941, 2345339, 2367326, 3934892, 5913112, 3616041, 12690244, 19605140, 24821918, 45411084, 54615808, 55111004, 94485989, 211536598, 241541784, 1548112783, 1620259125, 2507357126, 177303518960, 736175691264, 2526684042688, 10060283143158, 2489332104889344|
|practice_0045|multiplicative|128|2|139|143|4|143|63.90|18246, 187989, 124079, 294537, 423131, 8272, 239577, 239320, 282083, 2345339, 3934892, 5913112, 1774941, 3616041, 2367326, 24821918, 12690244, 19605140, 45411084, 54615808, 55111004, 241541784, 94485989, 211536598, 1548112783, 1620259125, 2507357126, 177303518960, 736175691264, 2526684042688, 10060283143158, 2489332104889344|
|practice_0046|independent|0|0|271|0|0|271|2.98|106857327016504, 120107009373960, 213727296656642, 267674849940558, 269150504629490, 284901015339268, 286005960631776, 294562493860331, 335327740827091, 344627056149928, 361676363214944, 407553114321277, 435828369024637, 436185658550699, 444619663795306, 448204359651896, 464010369447126, 491498225353097, 544886727197518, 595914041030441, 628835009200270, 631602058754337, 702738002324993, 732944065819547, 751356426860707, 820557627598087, 873237992325777, 942817669745124, 943543162245283, 997999249847235|
|practice_0046|independent|0|1|264|0|40|304|12.29|120107009373960, 213727296656642, 106857327016504, 267674849940558, 269150504629490, 286005960631776, 436185658550699, 751356426860707, 435828369024637, 284901015339268, 344627056149928, 294562493860331, 444619663795306, 448204359651896, 407553114321277, 335327740827091, 361676363214944, 544886727197518, 491498225353097, 628835009200270, 732944065819547, 631602058754337, 942817669745124, 943543162245283, 464010369447126, 702738002324993, 997999249847235, 820557627598087, 595914041030441, 873237992325777|
|practice_0046|independent|0|2|253|0|12|265|75.76|213727296656642, 335327740827091, 267674849940558, 106857327016504, 120107009373960, 344627056149928, 284901015339268, 361676363214944, 407553114321277, 294562493860331, 444619663795306, 464010369447126, 269150504629490, 628835009200270, 631602058754337, 436185658550699, 286005960631776, 751356426860707, 448204359651896, 435828369024637, 873237992325777, 942817669745124, 943543162245283, 997999249847235, 820557627598087, 595914041030441, 702738002324993, 491498225353097, 544886727197518, 732944065819547|
|practice_0046|independent|48|0|287|402|0|287|1.17|106857327016504, 120107009373960, 213727296656642, 267674849940558, 269150504629490, 284901015339268, 286005960631776, 294562493860331, 335327740827091, 344627056149928, 361676363214944, 407553114321277, 435828369024637, 436185658550699, 444619663795306, 448204359651896, 464010369447126, 491498225353097, 544886727197518, 595914041030441, 628835009200270, 631602058754337, 702738002324993, 732944065819547, 751356426860707, 820557627598087, 873237992325777, 942817669745124, 943543162245283, 997999249847235|
|practice_0046|independent|48|1|273|279|0|273|5.10|120107009373960, 213727296656642, 106857327016504, 628835009200270, 943543162245283, 595914041030441, 942817669745124, 294562493860331, 335327740827091, 407553114321277, 267674849940558, 286005960631776, 448204359651896, 269150504629490, 464010369447126, 436185658550699, 544886727197518, 491498225353097, 444619663795306, 361676363214944, 997999249847235, 284901015339268, 820557627598087, 344627056149928, 435828369024637, 702738002324993, 732944065819547, 751356426860707, 631602058754337, 873237992325777|
|practice_0046|independent|48|2|281|295|0|281|31.71|267674849940558, 269150504629490, 286005960631776, 213727296656642, 284901015339268, 344627056149928, 435828369024637, 407553114321277, 444619663795306, 120107009373960, 106857327016504, 544886727197518, 335327740827091, 631602058754337, 464010369447126, 361676363214944, 942817669745124, 595914041030441, 943543162245283, 820557627598087, 628835009200270, 732944065819547, 702738002324993, 491498225353097, 448204359651896, 873237992325777, 997999249847235, 436185658550699, 751356426860707, 294562493860331|
|practice_0046|independent|128|0|246|258|24|270|8.01|106857327016504, 120107009373960, 213727296656642, 267674849940558, 269150504629490, 284901015339268, 286005960631776, 294562493860331, 335327740827091, 344627056149928, 361676363214944, 407553114321277, 435828369024637, 436185658550699, 444619663795306, 448204359651896, 464010369447126, 491498225353097, 544886727197518, 595914041030441, 628835009200270, 631602058754337, 702738002324993, 732944065819547, 751356426860707, 820557627598087, 873237992325777, 942817669745124, 943543162245283, 997999249847235|
|practice_0046|independent|128|1|243|243|0|243|29.30|120107009373960, 213727296656642, 106857327016504, 267674849940558, 269150504629490, 286005960631776, 284901015339268, 631602058754337, 335327740827091, 361676363214944, 344627056149928, 435828369024637, 294562493860331, 436185658550699, 943543162245283, 407553114321277, 464010369447126, 444619663795306, 491498225353097, 448204359651896, 595914041030441, 702738002324993, 732944065819547, 820557627598087, 942817669745124, 751356426860707, 997999249847235, 628835009200270, 873237992325777, 544886727197518|
|practice_0046|independent|128|2|239|241|5|244|172.76|267674849940558, 286005960631776, 444619663795306, 106857327016504, 120107009373960, 213727296656642, 284901015339268, 269150504629490, 294562493860331, 436185658550699, 344627056149928, 407553114321277, 491498225353097, 595914041030441, 435828369024637, 464010369447126, 702738002324993, 628835009200270, 732944065819547, 544886727197518, 448204359651896, 751356426860707, 942817669745124, 997999249847235, 631602058754337, 335327740827091, 361676363214944, 820557627598087, 873237992325777, 943543162245283|
|practice_0047|tiny|0|0|5|0|0|5|0.01|1, 5, 7|
|practice_0047|tiny|0|1|5|0|0|5|0.01|1, 5, 7|
|practice_0047|tiny|0|2|5|0|0|5|0.01|1, 5, 7|
|practice_0047|tiny|48|0|5|6|0|5|0.01|1, 5, 7|
|practice_0047|tiny|48|1|5|6|0|5|0.01|1, 5, 7|
|practice_0047|tiny|48|2|5|6|0|5|0.01|1, 5, 7|
|practice_0047|tiny|128|0|5|6|0|5|0.01|1, 5, 7|
|practice_0047|tiny|128|1|5|6|0|5|0.01|1, 5, 7|
|practice_0047|tiny|128|2|5|6|0|5|0.01|1, 5, 7|
|practice_0048|multiplicative|0|0|124|0|0|124|2.72|6361, 7228, 7751, 13010, 14550, 16177, 16612, 25996, 86060, 97833, 199232, 252050, 898563, 994479, 1303680, 1396857, 1993912, 2729829, 3003826, 3702289, 4289598, 4465288, 10893170, 10954320, 11524779, 18176968, 26128370, 109050156, 272131200, 303019135, 429945808, 1321192312|
|practice_0048|multiplicative|0|1|125|0|0|125|3.17|6361, 199232, 7228, 7751, 13010, 14550, 16177, 16612, 25996, 86060, 252050, 97833, 898563, 994479, 1303680, 1396857, 1993912, 2729829, 3003826, 3702289, 10893170, 11524779, 4289598, 4465288, 10954320, 18176968, 26128370, 109050156, 272131200, 303019135, 429945808, 1321192312|
|practice_0048|multiplicative|0|2|120|0|0|120|22.21|13010, 14550, 7228, 7751, 86060, 199232, 252050, 16177, 898563, 994479, 6361, 16612, 25996, 1303680, 97833, 1396857, 11524779, 10893170, 1993912, 3702289, 4289598, 2729829, 26128370, 10954320, 18176968, 4465288, 3003826, 109050156, 272131200, 303019135, 429945808, 1321192312|
|practice_0048|multiplicative|48|0|132|169|0|132|0.95|6361, 7228, 7751, 13010, 14550, 16177, 16612, 25996, 86060, 97833, 199232, 252050, 898563, 994479, 1303680, 1396857, 1993912, 2729829, 3003826, 3702289, 4289598, 4465288, 10893170, 10954320, 11524779, 18176968, 26128370, 109050156, 272131200, 303019135, 429945808, 1321192312|
|practice_0048|multiplicative|48|1|128|134|0|128|1.51|6361, 199232, 7228, 7751, 13010, 14550, 16177, 16612, 25996, 86060, 97833, 252050, 898563, 994479, 1303680, 1396857, 2729829, 4289598, 1993912, 3702289, 3003826, 10893170, 10954320, 4465288, 109050156, 11524779, 18176968, 26128370, 272131200, 303019135, 429945808, 1321192312|
|practice_0048|multiplicative|48|2|124|128|0|124|9.23|13010, 14550, 7228, 7751, 86060, 199232, 6361, 25996, 97833, 16177, 1303680, 252050, 994479, 1396857, 898563, 2729829, 3003826, 1993912, 3702289, 272131200, 4289598, 16612, 4465288, 11524779, 18176968, 109050156, 10954320, 10893170, 1321192312, 303019135, 26128370, 429945808|
|practice_0048|multiplicative|128|0|123|125|0|123|4.40|6361, 7228, 7751, 13010, 14550, 16177, 16612, 25996, 86060, 97833, 199232, 252050, 898563, 994479, 1303680, 1396857, 1993912, 2729829, 3003826, 3702289, 4289598, 4465288, 10893170, 10954320, 11524779, 18176968, 26128370, 109050156, 272131200, 303019135, 429945808, 1321192312|
|practice_0048|multiplicative|128|1|123|125|0|123|5.62|6361, 199232, 7228, 7751, 13010, 14550, 16177, 16612, 25996, 86060, 97833, 252050, 994479, 898563, 1303680, 1396857, 1993912, 2729829, 3003826, 3702289, 4289598, 4465288, 10954320, 10893170, 11524779, 272131200, 18176968, 26128370, 109050156, 303019135, 429945808, 1321192312|
|practice_0048|multiplicative|128|2|117|119|0|117|33.49|13010, 14550, 7228, 7751, 86060, 199232, 6361, 25996, 16612, 16177, 994479, 97833, 898563, 252050, 2729829, 4465288, 3702289, 4289598, 10893170, 1303680, 1396857, 1993912, 3003826, 10954320, 11524779, 18176968, 26128370, 272131200, 109050156, 303019135, 429945808, 1321192312|
|practice_0049|multiplicative|0|0|131|0|0|131|3.46|131023, 184836, 225632, 274463, 308375, 372885, 374882, 377674, 485370, 630790, 824537, 942650, 1110124, 1316256, 1646563, 1688559, 5854336, 6037472, 8230332, 9166780, 9681580, 20121552, 48984319, 88119024, 180348374, 205642465, 228828265, 361557404, 1071349880, 1277451675, 166056836405|
|practice_0049|multiplicative|0|1|128|0|0|128|5.10|131023, 225632, 274463, 1110124, 184836, 308375, 372885, 374882, 377674, 485370, 630790, 824537, 1316256, 942650, 1646563, 5854336, 6037472, 9166780, 1688559, 8230332, 9681580, 20121552, 88119024, 48984319, 180348374, 205642465, 228828265, 361557404, 1071349880, 1277451675, 166056836405|
|practice_0049|multiplicative|0|2|126|0|0|126|31.89|308375, 131023, 274463, 5854336, 225632, 372885, 374882, 1316256, 184836, 1110124, 377674, 630790, 1688559, 824537, 485370, 942650, 6037472, 8230332, 9166780, 1646563, 9681580, 205642465, 20121552, 88119024, 180348374, 361557404, 48984319, 228828265, 1071349880, 1277451675, 166056836405|
|practice_0049|multiplicative|48|0|140|150|0|140|1.25|131023, 184836, 225632, 274463, 308375, 372885, 374882, 377674, 485370, 630790, 824537, 942650, 1110124, 1316256, 1646563, 1688559, 5854336, 6037472, 8230332, 9166780, 9681580, 20121552, 48984319, 88119024, 180348374, 205642465, 228828265, 361557404, 1071349880, 1277451675, 166056836405|
|practice_0049|multiplicative|48|1|135|143|0|135|2.14|131023, 225632, 274463, 1110124, 184836, 308375, 372885, 374882, 377674, 485370, 630790, 824537, 1316256, 942650, 5854336, 1646563, 1688559, 6037472, 8230332, 9166780, 9681580, 20121552, 48984319, 88119024, 205642465, 180348374, 228828265, 361557404, 1277451675, 1071349880, 166056836405|
|practice_0049|multiplicative|48|2|134|140|0|134|11.24|308375, 131023, 274463, 5854336, 225632, 374882, 1316256, 372885, 942650, 485370, 377674, 6037472, 1110124, 184836, 9681580, 1688559, 630790, 8230332, 20121552, 1646563, 824537, 180348374, 9166780, 166056836405, 48984319, 1277451675, 228828265, 88119024, 1071349880, 205642465, 361557404|
|practice_0049|multiplicative|128|0|128|130|1|129|4.82|131023, 184836, 225632, 274463, 308375, 372885, 374882, 377674, 485370, 630790, 824537, 942650, 1110124, 1316256, 1646563, 1688559, 5854336, 6037472, 8230332, 9166780, 9681580, 20121552, 48984319, 88119024, 180348374, 205642465, 228828265, 361557404, 1071349880, 1277451675, 166056836405|
|practice_0049|multiplicative|128|1|128|130|0|128|7.08|131023, 225632, 274463, 1110124, 184836, 308375, 372885, 374882, 377674, 485370, 630790, 824537, 1316256, 942650, 1646563, 5854336, 6037472, 1688559, 8230332, 9166780, 9681580, 20121552, 48984319, 88119024, 180348374, 205642465, 228828265, 361557404, 1071349880, 1277451675, 166056836405|
|practice_0049|multiplicative|128|2|125|127|1|126|35.78|308375, 131023, 274463, 5854336, 225632, 374882, 184836, 372885, 377674, 485370, 1110124, 1316256, 1688559, 8230332, 630790, 824537, 20121552, 942650, 1646563, 6037472, 9681580, 88119024, 180348374, 228828265, 9166780, 48984319, 205642465, 361557404, 1071349880, 1277451675, 166056836405|
|practice_0050|tiny|0|0|5|0|0|5|0.02|8, 14|
|practice_0050|tiny|0|1|5|0|0|5|0.01|8, 14|
|practice_0050|tiny|0|2|5|0|0|5|0.02|8, 14|
|practice_0050|tiny|48|0|5|10|0|5|0.01|8, 14|
|practice_0050|tiny|48|1|5|10|0|5|0.01|8, 14|
|practice_0050|tiny|48|2|5|10|0|5|0.01|8, 14|
|practice_0050|tiny|128|0|5|10|0|5|0.01|8, 14|
|practice_0050|tiny|128|1|5|10|0|5|0.02|8, 14|
|practice_0050|tiny|128|2|5|10|0|5|0.02|8, 14|
|practice_0051|tiny|0|0|7|0|0|7|0.01|2, 5, 7, 9|
|practice_0051|tiny|0|1|7|0|0|7|0.03|2, 5, 7, 9|
|practice_0051|tiny|0|2|7|0|0|7|0.02|2, 5, 7, 9|
|practice_0051|tiny|48|0|7|8|0|7|0.03|2, 5, 7, 9|
|practice_0051|tiny|48|1|7|8|0|7|0.01|2, 5, 7, 9|
|practice_0051|tiny|48|2|7|8|0|7|0.01|2, 5, 7, 9|
|practice_0051|tiny|128|0|7|8|0|7|0.01|2, 5, 7, 9|
|practice_0051|tiny|128|1|7|8|0|7|0.01|2, 5, 7, 9|
|practice_0051|tiny|128|2|7|8|0|7|0.01|2, 5, 7, 9|
|practice_0052|independent|0|0|214|0|0|214|3.00|140861124363, 147058888734, 181840266019, 196674421570, 208613163828, 237388457079, 273788851657, 281821332687, 284353400263, 284579042977, 332825364752, 401498001124, 512789332658, 527355910886, 529217634376, 599695884455, 603929584494, 627014145699, 633502426756, 640684295191, 710995776713, 711812025121, 715186272331, 716882918803, 782776662292, 815823438758, 857437397897, 878345872006, 905978771895, 993623864284|
|practice_0052|independent|0|1|201|0|33|234|7.81|140861124363, 147058888734, 208613163828, 196674421570, 181840266019, 281821332687, 716882918803, 237388457079, 284353400263, 284579042977, 273788851657, 815823438758, 710995776713, 711812025121, 332825364752, 401498001124, 527355910886, 529217634376, 627014145699, 603929584494, 599695884455, 633502426756, 512789332658, 640684295191, 715186272331, 782776662292, 993623864284, 857437397897, 878345872006, 905978771895|
|practice_0052|independent|0|2|192|0|9|201|47.74|147058888734, 237388457079, 181840266019, 281821332687, 284353400263, 603929584494, 284579042977, 273788851657, 711812025121, 208613163828, 710995776713, 196674421570, 633502426756, 512789332658, 599695884455, 401498001124, 140861124363, 815823438758, 878345872006, 640684295191, 905978771895, 993623864284, 527355910886, 332825364752, 529217634376, 782776662292, 627014145699, 716882918803, 857437397897, 715186272331|
|practice_0052|independent|48|0|209|239|0|209|1.06|140861124363, 147058888734, 181840266019, 196674421570, 208613163828, 237388457079, 273788851657, 281821332687, 284353400263, 284579042977, 332825364752, 401498001124, 512789332658, 527355910886, 529217634376, 599695884455, 603929584494, 627014145699, 633502426756, 640684295191, 710995776713, 711812025121, 715186272331, 716882918803, 782776662292, 815823438758, 857437397897, 878345872006, 905978771895, 993623864284|
|practice_0052|independent|48|1|207|216|0|207|3.59|140861124363, 147058888734, 208613163828, 196674421570, 281821332687, 284353400263, 284579042977, 273788851657, 401498001124, 627014145699, 640684295191, 237388457079, 181840266019, 715186272331, 512789332658, 527355910886, 711812025121, 716882918803, 332825364752, 710995776713, 905978771895, 529217634376, 599695884455, 878345872006, 633502426756, 782776662292, 815823438758, 857437397897, 603929584494, 993623864284|
|practice_0052|independent|48|2|213|226|0|213|26.35|147058888734, 237388457079, 401498001124, 332825364752, 782776662292, 196674421570, 181840266019, 284579042977, 208613163828, 512789332658, 603929584494, 527355910886, 273788851657, 284353400263, 640684295191, 140861124363, 715186272331, 716882918803, 993623864284, 281821332687, 711812025121, 599695884455, 529217634376, 878345872006, 815823438758, 905978771895, 857437397897, 633502426756, 627014145699, 710995776713|
|practice_0052|independent|128|0|198|214|32|230|6.52|140861124363, 147058888734, 181840266019, 196674421570, 208613163828, 237388457079, 273788851657, 281821332687, 284353400263, 284579042977, 332825364752, 401498001124, 512789332658, 527355910886, 529217634376, 599695884455, 603929584494, 627014145699, 633502426756, 640684295191, 710995776713, 711812025121, 715186272331, 716882918803, 782776662292, 815823438758, 857437397897, 878345872006, 905978771895, 993623864284|
|practice_0052|independent|128|1|189|193|9|198|19.80|140861124363, 147058888734, 208613163828, 196674421570, 181840266019, 281821332687, 716882918803, 237388457079, 284353400263, 284579042977, 715186272331, 273788851657, 332825364752, 710995776713, 599695884455, 603929584494, 711812025121, 993623864284, 401498001124, 527355910886, 529217634376, 512789332658, 627014145699, 815823438758, 782776662292, 633502426756, 640684295191, 857437397897, 878345872006, 905978771895|
|practice_0052|independent|128|2|186|188|17|203|132.05|147058888734, 237388457079, 181840266019, 281821332687, 284353400263, 208613163828, 711812025121, 993623864284, 284579042977, 603929584494, 599695884455, 273788851657, 332825364752, 140861124363, 782776662292, 857437397897, 529217634376, 710995776713, 640684295191, 527355910886, 512789332658, 627014145699, 633502426756, 196674421570, 878345872006, 715186272331, 401498001124, 716882918803, 815823438758, 905978771895|
|practice_0053|mixed|0|0|267|0|0|267|4.05|312309848579, 312309848855, 461187917056, 1059303922404, 3087070771848, 17356412143731, 24717091437428, 25630211933255, 81342246782628, 126231782192535, 488639157121920, 1701322226303637, 5016562983759040, 32607659459648000, 185906370046916411, 187618779130645501, 189584153629043862, 309242627709328530, 413626518333033642, 451660972543495975, 534822925696928622, 547643653274313896, 582308136961287848, 661962919002967074, 739315520667305675, 780100029175313347, 822513312036351031, 940621655055050255|
|practice_0053|mixed|0|1|255|0|0|255|14.71|461187917056, 1701322226303637, 312309848579, 312309848855, 3087070771848, 1059303922404, 24717091437428, 5016562983759040, 32607659459648000, 17356412143731, 25630211933255, 81342246782628, 126231782192535, 488639157121920, 309242627709328530, 582308136961287848, 185906370046916411, 187618779130645501, 189584153629043862, 413626518333033642, 547643653274313896, 451660972543495975, 534822925696928622, 661962919002967074, 739315520667305675, 780100029175313347, 940621655055050255, 822513312036351031|
|practice_0053|mixed|0|2|245|0|5|250|97.68|461187917056, 312309848579, 17356412143731, 312309848855, 1701322226303637, 1059303922404, 32607659459648000, 24717091437428, 25630211933255, 488639157121920, 126231782192535, 413626518333033642, 185906370046916411, 81342246782628, 3087070771848, 5016562983759040, 822513312036351031, 451660972543495975, 187618779130645501, 309242627709328530, 547643653274313896, 189584153629043862, 661962919002967074, 780100029175313347, 582308136961287848, 534822925696928622, 739315520667305675, 940621655055050255|
|practice_0053|mixed|48|0|272|319|0|272|1.94|312309848579, 312309848855, 461187917056, 1059303922404, 3087070771848, 17356412143731, 24717091437428, 25630211933255, 81342246782628, 126231782192535, 488639157121920, 1701322226303637, 5016562983759040, 32607659459648000, 185906370046916411, 187618779130645501, 189584153629043862, 309242627709328530, 413626518333033642, 451660972543495975, 534822925696928622, 547643653274313896, 582308136961287848, 661962919002967074, 739315520667305675, 780100029175313347, 822513312036351031, 940621655055050255|
|practice_0053|mixed|48|1|264|273|0|264|5.97|461187917056, 1701322226303637, 312309848579, 312309848855, 3087070771848, 5016562983759040, 17356412143731, 1059303922404, 32607659459648000, 25630211933255, 126231782192535, 488639157121920, 185906370046916411, 189584153629043862, 24717091437428, 81342246782628, 187618779130645501, 309242627709328530, 413626518333033642, 661962919002967074, 547643653274313896, 451660972543495975, 582308136961287848, 940621655055050255, 534822925696928622, 780100029175313347, 822513312036351031, 739315520667305675|
|practice_0053|mixed|48|2|263|305|0|263|34.57|461187917056, 312309848579, 312309848855, 1059303922404, 3087070771848, 17356412143731, 25630211933255, 547643653274313896, 1701322226303637, 81342246782628, 24717091437428, 5016562983759040, 32607659459648000, 185906370046916411, 582308136961287848, 780100029175313347, 126231782192535, 309242627709328530, 189584153629043862, 413626518333033642, 739315520667305675, 822513312036351031, 451660972543495975, 534822925696928622, 187618779130645501, 661962919002967074, 940621655055050255, 488639157121920|
|practice_0053|mixed|128|0|236|243|28|264|6.51|312309848579, 312309848855, 461187917056, 1059303922404, 3087070771848, 17356412143731, 24717091437428, 25630211933255, 81342246782628, 126231782192535, 488639157121920, 1701322226303637, 5016562983759040, 32607659459648000, 185906370046916411, 187618779130645501, 189584153629043862, 309242627709328530, 413626518333033642, 451660972543495975, 534822925696928622, 547643653274313896, 582308136961287848, 661962919002967074, 739315520667305675, 780100029175313347, 822513312036351031, 940621655055050255|
|practice_0053|mixed|128|1|232|236|3|235|26.73|461187917056, 1701322226303637, 312309848579, 312309848855, 3087070771848, 1059303922404, 24717091437428, 5016562983759040, 17356412143731, 32607659459648000, 25630211933255, 81342246782628, 488639157121920, 126231782192535, 185906370046916411, 187618779130645501, 309242627709328530, 413626518333033642, 739315520667305675, 582308136961287848, 189584153629043862, 451660972543495975, 534822925696928622, 547643653274313896, 780100029175313347, 661962919002967074, 822513312036351031, 940621655055050255|
|practice_0053|mixed|128|2|229|231|8|237|197.13|461187917056, 312309848579, 1059303922404, 312309848855, 17356412143731, 1701322226303637, 32607659459648000, 5016562983759040, 3087070771848, 488639157121920, 126231782192535, 25630211933255, 24717091437428, 81342246782628, 547643653274313896, 413626518333033642, 309242627709328530, 185906370046916411, 534822925696928622, 187618779130645501, 189584153629043862, 451660972543495975, 582308136961287848, 739315520667305675, 661962919002967074, 822513312036351031, 780100029175313347, 940621655055050255|
|practice_0054|independent|0|0|346|0|0|346|3.60|124411195495057076, 153520104060745523, 178842892673905490, 233285010469881014, 247403498650964427, 257978360939676536, 347929932959117668, 348293139463725273, 354267227154574639, 361262508925258038, 361571024067733918, 376956090201403825, 386380063147417267, 405769558406443173, 406312097839273758, 421078126317102856, 517172780177072708, 527613474283656403, 592891385548411092, 632739101074266214, 657030743401201865, 666368642704987422, 696927766891950677, 745893417771910205, 749303890846837867, 761394699024647193, 763965349153679446, 797222581966465314, 854080030213762869, 912108243438820683, 916352731084016013, 925036553511601437|
|practice_0054|independent|0|1|330|0|169|499|14.22|592891385548411092, 124411195495057076, 257978360939676536, 386380063147417267, 354267227154574639, 348293139463725273, 347929932959117668, 178842892673905490, 153520104060745523, 233285010469881014, 376956090201403825, 361262508925258038, 247403498650964427, 361571024067733918, 406312097839273758, 405769558406443173, 421078126317102856, 761394699024647193, 527613474283656403, 749303890846837867, 854080030213762869, 745893417771910205, 517172780177072708, 632739101074266214, 657030743401201865, 696927766891950677, 666368642704987422, 763965349153679446, 912108243438820683, 797222581966465314, 916352731084016013, 925036553511601437|
|practice_0054|independent|0|2|325|0|93|418|104.05|233285010469881014, 124411195495057076, 348293139463725273, 386380063147417267, 153520104060745523, 257978360939676536, 406312097839273758, 361262508925258038, 347929932959117668, 696927766891950677, 912108243438820683, 354267227154574639, 361571024067733918, 405769558406443173, 527613474283656403, 421078126317102856, 797222581966465314, 247403498650964427, 376956090201403825, 632739101074266214, 763965349153679446, 592891385548411092, 178842892673905490, 517172780177072708, 916352731084016013, 925036553511601437, 666368642704987422, 749303890846837867, 761394699024647193, 657030743401201865, 745893417771910205, 854080030213762869|
|practice_0054|independent|48|0|353|486|0|353|1.28|124411195495057076, 153520104060745523, 178842892673905490, 233285010469881014, 247403498650964427, 257978360939676536, 347929932959117668, 348293139463725273, 354267227154574639, 361262508925258038, 361571024067733918, 376956090201403825, 386380063147417267, 405769558406443173, 406312097839273758, 421078126317102856, 517172780177072708, 527613474283656403, 592891385548411092, 632739101074266214, 657030743401201865, 666368642704987422, 696927766891950677, 745893417771910205, 749303890846837867, 761394699024647193, 763965349153679446, 797222581966465314, 854080030213762869, 912108243438820683, 916352731084016013, 925036553511601437|
|practice_0054|independent|48|1|368|381|0|368|6.11|592891385548411092, 124411195495057076, 257978360939676536, 178842892673905490, 376956090201403825, 348293139463725273, 347929932959117668, 233285010469881014, 247403498650964427, 657030743401201865, 406312097839273758, 405769558406443173, 361262508925258038, 517172780177072708, 797222581966465314, 386380063147417267, 632739101074266214, 912108243438820683, 527613474283656403, 361571024067733918, 354267227154574639, 421078126317102856, 153520104060745523, 745893417771910205, 749303890846837867, 854080030213762869, 696927766891950677, 916352731084016013, 763965349153679446, 925036553511601437, 761394699024647193, 666368642704987422|
|practice_0054|independent|48|2|355|382|0|355|43.31|233285010469881014, 178842892673905490, 386380063147417267, 405769558406443173, 406312097839273758, 925036553511601437, 916352731084016013, 124411195495057076, 153520104060745523, 348293139463725273, 257978360939676536, 666368642704987422, 421078126317102856, 745893417771910205, 347929932959117668, 763965349153679446, 247403498650964427, 376956090201403825, 854080030213762869, 632739101074266214, 696927766891950677, 354267227154574639, 749303890846837867, 517172780177072708, 361571024067733918, 361262508925258038, 527613474283656403, 592891385548411092, 912108243438820683, 797222581966465314, 761394699024647193, 657030743401201865|
|practice_0054|independent|128|0|313|336|88|401|11.02|124411195495057076, 153520104060745523, 178842892673905490, 233285010469881014, 247403498650964427, 257978360939676536, 347929932959117668, 348293139463725273, 354267227154574639, 361262508925258038, 361571024067733918, 376956090201403825, 386380063147417267, 405769558406443173, 406312097839273758, 421078126317102856, 517172780177072708, 527613474283656403, 592891385548411092, 632739101074266214, 657030743401201865, 666368642704987422, 696927766891950677, 745893417771910205, 749303890846837867, 761394699024647193, 763965349153679446, 797222581966465314, 854080030213762869, 912108243438820683, 916352731084016013, 925036553511601437|
|practice_0054|independent|128|1|302|305|1|303|49.39|592891385548411092, 124411195495057076, 257978360939676536, 386380063147417267, 354267227154574639, 348293139463725273, 247403498650964427, 527613474283656403, 153520104060745523, 347929932959117668, 178842892673905490, 361262508925258038, 376956090201403825, 361571024067733918, 421078126317102856, 745893417771910205, 797222581966465314, 406312097839273758, 405769558406443173, 761394699024647193, 912108243438820683, 233285010469881014, 517172780177072708, 632739101074266214, 666368642704987422, 657030743401201865, 749303890846837867, 854080030213762869, 696927766891950677, 925036553511601437, 763965349153679446, 916352731084016013|
|practice_0054|independent|128|2|301|304|0|301|334.88|233285010469881014, 124411195495057076, 348293139463725273, 386380063147417267, 153520104060745523, 178842892673905490, 257978360939676536, 361262508925258038, 247403498650964427, 666368642704987422, 406312097839273758, 517172780177072708, 376956090201403825, 763965349153679446, 592891385548411092, 657030743401201865, 797222581966465314, 405769558406443173, 347929932959117668, 421078126317102856, 632739101074266214, 854080030213762869, 361571024067733918, 354267227154574639, 749303890846837867, 912108243438820683, 696927766891950677, 745893417771910205, 761394699024647193, 527613474283656403, 925036553511601437, 916352731084016013|
|practice_0055|independent|0|0|349|0|0|349|3.83|141776709497009448, 228937799460780901, 260012953831827660, 276371127369926414, 289820220517384280, 347588437546602445, 360593111186435464, 437353306960882731, 447943657482421715, 456500732363913214, 493967327458518250, 528681862507869629, 536377420058008717, 556580680015155818, 568238089681423194, 579010222098344056, 583904775909477515, 591841222573573082, 603179333030212653, 605636606439214487, 614854513131589828, 615470708188466964, 620994761051684957, 626266776167678095, 674935494893795113, 682221820665348905, 771644608462655034, 779086712456606426, 816232436682999090, 902564818205462992, 907294403490498993, 948863308767208178|
|practice_0055|independent|0|1|324|0|567|891|16.23|289820220517384280, 456500732363913214, 276371127369926414, 568238089681423194, 591841222573573082, 141776709497009448, 528681862507869629, 447943657482421715, 437353306960882731, 583904775909477515, 682221820665348905, 579010222098344056, 536377420058008717, 614854513131589828, 360593111186435464, 347588437546602445, 816232436682999090, 260012953831827660, 779086712456606426, 620994761051684957, 615470708188466964, 228937799460780901, 771644608462655034, 605636606439214487, 493967327458518250, 603179333030212653, 674935494893795113, 902564818205462992, 907294403490498993, 556580680015155818, 948863308767208178, 626266776167678095|
|practice_0055|independent|0|2|328|0|289|617|105.25|260012953831827660, 779086712456606426, 289820220517384280, 276371127369926414, 556580680015155818, 493967327458518250, 536377420058008717, 437353306960882731, 579010222098344056, 141776709497009448, 615470708188466964, 614854513131589828, 907294403490498993, 360593111186435464, 447943657482421715, 626266776167678095, 591841222573573082, 528681862507869629, 771644608462655034, 605636606439214487, 456500732363913214, 674935494893795113, 816232436682999090, 583904775909477515, 347588437546602445, 902564818205462992, 568238089681423194, 682221820665348905, 948863308767208178, 228937799460780901, 603179333030212653, 620994761051684957|
|practice_0055|independent|48|0|352|486|0|352|1.84|141776709497009448, 228937799460780901, 260012953831827660, 276371127369926414, 289820220517384280, 347588437546602445, 360593111186435464, 437353306960882731, 447943657482421715, 456500732363913214, 493967327458518250, 528681862507869629, 536377420058008717, 556580680015155818, 568238089681423194, 579010222098344056, 583904775909477515, 591841222573573082, 603179333030212653, 605636606439214487, 614854513131589828, 615470708188466964, 620994761051684957, 626266776167678095, 674935494893795113, 682221820665348905, 771644608462655034, 779086712456606426, 816232436682999090, 902564818205462992, 907294403490498993, 948863308767208178|
|practice_0055|independent|48|1|357|368|0|357|6.15|289820220517384280, 456500732363913214, 579010222098344056, 141776709497009448, 276371127369926414, 437353306960882731, 447943657482421715, 260012953831827660, 347588437546602445, 626266776167678095, 493967327458518250, 603179333030212653, 605636606439214487, 556580680015155818, 591841222573573082, 568238089681423194, 948863308767208178, 228937799460780901, 583904775909477515, 536377420058008717, 528681862507869629, 615470708188466964, 360593111186435464, 620994761051684957, 816232436682999090, 902564818205462992, 771644608462655034, 682221820665348905, 779086712456606426, 614854513131589828, 674935494893795113, 907294403490498993|
|practice_0055|independent|48|2|348|374|0|348|44.19|360593111186435464, 260012953831827660, 771644608462655034, 141776709497009448, 347588437546602445, 289820220517384280, 528681862507869629, 591841222573573082, 614854513131589828, 456500732363913214, 948863308767208178, 779086712456606426, 626266776167678095, 493967327458518250, 536377420058008717, 556580680015155818, 603179333030212653, 674935494893795113, 568238089681423194, 447943657482421715, 907294403490498993, 579010222098344056, 605636606439214487, 682221820665348905, 620994761051684957, 583904775909477515, 615470708188466964, 816232436682999090, 228937799460780901, 902564818205462992, 437353306960882731, 276371127369926414|
|practice_0055|independent|128|0|307|319|43|350|10.94|141776709497009448, 228937799460780901, 260012953831827660, 276371127369926414, 289820220517384280, 347588437546602445, 360593111186435464, 437353306960882731, 447943657482421715, 456500732363913214, 493967327458518250, 528681862507869629, 536377420058008717, 556580680015155818, 568238089681423194, 579010222098344056, 583904775909477515, 591841222573573082, 603179333030212653, 605636606439214487, 614854513131589828, 615470708188466964, 620994761051684957, 626266776167678095, 674935494893795113, 682221820665348905, 771644608462655034, 779086712456606426, 816232436682999090, 902564818205462992, 907294403490498993, 948863308767208178|
|practice_0055|independent|128|1|299|305|16|315|49.65|289820220517384280, 456500732363913214, 276371127369926414, 568238089681423194, 591841222573573082, 141776709497009448, 779086712456606426, 228937799460780901, 260012953831827660, 536377420058008717, 579010222098344056, 603179333030212653, 605636606439214487, 347588437546602445, 437353306960882731, 583904775909477515, 447943657482421715, 528681862507869629, 615470708188466964, 360593111186435464, 493967327458518250, 556580680015155818, 614854513131589828, 620994761051684957, 816232436682999090, 626266776167678095, 674935494893795113, 682221820665348905, 771644608462655034, 902564818205462992, 907294403490498993, 948863308767208178|
|practice_0055|independent|128|2|301|303|34|335|343.62|360593111186435464, 568238089681423194, 289820220517384280, 228937799460780901, 591841222573573082, 528681862507869629, 902564818205462992, 907294403490498993, 437353306960882731, 141776709497009448, 536377420058008717, 626266776167678095, 674935494893795113, 447943657482421715, 579010222098344056, 276371127369926414, 583904775909477515, 614854513131589828, 615470708188466964, 620994761051684957, 493967327458518250, 556580680015155818, 605636606439214487, 682221820665348905, 779086712456606426, 456500732363913214, 260012953831827660, 603179333030212653, 816232436682999090, 771644608462655034, 347588437546602445, 948863308767208178|
|practice_0056|tiny|0|0|7|0|0|7|0.01|3, 5, 7, 10|
|practice_0056|tiny|0|1|7|0|0|7|0.02|3, 5, 7, 10|
|practice_0056|tiny|0|2|7|0|0|7|0.01|3, 5, 7, 10|
|practice_0056|tiny|48|0|7|7|0|7|0.01|3, 5, 7, 10|
|practice_0056|tiny|48|1|7|7|0|7|0.01|3, 5, 7, 10|
|practice_0056|tiny|48|2|7|7|0|7|0.01|3, 5, 7, 10|
|practice_0056|tiny|128|0|7|7|0|7|0.01|3, 5, 7, 10|
|practice_0056|tiny|128|1|7|7|0|7|0.01|3, 5, 7, 10|
|practice_0056|tiny|128|2|7|7|0|7|0.01|3, 5, 7, 10|
|practice_0057|independent|0|0|256|0|0|256|3.08|104504919777964, 105379801099407, 165611582575133, 182358547509154, 186337530702355, 255397406951729, 283050455908476, 343862318099461, 349924871886378, 351088886932395, 387691393711870, 407136681167274, 411330369893617, 440196860141135, 499666468881750, 521768465808294, 537987395863962, 549214097407498, 599703308016578, 599750290225456, 614070527365412, 645628502545401, 685468551581345, 730111024224842, 821113324886310, 831603306061825, 863417377927926, 920460668468989, 971089861636539|
|practice_0057|independent|0|1|243|0|17|260|9.04|104504919777964, 105379801099407, 165611582575133, 255397406951729, 537987395863962, 182358547509154, 343862318099461, 521768465808294, 863417377927926, 283050455908476, 387691393711870, 349924871886378, 351088886932395, 407136681167274, 186337530702355, 440196860141135, 411330369893617, 599703308016578, 599750290225456, 614070527365412, 549214097407498, 499666468881750, 645628502545401, 730111024224842, 831603306061825, 685468551581345, 971089861636539, 821113324886310, 920460668468989|
|practice_0057|independent|0|2|244|0|60|304|71.56|599750290225456, 104504919777964, 105379801099407, 599703308016578, 186337530702355, 283050455908476, 351088886932395, 387691393711870, 182358547509154, 349924871886378, 920460668468989, 407136681167274, 255397406951729, 165611582575133, 440196860141135, 537987395863962, 730111024224842, 821113324886310, 411330369893617, 614070527365412, 549214097407498, 521768465808294, 645628502545401, 343862318099461, 863417377927926, 831603306061825, 685468551581345, 971089861636539, 499666468881750|
|practice_0057|independent|48|0|260|301|0|260|1.43|104504919777964, 105379801099407, 165611582575133, 182358547509154, 186337530702355, 255397406951729, 283050455908476, 343862318099461, 349924871886378, 351088886932395, 387691393711870, 407136681167274, 411330369893617, 440196860141135, 499666468881750, 521768465808294, 537987395863962, 549214097407498, 599703308016578, 599750290225456, 614070527365412, 645628502545401, 685468551581345, 730111024224842, 821113324886310, 831603306061825, 863417377927926, 920460668468989, 971089861636539|
|practice_0057|independent|48|1|257|269|0|257|3.09|104504919777964, 105379801099407, 165611582575133, 182358547509154, 255397406951729, 971089861636539, 186337530702355, 343862318099461, 283050455908476, 411330369893617, 407136681167274, 863417377927926, 645628502545401, 349924871886378, 387691393711870, 521768465808294, 351088886932395, 440196860141135, 537987395863962, 614070527365412, 549214097407498, 499666468881750, 599750290225456, 821113324886310, 920460668468989, 831603306061825, 599703308016578, 730111024224842, 685468551581345|
|practice_0057|independent|48|2|262|272|0|262|29.05|407136681167274, 685468551581345, 182358547509154, 186337530702355, 599703308016578, 165611582575133, 349924871886378, 599750290225456, 440196860141135, 645628502545401, 821113324886310, 614070527365412, 105379801099407, 104504919777964, 863417377927926, 283050455908476, 387691393711870, 521768465808294, 351088886932395, 411330369893617, 499666468881750, 730111024224842, 255397406951729, 920460668468989, 549214097407498, 537987395863962, 971089861636539, 831603306061825, 343862318099461|
|practice_0057|independent|128|0|235|240|13|248|7.58|104504919777964, 105379801099407, 165611582575133, 182358547509154, 186337530702355, 255397406951729, 283050455908476, 343862318099461, 349924871886378, 351088886932395, 387691393711870, 407136681167274, 411330369893617, 440196860141135, 499666468881750, 521768465808294, 537987395863962, 549214097407498, 599703308016578, 599750290225456, 614070527365412, 645628502545401, 685468551581345, 730111024224842, 821113324886310, 831603306061825, 863417377927926, 920460668468989, 971089861636539|
|practice_0057|independent|128|1|229|236|0|229|29.29|104504919777964, 105379801099407, 165611582575133, 255397406951729, 537987395863962, 182358547509154, 186337530702355, 343862318099461, 283050455908476, 407136681167274, 521768465808294, 863417377927926, 614070527365412, 387691393711870, 440196860141135, 349924871886378, 351088886932395, 411330369893617, 499666468881750, 821113324886310, 599703308016578, 599750290225456, 645628502545401, 831603306061825, 549214097407498, 920460668468989, 685468551581345, 730111024224842, 971089861636539|
|practice_0057|independent|128|2|225|227|29|254|163.64|104504919777964, 105379801099407, 255397406951729, 283050455908476, 863417377927926, 349924871886378, 387691393711870, 351088886932395, 599703308016578, 407136681167274, 186337530702355, 599750290225456, 549214097407498, 537987395863962, 440196860141135, 182358547509154, 821113324886310, 411330369893617, 165611582575133, 499666468881750, 730111024224842, 685468551581345, 614070527365412, 645628502545401, 920460668468989, 971089861636539, 831603306061825, 521768465808294, 343862318099461|
|practice_0058|independent|0|0|267|0|0|267|3.09|186802582793696, 230429597251946, 247234344088109, 271668741877750, 290189731028529, 350976467216784, 351217292601729, 419066656567192, 430400509231937, 468907368659499, 469072006883035, 475380981083679, 498725008259274, 509392720638813, 550414585868543, 565028939802867, 597986937572730, 608522489458393, 624416140593549, 641850016735797, 647494710891735, 663885562809018, 672063878747311, 793493863120639, 829537808529485, 892232049311132, 918823927500594, 925786502367773, 964133009734223, 985878108980081|
|practice_0058|independent|0|1|260|0|0|260|9.82|247234344088109, 186802582793696, 230429597251946, 350976467216784, 271668741877750, 351217292601729, 624416140593549, 565028939802867, 419066656567192, 430400509231937, 290189731028529, 475380981083679, 498725008259274, 468907368659499, 469072006883035, 509392720638813, 550414585868543, 597986937572730, 608522489458393, 647494710891735, 641850016735797, 672063878747311, 829537808529485, 663885562809018, 892232049311132, 925786502367773, 985878108980081, 964133009734223, 918823927500594, 793493863120639|
|practice_0058|independent|0|2|253|0|59|312|79.18|247234344088109, 350976467216784, 351217292601729, 475380981083679, 829537808529485, 186802582793696, 918823927500594, 271668741877750, 641850016735797, 290189731028529, 430400509231937, 468907368659499, 550414585868543, 565028939802867, 509392720638813, 608522489458393, 647494710891735, 663885562809018, 624416140593549, 230429597251946, 672063878747311, 793493863120639, 925786502367773, 469072006883035, 964133009734223, 892232049311132, 985878108980081, 419066656567192, 498725008259274, 597986937572730|
|practice_0058|independent|48|0|271|317|0|271|1.68|186802582793696, 230429597251946, 247234344088109, 271668741877750, 290189731028529, 350976467216784, 351217292601729, 419066656567192, 430400509231937, 468907368659499, 469072006883035, 475380981083679, 498725008259274, 509392720638813, 550414585868543, 565028939802867, 597986937572730, 608522489458393, 624416140593549, 641850016735797, 647494710891735, 663885562809018, 672063878747311, 793493863120639, 829537808529485, 892232049311132, 918823927500594, 925786502367773, 964133009734223, 985878108980081|
|practice_0058|independent|48|1|272|283|0|272|4.69|247234344088109, 186802582793696, 419066656567192, 230429597251946, 350976467216784, 271668741877750, 468907368659499, 469072006883035, 892232049311132, 498725008259274, 550414585868543, 509392720638813, 290189731028529, 351217292601729, 430400509231937, 475380981083679, 565028939802867, 608522489458393, 641850016735797, 597986937572730, 647494710891735, 663885562809018, 793493863120639, 672063878747311, 918823927500594, 925786502367773, 829537808529485, 624416140593549, 985878108980081, 964133009734223|
|practice_0058|independent|48|2|272|297|0|272|33.45|247234344088109, 271668741877750, 918823927500594, 829537808529485, 663885562809018, 647494710891735, 290189731028529, 597986937572730, 475380981083679, 351217292601729, 186802582793696, 469072006883035, 419066656567192, 925786502367773, 230429597251946, 468907368659499, 509392720638813, 430400509231937, 985878108980081, 565028939802867, 608522489458393, 550414585868543, 892232049311132, 793493863120639, 672063878747311, 964133009734223, 624416140593549, 350976467216784, 641850016735797, 498725008259274|
|practice_0058|independent|128|0|248|251|0|248|7.95|186802582793696, 230429597251946, 247234344088109, 271668741877750, 290189731028529, 350976467216784, 351217292601729, 419066656567192, 430400509231937, 468907368659499, 469072006883035, 475380981083679, 498725008259274, 509392720638813, 550414585868543, 565028939802867, 597986937572730, 608522489458393, 624416140593549, 641850016735797, 647494710891735, 663885562809018, 672063878747311, 793493863120639, 829537808529485, 892232049311132, 918823927500594, 925786502367773, 964133009734223, 985878108980081|
|practice_0058|independent|128|1|237|243|24|261|23.99|247234344088109, 186802582793696, 230429597251946, 350976467216784, 271668741877750, 351217292601729, 624416140593549, 597986937572730, 608522489458393, 419066656567192, 290189731028529, 430400509231937, 468907368659499, 469072006883035, 475380981083679, 498725008259274, 509392720638813, 672063878747311, 550414585868543, 565028939802867, 647494710891735, 663885562809018, 641850016735797, 793493863120639, 829537808529485, 964133009734223, 925786502367773, 892232049311132, 918823927500594, 985878108980081|
|practice_0058|independent|128|2|234|240|52|286|186.54|247234344088109, 350976467216784, 351217292601729, 475380981083679, 186802582793696, 271668741877750, 925786502367773, 597986937572730, 829537808529485, 509392720638813, 468907368659499, 550414585868543, 230429597251946, 469072006883035, 419066656567192, 290189731028529, 498725008259274, 892232049311132, 624416140593549, 641850016735797, 663885562809018, 793493863120639, 430400509231937, 985878108980081, 565028939802867, 608522489458393, 918823927500594, 647494710891735, 672063878747311, 964133009734223|
|practice_0059|multiplicative|0|0|151|0|0|151|3.17|150656, 180556, 186371, 223612, 291218, 849364, 931933, 1144172, 1416078, 1643551, 3344831, 3691291, 6989251, 11891502, 12545010, 25162110, 27058280, 60881184, 81422432, 100324800, 293548467, 541166171, 549605034, 1563074422, 3206301287, 6098643824, 6311933760, 9742365360, 10076935232, 19326757632, 315175124264, 2420071239005|
|practice_0059|multiplicative|0|1|146|0|0|146|5.57|150656, 180556, 186371, 223612, 931933, 11891502, 849364, 1643551, 291218, 3691291, 6989251, 1144172, 25162110, 1416078, 27058280, 60881184, 3344831, 12545010, 81422432, 100324800, 293548467, 541166171, 549605034, 19326757632, 6098643824, 1563074422, 3206301287, 6311933760, 9742365360, 10076935232, 315175124264, 2420071239005|
|practice_0059|multiplicative|0|2|136|0|0|136|30.73|186371, 180556, 150656, 25162110, 223612, 291218, 931933, 849364, 11891502, 6989251, 1144172, 60881184, 27058280, 12545010, 3344831, 19326757632, 100324800, 81422432, 1416078, 10076935232, 3691291, 1643551, 315175124264, 541166171, 293548467, 1563074422, 6098643824, 549605034, 6311933760, 3206301287, 9742365360, 2420071239005|
|practice_0059|multiplicative|48|0|149|162|0|149|1.32|150656, 180556, 186371, 223612, 291218, 849364, 931933, 1144172, 1416078, 1643551, 3344831, 3691291, 6989251, 11891502, 12545010, 25162110, 27058280, 60881184, 81422432, 100324800, 293548467, 541166171, 549605034, 1563074422, 3206301287, 6098643824, 6311933760, 9742365360, 10076935232, 19326757632, 315175124264, 2420071239005|
|practice_0059|multiplicative|48|1|152|162|0|152|2.25|150656, 180556, 186371, 931933, 25162110, 223612, 291218, 1144172, 3344831, 6989251, 293548467, 849364, 1416078, 1643551, 11891502, 3691291, 12545010, 60881184, 27058280, 100324800, 541166171, 81422432, 9742365360, 549605034, 1563074422, 6311933760, 3206301287, 6098643824, 10076935232, 19326757632, 315175124264, 2420071239005|
|practice_0059|multiplicative|48|2|140|149|0|140|14.51|186371, 25162110, 150656, 60881184, 223612, 931933, 291218, 1144172, 3344831, 180556, 849364, 6989251, 81422432, 27058280, 11891502, 3691291, 1416078, 12545010, 1643551, 100324800, 541166171, 9742365360, 549605034, 315175124264, 10076935232, 293548467, 19326757632, 2420071239005, 1563074422, 6098643824, 6311933760, 3206301287|
|practice_0059|multiplicative|128|0|144|146|17|161|4.72|150656, 180556, 186371, 223612, 291218, 849364, 931933, 1144172, 1416078, 1643551, 3344831, 3691291, 6989251, 11891502, 12545010, 25162110, 27058280, 60881184, 81422432, 100324800, 293548467, 541166171, 549605034, 1563074422, 3206301287, 6098643824, 6311933760, 9742365360, 10076935232, 19326757632, 315175124264, 2420071239005|
|practice_0059|multiplicative|128|1|144|145|0|144|9.96|150656, 180556, 186371, 931933, 25162110, 223612, 291218, 1144172, 3344831, 1643551, 3691291, 6989251, 11891502, 12545010, 293548467, 849364, 1416078, 27058280, 60881184, 81422432, 100324800, 541166171, 549605034, 6098643824, 9742365360, 6311933760, 19326757632, 10076935232, 1563074422, 3206301287, 315175124264, 2420071239005|
|practice_0059|multiplicative|128|2|130|133|0|130|42.74|186371, 25162110, 150656, 60881184, 223612, 931933, 3344831, 180556, 291218, 849364, 12545010, 1144172, 1643551, 1416078, 27058280, 3691291, 6989251, 11891502, 81422432, 315175124264, 541166171, 6311933760, 100324800, 293548467, 549605034, 1563074422, 6098643824, 9742365360, 10076935232, 19326757632, 3206301287, 2420071239005|
|practice_0060|mixed|0|0|265|0|0|265|3.87|45198759931, 167866408836, 167866408993, 278902221278, 750956352954, 4028793865872, 11977718447666, 231594939527316, 258850002369986, 1034057078135618, 1073773550526600, 2753151710273802, 3106200028373520, 20562902972006400, 118474215510715779, 134462208473505492, 174880424208072915, 217674465369178999, 240702944260902265, 257142605446931746, 312204279805031047, 332567133670871853, 419815013228158147, 470628066602324411, 475434761196233878, 503861468791084298, 578568846845492848, 847111895713423393, 871428646162469575|
|practice_0060|mixed|0|1|253|0|0|253|13.39|45198759931, 167866408836, 167866408993, 4028793865872, 3106200028373520, 1034057078135618, 278902221278, 20562902972006400, 258850002369986, 1073773550526600, 750956352954, 231594939527316, 11977718447666, 2753151710273802, 240702944260902265, 134462208473505492, 118474215510715779, 312204279805031047, 174880424208072915, 847111895713423393, 217674465369178999, 332567133670871853, 470628066602324411, 475434761196233878, 257142605446931746, 419815013228158147, 871428646162469575, 578568846845492848, 503861468791084298|
|practice_0060|mixed|0|2|240|0|48|288|89.70|167866408836, 278902221278, 45198759931, 3106200028373520, 4028793865872, 20562902972006400, 167866408993, 258850002369986, 174880424208072915, 1034057078135618, 11977718447666, 750956352954, 475434761196233878, 419815013228158147, 332567133670871853, 118474215510715779, 231594939527316, 134462208473505492, 1073773550526600, 240702944260902265, 470628066602324411, 257142605446931746, 2753151710273802, 503861468791084298, 217674465369178999, 312204279805031047, 578568846845492848, 847111895713423393, 871428646162469575|
|practice_0060|mixed|48|0|275|397|0|275|1.75|45198759931, 167866408836, 167866408993, 278902221278, 750956352954, 4028793865872, 11977718447666, 231594939527316, 258850002369986, 1034057078135618, 1073773550526600, 2753151710273802, 3106200028373520, 20562902972006400, 118474215510715779, 134462208473505492, 174880424208072915, 217674465369178999, 240702944260902265, 257142605446931746, 312204279805031047, 332567133670871853, 419815013228158147, 470628066602324411, 475434761196233878, 503861468791084298, 578568846845492848, 847111895713423393, 871428646162469575|
|practice_0060|mixed|48|1|263|301|0|263|5.80|45198759931, 167866408836, 167866408993, 4028793865872, 258850002369986, 3106200028373520, 1034057078135618, 1073773550526600, 278902221278, 20562902972006400, 11977718447666, 2753151710273802, 750956352954, 231594939527316, 240702944260902265, 118474215510715779, 503861468791084298, 134462208473505492, 475434761196233878, 332567133670871853, 578568846845492848, 174880424208072915, 257142605446931746, 217674465369178999, 470628066602324411, 847111895713423393, 871428646162469575, 312204279805031047, 419815013228158147|
|practice_0060|mixed|48|2|272|339|0|272|33.84|167866408836, 278902221278, 20562902972006400, 167866408993, 258850002369986, 3106200028373520, 11977718447666, 45198759931, 750956352954, 118474215510715779, 1073773550526600, 4028793865872, 847111895713423393, 174880424208072915, 240702944260902265, 134462208473505492, 231594939527316, 578568846845492848, 470628066602324411, 1034057078135618, 2753151710273802, 312204279805031047, 475434761196233878, 419815013228158147, 503861468791084298, 217674465369178999, 257142605446931746, 332567133670871853, 871428646162469575|
|practice_0060|mixed|128|0|246|251|75|321|6.87|45198759931, 167866408836, 167866408993, 278902221278, 750956352954, 4028793865872, 11977718447666, 231594939527316, 258850002369986, 1034057078135618, 1073773550526600, 2753151710273802, 3106200028373520, 20562902972006400, 118474215510715779, 134462208473505492, 174880424208072915, 217674465369178999, 240702944260902265, 257142605446931746, 312204279805031047, 332567133670871853, 419815013228158147, 470628066602324411, 475434761196233878, 503861468791084298, 578568846845492848, 847111895713423393, 871428646162469575|
|practice_0060|mixed|128|1|233|237|25|258|29.31|45198759931, 167866408836, 167866408993, 4028793865872, 3106200028373520, 258850002369986, 1034057078135618, 278902221278, 20562902972006400, 750956352954, 1073773550526600, 11977718447666, 231594939527316, 2753151710273802, 134462208473505492, 332567133670871853, 118474215510715779, 217674465369178999, 257142605446931746, 174880424208072915, 475434761196233878, 240702944260902265, 503861468791084298, 419815013228158147, 312204279805031047, 470628066602324411, 578568846845492848, 847111895713423393, 871428646162469575|
|practice_0060|mixed|128|2|228|232|18|246|193.23|167866408836, 278902221278, 45198759931, 258850002369986, 167866408993, 1034057078135618, 4028793865872, 3106200028373520, 20562902972006400, 1073773550526600, 11977718447666, 750956352954, 231594939527316, 217674465369178999, 2753151710273802, 174880424208072915, 257142605446931746, 578568846845492848, 240702944260902265, 419815013228158147, 134462208473505492, 332567133670871853, 475434761196233878, 871428646162469575, 847111895713423393, 470628066602324411, 118474215510715779, 503861468791084298, 312204279805031047|
|practice_0061|mixed|0|0|282|0|0|282|3.46|74341841380, 201293120174, 545035210290, 675511736860, 1929506140113, 5450352101700, 27442897057732, 185765727587275, 195267769582591, 515133965751554, 3016159904602035, 6721227594396122, 93207716959663424, 141156283531619038, 215715879278490148, 225550517107335582, 248271646464438670, 285421025813001864, 350083515163369866, 518428109375246994, 550174762406021003, 631896694021090995, 775959159771246988, 822935457371949841, 848323218193734737, 891682974382863072, 897865660889779270, 965641193700351464, 976340261031650713|
|practice_0061|mixed|0|1|265|0|3|268|15.48|545035210290, 5450352101700, 74341841380, 201293120174, 1929506140113, 675511736860, 185765727587275, 141156283531619038, 3016159904602035, 27442897057732, 195267769582591, 518428109375246994, 515133965751554, 6721227594396122, 93207716959663424, 248271646464438670, 225550517107335582, 215715879278490148, 285421025813001864, 350083515163369866, 965641193700351464, 822935457371949841, 891682974382863072, 631896694021090995, 775959159771246988, 976340261031650713, 550174762406021003, 848323218193734737, 897865660889779270|
|practice_0061|mixed|0|2|266|0|80|346|100.31|545035210290, 675511736860, 27442897057732, 185765727587275, 3016159904602035, 5450352101700, 201293120174, 515133965751554, 74341841380, 1929506140113, 215715879278490148, 195267769582591, 6721227594396122, 518428109375246994, 822935457371949841, 141156283531619038, 285421025813001864, 93207716959663424, 965641193700351464, 976340261031650713, 897865660889779270, 225550517107335582, 848323218193734737, 350083515163369866, 775959159771246988, 891682974382863072, 248271646464438670, 550174762406021003, 631896694021090995|
|practice_0061|mixed|48|0|281|432|0|281|1.75|74341841380, 201293120174, 545035210290, 675511736860, 1929506140113, 5450352101700, 27442897057732, 185765727587275, 195267769582591, 515133965751554, 3016159904602035, 6721227594396122, 93207716959663424, 141156283531619038, 215715879278490148, 225550517107335582, 248271646464438670, 285421025813001864, 350083515163369866, 518428109375246994, 550174762406021003, 631896694021090995, 775959159771246988, 822935457371949841, 848323218193734737, 891682974382863072, 897865660889779270, 965641193700351464, 976340261031650713|
|practice_0061|mixed|48|1|277|416|0|277|5.32|545035210290, 5450352101700, 74341841380, 201293120174, 675511736860, 185765727587275, 1929506140113, 27442897057732, 515133965751554, 195267769582591, 141156283531619038, 3016159904602035, 93207716959663424, 225550517107335582, 897865660889779270, 891682974382863072, 248271646464438670, 550174762406021003, 285421025813001864, 350083515163369866, 6721227594396122, 631896694021090995, 775959159771246988, 215715879278490148, 822935457371949841, 848323218193734737, 518428109375246994, 965641193700351464, 976340261031650713|
|practice_0061|mixed|48|2|292|313|0|292|36.46|545035210290, 675511736860, 185765727587275, 201293120174, 27442897057732, 3016159904602035, 285421025813001864, 1929506140113, 74341841380, 5450352101700, 93207716959663424, 897865660889779270, 195267769582591, 775959159771246988, 225550517107335582, 350083515163369866, 518428109375246994, 848323218193734737, 631896694021090995, 141156283531619038, 550174762406021003, 248271646464438670, 965641193700351464, 891682974382863072, 976340261031650713, 215715879278490148, 515133965751554, 6721227594396122, 822935457371949841|
|practice_0061|mixed|128|0|253|270|43|296|6.45|74341841380, 201293120174, 545035210290, 675511736860, 1929506140113, 5450352101700, 27442897057732, 185765727587275, 195267769582591, 515133965751554, 3016159904602035, 6721227594396122, 93207716959663424, 141156283531619038, 215715879278490148, 225550517107335582, 248271646464438670, 285421025813001864, 350083515163369866, 518428109375246994, 550174762406021003, 631896694021090995, 775959159771246988, 822935457371949841, 848323218193734737, 891682974382863072, 897865660889779270, 965641193700351464, 976340261031650713|
|practice_0061|mixed|128|1|246|248|1|247|29.55|545035210290, 5450352101700, 74341841380, 201293120174, 1929506140113, 675511736860, 185765727587275, 27442897057732, 3016159904602035, 141156283531619038, 195267769582591, 515133965751554, 6721227594396122, 93207716959663424, 285421025813001864, 518428109375246994, 891682974382863072, 225550517107335582, 550174762406021003, 631896694021090995, 897865660889779270, 775959159771246988, 248271646464438670, 822935457371949841, 215715879278490148, 848323218193734737, 965641193700351464, 350083515163369866, 976340261031650713|
|practice_0061|mixed|128|2|243|246|22|265|222.41|545035210290, 675511736860, 27442897057732, 185765727587275, 3016159904602035, 5450352101700, 1929506140113, 201293120174, 195267769582591, 74341841380, 515133965751554, 141156283531619038, 248271646464438670, 93207716959663424, 225550517107335582, 965641193700351464, 976340261031650713, 550174762406021003, 215715879278490148, 775959159771246988, 350083515163369866, 285421025813001864, 631896694021090995, 6721227594396122, 822935457371949841, 897865660889779270, 891682974382863072, 518428109375246994, 848323218193734737|
|practice_0062|independent|0|0|313|0|0|313|3.08|217361443446995809, 269379298509462913, 272968328320912336, 293860350109691496, 303847758267277208, 319968700254395169, 342404678813681359, 420994477073247576, 431203504305585335, 483406833671754707, 490504798322512607, 547855343572897979, 588533035873768395, 662258898435328422, 672276175927862908, 675799579395187684, 703967305336038065, 756660427737584906, 759749405930788048, 770383512273821745, 777242217060877998, 777828257276985925, 801152261965756910, 811750177002917517, 846939845109467856, 898823384316323885, 922937913578630290, 959828018626317934, 994177919648945667|
|practice_0062|independent|0|1|299|0|202|501|13.80|217361443446995809, 770383512273821745, 483406833671754707, 269379298509462913, 272968328320912336, 303847758267277208, 319968700254395169, 342404678813681359, 431203504305585335, 490504798322512607, 703967305336038065, 759749405930788048, 420994477073247576, 293860350109691496, 777242217060877998, 777828257276985925, 756660427737584906, 922937913578630290, 547855343572897979, 675799579395187684, 662258898435328422, 672276175927862908, 588533035873768395, 898823384316323885, 801152261965756910, 959828018626317934, 994177919648945667, 811750177002917517, 846939845109467856|
|practice_0062|independent|0|2|298|0|67|365|93.72|272968328320912336, 293860350109691496, 675799579395187684, 420994477073247576, 483406833671754707, 547855343572897979, 777242217060877998, 703967305336038065, 801152261965756910, 269379298509462913, 303847758267277208, 770383512273821745, 588533035873768395, 777828257276985925, 846939845109467856, 217361443446995809, 342404678813681359, 672276175927862908, 811750177002917517, 490504798322512607, 662258898435328422, 319968700254395169, 898823384316323885, 431203504305585335, 756660427737584906, 994177919648945667, 922937913578630290, 759749405930788048, 959828018626317934|
|practice_0062|independent|48|0|324|463|0|324|1.65|217361443446995809, 269379298509462913, 272968328320912336, 293860350109691496, 303847758267277208, 319968700254395169, 342404678813681359, 420994477073247576, 431203504305585335, 483406833671754707, 490504798322512607, 547855343572897979, 588533035873768395, 662258898435328422, 672276175927862908, 675799579395187684, 703967305336038065, 756660427737584906, 759749405930788048, 770383512273821745, 777242217060877998, 777828257276985925, 801152261965756910, 811750177002917517, 846939845109467856, 898823384316323885, 922937913578630290, 959828018626317934, 994177919648945667|
|practice_0062|independent|48|1|334|351|0|334|5.43|217361443446995809, 770383512273821745, 759749405930788048, 811750177002917517, 342404678813681359, 293860350109691496, 272968328320912336, 269379298509462913, 303847758267277208, 846939845109467856, 490504798322512607, 319968700254395169, 801152261965756910, 777828257276985925, 777242217060877998, 898823384316323885, 431203504305585335, 756660427737584906, 420994477073247576, 662258898435328422, 675799579395187684, 922937913578630290, 547855343572897979, 588533035873768395, 483406833671754707, 672276175927862908, 959828018626317934, 994177919648945667, 703967305336038065|
|practice_0062|independent|48|2|326|345|0|326|37.09|272968328320912336, 420994477073247576, 342404678813681359, 777242217060877998, 547855343572897979, 801152261965756910, 777828257276985925, 293860350109691496, 811750177002917517, 675799579395187684, 662258898435328422, 490504798322512607, 994177919648945667, 588533035873768395, 898823384316323885, 303847758267277208, 319968700254395169, 846939845109467856, 269379298509462913, 756660427737584906, 217361443446995809, 959828018626317934, 672276175927862908, 483406833671754707, 759749405930788048, 703967305336038065, 922937913578630290, 431203504305585335, 770383512273821745|
|practice_0062|independent|128|0|286|297|9|295|8.72|217361443446995809, 269379298509462913, 272968328320912336, 293860350109691496, 303847758267277208, 319968700254395169, 342404678813681359, 420994477073247576, 431203504305585335, 483406833671754707, 490504798322512607, 547855343572897979, 588533035873768395, 662258898435328422, 672276175927862908, 675799579395187684, 703967305336038065, 756660427737584906, 759749405930788048, 770383512273821745, 777242217060877998, 777828257276985925, 801152261965756910, 811750177002917517, 846939845109467856, 898823384316323885, 922937913578630290, 959828018626317934, 994177919648945667|
|practice_0062|independent|128|1|279|283|0|279|35.40|217361443446995809, 770383512273821745, 483406833671754707, 269379298509462913, 272968328320912336, 303847758267277208, 420994477073247576, 431203504305585335, 588533035873768395, 777828257276985925, 293860350109691496, 662258898435328422, 898823384316323885, 319968700254395169, 759749405930788048, 490504798322512607, 756660427737584906, 703967305336038065, 342404678813681359, 547855343572897979, 672276175927862908, 675799579395187684, 777242217060877998, 801152261965756910, 811750177002917517, 846939845109467856, 922937913578630290, 959828018626317934, 994177919648945667|
|practice_0062|independent|128|2|279|287|0|279|252.59|272968328320912336, 293860350109691496, 217361443446995809, 662258898435328422, 483406833671754707, 303847758267277208, 342404678813681359, 319968700254395169, 588533035873768395, 420994477073247576, 547855343572897979, 269379298509462913, 756660427737584906, 703967305336038065, 672276175927862908, 811750177002917517, 959828018626317934, 777242217060877998, 675799579395187684, 801152261965756910, 770383512273821745, 759749405930788048, 994177919648945667, 922937913578630290, 846939845109467856, 431203504305585335, 490504798322512607, 777828257276985925, 898823384316323885|
|practice_0063|multiplicative|0|0|131|0|0|131|2.87|1010485, 1679159, 5048175, 5788477, 6482682, 7556292, 13630639, 17631519, 19846343, 35981997, 60452952, 66997371, 68152941, 86827342, 100957800, 113343675, 158770236, 168755581, 330770797, 330770800, 337591967, 605963000, 676744198, 1190748960, 2037025190, 3375111600, 26698447421, 10756312683376|
|practice_0063|multiplicative|0|1|128|0|0|128|3.44|13630639, 68152941, 1010485, 1679159, 7556292, 17631519, 113343675, 5048175, 5788477, 86827342, 60452952, 6482682, 19846343, 158770236, 168755581, 1190748960, 3375111600, 66997371, 100957800, 330770797, 330770800, 337591967, 605963000, 676744198, 35981997, 2037025190, 26698447421, 10756312683376|
|practice_0063|multiplicative|0|2|127|0|0|127|18.50|19846343, 1679159, 5788477, 7556292, 13630639, 68152941, 17631519, 86827342, 113343675, 60452952, 158770236, 66997371, 168755581, 3375111600, 330770797, 330770800, 337591967, 605963000, 676744198, 1010485, 35981997, 100957800, 2037025190, 5048175, 6482682, 1190748960, 26698447421, 10756312683376|
|practice_0063|multiplicative|48|0|142|206|0|142|0.96|1010485, 1679159, 5048175, 5788477, 6482682, 7556292, 13630639, 17631519, 19846343, 35981997, 60452952, 66997371, 68152941, 86827342, 100957800, 113343675, 158770236, 168755581, 330770797, 330770800, 337591967, 605963000, 676744198, 1190748960, 2037025190, 3375111600, 26698447421, 10756312683376|
|practice_0063|multiplicative|48|1|135|147|0|135|1.85|13630639, 68152941, 1010485, 5048175, 1679159, 7556292, 17631519, 5788477, 6482682, 60452952, 19846343, 158770236, 66997371, 86827342, 330770797, 330770800, 113343675, 1190748960, 100957800, 337591967, 35981997, 605963000, 168755581, 3375111600, 2037025190, 676744198, 26698447421, 10756312683376|
|practice_0063|multiplicative|48|2|130|139|0|130|9.73|19846343, 1679159, 5788477, 7556292, 13630639, 68152941, 17631519, 113343675, 86827342, 66997371, 5048175, 676744198, 168755581, 60452952, 1190748960, 6482682, 337591967, 100957800, 3375111600, 158770236, 605963000, 26698447421, 1010485, 330770797, 330770800, 10756312683376, 35981997, 2037025190|
|practice_0063|multiplicative|128|0|128|130|0|128|3.67|1010485, 1679159, 5048175, 5788477, 6482682, 7556292, 13630639, 17631519, 19846343, 35981997, 60452952, 66997371, 68152941, 86827342, 100957800, 113343675, 158770236, 168755581, 330770797, 330770800, 337591967, 605963000, 676744198, 1190748960, 2037025190, 3375111600, 26698447421, 10756312683376|
|practice_0063|multiplicative|128|1|127|129|0|127|6.03|13630639, 68152941, 1010485, 5048175, 1679159, 7556292, 17631519, 100957800, 113343675, 5788477, 86827342, 60452952, 6482682, 19846343, 158770236, 35981997, 66997371, 168755581, 3375111600, 330770797, 330770800, 337591967, 605963000, 676744198, 1190748960, 2037025190, 26698447421, 10756312683376|
|practice_0063|multiplicative|128|2|121|125|0|121|31.25|19846343, 1679159, 5788477, 7556292, 13630639, 68152941, 17631519, 86827342, 113343675, 60452952, 158770236, 66997371, 168755581, 3375111600, 330770797, 330770800, 35981997, 6482682, 605963000, 337591967, 1190748960, 5048175, 100957800, 1010485, 676744198, 10756312683376, 26698447421, 2037025190|
|practice_0064|mixed|0|0|255|0|0|255|3.78|76928795681, 175612251366, 337696497227, 372524661214, 686334373127, 686334373223, 1077003140094, 1466343848400, 2745337492787, 3494118932035, 4650526712091, 7450493229252, 27691112765972, 1071432361313561, 137220771546205873, 203824908275587511, 218465650140026119, 243388753481194663, 440795364871210248, 445125886468410677, 495053605141950519, 571169417816152079, 594704731115649776, 637464565024787139, 669473413588665723, 676736597716696355, 730426457378415232, 797910559329631531, 910848877575662661|
|practice_0064|mixed|0|1|250|0|0|250|15.67|76928795681, 1077003140094, 337696497227, 27691112765972, 175612251366, 372524661214, 7450493229252, 686334373127, 686334373223, 2745337492787, 1466343848400, 3494118932035, 4650526712091, 1071432361313561, 440795364871210248, 137220771546205873, 637464565024787139, 203824908275587511, 218465650140026119, 243388753481194663, 594704731115649776, 571169417816152079, 445125886468410677, 676736597716696355, 730426457378415232, 495053605141950519, 669473413588665723, 797910559329631531, 910848877575662661|
|practice_0064|mixed|0|2|248|0|74|322|95.95|686334373223, 337696497227, 76928795681, 2745337492787, 686334373127, 175612251366, 27691112765972, 1077003140094, 372524661214, 7450493229252, 1466343848400, 4650526712091, 3494118932035, 218465650140026119, 440795364871210248, 1071432361313561, 137220771546205873, 594704731115649776, 571169417816152079, 637464565024787139, 243388753481194663, 910848877575662661, 203824908275587511, 445125886468410677, 676736597716696355, 730426457378415232, 797910559329631531, 495053605141950519, 669473413588665723|
|practice_0064|mixed|48|0|262|377|0|262|1.43|76928795681, 175612251366, 337696497227, 372524661214, 686334373127, 686334373223, 1077003140094, 1466343848400, 2745337492787, 3494118932035, 4650526712091, 7450493229252, 27691112765972, 1071432361313561, 137220771546205873, 203824908275587511, 218465650140026119, 243388753481194663, 440795364871210248, 445125886468410677, 495053605141950519, 571169417816152079, 594704731115649776, 637464565024787139, 669473413588665723, 676736597716696355, 730426457378415232, 797910559329631531, 910848877575662661|
|practice_0064|mixed|48|1|250|269|0|250|5.93|76928795681, 1077003140094, 337696497227, 27691112765972, 175612251366, 2745337492787, 686334373127, 686334373223, 1466343848400, 372524661214, 7450493229252, 1071432361313561, 3494118932035, 4650526712091, 218465650140026119, 137220771546205873, 243388753481194663, 203824908275587511, 571169417816152079, 730426457378415232, 440795364871210248, 445125886468410677, 495053605141950519, 594704731115649776, 669473413588665723, 676736597716696355, 637464565024787139, 797910559329631531, 910848877575662661|
|practice_0064|mixed|48|2|265|281|0|265|36.13|686334373127, 686334373223, 1077003140094, 1466343848400, 76928795681, 2745337492787, 175612251366, 337696497227, 372524661214, 203824908275587511, 4650526712091, 445125886468410677, 243388753481194663, 440795364871210248, 137220771546205873, 1071432361313561, 495053605141950519, 3494118932035, 218465650140026119, 637464565024787139, 571169417816152079, 7450493229252, 797910559329631531, 730426457378415232, 27691112765972, 594704731115649776, 676736597716696355, 910848877575662661, 669473413588665723|
|practice_0064|mixed|128|0|232|239|60|292|5.96|76928795681, 175612251366, 337696497227, 372524661214, 686334373127, 686334373223, 1077003140094, 1466343848400, 2745337492787, 3494118932035, 4650526712091, 7450493229252, 27691112765972, 1071432361313561, 137220771546205873, 203824908275587511, 218465650140026119, 243388753481194663, 440795364871210248, 445125886468410677, 495053605141950519, 571169417816152079, 594704731115649776, 637464565024787139, 669473413588665723, 676736597716696355, 730426457378415232, 797910559329631531, 910848877575662661|
|practice_0064|mixed|128|1|225|228|35|260|27.57|76928795681, 1077003140094, 337696497227, 27691112765972, 175612251366, 372524661214, 7450493229252, 686334373127, 686334373223, 2745337492787, 1466343848400, 3494118932035, 4650526712091, 1071432361313561, 137220771546205873, 243388753481194663, 495053605141950519, 203824908275587511, 440795364871210248, 218465650140026119, 571169417816152079, 730426457378415232, 445125886468410677, 594704731115649776, 637464565024787139, 676736597716696355, 910848877575662661, 797910559329631531, 669473413588665723|
|practice_0064|mixed|128|2|225|227|3|228|186.92|686334373127, 76928795681, 1466343848400, 337696497227, 27691112765972, 686334373223, 2745337492787, 372524661214, 1077003140094, 7450493229252, 1071432361313561, 175612251366, 4650526712091, 3494118932035, 445125886468410677, 571169417816152079, 137220771546205873, 203824908275587511, 440795364871210248, 218465650140026119, 243388753481194663, 730426457378415232, 495053605141950519, 797910559329631531, 594704731115649776, 676736597716696355, 669473413588665723, 637464565024787139, 910848877575662661|
|practice_0065|mixed|0|0|236|0|0|236|3.42|747100011, 779957065, 2971227603, 3101889185, 5259924195, 19536861137, 27916986997, 27917000399, 80223238983, 83194470548, 241205096264, 893344076347, 893344076352, 8361777456236, 21364492229760, 100772185191377, 116014969976117, 123761289124381, 160613528520118, 184954077303035, 213091601337958, 264417872134008, 418964901080258, 436631212765062, 742911033375445, 810023848383082, 810309364684747, 835733550307888, 878981695237916, 918457995013828, 934960808889110, 19707383693054592|
|practice_0065|mixed|0|1|231|0|0|231|12.59|3101889185, 893344076352, 893344076347, 27917000399, 27916986997, 747100011, 779957065, 2971227603, 83194470548, 80223238983, 5259924195, 19536861137, 241205096264, 8361777456236, 21364492229760, 160613528520118, 100772185191377, 116014969976117, 123761289124381, 184954077303035, 264417872134008, 835733550307888, 213091601337958, 934960808889110, 418964901080258, 436631212765062, 742911033375445, 810309364684747, 810023848383082, 918457995013828, 878981695237916, 19707383693054592|
|practice_0065|mixed|0|2|225|0|93|318|90.48|3101889185, 27916986997, 2971227603, 893344076352, 747100011, 27917000399, 893344076347, 80223238983, 5259924195, 83194470548, 779957065, 241205096264, 19536861137, 21364492229760, 264417872134008, 835733550307888, 100772185191377, 8361777456236, 810023848383082, 810309364684747, 160613528520118, 184954077303035, 918457995013828, 213091601337958, 123761289124381, 19707383693054592, 742911033375445, 116014969976117, 934960808889110, 436631212765062, 418964901080258, 878981695237916|
|practice_0065|mixed|48|0|241|294|0|241|1.71|747100011, 779957065, 2971227603, 3101889185, 5259924195, 19536861137, 27916986997, 27917000399, 80223238983, 83194470548, 241205096264, 893344076347, 893344076352, 8361777456236, 21364492229760, 100772185191377, 116014969976117, 123761289124381, 160613528520118, 184954077303035, 213091601337958, 264417872134008, 418964901080258, 436631212765062, 742911033375445, 810023848383082, 810309364684747, 835733550307888, 878981695237916, 918457995013828, 934960808889110, 19707383693054592|
|practice_0065|mixed|48|1|242|253|0|242|6.39|3101889185, 893344076352, 893344076347, 27917000399, 27916986997, 779957065, 2971227603, 83194470548, 80223238983, 747100011, 5259924195, 241205096264, 19536861137, 21364492229760, 8361777456236, 160613528520118, 213091601337958, 264417872134008, 418964901080258, 116014969976117, 100772185191377, 835733550307888, 184954077303035, 934960808889110, 878981695237916, 810023848383082, 810309364684747, 19707383693054592, 123761289124381, 742911033375445, 918457995013828, 436631212765062|
|practice_0065|mixed|48|2|236|252|0|236|38.61|747100011, 2971227603, 83194470548, 27916986997, 27917000399, 80223238983, 893344076352, 779957065, 21364492229760, 19536861137, 893344076347, 3101889185, 5259924195, 8361777456236, 436631212765062, 878981695237916, 100772185191377, 213091601337958, 116014969976117, 264417872134008, 742911033375445, 835733550307888, 918457995013828, 184954077303035, 123761289124381, 241205096264, 418964901080258, 810023848383082, 934960808889110, 810309364684747, 160613528520118, 19707383693054592|
|practice_0065|mixed|128|0|220|226|17|237|7.68|747100011, 779957065, 2971227603, 3101889185, 5259924195, 19536861137, 27916986997, 27917000399, 80223238983, 83194470548, 241205096264, 893344076347, 893344076352, 8361777456236, 21364492229760, 100772185191377, 116014969976117, 123761289124381, 160613528520118, 184954077303035, 213091601337958, 264417872134008, 418964901080258, 436631212765062, 742911033375445, 810023848383082, 810309364684747, 835733550307888, 878981695237916, 918457995013828, 934960808889110, 19707383693054592|
|practice_0065|mixed|128|1|216|219|2|218|22.89|3101889185, 893344076352, 893344076347, 27917000399, 27916986997, 747100011, 779957065, 2971227603, 83194470548, 80223238983, 5259924195, 19536861137, 241205096264, 8361777456236, 21364492229760, 116014969976117, 123761289124381, 160613528520118, 100772185191377, 436631212765062, 264417872134008, 418964901080258, 184954077303035, 213091601337958, 810023848383082, 810309364684747, 742911033375445, 835733550307888, 878981695237916, 918457995013828, 934960808889110, 19707383693054592|
|practice_0065|mixed|128|2|205|208|60|265|166.49|747100011, 27917000399, 779957065, 5259924195, 19536861137, 2971227603, 893344076352, 893344076347, 83194470548, 27916986997, 3101889185, 80223238983, 241205096264, 21364492229760, 8361777456236, 213091601337958, 264417872134008, 160613528520118, 184954077303035, 436631212765062, 742911033375445, 418964901080258, 934960808889110, 810023848383082, 100772185191377, 878981695237916, 835733550307888, 123761289124381, 116014969976117, 918457995013828, 810309364684747, 19707383693054592|
|practice_0066|multiplicative|0|0|112|0|0|112|2.14|2684, 3065, 4313, 6003, 6428, 11325, 11381, 11395, 13281, 18902, 23039, 36986, 65269, 65325, 239139, 242972, 613980, 1164473, 1578254, 2612264, 3431298, 5799705, 13856760, 25900248, 118359468, 186709463, 947177012, 6329484240, 20608902396, 21267310080|
|practice_0066|multiplicative|0|1|111|0|0|111|3.21|3065, 4313, 6003, 2684, 13281, 6428, 65269, 65325, 613980, 11325, 11381, 11395, 18902, 23039, 2612264, 36986, 239139, 242972, 1164473, 1578254, 5799705, 3431298, 13856760, 25900248, 118359468, 947177012, 21267310080, 186709463, 6329484240, 20608902396|
|practice_0066|multiplicative|0|2|107|0|0|107|19.79|2684, 13281, 3065, 65269, 4313, 613980, 11381, 11325, 65325, 23039, 36986, 6003, 6428, 2612264, 11395, 18902, 242972, 5799705, 239139, 1164473, 1578254, 25900248, 3431298, 13856760, 118359468, 947177012, 21267310080, 186709463, 6329484240, 20608902396|
|practice_0066|multiplicative|48|0|117|127|0|117|0.81|2684, 3065, 4313, 6003, 6428, 11325, 11381, 11395, 13281, 18902, 23039, 36986, 65269, 65325, 239139, 242972, 613980, 1164473, 1578254, 2612264, 3431298, 5799705, 13856760, 25900248, 118359468, 186709463, 947177012, 6329484240, 20608902396, 21267310080|
|practice_0066|multiplicative|48|1|115|119|0|115|1.69|3065, 4313, 6003, 2684, 13281, 6428, 65269, 65325, 613980, 11325, 11381, 11395, 18902, 23039, 36986, 239139, 242972, 2612264, 1164473, 1578254, 5799705, 3431298, 13856760, 25900248, 21267310080, 118359468, 6329484240, 186709463, 947177012, 20608902396|
|practice_0066|multiplicative|48|2|119|126|0|119|10.34|2684, 13281, 6428, 3065, 11325, 4313, 65269, 11381, 65325, 6003, 18902, 242972, 11395, 239139, 36986, 1164473, 2612264, 23039, 613980, 1578254, 3431298, 5799705, 947177012, 6329484240, 13856760, 118359468, 186709463, 20608902396, 25900248, 21267310080|
|practice_0066|multiplicative|128|0|108|112|0|108|2.95|2684, 3065, 4313, 6003, 6428, 11325, 11381, 11395, 13281, 18902, 23039, 36986, 65269, 65325, 239139, 242972, 613980, 1164473, 1578254, 2612264, 3431298, 5799705, 13856760, 25900248, 118359468, 186709463, 947177012, 6329484240, 20608902396, 21267310080|
|practice_0066|multiplicative|128|1|106|109|0|106|4.18|3065, 4313, 6003, 2684, 13281, 6428, 65269, 65325, 613980, 11325, 11381, 11395, 18902, 23039, 36986, 239139, 242972, 1164473, 1578254, 2612264, 3431298, 5799705, 13856760, 25900248, 118359468, 186709463, 21267310080, 947177012, 6329484240, 20608902396|
|practice_0066|multiplicative|128|2|111|112|0|111|25.97|2684, 13281, 6428, 3065, 11325, 4313, 65269, 11381, 65325, 6003, 11395, 242972, 18902, 239139, 613980, 36986, 2612264, 13856760, 3431298, 23039, 1164473, 1578254, 947177012, 5799705, 25900248, 6329484240, 20608902396, 118359468, 186709463, 21267310080|
|practice_0067|shared|0|0|173|0|0|173|3.94|450442995, 450445826, 450447054, 450447347, 669206889, 2435151664, 2435152039, 3617781934, 3617782029, 3713636290, 9368892941, 9368945684, 18088904840, 21414615310, 43242934656, 43413382792, 52251883174, 78297177848, 78297178343, 108533457991, 116239013763, 151946849823, 282477623179, 311945447520, 422521269840, 3397096767588, 8788022088930, 81573783429030, 128050029087176, 198518548064382, 794337587092284, 136892232855846876|
|practice_0067|shared|0|1|165|0|0|165|9.21|450447347, 450442995, 450445826, 450447054, 43242934656, 669206889, 21414615310, 9368892941, 9368945684, 78297177848, 78297178343, 2435151664, 2435152039, 3617781934, 3617782029, 43413382792, 108533457991, 18088904840, 198518548064382, 151946849823, 8788022088930, 52251883174, 422521269840, 116239013763, 3713636290, 311945447520, 282477623179, 3397096767588, 128050029087176, 81573783429030, 794337587092284, 136892232855846876|
|practice_0067|shared|0|2|160|0|0|160|60.92|450442995, 2435151664, 669206889, 9368892941, 450445826, 43242934656, 450447347, 78297177848, 3617782029, 3617781934, 2435152039, 450447054, 108533457991, 78297178343, 43413382792, 18088904840, 9368945684, 3713636290, 198518548064382, 52251883174, 3397096767588, 282477623179, 151946849823, 21414615310, 311945447520, 794337587092284, 8788022088930, 128050029087176, 81573783429030, 116239013763, 422521269840, 136892232855846876|
|practice_0067|shared|48|0|194|208|0|194|1.48|450442995, 450445826, 450447054, 450447347, 669206889, 2435151664, 2435152039, 3617781934, 3617782029, 3713636290, 9368892941, 9368945684, 18088904840, 21414615310, 43242934656, 43413382792, 52251883174, 78297177848, 78297178343, 108533457991, 116239013763, 151946849823, 282477623179, 311945447520, 422521269840, 3397096767588, 8788022088930, 81573783429030, 128050029087176, 198518548064382, 794337587092284, 136892232855846876|
|practice_0067|shared|48|1|172|178|0|172|4.10|450447347, 450442995, 450445826, 450447054, 43242934656, 52251883174, 669206889, 21414615310, 3617782029, 3617781934, 43413382792, 108533457991, 9368892941, 9368945684, 3713636290, 18088904840, 151946849823, 311945447520, 198518548064382, 2435151664, 2435152039, 282477623179, 8788022088930, 78297178343, 78297177848, 3397096767588, 116239013763, 422521269840, 81573783429030, 794337587092284, 128050029087176, 136892232855846876|
|practice_0067|shared|48|2|192|204|0|192|27.73|450442995, 669206889, 450445826, 450447347, 2435151664, 3617781934, 9368892941, 450447054, 9368945684, 108533457991, 43413382792, 3617782029, 198518548064382, 3713636290, 78297177848, 2435152039, 78297178343, 18088904840, 21414615310, 151946849823, 52251883174, 311945447520, 422521269840, 116239013763, 282477623179, 8788022088930, 81573783429030, 3397096767588, 43242934656, 128050029087176, 794337587092284, 136892232855846876|
|practice_0067|shared|128|0|163|169|23|186|5.12|450442995, 450445826, 450447054, 450447347, 669206889, 2435151664, 2435152039, 3617781934, 3617782029, 3713636290, 9368892941, 9368945684, 18088904840, 21414615310, 43242934656, 43413382792, 52251883174, 78297177848, 78297178343, 108533457991, 116239013763, 151946849823, 282477623179, 311945447520, 422521269840, 3397096767588, 8788022088930, 81573783429030, 128050029087176, 198518548064382, 794337587092284, 136892232855846876|
|practice_0067|shared|128|1|156|161|0|156|14.05|450447347, 450442995, 450445826, 450447054, 43242934656, 3713636290, 311945447520, 669206889, 9368892941, 21414615310, 9368945684, 78297177848, 78297178343, 8788022088930, 2435151664, 2435152039, 3617781934, 3617782029, 18088904840, 108533457991, 43413382792, 198518548064382, 151946849823, 52251883174, 116239013763, 282477623179, 422521269840, 3397096767588, 794337587092284, 81573783429030, 128050029087176, 136892232855846876|
|practice_0067|shared|128|2|150|155|0|150|75.01|450442995, 669206889, 450445826, 78297177848, 450447347, 43242934656, 3713636290, 78297178343, 2435151664, 2435152039, 9368945684, 3617781934, 3617782029, 9368892941, 198518548064382, 8788022088930, 18088904840, 450447054, 108533457991, 282477623179, 422521269840, 3397096767588, 43413382792, 794337587092284, 311945447520, 21414615310, 151946849823, 52251883174, 128050029087176, 116239013763, 81573783429030, 136892232855846876|
|practice_0068|mixed|0|0|198|0|0|198|2.89|13603244, 26982790, 29377549, 63459145, 81640132, 1821932980, 10338694640, 14577716768, 26818390364, 54203416253, 58479186740, 58479775780, 108395574980, 303345263661, 342932830219, 380346604074, 606642538829, 668414985766, 675030493219, 699747844260, 773532514651, 803121067085, 813390628807, 814198914533, 844714024625, 853945942587, 944954181639, 994240533810, 2777329960034, 7885453305753|
|practice_0068|mixed|0|1|193|0|0|193|7.90|29377549, 13603244, 26982790, 63459145, 81640132, 10338694640, 14577716768, 1821932980, 58479186740, 58479775780, 108395574980, 54203416253, 303345263661, 26818390364, 342932830219, 380346604074, 606642538829, 813390628807, 814198914533, 675030493219, 994240533810, 668414985766, 699747844260, 844714024625, 944954181639, 773532514651, 803121067085, 7885453305753, 853945942587, 2777329960034|
|practice_0068|mixed|0|2|184|0|0|184|46.35|13603244, 63459145, 14577716768, 26982790, 1821932980, 303345263661, 54203416253, 29377549, 58479775780, 108395574980, 58479186740, 26818390364, 10338694640, 813390628807, 675030493219, 994240533810, 81640132, 342932830219, 606642538829, 803121067085, 668414985766, 699747844260, 844714024625, 814198914533, 853945942587, 773532514651, 7885453305753, 380346604074, 944954181639, 2777329960034|
|practice_0068|mixed|48|0|200|240|0|200|1.33|13603244, 26982790, 29377549, 63459145, 81640132, 1821932980, 10338694640, 14577716768, 26818390364, 54203416253, 58479186740, 58479775780, 108395574980, 303345263661, 342932830219, 380346604074, 606642538829, 668414985766, 675030493219, 699747844260, 773532514651, 803121067085, 813390628807, 814198914533, 844714024625, 853945942587, 944954181639, 994240533810, 2777329960034, 7885453305753|
|practice_0068|mixed|48|1|204|210|0|204|3.99|29377549, 13603244, 26982790, 63459145, 81640132, 10338694640, 14577716768, 1821932980, 26818390364, 58479186740, 58479775780, 54203416253, 803121067085, 699747844260, 108395574980, 814198914533, 813390628807, 342932830219, 380346604074, 773532514651, 303345263661, 606642538829, 668414985766, 675030493219, 844714024625, 994240533810, 853945942587, 2777329960034, 944954181639, 7885453305753|
|practice_0068|mixed|48|2|205|212|0|205|22.56|63459145, 13603244, 1821932980, 26982790, 29377549, 14577716768, 10338694640, 303345263661, 58479775780, 606642538829, 803121067085, 26818390364, 54203416253, 58479186740, 853945942587, 342932830219, 81640132, 699747844260, 773532514651, 844714024625, 813390628807, 675030493219, 668414985766, 2777329960034, 814198914533, 108395574980, 380346604074, 994240533810, 944954181639, 7885453305753|
|practice_0068|mixed|128|0|185|195|26|211|6.12|13603244, 26982790, 29377549, 63459145, 81640132, 1821932980, 10338694640, 14577716768, 26818390364, 54203416253, 58479186740, 58479775780, 108395574980, 303345263661, 342932830219, 380346604074, 606642538829, 668414985766, 675030493219, 699747844260, 773532514651, 803121067085, 813390628807, 814198914533, 844714024625, 853945942587, 944954181639, 994240533810, 2777329960034, 7885453305753|
|practice_0068|mixed|128|1|180|182|1|181|11.35|29377549, 13603244, 26982790, 63459145, 81640132, 1821932980, 10338694640, 14577716768, 58479775780, 58479186740, 54203416253, 108395574980, 668414985766, 994240533810, 26818390364, 342932830219, 303345263661, 699747844260, 773532514651, 844714024625, 944954181639, 606642538829, 380346604074, 675030493219, 803121067085, 813390628807, 814198914533, 853945942587, 2777329960034, 7885453305753|
|practice_0068|mixed|128|2|180|182|25|205|108.88|63459145, 13603244, 1821932980, 81640132, 29377549, 58479186740, 26982790, 303345263661, 14577716768, 844714024625, 54203416253, 58479775780, 10338694640, 606642538829, 26818390364, 675030493219, 108395574980, 668414985766, 773532514651, 2777329960034, 380346604074, 813390628807, 699747844260, 944954181639, 853945942587, 994240533810, 814198914533, 803121067085, 342932830219, 7885453305753|
|practice_0069|independent|0|0|220|0|0|220|2.63|132011636453, 184756882024, 190151111233, 196501569938, 320044466726, 339634695568, 343005051804, 382758804322, 389439237224, 480972570935, 483531850892, 541277140804, 548428823502, 626542640865, 662543969299, 675672956502, 694401198341, 713373943733, 745120092786, 761395130054, 772954376336, 787657218325, 824460195518, 856616671530, 858406149168, 867232308754, 879101556421, 958296533093, 966519051693, 982710199409|
|practice_0069|independent|0|1|209|0|0|209|6.29|196501569938, 132011636453, 761395130054, 184756882024, 382758804322, 480972570935, 339634695568, 745120092786, 190151111233, 320044466726, 343005051804, 389439237224, 483531850892, 548428823502, 626542640865, 675672956502, 541277140804, 958296533093, 694401198341, 772954376336, 713373943733, 787657218325, 824460195518, 858406149168, 856616671530, 867232308754, 662543969299, 879101556421, 966519051693, 982710199409|
|practice_0069|independent|0|2|204|0|58|262|44.20|196501569938, 190151111233, 548428823502, 480972570935, 184756882024, 389439237224, 132011636453, 675672956502, 626542640865, 966519051693, 982710199409, 320044466726, 662543969299, 856616671530, 382758804322, 787657218325, 343005051804, 858406149168, 745120092786, 541277140804, 772954376336, 824460195518, 483531850892, 867232308754, 713373943733, 879101556421, 339634695568, 958296533093, 694401198341, 761395130054|
|practice_0069|independent|48|0|217|254|0|217|1.15|132011636453, 184756882024, 190151111233, 196501569938, 320044466726, 339634695568, 343005051804, 382758804322, 389439237224, 480972570935, 483531850892, 541277140804, 548428823502, 626542640865, 662543969299, 675672956502, 694401198341, 713373943733, 745120092786, 761395130054, 772954376336, 787657218325, 824460195518, 856616671530, 858406149168, 867232308754, 879101556421, 958296533093, 966519051693, 982710199409|
|practice_0069|independent|48|1|218|225|0|218|3.43|196501569938, 982710199409, 184756882024, 480972570935, 343005051804, 132011636453, 339634695568, 190151111233, 382758804322, 548428823502, 389439237224, 626542640865, 694401198341, 966519051693, 713373943733, 541277140804, 320044466726, 483531850892, 745120092786, 824460195518, 675672956502, 662543969299, 772954376336, 761395130054, 787657218325, 858406149168, 856616671530, 867232308754, 879101556421, 958296533093|
|practice_0069|independent|48|2|221|228|0|221|22.08|132011636453, 184756882024, 196501569938, 483531850892, 480972570935, 382758804322, 772954376336, 343005051804, 190151111233, 548428823502, 694401198341, 713373943733, 541277140804, 675672956502, 320044466726, 626542640865, 662543969299, 389439237224, 858406149168, 824460195518, 867232308754, 958296533093, 966519051693, 761395130054, 339634695568, 879101556421, 745120092786, 856616671530, 787657218325, 982710199409|
|practice_0069|independent|128|0|204|206|7|211|6.31|132011636453, 184756882024, 190151111233, 196501569938, 320044466726, 339634695568, 343005051804, 382758804322, 389439237224, 480972570935, 483531850892, 541277140804, 548428823502, 626542640865, 662543969299, 675672956502, 694401198341, 713373943733, 745120092786, 761395130054, 772954376336, 787657218325, 824460195518, 856616671530, 858406149168, 867232308754, 879101556421, 958296533093, 966519051693, 982710199409|
|practice_0069|independent|128|1|192|195|0|192|19.57|196501569938, 982710199409, 184756882024, 480972570935, 132011636453, 190151111233, 339634695568, 320044466726, 966519051693, 343005051804, 382758804322, 389439237224, 541277140804, 662543969299, 548428823502, 694401198341, 856616671530, 483531850892, 626542640865, 675672956502, 745120092786, 858406149168, 867232308754, 713373943733, 824460195518, 761395130054, 787657218325, 772954376336, 879101556421, 958296533093|
|practice_0069|independent|128|2|191|193|9|200|125.55|132011636453, 184756882024, 966519051693, 626542640865, 190151111233, 320044466726, 196501569938, 382758804322, 389439237224, 483531850892, 480972570935, 339634695568, 787657218325, 856616671530, 824460195518, 982710199409, 662543969299, 343005051804, 772954376336, 548428823502, 879101556421, 745120092786, 694401198341, 858406149168, 761395130054, 713373943733, 958296533093, 541277140804, 867232308754, 675672956502|
|practice_0070|independent|0|0|271|0|0|271|3.33|102177775498192, 118999834123640, 143649784701013, 199434480247716, 219125753040098, 224750598106835, 225006497695255, 230795762268426, 231416764203192, 280905794861165, 281391270604732, 319200480122429, 375858965178700, 404126946608115, 452361417896563, 503646731087839, 539137894336816, 638577161155880, 717515608138046, 736334386979470, 806593749956129, 811506502469537, 861488371939382, 868301225390495, 889708425079834, 926391117071842, 931675015983738, 958751607276494, 975420780711492, 979397606671605|
|practice_0070|independent|0|1|255|0|114|369|10.74|231416764203192, 224750598106835, 225006497695255, 230795762268426, 281391270604732, 280905794861165, 102177775498192, 143649784701013, 319200480122429, 118999834123640, 199434480247716, 638577161155880, 975420780711492, 717515608138046, 404126946608115, 979397606671605, 452361417896563, 219125753040098, 736334386979470, 539137894336816, 503646731087839, 806593749956129, 889708425079834, 958751607276494, 931675015983738, 868301225390495, 861488371939382, 375858965178700, 811506502469537, 926391117071842|
|practice_0070|independent|0|2|253|0|105|358|70.94|231416764203192, 926391117071842, 230795762268426, 102177775498192, 118999834123640, 219125753040098, 225006497695255, 281391270604732, 375858965178700, 811506502469537, 503646731087839, 199434480247716, 280905794861165, 224750598106835, 539137894336816, 452361417896563, 861488371939382, 736334386979470, 868301225390495, 638577161155880, 319200480122429, 717515608138046, 975420780711492, 931675015983738, 889708425079834, 979397606671605, 404126946608115, 143649784701013, 806593749956129, 958751607276494|
|practice_0070|independent|48|0|272|352|0|272|1.42|102177775498192, 118999834123640, 143649784701013, 199434480247716, 219125753040098, 224750598106835, 225006497695255, 230795762268426, 231416764203192, 280905794861165, 281391270604732, 319200480122429, 375858965178700, 404126946608115, 452361417896563, 503646731087839, 539137894336816, 638577161155880, 717515608138046, 736334386979470, 806593749956129, 811506502469537, 861488371939382, 868301225390495, 889708425079834, 926391117071842, 931675015983738, 958751607276494, 975420780711492, 979397606671605|
|practice_0070|independent|48|1|267|349|0|267|4.85|231416764203192, 230795762268426, 199434480247716, 281391270604732, 102177775498192, 219125753040098, 118999834123640, 143649784701013, 375858965178700, 404126946608115, 224750598106835, 225006497695255, 926391117071842, 979397606671605, 717515608138046, 638577161155880, 736334386979470, 889708425079834, 861488371939382, 280905794861165, 958751607276494, 811506502469537, 503646731087839, 868301225390495, 452361417896563, 975420780711492, 319200480122429, 539137894336816, 931675015983738, 806593749956129|
|practice_0070|independent|48|2|283|291|0|283|26.89|102177775498192, 199434480247716, 281391270604732, 224750598106835, 231416764203192, 280905794861165, 806593749956129, 118999834123640, 889708425079834, 979397606671605, 861488371939382, 868301225390495, 931675015983738, 143649784701013, 958751607276494, 375858965178700, 319200480122429, 811506502469537, 539137894336816, 452361417896563, 926391117071842, 975420780711492, 503646731087839, 717515608138046, 219125753040098, 736334386979470, 638577161155880, 230795762268426, 225006497695255, 404126946608115|
|practice_0070|independent|128|0|252|269|17|269|7.24|102177775498192, 118999834123640, 143649784701013, 199434480247716, 219125753040098, 224750598106835, 225006497695255, 230795762268426, 231416764203192, 280905794861165, 281391270604732, 319200480122429, 375858965178700, 404126946608115, 452361417896563, 503646731087839, 539137894336816, 638577161155880, 717515608138046, 736334386979470, 806593749956129, 811506502469537, 861488371939382, 868301225390495, 889708425079834, 926391117071842, 931675015983738, 958751607276494, 975420780711492, 979397606671605|
|practice_0070|independent|128|1|237|246|0|237|26.99|231416764203192, 230795762268426, 199434480247716, 281391270604732, 280905794861165, 118999834123640, 319200480122429, 404126946608115, 224750598106835, 225006497695255, 375858965178700, 102177775498192, 143649784701013, 219125753040098, 811506502469537, 889708425079834, 861488371939382, 926391117071842, 539137894336816, 503646731087839, 452361417896563, 638577161155880, 868301225390495, 975420780711492, 958751607276494, 931675015983738, 717515608138046, 736334386979470, 979397606671605, 806593749956129|
|practice_0070|independent|128|2|238|240|18|256|157.42|102177775498192, 224750598106835, 281391270604732, 219125753040098, 868301225390495, 230795762268426, 225006497695255, 319200480122429, 231416764203192, 143649784701013, 375858965178700, 452361417896563, 638577161155880, 118999834123640, 861488371939382, 503646731087839, 199434480247716, 280905794861165, 404126946608115, 539137894336816, 926391117071842, 975420780711492, 958751607276494, 889708425079834, 736334386979470, 806593749956129, 811506502469537, 717515608138046, 931675015983738, 979397606671605|
|practice_0071|shared|0|0|169|0|0|169|3.16|18823605446, 41168133432, 172172165140, 200049068111, 218758482883, 263530476272, 399932895833, 464687210700, 800196279824, 875033985520, 1574470656370, 2238238146768, 3717497682552, 4408517837392, 4646872103260, 4646872103417, 4820820610584, 5199127642593, 8531580831774, 9262823541300, 9838061112375, 29515413335524, 30007360223968, 32007848602210, 34280177387953, 209109244662450, 472226952870010, 12753651068060897|
|practice_0071|shared|0|1|168|0|0|168|8.62|18823605446, 263530476272, 29515413335524, 9838061112375, 172172165140, 2238238146768, 4820820610584, 41168133432, 218758482883, 8531580831774, 875033985520, 4646872103417, 4646872103260, 3717497682552, 464687210700, 209109244662450, 9262823541300, 399932895833, 5199127642593, 200049068111, 800196279824, 30007360223968, 32007848602210, 1574470656370, 4408517837392, 472226952870010, 34280177387953, 12753651068060897|
|practice_0071|shared|0|2|168|0|0|168|58.10|200049068111, 18823605446, 263530476272, 172172165140, 4820820610584, 2238238146768, 218758482883, 875033985520, 9838061112375, 8531580831774, 800196279824, 399932895833, 5199127642593, 464687210700, 4646872103260, 30007360223968, 4646872103417, 3717497682552, 1574470656370, 209109244662450, 4408517837392, 41168133432, 9262823541300, 34280177387953, 29515413335524, 32007848602210, 472226952870010, 12753651068060897|
|practice_0071|shared|48|0|202|241|0|202|1.41|18823605446, 41168133432, 172172165140, 200049068111, 218758482883, 263530476272, 399932895833, 464687210700, 800196279824, 875033985520, 1574470656370, 2238238146768, 3717497682552, 4408517837392, 4646872103260, 4646872103417, 4820820610584, 5199127642593, 8531580831774, 9262823541300, 9838061112375, 29515413335524, 30007360223968, 32007848602210, 34280177387953, 209109244662450, 472226952870010, 12753651068060897|
|practice_0071|shared|48|1|173|184|0|173|3.78|18823605446, 263530476272, 29515413335524, 9838061112375, 172172165140, 2238238146768, 4820820610584, 41168133432, 200049068111, 800196279824, 218758482883, 8531580831774, 875033985520, 30007360223968, 32007848602210, 399932895833, 5199127642593, 464687210700, 3717497682552, 4646872103260, 4646872103417, 209109244662450, 1574470656370, 4408517837392, 9262823541300, 34280177387953, 472226952870010, 12753651068060897|
|practice_0071|shared|48|2|186|208|0|186|26.32|200049068111, 18823605446, 32007848602210, 29515413335524, 263530476272, 30007360223968, 172172165140, 464687210700, 4646872103417, 3717497682552, 41168133432, 800196279824, 4646872103260, 2238238146768, 399932895833, 4820820610584, 5199127642593, 209109244662450, 1574470656370, 8531580831774, 9838061112375, 4408517837392, 9262823541300, 218758482883, 34280177387953, 875033985520, 472226952870010, 12753651068060897|
|practice_0071|shared|128|0|161|166|0|161|5.42|18823605446, 41168133432, 172172165140, 200049068111, 218758482883, 263530476272, 399932895833, 464687210700, 800196279824, 875033985520, 1574470656370, 2238238146768, 3717497682552, 4408517837392, 4646872103260, 4646872103417, 4820820610584, 5199127642593, 8531580831774, 9262823541300, 9838061112375, 29515413335524, 30007360223968, 32007848602210, 34280177387953, 209109244662450, 472226952870010, 12753651068060897|
|practice_0071|shared|128|1|165|168|0|165|15.95|18823605446, 263530476272, 29515413335524, 9838061112375, 172172165140, 2238238146768, 4820820610584, 41168133432, 218758482883, 8531580831774, 875033985520, 4646872103417, 4646872103260, 464687210700, 3717497682552, 209109244662450, 800196279824, 200049068111, 30007360223968, 399932895833, 5199127642593, 1574470656370, 4408517837392, 9262823541300, 32007848602210, 34280177387953, 472226952870010, 12753651068060897|
|practice_0071|shared|128|2|166|169|2|168|105.80|200049068111, 18823605446, 41168133432, 172172165140, 263530476272, 2238238146768, 218758482883, 875033985520, 8531580831774, 30007360223968, 464687210700, 4646872103260, 3717497682552, 4646872103417, 800196279824, 399932895833, 209109244662450, 9838061112375, 5199127642593, 1574470656370, 4408517837392, 4820820610584, 9262823541300, 29515413335524, 32007848602210, 34280177387953, 472226952870010, 12753651068060897|
|practice_0072|tiny|0|0|8|0|0|8|0.01|4, 6, 12, 15|
|practice_0072|tiny|0|1|8|0|0|8|0.01|4, 15, 6, 12|
|practice_0072|tiny|0|2|8|0|0|8|0.01|4, 15, 6, 12|
|practice_0072|tiny|48|0|10|13|0|10|0.01|4, 15, 6, 12|
|practice_0072|tiny|48|1|10|13|0|10|0.01|4, 15, 6, 12|
|practice_0072|tiny|48|2|10|13|0|10|0.01|4, 15, 6, 12|
|practice_0072|tiny|128|0|10|13|0|10|0.01|4, 15, 6, 12|
|practice_0072|tiny|128|1|10|13|0|10|0.01|4, 15, 6, 12|
|practice_0072|tiny|128|2|10|13|0|10|0.01|4, 15, 6, 12|
|practice_0073|shared|0|0|159|0|0|159|2.78|24564416, 24630263, 29225062, 29566026, 38635024, 46481100, 46481912, 46504754, 81881508, 86731533, 144921006, 197047234, 219187899, 631261533, 1108380690, 3309095228, 4207027248, 5601396979, 13264806240, 26655929203, 57186004680, 85044953667, 115981324333, 121102768077, 320982230513, 359760295952, 378586376010, 1259034795088, 74724663265056|
|practice_0073|shared|0|1|147|0|0|147|5.99|24564416, 24630263, 197047234, 29225062, 29566026, 46504754, 46481100, 46481912, 38635024, 81881508, 13264806240, 86731533, 144921006, 219187899, 1108380690, 631261533, 57186004680, 121102768077, 3309095228, 4207027248, 5601396979, 320982230513, 74724663265056, 85044953667, 115981324333, 359760295952, 378586376010, 26655929203, 1259034795088|
|practice_0073|shared|0|2|145|0|0|145|38.33|24630263, 46481100, 29566026, 24564416, 46504754, 29225062, 197047234, 46481912, 1108380690, 38635024, 219187899, 86731533, 631261533, 359760295952, 81881508, 144921006, 13264806240, 74724663265056, 3309095228, 320982230513, 26655929203, 4207027248, 57186004680, 5601396979, 121102768077, 115981324333, 378586376010, 85044953667, 1259034795088|
|practice_0073|shared|48|0|172|220|0|172|0.95|24564416, 24630263, 29225062, 29566026, 38635024, 46481100, 46481912, 46504754, 81881508, 86731533, 144921006, 197047234, 219187899, 631261533, 1108380690, 3309095228, 4207027248, 5601396979, 13264806240, 26655929203, 57186004680, 85044953667, 115981324333, 121102768077, 320982230513, 359760295952, 378586376010, 1259034795088, 74724663265056|
|practice_0073|shared|48|1|165|178|0|165|2.20|24564416, 24630263, 197047234, 29225062, 29566026, 46504754, 46481100, 46481912, 38635024, 219187899, 3309095228, 86731533, 81881508, 144921006, 631261533, 13264806240, 57186004680, 1108380690, 4207027248, 5601396979, 26655929203, 85044953667, 115981324333, 121102768077, 378586376010, 359760295952, 320982230513, 1259034795088, 74724663265056|
|practice_0073|shared|48|2|165|171|0|165|17.44|29225062, 24564416, 46481100, 24630263, 46481912, 29566026, 46504754, 1108380690, 81881508, 197047234, 219187899, 86731533, 13264806240, 5601396979, 121102768077, 3309095228, 38635024, 631261533, 4207027248, 378586376010, 144921006, 320982230513, 359760295952, 85044953667, 26655929203, 57186004680, 115981324333, 74724663265056, 1259034795088|
|practice_0073|shared|128|0|151|154|4|155|4.38|24564416, 24630263, 29225062, 29566026, 38635024, 46481100, 46481912, 46504754, 81881508, 86731533, 144921006, 197047234, 219187899, 631261533, 1108380690, 3309095228, 4207027248, 5601396979, 13264806240, 26655929203, 57186004680, 85044953667, 115981324333, 121102768077, 320982230513, 359760295952, 378586376010, 1259034795088, 74724663265056|
|practice_0073|shared|128|1|145|148|0|145|8.38|24564416, 24630263, 197047234, 29225062, 29566026, 46504754, 46481100, 46481912, 81881508, 38635024, 86731533, 144921006, 219187899, 3309095228, 13264806240, 57186004680, 121102768077, 4207027248, 631261533, 1108380690, 5601396979, 26655929203, 85044953667, 115981324333, 320982230513, 378586376010, 359760295952, 1259034795088, 74724663265056|
|practice_0073|shared|128|2|145|148|0|145|53.41|29225062, 24564416, 46481100, 24630263, 29566026, 46481912, 197047234, 219187899, 46504754, 81881508, 13264806240, 38635024, 144921006, 1108380690, 57186004680, 3309095228, 86731533, 631261533, 4207027248, 320982230513, 74724663265056, 121102768077, 26655929203, 5601396979, 115981324333, 1259034795088, 359760295952, 85044953667, 378586376010|
|practice_0074|shared|0|0|226|0|0|226|4.00|270132653033, 334401691341, 346820883948, 346820885496, 440163248904, 440163249044, 609333200433, 617641565064, 657003598755, 843521012352, 1755053180730, 1827999601539, 2628846496048, 2842414375271, 3510106362348, 8262088988295, 31536172739650, 36803851104370, 53756397968691, 138381532688484, 149567938711620, 212687607855160, 2562833809936800, 2866303374832128, 3922311484644960, 5188707998036472, 6438408049239936, 9064510622856810, 51380309438054415, 92435140906879260, 172863751769112327, 418382519719409532|
|practice_0074|shared|0|1|205|0|5|210|14.81|346820885496, 346820883948, 138381532688484, 270132653033, 440163248904, 440163249044, 609333200433, 1827999601539, 2866303374832128, 334401691341, 2842414375271, 617641565064, 3510106362348, 1755053180730, 657003598755, 3922311484644960, 31536172739650, 53756397968691, 418382519719409532, 92435140906879260, 9064510622856810, 51380309438054415, 5188707998036472, 843521012352, 2628846496048, 36803851104370, 8262088988295, 149567938711620, 2562833809936800, 212687607855160, 6438408049239936, 172863751769112327|
|practice_0074|shared|0|2|209|0|11|220|100.20|657003598755, 346820883948, 3922311484644960, 418382519719409532, 1827999601539, 346820885496, 609333200433, 334401691341, 440163248904, 440163249044, 2866303374832128, 2842414375271, 270132653033, 53756397968691, 1755053180730, 3510106362348, 31536172739650, 9064510622856810, 138381532688484, 92435140906879260, 51380309438054415, 2628846496048, 843521012352, 5188707998036472, 36803851104370, 8262088988295, 617641565064, 2562833809936800, 149567938711620, 6438408049239936, 172863751769112327, 212687607855160|
|practice_0074|shared|48|0|253|333|0|253|2.12|270132653033, 334401691341, 346820883948, 346820885496, 440163248904, 440163249044, 609333200433, 617641565064, 657003598755, 843521012352, 1755053180730, 1827999601539, 2628846496048, 2842414375271, 3510106362348, 8262088988295, 31536172739650, 36803851104370, 53756397968691, 138381532688484, 149567938711620, 212687607855160, 2562833809936800, 2866303374832128, 3922311484644960, 5188707998036472, 6438408049239936, 9064510622856810, 51380309438054415, 92435140906879260, 172863751769112327, 418382519719409532|
|practice_0074|shared|48|1|216|227|0|216|5.99|346820885496, 346820883948, 138381532688484, 9064510622856810, 440163248904, 440163249044, 617641565064, 843521012352, 334401691341, 609333200433, 2866303374832128, 1827999601539, 270132653033, 657003598755, 31536172739650, 53756397968691, 3922311484644960, 5188707998036472, 418382519719409532, 51380309438054415, 3510106362348, 1755053180730, 2628846496048, 8262088988295, 36803851104370, 2842414375271, 92435140906879260, 6438408049239936, 2562833809936800, 212687607855160, 149567938711620, 172863751769112327|
|practice_0074|shared|48|2|228|235|0|228|39.68|657003598755, 609333200433, 1827999601539, 346820885496, 3922311484644960, 418382519719409532, 346820883948, 270132653033, 53756397968691, 138381532688484, 51380309438054415, 31536172739650, 843521012352, 617641565064, 334401691341, 440163249044, 1755053180730, 3510106362348, 440163248904, 92435140906879260, 36803851104370, 2842414375271, 6438408049239936, 2628846496048, 8262088988295, 2866303374832128, 172863751769112327, 149567938711620, 212687607855160, 5188707998036472, 2562833809936800, 9064510622856810|
|practice_0074|shared|128|0|202|205|7|209|7.99|270132653033, 334401691341, 346820883948, 346820885496, 440163248904, 440163249044, 609333200433, 617641565064, 657003598755, 843521012352, 1755053180730, 1827999601539, 2628846496048, 2842414375271, 3510106362348, 8262088988295, 31536172739650, 36803851104370, 53756397968691, 138381532688484, 149567938711620, 212687607855160, 2562833809936800, 2866303374832128, 3922311484644960, 5188707998036472, 6438408049239936, 9064510622856810, 51380309438054415, 92435140906879260, 172863751769112327, 418382519719409532|
|practice_0074|shared|128|1|196|200|0|196|28.74|346820885496, 346820883948, 138381532688484, 9064510622856810, 36803851104370, 2628846496048, 8262088988295, 270132653033, 440163249044, 440163248904, 609333200433, 1827999601539, 2866303374832128, 334401691341, 617641565064, 657003598755, 3922311484644960, 31536172739650, 53756397968691, 418382519719409532, 843521012352, 1755053180730, 3510106362348, 51380309438054415, 92435140906879260, 5188707998036472, 2842414375271, 149567938711620, 212687607855160, 2562833809936800, 6438408049239936, 172863751769112327|
|practice_0074|shared|128|2|188|192|0|188|170.20|657003598755, 609333200433, 346820883948, 3922311484644960, 440163248904, 1827999601539, 440163249044, 1755053180730, 3510106362348, 346820885496, 138381532688484, 2866303374832128, 270132653033, 2628846496048, 843521012352, 8262088988295, 334401691341, 2842414375271, 617641565064, 53756397968691, 31536172739650, 418382519719409532, 149567938711620, 212687607855160, 36803851104370, 5188707998036472, 6438408049239936, 92435140906879260, 51380309438054415, 9064510622856810, 2562833809936800, 172863751769112327|
|practice_0075|mixed|0|0|199|0|0|199|2.94|22479735, 44261649, 44261962, 60454723, 87157870, 132786013, 177047944, 423164327, 454255497, 464006007, 3022754077, 8953273212, 21578130140, 26582247780, 141246805817, 156875976567, 231391033836, 286617534258, 324877258622, 336350775958, 345345399340, 479864966907, 605361636362, 678035205306, 701914273994, 722133179398, 723895506643, 756934975407, 833744814918, 891630399521, 924617535641, 965860181832|
|practice_0075|mixed|0|1|195|0|6|201|7.98|22479735, 87157870, 44261649, 44261962, 132786013, 177047944, 60454723, 423164327, 454255497, 464006007, 3022754077, 8953273212, 21578130140, 722133179398, 833744814918, 678035205306, 26582247780, 141246805817, 156875976567, 231391033836, 286617534258, 324877258622, 336350775958, 345345399340, 479864966907, 723895506643, 965860181832, 605361636362, 756934975407, 701914273994, 891630399521, 924617535641|
|practice_0075|mixed|0|2|190|0|0|190|48.41|44261649, 60454723, 177047944, 22479735, 44261962, 3022754077, 454255497, 132786013, 423164327, 464006007, 345345399340, 678035205306, 8953273212, 87157870, 26582247780, 21578130140, 605361636362, 141246805817, 324877258622, 156875976567, 723895506643, 479864966907, 701914273994, 286617534258, 231391033836, 722133179398, 924617535641, 336350775958, 756934975407, 891630399521, 833744814918, 965860181832|
|practice_0075|mixed|48|0|206|287|0|206|1.50|22479735, 44261649, 44261962, 60454723, 87157870, 132786013, 177047944, 423164327, 454255497, 464006007, 3022754077, 8953273212, 21578130140, 26582247780, 141246805817, 156875976567, 231391033836, 286617534258, 324877258622, 336350775958, 345345399340, 479864966907, 605361636362, 678035205306, 701914273994, 722133179398, 723895506643, 756934975407, 833744814918, 891630399521, 924617535641, 965860181832|
|practice_0075|mixed|48|1|206|214|0|206|3.88|22479735, 87157870, 44261649, 44261962, 132786013, 177047944, 423164327, 464006007, 60454723, 3022754077, 8953273212, 26582247780, 454255497, 21578130140, 141246805817, 231391033836, 156875976567, 336350775958, 756934975407, 345345399340, 286617534258, 324877258622, 833744814918, 701914273994, 722133179398, 723895506643, 924617535641, 965860181832, 479864966907, 605361636362, 891630399521, 678035205306|
|practice_0075|mixed|48|2|199|209|0|199|27.59|44261649, 60454723, 177047944, 454255497, 44261962, 464006007, 132786013, 87157870, 423164327, 22479735, 723895506643, 324877258622, 286617534258, 231391033836, 21578130140, 3022754077, 722133179398, 479864966907, 141246805817, 345345399340, 336350775958, 701914273994, 8953273212, 26582247780, 678035205306, 156875976567, 891630399521, 605361636362, 756934975407, 924617535641, 965860181832, 833744814918|
|practice_0075|mixed|128|0|185|188|49|234|5.89|22479735, 44261649, 44261962, 60454723, 87157870, 132786013, 177047944, 423164327, 454255497, 464006007, 3022754077, 8953273212, 21578130140, 26582247780, 141246805817, 156875976567, 231391033836, 286617534258, 324877258622, 336350775958, 345345399340, 479864966907, 605361636362, 678035205306, 701914273994, 722133179398, 723895506643, 756934975407, 833744814918, 891630399521, 924617535641, 965860181832|
|practice_0075|mixed|128|1|186|188|24|210|14.35|22479735, 87157870, 44261649, 44261962, 132786013, 177047944, 60454723, 423164327, 454255497, 464006007, 3022754077, 8953273212, 21578130140, 26582247780, 141246805817, 156875976567, 324877258622, 231391033836, 345345399340, 286617534258, 965860181832, 336350775958, 479864966907, 605361636362, 891630399521, 678035205306, 833744814918, 701914273994, 722133179398, 756934975407, 723895506643, 924617535641|
|practice_0075|mixed|128|2|180|182|34|214|117.05|44261649, 60454723, 177047944, 22479735, 44261962, 3022754077, 454255497, 423164327, 132786013, 21578130140, 231391033836, 26582247780, 87157870, 156875976567, 464006007, 8953273212, 324877258622, 286617534258, 345345399340, 833744814918, 701914273994, 722133179398, 141246805817, 723895506643, 756934975407, 924617535641, 605361636362, 336350775958, 479864966907, 965860181832, 678035205306, 891630399521|
|practice_0076|independent|0|0|337|0|0|337|3.17|121032726310201757, 140000013356187442, 142902919847282118, 169046316089929462, 253450914388385793, 262012459111832440, 361604076446460844, 402113361482517976, 411345267613133715, 476808211076154336, 484210682012179282, 490432817980320794, 524765515046508117, 535855866364080796, 538818366771098452, 557476646551456154, 568842455950189663, 619708027296050988, 662469938780307577, 679825957063564490, 740611496649372316, 743836624866246506, 837707209992655942, 844221743074733635, 848992007476054992, 858835033488762763, 900487941613231248, 904177791460641608, 906435635052655080, 982210488365816258, 984387539520305594|
|practice_0076|independent|0|1|318|0|304|622|16.18|253450914388385793, 140000013356187442, 262012459111832440, 121032726310201757, 142902919847282118, 361604076446460844, 484210682012179282, 848992007476054992, 476808211076154336, 490432817980320794, 743836624866246506, 619708027296050988, 662469938780307577, 740611496649372316, 844221743074733635, 169046316089929462, 900487941613231248, 411345267613133715, 524765515046508117, 402113361482517976, 906435635052655080, 538818366771098452, 535855866364080796, 679825957063564490, 557476646551456154, 837707209992655942, 858835033488762763, 904177791460641608, 568842455950189663, 982210488365816258, 984387539520305594|
|practice_0076|independent|0|2|306|0|296|602|115.44|619708027296050988, 662469938780307577, 253450914388385793, 361604076446460844, 121032726310201757, 568842455950189663, 142902919847282118, 476808211076154336, 557476646551456154, 982210488365816258, 679825957063564490, 848992007476054992, 538818366771098452, 906435635052655080, 140000013356187442, 535855866364080796, 411345267613133715, 169046316089929462, 490432817980320794, 900487941613231248, 844221743074733635, 740611496649372316, 984387539520305594, 402113361482517976, 743836624866246506, 837707209992655942, 858835033488762763, 524765515046508117, 262012459111832440, 484210682012179282, 904177791460641608|
|practice_0076|independent|48|0|348|552|0|348|2.06|121032726310201757, 140000013356187442, 142902919847282118, 169046316089929462, 253450914388385793, 262012459111832440, 361604076446460844, 402113361482517976, 411345267613133715, 476808211076154336, 484210682012179282, 490432817980320794, 524765515046508117, 535855866364080796, 538818366771098452, 557476646551456154, 568842455950189663, 619708027296050988, 662469938780307577, 679825957063564490, 740611496649372316, 743836624866246506, 837707209992655942, 844221743074733635, 848992007476054992, 858835033488762763, 900487941613231248, 904177791460641608, 906435635052655080, 982210488365816258, 984387539520305594|
|practice_0076|independent|48|1|348|359|0|348|6.28|253450914388385793, 900487941613231248, 262012459111832440, 906435635052655080, 361604076446460844, 538818366771098452, 402113361482517976, 169046316089929462, 121032726310201757, 568842455950189663, 476808211076154336, 535855866364080796, 557476646551456154, 679825957063564490, 743836624866246506, 484210682012179282, 844221743074733635, 837707209992655942, 848992007476054992, 142902919847282118, 140000013356187442, 619708027296050988, 490432817980320794, 662469938780307577, 984387539520305594, 858835033488762763, 740611496649372316, 982210488365816258, 524765515046508117, 411345267613133715, 904177791460641608|
|practice_0076|independent|48|2|340|372|0|340|40.77|619708027296050988, 476808211076154336, 142902919847282118, 904177791460641608, 900487941613231248, 121032726310201757, 253450914388385793, 743836624866246506, 568842455950189663, 262012459111832440, 740611496649372316, 169046316089929462, 535855866364080796, 490432817980320794, 411345267613133715, 524765515046508117, 982210488365816258, 402113361482517976, 484210682012179282, 906435635052655080, 557476646551456154, 538818366771098452, 844221743074733635, 662469938780307577, 984387539520305594, 837707209992655942, 679825957063564490, 361604076446460844, 140000013356187442, 858835033488762763, 848992007476054992|
|practice_0076|independent|128|0|301|306|13|314|11.27|121032726310201757, 140000013356187442, 142902919847282118, 169046316089929462, 253450914388385793, 262012459111832440, 361604076446460844, 402113361482517976, 411345267613133715, 476808211076154336, 484210682012179282, 490432817980320794, 524765515046508117, 535855866364080796, 538818366771098452, 557476646551456154, 568842455950189663, 619708027296050988, 662469938780307577, 679825957063564490, 740611496649372316, 743836624866246506, 837707209992655942, 844221743074733635, 848992007476054992, 858835033488762763, 900487941613231248, 904177791460641608, 906435635052655080, 982210488365816258, 984387539520305594|
|practice_0076|independent|128|1|292|295|18|310|49.81|253450914388385793, 900487941613231248, 848992007476054992, 142902919847282118, 121032726310201757, 140000013356187442, 169046316089929462, 262012459111832440, 476808211076154336, 619708027296050988, 361604076446460844, 538818366771098452, 679825957063564490, 535855866364080796, 662469938780307577, 484210682012179282, 411345267613133715, 490432817980320794, 524765515046508117, 982210488365816258, 402113361482517976, 906435635052655080, 743836624866246506, 984387539520305594, 557476646551456154, 568842455950189663, 740611496649372316, 904177791460641608, 837707209992655942, 844221743074733635, 858835033488762763|
|practice_0076|independent|128|2|287|290|27|314|295.35|619708027296050988, 476808211076154336, 140000013356187442, 121032726310201757, 568842455950189663, 982210488365816258, 262012459111832440, 142902919847282118, 411345267613133715, 169046316089929462, 524765515046508117, 361604076446460844, 402113361482517976, 662469938780307577, 984387539520305594, 848992007476054992, 253450914388385793, 906435635052655080, 535855866364080796, 904177791460641608, 740611496649372316, 538818366771098452, 679825957063564490, 844221743074733635, 858835033488762763, 484210682012179282, 743836624866246506, 557476646551456154, 900487941613231248, 490432817980320794, 837707209992655942|
|practice_0077|multiplicative|0|0|115|0|0|115|2.75|5839, 6697, 8999, 13071, 17024, 24578, 49158, 49230, 81888, 193367, 194040, 363648, 367646, 395710, 523911, 686426, 867517, 1099496, 1277710, 4699490, 8240544, 26019794, 43378115, 43733130, 55060077, 301043750, 323235360, 727747534, 3975718957, 34685884800|
|practice_0077|multiplicative|0|1|115|0|0|115|3.52|17024, 24578, 49158, 49230, 367646, 5839, 6697, 363648, 8999, 13071, 81888, 193367, 194040, 395710, 523911, 686426, 867517, 1099496, 1277710, 4699490, 8240544, 26019794, 43378115, 43733130, 55060077, 301043750, 323235360, 34685884800, 727747534, 3975718957|
|practice_0077|multiplicative|0|2|108|0|0|108|22.14|24578, 5839, 49158, 17024, 6697, 8999, 13071, 363648, 395710, 193367, 367646, 49230, 81888, 194040, 523911, 686426, 867517, 1099496, 34685884800, 43733130, 43378115, 4699490, 1277710, 8240544, 26019794, 55060077, 301043750, 323235360, 727747534, 3975718957|
|practice_0077|multiplicative|48|0|116|140|0|116|1.06|5839, 6697, 8999, 13071, 17024, 24578, 49158, 49230, 81888, 193367, 194040, 363648, 367646, 395710, 523911, 686426, 867517, 1099496, 1277710, 4699490, 8240544, 26019794, 43378115, 43733130, 55060077, 301043750, 323235360, 727747534, 3975718957, 34685884800|
|practice_0077|multiplicative|48|1|115|122|0|115|1.44|17024, 24578, 49158, 49230, 367646, 5839, 6697, 363648, 8999, 13071, 81888, 193367, 194040, 395710, 523911, 686426, 1099496, 1277710, 8240544, 867517, 323235360, 4699490, 43733130, 43378115, 301043750, 34685884800, 26019794, 727747534, 55060077, 3975718957|
|practice_0077|multiplicative|48|2|112|119|0|112|8.97|24578, 5839, 367646, 49158, 17024, 6697, 8999, 49230, 81888, 194040, 13071, 686426, 523911, 363648, 1099496, 193367, 395710, 4699490, 1277710, 8240544, 867517, 43733130, 26019794, 43378115, 3975718957, 55060077, 323235360, 727747534, 301043750, 34685884800|
|practice_0077|multiplicative|128|0|110|112|0|110|3.61|5839, 6697, 8999, 13071, 17024, 24578, 49158, 49230, 81888, 193367, 194040, 363648, 367646, 395710, 523911, 686426, 867517, 1099496, 1277710, 4699490, 8240544, 26019794, 43378115, 43733130, 55060077, 301043750, 323235360, 727747534, 3975718957, 34685884800|
|practice_0077|multiplicative|128|1|112|113|0|112|4.32|17024, 24578, 49158, 49230, 367646, 5839, 6697, 363648, 8999, 13071, 81888, 193367, 194040, 395710, 523911, 686426, 867517, 1099496, 1277710, 4699490, 8240544, 26019794, 43378115, 43733130, 55060077, 301043750, 323235360, 3975718957, 34685884800, 727747534|
|practice_0077|multiplicative|128|2|108|110|0|108|25.36|24578, 5839, 367646, 49158, 17024, 6697, 8999, 49230, 395710, 13071, 363648, 867517, 686426, 81888, 193367, 1099496, 194040, 523911, 1277710, 8240544, 4699490, 26019794, 34685884800, 43378115, 301043750, 323235360, 727747534, 43733130, 55060077, 3975718957|
|practice_0078|mixed|0|0|284|0|0|284|3.70|22178363631, 43766305072, 43766306117, 44317833141, 44317833591, 72715699982, 73632036772, 590366325046, 3542197991982, 11216960319775, 22405505404964, 32649083102338, 59563167128192, 538709871310644, 553906369900366, 20027880409692416, 125276061742606853, 167055692052918066, 182219648692168559, 367546164908432887, 420323999186545811, 432217258543539444, 462266767341623042, 585373520840821717, 600156588172550740, 611560671779502598, 775054298671110756, 901762444675444984, 908013534933575596, 915466260532015031, 947560085575417942, 998263746689377336|
|practice_0078|mixed|0|1|266|0|105|371|17.41|22178363631, 44317833141, 44317833591, 59563167128192, 43766305072, 43766306117, 553906369900366, 73632036772, 72715699982, 3542197991982, 590366325046, 11216960319775, 22405505404964, 32649083102338, 538709871310644, 20027880409692416, 367546164908432887, 420323999186545811, 125276061742606853, 182219648692168559, 462266767341623042, 167055692052918066, 432217258543539444, 947560085575417942, 585373520840821717, 775054298671110756, 611560671779502598, 600156588172550740, 901762444675444984, 915466260532015031, 998263746689377336, 908013534933575596|
|practice_0078|mixed|0|2|267|0|191|458|123.99|43766306117, 44317833141, 22178363631, 44317833591, 3542197991982, 590366325046, 11216960319775, 73632036772, 43766305072, 538709871310644, 32649083102338, 20027880409692416, 59563167128192, 553906369900366, 72715699982, 901762444675444984, 167055692052918066, 585373520840821717, 22405505404964, 367546164908432887, 998263746689377336, 182219648692168559, 420323999186545811, 611560671779502598, 775054298671110756, 432217258543539444, 908013534933575596, 915466260532015031, 462266767341623042, 600156588172550740, 947560085575417942, 125276061742606853|
|practice_0078|mixed|48|0|285|403|0|285|1.94|22178363631, 43766305072, 43766306117, 44317833141, 44317833591, 72715699982, 73632036772, 590366325046, 3542197991982, 11216960319775, 22405505404964, 32649083102338, 59563167128192, 538709871310644, 553906369900366, 20027880409692416, 125276061742606853, 167055692052918066, 182219648692168559, 367546164908432887, 420323999186545811, 432217258543539444, 462266767341623042, 585373520840821717, 600156588172550740, 611560671779502598, 775054298671110756, 901762444675444984, 908013534933575596, 915466260532015031, 947560085575417942, 998263746689377336|
|practice_0078|mixed|48|1|279|330|0|279|6.47|22178363631, 44317833141, 44317833591, 59563167128192, 43766305072, 43766306117, 72715699982, 73632036772, 590366325046, 11216960319775, 3542197991982, 22405505404964, 32649083102338, 901762444675444984, 538709871310644, 20027880409692416, 182219648692168559, 462266767341623042, 553906369900366, 998263746689377336, 167055692052918066, 432217258543539444, 600156588172550740, 585373520840821717, 915466260532015031, 367546164908432887, 908013534933575596, 611560671779502598, 775054298671110756, 420323999186545811, 947560085575417942, 125276061742606853|
|practice_0078|mixed|48|2|294|310|0|294|43.76|43766305072, 44317833141, 43766306117, 44317833591, 11216960319775, 3542197991982, 553906369900366, 22405505404964, 538709871310644, 59563167128192, 167055692052918066, 22178363631, 72715699982, 462266767341623042, 32649083102338, 20027880409692416, 73632036772, 947560085575417942, 125276061742606853, 182219648692168559, 420323999186545811, 585373520840821717, 590366325046, 915466260532015031, 600156588172550740, 908013534933575596, 611560671779502598, 432217258543539444, 901762444675444984, 367546164908432887, 775054298671110756, 998263746689377336|
|practice_0078|mixed|128|0|254|260|22|276|10.16|22178363631, 43766305072, 43766306117, 44317833141, 44317833591, 72715699982, 73632036772, 590366325046, 3542197991982, 11216960319775, 22405505404964, 32649083102338, 59563167128192, 538709871310644, 553906369900366, 20027880409692416, 125276061742606853, 167055692052918066, 182219648692168559, 367546164908432887, 420323999186545811, 432217258543539444, 462266767341623042, 585373520840821717, 600156588172550740, 611560671779502598, 775054298671110756, 901762444675444984, 908013534933575596, 915466260532015031, 947560085575417942, 998263746689377336|
|practice_0078|mixed|128|1|244|249|31|275|36.07|22178363631, 44317833141, 44317833591, 59563167128192, 43766305072, 43766306117, 73632036772, 72715699982, 3542197991982, 590366325046, 11216960319775, 22405505404964, 32649083102338, 538709871310644, 553906369900366, 20027880409692416, 167055692052918066, 182219648692168559, 367546164908432887, 432217258543539444, 125276061742606853, 462266767341623042, 611560671779502598, 600156588172550740, 585373520840821717, 908013534933575596, 420323999186545811, 775054298671110756, 901762444675444984, 915466260532015031, 947560085575417942, 998263746689377336|
|practice_0078|mixed|128|2|247|253|16|263|259.00|43766305072, 44317833141, 43766306117, 59563167128192, 590366325046, 44317833591, 3542197991982, 11216960319775, 72715699982, 73632036772, 22178363631, 32649083102338, 553906369900366, 538709871310644, 22405505404964, 20027880409692416, 420323999186545811, 585373520840821717, 432217258543539444, 167055692052918066, 462266767341623042, 611560671779502598, 600156588172550740, 775054298671110756, 367546164908432887, 908013534933575596, 901762444675444984, 947560085575417942, 915466260532015031, 998263746689377336, 125276061742606853, 182219648692168559|
|practice_0079|multiplicative|0|0|138|0|0|138|2.87|63253, 126855, 132350, 136241, 193008, 300274, 1268254, 1467725, 6649960, 7914210, 8895177, 9172168, 15017592, 27526109, 31146953, 62848345, 78975136, 180066080, 213683375, 405902730, 1596095796, 2054118667, 2415934224, 5148024498, 5522130788, 47174771420, 544012158820, 2734454635924|
|practice_0079|multiplicative|0|1|135|0|0|135|4.94|126855, 63253, 132350, 136241, 193008, 1268254, 1467725, 300274, 7914210, 213683375, 6649960, 8895177, 9172168, 15017592, 27526109, 31146953, 62848345, 78975136, 180066080, 405902730, 2415934224, 1596095796, 2054118667, 5148024498, 5522130788, 47174771420, 544012158820, 2734454635924|
|practice_0079|multiplicative|0|2|128|0|0|128|30.68|126855, 193008, 300274, 1268254, 132350, 7914210, 9172168, 213683375, 63253, 15017592, 136241, 1467725, 27526109, 405902730, 78975136, 6649960, 2054118667, 5148024498, 8895177, 31146953, 2415934224, 62848345, 5522130788, 180066080, 47174771420, 544012158820, 1596095796, 2734454635924|
|practice_0079|multiplicative|48|0|144|183|0|144|1.29|63253, 126855, 132350, 136241, 193008, 300274, 1268254, 1467725, 6649960, 7914210, 8895177, 9172168, 15017592, 27526109, 31146953, 62848345, 78975136, 180066080, 213683375, 405902730, 1596095796, 2054118667, 2415934224, 5148024498, 5522130788, 47174771420, 544012158820, 2734454635924|
|practice_0079|multiplicative|48|1|138|156|0|138|2.28|126855, 63253, 132350, 136241, 193008, 1268254, 1467725, 300274, 6649960, 7914210, 15017592, 8895177, 9172168, 27526109, 31146953, 62848345, 78975136, 213683375, 180066080, 405902730, 2415934224, 2054118667, 1596095796, 47174771420, 5522130788, 5148024498, 544012158820, 2734454635924|
|practice_0079|multiplicative|48|2|133|144|0|133|13.07|126855, 193008, 136241, 132350, 300274, 15017592, 1467725, 7914210, 63253, 9172168, 31146953, 1268254, 213683375, 62848345, 8895177, 6649960, 405902730, 2415934224, 180066080, 1596095796, 5522130788, 2054118667, 544012158820, 78975136, 27526109, 2734454635924, 5148024498, 47174771420|
|practice_0079|multiplicative|128|0|129|132|0|129|4.00|63253, 126855, 132350, 136241, 193008, 300274, 1268254, 1467725, 6649960, 7914210, 8895177, 9172168, 15017592, 27526109, 31146953, 62848345, 78975136, 180066080, 213683375, 405902730, 1596095796, 2054118667, 2415934224, 5148024498, 5522130788, 47174771420, 544012158820, 2734454635924|
|practice_0079|multiplicative|128|1|134|136|0|134|7.91|126855, 63253, 132350, 136241, 193008, 1268254, 1467725, 300274, 7914210, 213683375, 6649960, 8895177, 9172168, 15017592, 27526109, 31146953, 62848345, 78975136, 180066080, 405902730, 1596095796, 2415934224, 5148024498, 2054118667, 5522130788, 47174771420, 544012158820, 2734454635924|
|practice_0079|multiplicative|128|2|123|126|0|123|38.38|126855, 193008, 136241, 132350, 300274, 7914210, 15017592, 63253, 1467725, 1268254, 8895177, 6649960, 27526109, 31146953, 213683375, 405902730, 62848345, 78975136, 2415934224, 9172168, 5148024498, 180066080, 5522130788, 1596095796, 2734454635924, 2054118667, 47174771420, 544012158820|
|practice_0080|multiplicative|0|0|124|0|0|124|2.57|6548, 7637, 8710, 15191, 18927, 26995, 27620, 44122, 44703, 57016, 76097, 77721, 93072, 109791, 110405, 134882, 186660, 197592, 507041, 1373182, 1376790, 6895280, 16527198, 19953360, 22027104, 99188231, 148402248, 151436595, 829260355, 4361695996, 222774167104, 222873355269|
|practice_0080|multiplicative|0|1|122|0|0|122|3.45|6548, 18927, 27620, 8710, 15191, 7637, 26995, 44122, 44703, 57016, 76097, 77721, 197592, 93072, 109791, 110405, 134882, 186660, 507041, 1373182, 1376790, 6895280, 19953360, 22027104, 16527198, 99188231, 148402248, 151436595, 829260355, 4361695996, 222774167104, 222873355269|
|practice_0080|multiplicative|0|2|121|0|0|121|29.25|6548, 26995, 27620, 134882, 15191, 44122, 8710, 18927, 44703, 7637, 109791, 57016, 77721, 76097, 110405, 186660, 507041, 1373182, 93072, 197592, 16527198, 22027104, 1376790, 6895280, 19953360, 151436595, 99188231, 222873355269, 222774167104, 148402248, 829260355, 4361695996|
|practice_0080|multiplicative|48|0|127|137|0|127|1.14|6548, 7637, 8710, 15191, 18927, 26995, 27620, 44122, 44703, 57016, 76097, 77721, 93072, 109791, 110405, 134882, 186660, 197592, 507041, 1373182, 1376790, 6895280, 16527198, 19953360, 22027104, 99188231, 148402248, 151436595, 829260355, 4361695996, 222774167104, 222873355269|
|practice_0080|multiplicative|48|1|125|132|0|125|1.89|6548, 18927, 27620, 8710, 15191, 7637, 26995, 44122, 44703, 57016, 76097, 77721, 197592, 93072, 109791, 110405, 134882, 186660, 507041, 19953360, 6895280, 1373182, 1376790, 16527198, 22027104, 151436595, 99188231, 148402248, 829260355, 4361695996, 222774167104, 222873355269|
|practice_0080|multiplicative|48|2|119|125|0|119|13.30|6548, 26995, 7637, 27620, 134882, 15191, 57016, 8710, 18927, 76097, 186660, 197592, 44703, 109791, 93072, 1373182, 110405, 6895280, 1376790, 44122, 77721, 16527198, 22027104, 829260355, 99188231, 222873355269, 222774167104, 4361695996, 507041, 19953360, 151436595, 148402248|
|practice_0080|multiplicative|128|0|122|123|0|122|4.22|6548, 7637, 8710, 15191, 18927, 26995, 27620, 44122, 44703, 57016, 76097, 77721, 93072, 109791, 110405, 134882, 186660, 197592, 507041, 1373182, 1376790, 6895280, 16527198, 19953360, 22027104, 99188231, 148402248, 151436595, 829260355, 4361695996, 222774167104, 222873355269|
|practice_0080|multiplicative|128|1|119|121|0|119|5.89|6548, 18927, 27620, 8710, 15191, 7637, 26995, 44122, 44703, 57016, 76097, 77721, 197592, 93072, 186660, 109791, 110405, 134882, 507041, 1373182, 1376790, 6895280, 16527198, 19953360, 22027104, 99188231, 148402248, 151436595, 829260355, 4361695996, 222774167104, 222873355269|
|practice_0080|multiplicative|128|2|113|115|0|113|34.64|6548, 26995, 7637, 27620, 134882, 15191, 57016, 18927, 8710, 77721, 186660, 197592, 93072, 44703, 44122, 1376790, 22027104, 76097, 109791, 507041, 16527198, 1373182, 110405, 6895280, 19953360, 99188231, 222774167104, 222873355269, 148402248, 151436595, 829260355, 4361695996|
|practice_0081|shared|0|0|203|0|0|203|4.23|1672860812, 1747774781, 2547321839, 2780558355, 2780558427, 2780558861, 3129759631, 3129764943, 11710055947, 13079607920, 27964303080, 33366725689, 37557086178, 66534662950, 230721847267, 588582366120, 791316289566, 994791626884, 1989583244088, 2576222818191, 3240332330352, 4806755996543, 22345422151416, 23685765420724, 106426005554625, 169658297263488, 295896904818240, 728698380102000, 24530804203840003, 38319491533180600|
|practice_0081|shared|0|1|189|0|0|189|11.57|1672860812, 11710055947, 1747774781, 27964303080, 2547321839, 295896904818240, 2780558427, 2780558355, 2780558861, 3129759631, 3129764943, 33366725689, 37557086178, 13079607920, 588582366120, 791316289566, 169658297263488, 66534662950, 23685765420724, 994791626884, 1989583244088, 230721847267, 2576222818191, 3240332330352, 4806755996543, 106426005554625, 22345422151416, 728698380102000, 24530804203840003, 38319491533180600|
|practice_0081|shared|0|2|175|0|0|175|65.42|2780558861, 3129764943, 2780558427, 1672860812, 2780558355, 3129759631, 2547321839, 295896904818240, 37557086178, 13079607920, 588582366120, 791316289566, 3240332330352, 1747774781, 1989583244088, 994791626884, 11710055947, 728698380102000, 169658297263488, 33366725689, 27964303080, 2576222818191, 23685765420724, 24530804203840003, 66534662950, 22345422151416, 4806755996543, 230721847267, 38319491533180600, 106426005554625|
|practice_0081|shared|48|0|204|229|0|204|1.61|1672860812, 1747774781, 2547321839, 2780558355, 2780558427, 2780558861, 3129759631, 3129764943, 11710055947, 13079607920, 27964303080, 33366725689, 37557086178, 66534662950, 230721847267, 588582366120, 791316289566, 994791626884, 1989583244088, 2576222818191, 3240332330352, 4806755996543, 22345422151416, 23685765420724, 106426005554625, 169658297263488, 295896904818240, 728698380102000, 24530804203840003, 38319491533180600|
|practice_0081|shared|48|1|199|232|0|199|3.98|1672860812, 11710055947, 1747774781, 2780558355, 2780558427, 2780558861, 2547321839, 295896904818240, 3129764943, 3129759631, 13079607920, 588582366120, 33366725689, 37557086178, 27964303080, 230721847267, 791316289566, 66534662950, 2576222818191, 994791626884, 1989583244088, 22345422151416, 4806755996543, 3240332330352, 728698380102000, 169658297263488, 23685765420724, 106426005554625, 24530804203840003, 38319491533180600|
|practice_0081|shared|48|2|199|210|0|199|30.85|2780558861, 3129764943, 3129759631, 2780558427, 2780558355, 1672860812, 13079607920, 27964303080, 11710055947, 1747774781, 33366725689, 37557086178, 2547321839, 1989583244088, 994791626884, 295896904818240, 791316289566, 588582366120, 230721847267, 2576222818191, 66534662950, 4806755996543, 169658297263488, 106426005554625, 23685765420724, 22345422151416, 24530804203840003, 38319491533180600, 3240332330352, 728698380102000|
|practice_0081|shared|128|0|179|191|33|212|6.98|1672860812, 1747774781, 2547321839, 2780558355, 2780558427, 2780558861, 3129759631, 3129764943, 11710055947, 13079607920, 27964303080, 33366725689, 37557086178, 66534662950, 230721847267, 588582366120, 791316289566, 994791626884, 1989583244088, 2576222818191, 3240332330352, 4806755996543, 22345422151416, 23685765420724, 106426005554625, 169658297263488, 295896904818240, 728698380102000, 24530804203840003, 38319491533180600|
|practice_0081|shared|128|1|171|172|2|173|17.78|1672860812, 11710055947, 1747774781, 2780558355, 2780558427, 2780558861, 2547321839, 295896904818240, 3129764943, 3129759631, 13079607920, 588582366120, 27964303080, 33366725689, 37557086178, 66534662950, 791316289566, 994791626884, 1989583244088, 728698380102000, 22345422151416, 169658297263488, 230721847267, 3240332330352, 4806755996543, 2576222818191, 23685765420724, 24530804203840003, 106426005554625, 38319491533180600|
|practice_0081|shared|128|2|162|167|0|162|106.77|2780558861, 3129764943, 2780558427, 3129759631, 1672860812, 13079607920, 2780558355, 37557086178, 11710055947, 66534662950, 169658297263488, 2547321839, 295896904818240, 588582366120, 1747774781, 3240332330352, 27964303080, 33366725689, 791316289566, 22345422151416, 230721847267, 994791626884, 1989583244088, 728698380102000, 23685765420724, 24530804203840003, 4806755996543, 106426005554625, 38319491533180600, 2576222818191|
|practice_0082|mixed|0|0|243|0|0|243|3.88|125065031, 302457432, 1583451496, 1613989603, 1695779142, 1695779208, 1695779220, 2102056100, 4180209447, 4260826377, 68927767890, 305606437112, 5712067840224, 48552577926288, 150544332264013, 167890710916120, 201077263185159, 293831378093870, 304445120674711, 517345921746405, 521601572057051, 568016012462727, 605510119959146, 634347413737559, 670140269820952, 698884675896161, 800342796173875, 908240677041003, 942640489437206, 946525244010372, 1632577946475000|
|practice_0082|mixed|0|1|224|0|0|224|12.47|125065031, 1613989603, 302457432, 1695779208, 1695779142, 1695779220, 1583451496, 2102056100, 4260826377, 4180209447, 68927767890, 305606437112, 5712067840224, 48552577926288, 304445120674711, 150544332264013, 167890710916120, 201077263185159, 521601572057051, 517345921746405, 568016012462727, 1632577946475000, 293831378093870, 698884675896161, 605510119959146, 634347413737559, 670140269820952, 800342796173875, 908240677041003, 942640489437206, 946525244010372|
|practice_0082|mixed|0|2|217|0|32|249|78.81|1695779220, 1583451496, 125065031, 1695779208, 2102056100, 1695779142, 4260826377, 68927767890, 302457432, 305606437112, 1613989603, 4180209447, 5712067840224, 48552577926288, 517345921746405, 568016012462727, 1632577946475000, 800342796173875, 167890710916120, 942640489437206, 698884675896161, 304445120674711, 201077263185159, 670140269820952, 634347413737559, 521601572057051, 946525244010372, 605510119959146, 150544332264013, 908240677041003, 293831378093870|
|practice_0082|mixed|48|0|238|274|0|238|1.78|125065031, 302457432, 1583451496, 1613989603, 1695779142, 1695779208, 1695779220, 2102056100, 4180209447, 4260826377, 68927767890, 305606437112, 5712067840224, 48552577926288, 150544332264013, 167890710916120, 201077263185159, 293831378093870, 304445120674711, 517345921746405, 521601572057051, 568016012462727, 605510119959146, 634347413737559, 670140269820952, 698884675896161, 800342796173875, 908240677041003, 942640489437206, 946525244010372, 1632577946475000|
|practice_0082|mixed|48|1|231|244|0|231|4.36|125065031, 302457432, 1583451496, 1613989603, 1695779142, 1695779220, 1695779208, 4180209447, 2102056100, 68927767890, 4260826377, 305606437112, 304445120674711, 201077263185159, 5712067840224, 517345921746405, 48552577926288, 167890710916120, 1632577946475000, 946525244010372, 568016012462727, 670140269820952, 634347413737559, 150544332264013, 293831378093870, 605510119959146, 908240677041003, 521601572057051, 698884675896161, 942640489437206, 800342796173875|
|practice_0082|mixed|48|2|238|242|0|238|27.36|1695779220, 302457432, 1695779142, 1695779208, 2102056100, 4260826377, 125065031, 4180209447, 1613989603, 942640489437206, 201077263185159, 48552577926288, 5712067840224, 1583451496, 568016012462727, 68927767890, 800342796173875, 908240677041003, 634347413737559, 698884675896161, 305606437112, 304445120674711, 605510119959146, 293831378093870, 167890710916120, 517345921746405, 150544332264013, 521601572057051, 670140269820952, 1632577946475000, 946525244010372|
|practice_0082|mixed|128|0|219|227|36|255|5.63|125065031, 302457432, 1583451496, 1613989603, 1695779142, 1695779208, 1695779220, 2102056100, 4180209447, 4260826377, 68927767890, 305606437112, 5712067840224, 48552577926288, 150544332264013, 167890710916120, 201077263185159, 293831378093870, 304445120674711, 517345921746405, 521601572057051, 568016012462727, 605510119959146, 634347413737559, 670140269820952, 698884675896161, 800342796173875, 908240677041003, 942640489437206, 946525244010372, 1632577946475000|
|practice_0082|mixed|128|1|209|211|0|209|18.22|125065031, 302457432, 1583451496, 1613989603, 1695779142, 1695779208, 1695779220, 4260826377, 4180209447, 2102056100, 68927767890, 305606437112, 5712067840224, 201077263185159, 48552577926288, 167890710916120, 521601572057051, 946525244010372, 517345921746405, 293831378093870, 304445120674711, 150544332264013, 605510119959146, 670140269820952, 568016012462727, 634347413737559, 698884675896161, 800342796173875, 908240677041003, 1632577946475000, 942640489437206|
|practice_0082|mixed|128|2|204|206|42|246|118.30|1695779220, 302457432, 125065031, 4180209447, 1695779142, 1695779208, 1583451496, 2102056100, 5712067840224, 1613989603, 4260826377, 305606437112, 68927767890, 48552577926288, 167890710916120, 517345921746405, 521601572057051, 304445120674711, 634347413737559, 670140269820952, 293831378093870, 946525244010372, 698884675896161, 800342796173875, 942640489437206, 908240677041003, 605510119959146, 1632577946475000, 201077263185159, 150544332264013, 568016012462727|
|practice_0083|shared|0|0|155|0|0|155|2.60|56934058, 59257670, 59258437, 59258614, 59258679, 59613137, 296288645, 476904972, 834580894, 894197087, 1125915117, 1366417536, 1489140675, 1718501513, 1737319621, 2562032610, 3437000888, 4782460922, 5441066868, 6438223008, 7321561403, 10666558692, 11319105340, 29273782356, 56394056928, 58572039370, 159998252400, 219968187152, 422955404798, 5549246913193, 6317639236788|
|practice_0083|shared|0|1|147|0|0|147|5.47|1366417536, 56934058, 2562032610, 4782460922, 59257670, 59258679, 59258614, 59258437, 296288645, 1125915117, 29273782356, 10666558692, 59613137, 894197087, 476904972, 834580894, 1718501513, 1489140675, 3437000888, 6438223008, 56394056928, 159998252400, 219968187152, 1737319621, 11319105340, 58572039370, 5441066868, 7321561403, 422955404798, 5549246913193, 6317639236788|
|practice_0083|shared|0|2|141|0|0|141|34.65|59258679, 59258614, 1718501513, 59257670, 296288645, 59258437, 1125915117, 59613137, 476904972, 894197087, 3437000888, 10666558692, 56934058, 2562032610, 29273782356, 6438223008, 56394056928, 834580894, 159998252400, 4782460922, 1366417536, 1489140675, 11319105340, 219968187152, 58572039370, 1737319621, 422955404798, 5441066868, 6317639236788, 7321561403, 5549246913193|
|practice_0083|shared|48|0|161|185|0|161|1.11|56934058, 59257670, 59258437, 59258614, 59258679, 59613137, 296288645, 476904972, 834580894, 894197087, 1125915117, 1366417536, 1489140675, 1718501513, 1737319621, 2562032610, 3437000888, 4782460922, 5441066868, 6438223008, 7321561403, 10666558692, 11319105340, 29273782356, 56394056928, 58572039370, 159998252400, 219968187152, 422955404798, 5549246913193, 6317639236788|
|practice_0083|shared|48|1|153|159|0|153|2.75|1366417536, 56934058, 2562032610, 4782460922, 59257670, 59258679, 59258614, 59258437, 296288645, 59613137, 894197087, 476904972, 834580894, 1125915117, 1718501513, 3437000888, 10666558692, 29273782356, 56394056928, 1489140675, 6438223008, 159998252400, 11319105340, 1737319621, 5441066868, 219968187152, 7321561403, 58572039370, 422955404798, 6317639236788, 5549246913193|
|practice_0083|shared|48|2|153|162|0|153|19.93|59258679, 59258614, 59258437, 1718501513, 59257670, 1125915117, 219968187152, 59613137, 894197087, 296288645, 6438223008, 3437000888, 476904972, 834580894, 56934058, 56394056928, 1366417536, 2562032610, 10666558692, 1737319621, 7321561403, 5441066868, 159998252400, 422955404798, 4782460922, 29273782356, 58572039370, 1489140675, 5549246913193, 11319105340, 6317639236788|
|practice_0083|shared|128|0|141|147|0|141|4.66|56934058, 59257670, 59258437, 59258614, 59258679, 59613137, 296288645, 476904972, 834580894, 894197087, 1125915117, 1366417536, 1489140675, 1718501513, 1737319621, 2562032610, 3437000888, 4782460922, 5441066868, 6438223008, 7321561403, 10666558692, 11319105340, 29273782356, 56394056928, 58572039370, 159998252400, 219968187152, 422955404798, 5549246913193, 6317639236788|
|practice_0083|shared|128|1|139|142|6|145|9.23|1366417536, 56934058, 2562032610, 4782460922, 59257670, 59258679, 59258614, 59258437, 296288645, 1125915117, 29273782356, 10666558692, 59613137, 894197087, 476904972, 834580894, 1489140675, 1718501513, 3437000888, 6438223008, 159998252400, 1737319621, 56394056928, 219968187152, 5441066868, 7321561403, 11319105340, 58572039370, 422955404798, 6317639236788, 5549246913193|
|practice_0083|shared|128|2|137|140|0|137|50.92|59258679, 59258614, 59258437, 1718501513, 59257670, 1125915117, 10666558692, 56934058, 2562032610, 59613137, 894197087, 56394056928, 6438223008, 296288645, 476904972, 1366417536, 159998252400, 3437000888, 29273782356, 834580894, 4782460922, 1489140675, 219968187152, 1737319621, 422955404798, 6317639236788, 11319105340, 5441066868, 7321561403, 58572039370, 5549246913193|
|practice_0084|shared|0|0|168|0|0|168|3.60|214939802708, 362866681620, 431049286850, 431049294073, 612600505455, 727708074196, 727708077832, 727708081473, 727708082464, 864444353680, 1719518363641, 1728888718104, 1728888718242, 2183124246762, 2450402021180, 3009157136398, 5080133510984, 10187913140195, 12069380233788, 12102221040367, 13098745465923, 20690366117632, 97417140457786, 386090147331468, 1317151627202772, 3114047757028333, 23708729234345404, 24331870552229856, 148434244045523910|
|practice_0084|shared|0|1|150|0|0|150|10.46|1719518363641, 3009157136398, 214939802708, 431049286850, 431049294073, 12069380233788, 20690366117632, 97417140457786, 24331870552229856, 362866681620, 5080133510984, 386090147331468, 612600505455, 2450402021180, 727708082464, 727708074196, 727708077832, 727708081473, 2183124246762, 10187913140195, 13098745465923, 1317151627202772, 3114047757028333, 23708729234345404, 864444353680, 1728888718242, 1728888718104, 12102221040367, 148434244045523910|
|practice_0084|shared|0|2|152|0|0|152|53.22|727708081473, 13098745465923, 1719518363641, 727708074196, 727708082464, 20690366117632, 431049294073, 214939802708, 3009157136398, 431049286850, 12069380233788, 864444353680, 1728888718104, 1728888718242, 727708077832, 12102221040367, 612600505455, 2183124246762, 2450402021180, 10187913140195, 3114047757028333, 362866681620, 24331870552229856, 5080133510984, 97417140457786, 148434244045523910, 1317151627202772, 23708729234345404, 386090147331468|
|practice_0084|shared|48|0|206|374|0|206|1.60|214939802708, 362866681620, 431049286850, 431049294073, 612600505455, 727708074196, 727708077832, 727708081473, 727708082464, 864444353680, 1719518363641, 1728888718104, 1728888718242, 2183124246762, 2450402021180, 3009157136398, 5080133510984, 10187913140195, 12069380233788, 12102221040367, 13098745465923, 20690366117632, 97417140457786, 386090147331468, 1317151627202772, 3114047757028333, 23708729234345404, 24331870552229856, 148434244045523910|
|practice_0084|shared|48|1|157|171|0|157|5.05|1719518363641, 3009157136398, 214939802708, 431049286850, 431049294073, 12069380233788, 20690366117632, 97417140457786, 3114047757028333, 362866681620, 5080133510984, 612600505455, 2450402021180, 24331870552229856, 386090147331468, 727708077832, 727708074196, 727708082464, 727708081473, 2183124246762, 10187913140195, 13098745465923, 1317151627202772, 864444353680, 1728888718104, 1728888718242, 12102221040367, 23708729234345404, 148434244045523910|
|practice_0084|shared|48|2|152|166|0|152|23.39|727708081473, 13098745465923, 727708074196, 1719518363641, 214939802708, 2183124246762, 727708082464, 362866681620, 727708077832, 3114047757028333, 10187913140195, 431049294073, 431049286850, 23708729234345404, 3009157136398, 1728888718104, 1728888718242, 864444353680, 97417140457786, 2450402021180, 12069380233788, 5080133510984, 24331870552229856, 1317151627202772, 612600505455, 20690366117632, 386090147331468, 148434244045523910, 12102221040367|
|practice_0084|shared|128|0|158|163|2|160|5.43|214939802708, 362866681620, 431049286850, 431049294073, 612600505455, 727708074196, 727708077832, 727708081473, 727708082464, 864444353680, 1719518363641, 1728888718104, 1728888718242, 2183124246762, 2450402021180, 3009157136398, 5080133510984, 10187913140195, 12069380233788, 12102221040367, 13098745465923, 20690366117632, 97417140457786, 386090147331468, 1317151627202772, 3114047757028333, 23708729234345404, 24331870552229856, 148434244045523910|
|practice_0084|shared|128|1|145|149|0|145|14.61|1719518363641, 3009157136398, 214939802708, 431049286850, 431049294073, 12069380233788, 20690366117632, 97417140457786, 24331870552229856, 362866681620, 5080133510984, 386090147331468, 612600505455, 2450402021180, 1728888718104, 1728888718242, 864444353680, 12102221040367, 727708081473, 727708077832, 727708074196, 727708082464, 2183124246762, 10187913140195, 13098745465923, 3114047757028333, 1317151627202772, 148434244045523910, 23708729234345404|
|practice_0084|shared|128|2|145|147|15|160|74.72|727708081473, 13098745465923, 1719518363641, 727708074196, 3009157136398, 3114047757028333, 727708082464, 431049286850, 12069380233788, 431049294073, 727708077832, 362866681620, 97417140457786, 2183124246762, 10187913140195, 214939802708, 864444353680, 1728888718242, 1728888718104, 12102221040367, 2450402021180, 612600505455, 5080133510984, 20690366117632, 148434244045523910, 386090147331468, 23708729234345404, 1317151627202772, 24331870552229856|
|practice_0085|mixed|0|0|304|0|0|304|3.13|15624834220, 17408427482, 218747886593, 890616402824, 1818266294173, 3239545542674, 33426056283312, 37968383063475, 58242370834515, 103777651274715, 238972138120480, 360406349611053, 421967650413176, 840039015958282, 6591451942707315, 51400144829640599, 109437934602547936, 146460362920059099, 148653095924137494, 262559716287840971, 316777539231692342, 332916161490408878, 367073702051932678, 409355015188247262, 602510952758535363, 623186533847792485, 697093581344681066, 699134246929068657, 702871733542676587, 794089593950057693, 809289925174831616, 883995109806444736|
|practice_0085|mixed|0|1|280|0|22|302|18.98|15624834220, 218747886593, 890616402824, 17408427482, 37968383063475, 1818266294173, 3239545542674, 840039015958282, 6591451942707315, 33426056283312, 58242370834515, 238972138120480, 360406349611053, 421967650413176, 103777651274715, 883995109806444736, 51400144829640599, 809289925174831616, 262559716287840971, 109437934602547936, 146460362920059099, 148653095924137494, 316777539231692342, 367073702051932678, 409355015188247262, 332916161490408878, 602510952758535363, 623186533847792485, 794089593950057693, 699134246929068657, 697093581344681066, 702871733542676587|
|practice_0085|mixed|0|2|280|0|107|387|122.03|218747886593, 15624834220, 17408427482, 890616402824, 6591451942707315, 37968383063475, 33426056283312, 3239545542674, 238972138120480, 421967650413176, 360406349611053, 1818266294173, 148653095924137494, 840039015958282, 58242370834515, 103777651274715, 332916161490408878, 51400144829640599, 109437934602547936, 146460362920059099, 697093581344681066, 602510952758535363, 262559716287840971, 699134246929068657, 623186533847792485, 809289925174831616, 883995109806444736, 316777539231692342, 794089593950057693, 409355015188247262, 367073702051932678, 702871733542676587|
|practice_0085|mixed|48|0|304|404|0|304|1.58|15624834220, 17408427482, 218747886593, 890616402824, 1818266294173, 3239545542674, 33426056283312, 37968383063475, 58242370834515, 103777651274715, 238972138120480, 360406349611053, 421967650413176, 840039015958282, 6591451942707315, 51400144829640599, 109437934602547936, 146460362920059099, 148653095924137494, 262559716287840971, 316777539231692342, 332916161490408878, 367073702051932678, 409355015188247262, 602510952758535363, 623186533847792485, 697093581344681066, 699134246929068657, 702871733542676587, 794089593950057693, 809289925174831616, 883995109806444736|
|practice_0085|mixed|48|1|285|317|0|285|7.63|15624834220, 890616402824, 17408427482, 218747886593, 3239545542674, 1818266294173, 33426056283312, 58242370834515, 37968383063475, 103777651274715, 238972138120480, 840039015958282, 360406349611053, 421967650413176, 6591451942707315, 109437934602547936, 51400144829640599, 809289925174831616, 409355015188247262, 697093581344681066, 699134246929068657, 148653095924137494, 262559716287840971, 702871733542676587, 332916161490408878, 146460362920059099, 883995109806444736, 367073702051932678, 602510952758535363, 316777539231692342, 794089593950057693, 623186533847792485|
|practice_0085|mixed|48|2|301|323|0|301|43.55|218747886593, 15624834220, 3239545542674, 890616402824, 37968383063475, 1818266294173, 421967650413176, 58242370834515, 109437934602547936, 840039015958282, 6591451942707315, 623186533847792485, 146460362920059099, 17408427482, 794089593950057693, 238972138120480, 360406349611053, 103777651274715, 33426056283312, 316777539231692342, 332916161490408878, 602510952758535363, 262559716287840971, 409355015188247262, 883995109806444736, 148653095924137494, 809289925174831616, 702871733542676587, 51400144829640599, 697093581344681066, 699134246929068657, 367073702051932678|
|practice_0085|mixed|128|0|259|267|16|275|9.21|15624834220, 17408427482, 218747886593, 890616402824, 1818266294173, 3239545542674, 33426056283312, 37968383063475, 58242370834515, 103777651274715, 238972138120480, 360406349611053, 421967650413176, 840039015958282, 6591451942707315, 51400144829640599, 109437934602547936, 146460362920059099, 148653095924137494, 262559716287840971, 316777539231692342, 332916161490408878, 367073702051932678, 409355015188247262, 602510952758535363, 623186533847792485, 697093581344681066, 699134246929068657, 702871733542676587, 794089593950057693, 809289925174831616, 883995109806444736|
|practice_0085|mixed|128|1|258|261|7|265|42.19|15624834220, 890616402824, 17408427482, 218747886593, 3239545542674, 37968383063475, 1818266294173, 840039015958282, 33426056283312, 58242370834515, 103777651274715, 238972138120480, 360406349611053, 421967650413176, 6591451942707315, 51400144829640599, 332916161490408878, 109437934602547936, 809289925174831616, 883995109806444736, 367073702051932678, 146460362920059099, 148653095924137494, 316777539231692342, 262559716287840971, 602510952758535363, 409355015188247262, 697093581344681066, 699134246929068657, 794089593950057693, 623186533847792485, 702871733542676587|
|practice_0085|mixed|128|2|255|258|64|319|290.06|218747886593, 15624834220, 3239545542674, 1818266294173, 890616402824, 421967650413176, 17408427482, 840039015958282, 238972138120480, 33426056283312, 37968383063475, 360406349611053, 58242370834515, 103777651274715, 6591451942707315, 262559716287840971, 109437934602547936, 51400144829640599, 148653095924137494, 702871733542676587, 809289925174831616, 883995109806444736, 332916161490408878, 409355015188247262, 697093581344681066, 146460362920059099, 367073702051932678, 602510952758535363, 794089593950057693, 623186533847792485, 316777539231692342, 699134246929068657|
|practice_0086|shared|0|0|153|0|0|153|3.02|986599, 5349988, 7377251, 7378979, 13216342, 22138657, 26735989, 51066036, 57362228, 92517348, 158839688, 165370104, 261179868, 464010158, 475791039, 773080718, 951586416, 1188900575, 2655119874, 7639131798, 9292847598, 12373795980, 30556517944, 30558134032, 31918343744, 477684937184, 662642213623, 3779678074176, 13794117107604|
|practice_0086|shared|0|1|146|0|0|146|5.39|986599, 7377251, 7378979, 22138657, 5349988, 13216342, 92517348, 158839688, 26735989, 951586416, 51066036, 57362228, 165370104, 261179868, 475791039, 464010158, 477684937184, 30556517944, 7639131798, 12373795980, 30558134032, 773080718, 1188900575, 2655119874, 9292847598, 31918343744, 662642213623, 3779678074176, 13794117107604|
|practice_0086|shared|0|2|146|0|0|146|34.13|7378979, 13216342, 475791039, 7377251, 986599, 951586416, 92517348, 22138657, 7639131798, 30556517944, 5349988, 26735989, 2655119874, 57362228, 158839688, 165370104, 30558134032, 477684937184, 31918343744, 51066036, 464010158, 261179868, 9292847598, 773080718, 1188900575, 12373795980, 3779678074176, 13794117107604, 662642213623|
|practice_0086|shared|48|0|156|186|0|156|1.30|986599, 5349988, 7377251, 7378979, 13216342, 22138657, 26735989, 51066036, 57362228, 92517348, 158839688, 165370104, 261179868, 464010158, 475791039, 773080718, 951586416, 1188900575, 2655119874, 7639131798, 9292847598, 12373795980, 30556517944, 30558134032, 31918343744, 477684937184, 662642213623, 3779678074176, 13794117107604|
|practice_0086|shared|48|1|148|160|0|148|2.49|986599, 7377251, 7378979, 22138657, 5349988, 13216342, 92517348, 158839688, 51066036, 165370104, 261179868, 26735989, 30558134032, 30556517944, 7639131798, 951586416, 57362228, 475791039, 2655119874, 464010158, 1188900575, 773080718, 477684937184, 31918343744, 9292847598, 12373795980, 662642213623, 3779678074176, 13794117107604|
|practice_0086|shared|48|2|150|161|0|150|15.06|7378979, 477684937184, 986599, 7377251, 158839688, 5349988, 13216342, 951586416, 30556517944, 7639131798, 26735989, 30558134032, 165370104, 22138657, 92517348, 2655119874, 261179868, 57362228, 31918343744, 12373795980, 9292847598, 475791039, 773080718, 464010158, 13794117107604, 51066036, 1188900575, 662642213623, 3779678074176|
|practice_0086|shared|128|0|146|150|21|167|4.90|986599, 5349988, 7377251, 7378979, 13216342, 22138657, 26735989, 51066036, 57362228, 92517348, 158839688, 165370104, 261179868, 464010158, 475791039, 773080718, 951586416, 1188900575, 2655119874, 7639131798, 9292847598, 12373795980, 30556517944, 30558134032, 31918343744, 477684937184, 662642213623, 3779678074176, 13794117107604|
|practice_0086|shared|128|1|140|143|0|140|9.70|986599, 7377251, 7378979, 22138657, 5349988, 13216342, 92517348, 158839688, 26735989, 51066036, 57362228, 165370104, 261179868, 464010158, 475791039, 951586416, 773080718, 1188900575, 30558134032, 30556517944, 7639131798, 9292847598, 2655119874, 31918343744, 477684937184, 12373795980, 662642213623, 3779678074176, 13794117107604|
|practice_0086|shared|128|2|141|144|14|155|56.23|7378979, 477684937184, 986599, 7377251, 5349988, 951586416, 13216342, 30556517944, 7639131798, 92517348, 475791039, 158839688, 30558134032, 22138657, 26735989, 51066036, 165370104, 773080718, 261179868, 9292847598, 57362228, 464010158, 31918343744, 12373795980, 1188900575, 2655119874, 662642213623, 3779678074176, 13794117107604|
|practice_0087|mixed|0|0|255|0|0|255|3.26|228438854514, 228438856484, 444191287471, 444191290778, 444191294484, 469054218537, 469054225414, 469054229390, 5025654805172, 5939410216870, 1749572289485380, 2697864600811280, 6804875038469815, 66492657973358813, 133917248077122279, 202919113369677971, 203622275463572720, 237952896022508612, 247970022268506229, 356340179645825617, 384175020508201436, 393209443394965908, 669642033504956109, 732918045195987065, 751051099822368848, 751310836657643440, 796091087130310491, 827831894447955669, 901500367437635908|
|practice_0087|mixed|0|1|249|0|3|252|14.09|228438854514, 5939410216870, 228438856484, 5025654805172, 444191290778, 444191294484, 444191287471, 469054218537, 469054225414, 469054229390, 1749572289485380, 2697864600811280, 6804875038469815, 66492657973358813, 384175020508201436, 133917248077122279, 393209443394965908, 202919113369677971, 751310836657643440, 203622275463572720, 751051099822368848, 247970022268506229, 356340179645825617, 669642033504956109, 827831894447955669, 237952896022508612, 901500367437635908, 796091087130310491, 732918045195987065|
|practice_0087|mixed|0|2|240|0|97|337|98.06|228438856484, 444191287471, 469054225414, 5025654805172, 228438854514, 469054229390, 5939410216870, 469054218537, 444191294484, 444191290778, 6804875038469815, 66492657973358813, 237952896022508612, 202919113369677971, 247970022268506229, 2697864600811280, 203622275463572720, 1749572289485380, 751051099822368848, 133917248077122279, 356340179645825617, 393209443394965908, 751310836657643440, 827831894447955669, 384175020508201436, 669642033504956109, 732918045195987065, 796091087130310491, 901500367437635908|
|practice_0087|mixed|48|0|264|350|0|264|1.45|228438854514, 228438856484, 444191287471, 444191290778, 444191294484, 469054218537, 469054225414, 469054229390, 5025654805172, 5939410216870, 1749572289485380, 2697864600811280, 6804875038469815, 66492657973358813, 133917248077122279, 202919113369677971, 203622275463572720, 237952896022508612, 247970022268506229, 356340179645825617, 384175020508201436, 393209443394965908, 669642033504956109, 732918045195987065, 751051099822368848, 751310836657643440, 796091087130310491, 827831894447955669, 901500367437635908|
|practice_0087|mixed|48|1|249|296|0|249|5.70|228438854514, 5939410216870, 228438856484, 5025654805172, 444191287471, 444191290778, 444191294484, 469054229390, 469054225414, 469054218537, 6804875038469815, 1749572289485380, 203622275463572720, 2697864600811280, 356340179645825617, 202919113369677971, 237952896022508612, 247970022268506229, 66492657973358813, 732918045195987065, 384175020508201436, 133917248077122279, 393209443394965908, 751310836657643440, 751051099822368848, 901500367437635908, 669642033504956109, 827831894447955669, 796091087130310491|
|practice_0087|mixed|48|2|256|274|0|256|35.75|228438856484, 5025654805172, 228438854514, 444191294484, 5939410216870, 469054218537, 469054229390, 247970022268506229, 469054225414, 444191287471, 66492657973358813, 2697864600811280, 6804875038469815, 203622275463572720, 444191290778, 1749572289485380, 356340179645825617, 133917248077122279, 669642033504956109, 202919113369677971, 827831894447955669, 732918045195987065, 796091087130310491, 901500367437635908, 384175020508201436, 393209443394965908, 751310836657643440, 751051099822368848, 237952896022508612|
|practice_0087|mixed|128|0|229|238|15|244|8.50|228438854514, 228438856484, 444191287471, 444191290778, 444191294484, 469054218537, 469054225414, 469054229390, 5025654805172, 5939410216870, 1749572289485380, 2697864600811280, 6804875038469815, 66492657973358813, 133917248077122279, 202919113369677971, 203622275463572720, 237952896022508612, 247970022268506229, 356340179645825617, 384175020508201436, 393209443394965908, 669642033504956109, 732918045195987065, 751051099822368848, 751310836657643440, 796091087130310491, 827831894447955669, 901500367437635908|
|practice_0087|mixed|128|1|223|233|10|233|27.08|228438854514, 5939410216870, 228438856484, 5025654805172, 444191287471, 444191290778, 444191294484, 469054229390, 469054225414, 469054218537, 1749572289485380, 2697864600811280, 6804875038469815, 247970022268506229, 66492657973358813, 133917248077122279, 202919113369677971, 237952896022508612, 203622275463572720, 356340179645825617, 384175020508201436, 393209443394965908, 827831894447955669, 669642033504956109, 751051099822368848, 901500367437635908, 732918045195987065, 751310836657643440, 796091087130310491|
|practice_0087|mixed|128|2|221|225|34|255|208.27|228438856484, 444191287471, 469054225414, 5025654805172, 228438854514, 469054229390, 5939410216870, 469054218537, 444191294484, 444191290778, 1749572289485380, 2697864600811280, 6804875038469815, 751310836657643440, 66492657973358813, 247970022268506229, 202919113369677971, 356340179645825617, 203622275463572720, 393209443394965908, 133917248077122279, 732918045195987065, 669642033504956109, 384175020508201436, 751051099822368848, 796091087130310491, 827831894447955669, 237952896022508612, 901500367437635908|
|practice_0088|shared|0|0|233|0|0|233|3.68|134475483907, 134475494301, 160229456718, 160229460210, 160229460445, 160229467164, 164658205811, 190915676069, 392385202318, 1922753605620, 2550503784699, 3496362728202, 4281113418046, 5846845689360, 6992725514060, 10398207907961, 10483998794848, 20796415904962, 28643279460301, 34907540892272, 49991592688091, 273351465145528, 343955818237909, 1012318665097243, 1116545985017621, 2117064083316181, 3370882841177424, 7305720069312074, 119666340863245250, 203327500543763233, 239402824929575580, 958735261420279544|
|practice_0088|shared|0|1|228|0|0|228|14.80|134475494301, 134475483907, 6992725514060, 3496362728202, 160229456718, 160229460210, 160229460445, 160229467164, 1922753605620, 164658205811, 4281113418046, 49991592688091, 5846845689360, 3370882841177424, 190915676069, 2550503784699, 392385202318, 20796415904962, 10398207907961, 34907540892272, 28643279460301, 1012318665097243, 10483998794848, 273351465145528, 343955818237909, 1116545985017621, 2117064083316181, 7305720069312074, 239402824929575580, 119666340863245250, 203327500543763233, 958735261420279544|
|practice_0088|shared|0|2|208|0|7|215|98.81|160229456718, 134475494301, 160229467164, 3496362728202, 160229460210, 134475483907, 1922753605620, 160229460445, 392385202318, 20796415904962, 164658205811, 4281113418046, 2550503784699, 10398207907961, 34907540892272, 6992725514060, 273351465145528, 49991592688091, 190915676069, 3370882841177424, 119666340863245250, 2117064083316181, 10483998794848, 28643279460301, 239402824929575580, 1012318665097243, 343955818237909, 5846845689360, 958735261420279544, 203327500543763233, 1116545985017621, 7305720069312074|
|practice_0088|shared|48|0|245|311|0|245|1.87|134475483907, 134475494301, 160229456718, 160229460210, 160229460445, 160229467164, 164658205811, 190915676069, 392385202318, 1922753605620, 2550503784699, 3496362728202, 4281113418046, 5846845689360, 6992725514060, 10398207907961, 10483998794848, 20796415904962, 28643279460301, 34907540892272, 49991592688091, 273351465145528, 343955818237909, 1012318665097243, 1116545985017621, 2117064083316181, 3370882841177424, 7305720069312074, 119666340863245250, 203327500543763233, 239402824929575580, 958735261420279544|
|practice_0088|shared|48|1|221|235|0|221|6.39|134475494301, 134475483907, 6992725514060, 3496362728202, 28643279460301, 5846845689360, 190915676069, 160229456718, 160229467164, 1922753605620, 160229460210, 160229460445, 164658205811, 4281113418046, 49991592688091, 392385202318, 2550503784699, 10398207907961, 20796415904962, 273351465145528, 10483998794848, 1012318665097243, 34907540892272, 343955818237909, 1116545985017621, 7305720069312074, 119666340863245250, 3370882841177424, 239402824929575580, 2117064083316181, 958735261420279544, 203327500543763233|
|practice_0088|shared|48|2|245|259|0|245|37.30|160229456718, 134475483907, 160229467164, 1922753605620, 134475494301, 160229460445, 6992725514060, 3496362728202, 2550503784699, 392385202318, 10398207907961, 28643279460301, 160229460210, 20796415904962, 5846845689360, 190915676069, 164658205811, 1012318665097243, 4281113418046, 10483998794848, 273351465145528, 34907540892272, 7305720069312074, 1116545985017621, 239402824929575580, 343955818237909, 3370882841177424, 203327500543763233, 958735261420279544, 2117064083316181, 119666340863245250, 49991592688091|
|practice_0088|shared|128|0|211|216|0|211|7.47|134475483907, 134475494301, 160229456718, 160229460210, 160229460445, 160229467164, 164658205811, 190915676069, 392385202318, 1922753605620, 2550503784699, 3496362728202, 4281113418046, 5846845689360, 6992725514060, 10398207907961, 10483998794848, 20796415904962, 28643279460301, 34907540892272, 49991592688091, 273351465145528, 343955818237909, 1012318665097243, 1116545985017621, 2117064083316181, 3370882841177424, 7305720069312074, 119666340863245250, 203327500543763233, 239402824929575580, 958735261420279544|
|practice_0088|shared|128|1|208|212|5|213|29.07|134475494301, 134475483907, 6992725514060, 3496362728202, 28643279460301, 5846845689360, 190915676069, 392385202318, 2550503784699, 10398207907961, 20796415904962, 160229460445, 160229456718, 160229460210, 160229467164, 1922753605620, 164658205811, 4281113418046, 34907540892272, 49991592688091, 1012318665097243, 10483998794848, 273351465145528, 3370882841177424, 119666340863245250, 239402824929575580, 343955818237909, 1116545985017621, 7305720069312074, 203327500543763233, 2117064083316181, 958735261420279544|
|practice_0088|shared|128|2|195|197|10|205|160.34|160229456718, 134475483907, 160229467164, 6992725514060, 160229460210, 164658205811, 134475494301, 4281113418046, 3370882841177424, 160229460445, 1922753605620, 392385202318, 20796415904962, 3496362728202, 1012318665097243, 34907540892272, 10398207907961, 190915676069, 2550503784699, 28643279460301, 273351465145528, 958735261420279544, 119666340863245250, 49991592688091, 5846845689360, 7305720069312074, 10483998794848, 1116545985017621, 2117064083316181, 239402824929575580, 343955818237909, 203327500543763233|
|practice_0089|shared|0|0|165|0|0|165|2.99|1926499854, 1954116151, 1954117346, 1954119961, 1954119983, 5137329712, 5325809208, 5325811519, 7000720711, 17839244005, 21002003712, 29752921433, 76319570419, 196231618638, 255638848638, 1373752578976, 1599584987786, 1888440046428, 2106541943180, 3199171150430, 4216987453250, 7504755395520, 22837070466936, 41434618112134, 102563761935712, 507990382067417, 770037466196934, 821271236576128, 4318199920859136|
|practice_0089|shared|0|1|155|0|0|155|8.38|1954116151, 1954117346, 1954119961, 1954119983, 1926499854, 5137329712, 5325809208, 5325811519, 21002003712, 7000720711, 29752921433, 255638848638, 1373752578976, 4216987453250, 4318199920859136, 1888440046428, 2106541943180, 7504755395520, 76319570419, 22837070466936, 770037466196934, 17839244005, 196231618638, 507990382067417, 41434618112134, 102563761935712, 3199171150430, 1599584987786, 821271236576128|
|practice_0089|shared|0|2|155|0|0|155|44.72|1954119961, 1954117346, 5325809208, 5325811519, 1954116151, 21002003712, 4216987453250, 1954119983, 255638848638, 7000720711, 1888440046428, 29752921433, 1926499854, 17839244005, 7504755395520, 770037466196934, 821271236576128, 507990382067417, 102563761935712, 1599584987786, 196231618638, 4318199920859136, 41434618112134, 5137329712, 76319570419, 3199171150430, 1373752578976, 22837070466936, 2106541943180|
|practice_0089|shared|48|0|193|221|0|193|1.29|1926499854, 1954116151, 1954117346, 1954119961, 1954119983, 5137329712, 5325809208, 5325811519, 7000720711, 17839244005, 21002003712, 29752921433, 76319570419, 196231618638, 255638848638, 1373752578976, 1599584987786, 1888440046428, 2106541943180, 3199171150430, 4216987453250, 7504755395520, 22837070466936, 41434618112134, 102563761935712, 507990382067417, 770037466196934, 821271236576128, 4318199920859136|
|practice_0089|shared|48|1|181|191|0|181|3.46|1954116151, 1954117346, 1954119961, 1954119983, 1926499854, 5137329712, 21002003712, 5325811519, 5325809208, 7000720711, 255638848638, 22837070466936, 29752921433, 1373752578976, 76319570419, 17839244005, 196231618638, 1599584987786, 3199171150430, 41434618112134, 102563761935712, 821271236576128, 1888440046428, 507990382067417, 4216987453250, 7504755395520, 4318199920859136, 2106541943180, 770037466196934|
|practice_0089|shared|48|2|195|205|0|195|27.06|1954119961, 1954117346, 5325809208, 5325811519, 1954116151, 1954119983, 17839244005, 4216987453250, 196231618638, 76319570419, 5137329712, 1926499854, 255638848638, 21002003712, 1599584987786, 1373752578976, 7504755395520, 3199171150430, 29752921433, 102563761935712, 2106541943180, 4318199920859136, 1888440046428, 22837070466936, 7000720711, 507990382067417, 770037466196934, 41434618112134, 821271236576128|
|practice_0089|shared|128|0|178|184|31|209|6.14|1926499854, 1954116151, 1954117346, 1954119961, 1954119983, 5137329712, 5325809208, 5325811519, 7000720711, 17839244005, 21002003712, 29752921433, 76319570419, 196231618638, 255638848638, 1373752578976, 1599584987786, 1888440046428, 2106541943180, 3199171150430, 4216987453250, 7504755395520, 22837070466936, 41434618112134, 102563761935712, 507990382067417, 770037466196934, 821271236576128, 4318199920859136|
|practice_0089|shared|128|1|166|170|51|217|17.12|1954116151, 1954117346, 1954119961, 1954119983, 1926499854, 5137329712, 4216987453250, 4318199920859136, 5325811519, 5325809208, 21002003712, 7000720711, 29752921433, 76319570419, 255638848638, 1888440046428, 2106541943180, 17839244005, 196231618638, 7504755395520, 22837070466936, 1373752578976, 1599584987786, 3199171150430, 770037466196934, 41434618112134, 102563761935712, 821271236576128, 507990382067417|
|practice_0089|shared|128|2|158|163|7|165|82.88|1954119961, 1954117346, 5325809208, 5325811519, 1954116151, 4318199920859136, 1954119983, 255638848638, 2106541943180, 196231618638, 1599584987786, 17839244005, 3199171150430, 5137329712, 102563761935712, 1926499854, 22837070466936, 7000720711, 29752921433, 21002003712, 770037466196934, 4216987453250, 1888440046428, 7504755395520, 41434618112134, 507990382067417, 821271236576128, 76319570419, 1373752578976|
|practice_0090|shared|0|0|155|0|0|155|3.27|7354870, 7356109, 7356639, 29429456, 31882218, 36327643, 46198455, 85497303, 120020565, 127529666, 184787694, 765193080, 797056334, 1029031872, 1185359685, 1221365100, 1594112236, 4102769640, 7858989864, 10562547750, 12042637956, 13039662744, 15643072506, 22767607455, 69033827181, 70188509647, 78047654960, 1117633797504, 1389567390625, 74301367537152|
|practice_0090|shared|0|1|158|0|0|158|6.15|7354870, 7356639, 7356109, 29429456, 31882218, 46198455, 127529666, 184787694, 765193080, 85497303, 797056334, 1594112236, 36327643, 1117633797504, 7858989864, 120020565, 74301367537152, 1029031872, 1185359685, 1221365100, 4102769640, 12042637956, 10562547750, 13039662744, 78047654960, 15643072506, 22767607455, 69033827181, 70188509647, 1389567390625|
|practice_0090|shared|0|2|154|0|0|154|36.56|7356109, 31882218, 7354870, 46198455, 36327643, 797056334, 29429456, 1594112236, 7356639, 1029031872, 85497303, 184787694, 1221365100, 1185359685, 120020565, 127529666, 765193080, 7858989864, 1117633797504, 78047654960, 74301367537152, 70188509647, 13039662744, 10562547750, 4102769640, 22767607455, 69033827181, 1389567390625, 12042637956, 15643072506|
|practice_0090|shared|48|0|156|231|0|156|1.39|7354870, 7356109, 7356639, 29429456, 31882218, 36327643, 46198455, 85497303, 120020565, 127529666, 184787694, 765193080, 797056334, 1029031872, 1185359685, 1221365100, 1594112236, 4102769640, 7858989864, 10562547750, 12042637956, 13039662744, 15643072506, 22767607455, 69033827181, 70188509647, 78047654960, 1117633797504, 1389567390625, 74301367537152|
|practice_0090|shared|48|1|153|163|0|153|3.14|7354870, 7356639, 7356109, 29429456, 31882218, 46198455, 127529666, 765193080, 184787694, 1221365100, 120020565, 797056334, 1594112236, 10562547750, 1117633797504, 36327643, 85497303, 1185359685, 1029031872, 4102769640, 74301367537152, 78047654960, 12042637956, 7858989864, 13039662744, 70188509647, 69033827181, 15643072506, 22767607455, 1389567390625|
|practice_0090|shared|48|2|167|176|0|167|16.60|7356109, 31882218, 7354870, 46198455, 36327643, 29429456, 797056334, 127529666, 184787694, 120020565, 765193080, 1594112236, 7356639, 1221365100, 1185359685, 4102769640, 74301367537152, 85497303, 1029031872, 22767607455, 70188509647, 12042637956, 13039662744, 69033827181, 1117633797504, 1389567390625, 15643072506, 78047654960, 7858989864, 10562547750|
|practice_0090|shared|128|0|150|154|43|193|4.99|7354870, 7356109, 7356639, 29429456, 31882218, 36327643, 46198455, 85497303, 120020565, 127529666, 184787694, 765193080, 797056334, 1029031872, 1185359685, 1221365100, 1594112236, 4102769640, 7858989864, 10562547750, 12042637956, 13039662744, 15643072506, 22767607455, 69033827181, 70188509647, 78047654960, 1117633797504, 1389567390625, 74301367537152|
|practice_0090|shared|128|1|145|148|8|153|7.76|7354870, 7356639, 7356109, 29429456, 31882218, 46198455, 85497303, 127529666, 765193080, 36327643, 120020565, 184787694, 797056334, 1594112236, 4102769640, 7858989864, 1185359685, 1117633797504, 1029031872, 1221365100, 10562547750, 12042637956, 13039662744, 15643072506, 22767607455, 69033827181, 70188509647, 78047654960, 74301367537152, 1389567390625|
|practice_0090|shared|128|2|149|152|10|159|62.98|7356109, 31882218, 7354870, 46198455, 36327643, 797056334, 29429456, 1594112236, 7356639, 85497303, 127529666, 184787694, 1117633797504, 1029031872, 120020565, 10562547750, 765193080, 1185359685, 1221365100, 12042637956, 69033827181, 13039662744, 78047654960, 74301367537152, 15643072506, 7858989864, 4102769640, 22767607455, 70188509647, 1389567390625|
|practice_0091|shared|0|0|230|0|0|230|4.00|303476333263, 442214372758, 442214377616, 486321796923, 616342064366, 691768139031, 691768139076, 826590923804, 898110617772, 898110618006, 987690121101, 4248668727146, 7183697149210, 7184884943488, 7294826943742, 8406634964445, 67813522050395, 99190910838934, 117689874943654, 121152801810941, 517848713834761, 554707857879510, 754412919069600, 789578638079260, 858338103175250, 1286843833353930, 1579971267141870, 3486465415077342, 15689094397493944, 26430616992866772, 263252092595641721, 393699847705990200|
|practice_0091|shared|0|1|205|0|0|205|14.75|691768139076, 691768139031, 303476333263, 4248668727146, 442214372758, 442214377616, 898110617772, 898110618006, 7184884943488, 754412919069600, 486321796923, 7294826943742, 826590923804, 1286843833353930, 616342064366, 7183697149210, 99190910838934, 554707857879510, 858338103175250, 26430616992866772, 117689874943654, 3486465415077342, 15689094397493944, 987690121101, 8406634964445, 67813522050395, 517848713834761, 263252092595641721, 121152801810941, 789578638079260, 1579971267141870, 393699847705990200|
|practice_0091|shared|0|2|206|0|28|234|88.34|691768139076, 486321796923, 8406634964445, 616342064366, 67813522050395, 7294826943742, 303476333263, 691768139031, 898110618006, 7184884943488, 442214377616, 442214372758, 754412919069600, 517848713834761, 898110617772, 826590923804, 117689874943654, 121152801810941, 858338103175250, 554707857879510, 26430616992866772, 4248668727146, 263252092595641721, 1286843833353930, 3486465415077342, 7183697149210, 99190910838934, 987690121101, 789578638079260, 15689094397493944, 1579971267141870, 393699847705990200|
|practice_0091|shared|48|0|262|337|0|262|1.49|303476333263, 442214372758, 442214377616, 486321796923, 616342064366, 691768139031, 691768139076, 826590923804, 898110617772, 898110618006, 987690121101, 4248668727146, 7183697149210, 7184884943488, 7294826943742, 8406634964445, 67813522050395, 99190910838934, 117689874943654, 121152801810941, 517848713834761, 554707857879510, 754412919069600, 789578638079260, 858338103175250, 1286843833353930, 1579971267141870, 3486465415077342, 15689094397493944, 26430616992866772, 263252092595641721, 393699847705990200|
|practice_0091|shared|48|1|239|245|0|239|4.69|691768139076, 691768139031, 303476333263, 4248668727146, 442214372758, 442214377616, 616342064366, 486321796923, 898110618006, 898110617772, 7184884943488, 754412919069600, 554707857879510, 7183697149210, 987690121101, 7294826943742, 826590923804, 8406634964445, 67813522050395, 517848713834761, 99190910838934, 263252092595641721, 789578638079260, 3486465415077342, 1286843833353930, 1579971267141870, 117689874943654, 121152801810941, 393699847705990200, 26430616992866772, 858338103175250, 15689094397493944|
|practice_0091|shared|48|2|244|266|0|244|38.24|691768139076, 486321796923, 8406634964445, 691768139031, 117689874943654, 7294826943742, 898110618006, 7184884943488, 616342064366, 442214372758, 898110617772, 442214377616, 7183697149210, 826590923804, 26430616992866772, 4248668727146, 99190910838934, 3486465415077342, 858338103175250, 121152801810941, 754412919069600, 517848713834761, 987690121101, 303476333263, 554707857879510, 67813522050395, 1579971267141870, 393699847705990200, 15689094397493944, 1286843833353930, 789578638079260, 263252092595641721|
|practice_0091|shared|128|0|211|222|34|245|5.98|303476333263, 442214372758, 442214377616, 486321796923, 616342064366, 691768139031, 691768139076, 826590923804, 898110617772, 898110618006, 987690121101, 4248668727146, 7183697149210, 7184884943488, 7294826943742, 8406634964445, 67813522050395, 99190910838934, 117689874943654, 121152801810941, 517848713834761, 554707857879510, 754412919069600, 789578638079260, 858338103175250, 1286843833353930, 1579971267141870, 3486465415077342, 15689094397493944, 26430616992866772, 263252092595641721, 393699847705990200|
|practice_0091|shared|128|1|195|198|10|205|26.73|691768139076, 691768139031, 303476333263, 4248668727146, 442214372758, 442214377616, 898110617772, 898110618006, 7184884943488, 754412919069600, 7183697149210, 486321796923, 7294826943742, 616342064366, 117689874943654, 826590923804, 987690121101, 99190910838934, 8406634964445, 67813522050395, 517848713834761, 1286843833353930, 15689094397493944, 858338103175250, 26430616992866772, 263252092595641721, 3486465415077342, 554707857879510, 121152801810941, 393699847705990200, 789578638079260, 1579971267141870|
|practice_0091|shared|128|2|199|202|18|217|176.26|691768139076, 486321796923, 8406634964445, 616342064366, 898110617772, 67813522050395, 7184884943488, 691768139031, 898110618006, 754412919069600, 303476333263, 826590923804, 442214372758, 442214377616, 517848713834761, 7183697149210, 7294826943742, 858338103175250, 26430616992866772, 4248668727146, 117689874943654, 99190910838934, 554707857879510, 263252092595641721, 789578638079260, 3486465415077342, 1579971267141870, 121152801810941, 987690121101, 1286843833353930, 393699847705990200, 15689094397493944|
|practice_0092|shared|0|0|183|0|0|183|3.74|1531989240, 1531991575, 2540669703, 2540670265, 2692140704, 2692140707, 3736373619, 3736374206, 4464661895, 6565865736, 14644670848, 22866030448, 27323102616, 48458533104, 60409746375, 83891397376, 212679116602, 220606828416, 248182681953, 256068575021, 750064893876, 3134199722931, 3656565825003, 9270204569120, 15516013385904, 16588970679138, 16624759798779, 813927443420448, 2032820546218512, 72303050101192416|
|practice_0092|shared|0|1|174|0|0|174|11.63|1531991575, 1531989240, 220606828416, 248182681953, 15516013385904, 2540669703, 2540670265, 22866030448, 2692140704, 2692140707, 48458533104, 212679116602, 16588970679138, 27323102616, 83891397376, 4464661895, 3736373619, 3736374206, 6565865736, 14644670848, 60409746375, 256068575021, 3656565825003, 813927443420448, 750064893876, 3134199722931, 9270204569120, 16624759798779, 2032820546218512, 72303050101192416|
|practice_0092|shared|0|2|165|0|0|165|65.04|2692140704, 1531991575, 220606828416, 2540669703, 2692140707, 1531989240, 3736373619, 2540670265, 3736374206, 48458533104, 212679116602, 15516013385904, 4464661895, 22866030448, 248182681953, 3656565825003, 813927443420448, 16588970679138, 14644670848, 27323102616, 256068575021, 750064893876, 6565865736, 16624759798779, 3134199722931, 60409746375, 83891397376, 2032820546218512, 9270204569120, 72303050101192416|
|practice_0092|shared|48|0|195|262|0|195|1.67|1531989240, 1531991575, 2540669703, 2540670265, 2692140704, 2692140707, 3736373619, 3736374206, 4464661895, 6565865736, 14644670848, 22866030448, 27323102616, 48458533104, 60409746375, 83891397376, 212679116602, 220606828416, 248182681953, 256068575021, 750064893876, 3134199722931, 3656565825003, 9270204569120, 15516013385904, 16588970679138, 16624759798779, 813927443420448, 2032820546218512, 72303050101192416|
|practice_0092|shared|48|1|176|186|0|176|4.52|1531991575, 1531989240, 220606828416, 248182681953, 15516013385904, 2692140704, 2692140707, 48458533104, 212679116602, 16588970679138, 2540669703, 2540670265, 22866030448, 6565865736, 3736374206, 3736373619, 83891397376, 4464661895, 27323102616, 14644670848, 60409746375, 256068575021, 750064893876, 3656565825003, 3134199722931, 16624759798779, 9270204569120, 72303050101192416, 813927443420448, 2032820546218512|
|practice_0092|shared|48|2|184|191|0|184|28.54|2692140707, 1531991575, 2692140704, 220606828416, 1531989240, 48458533104, 2540670265, 22866030448, 248182681953, 2540669703, 3736373619, 15516013385904, 212679116602, 3736374206, 4464661895, 14644670848, 83891397376, 256068575021, 3134199722931, 6565865736, 16588970679138, 750064893876, 27323102616, 813927443420448, 60409746375, 3656565825003, 2032820546218512, 16624759798779, 9270204569120, 72303050101192416|
|practice_0092|shared|128|0|171|176|0|171|5.61|1531989240, 1531991575, 2540669703, 2540670265, 2692140704, 2692140707, 3736373619, 3736374206, 4464661895, 6565865736, 14644670848, 22866030448, 27323102616, 48458533104, 60409746375, 83891397376, 212679116602, 220606828416, 248182681953, 256068575021, 750064893876, 3134199722931, 3656565825003, 9270204569120, 15516013385904, 16588970679138, 16624759798779, 813927443420448, 2032820546218512, 72303050101192416|
|practice_0092|shared|128|1|156|160|0|156|15.69|1531991575, 1531989240, 220606828416, 248182681953, 15516013385904, 2540669703, 2540670265, 22866030448, 2692140704, 2692140707, 48458533104, 212679116602, 16588970679138, 6565865736, 3736374206, 3736373619, 4464661895, 27323102616, 14644670848, 60409746375, 83891397376, 256068575021, 3656565825003, 813927443420448, 3134199722931, 750064893876, 2032820546218512, 16624759798779, 9270204569120, 72303050101192416|
|practice_0092|shared|128|2|157|162|0|157|100.56|2692140707, 1531991575, 220606828416, 48458533104, 2540670265, 2540669703, 2692140704, 1531989240, 14644670848, 212679116602, 15516013385904, 248182681953, 4464661895, 27323102616, 3736374206, 3736373619, 22866030448, 6565865736, 256068575021, 16588970679138, 83891397376, 60409746375, 2032820546218512, 813927443420448, 3656565825003, 750064893876, 3134199722931, 16624759798779, 9270204569120, 72303050101192416|
|practice_0093|mixed|0|0|238|0|0|238|3.45|233337030, 496503213, 1390314551, 1821190179, 1829446500, 2396408691, 9566350599, 74669234112, 257409217354, 663339079350, 705792117368, 1327181587836, 7429783827881, 9979803632490, 32620200152178, 226598292249272, 286938075514284, 314097708700250, 327814041477178, 361902711037866, 468386101120041, 604851663363989, 772992003087076, 786804020079309, 794435468992810, 813201994067601, 842539926305810, 872942003139244, 875487223060537, 882191200150605|
|practice_0093|mixed|0|1|231|0|0|231|9.62|233337030, 496503213, 1390314551, 1829446500, 1821190179, 74669234112, 2396408691, 9566350599, 663339079350, 705792117368, 257409217354, 1327181587836, 7429783827881, 9979803632490, 32620200152178, 314097708700250, 226598292249272, 327814041477178, 286938075514284, 813201994067601, 468386101120041, 361902711037866, 875487223060537, 772992003087076, 794435468992810, 872942003139244, 786804020079309, 604851663363989, 882191200150605, 842539926305810|
|practice_0093|mixed|0|2|224|0|40|264|69.54|1821190179, 233337030, 2396408691, 257409217354, 496503213, 74669234112, 1390314551, 1829446500, 9566350599, 1327181587836, 663339079350, 32620200152178, 9979803632490, 842539926305810, 705792117368, 314097708700250, 872942003139244, 226598292249272, 7429783827881, 875487223060537, 361902711037866, 327814041477178, 882191200150605, 813201994067601, 468386101120041, 794435468992810, 604851663363989, 286938075514284, 772992003087076, 786804020079309|
|practice_0093|mixed|48|0|244|319|0|244|1.53|233337030, 496503213, 1390314551, 1821190179, 1829446500, 2396408691, 9566350599, 74669234112, 257409217354, 663339079350, 705792117368, 1327181587836, 7429783827881, 9979803632490, 32620200152178, 226598292249272, 286938075514284, 314097708700250, 327814041477178, 361902711037866, 468386101120041, 604851663363989, 772992003087076, 786804020079309, 794435468992810, 813201994067601, 842539926305810, 872942003139244, 875487223060537, 882191200150605|
|practice_0093|mixed|48|1|242|248|0|242|4.14|233337030, 496503213, 1390314551, 1829446500, 1821190179, 2396408691, 9566350599, 1327181587836, 257409217354, 663339079350, 705792117368, 74669234112, 9979803632490, 32620200152178, 314097708700250, 7429783827881, 286938075514284, 361902711037866, 327814041477178, 794435468992810, 772992003087076, 604851663363989, 468386101120041, 882191200150605, 813201994067601, 226598292249272, 875487223060537, 872942003139244, 842539926305810, 786804020079309|
|practice_0093|mixed|48|2|243|275|0|243|29.47|1821190179, 233337030, 9566350599, 2396408691, 663339079350, 74669234112, 496503213, 32620200152178, 1390314551, 7429783827881, 257409217354, 1829446500, 468386101120041, 226598292249272, 1327181587836, 286938075514284, 786804020079309, 314097708700250, 361902711037866, 772992003087076, 794435468992810, 604851663363989, 842539926305810, 875487223060537, 327814041477178, 872942003139244, 882191200150605, 813201994067601, 705792117368, 9979803632490|
|practice_0093|mixed|128|0|218|225|10|228|6.65|233337030, 496503213, 1390314551, 1821190179, 1829446500, 2396408691, 9566350599, 74669234112, 257409217354, 663339079350, 705792117368, 1327181587836, 7429783827881, 9979803632490, 32620200152178, 226598292249272, 286938075514284, 314097708700250, 327814041477178, 361902711037866, 468386101120041, 604851663363989, 772992003087076, 786804020079309, 794435468992810, 813201994067601, 842539926305810, 872942003139244, 875487223060537, 882191200150605|
|practice_0093|mixed|128|1|213|215|0|213|21.90|233337030, 496503213, 1390314551, 1829446500, 1821190179, 74669234112, 2396408691, 9566350599, 663339079350, 705792117368, 257409217354, 1327181587836, 7429783827881, 32620200152178, 314097708700250, 9979803632490, 842539926305810, 226598292249272, 468386101120041, 327814041477178, 286938075514284, 813201994067601, 604851663363989, 361902711037866, 772992003087076, 786804020079309, 794435468992810, 872942003139244, 875487223060537, 882191200150605|
|practice_0093|mixed|128|2|211|216|11|222|141.87|1821190179, 233337030, 1829446500, 496503213, 875487223060537, 257409217354, 2396408691, 9566350599, 74669234112, 1327181587836, 1390314551, 663339079350, 705792117368, 327814041477178, 7429783827881, 468386101120041, 32620200152178, 286938075514284, 794435468992810, 226598292249272, 786804020079309, 813201994067601, 842539926305810, 9979803632490, 604851663363989, 314097708700250, 361902711037866, 872942003139244, 882191200150605, 772992003087076|
|practice_0094|multiplicative|0|0|141|0|0|141|3.44|221376, 303536, 381251, 662612, 763028, 1484997, 1908134, 2085475, 3308838, 5947590, 6218359, 19688184, 39177770, 44606850, 78355894, 156697510, 376109412, 807112880, 1008276437, 1381575776, 1693275790, 2076009582, 2090430362, 2414464897, 90542425754, 286677130544, 7653152011232, 1194812383171680|
|practice_0094|multiplicative|0|1|137|0|0|137|6.60|221376, 303536, 381251, 763028, 662612, 1908134, 3308838, 1484997, 2085475, 5947590, 44606850, 6218359, 19688184, 39177770, 78355894, 156697510, 1381575776, 1008276437, 2414464897, 376109412, 807112880, 2076009582, 1693275790, 2090430362, 286677130544, 90542425754, 1194812383171680, 7653152011232|
|practice_0094|multiplicative|0|2|141|0|0|141|43.20|381251, 1908134, 303536, 221376, 763028, 5947590, 44606850, 2085475, 662612, 6218359, 807112880, 3308838, 39177770, 1484997, 1008276437, 78355894, 156697510, 19688184, 1381575776, 2076009582, 2090430362, 376109412, 1693275790, 1194812383171680, 2414464897, 90542425754, 286677130544, 7653152011232|
|practice_0094|multiplicative|48|0|147|167|0|147|1.42|221376, 303536, 381251, 662612, 763028, 1484997, 1908134, 2085475, 3308838, 5947590, 6218359, 19688184, 39177770, 44606850, 78355894, 156697510, 376109412, 807112880, 1008276437, 1381575776, 1693275790, 2076009582, 2090430362, 2414464897, 90542425754, 286677130544, 7653152011232, 1194812383171680|
|practice_0094|multiplicative|48|1|142|153|0|142|2.76|221376, 303536, 381251, 763028, 662612, 1908134, 3308838, 1484997, 5947590, 2085475, 19688184, 39177770, 78355894, 6218359, 156697510, 376109412, 44606850, 807112880, 1008276437, 2076009582, 1381575776, 1693275790, 2414464897, 90542425754, 286677130544, 7653152011232, 2090430362, 1194812383171680|
|practice_0094|multiplicative|48|2|147|155|0|147|16.81|381251, 1908134, 303536, 221376, 5947590, 44606850, 763028, 19688184, 1484997, 662612, 78355894, 3308838, 156697510, 1381575776, 807112880, 6218359, 39177770, 2090430362, 2085475, 2414464897, 376109412, 1693275790, 1008276437, 2076009582, 1194812383171680, 90542425754, 7653152011232, 286677130544|
|practice_0094|multiplicative|128|0|136|140|9|145|4.98|221376, 303536, 381251, 662612, 763028, 1484997, 1908134, 2085475, 3308838, 5947590, 6218359, 19688184, 39177770, 44606850, 78355894, 156697510, 376109412, 807112880, 1008276437, 1381575776, 1693275790, 2076009582, 2090430362, 2414464897, 90542425754, 286677130544, 7653152011232, 1194812383171680|
|practice_0094|multiplicative|128|1|137|144|14|151|11.43|221376, 303536, 381251, 763028, 662612, 1908134, 3308838, 1484997, 2085475, 5947590, 44606850, 6218359, 19688184, 39177770, 78355894, 156697510, 376109412, 1381575776, 2076009582, 2090430362, 2414464897, 807112880, 1008276437, 1693275790, 90542425754, 286677130544, 7653152011232, 1194812383171680|
|practice_0094|multiplicative|128|2|133|136|4|137|42.73|381251, 1908134, 303536, 221376, 763028, 5947590, 44606850, 19688184, 1484997, 39177770, 662612, 78355894, 807112880, 156697510, 3308838, 6218359, 2085475, 376109412, 1381575776, 2414464897, 2076009582, 2090430362, 1008276437, 1693275790, 90542425754, 286677130544, 1194812383171680, 7653152011232|
|practice_0095|mixed|0|0|261|0|0|261|4.62|833101208, 1709824812, 41091919346, 61312311585, 151651987869, 173480876446, 609094043505, 2540956261616, 6307569225465, 6939724357512, 11505741816520, 22917558730344, 28385557449200, 104278094163290, 116109530438404, 125736181115100, 151011995568139, 173741835757158, 237972529037602, 316508253660553, 433075401968856, 454619088852979, 575592072477537, 630246446838825, 672554083033645, 709054764118648, 896573468111885, 922345096588018, 950125181427392, 964150049896773, 39205096124218430|
|practice_0095|mixed|0|1|252|0|0|252|14.71|1709824812, 833101208, 173480876446, 41091919346, 61312311585, 151651987869, 609094043505, 2540956261616, 6307569225465, 6939724357512, 11505741816520, 22917558730344, 28385557449200, 104278094163290, 151011995568139, 173741835757158, 125736181115100, 116109530438404, 237972529037602, 630246446838825, 672554083033645, 709054764118648, 454619088852979, 316508253660553, 575592072477537, 433075401968856, 896573468111885, 922345096588018, 950125181427392, 964150049896773, 39205096124218430|
|practice_0095|mixed|0|2|246|0|111|357|107.71|41091919346, 833101208, 1709824812, 173480876446, 151651987869, 11505741816520, 61312311585, 609094043505, 22917558730344, 575592072477537, 6939724357512, 28385557449200, 2540956261616, 104278094163290, 6307569225465, 896573468111885, 709054764118648, 316508253660553, 151011995568139, 433075401968856, 964150049896773, 116109530438404, 125736181115100, 173741835757158, 454619088852979, 672554083033645, 237972529037602, 39205096124218430, 630246446838825, 950125181427392, 922345096588018|
|practice_0095|mixed|48|0|257|346|0|257|1.64|833101208, 1709824812, 41091919346, 61312311585, 151651987869, 173480876446, 609094043505, 2540956261616, 6307569225465, 6939724357512, 11505741816520, 22917558730344, 28385557449200, 104278094163290, 116109530438404, 125736181115100, 151011995568139, 173741835757158, 237972529037602, 316508253660553, 433075401968856, 454619088852979, 575592072477537, 630246446838825, 672554083033645, 709054764118648, 896573468111885, 922345096588018, 950125181427392, 964150049896773, 39205096124218430|
|practice_0095|mixed|48|1|249|263|0|249|5.50|1709824812, 833101208, 173480876446, 151651987869, 609094043505, 61312311585, 41091919346, 2540956261616, 6307569225465, 11505741816520, 22917558730344, 28385557449200, 6939724357512, 125736181115100, 316508253660553, 116109530438404, 104278094163290, 151011995568139, 433075401968856, 173741835757158, 964150049896773, 237972529037602, 672554083033645, 896573468111885, 950125181427392, 630246446838825, 922345096588018, 709054764118648, 39205096124218430, 454619088852979, 575592072477537|
|practice_0095|mixed|48|2|258|267|0|258|37.98|833101208, 61312311585, 173480876446, 151651987869, 41091919346, 22917558730344, 11505741816520, 151011995568139, 28385557449200, 950125181427392, 6307569225465, 609094043505, 1709824812, 237972529037602, 6939724357512, 173741835757158, 433075401968856, 964150049896773, 922345096588018, 454619088852979, 104278094163290, 316508253660553, 672554083033645, 896573468111885, 116109530438404, 630246446838825, 575592072477537, 125736181115100, 709054764118648, 2540956261616, 39205096124218430|
|practice_0095|mixed|128|0|230|250|16|246|7.96|833101208, 1709824812, 41091919346, 61312311585, 151651987869, 173480876446, 609094043505, 2540956261616, 6307569225465, 6939724357512, 11505741816520, 22917558730344, 28385557449200, 104278094163290, 116109530438404, 125736181115100, 151011995568139, 173741835757158, 237972529037602, 316508253660553, 433075401968856, 454619088852979, 575592072477537, 630246446838825, 672554083033645, 709054764118648, 896573468111885, 922345096588018, 950125181427392, 964150049896773, 39205096124218430|
|practice_0095|mixed|128|1|231|235|28|259|29.69|1709824812, 833101208, 173480876446, 41091919346, 61312311585, 151651987869, 11505741816520, 6939724357512, 28385557449200, 609094043505, 2540956261616, 6307569225465, 22917558730344, 116109530438404, 125736181115100, 151011995568139, 173741835757158, 104278094163290, 630246446838825, 575592072477537, 237972529037602, 316508253660553, 433075401968856, 709054764118648, 454619088852979, 672554083033645, 922345096588018, 896573468111885, 950125181427392, 964150049896773, 39205096124218430|
|practice_0095|mixed|128|2|224|227|1|225|178.01|833101208, 61312311585, 173480876446, 609094043505, 6307569225465, 11505741816520, 41091919346, 1709824812, 6939724357512, 28385557449200, 151011995568139, 151651987869, 39205096124218430, 2540956261616, 454619088852979, 433075401968856, 22917558730344, 104278094163290, 116109530438404, 237972529037602, 630246446838825, 125736181115100, 950125181427392, 672554083033645, 316508253660553, 173741835757158, 922345096588018, 709054764118648, 964150049896773, 575592072477537, 896573468111885|
|practice_0096|independent|0|0|214|0|0|214|2.79|137710494045, 183835446054, 192997193531, 217518533575, 286622259138, 365267696626, 488128500157, 497921642308, 502534580000, 541248377535, 580301183272, 594671116439, 605109201140, 630167269395, 638320228930, 639180199623, 672828648436, 705419875545, 713948366845, 727508128428, 732688007910, 741256397589, 748613751467, 762520906567, 787189555253, 807805343392, 811269276917, 939536632031, 955837218211, 990126344185|
|practice_0096|independent|0|1|202|0|0|202|8.87|137710494045, 183835446054, 639180199623, 192997193531, 217518533575, 497921642308, 286622259138, 638320228930, 365267696626, 488128500157, 502534580000, 541248377535, 990126344185, 580301183272, 741256397589, 594671116439, 605109201140, 630167269395, 939536632031, 672828648436, 748613751467, 807805343392, 727508128428, 762520906567, 732688007910, 705419875545, 811269276917, 713948366845, 787189555253, 955837218211|
|practice_0096|independent|0|2|206|0|0|206|59.44|217518533575, 192997193531, 286622259138, 672828648436, 183835446054, 365267696626, 497921642308, 488128500157, 630167269395, 502534580000, 990126344185, 137710494045, 541248377535, 741256397589, 713948366845, 732688007910, 639180199623, 580301183272, 638320228930, 605109201140, 727508128428, 594671116439, 748613751467, 705419875545, 939536632031, 787189555253, 762520906567, 811269276917, 807805343392, 955837218211|
|practice_0096|independent|48|0|220|330|0|220|1.55|137710494045, 183835446054, 192997193531, 217518533575, 286622259138, 365267696626, 488128500157, 497921642308, 502534580000, 541248377535, 580301183272, 594671116439, 605109201140, 630167269395, 638320228930, 639180199623, 672828648436, 705419875545, 713948366845, 727508128428, 732688007910, 741256397589, 748613751467, 762520906567, 787189555253, 807805343392, 811269276917, 939536632031, 955837218211, 990126344185|
|practice_0096|independent|48|1|213|221|0|213|4.05|137710494045, 183835446054, 192997193531, 541248377535, 639180199623, 580301183272, 488128500157, 605109201140, 594671116439, 630167269395, 638320228930, 217518533575, 286622259138, 502534580000, 705419875545, 762520906567, 727508128428, 732688007910, 807805343392, 497921642308, 741256397589, 713948366845, 672828648436, 365267696626, 990126344185, 748613751467, 955837218211, 787189555253, 939536632031, 811269276917|
|practice_0096|independent|48|2|219|226|0|219|26.00|286622259138, 192997193531, 502534580000, 580301183272, 605109201140, 594671116439, 672828648436, 137710494045, 639180199623, 183835446054, 488128500157, 713948366845, 638320228930, 748613751467, 727508128428, 787189555253, 541248377535, 741256397589, 217518533575, 732688007910, 939536632031, 762520906567, 705419875545, 955837218211, 811269276917, 630167269395, 497921642308, 990126344185, 365267696626, 807805343392|
|practice_0096|independent|128|0|199|205|18|217|7.92|137710494045, 183835446054, 192997193531, 217518533575, 286622259138, 365267696626, 488128500157, 497921642308, 502534580000, 541248377535, 580301183272, 594671116439, 605109201140, 630167269395, 638320228930, 639180199623, 672828648436, 705419875545, 713948366845, 727508128428, 732688007910, 741256397589, 748613751467, 762520906567, 787189555253, 807805343392, 811269276917, 939536632031, 955837218211, 990126344185|
|practice_0096|independent|128|1|202|205|56|258|22.15|137710494045, 183835446054, 192997193531, 541248377535, 365267696626, 639180199623, 502534580000, 488128500157, 217518533575, 497921642308, 286622259138, 594671116439, 580301183272, 605109201140, 787189555253, 630167269395, 638320228930, 732688007910, 672828648436, 705419875545, 713948366845, 727508128428, 741256397589, 748613751467, 762520906567, 807805343392, 811269276917, 939536632031, 990126344185, 955837218211|
|practice_0096|independent|128|2|190|193|37|227|152.33|286622259138, 192997193531, 502534580000, 605109201140, 488128500157, 217518533575, 713948366845, 137710494045, 365267696626, 630167269395, 541248377535, 580301183272, 705419875545, 638320228930, 639180199623, 727508128428, 183835446054, 594671116439, 990126344185, 672828648436, 497921642308, 811269276917, 748613751467, 787189555253, 762520906567, 807805343392, 939536632031, 741256397589, 732688007910, 955837218211|
|practice_0097|multiplicative|0|0|133|0|0|133|2.54|333408, 1396157, 1933641, 2763650, 2837397, 3798231, 3969818, 5671151, 9773596, 25634160, 33081345, 40958648, 45368536, 78184568, 94597248, 109857978, 261054528, 275825169, 405065429, 567102080, 846887552, 1270352384, 1544615959, 3059029198, 4127020819, 4136253112, 37528888316, 106708640416|
|practice_0097|multiplicative|0|1|131|0|0|131|3.93|333408, 1396157, 9773596, 78184568, 1933641, 3798231, 2763650, 2837397, 3969818, 5671151, 45368536, 261054528, 25634160, 33081345, 40958648, 94597248, 109857978, 567102080, 846887552, 1270352384, 1544615959, 405065429, 3059029198, 4127020819, 275825169, 4136253112, 37528888316, 106708640416|
|practice_0097|multiplicative|0|2|128|0|0|128|19.32|1396157, 2837397, 1933641, 333408, 9773596, 3798231, 5671151, 2763650, 45368536, 1270352384, 3969818, 78184568, 25634160, 33081345, 94597248, 275825169, 405065429, 846887552, 40958648, 109857978, 4136253112, 261054528, 567102080, 4127020819, 1544615959, 106708640416, 3059029198, 37528888316|
|practice_0097|multiplicative|48|0|140|157|0|140|1.05|333408, 1396157, 1933641, 2763650, 2837397, 3798231, 3969818, 5671151, 9773596, 25634160, 33081345, 40958648, 45368536, 78184568, 94597248, 109857978, 261054528, 275825169, 405065429, 567102080, 846887552, 1270352384, 1544615959, 3059029198, 4127020819, 4136253112, 37528888316, 106708640416|
|practice_0097|multiplicative|48|1|135|169|0|135|1.96|333408, 1396157, 9773596, 78184568, 1933641, 3798231, 2837397, 2763650, 3969818, 5671151, 45368536, 25634160, 33081345, 94597248, 567102080, 1270352384, 846887552, 261054528, 405065429, 40958648, 275825169, 109857978, 1544615959, 106708640416, 4127020819, 4136253112, 37528888316, 3059029198|
|practice_0097|multiplicative|48|2|129|138|0|129|10.15|1396157, 2837397, 333408, 5671151, 1933641, 2763650, 78184568, 3798231, 3969818, 94597248, 9773596, 1270352384, 33081345, 261054528, 45368536, 567102080, 25634160, 40958648, 1544615959, 4127020819, 275825169, 846887552, 4136253112, 37528888316, 109857978, 405065429, 3059029198, 106708640416|
|practice_0097|multiplicative|128|0|133|136|11|144|3.57|333408, 1396157, 1933641, 2763650, 2837397, 3798231, 3969818, 5671151, 9773596, 25634160, 33081345, 40958648, 45368536, 78184568, 94597248, 109857978, 261054528, 275825169, 405065429, 567102080, 846887552, 1270352384, 1544615959, 3059029198, 4127020819, 4136253112, 37528888316, 106708640416|
|practice_0097|multiplicative|128|1|134|134|0|134|5.46|333408, 1396157, 9773596, 78184568, 1933641, 3798231, 2763650, 2837397, 3969818, 5671151, 45368536, 261054528, 25634160, 33081345, 40958648, 94597248, 109857978, 275825169, 405065429, 567102080, 846887552, 1270352384, 1544615959, 4127020819, 4136253112, 3059029198, 37528888316, 106708640416|
|practice_0097|multiplicative|128|2|126|127|0|126|31.27|1396157, 2837397, 333408, 5671151, 1933641, 2763650, 3798231, 3969818, 9773596, 45368536, 40958648, 78184568, 109857978, 261054528, 275825169, 567102080, 25634160, 33081345, 94597248, 4136253112, 405065429, 846887552, 1270352384, 1544615959, 3059029198, 4127020819, 37528888316, 106708640416|
|practice_0098|tiny|0|0|2|0|0|2|0.01|2|
|practice_0098|tiny|0|1|2|0|0|2|0.01|2|
|practice_0098|tiny|0|2|2|0|0|2|0.01|2|
|practice_0098|tiny|48|0|2|3|0|2|0.01|2|
|practice_0098|tiny|48|1|2|3|0|2|0.01|2|
|practice_0098|tiny|48|2|2|3|0|2|0.01|2|
|practice_0098|tiny|128|0|2|3|0|2|0.01|2|
|practice_0098|tiny|128|1|2|3|0|2|0.01|2|
|practice_0098|tiny|128|2|2|3|0|2|0.01|2|
|practice_0099|multiplicative|0|0|132|0|0|132|2.41|180885, 358985, 1433070, 2098196, 2148263, 2845105, 2938494, 3751647, 4297693, 4298056, 4897755, 7163619, 7471000, 19205622, 33799613, 49698240, 63026892, 64471744, 275371728, 481383273, 576243072, 805146624, 1080468015, 7760358099, 15084558719, 30162306027, 34026684173, 41517969408|
|practice_0099|multiplicative|0|1|129|0|0|129|3.45|805146624, 180885, 358985, 1433070, 2938494, 2098196, 33799613, 2148263, 4297693, 4298056, 7163619, 2845105, 7471000, 49698240, 64471744, 3751647, 4897755, 19205622, 63026892, 275371728, 481383273, 576243072, 1080468015, 7760358099, 15084558719, 30162306027, 41517969408, 34026684173|
|practice_0099|multiplicative|0|2|127|0|0|127|21.37|180885, 4298056, 1433070, 2938494, 7163619, 4297693, 805146624, 358985, 2845105, 64471744, 4897755, 2098196, 7471000, 2148263, 3751647, 33799613, 63026892, 19205622, 49698240, 41517969408, 275371728, 1080468015, 30162306027, 481383273, 576243072, 34026684173, 7760358099, 15084558719|
|practice_0099|multiplicative|48|0|133|158|0|133|0.80|180885, 358985, 1433070, 2098196, 2148263, 2845105, 2938494, 3751647, 4297693, 4298056, 4897755, 7163619, 7471000, 19205622, 33799613, 49698240, 63026892, 64471744, 275371728, 481383273, 576243072, 805146624, 1080468015, 7760358099, 15084558719, 30162306027, 34026684173, 41517969408|
|practice_0099|multiplicative|48|1|138|143|0|138|1.63|805146624, 180885, 358985, 1433070, 2938494, 2098196, 33799613, 2148263, 4297693, 4298056, 7163619, 7471000, 2845105, 3751647, 19205622, 4897755, 49698240, 63026892, 64471744, 481383273, 275371728, 576243072, 1080468015, 7760358099, 41517969408, 15084558719, 30162306027, 34026684173|
|practice_0099|multiplicative|48|2|139|149|0|139|10.83|1433070, 358985, 180885, 2148263, 4298056, 4297693, 805146624, 2938494, 64471744, 2098196, 33799613, 3751647, 2845105, 19205622, 4897755, 7163619, 49698240, 576243072, 1080468015, 63026892, 15084558719, 7471000, 275371728, 7760358099, 481383273, 41517969408, 34026684173, 30162306027|
|practice_0099|multiplicative|128|0|127|129|0|127|3.72|180885, 358985, 1433070, 2098196, 2148263, 2845105, 2938494, 3751647, 4297693, 4298056, 4897755, 7163619, 7471000, 19205622, 33799613, 49698240, 63026892, 64471744, 275371728, 481383273, 576243072, 805146624, 1080468015, 7760358099, 15084558719, 30162306027, 34026684173, 41517969408|
|practice_0099|multiplicative|128|1|127|128|0|127|5.06|805146624, 180885, 358985, 1433070, 2938494, 2098196, 33799613, 2148263, 4297693, 4298056, 7163619, 2845105, 4897755, 7471000, 64471744, 3751647, 19205622, 49698240, 63026892, 275371728, 481383273, 576243072, 7760358099, 30162306027, 41517969408, 1080468015, 15084558719, 34026684173|
|practice_0099|multiplicative|128|2|125|127|0|125|29.91|1433070, 358985, 180885, 2148263, 4298056, 4297693, 2098196, 2938494, 2845105, 4897755, 7163619, 63026892, 64471744, 805146624, 3751647, 7471000, 49698240, 1080468015, 19205622, 33799613, 481383273, 275371728, 576243072, 7760358099, 41517969408, 15084558719, 30162306027, 34026684173|
|practice_0100|independent|0|0|342|0|0|342|3.74|125867224070491616, 138441496610859994, 143606351632728972, 164780631895145824, 190350173820163351, 192485007818548030, 217167267313224098, 223622759654173995, 229290164769898879, 229667744415993952, 236526068385871699, 264917354032648100, 321412598778939270, 361250679062729526, 398120826486896190, 525592575139715537, 530457039761122142, 530504879365159541, 549749463381938583, 561595283965183883, 605317056946648734, 617208682961924264, 620685837812176078, 636051366888137282, 649366289438767703, 692493891490775486, 764667801484923308, 817160275273569378, 823379440869269350, 908022472259176038, 935191169197221862|
|practice_0100|independent|0|1|318|0|305|623|16.19|138441496610859994, 164780631895145824, 125867224070491616, 143606351632728972, 649366289438767703, 692493891490775486, 636051366888137282, 190350173820163351, 229290164769898879, 236526068385871699, 229667744415993952, 192485007818548030, 223622759654173995, 264917354032648100, 530457039761122142, 530504879365159541, 217167267313224098, 764667801484923308, 561595283965183883, 817160275273569378, 321412598778939270, 361250679062729526, 525592575139715537, 823379440869269350, 908022472259176038, 398120826486896190, 617208682961924264, 935191169197221862, 549749463381938583, 605317056946648734, 620685837812176078|
|practice_0100|independent|0|2|312|0|273|585|104.58|164780631895145824, 138441496610859994, 525592575139715537, 530504879365159541, 125867224070491616, 229667744415993952, 264917354032648100, 223622759654173995, 649366289438767703, 398120826486896190, 361250679062729526, 192485007818548030, 817160275273569378, 321412598778939270, 549749463381938583, 908022472259176038, 229290164769898879, 530457039761122142, 190350173820163351, 823379440869269350, 236526068385871699, 620685837812176078, 605317056946648734, 935191169197221862, 617208682961924264, 692493891490775486, 143606351632728972, 636051366888137282, 561595283965183883, 217167267313224098, 764667801484923308|
|practice_0100|independent|48|0|354|546|0|354|1.67|125867224070491616, 138441496610859994, 143606351632728972, 164780631895145824, 190350173820163351, 192485007818548030, 217167267313224098, 223622759654173995, 229290164769898879, 229667744415993952, 236526068385871699, 264917354032648100, 321412598778939270, 361250679062729526, 398120826486896190, 525592575139715537, 530457039761122142, 530504879365159541, 549749463381938583, 561595283965183883, 605317056946648734, 617208682961924264, 620685837812176078, 636051366888137282, 649366289438767703, 692493891490775486, 764667801484923308, 817160275273569378, 823379440869269350, 908022472259176038, 935191169197221862|
|practice_0100|independent|48|1|346|402|0|346|6.62|138441496610859994, 164780631895145824, 236526068385871699, 223622759654173995, 229290164769898879, 605317056946648734, 217167267313224098, 617208682961924264, 361250679062729526, 549749463381938583, 692493891490775486, 125867224070491616, 143606351632728972, 229667744415993952, 398120826486896190, 525592575139715537, 190350173820163351, 561595283965183883, 192485007818548030, 620685837812176078, 636051366888137282, 530457039761122142, 530504879365159541, 321412598778939270, 264917354032648100, 649366289438767703, 764667801484923308, 817160275273569378, 908022472259176038, 823379440869269350, 935191169197221862|
|practice_0100|independent|48|2|339|374|0|339|42.13|164780631895145824, 649366289438767703, 236526068385871699, 229290164769898879, 143606351632728972, 823379440869269350, 223622759654173995, 229667744415993952, 125867224070491616, 530457039761122142, 321412598778939270, 361250679062729526, 530504879365159541, 549749463381938583, 525592575139715537, 561595283965183883, 264917354032648100, 190350173820163351, 636051366888137282, 138441496610859994, 398120826486896190, 620685837812176078, 605317056946648734, 817160275273569378, 908022472259176038, 217167267313224098, 192485007818548030, 692493891490775486, 617208682961924264, 764667801484923308, 935191169197221862|
|practice_0100|independent|128|0|308|346|18|326|11.03|125867224070491616, 138441496610859994, 143606351632728972, 164780631895145824, 190350173820163351, 192485007818548030, 217167267313224098, 223622759654173995, 229290164769898879, 229667744415993952, 236526068385871699, 264917354032648100, 321412598778939270, 361250679062729526, 398120826486896190, 525592575139715537, 530457039761122142, 530504879365159541, 549749463381938583, 561595283965183883, 605317056946648734, 617208682961924264, 620685837812176078, 636051366888137282, 649366289438767703, 692493891490775486, 764667801484923308, 817160275273569378, 823379440869269350, 908022472259176038, 935191169197221862|
|practice_0100|independent|128|1|297|299|12|309|44.74|138441496610859994, 164780631895145824, 605317056946648734, 321412598778939270, 190350173820163351, 143606351632728972, 125867224070491616, 192485007818548030, 217167267313224098, 223622759654173995, 229667744415993952, 229290164769898879, 236526068385871699, 264917354032648100, 361250679062729526, 530504879365159541, 530457039761122142, 525592575139715537, 398120826486896190, 549749463381938583, 561595283965183883, 617208682961924264, 636051366888137282, 649366289438767703, 620685837812176078, 692493891490775486, 823379440869269350, 817160275273569378, 764667801484923308, 908022472259176038, 935191169197221862|
|practice_0100|independent|128|2|284|288|43|327|302.86|164780631895145824, 138441496610859994, 525592575139715537, 530504879365159541, 143606351632728972, 229667744415993952, 190350173820163351, 125867224070491616, 264917354032648100, 223622759654173995, 530457039761122142, 229290164769898879, 236526068385871699, 620685837812176078, 561595283965183883, 692493891490775486, 321412598778939270, 617208682961924264, 649366289438767703, 549749463381938583, 361250679062729526, 764667801484923308, 605317056946648734, 908022472259176038, 636051366888137282, 823379440869269350, 398120826486896190, 192485007818548030, 817160275273569378, 935191169197221862, 217167267313224098|
|practice_0101|shared|0|0|168|0|0|168|2.72|23299063, 23299822, 29423977, 33466898, 33469997, 33470389, 33470432, 48078487, 96160699, 372802032, 549415451, 676202814, 1298169648, 1304795128, 1807495425, 1875134417, 2006049083, 2935791038, 5126557607, 9128746440, 16085528691, 242324290439, 463765744630, 1691019566199, 2180453932998, 3000502717450, 111537049938550, 124792193549232, 181931416747506|
|practice_0101|shared|0|1|161|0|0|161|6.83|33466898, 33469997, 33470389, 33470432, 372802032, 23299063, 23299822, 1304795128, 29423977, 48078487, 96160699, 1875134417, 1298169648, 549415451, 676202814, 1807495425, 2006049083, 2935791038, 5126557607, 9128746440, 242324290439, 16085528691, 3000502717450, 2180453932998, 463765744630, 1691019566199, 111537049938550, 181931416747506, 124792193549232|
|practice_0101|shared|0|2|155|0|0|155|40.19|33470432, 23299063, 23299822, 33470389, 33469997, 96160699, 372802032, 1304795128, 33466898, 48078487, 1298169648, 1875134417, 9128746440, 29423977, 676202814, 1807495425, 2935791038, 2006049083, 549415451, 5126557607, 16085528691, 242324290439, 1691019566199, 111537049938550, 3000502717450, 2180453932998, 463765744630, 181931416747506, 124792193549232|
|practice_0101|shared|48|0|175|217|0|175|1.42|23299063, 23299822, 29423977, 33466898, 33469997, 33470389, 33470432, 48078487, 96160699, 372802032, 549415451, 676202814, 1298169648, 1304795128, 1807495425, 1875134417, 2006049083, 2935791038, 5126557607, 9128746440, 16085528691, 242324290439, 463765744630, 1691019566199, 2180453932998, 3000502717450, 111537049938550, 124792193549232, 181931416747506|
|practice_0101|shared|48|1|163|172|0|163|3.12|33466898, 33469997, 33470389, 33470432, 372802032, 23299063, 23299822, 1304795128, 29423977, 48078487, 96160699, 676202814, 549415451, 1298169648, 1875134417, 2006049083, 2935791038, 9128746440, 1807495425, 5126557607, 16085528691, 1691019566199, 463765744630, 3000502717450, 111537049938550, 242324290439, 2180453932998, 181931416747506, 124792193549232|
|practice_0101|shared|48|2|166|173|0|166|20.49|33470432, 23299063, 23299822, 33470389, 33469997, 1304795128, 33466898, 549415451, 2935791038, 676202814, 372802032, 48078487, 1875134417, 9128746440, 1807495425, 1298169648, 96160699, 242324290439, 5126557607, 29423977, 463765744630, 16085528691, 3000502717450, 111537049938550, 2180453932998, 1691019566199, 181931416747506, 124792193549232, 2006049083|
|practice_0101|shared|128|0|153|159|3|156|4.45|23299063, 23299822, 29423977, 33466898, 33469997, 33470389, 33470432, 48078487, 96160699, 372802032, 549415451, 676202814, 1298169648, 1304795128, 1807495425, 1875134417, 2006049083, 2935791038, 5126557607, 9128746440, 16085528691, 242324290439, 463765744630, 1691019566199, 2180453932998, 3000502717450, 111537049938550, 124792193549232, 181931416747506|
|practice_0101|shared|128|1|149|152|18|167|9.37|33466898, 33469997, 33470389, 33470432, 372802032, 23299063, 23299822, 1304795128, 29423977, 48078487, 96160699, 1298169648, 1875134417, 549415451, 1807495425, 676202814, 2006049083, 2935791038, 16085528691, 9128746440, 5126557607, 1691019566199, 242324290439, 463765744630, 3000502717450, 2180453932998, 111537049938550, 124792193549232, 181931416747506|
|practice_0101|shared|128|2|148|152|1|149|63.81|33470432, 23299063, 23299822, 33470389, 33469997, 96160699, 48078487, 372802032, 1298169648, 33466898, 1875134417, 1304795128, 2935791038, 676202814, 29423977, 549415451, 463765744630, 5126557607, 9128746440, 2006049083, 1807495425, 16085528691, 1691019566199, 124792193549232, 242324290439, 3000502717450, 2180453932998, 111537049938550, 181931416747506|
|practice_0102|independent|0|0|307|0|0|307|3.17|119614911961970830, 122734130791731107, 139303265157253732, 167912908490272425, 183720330768192422, 218567313069197797, 260809672021753281, 271588698367259645, 292702636304383905, 324952554951602370, 334129737369164695, 432490634347295403, 464938691023305472, 482005754627286051, 515882383200782970, 545672012822009275, 569114968474305450, 686466144614481939, 705383743356564418, 709575084466384957, 820381880906546121, 827832391366968302, 835840983286674110, 837390662652426593, 847074509052636379, 902716231204869131, 915963903254103044, 973926015980588583|
|practice_0102|independent|0|1|288|0|52|340|11.81|271588698367259645, 432490634347295403, 119614911961970830, 260809672021753281, 324952554951602370, 686466144614481939, 464938691023305472, 292702636304383905, 139303265157253732, 167912908490272425, 334129737369164695, 122734130791731107, 569114968474305450, 515882383200782970, 902716231204869131, 709575084466384957, 218567313069197797, 183720330768192422, 482005754627286051, 545672012822009275, 705383743356564418, 837390662652426593, 820381880906546121, 915963903254103044, 835840983286674110, 827832391366968302, 847074509052636379, 973926015980588583|
|practice_0102|independent|0|2|285|0|146|431|86.73|902716231204869131, 183720330768192422, 915963903254103044, 686466144614481939, 464938691023305472, 324952554951602370, 218567313069197797, 119614911961970830, 705383743356564418, 139303265157253732, 837390662652426593, 847074509052636379, 260809672021753281, 569114968474305450, 167912908490272425, 709575084466384957, 334129737369164695, 271588698367259645, 482005754627286051, 122734130791731107, 820381880906546121, 292702636304383905, 835840983286674110, 515882383200782970, 432490634347295403, 827832391366968302, 545672012822009275, 973926015980588583|
|practice_0102|independent|48|0|309|443|0|309|1.44|119614911961970830, 122734130791731107, 139303265157253732, 167912908490272425, 183720330768192422, 218567313069197797, 260809672021753281, 271588698367259645, 292702636304383905, 324952554951602370, 334129737369164695, 432490634347295403, 464938691023305472, 482005754627286051, 515882383200782970, 545672012822009275, 569114968474305450, 686466144614481939, 705383743356564418, 709575084466384957, 820381880906546121, 827832391366968302, 835840983286674110, 837390662652426593, 847074509052636379, 902716231204869131, 915963903254103044, 973926015980588583|
|practice_0102|independent|48|1|310|319|0|310|5.01|271588698367259645, 432490634347295403, 119614911961970830, 260809672021753281, 334129737369164695, 686466144614481939, 218567313069197797, 292702636304383905, 183720330768192422, 482005754627286051, 324952554951602370, 820381880906546121, 847074509052636379, 515882383200782970, 122734130791731107, 705383743356564418, 569114968474305450, 709575084466384957, 835840983286674110, 139303265157253732, 902716231204869131, 837390662652426593, 167912908490272425, 827832391366968302, 464938691023305472, 545672012822009275, 973926015980588583, 915963903254103044|
|practice_0102|independent|48|2|311|330|0|311|36.56|902716231204869131, 167912908490272425, 324952554951602370, 271588698367259645, 139303265157253732, 432490634347295403, 709575084466384957, 705383743356564418, 569114968474305450, 482005754627286051, 820381880906546121, 835840983286674110, 119614911961970830, 837390662652426593, 218567313069197797, 827832391366968302, 915963903254103044, 515882383200782970, 686466144614481939, 122734130791731107, 334129737369164695, 260809672021753281, 183720330768192422, 545672012822009275, 464938691023305472, 292702636304383905, 847074509052636379, 973926015980588583|
|practice_0102|independent|128|0|275|291|20|295|9.47|119614911961970830, 122734130791731107, 139303265157253732, 167912908490272425, 183720330768192422, 218567313069197797, 260809672021753281, 271588698367259645, 292702636304383905, 324952554951602370, 334129737369164695, 432490634347295403, 464938691023305472, 482005754627286051, 515882383200782970, 545672012822009275, 569114968474305450, 686466144614481939, 705383743356564418, 709575084466384957, 820381880906546121, 827832391366968302, 835840983286674110, 837390662652426593, 847074509052636379, 902716231204869131, 915963903254103044, 973926015980588583|
|practice_0102|independent|128|1|268|271|18|286|35.37|271588698367259645, 432490634347295403, 119614911961970830, 260809672021753281, 837390662652426593, 827832391366968302, 122734130791731107, 464938691023305472, 139303265157253732, 515882383200782970, 183720330768192422, 324952554951602370, 569114968474305450, 686466144614481939, 709575084466384957, 292702636304383905, 847074509052636379, 218567313069197797, 334129737369164695, 545672012822009275, 167912908490272425, 705383743356564418, 973926015980588583, 915963903254103044, 482005754627286051, 835840983286674110, 820381880906546121, 902716231204869131|
|practice_0102|independent|128|2|263|269|94|357|258.13|902716231204869131, 183720330768192422, 218567313069197797, 139303265157253732, 119614911961970830, 167912908490272425, 122734130791731107, 847074509052636379, 686466144614481939, 271588698367259645, 334129737369164695, 432490634347295403, 835840983286674110, 464938691023305472, 705383743356564418, 482005754627286051, 324952554951602370, 820381880906546121, 515882383200782970, 260809672021753281, 545672012822009275, 837390662652426593, 569114968474305450, 827832391366968302, 709575084466384957, 292702636304383905, 915963903254103044, 973926015980588583|
|practice_0103|tiny|0|0|6|0|0|6|0.03|9, 16|
|practice_0103|tiny|0|1|6|0|0|6|0.01|9, 16|
|practice_0103|tiny|0|2|6|0|0|6|0.01|9, 16|
|practice_0103|tiny|48|0|6|11|0|6|0.01|9, 16|
|practice_0103|tiny|48|1|6|11|0|6|0.01|9, 16|
|practice_0103|tiny|48|2|6|11|0|6|0.01|9, 16|
|practice_0103|tiny|128|0|6|11|0|6|0.01|9, 16|
|practice_0103|tiny|128|1|6|11|0|6|0.01|9, 16|
|practice_0103|tiny|128|2|6|11|0|6|0.01|9, 16|
|practice_0104|independent|0|0|230|0|0|230|2.84|122871509429, 129370466977, 237488842048, 239727022678, 248180762582, 260792469119, 379856386862, 403520741852, 431454134182, 443304473751, 453653422200, 462831029226, 467230793317, 485962857655, 506904277366, 541380596563, 584943006053, 609222431448, 611152229468, 636525554817, 643336636139, 675356757633, 703517444325, 739834804694, 741102408089, 750926004373, 788994240219, 799874939406, 870366934130, 913438132415, 930677622303, 937987217429|
|practice_0104|independent|0|1|219|0|0|219|8.53|239727022678, 248180762582, 237488842048, 260792469119, 122871509429, 129370466977, 379856386862, 431454134182, 611152229468, 609222431448, 453653422200, 443304473751, 403520741852, 467230793317, 462831029226, 506904277366, 485962857655, 541380596563, 584943006053, 675356757633, 636525554817, 703517444325, 739834804694, 643336636139, 741102408089, 750926004373, 913438132415, 930677622303, 937987217429, 788994240219, 799874939406, 870366934130|
|practice_0104|independent|0|2|219|0|19|238|58.65|237488842048, 248180762582, 122871509429, 129370466977, 403520741852, 239727022678, 453653422200, 443304473751, 506904277366, 611152229468, 541380596563, 379856386862, 431454134182, 675356757633, 260792469119, 636525554817, 584943006053, 703517444325, 750926004373, 741102408089, 609222431448, 739834804694, 467230793317, 643336636139, 870366934130, 799874939406, 913438132415, 930677622303, 462831029226, 485962857655, 788994240219, 937987217429|
|practice_0104|independent|48|0|235|322|0|235|1.40|122871509429, 129370466977, 237488842048, 239727022678, 248180762582, 260792469119, 379856386862, 403520741852, 431454134182, 443304473751, 453653422200, 462831029226, 467230793317, 485962857655, 506904277366, 541380596563, 584943006053, 609222431448, 611152229468, 636525554817, 643336636139, 675356757633, 703517444325, 739834804694, 741102408089, 750926004373, 788994240219, 799874939406, 870366934130, 913438132415, 930677622303, 937987217429|
|practice_0104|independent|48|1|230|238|0|230|3.69|239727022678, 248180762582, 237488842048, 260792469119, 609222431448, 453653422200, 675356757633, 462831029226, 403520741852, 122871509429, 485962857655, 443304473751, 129370466977, 506904277366, 431454134182, 379856386862, 541380596563, 467230793317, 584943006053, 611152229468, 636525554817, 643336636139, 703517444325, 741102408089, 750926004373, 870366934130, 739834804694, 913438132415, 930677622303, 799874939406, 937987217429, 788994240219|
|practice_0104|independent|48|2|231|267|0|231|26.23|237488842048, 260792469119, 239727022678, 675356757633, 453653422200, 122871509429, 609222431448, 248180762582, 636525554817, 467230793317, 462831029226, 129370466977, 485962857655, 403520741852, 443304473751, 431454134182, 703517444325, 506904277366, 379856386862, 643336636139, 611152229468, 799874939406, 930677622303, 541380596563, 741102408089, 913438132415, 937987217429, 788994240219, 584943006053, 739834804694, 750926004373, 870366934130|
|practice_0104|independent|128|0|209|222|25|234|7.18|122871509429, 129370466977, 237488842048, 239727022678, 248180762582, 260792469119, 379856386862, 403520741852, 431454134182, 443304473751, 453653422200, 462831029226, 467230793317, 485962857655, 506904277366, 541380596563, 584943006053, 609222431448, 611152229468, 636525554817, 643336636139, 675356757633, 703517444325, 739834804694, 741102408089, 750926004373, 788994240219, 799874939406, 870366934130, 913438132415, 930677622303, 937987217429|
|practice_0104|independent|128|1|202|208|14|216|22.01|239727022678, 248180762582, 237488842048, 260792469119, 122871509429, 129370466977, 379856386862, 431454134182, 611152229468, 739834804694, 453653422200, 462831029226, 741102408089, 609222431448, 403520741852, 443304473751, 703517444325, 467230793317, 643336636139, 485962857655, 541380596563, 506904277366, 584943006053, 636525554817, 750926004373, 788994240219, 870366934130, 913438132415, 675356757633, 799874939406, 930677622303, 937987217429|
|practice_0104|independent|128|2|201|204|18|219|148.61|237488842048, 248180762582, 239727022678, 462831029226, 443304473751, 453653422200, 129370466977, 467230793317, 260792469119, 541380596563, 506904277366, 750926004373, 609222431448, 739834804694, 788994240219, 799874939406, 584943006053, 611152229468, 643336636139, 636525554817, 741102408089, 122871509429, 675356757633, 379856386862, 431454134182, 870366934130, 403520741852, 703517444325, 930677622303, 485962857655, 913438132415, 937987217429|
|practice_0105|mixed|0|0|195|0|0|195|2.48|1966292, 3928404, 7329225, 8225577, 87948160, 95285628, 273320117, 353698136, 613453640, 1104266019, 2350518718, 2377912530, 20152558300, 23957038272, 113427555540, 115811538618, 367731959522, 444304087114, 491904647816, 542761305214, 646968936180, 670035274115, 759443976528, 799839265418, 815299280249, 817437121300, 843879145060, 893114448217, 927088567158, 961381528015, 961385683549|
|practice_0105|mixed|0|1|188|0|0|188|6.09|1966292, 3928404, 7329225, 87948160, 8225577, 95285628, 273320117, 353698136, 613453640, 1104266019, 2377912530, 2350518718, 20152558300, 23957038272, 113427555540, 115811538618, 542761305214, 646968936180, 799839265418, 367731959522, 817437121300, 444304087114, 815299280249, 843879145060, 670035274115, 927088567158, 893114448217, 961385683549, 961381528015, 491904647816, 759443976528|
|practice_0105|mixed|0|2|186|0|0|186|51.59|7329225, 3928404, 8225577, 353698136, 20152558300, 95285628, 87948160, 1966292, 2377912530, 23957038272, 2350518718, 1104266019, 273320117, 613453640, 367731959522, 113427555540, 115811538618, 646968936180, 843879145060, 542761305214, 759443976528, 444304087114, 799839265418, 961381528015, 961385683549, 491904647816, 927088567158, 893114448217, 815299280249, 817437121300, 670035274115|
|practice_0105|mixed|48|0|195|235|0|195|1.34|1966292, 3928404, 7329225, 8225577, 87948160, 95285628, 273320117, 353698136, 613453640, 1104266019, 2350518718, 2377912530, 20152558300, 23957038272, 113427555540, 115811538618, 367731959522, 444304087114, 491904647816, 542761305214, 646968936180, 670035274115, 759443976528, 799839265418, 815299280249, 817437121300, 843879145060, 893114448217, 927088567158, 961381528015, 961385683549|
|practice_0105|mixed|48|1|193|202|0|193|3.13|1966292, 3928404, 7329225, 87948160, 8225577, 95285628, 273320117, 353698136, 2377912530, 613453640, 1104266019, 2350518718, 20152558300, 23957038272, 113427555540, 115811538618, 367731959522, 444304087114, 542761305214, 491904647816, 646968936180, 670035274115, 843879145060, 799839265418, 759443976528, 893114448217, 815299280249, 961381528015, 961385683549, 817437121300, 927088567158|
|practice_0105|mixed|48|2|199|207|0|199|22.53|7329225, 8225577, 1966292, 3928404, 353698136, 87948160, 1104266019, 20152558300, 273320117, 113427555540, 2350518718, 95285628, 613453640, 2377912530, 444304087114, 542761305214, 670035274115, 115811538618, 799839265418, 893114448217, 23957038272, 491904647816, 843879145060, 815299280249, 961385683549, 367731959522, 961381528015, 817437121300, 927088567158, 759443976528, 646968936180|
|practice_0105|mixed|128|0|178|183|58|236|5.39|1966292, 3928404, 7329225, 8225577, 87948160, 95285628, 273320117, 353698136, 613453640, 1104266019, 2350518718, 2377912530, 20152558300, 23957038272, 113427555540, 115811538618, 367731959522, 444304087114, 491904647816, 542761305214, 646968936180, 670035274115, 759443976528, 799839265418, 815299280249, 817437121300, 843879145060, 893114448217, 927088567158, 961381528015, 961385683549|
|practice_0105|mixed|128|1|172|176|0|172|15.40|1966292, 3928404, 7329225, 87948160, 8225577, 95285628, 273320117, 353698136, 2377912530, 613453640, 1104266019, 2350518718, 20152558300, 23957038272, 113427555540, 115811538618, 961381528015, 961385683549, 491904647816, 367731959522, 444304087114, 542761305214, 927088567158, 646968936180, 799839265418, 759443976528, 670035274115, 817437121300, 843879145060, 815299280249, 893114448217|
|practice_0105|mixed|128|2|176|177|26|202|109.68|7329225, 8225577, 1966292, 273320117, 87948160, 353698136, 3928404, 1104266019, 23957038272, 2377912530, 95285628, 670035274115, 613453640, 2350518718, 20152558300, 646968936180, 815299280249, 817437121300, 113427555540, 367731959522, 542761305214, 115811538618, 799839265418, 444304087114, 961381528015, 961385683549, 491904647816, 759443976528, 843879145060, 893114448217, 927088567158|
|practice_0106|shared|0|0|180|0|0|180|3.70|33272164377, 81285633208, 126070701838, 126070702875, 126070707607, 232905158682, 307997314540, 358326748947, 499082378054, 529474018722, 598898937924, 838422173936, 1197797874620, 1463141418916, 1512848477232, 5543951791400, 10707037617286, 11087903587500, 22175807162760, 26336545156476, 47652660730040, 70122362612389, 183614456158506, 360048869042751, 367228915765056, 1675535878208269, 8167629355804800, 420537079076077330, 715484331568500480|
|practice_0106|shared|0|1|176|0|0|176|12.24|33272164377, 232905158682, 598898937924, 1197797874620, 499082378054, 81285633208, 26336545156476, 1463141418916, 126070701838, 126070702875, 126070707607, 1512848477232, 307997314540, 5543951791400, 11087903587500, 22175807162760, 360048869042751, 529474018722, 47652660730040, 358326748947, 838422173936, 183614456158506, 367228915765056, 10707037617286, 1675535878208269, 8167629355804800, 715484331568500480, 420537079076077330, 70122362612389|
|practice_0106|shared|0|2|167|0|0|167|67.35|126070702875, 232905158682, 81285633208, 126070707607, 126070701838, 598898937924, 1197797874620, 307997314540, 22175807162760, 33272164377, 11087903587500, 26336545156476, 1463141418916, 5543951791400, 360048869042751, 838422173936, 358326748947, 8167629355804800, 715484331568500480, 367228915765056, 499082378054, 1512848477232, 529474018722, 420537079076077330, 10707037617286, 1675535878208269, 183614456158506, 47652660730040, 70122362612389|
|practice_0106|shared|48|0|195|243|0|195|1.43|33272164377, 81285633208, 126070701838, 126070702875, 126070707607, 232905158682, 307997314540, 358326748947, 499082378054, 529474018722, 598898937924, 838422173936, 1197797874620, 1463141418916, 1512848477232, 5543951791400, 10707037617286, 11087903587500, 22175807162760, 26336545156476, 47652660730040, 70122362612389, 183614456158506, 360048869042751, 367228915765056, 1675535878208269, 8167629355804800, 420537079076077330, 715484331568500480|
|practice_0106|shared|48|1|181|211|0|181|4.15|33272164377, 232905158682, 598898937924, 1197797874620, 499082378054, 81285633208, 1463141418916, 126070707607, 126070702875, 126070701838, 1512848477232, 26336545156476, 307997314540, 5543951791400, 11087903587500, 22175807162760, 360048869042751, 358326748947, 1675535878208269, 838422173936, 8167629355804800, 715484331568500480, 183614456158506, 367228915765056, 47652660730040, 529474018722, 70122362612389, 10707037617286, 420537079076077330|
|practice_0106|shared|48|2|192|204|0|192|29.71|126070702875, 81285633208, 33272164377, 1463141418916, 126070701838, 598898937924, 1197797874620, 1512848477232, 307997314540, 11087903587500, 499082378054, 529474018722, 22175807162760, 232905158682, 358326748947, 838422173936, 183614456158506, 5543951791400, 715484331568500480, 360048869042751, 126070707607, 10707037617286, 420537079076077330, 8167629355804800, 1675535878208269, 70122362612389, 26336545156476, 47652660730040, 367228915765056|
|practice_0106|shared|128|0|174|179|36|210|6.20|33272164377, 81285633208, 126070701838, 126070702875, 126070707607, 232905158682, 307997314540, 358326748947, 499082378054, 529474018722, 598898937924, 838422173936, 1197797874620, 1463141418916, 1512848477232, 5543951791400, 10707037617286, 11087903587500, 22175807162760, 26336545156476, 47652660730040, 70122362612389, 183614456158506, 360048869042751, 367228915765056, 1675535878208269, 8167629355804800, 420537079076077330, 715484331568500480|
|practice_0106|shared|128|1|166|169|0|166|21.57|33272164377, 232905158682, 598898937924, 1197797874620, 499082378054, 81285633208, 26336545156476, 1463141418916, 126070701838, 126070702875, 126070707607, 1512848477232, 307997314540, 5543951791400, 11087903587500, 22175807162760, 358326748947, 529474018722, 838422173936, 183614456158506, 367228915765056, 47652660730040, 360048869042751, 1675535878208269, 10707037617286, 70122362612389, 8167629355804800, 715484331568500480, 420537079076077330|
|practice_0106|shared|128|2|161|163|0|161|108.52|126070702875, 81285633208, 33272164377, 598898937924, 1463141418916, 1512848477232, 126070701838, 126070707607, 5543951791400, 11087903587500, 232905158682, 22175807162760, 307997314540, 26336545156476, 499082378054, 1197797874620, 838422173936, 358326748947, 360048869042751, 715484331568500480, 8167629355804800, 183614456158506, 529474018722, 367228915765056, 10707037617286, 47652660730040, 1675535878208269, 420537079076077330, 70122362612389|
|practice_0107|shared|0|0|125|0|0|125|2.30|221809, 633883, 1063316, 1063559, 1063579, 7660408, 15325770, 18372919, 35700808, 36746286, 71401288, 94656199, 133003795, 141834420, 189307984, 214203761, 238851067, 275596666, 368769297, 734925076, 881909904, 1310293298, 1616836945, 3270416227, 9639168750, 10174678125, 16848766943, 21991575760, 190855540386|
|practice_0107|shared|0|1|120|0|0|120|3.90|221809, 633883, 1063559, 1063316, 1063579, 7660408, 15325770, 18372919, 36746286, 238851067, 275596666, 734925076, 881909904, 35700808, 141834420, 368769297, 71401288, 214203761, 9639168750, 10174678125, 21991575760, 133003795, 94656199, 189307984, 1616836945, 3270416227, 190855540386, 1310293298, 16848766943|
|practice_0107|shared|0|2|119|0|0|119|20.83|1063579, 1063559, 18372919, 238851067, 1063316, 221809, 633883, 71401288, 10174678125, 36746286, 9639168750, 35700808, 214203761, 141834420, 881909904, 7660408, 15325770, 275596666, 133003795, 189307984, 368769297, 1310293298, 94656199, 734925076, 1616836945, 3270416227, 21991575760, 190855540386, 16848766943|
|practice_0107|shared|48|0|135|163|0|135|0.93|221809, 633883, 1063316, 1063559, 1063579, 7660408, 15325770, 18372919, 35700808, 36746286, 71401288, 94656199, 133003795, 141834420, 189307984, 214203761, 238851067, 275596666, 368769297, 734925076, 881909904, 1310293298, 1616836945, 3270416227, 9639168750, 10174678125, 16848766943, 21991575760, 190855540386|
|practice_0107|shared|48|1|126|132|0|126|2.07|221809, 633883, 1063559, 1063316, 1063579, 7660408, 18372919, 36746286, 238851067, 275596666, 15325770, 189307984, 35700808, 214203761, 71401288, 9639168750, 10174678125, 734925076, 94656199, 368769297, 141834420, 881909904, 1616836945, 3270416227, 21991575760, 133003795, 1310293298, 190855540386, 16848766943|
|practice_0107|shared|48|2|123|138|0|123|10.38|1063559, 1063579, 18372919, 238851067, 1063316, 35700808, 71401288, 214203761, 36746286, 9639168750, 734925076, 633883, 881909904, 7660408, 10174678125, 275596666, 21991575760, 94656199, 15325770, 221809, 133003795, 1616836945, 3270416227, 189307984, 141834420, 1310293298, 16848766943, 190855540386, 368769297|
|practice_0107|shared|128|0|124|126|0|124|3.48|221809, 633883, 1063316, 1063559, 1063579, 7660408, 15325770, 18372919, 35700808, 36746286, 71401288, 94656199, 133003795, 141834420, 189307984, 214203761, 238851067, 275596666, 368769297, 734925076, 881909904, 1310293298, 1616836945, 3270416227, 9639168750, 10174678125, 16848766943, 21991575760, 190855540386|
|practice_0107|shared|128|1|125|127|0|125|5.84|221809, 633883, 1063559, 1063316, 1063579, 7660408, 15325770, 18372919, 36746286, 238851067, 275596666, 734925076, 35700808, 71401288, 214203761, 9639168750, 10174678125, 21991575760, 133003795, 141834420, 189307984, 94656199, 368769297, 881909904, 1616836945, 3270416227, 190855540386, 1310293298, 16848766943|
|practice_0107|shared|128|2|111|115|0|111|26.76|1063559, 1063579, 18372919, 238851067, 1063316, 35700808, 633883, 71401288, 10174678125, 36746286, 9639168750, 189307984, 221809, 214203761, 275596666, 734925076, 881909904, 3270416227, 141834420, 368769297, 190855540386, 94656199, 7660408, 1616836945, 15325770, 133003795, 1310293298, 16848766943, 21991575760|
|practice_0108|multiplicative|0|0|119|0|0|119|2.17|30385, 71058, 71342, 125786, 149879, 172358, 172851, 175693, 204663, 221318, 226928, 233492, 614123, 702770, 843648, 904724, 1664749, 1810876, 2068872, 4960630, 8590574, 12021366, 14034228, 29883960, 32058896, 35312069, 49812448, 57495248, 70624008, 95825317|
|practice_0108|multiplicative|0|1|112|0|0|112|2.64|175693, 702770, 30385, 71058, 71342, 149879, 172358, 172851, 125786, 204663, 221318, 226928, 233492, 614123, 843648, 904724, 1664749, 1810876, 2068872, 8590574, 4960630, 14034228, 32058896, 29883960, 49812448, 70624008, 35312069, 57495248, 12021366, 95825317|
|practice_0108|multiplicative|0|2|113|0|0|113|16.08|175693, 71058, 30385, 71342, 125786, 702770, 172358, 233492, 614123, 204663, 221318, 172851, 904724, 1664749, 2068872, 1810876, 149879, 14034228, 4960630, 226928, 70624008, 843648, 8590574, 12021366, 32058896, 35312069, 57495248, 29883960, 49812448, 95825317|
|practice_0108|multiplicative|48|0|124|137|0|124|0.82|30385, 71058, 71342, 125786, 149879, 172358, 172851, 175693, 204663, 221318, 226928, 233492, 614123, 702770, 843648, 904724, 1664749, 1810876, 2068872, 4960630, 8590574, 12021366, 14034228, 29883960, 32058896, 35312069, 49812448, 57495248, 70624008, 95825317|
|practice_0108|multiplicative|48|1|111|118|0|111|1.25|175693, 702770, 30385, 71058, 71342, 149879, 172358, 172851, 221318, 125786, 204663, 226928, 233492, 614123, 904724, 843648, 1810876, 2068872, 4960630, 1664749, 14034228, 8590574, 12021366, 29883960, 32058896, 35312069, 49812448, 70624008, 95825317, 57495248|
|practice_0108|multiplicative|48|2|118|123|0|118|8.42|175693, 71058, 30385, 71342, 125786, 702770, 172358, 172851, 226928, 204663, 614123, 149879, 2068872, 233492, 843648, 1810876, 70624008, 221318, 35312069, 4960630, 904724, 1664749, 12021366, 14034228, 29883960, 8590574, 32058896, 57495248, 49812448, 95825317|
|practice_0108|multiplicative|128|0|114|115|0|114|3.77|30385, 71058, 71342, 125786, 149879, 172358, 172851, 175693, 204663, 221318, 226928, 233492, 614123, 702770, 843648, 904724, 1664749, 1810876, 2068872, 4960630, 8590574, 12021366, 14034228, 29883960, 32058896, 35312069, 49812448, 57495248, 70624008, 95825317|
|practice_0108|multiplicative|128|1|109|111|0|109|3.78|175693, 702770, 30385, 71058, 71342, 149879, 172358, 172851, 221318, 125786, 204663, 226928, 233492, 614123, 843648, 904724, 1664749, 1810876, 2068872, 4960630, 8590574, 12021366, 14034228, 29883960, 32058896, 49812448, 70624008, 35312069, 57495248, 95825317|
|practice_0108|multiplicative|128|2|110|113|0|110|25.10|175693, 71058, 30385, 71342, 125786, 702770, 172358, 172851, 226928, 843648, 904724, 233492, 204663, 1810876, 4960630, 149879, 29883960, 221318, 614123, 70624008, 14034228, 1664749, 2068872, 35312069, 8590574, 12021366, 32058896, 49812448, 57495248, 95825317|
|practice_0109|multiplicative|0|0|150|0|0|150|4.27|42829, 62022, 103504, 123234, 143739, 193901, 310575, 321554, 414228, 532518, 3489872, 4259514, 6433699, 9306153, 10858048, 11212045, 13251968, 37244348, 52502190, 56058564, 87888424, 93070078, 105453428, 196468470, 216990487, 1354686850, 4859299328, 4909921920, 7309689960, 9258719488, 8732726309508, 23409002618880|
|practice_0109|multiplicative|0|1|144|0|0|144|5.60|42829, 123234, 62022, 103504, 143739, 310575, 321554, 414228, 532518, 193901, 3489872, 6433699, 4259514, 13251968, 37244348, 10858048, 9306153, 11212045, 52502190, 56058564, 87888424, 93070078, 105453428, 196468470, 4859299328, 4909921920, 216990487, 1354686850, 7309689960, 9258719488, 23409002618880, 8732726309508|
|practice_0109|multiplicative|0|2|135|0|0|135|42.43|42829, 123234, 62022, 103504, 13251968, 310575, 321554, 37244348, 414228, 143739, 10858048, 193901, 9306153, 532518, 4259514, 6433699, 3489872, 196468470, 93070078, 216990487, 11212045, 52502190, 87888424, 105453428, 1354686850, 4859299328, 56058564, 4909921920, 7309689960, 9258719488, 23409002618880, 8732726309508|
|practice_0109|multiplicative|48|0|142|162|0|142|1.35|42829, 62022, 103504, 123234, 143739, 193901, 310575, 321554, 414228, 532518, 3489872, 4259514, 6433699, 9306153, 10858048, 11212045, 13251968, 37244348, 52502190, 56058564, 87888424, 93070078, 105453428, 196468470, 216990487, 1354686850, 4859299328, 4909921920, 7309689960, 9258719488, 8732726309508, 23409002618880|
|practice_0109|multiplicative|48|1|150|207|0|150|2.19|42829, 123234, 62022, 103504, 143739, 310575, 414228, 532518, 193901, 321554, 6433699, 13251968, 9306153, 3489872, 10858048, 4259514, 11212045, 37244348, 52502190, 87888424, 105453428, 56058564, 196468470, 93070078, 216990487, 1354686850, 4859299328, 4909921920, 9258719488, 7309689960, 23409002618880, 8732726309508|
|practice_0109|multiplicative|48|2|149|157|0|149|14.68|42829, 123234, 62022, 103504, 13251968, 310575, 37244348, 143739, 414228, 321554, 532518, 11212045, 4259514, 193901, 6433699, 56058564, 52502190, 3489872, 10858048, 216990487, 105453428, 9306153, 4909921920, 196468470, 4859299328, 87888424, 93070078, 9258719488, 1354686850, 7309689960, 23409002618880, 8732726309508|
|practice_0109|multiplicative|128|0|136|138|25|161|3.96|42829, 62022, 103504, 123234, 143739, 193901, 310575, 321554, 414228, 532518, 3489872, 4259514, 6433699, 9306153, 10858048, 11212045, 13251968, 37244348, 52502190, 56058564, 87888424, 93070078, 105453428, 196468470, 216990487, 1354686850, 4859299328, 4909921920, 7309689960, 9258719488, 8732726309508, 23409002618880|
|practice_0109|multiplicative|128|1|132|138|1|133|7.10|42829, 123234, 62022, 103504, 143739, 310575, 321554, 414228, 532518, 193901, 3489872, 6433699, 13251968, 37244348, 4259514, 9306153, 10858048, 11212045, 52502190, 56058564, 196468470, 87888424, 93070078, 105453428, 216990487, 1354686850, 4859299328, 4909921920, 7309689960, 9258719488, 23409002618880, 8732726309508|
|practice_0109|multiplicative|128|2|130|136|4|134|45.69|42829, 123234, 62022, 103504, 13251968, 310575, 532518, 321554, 414228, 10858048, 37244348, 143739, 193901, 4259514, 6433699, 3489872, 9306153, 87888424, 216990487, 11212045, 105453428, 56058564, 52502190, 9258719488, 93070078, 196468470, 1354686850, 4859299328, 4909921920, 7309689960, 23409002618880, 8732726309508|
|practice_0110|independent|0|0|268|0|0|268|3.87|109952327715897, 120997011882493, 160867708781278, 187215212799823, 195586250753441, 206990782150147, 236449218849510, 239681562089664, 276330957422573, 342859732309408, 398499633281304, 484409125906768, 506165313149232, 587251452295837, 604915989868811, 627357459238687, 641744016230072, 656602416766274, 658660703519880, 727237213018261, 787126016084085, 787798131932230, 823810258870502, 849350009314702, 850362365920313, 854798288289765, 856132710863010, 864592961764819, 870354465043280, 959494388105951|
|practice_0110|independent|0|1|252|0|65|317|12.27|120997011882493, 206990782150147, 959494388105951, 109952327715897, 276330957422573, 187215212799823, 236449218849510, 342859732309408, 604915989868811, 398499633281304, 864592961764819, 870354465043280, 160867708781278, 239681562089664, 854798288289765, 787798131932230, 856132710863010, 656602416766274, 506165313149232, 195586250753441, 484409125906768, 787126016084085, 587251452295837, 627357459238687, 641744016230072, 658660703519880, 823810258870502, 849350009314702, 850362365920313, 727237213018261|
|practice_0110|independent|0|2|249|0|164|413|85.39|120997011882493, 342859732309408, 604915989868811, 484409125906768, 195586250753441, 187215212799823, 506165313149232, 109952327715897, 239681562089664, 587251452295837, 856132710863010, 160867708781278, 236449218849510, 627357459238687, 849350009314702, 276330957422573, 850362365920313, 641744016230072, 727237213018261, 787798131932230, 787126016084085, 823810258870502, 656602416766274, 206990782150147, 658660703519880, 864592961764819, 398499633281304, 854798288289765, 870354465043280, 959494388105951|
|practice_0110|independent|48|0|267|375|0|267|1.58|109952327715897, 120997011882493, 160867708781278, 187215212799823, 195586250753441, 206990782150147, 236449218849510, 239681562089664, 276330957422573, 342859732309408, 398499633281304, 484409125906768, 506165313149232, 587251452295837, 604915989868811, 627357459238687, 641744016230072, 656602416766274, 658660703519880, 727237213018261, 787126016084085, 787798131932230, 823810258870502, 849350009314702, 850362365920313, 854798288289765, 856132710863010, 864592961764819, 870354465043280, 959494388105951|
|practice_0110|independent|48|1|272|284|0|272|4.54|120997011882493, 109952327715897, 187215212799823, 236449218849510, 239681562089664, 398499633281304, 506165313149232, 160867708781278, 195586250753441, 206990782150147, 276330957422573, 627357459238687, 856132710863010, 587251452295837, 727237213018261, 854798288289765, 870354465043280, 484409125906768, 342859732309408, 656602416766274, 658660703519880, 850362365920313, 849350009314702, 604915989868811, 823810258870502, 641744016230072, 787126016084085, 787798131932230, 959494388105951, 864592961764819|
|practice_0110|independent|48|2|269|278|0|269|32.65|239681562089664, 276330957422573, 484409125906768, 506165313149232, 587251452295837, 854798288289765, 195586250753441, 959494388105951, 856132710863010, 627357459238687, 109952327715897, 187215212799823, 641744016230072, 236449218849510, 823810258870502, 604915989868811, 658660703519880, 398499633281304, 656602416766274, 870354465043280, 727237213018261, 849350009314702, 864592961764819, 787798131932230, 850362365920313, 342859732309408, 160867708781278, 787126016084085, 206990782150147, 120997011882493|
|practice_0110|independent|128|0|244|257|8|252|8.30|109952327715897, 120997011882493, 160867708781278, 187215212799823, 195586250753441, 206990782150147, 236449218849510, 239681562089664, 276330957422573, 342859732309408, 398499633281304, 484409125906768, 506165313149232, 587251452295837, 604915989868811, 627357459238687, 641744016230072, 656602416766274, 658660703519880, 727237213018261, 787126016084085, 787798131932230, 823810258870502, 849350009314702, 850362365920313, 854798288289765, 856132710863010, 864592961764819, 870354465043280, 959494388105951|
|practice_0110|independent|128|1|232|239|6|238|26.26|120997011882493, 109952327715897, 236449218849510, 239681562089664, 398499633281304, 276330957422573, 604915989868811, 206990782150147, 506165313149232, 342859732309408, 160867708781278, 656602416766274, 195586250753441, 187215212799823, 484409125906768, 856132710863010, 727237213018261, 587251452295837, 627357459238687, 641744016230072, 849350009314702, 658660703519880, 850362365920313, 787798131932230, 870354465043280, 787126016084085, 823810258870502, 854798288289765, 864592961764819, 959494388105951|
|practice_0110|independent|128|2|235|238|48|283|192.94|239681562089664, 120997011882493, 195586250753441, 506165313149232, 641744016230072, 276330957422573, 484409125906768, 109952327715897, 187215212799823, 398499633281304, 236449218849510, 342859732309408, 627357459238687, 604915989868811, 787126016084085, 160867708781278, 656602416766274, 854798288289765, 587251452295837, 658660703519880, 864592961764819, 850362365920313, 823810258870502, 856132710863010, 849350009314702, 787798131932230, 206990782150147, 870354465043280, 727237213018261, 959494388105951|
|practice_0111|shared|0|0|183|0|0|183|3.16|5835218, 67819071, 79887696, 79887838, 789277317, 1213969548, 1213978738, 1278206608, 1488768495, 1544860784, 2607784200, 3301136109, 12628507568, 22288706267, 29102711780, 41275010634, 53684568323, 63484478641, 96384321520, 107370420730, 215510100444, 628262673371, 872586998325, 1355999323866, 3452033624854, 7054520474098, 56436204684273, 338164647023808, 520242864187248|
|practice_0111|shared|0|1|175|0|0|175|7.34|5835218, 67819071, 79887696, 79887838, 1278206608, 22288706267, 789277317, 12628507568, 1213969548, 1213978738, 2607784200, 1488768495, 1544860784, 3301136109, 41275010634, 29102711780, 53684568323, 63484478641, 96384321520, 107370420730, 215510100444, 628262673371, 520242864187248, 872586998325, 1355999323866, 3452033624854, 7054520474098, 56436204684273, 338164647023808|
|practice_0111|shared|0|2|171|0|0|171|51.16|1213969548, 79887838, 1278206608, 79887696, 67819071, 5835218, 1213978738, 789277317, 3301136109, 2607784200, 1488768495, 12628507568, 29102711780, 107370420730, 41275010634, 22288706267, 96384321520, 56436204684273, 53684568323, 1544860784, 63484478641, 7054520474098, 520242864187248, 215510100444, 338164647023808, 628262673371, 872586998325, 1355999323866, 3452033624854|
|practice_0111|shared|48|0|186|207|0|186|1.33|5835218, 67819071, 79887696, 79887838, 789277317, 1213969548, 1213978738, 1278206608, 1488768495, 1544860784, 2607784200, 3301136109, 12628507568, 22288706267, 29102711780, 41275010634, 53684568323, 63484478641, 96384321520, 107370420730, 215510100444, 628262673371, 872586998325, 1355999323866, 3452033624854, 7054520474098, 56436204684273, 338164647023808, 520242864187248|
|practice_0111|shared|48|1|184|190|0|184|3.49|5835218, 67819071, 79887696, 79887838, 1278206608, 22288706267, 789277317, 12628507568, 1213969548, 1213978738, 1544860784, 1488768495, 2607784200, 3301136109, 41275010634, 29102711780, 53684568323, 107370420730, 215510100444, 63484478641, 96384321520, 1355999323866, 56436204684273, 628262673371, 872586998325, 3452033624854, 520242864187248, 338164647023808, 7054520474098|
|practice_0111|shared|48|2|173|187|0|173|21.54|1213969548, 79887838, 1278206608, 79887696, 1488768495, 1213978738, 5835218, 67819071, 2607784200, 22288706267, 12628507568, 63484478641, 789277317, 29102711780, 215510100444, 872586998325, 53684568323, 628262673371, 41275010634, 107370420730, 3301136109, 338164647023808, 1355999323866, 1544860784, 7054520474098, 96384321520, 520242864187248, 3452033624854, 56436204684273|
|practice_0111|shared|128|0|165|171|17|182|5.88|5835218, 67819071, 79887696, 79887838, 789277317, 1213969548, 1213978738, 1278206608, 1488768495, 1544860784, 2607784200, 3301136109, 12628507568, 22288706267, 29102711780, 41275010634, 53684568323, 63484478641, 96384321520, 107370420730, 215510100444, 628262673371, 872586998325, 1355999323866, 3452033624854, 7054520474098, 56436204684273, 338164647023808, 520242864187248|
|practice_0111|shared|128|1|162|167|5|167|14.17|5835218, 67819071, 79887696, 79887838, 1278206608, 22288706267, 789277317, 12628507568, 1213969548, 1213978738, 2607784200, 1488768495, 1544860784, 3301136109, 29102711780, 41275010634, 53684568323, 107370420730, 63484478641, 96384321520, 215510100444, 872586998325, 7054520474098, 338164647023808, 56436204684273, 628262673371, 1355999323866, 520242864187248, 3452033624854|
|practice_0111|shared|128|2|163|166|13|176|83.55|1213969548, 79887838, 1278206608, 79887696, 67819071, 5835218, 1213978738, 789277317, 3301136109, 29102711780, 1488768495, 2607784200, 1544860784, 12628507568, 22288706267, 215510100444, 41275010634, 63484478641, 53684568323, 96384321520, 107370420730, 1355999323866, 872586998325, 7054520474098, 520242864187248, 628262673371, 3452033624854, 338164647023808, 56436204684273|
|practice_0112|multiplicative|0|0|129|0|0|129|3.35|7896, 9399, 14372, 15437, 17739, 17961, 25242, 26924, 28835, 33768, 44853, 54492, 71912, 75783, 85436, 102922, 154360, 246300, 291519, 566524, 827824, 9450882, 12350440, 34211451, 52957646, 99266648, 1773647215, 14553009998, 21804600384, 31197007840, 33896084480, 54692434170|
|practice_0112|multiplicative|0|1|124|0|0|124|4.99|7896, 9399, 14372, 33768, 85436, 15437, 17739, 26924, 17961, 28835, 25242, 44853, 71912, 75783, 246300, 291519, 54492, 102922, 154360, 566524, 827824, 12350440, 9450882, 34211451, 52957646, 99266648, 33896084480, 1773647215, 21804600384, 31197007840, 14553009998, 54692434170|
|practice_0112|multiplicative|0|2|118|0|0|118|26.50|17739, 7896, 17961, 9399, 14372, 25242, 44853, 75783, 26924, 71912, 28835, 33768, 102922, 246300, 54492, 85436, 291519, 15437, 154360, 827824, 99266648, 566524, 12350440, 9450882, 52957646, 1773647215, 54692434170, 34211451, 21804600384, 14553009998, 31197007840, 33896084480|
|practice_0112|multiplicative|48|0|131|153|0|131|1.13|7896, 9399, 14372, 15437, 17739, 17961, 25242, 26924, 28835, 33768, 44853, 54492, 71912, 75783, 85436, 102922, 154360, 246300, 291519, 566524, 827824, 9450882, 12350440, 34211451, 52957646, 99266648, 1773647215, 14553009998, 21804600384, 31197007840, 33896084480, 54692434170|
|practice_0112|multiplicative|48|1|132|138|0|132|1.58|7896, 9399, 14372, 33768, 85436, 15437, 17739, 17961, 25242, 28835, 71912, 75783, 291519, 26924, 44853, 54492, 102922, 246300, 154360, 827824, 566524, 9450882, 12350440, 34211451, 52957646, 99266648, 14553009998, 21804600384, 1773647215, 33896084480, 54692434170, 31197007840|
|practice_0112|multiplicative|48|2|117|128|0|117|11.79|17961, 7896, 25242, 71912, 14372, 9399, 33768, 17739, 75783, 15437, 54492, 291519, 28835, 26924, 44853, 85436, 827824, 154360, 246300, 102922, 12350440, 566524, 14553009998, 21804600384, 9450882, 99266648, 34211451, 54692434170, 52957646, 33896084480, 1773647215, 31197007840|
|practice_0112|multiplicative|128|0|123|126|0|123|2.90|7896, 9399, 14372, 15437, 17739, 17961, 25242, 26924, 28835, 33768, 44853, 54492, 71912, 75783, 85436, 102922, 154360, 246300, 291519, 566524, 827824, 9450882, 12350440, 34211451, 52957646, 99266648, 1773647215, 14553009998, 21804600384, 31197007840, 33896084480, 54692434170|
|practice_0112|multiplicative|128|1|118|123|0|118|4.86|7896, 9399, 14372, 33768, 85436, 15437, 17739, 17961, 25242, 28835, 71912, 75783, 291519, 26924, 44853, 54492, 102922, 154360, 246300, 566524, 827824, 9450882, 12350440, 52957646, 34211451, 99266648, 1773647215, 21804600384, 33896084480, 14553009998, 31197007840, 54692434170|
|practice_0112|multiplicative|128|2|119|120|0|119|35.22|17961, 7896, 25242, 71912, 14372, 9399, 33768, 17739, 75783, 54492, 44853, 85436, 15437, 26924, 28835, 102922, 291519, 154360, 827824, 566524, 12350440, 246300, 99266648, 9450882, 34211451, 52957646, 1773647215, 21804600384, 54692434170, 14553009998, 33896084480, 31197007840|
|practice_0113|independent|0|0|306|0|0|306|3.60|106907373320282887, 149034243297229819, 151957523941742539, 178100601637297302, 206157606269737769, 248286527686278076, 290734848754261734, 298418004292258424, 313771575625733225, 345659803003810929, 409222315068210638, 493218010036623592, 530198039684294237, 549589842753603784, 564724284905202967, 580163178249125900, 589068853161976409, 598210580718915529, 661272482554925132, 704248500990720514, 725270034778474526, 749774228096714557, 756862752484420839, 758446830291087755, 783455013347968041, 796152088988109730, 821873996335371104, 836838787830259029|
|practice_0113|independent|0|1|285|0|319|604|13.33|106907373320282887, 151957523941742539, 704248500990720514, 580163178249125900, 206157606269737769, 178100601637297302, 149034243297229819, 248286527686278076, 290734848754261734, 589068853161976409, 564724284905202967, 298418004292258424, 493218010036623592, 345659803003810929, 598210580718915529, 313771575625733225, 749774228096714557, 758446830291087755, 725270034778474526, 549589842753603784, 821873996335371104, 756862752484420839, 409222315068210638, 530198039684294237, 836838787830259029, 783455013347968041, 796152088988109730, 661272482554925132|
|practice_0113|independent|0|2|288|0|155|443|95.78|178100601637297302, 149034243297229819, 206157606269737769, 151957523941742539, 106907373320282887, 580163178249125900, 248286527686278076, 298418004292258424, 756862752484420839, 345659803003810929, 313771575625733225, 564724284905202967, 290734848754261734, 530198039684294237, 598210580718915529, 589068853161976409, 661272482554925132, 493218010036623592, 758446830291087755, 409222315068210638, 796152088988109730, 704248500990720514, 821873996335371104, 749774228096714557, 725270034778474526, 549589842753603784, 783455013347968041, 836838787830259029|
|practice_0113|independent|48|0|318|476|0|318|1.39|106907373320282887, 149034243297229819, 151957523941742539, 178100601637297302, 206157606269737769, 248286527686278076, 290734848754261734, 298418004292258424, 313771575625733225, 345659803003810929, 409222315068210638, 493218010036623592, 530198039684294237, 549589842753603784, 564724284905202967, 580163178249125900, 589068853161976409, 598210580718915529, 661272482554925132, 704248500990720514, 725270034778474526, 749774228096714557, 756862752484420839, 758446830291087755, 783455013347968041, 796152088988109730, 821873996335371104, 836838787830259029|
|practice_0113|independent|48|1|317|349|0|317|4.43|106907373320282887, 149034243297229819, 756862752484420839, 151957523941742539, 493218010036623592, 178100601637297302, 206157606269737769, 725270034778474526, 749774228096714557, 345659803003810929, 290734848754261734, 836838787830259029, 821873996335371104, 313771575625733225, 564724284905202967, 796152088988109730, 598210580718915529, 248286527686278076, 589068853161976409, 661272482554925132, 580163178249125900, 409222315068210638, 549589842753603784, 530198039684294237, 758446830291087755, 704248500990720514, 783455013347968041, 298418004292258424|
|practice_0113|independent|48|2|308|338|0|308|34.10|248286527686278076, 313771575625733225, 178100601637297302, 149034243297229819, 409222315068210638, 661272482554925132, 564724284905202967, 756862752484420839, 549589842753603784, 598210580718915529, 345659803003810929, 704248500990720514, 821873996335371104, 783455013347968041, 796152088988109730, 106907373320282887, 580163178249125900, 836838787830259029, 530198039684294237, 290734848754261734, 151957523941742539, 493218010036623592, 206157606269737769, 749774228096714557, 758446830291087755, 725270034778474526, 298418004292258424, 589068853161976409|
|practice_0113|independent|128|0|278|282|0|278|9.53|106907373320282887, 149034243297229819, 151957523941742539, 178100601637297302, 206157606269737769, 248286527686278076, 290734848754261734, 298418004292258424, 313771575625733225, 345659803003810929, 409222315068210638, 493218010036623592, 530198039684294237, 549589842753603784, 564724284905202967, 580163178249125900, 589068853161976409, 598210580718915529, 661272482554925132, 704248500990720514, 725270034778474526, 749774228096714557, 756862752484420839, 758446830291087755, 783455013347968041, 796152088988109730, 821873996335371104, 836838787830259029|
|practice_0113|independent|128|1|271|275|25|296|35.12|106907373320282887, 149034243297229819, 178100601637297302, 298418004292258424, 151957523941742539, 206157606269737769, 248286527686278076, 290734848754261734, 821873996335371104, 313771575625733225, 345659803003810929, 493218010036623592, 409222315068210638, 530198039684294237, 564724284905202967, 580163178249125900, 549589842753603784, 589068853161976409, 661272482554925132, 598210580718915529, 704248500990720514, 756862752484420839, 758446830291087755, 749774228096714557, 783455013347968041, 796152088988109730, 725270034778474526, 836838787830259029|
|practice_0113|independent|128|2|266|271|50|316|234.85|178100601637297302, 149034243297229819, 206157606269737769, 345659803003810929, 313771575625733225, 106907373320282887, 758446830291087755, 298418004292258424, 151957523941742539, 756862752484420839, 290734848754261734, 661272482554925132, 248286527686278076, 530198039684294237, 409222315068210638, 589068853161976409, 549589842753603784, 598210580718915529, 704248500990720514, 749774228096714557, 783455013347968041, 493218010036623592, 580163178249125900, 564724284905202967, 725270034778474526, 796152088988109730, 836838787830259029, 821873996335371104|
|practice_0114|mixed|0|0|57|0|0|57|0.26|174297, 191787, 196933, 202214, 202470, 202819, 213457, 717655, 744143, 893567, 903557, 958475, 10927224|
|practice_0114|mixed|0|1|50|0|0|50|0.28|196933, 202819, 202214, 202470, 958475, 191787, 213457, 174297, 744143, 717655, 903557, 893567, 10927224|
|practice_0114|mixed|0|2|54|0|0|54|1.76|202470, 202214, 958475, 191787, 202819, 213457, 174297, 196933, 744143, 903557, 717655, 893567, 10927224|
|practice_0114|mixed|48|0|56|61|0|56|0.25|174297, 191787, 196933, 202214, 202470, 202819, 213457, 717655, 744143, 893567, 903557, 958475, 10927224|
|practice_0114|mixed|48|1|51|58|0|51|0.25|196933, 202819, 202214, 202470, 958475, 191787, 213457, 174297, 744143, 717655, 893567, 903557, 10927224|
|practice_0114|mixed|48|2|53|59|0|53|1.34|202470, 202214, 958475, 191787, 202819, 213457, 174297, 903557, 196933, 717655, 744143, 893567, 10927224|
|practice_0114|mixed|128|0|55|60|0|55|0.37|174297, 191787, 196933, 202214, 202470, 202819, 213457, 717655, 744143, 893567, 903557, 958475, 10927224|
|practice_0114|mixed|128|1|52|58|0|52|0.32|196933, 202819, 202214, 202470, 958475, 191787, 213457, 174297, 744143, 717655, 893567, 903557, 10927224|
|practice_0114|mixed|128|2|55|59|0|55|1.71|202470, 202214, 958475, 191787, 202819, 213457, 174297, 903557, 196933, 717655, 744143, 893567, 10927224|
|practice_0115|shared|0|0|189|0|0|189|3.80|31609280, 113130074, 113149297, 113190772, 253268720, 253285597, 935772220, 1788782225, 2029770475, 2094063437, 3242700310, 9371664064, 16227197772, 28668286017, 64043350320, 257586106752, 931331841808, 1040225474016, 1256414973000, 2081625211992, 2150093796716, 10072455992950, 10391477824601, 21929674015470, 45060953271000, 5010189322495262, 17926278413547336, 259831113958883545|
|practice_0115|shared|0|1|184|0|0|184|11.34|31609280, 113130074, 113149297, 113190772, 253268720, 253285597, 935772220, 1788782225, 3242700310, 2029770475, 2094063437, 9371664064, 16227197772, 28668286017, 931331841808, 64043350320, 1040225474016, 2150093796716, 257586106752, 2081625211992, 1256414973000, 10072455992950, 21929674015470, 10391477824601, 45060953271000, 5010189322495262, 17926278413547336, 259831113958883545|
|practice_0115|shared|0|2|181|0|3|184|73.67|113149297, 31609280, 253268720, 935772220, 253285597, 113130074, 113190772, 9371664064, 3242700310, 1788782225, 2150093796716, 2029770475, 2094063437, 931331841808, 21929674015470, 10072455992950, 2081625211992, 28668286017, 64043350320, 1040225474016, 16227197772, 1256414973000, 10391477824601, 257586106752, 17926278413547336, 45060953271000, 5010189322495262, 259831113958883545|
|practice_0115|shared|48|0|197|307|0|197|1.63|31609280, 113130074, 113149297, 113190772, 253268720, 253285597, 935772220, 1788782225, 2029770475, 2094063437, 3242700310, 9371664064, 16227197772, 28668286017, 64043350320, 257586106752, 931331841808, 1040225474016, 1256414973000, 2081625211992, 2150093796716, 10072455992950, 10391477824601, 21929674015470, 45060953271000, 5010189322495262, 17926278413547336, 259831113958883545|
|practice_0115|shared|48|1|197|286|0|197|3.61|31609280, 113130074, 113149297, 113190772, 253268720, 253285597, 1788782225, 2029770475, 2094063437, 935772220, 3242700310, 64043350320, 9371664064, 16227197772, 1256414973000, 257586106752, 2081625211992, 2150093796716, 931331841808, 28668286017, 1040225474016, 45060953271000, 10391477824601, 10072455992950, 21929674015470, 17926278413547336, 259831113958883545, 5010189322495262|
|practice_0115|shared|48|2|203|212|0|203|24.70|113149297, 31609280, 253285597, 935772220, 113130074, 253268720, 113190772, 9371664064, 257586106752, 16227197772, 2094063437, 3242700310, 1040225474016, 2150093796716, 1256414973000, 10072455992950, 2081625211992, 2029770475, 64043350320, 28668286017, 931331841808, 5010189322495262, 45060953271000, 17926278413547336, 21929674015470, 10391477824601, 259831113958883545, 1788782225|
|practice_0115|shared|128|0|180|188|10|190|6.54|31609280, 113130074, 113149297, 113190772, 253268720, 253285597, 935772220, 1788782225, 2029770475, 2094063437, 3242700310, 9371664064, 16227197772, 28668286017, 64043350320, 257586106752, 931331841808, 1040225474016, 1256414973000, 2081625211992, 2150093796716, 10072455992950, 10391477824601, 21929674015470, 45060953271000, 5010189322495262, 17926278413547336, 259831113958883545|
|practice_0115|shared|128|1|180|185|12|192|16.94|31609280, 113130074, 113149297, 113190772, 253268720, 253285597, 935772220, 1788782225, 2029770475, 2094063437, 3242700310, 9371664064, 16227197772, 28668286017, 64043350320, 257586106752, 931331841808, 1040225474016, 1256414973000, 2081625211992, 2150093796716, 10072455992950, 21929674015470, 10391477824601, 45060953271000, 17926278413547336, 5010189322495262, 259831113958883545|
|practice_0115|shared|128|2|177|180|27|204|101.06|113149297, 31609280, 253268720, 935772220, 113130074, 253285597, 9371664064, 113190772, 3242700310, 2029770475, 1788782225, 257586106752, 2094063437, 16227197772, 931331841808, 2081625211992, 2150093796716, 1040225474016, 64043350320, 21929674015470, 28668286017, 1256414973000, 259831113958883545, 17926278413547336, 10391477824601, 5010189322495262, 10072455992950, 45060953271000|
|practice_0116|independent|0|0|71|0|0|71|0.45|180935502, 220842255, 435459409, 579537912, 606125165, 635276536, 659270750, 813890490, 855129615, 943258512|
|practice_0116|independent|0|1|69|0|0|69|0.63|220842255, 579537912, 606125165, 180935502, 435459409, 635276536, 659270750, 855129615, 813890490, 943258512|
|practice_0116|independent|0|2|71|0|0|71|2.73|435459409, 813890490, 855129615, 220842255, 579537912, 635276536, 943258512, 180935502, 606125165, 659270750|
|practice_0116|independent|48|0|71|80|0|71|0.28|180935502, 220842255, 435459409, 579537912, 606125165, 635276536, 659270750, 813890490, 855129615, 943258512|
|practice_0116|independent|48|1|68|72|0|68|0.42|220842255, 579537912, 606125165, 635276536, 659270750, 855129615, 435459409, 180935502, 813890490, 943258512|
|practice_0116|independent|48|2|67|73|0|67|2.13|435459409, 635276536, 220842255, 855129615, 180935502, 943258512, 579537912, 659270750, 813890490, 606125165|
|practice_0116|independent|128|0|72|74|0|72|0.50|180935502, 220842255, 435459409, 579537912, 606125165, 635276536, 659270750, 813890490, 855129615, 943258512|
|practice_0116|independent|128|1|69|72|0|69|0.65|220842255, 579537912, 606125165, 635276536, 659270750, 855129615, 180935502, 435459409, 813890490, 943258512|
|practice_0116|independent|128|2|71|72|0|71|3.07|435459409, 635276536, 220842255, 855129615, 579537912, 943258512, 180935502, 606125165, 659270750, 813890490|
|practice_0117|multiplicative|0|0|143|0|0|143|3.01|359747, 426956, 853579, 1096771, 1288837, 1288997, 2255323, 2742332, 2857664, 3865906, 4979658, 6204846, 6854709, 8225758, 9665396, 38809710, 66161812, 190901068, 260794147, 754961812, 1045242940, 2317112889, 2550030236, 5228822229, 9117543437, 12483814008, 12930075045, 34504371324, 91130306029, 2297620947390|
|practice_0117|multiplicative|0|1|145|0|0|145|5.26|359747, 426956, 853579, 1096771, 1288997, 1288837, 2742332, 2857664, 3865906, 9665396, 2255323, 4979658, 6204846, 6854709, 8225758, 38809710, 66161812, 190901068, 260794147, 754961812, 1045242940, 2317112889, 2550030236, 5228822229, 9117543437, 2297620947390, 12483814008, 12930075045, 34504371324, 91130306029|
|practice_0117|multiplicative|0|2|142|0|0|142|37.20|1096771, 1288837, 426956, 853579, 2857664, 1288997, 2742332, 6204846, 359747, 2255323, 3865906, 8225758, 9665396, 190901068, 754961812, 66161812, 6854709, 38809710, 1045242940, 4979658, 9117543437, 260794147, 34504371324, 2297620947390, 5228822229, 2550030236, 12483814008, 12930075045, 2317112889, 91130306029|
|practice_0117|multiplicative|48|0|151|161|0|151|0.75|359747, 426956, 853579, 1096771, 1288837, 1288997, 2255323, 2742332, 2857664, 3865906, 4979658, 6204846, 6854709, 8225758, 9665396, 38809710, 66161812, 190901068, 260794147, 754961812, 1045242940, 2317112889, 2550030236, 5228822229, 9117543437, 12483814008, 12930075045, 34504371324, 91130306029, 2297620947390|
|practice_0117|multiplicative|48|1|154|162|0|154|1.71|359747, 426956, 853579, 1096771, 1288837, 1288997, 2255323, 2742332, 3865906, 8225758, 9665396, 2857664, 4979658, 6204846, 6854709, 38809710, 66161812, 190901068, 1045242940, 260794147, 754961812, 2317112889, 5228822229, 2550030236, 9117543437, 12930075045, 34504371324, 2297620947390, 12483814008, 91130306029|
|practice_0117|multiplicative|48|2|159|164|0|159|14.10|1096771, 1288837, 426956, 853579, 1288997, 2742332, 8225758, 359747, 6854709, 2255323, 2857664, 4979658, 6204846, 190901068, 66161812, 1045242940, 2317112889, 9117543437, 38809710, 9665396, 260794147, 5228822229, 3865906, 91130306029, 12930075045, 34504371324, 12483814008, 2297620947390, 754961812, 2550030236|
|practice_0117|multiplicative|128|0|143|146|6|149|4.32|359747, 426956, 853579, 1096771, 1288837, 1288997, 2255323, 2742332, 2857664, 3865906, 4979658, 6204846, 6854709, 8225758, 9665396, 38809710, 66161812, 190901068, 260794147, 754961812, 1045242940, 2317112889, 2550030236, 5228822229, 9117543437, 12483814008, 12930075045, 34504371324, 91130306029, 2297620947390|
|practice_0117|multiplicative|128|1|144|146|25|169|8.43|359747, 426956, 853579, 1096771, 1288837, 1288997, 2255323, 2742332, 2857664, 3865906, 6854709, 8225758, 9665396, 4979658, 6204846, 38809710, 66161812, 190901068, 260794147, 754961812, 2550030236, 1045242940, 2317112889, 5228822229, 9117543437, 2297620947390, 12483814008, 12930075045, 34504371324, 91130306029|
|practice_0117|multiplicative|128|2|139|143|0|139|52.58|1096771, 1288837, 426956, 853579, 2857664, 1288997, 2742332, 3865906, 359747, 2255323, 9665396, 66161812, 6204846, 260794147, 4979658, 8225758, 38809710, 190901068, 6854709, 2550030236, 9117543437, 2297620947390, 1045242940, 2317112889, 5228822229, 91130306029, 754961812, 12483814008, 34504371324, 12930075045|
|practice_0118|mixed|0|0|43|0|0|43|0.13|110183, 181276, 478788, 666785, 698902, 713076, 10135398, 12331120|
|practice_0118|mixed|0|1|44|0|0|44|0.15|110183, 181276, 12331120, 478788, 666785, 698902, 713076, 10135398|
|practice_0118|mixed|0|2|43|0|0|43|0.75|181276, 110183, 12331120, 478788, 713076, 698902, 666785, 10135398|
|practice_0118|mixed|48|0|44|49|0|44|0.16|110183, 181276, 478788, 666785, 698902, 713076, 10135398, 12331120|
|practice_0118|mixed|48|1|43|48|0|43|0.15|110183, 181276, 12331120, 478788, 666785, 698902, 713076, 10135398|
|practice_0118|mixed|48|2|41|48|0|41|0.73|181276, 12331120, 110183, 713076, 666785, 478788, 10135398, 698902|
|practice_0118|mixed|128|0|44|49|0|44|0.16|110183, 181276, 478788, 666785, 698902, 713076, 10135398, 12331120|
|practice_0118|mixed|128|1|43|48|0|43|0.26|110183, 181276, 12331120, 478788, 666785, 698902, 713076, 10135398|
|practice_0118|mixed|128|2|41|48|0|41|0.65|181276, 12331120, 110183, 713076, 666785, 478788, 10135398, 698902|
|practice_0119|independent|0|0|267|0|0|267|2.55|134254779751698, 222739715152566, 263407718385353, 270687931436704, 293242637329202, 306955517383060, 343816673391161, 399773446367280, 431946322568429, 443538404993755, 453075669005094, 500497947499187, 562975385193696, 613793930724514, 626131463012629, 639240507610219, 649936579427990, 650143793284454, 651825553266437, 667513410946608, 679641236517294, 736561920176069, 765706910367504, 799116251695817, 865310157470772, 879173270125630, 891893688190504, 929879459475737, 998787901403962|
|practice_0119|independent|0|1|255|0|15|270|10.07|263407718385353, 562975385193696, 270687931436704, 443538404993755, 134254779751698, 399773446367280, 639240507610219, 306955517383060, 651825553266437, 222739715152566, 500497947499187, 293242637329202, 343816673391161, 431946322568429, 453075669005094, 613793930724514, 649936579427990, 626131463012629, 929879459475737, 650143793284454, 667513410946608, 736561920176069, 799116251695817, 879173270125630, 865310157470772, 679641236517294, 765706910367504, 891893688190504, 998787901403962|
|practice_0119|independent|0|2|247|0|45|292|70.59|222739715152566, 263407718385353, 443538404993755, 431946322568429, 453075669005094, 270687931436704, 293242637329202, 639240507610219, 134254779751698, 399773446367280, 667513410946608, 562975385193696, 865310157470772, 649936579427990, 500497947499187, 650143793284454, 679641236517294, 651825553266437, 343816673391161, 613793930724514, 891893688190504, 765706910367504, 306955517383060, 879173270125630, 626131463012629, 998787901403962, 736561920176069, 929879459475737, 799116251695817|
|practice_0119|independent|48|0|271|387|0|271|1.21|134254779751698, 222739715152566, 263407718385353, 270687931436704, 293242637329202, 306955517383060, 343816673391161, 399773446367280, 431946322568429, 443538404993755, 453075669005094, 500497947499187, 562975385193696, 613793930724514, 626131463012629, 639240507610219, 649936579427990, 650143793284454, 651825553266437, 667513410946608, 679641236517294, 736561920176069, 765706910367504, 799116251695817, 865310157470772, 879173270125630, 891893688190504, 929879459475737, 998787901403962|
|practice_0119|independent|48|1|269|275|0|269|4.05|263407718385353, 562975385193696, 270687931436704, 443538404993755, 431946322568429, 222739715152566, 306955517383060, 799116251695817, 765706910367504, 650143793284454, 613793930724514, 343816673391161, 134254779751698, 293242637329202, 399773446367280, 453075669005094, 626131463012629, 679641236517294, 667513410946608, 651825553266437, 649936579427990, 639240507610219, 865310157470772, 998787901403962, 879173270125630, 736561920176069, 500497947499187, 891893688190504, 929879459475737|
|practice_0119|independent|48|2|270|280|0|270|32.78|222739715152566, 270687931436704, 562975385193696, 431946322568429, 399773446367280, 799116251695817, 443538404993755, 929879459475737, 453075669005094, 293242637329202, 998787901403962, 134254779751698, 626131463012629, 650143793284454, 306955517383060, 649936579427990, 736561920176069, 613793930724514, 651825553266437, 343816673391161, 891893688190504, 639240507610219, 500497947499187, 679641236517294, 765706910367504, 879173270125630, 667513410946608, 865310157470772, 263407718385353|
|practice_0119|independent|128|0|242|255|32|274|9.04|134254779751698, 222739715152566, 263407718385353, 270687931436704, 293242637329202, 306955517383060, 343816673391161, 399773446367280, 431946322568429, 443538404993755, 453075669005094, 500497947499187, 562975385193696, 613793930724514, 626131463012629, 639240507610219, 649936579427990, 650143793284454, 651825553266437, 667513410946608, 679641236517294, 736561920176069, 765706910367504, 799116251695817, 865310157470772, 879173270125630, 891893688190504, 929879459475737, 998787901403962|
|practice_0119|independent|128|1|232|236|5|237|30.86|263407718385353, 562975385193696, 270687931436704, 443538404993755, 134254779751698, 639240507610219, 399773446367280, 651825553266437, 649936579427990, 293242637329202, 431946322568429, 453075669005094, 222739715152566, 500497947499187, 343816673391161, 306955517383060, 613793930724514, 650143793284454, 667513410946608, 626131463012629, 679641236517294, 929879459475737, 998787901403962, 736561920176069, 765706910367504, 891893688190504, 799116251695817, 879173270125630, 865310157470772|
|practice_0119|independent|128|2|234|236|22|256|191.83|222739715152566, 263407718385353, 443538404993755, 431946322568429, 453075669005094, 306955517383060, 998787901403962, 399773446367280, 293242637329202, 270687931436704, 679641236517294, 613793930724514, 626131463012629, 891893688190504, 639240507610219, 736561920176069, 649936579427990, 650143793284454, 799116251695817, 562975385193696, 343816673391161, 651825553266437, 865310157470772, 667513410946608, 929879459475737, 134254779751698, 879173270125630, 500497947499187, 765706910367504|
|practice_0120|shared|0|0|167|0|0|167|3.91|1062479, 1063197, 1765064, 1983227, 3529315, 4179386, 8987193, 9771997, 21570240, 22308627, 22308692, 52838316, 52838391, 161286091, 386287956, 514152638, 610262467, 681132749, 758472109, 1014688090, 1986103105, 2041903200, 2399101306, 3317705031, 3751523636, 5609056859, 6498747672, 13354423509, 19543256964, 32572077720, 35452642190, 59878811670625|
|practice_0120|shared|0|1|158|0|0|158|8.20|1062479, 1063197, 1765064, 1983227, 3529315, 4179386, 8987193, 9771997, 21570240, 22308627, 22308692, 52838316, 52838391, 514152638, 681132749, 758472109, 610262467, 1014688090, 386287956, 3751523636, 3317705031, 13354423509, 161286091, 1986103105, 2041903200, 2399101306, 19543256964, 5609056859, 6498747672, 32572077720, 35452642190, 59878811670625|
|practice_0120|shared|0|2|149|0|0|149|44.52|1062479, 1765064, 3529315, 1063197, 22308627, 4179386, 52838391, 22308692, 9771997, 52838316, 1983227, 610262467, 386287956, 8987193, 161286091, 21570240, 2041903200, 13354423509, 681132749, 1014688090, 514152638, 758472109, 6498747672, 5609056859, 2399101306, 19543256964, 32572077720, 1986103105, 3317705031, 59878811670625, 3751523636, 35452642190|
|practice_0120|shared|48|0|171|217|0|171|1.48|1062479, 1063197, 1765064, 1983227, 3529315, 4179386, 8987193, 9771997, 21570240, 22308627, 22308692, 52838316, 52838391, 161286091, 386287956, 514152638, 610262467, 681132749, 758472109, 1014688090, 1986103105, 2041903200, 2399101306, 3317705031, 3751523636, 5609056859, 6498747672, 13354423509, 19543256964, 32572077720, 35452642190, 59878811670625|
|practice_0120|shared|48|1|161|170|0|161|3.79|1062479, 1063197, 1765064, 1983227, 3529315, 4179386, 8987193, 9771997, 21570240, 22308627, 22308692, 52838391, 52838316, 161286091, 386287956, 32572077720, 610262467, 681132749, 514152638, 19543256964, 758472109, 3317705031, 3751523636, 1986103105, 2041903200, 1014688090, 6498747672, 2399101306, 13354423509, 5609056859, 35452642190, 59878811670625|
|practice_0120|shared|48|2|164|171|0|164|19.59|1062479, 1765064, 3529315, 8987193, 22308627, 52838391, 22308692, 1063197, 9771997, 52838316, 4179386, 514152638, 1014688090, 1983227, 610262467, 21570240, 2041903200, 386287956, 681132749, 758472109, 161286091, 32572077720, 3317705031, 1986103105, 2399101306, 6498747672, 19543256964, 59878811670625, 35452642190, 3751523636, 5609056859, 13354423509|
|practice_0120|shared|128|0|155|158|22|177|5.86|1062479, 1063197, 1765064, 1983227, 3529315, 4179386, 8987193, 9771997, 21570240, 22308627, 22308692, 52838316, 52838391, 161286091, 386287956, 514152638, 610262467, 681132749, 758472109, 1014688090, 1986103105, 2041903200, 2399101306, 3317705031, 3751523636, 5609056859, 6498747672, 13354423509, 19543256964, 32572077720, 35452642190, 59878811670625|
|practice_0120|shared|128|1|154|157|30|184|11.02|1062479, 1063197, 1765064, 1983227, 3529315, 4179386, 8987193, 9771997, 21570240, 22308627, 22308692, 52838316, 52838391, 514152638, 161286091, 386287956, 610262467, 681132749, 2041903200, 758472109, 1014688090, 3317705031, 3751523636, 5609056859, 1986103105, 2399101306, 6498747672, 13354423509, 32572077720, 19543256964, 35452642190, 59878811670625|
|practice_0120|shared|128|2|152|157|39|191|58.82|1062479, 1765064, 3529315, 4179386, 1063197, 8987193, 22308627, 9771997, 52838391, 52838316, 22308692, 610262467, 386287956, 514152638, 161286091, 21570240, 1983227, 2041903200, 1014688090, 681132749, 1986103105, 2399101306, 3317705031, 3751523636, 19543256964, 758472109, 13354423509, 5609056859, 6498747672, 32572077720, 35452642190, 59878811670625|
|practice_0121|independent|0|0|280|0|0|280|3.49|124404029787250, 152812443163904, 161448030306754, 205801929197303, 236608929624832, 262408382871215, 284297022954075, 316360021007192, 329364141269001, 337823059563756, 368157857913417, 368440308574544, 393149857974514, 397525761068108, 400731039033566, 430312758114924, 436920479732836, 457158723058651, 514082613012150, 517305227487278, 549515238248950, 581062238107274, 581232976942634, 657237739202682, 668770719324170, 690610158661503, 729757896613375, 753237698137745, 844327414118912, 850135947259471, 894153030650459, 896929446592441|
|practice_0121|independent|0|1|266|0|39|305|13.56|284297022954075, 729757896613375, 152812443163904, 124404029787250, 657237739202682, 236608929624832, 430312758114924, 397525761068108, 400731039033566, 436920479732836, 205801929197303, 337823059563756, 393149857974514, 161448030306754, 329364141269001, 368157857913417, 368440308574544, 844327414118912, 262408382871215, 514082613012150, 316360021007192, 517305227487278, 457158723058651, 581062238107274, 581232976942634, 668770719324170, 894153030650459, 850135947259471, 549515238248950, 690610158661503, 896929446592441, 753237698137745|
|practice_0121|independent|0|2|258|0|79|337|86.86|205801929197303, 152812443163904, 161448030306754, 729757896613375, 844327414118912, 393149857974514, 329364141269001, 400731039033566, 236608929624832, 896929446592441, 581232976942634, 657237739202682, 262408382871215, 337823059563756, 690610158661503, 368157857913417, 436920479732836, 284297022954075, 668770719324170, 368440308574544, 581062238107274, 124404029787250, 430312758114924, 457158723058651, 549515238248950, 514082613012150, 850135947259471, 316360021007192, 517305227487278, 894153030650459, 753237698137745, 397525761068108|
|practice_0121|independent|48|0|283|398|0|283|1.78|124404029787250, 152812443163904, 161448030306754, 205801929197303, 236608929624832, 262408382871215, 284297022954075, 316360021007192, 329364141269001, 337823059563756, 368157857913417, 368440308574544, 393149857974514, 397525761068108, 400731039033566, 430312758114924, 436920479732836, 457158723058651, 514082613012150, 517305227487278, 549515238248950, 581062238107274, 581232976942634, 657237739202682, 668770719324170, 690610158661503, 729757896613375, 753237698137745, 844327414118912, 850135947259471, 894153030650459, 896929446592441|
|practice_0121|independent|48|1|280|288|0|280|5.27|284297022954075, 729757896613375, 430312758114924, 152812443163904, 393149857974514, 337823059563756, 329364141269001, 457158723058651, 316360021007192, 517305227487278, 368440308574544, 262408382871215, 368157857913417, 124404029787250, 236608929624832, 205801929197303, 436920479732836, 514082613012150, 581232976942634, 161448030306754, 581062238107274, 549515238248950, 397525761068108, 400731039033566, 657237739202682, 668770719324170, 896929446592441, 753237698137745, 844327414118912, 894153030650459, 690610158661503, 850135947259471|
|practice_0121|independent|48|2|291|302|0|291|36.53|205801929197303, 161448030306754, 581232976942634, 236608929624832, 337823059563756, 581062238107274, 262408382871215, 436920479732836, 152812443163904, 284297022954075, 397525761068108, 400731039033566, 690610158661503, 329364141269001, 844327414118912, 124404029787250, 368440308574544, 368157857913417, 393149857974514, 517305227487278, 549515238248950, 316360021007192, 657237739202682, 457158723058651, 896929446592441, 668770719324170, 430312758114924, 850135947259471, 894153030650459, 729757896613375, 753237698137745, 514082613012150|
|practice_0121|independent|128|0|253|263|12|265|8.58|124404029787250, 152812443163904, 161448030306754, 205801929197303, 236608929624832, 262408382871215, 284297022954075, 316360021007192, 329364141269001, 337823059563756, 368157857913417, 368440308574544, 393149857974514, 397525761068108, 400731039033566, 430312758114924, 436920479732836, 457158723058651, 514082613012150, 517305227487278, 549515238248950, 581062238107274, 581232976942634, 657237739202682, 668770719324170, 690610158661503, 729757896613375, 753237698137745, 844327414118912, 850135947259471, 894153030650459, 896929446592441|
|practice_0121|independent|128|1|246|249|0|246|34.24|284297022954075, 729757896613375, 152812443163904, 236608929624832, 894153030650459, 457158723058651, 161448030306754, 753237698137745, 581062238107274, 124404029787250, 368440308574544, 368157857913417, 262408382871215, 205801929197303, 316360021007192, 329364141269001, 337823059563756, 393149857974514, 581232976942634, 844327414118912, 397525761068108, 400731039033566, 514082613012150, 517305227487278, 430312758114924, 436920479732836, 668770719324170, 850135947259471, 549515238248950, 896929446592441, 690610158661503, 657237739202682|
|practice_0121|independent|128|2|245|247|13|258|199.02|205801929197303, 152812443163904, 729757896613375, 161448030306754, 844327414118912, 393149857974514, 329364141269001, 316360021007192, 124404029787250, 368157857913417, 236608929624832, 368440308574544, 549515238248950, 397525761068108, 262408382871215, 436920479732836, 400731039033566, 514082613012150, 457158723058651, 896929446592441, 690610158661503, 581062238107274, 581232976942634, 430312758114924, 657237739202682, 850135947259471, 284297022954075, 337823059563756, 517305227487278, 753237698137745, 668770719324170, 894153030650459|
|practice_0122|independent|0|0|319|0|0|319|3.63|101290677712585636, 102403306942612856, 103453489777572028, 113338425107798834, 114782096154762233, 149524967065853537, 265352535172535674, 270784018929027815, 292575119379690373, 352363102777937593, 395007629086434753, 420796683317730775, 453999562888403126, 492975920564194340, 526872783139644834, 588500416494193932, 596314431238022152, 604290143464307571, 608516972083792837, 637465752675203066, 684656254771567127, 725126199902274223, 756415112998057713, 780594478705187334, 814409512288383905, 814950135506448394, 880018512292060444, 882071426324888161, 886546182892140048|
|practice_0122|independent|0|1|299|0|107|406|14.03|103453489777572028, 604290143464307571, 102403306942612856, 101290677712585636, 780594478705187334, 114782096154762233, 684656254771567127, 265352535172535674, 113338425107798834, 588500416494193932, 608516972083792837, 149524967065853537, 270784018929027815, 492975920564194340, 292575119379690373, 814950135506448394, 352363102777937593, 395007629086434753, 420796683317730775, 882071426324888161, 596314431238022152, 453999562888403126, 526872783139644834, 637465752675203066, 880018512292060444, 725126199902274223, 756415112998057713, 814409512288383905, 886546182892140048|
|practice_0122|independent|0|2|292|0|184|476|93.85|102403306942612856, 265352535172535674, 756415112998057713, 880018512292060444, 114782096154762233, 453999562888403126, 420796683317730775, 101290677712585636, 882071426324888161, 395007629086434753, 352363102777937593, 292575119379690373, 270784018929027815, 608516972083792837, 492975920564194340, 526872783139644834, 113338425107798834, 780594478705187334, 588500416494193932, 103453489777572028, 149524967065853537, 725126199902274223, 604290143464307571, 886546182892140048, 814950135506448394, 637465752675203066, 596314431238022152, 684656254771567127, 814409512288383905|
|practice_0122|independent|48|0|324|495|0|324|2.01|101290677712585636, 102403306942612856, 103453489777572028, 113338425107798834, 114782096154762233, 149524967065853537, 265352535172535674, 270784018929027815, 292575119379690373, 352363102777937593, 395007629086434753, 420796683317730775, 453999562888403126, 492975920564194340, 526872783139644834, 588500416494193932, 596314431238022152, 604290143464307571, 608516972083792837, 637465752675203066, 684656254771567127, 725126199902274223, 756415112998057713, 780594478705187334, 814409512288383905, 814950135506448394, 880018512292060444, 882071426324888161, 886546182892140048|
|practice_0122|independent|48|1|326|464|0|326|6.60|103453489777572028, 604290143464307571, 588500416494193932, 101290677712585636, 113338425107798834, 102403306942612856, 149524967065853537, 886546182892140048, 270784018929027815, 265352535172535674, 420796683317730775, 395007629086434753, 880018512292060444, 725126199902274223, 352363102777937593, 637465752675203066, 596314431238022152, 114782096154762233, 814409512288383905, 882071426324888161, 608516972083792837, 756415112998057713, 684656254771567127, 492975920564194340, 453999562888403126, 292575119379690373, 780594478705187334, 526872783139644834, 814950135506448394|
|practice_0122|independent|48|2|327|338|0|327|39.28|102403306942612856, 265352535172535674, 492975920564194340, 608516972083792837, 882071426324888161, 149524967065853537, 113338425107798834, 814409512288383905, 103453489777572028, 270784018929027815, 725126199902274223, 292575119379690373, 637465752675203066, 604290143464307571, 780594478705187334, 526872783139644834, 588500416494193932, 114782096154762233, 886546182892140048, 453999562888403126, 756415112998057713, 814950135506448394, 420796683317730775, 352363102777937593, 880018512292060444, 395007629086434753, 101290677712585636, 596314431238022152, 684656254771567127|
|practice_0122|independent|128|0|293|317|81|374|10.57|101290677712585636, 102403306942612856, 103453489777572028, 113338425107798834, 114782096154762233, 149524967065853537, 265352535172535674, 270784018929027815, 292575119379690373, 352363102777937593, 395007629086434753, 420796683317730775, 453999562888403126, 492975920564194340, 526872783139644834, 588500416494193932, 596314431238022152, 604290143464307571, 608516972083792837, 637465752675203066, 684656254771567127, 725126199902274223, 756415112998057713, 780594478705187334, 814409512288383905, 814950135506448394, 880018512292060444, 882071426324888161, 886546182892140048|
|practice_0122|independent|128|1|281|283|12|293|38.02|103453489777572028, 604290143464307571, 102403306942612856, 101290677712585636, 780594478705187334, 114782096154762233, 113338425107798834, 149524967065853537, 270784018929027815, 265352535172535674, 684656254771567127, 756415112998057713, 637465752675203066, 492975920564194340, 352363102777937593, 292575119379690373, 395007629086434753, 453999562888403126, 420796683317730775, 588500416494193932, 596314431238022152, 526872783139644834, 608516972083792837, 725126199902274223, 886546182892140048, 882071426324888161, 814409512288383905, 814950135506448394, 880018512292060444|
|practice_0122|independent|128|2|273|276|0|273|250.02|102403306942612856, 265352535172535674, 756415112998057713, 880018512292060444, 113338425107798834, 103453489777572028, 814409512288383905, 270784018929027815, 814950135506448394, 114782096154762233, 882071426324888161, 352363102777937593, 886546182892140048, 292575119379690373, 149524967065853537, 492975920564194340, 596314431238022152, 101290677712585636, 453999562888403126, 395007629086434753, 604290143464307571, 637465752675203066, 526872783139644834, 608516972083792837, 588500416494193932, 780594478705187334, 420796683317730775, 725126199902274223, 684656254771567127|
|practice_0123|tiny|0|0|3|0|0|3|0.01|3|
|practice_0123|tiny|0|1|3|0|0|3|0.03|3|
|practice_0123|tiny|0|2|3|0|0|3|0.02|3|
|practice_0123|tiny|48|0|3|8|0|3|0.01|3|
|practice_0123|tiny|48|1|3|8|0|3|0.01|3|
|practice_0123|tiny|48|2|3|8|0|3|0.01|3|
|practice_0123|tiny|128|0|3|8|0|3|0.01|3|
|practice_0123|tiny|128|1|3|8|0|3|0.01|3|
|practice_0123|tiny|128|2|3|8|0|3|0.01|3|
|practice_0124|mixed|0|0|226|0|0|226|3.89|44062827, 44064002, 93915375, 385973358, 643288531, 836274946, 1057513313, 3419914266, 9404766617, 77762018252, 176980607668, 603199839774, 4364121557556, 8009487881555, 33955987694511, 146917206689707, 151204906225379, 187111647850469, 248742657127545, 290422158732453, 307464843335820, 310743524407807, 331986275210566, 361634780196699, 372283828242399, 378358827641305, 688989212070127, 728306598165772, 734236525498049, 995725705796467|
|practice_0124|mixed|0|1|216|0|0|216|9.43|44064002, 44062827, 385973358, 93915375, 643288531, 836274946, 1057513313, 3419914266, 9404766617, 176980607668, 77762018252, 8009487881555, 603199839774, 4364121557556, 187111647850469, 307464843335820, 33955987694511, 310743524407807, 331986275210566, 151204906225379, 146917206689707, 248742657127545, 290422158732453, 361634780196699, 372283828242399, 378358827641305, 995725705796467, 734236525498049, 688989212070127, 728306598165772|
|practice_0124|mixed|0|2|211|0|0|211|69.78|836274946, 44064002, 44062827, 77762018252, 3419914266, 643288531, 1057513313, 385973358, 8009487881555, 176980607668, 93915375, 9404766617, 248742657127545, 603199839774, 33955987694511, 4364121557556, 146917206689707, 372283828242399, 187111647850469, 290422158732453, 728306598165772, 378358827641305, 331986275210566, 310743524407807, 734236525498049, 995725705796467, 688989212070127, 307464843335820, 361634780196699, 151204906225379|
|practice_0124|mixed|48|0|227|280|0|227|1.81|44062827, 44064002, 93915375, 385973358, 643288531, 836274946, 1057513313, 3419914266, 9404766617, 77762018252, 176980607668, 603199839774, 4364121557556, 8009487881555, 33955987694511, 146917206689707, 151204906225379, 187111647850469, 248742657127545, 290422158732453, 307464843335820, 310743524407807, 331986275210566, 361634780196699, 372283828242399, 378358827641305, 688989212070127, 728306598165772, 734236525498049, 995725705796467|
|practice_0124|mixed|48|1|223|235|0|223|4.95|44064002, 44062827, 93915375, 385973358, 643288531, 836274946, 1057513313, 9404766617, 3419914266, 4364121557556, 77762018252, 8009487881555, 290422158732453, 176980607668, 603199839774, 310743524407807, 146917206689707, 33955987694511, 151204906225379, 307464843335820, 331986275210566, 728306598165772, 248742657127545, 361634780196699, 734236525498049, 187111647850469, 378358827641305, 688989212070127, 372283828242399, 995725705796467|
|practice_0124|mixed|48|2|222|238|0|222|29.47|836274946, 44064002, 44062827, 93915375, 643288531, 77762018252, 176980607668, 603199839774, 385973358, 9404766617, 8009487881555, 1057513313, 688989212070127, 151204906225379, 3419914266, 307464843335820, 248742657127545, 310743524407807, 4364121557556, 290422158732453, 378358827641305, 734236525498049, 728306598165772, 361634780196699, 146917206689707, 187111647850469, 33955987694511, 995725705796467, 331986275210566, 372283828242399|
|practice_0124|mixed|128|0|205|214|19|224|6.30|44062827, 44064002, 93915375, 385973358, 643288531, 836274946, 1057513313, 3419914266, 9404766617, 77762018252, 176980607668, 603199839774, 4364121557556, 8009487881555, 33955987694511, 146917206689707, 151204906225379, 187111647850469, 248742657127545, 290422158732453, 307464843335820, 310743524407807, 331986275210566, 361634780196699, 372283828242399, 378358827641305, 688989212070127, 728306598165772, 734236525498049, 995725705796467|
|practice_0124|mixed|128|1|202|205|4|206|16.60|44064002, 44062827, 93915375, 385973358, 643288531, 836274946, 1057513313, 3419914266, 9404766617, 176980607668, 77762018252, 8009487881555, 603199839774, 4364121557556, 33955987694511, 151204906225379, 146917206689707, 187111647850469, 310743524407807, 995725705796467, 248742657127545, 290422158732453, 307464843335820, 331986275210566, 361634780196699, 372283828242399, 378358827641305, 688989212070127, 728306598165772, 734236525498049|
|practice_0124|mixed|128|2|199|203|61|260|132.50|836274946, 44064002, 44062827, 77762018252, 643288531, 93915375, 9404766617, 3419914266, 385973358, 1057513313, 8009487881555, 176980607668, 603199839774, 4364121557556, 734236525498049, 310743524407807, 372283828242399, 290422158732453, 187111647850469, 378358827641305, 33955987694511, 331986275210566, 688989212070127, 146917206689707, 151204906225379, 361634780196699, 307464843335820, 728306598165772, 995725705796467, 248742657127545|
|practice_0125|tiny|0|0|4|0|0|4|0.02|3, 5, 7|
|practice_0125|tiny|0|1|4|0|0|4|0.02|3, 5, 7|
|practice_0125|tiny|0|2|4|0|0|4|0.01|3, 5, 7|
|practice_0125|tiny|48|0|4|7|0|4|0.01|3, 5, 7|
|practice_0125|tiny|48|1|4|7|0|4|0.01|3, 5, 7|
|practice_0125|tiny|48|2|4|7|0|4|0.01|3, 5, 7|
|practice_0125|tiny|128|0|4|7|0|4|0.01|3, 5, 7|
|practice_0125|tiny|128|1|4|7|0|4|0.01|3, 5, 7|
|practice_0125|tiny|128|2|4|7|0|4|0.01|3, 5, 7|
|practice_0126|shared|0|0|156|0|0|156|3.29|2756459439, 3103707936, 3103721377, 3103721546, 3494701617, 5195545392, 5512923298, 5850047101, 5850056957, 5850060291, 5850060346, 9311124075, 10391104379, 11025840322, 24462910195, 40950421720, 52763040158, 68550184127, 68550185851, 2527226026008, 3995496541848, 4015082235448, 4260347232921, 5090086920677, 10183266722521, 17041388971052, 34093168954172, 40733066759626, 54177405025842, 68186338078953, 8350278698754360|
|practice_0126|shared|0|1|155|0|0|155|8.88|2756459439, 5512923298, 3103707936, 9311124075, 3103721377, 3103721546, 11025840322, 52763040158, 5195545392, 10391104379, 5850047101, 5850060291, 5850060346, 5850056957, 40950421720, 2527226026008, 34093168954172, 68186338078953, 4260347232921, 17041388971052, 5090086920677, 3494701617, 24462910195, 68550184127, 68550185851, 54177405025842, 3995496541848, 4015082235448, 10183266722521, 40733066759626, 8350278698754360|
|practice_0126|shared|0|2|155|0|0|155|57.68|3103707936, 2756459439, 3494701617, 3103721546, 24462910195, 5850056957, 5850060346, 3103721377, 5512923298, 5850060291, 5195545392, 5850047101, 40950421720, 9311124075, 68550184127, 4260347232921, 68550185851, 52763040158, 17041388971052, 5090086920677, 4015082235448, 10391104379, 3995496541848, 8350278698754360, 68186338078953, 34093168954172, 10183266722521, 11025840322, 54177405025842, 2527226026008, 40733066759626|
|practice_0126|shared|48|0|179|200|0|179|1.41|2756459439, 3103707936, 3103721377, 3103721546, 3494701617, 5195545392, 5512923298, 5850047101, 5850056957, 5850060291, 5850060346, 9311124075, 10391104379, 11025840322, 24462910195, 40950421720, 52763040158, 68550184127, 68550185851, 2527226026008, 3995496541848, 4015082235448, 4260347232921, 5090086920677, 10183266722521, 17041388971052, 34093168954172, 40733066759626, 54177405025842, 68186338078953, 8350278698754360|
|practice_0126|shared|48|1|159|167|0|159|3.92|2756459439, 5512923298, 3103721377, 3103721546, 3103707936, 9311124075, 11025840322, 52763040158, 5195545392, 10391104379, 4260347232921, 17041388971052, 68186338078953, 34093168954172, 5850060346, 5850047101, 5850060291, 40950421720, 5850056957, 2527226026008, 54177405025842, 3494701617, 24462910195, 68550184127, 68550185851, 3995496541848, 4015082235448, 5090086920677, 10183266722521, 40733066759626, 8350278698754360|
|practice_0126|shared|48|2|167|177|0|167|25.80|3103721546, 2756459439, 3103707936, 11025840322, 3103721377, 5850060291, 5850047101, 40950421720, 5195545392, 5850060346, 3494701617, 5512923298, 54177405025842, 34093168954172, 5850056957, 24462910195, 10391104379, 68550185851, 2527226026008, 68550184127, 4260347232921, 52763040158, 3995496541848, 10183266722521, 40733066759626, 68186338078953, 17041388971052, 9311124075, 4015082235448, 8350278698754360, 5090086920677|
|practice_0126|shared|128|0|158|163|16|174|5.68|2756459439, 3103707936, 3103721377, 3103721546, 3494701617, 5195545392, 5512923298, 5850047101, 5850056957, 5850060291, 5850060346, 9311124075, 10391104379, 11025840322, 24462910195, 40950421720, 52763040158, 68550184127, 68550185851, 2527226026008, 3995496541848, 4015082235448, 4260347232921, 5090086920677, 10183266722521, 17041388971052, 34093168954172, 40733066759626, 54177405025842, 68186338078953, 8350278698754360|
|practice_0126|shared|128|1|154|158|0|154|15.89|2756459439, 5512923298, 3103721377, 3103721546, 3103707936, 9311124075, 11025840322, 52763040158, 5195545392, 10391104379, 5850047101, 5850056957, 5850060291, 5850060346, 40950421720, 3494701617, 24462910195, 68550184127, 68550185851, 2527226026008, 4260347232921, 17041388971052, 34093168954172, 68186338078953, 4015082235448, 3995496541848, 5090086920677, 10183266722521, 40733066759626, 8350278698754360, 54177405025842|
|practice_0126|shared|128|2|148|152|0|148|81.59|3103721546, 2756459439, 3103707936, 3103721377, 5195545392, 3494701617, 24462910195, 5850047101, 5850060291, 5850056957, 5850060346, 5512923298, 34093168954172, 4260347232921, 9311124075, 10391104379, 40950421720, 11025840322, 68550184127, 68550185851, 17041388971052, 10183266722521, 52763040158, 5090086920677, 68186338078953, 3995496541848, 40733066759626, 2527226026008, 4015082235448, 8350278698754360, 54177405025842|
|practice_0127|mixed|0|0|196|0|0|196|3.28|1421726, 2354109, 6833839, 29894174, 30568291, 158420325, 186150642, 372018297, 1066008632, 1278680202, 1504655806, 1935967728, 2533505184, 4271103335, 115604133007, 119403762973, 128368695695, 245132166548, 260433653309, 274340878354, 306830884472, 317075214233, 482508103790, 546879747444, 605670394695, 651129023152, 823906998938, 848759957892, 873272512246, 922876333042, 1318331805518|
|practice_0127|mixed|0|1|196|0|0|196|7.48|2354109, 1421726, 6833839, 29894174, 30568291, 1504655806, 158420325, 372018297, 186150642, 1066008632, 1278680202, 1935967728, 2533505184, 4271103335, 128368695695, 651129023152, 274340878354, 115604133007, 119403762973, 260433653309, 823906998938, 848759957892, 306830884472, 317075214233, 245132166548, 546879747444, 482508103790, 605670394695, 873272512246, 922876333042, 1318331805518|
|practice_0127|mixed|0|2|183|0|16|199|47.40|2354109, 6833839, 372018297, 186150642, 1421726, 158420325, 30568291, 1066008632, 29894174, 4271103335, 2533505184, 1504655806, 306830884472, 128368695695, 317075214233, 115604133007, 1278680202, 1935967728, 605670394695, 260433653309, 848759957892, 245132166548, 119403762973, 546879747444, 482508103790, 274340878354, 823906998938, 922876333042, 651129023152, 873272512246, 1318331805518|
|practice_0127|mixed|48|0|200|276|0|200|1.41|1421726, 2354109, 6833839, 29894174, 30568291, 158420325, 186150642, 372018297, 1066008632, 1278680202, 1504655806, 1935967728, 2533505184, 4271103335, 115604133007, 119403762973, 128368695695, 245132166548, 260433653309, 274340878354, 306830884472, 317075214233, 482508103790, 546879747444, 605670394695, 651129023152, 823906998938, 848759957892, 873272512246, 922876333042, 1318331805518|
|practice_0127|mixed|48|1|199|219|0|199|3.61|2354109, 1421726, 30568291, 6833839, 29894174, 158420325, 186150642, 1066008632, 372018297, 4271103335, 1278680202, 1504655806, 1935967728, 2533505184, 119403762973, 115604133007, 128368695695, 245132166548, 482508103790, 260433653309, 274340878354, 306830884472, 317075214233, 546879747444, 605670394695, 848759957892, 922876333042, 873272512246, 651129023152, 1318331805518, 823906998938|
|practice_0127|mixed|48|2|197|207|0|197|23.76|1421726, 2354109, 6833839, 158420325, 29894174, 186150642, 1935967728, 1504655806, 2533505184, 4271103335, 30568291, 1066008632, 245132166548, 372018297, 128368695695, 115604133007, 119403762973, 317075214233, 306830884472, 605670394695, 848759957892, 274340878354, 651129023152, 260433653309, 1278680202, 873272512246, 823906998938, 1318331805518, 922876333042, 546879747444, 482508103790|
|practice_0127|mixed|128|0|181|185|12|193|6.82|1421726, 2354109, 6833839, 29894174, 30568291, 158420325, 186150642, 372018297, 1066008632, 1278680202, 1504655806, 1935967728, 2533505184, 4271103335, 115604133007, 119403762973, 128368695695, 245132166548, 260433653309, 274340878354, 306830884472, 317075214233, 482508103790, 546879747444, 605670394695, 651129023152, 823906998938, 848759957892, 873272512246, 922876333042, 1318331805518|
|practice_0127|mixed|128|1|180|184|3|183|13.29|2354109, 1421726, 30568291, 6833839, 29894174, 158420325, 186150642, 372018297, 1066008632, 1278680202, 1504655806, 1935967728, 2533505184, 4271103335, 128368695695, 260433653309, 274340878354, 482508103790, 115604133007, 119403762973, 245132166548, 651129023152, 823906998938, 922876333042, 306830884472, 317075214233, 546879747444, 605670394695, 873272512246, 848759957892, 1318331805518|
|practice_0127|mixed|128|2|177|181|2|179|110.40|1421726, 2354109, 6833839, 29894174, 30568291, 372018297, 186150642, 158420325, 1278680202, 1504655806, 2533505184, 4271103335, 260433653309, 1066008632, 1935967728, 119403762973, 115604133007, 848759957892, 873272512246, 605670394695, 245132166548, 306830884472, 922876333042, 651129023152, 274340878354, 823906998938, 317075214233, 128368695695, 482508103790, 1318331805518, 546879747444|
|practice_0128|tiny|0|0|4|0|0|4|0.01|3, 8|
|practice_0128|tiny|0|1|4|0|0|4|0.02|3, 8|
|practice_0128|tiny|0|2|4|0|0|4|0.01|3, 8|
|practice_0128|tiny|48|0|4|6|0|4|0.01|3, 8|
|practice_0128|tiny|48|1|4|6|0|4|0.01|3, 8|
|practice_0128|tiny|48|2|4|6|0|4|0.01|3, 8|
|practice_0128|tiny|128|0|4|6|0|4|0.01|3, 8|
|practice_0128|tiny|128|1|4|6|0|4|0.01|3, 8|
|practice_0128|tiny|128|2|4|6|0|4|0.01|3, 8|
|practice_0129|multiplicative|0|0|125|0|0|125|2.87|50007, 53883, 269278, 444048, 530500, 580757, 805904, 832712, 1045632, 1201815, 1610520, 1717420, 2169231, 2617785, 4113653, 4996031, 8845881, 13437921, 24824320, 29042505, 40388856, 67749360, 78178368, 149509728, 1011533754, 13940942384, 36071990926, 52213600320, 1915739844480|
|practice_0129|multiplicative|0|1|124|0|0|124|4.02|1045632, 50007, 53883, 269278, 530500, 580757, 805904, 832712, 1201815, 444048, 1717420, 2169231, 2617785, 4113653, 4996031, 8845881, 1610520, 24824320, 13437921, 29042505, 40388856, 67749360, 78178368, 149509728, 1011533754, 13940942384, 52213600320, 1915739844480, 36071990926|
|practice_0129|multiplicative|0|2|122|0|0|122|29.84|50007, 13437921, 53883, 269278, 530500, 1201815, 1045632, 832712, 1610520, 805904, 444048, 1717420, 580757, 67749360, 4113653, 2169231, 2617785, 4996031, 24824320, 78178368, 52213600320, 1915739844480, 8845881, 149509728, 29042505, 13940942384, 40388856, 1011533754, 36071990926|
|practice_0129|multiplicative|48|0|132|141|0|132|1.20|50007, 53883, 269278, 444048, 530500, 580757, 805904, 832712, 1045632, 1201815, 1610520, 1717420, 2169231, 2617785, 4113653, 4996031, 8845881, 13437921, 24824320, 29042505, 40388856, 67749360, 78178368, 149509728, 1011533754, 13940942384, 36071990926, 52213600320, 1915739844480|
|practice_0129|multiplicative|48|1|128|141|0|128|1.94|1045632, 50007, 53883, 269278, 530500, 580757, 805904, 832712, 1201815, 2617785, 4113653, 4996031, 1717420, 444048, 1610520, 2169231, 8845881, 13437921, 24824320, 29042505, 67749360, 40388856, 78178368, 149509728, 1011533754, 52213600320, 36071990926, 13940942384, 1915739844480|
|practice_0129|multiplicative|48|2|128|135|0|128|12.12|50007, 13437921, 269278, 1201815, 1045632, 1610520, 53883, 580757, 444048, 805904, 832712, 530500, 2617785, 4113653, 8845881, 29042505, 2169231, 24824320, 149509728, 1717420, 4996031, 40388856, 1915739844480, 36071990926, 78178368, 52213600320, 67749360, 1011533754, 13940942384|
|practice_0129|multiplicative|128|0|124|128|0|124|4.02|50007, 53883, 269278, 444048, 530500, 580757, 805904, 832712, 1045632, 1201815, 1610520, 1717420, 2169231, 2617785, 4113653, 4996031, 8845881, 13437921, 24824320, 29042505, 40388856, 67749360, 78178368, 149509728, 1011533754, 13940942384, 36071990926, 52213600320, 1915739844480|
|practice_0129|multiplicative|128|1|119|123|0|119|5.79|1045632, 50007, 53883, 269278, 530500, 580757, 805904, 832712, 1201815, 444048, 1717420, 2169231, 2617785, 4113653, 4996031, 8845881, 24824320, 1610520, 13437921, 29042505, 40388856, 67749360, 78178368, 149509728, 1011533754, 52213600320, 1915739844480, 13940942384, 36071990926|
|practice_0129|multiplicative|128|2|120|123|0|120|37.27|50007, 13437921, 269278, 1201815, 1045632, 1610520, 832712, 53883, 444048, 530500, 4996031, 805904, 580757, 1717420, 2169231, 2617785, 4113653, 8845881, 24824320, 40388856, 78178368, 29042505, 1011533754, 67749360, 149509728, 13940942384, 52213600320, 1915739844480, 36071990926|
|practice_0130|multiplicative|0|0|111|0|0|111|3.08|11627, 16779, 25385, 40142, 111786, 114516, 180642, 254305, 301118, 360532, 361340, 372196, 540386, 614016, 632484, 903586, 963552, 1378997, 1831087, 2406336, 6384708, 9636172, 33607131, 250435845, 834731566, 40168333522, 53128868760, 35132395233040|
|practice_0130|multiplicative|0|1|112|0|0|112|4.90|11627, 16779, 25385, 40142, 360532, 111786, 114516, 180642, 361340, 254305, 301118, 963552, 903586, 372196, 540386, 614016, 632484, 1378997, 1831087, 2406336, 6384708, 9636172, 33607131, 250435845, 834731566, 53128868760, 40168333522, 35132395233040|
|practice_0130|multiplicative|0|2|107|0|0|107|32.54|25385, 114516, 16779, 180642, 963552, 301118, 361340, 11627, 40142, 360532, 254305, 111786, 372196, 540386, 903586, 2406336, 614016, 632484, 1378997, 1831087, 6384708, 9636172, 33607131, 250435845, 834731566, 53128868760, 40168333522, 35132395233040|
|practice_0130|multiplicative|48|0|118|131|0|118|1.23|11627, 16779, 25385, 40142, 111786, 114516, 180642, 254305, 301118, 360532, 361340, 372196, 540386, 614016, 632484, 903586, 963552, 1378997, 1831087, 2406336, 6384708, 9636172, 33607131, 250435845, 834731566, 40168333522, 53128868760, 35132395233040|
|practice_0130|multiplicative|48|1|115|123|0|115|1.94|11627, 16779, 25385, 40142, 360532, 111786, 114516, 180642, 361340, 254305, 301118, 372196, 540386, 614016, 632484, 903586, 963552, 1378997, 2406336, 9636172, 1831087, 6384708, 33607131, 250435845, 53128868760, 834731566, 35132395233040, 40168333522|
|practice_0130|multiplicative|48|2|108|115|0|108|12.80|25385, 114516, 16779, 111786, 11627, 301118, 361340, 963552, 40142, 903586, 372196, 180642, 360532, 254305, 2406336, 1831087, 632484, 540386, 9636172, 6384708, 614016, 1378997, 834731566, 250435845, 40168333522, 33607131, 53128868760, 35132395233040|
|practice_0130|multiplicative|128|0|111|114|0|111|3.62|11627, 16779, 25385, 40142, 111786, 114516, 180642, 254305, 301118, 360532, 361340, 372196, 540386, 614016, 632484, 903586, 963552, 1378997, 1831087, 2406336, 6384708, 9636172, 33607131, 250435845, 834731566, 40168333522, 53128868760, 35132395233040|
|practice_0130|multiplicative|128|1|108|112|0|108|5.19|11627, 16779, 25385, 40142, 360532, 111786, 114516, 180642, 361340, 254305, 301118, 963552, 903586, 372196, 540386, 614016, 632484, 1378997, 1831087, 2406336, 6384708, 9636172, 33607131, 250435845, 834731566, 40168333522, 53128868760, 35132395233040|
|practice_0130|multiplicative|128|2|109|111|0|109|32.47|25385, 114516, 16779, 111786, 11627, 301118, 361340, 963552, 903586, 40142, 360532, 372196, 1378997, 1831087, 180642, 2406336, 254305, 540386, 614016, 632484, 6384708, 9636172, 33607131, 250435845, 834731566, 40168333522, 53128868760, 35132395233040|
|practice_0131|mixed|0|0|228|0|0|228|3.60|3286709200, 4453885416, 5383914506, 7295850977, 32303472174, 43071291600, 48309535308, 70779474923, 106893247098, 115942890528, 1174921334883, 2084816899107, 3512566495080, 203479313316730, 265095698635928, 283942958585736, 312693558611849, 326154042096430, 399957485856517, 412964249158266, 507855444232721, 543265354379831, 553324831822924, 709033974647465, 717048969276903, 815064768070002, 817574801714314, 995015701928045|
|practice_0131|mixed|0|1|214|0|0|214|11.14|4453885416, 106893247098, 3286709200, 5383914506, 32303472174, 43071291600, 7295850977, 48309535308, 70779474923, 115942890528, 1174921334883, 2084816899107, 3512566495080, 265095698635928, 326154042096430, 203479313316730, 283942958585736, 312693558611849, 553324831822924, 412964249158266, 709033974647465, 399957485856517, 817574801714314, 507855444232721, 543265354379831, 815064768070002, 717048969276903, 995015701928045|
|practice_0131|mixed|0|2|212|0|0|212|67.67|43071291600, 7295850977, 4453885416, 106893247098, 5383914506, 32303472174, 2084816899107, 115942890528, 3286709200, 1174921334883, 48309535308, 265095698635928, 70779474923, 203479313316730, 399957485856517, 283942958585736, 543265354379831, 3512566495080, 717048969276903, 817574801714314, 507855444232721, 815064768070002, 326154042096430, 995015701928045, 709033974647465, 412964249158266, 312693558611849, 553324831822924|
|practice_0131|mixed|48|0|231|272|0|231|1.47|3286709200, 4453885416, 5383914506, 7295850977, 32303472174, 43071291600, 48309535308, 70779474923, 106893247098, 115942890528, 1174921334883, 2084816899107, 3512566495080, 203479313316730, 265095698635928, 283942958585736, 312693558611849, 326154042096430, 399957485856517, 412964249158266, 507855444232721, 543265354379831, 553324831822924, 709033974647465, 717048969276903, 815064768070002, 817574801714314, 995015701928045|
|practice_0131|mixed|48|1|224|239|0|224|4.39|4453885416, 106893247098, 3286709200, 43071291600, 5383914506, 32303472174, 70779474923, 3512566495080, 7295850977, 115942890528, 48309535308, 1174921334883, 2084816899107, 283942958585736, 543265354379831, 507855444232721, 326154042096430, 265095698635928, 203479313316730, 312693558611849, 399957485856517, 553324831822924, 717048969276903, 709033974647465, 817574801714314, 815064768070002, 412964249158266, 995015701928045|
|practice_0131|mixed|48|2|229|252|0|229|29.01|43071291600, 4453885416, 7295850977, 5383914506, 106893247098, 3286709200, 412964249158266, 48309535308, 2084816899107, 3512566495080, 265095698635928, 399957485856517, 1174921334883, 70779474923, 115942890528, 717048969276903, 312693558611849, 32303472174, 203479313316730, 815064768070002, 709033974647465, 507855444232721, 543265354379831, 553324831822924, 817574801714314, 283942958585736, 326154042096430, 995015701928045|
|practice_0131|mixed|128|0|208|215|26|234|7.63|3286709200, 4453885416, 5383914506, 7295850977, 32303472174, 43071291600, 48309535308, 70779474923, 106893247098, 115942890528, 1174921334883, 2084816899107, 3512566495080, 203479313316730, 265095698635928, 283942958585736, 312693558611849, 326154042096430, 399957485856517, 412964249158266, 507855444232721, 543265354379831, 553324831822924, 709033974647465, 717048969276903, 815064768070002, 817574801714314, 995015701928045|
|practice_0131|mixed|128|1|202|206|0|202|19.86|4453885416, 106893247098, 3286709200, 5383914506, 32303472174, 43071291600, 7295850977, 48309535308, 70779474923, 115942890528, 1174921334883, 2084816899107, 3512566495080, 265095698635928, 203479313316730, 283942958585736, 312693558611849, 326154042096430, 399957485856517, 412964249158266, 543265354379831, 709033974647465, 553324831822924, 717048969276903, 815064768070002, 817574801714314, 507855444232721, 995015701928045|
|practice_0131|mixed|128|2|198|202|27|225|126.99|43071291600, 4453885416, 70779474923, 3286709200, 5383914506, 7295850977, 32303472174, 106893247098, 48309535308, 1174921334883, 115942890528, 2084816899107, 3512566495080, 265095698635928, 203479313316730, 312693558611849, 412964249158266, 817574801714314, 553324831822924, 326154042096430, 709033974647465, 399957485856517, 815064768070002, 543265354379831, 283942958585736, 507855444232721, 717048969276903, 995015701928045|
|practice_0132|mixed|0|0|179|0|0|179|3.33|4321209, 7158484, 19097731, 22595372, 22595510, 112975569, 138388978, 172986455, 196332232, 220477522, 1099604955, 2387113994, 10607930340, 190683869115, 212948545996, 254002230164, 315259085923, 394004981039, 430754991719, 448510015470, 476309701550, 616466922200, 712732660299, 814611284063, 884805273737, 917029488363, 918268708816, 980814002854, 3190633663581|
|practice_0132|mixed|0|1|175|0|7|182|8.19|4321209, 7158484, 19097731, 2387113994, 22595372, 22595510, 112975569, 220477522, 138388978, 172986455, 196332232, 1099604955, 10607930340, 616466922200, 190683869115, 448510015470, 315259085923, 394004981039, 254002230164, 212948545996, 430754991719, 917029488363, 918268708816, 814611284063, 884805273737, 476309701550, 712732660299, 980814002854, 3190633663581|
|practice_0132|mixed|0|2|170|0|0|170|50.12|19097731, 22595372, 4321209, 112975569, 22595510, 7158484, 220477522, 196332232, 138388978, 172986455, 2387113994, 315259085923, 10607930340, 1099604955, 476309701550, 448510015470, 616466922200, 430754991719, 254002230164, 917029488363, 884805273737, 712732660299, 190683869115, 212948545996, 918268708816, 394004981039, 814611284063, 980814002854, 3190633663581|
|practice_0132|mixed|48|0|186|226|0|186|1.27|4321209, 7158484, 19097731, 22595372, 22595510, 112975569, 138388978, 172986455, 196332232, 220477522, 1099604955, 2387113994, 10607930340, 190683869115, 212948545996, 254002230164, 315259085923, 394004981039, 430754991719, 448510015470, 476309701550, 616466922200, 712732660299, 814611284063, 884805273737, 917029488363, 918268708816, 980814002854, 3190633663581|
|practice_0132|mixed|48|1|183|189|0|183|3.37|4321209, 7158484, 19097731, 138388978, 2387113994, 22595372, 22595510, 112975569, 196332232, 172986455, 220477522, 1099604955, 10607930340, 190683869115, 212948545996, 254002230164, 616466922200, 394004981039, 315259085923, 448510015470, 430754991719, 476309701550, 712732660299, 918268708816, 917029488363, 884805273737, 980814002854, 3190633663581, 814611284063|
|practice_0132|mixed|48|2|180|189|0|180|21.71|22595372, 7158484, 22595510, 112975569, 4321209, 19097731, 196332232, 138388978, 172986455, 2387113994, 616466922200, 212948545996, 1099604955, 394004981039, 254002230164, 430754991719, 10607930340, 315259085923, 476309701550, 448510015470, 712732660299, 884805273737, 980814002854, 190683869115, 814611284063, 917029488363, 918268708816, 220477522, 3190633663581|
|practice_0132|mixed|128|0|166|171|25|191|6.00|4321209, 7158484, 19097731, 22595372, 22595510, 112975569, 138388978, 172986455, 196332232, 220477522, 1099604955, 2387113994, 10607930340, 190683869115, 212948545996, 254002230164, 315259085923, 394004981039, 430754991719, 448510015470, 476309701550, 616466922200, 712732660299, 814611284063, 884805273737, 917029488363, 918268708816, 980814002854, 3190633663581|
|practice_0132|mixed|128|1|166|168|3|169|13.70|4321209, 7158484, 19097731, 138388978, 2387113994, 22595372, 22595510, 112975569, 172986455, 196332232, 220477522, 1099604955, 10607930340, 190683869115, 315259085923, 394004981039, 476309701550, 712732660299, 212948545996, 254002230164, 430754991719, 918268708816, 884805273737, 448510015470, 814611284063, 917029488363, 616466922200, 980814002854, 3190633663581|
|practice_0132|mixed|128|2|163|165|19|182|88.45|22595372, 7158484, 22595510, 4321209, 19097731, 220477522, 112975569, 172986455, 138388978, 196332232, 616466922200, 2387113994, 10607930340, 254002230164, 918268708816, 1099604955, 394004981039, 190683869115, 212948545996, 430754991719, 884805273737, 448510015470, 814611284063, 3190633663581, 980814002854, 315259085923, 476309701550, 917029488363, 712732660299|
|practice_0133|shared|0|0|189|0|0|189|3.42|83742520244, 209565661837, 209565670282, 272138950762, 272138955692, 524438120755, 524438120766, 524438123677, 524438132456, 628697010705, 681027778766, 884372849044, 1362055556788, 3809945392607, 7347751784973, 7369342058005, 73767115963199, 95792912657030, 96065051613784, 202859568881866, 473731439588586, 481486639322430, 1302531166797593, 6491506203925256, 10577723454466872, 18475513750194024, 42370823886526278, 65274190700452667, 385291465624708250|
|practice_0133|shared|0|1|178|0|59|237|10.55|272138950762, 272138955692, 7347751784973, 3809945392607, 95792912657030, 96065051613784, 83742520244, 524438132456, 524438120766, 524438120755, 524438123677, 1362055556788, 681027778766, 481486639322430, 385291465624708250, 209565661837, 209565670282, 628697010705, 73767115963199, 6491506203925256, 202859568881866, 10577723454466872, 7369342058005, 884372849044, 42370823886526278, 473731439588586, 65274190700452667, 18475513750194024, 1302531166797593|
|practice_0133|shared|0|2|174|0|6|180|68.73|272138955692, 209565670282, 73767115963199, 272138950762, 209565661837, 628697010705, 96065051613784, 7347751784973, 3809945392607, 524438120755, 524438120766, 681027778766, 1362055556788, 524438132456, 481486639322430, 524438123677, 95792912657030, 385291465624708250, 6491506203925256, 10577723454466872, 83742520244, 884372849044, 7369342058005, 473731439588586, 202859568881866, 42370823886526278, 18475513750194024, 65274190700452667, 1302531166797593|
|practice_0133|shared|48|0|215|251|0|215|1.41|83742520244, 209565661837, 209565670282, 272138950762, 272138955692, 524438120755, 524438120766, 524438123677, 524438132456, 628697010705, 681027778766, 884372849044, 1362055556788, 3809945392607, 7347751784973, 7369342058005, 73767115963199, 95792912657030, 96065051613784, 202859568881866, 473731439588586, 481486639322430, 1302531166797593, 6491506203925256, 10577723454466872, 18475513750194024, 42370823886526278, 65274190700452667, 385291465624708250|
|practice_0133|shared|48|1|184|194|0|184|4.09|272138950762, 272138955692, 7347751784973, 3809945392607, 95792912657030, 96065051613784, 209565661837, 209565670282, 628697010705, 73767115963199, 6491506203925256, 202859568881866, 83742520244, 7369342058005, 524438120766, 524438120755, 524438132456, 524438123677, 681027778766, 1362055556788, 481486639322430, 10577723454466872, 884372849044, 473731439588586, 385291465624708250, 42370823886526278, 1302531166797593, 18475513750194024, 65274190700452667|
|practice_0133|shared|48|2|198|210|0|198|27.46|272138955692, 209565670282, 73767115963199, 3809945392607, 209565661837, 96065051613784, 628697010705, 681027778766, 524438123677, 272138950762, 7347751784973, 524438120755, 524438120766, 202859568881866, 524438132456, 6491506203925256, 95792912657030, 473731439588586, 42370823886526278, 83742520244, 7369342058005, 1362055556788, 884372849044, 10577723454466872, 1302531166797593, 18475513750194024, 481486639322430, 385291465624708250, 65274190700452667|
|practice_0133|shared|128|0|183|189|19|202|6.62|83742520244, 209565661837, 209565670282, 272138950762, 272138955692, 524438120755, 524438120766, 524438123677, 524438132456, 628697010705, 681027778766, 884372849044, 1362055556788, 3809945392607, 7347751784973, 7369342058005, 73767115963199, 95792912657030, 96065051613784, 202859568881866, 473731439588586, 481486639322430, 1302531166797593, 6491506203925256, 10577723454466872, 18475513750194024, 42370823886526278, 65274190700452667, 385291465624708250|
|practice_0133|shared|128|1|162|167|0|162|19.40|272138950762, 272138955692, 7347751784973, 3809945392607, 95792912657030, 96065051613784, 83742520244, 209565661837, 209565670282, 628697010705, 73767115963199, 202859568881866, 6491506203925256, 524438123677, 524438120755, 524438120766, 524438132456, 681027778766, 1362055556788, 481486639322430, 385291465624708250, 7369342058005, 884372849044, 473731439588586, 10577723454466872, 42370823886526278, 18475513750194024, 1302531166797593, 65274190700452667|
|practice_0133|shared|128|2|164|168|3|167|112.98|272138955692, 209565670282, 73767115963199, 272138950762, 628697010705, 209565661837, 7347751784973, 202859568881866, 3809945392607, 524438120755, 524438120766, 524438123677, 681027778766, 1362055556788, 524438132456, 481486639322430, 95792912657030, 96065051613784, 884372849044, 83742520244, 385291465624708250, 7369342058005, 18475513750194024, 65274190700452667, 6491506203925256, 10577723454466872, 473731439588586, 1302531166797593, 42370823886526278|
|practice_0134|independent|0|0|215|0|0|215|2.77|101488932214, 124798309420, 136398152533, 144100653396, 186818978252, 200012654520, 217589653092, 262423641955, 289031822279, 320323169546, 368916916140, 398054604485, 477647515103, 634073929916, 680158067821, 685302444338, 698916540993, 727503635163, 788142447754, 807577763684, 829205322584, 835668175186, 855934709941, 860501961807, 873053152557, 879107370295, 889208335956, 912027107957, 933906904155, 965052143411|
|practice_0134|independent|0|1|207|0|0|207|8.70|124798309420, 368916916140, 101488932214, 136398152533, 144100653396, 186818978252, 200012654520, 217589653092, 262423641955, 398054604485, 477647515103, 680158067821, 289031822279, 320323169546, 685302444338, 634073929916, 698916540993, 807577763684, 829205322584, 788142447754, 835668175186, 855934709941, 933906904155, 879107370295, 860501961807, 889208335956, 873053152557, 912027107957, 965052143411, 727503635163|
|practice_0134|independent|0|2|200|0|19|219|55.91|144100653396, 101488932214, 200012654520, 136398152533, 262423641955, 124798309420, 320323169546, 217589653092, 289031822279, 186818978252, 368916916140, 680158067821, 634073929916, 807577763684, 873053152557, 727503635163, 685302444338, 912027107957, 788142447754, 860501961807, 855934709941, 698916540993, 879107370295, 835668175186, 965052143411, 398054604485, 933906904155, 829205322584, 477647515103, 889208335956|
|practice_0134|independent|48|0|219|245|0|219|1.41|101488932214, 124798309420, 136398152533, 144100653396, 186818978252, 200012654520, 217589653092, 262423641955, 289031822279, 320323169546, 368916916140, 398054604485, 477647515103, 634073929916, 680158067821, 685302444338, 698916540993, 727503635163, 788142447754, 807577763684, 829205322584, 835668175186, 855934709941, 860501961807, 873053152557, 879107370295, 889208335956, 912027107957, 933906904155, 965052143411|
|practice_0134|independent|48|1|219|224|0|219|3.42|124798309420, 144100653396, 217589653092, 289031822279, 634073929916, 101488932214, 200012654520, 186818978252, 136398152533, 807577763684, 855934709941, 320323169546, 788142447754, 829205322584, 477647515103, 368916916140, 685302444338, 262423641955, 398054604485, 698916540993, 860501961807, 835668175186, 873053152557, 727503635163, 879107370295, 680158067821, 933906904155, 912027107957, 965052143411, 889208335956|
|practice_0134|independent|48|2|219|226|0|219|26.55|144100653396, 217589653092, 136398152533, 680158067821, 320323169546, 262423641955, 200012654520, 101488932214, 788142447754, 807577763684, 186818978252, 368916916140, 829205322584, 634073929916, 124798309420, 860501961807, 879107370295, 873053152557, 289031822279, 855934709941, 889208335956, 912027107957, 835668175186, 727503635163, 698916540993, 477647515103, 933906904155, 398054604485, 965052143411, 685302444338|
|practice_0134|independent|128|0|201|204|43|244|8.54|101488932214, 124798309420, 136398152533, 144100653396, 186818978252, 200012654520, 217589653092, 262423641955, 289031822279, 320323169546, 368916916140, 398054604485, 477647515103, 634073929916, 680158067821, 685302444338, 698916540993, 727503635163, 788142447754, 807577763684, 829205322584, 835668175186, 855934709941, 860501961807, 873053152557, 879107370295, 889208335956, 912027107957, 933906904155, 965052143411|
|practice_0134|independent|128|1|195|197|29|224|23.89|124798309420, 144100653396, 217589653092, 289031822279, 262423641955, 398054604485, 101488932214, 136398152533, 200012654520, 186818978252, 320323169546, 368916916140, 889208335956, 933906904155, 680158067821, 788142447754, 634073929916, 685302444338, 477647515103, 698916540993, 873053152557, 912027107957, 727503635163, 860501961807, 829205322584, 807577763684, 835668175186, 855934709941, 879107370295, 965052143411|
|practice_0134|independent|128|2|188|190|38|226|146.12|144100653396, 217589653092, 124798309420, 262423641955, 320323169546, 101488932214, 698916540993, 136398152533, 398054604485, 680158067821, 200012654520, 933906904155, 368916916140, 727503635163, 186818978252, 860501961807, 289031822279, 807577763684, 829205322584, 634073929916, 912027107957, 477647515103, 873053152557, 879107370295, 788142447754, 965052143411, 685302444338, 835668175186, 855934709941, 889208335956|
|practice_0135|independent|0|0|344|0|0|344|4.09|124135467626659721, 140313627463291012, 172179701896309054, 189870680774386158, 231208999096928564, 273052960209285473, 298868336222432507, 302038192989410070, 312168601611082185, 313492656165030267, 362889858852470727, 396067713321975751, 419902324508535523, 420157490537909631, 441984185557584724, 478078758817456364, 511238410377897948, 520177610979190938, 551890803958774692, 611782802261086688, 662544736511445942, 690776549101961323, 706907048025176221, 758853903986316582, 778135972479139480, 836245934200209715, 875779822043179619, 929900945553061311, 952060902169846540, 966600386608709992, 976702807412068467, 982202421089663201|
|practice_0135|independent|0|1|330|0|149|479|16.56|124135467626659721, 611782802261086688, 140313627463291012, 302038192989410070, 982202421089663201, 313492656165030267, 189870680774386158, 312168601611082185, 420157490537909631, 172179701896309054, 231208999096928564, 419902324508535523, 362889858852470727, 298868336222432507, 706907048025176221, 273052960209285473, 396067713321975751, 952060902169846540, 551890803958774692, 441984185557584724, 966600386608709992, 478078758817456364, 520177610979190938, 875779822043179619, 758853903986316582, 511238410377897948, 662544736511445942, 690776549101961323, 778135972479139480, 836245934200209715, 929900945553061311, 976702807412068467|
|practice_0135|independent|0|2|325|0|159|484|116.33|298868336222432507, 758853903986316582, 140313627463291012, 313492656165030267, 312168601611082185, 302038192989410070, 420157490537909631, 778135972479139480, 929900945553061311, 662544736511445942, 419902324508535523, 189870680774386158, 611782802261086688, 231208999096928564, 124135467626659721, 690776549101961323, 982202421089663201, 706907048025176221, 273052960209285473, 172179701896309054, 396067713321975751, 441984185557584724, 551890803958774692, 966600386608709992, 511238410377897948, 478078758817456364, 836245934200209715, 952060902169846540, 362889858852470727, 875779822043179619, 520177610979190938, 976702807412068467|
|practice_0135|independent|48|0|352|506|0|352|2.01|124135467626659721, 140313627463291012, 172179701896309054, 189870680774386158, 231208999096928564, 273052960209285473, 298868336222432507, 302038192989410070, 312168601611082185, 313492656165030267, 362889858852470727, 396067713321975751, 419902324508535523, 420157490537909631, 441984185557584724, 478078758817456364, 511238410377897948, 520177610979190938, 551890803958774692, 611782802261086688, 662544736511445942, 690776549101961323, 706907048025176221, 758853903986316582, 778135972479139480, 836245934200209715, 875779822043179619, 929900945553061311, 952060902169846540, 966600386608709992, 976702807412068467, 982202421089663201|
|practice_0135|independent|48|1|347|359|0|347|6.80|124135467626659721, 611782802261086688, 189870680774386158, 231208999096928564, 511238410377897948, 172179701896309054, 273052960209285473, 419902324508535523, 298868336222432507, 362889858852470727, 758853903986316582, 662544736511445942, 313492656165030267, 312168601611082185, 302038192989410070, 441984185557584724, 836245934200209715, 396067713321975751, 520177610979190938, 478078758817456364, 140313627463291012, 706907048025176221, 929900945553061311, 875779822043179619, 976702807412068467, 966600386608709992, 551890803958774692, 982202421089663201, 420157490537909631, 690776549101961323, 778135972479139480, 952060902169846540|
|practice_0135|independent|48|2|349|365|0|349|43.34|298868336222432507, 511238410377897948, 189870680774386158, 362889858852470727, 929900945553061311, 396067713321975751, 441984185557584724, 836245934200209715, 172179701896309054, 520177610979190938, 976702807412068467, 302038192989410070, 313492656165030267, 982202421089663201, 231208999096928564, 312168601611082185, 706907048025176221, 875779822043179619, 140313627463291012, 662544736511445942, 124135467626659721, 952060902169846540, 419902324508535523, 611782802261086688, 420157490537909631, 551890803958774692, 478078758817456364, 778135972479139480, 966600386608709992, 690776549101961323, 758853903986316582, 273052960209285473|
|practice_0135|independent|128|0|314|320|75|389|11.13|124135467626659721, 140313627463291012, 172179701896309054, 189870680774386158, 231208999096928564, 273052960209285473, 298868336222432507, 302038192989410070, 312168601611082185, 313492656165030267, 362889858852470727, 396067713321975751, 419902324508535523, 420157490537909631, 441984185557584724, 478078758817456364, 511238410377897948, 520177610979190938, 551890803958774692, 611782802261086688, 662544736511445942, 690776549101961323, 706907048025176221, 758853903986316582, 778135972479139480, 836245934200209715, 875779822043179619, 929900945553061311, 952060902169846540, 966600386608709992, 976702807412068467, 982202421089663201|
|practice_0135|independent|128|1|305|308|32|337|49.35|124135467626659721, 611782802261086688, 140313627463291012, 302038192989410070, 982202421089663201, 172179701896309054, 298868336222432507, 189870680774386158, 273052960209285473, 231208999096928564, 312168601611082185, 362889858852470727, 313492656165030267, 396067713321975751, 419902324508535523, 420157490537909631, 441984185557584724, 478078758817456364, 511238410377897948, 551890803958774692, 690776549101961323, 929900945553061311, 520177610979190938, 976702807412068467, 706907048025176221, 952060902169846540, 778135972479139480, 662544736511445942, 758853903986316582, 966600386608709992, 875779822043179619, 836245934200209715|
|practice_0135|independent|128|2|296|301|24|320|354.37|298868336222432507, 758853903986316582, 140313627463291012, 520177610979190938, 312168601611082185, 273052960209285473, 231208999096928564, 124135467626659721, 551890803958774692, 420157490537909631, 189870680774386158, 929900945553061311, 362889858852470727, 966600386608709992, 302038192989410070, 478078758817456364, 511238410377897948, 419902324508535523, 662544736511445942, 313492656165030267, 976702807412068467, 396067713321975751, 611782802261086688, 172179701896309054, 836245934200209715, 778135972479139480, 706907048025176221, 441984185557584724, 982202421089663201, 690776549101961323, 875779822043179619, 952060902169846540|
|practice_0136|mixed|0|0|56|0|0|56|0.25|67596, 83987, 266252, 266514, 426781, 445577, 532494, 564530, 716298, 814964, 842229, 3219840|
|practice_0136|mixed|0|1|55|0|0|55|0.22|266252, 532494, 266514, 67596, 83987, 564530, 426781, 814964, 842229, 3219840, 445577, 716298|
|practice_0136|mixed|0|2|52|0|0|52|0.99|266514, 266252, 67596, 532494, 83987, 564530, 426781, 445577, 716298, 814964, 842229, 3219840|
|practice_0136|mixed|48|0|57|60|0|57|0.21|67596, 83987, 266252, 266514, 426781, 445577, 532494, 564530, 716298, 814964, 842229, 3219840|
|practice_0136|mixed|48|1|55|59|0|55|0.19|266252, 532494, 266514, 67596, 83987, 445577, 564530, 426781, 814964, 3219840, 716298, 842229|
|practice_0136|mixed|48|2|51|55|0|51|0.86|266514, 266252, 67596, 532494, 83987, 564530, 426781, 445577, 814964, 716298, 842229, 3219840|
|practice_0136|mixed|128|0|58|59|0|58|0.22|67596, 83987, 266252, 266514, 426781, 445577, 532494, 564530, 716298, 814964, 842229, 3219840|
|practice_0136|mixed|128|1|55|57|0|55|0.19|266252, 532494, 266514, 67596, 83987, 445577, 564530, 426781, 814964, 3219840, 716298, 842229|
|practice_0136|mixed|128|2|52|55|0|52|0.92|266514, 266252, 67596, 532494, 83987, 564530, 426781, 445577, 716298, 814964, 842229, 3219840|
|practice_0137|multiplicative|0|0|106|0|0|106|1.96|13285, 21928, 35767, 37752, 37780, 46877, 62385, 75459, 77818, 113218, 113477, 226474, 232980, 265314, 381495, 432310, 565905, 584324, 610324, 1207844, 1358819, 2533961, 5160666, 15626621, 19667807, 61521110, 68215086, 204608104, 793166320|
|practice_0137|multiplicative|0|1|107|0|0|107|2.47|13285, 77818, 232980, 21928, 35767, 610324, 37752, 37780, 46877, 62385, 75459, 113218, 113477, 226474, 265314, 381495, 432310, 565905, 584324, 1207844, 1358819, 2533961, 5160666, 19667807, 15626621, 61521110, 68215086, 204608104, 793166320|
|practice_0137|multiplicative|0|2|103|0|0|103|16.74|13285, 37752, 46877, 37780, 35767, 62385, 113218, 113477, 75459, 77818, 226474, 232980, 610324, 265314, 584324, 21928, 381495, 5160666, 432310, 565905, 1207844, 1358819, 2533961, 19667807, 68215086, 15626621, 204608104, 61521110, 793166320|
|practice_0137|multiplicative|48|0|111|116|0|111|0.88|13285, 21928, 35767, 37752, 37780, 46877, 62385, 75459, 77818, 113218, 113477, 226474, 232980, 265314, 381495, 432310, 565905, 584324, 610324, 1207844, 1358819, 2533961, 5160666, 15626621, 19667807, 61521110, 68215086, 204608104, 793166320|
|practice_0137|multiplicative|48|1|106|144|0|106|1.22|13285, 77818, 232980, 21928, 35767, 610324, 37752, 37780, 46877, 62385, 75459, 113218, 226474, 565905, 113477, 265314, 5160666, 381495, 432310, 584324, 1207844, 1358819, 2533961, 15626621, 19667807, 61521110, 68215086, 204608104, 793166320|
|practice_0137|multiplicative|48|2|102|107|0|102|7.17|37752, 13285, 46877, 75459, 113218, 37780, 113477, 35767, 77818, 610324, 565905, 226474, 265314, 232980, 381495, 62385, 21928, 1358819, 432310, 5160666, 584324, 1207844, 19667807, 2533961, 204608104, 15626621, 61521110, 68215086, 793166320|
|practice_0137|multiplicative|128|0|104|107|0|104|3.35|13285, 21928, 35767, 37752, 37780, 46877, 62385, 75459, 77818, 113218, 113477, 226474, 232980, 265314, 381495, 432310, 565905, 584324, 610324, 1207844, 1358819, 2533961, 5160666, 15626621, 19667807, 61521110, 68215086, 204608104, 793166320|
|practice_0137|multiplicative|128|1|106|108|0|106|3.54|13285, 77818, 232980, 21928, 35767, 610324, 37752, 37780, 46877, 62385, 75459, 113218, 113477, 226474, 565905, 265314, 381495, 432310, 584324, 1207844, 1358819, 2533961, 5160666, 15626621, 19667807, 61521110, 68215086, 204608104, 793166320|
|practice_0137|multiplicative|128|2|100|102|0|100|17.19|37752, 13285, 46877, 75459, 113218, 37780, 113477, 35767, 226474, 77818, 232980, 565905, 610324, 62385, 265314, 1358819, 1207844, 2533961, 21928, 381495, 432310, 584324, 5160666, 19667807, 61521110, 15626621, 68215086, 204608104, 793166320|
|practice_0138|independent|0|0|346|0|0|346|3.90|109043512172482118, 133146592939631857, 188094863066545394, 216499168346711300, 272579870831984343, 294295019843106974, 341436522262830305, 370013064724646303, 380011025269908502, 419322506321694936, 423493384297253099, 427494485270854716, 458146375000742171, 463122809908196037, 526637939803736991, 550121650131232081, 554582926157527683, 623330854068294276, 623685194843148515, 626884079414117374, 768323472289316550, 782068093292473575, 792898340269867212, 833169166523543804, 858004634714158858, 867467640733493317, 883716370987495099, 893966448159572461, 937507446966936710, 938975939109579325, 958246263081504287, 984619727015287499|
|practice_0138|independent|0|1|325|0|234|559|15.67|109043512172482118, 272579870831984343, 133146592939631857, 626884079414117374, 458146375000742171, 423493384297253099, 419322506321694936, 294295019843106974, 623685194843148515, 782068093292473575, 188094863066545394, 370013064724646303, 380011025269908502, 623330854068294276, 216499168346711300, 341436522262830305, 554582926157527683, 427494485270854716, 768323472289316550, 937507446966936710, 833169166523543804, 984619727015287499, 463122809908196037, 526637939803736991, 867467640733493317, 858004634714158858, 792898340269867212, 550121650131232081, 938975939109579325, 958246263081504287, 883716370987495099, 893966448159572461|
|practice_0138|independent|0|2|321|0|83|404|118.20|188094863066545394, 341436522262830305, 423493384297253099, 380011025269908502, 109043512172482118, 272579870831984343, 216499168346711300, 427494485270854716, 623330854068294276, 623685194843148515, 782068093292473575, 463122809908196037, 768323472289316550, 554582926157527683, 370013064724646303, 294295019843106974, 958246263081504287, 550121650131232081, 883716370987495099, 893966448159572461, 858004634714158858, 867467640733493317, 792898340269867212, 626884079414117374, 133146592939631857, 526637939803736991, 984619727015287499, 833169166523543804, 419322506321694936, 937507446966936710, 458146375000742171, 938975939109579325|
|practice_0138|independent|48|0|363|721|0|363|1.82|109043512172482118, 133146592939631857, 188094863066545394, 216499168346711300, 272579870831984343, 294295019843106974, 341436522262830305, 370013064724646303, 380011025269908502, 419322506321694936, 423493384297253099, 427494485270854716, 458146375000742171, 463122809908196037, 526637939803736991, 550121650131232081, 554582926157527683, 623330854068294276, 623685194843148515, 626884079414117374, 768323472289316550, 782068093292473575, 792898340269867212, 833169166523543804, 858004634714158858, 867467640733493317, 883716370987495099, 893966448159572461, 937507446966936710, 938975939109579325, 958246263081504287, 984619727015287499|
|practice_0138|independent|48|1|366|461|0|366|6.43|109043512172482118, 272579870831984343, 380011025269908502, 768323472289316550, 133146592939631857, 419322506321694936, 427494485270854716, 294295019843106974, 623685194843148515, 782068093292473575, 463122809908196037, 623330854068294276, 526637939803736991, 341436522262830305, 554582926157527683, 550121650131232081, 858004634714158858, 833169166523543804, 458146375000742171, 958246263081504287, 937507446966936710, 883716370987495099, 188094863066545394, 938975939109579325, 216499168346711300, 867467640733493317, 370013064724646303, 792898340269867212, 893966448159572461, 626884079414117374, 423493384297253099, 984619727015287499|
|practice_0138|independent|48|2|359|385|0|359|46.72|188094863066545394, 833169166523543804, 370013064724646303, 463122809908196037, 458146375000742171, 380011025269908502, 427494485270854716, 554582926157527683, 419322506321694936, 623330854068294276, 623685194843148515, 937507446966936710, 792898340269867212, 883716370987495099, 272579870831984343, 984619727015287499, 768323472289316550, 550121650131232081, 341436522262830305, 216499168346711300, 893966448159572461, 958246263081504287, 526637939803736991, 133146592939631857, 782068093292473575, 938975939109579325, 867467640733493317, 423493384297253099, 626884079414117374, 294295019843106974, 109043512172482118, 858004634714158858|
|practice_0138|independent|128|0|314|323|32|346|11.78|109043512172482118, 133146592939631857, 188094863066545394, 216499168346711300, 272579870831984343, 294295019843106974, 341436522262830305, 370013064724646303, 380011025269908502, 419322506321694936, 423493384297253099, 427494485270854716, 458146375000742171, 463122809908196037, 526637939803736991, 550121650131232081, 554582926157527683, 623330854068294276, 623685194843148515, 626884079414117374, 768323472289316550, 782068093292473575, 792898340269867212, 833169166523543804, 858004634714158858, 867467640733493317, 883716370987495099, 893966448159572461, 937507446966936710, 938975939109579325, 958246263081504287, 984619727015287499|
|practice_0138|independent|128|1|305|309|30|335|53.63|109043512172482118, 272579870831984343, 133146592939631857, 427494485270854716, 883716370987495099, 188094863066545394, 216499168346711300, 294295019843106974, 341436522262830305, 370013064724646303, 626884079414117374, 458146375000742171, 623330854068294276, 423493384297253099, 623685194843148515, 463122809908196037, 380011025269908502, 526637939803736991, 867467640733493317, 937507446966936710, 550121650131232081, 554582926157527683, 419322506321694936, 792898340269867212, 958246263081504287, 768323472289316550, 782068093292473575, 858004634714158858, 833169166523543804, 938975939109579325, 893966448159572461, 984619727015287499|
|practice_0138|independent|128|2|294|296|69|363|327.97|188094863066545394, 341436522262830305, 423493384297253099, 380011025269908502, 937507446966936710, 294295019843106974, 272579870831984343, 109043512172482118, 463122809908196037, 554582926157527683, 938975939109579325, 550121650131232081, 867467640733493317, 427494485270854716, 216499168346711300, 623685194843148515, 768323472289316550, 984619727015287499, 133146592939631857, 623330854068294276, 419322506321694936, 833169166523543804, 526637939803736991, 792898340269867212, 893966448159572461, 858004634714158858, 458146375000742171, 626884079414117374, 370013064724646303, 782068093292473575, 958246263081504287, 883716370987495099|
|practice_0139|mixed|0|0|218|0|0|218|2.95|2409533406, 2409533573, 4097831852, 4097831964, 4819067132, 6030750920, 30153754578, 33733473031, 81560156848, 104536154702, 662062312910, 679488457901, 5436162049878, 226571913217577, 227272628441446, 241617900093270, 258282958404928, 383035323402593, 410715588589575, 571862962837017, 623469352588386, 640921983123953, 694505498941329, 800019175280092, 842717403000419, 849974684750025, 904089036358007, 920050573372680, 3040137658587960|
|practice_0139|mixed|0|1|202|0|6|208|10.20|2409533406, 2409533573, 4819067132, 33733473031, 4097831852, 4097831964, 679488457901, 6030750920, 30153754578, 81560156848, 5436162049878, 104536154702, 662062312910, 258282958404928, 226571913217577, 227272628441446, 920050573372680, 571862962837017, 241617900093270, 383035323402593, 410715588589575, 623469352588386, 800019175280092, 694505498941329, 3040137658587960, 640921983123953, 842717403000419, 849974684750025, 904089036358007|
|practice_0139|mixed|0|2|205|0|29|234|65.30|2409533406, 4097831852, 6030750920, 4097831964, 2409533573, 81560156848, 662062312910, 4819067132, 33733473031, 30153754578, 104536154702, 5436162049878, 679488457901, 227272628441446, 849974684750025, 410715588589575, 623469352588386, 226571913217577, 241617900093270, 571862962837017, 842717403000419, 904089036358007, 258282958404928, 800019175280092, 694505498941329, 640921983123953, 920050573372680, 3040137658587960, 383035323402593|
|practice_0139|mixed|48|0|221|247|0|221|1.45|2409533406, 2409533573, 4097831852, 4097831964, 4819067132, 6030750920, 30153754578, 33733473031, 81560156848, 104536154702, 662062312910, 679488457901, 5436162049878, 226571913217577, 227272628441446, 241617900093270, 258282958404928, 383035323402593, 410715588589575, 571862962837017, 623469352588386, 640921983123953, 694505498941329, 800019175280092, 842717403000419, 849974684750025, 904089036358007, 920050573372680, 3040137658587960|
|practice_0139|mixed|48|1|218|235|0|218|4.63|2409533406, 2409533573, 4819067132, 33733473031, 679488457901, 6030750920, 30153754578, 4097831852, 4097831964, 104536154702, 81560156848, 662062312910, 226571913217577, 227272628441446, 5436162049878, 258282958404928, 383035323402593, 842717403000419, 241617900093270, 571862962837017, 410715588589575, 640921983123953, 623469352588386, 920050573372680, 849974684750025, 694505498941329, 904089036358007, 800019175280092, 3040137658587960|
|practice_0139|mixed|48|2|218|231|0|218|30.68|2409533406, 679488457901, 2409533573, 6030750920, 4097831852, 4097831964, 662062312910, 104536154702, 5436162049878, 4819067132, 81560156848, 33733473031, 226571913217577, 571862962837017, 383035323402593, 30153754578, 800019175280092, 640921983123953, 258282958404928, 227272628441446, 241617900093270, 3040137658587960, 904089036358007, 623469352588386, 410715588589575, 694505498941329, 920050573372680, 842717403000419, 849974684750025|
|practice_0139|mixed|128|0|195|203|5|200|6.45|2409533406, 2409533573, 4097831852, 4097831964, 4819067132, 6030750920, 30153754578, 33733473031, 81560156848, 104536154702, 662062312910, 679488457901, 5436162049878, 226571913217577, 227272628441446, 241617900093270, 258282958404928, 383035323402593, 410715588589575, 571862962837017, 623469352588386, 640921983123953, 694505498941329, 800019175280092, 842717403000419, 849974684750025, 904089036358007, 920050573372680, 3040137658587960|
|practice_0139|mixed|128|1|191|195|1|192|20.96|2409533406, 2409533573, 4819067132, 33733473031, 679488457901, 4097831852, 4097831964, 6030750920, 30153754578, 81560156848, 662062312910, 104536154702, 5436162049878, 241617900093270, 227272628441446, 226571913217577, 258282958404928, 410715588589575, 383035323402593, 849974684750025, 623469352588386, 571862962837017, 694505498941329, 640921983123953, 800019175280092, 904089036358007, 3040137658587960, 842717403000419, 920050573372680|
|practice_0139|mixed|128|2|191|195|2|193|139.70|2409533406, 4097831852, 6030750920, 4097831964, 679488457901, 2409533573, 4819067132, 81560156848, 104536154702, 662062312910, 30153754578, 33733473031, 227272628441446, 5436162049878, 694505498941329, 258282958404928, 226571913217577, 920050573372680, 571862962837017, 410715588589575, 3040137658587960, 623469352588386, 383035323402593, 842717403000419, 800019175280092, 904089036358007, 849974684750025, 241617900093270, 640921983123953|
|practice_0140|tiny|0|0|1|0|0|1|0.01|1, 9|
|practice_0140|tiny|0|1|1|0|0|1|0.02|1, 9|
|practice_0140|tiny|0|2|1|0|0|1|0.01|1, 9|
|practice_0140|tiny|48|0|1|8|0|1|0.01|1, 9|
|practice_0140|tiny|48|1|1|8|0|1|0.01|1, 9|
|practice_0140|tiny|48|2|1|8|0|1|0.01|1, 9|
|practice_0140|tiny|128|0|1|8|0|1|0.01|1, 9|
|practice_0140|tiny|128|1|1|8|0|1|0.01|1, 9|
|practice_0140|tiny|128|2|1|8|0|1|0.01|1, 9|
|practice_0141|shared|0|0|157|0|0|157|3.40|317395, 10674048, 17909536, 23023134, 23380060, 28107806, 32973715, 70343848, 73407977, 167086570, 174246430, 232002209, 314904755, 584748288, 626404858, 669892436, 1242599428, 3587999599, 5044212036, 8970026880, 33543414240, 123599679892, 527596589726, 695392572837, 2318896470912, 2463817337338, 10517434251862, 42486071912256|
|practice_0141|shared|0|1|155|0|0|155|7.72|317395, 10674048, 17909536, 23380060, 23023134, 28107806, 32973715, 70343848, 73407977, 167086570, 314904755, 584748288, 626404858, 669892436, 5044212036, 8970026880, 174246430, 232002209, 1242599428, 3587999599, 33543414240, 123599679892, 527596589726, 695392572837, 2463817337338, 2318896470912, 42486071912256, 10517434251862|
|practice_0141|shared|0|2|152|0|0|152|41.43|10674048, 317395, 23023134, 584748288, 73407977, 17909536, 32973715, 28107806, 23380060, 70343848, 167086570, 5044212036, 3587999599, 8970026880, 669892436, 314904755, 174246430, 232002209, 626404858, 33543414240, 123599679892, 1242599428, 527596589726, 2463817337338, 695392572837, 2318896470912, 42486071912256, 10517434251862|
|practice_0141|shared|48|0|166|193|0|166|1.06|317395, 10674048, 17909536, 23023134, 23380060, 28107806, 32973715, 70343848, 73407977, 167086570, 174246430, 232002209, 314904755, 584748288, 626404858, 669892436, 1242599428, 3587999599, 5044212036, 8970026880, 33543414240, 123599679892, 527596589726, 695392572837, 2318896470912, 2463817337338, 10517434251862, 42486071912256|
|practice_0141|shared|48|1|162|173|0|162|2.54|317395, 10674048, 17909536, 23380060, 23023134, 28107806, 32973715, 70343848, 167086570, 232002209, 584748288, 174246430, 73407977, 314904755, 669892436, 3587999599, 8970026880, 1242599428, 626404858, 5044212036, 33543414240, 695392572837, 2463817337338, 42486071912256, 527596589726, 123599679892, 2318896470912, 10517434251862|
|practice_0141|shared|48|2|162|169|0|162|16.95|10674048, 317395, 23023134, 17909536, 23380060, 70343848, 73407977, 167086570, 5044212036, 174246430, 584748288, 1242599428, 28107806, 8970026880, 2318896470912, 314904755, 3587999599, 669892436, 32973715, 33543414240, 626404858, 123599679892, 527596589726, 2463817337338, 232002209, 10517434251862, 42486071912256, 695392572837|
|practice_0141|shared|128|0|153|159|52|205|3.90|317395, 10674048, 17909536, 23023134, 23380060, 28107806, 32973715, 70343848, 73407977, 167086570, 174246430, 232002209, 314904755, 584748288, 626404858, 669892436, 1242599428, 3587999599, 5044212036, 8970026880, 33543414240, 123599679892, 527596589726, 695392572837, 2318896470912, 2463817337338, 10517434251862, 42486071912256|
|practice_0141|shared|128|1|148|150|0|148|11.41|317395, 10674048, 17909536, 23380060, 23023134, 28107806, 32973715, 70343848, 73407977, 167086570, 232002209, 314904755, 584748288, 626404858, 669892436, 1242599428, 5044212036, 174246430, 8970026880, 123599679892, 3587999599, 33543414240, 527596589726, 695392572837, 2318896470912, 10517434251862, 2463817337338, 42486071912256|
|practice_0141|shared|128|2|145|149|14|159|66.11|10674048, 317395, 23023134, 584748288, 73407977, 17909536, 32973715, 28107806, 23380060, 167086570, 174246430, 232002209, 5044212036, 8970026880, 3587999599, 33543414240, 70343848, 669892436, 314904755, 1242599428, 527596589726, 626404858, 2318896470912, 123599679892, 42486071912256, 2463817337338, 695392572837, 10517434251862|
|practice_0142|independent|0|0|264|0|0|264|3.00|150969524630344, 191359198622406, 211262135148758, 280919862123017, 287866052184623, 332371775023707, 349226917777691, 365984805202671, 371053555423429, 423337231014750, 495816805612011, 514770095657472, 524045219117091, 526466283536429, 540383064137950, 638169857520724, 651297280637475, 659393683214408, 674491180158905, 689179876347014, 730111591697483, 739682569233018, 742418919477640, 786349098132365, 842027413513487, 865051623969528, 896318856228745, 923295789969061, 933925509982725, 964616967032530|
|practice_0142|independent|0|1|255|0|25|280|10.43|638169857520724, 150969524630344, 191359198622406, 211262135148758, 287866052184623, 280919862123017, 526466283536429, 730111591697483, 365984805202671, 739682569233018, 332371775023707, 349226917777691, 495816805612011, 423337231014750, 514770095657472, 842027413513487, 371053555423429, 524045219117091, 540383064137950, 651297280637475, 896318856228745, 659393683214408, 674491180158905, 742418919477640, 865051623969528, 689179876347014, 923295789969061, 933925509982725, 964616967032530, 786349098132365|
|practice_0142|independent|0|2|249|0|63|312|74.08|191359198622406, 150969524630344, 638169857520724, 651297280637475, 211262135148758, 280919862123017, 371053555423429, 423337231014750, 742418919477640, 526466283536429, 739682569233018, 514770095657472, 933925509982725, 842027413513487, 964616967032530, 495816805612011, 349226917777691, 332371775023707, 659393683214408, 365984805202671, 923295789969061, 674491180158905, 540383064137950, 689179876347014, 865051623969528, 524045219117091, 730111591697483, 287866052184623, 786349098132365, 896318856228745|
|practice_0142|independent|48|0|271|328|0|271|1.58|150969524630344, 191359198622406, 211262135148758, 280919862123017, 287866052184623, 332371775023707, 349226917777691, 365984805202671, 371053555423429, 423337231014750, 495816805612011, 514770095657472, 524045219117091, 526466283536429, 540383064137950, 638169857520724, 651297280637475, 659393683214408, 674491180158905, 689179876347014, 730111591697483, 739682569233018, 742418919477640, 786349098132365, 842027413513487, 865051623969528, 896318856228745, 923295789969061, 933925509982725, 964616967032530|
|practice_0142|independent|48|1|270|299|0|270|4.83|638169857520724, 150969524630344, 191359198622406, 211262135148758, 287866052184623, 842027413513487, 371053555423429, 423337231014750, 349226917777691, 280919862123017, 514770095657472, 495816805612011, 332371775023707, 524045219117091, 365984805202671, 896318856228745, 526466283536429, 540383064137950, 651297280637475, 786349098132365, 659393683214408, 689179876347014, 933925509982725, 674491180158905, 730111591697483, 739682569233018, 964616967032530, 923295789969061, 742418919477640, 865051623969528|
|practice_0142|independent|48|2|272|289|0|272|32.97|150969524630344, 280919862123017, 371053555423429, 211262135148758, 287866052184623, 349226917777691, 365984805202671, 540383064137950, 526466283536429, 423337231014750, 191359198622406, 896318856228745, 524045219117091, 659393683214408, 638169857520724, 495816805612011, 651297280637475, 730111591697483, 689179876347014, 514770095657472, 933925509982725, 332371775023707, 674491180158905, 923295789969061, 865051623969528, 842027413513487, 742418919477640, 964616967032530, 786349098132365, 739682569233018|
|practice_0142|independent|128|0|248|260|22|270|8.37|150969524630344, 191359198622406, 211262135148758, 280919862123017, 287866052184623, 332371775023707, 349226917777691, 365984805202671, 371053555423429, 423337231014750, 495816805612011, 514770095657472, 524045219117091, 526466283536429, 540383064137950, 638169857520724, 651297280637475, 659393683214408, 674491180158905, 689179876347014, 730111591697483, 739682569233018, 742418919477640, 786349098132365, 842027413513487, 865051623969528, 896318856228745, 923295789969061, 933925509982725, 964616967032530|
|practice_0142|independent|128|1|239|243|14|253|25.81|638169857520724, 150969524630344, 191359198622406, 211262135148758, 349226917777691, 280919862123017, 332371775023707, 371053555423429, 423337231014750, 495816805612011, 514770095657472, 287866052184623, 365984805202671, 524045219117091, 659393683214408, 526466283536429, 674491180158905, 651297280637475, 689179876347014, 730111591697483, 540383064137950, 739682569233018, 742418919477640, 786349098132365, 842027413513487, 865051623969528, 964616967032530, 923295789969061, 933925509982725, 896318856228745|
|practice_0142|independent|128|2|236|240|44|280|202.73|150969524630344, 638169857520724, 211262135148758, 524045219117091, 742418919477640, 730111591697483, 423337231014750, 280919862123017, 191359198622406, 349226917777691, 371053555423429, 514770095657472, 365984805202671, 495816805612011, 287866052184623, 526466283536429, 865051623969528, 659393683214408, 651297280637475, 739682569233018, 689179876347014, 964616967032530, 674491180158905, 842027413513487, 540383064137950, 923295789969061, 933925509982725, 332371775023707, 896318856228745, 786349098132365|
|practice_0143|shared|0|0|156|0|0|156|2.73|32046448, 32046890, 56049449, 56049526, 56049557, 91109003, 91109036, 98027572, 256375672, 273327839, 416737537, 1457739424, 1793585940, 1961713425, 3097708369, 3279922920, 13028587854, 13459978560, 14950551550, 18427061792, 25874956568, 29953137475, 64457011250, 96870263776, 97378324586, 146067435150, 172740682440, 528895377996, 835698951528, 2096722640775, 3945699489100, 6232212795100|
|practice_0143|shared|0|1|160|0|0|160|6.42|32046448, 32046890, 256375672, 56049449, 56049526, 56049557, 91109003, 91109036, 1457739424, 273327839, 1793585940, 3097708369, 3279922920, 25874956568, 13028587854, 98027572, 1961713425, 416737537, 13459978560, 14950551550, 18427061792, 64457011250, 835698951528, 29953137475, 146067435150, 96870263776, 97378324586, 6232212795100, 172740682440, 528895377996, 2096722640775, 3945699489100|
|practice_0143|shared|0|2|152|0|0|152|40.54|91109003, 1457739424, 56049526, 25874956568, 91109036, 1793585940, 32046448, 56049557, 32046890, 256375672, 56049449, 273327839, 3279922920, 416737537, 18427061792, 3097708369, 64457011250, 98027572, 1961713425, 13459978560, 96870263776, 97378324586, 146067435150, 6232212795100, 13028587854, 172740682440, 835698951528, 29953137475, 14950551550, 528895377996, 2096722640775, 3945699489100|
|practice_0143|shared|48|0|165|213|0|165|1.05|32046448, 32046890, 56049449, 56049526, 56049557, 91109003, 91109036, 98027572, 256375672, 273327839, 416737537, 1457739424, 1793585940, 1961713425, 3097708369, 3279922920, 13028587854, 13459978560, 14950551550, 18427061792, 25874956568, 29953137475, 64457011250, 96870263776, 97378324586, 146067435150, 172740682440, 528895377996, 835698951528, 2096722640775, 3945699489100, 6232212795100|
|practice_0143|shared|48|1|167|175|0|167|3.40|32046448, 32046890, 256375672, 56049449, 56049526, 56049557, 91109003, 91109036, 1457739424, 273327839, 1793585940, 3097708369, 3279922920, 13028587854, 25874956568, 98027572, 1961713425, 64457011250, 416737537, 18427061792, 835698951528, 13459978560, 29953137475, 14950551550, 96870263776, 97378324586, 146067435150, 172740682440, 528895377996, 3945699489100, 2096722640775, 6232212795100|
|practice_0143|shared|48|2|151|157|0|151|20.43|56049557, 32046890, 256375672, 91109003, 91109036, 3279922920, 56049526, 273327839, 56049449, 1457739424, 1961713425, 32046448, 835698951528, 13028587854, 13459978560, 1793585940, 3097708369, 64457011250, 14950551550, 96870263776, 97378324586, 172740682440, 98027572, 25874956568, 6232212795100, 416737537, 29953137475, 146067435150, 528895377996, 18427061792, 2096722640775, 3945699489100|
|practice_0143|shared|128|0|150|153|0|150|5.21|32046448, 32046890, 56049449, 56049526, 56049557, 91109003, 91109036, 98027572, 256375672, 273327839, 416737537, 1457739424, 1793585940, 1961713425, 3097708369, 3279922920, 13028587854, 13459978560, 14950551550, 18427061792, 25874956568, 29953137475, 64457011250, 96870263776, 97378324586, 146067435150, 172740682440, 528895377996, 835698951528, 2096722640775, 3945699489100, 6232212795100|
|practice_0143|shared|128|1|155|157|0|155|13.33|32046448, 32046890, 256375672, 56049449, 56049526, 56049557, 91109003, 91109036, 1457739424, 273327839, 1793585940, 3097708369, 3279922920, 25874956568, 13028587854, 98027572, 1961713425, 416737537, 13459978560, 14950551550, 18427061792, 29953137475, 64457011250, 96870263776, 97378324586, 6232212795100, 835698951528, 146067435150, 172740682440, 528895377996, 2096722640775, 3945699489100|
|practice_0143|shared|128|2|145|148|0|145|58.05|56049557, 32046890, 256375672, 91109003, 91109036, 32046448, 56049526, 1457739424, 56049449, 1793585940, 3279922920, 25874956568, 96870263776, 1961713425, 13028587854, 64457011250, 273327839, 416737537, 3097708369, 98027572, 97378324586, 6232212795100, 13459978560, 18427061792, 172740682440, 835698951528, 14950551550, 146067435150, 29953137475, 2096722640775, 528895377996, 3945699489100|
|practice_0144|multiplicative|0|0|116|0|0|116|2.52|9661, 12065, 12929, 21827, 23158, 25595, 30011, 82643, 120993, 121475, 222108, 458275, 462530, 523848, 922473, 1443704, 1579344, 1582603, 2790053, 4188912, 4508807, 6407331, 7101815, 10626000, 28784450, 31571496, 45366360, 77741740, 189722462, 1330025346, 3596716112|
|practice_0144|multiplicative|0|1|119|0|0|119|3.50|9661, 12065, 12929, 21827, 523848, 23158, 25595, 30011, 82643, 120993, 121475, 1579344, 222108, 458275, 462530, 1443704, 1582603, 2790053, 4188912, 922473, 4508807, 6407331, 7101815, 10626000, 28784450, 31571496, 45366360, 77741740, 189722462, 3596716112, 1330025346|
|practice_0144|multiplicative|0|2|111|0|0|111|21.06|12065, 30011, 23158, 21827, 25595, 12929, 121475, 1579344, 2790053, 9661, 458275, 922473, 462530, 523848, 120993, 6407331, 1582603, 10626000, 82643, 4188912, 222108, 1443704, 4508807, 7101815, 28784450, 1330025346, 31571496, 45366360, 77741740, 189722462, 3596716112|
|practice_0144|multiplicative|48|0|125|134|0|125|1.08|9661, 12065, 12929, 21827, 23158, 25595, 30011, 82643, 120993, 121475, 222108, 458275, 462530, 523848, 922473, 1443704, 1579344, 1582603, 2790053, 4188912, 4508807, 6407331, 7101815, 10626000, 28784450, 31571496, 45366360, 77741740, 189722462, 1330025346, 3596716112|
|practice_0144|multiplicative|48|1|118|125|0|118|1.75|9661, 12065, 12929, 21827, 523848, 23158, 25595, 2790053, 30011, 82643, 120993, 222108, 121475, 1579344, 458275, 462530, 1443704, 1582603, 4188912, 6407331, 7101815, 922473, 4508807, 10626000, 28784450, 31571496, 45366360, 77741740, 189722462, 1330025346, 3596716112|
|practice_0144|multiplicative|48|2|114|121|0|114|9.45|12065, 30011, 23158, 25595, 121475, 21827, 1579344, 222108, 9661, 12929, 523848, 120993, 1582603, 462530, 458275, 10626000, 6407331, 1443704, 82643, 922473, 2790053, 4508807, 31571496, 7101815, 4188912, 45366360, 189722462, 3596716112, 28784450, 1330025346, 77741740|
|practice_0144|multiplicative|128|0|114|116|0|114|3.78|9661, 12065, 12929, 21827, 23158, 25595, 30011, 82643, 120993, 121475, 222108, 458275, 462530, 523848, 922473, 1443704, 1579344, 1582603, 2790053, 4188912, 4508807, 6407331, 7101815, 10626000, 28784450, 31571496, 45366360, 77741740, 189722462, 1330025346, 3596716112|
|practice_0144|multiplicative|128|1|115|117|0|115|4.24|9661, 12065, 12929, 21827, 523848, 23158, 25595, 2790053, 30011, 82643, 120993, 121475, 1579344, 222108, 458275, 462530, 1443704, 922473, 1582603, 4188912, 4508807, 6407331, 7101815, 10626000, 28784450, 31571496, 45366360, 77741740, 189722462, 1330025346, 3596716112|
|practice_0144|multiplicative|128|2|111|114|0|111|26.48|12065, 30011, 23158, 25595, 121475, 21827, 1579344, 222108, 9661, 12929, 523848, 1582603, 458275, 462530, 82643, 120993, 2790053, 7101815, 922473, 6407331, 1443704, 10626000, 28784450, 4188912, 4508807, 31571496, 45366360, 77741740, 189722462, 1330025346, 3596716112|
|practice_0145|shared|0|0|207|0|0|207|4.20|3858987186, 27655787728, 36057642596, 38918608619, 38918615226, 70438880709, 258410572271, 278914002113, 301044255588, 301044257598, 495991876052, 535346040210, 602088511235, 700535071746, 957028164930, 991983742296, 1673483988995, 7809591948931, 8568037346657, 15320168457063, 27043232859000, 158630224574736, 214615096153536, 334696797809981, 554422422955776, 796594904910048, 984543866154230, 984543866158200, 2217689324167072, 4435378650030467, 254292541454269440, 548938449400166400|
|practice_0145|shared|0|1|189|0|0|189|12.85|3858987186, 27655787728, 38918608619, 38918615226, 700535071746, 36057642596, 70438880709, 258410572271, 254292541454269440, 27043232859000, 214615096153536, 4435378650030467, 301044255588, 301044257598, 602088511235, 535346040210, 957028164930, 1673483988995, 7809591948931, 334696797809981, 548938449400166400, 278914002113, 554422422955776, 796594904910048, 495991876052, 991983742296, 984543866154230, 984543866158200, 2217689324167072, 8568037346657, 15320168457063, 158630224574736|
|practice_0145|shared|0|2|180|0|0|180|77.57|36057642596, 38918608619, 214615096153536, 495991876052, 38918615226, 984543866158200, 27043232859000, 4435378650030467, 984543866154230, 3858987186, 991983742296, 258410572271, 254292541454269440, 278914002113, 548938449400166400, 7809591948931, 301044257598, 15320168457063, 700535071746, 301044255588, 957028164930, 1673483988995, 602088511235, 334696797809981, 535346040210, 554422422955776, 796594904910048, 70438880709, 27655787728, 2217689324167072, 8568037346657, 158630224574736|
|practice_0145|shared|48|0|234|343|0|234|1.77|3858987186, 27655787728, 36057642596, 38918608619, 38918615226, 70438880709, 258410572271, 278914002113, 301044255588, 301044257598, 495991876052, 535346040210, 602088511235, 700535071746, 957028164930, 991983742296, 1673483988995, 7809591948931, 8568037346657, 15320168457063, 27043232859000, 158630224574736, 214615096153536, 334696797809981, 554422422955776, 796594904910048, 984543866154230, 984543866158200, 2217689324167072, 4435378650030467, 254292541454269440, 548938449400166400|
|practice_0145|shared|48|1|191|201|0|191|5.13|3858987186, 27655787728, 36057642596, 38918608619, 38918615226, 700535071746, 70438880709, 957028164930, 214615096153536, 991983742296, 495991876052, 258410572271, 984543866154230, 984543866158200, 27043232859000, 254292541454269440, 2217689324167072, 535346040210, 796594904910048, 4435378650030467, 554422422955776, 301044255588, 301044257598, 602088511235, 278914002113, 7809591948931, 548938449400166400, 1673483988995, 8568037346657, 334696797809981, 15320168457063, 158630224574736|
|practice_0145|shared|48|2|200|214|0|200|34.94|36057642596, 27655787728, 301044255588, 301044257598, 602088511235, 27043232859000, 214615096153536, 38918615226, 2217689324167072, 535346040210, 278914002113, 70438880709, 4435378650030467, 1673483988995, 38918608619, 7809591948931, 700535071746, 495991876052, 3858987186, 158630224574736, 258410572271, 957028164930, 991983742296, 254292541454269440, 984543866158200, 15320168457063, 548938449400166400, 984543866154230, 334696797809981, 796594904910048, 8568037346657, 554422422955776|
|practice_0145|shared|128|0|189|195|0|189|6.83|3858987186, 27655787728, 36057642596, 38918608619, 38918615226, 70438880709, 258410572271, 278914002113, 301044255588, 301044257598, 495991876052, 535346040210, 602088511235, 700535071746, 957028164930, 991983742296, 1673483988995, 7809591948931, 8568037346657, 15320168457063, 27043232859000, 158630224574736, 214615096153536, 334696797809981, 554422422955776, 796594904910048, 984543866154230, 984543866158200, 2217689324167072, 4435378650030467, 254292541454269440, 548938449400166400|
|practice_0145|shared|128|1|180|184|0|180|24.42|3858987186, 27655787728, 36057642596, 38918608619, 38918615226, 700535071746, 70438880709, 278914002113, 7809591948931, 1673483988995, 334696797809981, 535346040210, 548938449400166400, 258410572271, 254292541454269440, 301044255588, 301044257598, 602088511235, 27043232859000, 554422422955776, 214615096153536, 796594904910048, 991983742296, 495991876052, 2217689324167072, 984543866154230, 984543866158200, 4435378650030467, 957028164930, 15320168457063, 8568037346657, 158630224574736|
|practice_0145|shared|128|2|174|177|24|198|137.40|36057642596, 27655787728, 301044255588, 27043232859000, 38918608619, 301044257598, 214615096153536, 258410572271, 38918615226, 254292541454269440, 278914002113, 602088511235, 7809591948931, 548938449400166400, 334696797809981, 700535071746, 495991876052, 554422422955776, 984543866154230, 984543866158200, 3858987186, 4435378650030467, 1673483988995, 991983742296, 15320168457063, 70438880709, 957028164930, 2217689324167072, 535346040210, 796594904910048, 8568037346657, 158630224574736|
|practice_0146|mixed|0|0|47|0|0|47|0.16|47459, 449767, 573558, 598688, 636179, 971257, 3812386, 122493650|
|practice_0146|mixed|0|1|47|0|0|47|0.17|47459, 449767, 598688, 573558, 636179, 971257, 3812386, 122493650|
|practice_0146|mixed|0|2|43|0|0|43|0.60|47459, 598688, 449767, 971257, 636179, 573558, 3812386, 122493650|
|practice_0146|mixed|48|0|47|52|0|47|0.15|47459, 449767, 573558, 598688, 636179, 971257, 3812386, 122493650|
|practice_0146|mixed|48|1|47|52|0|47|0.18|47459, 449767, 598688, 573558, 636179, 971257, 3812386, 122493650|
|practice_0146|mixed|48|2|45|51|0|45|0.59|47459, 598688, 449767, 971257, 636179, 573558, 3812386, 122493650|
|practice_0146|mixed|128|0|47|51|0|47|0.14|47459, 449767, 573558, 598688, 636179, 971257, 3812386, 122493650|
|practice_0146|mixed|128|1|47|51|0|47|0.17|47459, 449767, 598688, 573558, 636179, 971257, 3812386, 122493650|
|practice_0146|mixed|128|2|45|51|0|45|0.74|47459, 598688, 449767, 971257, 636179, 573558, 3812386, 122493650|
|practice_0147|shared|0|0|188|0|0|188|3.21|68830934658, 95607113627, 95607113728, 147817755242, 240892487872, 372442858870, 423163950020, 423163963026, 509787681172, 623268686883, 785196507375, 967231931088, 1682563958070, 1906905818714, 1996097109983, 3725753717829, 5164226289290, 26895787553538, 55887057194441, 56341259098612, 101304457723803, 212085498147233, 275827930898810, 383250645077752, 392696157746186, 551655861795285, 626310467834112, 28686104813306642|
|practice_0147|shared|0|1|175|0|0|175|10.46|95607113728, 95607113627, 68830934658, 1996097109983, 383250645077752, 372442858870, 240892487872, 55887057194441, 147817755242, 423163950020, 423163963026, 967231931088, 101304457723803, 392696157746186, 551655861795285, 56341259098612, 275827930898810, 28686104813306642, 509787681172, 623268686883, 785196507375, 1682563958070, 1906905818714, 3725753717829, 626310467834112, 26895787553538, 5164226289290, 212085498147233|
|practice_0147|shared|0|2|175|0|0|175|66.30|68830934658, 423163950020, 95607113728, 1996097109983, 423163963026, 147817755242, 240892487872, 95607113627, 56341259098612, 785196507375, 1682563958070, 967231931088, 1906905818714, 372442858870, 383250645077752, 392696157746186, 55887057194441, 275827930898810, 623268686883, 551655861795285, 626310467834112, 26895787553538, 3725753717829, 28686104813306642, 101304457723803, 509787681172, 212085498147233, 5164226289290|
|practice_0147|shared|48|0|207|250|0|207|1.49|68830934658, 95607113627, 95607113728, 147817755242, 240892487872, 372442858870, 423163950020, 423163963026, 509787681172, 623268686883, 785196507375, 967231931088, 1682563958070, 1906905818714, 1996097109983, 3725753717829, 5164226289290, 26895787553538, 55887057194441, 56341259098612, 101304457723803, 212085498147233, 275827930898810, 383250645077752, 392696157746186, 551655861795285, 626310467834112, 28686104813306642|
|practice_0147|shared|48|1|186|197|0|186|4.05|95607113728, 95607113627, 68830934658, 1996097109983, 383250645077752, 240892487872, 55887057194441, 147817755242, 423163950020, 423163963026, 967231931088, 1906905818714, 392696157746186, 509787681172, 1682563958070, 785196507375, 56341259098612, 372442858870, 623268686883, 3725753717829, 26895787553538, 626310467834112, 101304457723803, 5164226289290, 275827930898810, 551655861795285, 28686104813306642, 212085498147233|
|practice_0147|shared|48|2|196|211|0|196|27.52|68830934658, 147817755242, 423163963026, 1996097109983, 392696157746186, 423163950020, 240892487872, 967231931088, 95607113627, 509787681172, 95607113728, 623268686883, 1906905818714, 372442858870, 5164226289290, 785196507375, 56341259098612, 26895787553538, 55887057194441, 1682563958070, 551655861795285, 275827930898810, 212085498147233, 383250645077752, 626310467834112, 28686104813306642, 101304457723803, 3725753717829|
|practice_0147|shared|128|0|179|182|12|191|6.12|68830934658, 95607113627, 95607113728, 147817755242, 240892487872, 372442858870, 423163950020, 423163963026, 509787681172, 623268686883, 785196507375, 967231931088, 1682563958070, 1906905818714, 1996097109983, 3725753717829, 5164226289290, 26895787553538, 55887057194441, 56341259098612, 101304457723803, 212085498147233, 275827930898810, 383250645077752, 392696157746186, 551655861795285, 626310467834112, 28686104813306642|
|practice_0147|shared|128|1|170|173|16|186|18.13|95607113728, 95607113627, 68830934658, 1996097109983, 383250645077752, 240892487872, 55887057194441, 785196507375, 1682563958070, 1906905818714, 147817755242, 372442858870, 101304457723803, 423163950020, 423163963026, 967231931088, 509787681172, 623268686883, 5164226289290, 26895787553538, 626310467834112, 392696157746186, 3725753717829, 551655861795285, 275827930898810, 56341259098612, 28686104813306642, 212085498147233|
|practice_0147|shared|128|2|168|173|0|168|113.43|68830934658, 95607113627, 95607113728, 423163950020, 423163963026, 1996097109983, 240892487872, 56341259098612, 392696157746186, 147817755242, 967231931088, 55887057194441, 1906905818714, 372442858870, 1682563958070, 785196507375, 383250645077752, 551655861795285, 28686104813306642, 509787681172, 101304457723803, 275827930898810, 626310467834112, 623268686883, 3725753717829, 26895787553538, 5164226289290, 212085498147233|
|practice_0148|shared|0|0|176|0|0|176|4.41|2999838, 5941142, 6212670, 6895082, 6895274, 9792268, 12297350, 12866919, 13647310, 13650683, 13650808, 20289538, 20291293, 22519854, 78381987, 157013207, 819093413, 2164767748, 2374124120, 4329531334, 5578623059, 8684177712, 45250326764, 69580138350, 315616479926, 623441904120, 1528326930944, 19313123180736, 125222037998387, 245965650442225, 20099952556578048|
|practice_0148|shared|0|1|168|0|0|168|11.04|2999838, 6212670, 12866919, 6895274, 6895082, 20289538, 20291293, 22519854, 5941142, 13647310, 13650683, 13650808, 9792268, 12297350, 78381987, 819093413, 4329531334, 2164767748, 157013207, 2374124120, 5578623059, 8684177712, 45250326764, 315616479926, 69580138350, 623441904120, 1528326930944, 19313123180736, 125222037998387, 245965650442225, 20099952556578048|
|practice_0148|shared|0|2|164|0|0|164|62.55|6895082, 6212670, 13650808, 6895274, 2999838, 13647310, 13650683, 9792268, 20289538, 20291293, 5941142, 12297350, 2164767748, 12866919, 22519854, 157013207, 819093413, 2374124120, 1528326930944, 45250326764, 4329531334, 78381987, 623441904120, 315616479926, 8684177712, 69580138350, 5578623059, 245965650442225, 19313123180736, 125222037998387, 20099952556578048|
|practice_0148|shared|48|0|179|230|0|179|1.52|2999838, 5941142, 6212670, 6895082, 6895274, 9792268, 12297350, 12866919, 13647310, 13650683, 13650808, 20289538, 20291293, 22519854, 78381987, 157013207, 819093413, 2164767748, 2374124120, 4329531334, 5578623059, 8684177712, 45250326764, 69580138350, 315616479926, 623441904120, 1528326930944, 19313123180736, 125222037998387, 245965650442225, 20099952556578048|
|practice_0148|shared|48|1|175|184|0|175|3.84|2999838, 13650808, 13650683, 6212670, 5941142, 6895082, 6895274, 9792268, 12866919, 12297350, 13647310, 20289538, 20291293, 22519854, 819093413, 78381987, 157013207, 2164767748, 2374124120, 4329531334, 8684177712, 5578623059, 623441904120, 45250326764, 69580138350, 1528326930944, 125222037998387, 315616479926, 19313123180736, 20099952556578048, 245965650442225|
|practice_0148|shared|48|2|176|193|0|176|22.38|6895082, 6212670, 13650808, 13647310, 9792268, 6895274, 13650683, 12866919, 78381987, 20291293, 12297350, 2164767748, 5941142, 2999838, 22519854, 8684177712, 4329531334, 819093413, 623441904120, 45250326764, 5578623059, 125222037998387, 20289538, 315616479926, 157013207, 69580138350, 2374124120, 1528326930944, 245965650442225, 20099952556578048, 19313123180736|
|practice_0148|shared|128|0|163|166|21|184|6.02|2999838, 5941142, 6212670, 6895082, 6895274, 9792268, 12297350, 12866919, 13647310, 13650683, 13650808, 20289538, 20291293, 22519854, 78381987, 157013207, 819093413, 2164767748, 2374124120, 4329531334, 5578623059, 8684177712, 45250326764, 69580138350, 315616479926, 623441904120, 1528326930944, 19313123180736, 125222037998387, 245965650442225, 20099952556578048|
|practice_0148|shared|128|1|160|163|41|201|14.00|2999838, 13650808, 13650683, 6212670, 5941142, 6895082, 6895274, 9792268, 12866919, 13647310, 12297350, 20289538, 20291293, 22519854, 157013207, 819093413, 2164767748, 2374124120, 4329531334, 8684177712, 78381987, 5578623059, 45250326764, 623441904120, 69580138350, 315616479926, 1528326930944, 19313123180736, 125222037998387, 245965650442225, 20099952556578048|
|practice_0148|shared|128|2|156|162|28|184|88.90|6895082, 6212670, 13650808, 2999838, 9792268, 6895274, 13650683, 13647310, 20289538, 5941142, 78381987, 20291293, 22519854, 12866919, 157013207, 819093413, 12297350, 2164767748, 4329531334, 2374124120, 8684177712, 5578623059, 45250326764, 69580138350, 623441904120, 1528326930944, 315616479926, 19313123180736, 245965650442225, 125222037998387, 20099952556578048|
|practice_0149|shared|0|0|203|0|0|203|4.14|24438129715, 116316042358, 116316042646, 415269398474, 415269403396, 415269404730, 447342532566, 447342532596, 516512511614, 516512514551, 553619329450, 553619331123, 562077012086, 1789370130377, 15343363301124, 35288661072954, 66197113685610, 78509902929648, 119355837419391, 119805923774022, 157536316883765, 157536316891695, 168300276293550, 272878944833540, 1170890409071787, 2771163106529636, 3277231393758000, 5724387789277748, 18919853401006590, 43933510125691983, 717837173925810305|
|practice_0149|shared|0|1|194|0|0|194|13.71|24438129715, 562077012086, 116316042358, 116316042646, 415269398474, 415269404730, 415269403396, 447342532566, 447342532596, 1789370130377, 516512511614, 516512514551, 157536316891695, 157536316883765, 272878944833540, 553619331123, 553619329450, 78509902929648, 168300276293550, 18919853401006590, 43933510125691983, 15343363301124, 35288661072954, 66197113685610, 119355837419391, 119805923774022, 3277231393758000, 1170890409071787, 2771163106529636, 5724387789277748, 717837173925810305|
|practice_0149|shared|0|2|193|0|10|203|81.64|516512514551, 24438129715, 157536316891695, 116316042358, 415269398474, 157536316883765, 78509902929648, 447342532596, 447342532566, 116316042646, 415269403396, 1789370130377, 553619329450, 415269404730, 516512511614, 272878944833540, 553619331123, 562077012086, 66197113685610, 168300276293550, 18919853401006590, 3277231393758000, 2771163106529636, 15343363301124, 43933510125691983, 119805923774022, 35288661072954, 119355837419391, 1170890409071787, 717837173925810305, 5724387789277748|
|practice_0149|shared|48|0|213|311|0|213|1.50|24438129715, 116316042358, 116316042646, 415269398474, 415269403396, 415269404730, 447342532566, 447342532596, 516512511614, 516512514551, 553619329450, 553619331123, 562077012086, 1789370130377, 15343363301124, 35288661072954, 66197113685610, 78509902929648, 119355837419391, 119805923774022, 157536316883765, 157536316891695, 168300276293550, 272878944833540, 1170890409071787, 2771163106529636, 3277231393758000, 5724387789277748, 18919853401006590, 43933510125691983, 717837173925810305|
|practice_0149|shared|48|1|197|212|0|197|4.46|24438129715, 562077012086, 116316042358, 116316042646, 415269398474, 415269404730, 415269403396, 447342532566, 447342532596, 1789370130377, 272878944833540, 516512514551, 157536316891695, 157536316883765, 516512511614, 78509902929648, 553619329450, 553619331123, 168300276293550, 15343363301124, 66197113685610, 119805923774022, 2771163106529636, 35288661072954, 119355837419391, 3277231393758000, 43933510125691983, 5724387789277748, 1170890409071787, 18919853401006590, 717837173925810305|
|practice_0149|shared|48|2|215|227|0|215|31.83|516512514551, 24438129715, 516512511614, 157536316891695, 78509902929648, 157536316883765, 447342532566, 1789370130377, 447342532596, 415269403396, 553619331123, 553619329450, 116316042646, 562077012086, 35288661072954, 415269404730, 415269398474, 5724387789277748, 3277231393758000, 116316042358, 119805923774022, 168300276293550, 2771163106529636, 66197113685610, 43933510125691983, 18919853401006590, 119355837419391, 15343363301124, 272878944833540, 1170890409071787, 717837173925810305|
|practice_0149|shared|128|0|187|194|15|202|5.63|24438129715, 116316042358, 116316042646, 415269398474, 415269403396, 415269404730, 447342532566, 447342532596, 516512511614, 516512514551, 553619329450, 553619331123, 562077012086, 1789370130377, 15343363301124, 35288661072954, 66197113685610, 78509902929648, 119355837419391, 119805923774022, 157536316883765, 157536316891695, 168300276293550, 272878944833540, 1170890409071787, 2771163106529636, 3277231393758000, 5724387789277748, 18919853401006590, 43933510125691983, 717837173925810305|
|practice_0149|shared|128|1|181|186|11|192|20.92|24438129715, 562077012086, 116316042358, 116316042646, 415269398474, 415269404730, 415269403396, 447342532566, 447342532596, 1789370130377, 516512511614, 516512514551, 157536316891695, 157536316883765, 78509902929648, 272878944833540, 553619331123, 553619329450, 43933510125691983, 119805923774022, 119355837419391, 168300276293550, 15343363301124, 35288661072954, 66197113685610, 2771163106529636, 1170890409071787, 5724387789277748, 3277231393758000, 18919853401006590, 717837173925810305|
|practice_0149|shared|128|2|172|175|0|172|145.68|516512514551, 24438129715, 157536316891695, 116316042358, 415269398474, 157536316883765, 116316042646, 447342532596, 447342532566, 1789370130377, 415269403396, 415269404730, 553619331123, 78509902929648, 553619329450, 516512511614, 272878944833540, 119805923774022, 562077012086, 3277231393758000, 119355837419391, 18919853401006590, 43933510125691983, 66197113685610, 35288661072954, 168300276293550, 2771163106529636, 15343363301124, 1170890409071787, 5724387789277748, 717837173925810305|
|practice_0150|independent|0|0|203|0|0|203|2.56|108622219485, 126507824386, 135432090117, 158652448913, 223508630087, 224857834348, 275421090138, 288666283189, 312484281709, 357508749528, 370191085635, 383714347738, 417788026014, 501975278663, 650067275167, 662966330461, 692401027915, 719010386830, 745853918562, 753542924136, 797079106014, 829727437948, 841566957294, 855229793039, 861979151057, 863719469726, 978629683832, 982758270228|
|practice_0150|independent|0|1|194|0|6|200|5.89|135432090117, 370191085635, 383714347738, 108622219485, 126507824386, 158652448913, 863719469726, 855229793039, 223508630087, 224857834348, 288666283189, 275421090138, 312484281709, 357508749528, 745853918562, 719010386830, 797079106014, 417788026014, 861979151057, 978629683832, 692401027915, 662966330461, 753542924136, 650067275167, 982758270228, 501975278663, 829727437948, 841566957294|
|practice_0150|independent|0|2|186|0|24|210|46.62|288666283189, 126507824386, 223508630087, 135432090117, 370191085635, 275421090138, 108622219485, 650067275167, 312484281709, 662966330461, 855229793039, 357508749528, 692401027915, 417788026014, 158652448913, 501975278663, 841566957294, 745853918562, 719010386830, 978629683832, 982758270228, 863719469726, 861979151057, 224857834348, 829727437948, 797079106014, 383714347738, 753542924136|
|practice_0150|independent|48|0|213|251|0|213|1.27|108622219485, 126507824386, 135432090117, 158652448913, 223508630087, 224857834348, 275421090138, 288666283189, 312484281709, 357508749528, 370191085635, 383714347738, 417788026014, 501975278663, 650067275167, 662966330461, 692401027915, 719010386830, 745853918562, 753542924136, 797079106014, 829727437948, 841566957294, 855229793039, 861979151057, 863719469726, 978629683832, 982758270228|
|practice_0150|independent|48|1|196|204|0|196|3.59|135432090117, 370191085635, 383714347738, 417788026014, 108622219485, 275421090138, 692401027915, 312484281709, 158652448913, 855229793039, 863719469726, 126507824386, 288666283189, 753542924136, 861979151057, 223508630087, 224857834348, 719010386830, 501975278663, 982758270228, 745853918562, 978629683832, 650067275167, 797079106014, 357508749528, 841566957294, 829727437948, 662966330461|
|practice_0150|independent|48|2|205|213|0|205|22.18|288666283189, 126507824386, 223508630087, 357508749528, 855229793039, 275421090138, 135432090117, 370191085635, 108622219485, 501975278663, 650067275167, 692401027915, 158652448913, 829727437948, 224857834348, 417788026014, 312484281709, 753542924136, 861979151057, 863719469726, 745853918562, 383714347738, 982758270228, 841566957294, 797079106014, 662966330461, 719010386830, 978629683832|
|practice_0150|independent|128|0|189|192|37|226|6.89|108622219485, 126507824386, 135432090117, 158652448913, 223508630087, 224857834348, 275421090138, 288666283189, 312484281709, 357508749528, 370191085635, 383714347738, 417788026014, 501975278663, 650067275167, 662966330461, 692401027915, 719010386830, 745853918562, 753542924136, 797079106014, 829727437948, 841566957294, 855229793039, 861979151057, 863719469726, 978629683832, 982758270228|
|practice_0150|independent|128|1|185|187|12|197|14.85|135432090117, 370191085635, 383714347738, 417788026014, 108622219485, 126507824386, 158652448913, 224857834348, 223508630087, 275421090138, 288666283189, 312484281709, 357508749528, 501975278663, 861979151057, 650067275167, 662966330461, 692401027915, 719010386830, 745853918562, 829727437948, 841566957294, 797079106014, 855229793039, 753542924136, 863719469726, 982758270228, 978629683832|
|practice_0150|independent|128|2|181|183|46|227|118.07|288666283189, 126507824386, 223508630087, 275421090138, 370191085635, 108622219485, 158652448913, 745853918562, 312484281709, 224857834348, 417788026014, 855229793039, 383714347738, 357508749528, 135432090117, 753542924136, 501975278663, 650067275167, 863719469726, 861979151057, 841566957294, 829727437948, 982758270228, 797079106014, 692401027915, 719010386830, 662966330461, 978629683832|
|practice_0151|independent|0|0|333|0|0|333|3.72|228073665184200373, 245480518894856863, 262135379333224052, 269542520826739969, 288592549037545500, 310984232959383766, 364520575349403430, 385430482562541331, 412040031216855672, 477833329761646733, 478763429188357236, 602161800163935813, 618128206223077243, 641057167840742524, 652391820204727827, 657695179530609392, 659485754871618591, 672798757774377801, 683593058653459510, 687125451345832020, 688257590652762727, 726907939125155284, 741547111817875034, 759049211465592702, 778998225535244045, 836213802859714921, 895348890120319049, 917119938494255921, 930904873208583016, 954122190137257434|
|practice_0151|independent|0|1|312|0|100|412|15.55|310984232959383766, 657695179530609392, 228073665184200373, 364520575349403430, 683593058653459510, 262135379333224052, 245480518894856863, 385430482562541331, 641057167840742524, 288592549037545500, 688257590652762727, 741547111817875034, 269542520826739969, 687125451345832020, 954122190137257434, 412040031216855672, 477833329761646733, 478763429188357236, 726907939125155284, 618128206223077243, 672798757774377801, 659485754871618591, 652391820204727827, 602161800163935813, 836213802859714921, 759049211465592702, 930904873208583016, 895348890120319049, 778998225535244045, 917119938494255921|
|practice_0151|independent|0|2|307|0|145|452|102.25|310984232959383766, 245480518894856863, 652391820204727827, 930904873208583016, 657695179530609392, 683593058653459510, 641057167840742524, 288592549037545500, 659485754871618591, 269542520826739969, 262135379333224052, 726907939125155284, 836213802859714921, 364520575349403430, 687125451345832020, 618128206223077243, 478763429188357236, 778998225535244045, 688257590652762727, 895348890120319049, 672798757774377801, 602161800163935813, 759049211465592702, 412040031216855672, 741547111817875034, 228073665184200373, 477833329761646733, 385430482562541331, 954122190137257434, 917119938494255921|
|practice_0151|independent|48|0|336|613|0|336|1.59|228073665184200373, 245480518894856863, 262135379333224052, 269542520826739969, 288592549037545500, 310984232959383766, 364520575349403430, 385430482562541331, 412040031216855672, 477833329761646733, 478763429188357236, 602161800163935813, 618128206223077243, 641057167840742524, 652391820204727827, 657695179530609392, 659485754871618591, 672798757774377801, 683593058653459510, 687125451345832020, 688257590652762727, 726907939125155284, 741547111817875034, 759049211465592702, 778998225535244045, 836213802859714921, 895348890120319049, 917119938494255921, 930904873208583016, 954122190137257434|
|practice_0151|independent|48|1|336|612|0|336|5.01|310984232959383766, 657695179530609392, 228073665184200373, 269542520826739969, 659485754871618591, 641057167840742524, 672798757774377801, 741547111817875034, 262135379333224052, 412040031216855672, 245480518894856863, 364520575349403430, 683593058653459510, 618128206223077243, 385430482562541331, 288592549037545500, 687125451345832020, 688257590652762727, 478763429188357236, 602161800163935813, 895348890120319049, 917119938494255921, 477833329761646733, 652391820204727827, 726907939125155284, 836213802859714921, 759049211465592702, 930904873208583016, 954122190137257434, 778998225535244045|
|practice_0151|independent|48|2|347|360|0|347|34.01|310984232959383766, 269542520826739969, 228073665184200373, 412040031216855672, 245480518894856863, 478763429188357236, 659485754871618591, 726907939125155284, 618128206223077243, 759049211465592702, 652391820204727827, 778998225535244045, 688257590652762727, 262135379333224052, 657695179530609392, 288592549037545500, 917119938494255921, 385430482562541331, 672798757774377801, 687125451345832020, 895348890120319049, 364520575349403430, 641057167840742524, 741547111817875034, 836213802859714921, 683593058653459510, 954122190137257434, 477833329761646733, 930904873208583016, 602161800163935813|
|practice_0151|independent|128|0|294|303|1|295|9.95|228073665184200373, 245480518894856863, 262135379333224052, 269542520826739969, 288592549037545500, 310984232959383766, 364520575349403430, 385430482562541331, 412040031216855672, 477833329761646733, 478763429188357236, 602161800163935813, 618128206223077243, 641057167840742524, 652391820204727827, 657695179530609392, 659485754871618591, 672798757774377801, 683593058653459510, 687125451345832020, 688257590652762727, 726907939125155284, 741547111817875034, 759049211465592702, 778998225535244045, 836213802859714921, 895348890120319049, 917119938494255921, 930904873208583016, 954122190137257434|
|practice_0151|independent|128|1|284|293|61|345|41.95|310984232959383766, 657695179530609392, 930904873208583016, 683593058653459510, 245480518894856863, 672798757774377801, 269542520826739969, 477833329761646733, 228073665184200373, 364520575349403430, 478763429188357236, 688257590652762727, 602161800163935813, 412040031216855672, 385430482562541331, 618128206223077243, 741547111817875034, 262135379333224052, 726907939125155284, 687125451345832020, 652391820204727827, 895348890120319049, 659485754871618591, 288592549037545500, 641057167840742524, 759049211465592702, 778998225535244045, 836213802859714921, 917119938494255921, 954122190137257434|
|practice_0151|independent|128|2|284|290|53|337|289.19|310984232959383766, 657695179530609392, 659485754871618591, 269542520826739969, 262135379333224052, 364520575349403430, 641057167840742524, 288592549037545500, 245480518894856863, 672798757774377801, 683593058653459510, 412040031216855672, 688257590652762727, 618128206223077243, 895348890120319049, 228073665184200373, 759049211465592702, 687125451345832020, 778998225535244045, 477833329761646733, 726907939125155284, 741547111817875034, 954122190137257434, 385430482562541331, 836213802859714921, 478763429188357236, 652391820204727827, 917119938494255921, 930904873208583016, 602161800163935813|
|practice_0152|shared|0|0|162|0|0|162|2.89|1956223080, 2608300656, 2934338396, 2988013521, 4097966247, 4097966892, 4097969416, 4783609140, 5577245567, 8964043185, 9204507254, 13806760792, 16097411604, 25755290552, 59503961740, 62186925396, 65567470015, 140848243104, 167257288323, 186336184125, 717951563292, 1268853549896, 1973024702460, 2899419772734, 2952158230164, 16021490638860, 52704982316700, 548131817651040|
|practice_0152|shared|0|1|158|0|0|158|6.34|1956223080, 2934338396, 140848243104, 167257288323, 2608300656, 2988013521, 8964043185, 2952158230164, 4097966892, 4097966247, 65567470015, 4097969416, 9204507254, 13806760792, 717951563292, 2899419772734, 4783609140, 62186925396, 5577245567, 59503961740, 16097411604, 25755290552, 1268853549896, 186336184125, 548131817651040, 52704982316700, 1973024702460, 16021490638860|
|practice_0152|shared|0|2|148|0|0|148|42.09|4097966247, 2934338396, 1956223080, 4097966892, 65567470015, 4097969416, 140848243104, 9204507254, 2608300656, 167257288323, 8964043185, 13806760792, 2988013521, 4783609140, 717951563292, 2952158230164, 16021490638860, 62186925396, 2899419772734, 5577245567, 548131817651040, 1973024702460, 52704982316700, 1268853549896, 186336184125, 59503961740, 25755290552, 16097411604|
|practice_0152|shared|48|0|178|199|0|178|1.25|1956223080, 2608300656, 2934338396, 2988013521, 4097966247, 4097966892, 4097969416, 4783609140, 5577245567, 8964043185, 9204507254, 13806760792, 16097411604, 25755290552, 59503961740, 62186925396, 65567470015, 140848243104, 167257288323, 186336184125, 717951563292, 1268853549896, 1973024702460, 2899419772734, 2952158230164, 16021490638860, 52704982316700, 548131817651040|
|practice_0152|shared|48|1|156|168|0|156|3.58|1956223080, 2934338396, 140848243104, 2608300656, 2988013521, 8964043185, 2952158230164, 167257288323, 25755290552, 65567470015, 4097966892, 4097969416, 4097966247, 16021490638860, 5577245567, 4783609140, 1973024702460, 13806760792, 717951563292, 62186925396, 16097411604, 59503961740, 52704982316700, 186336184125, 548131817651040, 9204507254, 2899419772734, 1268853549896|
|practice_0152|shared|48|2|167|179|0|167|20.50|4097966247, 2934338396, 1956223080, 4097966892, 140848243104, 4783609140, 2988013521, 4097969416, 9204507254, 717951563292, 65567470015, 5577245567, 8964043185, 2899419772734, 52704982316700, 13806760792, 59503961740, 2608300656, 548131817651040, 167257288323, 16097411604, 2952158230164, 62186925396, 1268853549896, 25755290552, 186336184125, 1973024702460, 16021490638860|
|practice_0152|shared|128|0|158|161|29|187|4.45|1956223080, 2608300656, 2934338396, 2988013521, 4097966247, 4097966892, 4097969416, 4783609140, 5577245567, 8964043185, 9204507254, 13806760792, 16097411604, 25755290552, 59503961740, 62186925396, 65567470015, 140848243104, 167257288323, 186336184125, 717951563292, 1268853549896, 1973024702460, 2899419772734, 2952158230164, 16021490638860, 52704982316700, 548131817651040|
|practice_0152|shared|128|1|151|155|0|151|12.34|1956223080, 2934338396, 140848243104, 2608300656, 167257288323, 2988013521, 8964043185, 2952158230164, 4097966892, 4097966247, 65567470015, 4097969416, 4783609140, 62186925396, 5577245567, 13806760792, 717951563292, 16097411604, 25755290552, 59503961740, 186336184125, 2899419772734, 9204507254, 16021490638860, 52704982316700, 1268853549896, 1973024702460, 548131817651040|
|practice_0152|shared|128|2|152|155|0|152|78.47|4097966247, 2934338396, 1956223080, 4097966892, 65567470015, 4097969416, 8964043185, 2988013521, 140848243104, 9204507254, 167257288323, 5577245567, 16097411604, 13806760792, 2608300656, 2952158230164, 2899419772734, 717951563292, 4783609140, 62186925396, 52704982316700, 25755290552, 186336184125, 59503961740, 1973024702460, 1268853549896, 16021490638860, 548131817651040|
|practice_0153|shared|0|0|149|0|0|149|3.37|2232275445, 3591239137, 3591244483, 5777520144, 28729954635, 38182285080, 38182289816, 38182289864, 61426868374, 61426875879, 143649779160, 161770562828, 165188801178, 305458280730, 531504170608, 653093491567, 653093492124, 653093492167, 1063008365784, 1374562262592, 1719952455757, 4259215950916, 4545588631370, 5877841434849, 10309216949700, 11301956381895, 22973650660284, 61918288407360, 101717607437136, 126764568514100, 1632292685330564|
|practice_0153|shared|0|1|148|0|0|148|8.75|2232275445, 3591239137, 3591244483, 143649779160, 1063008365784, 28729954635, 531504170608, 4259215950916, 5777520144, 161770562828, 38182285080, 38182289816, 38182289864, 305458280730, 1374562262592, 11301956381895, 101717607437136, 10309216949700, 126764568514100, 61426875879, 61426868374, 1719952455757, 61918288407360, 22973650660284, 4545588631370, 165188801178, 653093492124, 653093492167, 653093491567, 5877841434849, 1632292685330564|
|practice_0153|shared|0|2|150|0|0|150|49.53|38182285080, 3591239137, 3591244483, 38182289816, 38182289864, 305458280730, 11301956381895, 101717607437136, 28729954635, 143649779160, 1063008365784, 5777520144, 161770562828, 2232275445, 531504170608, 61426868374, 1719952455757, 61918288407360, 61426875879, 126764568514100, 653093491567, 653093492167, 653093492124, 1374562262592, 4545588631370, 5877841434849, 22973650660284, 4259215950916, 10309216949700, 165188801178, 1632292685330564|
|practice_0153|shared|48|0|185|221|0|185|1.45|2232275445, 3591239137, 3591244483, 5777520144, 28729954635, 38182285080, 38182289816, 38182289864, 61426868374, 61426875879, 143649779160, 161770562828, 165188801178, 305458280730, 531504170608, 653093491567, 653093492124, 653093492167, 1063008365784, 1374562262592, 1719952455757, 4259215950916, 4545588631370, 5877841434849, 10309216949700, 11301956381895, 22973650660284, 61918288407360, 101717607437136, 126764568514100, 1632292685330564|
|practice_0153|shared|48|1|146|172|0|146|4.35|2232275445, 3591239137, 3591244483, 143649779160, 1063008365784, 28729954635, 531504170608, 165188801178, 4259215950916, 5777520144, 161770562828, 126764568514100, 38182285080, 38182289816, 38182289864, 305458280730, 1374562262592, 101717607437136, 11301956381895, 10309216949700, 61426875879, 61426868374, 4545588631370, 1719952455757, 61918288407360, 22973650660284, 5877841434849, 653093491567, 653093492124, 653093492167, 1632292685330564|
|practice_0153|shared|48|2|160|171|0|160|25.22|38182285080, 3591239137, 3591244483, 38182289864, 1374562262592, 101717607437136, 38182289816, 531504170608, 11301956381895, 10309216949700, 305458280730, 143649779160, 653093492124, 653093492167, 5777520144, 1063008365784, 161770562828, 653093491567, 28729954635, 61426868374, 61426875879, 1719952455757, 61918288407360, 126764568514100, 4259215950916, 165188801178, 4545588631370, 22973650660284, 2232275445, 5877841434849, 1632292685330564|
|practice_0153|shared|128|0|141|145|0|141|5.38|2232275445, 3591239137, 3591244483, 5777520144, 28729954635, 38182285080, 38182289816, 38182289864, 61426868374, 61426875879, 143649779160, 161770562828, 165188801178, 305458280730, 531504170608, 653093491567, 653093492124, 653093492167, 1063008365784, 1374562262592, 1719952455757, 4259215950916, 4545588631370, 5877841434849, 10309216949700, 11301956381895, 22973650660284, 61918288407360, 101717607437136, 126764568514100, 1632292685330564|
|practice_0153|shared|128|1|138|143|0|138|15.98|2232275445, 3591239137, 3591244483, 143649779160, 1063008365784, 28729954635, 531504170608, 165188801178, 4259215950916, 5777520144, 161770562828, 126764568514100, 38182285080, 38182289816, 38182289864, 305458280730, 1374562262592, 101717607437136, 11301956381895, 10309216949700, 61426868374, 61426875879, 1719952455757, 61918288407360, 4545588631370, 22973650660284, 653093491567, 653093492124, 653093492167, 5877841434849, 1632292685330564|
|practice_0153|shared|128|2|138|143|0|138|75.43|38182285080, 3591239137, 3591244483, 38182289816, 38182289864, 305458280730, 1063008365784, 531504170608, 143649779160, 653093492167, 653093492124, 653093491567, 11301956381895, 101717607437136, 28729954635, 5777520144, 161770562828, 126764568514100, 61426868374, 1719952455757, 61918288407360, 1374562262592, 61426875879, 5877841434849, 4259215950916, 10309216949700, 4545588631370, 22973650660284, 2232275445, 165188801178, 1632292685330564|
|practice_0154|independent|0|0|336|0|0|336|3.85|120379281037919196, 122573911577741216, 263667488946180639, 281137696426845581, 293574525589296472, 297983335322499488, 318965408582764562, 390487782324131955, 401664375044909210, 424812700016566932, 427610032107519393, 433104428305477902, 453765298331624591, 464287523124354793, 488936605037698230, 507964979146371262, 509197042059934047, 540240602879632139, 590154702998749219, 597545959448387011, 606546754528613285, 663975061408322262, 749109101734196728, 777811218416055790, 780368636561637725, 802349172776291258, 888095901008837352, 915982347960795289, 965418636640725562, 966650107460464790, 968667652799149400|
|practice_0154|independent|0|1|314|0|115|429|18.28|122573911577741216, 263667488946180639, 120379281037919196, 281137696426845581, 318965408582764562, 297983335322499488, 777811218416055790, 453765298331624591, 424812700016566932, 390487782324131955, 401664375044909210, 293574525589296472, 780368636561637725, 507964979146371262, 597545959448387011, 965418636640725562, 433104428305477902, 509197042059934047, 663975061408322262, 464287523124354793, 488936605037698230, 590154702998749219, 427610032107519393, 540240602879632139, 606546754528613285, 749109101734196728, 888095901008837352, 966650107460464790, 968667652799149400, 915982347960795289, 802349172776291258|
|practice_0154|independent|0|2|314|0|229|543|112.05|120379281037919196, 263667488946180639, 293574525589296472, 606546754528613285, 281137696426845581, 401664375044909210, 427610032107519393, 965418636640725562, 507964979146371262, 509197042059934047, 122573911577741216, 297983335322499488, 424812700016566932, 464287523124354793, 777811218416055790, 968667652799149400, 318965408582764562, 749109101734196728, 453765298331624591, 390487782324131955, 540240602879632139, 663975061408322262, 590154702998749219, 888095901008837352, 915982347960795289, 802349172776291258, 966650107460464790, 780368636561637725, 597545959448387011, 433104428305477902, 488936605037698230|
|practice_0154|independent|48|0|351|484|0|351|1.54|120379281037919196, 122573911577741216, 263667488946180639, 281137696426845581, 293574525589296472, 297983335322499488, 318965408582764562, 390487782324131955, 401664375044909210, 424812700016566932, 427610032107519393, 433104428305477902, 453765298331624591, 464287523124354793, 488936605037698230, 507964979146371262, 509197042059934047, 540240602879632139, 590154702998749219, 597545959448387011, 606546754528613285, 663975061408322262, 749109101734196728, 777811218416055790, 780368636561637725, 802349172776291258, 888095901008837352, 915982347960795289, 965418636640725562, 966650107460464790, 968667652799149400|
|practice_0154|independent|48|1|336|353|0|336|6.57|122573911577741216, 263667488946180639, 453765298331624591, 297983335322499488, 318965408582764562, 424812700016566932, 433104428305477902, 464287523124354793, 293574525589296472, 597545959448387011, 390487782324131955, 120379281037919196, 427610032107519393, 968667652799149400, 966650107460464790, 540240602879632139, 509197042059934047, 965418636640725562, 507964979146371262, 749109101734196728, 401664375044909210, 281137696426845581, 488936605037698230, 590154702998749219, 606546754528613285, 663975061408322262, 777811218416055790, 780368636561637725, 915982347960795289, 888095901008837352, 802349172776291258|
|practice_0154|independent|48|2|343|421|0|343|37.71|263667488946180639, 122573911577741216, 540240602879632139, 120379281037919196, 390487782324131955, 780368636561637725, 597545959448387011, 318965408582764562, 749109101734196728, 464287523124354793, 507964979146371262, 453765298331624591, 427610032107519393, 509197042059934047, 488936605037698230, 802349172776291258, 663975061408322262, 433104428305477902, 968667652799149400, 966650107460464790, 888095901008837352, 965418636640725562, 915982347960795289, 606546754528613285, 297983335322499488, 590154702998749219, 424812700016566932, 777811218416055790, 281137696426845581, 401664375044909210, 293574525589296472|
|practice_0154|independent|128|0|306|312|35|341|11.06|120379281037919196, 122573911577741216, 263667488946180639, 281137696426845581, 293574525589296472, 297983335322499488, 318965408582764562, 390487782324131955, 401664375044909210, 424812700016566932, 427610032107519393, 433104428305477902, 453765298331624591, 464287523124354793, 488936605037698230, 507964979146371262, 509197042059934047, 540240602879632139, 590154702998749219, 597545959448387011, 606546754528613285, 663975061408322262, 749109101734196728, 777811218416055790, 780368636561637725, 802349172776291258, 888095901008837352, 915982347960795289, 965418636640725562, 966650107460464790, 968667652799149400|
|practice_0154|independent|128|1|295|297|11|306|53.74|122573911577741216, 263667488946180639, 453765298331624591, 424812700016566932, 281137696426845581, 390487782324131955, 293574525589296472, 120379281037919196, 297983335322499488, 318965408582764562, 606546754528613285, 597545959448387011, 401664375044909210, 590154702998749219, 888095901008837352, 433104428305477902, 540240602879632139, 427610032107519393, 464287523124354793, 968667652799149400, 663975061408322262, 488936605037698230, 509197042059934047, 507964979146371262, 749109101734196728, 802349172776291258, 777811218416055790, 780368636561637725, 966650107460464790, 965418636640725562, 915982347960795289|
|practice_0154|independent|128|2|287|290|12|299|313.09|120379281037919196, 281137696426845581, 318965408582764562, 122573911577741216, 401664375044909210, 297983335322499488, 966650107460464790, 390487782324131955, 590154702998749219, 507964979146371262, 965418636640725562, 433104428305477902, 888095901008837352, 802349172776291258, 293574525589296472, 453765298331624591, 597545959448387011, 968667652799149400, 427610032107519393, 540240602879632139, 488936605037698230, 464287523124354793, 663975061408322262, 780368636561637725, 263667488946180639, 777811218416055790, 424812700016566932, 915982347960795289, 509197042059934047, 749109101734196728, 606546754528613285|
|practice_0155|multiplicative|0|0|160|0|0|160|3.40|1293348, 1626420, 2628274, 2708196, 3255470, 3793728, 3800520, 4211056, 4260326, 4380411, 6700244, 12613655, 23137521, 52028474, 69396332, 112139260, 156085342, 260143033, 416227072, 2312918784, 3707884796, 6594711372, 9015739392, 12049506715, 13742215996, 88977969264, 152938682160, 281962992896, 381881511262, 384978147840, 64802296863178656|
|practice_0155|multiplicative|0|1|159|0|0|159|9.39|3255470, 1626420, 1293348, 4211056, 2628274, 3800520, 2708196, 3793728, 4260326, 4380411, 12613655, 6700244, 23137521, 52028474, 156085342, 260143033, 416227072, 69396332, 112139260, 2312918784, 384978147840, 3707884796, 9015739392, 6594711372, 12049506715, 13742215996, 152938682160, 281962992896, 88977969264, 381881511262, 64802296863178656|
|practice_0155|multiplicative|0|2|155|0|0|155|60.08|3793728, 1626420, 4211056, 2628274, 3255470, 1293348, 3800520, 2708196, 52028474, 4380411, 12613655, 156085342, 260143033, 416227072, 2312918784, 4260326, 69396332, 9015739392, 6594711372, 112139260, 152938682160, 13742215996, 6700244, 3707884796, 23137521, 12049506715, 281962992896, 381881511262, 384978147840, 88977969264, 64802296863178656|
|practice_0155|multiplicative|48|0|168|211|0|168|1.36|1293348, 1626420, 2628274, 2708196, 3255470, 3793728, 3800520, 4211056, 4260326, 4380411, 6700244, 12613655, 23137521, 52028474, 69396332, 112139260, 156085342, 260143033, 416227072, 2312918784, 3707884796, 6594711372, 9015739392, 12049506715, 13742215996, 88977969264, 152938682160, 281962992896, 381881511262, 384978147840, 64802296863178656|
|practice_0155|multiplicative|48|1|162|178|0|162|3.33|3255470, 1626420, 1293348, 4211056, 2628274, 2708196, 3793728, 3800520, 4260326, 12613655, 4380411, 6700244, 23137521, 52028474, 69396332, 260143033, 112139260, 416227072, 156085342, 2312918784, 6594711372, 9015739392, 3707884796, 13742215996, 88977969264, 152938682160, 281962992896, 12049506715, 384978147840, 64802296863178656, 381881511262|
|practice_0155|multiplicative|48|2|163|173|0|163|20.00|3793728, 1626420, 4211056, 2708196, 1293348, 3255470, 2628274, 6700244, 4260326, 23137521, 416227072, 52028474, 12613655, 69396332, 3800520, 2312918784, 156085342, 3707884796, 112139260, 12049506715, 4380411, 260143033, 13742215996, 9015739392, 384978147840, 381881511262, 88977969264, 152938682160, 281962992896, 6594711372, 64802296863178656|
|practice_0155|multiplicative|128|0|155|163|57|212|4.96|1293348, 1626420, 2628274, 2708196, 3255470, 3793728, 3800520, 4211056, 4260326, 4380411, 6700244, 12613655, 23137521, 52028474, 69396332, 112139260, 156085342, 260143033, 416227072, 2312918784, 3707884796, 6594711372, 9015739392, 12049506715, 13742215996, 88977969264, 152938682160, 281962992896, 381881511262, 384978147840, 64802296863178656|
|practice_0155|multiplicative|128|1|152|156|2|154|12.05|3255470, 1626420, 1293348, 4211056, 2628274, 2708196, 3793728, 3800520, 4260326, 4380411, 12613655, 6700244, 23137521, 52028474, 156085342, 260143033, 416227072, 69396332, 112139260, 2312918784, 9015739392, 152938682160, 3707884796, 6594711372, 12049506715, 13742215996, 88977969264, 281962992896, 384978147840, 381881511262, 64802296863178656|
|practice_0155|multiplicative|128|2|151|154|28|179|69.53|3793728, 1626420, 4211056, 2628274, 3255470, 1293348, 3800520, 2708196, 52028474, 4380411, 12613655, 156085342, 260143033, 416227072, 69396332, 6700244, 23137521, 4260326, 9015739392, 2312918784, 112139260, 152938682160, 6594711372, 3707884796, 281962992896, 12049506715, 88977969264, 384978147840, 13742215996, 381881511262, 64802296863178656|
|practice_0156|multiplicative|0|0|111|0|0|111|1.91|35116, 36474, 37861, 42015, 48420, 60504, 131511, 139888, 193612, 301590, 310160, 484885, 489345, 489819, 524810, 542556, 1187424, 2448945, 2938986, 4255249, 6510990, 8415936, 11634650, 17659246, 44279424, 69797851, 150237864, 179002780|
|practice_0156|multiplicative|0|1|110|0|0|110|2.27|37861, 36474, 35116, 42015, 48420, 193612, 60504, 131511, 8415936, 139888, 301590, 310160, 484885, 489345, 489819, 524810, 542556, 1187424, 2448945, 2938986, 4255249, 6510990, 44279424, 11634650, 17659246, 69797851, 150237864, 179002780|
|practice_0156|multiplicative|0|2|109|0|0|109|15.41|48420, 131511, 42015, 37861, 36474, 193612, 8415936, 35116, 60504, 489819, 310160, 524810, 139888, 301590, 484885, 4255249, 489345, 542556, 1187424, 2448945, 2938986, 6510990, 11634650, 44279424, 17659246, 69797851, 150237864, 179002780|
|practice_0156|multiplicative|48|0|114|149|0|114|0.74|35116, 36474, 37861, 42015, 48420, 60504, 131511, 139888, 193612, 301590, 310160, 484885, 489345, 489819, 524810, 542556, 1187424, 2448945, 2938986, 4255249, 6510990, 8415936, 11634650, 17659246, 44279424, 69797851, 150237864, 179002780|
|practice_0156|multiplicative|48|1|110|115|0|110|1.08|37861, 36474, 35116, 42015, 48420, 60504, 193612, 131511, 139888, 484885, 301590, 489345, 310160, 524810, 489819, 542556, 2448945, 2938986, 1187424, 6510990, 150237864, 4255249, 8415936, 11634650, 17659246, 44279424, 179002780, 69797851|
|practice_0156|multiplicative|48|2|107|112|0|107|7.05|48420, 131511, 42015, 37861, 36474, 193612, 8415936, 139888, 310160, 489819, 524810, 35116, 489345, 60504, 301590, 1187424, 484885, 6510990, 2938986, 542556, 17659246, 44279424, 11634650, 2448945, 4255249, 150237864, 69797851, 179002780|
|practice_0156|multiplicative|128|0|108|110|0|108|3.25|35116, 36474, 37861, 42015, 48420, 60504, 131511, 139888, 193612, 301590, 310160, 484885, 489345, 489819, 524810, 542556, 1187424, 2448945, 2938986, 4255249, 6510990, 8415936, 11634650, 17659246, 44279424, 69797851, 150237864, 179002780|
|practice_0156|multiplicative|128|1|107|109|0|107|3.61|37861, 36474, 35116, 42015, 48420, 60504, 193612, 131511, 8415936, 139888, 301590, 310160, 484885, 489345, 489819, 524810, 542556, 1187424, 2448945, 2938986, 4255249, 6510990, 44279424, 11634650, 17659246, 69797851, 150237864, 179002780|
|practice_0156|multiplicative|128|2|106|108|0|106|23.04|48420, 131511, 42015, 37861, 36474, 193612, 8415936, 35116, 310160, 489819, 489345, 524810, 2448945, 1187424, 60504, 139888, 301590, 484885, 542556, 2938986, 44279424, 4255249, 6510990, 11634650, 179002780, 17659246, 69797851, 150237864|
|practice_0157|shared|0|0|146|0|0|146|3.09|5707606, 5708513, 5708915, 11522007, 11522136, 11522156, 20698593, 20699503, 23260434, 41786397, 41787069, 75064881, 334291792, 459649542, 585010601, 765836802, 837386676, 852639480, 2980559728, 3133990950, 4689513829, 6017252256, 6209488818, 6722524081, 13872308604, 27284444096, 97697581800, 110981387313, 138841880965, 249207328524, 423652722528, 30287169197376|
|practice_0157|shared|0|1|130|0|0|130|5.38|11522136, 11522007, 11522156, 5708513, 5707606, 5708915, 41786397, 41787069, 334291792, 6017252256, 459649542, 585010601, 852639480, 3133990950, 27284444096, 4689513829, 20698593, 20699503, 75064881, 23260434, 6722524081, 97697581800, 837386676, 6209488818, 765836802, 2980559728, 110981387313, 30287169197376, 13872308604, 423652722528, 249207328524, 138841880965|
|practice_0157|shared|0|2|128|0|0|128|33.73|11522136, 5707606, 11522156, 5708915, 11522007, 20698593, 5708513, 41786397, 334291792, 852639480, 20699503, 585010601, 27284444096, 4689513829, 6017252256, 41787069, 23260434, 6209488818, 459649542, 765836802, 3133990950, 13872308604, 30287169197376, 837386676, 6722524081, 2980559728, 97697581800, 423652722528, 75064881, 110981387313, 249207328524, 138841880965|
|practice_0157|shared|48|0|154|200|0|154|1.22|5707606, 5708513, 5708915, 11522007, 11522136, 11522156, 20698593, 20699503, 23260434, 41786397, 41787069, 75064881, 334291792, 459649542, 585010601, 765836802, 837386676, 852639480, 2980559728, 3133990950, 4689513829, 6017252256, 6209488818, 6722524081, 13872308604, 27284444096, 97697581800, 110981387313, 138841880965, 249207328524, 423652722528, 30287169197376|
|practice_0157|shared|48|1|144|151|0|144|2.87|11522136, 11522007, 11522156, 5708513, 5707606, 5708915, 41786397, 41787069, 334291792, 6017252256, 459649542, 585010601, 3133990950, 20698593, 20699503, 23260434, 75064881, 6722524081, 97697581800, 837386676, 765836802, 852639480, 2980559728, 4689513829, 27284444096, 6209488818, 13872308604, 110981387313, 249207328524, 423652722528, 30287169197376, 138841880965|
|practice_0157|shared|48|2|137|147|0|137|18.96|11522007, 11522136, 5708915, 4689513829, 11522156, 5708513, 5707606, 41786397, 23260434, 459649542, 20699503, 20698593, 334291792, 6017252256, 27284444096, 852639480, 3133990950, 837386676, 585010601, 765836802, 97697581800, 41787069, 6722524081, 30287169197376, 75064881, 6209488818, 2980559728, 423652722528, 13872308604, 110981387313, 249207328524, 138841880965|
|practice_0157|shared|128|0|132|136|0|132|5.05|5707606, 5708513, 5708915, 11522007, 11522136, 11522156, 20698593, 20699503, 23260434, 41786397, 41787069, 75064881, 334291792, 459649542, 585010601, 765836802, 837386676, 852639480, 2980559728, 3133990950, 4689513829, 6017252256, 6209488818, 6722524081, 13872308604, 27284444096, 97697581800, 110981387313, 138841880965, 249207328524, 423652722528, 30287169197376|
|practice_0157|shared|128|1|127|131|0|127|8.49|11522136, 11522007, 11522156, 5708513, 5707606, 5708915, 41786397, 41787069, 334291792, 6017252256, 459649542, 585010601, 3133990950, 852639480, 20698593, 20699503, 23260434, 75064881, 765836802, 837386676, 4689513829, 27284444096, 6722524081, 97697581800, 13872308604, 110981387313, 423652722528, 6209488818, 2980559728, 249207328524, 30287169197376, 138841880965|
|practice_0157|shared|128|2|130|134|0|130|47.76|11522007, 11522136, 5708915, 4689513829, 11522156, 5707606, 20698593, 41787069, 41786397, 334291792, 6017252256, 5708513, 23260434, 27284444096, 20699503, 852639480, 585010601, 3133990950, 459649542, 13872308604, 30287169197376, 6722524081, 97697581800, 75064881, 765836802, 837386676, 2980559728, 110981387313, 423652722528, 6209488818, 249207328524, 138841880965|
|practice_0158|shared|0|0|239|0|0|239|4.93|12590354512, 27227347342, 304513443669, 304513444191, 304513453600, 353275172491, 658527380611, 664735207384, 706550337540, 777109232863, 2037272956461, 2331327736224, 2436107551248, 2537410530657, 4872215206915, 6254279127756, 8584385307456, 9971561428476, 30229441048252, 38078353649497, 50422505045220, 159633097579902, 319266208101891, 807014669519800, 1048293365588161, 1056638895843166, 13215289145294016, 18130221771824050, 33138077723757864, 73906385916333563, 224361494423908980, 821602623036856320|
|practice_0158|shared|0|1|226|0|0|226|18.86|2331327736224, 777109232863, 38078353649497, 12590354512, 27227347342, 304513443669, 304513444191, 304513453600, 2436107551248, 4872215206915, 9971561428476, 664735207384, 658527380611, 706550337540, 353275172491, 2037272956461, 8584385307456, 50422505045220, 6254279127756, 2537410530657, 30229441048252, 13215289145294016, 159633097579902, 319266208101891, 1056638895843166, 1048293365588161, 807014669519800, 18130221771824050, 33138077723757864, 821602623036856320, 224361494423908980, 73906385916333563|
|practice_0158|shared|0|2|218|0|29|247|107.22|304513443669, 12590354512, 304513444191, 8584385307456, 2436107551248, 304513453600, 2331327736224, 777109232863, 4872215206915, 664735207384, 706550337540, 27227347342, 353275172491, 38078353649497, 6254279127756, 50422505045220, 2037272956461, 159633097579902, 319266208101891, 13215289145294016, 658527380611, 807014669519800, 9971561428476, 1056638895843166, 1048293365588161, 33138077723757864, 30229441048252, 821602623036856320, 2537410530657, 224361494423908980, 18130221771824050, 73906385916333563|
|practice_0158|shared|48|0|251|302|0|251|1.75|12590354512, 27227347342, 304513443669, 304513444191, 304513453600, 353275172491, 658527380611, 664735207384, 706550337540, 777109232863, 2037272956461, 2331327736224, 2436107551248, 2537410530657, 4872215206915, 6254279127756, 8584385307456, 9971561428476, 30229441048252, 38078353649497, 50422505045220, 159633097579902, 319266208101891, 807014669519800, 1048293365588161, 1056638895843166, 13215289145294016, 18130221771824050, 33138077723757864, 73906385916333563, 224361494423908980, 821602623036856320|
|practice_0158|shared|48|1|251|274|0|251|6.66|2331327736224, 777109232863, 38078353649497, 12590354512, 304513443669, 304513444191, 2436107551248, 304513453600, 4872215206915, 27227347342, 8584385307456, 664735207384, 658527380611, 353275172491, 706550337540, 2537410530657, 2037272956461, 6254279127756, 9971561428476, 50422505045220, 30229441048252, 1056638895843166, 807014669519800, 13215289145294016, 319266208101891, 1048293365588161, 33138077723757864, 18130221771824050, 159633097579902, 73906385916333563, 821602623036856320, 224361494423908980|
|practice_0158|shared|48|2|244|257|0|244|40.53|304513443669, 304513444191, 304513453600, 706550337540, 353275172491, 12590354512, 4872215206915, 2436107551248, 2331327736224, 8584385307456, 6254279127756, 777109232863, 27227347342, 50422505045220, 2537410530657, 658527380611, 9971561428476, 807014669519800, 1048293365588161, 13215289145294016, 2037272956461, 1056638895843166, 319266208101891, 73906385916333563, 821602623036856320, 664735207384, 159633097579902, 18130221771824050, 38078353649497, 224361494423908980, 30229441048252, 33138077723757864|
|practice_0158|shared|128|0|215|222|68|283|8.75|12590354512, 27227347342, 304513443669, 304513444191, 304513453600, 353275172491, 658527380611, 664735207384, 706550337540, 777109232863, 2037272956461, 2331327736224, 2436107551248, 2537410530657, 4872215206915, 6254279127756, 8584385307456, 9971561428476, 30229441048252, 38078353649497, 50422505045220, 159633097579902, 319266208101891, 807014669519800, 1048293365588161, 1056638895843166, 13215289145294016, 18130221771824050, 33138077723757864, 73906385916333563, 224361494423908980, 821602623036856320|
|practice_0158|shared|128|1|210|215|0|210|32.33|2331327736224, 777109232863, 38078353649497, 12590354512, 27227347342, 304513443669, 304513444191, 304513453600, 2436107551248, 4872215206915, 9971561428476, 353275172491, 706550337540, 2037272956461, 658527380611, 664735207384, 6254279127756, 2537410530657, 8584385307456, 50422505045220, 159633097579902, 319266208101891, 30229441048252, 807014669519800, 13215289145294016, 1048293365588161, 18130221771824050, 1056638895843166, 33138077723757864, 821602623036856320, 73906385916333563, 224361494423908980|
|practice_0158|shared|128|2|203|208|4|207|185.45|304513443669, 12590354512, 304513444191, 304513453600, 664735207384, 2436107551248, 353275172491, 706550337540, 777109232863, 2037272956461, 4872215206915, 27227347342, 38078353649497, 2331327736224, 2537410530657, 30229441048252, 319266208101891, 159633097579902, 50422505045220, 658527380611, 9971561428476, 13215289145294016, 6254279127756, 33138077723757864, 8584385307456, 1056638895843166, 18130221771824050, 1048293365588161, 807014669519800, 821602623036856320, 73906385916333563, 224361494423908980|
|practice_0159|shared|0|0|39|0|0|39|0.09|562697, 590825, 590998, 591049, 620846, 7683000, 28563436|
|practice_0159|shared|0|1|34|0|0|34|0.08|590825, 591049, 590998, 7683000, 562697, 620846, 28563436|
|practice_0159|shared|0|2|38|0|0|38|0.27|590998, 590825, 591049, 7683000, 562697, 620846, 28563436|
|practice_0159|shared|48|0|39|44|0|39|0.10|562697, 590825, 590998, 591049, 620846, 7683000, 28563436|
|practice_0159|shared|48|1|33|40|0|33|0.09|590825, 591049, 590998, 7683000, 562697, 620846, 28563436|
|practice_0159|shared|48|2|39|43|0|39|0.46|590998, 590825, 591049, 7683000, 562697, 620846, 28563436|
|practice_0159|shared|128|0|39|44|0|39|0.09|562697, 590825, 590998, 591049, 620846, 7683000, 28563436|
|practice_0159|shared|128|1|33|40|0|33|0.08|590825, 591049, 590998, 7683000, 562697, 620846, 28563436|
|practice_0159|shared|128|2|39|43|0|39|0.47|590998, 590825, 591049, 7683000, 562697, 620846, 28563436|
|practice_0160|mixed|0|0|187|0|0|187|3.15|15050316, 15051411, 18885089, 56653552, 80029880, 640247458, 2115120672, 5762230632, 16328955144, 16775449741, 83232152787, 134654546990, 238677436207, 324174132392, 325912213988, 366495656522, 415187029532, 428863453039, 466060717572, 487261080412, 631129862656, 634720324610, 720335604439, 758020236408, 862063407339, 909051596098, 1291356641836, 141921179990038|
|practice_0160|mixed|0|1|180|0|0|180|8.65|18885089, 56653552, 2115120672, 15050316, 15051411, 80029880, 640247458, 5762230632, 83232152787, 16328955144, 16775449741, 134654546990, 238677436207, 324174132392, 325912213988, 631129862656, 466060717572, 428863453039, 415187029532, 634720324610, 758020236408, 720335604439, 366495656522, 487261080412, 862063407339, 909051596098, 1291356641836, 141921179990038|
|practice_0160|mixed|0|2|178|0|0|178|51.08|15051411, 18885089, 80029880, 15050316, 56653552, 640247458, 2115120672, 16328955144, 83232152787, 5762230632, 720335604439, 16775449741, 487261080412, 134654546990, 366495656522, 415187029532, 466060717572, 324174132392, 631129862656, 325912213988, 909051596098, 1291356641836, 428863453039, 238677436207, 634720324610, 758020236408, 862063407339, 141921179990038|
|practice_0160|mixed|48|0|184|217|0|184|1.38|15050316, 15051411, 18885089, 56653552, 80029880, 640247458, 2115120672, 5762230632, 16328955144, 16775449741, 83232152787, 134654546990, 238677436207, 324174132392, 325912213988, 366495656522, 415187029532, 428863453039, 466060717572, 487261080412, 631129862656, 634720324610, 720335604439, 758020236408, 862063407339, 909051596098, 1291356641836, 141921179990038|
|practice_0160|mixed|48|1|182|190|0|182|3.88|18885089, 56653552, 80029880, 640247458, 2115120672, 5762230632, 83232152787, 15051411, 15050316, 16328955144, 16775449741, 415187029532, 366495656522, 325912213988, 428863453039, 134654546990, 324174132392, 487261080412, 238677436207, 634720324610, 631129862656, 909051596098, 862063407339, 466060717572, 720335604439, 758020236408, 1291356641836, 141921179990038|
|practice_0160|mixed|48|2|183|196|0|183|20.15|15051411, 18885089, 15050316, 56653552, 80029880, 16328955144, 2115120672, 640247458, 415187029532, 631129862656, 5762230632, 634720324610, 83232152787, 238677436207, 324174132392, 466060717572, 325912213988, 16775449741, 909051596098, 720335604439, 134654546990, 487261080412, 366495656522, 1291356641836, 428863453039, 862063407339, 141921179990038, 758020236408|
|practice_0160|mixed|128|0|171|175|11|182|5.51|15050316, 15051411, 18885089, 56653552, 80029880, 640247458, 2115120672, 5762230632, 16328955144, 16775449741, 83232152787, 134654546990, 238677436207, 324174132392, 325912213988, 366495656522, 415187029532, 428863453039, 466060717572, 487261080412, 631129862656, 634720324610, 720335604439, 758020236408, 862063407339, 909051596098, 1291356641836, 141921179990038|
|practice_0160|mixed|128|1|167|171|7|174|11.45|18885089, 56653552, 80029880, 640247458, 2115120672, 5762230632, 15050316, 15051411, 83232152787, 16328955144, 16775449741, 1291356641836, 415187029532, 134654546990, 238677436207, 428863453039, 325912213988, 466060717572, 324174132392, 366495656522, 631129862656, 487261080412, 634720324610, 720335604439, 758020236408, 862063407339, 909051596098, 141921179990038|
|practice_0160|mixed|128|2|163|166|0|163|81.46|15051411, 18885089, 80029880, 15050316, 640247458, 56653552, 83232152787, 2115120672, 5762230632, 720335604439, 631129862656, 16328955144, 16775449741, 634720324610, 324174132392, 238677436207, 466060717572, 428863453039, 325912213988, 366495656522, 1291356641836, 415187029532, 862063407339, 909051596098, 134654546990, 487261080412, 758020236408, 141921179990038|
|practice_0161|mixed|0|0|274|0|0|274|3.52|42418406101, 89078683230, 154946889483, 254158987381, 282121588389, 899119235789, 1913972713028, 2036084663652, 2302205126233, 236185822410240, 1685543374908150, 22120131174680124, 125810949519001281, 210135314550808495, 238245469826085238, 308950305198811200, 374694249240572089, 455179725236300684, 486184993838179026, 517789639582129336, 566561755614653902, 580430101104173558, 649794434516738717, 723861253440532319, 782467130834547122, 823061349451565856, 929733096795523467, 996600424723016161|
|practice_0161|mixed|0|1|259|0|53|312|14.23|42418406101, 254158987381, 2036084663652, 89078683230, 282121588389, 899119235789, 236185822410240, 154946889483, 1913972713028, 2302205126233, 1685543374908150, 22120131174680124, 455179725236300684, 238245469826085238, 308950305198811200, 823061349451565856, 125810949519001281, 517789639582129336, 580430101104173558, 566561755614653902, 782467130834547122, 486184993838179026, 649794434516738717, 210135314550808495, 929733096795523467, 374694249240572089, 723861253440532319, 996600424723016161|
|practice_0161|mixed|0|2|261|0|232|493|101.72|282121588389, 899119235789, 42418406101, 2036084663652, 154946889483, 254158987381, 2302205126233, 89078683230, 1685543374908150, 1913972713028, 22120131174680124, 236185822410240, 125810949519001281, 723861253440532319, 308950305198811200, 929733096795523467, 455179725236300684, 210135314550808495, 517789639582129336, 566561755614653902, 996600424723016161, 782467130834547122, 238245469826085238, 374694249240572089, 580430101104173558, 649794434516738717, 486184993838179026, 823061349451565856|
|practice_0161|mixed|48|0|274|356|0|274|1.60|42418406101, 89078683230, 154946889483, 254158987381, 282121588389, 899119235789, 1913972713028, 2036084663652, 2302205126233, 236185822410240, 1685543374908150, 22120131174680124, 125810949519001281, 210135314550808495, 238245469826085238, 308950305198811200, 374694249240572089, 455179725236300684, 486184993838179026, 517789639582129336, 566561755614653902, 580430101104173558, 649794434516738717, 723861253440532319, 782467130834547122, 823061349451565856, 929733096795523467, 996600424723016161|
|practice_0161|mixed|48|1|265|302|0|265|5.70|42418406101, 154946889483, 2302205126233, 89078683230, 2036084663652, 254158987381, 282121588389, 236185822410240, 899119235789, 22120131174680124, 1913972713028, 649794434516738717, 1685543374908150, 308950305198811200, 125810949519001281, 238245469826085238, 580430101104173558, 374694249240572089, 517789639582129336, 723861253440532319, 782467130834547122, 455179725236300684, 566561755614653902, 823061349451565856, 210135314550808495, 486184993838179026, 929733096795523467, 996600424723016161|
|practice_0161|mixed|48|2|267|326|0|267|35.11|42418406101, 254158987381, 2302205126233, 154946889483, 282121588389, 1913972713028, 899119235789, 2036084663652, 308950305198811200, 236185822410240, 455179725236300684, 517789639582129336, 89078683230, 22120131174680124, 782467130834547122, 486184993838179026, 125810949519001281, 1685543374908150, 823061349451565856, 566561755614653902, 723861253440532319, 210135314550808495, 374694249240572089, 996600424723016161, 238245469826085238, 649794434516738717, 929733096795523467, 580430101104173558|
|practice_0161|mixed|128|0|243|253|50|293|8.26|42418406101, 89078683230, 154946889483, 254158987381, 282121588389, 899119235789, 1913972713028, 2036084663652, 2302205126233, 236185822410240, 1685543374908150, 22120131174680124, 125810949519001281, 210135314550808495, 238245469826085238, 308950305198811200, 374694249240572089, 455179725236300684, 486184993838179026, 517789639582129336, 566561755614653902, 580430101104173558, 649794434516738717, 723861253440532319, 782467130834547122, 823061349451565856, 929733096795523467, 996600424723016161|
|practice_0161|mixed|128|1|237|241|0|237|30.77|42418406101, 154946889483, 2302205126233, 89078683230, 2036084663652, 254158987381, 282121588389, 899119235789, 1913972713028, 236185822410240, 1685543374908150, 22120131174680124, 517789639582129336, 125810949519001281, 210135314550808495, 566561755614653902, 455179725236300684, 649794434516738717, 308950305198811200, 374694249240572089, 996600424723016161, 238245469826085238, 782467130834547122, 929733096795523467, 486184993838179026, 580430101104173558, 823061349451565856, 723861253440532319|
|practice_0161|mixed|128|2|235|239|37|272|204.11|42418406101, 899119235789, 2036084663652, 154946889483, 89078683230, 2302205126233, 1913972713028, 254158987381, 282121588389, 236185822410240, 308950305198811200, 22120131174680124, 1685543374908150, 125810949519001281, 580430101104173558, 517789639582129336, 455179725236300684, 238245469826085238, 723861253440532319, 823061349451565856, 566561755614653902, 210135314550808495, 996600424723016161, 649794434516738717, 486184993838179026, 374694249240572089, 929733096795523467, 782467130834547122|
|practice_0162|independent|0|0|285|0|0|285|3.73|140351197653162, 162081973704399, 204342717232099, 242896297631890, 248094163488979, 258069297217901, 283069207400407, 329463793747349, 357757835588338, 371424554589356, 407034357982502, 441271749952025, 460154283054601, 464265677077166, 465599230083401, 511032829218824, 533292074685494, 533942220760360, 562153617470829, 592155441988627, 629842648212757, 636427934131681, 641803928937016, 648016261939397, 728145405865800, 735711278977324, 738312573489368, 757832546422507, 820055235158256, 856814361530697, 991632366220999, 998004337955847|
|practice_0162|independent|0|1|268|0|22|290|13.31|140351197653162, 162081973704399, 283069207400407, 728145405865800, 204342717232099, 460154283054601, 465599230083401, 464265677077166, 735711278977324, 738312573489368, 533292074685494, 441271749952025, 242896297631890, 641803928937016, 258069297217901, 329463793747349, 357757835588338, 248094163488979, 533942220760360, 407034357982502, 371424554589356, 511032829218824, 636427934131681, 998004337955847, 562153617470829, 592155441988627, 991632366220999, 629842648212757, 648016261939397, 757832546422507, 820055235158256, 856814361530697|
|practice_0162|independent|0|2|258|0|148|406|87.14|371424554589356, 460154283054601, 465599230083401, 248094163488979, 728145405865800, 162081973704399, 533942220760360, 562153617470829, 329463793747349, 533292074685494, 735711278977324, 592155441988627, 140351197653162, 258069297217901, 357757835588338, 641803928937016, 511032829218824, 991632366220999, 464265677077166, 204342717232099, 757832546422507, 738312573489368, 636427934131681, 407034357982502, 242896297631890, 441271749952025, 629842648212757, 998004337955847, 283069207400407, 648016261939397, 820055235158256, 856814361530697|
|practice_0162|independent|48|0|289|350|0|289|1.92|140351197653162, 162081973704399, 204342717232099, 242896297631890, 248094163488979, 258069297217901, 283069207400407, 329463793747349, 357757835588338, 371424554589356, 407034357982502, 441271749952025, 460154283054601, 464265677077166, 465599230083401, 511032829218824, 533292074685494, 533942220760360, 562153617470829, 592155441988627, 629842648212757, 636427934131681, 641803928937016, 648016261939397, 728145405865800, 735711278977324, 738312573489368, 757832546422507, 820055235158256, 856814361530697, 991632366220999, 998004337955847|
|practice_0162|independent|48|1|283|293|0|283|5.76|140351197653162, 162081973704399, 204342717232099, 991632366220999, 757832546422507, 242896297631890, 441271749952025, 856814361530697, 248094163488979, 357757835588338, 283069207400407, 371424554589356, 464265677077166, 258069297217901, 465599230083401, 511032829218824, 533942220760360, 533292074685494, 460154283054601, 407034357982502, 562153617470829, 329463793747349, 629842648212757, 641803928937016, 636427934131681, 738312573489368, 592155441988627, 648016261939397, 728145405865800, 735711278977324, 820055235158256, 998004337955847|
|practice_0162|independent|48|2|285|304|0|285|34.80|371424554589356, 460154283054601, 465599230083401, 140351197653162, 441271749952025, 329463793747349, 728145405865800, 204342717232099, 636427934131681, 562153617470829, 735711278977324, 258069297217901, 592155441988627, 738312573489368, 407034357982502, 641803928937016, 820055235158256, 464265677077166, 629842648212757, 533942220760360, 162081973704399, 242896297631890, 856814361530697, 248094163488979, 998004337955847, 991632366220999, 357757835588338, 648016261939397, 533292074685494, 511032829218824, 283069207400407, 757832546422507|
|practice_0162|independent|128|0|261|267|35|296|9.89|140351197653162, 162081973704399, 204342717232099, 242896297631890, 248094163488979, 258069297217901, 283069207400407, 329463793747349, 357757835588338, 371424554589356, 407034357982502, 441271749952025, 460154283054601, 464265677077166, 465599230083401, 511032829218824, 533292074685494, 533942220760360, 562153617470829, 592155441988627, 629842648212757, 636427934131681, 641803928937016, 648016261939397, 728145405865800, 735711278977324, 738312573489368, 757832546422507, 820055235158256, 856814361530697, 991632366220999, 998004337955847|
|practice_0162|independent|128|1|254|256|1|255|28.49|140351197653162, 162081973704399, 204342717232099, 991632366220999, 283069207400407, 371424554589356, 248094163488979, 441271749952025, 258069297217901, 329463793747349, 460154283054601, 242896297631890, 357757835588338, 407034357982502, 465599230083401, 464265677077166, 629842648212757, 533292074685494, 533942220760360, 511032829218824, 636427934131681, 592155441988627, 641803928937016, 648016261939397, 562153617470829, 728145405865800, 735711278977324, 738312573489368, 757832546422507, 820055235158256, 856814361530697, 998004337955847|
|practice_0162|independent|128|2|241|247|86|327|206.35|371424554589356, 991632366220999, 140351197653162, 242896297631890, 464265677077166, 641803928937016, 460154283054601, 648016261939397, 162081973704399, 636427934131681, 465599230083401, 283069207400407, 258069297217901, 533292074685494, 204342717232099, 533942220760360, 728145405865800, 248094163488979, 329463793747349, 441271749952025, 562153617470829, 407034357982502, 629842648212757, 511032829218824, 738312573489368, 820055235158256, 735711278977324, 998004337955847, 757832546422507, 357757835588338, 592155441988627, 856814361530697|
|practice_0163|independent|0|0|263|0|0|263|3.68|174735510346178, 199329268564145, 314810370068965, 336282594540678, 353522935134789, 357398143088684, 358793972903350, 384689015589443, 456464349478030, 519648163063652, 533578367218053, 535312895831915, 569487312441038, 588817630152934, 680731332672148, 726817366593915, 766092747340758, 771729160553321, 787228669766323, 820875896206500, 901160130579036, 903199107764957, 905351509069907, 923101314391481, 924756422479934, 937383891229928, 961626368059700, 965576132639075, 981216432874878|
|practice_0163|independent|0|1|247|0|146|393|11.60|199329268564145, 174735510346178, 384689015589443, 569487312441038, 535312895831915, 533578367218053, 588817630152934, 358793972903350, 353522935134789, 336282594540678, 357398143088684, 766092747340758, 961626368059700, 981216432874878, 923101314391481, 456464349478030, 905351509069907, 924756422479934, 965576132639075, 519648163063652, 771729160553321, 680731332672148, 901160130579036, 787228669766323, 314810370068965, 937383891229928, 903199107764957, 726817366593915, 820875896206500|
|practice_0163|independent|0|2|244|0|126|370|72.40|199329268564145, 535312895831915, 353522935134789, 961626368059700, 965576132639075, 384689015589443, 680731332672148, 174735510346178, 533578367218053, 771729160553321, 981216432874878, 357398143088684, 787228669766323, 766092747340758, 314810370068965, 905351509069907, 924756422479934, 726817366593915, 358793972903350, 588817630152934, 901160130579036, 903199107764957, 923101314391481, 456464349478030, 820875896206500, 519648163063652, 336282594540678, 937383891229928, 569487312441038|
|practice_0163|independent|48|0|267|333|0|267|1.34|174735510346178, 199329268564145, 314810370068965, 336282594540678, 353522935134789, 357398143088684, 358793972903350, 384689015589443, 456464349478030, 519648163063652, 533578367218053, 535312895831915, 569487312441038, 588817630152934, 680731332672148, 726817366593915, 766092747340758, 771729160553321, 787228669766323, 820875896206500, 901160130579036, 903199107764957, 905351509069907, 923101314391481, 924756422479934, 937383891229928, 961626368059700, 965576132639075, 981216432874878|
|practice_0163|independent|48|1|257|286|0|257|4.82|199329268564145, 174735510346178, 535312895831915, 353522935134789, 358793972903350, 357398143088684, 533578367218053, 384689015589443, 569487312441038, 336282594540678, 588817630152934, 766092747340758, 771729160553321, 937383891229928, 961626368059700, 726817366593915, 787228669766323, 901160130579036, 965576132639075, 519648163063652, 680731332672148, 905351509069907, 456464349478030, 903199107764957, 924756422479934, 923101314391481, 820875896206500, 981216432874878, 314810370068965|
|practice_0163|independent|48|2|261|274|0|261|28.61|199329268564145, 535312895831915, 903199107764957, 174735510346178, 924756422479934, 923101314391481, 314810370068965, 357398143088684, 961626368059700, 358793972903350, 353522935134789, 456464349478030, 820875896206500, 533578367218053, 588817630152934, 901160130579036, 519648163063652, 787228669766323, 680731332672148, 766092747340758, 336282594540678, 981216432874878, 771729160553321, 384689015589443, 569487312441038, 965576132639075, 937383891229928, 726817366593915, 905351509069907|
|practice_0163|independent|128|0|239|241|2|241|7.70|174735510346178, 199329268564145, 314810370068965, 336282594540678, 353522935134789, 357398143088684, 358793972903350, 384689015589443, 456464349478030, 519648163063652, 533578367218053, 535312895831915, 569487312441038, 588817630152934, 680731332672148, 726817366593915, 766092747340758, 771729160553321, 787228669766323, 820875896206500, 901160130579036, 903199107764957, 905351509069907, 923101314391481, 924756422479934, 937383891229928, 961626368059700, 965576132639075, 981216432874878|
|practice_0163|independent|128|1|231|234|0|231|26.96|199329268564145, 174735510346178, 384689015589443, 569487312441038, 535312895831915, 533578367218053, 358793972903350, 357398143088684, 726817366593915, 588817630152934, 353522935134789, 766092747340758, 771729160553321, 314810370068965, 456464349478030, 336282594540678, 519648163063652, 820875896206500, 901160130579036, 903199107764957, 905351509069907, 937383891229928, 924756422479934, 923101314391481, 680731332672148, 787228669766323, 961626368059700, 965576132639075, 981216432874878|
|practice_0163|independent|128|2|228|231|30|258|188.44|199329268564145, 535312895831915, 353522935134789, 961626368059700, 965576132639075, 384689015589443, 820875896206500, 174735510346178, 358793972903350, 726817366593915, 680731332672148, 533578367218053, 569487312441038, 519648163063652, 771729160553321, 336282594540678, 766092747340758, 357398143088684, 314810370068965, 923101314391481, 901160130579036, 924756422479934, 456464349478030, 787228669766323, 905351509069907, 588817630152934, 937383891229928, 903199107764957, 981216432874878|
|practice_0164|mixed|0|0|256|0|0|256|3.33|85933782799, 113258582040, 113258582260, 157175770730, 945271615411, 1733833821132, 26131352989130, 71520645622668, 143385365150025, 145056226080424, 824522478896200, 2743792078569686, 12486758694826500, 146995386281861861, 325462030314932389, 345771152532184240, 451769220170265128, 529160647204624490, 583524072033077916, 611526930393934441, 618905517433095760, 626277426288568576, 697398739626569752, 704457530172297927, 754152624453126440, 771962204616370648, 786038266831157350, 925048194618706300, 965731400585228879|
|practice_0164|mixed|0|1|243|0|0|243|14.21|85933782799, 945271615411, 145056226080424, 113258582040, 113258582260, 143385365150025, 824522478896200, 12486758694826500, 583524072033077916, 157175770730, 1733833821132, 26131352989130, 2743792078569686, 71520645622668, 146995386281861861, 697398739626569752, 325462030314932389, 626277426288568576, 611526930393934441, 345771152532184240, 451769220170265128, 704457530172297927, 618905517433095760, 754152624453126440, 529160647204624490, 771962204616370648, 925048194618706300, 786038266831157350, 965731400585228879|
|practice_0164|mixed|0|2|243|0|4|247|92.25|113258582040, 157175770730, 113258582260, 12486758694826500, 85933782799, 945271615411, 145056226080424, 26131352989130, 71520645622668, 824522478896200, 1733833821132, 143385365150025, 146995386281861861, 583524072033077916, 2743792078569686, 451769220170265128, 325462030314932389, 618905517433095760, 704457530172297927, 345771152532184240, 626277426288568576, 965731400585228879, 771962204616370648, 611526930393934441, 697398739626569752, 754152624453126440, 786038266831157350, 529160647204624490, 925048194618706300|
|practice_0164|mixed|48|0|281|380|0|281|1.81|85933782799, 113258582040, 113258582260, 157175770730, 945271615411, 1733833821132, 26131352989130, 71520645622668, 143385365150025, 145056226080424, 824522478896200, 2743792078569686, 12486758694826500, 146995386281861861, 325462030314932389, 345771152532184240, 451769220170265128, 529160647204624490, 583524072033077916, 611526930393934441, 618905517433095760, 626277426288568576, 697398739626569752, 704457530172297927, 754152624453126440, 771962204616370648, 786038266831157350, 925048194618706300, 965731400585228879|
|practice_0164|mixed|48|1|259|268|0|259|6.57|85933782799, 945271615411, 145056226080424, 157175770730, 113258582260, 113258582040, 824522478896200, 12486758694826500, 143385365150025, 1733833821132, 26131352989130, 583524072033077916, 71520645622668, 325462030314932389, 345771152532184240, 451769220170265128, 611526930393934441, 2743792078569686, 146995386281861861, 529160647204624490, 626277426288568576, 697398739626569752, 704457530172297927, 618905517433095760, 754152624453126440, 786038266831157350, 771962204616370648, 965731400585228879, 925048194618706300|
|practice_0164|mixed|48|2|274|292|0|274|37.73|113258582260, 85933782799, 945271615411, 113258582040, 157175770730, 26131352989130, 1733833821132, 824522478896200, 626277426288568576, 143385365150025, 12486758694826500, 145056226080424, 2743792078569686, 618905517433095760, 71520645622668, 925048194618706300, 611526930393934441, 697398739626569752, 771962204616370648, 786038266831157350, 754152624453126440, 345771152532184240, 965731400585228879, 146995386281861861, 583524072033077916, 451769220170265128, 704457530172297927, 325462030314932389, 529160647204624490|
|practice_0164|mixed|128|0|236|256|11|247|8.97|85933782799, 113258582040, 113258582260, 157175770730, 945271615411, 1733833821132, 26131352989130, 71520645622668, 143385365150025, 145056226080424, 824522478896200, 2743792078569686, 12486758694826500, 146995386281861861, 325462030314932389, 345771152532184240, 451769220170265128, 529160647204624490, 583524072033077916, 611526930393934441, 618905517433095760, 626277426288568576, 697398739626569752, 704457530172297927, 754152624453126440, 771962204616370648, 786038266831157350, 925048194618706300, 965731400585228879|
|practice_0164|mixed|128|1|230|233|37|267|30.38|85933782799, 945271615411, 145056226080424, 157175770730, 113258582040, 113258582260, 143385365150025, 824522478896200, 12486758694826500, 1733833821132, 26131352989130, 71520645622668, 2743792078569686, 771962204616370648, 583524072033077916, 146995386281861861, 345771152532184240, 451769220170265128, 618905517433095760, 325462030314932389, 754152624453126440, 626277426288568576, 786038266831157350, 611526930393934441, 529160647204624490, 697398739626569752, 704457530172297927, 965731400585228879, 925048194618706300|
|practice_0164|mixed|128|2|220|227|0|220|199.38|113258582260, 85933782799, 12486758694826500, 113258582040, 824522478896200, 143385365150025, 26131352989130, 2743792078569686, 945271615411, 145056226080424, 583524072033077916, 71520645622668, 1733833821132, 611526930393934441, 345771152532184240, 157175770730, 697398739626569752, 451769220170265128, 618905517433095760, 626277426288568576, 965731400585228879, 786038266831157350, 146995386281861861, 925048194618706300, 704457530172297927, 771962204616370648, 325462030314932389, 529160647204624490, 754152624453126440|
|practice_0165|mixed|0|0|244|0|0|244|3.78|93857108, 793137363, 1032430608, 1689436044, 6239168587, 6702369612, 28552938828, 46169849456, 56783676152, 60321325221, 87130802438, 369358805971, 624540804594, 5540381965440, 8141520516372, 29109466423728, 211416579123327, 273486270203587, 292078253938012, 334399782209879, 348135430580822, 478755644087116, 497988479802881, 604819204885970, 645817684866744, 706417489393627, 716961819468829, 739764283212716, 827143571646334, 888252783648734, 895493003502826, 960281648806151|
|practice_0165|mixed|0|1|224|0|0|224|11.65|793137363, 1032430608, 93857108, 1689436044, 28552938828, 6239168587, 56783676152, 6702369612, 60321325221, 87130802438, 46169849456, 5540381965440, 369358805971, 624540804594, 29109466423728, 8141520516372, 211416579123327, 273486270203587, 292078253938012, 716961819468829, 334399782209879, 348135430580822, 478755644087116, 497988479802881, 604819204885970, 739764283212716, 706417489393627, 645817684866744, 960281648806151, 895493003502826, 827143571646334, 888252783648734|
|practice_0165|mixed|0|2|219|0|59|278|83.46|93857108, 6702369612, 56783676152, 87130802438, 1032430608, 46169849456, 5540381965440, 6239168587, 60321325221, 1689436044, 793137363, 369358805971, 28552938828, 624540804594, 29109466423728, 645817684866744, 8141520516372, 334399782209879, 292078253938012, 211416579123327, 497988479802881, 478755644087116, 348135430580822, 827143571646334, 706417489393627, 739764283212716, 888252783648734, 895493003502826, 716961819468829, 273486270203587, 960281648806151, 604819204885970|
|practice_0165|mixed|48|0|238|289|0|238|1.76|93857108, 793137363, 1032430608, 1689436044, 6239168587, 6702369612, 28552938828, 46169849456, 56783676152, 60321325221, 87130802438, 369358805971, 624540804594, 5540381965440, 8141520516372, 29109466423728, 211416579123327, 273486270203587, 292078253938012, 334399782209879, 348135430580822, 478755644087116, 497988479802881, 604819204885970, 645817684866744, 706417489393627, 716961819468829, 739764283212716, 827143571646334, 888252783648734, 895493003502826, 960281648806151|
|practice_0165|mixed|48|1|230|259|0|230|5.15|793137363, 28552938828, 93857108, 1032430608, 1689436044, 6239168587, 56783676152, 5540381965440, 46169849456, 369358805971, 6702369612, 60321325221, 87130802438, 624540804594, 29109466423728, 895493003502826, 604819204885970, 8141520516372, 348135430580822, 334399782209879, 888252783648734, 292078253938012, 211416579123327, 273486270203587, 497988479802881, 478755644087116, 645817684866744, 716961819468829, 706417489393627, 960281648806151, 739764283212716, 827143571646334|
|practice_0165|mixed|48|2|242|255|0|242|34.26|1032430608, 793137363, 93857108, 1689436044, 56783676152, 60321325221, 87130802438, 46169849456, 6239168587, 28552938828, 6702369612, 369358805971, 624540804594, 273486270203587, 645817684866744, 8141520516372, 348135430580822, 478755644087116, 211416579123327, 5540381965440, 292078253938012, 29109466423728, 827143571646334, 895493003502826, 497988479802881, 604819204885970, 706417489393627, 334399782209879, 739764283212716, 960281648806151, 888252783648734, 716961819468829|
|practice_0165|mixed|128|0|219|231|38|257|6.32|93857108, 793137363, 1032430608, 1689436044, 6239168587, 6702369612, 28552938828, 46169849456, 56783676152, 60321325221, 87130802438, 369358805971, 624540804594, 5540381965440, 8141520516372, 29109466423728, 211416579123327, 273486270203587, 292078253938012, 334399782209879, 348135430580822, 478755644087116, 497988479802881, 604819204885970, 645817684866744, 706417489393627, 716961819468829, 739764283212716, 827143571646334, 888252783648734, 895493003502826, 960281648806151|
|practice_0165|mixed|128|1|208|223|18|226|22.16|793137363, 28552938828, 93857108, 1032430608, 1689436044, 56783676152, 6239168587, 6702369612, 60321325221, 87130802438, 46169849456, 5540381965440, 369358805971, 624540804594, 29109466423728, 8141520516372, 211416579123327, 292078253938012, 334399782209879, 895493003502826, 348135430580822, 478755644087116, 960281648806151, 273486270203587, 604819204885970, 645817684866744, 497988479802881, 706417489393627, 716961819468829, 739764283212716, 827143571646334, 888252783648734|
|practice_0165|mixed|128|2|204|209|1|205|144.99|1032430608, 46169849456, 93857108, 6702369612, 60321325221, 87130802438, 369358805971, 5540381965440, 1689436044, 56783676152, 6239168587, 28552938828, 29109466423728, 793137363, 604819204885970, 624540804594, 8141520516372, 478755644087116, 273486270203587, 292078253938012, 706417489393627, 211416579123327, 716961819468829, 497988479802881, 334399782209879, 739764283212716, 645817684866744, 888252783648734, 960281648806151, 348135430580822, 827143571646334, 895493003502826|
|practice_0166|shared|0|0|54|0|0|54|0.21|56689, 166752, 166912, 167099, 492572, 492610, 5913648, 7017235, 7184268, 13799100, 19882044|
|practice_0166|shared|0|1|49|0|0|49|0.17|166912, 166752, 167099, 56689, 492572, 492610, 7184268, 7017235, 19882044, 5913648, 13799100|
|practice_0166|shared|0|2|49|0|0|49|0.84|166912, 167099, 7184268, 492572, 166752, 492610, 7017235, 19882044, 56689, 5913648, 13799100|
|practice_0166|shared|48|0|54|59|0|54|0.19|56689, 166752, 166912, 167099, 492572, 492610, 5913648, 7017235, 7184268, 13799100, 19882044|
|practice_0166|shared|48|1|49|53|0|49|0.17|166912, 166752, 167099, 56689, 492572, 492610, 7184268, 7017235, 19882044, 5913648, 13799100|
|practice_0166|shared|48|2|47|53|0|47|0.82|166912, 167099, 7184268, 492572, 166752, 492610, 7017235, 19882044, 56689, 5913648, 13799100|
|practice_0166|shared|128|0|53|59|0|53|0.31|56689, 166752, 166912, 167099, 492572, 492610, 5913648, 7017235, 7184268, 13799100, 19882044|
|practice_0166|shared|128|1|49|53|0|49|0.16|166912, 166752, 167099, 56689, 492572, 492610, 7184268, 7017235, 19882044, 5913648, 13799100|
|practice_0166|shared|128|2|47|53|0|47|0.67|166912, 167099, 7184268, 492572, 166752, 492610, 7017235, 19882044, 56689, 5913648, 13799100|
|practice_0167|mixed|0|0|300|0|0|300|4.20|205356701409, 788588400753, 964122526623, 2169428202904, 7864551946037, 11570283746799, 11807901178920, 26144258128653, 29412290397228, 65649717828399, 586746358770090, 653034319269344, 1772047803978950, 2127391315479966, 2299593913677000, 104939363799424248, 110342883540903522, 147522302373850478, 155876053032623367, 189727826837785091, 226381425613787180, 229812654854996769, 317798351310297671, 357870528533677178, 366250808749458822, 376647041997338629, 437071443271445863, 471563025880358711, 596586361938862383, 807851174824112213, 829936280938409243|
|practice_0167|mixed|0|1|290|0|114|404|18.53|205356701409, 788588400753, 964122526623, 11807901178920, 2169428202904, 7864551946037, 11570283746799, 26144258128653, 29412290397228, 65649717828399, 586746358770090, 653034319269344, 2299593913677000, 1772047803978950, 2127391315479966, 147522302373850478, 155876053032623367, 189727826837785091, 110342883540903522, 357870528533677178, 317798351310297671, 104939363799424248, 229812654854996769, 226381425613787180, 366250808749458822, 376647041997338629, 471563025880358711, 437071443271445863, 596586361938862383, 807851174824112213, 829936280938409243|
|practice_0167|mixed|0|2|280|0|138|418|124.95|788588400753, 2169428202904, 964122526623, 2299593913677000, 11807901178920, 11570283746799, 26144258128653, 205356701409, 653034319269344, 29412290397228, 7864551946037, 586746358770090, 1772047803978950, 110342883540903522, 65649717828399, 596586361938862383, 2127391315479966, 229812654854996769, 376647041997338629, 189727826837785091, 317798351310297671, 226381425613787180, 366250808749458822, 104939363799424248, 437071443271445863, 155876053032623367, 147522302373850478, 357870528533677178, 829936280938409243, 471563025880358711, 807851174824112213|
|practice_0167|mixed|48|0|304|413|0|304|1.64|205356701409, 788588400753, 964122526623, 2169428202904, 7864551946037, 11570283746799, 11807901178920, 26144258128653, 29412290397228, 65649717828399, 586746358770090, 653034319269344, 1772047803978950, 2127391315479966, 2299593913677000, 104939363799424248, 110342883540903522, 147522302373850478, 155876053032623367, 189727826837785091, 226381425613787180, 229812654854996769, 317798351310297671, 357870528533677178, 366250808749458822, 376647041997338629, 437071443271445863, 471563025880358711, 596586361938862383, 807851174824112213, 829936280938409243|
|practice_0167|mixed|48|1|312|318|0|312|6.40|205356701409, 788588400753, 964122526623, 11570283746799, 2169428202904, 11807901178920, 26144258128653, 7864551946037, 2299593913677000, 29412290397228, 653034319269344, 65649717828399, 586746358770090, 104939363799424248, 2127391315479966, 1772047803978950, 110342883540903522, 155876053032623367, 437071443271445863, 226381425613787180, 147522302373850478, 471563025880358711, 357870528533677178, 376647041997338629, 229812654854996769, 366250808749458822, 829936280938409243, 189727826837785091, 807851174824112213, 317798351310297671, 596586361938862383|
|practice_0167|mixed|48|2|304|316|0|304|41.51|11570283746799, 2169428202904, 26144258128653, 29412290397228, 964122526623, 205356701409, 65649717828399, 1772047803978950, 147522302373850478, 788588400753, 653034319269344, 7864551946037, 11807901178920, 2299593913677000, 155876053032623367, 2127391315479966, 357870528533677178, 437071443271445863, 366250808749458822, 317798351310297671, 586746358770090, 189727826837785091, 596586361938862383, 229812654854996769, 376647041997338629, 110342883540903522, 829936280938409243, 226381425613787180, 471563025880358711, 104939363799424248, 807851174824112213|
|practice_0167|mixed|128|0|272|283|50|322|9.39|205356701409, 788588400753, 964122526623, 2169428202904, 7864551946037, 11570283746799, 11807901178920, 26144258128653, 29412290397228, 65649717828399, 586746358770090, 653034319269344, 1772047803978950, 2127391315479966, 2299593913677000, 104939363799424248, 110342883540903522, 147522302373850478, 155876053032623367, 189727826837785091, 226381425613787180, 229812654854996769, 317798351310297671, 357870528533677178, 366250808749458822, 376647041997338629, 437071443271445863, 471563025880358711, 596586361938862383, 807851174824112213, 829936280938409243|
|practice_0167|mixed|128|1|264|266|11|275|37.72|205356701409, 788588400753, 964122526623, 11807901178920, 2169428202904, 7864551946037, 11570283746799, 26144258128653, 653034319269344, 2299593913677000, 29412290397228, 65649717828399, 1772047803978950, 586746358770090, 2127391315479966, 110342883540903522, 155876053032623367, 104939363799424248, 189727826837785091, 229812654854996769, 357870528533677178, 317798351310297671, 366250808749458822, 226381425613787180, 147522302373850478, 437071443271445863, 376647041997338629, 471563025880358711, 596586361938862383, 807851174824112213, 829936280938409243|
|practice_0167|mixed|128|2|264|268|47|311|257.29|11570283746799, 11807901178920, 653034319269344, 26144258128653, 205356701409, 788588400753, 964122526623, 2169428202904, 29412290397228, 1772047803978950, 7864551946037, 65649717828399, 189727826837785091, 586746358770090, 229812654854996769, 2299593913677000, 357870528533677178, 147522302373850478, 226381425613787180, 596586361938862383, 807851174824112213, 104939363799424248, 437071443271445863, 110342883540903522, 2127391315479966, 317798351310297671, 366250808749458822, 376647041997338629, 471563025880358711, 829936280938409243, 155876053032623367|
|practice_0168|multiplicative|0|0|130|0|0|130|2.81|155568, 232926, 275481, 275514, 467267, 642871, 643415, 1469446, 1869536, 3202160, 5511814, 6541186, 6728380, 8177313, 10101982, 16397683, 20180880, 66142080, 197297380, 394661706, 449565626, 678859104, 1388983686, 2943702256, 8777873399, 38458856576, 43411250408, 220937781142|
|practice_0168|multiplicative|0|1|127|0|0|127|3.94|155568, 232926, 3202160, 275481, 275514, 642871, 643415, 467267, 1469446, 1869536, 5511814, 6541186, 10101982, 678859104, 6728380, 8177313, 16397683, 66142080, 20180880, 1388983686, 197297380, 394661706, 449565626, 2943702256, 8777873399, 38458856576, 43411250408, 220937781142|
|practice_0168|multiplicative|0|2|122|0|0|122|19.51|275514, 642871, 155568, 1469446, 5511814, 66142080, 643415, 467267, 275481, 3202160, 232926, 6541186, 6728380, 678859104, 20180880, 1869536, 197297380, 8177313, 1388983686, 10101982, 16397683, 2943702256, 43411250408, 394661706, 38458856576, 449565626, 8777873399, 220937781142|
|practice_0168|multiplicative|48|0|132|168|0|132|0.94|155568, 232926, 275481, 275514, 467267, 642871, 643415, 1469446, 1869536, 3202160, 5511814, 6541186, 6728380, 8177313, 10101982, 16397683, 20180880, 66142080, 197297380, 394661706, 449565626, 678859104, 1388983686, 2943702256, 8777873399, 38458856576, 43411250408, 220937781142|
|practice_0168|multiplicative|48|1|129|139|0|129|1.72|155568, 232926, 275481, 275514, 642871, 643415, 467267, 1469446, 1869536, 3202160, 6541186, 678859104, 10101982, 5511814, 6728380, 66142080, 8177313, 16397683, 20180880, 1388983686, 197297380, 394661706, 449565626, 8777873399, 38458856576, 43411250408, 2943702256, 220937781142|
|practice_0168|multiplicative|48|2|121|130|0|121|10.49|275514, 642871, 155568, 1469446, 643415, 5511814, 275481, 66142080, 467267, 1869536, 3202160, 232926, 6541186, 197297380, 16397683, 43411250408, 678859104, 394661706, 1388983686, 10101982, 8177313, 6728380, 20180880, 220937781142, 449565626, 2943702256, 8777873399, 38458856576|
|practice_0168|multiplicative|128|0|128|128|0|128|3.66|155568, 232926, 275481, 275514, 467267, 642871, 643415, 1469446, 1869536, 3202160, 5511814, 6541186, 6728380, 8177313, 10101982, 16397683, 20180880, 66142080, 197297380, 394661706, 449565626, 678859104, 1388983686, 2943702256, 8777873399, 38458856576, 43411250408, 220937781142|
|practice_0168|multiplicative|128|1|123|126|0|123|5.38|155568, 232926, 275481, 275514, 642871, 643415, 467267, 1469446, 1869536, 3202160, 6541186, 10101982, 678859104, 16397683, 5511814, 8177313, 6728380, 20180880, 66142080, 197297380, 394661706, 449565626, 1388983686, 2943702256, 8777873399, 38458856576, 43411250408, 220937781142|
|practice_0168|multiplicative|128|2|119|122|0|119|28.65|275514, 642871, 155568, 1469446, 5511814, 643415, 66142080, 275481, 678859104, 467267, 10101982, 3202160, 1388983686, 6541186, 8177313, 20180880, 232926, 1869536, 16397683, 43411250408, 197297380, 6728380, 394661706, 449565626, 2943702256, 38458856576, 8777873399, 220937781142|
|practice_0169|tiny|0|0|6|0|0|6|0.01|3, 4, 5, 9|
|practice_0169|tiny|0|1|6|0|0|6|0.01|3, 4, 5, 9|
|practice_0169|tiny|0|2|6|0|0|6|0.01|3, 4, 5, 9|
|practice_0169|tiny|48|0|6|8|0|6|0.01|3, 4, 5, 9|
|practice_0169|tiny|48|1|6|8|0|6|0.01|3, 4, 5, 9|
|practice_0169|tiny|48|2|6|8|0|6|0.02|3, 4, 5, 9|
|practice_0169|tiny|128|0|6|8|0|6|0.02|3, 4, 5, 9|
|practice_0169|tiny|128|1|6|8|0|6|0.02|3, 4, 5, 9|
|practice_0169|tiny|128|2|6|8|0|6|0.01|3, 4, 5, 9|
|practice_0170|multiplicative|0|0|41|0|0|41|0.07|291, 321, 416, 663, 839, 1110, 1395, 1439, 2548, 19390, 25381|
|practice_0170|multiplicative|0|1|41|0|0|41|0.07|416, 291, 321, 663, 839, 1110, 1395, 1439, 2548, 19390, 25381|
|practice_0170|multiplicative|0|2|41|0|0|41|0.39|416, 321, 291, 663, 839, 1110, 1395, 1439, 2548, 19390, 25381|
|practice_0170|multiplicative|48|0|39|44|0|39|0.06|291, 321, 416, 663, 839, 1110, 1395, 1439, 2548, 19390, 25381|
|practice_0170|multiplicative|48|1|39|44|0|39|0.14|416, 291, 321, 663, 839, 1110, 1395, 1439, 2548, 25381, 19390|
|practice_0170|multiplicative|48|2|37|42|0|37|0.38|416, 1439, 1395, 291, 321, 663, 839, 1110, 2548, 19390, 25381|
|practice_0170|multiplicative|128|0|39|44|0|39|0.09|291, 321, 416, 663, 839, 1110, 1395, 1439, 2548, 19390, 25381|
|practice_0170|multiplicative|128|1|39|44|0|39|0.06|416, 291, 321, 663, 839, 1110, 1395, 1439, 2548, 25381, 19390|
|practice_0170|multiplicative|128|2|37|42|0|37|0.26|416, 1439, 1395, 291, 321, 663, 839, 1110, 2548, 19390, 25381|
|practice_0171|mixed|0|0|292|0|0|292|3.73|8851699716, 8851712313, 23848433704, 59189511810, 59189553142, 79734741406, 79734745028, 1240117221876, 13862423488499, 40685381243336, 50429476575444, 52853495194738, 55449693959400, 325518823279645, 43016343403300060, 76412488363451562, 148328288064976704, 160849395739770682, 197268862868876555, 223457741727681311, 245365988013753065, 348284770246567750, 387969256191109088, 407661245537133055, 494294545050102091, 572274025942904416, 598025092441351609, 680960675285986054, 738284318807194065, 810637843540198054, 881709564646682383, 961078676688255157|
|practice_0171|mixed|0|1|273|0|9|282|16.97|8851699716, 8851712313, 23848433704, 1240117221876, 59189511810, 59189553142, 79734741406, 79734745028, 55449693959400, 13862423488499, 50429476575444, 40685381243336, 52853495194738, 43016343403300060, 325518823279645, 76412488363451562, 148328288064976704, 810637843540198054, 223457741727681311, 598025092441351609, 160849395739770682, 348284770246567750, 494294545050102091, 197268862868876555, 572274025942904416, 245365988013753065, 738284318807194065, 407661245537133055, 387969256191109088, 881709564646682383, 680960675285986054, 961078676688255157|
|practice_0171|mixed|0|2|271|0|19|290|124.18|8851699716, 59189553142, 79734741406, 59189511810, 79734745028, 1240117221876, 55449693959400, 23848433704, 8851712313, 13862423488499, 40685381243336, 50429476575444, 52853495194738, 43016343403300060, 598025092441351609, 76412488363451562, 245365988013753065, 572274025942904416, 148328288064976704, 325518823279645, 160849395739770682, 197268862868876555, 407661245537133055, 881709564646682383, 680960675285986054, 348284770246567750, 961078676688255157, 810637843540198054, 387969256191109088, 223457741727681311, 738284318807194065, 494294545050102091|
|practice_0171|mixed|48|0|293|426|0|293|1.85|8851699716, 8851712313, 23848433704, 59189511810, 59189553142, 79734741406, 79734745028, 1240117221876, 13862423488499, 40685381243336, 50429476575444, 52853495194738, 55449693959400, 325518823279645, 43016343403300060, 76412488363451562, 148328288064976704, 160849395739770682, 197268862868876555, 223457741727681311, 245365988013753065, 348284770246567750, 387969256191109088, 407661245537133055, 494294545050102091, 572274025942904416, 598025092441351609, 680960675285986054, 738284318807194065, 810637843540198054, 881709564646682383, 961078676688255157|
|practice_0171|mixed|48|1|291|300|0|291|6.07|8851699716, 8851712313, 23848433704, 1240117221876, 59189511810, 59189553142, 79734741406, 79734745028, 50429476575444, 52853495194738, 13862423488499, 55449693959400, 40685381243336, 325518823279645, 43016343403300060, 197268862868876555, 680960675285986054, 223457741727681311, 572274025942904416, 494294545050102091, 76412488363451562, 738284318807194065, 148328288064976704, 810637843540198054, 245365988013753065, 387969256191109088, 160849395739770682, 348284770246567750, 407661245537133055, 598025092441351609, 881709564646682383, 961078676688255157|
|practice_0171|mixed|48|2|289|324|0|289|42.53|8851699716, 23848433704, 79734741406, 79734745028, 8851712313, 55449693959400, 59189553142, 59189511810, 13862423488499, 40685381243336, 50429476575444, 52853495194738, 43016343403300060, 810637843540198054, 325518823279645, 598025092441351609, 1240117221876, 197268862868876555, 245365988013753065, 572274025942904416, 76412488363451562, 407661245537133055, 160849395739770682, 387969256191109088, 223457741727681311, 348284770246567750, 148328288064976704, 494294545050102091, 738284318807194065, 961078676688255157, 881709564646682383, 680960675285986054|
|practice_0171|mixed|128|0|260|266|15|275|8.59|8851699716, 8851712313, 23848433704, 59189511810, 59189553142, 79734741406, 79734745028, 1240117221876, 13862423488499, 40685381243336, 50429476575444, 52853495194738, 55449693959400, 325518823279645, 43016343403300060, 76412488363451562, 148328288064976704, 160849395739770682, 197268862868876555, 223457741727681311, 245365988013753065, 348284770246567750, 387969256191109088, 407661245537133055, 494294545050102091, 572274025942904416, 598025092441351609, 680960675285986054, 738284318807194065, 810637843540198054, 881709564646682383, 961078676688255157|
|practice_0171|mixed|128|1|246|250|0|246|38.83|8851699716, 8851712313, 23848433704, 1240117221876, 59189511810, 59189553142, 79734741406, 79734745028, 55449693959400, 13862423488499, 52853495194738, 50429476575444, 40685381243336, 43016343403300060, 325518823279645, 148328288064976704, 76412488363451562, 197268862868876555, 160849395739770682, 223457741727681311, 494294545050102091, 738284318807194065, 348284770246567750, 245365988013753065, 407661245537133055, 387969256191109088, 598025092441351609, 572274025942904416, 680960675285986054, 810637843540198054, 881709564646682383, 961078676688255157|
|practice_0171|mixed|128|2|261|265|82|343|238.47|8851699716, 23848433704, 59189511810, 79734741406, 59189553142, 79734745028, 8851712313, 13862423488499, 55449693959400, 52853495194738, 1240117221876, 40685381243336, 50429476575444, 43016343403300060, 76412488363451562, 325518823279645, 148328288064976704, 223457741727681311, 160849395739770682, 494294545050102091, 387969256191109088, 348284770246567750, 245365988013753065, 680960675285986054, 197268862868876555, 572274025942904416, 961078676688255157, 810637843540198054, 598025092441351609, 881709564646682383, 407661245537133055, 738284318807194065|
|practice_0172|tiny|0|0|2|0|0|2|0.01|2|
|practice_0172|tiny|0|1|2|0|0|2|0.01|2|
|practice_0172|tiny|0|2|2|0|0|2|0.01|2|
|practice_0172|tiny|48|0|2|8|0|2|0.01|2|
|practice_0172|tiny|48|1|2|8|0|2|0.01|2|
|practice_0172|tiny|48|2|2|8|0|2|0.01|2|
|practice_0172|tiny|128|0|2|8|0|2|0.01|2|
|practice_0172|tiny|128|1|2|8|0|2|0.01|2|
|practice_0172|tiny|128|2|2|8|0|2|0.01|2|
|practice_0173|shared|0|0|147|0|0|147|2.83|494538789, 494539233, 494539558, 726510146, 726511694, 726511852, 5085584272, 6507971509, 6614074956, 7066559840, 7785939176, 8718155853, 10171165815, 11483159766, 11483160689, 12366480658, 13829439671, 15016439575, 17308883464, 17666399681, 25427914528, 30916199686, 50855829495, 555862415957, 816599349735, 5580585743432, 6916392783440, 24207375302600, 64005900528635, 347138693385848|
|practice_0173|shared|0|1|152|0|0|152|6.80|494538789, 494539233, 494539558, 17308883464, 726511694, 726510146, 726511852, 25427914528, 10171165815, 5085584272, 50855829495, 8718155853, 11483160689, 11483159766, 7066559840, 12366480658, 15016439575, 17666399681, 30916199686, 6916392783440, 6507971509, 7785939176, 6614074956, 816599349735, 13829439671, 24207375302600, 555862415957, 5580585743432, 347138693385848, 64005900528635|
|practice_0173|shared|0|2|136|0|0|136|39.09|726511852, 726510146, 10171165815, 726511694, 494539558, 494538789, 494539233, 6916392783440, 11483160689, 7066559840, 11483159766, 15016439575, 17308883464, 12366480658, 5085584272, 25427914528, 6507971509, 50855829495, 17666399681, 30916199686, 64005900528635, 816599349735, 555862415957, 7785939176, 5580585743432, 8718155853, 13829439671, 6614074956, 24207375302600, 347138693385848|
|practice_0173|shared|48|0|169|225|0|169|1.31|494538789, 494539233, 494539558, 726510146, 726511694, 726511852, 5085584272, 6507971509, 6614074956, 7066559840, 7785939176, 8718155853, 10171165815, 11483159766, 11483160689, 12366480658, 13829439671, 15016439575, 17308883464, 17666399681, 25427914528, 30916199686, 50855829495, 555862415957, 816599349735, 5580585743432, 6916392783440, 24207375302600, 64005900528635, 347138693385848|
|practice_0173|shared|48|1|154|168|0|154|3.53|494538789, 494539233, 494539558, 17308883464, 726511694, 726510146, 726511852, 12366480658, 25427914528, 50855829495, 10171165815, 5085584272, 8718155853, 6916392783440, 6507971509, 64005900528635, 7066559840, 11483160689, 11483159766, 17666399681, 30916199686, 13829439671, 15016439575, 7785939176, 6614074956, 5580585743432, 555862415957, 24207375302600, 816599349735, 347138693385848|
|practice_0173|shared|48|2|161|167|0|161|22.91|726511852, 726510146, 10171165815, 726511694, 494539558, 494538789, 50855829495, 12366480658, 494539233, 8718155853, 11483160689, 7066559840, 15016439575, 5085584272, 17666399681, 17308883464, 6614074956, 30916199686, 6916392783440, 25427914528, 11483159766, 555862415957, 6507971509, 816599349735, 13829439671, 7785939176, 64005900528635, 347138693385848, 24207375302600, 5580585743432|
|practice_0173|shared|128|0|150|153|17|167|5.16|494538789, 494539233, 494539558, 726510146, 726511694, 726511852, 5085584272, 6507971509, 6614074956, 7066559840, 7785939176, 8718155853, 10171165815, 11483159766, 11483160689, 12366480658, 13829439671, 15016439575, 17308883464, 17666399681, 25427914528, 30916199686, 50855829495, 555862415957, 816599349735, 5580585743432, 6916392783440, 24207375302600, 64005900528635, 347138693385848|
|practice_0173|shared|128|1|143|149|5|148|12.11|494538789, 494539233, 494539558, 17308883464, 726511694, 726510146, 726511852, 25427914528, 10171165815, 5085584272, 50855829495, 8718155853, 11483160689, 11483159766, 7066559840, 12366480658, 15016439575, 17666399681, 30916199686, 6916392783440, 6507971509, 7785939176, 6614074956, 816599349735, 24207375302600, 555862415957, 13829439671, 5580585743432, 347138693385848, 64005900528635|
|practice_0173|shared|128|2|143|147|10|153|66.97|726511852, 726510146, 10171165815, 726511694, 494539558, 494538789, 17308883464, 25427914528, 17666399681, 11483159766, 11483160689, 494539233, 7066559840, 12366480658, 15016439575, 5085584272, 30916199686, 50855829495, 8718155853, 555862415957, 7785939176, 6916392783440, 816599349735, 6507971509, 6614074956, 13829439671, 24207375302600, 64005900528635, 347138693385848, 5580585743432|
|practice_0174|tiny|0|0|2|0|0|2|0.01|2|
|practice_0174|tiny|0|1|2|0|0|2|0.01|2|
|practice_0174|tiny|0|2|2|0|0|2|0.01|2|
|practice_0174|tiny|48|0|2|8|0|2|0.01|2|
|practice_0174|tiny|48|1|2|8|0|2|0.01|2|
|practice_0174|tiny|48|2|2|8|0|2|0.01|2|
|practice_0174|tiny|128|0|2|8|0|2|0.01|2|
|practice_0174|tiny|128|1|2|8|0|2|0.01|2|
|practice_0174|tiny|128|2|2|8|0|2|0.01|2|
|practice_0175|tiny|0|0|6|0|0|6|0.01|1, 9, 12, 15|
|practice_0175|tiny|0|1|6|0|0|6|0.01|1, 9, 15, 12|
|practice_0175|tiny|0|2|6|0|0|6|0.01|1, 9, 15, 12|
|practice_0175|tiny|48|0|7|11|0|7|0.01|1, 9, 15, 12|
|practice_0175|tiny|48|1|7|11|0|7|0.01|1, 9, 15, 12|
|practice_0175|tiny|48|2|7|11|0|7|0.01|1, 9, 15, 12|
|practice_0175|tiny|128|0|7|11|0|7|0.01|1, 9, 15, 12|
|practice_0175|tiny|128|1|7|11|0|7|0.01|1, 9, 15, 12|
|practice_0175|tiny|128|2|7|11|0|7|0.01|1, 9, 15, 12|
|practice_0176|independent|0|0|318|0|0|318|3.99|105280150461287758, 106871143279643138, 145038212348708135, 193270389637861274, 195409765175471518, 199528619547510510, 209579026317407881, 229534785594847358, 261732718651120582, 355038100652596481, 365477786511568456, 421069163879798217, 449505030529960947, 469700570914310637, 494485127540215482, 508200384219223082, 512847424116947380, 558248951314970078, 592288466761680293, 623243467656180578, 671323428265898708, 752607559260641172, 768659912298963167, 777879507677340677, 789062749853070988, 802303796192799541, 811911796452247216, 860025318882886970, 879660044379391844, 929587356674553680|
|practice_0176|independent|0|1|306|0|66|372|15.61|469700570914310637, 105280150461287758, 209579026317407881, 145038212348708135, 199528619547510510, 106871143279643138, 193270389637861274, 365477786511568456, 195409765175471518, 229534785594847358, 261732718651120582, 355038100652596481, 421069163879798217, 494485127540215482, 508200384219223082, 512847424116947380, 449505030529960947, 592288466761680293, 768659912298963167, 752607559260641172, 671323428265898708, 860025318882886970, 777879507677340677, 811911796452247216, 558248951314970078, 789062749853070988, 623243467656180578, 802303796192799541, 929587356674553680, 879660044379391844|
|practice_0176|independent|0|2|301|0|200|501|109.16|195409765175471518, 145038212348708135, 105280150461287758, 229534785594847358, 209579026317407881, 199528619547510510, 558248951314970078, 494485127540215482, 193270389637861274, 449505030529960947, 365477786511568456, 671323428265898708, 777879507677340677, 106871143279643138, 261732718651120582, 421069163879798217, 752607559260641172, 512847424116947380, 508200384219223082, 879660044379391844, 789062749853070988, 355038100652596481, 592288466761680293, 768659912298963167, 929587356674553680, 469700570914310637, 802303796192799541, 860025318882886970, 623243467656180578, 811911796452247216|
|practice_0176|independent|48|0|334|445|0|334|2.12|105280150461287758, 106871143279643138, 145038212348708135, 193270389637861274, 195409765175471518, 199528619547510510, 209579026317407881, 229534785594847358, 261732718651120582, 355038100652596481, 365477786511568456, 421069163879798217, 449505030529960947, 469700570914310637, 494485127540215482, 508200384219223082, 512847424116947380, 558248951314970078, 592288466761680293, 623243467656180578, 671323428265898708, 752607559260641172, 768659912298963167, 777879507677340677, 789062749853070988, 802303796192799541, 811911796452247216, 860025318882886970, 879660044379391844, 929587356674553680|
|practice_0176|independent|48|1|326|380|0|326|6.29|469700570914310637, 508200384219223082, 145038212348708135, 106871143279643138, 105280150461287758, 494485127540215482, 802303796192799541, 752607559260641172, 623243467656180578, 199528619547510510, 193270389637861274, 860025318882886970, 929587356674553680, 195409765175471518, 209579026317407881, 355038100652596481, 592288466761680293, 768659912298963167, 261732718651120582, 512847424116947380, 229534785594847358, 365477786511568456, 811911796452247216, 421069163879798217, 879660044379391844, 777879507677340677, 671323428265898708, 449505030529960947, 558248951314970078, 789062749853070988|
|practice_0176|independent|48|2|330|362|0|330|37.16|195409765175471518, 199528619547510510, 106871143279643138, 512847424116947380, 193270389637861274, 145038212348708135, 421069163879798217, 365477786511568456, 449505030529960947, 494485127540215482, 623243467656180578, 671323428265898708, 752607559260641172, 105280150461287758, 811911796452247216, 261732718651120582, 879660044379391844, 209579026317407881, 802303796192799541, 558248951314970078, 768659912298963167, 929587356674553680, 469700570914310637, 789062749853070988, 860025318882886970, 508200384219223082, 777879507677340677, 592288466761680293, 229534785594847358, 355038100652596481|
|practice_0176|independent|128|0|297|312|65|362|9.16|105280150461287758, 106871143279643138, 145038212348708135, 193270389637861274, 195409765175471518, 199528619547510510, 209579026317407881, 229534785594847358, 261732718651120582, 355038100652596481, 365477786511568456, 421069163879798217, 449505030529960947, 469700570914310637, 494485127540215482, 508200384219223082, 512847424116947380, 558248951314970078, 592288466761680293, 623243467656180578, 671323428265898708, 752607559260641172, 768659912298963167, 777879507677340677, 789062749853070988, 802303796192799541, 811911796452247216, 860025318882886970, 879660044379391844, 929587356674553680|
|practice_0176|independent|128|1|287|332|30|317|38.54|469700570914310637, 105280150461287758, 209579026317407881, 145038212348708135, 199528619547510510, 106871143279643138, 193270389637861274, 195409765175471518, 229534785594847358, 355038100652596481, 365477786511568456, 421069163879798217, 512847424116947380, 261732718651120582, 449505030529960947, 494485127540215482, 508200384219223082, 592288466761680293, 929587356674553680, 860025318882886970, 558248951314970078, 752607559260641172, 802303796192799541, 768659912298963167, 671323428265898708, 777879507677340677, 789062749853070988, 623243467656180578, 811911796452247216, 879660044379391844|
|practice_0176|independent|128|2|275|280|25|300|295.61|195409765175471518, 199528619547510510, 106871143279643138, 105280150461287758, 355038100652596481, 261732718651120582, 777879507677340677, 209579026317407881, 671323428265898708, 365477786511568456, 494485127540215482, 512847424116947380, 421069163879798217, 449505030529960947, 469700570914310637, 802303796192799541, 768659912298963167, 592288466761680293, 558248951314970078, 145038212348708135, 229534785594847358, 752607559260641172, 508200384219223082, 789062749853070988, 879660044379391844, 811911796452247216, 193270389637861274, 860025318882886970, 623243467656180578, 929587356674553680|
|practice_0177|mixed|0|0|179|0|0|179|2.45|18076319, 26273011, 53202514, 79796782, 153489304, 506086891, 1684421060, 5898109806, 16261069603, 16263656652, 38240201572, 105781797681, 112283683930, 113973106291, 124485151827, 208566153062, 224663742526, 226542767606, 227678184048, 286901029002, 288584622880, 347270775403, 359549184789, 412844726394, 470592217262, 485694910309, 707519825769, 778746749511|
|practice_0177|mixed|0|1|177|0|0|177|5.68|18076319, 26273011, 53202514, 79796782, 153489304, 506086891, 1684421060, 16263656652, 16261069603, 38240201572, 112283683930, 5898109806, 113973106291, 105781797681, 124485151827, 208566153062, 226542767606, 224663742526, 227678184048, 286901029002, 288584622880, 359549184789, 347270775403, 412844726394, 470592217262, 485694910309, 707519825769, 778746749511|
|practice_0177|mixed|0|2|171|0|0|171|40.28|26273011, 79796782, 153489304, 18076319, 506086891, 5898109806, 53202514, 16261069603, 16263656652, 347270775403, 1684421060, 112283683930, 113973106291, 470592217262, 38240201572, 224663742526, 288584622880, 208566153062, 105781797681, 485694910309, 286901029002, 226542767606, 227678184048, 778746749511, 412844726394, 124485151827, 707519825769, 359549184789|
|practice_0177|mixed|48|0|188|209|0|188|1.07|18076319, 26273011, 53202514, 79796782, 153489304, 506086891, 1684421060, 5898109806, 16261069603, 16263656652, 38240201572, 105781797681, 112283683930, 113973106291, 124485151827, 208566153062, 224663742526, 226542767606, 227678184048, 286901029002, 288584622880, 347270775403, 359549184789, 412844726394, 470592217262, 485694910309, 707519825769, 778746749511|
|practice_0177|mixed|48|1|187|215|0|187|2.26|18076319, 26273011, 53202514, 79796782, 506086891, 153489304, 5898109806, 1684421060, 16261069603, 16263656652, 38240201572, 112283683930, 113973106291, 105781797681, 227678184048, 412844726394, 286901029002, 224663742526, 226542767606, 208566153062, 124485151827, 485694910309, 359549184789, 288584622880, 707519825769, 778746749511, 347270775403, 470592217262|
|practice_0177|mixed|48|2|189|193|0|189|19.08|26273011, 18076319, 53202514, 153489304, 16263656652, 113973106291, 347270775403, 5898109806, 208566153062, 79796782, 16261069603, 112283683930, 506086891, 38240201572, 1684421060, 105781797681, 226542767606, 359549184789, 227678184048, 470592217262, 286901029002, 707519825769, 124485151827, 412844726394, 288584622880, 485694910309, 224663742526, 778746749511|
|practice_0177|mixed|128|0|172|178|43|215|4.68|18076319, 26273011, 53202514, 79796782, 153489304, 506086891, 1684421060, 5898109806, 16261069603, 16263656652, 38240201572, 105781797681, 112283683930, 113973106291, 124485151827, 208566153062, 224663742526, 226542767606, 227678184048, 286901029002, 288584622880, 347270775403, 359549184789, 412844726394, 470592217262, 485694910309, 707519825769, 778746749511|
|practice_0177|mixed|128|1|166|170|0|166|10.13|18076319, 26273011, 53202514, 79796782, 506086891, 153489304, 1684421060, 5898109806, 16261069603, 16263656652, 112283683930, 38240201572, 113973106291, 227678184048, 105781797681, 347270775403, 224663742526, 208566153062, 124485151827, 286901029002, 288584622880, 226542767606, 412844726394, 359549184789, 470592217262, 485694910309, 707519825769, 778746749511|
|practice_0177|mixed|128|2|168|169|29|197|83.51|26273011, 18076319, 53202514, 153489304, 79796782, 16263656652, 347270775403, 506086891, 1684421060, 16261069603, 5898109806, 112283683930, 227678184048, 226542767606, 38240201572, 113973106291, 124485151827, 286901029002, 224663742526, 288584622880, 359549184789, 208566153062, 470592217262, 105781797681, 707519825769, 412844726394, 485694910309, 778746749511|
|practice_0178|multiplicative|0|0|93|0|0|93|1.79|667, 1397, 2660, 4134, 16593, 32329, 32342, 34039, 34084, 37643, 42535, 59549, 123445, 212728, 225872, 280357, 346101, 389864, 510520, 1355256, 1807083, 2144772, 3274992, 6132060, 49780584, 69705642, 174602856, 1659369600|
|practice_0178|multiplicative|0|1|94|0|0|94|2.12|667, 1397, 2660, 4134, 16593, 32329, 32342, 34039, 34084, 37643, 42535, 59549, 225872, 123445, 212728, 280357, 346101, 389864, 510520, 1355256, 1807083, 2144772, 3274992, 6132060, 49780584, 69705642, 174602856, 1659369600|
|practice_0178|multiplicative|0|2|96|0|0|96|13.80|32329, 16593, 667, 34039, 32342, 37643, 1397, 34084, 42535, 2660, 4134, 59549, 1807083, 212728, 123445, 225872, 280357, 346101, 389864, 510520, 1355256, 2144772, 3274992, 6132060, 49780584, 69705642, 174602856, 1659369600|
|practice_0178|multiplicative|48|0|94|102|0|94|0.80|667, 1397, 2660, 4134, 16593, 32329, 32342, 34039, 34084, 37643, 42535, 59549, 123445, 212728, 225872, 280357, 346101, 389864, 510520, 1355256, 1807083, 2144772, 3274992, 6132060, 49780584, 69705642, 174602856, 1659369600|
|practice_0178|multiplicative|48|1|93|115|0|93|1.11|667, 1397, 2660, 4134, 16593, 32329, 32342, 34039, 34084, 37643, 42535, 59549, 225872, 123445, 212728, 280357, 346101, 389864, 510520, 1355256, 2144772, 3274992, 6132060, 1807083, 49780584, 69705642, 1659369600, 174602856|
|practice_0178|multiplicative|48|2|94|98|0|94|7.87|32329, 16593, 667, 34039, 32342, 37643, 2660, 1397, 34084, 42535, 1807083, 4134, 59549, 280357, 123445, 346101, 212728, 225872, 510520, 389864, 2144772, 69705642, 3274992, 1355256, 6132060, 49780584, 174602856, 1659369600|
|practice_0178|multiplicative|128|0|95|98|0|95|2.14|667, 1397, 2660, 4134, 16593, 32329, 32342, 34039, 34084, 37643, 42535, 59549, 123445, 212728, 225872, 280357, 346101, 389864, 510520, 1355256, 1807083, 2144772, 3274992, 6132060, 49780584, 69705642, 174602856, 1659369600|
|practice_0178|multiplicative|128|1|93|97|0|93|2.97|667, 1397, 2660, 4134, 16593, 32329, 32342, 34039, 34084, 37643, 42535, 59549, 225872, 123445, 212728, 280357, 346101, 389864, 510520, 1355256, 1807083, 2144772, 3274992, 6132060, 49780584, 69705642, 174602856, 1659369600|
|practice_0178|multiplicative|128|2|93|95|0|93|17.88|32329, 16593, 667, 34039, 32342, 37643, 2660, 1397, 123445, 42535, 212728, 4134, 34084, 59549, 1807083, 346101, 225872, 280357, 389864, 1355256, 510520, 2144772, 3274992, 6132060, 49780584, 69705642, 174602856, 1659369600|
|practice_0179|independent|0|0|342|0|0|342|4.05|101365548412351182, 154763179100593987, 206490173466340054, 228687122727270486, 235370275999927823, 270281744853929697, 337293441466679895, 346196826960208996, 349566780439568727, 370489054040967369, 373362050825167132, 391388003822650766, 401180794243619375, 422431597642192314, 434513983494371408, 476529286207295488, 527001047561150053, 558578924358814141, 596314329763881662, 606757782097573194, 678065131953322204, 686581515109531259, 714990940963585267, 718920040312605251, 750734232377981013, 759764243641524717, 811507577324896930, 815690789500205270, 833059716518573809, 862642086109042398, 910083540825701073, 934919511769657464|
|practice_0179|independent|0|1|320|0|98|418|18.01|476529286207295488, 101365548412351182, 910083540825701073, 235370275999927823, 337293441466679895, 686581515109531259, 270281744853929697, 206490173466340054, 228687122727270486, 373362050825167132, 349566780439568727, 370489054040967369, 422431597642192314, 346196826960208996, 154763179100593987, 434513983494371408, 558578924358814141, 934919511769657464, 391388003822650766, 759764243641524717, 678065131953322204, 718920040312605251, 714990940963585267, 750734232377981013, 606757782097573194, 401180794243619375, 815690789500205270, 527001047561150053, 811507577324896930, 596314329763881662, 833059716518573809, 862642086109042398|
|practice_0179|independent|0|2|315|0|206|521|121.49|476529286207295488, 270281744853929697, 686581515109531259, 714990940963585267, 718920040312605251, 228687122727270486, 391388003822650766, 422431597642192314, 811507577324896930, 759764243641524717, 606757782097573194, 373362050825167132, 678065131953322204, 346196826960208996, 206490173466340054, 401180794243619375, 337293441466679895, 235370275999927823, 862642086109042398, 101365548412351182, 558578924358814141, 434513983494371408, 154763179100593987, 750734232377981013, 370489054040967369, 910083540825701073, 815690789500205270, 934919511769657464, 833059716518573809, 349566780439568727, 527001047561150053, 596314329763881662|
|practice_0179|independent|48|0|358|681|0|358|2.04|101365548412351182, 154763179100593987, 206490173466340054, 228687122727270486, 235370275999927823, 270281744853929697, 337293441466679895, 346196826960208996, 349566780439568727, 370489054040967369, 373362050825167132, 391388003822650766, 401180794243619375, 422431597642192314, 434513983494371408, 476529286207295488, 527001047561150053, 558578924358814141, 596314329763881662, 606757782097573194, 678065131953322204, 686581515109531259, 714990940963585267, 718920040312605251, 750734232377981013, 759764243641524717, 811507577324896930, 815690789500205270, 833059716518573809, 862642086109042398, 910083540825701073, 934919511769657464|
|practice_0179|independent|48|1|346|383|0|346|7.30|476529286207295488, 101365548412351182, 678065131953322204, 270281744853929697, 910083540825701073, 228687122727270486, 154763179100593987, 235370275999927823, 206490173466340054, 401180794243619375, 346196826960208996, 714990940963585267, 337293441466679895, 349566780439568727, 815690789500205270, 558578924358814141, 373362050825167132, 527001047561150053, 391388003822650766, 759764243641524717, 422431597642192314, 862642086109042398, 596314329763881662, 370489054040967369, 606757782097573194, 718920040312605251, 686581515109531259, 434513983494371408, 934919511769657464, 811507577324896930, 833059716518573809, 750734232377981013|
|practice_0179|independent|48|2|345|370|0|345|46.92|476529286207295488, 391388003822650766, 678065131953322204, 422431597642192314, 235370275999927823, 527001047561150053, 718920040312605251, 101365548412351182, 349566780439568727, 228687122727270486, 558578924358814141, 154763179100593987, 373362050825167132, 596314329763881662, 714990940963585267, 346196826960208996, 815690789500205270, 434513983494371408, 606757782097573194, 206490173466340054, 862642086109042398, 270281744853929697, 811507577324896930, 401180794243619375, 934919511769657464, 686581515109531259, 750734232377981013, 337293441466679895, 759764243641524717, 833059716518573809, 910083540825701073, 370489054040967369|
|practice_0179|independent|128|0|308|337|31|339|11.92|101365548412351182, 154763179100593987, 206490173466340054, 228687122727270486, 235370275999927823, 270281744853929697, 337293441466679895, 346196826960208996, 349566780439568727, 370489054040967369, 373362050825167132, 391388003822650766, 401180794243619375, 422431597642192314, 434513983494371408, 476529286207295488, 527001047561150053, 558578924358814141, 596314329763881662, 606757782097573194, 678065131953322204, 686581515109531259, 714990940963585267, 718920040312605251, 750734232377981013, 759764243641524717, 811507577324896930, 815690789500205270, 833059716518573809, 862642086109042398, 910083540825701073, 934919511769657464|
|practice_0179|independent|128|1|290|294|36|326|53.83|476529286207295488, 101365548412351182, 270281744853929697, 206490173466340054, 349566780439568727, 678065131953322204, 718920040312605251, 154763179100593987, 228687122727270486, 346196826960208996, 686581515109531259, 337293441466679895, 558578924358814141, 401180794243619375, 434513983494371408, 370489054040967369, 596314329763881662, 235370275999927823, 750734232377981013, 391388003822650766, 422431597642192314, 527001047561150053, 714990940963585267, 606757782097573194, 833059716518573809, 373362050825167132, 910083540825701073, 759764243641524717, 815690789500205270, 811507577324896930, 934919511769657464, 862642086109042398|
|practice_0179|independent|128|2|293|298|26|319|356.03|270281744853929697, 228687122727270486, 476529286207295488, 337293441466679895, 373362050825167132, 401180794243619375, 422431597642192314, 349566780439568727, 678065131953322204, 154763179100593987, 346196826960208996, 759764243641524717, 101365548412351182, 206490173466340054, 606757782097573194, 434513983494371408, 370489054040967369, 235370275999927823, 815690789500205270, 527001047561150053, 558578924358814141, 718920040312605251, 686581515109531259, 391388003822650766, 596314329763881662, 811507577324896930, 833059716518573809, 750734232377981013, 714990940963585267, 862642086109042398, 934919511769657464, 910083540825701073|
|practice_0180|multiplicative|0|0|155|0|0|155|3.46|361767, 538506, 1448580, 1449122, 1801199, 1801789, 2897324, 3266199, 5794231, 5998122, 11596323, 15134400, 34680889, 43188534, 52140024, 52148108, 80974943, 157928054, 211491879, 262716495, 485727705, 1898379989, 2104408200, 2825722535, 324501109482, 409349137536, 511426432128, 661588992090, 923188578840, 4103517720948, 8918876545920|
|practice_0180|multiplicative|0|1|151|0|0|151|7.31|361767, 1449122, 1448580, 2897324, 5794231, 52140024, 11596323, 52148108, 538506, 15134400, 1801199, 1801789, 3266199, 5998122, 34680889, 80974943, 43188534, 262716495, 157928054, 485727705, 211491879, 1898379989, 2104408200, 2825722535, 409349137536, 661588992090, 324501109482, 511426432128, 923188578840, 4103517720948, 8918876545920|
|practice_0180|multiplicative|0|2|148|0|0|148|38.45|1448580, 1801199, 1801789, 5794231, 52148108, 211491879, 1449122, 52140024, 5998122, 2897324, 538506, 361767, 80974943, 3266199, 11596323, 15134400, 485727705, 157928054, 43188534, 262716495, 34680889, 409349137536, 2104408200, 1898379989, 923188578840, 2825722535, 661588992090, 324501109482, 511426432128, 8918876545920, 4103517720948|
|practice_0180|multiplicative|48|0|160|195|0|160|1.36|361767, 538506, 1448580, 1449122, 1801199, 1801789, 2897324, 3266199, 5794231, 5998122, 11596323, 15134400, 34680889, 43188534, 52140024, 52148108, 80974943, 157928054, 211491879, 262716495, 485727705, 1898379989, 2104408200, 2825722535, 324501109482, 409349137536, 511426432128, 661588992090, 923188578840, 4103517720948, 8918876545920|
|practice_0180|multiplicative|48|1|156|165|0|156|3.03|361767, 538506, 1448580, 1449122, 2897324, 5794231, 52148108, 52140024, 1801199, 1801789, 3266199, 5998122, 15134400, 211491879, 80974943, 34680889, 11596323, 43188534, 157928054, 262716495, 485727705, 2104408200, 409349137536, 1898379989, 2825722535, 324501109482, 511426432128, 661588992090, 923188578840, 8918876545920, 4103517720948|
|practice_0180|multiplicative|48|2|155|167|0|155|18.95|1448580, 361767, 1801789, 2897324, 1449122, 538506, 5794231, 52148108, 5998122, 52140024, 11596323, 15134400, 1801199, 3266199, 157928054, 1898379989, 43188534, 211491879, 511426432128, 262716495, 34680889, 80974943, 2825722535, 2104408200, 409349137536, 485727705, 324501109482, 923188578840, 8918876545920, 661588992090, 4103517720948|
|practice_0180|multiplicative|128|0|145|149|40|185|5.16|361767, 538506, 1448580, 1449122, 1801199, 1801789, 2897324, 3266199, 5794231, 5998122, 11596323, 15134400, 34680889, 43188534, 52140024, 52148108, 80974943, 157928054, 211491879, 262716495, 485727705, 1898379989, 2104408200, 2825722535, 324501109482, 409349137536, 511426432128, 661588992090, 923188578840, 4103517720948, 8918876545920|
|practice_0180|multiplicative|128|1|146|150|39|185|10.23|361767, 538506, 1448580, 1449122, 2897324, 5794231, 52148108, 52140024, 1801199, 1801789, 5998122, 3266199, 11596323, 15134400, 34680889, 80974943, 43188534, 157928054, 211491879, 262716495, 485727705, 1898379989, 2104408200, 2825722535, 324501109482, 409349137536, 511426432128, 661588992090, 923188578840, 4103517720948, 8918876545920|
|practice_0180|multiplicative|128|2|143|146|26|169|63.95|1448580, 361767, 1801789, 2897324, 1449122, 5794231, 52148108, 1801199, 538506, 5998122, 211491879, 3266199, 52140024, 157928054, 80974943, 11596323, 43188534, 485727705, 15134400, 34680889, 2104408200, 2825722535, 262716495, 324501109482, 409349137536, 661588992090, 8918876545920, 1898379989, 511426432128, 923188578840, 4103517720948|
|practice_0181|independent|0|0|281|0|0|281|3.00|168976434426031, 180882684015183, 187685900572100, 193035964721473, 210646715967013, 211571345817361, 304313550924739, 331473947784336, 389154974497619, 398280235105160, 411772023869177, 412891216722008, 447315716664415, 481080769504069, 558253850837932, 608356226339265, 629354968020019, 654025300417150, 663579941488831, 688599686345799, 726403742840221, 806830395265575, 817728086487864, 828691257260985, 838400457402484, 873590453481353, 874882096672526, 877513852251227, 925922011323929, 932491728031672, 944951224682090|
|practice_0181|independent|0|1|264|0|108|372|11.24|629354968020019, 168976434426031, 187685900572100, 412891216722008, 411772023869177, 180882684015183, 806830395265575, 828691257260985, 193035964721473, 210646715967013, 331473947784336, 481080769504069, 663579941488831, 877513852251227, 817728086487864, 211571345817361, 398280235105160, 304313550924739, 873590453481353, 874882096672526, 925922011323929, 838400457402484, 389154974497619, 688599686345799, 608356226339265, 447315716664415, 558253850837932, 654025300417150, 932491728031672, 726403742840221, 944951224682090|
|practice_0181|independent|0|2|255|0|144|399|72.41|838400457402484, 168976434426031, 398280235105160, 304313550924739, 180882684015183, 817728086487864, 412891216722008, 828691257260985, 210646715967013, 877513852251227, 193035964721473, 331473947784336, 447315716664415, 211571345817361, 944951224682090, 411772023869177, 629354968020019, 873590453481353, 726403742840221, 187685900572100, 806830395265575, 925922011323929, 654025300417150, 663579941488831, 608356226339265, 932491728031672, 481080769504069, 688599686345799, 558253850837932, 389154974497619, 874882096672526|
|practice_0181|independent|48|0|293|390|0|293|1.43|168976434426031, 180882684015183, 187685900572100, 193035964721473, 210646715967013, 211571345817361, 304313550924739, 331473947784336, 389154974497619, 398280235105160, 411772023869177, 412891216722008, 447315716664415, 481080769504069, 558253850837932, 608356226339265, 629354968020019, 654025300417150, 663579941488831, 688599686345799, 726403742840221, 806830395265575, 817728086487864, 828691257260985, 838400457402484, 873590453481353, 874882096672526, 877513852251227, 925922011323929, 932491728031672, 944951224682090|
|practice_0181|independent|48|1|285|290|0|285|4.81|629354968020019, 168976434426031, 187685900572100, 412891216722008, 411772023869177, 193035964721473, 726403742840221, 389154974497619, 331473947784336, 180882684015183, 211571345817361, 481080769504069, 398280235105160, 447315716664415, 210646715967013, 558253850837932, 608356226339265, 838400457402484, 944951224682090, 663579941488831, 654025300417150, 304313550924739, 806830395265575, 873590453481353, 828691257260985, 688599686345799, 932491728031672, 874882096672526, 877513852251227, 925922011323929, 817728086487864|
|practice_0181|independent|48|2|287|296|0|287|35.40|838400457402484, 193035964721473, 210646715967013, 304313550924739, 187685900572100, 398280235105160, 331473947784336, 412891216722008, 389154974497619, 411772023869177, 663579941488831, 806830395265575, 817728086487864, 558253850837932, 828691257260985, 726403742840221, 873590453481353, 629354968020019, 447315716664415, 874882096672526, 481080769504069, 932491728031672, 944951224682090, 211571345817361, 180882684015183, 168976434426031, 877513852251227, 654025300417150, 925922011323929, 608356226339265, 688599686345799|
|practice_0181|independent|128|0|260|262|0|260|7.92|168976434426031, 180882684015183, 187685900572100, 193035964721473, 210646715967013, 211571345817361, 304313550924739, 331473947784336, 389154974497619, 398280235105160, 411772023869177, 412891216722008, 447315716664415, 481080769504069, 558253850837932, 608356226339265, 629354968020019, 654025300417150, 663579941488831, 688599686345799, 726403742840221, 806830395265575, 817728086487864, 828691257260985, 838400457402484, 873590453481353, 874882096672526, 877513852251227, 925922011323929, 932491728031672, 944951224682090|
|practice_0181|independent|128|1|246|249|18|264|32.47|629354968020019, 168976434426031, 187685900572100, 412891216722008, 411772023869177, 180882684015183, 806830395265575, 817728086487864, 210646715967013, 211571345817361, 447315716664415, 331473947784336, 558253850837932, 193035964721473, 304313550924739, 389154974497619, 398280235105160, 663579941488831, 688599686345799, 654025300417150, 608356226339265, 828691257260985, 481080769504069, 838400457402484, 726403742840221, 877513852251227, 925922011323929, 873590453481353, 874882096672526, 932491728031672, 944951224682090|
|practice_0181|independent|128|2|239|243|18|257|177.21|838400457402484, 168976434426031, 398280235105160, 304313550924739, 180882684015183, 558253850837932, 688599686345799, 412891216722008, 187685900572100, 193035964721473, 389154974497619, 608356226339265, 874882096672526, 211571345817361, 932491728031672, 629354968020019, 210646715967013, 873590453481353, 828691257260985, 447315716664415, 726403742840221, 411772023869177, 331473947784336, 877513852251227, 654025300417150, 806830395265575, 481080769504069, 944951224682090, 925922011323929, 663579941488831, 817728086487864|
|practice_0182|mixed|0|0|220|0|0|220|3.10|2242212112, 3208666198, 3208666392, 6670968962, 18667540078, 27056457170, 45390310876, 82650395640, 116107988131, 252011778174, 852883226496, 2275755850791, 2305441157396, 3397492254026, 270832182212655, 300325973054222, 354620346528232, 371329442091396, 405077402515214, 489567374108294, 507228210022806, 555930623620778, 572730235577967, 628326508515550, 723109468191104, 733065091142443, 755552156016049, 953147153715086|
|practice_0182|mixed|0|1|212|0|0|212|9.46|2242212112, 3208666198, 3208666392, 6670968962, 18667540078, 2305441157396, 252011778174, 3397492254026, 27056457170, 45390310876, 82650395640, 116107988131, 852883226496, 2275755850791, 300325973054222, 354620346528232, 489567374108294, 507228210022806, 270832182212655, 405077402515214, 755552156016049, 733065091142443, 371329442091396, 555930623620778, 723109468191104, 572730235577967, 628326508515550, 953147153715086|
|practice_0182|mixed|0|2|211|0|0|211|64.51|18667540078, 3208666392, 2305441157396, 27056457170, 3208666198, 6670968962, 82650395640, 2242212112, 852883226496, 252011778174, 2275755850791, 723109468191104, 45390310876, 354620346528232, 3397492254026, 405077402515214, 116107988131, 371329442091396, 270832182212655, 507228210022806, 572730235577967, 489567374108294, 555930623620778, 300325973054222, 755552156016049, 733065091142443, 628326508515550, 953147153715086|
|practice_0182|mixed|48|0|222|346|0|222|1.62|2242212112, 3208666198, 3208666392, 6670968962, 18667540078, 27056457170, 45390310876, 82650395640, 116107988131, 252011778174, 852883226496, 2275755850791, 2305441157396, 3397492254026, 270832182212655, 300325973054222, 354620346528232, 371329442091396, 405077402515214, 489567374108294, 507228210022806, 555930623620778, 572730235577967, 628326508515550, 723109468191104, 733065091142443, 755552156016049, 953147153715086|
|practice_0182|mixed|48|1|227|235|0|227|4.27|2242212112, 3208666198, 3208666392, 6670968962, 852883226496, 45390310876, 2275755850791, 18667540078, 2305441157396, 27056457170, 252011778174, 3397492254026, 116107988131, 82650395640, 300325973054222, 489567374108294, 405077402515214, 628326508515550, 507228210022806, 371329442091396, 723109468191104, 755552156016049, 555930623620778, 733065091142443, 270832182212655, 354620346528232, 953147153715086, 572730235577967|
|practice_0182|mixed|48|2|223|244|0|223|27.58|18667540078, 3208666198, 6670968962, 3208666392, 2305441157396, 2242212112, 82650395640, 27056457170, 116107988131, 555930623620778, 852883226496, 252011778174, 2275755850791, 270832182212655, 45390310876, 371329442091396, 405077402515214, 507228210022806, 628326508515550, 723109468191104, 572730235577967, 489567374108294, 354620346528232, 755552156016049, 3397492254026, 300325973054222, 733065091142443, 953147153715086|
|practice_0182|mixed|128|0|204|211|7|211|6.29|2242212112, 3208666198, 3208666392, 6670968962, 18667540078, 27056457170, 45390310876, 82650395640, 116107988131, 252011778174, 852883226496, 2275755850791, 2305441157396, 3397492254026, 270832182212655, 300325973054222, 354620346528232, 371329442091396, 405077402515214, 489567374108294, 507228210022806, 555930623620778, 572730235577967, 628326508515550, 723109468191104, 733065091142443, 755552156016049, 953147153715086|
|practice_0182|mixed|128|1|200|202|0|200|19.01|2242212112, 3208666198, 3208666392, 6670968962, 852883226496, 18667540078, 252011778174, 2305441157396, 3397492254026, 116107988131, 27056457170, 45390310876, 82650395640, 2275755850791, 300325973054222, 354620346528232, 270832182212655, 371329442091396, 405077402515214, 555930623620778, 628326508515550, 723109468191104, 733065091142443, 755552156016049, 489567374108294, 507228210022806, 572730235577967, 953147153715086|
|practice_0182|mixed|128|2|194|198|14|208|133.54|18667540078, 3208666198, 6670968962, 252011778174, 3208666392, 82650395640, 2305441157396, 27056457170, 3397492254026, 2242212112, 116107988131, 852883226496, 2275755850791, 507228210022806, 45390310876, 733065091142443, 354620346528232, 371329442091396, 755552156016049, 300325973054222, 628326508515550, 555930623620778, 723109468191104, 405077402515214, 489567374108294, 270832182212655, 572730235577967, 953147153715086|
|practice_0183|mixed|0|0|287|0|0|287|4.40|80429374640, 160858749170, 324285375882, 515268525309, 986744222170, 1242019278529, 6183222304248, 40744759086811, 43723869895976, 176321257954911, 235812717638003, 359498763105408, 457063094633682, 29180837967993675, 51893936991821600, 130197354169487224, 303889950743201647, 343939983301228248, 484619904996210387, 546898263528566077, 554614963800950925, 611597832941239283, 627658405169574298, 791219176594158930, 822885923034341060, 831236728238281441, 904038626273660184, 920741665271519496, 965259995330349553, 976083889603034530|
|practice_0183|mixed|0|1|276|0|82|358|18.50|324285375882, 80429374640, 160858749170, 515268525309, 6183222304248, 986744222170, 1242019278529, 457063094633682, 40744759086811, 43723869895976, 176321257954911, 235812717638003, 359498763105408, 51893936991821600, 343939983301228248, 976083889603034530, 303889950743201647, 29180837967993675, 546898263528566077, 831236728238281441, 130197354169487224, 554614963800950925, 611597832941239283, 791219176594158930, 904038626273660184, 627658405169574298, 822885923034341060, 920741665271519496, 965259995330349553, 484619904996210387|
|practice_0183|mixed|0|2|268|0|41|309|118.29|515268525309, 1242019278529, 160858749170, 359498763105408, 80429374640, 6183222304248, 457063094633682, 324285375882, 986744222170, 176321257954911, 235812717638003, 40744759086811, 611597832941239283, 130197354169487224, 43723869895976, 627658405169574298, 791219176594158930, 51893936991821600, 303889950743201647, 822885923034341060, 343939983301228248, 976083889603034530, 554614963800950925, 920741665271519496, 29180837967993675, 904038626273660184, 831236728238281441, 484619904996210387, 965259995330349553, 546898263528566077|
|practice_0183|mixed|48|0|290|399|0|290|1.97|80429374640, 160858749170, 324285375882, 515268525309, 986744222170, 1242019278529, 6183222304248, 40744759086811, 43723869895976, 176321257954911, 235812717638003, 359498763105408, 457063094633682, 29180837967993675, 51893936991821600, 130197354169487224, 303889950743201647, 343939983301228248, 484619904996210387, 546898263528566077, 554614963800950925, 611597832941239283, 627658405169574298, 791219176594158930, 822885923034341060, 831236728238281441, 904038626273660184, 920741665271519496, 965259995330349553, 976083889603034530|
|practice_0183|mixed|48|1|283|295|0|283|6.51|324285375882, 80429374640, 160858749170, 986744222170, 6183222304248, 515268525309, 1242019278529, 457063094633682, 176321257954911, 359498763105408, 235812717638003, 43723869895976, 40744759086811, 51893936991821600, 554614963800950925, 29180837967993675, 303889950743201647, 822885923034341060, 343939983301228248, 831236728238281441, 484619904996210387, 546898263528566077, 130197354169487224, 611597832941239283, 627658405169574298, 791219176594158930, 976083889603034530, 965259995330349553, 904038626273660184, 920741665271519496|
|practice_0183|mixed|48|2|286|318|0|286|39.51|80429374640, 6183222304248, 324285375882, 515268525309, 176321257954911, 43723869895976, 1242019278529, 986744222170, 40744759086811, 51893936991821600, 554614963800950925, 160858749170, 343939983301228248, 29180837967993675, 976083889603034530, 457063094633682, 235812717638003, 627658405169574298, 359498763105408, 831236728238281441, 546898263528566077, 920741665271519496, 130197354169487224, 484619904996210387, 822885923034341060, 791219176594158930, 303889950743201647, 611597832941239283, 965259995330349553, 904038626273660184|
|practice_0183|mixed|128|0|252|262|42|294|9.57|80429374640, 160858749170, 324285375882, 515268525309, 986744222170, 1242019278529, 6183222304248, 40744759086811, 43723869895976, 176321257954911, 235812717638003, 359498763105408, 457063094633682, 29180837967993675, 51893936991821600, 130197354169487224, 303889950743201647, 343939983301228248, 484619904996210387, 546898263528566077, 554614963800950925, 611597832941239283, 627658405169574298, 791219176594158930, 822885923034341060, 831236728238281441, 904038626273660184, 920741665271519496, 965259995330349553, 976083889603034530|
|practice_0183|mixed|128|1|244|261|28|272|31.29|324285375882, 80429374640, 160858749170, 515268525309, 6183222304248, 986744222170, 1242019278529, 457063094633682, 359498763105408, 43723869895976, 40744759086811, 176321257954911, 235812717638003, 51893936991821600, 130197354169487224, 484619904996210387, 29180837967993675, 343939983301228248, 546898263528566077, 554614963800950925, 611597832941239283, 831236728238281441, 920741665271519496, 303889950743201647, 791219176594158930, 822885923034341060, 627658405169574298, 965259995330349553, 976083889603034530, 904038626273660184|
|practice_0183|mixed|128|2|247|251|41|288|237.34|80429374640, 6183222304248, 1242019278529, 160858749170, 324285375882, 457063094633682, 515268525309, 40744759086811, 43723869895976, 359498763105408, 986744222170, 51893936991821600, 176321257954911, 343939983301228248, 130197354169487224, 29180837967993675, 303889950743201647, 554614963800950925, 235812717638003, 791219176594158930, 611597832941239283, 484619904996210387, 904038626273660184, 627658405169574298, 965259995330349553, 546898263528566077, 920741665271519496, 976083889603034530, 822885923034341060, 831236728238281441|
|practice_0184|mixed|0|0|256|0|0|256|3.45|947785254, 1713664503, 2619857802, 2833295484, 2833296004, 3957095924, 6542472615, 7241838810, 99841056832, 244466024040, 679990922165, 4365773250245, 5971000129230, 8529122784964, 19039745943840, 191231248883347, 246063852624987, 255856792923601, 257392442054087, 282252394004968, 327224324269364, 406933866677313, 408762012789823, 448015513666599, 504164377753036, 555132911645971, 566849651513963, 597334133423880, 621743421356862, 724954598954424, 760055144938244, 871229076464132|
|practice_0184|mixed|0|1|238|0|61|299|9.38|99841056832, 2833295484, 2833296004, 679990922165, 947785254, 1713664503, 2619857802, 3957095924, 6542472615, 7241838810, 244466024040, 19039745943840, 4365773250245, 5971000129230, 8529122784964, 246063852624987, 282252394004968, 191231248883347, 504164377753036, 566849651513963, 597334133423880, 621743421356862, 871229076464132, 448015513666599, 327224324269364, 257392442054087, 555132911645971, 255856792923601, 760055144938244, 724954598954424, 408762012789823, 406933866677313|
|practice_0184|mixed|0|2|240|0|35|275|82.23|2833295484, 947785254, 6542472615, 2833296004, 2619857802, 99841056832, 3957095924, 1713664503, 246063852624987, 679990922165, 7241838810, 4365773250245, 19039745943840, 244466024040, 8529122784964, 566849651513963, 5971000129230, 282252394004968, 448015513666599, 408762012789823, 597334133423880, 555132911645971, 621743421356862, 257392442054087, 724954598954424, 406933866677313, 191231248883347, 504164377753036, 327224324269364, 255856792923601, 760055144938244, 871229076464132|
|practice_0184|mixed|48|0|249|350|0|249|1.52|947785254, 1713664503, 2619857802, 2833295484, 2833296004, 3957095924, 6542472615, 7241838810, 99841056832, 244466024040, 679990922165, 4365773250245, 5971000129230, 8529122784964, 19039745943840, 191231248883347, 246063852624987, 255856792923601, 257392442054087, 282252394004968, 327224324269364, 406933866677313, 408762012789823, 448015513666599, 504164377753036, 555132911645971, 566849651513963, 597334133423880, 621743421356862, 724954598954424, 760055144938244, 871229076464132|
|practice_0184|mixed|48|1|249|371|0|249|5.04|99841056832, 2833295484, 2833296004, 947785254, 1713664503, 679990922165, 19039745943840, 2619857802, 3957095924, 6542472615, 7241838810, 244466024040, 191231248883347, 255856792923601, 4365773250245, 8529122784964, 327224324269364, 246063852624987, 5971000129230, 504164377753036, 406933866677313, 621743421356862, 282252394004968, 566849651513963, 448015513666599, 257392442054087, 597334133423880, 408762012789823, 724954598954424, 760055144938244, 555132911645971, 871229076464132|
|practice_0184|mixed|48|2|250|272|0|250|32.77|2833295484, 6542472615, 2619857802, 2833296004, 679990922165, 1713664503, 947785254, 621743421356862, 99841056832, 244466024040, 7241838810, 3957095924, 8529122784964, 257392442054087, 255856792923601, 282252394004968, 760055144938244, 246063852624987, 504164377753036, 408762012789823, 555132911645971, 4365773250245, 191231248883347, 327224324269364, 724954598954424, 448015513666599, 5971000129230, 566849651513963, 597334133423880, 871229076464132, 406933866677313, 19039745943840|
|practice_0184|mixed|128|0|222|229|0|222|6.17|947785254, 1713664503, 2619857802, 2833295484, 2833296004, 3957095924, 6542472615, 7241838810, 99841056832, 244466024040, 679990922165, 4365773250245, 5971000129230, 8529122784964, 19039745943840, 191231248883347, 246063852624987, 255856792923601, 257392442054087, 282252394004968, 327224324269364, 406933866677313, 408762012789823, 448015513666599, 504164377753036, 555132911645971, 566849651513963, 597334133423880, 621743421356862, 724954598954424, 760055144938244, 871229076464132|
|practice_0184|mixed|128|1|219|224|6|225|23.09|99841056832, 2833295484, 2833296004, 947785254, 2619857802, 7241838810, 6542472615, 1713664503, 679990922165, 3957095924, 244466024040, 19039745943840, 4365773250245, 5971000129230, 8529122784964, 191231248883347, 871229076464132, 255856792923601, 257392442054087, 246063852624987, 597334133423880, 327224324269364, 406933866677313, 408762012789823, 555132911645971, 448015513666599, 504164377753036, 566849651513963, 760055144938244, 282252394004968, 621743421356862, 724954598954424|
|practice_0184|mixed|128|2|222|227|8|230|169.00|2833295484, 947785254, 3957095924, 7241838810, 2833296004, 1713664503, 6542472615, 2619857802, 679990922165, 4365773250245, 99841056832, 19039745943840, 255856792923601, 5971000129230, 246063852624987, 8529122784964, 244466024040, 406933866677313, 408762012789823, 191231248883347, 257392442054087, 504164377753036, 566849651513963, 621743421356862, 327224324269364, 760055144938244, 448015513666599, 555132911645971, 597334133423880, 724954598954424, 282252394004968, 871229076464132|
|practice_0185|shared|0|0|177|0|0|177|3.39|30755471, 38275135, 38280391, 52531798, 76870506, 79656638, 79656650, 95322779, 159312712, 584409707, 618256494, 1375631791, 1622923000, 3323098222, 3323120625, 3345565323, 14980977578, 37620346503, 46063964024, 62175567762, 88712260867, 172671741107, 177424531870, 205976843445, 267782230485, 1903874458256, 3681085276475, 38213444335867, 472883057187347, 504019713320964|
|practice_0185|shared|0|1|171|0|0|171|7.73|159312712, 79656638, 79656650, 30755471, 95322779, 76870506, 618256494, 52531798, 3323098222, 38280391, 38275135, 584409707, 3323120625, 1622923000, 267782230485, 1375631791, 3345565323, 62175567762, 14980977578, 46063964024, 1903874458256, 37620346503, 177424531870, 88712260867, 172671741107, 205976843445, 3681085276475, 472883057187347, 38213444335867, 504019713320964|
|practice_0185|shared|0|2|168|0|0|168|55.78|38280391, 76870506, 79656650, 38275135, 79656638, 159312712, 1375631791, 30755471, 3323120625, 3345565323, 1622923000, 267782230485, 584409707, 3323098222, 95322779, 52531798, 618256494, 1903874458256, 88712260867, 177424531870, 46063964024, 14980977578, 37620346503, 205976843445, 3681085276475, 62175567762, 172671741107, 504019713320964, 38213444335867, 472883057187347|
|practice_0185|shared|48|0|189|233|0|189|1.46|30755471, 38275135, 38280391, 52531798, 76870506, 79656638, 79656650, 95322779, 159312712, 584409707, 618256494, 1375631791, 1622923000, 3323098222, 3323120625, 3345565323, 14980977578, 37620346503, 46063964024, 62175567762, 88712260867, 172671741107, 177424531870, 205976843445, 267782230485, 1903874458256, 3681085276475, 38213444335867, 472883057187347, 504019713320964|
|practice_0185|shared|48|1|181|191|0|181|3.46|159312712, 79656638, 79656650, 30755471, 3323098222, 38280391, 38275135, 3323120625, 52531798, 76870506, 95322779, 618256494, 584409707, 3345565323, 1375631791, 177424531870, 1622923000, 37620346503, 46063964024, 14980977578, 62175567762, 88712260867, 267782230485, 172671741107, 205976843445, 1903874458256, 3681085276475, 504019713320964, 38213444335867, 472883057187347|
|practice_0185|shared|48|2|179|188|0|179|23.03|38280391, 76870506, 159312712, 79656650, 38275135, 79656638, 1375631791, 30755471, 584409707, 62175567762, 95322779, 3323098222, 1622923000, 618256494, 88712260867, 3323120625, 37620346503, 177424531870, 267782230485, 52531798, 1903874458256, 14980977578, 3345565323, 472883057187347, 3681085276475, 46063964024, 38213444335867, 172671741107, 205976843445, 504019713320964|
|practice_0185|shared|128|0|167|174|71|238|5.04|30755471, 38275135, 38280391, 52531798, 76870506, 79656638, 79656650, 95322779, 159312712, 584409707, 618256494, 1375631791, 1622923000, 3323098222, 3323120625, 3345565323, 14980977578, 37620346503, 46063964024, 62175567762, 88712260867, 172671741107, 177424531870, 205976843445, 267782230485, 1903874458256, 3681085276475, 38213444335867, 472883057187347, 504019713320964|
|practice_0185|shared|128|1|164|167|0|164|11.76|159312712, 79656638, 79656650, 30755471, 3323098222, 38275135, 38280391, 3323120625, 14980977578, 76870506, 52531798, 95322779, 584409707, 618256494, 1622923000, 267782230485, 1375631791, 3345565323, 62175567762, 37620346503, 46063964024, 172671741107, 88712260867, 177424531870, 1903874458256, 205976843445, 38213444335867, 3681085276475, 472883057187347, 504019713320964|
|practice_0185|shared|128|2|163|167|15|178|85.39|38280391, 76870506, 79656650, 38275135, 79656638, 618256494, 30755471, 159312712, 1622923000, 267782230485, 584409707, 1375631791, 3323098222, 52531798, 3345565323, 95322779, 177424531870, 88712260867, 37620346503, 3323120625, 46063964024, 1903874458256, 14980977578, 62175567762, 205976843445, 38213444335867, 172671741107, 3681085276475, 472883057187347, 504019713320964|
|practice_0186|independent|0|0|118|0|0|118|1.51|111916243653, 141295673122, 192359786047, 194154478998, 233116249155, 268935437326, 302681606070, 344415376536, 550705227817, 619211682640, 758458423387, 941837545850, 953183920456, 969776438323, 980570890393|
|practice_0186|independent|0|1|118|0|0|118|2.84|141295673122, 941837545850, 111916243653, 233116249155, 194154478998, 192359786047, 268935437326, 302681606070, 344415376536, 550705227817, 953183920456, 980570890393, 619211682640, 758458423387, 969776438323|
|practice_0186|independent|0|2|111|0|0|111|14.87|141295673122, 194154478998, 941837545850, 953183920456, 758458423387, 111916243653, 192359786047, 619211682640, 980570890393, 969776438323, 233116249155, 302681606070, 268935437326, 344415376536, 550705227817|
|practice_0186|independent|48|0|118|128|0|118|0.68|111916243653, 141295673122, 192359786047, 194154478998, 233116249155, 268935437326, 302681606070, 344415376536, 550705227817, 619211682640, 758458423387, 941837545850, 953183920456, 969776438323, 980570890393|
|practice_0186|independent|48|1|109|123|0|109|1.30|141295673122, 111916243653, 344415376536, 192359786047, 194154478998, 268935437326, 302681606070, 619211682640, 550705227817, 953183920456, 758458423387, 980570890393, 233116249155, 941837545850, 969776438323|
|practice_0186|independent|48|2|111|124|0|111|7.90|141295673122, 194154478998, 192359786047, 941837545850, 344415376536, 302681606070, 233116249155, 953183920456, 619211682640, 111916243653, 268935437326, 980570890393, 550705227817, 758458423387, 969776438323|
|practice_0186|independent|128|0|112|115|0|112|2.68|111916243653, 141295673122, 192359786047, 194154478998, 233116249155, 268935437326, 302681606070, 344415376536, 550705227817, 619211682640, 758458423387, 941837545850, 953183920456, 969776438323, 980570890393|
|practice_0186|independent|128|1|110|115|0|110|4.58|141295673122, 111916243653, 344415376536, 192359786047, 194154478998, 233116249155, 268935437326, 953183920456, 302681606070, 550705227817, 619211682640, 758458423387, 941837545850, 969776438323, 980570890393|
|practice_0186|independent|128|2|107|110|0|107|22.66|141295673122, 111916243653, 192359786047, 194154478998, 953183920456, 344415376536, 268935437326, 233116249155, 619211682640, 969776438323, 941837545850, 550705227817, 980570890393, 302681606070, 758458423387|
|practice_0187|independent|0|0|218|0|0|218|3.07|126841179221, 141669752406, 152503948574, 169544632676, 210885707724, 245697424705, 305966532529, 365059335722, 392118417572, 407943399316, 435739849249, 521041255277, 572434087382, 575337675207, 577325777601, 672233815740, 706755199265, 709588605521, 790740972654, 832617796518, 846906914529, 872435529648, 908856242821, 909756235197, 916081777803, 927486196291, 928038116459, 962020432988, 980474624918|
|practice_0187|independent|0|1|201|0|63|264|5.63|126841179221, 141669752406, 392118417572, 928038116459, 435739849249, 152503948574, 169544632676, 245697424705, 210885707724, 916081777803, 980474624918, 407943399316, 521041255277, 572434087382, 305966532529, 575337675207, 577325777601, 709588605521, 365059335722, 706755199265, 790740972654, 846906914529, 927486196291, 672233815740, 832617796518, 962020432988, 872435529648, 909756235197, 908856242821|
|practice_0187|independent|0|2|200|0|9|209|49.86|305966532529, 392118417572, 169544632676, 141669752406, 407943399316, 577325777601, 575337675207, 126841179221, 435739849249, 210885707724, 245697424705, 365059335722, 521041255277, 832617796518, 152503948574, 709588605521, 928038116459, 962020432988, 790740972654, 908856242821, 846906914529, 909756235197, 572434087382, 927486196291, 706755199265, 672233815740, 872435529648, 980474624918, 916081777803|
|practice_0187|independent|48|0|212|239|0|212|1.22|126841179221, 141669752406, 152503948574, 169544632676, 210885707724, 245697424705, 305966532529, 365059335722, 392118417572, 407943399316, 435739849249, 521041255277, 572434087382, 575337675207, 577325777601, 672233815740, 706755199265, 709588605521, 790740972654, 832617796518, 846906914529, 872435529648, 908856242821, 909756235197, 916081777803, 927486196291, 928038116459, 962020432988, 980474624918|
|practice_0187|independent|48|1|212|220|0|212|3.44|126841179221, 141669752406, 392118417572, 152503948574, 245697424705, 169544632676, 365059335722, 210885707724, 832617796518, 305966532529, 790740972654, 407943399316, 672233815740, 521041255277, 709588605521, 577325777601, 435739849249, 928038116459, 706755199265, 575337675207, 927486196291, 916081777803, 980474624918, 846906914529, 872435529648, 962020432988, 908856242821, 572434087382, 909756235197|
|practice_0187|independent|48|2|217|223|0|217|21.91|305966532529, 392118417572, 169544632676, 126841179221, 152503948574, 435739849249, 210885707724, 575337675207, 521041255277, 706755199265, 846906914529, 577325777601, 672233815740, 572434087382, 908856242821, 407943399316, 245697424705, 927486196291, 365059335722, 916081777803, 872435529648, 790740972654, 980474624918, 928038116459, 709588605521, 962020432988, 832617796518, 909756235197, 141669752406|
|practice_0187|independent|128|0|201|206|47|248|6.86|126841179221, 141669752406, 152503948574, 169544632676, 210885707724, 245697424705, 305966532529, 365059335722, 392118417572, 407943399316, 435739849249, 521041255277, 572434087382, 575337675207, 577325777601, 672233815740, 706755199265, 709588605521, 790740972654, 832617796518, 846906914529, 872435529648, 908856242821, 909756235197, 916081777803, 927486196291, 928038116459, 962020432988, 980474624918|
|practice_0187|independent|128|1|189|193|1|190|17.83|126841179221, 141669752406, 392118417572, 928038116459, 435739849249, 305966532529, 169544632676, 152503948574, 210885707724, 927486196291, 245697424705, 365059335722, 407943399316, 832617796518, 846906914529, 521041255277, 572434087382, 575337675207, 577325777601, 709588605521, 672233815740, 706755199265, 790740972654, 872435529648, 908856242821, 916081777803, 980474624918, 909756235197, 962020432988|
|practice_0187|independent|128|2|185|188|27|212|119.43|305966532529, 392118417572, 152503948574, 435739849249, 141669752406, 365059335722, 126841179221, 577325777601, 706755199265, 169544632676, 872435529648, 407943399316, 210885707724, 916081777803, 908856242821, 790740972654, 709588605521, 928038116459, 980474624918, 909756235197, 572434087382, 927486196291, 575337675207, 521041255277, 672233815740, 962020432988, 245697424705, 832617796518, 846906914529|
|practice_0188|independent|0|0|224|0|0|224|2.67|120178777495, 129589319159, 131006534682, 166650305454, 182747541484, 200197625501, 237934048222, 253518773235, 254062200681, 359554499899, 360117771562, 385237676639, 471881751669, 515843837582, 590112216519, 667809541678, 718433738564, 730973293858, 756439618427, 759604455987, 768310067715, 772701612865, 810710470762, 839418058242, 868192839954, 871934170019, 909124381722, 911085469949, 917788223327, 953835573549, 966083894617, 997767165730|
|practice_0188|independent|0|1|220|0|0|220|8.12|120178777495, 129589319159, 131006534682, 166650305454, 182747541484, 200197625501, 730973293858, 254062200681, 237934048222, 253518773235, 359554499899, 360117771562, 471881751669, 385237676639, 590112216519, 515843837582, 667809541678, 756439618427, 772701612865, 759604455987, 768310067715, 868192839954, 718433738564, 810710470762, 871934170019, 911085469949, 909124381722, 917788223327, 839418058242, 997767165730, 953835573549, 966083894617|
|practice_0188|independent|0|2|210|0|69|279|58.44|129589319159, 182747541484, 360117771562, 120178777495, 200197625501, 237934048222, 385237676639, 730973293858, 254062200681, 359554499899, 718433738564, 166650305454, 253518773235, 759604455987, 839418058242, 768310067715, 772701612865, 868192839954, 909124381722, 997767165730, 667809541678, 911085469949, 756439618427, 810710470762, 871934170019, 917788223327, 966083894617, 471881751669, 953835573549, 131006534682, 515843837582, 590112216519|
|practice_0188|independent|48|0|233|261|0|233|1.41|120178777495, 129589319159, 131006534682, 166650305454, 182747541484, 200197625501, 237934048222, 253518773235, 254062200681, 359554499899, 360117771562, 385237676639, 471881751669, 515843837582, 590112216519, 667809541678, 718433738564, 730973293858, 756439618427, 759604455987, 768310067715, 772701612865, 810710470762, 839418058242, 868192839954, 871934170019, 909124381722, 911085469949, 917788223327, 953835573549, 966083894617, 997767165730|
|practice_0188|independent|48|1|234|239|0|234|3.47|120178777495, 129589319159, 131006534682, 166650305454, 182747541484, 200197625501, 254062200681, 237934048222, 253518773235, 359554499899, 360117771562, 810710470762, 471881751669, 515843837582, 385237676639, 756439618427, 718433738564, 772701612865, 768310067715, 909124381722, 730973293858, 911085469949, 590112216519, 953835573549, 759604455987, 871934170019, 917788223327, 966083894617, 868192839954, 997767165730, 667809541678, 839418058242|
|practice_0188|independent|48|2|230|234|0|230|31.23|129589319159, 182747541484, 730973293858, 718433738564, 120178777495, 759604455987, 254062200681, 200197625501, 131006534682, 756439618427, 515843837582, 253518773235, 768310067715, 360117771562, 667809541678, 166650305454, 810710470762, 909124381722, 237934048222, 471881751669, 953835573549, 868192839954, 966083894617, 590112216519, 917788223327, 839418058242, 997767165730, 359554499899, 772701612865, 871934170019, 911085469949, 385237676639|
|practice_0188|independent|128|0|212|218|47|259|9.08|120178777495, 129589319159, 131006534682, 166650305454, 182747541484, 200197625501, 237934048222, 253518773235, 254062200681, 359554499899, 360117771562, 385237676639, 471881751669, 515843837582, 590112216519, 667809541678, 718433738564, 730973293858, 756439618427, 759604455987, 768310067715, 772701612865, 810710470762, 839418058242, 868192839954, 871934170019, 909124381722, 911085469949, 917788223327, 953835573549, 966083894617, 997767165730|
|practice_0188|independent|128|1|207|209|11|218|24.37|120178777495, 129589319159, 131006534682, 166650305454, 182747541484, 200197625501, 237934048222, 253518773235, 254062200681, 730973293858, 359554499899, 360117771562, 385237676639, 471881751669, 515843837582, 718433738564, 759604455987, 768310067715, 966083894617, 772701612865, 590112216519, 911085469949, 909124381722, 667809541678, 810710470762, 756439618427, 839418058242, 868192839954, 871934170019, 953835573549, 917788223327, 997767165730|
|practice_0188|independent|128|2|193|196|34|227|151.98|129589319159, 182747541484, 360117771562, 359554499899, 166650305454, 253518773235, 237934048222, 200197625501, 254062200681, 756439618427, 768310067715, 911085469949, 759604455987, 953835573549, 120178777495, 590112216519, 131006534682, 730973293858, 385237676639, 839418058242, 471881751669, 772701612865, 810710470762, 997767165730, 515843837582, 868192839954, 909124381722, 871934170019, 917788223327, 966083894617, 718433738564, 667809541678|
|practice_0189|shared|0|0|187|0|0|187|4.65|46795365, 540466710, 550767728, 550768138, 647106611, 692409091, 692412080, 1772779746, 5945099236, 6435153517, 7616439710, 7616532924, 8149575996, 8647415616, 17883985630, 45297240547, 52462752016, 158310163470, 208382256930, 243728000032, 535999694400, 1035848441584, 1916741494304, 21046786310530, 67955213547583, 138071588278283, 138267756368896, 372124335215580, 409544449933644656, 756444388499078444|
|practice_0189|shared|0|1|186|0|0|186|13.28|692409091, 243728000032, 692412080, 7616532924, 7616439710, 46795365, 540466710, 5945099236, 8647415616, 1035848441584, 550767728, 550768138, 647106611, 1772779746, 8149575996, 6435153517, 158310163470, 17883985630, 52462752016, 45297240547, 535999694400, 1916741494304, 208382256930, 67955213547583, 138267756368896, 21046786310530, 138071588278283, 372124335215580, 409544449933644656, 756444388499078444|
|practice_0189|shared|0|2|182|0|0|182|75.16|692409091, 692412080, 540466710, 243728000032, 8647415616, 5945099236, 550767728, 7616532924, 647106611, 550768138, 46795365, 7616439710, 17883985630, 1035848441584, 8149575996, 1772779746, 6435153517, 45297240547, 1916741494304, 138267756368896, 208382256930, 158310163470, 535999694400, 52462752016, 138071588278283, 67955213547583, 409544449933644656, 21046786310530, 372124335215580, 756444388499078444|
|practice_0189|shared|48|0|208|255|0|208|1.91|46795365, 540466710, 550767728, 550768138, 647106611, 692409091, 692412080, 1772779746, 5945099236, 6435153517, 7616439710, 7616532924, 8149575996, 8647415616, 17883985630, 45297240547, 52462752016, 158310163470, 208382256930, 243728000032, 535999694400, 1035848441584, 1916741494304, 21046786310530, 67955213547583, 138071588278283, 138267756368896, 372124335215580, 409544449933644656, 756444388499078444|
|practice_0189|shared|48|1|191|198|0|191|4.91|692409091, 243728000032, 692412080, 7616532924, 7616439710, 46795365, 540466710, 5945099236, 8647415616, 1035848441584, 550767728, 550768138, 17883985630, 647106611, 1772779746, 6435153517, 8149575996, 158310163470, 208382256930, 45297240547, 52462752016, 535999694400, 138071588278283, 1916741494304, 138267756368896, 21046786310530, 67955213547583, 372124335215580, 409544449933644656, 756444388499078444|
|practice_0189|shared|48|2|193|199|0|193|29.65|692409091, 692412080, 540466710, 7616532924, 243728000032, 46795365, 550767728, 7616439710, 550768138, 1772779746, 647106611, 8647415616, 45297240547, 8149575996, 6435153517, 208382256930, 1916741494304, 535999694400, 17883985630, 67955213547583, 5945099236, 138071588278283, 158310163470, 409544449933644656, 1035848441584, 52462752016, 756444388499078444, 138267756368896, 21046786310530, 372124335215580|
|practice_0189|shared|128|0|182|186|44|226|7.61|46795365, 540466710, 550767728, 550768138, 647106611, 692409091, 692412080, 1772779746, 5945099236, 6435153517, 7616439710, 7616532924, 8149575996, 8647415616, 17883985630, 45297240547, 52462752016, 158310163470, 208382256930, 243728000032, 535999694400, 1035848441584, 1916741494304, 21046786310530, 67955213547583, 138071588278283, 138267756368896, 372124335215580, 409544449933644656, 756444388499078444|
|practice_0189|shared|128|1|174|177|6|180|18.43|692409091, 243728000032, 692412080, 7616532924, 7616439710, 46795365, 540466710, 5945099236, 8647415616, 17883985630, 550767728, 550768138, 1035848441584, 647106611, 1772779746, 6435153517, 8149575996, 52462752016, 45297240547, 158310163470, 208382256930, 138267756368896, 1916741494304, 535999694400, 138071588278283, 67955213547583, 21046786310530, 372124335215580, 409544449933644656, 756444388499078444|
|practice_0189|shared|128|2|171|175|0|171|122.21|692409091, 692412080, 540466710, 7616439710, 550767728, 550768138, 243728000032, 7616532924, 647106611, 46795365, 5945099236, 8647415616, 1035848441584, 1772779746, 535999694400, 17883985630, 6435153517, 8149575996, 52462752016, 158310163470, 138267756368896, 1916741494304, 67955213547583, 138071588278283, 45297240547, 21046786310530, 208382256930, 372124335215580, 409544449933644656, 756444388499078444|
|practice_0190|mixed|0|0|295|0|0|295|4.85|321319127477, 469063807896, 485219514258, 526715306061, 1743991089450, 2489897769341, 3055067773171, 8507150710299, 8733951297984, 56335537939110, 147754864021821, 272228822705952, 863813100580430, 1164115612572532, 180473232707285041, 188750336402031444, 268972324451083911, 275122027104930847, 306059030899686137, 398467922451629174, 433176162207306846, 467381172426686927, 470643461349276533, 512768056677152923, 536332007659516937, 542206378483391675, 753070405673508721, 889385646645123251, 916964174608356770, 953436752559662957, 982865875100419947|
|practice_0190|mixed|0|1|284|0|75|359|19.09|321319127477, 469063807896, 485219514258, 8733951297984, 526715306061, 1743991089450, 2489897769341, 3055067773171, 8507150710299, 272228822705952, 56335537939110, 147754864021821, 863813100580430, 1164115612572532, 306059030899686137, 398467922451629174, 953436752559662957, 542206378483391675, 188750336402031444, 180473232707285041, 268972324451083911, 512768056677152923, 982865875100419947, 433176162207306846, 275122027104930847, 536332007659516937, 467381172426686927, 753070405673508721, 470643461349276533, 889385646645123251, 916964174608356770|
|practice_0190|mixed|0|2|278|0|11|289|126.47|321319127477, 485219514258, 272228822705952, 469063807896, 526715306061, 8733951297984, 1743991089450, 8507150710299, 863813100580430, 2489897769341, 147754864021821, 56335537939110, 3055067773171, 542206378483391675, 275122027104930847, 467381172426686927, 180473232707285041, 953436752559662957, 306059030899686137, 1164115612572532, 268972324451083911, 188750336402031444, 470643461349276533, 536332007659516937, 753070405673508721, 433176162207306846, 916964174608356770, 889385646645123251, 512768056677152923, 398467922451629174, 982865875100419947|
|practice_0190|mixed|48|0|297|485|0|297|1.89|321319127477, 469063807896, 485219514258, 526715306061, 1743991089450, 2489897769341, 3055067773171, 8507150710299, 8733951297984, 56335537939110, 147754864021821, 272228822705952, 863813100580430, 1164115612572532, 180473232707285041, 188750336402031444, 268972324451083911, 275122027104930847, 306059030899686137, 398467922451629174, 433176162207306846, 467381172426686927, 470643461349276533, 512768056677152923, 536332007659516937, 542206378483391675, 753070405673508721, 889385646645123251, 916964174608356770, 953436752559662957, 982865875100419947|
|practice_0190|mixed|48|1|303|315|0|303|5.36|321319127477, 469063807896, 485219514258, 8733951297984, 1743991089450, 526715306061, 2489897769341, 8507150710299, 3055067773171, 272228822705952, 863813100580430, 147754864021821, 56335537939110, 1164115612572532, 180473232707285041, 188750336402031444, 268972324451083911, 433176162207306846, 275122027104930847, 542206378483391675, 536332007659516937, 467381172426686927, 953436752559662957, 398467922451629174, 753070405673508721, 889385646645123251, 916964174608356770, 306059030899686137, 470643461349276533, 512768056677152923, 982865875100419947|
|practice_0190|mixed|48|2|297|315|0|297|38.42|321319127477, 485219514258, 272228822705952, 2489897769341, 8507150710299, 469063807896, 8733951297984, 3055067773171, 147754864021821, 1743991089450, 526715306061, 268972324451083911, 863813100580430, 467381172426686927, 180473232707285041, 188750336402031444, 953436752559662957, 56335537939110, 536332007659516937, 275122027104930847, 542206378483391675, 982865875100419947, 753070405673508721, 1164115612572532, 512768056677152923, 398467922451629174, 889385646645123251, 470643461349276533, 306059030899686137, 433176162207306846, 916964174608356770|
|practice_0190|mixed|128|0|266|275|64|330|8.69|321319127477, 469063807896, 485219514258, 526715306061, 1743991089450, 2489897769341, 3055067773171, 8507150710299, 8733951297984, 56335537939110, 147754864021821, 272228822705952, 863813100580430, 1164115612572532, 180473232707285041, 188750336402031444, 268972324451083911, 275122027104930847, 306059030899686137, 398467922451629174, 433176162207306846, 467381172426686927, 470643461349276533, 512768056677152923, 536332007659516937, 542206378483391675, 753070405673508721, 889385646645123251, 916964174608356770, 953436752559662957, 982865875100419947|
|practice_0190|mixed|128|1|260|264|22|282|35.75|321319127477, 469063807896, 485219514258, 8733951297984, 526715306061, 1743991089450, 2489897769341, 3055067773171, 8507150710299, 272228822705952, 147754864021821, 863813100580430, 56335537939110, 1164115612572532, 306059030899686137, 536332007659516937, 916964174608356770, 398467922451629174, 180473232707285041, 188750336402031444, 268972324451083911, 275122027104930847, 433176162207306846, 512768056677152923, 889385646645123251, 470643461349276533, 467381172426686927, 753070405673508721, 542206378483391675, 953436752559662957, 982865875100419947|
|practice_0190|mixed|128|2|250|255|85|335|246.83|321319127477, 485219514258, 272228822705952, 8507150710299, 8733951297984, 469063807896, 526715306061, 863813100580430, 3055067773171, 1164115612572532, 2489897769341, 147754864021821, 180473232707285041, 1743991089450, 56335537939110, 512768056677152923, 536332007659516937, 268972324451083911, 542206378483391675, 467381172426686927, 916964174608356770, 188750336402031444, 398467922451629174, 433176162207306846, 953436752559662957, 306059030899686137, 275122027104930847, 470643461349276533, 753070405673508721, 982865875100419947, 889385646645123251|
|practice_0191|independent|0|0|313|0|0|313|3.85|101820082912984330, 136806375144442246, 155777593432933436, 182319563625086810, 260368897778667026, 340725213721118157, 348643307029642397, 394373075768923185, 425208900371685237, 432912814033083557, 458751954806588444, 563394527463555286, 564166092075121281, 605799589366170187, 637095762015427238, 649084535119327512, 658159088860784063, 697731484520701374, 741494494697728917, 781132107142841991, 797860241560168683, 845853295456176534, 863749769562227799, 898172804753715088, 903627912701830798, 966528720219226766, 978397760341008103, 983299434622757164|
|practice_0191|independent|0|1|288|0|288|576|12.05|649084535119327512, 136806375144442246, 348643307029642397, 425208900371685237, 605799589366170187, 563394527463555286, 340725213721118157, 394373075768923185, 101820082912984330, 155777593432933436, 564166092075121281, 903627912701830798, 966528720219226766, 432912814033083557, 260368897778667026, 781132107142841991, 658159088860784063, 697731484520701374, 637095762015427238, 983299434622757164, 182319563625086810, 845853295456176534, 863749769562227799, 797860241560168683, 898172804753715088, 458751954806588444, 978397760341008103, 741494494697728917|
|practice_0191|independent|0|2|293|0|255|548|87.89|394373075768923185, 182319563625086810, 432912814033083557, 348643307029642397, 649084535119327512, 697731484520701374, 563394527463555286, 564166092075121281, 863749769562227799, 978397760341008103, 845853295456176534, 136806375144442246, 458751954806588444, 155777593432933436, 260368897778667026, 101820082912984330, 425208900371685237, 781132107142841991, 741494494697728917, 966528720219226766, 637095762015427238, 658159088860784063, 983299434622757164, 898172804753715088, 605799589366170187, 903627912701830798, 797860241560168683, 340725213721118157|
|practice_0191|independent|48|0|322|669|0|322|1.74|101820082912984330, 136806375144442246, 155777593432933436, 182319563625086810, 260368897778667026, 340725213721118157, 348643307029642397, 394373075768923185, 425208900371685237, 432912814033083557, 458751954806588444, 563394527463555286, 564166092075121281, 605799589366170187, 637095762015427238, 649084535119327512, 658159088860784063, 697731484520701374, 741494494697728917, 781132107142841991, 797860241560168683, 845853295456176534, 863749769562227799, 898172804753715088, 903627912701830798, 966528720219226766, 978397760341008103, 983299434622757164|
|practice_0191|independent|48|1|328|388|0|328|4.86|649084535119327512, 136806375144442246, 182319563625086810, 394373075768923185, 898172804753715088, 432912814033083557, 101820082912984330, 348643307029642397, 155777593432933436, 605799589366170187, 458751954806588444, 563394527463555286, 564166092075121281, 697731484520701374, 658159088860784063, 260368897778667026, 983299434622757164, 845853295456176534, 637095762015427238, 781132107142841991, 978397760341008103, 903627912701830798, 966528720219226766, 340725213721118157, 741494494697728917, 797860241560168683, 425208900371685237, 863749769562227799|
|practice_0191|independent|48|2|317|387|0|317|35.44|394373075768923185, 155777593432933436, 136806375144442246, 340725213721118157, 348643307029642397, 425208900371685237, 845853295456176534, 605799589366170187, 563394527463555286, 898172804753715088, 781132107142841991, 637095762015427238, 978397760341008103, 983299434622757164, 101820082912984330, 966528720219226766, 649084535119327512, 797860241560168683, 182319563625086810, 903627912701830798, 260368897778667026, 564166092075121281, 697731484520701374, 458751954806588444, 658159088860784063, 863749769562227799, 741494494697728917, 432912814033083557|
|practice_0191|independent|128|0|280|288|41|321|8.77|101820082912984330, 136806375144442246, 155777593432933436, 182319563625086810, 260368897778667026, 340725213721118157, 348643307029642397, 394373075768923185, 425208900371685237, 432912814033083557, 458751954806588444, 563394527463555286, 564166092075121281, 605799589366170187, 637095762015427238, 649084535119327512, 658159088860784063, 697731484520701374, 741494494697728917, 781132107142841991, 797860241560168683, 845853295456176534, 863749769562227799, 898172804753715088, 903627912701830798, 966528720219226766, 978397760341008103, 983299434622757164|
|practice_0191|independent|128|1|265|269|10|275|38.05|649084535119327512, 136806375144442246, 348643307029642397, 425208900371685237, 605799589366170187, 260368897778667026, 182319563625086810, 340725213721118157, 155777593432933436, 898172804753715088, 432912814033083557, 637095762015427238, 563394527463555286, 101820082912984330, 564166092075121281, 741494494697728917, 903627912701830798, 966528720219226766, 978397760341008103, 394373075768923185, 863749769562227799, 458751954806588444, 658159088860784063, 697731484520701374, 845853295456176534, 797860241560168683, 781132107142841991, 983299434622757164|
|practice_0191|independent|128|2|269|272|7|276|228.94|898172804753715088, 340725213721118157, 260368897778667026, 101820082912984330, 781132107142841991, 564166092075121281, 563394527463555286, 458751954806588444, 136806375144442246, 697731484520701374, 348643307029642397, 182319563625086810, 649084535119327512, 394373075768923185, 432912814033083557, 637095762015427238, 605799589366170187, 903627912701830798, 658159088860784063, 155777593432933436, 863749769562227799, 966528720219226766, 845853295456176534, 978397760341008103, 425208900371685237, 983299434622757164, 741494494697728917, 797860241560168683|
|practice_0192|multiplicative|0|0|125|0|0|125|2.74|850, 10061, 21037, 24325, 24460, 42180, 47074, 55009, 79650, 81138, 88224, 116860, 219190, 352065, 576978, 922222, 1144502, 7396928, 11897349, 12082177, 22201734, 23805018, 30343464, 41691978, 66897632, 133188480, 158694366, 236828160, 518061600, 2410209271, 8549496652|
|practice_0192|multiplicative|0|1|123|0|0|123|3.67|850, 10061, 21037, 7396928, 24325, 24460, 42180, 47074, 79650, 55009, 81138, 88224, 116860, 219190, 352065, 576978, 922222, 1144502, 22201734, 236828160, 518061600, 11897349, 12082177, 23805018, 30343464, 41691978, 66897632, 133188480, 158694366, 2410209271, 8549496652|
|practice_0192|multiplicative|0|2|118|0|0|118|23.21|24325, 24460, 850, 10061, 79650, 21037, 116860, 219190, 55009, 88224, 576978, 42180, 47074, 81138, 133188480, 11897349, 352065, 922222, 236828160, 518061600, 1144502, 7396928, 23805018, 158694366, 12082177, 22201734, 30343464, 8549496652, 41691978, 2410209271, 66897632|
|practice_0192|multiplicative|48|0|122|153|0|122|1.14|850, 10061, 21037, 24325, 24460, 42180, 47074, 55009, 79650, 81138, 88224, 116860, 219190, 352065, 576978, 922222, 1144502, 7396928, 11897349, 12082177, 22201734, 23805018, 30343464, 41691978, 66897632, 133188480, 158694366, 236828160, 518061600, 2410209271, 8549496652|
|practice_0192|multiplicative|48|1|125|132|0|125|1.80|850, 10061, 21037, 7396928, 24325, 24460, 42180, 47074, 79650, 55009, 81138, 88224, 116860, 352065, 219190, 576978, 922222, 1144502, 22201734, 23805018, 11897349, 12082177, 30343464, 41691978, 158694366, 236828160, 518061600, 66897632, 133188480, 2410209271, 8549496652|
|practice_0192|multiplicative|48|2|123|127|0|123|9.78|24325, 24460, 850, 10061, 79650, 21037, 116860, 47074, 55009, 352065, 576978, 219190, 922222, 88224, 81138, 11897349, 23805018, 22201734, 7396928, 41691978, 133188480, 42180, 236828160, 518061600, 12082177, 8549496652, 1144502, 30343464, 158694366, 66897632, 2410209271|
|practice_0192|multiplicative|128|0|123|125|0|123|4.05|850, 10061, 21037, 24325, 24460, 42180, 47074, 55009, 79650, 81138, 88224, 116860, 219190, 352065, 576978, 922222, 1144502, 7396928, 11897349, 12082177, 22201734, 23805018, 30343464, 41691978, 66897632, 133188480, 158694366, 236828160, 518061600, 2410209271, 8549496652|
|practice_0192|multiplicative|128|1|123|124|0|123|5.68|850, 10061, 21037, 7396928, 24325, 24460, 42180, 47074, 79650, 55009, 81138, 88224, 116860, 219190, 352065, 922222, 576978, 1144502, 12082177, 30343464, 66897632, 11897349, 22201734, 23805018, 41691978, 133188480, 158694366, 236828160, 518061600, 2410209271, 8549496652|
|practice_0192|multiplicative|128|2|117|121|0|117|30.69|24325, 24460, 850, 10061, 79650, 21037, 116860, 219190, 55009, 88224, 81138, 352065, 47074, 11897349, 42180, 133188480, 576978, 922222, 1144502, 7396928, 12082177, 22201734, 30343464, 236828160, 66897632, 518061600, 23805018, 41691978, 158694366, 2410209271, 8549496652|
|practice_0193|multiplicative|0|0|141|0|0|141|3.55|35642, 102096, 102105, 141942, 205200, 274913, 320553, 612546, 1099451, 1578593, 1610367, 1837880, 2563516, 3676048, 4838291, 5493234, 6283243, 7873055, 9467960, 9798048, 33839176, 67043712, 72496112, 93751286, 103198212, 906866194, 2810582800, 4865720320, 357168974400, 1125080442400, 3075042549809|
|practice_0193|multiplicative|0|1|134|0|0|134|6.13|35642, 205200, 102096, 102105, 320553, 612546, 1837880, 141942, 1578593, 3676048, 274913, 1099451, 1610367, 5493234, 9798048, 72496112, 2563516, 4838291, 6283243, 7873055, 9467960, 33839176, 67043712, 93751286, 103198212, 4865720320, 2810582800, 906866194, 357168974400, 3075042549809, 1125080442400|
|practice_0193|multiplicative|0|2|128|0|0|128|27.14|102105, 35642, 205200, 274913, 102096, 9798048, 320553, 1837880, 1578593, 1610367, 3676048, 141942, 612546, 67043712, 1099451, 5493234, 6283243, 9467960, 72496112, 2563516, 7873055, 4865720320, 4838291, 93751286, 33839176, 103198212, 2810582800, 906866194, 357168974400, 3075042549809, 1125080442400|
|practice_0193|multiplicative|48|0|144|203|0|144|1.47|35642, 102096, 102105, 141942, 205200, 274913, 320553, 612546, 1099451, 1578593, 1610367, 1837880, 2563516, 3676048, 4838291, 5493234, 6283243, 7873055, 9467960, 9798048, 33839176, 67043712, 72496112, 93751286, 103198212, 906866194, 2810582800, 4865720320, 357168974400, 1125080442400, 3075042549809|
|practice_0193|multiplicative|48|1|136|148|0|136|2.38|35642, 205200, 102096, 102105, 320553, 612546, 1837880, 141942, 1578593, 3676048, 9798048, 72496112, 274913, 1610367, 1099451, 103198212, 2563516, 4838291, 5493234, 6283243, 9467960, 93751286, 33839176, 2810582800, 67043712, 7873055, 906866194, 4865720320, 357168974400, 1125080442400, 3075042549809|
|practice_0193|multiplicative|48|2|135|147|0|135|12.56|102105, 35642, 274913, 102096, 320553, 612546, 205200, 1837880, 141942, 1578593, 3676048, 1610367, 4838291, 5493234, 1099451, 2563516, 9798048, 6283243, 4865720320, 103198212, 9467960, 72496112, 906866194, 93751286, 357168974400, 33839176, 7873055, 3075042549809, 67043712, 2810582800, 1125080442400|
|practice_0193|multiplicative|128|0|133|136|1|134|3.99|35642, 102096, 102105, 141942, 205200, 274913, 320553, 612546, 1099451, 1578593, 1610367, 1837880, 2563516, 3676048, 4838291, 5493234, 6283243, 7873055, 9467960, 9798048, 33839176, 67043712, 72496112, 93751286, 103198212, 906866194, 2810582800, 4865720320, 357168974400, 1125080442400, 3075042549809|
|practice_0193|multiplicative|128|1|128|131|0|128|6.90|35642, 205200, 102096, 102105, 320553, 612546, 1837880, 141942, 1578593, 3676048, 274913, 1099451, 1610367, 5493234, 9798048, 72496112, 2563516, 4838291, 33839176, 6283243, 7873055, 9467960, 103198212, 67043712, 93751286, 906866194, 4865720320, 2810582800, 357168974400, 3075042549809, 1125080442400|
|practice_0193|multiplicative|128|2|126|130|0|126|32.72|102105, 35642, 274913, 102096, 320553, 612546, 1837880, 205200, 1578593, 9798048, 1610367, 5493234, 2563516, 67043712, 1099451, 141942, 3676048, 7873055, 9467960, 103198212, 4838291, 6283243, 2810582800, 72496112, 33839176, 93751286, 906866194, 4865720320, 357168974400, 1125080442400, 3075042549809|
|practice_0194|mixed|0|0|249|0|0|249|3.60|16240740892, 26008713295, 27067904472, 259412740786, 259412742900, 259412743070, 277173952895, 389605665846, 778238228911, 784029889432, 2333921716044, 3434316541665, 4582668765600, 3807408205537232, 319625035512395402, 328459118409053944, 374855517519907790, 385687971678785004, 413956826422147456, 415399148292321536, 460916722833655479, 476876563643924133, 508165376573780056, 513055160567618935, 603299706328452441, 606206348574387586, 809661110308353313, 878188438346647256|
|practice_0194|mixed|0|1|242|0|6|248|12.71|16240740892, 27067904472, 26008713295, 259412742900, 259412740786, 259412743070, 778238228911, 277173952895, 784029889432, 389605665846, 2333921716044, 3434316541665, 4582668765600, 3807408205537232, 328459118409053944, 413956826422147456, 603299706328452441, 415399148292321536, 476876563643924133, 374855517519907790, 319625035512395402, 513055160567618935, 809661110308353313, 508165376573780056, 385687971678785004, 460916722833655479, 606206348574387586, 878188438346647256|
|practice_0194|mixed|0|2|242|0|47|289|97.12|259412742900, 27067904472, 259412740786, 259412743070, 778238228911, 26008713295, 784029889432, 16240740892, 277173952895, 4582668765600, 389605665846, 2333921716044, 415399148292321536, 3434316541665, 319625035512395402, 328459118409053944, 385687971678785004, 413956826422147456, 460916722833655479, 3807408205537232, 476876563643924133, 603299706328452441, 606206348574387586, 809661110308353313, 508165376573780056, 374855517519907790, 513055160567618935, 878188438346647256|
|practice_0194|mixed|48|0|250|356|0|250|1.63|16240740892, 26008713295, 27067904472, 259412740786, 259412742900, 259412743070, 277173952895, 389605665846, 778238228911, 784029889432, 2333921716044, 3434316541665, 4582668765600, 3807408205537232, 319625035512395402, 328459118409053944, 374855517519907790, 385687971678785004, 413956826422147456, 415399148292321536, 460916722833655479, 476876563643924133, 508165376573780056, 513055160567618935, 603299706328452441, 606206348574387586, 809661110308353313, 878188438346647256|
|practice_0194|mixed|48|1|246|283|0|246|3.91|16240740892, 27067904472, 26008713295, 259412740786, 259412743070, 259412742900, 778238228911, 784029889432, 2333921716044, 277173952895, 389605665846, 4582668765600, 3434316541665, 603299706328452441, 385687971678785004, 3807408205537232, 460916722833655479, 319625035512395402, 328459118409053944, 413956826422147456, 374855517519907790, 606206348574387586, 415399148292321536, 508165376573780056, 513055160567618935, 878188438346647256, 809661110308353313, 476876563643924133|
|practice_0194|mixed|48|2|256|274|0|256|31.50|259412742900, 16240740892, 259412743070, 259412740786, 778238228911, 277173952895, 2333921716044, 26008713295, 784029889432, 27067904472, 3434316541665, 385687971678785004, 415399148292321536, 389605665846, 878188438346647256, 374855517519907790, 3807408205537232, 4582668765600, 809661110308353313, 513055160567618935, 603299706328452441, 460916722833655479, 606206348574387586, 476876563643924133, 508165376573780056, 319625035512395402, 328459118409053944, 413956826422147456|
|practice_0194|mixed|128|0|224|237|19|243|8.43|16240740892, 26008713295, 27067904472, 259412740786, 259412742900, 259412743070, 277173952895, 389605665846, 778238228911, 784029889432, 2333921716044, 3434316541665, 4582668765600, 3807408205537232, 319625035512395402, 328459118409053944, 374855517519907790, 385687971678785004, 413956826422147456, 415399148292321536, 460916722833655479, 476876563643924133, 508165376573780056, 513055160567618935, 603299706328452441, 606206348574387586, 809661110308353313, 878188438346647256|
|practice_0194|mixed|128|1|220|223|0|220|25.97|16240740892, 27067904472, 26008713295, 259412740786, 259412742900, 259412743070, 778238228911, 389605665846, 784029889432, 3434316541665, 277173952895, 2333921716044, 4582668765600, 328459118409053944, 3807408205537232, 413956826422147456, 460916722833655479, 513055160567618935, 508165376573780056, 319625035512395402, 415399148292321536, 476876563643924133, 606206348574387586, 385687971678785004, 374855517519907790, 603299706328452441, 809661110308353313, 878188438346647256|
|practice_0194|mixed|128|2|218|221|13|231|158.52|259412742900, 27067904472, 259412740786, 259412743070, 26008713295, 778238228911, 389605665846, 4582668765600, 16240740892, 277173952895, 784029889432, 2333921716044, 3434316541665, 328459118409053944, 415399148292321536, 606206348574387586, 413956826422147456, 3807408205537232, 878188438346647256, 508165376573780056, 374855517519907790, 319625035512395402, 385687971678785004, 809661110308353313, 603299706328452441, 476876563643924133, 513055160567618935, 460916722833655479|
|practice_0195|independent|0|0|262|0|0|262|3.36|126221220170716, 159522232942741, 214384273901320, 215270473433922, 221742375609709, 239315415773830, 260789681824907, 382777452425856, 386504009457796, 455918650508655, 464072167169821, 471106293617060, 472077364949775, 583016799179608, 590396232116310, 654735428499688, 738645438208344, 781154524866282, 842378175346634, 882066412524287, 892648645163610, 894539381575166, 894692356936423, 897401123055381, 908095471582681, 914273363906683, 921865669108033, 937627741577359, 964198583348750|
|practice_0195|independent|0|1|250|0|100|350|10.33|897401123055381, 126221220170716, 260789681824907, 215270473433922, 464072167169821, 214384273901320, 382777452425856, 159522232942741, 738645438208344, 386504009457796, 892648645163610, 894539381575166, 239315415773830, 221742375609709, 590396232116310, 583016799179608, 455918650508655, 894692356936423, 908095471582681, 914273363906683, 882066412524287, 921865669108033, 472077364949775, 842378175346634, 471106293617060, 654735428499688, 781154524866282, 937627741577359, 964198583348750|
|practice_0195|independent|0|2|244|0|89|333|71.17|126221220170716, 583016799179608, 215270473433922, 382777452425856, 214384273901320, 386504009457796, 455918650508655, 781154524866282, 654735428499688, 464072167169821, 590396232116310, 239315415773830, 842378175346634, 908095471582681, 882066412524287, 472077364949775, 221742375609709, 897401123055381, 471106293617060, 894692356936423, 894539381575166, 159522232942741, 260789681824907, 964198583348750, 921865669108033, 738645438208344, 914273363906683, 937627741577359, 892648645163610|
|practice_0195|independent|48|0|271|313|0|271|1.40|126221220170716, 159522232942741, 214384273901320, 215270473433922, 221742375609709, 239315415773830, 260789681824907, 382777452425856, 386504009457796, 455918650508655, 464072167169821, 471106293617060, 472077364949775, 583016799179608, 590396232116310, 654735428499688, 738645438208344, 781154524866282, 842378175346634, 882066412524287, 892648645163610, 894539381575166, 894692356936423, 897401123055381, 908095471582681, 914273363906683, 921865669108033, 937627741577359, 964198583348750|
|practice_0195|independent|48|1|269|277|0|269|3.56|897401123055381, 126221220170716, 260789681824907, 382777452425856, 159522232942741, 464072167169821, 239315415773830, 921865669108033, 908095471582681, 221742375609709, 215270473433922, 894539381575166, 842378175346634, 472077364949775, 892648645163610, 882066412524287, 455918650508655, 738645438208344, 471106293617060, 386504009457796, 583016799179608, 937627741577359, 781154524866282, 894692356936423, 914273363906683, 654735428499688, 214384273901320, 964198583348750, 590396232116310|
|practice_0195|independent|48|2|261|314|0|261|28.93|126221220170716, 215270473433922, 583016799179608, 654735428499688, 464072167169821, 221742375609709, 471106293617060, 914273363906683, 908095471582681, 472077364949775, 892648645163610, 897401123055381, 894539381575166, 239315415773830, 921865669108033, 964198583348750, 937627741577359, 382777452425856, 894692356936423, 260789681824907, 386504009457796, 882066412524287, 842378175346634, 590396232116310, 214384273901320, 159522232942741, 455918650508655, 738645438208344, 781154524866282|
|practice_0195|independent|128|0|236|244|15|251|8.11|126221220170716, 159522232942741, 214384273901320, 215270473433922, 221742375609709, 239315415773830, 260789681824907, 382777452425856, 386504009457796, 455918650508655, 464072167169821, 471106293617060, 472077364949775, 583016799179608, 590396232116310, 654735428499688, 738645438208344, 781154524866282, 842378175346634, 882066412524287, 892648645163610, 894539381575166, 894692356936423, 897401123055381, 908095471582681, 914273363906683, 921865669108033, 937627741577359, 964198583348750|
|practice_0195|independent|128|1|231|236|23|254|27.92|897401123055381, 126221220170716, 260789681824907, 215270473433922, 464072167169821, 214384273901320, 382777452425856, 386504009457796, 583016799179608, 894539381575166, 239315415773830, 654735428499688, 472077364949775, 221742375609709, 455918650508655, 590396232116310, 471106293617060, 892648645163610, 159522232942741, 781154524866282, 842378175346634, 738645438208344, 882066412524287, 894692356936423, 914273363906683, 908095471582681, 921865669108033, 964198583348750, 937627741577359|
|practice_0195|independent|128|2|229|233|29|258|150.04|126221220170716, 159522232942741, 214384273901320, 221742375609709, 455918650508655, 215270473433922, 239315415773830, 382777452425856, 583016799179608, 464072167169821, 590396232116310, 842378175346634, 472077364949775, 897401123055381, 386504009457796, 937627741577359, 882066412524287, 892648645163610, 471106293617060, 738645438208344, 260789681824907, 654735428499688, 781154524866282, 894692356936423, 921865669108033, 894539381575166, 908095471582681, 914273363906683, 964198583348750|
|practice_0196|shared|0|0|137|0|0|137|2.37|1197575216, 1197575337, 1974237726, 1974237747, 2689355850, 2689355905, 2958503917, 2958504378, 8068067582, 8973162841, 10778177193, 15793901864, 21514846920, 56211585196, 78110122182, 81364710033, 153289628565, 181257882920, 197423764718, 295850421500, 300084129488, 387640631664, 598787617855, 725031552233, 730750510116, 860593881921, 1153816595642, 1379639568276|
|practice_0196|shared|0|1|129|0|0|129|4.12|2958503917, 2958504378, 56211585196, 2689355850, 2689355905, 8068067582, 21514846920, 860593881921, 1379639568276, 730750510116, 295850421500, 1197575216, 1197575337, 10778177193, 153289628565, 1974237747, 1974237726, 15793901864, 300084129488, 81364710033, 598787617855, 197423764718, 8973162841, 181257882920, 387640631664, 725031552233, 1153816595642, 78110122182|
|practice_0196|shared|0|2|135|0|0|135|28.64|1197575216, 1974237726, 1974237747, 2958503917, 2958504378, 1197575337, 10778177193, 2689355850, 2689355905, 8068067582, 15793901864, 21514846920, 295850421500, 56211585196, 300084129488, 153289628565, 730750510116, 1379639568276, 197423764718, 860593881921, 181257882920, 725031552233, 1153816595642, 8973162841, 387640631664, 78110122182, 81364710033, 598787617855|
|practice_0196|shared|48|0|160|198|0|160|1.10|1197575216, 1197575337, 1974237726, 1974237747, 2689355850, 2689355905, 2958503917, 2958504378, 8068067582, 8973162841, 10778177193, 15793901864, 21514846920, 56211585196, 78110122182, 81364710033, 153289628565, 181257882920, 197423764718, 295850421500, 300084129488, 387640631664, 598787617855, 725031552233, 730750510116, 860593881921, 1153816595642, 1379639568276|
|practice_0196|shared|48|1|140|145|0|140|2.68|2958503917, 2958504378, 56211585196, 2689355850, 2689355905, 8068067582, 21514846920, 860593881921, 1379639568276, 1197575216, 1197575337, 10778177193, 153289628565, 295850421500, 81364710033, 1974237726, 1974237747, 300084129488, 15793901864, 197423764718, 78110122182, 387640631664, 8973162841, 181257882920, 725031552233, 730750510116, 1153816595642, 598787617855|
|practice_0196|shared|48|2|145|151|0|145|15.98|2689355850, 2958504378, 8068067582, 21514846920, 56211585196, 730750510116, 860593881921, 300084129488, 2689355905, 1197575216, 2958503917, 1974237747, 15793901864, 10778177193, 1974237726, 598787617855, 81364710033, 295850421500, 153289628565, 197423764718, 1153816595642, 181257882920, 1197575337, 8973162841, 78110122182, 1379639568276, 725031552233, 387640631664|
|practice_0196|shared|128|0|133|134|0|133|3.99|1197575216, 1197575337, 1974237726, 1974237747, 2689355850, 2689355905, 2958503917, 2958504378, 8068067582, 8973162841, 10778177193, 15793901864, 21514846920, 56211585196, 78110122182, 81364710033, 153289628565, 181257882920, 197423764718, 295850421500, 300084129488, 387640631664, 598787617855, 725031552233, 730750510116, 860593881921, 1153816595642, 1379639568276|
|practice_0196|shared|128|1|133|136|0|133|8.72|2958503917, 2958504378, 56211585196, 2689355850, 2689355905, 8068067582, 21514846920, 860593881921, 1379639568276, 1197575216, 1197575337, 10778177193, 153289628565, 295850421500, 1974237747, 1974237726, 15793901864, 300084129488, 197423764718, 598787617855, 730750510116, 725031552233, 8973162841, 1153816595642, 78110122182, 81364710033, 181257882920, 387640631664|
|practice_0196|shared|128|2|129|131|0|129|51.45|2689355850, 2958504378, 8068067582, 21514846920, 2958503917, 56211585196, 1197575216, 1197575337, 2689355905, 10778177193, 730750510116, 300084129488, 1974237726, 1974237747, 15793901864, 153289628565, 295850421500, 860593881921, 197423764718, 598787617855, 8973162841, 1379639568276, 387640631664, 181257882920, 725031552233, 78110122182, 81364710033, 1153816595642|
|practice_0197|mixed|0|0|209|0|0|209|3.01|878205155, 878205616, 3915880919, 5269227852, 5296329076, 13173084360, 23495274118, 71267739522, 214282170816, 323076102484, 858006888825, 4347332109012, 7643799573428, 19285395386592, 202980950372592, 229854309618713, 235921185205208, 318284828531065, 364207295517863, 384793418834664, 613535702016050, 835165666587903, 885255470368409, 907435396447276, 916565695286074, 928237519293186, 939458041009102, 959224756038277|
|practice_0197|mixed|0|1|199|0|0|199|9.32|878205155, 878205616, 13173084360, 5269227852, 214282170816, 858006888825, 19285395386592, 5296329076, 3915880919, 23495274118, 323076102484, 7643799573428, 71267739522, 4347332109012, 907435396447276, 202980950372592, 959224756038277, 229854309618713, 235921185205208, 364207295517863, 885255470368409, 384793418834664, 318284828531065, 613535702016050, 916565695286074, 835165666587903, 939458041009102, 928237519293186|
|practice_0197|mixed|0|2|198|0|1|199|63.21|878205616, 858006888825, 3915880919, 7643799573428, 13173084360, 878205155, 5269227852, 5296329076, 71267739522, 4347332109012, 214282170816, 323076102484, 19285395386592, 959224756038277, 23495274118, 885255470368409, 384793418834664, 229854309618713, 235921185205208, 318284828531065, 202980950372592, 613535702016050, 928237519293186, 835165666587903, 916565695286074, 939458041009102, 364207295517863, 907435396447276|
|practice_0197|mixed|48|0|214|272|0|214|1.44|878205155, 878205616, 3915880919, 5269227852, 5296329076, 13173084360, 23495274118, 71267739522, 214282170816, 323076102484, 858006888825, 4347332109012, 7643799573428, 19285395386592, 202980950372592, 229854309618713, 235921185205208, 318284828531065, 364207295517863, 384793418834664, 613535702016050, 835165666587903, 885255470368409, 907435396447276, 916565695286074, 928237519293186, 939458041009102, 959224756038277|
|practice_0197|mixed|48|1|209|216|0|209|3.95|878205155, 878205616, 13173084360, 5269227852, 214282170816, 858006888825, 19285395386592, 5296329076, 323076102484, 3915880919, 23495274118, 7643799573428, 71267739522, 4347332109012, 202980950372592, 235921185205208, 229854309618713, 318284828531065, 835165666587903, 384793418834664, 885255470368409, 907435396447276, 916565695286074, 959224756038277, 364207295517863, 613535702016050, 939458041009102, 928237519293186|
|practice_0197|mixed|48|2|215|234|0|215|28.70|13173084360, 878205616, 214282170816, 3915880919, 858006888825, 878205155, 7643799573428, 5296329076, 71267739522, 323076102484, 202980950372592, 23495274118, 229854309618713, 5269227852, 235921185205208, 928237519293186, 835165666587903, 959224756038277, 364207295517863, 19285395386592, 4347332109012, 907435396447276, 885255470368409, 613535702016050, 916565695286074, 939458041009102, 318284828531065, 384793418834664|
|practice_0197|mixed|128|0|190|198|8|198|6.86|878205155, 878205616, 3915880919, 5269227852, 5296329076, 13173084360, 23495274118, 71267739522, 214282170816, 323076102484, 858006888825, 4347332109012, 7643799573428, 19285395386592, 202980950372592, 229854309618713, 235921185205208, 318284828531065, 364207295517863, 384793418834664, 613535702016050, 835165666587903, 885255470368409, 907435396447276, 916565695286074, 928237519293186, 939458041009102, 959224756038277|
|practice_0197|mixed|128|1|182|186|0|182|17.26|878205155, 878205616, 13173084360, 5269227852, 214282170816, 858006888825, 19285395386592, 5296329076, 3915880919, 23495274118, 323076102484, 7643799573428, 71267739522, 4347332109012, 318284828531065, 364207295517863, 235921185205208, 229854309618713, 613535702016050, 202980950372592, 384793418834664, 835165666587903, 885255470368409, 907435396447276, 916565695286074, 928237519293186, 939458041009102, 959224756038277|
|practice_0197|mixed|128|2|183|186|114|297|107.73|13173084360, 3915880919, 878205616, 5296329076, 214282170816, 71267739522, 858006888825, 4347332109012, 19285395386592, 323076102484, 878205155, 5269227852, 7643799573428, 23495274118, 318284828531065, 384793418834664, 928237519293186, 835165666587903, 202980950372592, 229854309618713, 235921185205208, 907435396447276, 885255470368409, 959224756038277, 939458041009102, 613535702016050, 364207295517863, 916565695286074|
|practice_0198|multiplicative|0|0|47|0|0|47|0.17|1820, 2069, 3000, 3482, 4985, 7470, 83454, 88623, 89139, 145108, 476790, 1430370|
|practice_0198|multiplicative|0|1|47|0|0|47|0.19|1820, 2069, 7470, 3000, 3482, 4985, 83454, 88623, 89139, 145108, 476790, 1430370|
|practice_0198|multiplicative|0|2|43|0|0|43|0.92|2069, 1820, 89139, 4985, 476790, 1430370, 3000, 88623, 3482, 7470, 83454, 145108|
|practice_0198|multiplicative|48|0|50|54|0|50|0.19|1820, 2069, 3000, 3482, 4985, 7470, 83454, 88623, 89139, 145108, 476790, 1430370|
|practice_0198|multiplicative|48|1|49|53|0|49|0.25|1820, 2069, 7470, 3000, 3482, 4985, 83454, 88623, 89139, 145108, 1430370, 476790|
|practice_0198|multiplicative|48|2|43|49|0|43|0.85|2069, 1820, 89139, 4985, 476790, 1430370, 88623, 3482, 3000, 7470, 145108, 83454|
|practice_0198|multiplicative|128|0|48|51|0|48|0.24|1820, 2069, 3000, 3482, 4985, 7470, 83454, 88623, 89139, 145108, 476790, 1430370|
|practice_0198|multiplicative|128|1|48|51|0|48|0.20|1820, 2069, 7470, 3000, 3482, 4985, 83454, 88623, 89139, 145108, 476790, 1430370|
|practice_0198|multiplicative|128|2|44|49|0|44|0.77|2069, 1820, 89139, 4985, 476790, 1430370, 88623, 3482, 3000, 7470, 83454, 145108|
|practice_0199|independent|0|0|291|0|0|291|3.71|122146382672925, 130652565620121, 133375024804073, 150381485058867, 255841160386830, 279322876601353, 293736156646090, 354308239491401, 417664143278984, 443631125100542, 479119019984135, 505136880513525, 550216635733217, 565489639235006, 569736585793933, 576047502841171, 587998988469770, 591112804314900, 593613616953617, 636099074601732, 638064198072636, 645985661789153, 764921743947741, 777944691102156, 789238113722076, 795675058022501, 807893927797484, 830363410482568, 875984604876813, 913986038450337, 958341176477383, 984301110720979|
|practice_0199|independent|0|1|272|0|143|415|12.83|122146382672925, 958341176477383, 130652565620121, 133375024804073, 279322876601353, 417664143278984, 150381485058867, 764921743947741, 505136880513525, 354308239491401, 443631125100542, 984301110720979, 255841160386830, 795675058022501, 479119019984135, 293736156646090, 576047502841171, 565489639235006, 807893927797484, 550216635733217, 587998988469770, 636099074601732, 777944691102156, 569736585793933, 638064198072636, 591112804314900, 593613616953617, 645985661789153, 789238113722076, 830363410482568, 875984604876813, 913986038450337|
|practice_0199|independent|0|2|271|0|103|374|71.80|122146382672925, 958341176477383, 591112804314900, 133375024804073, 150381485058867, 789238113722076, 777944691102156, 293736156646090, 255841160386830, 130652565620121, 638064198072636, 417664143278984, 443631125100542, 279322876601353, 807893927797484, 354308239491401, 479119019984135, 505136880513525, 645985661789153, 565489639235006, 795675058022501, 593613616953617, 569736585793933, 576047502841171, 587998988469770, 636099074601732, 875984604876813, 550216635733217, 984301110720979, 764921743947741, 830363410482568, 913986038450337|
|practice_0199|independent|48|0|291|422|0|291|1.67|122146382672925, 130652565620121, 133375024804073, 150381485058867, 255841160386830, 279322876601353, 293736156646090, 354308239491401, 417664143278984, 443631125100542, 479119019984135, 505136880513525, 550216635733217, 565489639235006, 569736585793933, 576047502841171, 587998988469770, 591112804314900, 593613616953617, 636099074601732, 638064198072636, 645985661789153, 764921743947741, 777944691102156, 789238113722076, 795675058022501, 807893927797484, 830363410482568, 875984604876813, 913986038450337, 958341176477383, 984301110720979|
|practice_0199|independent|48|1|296|400|0|296|5.58|122146382672925, 958341176477383, 133375024804073, 279322876601353, 150381485058867, 130652565620121, 255841160386830, 354308239491401, 638064198072636, 479119019984135, 591112804314900, 587998988469770, 984301110720979, 505136880513525, 417664143278984, 443631125100542, 795675058022501, 789238113722076, 777944691102156, 293736156646090, 565489639235006, 593613616953617, 550216635733217, 807893927797484, 764921743947741, 875984604876813, 913986038450337, 569736585793933, 830363410482568, 636099074601732, 645985661789153, 576047502841171|
|practice_0199|independent|48|2|289|299|0|289|33.49|122146382672925, 130652565620121, 591112804314900, 569736585793933, 293736156646090, 638064198072636, 565489639235006, 576047502841171, 550216635733217, 636099074601732, 593613616953617, 777944691102156, 354308239491401, 789238113722076, 984301110720979, 255841160386830, 807893927797484, 443631125100542, 505136880513525, 875984604876813, 279322876601353, 830363410482568, 150381485058867, 417664143278984, 587998988469770, 764921743947741, 133375024804073, 795675058022501, 479119019984135, 958341176477383, 913986038450337, 645985661789153|
|practice_0199|independent|128|0|260|266|26|286|7.68|122146382672925, 130652565620121, 133375024804073, 150381485058867, 255841160386830, 279322876601353, 293736156646090, 354308239491401, 417664143278984, 443631125100542, 479119019984135, 505136880513525, 550216635733217, 565489639235006, 569736585793933, 576047502841171, 587998988469770, 591112804314900, 593613616953617, 636099074601732, 638064198072636, 645985661789153, 764921743947741, 777944691102156, 789238113722076, 795675058022501, 807893927797484, 830363410482568, 875984604876813, 913986038450337, 958341176477383, 984301110720979|
|practice_0199|independent|128|1|256|259|23|279|31.03|122146382672925, 958341176477383, 133375024804073, 130652565620121, 150381485058867, 591112804314900, 636099074601732, 293736156646090, 638064198072636, 255841160386830, 354308239491401, 279322876601353, 443631125100542, 505136880513525, 479119019984135, 417664143278984, 550216635733217, 593613616953617, 565489639235006, 576047502841171, 569736585793933, 645985661789153, 984301110720979, 830363410482568, 807893927797484, 789238113722076, 795675058022501, 587998988469770, 764921743947741, 913986038450337, 777944691102156, 875984604876813|
|practice_0199|independent|128|2|250|258|17|267|121.82|122146382672925, 130652565620121, 569736585793933, 133375024804073, 636099074601732, 565489639235006, 279322876601353, 417664143278984, 795675058022501, 354308239491401, 293736156646090, 150381485058867, 505136880513525, 443631125100542, 576047502841171, 550216635733217, 591112804314900, 645985661789153, 638064198072636, 587998988469770, 789238113722076, 875984604876813, 255841160386830, 777944691102156, 830363410482568, 593613616953617, 764921743947741, 807893927797484, 913986038450337, 958341176477383, 479119019984135, 984301110720979|
|practice_0200|shared|0|0|145|0|0|145|2.90|10942875, 15309437, 18974564, 18974633, 21417766, 26546224, 26546820, 31836014, 44539854, 107170751, 267239334, 322569271, 555727356, 663655025, 1603435266, 1647975260, 3282864300, 4944967848, 8538601104, 9185655000, 10283107781, 97252602475, 124956564015, 159387379200, 343831683600, 409853264400, 919134481797, 476609925905304|
|practice_0200|shared|0|1|142|0|0|142|5.45|10942875, 3282864300, 26546224, 663655025, 26546820, 18974564, 18974633, 322569271, 15309437, 107170751, 9185655000, 21417766, 31836014, 44539854, 267239334, 1603435266, 1647975260, 8538601104, 10283107781, 4944967848, 555727356, 343831683600, 97252602475, 124956564015, 159387379200, 409853264400, 919134481797, 476609925905304|
|practice_0200|shared|0|2|135|0|0|135|30.37|18974564, 26546224, 322569271, 26546820, 44539854, 15309437, 267239334, 1603435266, 31836014, 18974633, 663655025, 1647975260, 10942875, 159387379200, 409853264400, 21417766, 9185655000, 107170751, 3282864300, 124956564015, 555727356, 8538601104, 10283107781, 4944967848, 97252602475, 919134481797, 343831683600, 476609925905304|
|practice_0200|shared|48|0|148|182|0|148|1.22|10942875, 15309437, 18974564, 18974633, 21417766, 26546224, 26546820, 31836014, 44539854, 107170751, 267239334, 322569271, 555727356, 663655025, 1603435266, 1647975260, 3282864300, 4944967848, 8538601104, 9185655000, 10283107781, 97252602475, 124956564015, 159387379200, 343831683600, 409853264400, 919134481797, 476609925905304|
|practice_0200|shared|48|1|142|153|0|142|1.96|10942875, 3282864300, 26546224, 663655025, 26546820, 18974564, 18974633, 322569271, 15309437, 9185655000, 21417766, 267239334, 44539854, 107170751, 1603435266, 1647975260, 31836014, 555727356, 97252602475, 8538601104, 343831683600, 4944967848, 409853264400, 159387379200, 10283107781, 124956564015, 919134481797, 476609925905304|
|practice_0200|shared|48|2|150|159|0|150|16.29|18974633, 10942875, 26546224, 322569271, 18974564, 15309437, 663655025, 44539854, 267239334, 26546820, 31836014, 1603435266, 159387379200, 555727356, 1647975260, 107170751, 4944967848, 97252602475, 409853264400, 9185655000, 21417766, 124956564015, 343831683600, 8538601104, 3282864300, 919134481797, 10283107781, 476609925905304|
|practice_0200|shared|128|0|131|137|0|131|4.54|10942875, 15309437, 18974564, 18974633, 21417766, 26546224, 26546820, 31836014, 44539854, 107170751, 267239334, 322569271, 555727356, 663655025, 1603435266, 1647975260, 3282864300, 4944967848, 8538601104, 9185655000, 10283107781, 97252602475, 124956564015, 159387379200, 343831683600, 409853264400, 919134481797, 476609925905304|
|practice_0200|shared|128|1|131|136|1|132|7.23|10942875, 3282864300, 26546224, 663655025, 26546820, 18974564, 18974633, 322569271, 15309437, 107170751, 9185655000, 21417766, 31836014, 44539854, 267239334, 1603435266, 1647975260, 8538601104, 10283107781, 4944967848, 555727356, 97252602475, 124956564015, 343831683600, 409853264400, 159387379200, 476609925905304, 919134481797|
|practice_0200|shared|128|2|130|136|0|130|41.37|18974633, 10942875, 15309437, 26546224, 26546820, 18974564, 44539854, 1603435266, 267239334, 31836014, 663655025, 107170751, 1647975260, 3282864300, 9185655000, 322569271, 8538601104, 10283107781, 343831683600, 21417766, 555727356, 4944967848, 159387379200, 409853264400, 124956564015, 97252602475, 919134481797, 476609925905304|

## Planificador y pulido final (practice)

200 programas válidos. Histórico → crudo → pos → A4 → ventanas: [35880, 32892, 32523, 32521, 32520]. IDs mejorados: 176.

|ID|Histórico|Crudo|Pos|A4|Ventanas|Ganador|Control pulido|0-0|0-1|0-2|48-0|48-1|48-2|128-0|128-1|128-2|
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
|practice_0001|153|143|142|142|142|0-2|153|154|146|142|162|149|153|164|149|142|
|practice_0002|267|238|236|236|236|128-2|267|276|313|318|277|271|271|254|240|236|
|practice_0003|284|248|248|248|248|128-2|284|285|314|475|293|287|292|261|251|248|
|practice_0004|129|122|122|122|122|128-2|129|130|125|123|137|131|128|124|123|122|
|practice_0005|232|203|202|202|202|128-2|232|234|219|272|232|227|228|223|203|202|
|practice_0006|36|36|34|34|34|0-2|36|40|39|34|39|37|36|39|37|36|
|practice_0007|104|98|98|97|97|128-2|104|105|108|103|109|104|102|105|103|97|
|practice_0008|258|241|240|240|240|128-0|258|258|257|272|271|260|261|240|240|274|
|practice_0009|137|137|137|137|137|hist|137|139|140|140|142|143|143|154|142|141|
|practice_0010|139|134|133|133|133|128-2|139|140|137|142|150|146|145|140|134|133|
|practice_0011|128|120|120|120|120|0-2|128|128|124|120|130|125|122|123|121|123|
|practice_0012|258|237|233|233|233|128-2|258|258|275|288|271|252|258|241|236|233|
|practice_0013|215|197|197|197|197|128-1|215|217|206|240|218|223|216|221|197|236|
|practice_0014|138|131|130|130|130|128-2|138|138|137|134|152|146|137|139|132|130|
|practice_0015|199|186|186|186|186|128-1|199|205|191|202|244|206|227|219|186|195|
|practice_0016|229|209|208|208|208|128-1|229|235|221|321|255|224|244|227|208|212|
|practice_0017|175|158|153|153|153|128-2|175|176|168|168|189|183|172|168|169|153|
|practice_0018|342|304|304|304|304|128-2|342|342|603|506|358|358|348|345|356|304|
|practice_0019|207|176|175|175|175|128-2|207|211|201|196|228|192|212|180|203|175|
|practice_0020|111|107|106|106|106|128-2|111|113|111|109|111|114|110|109|108|106|
|practice_0021|132|128|128|128|128|128-0|132|136|131|132|140|140|132|128|130|129|
|practice_0022|143|135|134|134|134|0-2|143|143|136|134|148|143|145|148|135|143|
|practice_0023|192|181|180|180|180|128-1|192|193|183|181|200|193|192|204|180|193|
|practice_0024|37|37|37|37|37|hist|37|42|42|38|42|42|37|42|42|37|
|practice_0025|182|175|175|175|175|0-2|182|182|181|175|196|186|188|193|184|175|
|practice_0026|152|147|147|147|147|0-2|152|153|154|147|158|154|157|172|161|152|
|practice_0027|204|187|184|184|184|128-2|204|205|197|188|220|216|202|186|197|184|
|practice_0028|240|228|216|216|216|128-1|240|240|233|228|244|232|243|253|216|232|
|practice_0029|211|193|192|192|192|128-1|211|214|205|206|214|207|210|194|192|193|
|practice_0030|139|134|134|134|134|0-2|139|141|141|134|144|141|144|141|141|142|
|practice_0031|226|198|198|198|198|128-2|226|226|213|239|229|228|220|216|213|198|
|practice_0032|140|136|136|136|136|128-2|140|140|140|138|146|150|147|144|143|136|
|practice_0033|41|39|36|35|35|48-2|41|41|40|36|38|37|35|38|37|35|
|practice_0034|318|281|280|280|280|128-1|318|319|368|437|318|318|316|322|280|286|
|practice_0035|264|243|239|239|239|128-2|264|265|265|246|266|254|278|243|241|239|
|practice_0036|132|126|125|125|125|0-2|132|133|131|125|137|138|135|130|126|128|
|practice_0037|293|259|259|259|259|128-1|293|293|322|435|300|294|296|266|259|274|
|practice_0038|43|43|43|43|43|hist|43|43|43|44|44|44|44|44|44|44|
|practice_0039|6|6|6|6|6|hist|6|8|8|8|7|7|7|7|7|7|
|practice_0040|198|174|174|174|174|128-2|198|203|195|187|204|194|196|200|183|174|
|practice_0041|188|165|164|164|164|128-2|188|188|182|197|189|189|192|205|183|164|
|practice_0042|182|156|156|156|156|128-2|182|185|170|169|198|184|181|180|178|156|
|practice_0043|122|122|122|122|122|hist|122|126|123|125|130|136|129|126|127|122|
|practice_0044|230|216|208|208|208|128-1|230|233|216|319|238|223|231|228|208|228|
|practice_0045|147|141|141|141|141|0-2|147|150|148|141|153|150|152|148|149|143|
|practice_0046|271|243|240|240|240|128-1|271|271|292|258|285|273|279|259|240|241|
|practice_0047|2|2|2|2|2|hist|2|2|2|2|2|2|2|2|2|2|
|practice_0048|120|117|116|116|116|128-2|120|122|125|120|130|126|123|122|123|116|
|practice_0049|129|126|124|124|124|128-2|129|131|128|126|138|134|134|129|126|124|
|practice_0050|3|3|3|3|3|hist|3|3|3|3|3|3|3|3|3|3|
|practice_0051|5|5|5|5|5|hist|5|5|5|5|5|5|5|5|5|5|
|practice_0052|208|198|195|195|195|0-2|208|214|226|195|209|206|212|218|198|195|
|practice_0053|267|235|232|232|232|128-1|267|267|255|249|272|263|262|259|232|237|
|practice_0054|346|301|301|301|301|128-2|346|346|456|394|353|365|354|345|303|301|
|practice_0055|349|315|311|311|311|128-1|349|349|680|563|351|354|345|338|311|314|
|practice_0056|4|4|4|4|4|hist|4|6|6|6|6|6|6|6|6|6|
|practice_0057|255|229|228|228|228|128-1|255|255|255|294|260|257|259|240|228|244|
|practice_0058|267|248|241|241|241|128-1|267|267|260|297|271|272|268|248|241|273|
|practice_0059|147|130|129|129|129|128-2|147|150|146|134|148|150|137|154|143|129|
|practice_0060|265|246|235|235|235|128-2|265|265|253|271|273|262|269|293|247|235|
|practice_0061|282|247|246|246|246|128-1|282|282|268|309|281|275|291|278|246|250|
|practice_0062|313|279|279|279|279|128-1|313|313|478|346|324|333|324|287|279|279|
|practice_0063|128|121|121|121|121|128-2|128|131|128|127|140|133|130|128|126|121|
|practice_0064|255|228|226|226|226|128-2|255|255|250|280|262|250|263|259|248|226|
|practice_0065|236|218|216|216|216|128-1|236|236|231|285|241|238|236|228|216|221|
|practice_0066|106|106|106|106|106|hist|106|111|109|107|117|114|119|108|106|110|
|practice_0067|171|150|150|150|150|128-2|171|173|165|160|192|171|190|165|155|150|
|practice_0068|196|181|181|181|181|128-1|196|197|193|184|197|202|201|201|181|194|
|practice_0069|220|192|192|192|192|128-1|220|220|208|234|216|215|220|209|192|195|
|practice_0070|270|237|237|237|237|128-1|270|271|357|293|271|266|280|263|237|247|
|practice_0071|167|161|161|161|161|128-0|167|168|167|168|200|171|186|161|165|166|
|practice_0072|5|5|5|5|5|hist|5|6|6|6|6|6|6|6|6|6|
|practice_0073|158|145|144|144|144|128-1|158|159|147|145|170|165|162|155|144|144|
|practice_0074|216|188|188|188|188|128-2|216|226|206|217|253|216|225|204|196|188|
|practice_0075|198|190|190|190|190|0-2|198|199|195|190|205|205|197|212|198|200|
|practice_0076|332|310|299|299|299|128-2|332|337|571|505|347|347|338|309|303|299|
|practice_0077|111|108|107|107|107|0-2|111|114|115|107|116|113|111|108|112|108|
|practice_0078|284|263|257|257|257|128-2|284|284|353|378|285|278|292|270|262|257|
|practice_0079|137|123|120|120|120|128-2|137|138|135|128|144|138|131|127|134|120|
|practice_0080|119|113|113|113|113|128-2|119|124|122|121|125|124|119|122|118|113|
|practice_0081|196|162|162|162|162|128-2|196|203|188|175|203|195|198|196|173|162|
|practice_0082|239|209|209|209|209|128-1|239|243|224|228|237|230|236|238|209|216|
|practice_0083|151|137|136|136|136|128-2|151|153|147|141|161|151|152|141|142|136|
|practice_0084|158|145|144|144|144|128-1|158|167|149|151|204|155|152|160|144|155|
|practice_0085|304|265|260|260|260|128-1|304|304|294|364|303|283|299|264|260|274|
|practice_0086|151|140|138|138|138|128-1|151|153|145|146|156|147|150|151|138|149|
|practice_0087|255|233|227|227|227|128-1|255|255|251|303|264|249|255|236|227|231|
|practice_0088|232|205|200|200|200|128-2|232|233|227|212|244|219|244|211|213|200|
|practice_0089|162|155|154|154|154|0-1|162|165|154|155|192|181|195|190|198|161|
|practice_0090|154|153|148|148|148|128-1|154|154|157|154|156|152|165|168|148|154|
|practice_0091|223|205|197|197|197|128-1|223|229|204|218|260|236|242|224|197|216|
|practice_0092|181|156|156|156|156|128-1|181|183|173|165|195|175|183|171|156|157|
|practice_0093|238|213|212|212|212|128-1|238|238|231|250|241|240|243|224|212|215|
|practice_0094|141|137|135|135|135|128-2|141|141|137|141|146|140|144|141|147|135|
|practice_0095|259|225|225|225|225|128-2|259|261|252|308|256|247|256|234|248|225|
|practice_0096|214|202|202|202|202|0-1|214|214|202|206|219|210|218|209|231|205|
|practice_0097|131|126|126|126|126|128-2|131|133|131|128|140|135|128|141|134|126|
|practice_0098|1|1|1|1|1|hist|1|1|1|1|1|1|1|1|1|1|
|practice_0099|130|125|124|124|124|128-2|130|131|129|127|133|136|138|127|127|124|
|practice_0100|341|309|304|304|304|128-1|341|342|499|533|350|343|339|322|304|310|
|practice_0101|166|149|149|149|149|128-2|166|168|161|154|174|162|163|153|166|149|
|practice_0102|306|286|273|273|273|128-1|306|307|339|387|309|306|308|291|273|326|
|practice_0103|5|5|5|5|5|hist|5|5|5|5|5|5|5|5|5|5|
|practice_0104|230|216|209|209|209|128-1|230|230|219|228|234|228|229|229|209|211|
|practice_0105|193|172|171|171|171|128-1|193|193|188|185|193|191|199|200|171|191|
|practice_0106|179|161|161|161|161|128-2|179|180|176|166|195|181|191|200|165|161|
|practice_0107|124|111|111|111|111|128-2|124|125|118|119|135|125|123|124|125|111|
|practice_0108|113|109|107|107|107|128-1|113|119|111|113|124|111|116|114|107|110|
|practice_0109|147|133|133|133|133|128-1|147|150|144|135|141|150|148|148|133|134|
|practice_0110|268|238|237|237|237|128-1|268|268|294|371|267|271|267|247|237|263|
|practice_0111|180|167|161|161|161|128-1|180|182|175|171|183|179|170|178|161|171|
|practice_0112|126|117|117|117|117|48-2|125|129|123|118|131|130|117|122|117|119|
|practice_0113|306|278|277|277|277|128-0|306|306|498|396|318|316|305|277|288|291|
|practice_0114|55|50|49|49|49|0-1|55|56|49|53|55|51|52|55|52|53|
|practice_0115|189|184|183|183|182|128-0|189|189|184|184|197|195|201|182|185|197|
|practice_0116|67|67|64|64|64|48-2|67|70|68|68|69|67|64|70|68|69|
|practice_0117|142|139|139|139|139|128-2|142|143|145|142|150|151|157|147|155|139|
|practice_0118|40|40|40|40|40|hist|40|40|42|42|43|41|40|43|41|40|
|practice_0119|266|237|236|236|236|128-1|266|267|270|272|270|267|269|253|236|249|
|practice_0120|166|149|149|149|149|0-2|166|167|158|149|171|160|164|166|168|177|
|practice_0121|280|246|246|246|246|128-1|280|280|298|316|281|279|288|265|246|251|
|practice_0122|319|273|271|271|271|128-2|319|319|401|446|324|325|323|330|285|271|
|practice_0123|2|2|2|2|2|hist|2|2|2|2|2|2|2|2|2|2|
|practice_0124|224|206|205|205|205|128-1|224|226|216|211|226|223|220|218|205|230|
|practice_0125|3|3|3|3|3|hist|3|3|3|3|3|3|3|3|3|3|
|practice_0126|154|148|147|147|147|128-2|154|155|154|154|177|158|166|163|154|147|
|practice_0127|196|179|179|179|179|128-2|196|196|196|196|200|197|197|185|183|179|
|practice_0128|2|2|2|2|2|hist|2|2|2|2|2|2|2|2|2|2|
|practice_0129|124|119|119|119|119|128-1|124|124|124|122|131|127|127|123|119|120|
|practice_0130|110|107|106|106|106|0-2|110|111|112|106|117|114|107|111|108|107|
|practice_0131|226|202|202|202|202|128-1|226|228|214|212|230|223|228|214|202|214|
|practice_0132|179|169|169|169|169|128-1|179|179|178|170|186|183|180|172|169|174|
|practice_0133|187|162|162|162|162|128-1|187|189|208|175|210|183|197|197|162|166|
|practice_0134|215|207|207|207|207|0-1|215|215|207|211|217|217|218|211|217|210|
|practice_0135|344|320|311|311|311|128-2|344|344|421|426|352|345|346|350|313|311|
|practice_0136|53|51|50|50|50|48-2|53|54|54|52|56|54|50|57|54|51|
|practice_0137|104|100|99|99|99|128-2|104|106|107|102|109|104|102|104|106|99|
|practice_0138|345|335|322|322|322|128-1|345|346|528|389|359|361|358|332|322|341|
|practice_0139|218|192|191|191|191|128-1|218|218|206|221|220|217|217|195|191|193|
|practice_0140|1|1|1|1|1|hist|1|1|1|1|1|1|1|1|1|1|
|practice_0141|154|148|148|148|148|128-1|154|156|154|152|166|162|160|186|148|149|
|practice_0142|264|253|247|247|247|128-1|264|264|270|293|269|270|270|259|247|264|
|practice_0143|152|145|145|145|145|128-2|152|156|159|151|165|165|150|150|152|145|
|practice_0144|113|111|111|111|111|0-2|113|115|118|111|123|118|113|113|114|111|
|practice_0145|198|180|179|179|179|128-1|198|207|188|180|234|187|199|189|179|185|
|practice_0146|44|43|42|42|42|0-2|44|45|45|42|45|45|42|45|45|43|
|practice_0147|187|168|167|167|167|128-2|187|187|174|175|204|183|193|186|171|167|
|practice_0148|176|164|164|164|164|0-2|176|176|168|164|179|173|175|172|182|168|
|practice_0149|201|172|172|172|172|128-2|201|203|194|202|213|196|214|196|190|172|
|practice_0150|203|196|191|191|191|128-1|203|203|195|201|211|195|203|210|191|204|
|practice_0151|333|295|293|293|293|128-0|333|333|388|414|334|335|346|293|319|314|
|practice_0152|161|148|148|148|148|0-2|161|162|158|148|176|156|167|172|151|152|
|practice_0153|144|138|138|138|138|128-1|144|147|147|150|185|144|156|141|138|138|
|practice_0154|335|299|296|296|296|128-2|335|335|387|479|349|333|341|330|300|296|
|practice_0155|158|154|154|154|154|128-1|158|160|159|155|167|161|162|198|154|165|
|practice_0156|109|106|106|106|106|48-2|109|111|109|108|113|108|106|108|106|106|
|practice_0157|137|127|127|127|127|128-1|137|146|129|128|154|140|136|132|127|130|
|practice_0158|235|207|205|205|205|128-2|235|239|226|234|249|250|241|249|210|205|
|practice_0159|34|33|32|32|32|0-1|34|38|32|37|37|32|38|37|32|38|
|practice_0160|185|163|163|163|163|128-2|185|187|180|178|184|180|182|178|172|163|
|practice_0161|274|237|235|235|235|128-1|274|274|290|388|273|265|266|264|235|246|
|practice_0162|285|255|255|255|255|128-1|285|285|285|334|286|280|284|282|255|308|
|practice_0163|262|231|231|231|231|128-1|262|262|331|342|265|255|260|239|231|249|
|practice_0164|254|220|220|220|220|128-2|254|256|243|243|280|258|274|246|246|220|
|practice_0165|238|205|205|205|205|128-2|238|244|224|236|238|230|241|244|214|205|
|practice_0166|48|47|43|43|43|48-2|48|52|48|48|50|47|43|51|47|43|
|practice_0167|300|275|267|267|267|128-1|300|300|359|360|304|311|301|292|267|290|
|practice_0168|128|119|117|117|117|128-2|128|130|127|120|131|128|121|128|122|117|
|practice_0169|4|4|4|4|4|hist|4|4|4|4|4|4|4|4|4|4|
|practice_0170|36|36|34|34|34|48-0|36|36|36|37|34|34|35|34|34|35|
|practice_0171|292|246|245|245|245|128-1|292|292|278|282|293|290|286|270|245|302|
|practice_0172|2|2|2|2|2|hist|2|2|2|2|2|2|2|2|2|2|
|practice_0173|147|136|135|135|135|0-2|147|147|152|135|168|153|159|162|146|146|
|practice_0174|2|2|2|2|2|hist|2|2|2|2|2|2|2|2|2|2|
|practice_0175|4|4|4|4|4|hist|4|5|5|5|5|5|5|5|5|5|
|practice_0176|317|300|295|295|295|128-2|317|318|353|402|331|324|329|346|305|295|
|practice_0177|178|166|165|165|165|128-1|178|179|177|171|187|184|187|193|165|181|
|practice_0178|92|92|92|92|92|hist|92|93|94|95|93|92|92|95|92|92|
|practice_0179|342|319|306|306|306|128-2|342|342|393|477|358|343|344|335|307|306|
|practice_0180|154|148|147|147|147|0-2|154|155|151|147|158|154|155|155|160|157|
|practice_0181|281|257|250|250|250|128-2|281|281|330|376|292|281|285|260|252|250|
|practice_0182|220|200|198|198|198|128-1|220|220|211|211|220|225|223|211|198|203|
|practice_0183|286|272|251|251|251|128-1|286|287|342|287|290|283|283|270|251|271|
|practice_0184|254|222|222|222|222|128-0|254|256|274|269|249|249|250|222|224|225|
|practice_0185|170|164|164|164|164|128-1|170|177|171|168|188|178|178|213|164|173|
|practice_0186|117|107|106|106|106|128-2|117|117|116|111|116|108|110|112|108|106|
|practice_0187|218|190|190|190|190|128-1|218|218|237|203|210|211|214|228|190|197|
|practice_0188|224|218|212|212|212|128-1|224|224|220|262|229|230|227|246|212|214|
|practice_0189|184|171|170|170|170|128-2|184|186|186|182|206|190|191|210|177|170|
|practice_0190|295|282|269|269|269|128-1|295|295|344|287|296|301|296|298|269|300|
|practice_0191|313|275|271|271|271|128-1|313|313|533|453|322|327|316|307|271|274|
|practice_0192|119|117|116|116|116|128-2|119|125|123|118|121|124|121|121|123|116|
|practice_0193|132|126|126|126|126|128-2|132|140|134|128|144|135|135|133|128|126|
|practice_0194|249|220|220|220|220|128-1|249|249|245|258|249|246|253|238|220|228|
|practice_0195|262|251|239|239|239|128-1|262|262|284|310|270|268|261|244|239|251|
|practice_0196|137|129|128|128|128|0-1|137|137|128|135|158|139|142|133|133|129|
|practice_0197|207|182|182|182|182|128-1|207|209|199|199|213|206|213|192|182|220|
|practice_0198|41|41|41|41|41|hist|41|46|46|43|50|49|41|47|47|42|
|practice_0199|291|267|260|260|260|128-2|291|291|379|321|291|296|287|270|271|260|
|practice_0200|138|130|129|129|129|128-2|138|145|142|135|145|141|150|131|130|129|

## Planificador y pulido final (scored)

1000 programas válidos. Histórico → crudo → pos → A4 → ventanas: [176594, 162189, 160464, 160451, 160445]. IDs mejorados: 941. [Entrega](artifacts/submission_planner_160445.csv)

|ID|Histórico|Crudo|Pos|A4|Ventanas|Ganador|Control pulido|0-0|0-1|0-2|48-0|48-1|48-2|128-0|128-1|128-2|
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
|scored_0001|289|275|267|267|267|128-2|289|289|284|341|294|291|291|273|268|267|
|scored_0002|304|269|268|268|268|128-1|304|304|339|399|309|304|306|306|268|275|
|scored_0003|265|248|238|238|238|128-1|265|266|257|375|262|267|264|264|238|239|
|scored_0004|268|232|231|231|231|128-1|268|269|280|351|271|267|268|265|231|249|
|scored_0005|67|65|63|63|63|48-1|67|69|66|64|68|63|64|68|68|69|
|scored_0006|174|158|157|157|157|128-1|174|175|169|165|237|167|181|170|157|158|
|scored_0007|221|196|196|196|196|0-2|221|222|220|196|259|234|215|214|199|211|
|scored_0008|156|143|143|143|143|128-2|156|162|157|146|179|157|151|156|148|143|
|scored_0009|48|43|41|41|41|48-2|48|48|48|45|48|48|41|48|48|41|
|scored_0010|223|213|209|209|209|128-2|223|224|215|230|236|218|228|222|228|209|
|scored_0011|175|153|153|153|153|128-2|175|181|166|159|220|181|192|180|160|153|
|scored_0012|226|200|200|200|200|128-1|226|226|219|236|228|227|231|207|200|203|
|scored_0013|217|193|193|193|193|0-2|217|222|205|193|224|211|213|221|209|194|
|scored_0014|39|37|35|35|35|48-2|39|39|39|36|38|38|35|38|38|35|
|scored_0015|275|241|241|241|241|128-1|275|275|390|303|279|280|281|268|241|243|
|scored_0016|202|182|181|181|181|128-2|202|203|207|202|230|218|212|222|194|181|
|scored_0017|161|147|147|147|147|128-2|161|161|160|157|163|165|163|157|152|147|
|scored_0018|165|150|150|150|150|0-2|165|169|162|150|172|166|159|183|151|158|
|scored_0019|36|35|35|35|35|0-1|36|36|35|35|36|35|35|36|35|35|
|scored_0020|301|269|267|267|267|128-1|301|303|415|445|307|302|293|292|267|270|
|scored_0021|51|51|51|51|51|hist|51|54|52|52|54|54|51|54|54|51|
|scored_0022|155|139|139|139|139|128-2|155|156|151|143|160|151|156|155|144|139|
|scored_0023|201|186|185|185|185|128-1|201|201|193|217|202|198|199|193|185|214|
|scored_0024|40|40|40|39|39|48-1|40|42|41|41|43|39|39|42|39|39|
|scored_0025|175|154|154|154|154|128-2|175|177|159|165|176|176|166|171|177|154|
|scored_0026|227|213|209|209|209|128-1|227|227|216|214|233|226|233|234|209|213|
|scored_0027|136|130|130|130|130|128-2|136|138|133|135|152|139|136|134|132|130|
|scored_0028|183|167|167|167|167|128-2|183|188|176|180|233|198|192|176|169|167|
|scored_0029|48|48|48|48|48|hist|48|50|51|51|50|54|51|50|53|51|
|scored_0030|249|226|224|224|224|128-1|249|250|262|235|256|254|251|226|224|224|
|scored_0031|301|263|263|263|263|128-2|301|301|319|409|309|303|318|298|303|263|
|scored_0032|132|126|124|124|124|128-2|132|136|135|128|151|132|140|134|129|124|
|scored_0033|260|232|230|230|230|128-2|260|263|291|271|274|266|257|236|246|230|
|scored_0034|214|204|203|203|203|48-2|214|214|209|205|216|208|203|215|206|209|
|scored_0035|135|132|132|132|132|128-2|135|141|141|137|145|141|143|151|137|132|
|scored_0036|295|286|277|277|277|128-1|295|295|386|435|295|303|296|310|277|307|
|scored_0037|251|216|216|216|216|128-1|251|251|241|281|251|237|240|222|216|232|
|scored_0038|179|172|169|169|169|128-1|179|179|177|171|185|178|180|180|169|173|
|scored_0039|136|129|129|129|129|128-1|136|140|136|132|145|138|139|151|129|132|
|scored_0040|217|199|199|199|199|128-2|217|217|218|216|221|218|215|228|203|199|
|scored_0041|181|165|163|163|163|128-1|181|183|172|170|188|182|184|165|163|169|
|scored_0042|202|177|177|177|177|128-2|202|202|191|218|209|200|205|193|194|177|
|scored_0043|267|235|232|232|232|128-2|267|268|288|373|274|266|272|272|235|232|
|scored_0044|102|99|98|98|98|48-2|102|102|102|100|103|100|98|102|102|98|
|scored_0045|201|190|189|189|189|0-2|201|201|193|189|203|203|206|206|203|197|
|scored_0046|39|39|37|37|37|0-1|39|39|37|39|45|37|37|45|37|37|
|scored_0047|208|190|189|189|189|128-2|208|208|206|222|214|214|215|202|194|189|
|scored_0048|183|174|167|167|167|128-1|183|183|177|173|185|178|186|172|167|176|
|scored_0049|193|189|184|184|184|128-0|193|194|190|189|195|191|196|184|202|193|
|scored_0050|34|34|34|34|34|hist|34|39|38|34|40|38|37|40|38|37|
|scored_0051|352|305|302|302|302|128-1|352|352|348|615|353|352|358|356|302|318|
|scored_0052|137|131|131|131|131|128-2|137|139|140|135|148|137|137|142|139|131|
|scored_0053|193|157|157|157|157|128-2|193|197|166|171|216|183|199|189|161|157|
|scored_0054|242|218|212|212|212|128-1|242|242|230|264|243|244|238|221|212|216|
|scored_0055|99|93|91|91|91|128-2|99|100|98|95|98|96|95|95|94|91|
|scored_0056|36|34|34|33|33|0-1|36|36|33|35|38|37|34|38|37|34|
|scored_0057|270|238|236|236|236|128-2|270|270|300|286|276|264|275|248|242|236|
|scored_0058|333|305|301|301|301|128-1|333|333|374|487|335|327|339|338|301|303|
|scored_0059|183|166|166|166|166|128-1|183|186|178|194|184|181|174|171|166|183|
|scored_0060|316|279|278|278|278|128-2|316|317|425|447|329|323|324|299|280|278|
|scored_0061|4|4|4|4|4|hist|4|4|4|4|4|4|4|4|4|4|
|scored_0062|182|167|166|166|166|128-0|182|182|178|173|187|179|183|166|182|169|
|scored_0063|154|140|140|140|140|0-2|154|154|150|140|159|158|154|212|183|140|
|scored_0064|167|154|153|153|153|128-2|167|168|162|159|173|165|173|200|161|153|
|scored_0065|47|47|47|47|47|hist|47|49|49|49|49|49|48|49|49|48|
|scored_0066|52|48|48|48|48|0-1|52|52|48|48|52|49|48|52|50|48|
|scored_0067|321|284|280|280|280|128-1|321|321|457|370|321|329|335|335|280|283|
|scored_0068|194|169|169|169|169|128-2|194|195|177|180|232|184|191|189|178|169|
|scored_0069|353|298|298|298|298|128-1|353|353|451|577|351|343|344|358|298|322|
|scored_0070|183|174|172|172|172|128-2|183|183|189|177|200|179|195|180|175|172|
|scored_0071|145|129|129|129|129|128-1|145|146|139|134|149|136|139|143|129|130|
|scored_0072|161|146|146|146|146|128-2|161|161|164|156|167|164|170|156|169|146|
|scored_0073|155|149|149|149|149|0-2|155|160|154|149|168|159|153|149|152|157|
|scored_0074|259|242|238|238|238|128-2|259|260|268|307|254|258|261|261|294|238|
|scored_0075|318|278|275|275|275|128-2|318|318|413|395|324|327|321|310|307|275|
|scored_0076|174|159|158|158|158|128-2|174|178|177|165|178|180|178|174|169|158|
|scored_0077|133|129|129|129|129|128-0|133|135|133|131|143|140|135|129|134|134|
|scored_0078|129|125|125|125|125|128-1|129|130|127|131|131|134|131|130|125|125|
|scored_0079|191|170|170|170|170|128-2|191|191|187|187|196|191|198|193|181|170|
|scored_0080|183|173|170|170|170|128-2|183|185|180|173|194|183|185|189|187|170|
|scored_0081|148|138|137|137|137|128-1|148|149|144|146|159|156|150|147|137|143|
|scored_0082|154|138|138|138|138|128-1|154|158|145|143|199|141|143|166|138|138|
|scored_0083|219|197|197|197|197|128-1|219|219|207|232|211|215|217|211|197|214|
|scored_0084|50|50|49|49|49|48-2|50|50|51|51|50|53|49|49|51|50|
|scored_0085|116|114|114|114|114|0-2|116|116|120|114|121|115|122|114|114|114|
|scored_0086|132|126|126|126|126|0-2|132|132|131|126|144|132|137|132|126|129|
|scored_0087|304|277|268|268|268|128-1|304|306|303|377|307|309|305|277|268|285|
|scored_0088|264|247|239|239|239|128-2|264|264|249|264|263|258|261|254|240|239|
|scored_0089|152|144|142|142|142|128-2|152|154|146|144|160|154|152|144|151|142|
|scored_0090|238|226|216|216|216|128-2|238|238|296|225|234|237|232|244|255|216|
|scored_0091|40|32|29|29|29|0-2|40|40|34|29|40|34|29|40|34|29|
|scored_0092|53|49|47|47|47|0-2|53|55|54|47|54|53|47|55|54|47|
|scored_0093|135|125|125|125|125|128-2|135|137|135|129|141|144|129|142|132|125|
|scored_0094|313|281|281|281|281|128-0|313|313|330|376|322|313|313|281|345|300|
|scored_0095|44|44|44|44|44|hist|44|44|45|44|44|46|44|44|46|45|
|scored_0096|220|186|186|186|186|128-2|220|221|209|247|221|215|216|227|190|186|
|scored_0097|125|114|113|113|113|128-2|125|127|116|114|128|114|116|124|116|113|
|scored_0098|233|208|202|202|202|128-1|233|235|232|283|223|230|234|222|202|218|
|scored_0099|190|177|176|176|176|0-2|190|197|196|176|219|209|221|198|190|180|
|scored_0100|185|171|171|171|171|128-1|185|188|179|173|196|187|186|212|171|177|
|scored_0101|152|144|144|144|144|128-2|152|154|150|151|162|154|152|154|150|144|
|scored_0102|190|163|163|163|163|128-2|190|190|172|177|203|193|206|192|170|163|
|scored_0103|59|56|53|53|53|48-2|59|59|60|56|58|58|53|59|60|54|
|scored_0104|286|251|251|251|251|128-1|286|286|272|380|287|283|287|298|251|284|
|scored_0105|188|170|170|170|170|128-2|188|189|179|179|186|184|185|177|183|170|
|scored_0106|288|244|244|244|244|128-2|288|289|288|297|290|286|287|276|253|244|
|scored_0107|300|264|262|262|262|128-2|300|300|368|370|310|316|301|308|266|262|
|scored_0108|228|207|206|206|206|128-2|228|228|220|249|233|228|226|222|213|206|
|scored_0109|177|161|161|161|161|128-2|177|177|175|171|197|188|179|169|163|161|
|scored_0110|213|187|186|186|186|128-2|213|213|207|206|221|212|212|204|202|186|
|scored_0111|266|244|244|244|244|128-0|266|266|302|332|269|271|262|244|261|247|
|scored_0112|262|238|238|238|238|128-2|262|264|248|271|273|266|259|246|241|238|
|scored_0113|34|34|33|33|33|0-2|34|38|34|33|39|35|38|39|35|38|
|scored_0114|211|198|194|194|194|128-2|211|211|203|224|212|208|205|212|199|194|
|scored_0115|254|228|227|227|227|128-2|254|259|250|384|259|257|253|246|244|227|
|scored_0116|109|102|102|102|102|0-2|109|110|108|102|112|109|110|109|105|104|
|scored_0117|241|223|214|214|214|128-2|241|242|234|285|243|239|240|228|229|214|
|scored_0118|131|122|122|122|122|0-2|131|132|125|122|131|126|133|128|122|122|
|scored_0119|287|275|267|267|267|128-1|287|287|274|345|292|289|288|273|267|268|
|scored_0120|247|227|218|218|218|128-2|247|247|366|233|256|247|255|244|222|218|
|scored_0121|64|60|58|58|58|48-2|64|66|65|61|63|61|58|64|63|60|
|scored_0122|198|192|187|187|187|128-2|198|199|192|201|201|195|202|212|205|187|
|scored_0123|269|243|240|240|240|128-1|269|271|263|327|279|271|268|312|240|249|
|scored_0124|35|35|33|33|33|0-2|35|35|34|33|35|36|34|35|36|34|
|scored_0125|145|134|134|134|134|0-2|145|148|145|134|145|151|135|140|146|134|
|scored_0126|4|4|4|4|4|hist|4|4|4|4|4|4|4|4|4|4|
|scored_0127|140|134|134|134|134|0-2|140|144|144|134|162|149|145|139|137|135|
|scored_0128|160|142|141|141|141|128-2|160|162|149|144|162|158|157|157|152|141|
|scored_0129|340|337|318|318|318|128-1|340|340|343|457|339|339|348|344|318|325|
|scored_0130|33|33|32|32|32|0-1|33|35|32|33|36|32|35|36|32|35|
|scored_0131|118|111|111|111|111|128-2|118|122|119|114|122|121|118|116|115|111|
|scored_0132|133|125|124|124|124|128-2|133|132|130|126|138|141|132|130|130|124|
|scored_0133|288|265|258|258|258|128-1|288|289|282|323|295|289|291|272|258|266|
|scored_0134|41|38|36|36|36|0-1|41|43|36|41|41|40|37|41|40|37|
|scored_0135|118|110|109|109|109|128-2|118|120|118|113|116|116|114|111|110|109|
|scored_0136|155|144|144|144|144|0-2|155|165|153|144|189|161|175|164|154|148|
|scored_0137|70|67|67|67|67|0-2|70|73|69|67|71|68|67|73|70|68|
|scored_0138|118|115|115|115|115|0-2|118|119|119|115|126|122|118|117|117|116|
|scored_0139|228|212|212|212|212|0-1|228|228|212|217|226|218|222|220|217|254|
|scored_0140|198|188|188|188|188|0-2|198|199|195|188|212|199|206|193|203|196|
|scored_0141|167|158|158|158|158|0-2|167|168|164|158|179|169|171|170|170|167|
|scored_0142|148|141|140|140|140|128-2|148|148|150|145|154|151|146|151|143|140|
|scored_0143|184|159|158|158|158|0-2|184|189|173|158|231|178|185|181|160|166|
|scored_0144|141|130|129|129|129|128-2|141|144|138|137|157|150|146|146|139|129|
|scored_0145|269|251|250|250|250|128-1|269|269|343|508|276|270|275|266|250|262|
|scored_0146|327|282|281|281|281|128-1|327|329|411|398|327|328|331|300|281|350|
|scored_0147|119|112|112|112|112|128-1|119|123|122|113|128|117|122|119|112|115|
|scored_0148|217|197|197|197|197|128-1|217|217|211|212|215|224|222|215|197|200|
|scored_0149|26|26|26|26|26|hist|26|26|26|26|26|26|26|26|26|26|
|scored_0150|204|193|189|189|189|128-2|204|207|197|227|216|203|203|204|198|189|
|scored_0151|43|41|40|40|40|0-2|43|43|41|40|45|41|40|45|41|40|
|scored_0152|154|141|140|140|140|128-2|154|158|146|143|151|157|154|152|156|140|
|scored_0153|137|122|120|120|120|128-2|137|137|132|126|141|137|140|136|146|120|
|scored_0154|206|187|186|186|186|128-1|206|206|199|225|208|207|207|191|186|200|
|scored_0155|281|247|246|246|246|128-1|281|281|301|321|279|282|291|271|246|277|
|scored_0156|171|163|162|162|162|0-2|171|173|166|162|177|166|164|172|165|175|
|scored_0157|198|181|178|178|178|128-2|198|198|196|182|206|194|196|187|197|178|
|scored_0158|190|175|169|169|169|128-2|190|192|181|179|189|187|189|175|182|169|
|scored_0159|284|264|254|254|254|128-1|284|286|275|408|293|284|286|270|254|270|
|scored_0160|229|207|205|205|205|128-2|229|229|229|241|231|231|236|214|207|205|
|scored_0161|196|178|178|178|178|128-2|196|197|190|180|203|196|197|215|216|178|
|scored_0162|229|196|196|196|196|128-1|229|229|260|226|252|219|233|219|196|211|
|scored_0163|312|263|263|263|263|128-1|312|314|296|341|310|315|318|299|263|285|
|scored_0164|161|146|145|145|145|0-2|161|161|150|145|168|161|146|154|155|147|
|scored_0165|302|263|263|263|263|128-2|302|302|361|437|307|306|308|279|299|263|
|scored_0166|257|226|225|225|225|128-1|257|257|299|304|258|259|262|251|225|225|
|scored_0167|310|310|280|280|280|128-1|310|312|365|360|321|314|307|333|280|356|
|scored_0168|337|302|298|298|298|128-2|337|337|392|407|336|342|352|319|306|298|
|scored_0169|129|127|127|127|127|128-1|129|132|130|128|142|134|138|130|127|127|
|scored_0170|283|248|248|248|248|128-2|283|285|362|284|285|288|286|262|255|248|
|scored_0171|159|149|147|147|147|128-2|159|164|161|149|169|172|165|167|157|147|
|scored_0172|204|176|175|175|175|128-2|204|204|236|195|209|202|203|202|215|175|
|scored_0173|234|204|204|204|204|128-2|234|236|224|250|243|232|236|233|210|204|
|scored_0174|175|165|161|161|161|128-2|175|175|173|165|182|178|176|194|179|161|
|scored_0175|131|119|118|118|118|0-2|131|133|122|118|138|136|123|126|125|121|
|scored_0176|144|129|129|129|129|0-2|144|144|143|129|152|141|134|139|136|129|
|scored_0177|169|151|151|151|151|128-2|169|171|158|153|192|164|170|161|158|151|
|scored_0178|134|124|123|123|123|128-2|134|136|132|129|140|139|132|134|128|123|
|scored_0179|262|229|229|229|229|128-2|262|263|259|342|273|271|261|261|238|229|
|scored_0180|108|103|101|101|101|0-2|108|113|113|101|113|110|106|109|103|106|
|scored_0181|39|39|37|37|37|0-2|39|39|39|37|40|40|42|40|40|42|
|scored_0182|204|184|184|184|184|128-1|204|204|199|194|210|200|199|188|184|184|
|scored_0183|352|352|321|321|321|128-2|352|352|476|567|364|350|356|364|357|321|
|scored_0184|172|163|162|162|162|128-1|172|175|168|164|185|174|165|168|162|171|
|scored_0185|224|194|193|193|193|0-2|224|226|217|193|234|221|227|207|199|198|
|scored_0186|166|155|155|155|155|128-2|166|168|161|159|206|176|175|156|162|155|
|scored_0187|119|112|112|112|112|0-2|119|122|119|112|126|118|122|121|114|113|
|scored_0188|123|115|115|115|115|128-2|123|124|124|117|126|126|123|122|119|115|
|scored_0189|197|161|161|161|161|128-2|197|197|189|173|233|209|218|195|186|161|
|scored_0190|154|148|148|148|148|0-2|154|156|152|148|165|159|165|161|150|150|
|scored_0191|158|151|149|149|149|128-2|158|160|160|155|190|174|177|153|153|149|
|scored_0192|181|169|169|169|169|128-1|181|182|181|174|208|182|198|197|169|170|
|scored_0193|152|139|139|139|139|128-2|152|152|150|147|158|162|151|178|150|139|
|scored_0194|188|181|178|178|178|128-2|188|190|188|181|199|196|188|202|188|178|
|scored_0195|154|144|144|144|144|128-1|154|155|154|149|164|159|152|145|144|164|
|scored_0196|39|39|38|38|38|0-2|39|40|40|38|38|39|38|38|39|38|
|scored_0197|154|143|143|143|143|128-1|154|155|151|149|162|151|149|164|143|144|
|scored_0198|269|232|231|231|231|128-1|269|271|253|260|274|265|274|265|231|243|
|scored_0199|117|115|114|114|114|128-2|117|117|119|119|122|117|122|117|116|114|
|scored_0200|167|156|156|156|156|128-2|167|172|169|165|206|171|165|163|157|156|
|scored_0201|194|179|179|179|179|0-2|194|195|183|179|196|191|189|190|186|180|
|scored_0202|146|144|141|141|141|128-2|146|149|148|145|163|150|152|158|156|141|
|scored_0203|275|239|239|239|239|128-2|275|275|392|314|276|270|272|287|245|239|
|scored_0204|271|239|239|239|239|128-1|271|273|261|287|278|268|277|255|239|260|
|scored_0205|164|156|156|156|156|0-2|164|174|167|156|176|175|166|177|168|165|
|scored_0206|198|184|182|182|182|128-1|198|199|185|183|199|192|196|210|182|182|
|scored_0207|276|247|247|247|247|128-1|276|276|265|293|272|277|280|274|247|261|
|scored_0208|285|245|245|245|245|128-2|285|285|291|283|287|284|294|264|268|245|
|scored_0209|280|256|247|247|247|128-1|280|280|298|331|280|270|280|252|247|267|
|scored_0210|119|112|111|111|111|128-1|119|123|116|114|113|117|121|114|111|112|
|scored_0211|51|51|50|50|50|128-1|51|52|51|55|51|51|54|52|50|55|
|scored_0212|25|25|25|25|25|hist|25|28|28|28|28|28|29|28|28|29|
|scored_0213|146|137|137|137|137|0-2|146|148|142|137|158|146|150|145|137|137|
|scored_0214|213|190|190|190|190|128-2|213|215|205|199|217|214|209|201|193|190|
|scored_0215|203|182|182|182|182|128-1|203|203|193|188|208|194|204|189|182|190|
|scored_0216|189|159|159|159|159|128-2|189|189|193|176|214|199|187|196|168|159|
|scored_0217|300|291|273|273|273|128-2|300|301|398|339|305|300|296|291|314|273|
|scored_0218|348|321|310|310|310|128-2|348|348|455|554|362|360|345|344|318|310|
|scored_0219|149|143|143|143|143|0-1|149|154|143|151|156|156|160|151|150|155|
|scored_0220|140|131|131|131|131|128-2|140|141|137|132|141|139|139|136|132|131|
|scored_0221|51|49|47|47|47|48-0|51|51|50|50|47|47|49|48|49|50|
|scored_0222|49|47|47|47|47|0-1|49|50|47|49|53|47|49|53|47|49|
|scored_0223|262|231|231|231|231|128-1|262|263|331|301|275|263|266|241|231|232|
|scored_0224|225|199|197|197|197|128-2|225|225|210|210|227|224|218|235|200|197|
|scored_0225|129|127|127|127|127|128-2|129|132|131|128|137|137|141|130|132|127|
|scored_0226|154|142|142|142|142|128-2|154|154|152|145|164|160|166|184|153|142|
|scored_0227|185|166|165|165|165|128-2|185|189|189|174|221|187|166|186|167|165|
|scored_0228|144|136|135|135|135|128-2|144|145|143|143|160|163|154|141|139|135|
|scored_0229|179|159|159|159|159|128-2|179|182|174|167|177|179|177|179|162|159|
|scored_0230|266|247|242|242|242|128-2|266|266|271|313|268|261|264|281|288|242|
|scored_0231|221|197|197|197|197|128-1|221|221|212|379|219|218|217|220|197|219|
|scored_0232|203|190|187|187|187|128-2|203|203|195|206|211|203|204|215|197|187|
|scored_0233|49|48|47|47|47|48-2|49|49|50|48|50|49|47|51|48|47|
|scored_0234|227|214|214|214|214|128-0|227|227|230|240|236|228|226|214|217|215|
|scored_0235|264|228|226|226|226|128-2|264|264|287|279|267|259|273|247|234|226|
|scored_0236|135|124|124|124|124|128-2|135|137|130|130|138|132|137|134|128|124|
|scored_0237|42|42|41|41|41|0-2|42|44|44|41|43|42|43|43|42|43|
|scored_0238|183|163|162|162|162|128-1|183|182|172|175|181|174|180|170|162|176|
|scored_0239|212|188|188|188|188|128-2|212|212|204|219|220|213|217|211|190|188|
|scored_0240|152|139|139|139|139|128-1|152|154|146|143|152|156|153|147|139|139|
|scored_0241|39|39|39|39|39|hist|39|41|39|41|44|40|40|44|40|41|
|scored_0242|201|180|180|180|180|128-1|201|201|208|210|202|196|194|221|180|193|
|scored_0243|148|138|137|137|137|128-2|148|153|145|146|156|154|161|159|151|137|
|scored_0244|232|217|208|208|208|128-2|232|232|229|222|242|241|243|219|226|208|
|scored_0245|142|132|132|132|132|128-1|142|142|141|138|148|145|145|135|132|135|
|scored_0246|167|159|159|159|159|128-2|167|167|166|162|179|168|182|172|164|159|
|scored_0247|173|156|153|153|153|128-2|173|176|165|156|179|176|169|192|172|153|
|scored_0248|217|204|200|200|200|128-1|217|217|210|209|222|213|222|219|200|200|
|scored_0249|322|297|291|291|291|128-1|322|322|307|444|327|316|323|292|291|344|
|scored_0250|31|31|31|31|31|hist|31|34|33|32|33|33|32|33|33|32|
|scored_0251|267|250|246|246|246|128-1|267|272|290|463|275|272|273|252|246|249|
|scored_0252|88|82|81|81|81|128-2|88|88|87|83|86|86|84|87|86|81|
|scored_0253|97|94|92|92|92|128-0|97|99|97|95|94|96|95|92|93|93|
|scored_0254|286|253|246|246|246|128-2|286|286|273|355|289|287|284|264|251|246|
|scored_0255|346|301|301|301|301|128-1|346|346|466|390|346|345|348|322|301|307|
|scored_0256|112|101|100|100|100|48-2|112|113|112|103|113|109|100|106|106|101|
|scored_0257|241|213|213|213|213|128-1|241|242|230|228|247|241|242|243|213|218|
|scored_0258|342|312|303|303|303|128-2|342|342|369|542|342|338|338|338|335|303|
|scored_0259|133|127|127|127|127|128-1|133|134|134|132|142|141|135|131|127|134|
|scored_0260|112|110|109|109|109|128-1|112|118|114|112|113|114|114|112|109|111|
|scored_0261|213|192|192|192|192|128-1|213|214|204|251|217|207|208|210|192|247|
|scored_0262|277|273|262|262|262|128-0|277|277|391|282|285|273|282|262|270|277|
|scored_0263|299|264|263|263|263|128-2|299|299|293|357|300|298|315|291|328|263|
|scored_0264|269|238|238|238|238|128-1|269|269|261|312|287|268|276|265|238|245|
|scored_0265|233|224|222|222|222|128-1|233|239|235|288|251|245|248|224|222|226|
|scored_0266|142|127|127|127|127|128-2|142|143|142|135|148|139|144|146|141|127|
|scored_0267|143|135|135|135|135|128-1|143|148|138|137|164|137|149|140|135|136|
|scored_0268|260|241|235|235|235|128-1|260|260|248|286|267|256|258|244|235|236|
|scored_0269|347|322|318|318|318|128-0|347|347|600|422|359|352|354|318|336|364|
|scored_0270|152|148|146|146|146|128-1|152|157|148|147|166|157|156|148|146|163|
|scored_0271|56|54|52|52|52|48-0|56|58|56|53|52|56|54|54|56|54|
|scored_0272|32|32|30|30|30|48-1|32|32|33|33|32|30|30|32|30|30|
|scored_0273|230|216|216|216|216|0-1|230|231|216|261|231|223|221|226|220|232|
|scored_0274|207|204|193|193|191|128-2|207|211|208|201|222|213|214|216|204|191|
|scored_0275|183|164|164|164|164|128-2|183|184|179|213|183|182|188|178|190|164|
|scored_0276|90|86|86|86|86|128-2|90|90|88|88|90|89|87|91|87|86|
|scored_0277|170|156|154|154|154|128-1|170|171|168|156|192|171|174|170|154|165|
|scored_0278|101|97|97|97|97|128-2|101|104|105|100|104|104|99|100|102|97|
|scored_0279|320|292|282|282|282|128-2|320|320|416|394|324|330|339|310|431|282|
|scored_0280|184|164|164|164|164|128-2|184|190|179|175|224|184|198|184|175|164|
|scored_0281|230|219|213|213|213|128-2|230|230|219|325|230|224|224|230|239|213|
|scored_0282|60|57|55|55|55|0-2|60|61|59|55|63|59|56|61|61|57|
|scored_0283|269|256|253|253|253|128-2|269|273|270|337|270|275|258|254|254|253|
|scored_0284|176|164|164|164|164|128-1|176|176|175|171|205|181|189|179|164|176|
|scored_0285|256|236|235|235|235|128-1|256|256|260|297|261|261|266|250|235|257|
|scored_0286|155|148|148|148|148|128-2|155|162|159|159|176|160|170|163|155|148|
|scored_0287|243|205|205|205|205|128-2|243|244|290|264|249|237|241|237|209|205|
|scored_0288|167|154|153|153|153|128-1|167|170|167|163|177|168|174|172|153|171|
|scored_0289|24|24|24|24|24|hist|24|25|25|25|30|30|30|30|30|30|
|scored_0290|181|174|174|174|174|0-2|181|181|175|174|189|182|176|179|176|180|
|scored_0291|203|188|187|187|187|0-2|203|207|202|187|214|210|208|207|254|205|
|scored_0292|66|65|61|61|61|128-1|66|66|65|65|66|63|64|67|61|63|
|scored_0293|245|218|218|218|218|128-1|245|245|235|235|250|247|246|243|218|226|
|scored_0294|194|177|177|177|177|128-2|194|194|191|191|204|199|193|190|181|177|
|scored_0295|163|143|142|142|142|128-2|163|163|162|150|172|173|165|182|155|142|
|scored_0296|54|53|52|52|52|48-0|54|54|54|53|52|52|53|54|54|55|
|scored_0297|153|143|143|143|143|128-2|153|154|152|150|160|147|153|158|152|143|
|scored_0298|126|119|118|118|118|48-2|126|127|126|121|131|126|118|128|123|119|
|scored_0299|116|108|107|107|107|128-2|116|116|111|111|114|111|111|113|109|107|
|scored_0300|54|51|48|48|48|0-2|54|56|55|48|53|51|49|56|55|49|
|scored_0301|4|4|4|4|4|hist|4|4|4|4|4|4|4|4|4|4|
|scored_0302|200|170|170|170|170|128-2|200|207|190|190|224|204|208|193|174|170|
|scored_0303|255|229|226|226|226|128-2|255|255|295|285|264|251|245|247|248|226|
|scored_0304|196|181|178|178|178|128-1|196|197|195|181|206|199|200|194|178|203|
|scored_0305|204|183|183|183|183|128-1|204|204|192|192|221|205|210|205|183|189|
|scored_0306|328|294|294|294|294|128-2|328|328|327|363|329|337|326|335|351|294|
|scored_0307|163|140|140|140|140|128-2|163|165|152|150|181|165|151|150|146|140|
|scored_0308|42|42|41|41|41|48-1|42|44|42|42|42|41|41|42|41|41|
|scored_0309|162|150|149|149|149|128-2|162|162|161|150|184|176|167|162|154|149|
|scored_0310|145|139|139|139|139|0-2|145|146|147|139|168|153|144|149|145|142|
|scored_0311|134|127|127|127|127|0-2|134|135|136|127|142|141|138|137|131|133|
|scored_0312|307|287|280|280|280|128-1|307|307|328|425|315|316|316|284|280|295|
|scored_0313|212|202|191|191|191|128-2|212|212|202|238|217|208|209|203|212|191|
|scored_0314|243|208|208|208|208|128-2|243|243|232|263|250|244|240|221|217|208|
|scored_0315|283|246|245|245|245|128-1|283|283|275|356|288|285|284|311|245|297|
|scored_0316|141|128|128|128|128|128-2|141|141|138|131|146|138|135|134|136|128|
|scored_0317|224|204|201|201|201|128-2|224|224|210|227|223|216|220|226|220|201|
|scored_0318|226|212|208|208|208|128-0|226|226|222|272|226|221|228|208|214|250|
|scored_0319|281|263|251|251|251|128-1|281|282|316|302|281|278|276|263|251|259|
|scored_0320|40|38|37|37|37|0-1|40|40|37|38|41|40|40|41|40|40|
|scored_0321|340|321|314|314|314|128-1|340|340|548|501|341|346|342|324|314|326|
|scored_0322|219|194|193|193|193|128-1|219|222|270|251|225|217|222|210|193|196|
|scored_0323|183|172|170|170|170|128-1|183|184|181|174|210|175|187|188|170|187|
|scored_0324|223|210|203|203|203|128-0|223|224|216|226|220|222|218|203|209|206|
|scored_0325|154|144|143|143|143|0-1|154|154|143|149|167|155|158|155|149|145|
|scored_0326|176|167|166|166|166|0-2|176|182|172|166|203|186|190|203|179|171|
|scored_0327|164|155|155|155|155|0-2|164|164|164|155|171|171|164|169|184|156|
|scored_0328|286|250|249|249|249|128-1|286|286|321|365|294|281|293|287|249|267|
|scored_0329|156|150|150|150|150|0-2|156|158|157|150|159|160|154|155|161|157|
|scored_0330|194|164|164|164|164|128-1|194|214|188|190|224|221|186|217|164|176|
|scored_0331|309|270|268|268|268|128-2|309|309|451|590|306|305|314|342|271|268|
|scored_0332|181|165|165|165|165|128-1|181|181|181|171|185|181|184|184|165|176|
|scored_0333|264|253|247|247|247|128-2|264|266|253|405|269|267|265|256|297|247|
|scored_0334|285|252|252|252|252|128-1|285|285|296|385|299|287|296|281|252|259|
|scored_0335|232|211|211|211|211|128-2|232|233|235|240|244|237|250|218|227|211|
|scored_0336|189|180|180|180|180|0-1|189|193|180|183|199|190|199|184|182|183|
|scored_0337|150|147|146|146|146|0-2|150|154|153|146|160|153|153|157|154|161|
|scored_0338|221|199|198|198|198|128-1|221|221|408|244|226|215|219|207|198|211|
|scored_0339|112|111|110|110|110|0-2|112|117|117|110|120|115|113|116|113|113|
|scored_0340|144|129|128|128|128|128-2|144|147|139|135|158|137|141|136|133|128|
|scored_0341|173|146|146|146|146|128-2|173|182|166|159|172|174|161|182|166|146|
|scored_0342|269|241|241|241|241|128-1|269|269|258|346|278|277|280|259|241|255|
|scored_0343|143|133|132|132|132|128-2|143|143|138|137|145|140|139|144|133|132|
|scored_0344|202|184|184|184|184|128-2|202|203|195|192|203|196|190|199|198|184|
|scored_0345|245|242|224|224|224|128-2|245|246|243|352|254|257|256|247|286|224|
|scored_0346|213|192|189|189|189|128-2|213|213|210|266|221|203|211|208|203|189|
|scored_0347|141|135|135|135|135|128-2|141|145|143|140|162|149|154|143|145|135|
|scored_0348|3|3|3|3|3|hist|3|3|3|3|3|3|3|3|3|3|
|scored_0349|235|228|218|218|218|128-2|235|235|227|251|235|236|239|229|230|218|
|scored_0350|73|69|68|68|68|48-2|73|73|72|70|70|69|68|71|70|71|
|scored_0351|198|183|180|180|180|128-2|198|202|191|229|200|190|200|199|192|180|
|scored_0352|288|269|260|260|260|128-0|288|288|319|291|285|288|284|260|270|262|
|scored_0353|181|169|169|169|169|128-1|181|183|176|172|184|183|184|193|169|169|
|scored_0354|180|172|172|172|172|128-1|180|189|180|178|236|182|199|184|172|172|
|scored_0355|147|138|136|136|136|128-2|147|148|142|141|156|150|144|150|146|136|
|scored_0356|102|101|100|100|100|0-1|102|104|100|103|106|103|103|102|101|101|
|scored_0357|285|271|265|265|265|128-1|285|285|302|321|290|282|284|279|265|268|
|scored_0358|180|165|165|165|165|128-1|180|188|191|171|218|190|198|192|165|173|
|scored_0359|142|134|131|131|131|128-2|142|145|137|136|155|140|139|143|133|131|
|scored_0360|160|153|153|153|153|0-2|160|161|155|153|162|160|166|169|156|157|
|scored_0361|222|192|190|190|190|128-2|222|226|207|205|237|224|229|211|205|190|
|scored_0362|139|130|130|130|130|128-2|139|142|135|133|148|136|143|140|132|130|
|scored_0363|205|192|187|187|187|128-1|205|205|192|192|208|203|203|199|187|189|
|scored_0364|217|178|178|178|178|128-2|217|217|186|196|231|217|195|214|195|178|
|scored_0365|4|4|4|4|4|hist|4|4|4|4|4|4|4|4|4|4|
|scored_0366|145|143|143|143|143|128-1|145|151|145|147|160|154|156|150|143|146|
|scored_0367|149|139|139|139|139|0-1|149|153|139|142|153|150|152|144|153|166|
|scored_0368|34|33|33|33|33|0-1|34|39|33|42|38|33|41|39|33|41|
|scored_0369|294|250|248|248|248|128-1|294|298|337|359|303|293|301|278|248|263|
|scored_0370|327|301|292|292|292|128-2|327|327|327|405|341|326|334|322|294|292|
|scored_0371|157|146|146|146|146|128-2|157|161|157|152|183|165|165|158|150|146|
|scored_0372|141|130|130|130|130|128-2|141|143|139|131|147|144|138|138|136|130|
|scored_0373|273|233|233|233|233|128-2|273|281|261|338|283|284|278|251|240|233|
|scored_0374|273|241|235|235|235|128-1|273|273|298|337|278|270|281|251|235|239|
|scored_0375|263|232|232|232|232|128-1|263|263|257|270|271|265|272|246|232|232|
|scored_0376|33|33|33|33|33|hist|33|33|33|33|33|35|34|33|35|34|
|scored_0377|104|100|99|99|99|0-2|104|105|106|99|110|111|109|107|108|101|
|scored_0378|104|98|98|98|98|128-2|104|106|106|100|103|102|104|102|101|98|
|scored_0379|145|137|137|137|137|128-0|145|148|146|145|150|148|147|137|142|139|
|scored_0380|195|188|183|183|183|128-1|195|195|192|188|202|195|203|219|183|219|
|scored_0381|243|234|227|227|227|128-2|243|244|246|303|253|246|250|234|254|227|
|scored_0382|178|167|167|167|167|128-1|178|179|177|171|183|175|174|188|167|168|
|scored_0383|182|171|168|168|168|128-1|182|183|175|175|203|189|198|171|168|172|
|scored_0384|180|166|165|165|165|128-1|180|180|178|175|188|182|181|192|165|169|
|scored_0385|187|163|163|163|163|128-2|187|188|187|183|200|188|186|183|164|163|
|scored_0386|240|226|216|216|216|128-2|240|244|295|229|246|248|246|224|226|216|
|scored_0387|32|32|32|32|32|hist|32|37|35|34|35|36|36|35|36|36|
|scored_0388|30|25|25|25|25|48-2|30|30|28|28|32|26|25|32|26|25|
|scored_0389|219|201|196|196|196|128-2|219|219|210|201|221|209|216|218|199|196|
|scored_0390|125|119|118|118|118|128-1|125|128|124|123|140|125|133|127|118|120|
|scored_0391|269|233|231|231|231|128-2|269|270|273|247|269|264|269|260|266|231|
|scored_0392|142|131|130|130|130|128-2|142|144|139|138|149|143|138|135|133|130|
|scored_0393|34|34|33|33|33|0-2|34|34|34|33|36|35|34|36|35|34|
|scored_0394|171|162|162|162|162|0-2|171|172|164|162|182|180|170|206|168|167|
|scored_0395|112|109|109|109|109|0-2|112|114|118|109|116|118|115|112|112|109|
|scored_0396|48|48|47|47|47|0-2|48|48|50|47|51|50|47|50|51|48|
|scored_0397|239|219|218|218|218|128-1|239|240|279|227|239|248|241|255|218|236|
|scored_0398|254|222|220|220|220|128-2|254|255|271|357|261|253|252|244|230|220|
|scored_0399|28|28|28|28|28|hist|28|31|30|30|31|32|31|31|32|31|
|scored_0400|284|252|251|251|251|128-1|284|305|274|359|284|267|271|270|251|257|
|scored_0401|38|37|34|34|34|0-2|38|38|41|34|38|38|34|38|38|34|
|scored_0402|118|109|108|108|108|128-2|118|118|116|112|118|118|114|112|111|108|
|scored_0403|291|283|271|271|271|128-1|291|295|502|359|293|293|288|272|271|277|
|scored_0404|130|122|122|122|122|128-2|130|130|129|128|138|134|131|124|126|122|
|scored_0405|336|297|297|297|297|128-1|336|336|387|457|347|347|352|316|297|314|
|scored_0406|172|154|154|154|154|0-2|172|175|163|154|186|169|176|162|173|162|
|scored_0407|198|186|186|186|186|0-2|198|198|195|186|200|199|202|216|198|298|
|scored_0408|206|189|187|187|187|128-2|206|206|196|203|213|202|206|202|188|187|
|scored_0409|30|30|30|30|30|hist|30|32|33|32|34|32|31|34|32|31|
|scored_0410|225|211|206|206|206|128-1|225|228|214|220|230|227|229|225|206|212|
|scored_0411|225|197|197|197|197|128-2|225|226|268|258|226|230|222|226|205|197|
|scored_0412|218|198|198|198|198|128-1|218|218|211|203|224|216|223|220|198|199|
|scored_0413|189|170|170|170|170|128-2|189|191|188|187|194|198|196|179|175|170|
|scored_0414|161|142|142|142|142|128-2|161|162|155|145|170|153|159|147|151|142|
|scored_0415|187|169|169|169|169|128-1|187|187|179|175|196|191|189|183|169|172|
|scored_0416|224|186|186|186|186|128-2|224|227|216|224|244|217|233|220|201|186|
|scored_0417|213|188|188|188|188|128-1|213|214|204|211|216|210|217|232|188|192|
|scored_0418|148|140|139|139|139|128-2|148|149|144|147|157|148|151|145|140|139|
|scored_0419|31|30|29|29|29|0-2|31|31|31|29|31|31|29|31|31|29|
|scored_0420|190|175|175|175|175|0-2|190|191|189|175|197|193|190|200|186|188|
|scored_0421|195|162|162|162|162|128-2|195|196|188|178|209|183|186|186|169|162|
|scored_0422|200|198|193|193|193|128-0|200|200|198|207|208|202|203|193|196|248|
|scored_0423|115|112|112|112|112|128-2|115|116|114|113|121|117|117|113|114|112|
|scored_0424|275|252|248|248|248|128-2|275|275|373|386|276|284|284|250|249|248|
|scored_0425|36|36|36|36|36|hist|36|39|39|39|38|38|38|38|38|38|
|scored_0426|170|167|163|163|163|128-2|170|170|167|169|175|172|183|184|174|163|
|scored_0427|175|171|169|169|169|128-1|175|183|173|171|215|179|202|185|169|177|
|scored_0428|264|232|231|231|231|128-1|264|264|266|300|271|262|268|244|231|237|
|scored_0429|138|133|133|133|133|128-1|138|141|139|134|145|148|141|138|133|134|
|scored_0430|110|109|109|109|109|0-2|110|110|111|109|115|114|116|110|109|109|
|scored_0431|106|96|96|96|96|128-2|106|106|101|105|110|105|99|104|102|96|
|scored_0432|169|152|152|152|152|128-2|169|172|170|162|174|175|167|164|166|152|
|scored_0433|163|158|158|158|158|0-2|163|164|161|158|178|170|171|172|186|163|
|scored_0434|135|127|127|127|127|0-2|135|137|135|127|138|134|133|164|136|136|
|scored_0435|191|173|173|173|173|128-1|191|191|189|180|198|203|199|192|173|179|
|scored_0436|170|158|158|158|158|128-2|170|181|170|169|222|176|175|181|167|158|
|scored_0437|209|186|185|185|185|128-2|209|209|200|189|225|204|222|199|186|185|
|scored_0438|245|233|221|221|221|128-1|245|245|280|273|257|241|255|228|221|237|
|scored_0439|192|176|175|175|175|128-1|192|198|202|185|234|196|192|195|175|180|
|scored_0440|237|214|209|208|208|128-2|237|237|305|240|239|233|232|213|219|208|
|scored_0441|226|199|199|199|199|128-2|226|226|220|207|229|220|228|225|209|199|
|scored_0442|219|200|199|199|199|128-1|219|220|220|229|220|220|222|208|199|199|
|scored_0443|45|43|41|41|41|0-2|45|45|44|41|43|42|42|43|42|42|
|scored_0444|298|262|261|261|261|128-1|298|298|293|317|309|299|306|275|261|278|
|scored_0445|289|253|252|252|252|128-1|289|292|320|457|294|296|289|290|252|273|
|scored_0446|310|301|289|289|289|128-0|310|310|383|437|315|310|301|289|351|348|
|scored_0447|272|250|247|247|247|128-1|272|272|294|330|278|277|279|282|247|285|
|scored_0448|260|223|222|222|222|128-1|260|260|242|279|252|250|258|234|222|232|
|scored_0449|123|115|114|114|114|0-2|123|125|118|114|126|124|118|116|115|115|
|scored_0450|342|300|300|300|300|128-1|342|342|395|475|353|341|348|338|300|345|
|scored_0451|146|137|137|137|137|0-2|146|148|144|137|149|152|148|151|140|141|
|scored_0452|253|224|223|223|223|128-1|253|254|257|249|260|257|255|247|223|225|
|scored_0453|177|165|164|164|164|128-2|177|185|165|174|202|193|167|170|166|164|
|scored_0454|258|223|223|223|223|128-1|258|258|255|286|260|263|265|238|223|231|
|scored_0455|37|37|37|37|37|hist|37|37|41|42|37|43|41|37|43|41|
|scored_0456|148|144|143|143|143|128-0|148|149|145|145|158|157|156|143|143|161|
|scored_0457|174|154|154|154|154|128-2|174|174|181|159|190|166|173|165|169|154|
|scored_0458|188|172|168|168|168|128-2|188|189|178|177|196|184|182|178|171|168|
|scored_0459|105|101|101|100|100|0-2|105|108|110|100|113|107|103|107|108|102|
|scored_0460|202|173|173|173|173|128-2|202|202|196|241|208|200|202|196|184|173|
|scored_0461|45|44|41|41|41|48-2|45|45|45|43|45|44|41|45|44|41|
|scored_0462|144|143|143|143|143|0-2|144|146|147|143|154|154|148|148|163|158|
|scored_0463|131|126|126|126|126|128-1|131|132|131|129|141|128|132|131|126|127|
|scored_0464|99|95|94|94|94|0-2|99|101|98|94|107|101|102|101|100|97|
|scored_0465|175|157|157|157|157|128-1|175|177|170|169|215|173|180|171|157|157|
|scored_0466|231|209|209|209|209|128-1|231|231|314|244|235|227|237|217|209|223|
|scored_0467|119|117|115|115|115|128-1|119|122|123|118|128|124|119|120|115|124|
|scored_0468|193|175|173|173|173|128-2|193|193|185|183|194|189|192|179|175|173|
|scored_0469|107|103|102|102|102|48-2|107|110|112|106|108|107|102|104|106|107|
|scored_0470|327|308|293|293|293|128-1|327|327|408|360|331|323|324|311|293|373|
|scored_0471|124|117|117|117|117|128-2|124|125|127|122|131|127|126|123|123|117|
|scored_0472|30|27|25|25|25|0-2|30|30|29|25|35|30|28|35|30|28|
|scored_0473|35|35|33|33|33|0-2|35|36|36|33|35|35|35|35|35|35|
|scored_0474|106|106|104|104|104|128-0|106|109|109|109|117|116|106|104|106|106|
|scored_0475|248|228|227|227|227|128-2|248|248|239|299|254|248|248|230|254|227|
|scored_0476|262|249|242|242|242|128-2|262|275|255|252|269|257|268|250|247|242|
|scored_0477|206|198|192|192|192|128-1|206|209|204|218|211|204|208|196|192|192|
|scored_0478|98|94|94|94|94|128-0|98|98|95|97|95|95|97|94|96|96|
|scored_0479|142|136|136|136|136|128-1|142|147|145|141|153|149|147|141|136|171|
|scored_0480|182|173|172|172|172|128-2|182|184|181|178|191|183|190|182|190|172|
|scored_0481|224|207|205|205|205|128-1|224|224|220|261|230|218|231|240|205|228|
|scored_0482|234|207|207|207|207|128-1|234|235|220|302|234|230|232|216|207|215|
|scored_0483|205|187|187|187|187|128-1|205|210|200|210|214|202|208|223|187|195|
|scored_0484|176|164|164|164|164|0-2|176|176|172|164|181|170|178|196|178|168|
|scored_0485|2|2|2|2|2|hist|2|2|2|2|2|2|2|2|2|2|
|scored_0486|139|126|125|125|125|128-2|139|143|134|133|144|137|133|134|134|125|
|scored_0487|37|37|37|37|37|hist|37|44|44|39|43|42|41|43|42|41|
|scored_0488|176|175|175|175|175|0-1|176|179|175|180|183|181|181|183|203|182|
|scored_0489|178|163|163|163|163|128-2|178|179|178|172|186|187|184|184|187|163|
|scored_0490|4|4|4|4|4|hist|4|4|4|4|6|6|6|6|6|6|
|scored_0491|236|220|210|210|210|128-2|236|236|281|232|243|239|232|232|214|210|
|scored_0492|51|46|45|45|45|0-2|51|51|50|45|46|48|45|46|48|45|
|scored_0493|301|275|268|268|268|128-2|301|301|310|344|305|306|303|315|285|268|
|scored_0494|277|262|258|258|258|128-1|277|277|262|346|282|277|279|266|258|268|
|scored_0495|141|128|127|127|127|128-0|141|144|137|131|153|142|139|127|130|133|
|scored_0496|160|152|152|152|152|0-2|160|166|157|152|184|172|171|162|153|164|
|scored_0497|213|195|195|195|195|128-1|213|218|206|204|213|214|222|201|195|211|
|scored_0498|192|170|170|170|170|128-1|192|193|192|208|237|205|204|174|170|173|
|scored_0499|174|155|155|155|155|128-1|174|174|173|168|239|180|183|179|155|163|
|scored_0500|101|101|101|101|101|hist|101|106|106|101|106|108|105|106|107|104|
|scored_0501|170|153|153|153|153|128-1|170|171|161|161|180|167|181|184|153|153|
|scored_0502|200|183|182|182|182|128-2|200|202|195|201|205|195|206|198|191|182|
|scored_0503|105|103|103|103|103|0-2|105|107|104|103|109|103|103|104|103|103|
|scored_0504|238|225|219|219|219|128-0|238|238|267|268|232|225|232|219|230|224|
|scored_0505|150|146|142|142|142|128-2|150|153|152|147|158|154|151|148|145|142|
|scored_0506|147|138|138|138|138|0-2|147|147|147|138|155|150|141|162|168|142|
|scored_0507|205|193|190|190|190|128-2|205|205|196|274|207|206|203|192|203|190|
|scored_0508|252|242|235|235|235|128-0|252|254|253|251|259|248|249|235|241|237|
|scored_0509|166|151|151|151|151|128-1|166|166|158|158|192|171|172|176|151|151|
|scored_0510|106|106|104|104|104|128-0|106|107|106|106|115|110|111|104|105|108|
|scored_0511|304|304|302|302|302|48-1|304|306|341|388|315|302|304|307|307|302|
|scored_0512|187|174|174|174|174|0-2|187|187|178|174|216|183|197|204|174|182|
|scored_0513|183|169|169|169|169|128-1|183|183|179|172|189|183|182|174|169|174|
|scored_0514|267|251|245|245|245|128-2|267|267|291|294|274|259|262|249|250|245|
|scored_0515|38|38|37|37|37|48-1|38|38|40|40|38|37|42|38|39|42|
|scored_0516|199|190|186|186|185|128-2|199|199|190|212|198|196|199|191|214|185|
|scored_0517|198|172|172|172|172|128-2|198|198|189|182|210|203|200|213|193|172|
|scored_0518|244|215|213|213|213|128-2|244|245|231|229|246|251|248|246|222|213|
|scored_0519|112|112|110|110|110|128-2|112|112|114|111|118|120|115|117|117|110|
|scored_0520|219|213|209|209|208|128-2|219|219|228|227|220|215|212|219|226|208|
|scored_0521|195|176|176|176|176|0-2|195|198|182|176|231|199|213|197|187|190|
|scored_0522|113|109|106|106|106|128-2|113|118|115|112|114|111|116|111|110|106|
|scored_0523|154|140|139|138|138|128-2|154|154|150|139|164|156|145|149|144|138|
|scored_0524|156|142|142|142|142|128-2|156|160|157|154|195|173|175|157|145|142|
|scored_0525|150|136|136|136|136|0-2|150|152|145|136|155|147|149|175|136|136|
|scored_0526|265|228|228|228|228|128-2|265|265|278|293|265|266|266|247|237|228|
|scored_0527|276|245|243|243|243|128-1|276|276|259|313|297|267|276|279|243|246|
|scored_0528|170|168|168|168|168|0-1|170|170|168|168|180|176|175|174|183|171|
|scored_0529|186|170|169|169|169|128-1|186|186|187|183|213|189|190|176|169|173|
|scored_0530|194|179|178|178|178|128-1|194|196|188|186|204|194|194|204|178|210|
|scored_0531|238|210|210|210|210|128-1|238|238|229|273|246|234|241|238|210|244|
|scored_0532|102|98|94|94|94|48-2|102|108|101|95|109|101|94|106|101|96|
|scored_0533|143|134|134|134|134|0-2|143|144|139|134|139|146|146|141|161|135|
|scored_0534|161|147|147|147|147|128-2|161|161|156|154|176|159|166|161|159|147|
|scored_0535|167|155|154|154|154|128-1|167|167|169|160|181|163|179|162|154|156|
|scored_0536|308|269|269|269|269|128-2|308|309|298|386|303|308|306|282|290|269|
|scored_0537|2|2|2|2|2|hist|2|2|2|2|2|2|2|2|2|2|
|scored_0538|210|205|200|200|200|128-2|210|210|205|204|212|215|213|217|204|200|
|scored_0539|117|111|109|109|109|128-2|117|119|113|111|123|115|120|116|110|109|
|scored_0540|212|199|196|196|196|128-1|212|214|199|246|214|211|211|206|196|196|
|scored_0541|69|63|63|63|63|48-2|69|68|68|65|66|65|63|68|68|67|
|scored_0542|136|129|129|129|129|128-1|136|139|134|132|141|133|135|132|129|129|
|scored_0543|127|119|116|116|116|128-1|127|127|118|119|131|121|121|122|116|119|
|scored_0544|217|201|196|196|196|128-2|217|219|210|249|223|213|216|214|213|196|
|scored_0545|138|127|127|127|127|128-2|138|140|134|133|142|138|138|133|130|127|
|scored_0546|184|168|168|168|168|128-1|184|184|178|174|192|184|185|180|168|183|
|scored_0547|132|125|125|125|125|128-1|132|133|132|129|143|132|136|143|125|126|
|scored_0548|48|48|46|46|45|0-2|48|48|50|45|49|49|47|49|49|47|
|scored_0549|139|131|128|128|128|128-2|139|139|138|131|145|141|135|142|133|128|
|scored_0550|133|128|128|128|128|0-2|133|133|134|128|160|137|133|132|130|128|
|scored_0551|220|198|196|196|196|128-1|220|220|211|215|221|212|215|246|196|203|
|scored_0552|271|255|250|250|250|128-1|271|271|260|273|280|271|279|263|250|252|
|scored_0553|219|195|193|193|193|128-2|219|219|210|219|236|208|221|209|195|193|
|scored_0554|139|125|124|124|124|128-2|139|139|131|128|152|139|131|136|127|124|
|scored_0555|153|145|145|145|145|128-1|153|157|155|148|174|158|157|160|145|152|
|scored_0556|36|36|36|36|36|hist|36|40|38|38|40|39|39|40|39|39|
|scored_0557|161|141|140|140|140|128-1|161|163|161|151|161|164|166|146|140|146|
|scored_0558|148|141|141|141|141|128-2|148|153|145|145|151|152|149|155|148|141|
|scored_0559|45|45|44|44|44|48-2|45|45|46|45|47|51|44|47|51|44|
|scored_0560|48|48|47|45|45|48-2|48|53|49|47|52|50|45|53|50|45|
|scored_0561|120|116|116|116|116|0-2|120|123|125|116|129|124|126|121|121|117|
|scored_0562|116|108|108|108|108|0-2|116|116|113|108|118|114|116|113|110|111|
|scored_0563|131|127|126|125|125|128-2|131|133|131|129|139|130|138|143|134|125|
|scored_0564|36|36|36|36|36|hist|36|39|40|37|38|40|37|38|40|37|
|scored_0565|264|235|234|234|234|128-2|264|264|251|317|270|271|265|271|245|234|
|scored_0566|30|30|29|29|29|0-2|30|30|33|29|30|31|30|30|31|30|
|scored_0567|137|130|129|129|129|128-1|137|145|135|136|172|139|153|136|129|135|
|scored_0568|140|132|132|132|132|128-2|140|140|138|138|156|152|148|146|141|132|
|scored_0569|187|171|171|171|171|128-1|187|190|187|177|208|190|200|173|171|185|
|scored_0570|110|100|100|100|100|0-2|110|113|106|100|111|105|109|104|106|103|
|scored_0571|240|231|229|229|229|128-1|240|240|231|242|242|238|240|243|229|237|
|scored_0572|286|253|252|252|252|128-1|286|286|293|316|287|275|283|266|252|261|
|scored_0573|231|207|205|205|205|128-2|231|231|224|255|234|225|231|247|285|205|
|scored_0574|126|121|121|121|121|0-2|126|130|124|121|130|131|134|124|127|134|
|scored_0575|158|142|140|140|140|128-2|158|165|153|149|184|147|172|168|146|140|
|scored_0576|207|193|193|193|193|0-1|207|208|193|193|211|202|202|196|228|197|
|scored_0577|209|174|173|173|173|128-2|209|211|206|188|222|213|206|192|178|173|
|scored_0578|152|136|136|136|136|128-2|152|154|143|137|168|145|158|147|141|136|
|scored_0579|203|203|198|198|198|128-2|203|203|206|210|212|203|204|215|205|198|
|scored_0580|282|246|246|246|246|128-1|282|282|275|304|291|283|289|254|246|257|
|scored_0581|312|274|273|273|273|128-1|312|314|410|415|319|314|318|323|273|289|
|scored_0582|226|201|201|201|201|128-1|226|228|220|214|220|230|227|206|201|203|
|scored_0583|50|50|48|48|48|48-0|50|50|50|49|48|48|52|48|48|52|
|scored_0584|57|57|56|56|56|48-1|57|57|57|58|60|56|58|61|60|59|
|scored_0585|226|204|203|203|203|128-1|226|226|231|212|227|218|220|219|203|210|
|scored_0586|105|102|102|102|102|128-2|105|107|106|107|108|113|105|109|106|102|
|scored_0587|182|180|166|166|166|128-1|182|185|180|179|189|178|186|182|166|196|
|scored_0588|168|163|158|158|158|128-2|168|171|167|163|178|175|171|187|172|158|
|scored_0589|111|102|102|102|102|128-2|111|113|113|105|107|107|106|107|107|102|
|scored_0590|170|151|151|151|151|128-2|170|172|167|163|207|166|173|168|160|151|
|scored_0591|153|146|146|146|146|128-1|153|156|156|152|171|162|164|150|146|160|
|scored_0592|259|228|228|228|228|128-2|259|259|252|261|258|260|265|235|234|228|
|scored_0593|282|260|260|260|260|128-1|282|282|354|343|296|281|286|270|260|267|
|scored_0594|284|275|267|267|267|128-0|284|284|318|270|283|290|289|267|272|268|
|scored_0595|144|129|129|129|129|0-2|144|145|139|129|146|145|136|145|140|130|
|scored_0596|152|132|132|132|132|128-2|152|153|144|135|166|144|152|146|138|132|
|scored_0597|3|3|3|3|3|hist|3|3|3|3|3|3|3|3|3|3|
|scored_0598|99|97|97|97|97|0-2|99|101|103|97|102|106|102|104|101|99|
|scored_0599|196|171|170|170|170|128-2|196|196|189|182|244|210|209|191|179|170|
|scored_0600|3|3|3|3|3|hist|3|3|3|3|3|3|3|3|3|3|
|scored_0601|146|142|142|142|142|0-2|146|147|146|142|153|152|149|166|158|145|
|scored_0602|132|123|123|123|123|128-2|132|135|125|125|145|132|128|126|126|123|
|scored_0603|186|179|174|174|174|128-2|186|190|180|178|206|202|181|185|178|174|
|scored_0604|157|141|140|140|140|128-2|157|160|151|144|177|160|157|166|147|140|
|scored_0605|207|191|185|185|185|128-2|207|207|197|203|208|201|209|191|225|185|
|scored_0606|205|189|189|189|189|128-0|205|207|199|207|208|204|202|189|195|194|
|scored_0607|154|139|138|138|138|128-2|154|156|154|146|165|158|152|170|151|138|
|scored_0608|187|174|170|170|170|128-2|187|187|182|174|189|190|181|181|173|170|
|scored_0609|188|171|170|170|170|128-1|188|188|180|176|195|191|185|183|170|184|
|scored_0610|335|291|291|291|291|128-1|335|336|352|608|350|353|342|311|291|303|
|scored_0611|39|39|39|39|39|hist|39|40|40|41|40|40|41|40|40|41|
|scored_0612|321|292|289|289|289|128-1|321|321|377|356|323|322|334|290|289|315|
|scored_0613|131|121|119|119|119|48-2|131|132|130|125|141|126|119|131|121|121|
|scored_0614|26|26|26|26|26|hist|26|30|31|31|31|31|31|31|31|31|
|scored_0615|140|133|133|133|133|0-2|140|139|139|133|142|141|143|146|134|135|
|scored_0616|185|167|167|165|165|128-2|185|186|176|168|213|185|190|179|179|165|
|scored_0617|125|123|122|122|122|0-1|125|127|122|127|137|130|136|127|124|124|
|scored_0618|129|124|124|124|124|128-1|129|130|127|129|135|129|134|128|124|124|
|scored_0619|191|171|171|171|171|128-1|191|193|180|178|217|199|195|193|171|172|
|scored_0620|50|49|48|48|48|48-1|50|51|49|49|51|48|49|52|48|49|
|scored_0621|214|208|206|206|206|128-2|214|217|208|216|220|209|207|229|214|206|
|scored_0622|201|201|194|194|194|128-2|201|201|202|205|211|204|209|197|201|194|
|scored_0623|102|96|96|96|96|0-2|102|102|97|96|101|99|103|103|97|96|
|scored_0624|143|133|132|132|132|128-2|143|144|144|140|153|149|145|149|146|132|
|scored_0625|119|117|116|116|116|128-2|119|121|121|118|121|122|120|117|117|116|
|scored_0626|211|187|186|186|186|128-1|211|218|224|242|216|218|214|206|186|188|
|scored_0627|159|152|151|151|151|48-1|159|169|155|163|178|151|168|162|168|160|
|scored_0628|255|222|221|221|221|128-2|255|256|244|317|260|251|254|235|232|221|
|scored_0629|195|180|177|177|177|128-2|195|195|187|186|204|197|192|188|187|177|
|scored_0630|39|39|39|39|39|hist|39|41|39|39|42|41|40|42|41|39|
|scored_0631|162|147|146|146|146|128-2|162|165|163|158|196|164|169|181|155|146|
|scored_0632|236|223|214|214|214|128-2|236|238|227|226|244|237|239|222|233|214|
|scored_0633|207|189|189|189|189|128-1|207|207|194|196|207|212|204|200|189|193|
|scored_0634|205|182|182|182|182|128-2|205|207|197|204|210|206|210|202|192|182|
|scored_0635|46|46|46|46|46|hist|46|50|48|48|48|47|46|51|48|47|
|scored_0636|314|290|288|288|288|128-0|314|314|402|411|315|317|322|288|293|295|
|scored_0637|43|42|42|42|42|0-2|43|43|45|42|43|45|42|43|45|42|
|scored_0638|340|302|296|296|296|128-2|340|340|370|420|343|338|336|334|354|296|
|scored_0639|151|137|137|137|137|128-2|151|153|147|142|160|153|141|149|150|137|
|scored_0640|260|247|235|235|235|128-1|260|260|250|284|260|255|267|248|235|242|
|scored_0641|138|133|133|133|133|128-2|138|142|141|139|145|143|142|135|135|133|
|scored_0642|151|141|141|141|141|0-2|151|152|153|141|157|153|151|179|146|165|
|scored_0643|227|193|193|193|193|128-2|227|227|227|209|254|217|243|221|199|193|
|scored_0644|174|168|163|163|163|128-1|174|176|171|172|180|178|185|171|163|190|
|scored_0645|246|221|221|221|221|128-1|246|247|264|231|250|247|250|224|221|247|
|scored_0646|202|185|185|185|185|128-1|202|206|200|198|229|207|215|215|185|187|
|scored_0647|259|227|227|227|227|128-1|259|259|353|326|263|259|258|237|227|239|
|scored_0648|124|120|119|119|119|128-2|124|127|120|121|134|130|134|123|124|119|
|scored_0649|150|137|137|137|137|0-2|150|151|144|137|157|149|143|149|158|146|
|scored_0650|44|44|44|44|44|hist|44|45|45|44|45|45|44|45|45|44|
|scored_0651|205|182|181|181|181|128-1|205|205|201|197|238|208|205|199|181|186|
|scored_0652|121|114|114|113|113|128-2|121|126|123|114|127|124|123|119|117|113|
|scored_0653|209|191|190|190|190|128-2|209|212|196|202|227|206|214|207|195|190|
|scored_0654|42|42|42|42|42|hist|42|44|46|46|48|46|44|48|46|44|
|scored_0655|325|289|289|289|289|128-1|325|326|365|408|328|338|335|341|289|318|
|scored_0656|46|41|41|40|40|0-2|46|47|44|40|47|43|44|47|43|44|
|scored_0657|119|114|114|114|114|128-1|119|122|119|116|127|124|116|119|114|114|
|scored_0658|215|196|194|194|194|128-1|215|215|227|219|215|210|210|199|194|248|
|scored_0659|104|104|104|104|104|hist|104|107|109|105|116|110|108|108|107|104|
|scored_0660|209|209|195|195|195|128-1|209|215|208|286|220|212|215|215|195|206|
|scored_0661|282|247|246|246|246|128-2|282|282|303|296|296|279|287|271|257|246|
|scored_0662|203|187|187|187|187|128-1|203|206|201|196|215|203|214|198|187|203|
|scored_0663|224|212|204|204|204|128-1|224|225|224|224|233|226|229|232|204|211|
|scored_0664|269|237|232|232|232|128-1|269|283|300|308|265|257|256|245|232|290|
|scored_0665|117|107|106|106|106|128-2|117|117|112|108|124|114|113|117|110|106|
|scored_0666|186|177|177|177|177|0-2|186|187|179|177|196|185|180|182|189|178|
|scored_0667|208|184|183|183|183|128-1|208|212|195|206|242|213|217|212|183|198|
|scored_0668|148|130|130|130|130|128-2|148|149|141|137|149|148|140|135|139|130|
|scored_0669|159|149|149|149|149|128-1|159|160|156|155|185|165|196|171|149|151|
|scored_0670|221|209|205|205|205|128-2|221|221|213|209|220|225|226|212|243|205|
|scored_0671|240|204|203|203|203|128-2|240|242|235|269|239|233|235|217|231|203|
|scored_0672|72|67|67|67|67|48-2|72|72|68|68|69|69|67|70|68|68|
|scored_0673|207|197|193|193|193|128-2|207|208|197|212|211|212|205|200|195|193|
|scored_0674|155|144|143|143|143|128-0|155|156|149|146|158|157|155|143|144|152|
|scored_0675|179|159|156|156|156|128-2|179|183|175|175|199|177|183|191|163|156|
|scored_0676|6|6|6|6|6|hist|6|6|6|6|6|6|6|6|6|6|
|scored_0677|285|260|247|247|247|128-2|285|285|418|325|288|284|283|287|271|247|
|scored_0678|206|183|183|183|183|128-1|206|206|188|203|206|199|205|193|183|191|
|scored_0679|146|139|139|139|139|0-2|146|147|146|139|151|151|144|146|151|145|
|scored_0680|244|219|219|219|219|128-1|244|245|247|268|258|247|251|225|219|220|
|scored_0681|254|225|224|224|224|128-2|254|254|298|269|261|262|268|240|241|224|
|scored_0682|327|297|294|294|294|128-1|327|327|393|403|332|327|324|326|294|310|
|scored_0683|176|160|157|157|157|128-2|176|180|170|166|191|178|173|174|162|157|
|scored_0684|138|130|128|128|128|128-2|138|139|131|129|153|140|134|131|131|128|
|scored_0685|220|212|202|202|202|128-1|220|220|216|237|219|210|221|219|202|226|
|scored_0686|115|114|114|114|114|128-2|115|117|119|116|118|116|116|116|115|114|
|scored_0687|286|262|257|257|257|128-1|286|286|340|330|286|284|288|288|257|291|
|scored_0688|202|180|180|180|180|128-2|202|204|213|199|203|203|200|213|187|180|
|scored_0689|225|195|195|195|195|128-1|225|225|222|257|234|223|226|214|195|225|
|scored_0690|331|279|278|278|278|128-2|331|331|566|490|335|323|335|339|301|278|
|scored_0691|174|166|166|166|166|0-2|174|176|175|166|180|179|182|169|176|174|
|scored_0692|249|233|224|224|224|128-2|249|249|279|261|261|254|253|244|233|224|
|scored_0693|208|171|171|171|171|128-2|208|209|201|174|217|208|213|198|213|171|
|scored_0694|192|172|171|171|171|128-2|192|193|194|185|202|194|193|214|199|171|
|scored_0695|142|135|135|135|135|128-1|142|144|143|140|166|146|150|140|135|140|
|scored_0696|101|96|95|95|95|0-2|101|103|101|95|106|103|97|100|101|97|
|scored_0697|134|124|124|124|124|128-1|134|135|129|130|145|133|142|131|124|125|
|scored_0698|291|246|246|246|246|128-2|291|291|281|418|290|292|291|271|247|246|
|scored_0699|169|161|160|160|160|128-1|169|176|168|166|193|171|182|170|160|170|
|scored_0700|212|199|199|199|198|128-1|212|213|199|216|214|209|210|202|198|202|
|scored_0701|41|40|37|37|37|0-1|41|41|37|42|41|37|37|41|37|37|
|scored_0702|262|227|227|227|227|128-1|262|262|341|333|261|265|268|247|227|239|
|scored_0703|214|198|187|187|187|128-2|214|214|202|198|217|205|211|214|194|187|
|scored_0704|156|143|143|143|143|128-2|156|156|153|147|163|160|161|165|165|143|
|scored_0705|191|173|172|172|172|128-2|191|191|188|187|216|204|211|173|180|172|
|scored_0706|228|220|217|217|217|128-1|228|229|219|292|231|231|226|237|217|234|
|scored_0707|115|107|107|107|107|128-2|115|116|112|111|116|110|110|116|108|107|
|scored_0708|186|165|164|164|164|128-2|186|190|178|176|192|181|181|194|168|164|
|scored_0709|157|151|150|150|150|128-1|157|157|164|153|199|171|172|162|150|155|
|scored_0710|148|135|135|135|135|128-2|148|148|146|141|159|149|149|138|140|135|
|scored_0711|26|26|26|26|26|hist|26|29|28|28|29|29|28|29|29|28|
|scored_0712|233|205|204|204|204|128-1|233|237|221|231|235|241|240|222|204|216|
|scored_0713|121|116|116|116|116|128-2|121|121|121|120|127|124|122|122|119|116|
|scored_0714|232|217|216|216|216|128-1|232|232|232|268|266|246|245|226|216|221|
|scored_0715|197|179|179|179|179|128-2|197|200|195|187|211|207|208|194|195|179|
|scored_0716|147|139|138|138|138|0-2|147|148|142|138|153|150|146|155|142|140|
|scored_0717|151|144|144|144|144|128-1|151|155|149|146|160|157|155|162|144|148|
|scored_0718|169|153|153|153|153|128-2|169|173|166|156|186|170|174|164|175|153|
|scored_0719|309|290|287|287|287|128-0|309|309|304|400|316|309|319|287|323|316|
|scored_0720|37|37|37|37|37|hist|37|37|37|37|38|38|37|38|38|37|
|scored_0721|4|4|4|4|4|hist|4|4|4|4|4|4|4|4|4|4|
|scored_0722|198|194|193|193|193|128-1|198|198|194|197|229|209|220|196|193|207|
|scored_0723|122|116|115|115|115|128-2|122|124|123|118|123|119|125|121|118|115|
|scored_0724|199|189|189|189|189|128-1|199|199|194|202|204|199|196|212|189|207|
|scored_0725|273|239|239|239|239|128-2|273|273|266|341|282|274|280|249|246|239|
|scored_0726|186|167|166|166|166|128-2|186|186|176|176|187|178|188|169|167|166|
|scored_0727|101|98|97|97|97|48-1|101|104|101|98|101|97|98|100|101|98|
|scored_0728|315|284|278|278|278|128-1|315|315|581|381|321|320|327|295|278|301|
|scored_0729|352|300|298|298|298|128-1|352|352|478|500|355|354|361|357|298|301|
|scored_0730|158|147|147|147|147|128-2|158|161|152|154|176|166|162|150|148|147|
|scored_0731|195|178|178|178|178|128-1|195|195|194|187|211|195|213|206|178|182|
|scored_0732|154|143|143|143|143|128-2|154|154|148|146|160|156|155|150|161|143|
|scored_0733|116|107|107|107|107|128-2|116|117|111|111|111|110|112|113|111|107|
|scored_0734|178|162|159|159|159|128-2|178|178|177|171|181|179|172|183|180|159|
|scored_0735|140|136|136|136|136|128-2|140|148|143|140|151|147|147|143|160|136|
|scored_0736|195|179|178|178|178|0-2|195|196|179|178|208|202|182|183|178|190|
|scored_0737|143|131|131|131|131|128-2|143|147|143|134|148|145|139|146|142|131|
|scored_0738|146|138|138|138|138|128-2|146|146|144|139|183|151|155|142|141|138|
|scored_0739|187|169|169|169|169|128-1|187|191|199|179|190|187|188|200|169|208|
|scored_0740|163|159|159|159|159|0-1|163|167|159|160|172|166|163|174|181|159|
|scored_0741|166|163|163|163|163|0-2|166|166|164|163|177|179|164|165|172|179|
|scored_0742|224|224|220|220|220|128-2|224|224|224|268|229|225|229|226|222|220|
|scored_0743|130|123|121|121|121|128-2|130|132|132|126|141|135|134|134|130|121|
|scored_0744|343|309|306|306|306|128-2|343|343|340|745|350|352|349|312|324|306|
|scored_0745|220|209|208|208|208|128-1|220|220|213|250|224|227|221|264|208|210|
|scored_0746|183|159|159|159|159|128-1|183|186|184|168|196|178|185|174|159|161|
|scored_0747|190|182|177|177|177|128-2|190|191|184|182|195|192|194|180|179|177|
|scored_0748|288|253|250|250|250|128-2|288|288|297|387|291|286|281|258|266|250|
|scored_0749|118|112|110|110|110|128-1|118|118|113|112|119|116|115|115|110|113|
|scored_0750|139|135|135|135|135|0-2|139|139|142|135|148|142|146|147|138|135|
|scored_0751|268|236|236|236|236|128-2|268|268|257|309|272|261|268|270|248|236|
|scored_0752|126|118|118|118|118|128-2|126|128|127|122|124|125|119|123|122|118|
|scored_0753|119|114|112|112|112|128-2|119|120|117|116|122|120|118|116|117|112|
|scored_0754|208|189|188|188|188|128-1|208|208|201|228|214|211|210|200|188|196|
|scored_0755|142|122|122|122|122|128-2|142|144|136|135|156|131|139|138|136|122|
|scored_0756|191|175|172|172|172|128-1|191|190|186|175|189|185|185|194|172|180|
|scored_0757|161|153|148|148|148|128-2|161|163|165|157|168|169|161|163|159|148|
|scored_0758|136|127|127|127|127|128-1|136|138|132|130|137|134|135|132|127|127|
|scored_0759|244|239|237|237|237|128-2|244|244|239|268|260|256|250|246|250|237|
|scored_0760|136|132|132|132|132|0-2|136|138|134|132|144|137|138|132|136|140|
|scored_0761|122|115|113|113|113|0-2|122|124|122|113|128|120|121|123|117|116|
|scored_0762|341|291|288|288|288|128-2|341|347|741|415|344|350|342|351|308|288|
|scored_0763|115|106|106|106|106|128-2|115|115|112|110|115|112|113|114|107|106|
|scored_0764|248|213|213|213|213|128-2|248|249|228|286|244|242|247|228|221|213|
|scored_0765|228|205|200|200|200|128-2|228|231|221|312|237|228|234|214|218|200|
|scored_0766|262|246|230|230|230|128-1|262|262|290|307|269|263|274|261|230|242|
|scored_0767|147|139|138|138|138|128-2|147|149|146|145|155|149|152|150|142|138|
|scored_0768|149|139|138|138|138|128-2|149|151|150|146|176|154|163|153|143|138|
|scored_0769|143|135|134|134|134|128-2|143|144|138|138|175|138|137|156|135|134|
|scored_0770|182|167|166|166|166|128-2|182|190|180|200|219|179|183|189|168|166|
|scored_0771|321|283|282|282|282|128-1|321|321|319|465|322|324|326|327|282|369|
|scored_0772|145|130|130|130|130|128-2|145|148|144|135|163|144|142|153|134|130|
|scored_0773|60|51|51|51|51|0-2|60|60|60|51|57|57|57|60|60|57|
|scored_0774|344|298|297|297|297|128-1|344|344|363|402|348|355|354|345|297|333|
|scored_0775|184|164|164|164|164|128-2|184|186|172|169|207|197|200|174|166|164|
|scored_0776|273|273|273|273|273|hist|273|273|280|283|281|276|285|278|310|296|
|scored_0777|49|49|48|48|48|48-2|49|50|51|51|49|49|48|49|49|49|
|scored_0778|163|140|140|140|140|128-2|163|163|155|148|168|162|154|173|150|140|
|scored_0779|178|167|166|166|166|128-2|178|180|181|176|208|193|183|186|170|166|
|scored_0780|331|289|288|288|288|128-1|331|333|330|499|334|323|327|302|288|289|
|scored_0781|305|285|282|282|282|128-2|305|305|424|454|312|313|304|291|283|282|
|scored_0782|185|172|172|172|172|0-2|185|187|178|172|190|187|192|178|178|183|
|scored_0783|137|132|131|131|131|0-2|137|139|138|131|142|144|137|136|135|136|
|scored_0784|275|255|248|248|248|128-2|275|281|414|302|286|281|284|272|273|248|
|scored_0785|134|126|123|123|123|128-2|134|134|135|129|142|140|134|127|126|123|
|scored_0786|298|278|275|275|275|128-1|298|298|430|381|311|304|306|288|275|306|
|scored_0787|125|121|120|120|120|128-1|125|127|125|123|128|130|127|124|120|121|
|scored_0788|285|243|243|243|243|128-2|285|286|295|340|286|275|286|273|279|243|
|scored_0789|219|190|190|190|190|128-1|219|219|214|210|226|217|217|208|190|190|
|scored_0790|272|248|248|248|248|128-1|272|272|272|284|274|276|272|261|248|251|
|scored_0791|192|169|169|169|169|128-2|192|193|190|176|205|196|185|183|174|169|
|scored_0792|167|163|157|157|157|128-2|167|168|169|168|195|171|175|177|168|157|
|scored_0793|272|240|239|239|239|128-1|272|277|261|281|277|280|278|267|239|243|
|scored_0794|210|188|187|187|187|128-1|210|210|210|189|234|211|234|198|187|242|
|scored_0795|220|206|203|203|203|128-2|220|221|214|220|234|224|228|218|219|203|
|scored_0796|278|249|248|248|248|128-1|278|278|273|317|278|278|280|254|248|278|
|scored_0797|90|84|83|83|83|48-2|90|89|87|85|88|91|83|86|89|84|
|scored_0798|2|2|2|2|2|hist|2|4|4|4|4|4|4|4|4|4|
|scored_0799|285|247|246|246|246|128-1|285|286|274|492|290|287|289|260|246|261|
|scored_0800|101|96|95|95|95|0-2|101|102|101|95|99|96|96|102|101|97|
|scored_0801|125|117|117|117|117|128-2|125|126|124|120|132|124|119|122|123|117|
|scored_0802|232|209|207|207|207|128-0|232|232|220|234|225|233|233|207|214|213|
|scored_0803|173|161|160|160|160|0-2|173|177|171|160|180|176|174|206|186|166|
|scored_0804|141|136|135|135|135|128-1|141|143|144|139|146|143|145|144|135|135|
|scored_0805|199|175|174|174|174|128-2|199|200|191|187|212|197|194|199|181|174|
|scored_0806|118|106|105|105|105|128-2|118|119|114|112|117|117|112|113|111|105|
|scored_0807|175|166|160|160|160|128-2|175|177|177|166|186|187|178|186|174|160|
|scored_0808|305|285|271|271|271|128-1|305|307|528|439|305|311|313|289|271|279|
|scored_0809|166|153|152|152|152|128-1|166|166|163|163|183|168|169|179|152|174|
|scored_0810|41|41|38|38|38|0-2|41|43|42|38|45|42|40|45|42|40|
|scored_0811|110|105|105|105|105|128-0|110|112|107|109|110|109|107|105|106|108|
|scored_0812|131|120|120|120|120|128-2|131|131|121|122|132|126|125|126|124|120|
|scored_0813|279|239|239|239|239|128-2|279|279|348|294|278|281|272|258|240|239|
|scored_0814|198|187|184|184|184|128-2|198|203|194|207|232|213|208|214|192|184|
|scored_0815|261|231|231|231|231|128-2|261|262|255|282|275|262|265|253|232|231|
|scored_0816|131|122|122|122|122|128-2|131|133|130|129|137|136|133|128|129|122|
|scored_0817|209|186|185|185|185|128-2|209|209|218|229|219|210|213|193|187|185|
|scored_0818|31|31|29|29|29|0-1|31|32|29|29|32|32|32|32|32|32|
|scored_0819|282|252|252|252|252|128-1|282|282|269|316|289|288|283|264|252|263|
|scored_0820|275|241|241|241|241|128-1|275|276|344|334|297|282|277|276|241|244|
|scored_0821|125|120|120|120|120|128-0|125|126|124|121|129|129|121|120|120|120|
|scored_0822|46|46|45|45|45|48-2|46|46|46|51|47|46|45|47|47|47|
|scored_0823|189|187|187|187|187|0-1|189|209|187|224|227|200|214|208|192|210|
|scored_0824|176|162|162|162|162|0-2|176|178|164|162|181|181|177|193|179|174|
|scored_0825|150|142|142|142|142|0-2|150|151|146|142|154|158|151|142|142|144|
|scored_0826|259|240|230|229|229|128-1|259|259|249|289|266|258|264|260|229|238|
|scored_0827|240|217|212|212|212|128-1|240|241|225|238|247|233|238|263|212|217|
|scored_0828|42|42|42|42|42|hist|42|42|43|46|44|42|46|42|42|46|
|scored_0829|212|201|195|195|195|128-2|212|213|207|207|223|212|214|207|200|195|
|scored_0830|126|120|120|120|120|128-2|126|126|126|124|132|124|121|121|122|120|
|scored_0831|119|109|108|108|108|0-2|119|119|117|108|121|120|116|114|114|110|
|scored_0832|230|208|206|206|206|128-2|230|229|216|301|228|228|227|222|213|206|
|scored_0833|157|149|148|148|148|128-1|157|157|155|151|166|160|164|152|148|150|
|scored_0834|262|246|239|239|239|128-2|262|267|288|344|273|269|280|252|245|239|
|scored_0835|141|138|134|134|134|128-2|141|142|139|142|150|145|144|145|147|134|
|scored_0836|232|222|217|217|217|128-0|232|232|241|262|234|232|234|217|235|253|
|scored_0837|288|259|253|253|253|128-1|288|289|352|358|286|273|288|269|253|265|
|scored_0838|172|155|154|154|154|128-2|172|172|167|163|181|171|173|170|159|154|
|scored_0839|209|207|200|200|200|128-0|209|212|219|235|220|209|209|200|224|211|
|scored_0840|281|257|257|257|257|128-1|281|283|332|434|294|281|289|285|257|274|
|scored_0841|189|171|171|171|171|128-1|189|189|181|189|220|206|203|195|171|184|
|scored_0842|32|32|32|32|32|hist|32|39|38|37|39|38|36|39|38|36|
|scored_0843|176|163|162|162|162|128-1|176|178|172|168|180|179|176|181|162|169|
|scored_0844|318|270|270|270|270|128-1|318|318|358|544|318|318|319|316|270|270|
|scored_0845|265|239|239|239|239|128-2|265|265|263|318|279|261|267|253|242|239|
|scored_0846|125|118|117|117|117|128-2|125|127|124|118|131|123|121|123|124|117|
|scored_0847|216|200|198|198|198|128-2|216|220|199|243|218|209|210|220|199|198|
|scored_0848|312|278|269|269|269|128-1|312|312|348|418|319|305|317|285|269|275|
|scored_0849|240|214|213|213|213|128-1|240|243|231|227|246|247|243|254|213|256|
|scored_0850|218|189|188|188|188|128-2|218|218|210|259|220|214|213|207|201|188|
|scored_0851|41|40|39|39|39|0-1|41|41|39|39|40|39|39|40|39|39|
|scored_0852|198|180|179|179|179|128-2|198|199|187|188|205|199|195|196|186|179|
|scored_0853|217|207|198|198|198|128-2|217|217|207|237|220|216|221|209|205|198|
|scored_0854|196|185|182|182|182|128-2|196|201|195|222|210|202|199|195|194|182|
|scored_0855|88|83|83|83|83|48-2|88|90|88|84|92|88|83|91|90|83|
|scored_0856|266|234|234|234|234|128-1|266|268|316|316|271|263|273|251|234|234|
|scored_0857|280|241|241|241|241|128-2|280|280|271|300|283|270|282|271|245|241|
|scored_0858|56|53|53|53|53|0-1|56|56|53|53|55|54|53|55|55|53|
|scored_0859|267|244|243|243|243|128-1|267|267|415|304|282|271|286|260|243|255|
|scored_0860|180|167|166|166|166|0-2|180|180|171|166|180|183|172|172|167|167|
|scored_0861|197|186|183|183|183|128-2|197|199|191|188|202|196|194|215|201|183|
|scored_0862|162|150|149|149|149|128-2|162|165|157|151|213|165|160|153|159|149|
|scored_0863|43|43|43|43|43|hist|43|43|43|45|44|44|46|44|44|47|
|scored_0864|230|204|204|204|204|128-2|230|230|226|222|231|226|228|216|207|204|
|scored_0865|293|265|265|265|265|128-0|293|294|309|370|300|296|294|265|290|272|
|scored_0866|41|41|36|36|36|48-2|41|41|38|38|38|40|36|38|40|36|
|scored_0867|111|104|102|102|102|0-2|111|111|105|102|111|109|104|107|105|103|
|scored_0868|39|39|39|39|39|hist|39|42|42|40|40|40|41|40|40|41|
|scored_0869|154|146|141|141|141|128-0|154|156|149|153|163|153|158|141|146|148|
|scored_0870|269|250|246|246|246|128-1|269|269|262|303|271|271|273|250|246|250|
|scored_0871|140|134|133|133|133|0-2|140|140|140|133|148|146|147|149|136|138|
|scored_0872|130|123|123|123|123|0-2|130|132|128|123|137|137|133|127|126|131|
|scored_0873|48|44|44|44|44|48-2|48|48|46|46|51|47|44|50|49|45|
|scored_0874|42|36|36|36|36|0-2|42|46|42|36|44|44|37|47|44|37|
|scored_0875|197|194|186|186|186|128-1|197|197|197|194|208|201|204|206|186|196|
|scored_0876|245|226|224|224|224|128-2|245|245|239|294|251|245|250|226|238|224|
|scored_0877|74|66|65|65|65|48-1|74|75|70|70|72|65|66|75|69|67|
|scored_0878|193|178|177|177|177|128-2|193|193|192|193|200|200|193|199|178|177|
|scored_0879|128|125|125|125|125|128-0|128|130|128|128|135|137|132|125|125|126|
|scored_0880|48|42|42|42|42|48-2|48|48|45|44|46|47|42|46|47|42|
|scored_0881|109|105|104|104|104|128-2|109|111|111|105|117|111|108|105|106|104|
|scored_0882|31|31|31|31|31|hist|31|36|36|36|36|35|36|36|35|36|
|scored_0883|7|7|7|7|7|hist|7|7|7|7|7|7|7|7|7|7|
|scored_0884|135|128|127|127|127|0-2|135|137|132|127|133|137|132|135|139|127|
|scored_0885|202|183|183|183|183|128-2|202|202|198|195|204|206|205|212|198|183|
|scored_0886|191|173|171|171|171|128-2|191|194|183|176|206|202|196|195|177|171|
|scored_0887|4|4|4|4|4|hist|4|4|4|4|4|4|4|4|4|4|
|scored_0888|155|154|154|154|154|0-2|155|158|155|154|172|156|159|166|173|165|
|scored_0889|135|127|126|126|126|128-2|135|136|136|127|136|139|133|131|134|126|
|scored_0890|113|109|108|108|108|128-0|113|116|116|109|113|113|111|108|108|109|
|scored_0891|135|129|128|128|128|128-1|135|136|132|132|143|136|138|132|128|140|
|scored_0892|21|21|21|21|21|hist|21|27|27|27|26|26|26|26|26|26|
|scored_0893|161|145|145|145|145|128-1|161|163|156|150|167|161|159|147|145|172|
|scored_0894|153|144|143|143|143|0-2|153|154|154|143|169|159|158|157|147|144|
|scored_0895|146|142|142|142|142|128-2|146|147|147|144|154|146|155|181|146|142|
|scored_0896|213|186|186|186|186|128-2|213|214|207|195|242|233|225|218|199|186|
|scored_0897|47|47|44|44|44|128-2|47|48|47|49|48|47|45|46|48|44|
|scored_0898|102|96|95|95|95|128-2|102|102|103|97|102|97|100|102|100|95|
|scored_0899|289|257|252|252|252|128-1|289|289|284|331|288|279|298|270|252|273|
|scored_0900|208|203|201|201|201|128-1|208|211|233|217|214|203|208|210|201|202|
|scored_0901|21|21|21|21|21|hist|21|24|24|24|24|24|24|24|24|24|
|scored_0902|262|228|227|227|227|128-2|262|262|256|313|263|259|269|255|238|227|
|scored_0903|190|179|179|179|179|0-2|190|192|185|179|190|186|187|199|192|184|
|scored_0904|169|150|150|150|150|0-2|169|169|166|150|178|174|198|157|153|154|
|scored_0905|296|263|262|262|262|128-0|296|296|456|374|299|297|292|262|267|312|
|scored_0906|258|226|224|224|224|128-1|258|258|240|267|255|253|255|248|224|233|
|scored_0907|145|140|138|138|138|128-2|145|145|142|140|158|142|144|169|150|138|
|scored_0908|43|43|41|41|41|0-2|43|43|42|41|42|42|41|42|42|41|
|scored_0909|199|165|165|165|165|128-2|199|201|191|184|232|198|179|203|180|165|
|scored_0910|45|42|42|42|42|0-2|45|45|45|42|46|44|43|46|45|43|
|scored_0911|103|97|97|97|97|0-2|103|105|101|97|101|103|104|101|98|98|
|scored_0912|260|232|231|231|231|128-1|260|260|257|318|263|261|258|268|231|241|
|scored_0913|34|34|34|34|34|hist|34|39|39|39|39|37|36|39|37|36|
|scored_0914|50|48|48|48|48|128-1|50|54|49|52|51|49|53|51|48|54|
|scored_0915|235|215|215|215|215|128-1|235|235|232|279|239|243|247|220|215|233|
|scored_0916|284|251|251|251|251|128-1|284|286|316|309|293|290|282|273|251|258|
|scored_0917|264|239|236|236|236|128-1|264|265|264|254|272|267|267|250|236|253|
|scored_0918|264|244|240|240|240|128-1|264|266|253|333|268|270|265|260|240|242|
|scored_0919|185|168|168|168|168|0-1|185|193|168|169|217|174|182|184|185|183|
|scored_0920|182|168|159|159|159|128-2|182|182|171|168|185|178|171|188|166|159|
|scored_0921|207|188|186|186|186|128-1|207|208|200|192|205|205|206|201|186|216|
|scored_0922|34|34|34|34|34|hist|34|34|34|35|34|34|34|34|34|34|
|scored_0923|247|224|219|219|219|128-2|247|247|240|230|260|251|244|225|236|219|
|scored_0924|141|133|132|132|132|128-2|141|142|140|138|156|139|146|141|138|132|
|scored_0925|196|170|170|170|170|128-2|196|196|186|192|209|204|209|212|182|170|
|scored_0926|31|31|30|30|30|48-1|31|32|31|31|32|30|33|32|30|33|
|scored_0927|170|147|146|146|146|128-2|170|171|171|164|179|178|173|162|154|146|
|scored_0928|41|41|41|41|41|hist|41|43|41|41|44|41|41|44|41|41|
|scored_0929|171|166|166|166|166|0-2|171|176|167|166|184|181|177|184|168|187|
|scored_0930|106|101|100|100|100|128-2|106|107|110|102|115|107|107|110|107|100|
|scored_0931|2|2|2|2|2|hist|2|2|2|2|2|2|2|2|2|2|
|scored_0932|283|274|269|269|269|128-1|283|288|278|330|294|283|293|274|269|301|
|scored_0933|196|182|181|181|181|128-1|196|196|194|185|208|196|204|188|181|183|
|scored_0934|177|156|156|156|156|128-2|177|178|169|168|179|178|169|211|164|156|
|scored_0935|210|185|183|183|183|128-2|210|213|210|220|222|214|209|216|203|183|
|scored_0936|187|180|173|173|173|128-2|187|192|179|181|198|193|194|188|176|173|
|scored_0937|38|38|36|36|36|48-2|38|38|38|37|41|38|36|41|38|36|
|scored_0938|163|149|149|149|149|0-2|163|171|155|149|194|155|175|156|150|149|
|scored_0939|289|266|265|265|265|128-0|289|289|286|400|298|292|290|265|272|307|
|scored_0940|155|148|148|148|148|128-1|155|157|160|155|174|167|167|150|148|166|
|scored_0941|290|254|253|253|253|128-1|290|290|291|344|297|291|293|275|253|328|
|scored_0942|174|160|152|152|152|128-0|174|176|166|176|210|176|188|152|167|180|
|scored_0943|116|108|106|106|106|48-2|116|117|114|111|116|119|106|112|110|110|
|scored_0944|280|247|245|245|245|128-2|280|280|264|312|278|263|268|264|290|245|
|scored_0945|34|34|34|34|34|hist|34|40|39|40|39|44|40|39|44|42|
|scored_0946|267|259|250|250|250|128-2|267|267|297|324|271|267|264|259|254|250|
|scored_0947|115|110|109|109|109|128-2|115|116|116|110|117|116|116|113|113|109|
|scored_0948|163|152|152|152|152|0-2|163|163|159|152|167|159|165|197|173|170|
|scored_0949|106|102|102|102|102|0-1|106|106|102|102|107|108|106|108|104|106|
|scored_0950|154|135|133|133|133|128-2|154|158|149|143|172|162|156|164|148|133|
|scored_0951|111|105|102|102|102|128-1|111|114|107|109|108|109|110|108|102|105|
|scored_0952|267|229|229|229|229|128-2|267|269|254|262|277|267|270|260|234|229|
|scored_0953|224|205|201|201|201|128-1|224|224|213|269|228|214|224|218|201|225|
|scored_0954|99|98|96|96|96|128-1|99|101|98|97|104|102|97|100|96|97|
|scored_0955|111|103|102|102|102|0-2|111|113|110|102|110|108|103|109|106|104|
|scored_0956|279|263|257|257|257|128-1|279|287|265|397|281|280|287|263|257|273|
|scored_0957|39|38|36|36|36|0-2|39|39|38|36|40|38|37|40|38|37|
|scored_0958|36|35|33|33|33|48-1|36|36|36|36|40|33|33|40|33|33|
|scored_0959|289|289|288|288|288|128-0|289|289|343|415|295|290|294|288|334|311|
|scored_0960|155|150|150|150|150|0-2|155|157|151|150|157|153|155|174|155|172|
|scored_0961|155|135|134|134|134|128-2|155|155|143|146|151|142|146|141|143|134|
|scored_0962|43|42|41|41|41|48-1|43|47|43|42|47|41|42|47|41|42|
|scored_0963|154|154|149|149|149|128-2|154|154|153|154|161|162|163|174|175|149|
|scored_0964|258|229|228|228|228|128-1|258|258|248|243|259|261|257|236|228|234|
|scored_0965|254|232|230|230|230|128-1|254|257|242|275|263|261|257|237|230|245|
|scored_0966|109|106|105|105|105|0-2|109|110|109|105|112|114|112|110|108|108|
|scored_0967|137|124|124|124|124|128-2|137|138|139|133|137|141|137|129|129|124|
|scored_0968|205|175|174|174|174|128-2|205|206|190|196|205|196|195|198|181|174|
|scored_0969|184|174|173|173|173|128-1|184|184|184|177|187|182|188|195|173|174|
|scored_0970|289|254|248|248|248|128-2|289|289|319|317|289|288|285|276|274|248|
|scored_0971|23|23|23|23|23|hist|23|23|23|23|23|23|23|23|23|23|
|scored_0972|154|145|145|145|145|128-1|154|155|154|151|158|166|154|154|145|146|
|scored_0973|136|126|125|125|125|128-2|136|136|138|129|140|138|136|132|130|125|
|scored_0974|286|256|256|256|256|128-1|286|286|285|393|288|291|294|277|256|294|
|scored_0975|157|141|141|141|141|128-1|157|158|148|142|169|158|147|150|141|141|
|scored_0976|149|133|132|132|132|128-1|149|151|143|139|142|148|146|138|132|133|
|scored_0977|207|196|191|191|191|128-2|207|207|196|247|212|200|212|201|196|191|
|scored_0978|132|125|125|125|125|128-1|132|132|128|127|134|131|140|127|125|125|
|scored_0979|220|203|201|201|201|128-0|220|220|215|230|220|221|224|201|216|203|
|scored_0980|157|155|155|155|155|0-1|157|158|155|156|169|166|169|170|164|174|
|scored_0981|33|33|33|33|33|hist|33|34|35|34|37|36|35|37|36|35|
|scored_0982|342|308|303|303|303|128-1|342|342|420|510|339|338|347|306|303|331|
|scored_0983|171|161|161|161|161|128-2|171|174|173|167|184|183|185|169|168|161|
|scored_0984|168|158|158|158|158|0-2|168|170|161|158|167|164|165|173|185|165|
|scored_0985|165|151|151|151|151|128-1|165|167|156|161|174|164|166|166|151|153|
|scored_0986|211|188|186|186|186|128-2|211|211|202|225|217|208|209|219|210|186|
|scored_0987|176|153|152|152|152|128-2|176|180|165|156|185|186|165|168|161|152|
|scored_0988|181|164|164|164|164|128-1|181|184|184|172|207|172|182|189|164|167|
|scored_0989|100|95|94|94|94|48-1|100|102|100|96|103|94|97|102|94|96|
|scored_0990|222|208|200|200|200|128-2|222|222|215|222|230|218|227|229|210|200|
|scored_0991|218|206|203|203|203|128-1|218|219|209|206|219|211|219|217|203|208|
|scored_0992|326|290|284|284|284|128-1|326|326|373|358|330|326|330|331|284|306|
|scored_0993|274|251|247|247|247|128-1|274|274|265|328|282|274|272|251|247|269|
|scored_0994|189|173|173|173|173|128-1|189|200|181|174|201|200|187|198|173|180|
|scored_0995|177|169|169|169|169|0-2|177|181|173|169|194|182|188|194|169|186|
|scored_0996|160|150|150|150|150|0-2|160|160|162|150|170|169|162|160|158|155|
|scored_0997|161|149|147|147|147|128-2|161|161|152|149|167|164|156|162|153|147|
|scored_0998|255|217|217|217|217|128-2|255|255|265|258|255|256|261|238|225|217|
|scored_0999|132|125|123|123|123|128-2|132|132|132|126|135|131|135|132|126|123|
|scored_1000|297|254|250|250|250|128-2|297|299|364|352|303|290|294|268|256|250|

## Colas, recombinación y registros en práctica

200 programas reproducidos: 31834 pasos. Independientes: 31991. Ahorro adicional del cruce: 157. [Detalle](reports/tail_cross_practice/comparison.csv) · [Programas](reports/tail_cross_practice/portfolio.csv)

Aporte exclusivo entre independientes: E1 13, E2 181, E3 204

Ahorro final por familia: independent 193, mixed 146, multiplicative 137, shared 209, tiny 1

| Método | IDs mejorados | Pasos ahorrados | Segundos acumulados por paso |
|---|---:|---:|---:|
|E1|40|102|165.8|
|E2|95|266|41.4|
|E3|73|309|2.2|
|Cruce|158|686|8.8|

| Familia | Histórico | E1 | E2 | E3 | Cruce | Cartera |
|---|---:|---:|---:|---:|---:|---:|
|independent|10849|10816|10836|10685|10656|10656|
|mixed|9519|9500|9482|9447|9373|9373|
|multiplicative|4723|4705|4627|4698|4586|4586|
|shared|7378|7346|7259|7330|7169|7169|
|tiny|51|51|50|51|50|50|

CP-SAT antes del pulido, conservando la referencia: 32488 pasos con recetas observadas y 32276 al incorporar también las cruzadas.

| Método | Completos | Histórico en esos IDs | Antes del pulido | Después | Segundos acumulados |
|---|---:|---:|---:|---:|---:|
|E1|200/200|32520|32452|32418|16910.9|
|E2|200/200|32520|32276|32254|10999.9|
|E3|200/200|32520|32328|32211|670.0|
|Cruce|200/200|32520|31848|31834|6031.4|

Comparación directa, 200 IDs comunes: E1 32418, E2 32254, E3 32211

| ID | Familia | Método | Configuración | Pasos | Operaciones DAG | Estado |
|---|---|---|---|---:|---:|---|
|practice_0001|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0001|multiplicative|Cruce|CP True/coste0/orden0|133|133|válido|
|practice_0001|multiplicative|Cruce|CP True/coste1/orden0|133|133|válido|
|practice_0001|multiplicative|Cruce|CP True/coste0/orden1|133|133|válido|
|practice_0001|multiplicative|Cruce|CP True/coste1/orden1|133|133|válido|
|practice_0001|multiplicative|Cruce|mejor independiente/coste0/orden0|136|136|válido|
|practice_0001|multiplicative|Cruce|mejor independiente/coste1/orden0|136|136|válido|
|practice_0001|multiplicative|Cruce|mejor independiente/coste0/orden1|136|136|válido|
|practice_0001|multiplicative|Cruce|mejor independiente/coste1/orden1|136|136|válido|
|practice_0001|multiplicative|E1|cola10/pol1/inicio0|144|48|válido|
|practice_0001|multiplicative|E1|cola10/pol1/inicio1|144|48|válido|
|practice_0001|multiplicative|E1|cola10/pol2/inicio0|144|48|válido|
|practice_0001|multiplicative|E1|cola10/pol2/inicio1|144|48|válido|
|practice_0001|multiplicative|E1|cola8/pol1/inicio0|142|39|válido|
|practice_0001|multiplicative|E1|cola8/pol1/inicio1|142|39|válido|
|practice_0001|multiplicative|E1|cola8/pol2/inicio0|142|39|válido|
|practice_0001|multiplicative|E1|cola8/pol2/inicio1|142|39|válido|
|practice_0001|multiplicative|E1|cola6/pol1/inicio0|143|33|válido|
|practice_0001|multiplicative|E1|cola6/pol1/inicio1|142|32|válido|
|practice_0001|multiplicative|E1|cola6/pol2/inicio0|143|33|válido|
|practice_0001|multiplicative|E1|cola6/pol2/inicio1|142|32|válido|
|practice_0001|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0001|multiplicative|E2|CP False/coste0/orden0|136|136|válido|
|practice_0001|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0001|multiplicative|E2|CP True/coste0/orden0|142|142|válido|
|practice_0001|multiplicative|E3|donante0/coste0/orden0|142|142|válido|
|practice_0001|multiplicative|E3|donante0/coste1/orden0|142|142|válido|
|practice_0001|multiplicative|E3|donante0/coste0/orden1|142|142|válido|
|practice_0001|multiplicative|E3|donante0/coste1/orden1|142|142|válido|
|practice_0001|multiplicative|E3|donante1/coste0/orden0|154|154|válido|
|practice_0001|multiplicative|E3|donante1/coste1/orden0|154|154|válido|
|practice_0001|multiplicative|E3|donante1/coste0/orden1|154|154|válido|
|practice_0001|multiplicative|E3|donante1/coste1/orden1|154|154|válido|
|practice_0001|multiplicative|E3|donante2/coste0/orden0|146|146|válido|
|practice_0001|multiplicative|E3|donante2/coste1/orden0|146|146|válido|
|practice_0001|multiplicative|E3|donante2/coste0/orden1|146|146|válido|
|practice_0001|multiplicative|E3|donante2/coste1/orden1|146|146|válido|
|practice_0001|multiplicative|E3|donante3/coste0/orden0|143|143|válido|
|practice_0001|multiplicative|E3|donante3/coste1/orden0|143|143|válido|
|practice_0001|multiplicative|E3|donante3/coste0/orden1|143|143|válido|
|practice_0001|multiplicative|E3|donante3/coste1/orden1|143|143|válido|
|practice_0001|multiplicative|E3|donante4/coste0/orden0|164|164|válido|
|practice_0001|multiplicative|E3|donante4/coste1/orden0|164|164|válido|
|practice_0001|multiplicative|E3|donante4/coste0/orden1|164|164|válido|
|practice_0001|multiplicative|E3|donante4/coste1/orden1|164|164|válido|
|practice_0001|multiplicative|E3|donante5/coste0/orden0|151|151|válido|
|practice_0001|multiplicative|E3|donante5/coste1/orden0|151|151|válido|
|practice_0001|multiplicative|E3|donante5/coste0/orden1|151|151|válido|
|practice_0001|multiplicative|E3|donante5/coste1/orden1|151|151|válido|
|practice_0001|multiplicative|E3|donante6/coste0/orden0|154|154|válido|
|practice_0001|multiplicative|E3|donante6/coste1/orden0|154|154|válido|
|practice_0001|multiplicative|E3|donante6/coste0/orden1|154|154|válido|
|practice_0001|multiplicative|E3|donante6/coste1/orden1|154|154|válido|
|practice_0001|multiplicative|E3|donante7/coste0/orden0|188|149|válido|
|practice_0001|multiplicative|E3|donante7/coste1/orden0|159|149|válido|
|practice_0001|multiplicative|E3|donante7/coste0/orden1|178|149|válido|
|practice_0001|multiplicative|E3|donante7/coste1/orden1|160|149|válido|
|practice_0001|multiplicative|E3|donante8/coste0/orden0|149|147|válido|
|practice_0001|multiplicative|E3|donante8/coste1/orden0|148|147|válido|
|practice_0001|multiplicative|E3|donante8/coste0/orden1|148|147|válido|
|practice_0001|multiplicative|E3|donante8/coste1/orden1|148|147|válido|
|practice_0001|multiplicative|E3|donante9/coste0/orden0|144|139|válido|
|practice_0001|multiplicative|E3|donante9/coste1/orden0|140|139|válido|
|practice_0001|multiplicative|E3|donante9/coste0/orden1|139|139|válido|
|practice_0001|multiplicative|E3|donante9/coste1/orden1|139|139|válido|
|practice_0002|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0002|mixed|Cruce|CP True/coste0/orden0|274|232|válido|
|practice_0002|mixed|Cruce|CP True/coste1/orden0|279|232|válido|
|practice_0002|mixed|Cruce|CP True/coste0/orden1|244|232|válido|
|practice_0002|mixed|Cruce|CP True/coste1/orden1|261|232|válido|
|practice_0002|mixed|Cruce|mejor independiente/coste0/orden0|279|234|válido|
|practice_0002|mixed|Cruce|mejor independiente/coste1/orden0|274|234|válido|
|practice_0002|mixed|Cruce|mejor independiente/coste0/orden1|241|234|válido|
|practice_0002|mixed|Cruce|mejor independiente/coste1/orden1|235|234|válido|
|practice_0002|mixed|E1|cola10/pol1/inicio0|245|86|válido|
|practice_0002|mixed|E1|cola10/pol1/inicio1|245|86|válido|
|practice_0002|mixed|E1|cola10/pol2/inicio0|243|84|válido|
|practice_0002|mixed|E1|cola10/pol2/inicio1|243|84|válido|
|practice_0002|mixed|E1|cola8/pol1/inicio0|246|71|válido|
|practice_0002|mixed|E1|cola8/pol1/inicio1|246|71|válido|
|practice_0002|mixed|E1|cola8/pol2/inicio0|244|69|válido|
|practice_0002|mixed|E1|cola8/pol2/inicio1|246|71|válido|
|practice_0002|mixed|E1|cola6/pol1/inicio0|248|57|válido|
|practice_0002|mixed|E1|cola6/pol1/inicio1|248|57|válido|
|practice_0002|mixed|E1|cola6/pol2/inicio0|242|51|válido|
|practice_0002|mixed|E1|cola6/pol2/inicio1|242|51|válido|
|practice_0002|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0002|mixed|E2|CP False/coste0/orden0|321|258|válido|
|practice_0002|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0002|mixed|E2|CP True/coste0/orden0|302|234|válido|
|practice_0002|mixed|E3|donante0/coste0/orden0|238|234|válido|
|practice_0002|mixed|E3|donante0/coste1/orden0|261|234|válido|
|practice_0002|mixed|E3|donante0/coste0/orden1|247|234|válido|
|practice_0002|mixed|E3|donante0/coste1/orden1|246|234|válido|
|practice_0002|mixed|E3|donante1/coste0/orden0|276|276|válido|
|practice_0002|mixed|E3|donante1/coste1/orden0|276|276|válido|
|practice_0002|mixed|E3|donante1/coste0/orden1|276|276|válido|
|practice_0002|mixed|E3|donante1/coste1/orden1|276|276|válido|
|practice_0002|mixed|E3|donante2/coste0/orden0|323|258|válido|
|practice_0002|mixed|E3|donante2/coste1/orden0|264|258|válido|
|practice_0002|mixed|E3|donante2/coste0/orden1|258|258|válido|
|practice_0002|mixed|E3|donante2/coste1/orden1|258|258|válido|
|practice_0002|mixed|E3|donante3/coste0/orden0|285|251|válido|
|practice_0002|mixed|E3|donante3/coste1/orden0|253|251|válido|
|practice_0002|mixed|E3|donante3/coste0/orden1|251|251|válido|
|practice_0002|mixed|E3|donante3/coste1/orden1|251|251|válido|
|practice_0002|mixed|E3|donante4/coste0/orden0|278|278|válido|
|practice_0002|mixed|E3|donante4/coste1/orden0|278|278|válido|
|practice_0002|mixed|E3|donante4/coste0/orden1|278|278|válido|
|practice_0002|mixed|E3|donante4/coste1/orden1|278|278|válido|
|practice_0002|mixed|E3|donante5/coste0/orden0|272|272|válido|
|practice_0002|mixed|E3|donante5/coste1/orden0|272|272|válido|
|practice_0002|mixed|E3|donante5/coste0/orden1|272|272|válido|
|practice_0002|mixed|E3|donante5/coste1/orden1|272|272|válido|
|practice_0002|mixed|E3|donante6/coste0/orden0|272|272|válido|
|practice_0002|mixed|E3|donante6/coste1/orden0|272|272|válido|
|practice_0002|mixed|E3|donante6/coste0/orden1|272|272|válido|
|practice_0002|mixed|E3|donante6/coste1/orden1|272|272|válido|
|practice_0002|mixed|E3|donante7/coste0/orden0|269|239|válido|
|practice_0002|mixed|E3|donante7/coste1/orden0|260|239|válido|
|practice_0002|mixed|E3|donante7/coste0/orden1|273|239|válido|
|practice_0002|mixed|E3|donante7/coste1/orden1|261|239|válido|
|practice_0002|mixed|E3|donante8/coste0/orden0|258|233|válido|
|practice_0002|mixed|E3|donante8/coste1/orden0|290|233|válido|
|practice_0002|mixed|E3|donante8/coste0/orden1|243|233|válido|
|practice_0002|mixed|E3|donante8/coste1/orden1|254|233|válido|
|practice_0002|mixed|E3|donante9/coste0/orden0|238|234|válido|
|practice_0002|mixed|E3|donante9/coste1/orden0|261|234|válido|
|practice_0002|mixed|E3|donante9/coste0/orden1|247|234|válido|
|practice_0002|mixed|E3|donante9/coste1/orden1|246|234|válido|
|practice_0003|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0003|mixed|Cruce|CP True/coste0/orden0|297|245|válido|
|practice_0003|mixed|Cruce|CP True/coste1/orden0|279|245|válido|
|practice_0003|mixed|Cruce|CP True/coste0/orden1|245|245|válido|
|practice_0003|mixed|Cruce|CP True/coste1/orden1|245|245|válido|
|practice_0003|mixed|Cruce|mejor independiente/coste0/orden0|299|245|válido|
|practice_0003|mixed|Cruce|mejor independiente/coste1/orden0|280|245|válido|
|practice_0003|mixed|Cruce|mejor independiente/coste0/orden1|245|245|válido|
|practice_0003|mixed|Cruce|mejor independiente/coste1/orden1|245|245|válido|
|practice_0003|mixed|E1|cola10/pol1/inicio0|256|87|válido|
|practice_0003|mixed|E1|cola10/pol1/inicio1|256|87|válido|
|practice_0003|mixed|E1|cola10/pol2/inicio0|253|84|válido|
|practice_0003|mixed|E1|cola10/pol2/inicio1|253|84|válido|
|practice_0003|mixed|E1|cola8/pol1/inicio0|253|63|válido|
|practice_0003|mixed|E1|cola8/pol1/inicio1|253|63|válido|
|practice_0003|mixed|E1|cola8/pol2/inicio0|256|66|válido|
|practice_0003|mixed|E1|cola8/pol2/inicio1|256|66|válido|
|practice_0003|mixed|E1|cola6/pol1/inicio0|258|52|válido|
|practice_0003|mixed|E1|cola6/pol1/inicio1|253|47|válido|
|practice_0003|mixed|E1|cola6/pol2/inicio0|257|51|válido|
|practice_0003|mixed|E1|cola6/pol2/inicio1|252|46|válido|
|practice_0003|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0003|mixed|E2|CP False/coste0/orden0|285|285|válido|
|practice_0003|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0003|mixed|E2|CP True/coste0/orden0|351|243|válido|
|practice_0003|mixed|E3|donante0/coste0/orden0|248|245|válido|
|practice_0003|mixed|E3|donante0/coste1/orden0|247|245|válido|
|practice_0003|mixed|E3|donante0/coste0/orden1|257|245|válido|
|practice_0003|mixed|E3|donante0/coste1/orden1|251|245|válido|
|practice_0003|mixed|E3|donante1/coste0/orden0|285|285|válido|
|practice_0003|mixed|E3|donante1/coste1/orden0|285|285|válido|
|practice_0003|mixed|E3|donante1/coste0/orden1|285|285|válido|
|practice_0003|mixed|E3|donante1/coste1/orden1|285|285|válido|
|practice_0003|mixed|E3|donante2/coste0/orden0|335|274|válido|
|practice_0003|mixed|E3|donante2/coste1/orden0|316|274|válido|
|practice_0003|mixed|E3|donante2/coste0/orden1|274|274|válido|
|practice_0003|mixed|E3|donante2/coste1/orden1|274|274|válido|
|practice_0003|mixed|E3|donante3/coste0/orden0|573|270|válido|
|practice_0003|mixed|E3|donante3/coste1/orden0|317|270|válido|
|practice_0003|mixed|E3|donante3/coste0/orden1|270|270|válido|
|practice_0003|mixed|E3|donante3/coste1/orden1|270|270|válido|
|practice_0003|mixed|E3|donante4/coste0/orden0|293|293|válido|
|practice_0003|mixed|E3|donante4/coste1/orden0|293|293|válido|
|practice_0003|mixed|E3|donante4/coste0/orden1|293|293|válido|
|practice_0003|mixed|E3|donante4/coste1/orden1|293|293|válido|
|practice_0003|mixed|E3|donante5/coste0/orden0|288|288|válido|
|practice_0003|mixed|E3|donante5/coste1/orden0|288|288|válido|
|practice_0003|mixed|E3|donante5/coste0/orden1|288|288|válido|
|practice_0003|mixed|E3|donante5/coste1/orden1|288|288|válido|
|practice_0003|mixed|E3|donante6/coste0/orden0|292|292|válido|
|practice_0003|mixed|E3|donante6/coste1/orden0|292|292|válido|
|practice_0003|mixed|E3|donante6/coste0/orden1|292|292|válido|
|practice_0003|mixed|E3|donante6/coste1/orden1|292|292|válido|
|practice_0003|mixed|E3|donante7/coste0/orden0|275|257|válido|
|practice_0003|mixed|E3|donante7/coste1/orden0|262|257|válido|
|practice_0003|mixed|E3|donante7/coste0/orden1|275|257|válido|
|practice_0003|mixed|E3|donante7/coste1/orden1|262|257|válido|
|practice_0003|mixed|E3|donante8/coste0/orden0|256|246|válido|
|practice_0003|mixed|E3|donante8/coste1/orden0|247|246|válido|
|practice_0003|mixed|E3|donante8/coste0/orden1|246|246|válido|
|practice_0003|mixed|E3|donante8/coste1/orden1|246|246|válido|
|practice_0004|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0004|shared|Cruce|CP True/coste0/orden0|119|119|válido|
|practice_0004|shared|Cruce|CP True/coste1/orden0|119|119|válido|
|practice_0004|shared|Cruce|CP True/coste0/orden1|119|119|válido|
|practice_0004|shared|Cruce|CP True/coste1/orden1|119|119|válido|
|practice_0004|shared|Cruce|mejor independiente/coste0/orden0|119|119|válido|
|practice_0004|shared|Cruce|mejor independiente/coste1/orden0|119|119|válido|
|practice_0004|shared|Cruce|mejor independiente/coste0/orden1|119|119|válido|
|practice_0004|shared|Cruce|mejor independiente/coste1/orden1|119|119|válido|
|practice_0004|shared|E1|cola10/pol1/inicio0|126|43|válido|
|practice_0004|shared|E1|cola10/pol1/inicio1|126|43|válido|
|practice_0004|shared|E1|cola10/pol2/inicio0|126|43|válido|
|practice_0004|shared|E1|cola10/pol2/inicio1|126|43|válido|
|practice_0004|shared|E1|cola8/pol1/inicio0|126|38|válido|
|practice_0004|shared|E1|cola8/pol1/inicio1|126|38|válido|
|practice_0004|shared|E1|cola8/pol2/inicio0|126|38|válido|
|practice_0004|shared|E1|cola8/pol2/inicio1|126|38|válido|
|practice_0004|shared|E1|cola6/pol1/inicio0|123|27|válido|
|practice_0004|shared|E1|cola6/pol1/inicio1|123|27|válido|
|practice_0004|shared|E1|cola6/pol2/inicio0|123|27|válido|
|practice_0004|shared|E1|cola6/pol2/inicio1|123|27|válido|
|practice_0004|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0004|shared|E2|CP False/coste0/orden0|122|122|válido|
|practice_0004|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0004|shared|E2|CP True/coste0/orden0|119|119|válido|
|practice_0004|shared|E3|donante0/coste0/orden0|122|122|válido|
|practice_0004|shared|E3|donante0/coste1/orden0|122|122|válido|
|practice_0004|shared|E3|donante0/coste0/orden1|122|122|válido|
|practice_0004|shared|E3|donante0/coste1/orden1|122|122|válido|
|practice_0004|shared|E3|donante1/coste0/orden0|130|130|válido|
|practice_0004|shared|E3|donante1/coste1/orden0|130|130|válido|
|practice_0004|shared|E3|donante1/coste0/orden1|130|130|válido|
|practice_0004|shared|E3|donante1/coste1/orden1|130|130|válido|
|practice_0004|shared|E3|donante2/coste0/orden0|125|125|válido|
|practice_0004|shared|E3|donante2/coste1/orden0|125|125|válido|
|practice_0004|shared|E3|donante2/coste0/orden1|125|125|válido|
|practice_0004|shared|E3|donante2/coste1/orden1|125|125|válido|
|practice_0004|shared|E3|donante3/coste0/orden0|123|123|válido|
|practice_0004|shared|E3|donante3/coste1/orden0|123|123|válido|
|practice_0004|shared|E3|donante3/coste0/orden1|123|123|válido|
|practice_0004|shared|E3|donante3/coste1/orden1|123|123|válido|
|practice_0004|shared|E3|donante4/coste0/orden0|138|138|válido|
|practice_0004|shared|E3|donante4/coste1/orden0|138|138|válido|
|practice_0004|shared|E3|donante4/coste0/orden1|138|138|válido|
|practice_0004|shared|E3|donante4/coste1/orden1|138|138|válido|
|practice_0004|shared|E3|donante5/coste0/orden0|132|132|válido|
|practice_0004|shared|E3|donante5/coste1/orden0|132|132|válido|
|practice_0004|shared|E3|donante5/coste0/orden1|132|132|válido|
|practice_0004|shared|E3|donante5/coste1/orden1|132|132|válido|
|practice_0004|shared|E3|donante6/coste0/orden0|130|130|válido|
|practice_0004|shared|E3|donante6/coste1/orden0|130|130|válido|
|practice_0004|shared|E3|donante6/coste0/orden1|130|130|válido|
|practice_0004|shared|E3|donante6/coste1/orden1|130|130|válido|
|practice_0004|shared|E3|donante7/coste0/orden0|124|124|válido|
|practice_0004|shared|E3|donante7/coste1/orden0|124|124|válido|
|practice_0004|shared|E3|donante7/coste0/orden1|124|124|válido|
|practice_0004|shared|E3|donante7/coste1/orden1|124|124|válido|
|practice_0004|shared|E3|donante8/coste0/orden0|123|123|válido|
|practice_0004|shared|E3|donante8/coste1/orden0|123|123|válido|
|practice_0004|shared|E3|donante8/coste0/orden1|123|123|válido|
|practice_0004|shared|E3|donante8/coste1/orden1|123|123|válido|
|practice_0005|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0005|independent|Cruce|CP True/coste0/orden0|293|197|válido|
|practice_0005|independent|Cruce|CP True/coste1/orden0|282|197|válido|
|practice_0005|independent|Cruce|CP True/coste0/orden1|208|197|válido|
|practice_0005|independent|Cruce|CP True/coste1/orden1|215|197|válido|
|practice_0005|independent|Cruce|mejor independiente/coste0/orden0|302|200|válido|
|practice_0005|independent|Cruce|mejor independiente/coste1/orden0|290|200|válido|
|practice_0005|independent|Cruce|mejor independiente/coste0/orden1|209|200|válido|
|practice_0005|independent|Cruce|mejor independiente/coste1/orden1|209|200|válido|
|practice_0005|independent|E1|cola10/pol1/inicio0|211|62|válido|
|practice_0005|independent|E1|cola10/pol1/inicio1|211|62|válido|
|practice_0005|independent|E1|cola10/pol2/inicio0|211|62|válido|
|practice_0005|independent|E1|cola10/pol2/inicio1|211|62|válido|
|practice_0005|independent|E1|cola8/pol1/inicio0|212|50|válido|
|practice_0005|independent|E1|cola8/pol1/inicio1|212|50|válido|
|practice_0005|independent|E1|cola8/pol2/inicio0|212|50|válido|
|practice_0005|independent|E1|cola8/pol2/inicio1|212|50|válido|
|practice_0005|independent|E1|cola6/pol1/inicio0|210|38|válido|
|practice_0005|independent|E1|cola6/pol1/inicio1|210|38|válido|
|practice_0005|independent|E1|cola6/pol2/inicio0|207|35|válido|
|practice_0005|independent|E1|cola6/pol2/inicio1|207|35|válido|
|practice_0005|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0005|independent|E2|CP False/coste0/orden0|319|227|válido|
|practice_0005|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0005|independent|E2|CP True/coste0/orden0|293|197|válido|
|practice_0005|independent|E3|donante0/coste0/orden0|211|200|válido|
|practice_0005|independent|E3|donante0/coste1/orden0|211|200|válido|
|practice_0005|independent|E3|donante0/coste0/orden1|209|200|válido|
|practice_0005|independent|E3|donante0/coste1/orden1|209|200|válido|
|practice_0005|independent|E3|donante1/coste0/orden0|234|234|válido|
|practice_0005|independent|E3|donante1/coste1/orden0|234|234|válido|
|practice_0005|independent|E3|donante1/coste0/orden1|234|234|válido|
|practice_0005|independent|E3|donante1/coste1/orden1|234|234|válido|
|practice_0005|independent|E3|donante2/coste0/orden0|219|219|válido|
|practice_0005|independent|E3|donante2/coste1/orden0|219|219|válido|
|practice_0005|independent|E3|donante2/coste0/orden1|219|219|válido|
|practice_0005|independent|E3|donante2/coste1/orden1|219|219|válido|
|practice_0005|independent|E3|donante3/coste0/orden0|289|211|válido|
|practice_0005|independent|E3|donante3/coste1/orden0|265|211|válido|
|practice_0005|independent|E3|donante3/coste0/orden1|214|211|válido|
|practice_0005|independent|E3|donante3/coste1/orden1|214|211|válido|
|practice_0005|independent|E3|donante4/coste0/orden0|233|233|válido|
|practice_0005|independent|E3|donante4/coste1/orden0|233|233|válido|
|practice_0005|independent|E3|donante4/coste0/orden1|233|233|válido|
|practice_0005|independent|E3|donante4/coste1/orden1|233|233|válido|
|practice_0005|independent|E3|donante5/coste0/orden0|228|228|válido|
|practice_0005|independent|E3|donante5/coste1/orden0|228|228|válido|
|practice_0005|independent|E3|donante5/coste0/orden1|228|228|válido|
|practice_0005|independent|E3|donante5/coste1/orden1|228|228|válido|
|practice_0005|independent|E3|donante6/coste0/orden0|228|228|válido|
|practice_0005|independent|E3|donante6/coste1/orden0|228|228|válido|
|practice_0005|independent|E3|donante6/coste0/orden1|228|228|válido|
|practice_0005|independent|E3|donante6/coste1/orden1|228|228|válido|
|practice_0005|independent|E3|donante7/coste0/orden0|230|212|válido|
|practice_0005|independent|E3|donante7/coste1/orden0|233|212|válido|
|practice_0005|independent|E3|donante7/coste0/orden1|230|212|válido|
|practice_0005|independent|E3|donante7/coste1/orden1|228|212|válido|
|practice_0005|independent|E3|donante8/coste0/orden0|203|202|válido|
|practice_0005|independent|E3|donante8/coste1/orden0|203|202|válido|
|practice_0005|independent|E3|donante8/coste0/orden1|229|202|válido|
|practice_0005|independent|E3|donante8/coste1/orden1|229|202|válido|
|practice_0005|independent|E3|donante9/coste0/orden0|211|200|válido|
|practice_0005|independent|E3|donante9/coste1/orden0|211|200|válido|
|practice_0005|independent|E3|donante9/coste0/orden1|209|200|válido|
|practice_0005|independent|E3|donante9/coste1/orden1|209|200|válido|
|practice_0006|shared|Cruce|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0006|shared|Cruce|CP True/coste0/orden0|28|28|válido|
|practice_0006|shared|Cruce|CP True/coste1/orden0|28|28|válido|
|practice_0006|shared|Cruce|CP True/coste0/orden1|28|28|válido|
|practice_0006|shared|Cruce|CP True/coste1/orden1|28|28|válido|
|practice_0006|shared|Cruce|mejor independiente/coste0/orden0|28|28|válido|
|practice_0006|shared|Cruce|mejor independiente/coste1/orden0|28|28|válido|
|practice_0006|shared|Cruce|mejor independiente/coste0/orden1|28|28|válido|
|practice_0006|shared|Cruce|mejor independiente/coste1/orden1|28|28|válido|
|practice_0006|shared|E1|cola9/pol1/inicio0|33|20|válido|
|practice_0006|shared|E1|cola9/pol1/inicio1|33|20|válido|
|practice_0006|shared|E1|cola9/pol2/inicio0|35|22|válido|
|practice_0006|shared|E1|cola9/pol2/inicio1|33|20|válido|
|practice_0006|shared|E1|cola8/pol1/inicio0|33|16|válido|
|practice_0006|shared|E1|cola8/pol1/inicio1|36|19|válido|
|practice_0006|shared|E1|cola8/pol2/inicio0|35|18|válido|
|practice_0006|shared|E1|cola8/pol2/inicio1|35|18|válido|
|practice_0006|shared|E1|cola6/pol1/inicio0|34|10|válido|
|practice_0006|shared|E1|cola6/pol1/inicio1|34|10|válido|
|practice_0006|shared|E1|cola6/pol2/inicio0|34|10|válido|
|practice_0006|shared|E1|cola6/pol2/inicio1|34|10|válido|
|practice_0006|shared|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0006|shared|E2|CP False/coste0/orden0|34|34|válido|
|practice_0006|shared|E2|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0006|shared|E2|CP True/coste0/orden0|28|28|válido|
|practice_0006|shared|E3|donante0/coste0/orden0|34|34|válido|
|practice_0006|shared|E3|donante0/coste1/orden0|34|34|válido|
|practice_0006|shared|E3|donante0/coste0/orden1|34|34|válido|
|practice_0006|shared|E3|donante0/coste1/orden1|34|34|válido|
|practice_0006|shared|E3|donante1/coste0/orden0|40|40|válido|
|practice_0006|shared|E3|donante1/coste1/orden0|40|40|válido|
|practice_0006|shared|E3|donante1/coste0/orden1|40|40|válido|
|practice_0006|shared|E3|donante1/coste1/orden1|40|40|válido|
|practice_0006|shared|E3|donante2/coste0/orden0|39|39|válido|
|practice_0006|shared|E3|donante2/coste1/orden0|39|39|válido|
|practice_0006|shared|E3|donante2/coste0/orden1|39|39|válido|
|practice_0006|shared|E3|donante2/coste1/orden1|39|39|válido|
|practice_0006|shared|E3|donante3/coste0/orden0|36|36|válido|
|practice_0006|shared|E3|donante3/coste1/orden0|36|36|válido|
|practice_0006|shared|E3|donante3/coste0/orden1|36|36|válido|
|practice_0006|shared|E3|donante3/coste1/orden1|36|36|válido|
|practice_0006|shared|E3|donante4/coste0/orden0|41|41|válido|
|practice_0006|shared|E3|donante4/coste1/orden0|41|41|válido|
|practice_0006|shared|E3|donante4/coste0/orden1|41|41|válido|
|practice_0006|shared|E3|donante4/coste1/orden1|41|41|válido|
|practice_0006|shared|E3|donante5/coste0/orden0|39|39|válido|
|practice_0006|shared|E3|donante5/coste1/orden0|39|39|válido|
|practice_0006|shared|E3|donante5/coste0/orden1|39|39|válido|
|practice_0006|shared|E3|donante5/coste1/orden1|39|39|válido|
|practice_0006|shared|E3|donante6/coste0/orden0|38|38|válido|
|practice_0006|shared|E3|donante6/coste1/orden0|38|38|válido|
|practice_0006|shared|E3|donante6/coste0/orden1|38|38|válido|
|practice_0006|shared|E3|donante6/coste1/orden1|38|38|válido|
|practice_0007|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0007|multiplicative|Cruce|CP True/coste0/orden0|95|95|válido|
|practice_0007|multiplicative|Cruce|CP True/coste1/orden0|95|95|válido|
|practice_0007|multiplicative|Cruce|CP True/coste0/orden1|95|95|válido|
|practice_0007|multiplicative|Cruce|CP True/coste1/orden1|95|95|válido|
|practice_0007|multiplicative|Cruce|mejor independiente/coste0/orden0|95|95|válido|
|practice_0007|multiplicative|Cruce|mejor independiente/coste1/orden0|95|95|válido|
|practice_0007|multiplicative|Cruce|mejor independiente/coste0/orden1|95|95|válido|
|practice_0007|multiplicative|Cruce|mejor independiente/coste1/orden1|95|95|válido|
|practice_0007|multiplicative|E1|cola10/pol1/inicio0|104|38|válido|
|practice_0007|multiplicative|E1|cola10/pol1/inicio1|104|38|válido|
|practice_0007|multiplicative|E1|cola10/pol2/inicio0|105|39|válido|
|practice_0007|multiplicative|E1|cola10/pol2/inicio1|105|39|válido|
|practice_0007|multiplicative|E1|cola8/pol1/inicio0|106|34|válido|
|practice_0007|multiplicative|E1|cola8/pol1/inicio1|106|34|válido|
|practice_0007|multiplicative|E1|cola8/pol2/inicio0|104|32|válido|
|practice_0007|multiplicative|E1|cola8/pol2/inicio1|104|32|válido|
|practice_0007|multiplicative|E1|cola6/pol1/inicio0|105|27|válido|
|practice_0007|multiplicative|E1|cola6/pol1/inicio1|105|27|válido|
|practice_0007|multiplicative|E1|cola6/pol2/inicio0|104|26|válido|
|practice_0007|multiplicative|E1|cola6/pol2/inicio1|104|26|válido|
|practice_0007|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0007|multiplicative|E2|CP False/coste0/orden0|98|98|válido|
|practice_0007|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0007|multiplicative|E2|CP True/coste0/orden0|95|95|válido|
|practice_0007|multiplicative|E3|donante0/coste0/orden0|97|97|válido|
|practice_0007|multiplicative|E3|donante0/coste1/orden0|97|97|válido|
|practice_0007|multiplicative|E3|donante0/coste0/orden1|97|97|válido|
|practice_0007|multiplicative|E3|donante0/coste1/orden1|97|97|válido|
|practice_0007|multiplicative|E3|donante1/coste0/orden0|106|106|válido|
|practice_0007|multiplicative|E3|donante1/coste1/orden0|106|106|válido|
|practice_0007|multiplicative|E3|donante1/coste0/orden1|106|106|válido|
|practice_0007|multiplicative|E3|donante1/coste1/orden1|106|106|válido|
|practice_0007|multiplicative|E3|donante2/coste0/orden0|108|108|válido|
|practice_0007|multiplicative|E3|donante2/coste1/orden0|108|108|válido|
|practice_0007|multiplicative|E3|donante2/coste0/orden1|108|108|válido|
|practice_0007|multiplicative|E3|donante2/coste1/orden1|108|108|válido|
|practice_0007|multiplicative|E3|donante3/coste0/orden0|103|103|válido|
|practice_0007|multiplicative|E3|donante3/coste1/orden0|103|103|válido|
|practice_0007|multiplicative|E3|donante3/coste0/orden1|103|103|válido|
|practice_0007|multiplicative|E3|donante3/coste1/orden1|103|103|válido|
|practice_0007|multiplicative|E3|donante4/coste0/orden0|110|110|válido|
|practice_0007|multiplicative|E3|donante4/coste1/orden0|110|110|válido|
|practice_0007|multiplicative|E3|donante4/coste0/orden1|110|110|válido|
|practice_0007|multiplicative|E3|donante4/coste1/orden1|110|110|válido|
|practice_0007|multiplicative|E3|donante5/coste0/orden0|106|106|válido|
|practice_0007|multiplicative|E3|donante5/coste1/orden0|106|106|válido|
|practice_0007|multiplicative|E3|donante5/coste0/orden1|106|106|válido|
|practice_0007|multiplicative|E3|donante5/coste1/orden1|106|106|válido|
|practice_0007|multiplicative|E3|donante6/coste0/orden0|104|104|válido|
|practice_0007|multiplicative|E3|donante6/coste1/orden0|104|104|válido|
|practice_0007|multiplicative|E3|donante6/coste0/orden1|104|104|válido|
|practice_0007|multiplicative|E3|donante6/coste1/orden1|104|104|válido|
|practice_0007|multiplicative|E3|donante7/coste0/orden0|106|106|válido|
|practice_0007|multiplicative|E3|donante7/coste1/orden0|106|106|válido|
|practice_0007|multiplicative|E3|donante7/coste0/orden1|106|106|válido|
|practice_0007|multiplicative|E3|donante7/coste1/orden1|106|106|válido|
|practice_0007|multiplicative|E3|donante8/coste0/orden0|104|104|válido|
|practice_0007|multiplicative|E3|donante8/coste1/orden0|104|104|válido|
|practice_0007|multiplicative|E3|donante8/coste0/orden1|104|104|válido|
|practice_0007|multiplicative|E3|donante8/coste1/orden1|104|104|válido|
|practice_0007|multiplicative|E3|donante9/coste0/orden0|98|98|válido|
|practice_0007|multiplicative|E3|donante9/coste1/orden0|98|98|válido|
|practice_0007|multiplicative|E3|donante9/coste0/orden1|98|98|válido|
|practice_0007|multiplicative|E3|donante9/coste1/orden1|98|98|válido|
|practice_0008|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0008|independent|Cruce|CP True/coste0/orden0|295|225|válido|
|practice_0008|independent|Cruce|CP True/coste1/orden0|287|225|válido|
|practice_0008|independent|Cruce|CP True/coste0/orden1|247|225|válido|
|practice_0008|independent|Cruce|CP True/coste1/orden1|228|225|válido|
|practice_0008|independent|Cruce|mejor independiente/coste0/orden0|321|227|válido|
|practice_0008|independent|Cruce|mejor independiente/coste1/orden0|307|227|válido|
|practice_0008|independent|Cruce|mejor independiente/coste0/orden1|255|227|válido|
|practice_0008|independent|Cruce|mejor independiente/coste1/orden1|230|227|válido|
|practice_0008|independent|E1|cola10/pol1/inicio0|243|73|válido|
|practice_0008|independent|E1|cola10/pol1/inicio1|243|73|válido|
|practice_0008|independent|E1|cola10/pol2/inicio0|242|72|válido|
|practice_0008|independent|E1|cola10/pol2/inicio1|242|72|válido|
|practice_0008|independent|E1|cola8/pol1/inicio0|240|56|válido|
|practice_0008|independent|E1|cola8/pol1/inicio1|240|56|válido|
|practice_0008|independent|E1|cola8/pol2/inicio0|238|54|válido|
|practice_0008|independent|E1|cola8/pol2/inicio1|238|54|válido|
|practice_0008|independent|E1|cola6/pol1/inicio0|245|41|válido|
|practice_0008|independent|E1|cola6/pol1/inicio1|245|41|válido|
|practice_0008|independent|E1|cola6/pol2/inicio0|245|41|válido|
|practice_0008|independent|E1|cola6/pol2/inicio1|241|37|válido|
|practice_0008|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0008|independent|E2|CP False/coste0/orden0|242|237|válido|
|practice_0008|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0008|independent|E2|CP True/coste0/orden0|243|233|válido|
|practice_0008|independent|E3|donante0/coste0/orden0|241|234|válido|
|practice_0008|independent|E3|donante0/coste1/orden0|241|234|válido|
|practice_0008|independent|E3|donante0/coste0/orden1|251|234|válido|
|practice_0008|independent|E3|donante0/coste1/orden1|243|234|válido|
|practice_0008|independent|E3|donante1/coste0/orden0|258|258|válido|
|practice_0008|independent|E3|donante1/coste1/orden0|258|258|válido|
|practice_0008|independent|E3|donante1/coste0/orden1|258|258|válido|
|practice_0008|independent|E3|donante1/coste1/orden1|258|258|válido|
|practice_0008|independent|E3|donante2/coste0/orden0|260|249|válido|
|practice_0008|independent|E3|donante2/coste1/orden0|259|249|válido|
|practice_0008|independent|E3|donante2/coste0/orden1|249|249|válido|
|practice_0008|independent|E3|donante2/coste1/orden1|249|249|válido|
|practice_0008|independent|E3|donante3/coste0/orden0|281|238|válido|
|practice_0008|independent|E3|donante3/coste1/orden0|274|238|válido|
|practice_0008|independent|E3|donante3/coste0/orden1|238|238|válido|
|practice_0008|independent|E3|donante3/coste1/orden1|238|238|válido|
|practice_0008|independent|E3|donante4/coste0/orden0|274|274|válido|
|practice_0008|independent|E3|donante4/coste1/orden0|274|274|válido|
|practice_0008|independent|E3|donante4/coste0/orden1|274|274|válido|
|practice_0008|independent|E3|donante4/coste1/orden1|274|274|válido|
|practice_0008|independent|E3|donante5/coste0/orden0|264|264|válido|
|practice_0008|independent|E3|donante5/coste1/orden0|264|264|válido|
|practice_0008|independent|E3|donante5/coste0/orden1|264|264|válido|
|practice_0008|independent|E3|donante5/coste1/orden1|264|264|válido|
|practice_0008|independent|E3|donante6/coste0/orden0|262|262|válido|
|practice_0008|independent|E3|donante6/coste1/orden0|262|262|válido|
|practice_0008|independent|E3|donante6/coste0/orden1|262|262|válido|
|practice_0008|independent|E3|donante6/coste1/orden1|262|262|válido|
|practice_0008|independent|E3|donante7/coste0/orden0|241|234|válido|
|practice_0008|independent|E3|donante7/coste1/orden0|241|234|válido|
|practice_0008|independent|E3|donante7/coste0/orden1|251|234|válido|
|practice_0008|independent|E3|donante7/coste1/orden1|243|234|válido|
|practice_0008|independent|E3|donante8/coste0/orden0|245|234|válido|
|practice_0008|independent|E3|donante8/coste1/orden0|250|234|válido|
|practice_0008|independent|E3|donante8/coste0/orden1|258|234|válido|
|practice_0008|independent|E3|donante8/coste1/orden1|250|234|válido|
|practice_0008|independent|E3|donante9/coste0/orden0|291|227|válido|
|practice_0008|independent|E3|donante9/coste1/orden0|269|227|válido|
|practice_0008|independent|E3|donante9/coste0/orden1|255|227|válido|
|practice_0008|independent|E3|donante9/coste1/orden1|230|227|válido|
|practice_0009|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0009|multiplicative|Cruce|CP True/coste0/orden0|134|134|válido|
|practice_0009|multiplicative|Cruce|CP True/coste1/orden0|134|134|válido|
|practice_0009|multiplicative|Cruce|CP True/coste0/orden1|134|134|válido|
|practice_0009|multiplicative|Cruce|CP True/coste1/orden1|134|134|válido|
|practice_0009|multiplicative|Cruce|mejor independiente/coste0/orden0|134|134|válido|
|practice_0009|multiplicative|Cruce|mejor independiente/coste1/orden0|134|134|válido|
|practice_0009|multiplicative|Cruce|mejor independiente/coste0/orden1|134|134|válido|
|practice_0009|multiplicative|Cruce|mejor independiente/coste1/orden1|134|134|válido|
|practice_0009|multiplicative|E1|cola10/pol1/inicio0|140|44|válido|
|practice_0009|multiplicative|E1|cola10/pol1/inicio1|140|44|válido|
|practice_0009|multiplicative|E1|cola10/pol2/inicio0|139|43|válido|
|practice_0009|multiplicative|E1|cola10/pol2/inicio1|139|43|válido|
|practice_0009|multiplicative|E1|cola8/pol1/inicio0|137|33|válido|
|practice_0009|multiplicative|E1|cola8/pol1/inicio1|137|33|válido|
|practice_0009|multiplicative|E1|cola8/pol2/inicio0|137|33|válido|
|practice_0009|multiplicative|E1|cola8/pol2/inicio1|137|33|válido|
|practice_0009|multiplicative|E1|cola6/pol1/inicio0|137|26|válido|
|practice_0009|multiplicative|E1|cola6/pol1/inicio1|137|26|válido|
|practice_0009|multiplicative|E1|cola6/pol2/inicio0|137|26|válido|
|practice_0009|multiplicative|E1|cola6/pol2/inicio1|137|26|válido|
|practice_0009|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0009|multiplicative|E2|CP False/coste0/orden0|139|139|válido|
|practice_0009|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0009|multiplicative|E2|CP True/coste0/orden0|136|136|válido|
|practice_0009|multiplicative|E3|donante0/coste0/orden0|137|137|válido|
|practice_0009|multiplicative|E3|donante0/coste1/orden0|137|137|válido|
|practice_0009|multiplicative|E3|donante0/coste0/orden1|137|137|válido|
|practice_0009|multiplicative|E3|donante0/coste1/orden1|137|137|válido|
|practice_0009|multiplicative|E3|donante1/coste0/orden0|139|139|válido|
|practice_0009|multiplicative|E3|donante1/coste1/orden0|139|139|válido|
|practice_0009|multiplicative|E3|donante1/coste0/orden1|139|139|válido|
|practice_0009|multiplicative|E3|donante1/coste1/orden1|139|139|válido|
|practice_0009|multiplicative|E3|donante2/coste0/orden0|140|140|válido|
|practice_0009|multiplicative|E3|donante2/coste1/orden0|140|140|válido|
|practice_0009|multiplicative|E3|donante2/coste0/orden1|140|140|válido|
|practice_0009|multiplicative|E3|donante2/coste1/orden1|140|140|válido|
|practice_0009|multiplicative|E3|donante3/coste0/orden0|140|140|válido|
|practice_0009|multiplicative|E3|donante3/coste1/orden0|140|140|válido|
|practice_0009|multiplicative|E3|donante3/coste0/orden1|140|140|válido|
|practice_0009|multiplicative|E3|donante3/coste1/orden1|140|140|válido|
|practice_0009|multiplicative|E3|donante4/coste0/orden0|145|145|válido|
|practice_0009|multiplicative|E3|donante4/coste1/orden0|145|145|válido|
|practice_0009|multiplicative|E3|donante4/coste0/orden1|145|145|válido|
|practice_0009|multiplicative|E3|donante4/coste1/orden1|145|145|válido|
|practice_0009|multiplicative|E3|donante5/coste0/orden0|146|146|válido|
|practice_0009|multiplicative|E3|donante5/coste1/orden0|146|146|válido|
|practice_0009|multiplicative|E3|donante5/coste0/orden1|146|146|válido|
|practice_0009|multiplicative|E3|donante5/coste1/orden1|146|146|válido|
|practice_0009|multiplicative|E3|donante6/coste0/orden0|143|143|válido|
|practice_0009|multiplicative|E3|donante6/coste1/orden0|143|143|válido|
|practice_0009|multiplicative|E3|donante6/coste0/orden1|143|143|válido|
|practice_0009|multiplicative|E3|donante6/coste1/orden1|143|143|válido|
|practice_0009|multiplicative|E3|donante7/coste0/orden0|163|136|válido|
|practice_0009|multiplicative|E3|donante7/coste1/orden0|149|136|válido|
|practice_0009|multiplicative|E3|donante7/coste0/orden1|140|136|válido|
|practice_0009|multiplicative|E3|donante7/coste1/orden1|139|136|válido|
|practice_0009|multiplicative|E3|donante8/coste0/orden0|144|138|válido|
|practice_0009|multiplicative|E3|donante8/coste1/orden0|144|138|válido|
|practice_0009|multiplicative|E3|donante8/coste0/orden1|138|138|válido|
|practice_0009|multiplicative|E3|donante8/coste1/orden1|138|138|válido|
|practice_0009|multiplicative|E3|donante9/coste0/orden0|150|134|válido|
|practice_0009|multiplicative|E3|donante9/coste1/orden0|150|134|válido|
|practice_0009|multiplicative|E3|donante9/coste0/orden1|134|134|válido|
|practice_0009|multiplicative|E3|donante9/coste1/orden1|134|134|válido|
|practice_0010|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0010|multiplicative|Cruce|CP True/coste0/orden0|129|129|válido|
|practice_0010|multiplicative|Cruce|CP True/coste1/orden0|129|129|válido|
|practice_0010|multiplicative|Cruce|CP True/coste0/orden1|129|129|válido|
|practice_0010|multiplicative|Cruce|CP True/coste1/orden1|129|129|válido|
|practice_0010|multiplicative|Cruce|mejor independiente/coste0/orden0|130|130|válido|
|practice_0010|multiplicative|Cruce|mejor independiente/coste1/orden0|130|130|válido|
|practice_0010|multiplicative|Cruce|mejor independiente/coste0/orden1|130|130|válido|
|practice_0010|multiplicative|Cruce|mejor independiente/coste1/orden1|130|130|válido|
|practice_0010|multiplicative|E1|cola10/pol1/inicio0|130|39|válido|
|practice_0010|multiplicative|E1|cola10/pol1/inicio1|130|39|válido|
|practice_0010|multiplicative|E1|cola10/pol2/inicio0|130|39|válido|
|practice_0010|multiplicative|E1|cola10/pol2/inicio1|130|39|válido|
|practice_0010|multiplicative|E1|cola8/pol1/inicio0|130|31|válido|
|practice_0010|multiplicative|E1|cola8/pol1/inicio1|130|31|válido|
|practice_0010|multiplicative|E1|cola8/pol2/inicio0|130|31|válido|
|practice_0010|multiplicative|E1|cola8/pol2/inicio1|130|31|válido|
|practice_0010|multiplicative|E1|cola6/pol1/inicio0|131|24|válido|
|practice_0010|multiplicative|E1|cola6/pol1/inicio1|131|24|válido|
|practice_0010|multiplicative|E1|cola6/pol2/inicio0|131|24|válido|
|practice_0010|multiplicative|E1|cola6/pol2/inicio1|131|24|válido|
|practice_0010|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0010|multiplicative|E2|CP False/coste0/orden0|134|134|válido|
|practice_0010|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0010|multiplicative|E2|CP True/coste0/orden0|130|130|válido|
|practice_0010|multiplicative|E3|donante0/coste0/orden0|134|131|válido|
|practice_0010|multiplicative|E3|donante0/coste1/orden0|134|131|válido|
|practice_0010|multiplicative|E3|donante0/coste0/orden1|131|131|válido|
|practice_0010|multiplicative|E3|donante0/coste1/orden1|131|131|válido|
|practice_0010|multiplicative|E3|donante1/coste0/orden0|141|141|válido|
|practice_0010|multiplicative|E3|donante1/coste1/orden0|141|141|válido|
|practice_0010|multiplicative|E3|donante1/coste0/orden1|141|141|válido|
|practice_0010|multiplicative|E3|donante1/coste1/orden1|141|141|válido|
|practice_0010|multiplicative|E3|donante2/coste0/orden0|137|137|válido|
|practice_0010|multiplicative|E3|donante2/coste1/orden0|137|137|válido|
|practice_0010|multiplicative|E3|donante2/coste0/orden1|137|137|válido|
|practice_0010|multiplicative|E3|donante2/coste1/orden1|137|137|válido|
|practice_0010|multiplicative|E3|donante3/coste0/orden0|142|142|válido|
|practice_0010|multiplicative|E3|donante3/coste1/orden0|142|142|válido|
|practice_0010|multiplicative|E3|donante3/coste0/orden1|142|142|válido|
|practice_0010|multiplicative|E3|donante3/coste1/orden1|142|142|válido|
|practice_0010|multiplicative|E3|donante4/coste0/orden0|151|151|válido|
|practice_0010|multiplicative|E3|donante4/coste1/orden0|151|151|válido|
|practice_0010|multiplicative|E3|donante4/coste0/orden1|151|151|válido|
|practice_0010|multiplicative|E3|donante4/coste1/orden1|151|151|válido|
|practice_0010|multiplicative|E3|donante5/coste0/orden0|148|148|válido|
|practice_0010|multiplicative|E3|donante5/coste1/orden0|148|148|válido|
|practice_0010|multiplicative|E3|donante5/coste0/orden1|148|148|válido|
|practice_0010|multiplicative|E3|donante5/coste1/orden1|148|148|válido|
|practice_0010|multiplicative|E3|donante6/coste0/orden0|145|145|válido|
|practice_0010|multiplicative|E3|donante6/coste1/orden0|145|145|válido|
|practice_0010|multiplicative|E3|donante6/coste0/orden1|145|145|válido|
|practice_0010|multiplicative|E3|donante6/coste1/orden1|145|145|válido|
|practice_0010|multiplicative|E3|donante7/coste0/orden0|142|137|válido|
|practice_0010|multiplicative|E3|donante7/coste1/orden0|142|137|válido|
|practice_0010|multiplicative|E3|donante7/coste0/orden1|142|137|válido|
|practice_0010|multiplicative|E3|donante7/coste1/orden1|142|137|válido|
|practice_0010|multiplicative|E3|donante8/coste0/orden0|134|134|válido|
|practice_0010|multiplicative|E3|donante8/coste1/orden0|134|134|válido|
|practice_0010|multiplicative|E3|donante8/coste0/orden1|134|134|válido|
|practice_0010|multiplicative|E3|donante8/coste1/orden1|134|134|válido|
|practice_0010|multiplicative|E3|donante9/coste0/orden0|135|131|válido|
|practice_0010|multiplicative|E3|donante9/coste1/orden0|136|131|válido|
|practice_0010|multiplicative|E3|donante9/coste0/orden1|131|131|válido|
|practice_0010|multiplicative|E3|donante9/coste1/orden1|131|131|válido|
|practice_0011|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0011|multiplicative|Cruce|CP True/coste0/orden0|115|115|válido|
|practice_0011|multiplicative|Cruce|CP True/coste1/orden0|115|115|válido|
|practice_0011|multiplicative|Cruce|CP True/coste0/orden1|115|115|válido|
|practice_0011|multiplicative|Cruce|CP True/coste1/orden1|115|115|válido|
|practice_0011|multiplicative|Cruce|mejor independiente/coste0/orden0|115|115|válido|
|practice_0011|multiplicative|Cruce|mejor independiente/coste1/orden0|115|115|válido|
|practice_0011|multiplicative|Cruce|mejor independiente/coste0/orden1|115|115|válido|
|practice_0011|multiplicative|Cruce|mejor independiente/coste1/orden1|115|115|válido|
|practice_0011|multiplicative|E1|cola10/pol1/inicio0|123|38|válido|
|practice_0011|multiplicative|E1|cola10/pol1/inicio1|123|38|válido|
|practice_0011|multiplicative|E1|cola10/pol2/inicio0|120|35|válido|
|practice_0011|multiplicative|E1|cola10/pol2/inicio1|120|35|válido|
|practice_0011|multiplicative|E1|cola8/pol1/inicio0|123|33|válido|
|practice_0011|multiplicative|E1|cola8/pol1/inicio1|123|33|válido|
|practice_0011|multiplicative|E1|cola8/pol2/inicio0|120|30|válido|
|practice_0011|multiplicative|E1|cola8/pol2/inicio1|120|30|válido|
|practice_0011|multiplicative|E1|cola6/pol1/inicio0|123|25|válido|
|practice_0011|multiplicative|E1|cola6/pol1/inicio1|123|25|válido|
|practice_0011|multiplicative|E1|cola6/pol2/inicio0|121|23|válido|
|practice_0011|multiplicative|E1|cola6/pol2/inicio1|121|23|válido|
|practice_0011|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0011|multiplicative|E2|CP False/coste0/orden0|122|122|válido|
|practice_0011|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0011|multiplicative|E2|CP True/coste0/orden0|116|116|válido|
|practice_0011|multiplicative|E3|donante0/coste0/orden0|120|120|válido|
|practice_0011|multiplicative|E3|donante0/coste1/orden0|120|120|válido|
|practice_0011|multiplicative|E3|donante0/coste0/orden1|120|120|válido|
|practice_0011|multiplicative|E3|donante0/coste1/orden1|120|120|válido|
|practice_0011|multiplicative|E3|donante1/coste0/orden0|128|128|válido|
|practice_0011|multiplicative|E3|donante1/coste1/orden0|128|128|válido|
|practice_0011|multiplicative|E3|donante1/coste0/orden1|128|128|válido|
|practice_0011|multiplicative|E3|donante1/coste1/orden1|128|128|válido|
|practice_0011|multiplicative|E3|donante2/coste0/orden0|124|124|válido|
|practice_0011|multiplicative|E3|donante2/coste1/orden0|124|124|válido|
|practice_0011|multiplicative|E3|donante2/coste0/orden1|124|124|válido|
|practice_0011|multiplicative|E3|donante2/coste1/orden1|124|124|válido|
|practice_0011|multiplicative|E3|donante3/coste0/orden0|131|131|válido|
|practice_0011|multiplicative|E3|donante3/coste1/orden0|131|131|válido|
|practice_0011|multiplicative|E3|donante3/coste0/orden1|131|131|válido|
|practice_0011|multiplicative|E3|donante3/coste1/orden1|131|131|válido|
|practice_0011|multiplicative|E3|donante4/coste0/orden0|128|128|válido|
|practice_0011|multiplicative|E3|donante4/coste1/orden0|128|128|válido|
|practice_0011|multiplicative|E3|donante4/coste0/orden1|128|128|válido|
|practice_0011|multiplicative|E3|donante4/coste1/orden1|128|128|válido|
|practice_0011|multiplicative|E3|donante5/coste0/orden0|124|124|válido|
|practice_0011|multiplicative|E3|donante5/coste1/orden0|124|124|válido|
|practice_0011|multiplicative|E3|donante5/coste0/orden1|124|124|válido|
|practice_0011|multiplicative|E3|donante5/coste1/orden1|124|124|válido|
|practice_0011|multiplicative|E3|donante6/coste0/orden0|124|124|válido|
|practice_0011|multiplicative|E3|donante6/coste1/orden0|124|124|válido|
|practice_0011|multiplicative|E3|donante6/coste0/orden1|124|124|válido|
|practice_0011|multiplicative|E3|donante6/coste1/orden1|124|124|válido|
|practice_0011|multiplicative|E3|donante7/coste0/orden0|121|121|válido|
|practice_0011|multiplicative|E3|donante7/coste1/orden0|121|121|válido|
|practice_0011|multiplicative|E3|donante7/coste0/orden1|121|121|válido|
|practice_0011|multiplicative|E3|donante7/coste1/orden1|121|121|válido|
|practice_0011|multiplicative|E3|donante8/coste0/orden0|123|123|válido|
|practice_0011|multiplicative|E3|donante8/coste1/orden0|123|123|válido|
|practice_0011|multiplicative|E3|donante8/coste0/orden1|123|123|válido|
|practice_0011|multiplicative|E3|donante8/coste1/orden1|123|123|válido|
|practice_0012|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0012|mixed|Cruce|CP True/coste0/orden0|253|218|válido|
|practice_0012|mixed|Cruce|CP True/coste1/orden0|235|218|válido|
|practice_0012|mixed|Cruce|CP True/coste0/orden1|269|218|válido|
|practice_0012|mixed|Cruce|CP True/coste1/orden1|267|218|válido|
|practice_0012|mixed|Cruce|mejor independiente/coste0/orden0|249|222|válido|
|practice_0012|mixed|Cruce|mejor independiente/coste1/orden0|244|222|válido|
|practice_0012|mixed|Cruce|mejor independiente/coste0/orden1|232|222|válido|
|practice_0012|mixed|Cruce|mejor independiente/coste1/orden1|249|222|válido|
|practice_0012|mixed|E1|cola10/pol1/inicio0|230|85|válido|
|practice_0012|mixed|E1|cola10/pol1/inicio1|230|85|válido|
|practice_0012|mixed|E1|cola10/pol2/inicio0|228|83|válido|
|practice_0012|mixed|E1|cola10/pol2/inicio1|228|83|válido|
|practice_0012|mixed|E1|cola8/pol1/inicio0|234|73|válido|
|practice_0012|mixed|E1|cola8/pol1/inicio1|234|73|válido|
|practice_0012|mixed|E1|cola8/pol2/inicio0|233|72|válido|
|practice_0012|mixed|E1|cola8/pol2/inicio1|233|72|válido|
|practice_0012|mixed|E1|cola6/pol1/inicio0|232|56|válido|
|practice_0012|mixed|E1|cola6/pol1/inicio1|232|56|válido|
|practice_0012|mixed|E1|cola6/pol2/inicio0|232|56|válido|
|practice_0012|mixed|E1|cola6/pol2/inicio1|232|56|válido|
|practice_0012|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0012|mixed|E2|CP False/coste0/orden0|368|263|válido|
|practice_0012|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0012|mixed|E2|CP True/coste0/orden0|245|217|válido|
|practice_0012|mixed|E3|donante0/coste0/orden0|237|220|válido|
|practice_0012|mixed|E3|donante0/coste1/orden0|237|220|válido|
|practice_0012|mixed|E3|donante0/coste0/orden1|241|220|válido|
|practice_0012|mixed|E3|donante0/coste1/orden1|252|220|válido|
|practice_0012|mixed|E3|donante1/coste0/orden0|259|259|válido|
|practice_0012|mixed|E3|donante1/coste1/orden0|259|259|válido|
|practice_0012|mixed|E3|donante1/coste0/orden1|259|259|válido|
|practice_0012|mixed|E3|donante1/coste1/orden1|259|259|válido|
|practice_0012|mixed|E3|donante2/coste0/orden0|288|246|válido|
|practice_0012|mixed|E3|donante2/coste1/orden0|250|246|válido|
|practice_0012|mixed|E3|donante2/coste0/orden1|246|246|válido|
|practice_0012|mixed|E3|donante2/coste1/orden1|246|246|válido|
|practice_0012|mixed|E3|donante3/coste0/orden0|297|246|válido|
|practice_0012|mixed|E3|donante3/coste1/orden0|258|246|válido|
|practice_0012|mixed|E3|donante3/coste0/orden1|246|246|válido|
|practice_0012|mixed|E3|donante3/coste1/orden1|246|246|válido|
|practice_0012|mixed|E3|donante4/coste0/orden0|272|272|válido|
|practice_0012|mixed|E3|donante4/coste1/orden0|272|272|válido|
|practice_0012|mixed|E3|donante4/coste0/orden1|272|272|válido|
|practice_0012|mixed|E3|donante4/coste1/orden1|272|272|válido|
|practice_0012|mixed|E3|donante5/coste0/orden0|254|254|válido|
|practice_0012|mixed|E3|donante5/coste1/orden0|254|254|válido|
|practice_0012|mixed|E3|donante5/coste0/orden1|254|254|válido|
|practice_0012|mixed|E3|donante5/coste1/orden1|254|254|válido|
|practice_0012|mixed|E3|donante6/coste0/orden0|261|261|válido|
|practice_0012|mixed|E3|donante6/coste1/orden0|261|261|válido|
|practice_0012|mixed|E3|donante6/coste0/orden1|261|261|válido|
|practice_0012|mixed|E3|donante6/coste1/orden1|261|261|válido|
|practice_0012|mixed|E3|donante7/coste0/orden0|250|230|válido|
|practice_0012|mixed|E3|donante7/coste1/orden0|236|230|válido|
|practice_0012|mixed|E3|donante7/coste0/orden1|268|230|válido|
|practice_0012|mixed|E3|donante7/coste1/orden1|266|230|válido|
|practice_0012|mixed|E3|donante8/coste0/orden0|240|228|válido|
|practice_0012|mixed|E3|donante8/coste1/orden0|242|228|válido|
|practice_0012|mixed|E3|donante8/coste0/orden1|268|228|válido|
|practice_0012|mixed|E3|donante8/coste1/orden1|240|228|válido|
|practice_0012|mixed|E3|donante9/coste0/orden0|237|220|válido|
|practice_0012|mixed|E3|donante9/coste1/orden0|237|220|válido|
|practice_0012|mixed|E3|donante9/coste0/orden1|241|220|válido|
|practice_0012|mixed|E3|donante9/coste1/orden1|252|220|válido|
|practice_0013|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0013|independent|Cruce|CP True/coste0/orden0|234|190|válido|
|practice_0013|independent|Cruce|CP True/coste1/orden0|218|190|válido|
|practice_0013|independent|Cruce|CP True/coste0/orden1|190|190|válido|
|practice_0013|independent|Cruce|CP True/coste1/orden1|190|190|válido|
|practice_0013|independent|Cruce|mejor independiente/coste0/orden0|234|190|válido|
|practice_0013|independent|Cruce|mejor independiente/coste1/orden0|218|190|válido|
|practice_0013|independent|Cruce|mejor independiente/coste0/orden1|190|190|válido|
|practice_0013|independent|Cruce|mejor independiente/coste1/orden1|190|190|válido|
|practice_0013|independent|E1|cola10/pol1/inicio0|208|63|válido|
|practice_0013|independent|E1|cola10/pol1/inicio1|208|63|válido|
|practice_0013|independent|E1|cola10/pol2/inicio0|208|63|válido|
|practice_0013|independent|E1|cola10/pol2/inicio1|208|63|válido|
|practice_0013|independent|E1|cola8/pol1/inicio0|209|53|válido|
|practice_0013|independent|E1|cola8/pol1/inicio1|209|53|válido|
|practice_0013|independent|E1|cola8/pol2/inicio0|209|53|válido|
|practice_0013|independent|E1|cola8/pol2/inicio1|209|53|válido|
|practice_0013|independent|E1|cola6/pol1/inicio0|210|43|válido|
|practice_0013|independent|E1|cola6/pol1/inicio1|210|43|válido|
|practice_0013|independent|E1|cola6/pol2/inicio0|210|43|válido|
|practice_0013|independent|E1|cola6/pol2/inicio1|210|43|válido|
|practice_0013|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0013|independent|E2|CP False/coste0/orden0|228|220|válido|
|practice_0013|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0013|independent|E2|CP True/coste0/orden0|238|196|válido|
|practice_0013|independent|E3|donante0/coste0/orden0|197|197|válido|
|practice_0013|independent|E3|donante0/coste1/orden0|197|197|válido|
|practice_0013|independent|E3|donante0/coste0/orden1|203|197|válido|
|practice_0013|independent|E3|donante0/coste1/orden1|206|197|válido|
|practice_0013|independent|E3|donante1/coste0/orden0|217|217|válido|
|practice_0013|independent|E3|donante1/coste1/orden0|217|217|válido|
|practice_0013|independent|E3|donante1/coste0/orden1|217|217|válido|
|practice_0013|independent|E3|donante1/coste1/orden1|217|217|válido|
|practice_0013|independent|E3|donante2/coste0/orden0|206|206|válido|
|practice_0013|independent|E3|donante2/coste1/orden0|206|206|válido|
|practice_0013|independent|E3|donante2/coste0/orden1|206|206|válido|
|practice_0013|independent|E3|donante2/coste1/orden1|206|206|válido|
|practice_0013|independent|E3|donante3/coste0/orden0|269|203|válido|
|practice_0013|independent|E3|donante3/coste1/orden0|269|203|válido|
|practice_0013|independent|E3|donante3/coste0/orden1|203|203|válido|
|practice_0013|independent|E3|donante3/coste1/orden1|203|203|válido|
|practice_0013|independent|E3|donante4/coste0/orden0|219|219|válido|
|practice_0013|independent|E3|donante4/coste1/orden0|219|219|válido|
|practice_0013|independent|E3|donante4/coste0/orden1|219|219|válido|
|practice_0013|independent|E3|donante4/coste1/orden1|219|219|válido|
|practice_0013|independent|E3|donante5/coste0/orden0|224|224|válido|
|practice_0013|independent|E3|donante5/coste1/orden0|224|224|válido|
|practice_0013|independent|E3|donante5/coste0/orden1|224|224|válido|
|practice_0013|independent|E3|donante5/coste1/orden1|224|224|válido|
|practice_0013|independent|E3|donante6/coste0/orden0|217|217|válido|
|practice_0013|independent|E3|donante6/coste1/orden0|217|217|válido|
|practice_0013|independent|E3|donante6/coste0/orden1|217|217|válido|
|practice_0013|independent|E3|donante6/coste1/orden1|217|217|válido|
|practice_0013|independent|E3|donante7/coste0/orden0|250|203|válido|
|practice_0013|independent|E3|donante7/coste1/orden0|248|203|válido|
|practice_0013|independent|E3|donante7/coste0/orden1|245|203|válido|
|practice_0013|independent|E3|donante7/coste1/orden1|236|203|válido|
|practice_0013|independent|E3|donante8/coste0/orden0|263|192|válido|
|practice_0013|independent|E3|donante8/coste1/orden0|206|192|válido|
|practice_0013|independent|E3|donante8/coste0/orden1|192|192|válido|
|practice_0013|independent|E3|donante8/coste1/orden1|192|192|válido|
|practice_0014|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0014|multiplicative|Cruce|CP True/coste0/orden0|128|128|válido|
|practice_0014|multiplicative|Cruce|CP True/coste1/orden0|128|128|válido|
|practice_0014|multiplicative|Cruce|CP True/coste0/orden1|128|128|válido|
|practice_0014|multiplicative|Cruce|CP True/coste1/orden1|128|128|válido|
|practice_0014|multiplicative|Cruce|mejor independiente/coste0/orden0|128|128|válido|
|practice_0014|multiplicative|Cruce|mejor independiente/coste1/orden0|128|128|válido|
|practice_0014|multiplicative|Cruce|mejor independiente/coste0/orden1|128|128|válido|
|practice_0014|multiplicative|Cruce|mejor independiente/coste1/orden1|128|128|válido|
|practice_0014|multiplicative|E1|cola10/pol1/inicio0|131|39|válido|
|practice_0014|multiplicative|E1|cola10/pol1/inicio1|131|39|válido|
|practice_0014|multiplicative|E1|cola10/pol2/inicio0|131|39|válido|
|practice_0014|multiplicative|E1|cola10/pol2/inicio1|131|39|válido|
|practice_0014|multiplicative|E1|cola8/pol1/inicio0|130|32|válido|
|practice_0014|multiplicative|E1|cola8/pol1/inicio1|130|32|válido|
|practice_0014|multiplicative|E1|cola8/pol2/inicio0|130|32|válido|
|practice_0014|multiplicative|E1|cola8/pol2/inicio1|130|32|válido|
|practice_0014|multiplicative|E1|cola6/pol1/inicio0|130|22|válido|
|practice_0014|multiplicative|E1|cola6/pol1/inicio1|130|22|válido|
|practice_0014|multiplicative|E1|cola6/pol2/inicio0|130|22|válido|
|practice_0014|multiplicative|E1|cola6/pol2/inicio1|130|22|válido|
|practice_0014|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0014|multiplicative|E2|CP False/coste0/orden0|136|136|válido|
|practice_0014|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0014|multiplicative|E2|CP True/coste0/orden0|128|128|válido|
|practice_0014|multiplicative|E3|donante0/coste0/orden0|130|130|válido|
|practice_0014|multiplicative|E3|donante0/coste1/orden0|130|130|válido|
|practice_0014|multiplicative|E3|donante0/coste0/orden1|130|130|válido|
|practice_0014|multiplicative|E3|donante0/coste1/orden1|130|130|válido|
|practice_0014|multiplicative|E3|donante1/coste0/orden0|138|138|válido|
|practice_0014|multiplicative|E3|donante1/coste1/orden0|138|138|válido|
|practice_0014|multiplicative|E3|donante1/coste0/orden1|138|138|válido|
|practice_0014|multiplicative|E3|donante1/coste1/orden1|138|138|válido|
|practice_0014|multiplicative|E3|donante2/coste0/orden0|137|137|válido|
|practice_0014|multiplicative|E3|donante2/coste1/orden0|137|137|válido|
|practice_0014|multiplicative|E3|donante2/coste0/orden1|137|137|válido|
|practice_0014|multiplicative|E3|donante2/coste1/orden1|137|137|válido|
|practice_0014|multiplicative|E3|donante3/coste0/orden0|134|134|válido|
|practice_0014|multiplicative|E3|donante3/coste1/orden0|134|134|válido|
|practice_0014|multiplicative|E3|donante3/coste0/orden1|134|134|válido|
|practice_0014|multiplicative|E3|donante3/coste1/orden1|134|134|válido|
|practice_0014|multiplicative|E3|donante4/coste0/orden0|152|152|válido|
|practice_0014|multiplicative|E3|donante4/coste1/orden0|152|152|válido|
|practice_0014|multiplicative|E3|donante4/coste0/orden1|152|152|válido|
|practice_0014|multiplicative|E3|donante4/coste1/orden1|152|152|válido|
|practice_0014|multiplicative|E3|donante5/coste0/orden0|146|146|válido|
|practice_0014|multiplicative|E3|donante5/coste1/orden0|146|146|válido|
|practice_0014|multiplicative|E3|donante5/coste0/orden1|146|146|válido|
|practice_0014|multiplicative|E3|donante5/coste1/orden1|146|146|válido|
|practice_0014|multiplicative|E3|donante6/coste0/orden0|137|137|válido|
|practice_0014|multiplicative|E3|donante6/coste1/orden0|137|137|válido|
|practice_0014|multiplicative|E3|donante6/coste0/orden1|137|137|válido|
|practice_0014|multiplicative|E3|donante6/coste1/orden1|137|137|válido|
|practice_0014|multiplicative|E3|donante7/coste0/orden0|139|136|válido|
|practice_0014|multiplicative|E3|donante7/coste1/orden0|139|136|válido|
|practice_0014|multiplicative|E3|donante7/coste0/orden1|136|136|válido|
|practice_0014|multiplicative|E3|donante7/coste1/orden1|136|136|válido|
|practice_0014|multiplicative|E3|donante8/coste0/orden0|132|132|válido|
|practice_0014|multiplicative|E3|donante8/coste1/orden0|132|132|válido|
|practice_0014|multiplicative|E3|donante8/coste0/orden1|132|132|válido|
|practice_0014|multiplicative|E3|donante8/coste1/orden1|132|132|válido|
|practice_0014|multiplicative|E3|donante9/coste0/orden0|131|131|válido|
|practice_0014|multiplicative|E3|donante9/coste1/orden0|131|131|válido|
|practice_0014|multiplicative|E3|donante9/coste0/orden1|131|131|válido|
|practice_0014|multiplicative|E3|donante9/coste1/orden1|131|131|válido|
|practice_0015|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0015|shared|Cruce|CP True/coste0/orden0|204|182|válido|
|practice_0015|shared|Cruce|CP True/coste1/orden0|187|182|válido|
|practice_0015|shared|Cruce|CP True/coste0/orden1|182|182|válido|
|practice_0015|shared|Cruce|CP True/coste1/orden1|182|182|válido|
|practice_0015|shared|Cruce|mejor independiente/coste0/orden0|239|185|válido|
|practice_0015|shared|Cruce|mejor independiente/coste1/orden0|215|185|válido|
|practice_0015|shared|Cruce|mejor independiente/coste0/orden1|185|185|válido|
|practice_0015|shared|Cruce|mejor independiente/coste1/orden1|185|185|válido|
|practice_0015|shared|E1|cola10/pol1/inicio0|191|60|válido|
|practice_0015|shared|E1|cola10/pol1/inicio1|190|59|válido|
|practice_0015|shared|E1|cola10/pol2/inicio0|192|61|válido|
|practice_0015|shared|E1|cola10/pol2/inicio1|190|59|válido|
|practice_0015|shared|E1|cola8/pol1/inicio0|191|48|válido|
|practice_0015|shared|E1|cola8/pol1/inicio1|191|48|válido|
|practice_0015|shared|E1|cola8/pol2/inicio0|191|48|válido|
|practice_0015|shared|E1|cola8/pol2/inicio1|191|48|válido|
|practice_0015|shared|E1|cola6/pol1/inicio0|185|30|válido|
|practice_0015|shared|E1|cola6/pol1/inicio1|185|30|válido|
|practice_0015|shared|E1|cola6/pol2/inicio0|185|30|válido|
|practice_0015|shared|E1|cola6/pol2/inicio1|185|30|válido|
|practice_0015|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0015|shared|E2|CP False/coste0/orden0|196|196|válido|
|practice_0015|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0015|shared|E2|CP True/coste0/orden0|212|182|válido|
|practice_0015|shared|E3|donante0/coste0/orden0|186|186|válido|
|practice_0015|shared|E3|donante0/coste1/orden0|186|186|válido|
|practice_0015|shared|E3|donante0/coste0/orden1|188|186|válido|
|practice_0015|shared|E3|donante0/coste1/orden1|188|186|válido|
|practice_0015|shared|E3|donante1/coste0/orden0|205|205|válido|
|practice_0015|shared|E3|donante1/coste1/orden0|205|205|válido|
|practice_0015|shared|E3|donante1/coste0/orden1|205|205|válido|
|practice_0015|shared|E3|donante1/coste1/orden1|205|205|válido|
|practice_0015|shared|E3|donante2/coste0/orden0|191|191|válido|
|practice_0015|shared|E3|donante2/coste1/orden0|191|191|válido|
|practice_0015|shared|E3|donante2/coste0/orden1|191|191|válido|
|practice_0015|shared|E3|donante2/coste1/orden1|191|191|válido|
|practice_0015|shared|E3|donante3/coste0/orden0|218|194|válido|
|practice_0015|shared|E3|donante3/coste1/orden0|213|194|válido|
|practice_0015|shared|E3|donante3/coste0/orden1|194|194|válido|
|practice_0015|shared|E3|donante3/coste1/orden1|194|194|válido|
|practice_0015|shared|E3|donante4/coste0/orden0|245|245|válido|
|practice_0015|shared|E3|donante4/coste1/orden0|245|245|válido|
|practice_0015|shared|E3|donante4/coste0/orden1|245|245|válido|
|practice_0015|shared|E3|donante4/coste1/orden1|245|245|válido|
|practice_0015|shared|E3|donante5/coste0/orden0|208|208|válido|
|practice_0015|shared|E3|donante5/coste1/orden0|208|208|válido|
|practice_0015|shared|E3|donante5/coste0/orden1|208|208|válido|
|practice_0015|shared|E3|donante5/coste1/orden1|208|208|válido|
|practice_0015|shared|E3|donante6/coste0/orden0|229|229|válido|
|practice_0015|shared|E3|donante6/coste1/orden0|229|229|válido|
|practice_0015|shared|E3|donante6/coste0/orden1|229|229|válido|
|practice_0015|shared|E3|donante6/coste1/orden1|229|229|válido|
|practice_0015|shared|E3|donante7/coste0/orden0|229|203|válido|
|practice_0015|shared|E3|donante7/coste1/orden0|217|203|válido|
|practice_0015|shared|E3|donante7/coste0/orden1|235|203|válido|
|practice_0015|shared|E3|donante7/coste1/orden1|219|203|válido|
|practice_0015|shared|E3|donante8/coste0/orden0|196|196|válido|
|practice_0015|shared|E3|donante8/coste1/orden0|196|196|válido|
|practice_0015|shared|E3|donante8/coste0/orden1|228|196|válido|
|practice_0015|shared|E3|donante8/coste1/orden1|218|196|válido|
|practice_0016|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0016|shared|Cruce|CP True/coste0/orden0|226|202|válido|
|practice_0016|shared|Cruce|CP True/coste1/orden0|207|202|válido|
|practice_0016|shared|Cruce|CP True/coste0/orden1|208|202|válido|
|practice_0016|shared|Cruce|CP True/coste1/orden1|207|202|válido|
|practice_0016|shared|Cruce|mejor independiente/coste0/orden0|254|204|válido|
|practice_0016|shared|Cruce|mejor independiente/coste1/orden0|216|204|válido|
|practice_0016|shared|Cruce|mejor independiente/coste0/orden1|204|204|válido|
|practice_0016|shared|Cruce|mejor independiente/coste1/orden1|204|204|válido|
|practice_0016|shared|E1|cola10/pol1/inicio0|211|71|válido|
|practice_0016|shared|E1|cola10/pol1/inicio1|211|71|válido|
|practice_0016|shared|E1|cola10/pol2/inicio0|210|70|válido|
|practice_0016|shared|E1|cola10/pol2/inicio1|210|70|válido|
|practice_0016|shared|E1|cola8/pol1/inicio0|220|62|válido|
|practice_0016|shared|E1|cola8/pol1/inicio1|223|65|válido|
|practice_0016|shared|E1|cola8/pol2/inicio0|222|64|válido|
|practice_0016|shared|E1|cola8/pol2/inicio1|214|56|válido|
|practice_0016|shared|E1|cola6/pol1/inicio0|212|42|válido|
|practice_0016|shared|E1|cola6/pol1/inicio1|212|42|válido|
|practice_0016|shared|E1|cola6/pol2/inicio0|212|42|válido|
|practice_0016|shared|E1|cola6/pol2/inicio1|212|42|válido|
|practice_0016|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0016|shared|E2|CP False/coste0/orden0|237|211|válido|
|practice_0016|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0016|shared|E2|CP True/coste0/orden0|226|202|válido|
|practice_0016|shared|E3|donante0/coste0/orden0|209|204|válido|
|practice_0016|shared|E3|donante0/coste1/orden0|209|204|válido|
|practice_0016|shared|E3|donante0/coste0/orden1|204|204|válido|
|practice_0016|shared|E3|donante0/coste1/orden1|204|204|válido|
|practice_0016|shared|E3|donante1/coste0/orden0|235|235|válido|
|practice_0016|shared|E3|donante1/coste1/orden0|235|235|válido|
|practice_0016|shared|E3|donante1/coste0/orden1|235|235|válido|
|practice_0016|shared|E3|donante1/coste1/orden1|235|235|válido|
|practice_0016|shared|E3|donante2/coste0/orden0|221|221|válido|
|practice_0016|shared|E3|donante2/coste1/orden0|221|221|válido|
|practice_0016|shared|E3|donante2/coste0/orden1|221|221|válido|
|practice_0016|shared|E3|donante2/coste1/orden1|221|221|válido|
|practice_0016|shared|E3|donante3/coste0/orden0|383|226|válido|
|practice_0016|shared|E3|donante3/coste1/orden0|243|226|válido|
|practice_0016|shared|E3|donante3/coste0/orden1|226|226|válido|
|practice_0016|shared|E3|donante3/coste1/orden1|226|226|válido|
|practice_0016|shared|E3|donante4/coste0/orden0|255|255|válido|
|practice_0016|shared|E3|donante4/coste1/orden0|255|255|válido|
|practice_0016|shared|E3|donante4/coste0/orden1|255|255|válido|
|practice_0016|shared|E3|donante4/coste1/orden1|255|255|válido|
|practice_0016|shared|E3|donante5/coste0/orden0|227|227|válido|
|practice_0016|shared|E3|donante5/coste1/orden0|227|227|válido|
|practice_0016|shared|E3|donante5/coste0/orden1|227|227|válido|
|practice_0016|shared|E3|donante5/coste1/orden1|227|227|válido|
|practice_0016|shared|E3|donante6/coste0/orden0|247|247|válido|
|practice_0016|shared|E3|donante6/coste1/orden0|247|247|válido|
|practice_0016|shared|E3|donante6/coste0/orden1|247|247|válido|
|practice_0016|shared|E3|donante6/coste1/orden1|247|247|válido|
|practice_0016|shared|E3|donante7/coste0/orden0|238|215|válido|
|practice_0016|shared|E3|donante7/coste1/orden0|221|215|válido|
|practice_0016|shared|E3|donante7/coste0/orden1|250|215|válido|
|practice_0016|shared|E3|donante7/coste1/orden1|238|215|válido|
|practice_0016|shared|E3|donante8/coste0/orden0|209|204|válido|
|practice_0016|shared|E3|donante8/coste1/orden0|209|204|válido|
|practice_0016|shared|E3|donante8/coste0/orden1|204|204|válido|
|practice_0016|shared|E3|donante8/coste1/orden1|204|204|válido|
|practice_0016|shared|E3|donante9/coste0/orden0|225|205|válido|
|practice_0016|shared|E3|donante9/coste1/orden0|207|205|válido|
|practice_0016|shared|E3|donante9/coste0/orden1|225|205|válido|
|practice_0016|shared|E3|donante9/coste1/orden1|223|205|válido|
|practice_0017|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0017|shared|Cruce|CP True/coste0/orden0|148|147|válido|
|practice_0017|shared|Cruce|CP True/coste1/orden0|148|147|válido|
|practice_0017|shared|Cruce|CP True/coste0/orden1|147|147|válido|
|practice_0017|shared|Cruce|CP True/coste1/orden1|147|147|válido|
|practice_0017|shared|Cruce|mejor independiente/coste0/orden0|164|149|válido|
|practice_0017|shared|Cruce|mejor independiente/coste1/orden0|156|149|válido|
|practice_0017|shared|Cruce|mejor independiente/coste0/orden1|149|149|válido|
|practice_0017|shared|Cruce|mejor independiente/coste1/orden1|149|149|válido|
|practice_0017|shared|E1|cola10/pol1/inicio0|158|55|válido|
|practice_0017|shared|E1|cola10/pol1/inicio1|158|55|válido|
|practice_0017|shared|E1|cola10/pol2/inicio0|158|55|válido|
|practice_0017|shared|E1|cola10/pol2/inicio1|158|55|válido|
|practice_0017|shared|E1|cola8/pol1/inicio0|156|44|válido|
|practice_0017|shared|E1|cola8/pol1/inicio1|156|44|válido|
|practice_0017|shared|E1|cola8/pol2/inicio0|156|44|válido|
|practice_0017|shared|E1|cola8/pol2/inicio1|156|44|válido|
|practice_0017|shared|E1|cola6/pol1/inicio0|154|33|válido|
|practice_0017|shared|E1|cola6/pol1/inicio1|154|33|válido|
|practice_0017|shared|E1|cola6/pol2/inicio0|150|29|válido|
|practice_0017|shared|E1|cola6/pol2/inicio1|150|29|válido|
|practice_0017|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0017|shared|E2|CP False/coste0/orden0|173|165|válido|
|practice_0017|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0017|shared|E2|CP True/coste0/orden0|155|146|válido|
|practice_0017|shared|E3|donante0/coste0/orden0|158|149|válido|
|practice_0017|shared|E3|donante0/coste1/orden0|158|149|válido|
|practice_0017|shared|E3|donante0/coste0/orden1|149|149|válido|
|practice_0017|shared|E3|donante0/coste1/orden1|149|149|válido|
|practice_0017|shared|E3|donante1/coste0/orden0|176|176|válido|
|practice_0017|shared|E3|donante1/coste1/orden0|176|176|válido|
|practice_0017|shared|E3|donante1/coste0/orden1|176|176|válido|
|practice_0017|shared|E3|donante1/coste1/orden1|176|176|válido|
|practice_0017|shared|E3|donante2/coste0/orden0|169|169|válido|
|practice_0017|shared|E3|donante2/coste1/orden0|169|169|válido|
|practice_0017|shared|E3|donante2/coste0/orden1|169|169|válido|
|practice_0017|shared|E3|donante2/coste1/orden1|169|169|válido|
|practice_0017|shared|E3|donante3/coste0/orden0|169|169|válido|
|practice_0017|shared|E3|donante3/coste1/orden0|169|169|válido|
|practice_0017|shared|E3|donante3/coste0/orden1|169|169|válido|
|practice_0017|shared|E3|donante3/coste1/orden1|169|169|válido|
|practice_0017|shared|E3|donante4/coste0/orden0|189|189|válido|
|practice_0017|shared|E3|donante4/coste1/orden0|189|189|válido|
|practice_0017|shared|E3|donante4/coste0/orden1|189|189|válido|
|practice_0017|shared|E3|donante4/coste1/orden1|189|189|válido|
|practice_0017|shared|E3|donante5/coste0/orden0|184|184|válido|
|practice_0017|shared|E3|donante5/coste1/orden0|184|184|válido|
|practice_0017|shared|E3|donante5/coste0/orden1|184|184|válido|
|practice_0017|shared|E3|donante5/coste1/orden1|184|184|válido|
|practice_0017|shared|E3|donante6/coste0/orden0|174|174|válido|
|practice_0017|shared|E3|donante6/coste1/orden0|174|174|válido|
|practice_0017|shared|E3|donante6/coste0/orden1|174|174|válido|
|practice_0017|shared|E3|donante6/coste1/orden1|174|174|válido|
|practice_0017|shared|E3|donante7/coste0/orden0|179|162|válido|
|practice_0017|shared|E3|donante7/coste1/orden0|179|162|válido|
|practice_0017|shared|E3|donante7/coste0/orden1|183|162|válido|
|practice_0017|shared|E3|donante7/coste1/orden1|173|162|válido|
|practice_0017|shared|E3|donante8/coste0/orden0|176|163|válido|
|practice_0017|shared|E3|donante8/coste1/orden0|169|163|válido|
|practice_0017|shared|E3|donante8/coste0/orden1|163|163|válido|
|practice_0017|shared|E3|donante8/coste1/orden1|163|163|válido|
|practice_0017|shared|E3|donante9/coste0/orden0|158|149|válido|
|practice_0017|shared|E3|donante9/coste1/orden0|158|149|válido|
|practice_0017|shared|E3|donante9/coste0/orden1|149|149|válido|
|practice_0017|shared|E3|donante9/coste1/orden1|149|149|válido|
|practice_0018|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0018|independent|Cruce|CP True/coste0/orden0|518|297|válido|
|practice_0018|independent|Cruce|CP True/coste1/orden0|492|297|válido|
|practice_0018|independent|Cruce|CP True/coste0/orden1|343|297|válido|
|practice_0018|independent|Cruce|CP True/coste1/orden1|342|297|válido|
|practice_0018|independent|Cruce|mejor independiente/coste0/orden0|518|297|válido|
|practice_0018|independent|Cruce|mejor independiente/coste1/orden0|492|297|válido|
|practice_0018|independent|Cruce|mejor independiente/coste0/orden1|343|297|válido|
|practice_0018|independent|Cruce|mejor independiente/coste1/orden1|342|297|válido|
|practice_0018|independent|E1|cola10/pol1/inicio0|314|92|válido|
|practice_0018|independent|E1|cola10/pol1/inicio1|314|92|válido|
|practice_0018|independent|E1|cola10/pol2/inicio0|309|87|válido|
|practice_0018|independent|E1|cola10/pol2/inicio1|314|92|válido|
|practice_0018|independent|E1|cola8/pol1/inicio0|318|74|válido|
|practice_0018|independent|E1|cola8/pol1/inicio1|319|75|válido|
|practice_0018|independent|E1|cola8/pol2/inicio0|320|76|válido|
|practice_0018|independent|E1|cola8/pol2/inicio1|318|74|válido|
|practice_0018|independent|E1|cola6/pol1/inicio0|315|54|válido|
|practice_0018|independent|E1|cola6/pol1/inicio1|317|56|válido|
|practice_0018|independent|E1|cola6/pol2/inicio0|314|53|válido|
|practice_0018|independent|E1|cola6/pol2/inicio1|314|53|válido|
|practice_0018|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0018|independent|E2|CP False/coste0/orden0|740|348|válido|
|practice_0018|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0018|independent|E2|CP True/coste0/orden0|527|296|válido|
|practice_0018|independent|E3|donante0/coste0/orden0|304|296|válido|
|practice_0018|independent|E3|donante0/coste1/orden0|304|296|válido|
|practice_0018|independent|E3|donante0/coste0/orden1|343|296|válido|
|practice_0018|independent|E3|donante0/coste1/orden1|342|296|válido|
|practice_0018|independent|E3|donante1/coste0/orden0|342|342|válido|
|practice_0018|independent|E3|donante1/coste1/orden0|342|342|válido|
|practice_0018|independent|E3|donante1/coste0/orden1|342|342|válido|
|practice_0018|independent|E3|donante1/coste1/orden1|342|342|válido|
|practice_0018|independent|E3|donante2/coste0/orden0|661|320|válido|
|practice_0018|independent|E3|donante2/coste1/orden0|384|320|válido|
|practice_0018|independent|E3|donante2/coste0/orden1|343|320|válido|
|practice_0018|independent|E3|donante2/coste1/orden1|321|320|válido|
|practice_0018|independent|E3|donante3/coste0/orden0|564|325|válido|
|practice_0018|independent|E3|donante3/coste1/orden0|476|325|válido|
|practice_0018|independent|E3|donante3/coste0/orden1|525|325|válido|
|practice_0018|independent|E3|donante3/coste1/orden1|338|325|válido|
|practice_0018|independent|E3|donante4/coste0/orden0|358|358|válido|
|practice_0018|independent|E3|donante4/coste1/orden0|358|358|válido|
|practice_0018|independent|E3|donante4/coste0/orden1|358|358|válido|
|practice_0018|independent|E3|donante4/coste1/orden1|358|358|válido|
|practice_0018|independent|E3|donante5/coste0/orden0|358|358|válido|
|practice_0018|independent|E3|donante5/coste1/orden0|358|358|válido|
|practice_0018|independent|E3|donante5/coste0/orden1|358|358|válido|
|practice_0018|independent|E3|donante5/coste1/orden1|358|358|válido|
|practice_0018|independent|E3|donante6/coste0/orden0|348|348|válido|
|practice_0018|independent|E3|donante6/coste1/orden0|348|348|válido|
|practice_0018|independent|E3|donante6/coste0/orden1|348|348|válido|
|practice_0018|independent|E3|donante6/coste1/orden1|348|348|válido|
|practice_0018|independent|E3|donante7/coste0/orden0|361|315|válido|
|practice_0018|independent|E3|donante7/coste1/orden0|335|315|válido|
|practice_0018|independent|E3|donante7/coste0/orden1|377|315|válido|
|practice_0018|independent|E3|donante7/coste1/orden1|385|315|válido|
|practice_0018|independent|E3|donante8/coste0/orden0|444|303|válido|
|practice_0018|independent|E3|donante8/coste1/orden0|403|303|válido|
|practice_0018|independent|E3|donante8/coste0/orden1|459|303|válido|
|practice_0018|independent|E3|donante8/coste1/orden1|385|303|válido|
|practice_0019|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0019|shared|Cruce|CP True/coste0/orden0|170|170|válido|
|practice_0019|shared|Cruce|CP True/coste1/orden0|170|170|válido|
|practice_0019|shared|Cruce|CP True/coste0/orden1|170|170|válido|
|practice_0019|shared|Cruce|CP True/coste1/orden1|170|170|válido|
|practice_0019|shared|Cruce|mejor independiente/coste0/orden0|172|172|válido|
|practice_0019|shared|Cruce|mejor independiente/coste1/orden0|172|172|válido|
|practice_0019|shared|Cruce|mejor independiente/coste0/orden1|172|172|válido|
|practice_0019|shared|Cruce|mejor independiente/coste1/orden1|172|172|válido|
|practice_0019|shared|E1|cola10/pol1/inicio0|182|65|válido|
|practice_0019|shared|E1|cola10/pol1/inicio1|182|65|válido|
|practice_0019|shared|E1|cola10/pol2/inicio0|182|65|válido|
|practice_0019|shared|E1|cola10/pol2/inicio1|182|65|válido|
|practice_0019|shared|E1|cola8/pol1/inicio0|176|46|válido|
|practice_0019|shared|E1|cola8/pol1/inicio1|176|46|válido|
|practice_0019|shared|E1|cola8/pol2/inicio0|181|51|válido|
|practice_0019|shared|E1|cola8/pol2/inicio1|181|51|válido|
|practice_0019|shared|E1|cola6/pol1/inicio0|178|39|válido|
|practice_0019|shared|E1|cola6/pol1/inicio1|178|39|válido|
|practice_0019|shared|E1|cola6/pol2/inicio0|178|39|válido|
|practice_0019|shared|E1|cola6/pol2/inicio1|178|39|válido|
|practice_0019|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0019|shared|E2|CP False/coste0/orden0|196|196|válido|
|practice_0019|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0019|shared|E2|CP True/coste0/orden0|172|172|válido|
|practice_0019|shared|E3|donante0/coste0/orden0|175|175|válido|
|practice_0019|shared|E3|donante0/coste1/orden0|175|175|válido|
|practice_0019|shared|E3|donante0/coste0/orden1|175|175|válido|
|practice_0019|shared|E3|donante0/coste1/orden1|175|175|válido|
|practice_0019|shared|E3|donante1/coste0/orden0|211|211|válido|
|practice_0019|shared|E3|donante1/coste1/orden0|211|211|válido|
|practice_0019|shared|E3|donante1/coste0/orden1|211|211|válido|
|practice_0019|shared|E3|donante1/coste1/orden1|211|211|válido|
|practice_0019|shared|E3|donante2/coste0/orden0|201|201|válido|
|practice_0019|shared|E3|donante2/coste1/orden0|201|201|válido|
|practice_0019|shared|E3|donante2/coste0/orden1|201|201|válido|
|practice_0019|shared|E3|donante2/coste1/orden1|201|201|válido|
|practice_0019|shared|E3|donante3/coste0/orden0|196|196|válido|
|practice_0019|shared|E3|donante3/coste1/orden0|196|196|válido|
|practice_0019|shared|E3|donante3/coste0/orden1|196|196|válido|
|practice_0019|shared|E3|donante3/coste1/orden1|196|196|válido|
|practice_0019|shared|E3|donante4/coste0/orden0|228|228|válido|
|practice_0019|shared|E3|donante4/coste1/orden0|228|228|válido|
|practice_0019|shared|E3|donante4/coste0/orden1|228|228|válido|
|practice_0019|shared|E3|donante4/coste1/orden1|228|228|válido|
|practice_0019|shared|E3|donante5/coste0/orden0|194|194|válido|
|practice_0019|shared|E3|donante5/coste1/orden0|194|194|válido|
|practice_0019|shared|E3|donante5/coste0/orden1|194|194|válido|
|practice_0019|shared|E3|donante5/coste1/orden1|194|194|válido|
|practice_0019|shared|E3|donante6/coste0/orden0|215|215|válido|
|practice_0019|shared|E3|donante6/coste1/orden0|215|215|válido|
|practice_0019|shared|E3|donante6/coste0/orden1|215|215|válido|
|practice_0019|shared|E3|donante6/coste1/orden1|215|215|válido|
|practice_0019|shared|E3|donante7/coste0/orden0|182|180|válido|
|practice_0019|shared|E3|donante7/coste1/orden0|182|180|válido|
|practice_0019|shared|E3|donante7/coste0/orden1|180|180|válido|
|practice_0019|shared|E3|donante7/coste1/orden1|180|180|válido|
|practice_0019|shared|E3|donante8/coste0/orden0|251|177|válido|
|practice_0019|shared|E3|donante8/coste1/orden0|192|177|válido|
|practice_0019|shared|E3|donante8/coste0/orden1|248|177|válido|
|practice_0019|shared|E3|donante8/coste1/orden1|187|177|válido|
|practice_0019|shared|E3|donante9/coste0/orden0|176|176|válido|
|practice_0019|shared|E3|donante9/coste1/orden0|176|176|válido|
|practice_0019|shared|E3|donante9/coste0/orden1|176|176|válido|
|practice_0019|shared|E3|donante9/coste1/orden1|176|176|válido|
|practice_0020|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0020|multiplicative|Cruce|CP True/coste0/orden0|106|106|válido|
|practice_0020|multiplicative|Cruce|CP True/coste1/orden0|106|106|válido|
|practice_0020|multiplicative|Cruce|CP True/coste0/orden1|106|106|válido|
|practice_0020|multiplicative|Cruce|CP True/coste1/orden1|106|106|válido|
|practice_0020|multiplicative|Cruce|mejor independiente/coste0/orden0|106|106|válido|
|practice_0020|multiplicative|Cruce|mejor independiente/coste1/orden0|106|106|válido|
|practice_0020|multiplicative|Cruce|mejor independiente/coste0/orden1|106|106|válido|
|practice_0020|multiplicative|Cruce|mejor independiente/coste1/orden1|106|106|válido|
|practice_0020|multiplicative|E1|cola10/pol1/inicio0|—|—|registros ocupados|
|practice_0020|multiplicative|E1|cola10/pol1/inicio1|—|—|registros ocupados|
|practice_0020|multiplicative|E1|cola10/pol2/inicio0|—|—|registros ocupados|
|practice_0020|multiplicative|E1|cola10/pol2/inicio1|—|—|registros ocupados|
|practice_0020|multiplicative|E1|cola8/pol1/inicio0|—|—|registros ocupados|
|practice_0020|multiplicative|E1|cola8/pol1/inicio1|—|—|registros ocupados|
|practice_0020|multiplicative|E1|cola8/pol2/inicio0|—|—|registros ocupados|
|practice_0020|multiplicative|E1|cola8/pol2/inicio1|—|—|registros ocupados|
|practice_0020|multiplicative|E1|cola6/pol1/inicio0|—|—|registros ocupados|
|practice_0020|multiplicative|E1|cola6/pol1/inicio1|—|—|registros ocupados|
|practice_0020|multiplicative|E1|cola6/pol2/inicio0|—|—|registros ocupados|
|practice_0020|multiplicative|E1|cola6/pol2/inicio1|—|—|registros ocupados|
|practice_0020|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0020|multiplicative|E2|CP False/coste0/orden0|108|108|válido|
|practice_0020|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0020|multiplicative|E2|CP True/coste0/orden0|106|106|válido|
|practice_0020|multiplicative|E3|donante0/coste0/orden0|106|106|válido|
|practice_0020|multiplicative|E3|donante0/coste1/orden0|106|106|válido|
|practice_0020|multiplicative|E3|donante0/coste0/orden1|106|106|válido|
|practice_0020|multiplicative|E3|donante0/coste1/orden1|106|106|válido|
|practice_0020|multiplicative|E3|donante1/coste0/orden0|114|114|válido|
|practice_0020|multiplicative|E3|donante1/coste1/orden0|114|114|válido|
|practice_0020|multiplicative|E3|donante1/coste0/orden1|114|114|válido|
|practice_0020|multiplicative|E3|donante1/coste1/orden1|114|114|válido|
|practice_0020|multiplicative|E3|donante2/coste0/orden0|112|112|válido|
|practice_0020|multiplicative|E3|donante2/coste1/orden0|112|112|válido|
|practice_0020|multiplicative|E3|donante2/coste0/orden1|112|112|válido|
|practice_0020|multiplicative|E3|donante2/coste1/orden1|112|112|válido|
|practice_0020|multiplicative|E3|donante3/coste0/orden0|109|109|válido|
|practice_0020|multiplicative|E3|donante3/coste1/orden0|109|109|válido|
|practice_0020|multiplicative|E3|donante3/coste0/orden1|109|109|válido|
|practice_0020|multiplicative|E3|donante3/coste1/orden1|109|109|válido|
|practice_0020|multiplicative|E3|donante4/coste0/orden0|112|112|válido|
|practice_0020|multiplicative|E3|donante4/coste1/orden0|112|112|válido|
|practice_0020|multiplicative|E3|donante4/coste0/orden1|112|112|válido|
|practice_0020|multiplicative|E3|donante4/coste1/orden1|112|112|válido|
|practice_0020|multiplicative|E3|donante5/coste0/orden0|115|115|válido|
|practice_0020|multiplicative|E3|donante5/coste1/orden0|115|115|válido|
|practice_0020|multiplicative|E3|donante5/coste0/orden1|115|115|válido|
|practice_0020|multiplicative|E3|donante5/coste1/orden1|115|115|válido|
|practice_0020|multiplicative|E3|donante6/coste0/orden0|111|111|válido|
|practice_0020|multiplicative|E3|donante6/coste1/orden0|111|111|válido|
|practice_0020|multiplicative|E3|donante6/coste0/orden1|111|111|válido|
|practice_0020|multiplicative|E3|donante6/coste1/orden1|111|111|válido|
|practice_0020|multiplicative|E3|donante7/coste0/orden0|110|110|válido|
|practice_0020|multiplicative|E3|donante7/coste1/orden0|110|110|válido|
|practice_0020|multiplicative|E3|donante7/coste0/orden1|110|110|válido|
|practice_0020|multiplicative|E3|donante7/coste1/orden1|110|110|válido|
|practice_0020|multiplicative|E3|donante8/coste0/orden0|108|108|válido|
|practice_0020|multiplicative|E3|donante8/coste1/orden0|108|108|válido|
|practice_0020|multiplicative|E3|donante8/coste0/orden1|108|108|válido|
|practice_0020|multiplicative|E3|donante8/coste1/orden1|108|108|válido|
|practice_0020|multiplicative|E3|donante9/coste0/orden0|107|107|válido|
|practice_0020|multiplicative|E3|donante9/coste1/orden0|107|107|válido|
|practice_0020|multiplicative|E3|donante9/coste0/orden1|107|107|válido|
|practice_0020|multiplicative|E3|donante9/coste1/orden1|107|107|válido|
|practice_0021|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0021|multiplicative|Cruce|CP True/coste0/orden0|128|128|válido|
|practice_0021|multiplicative|Cruce|CP True/coste1/orden0|128|128|válido|
|practice_0021|multiplicative|Cruce|CP True/coste0/orden1|128|128|válido|
|practice_0021|multiplicative|Cruce|CP True/coste1/orden1|128|128|válido|
|practice_0021|multiplicative|Cruce|mejor independiente/coste0/orden0|128|128|válido|
|practice_0021|multiplicative|Cruce|mejor independiente/coste1/orden0|128|128|válido|
|practice_0021|multiplicative|Cruce|mejor independiente/coste0/orden1|128|128|válido|
|practice_0021|multiplicative|Cruce|mejor independiente/coste1/orden1|128|128|válido|
|practice_0021|multiplicative|E1|cola10/pol1/inicio0|137|50|válido|
|practice_0021|multiplicative|E1|cola10/pol1/inicio1|137|50|válido|
|practice_0021|multiplicative|E1|cola10/pol2/inicio0|137|50|válido|
|practice_0021|multiplicative|E1|cola10/pol2/inicio1|137|50|válido|
|practice_0021|multiplicative|E1|cola8/pol1/inicio0|133|38|válido|
|practice_0021|multiplicative|E1|cola8/pol1/inicio1|133|38|válido|
|practice_0021|multiplicative|E1|cola8/pol2/inicio0|128|33|válido|
|practice_0021|multiplicative|E1|cola8/pol2/inicio1|128|33|válido|
|practice_0021|multiplicative|E1|cola6/pol1/inicio0|128|25|válido|
|practice_0021|multiplicative|E1|cola6/pol1/inicio1|128|25|válido|
|practice_0021|multiplicative|E1|cola6/pol2/inicio0|128|25|válido|
|practice_0021|multiplicative|E1|cola6/pol2/inicio1|128|25|válido|
|practice_0021|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0021|multiplicative|E2|CP False/coste0/orden0|128|128|válido|
|practice_0021|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0021|multiplicative|E2|CP True/coste0/orden0|128|128|válido|
|practice_0021|multiplicative|E3|donante0/coste0/orden0|128|128|válido|
|practice_0021|multiplicative|E3|donante0/coste1/orden0|128|128|válido|
|practice_0021|multiplicative|E3|donante0/coste0/orden1|128|128|válido|
|practice_0021|multiplicative|E3|donante0/coste1/orden1|128|128|válido|
|practice_0021|multiplicative|E3|donante1/coste0/orden0|136|136|válido|
|practice_0021|multiplicative|E3|donante1/coste1/orden0|136|136|válido|
|practice_0021|multiplicative|E3|donante1/coste0/orden1|136|136|válido|
|practice_0021|multiplicative|E3|donante1/coste1/orden1|136|136|válido|
|practice_0021|multiplicative|E3|donante2/coste0/orden0|131|131|válido|
|practice_0021|multiplicative|E3|donante2/coste1/orden0|131|131|válido|
|practice_0021|multiplicative|E3|donante2/coste0/orden1|131|131|válido|
|practice_0021|multiplicative|E3|donante2/coste1/orden1|131|131|válido|
|practice_0021|multiplicative|E3|donante3/coste0/orden0|132|132|válido|
|practice_0021|multiplicative|E3|donante3/coste1/orden0|132|132|válido|
|practice_0021|multiplicative|E3|donante3/coste0/orden1|132|132|válido|
|practice_0021|multiplicative|E3|donante3/coste1/orden1|132|132|válido|
|practice_0021|multiplicative|E3|donante4/coste0/orden0|141|141|válido|
|practice_0021|multiplicative|E3|donante4/coste1/orden0|141|141|válido|
|practice_0021|multiplicative|E3|donante4/coste0/orden1|141|141|válido|
|practice_0021|multiplicative|E3|donante4/coste1/orden1|141|141|válido|
|practice_0021|multiplicative|E3|donante5/coste0/orden0|140|140|válido|
|practice_0021|multiplicative|E3|donante5/coste1/orden0|140|140|válido|
|practice_0021|multiplicative|E3|donante5/coste0/orden1|140|140|válido|
|practice_0021|multiplicative|E3|donante5/coste1/orden1|140|140|válido|
|practice_0021|multiplicative|E3|donante6/coste0/orden0|133|133|válido|
|practice_0021|multiplicative|E3|donante6/coste1/orden0|133|133|válido|
|practice_0021|multiplicative|E3|donante6/coste0/orden1|133|133|válido|
|practice_0021|multiplicative|E3|donante6/coste1/orden1|133|133|válido|
|practice_0021|multiplicative|E3|donante7/coste0/orden0|130|130|válido|
|practice_0021|multiplicative|E3|donante7/coste1/orden0|130|130|válido|
|practice_0021|multiplicative|E3|donante7/coste0/orden1|130|130|válido|
|practice_0021|multiplicative|E3|donante7/coste1/orden1|130|130|válido|
|practice_0021|multiplicative|E3|donante8/coste0/orden0|129|129|válido|
|practice_0021|multiplicative|E3|donante8/coste1/orden0|129|129|válido|
|practice_0021|multiplicative|E3|donante8/coste0/orden1|129|129|válido|
|practice_0021|multiplicative|E3|donante8/coste1/orden1|129|129|válido|
|practice_0022|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0022|multiplicative|Cruce|CP True/coste0/orden0|134|134|válido|
|practice_0022|multiplicative|Cruce|CP True/coste1/orden0|134|134|válido|
|practice_0022|multiplicative|Cruce|CP True/coste0/orden1|134|134|válido|
|practice_0022|multiplicative|Cruce|CP True/coste1/orden1|134|134|válido|
|practice_0022|multiplicative|Cruce|mejor independiente/coste0/orden0|134|134|válido|
|practice_0022|multiplicative|Cruce|mejor independiente/coste1/orden0|134|134|válido|
|practice_0022|multiplicative|Cruce|mejor independiente/coste0/orden1|134|134|válido|
|practice_0022|multiplicative|Cruce|mejor independiente/coste1/orden1|134|134|válido|
|practice_0022|multiplicative|E1|cola10/pol1/inicio0|—|—|registros ocupados|
|practice_0022|multiplicative|E1|cola10/pol1/inicio1|—|—|registros ocupados|
|practice_0022|multiplicative|E1|cola10/pol2/inicio0|—|—|registros ocupados|
|practice_0022|multiplicative|E1|cola10/pol2/inicio1|—|—|registros ocupados|
|practice_0022|multiplicative|E1|cola8/pol1/inicio0|—|—|registros ocupados|
|practice_0022|multiplicative|E1|cola8/pol1/inicio1|—|—|registros ocupados|
|practice_0022|multiplicative|E1|cola8/pol2/inicio0|—|—|registros ocupados|
|practice_0022|multiplicative|E1|cola8/pol2/inicio1|—|—|registros ocupados|
|practice_0022|multiplicative|E1|cola6/pol1/inicio0|—|—|registros ocupados|
|practice_0022|multiplicative|E1|cola6/pol1/inicio1|—|—|registros ocupados|
|practice_0022|multiplicative|E1|cola6/pol2/inicio0|—|—|registros ocupados|
|practice_0022|multiplicative|E1|cola6/pol2/inicio1|—|—|registros ocupados|
|practice_0022|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0022|multiplicative|E2|CP False/coste0/orden0|141|141|válido|
|practice_0022|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0022|multiplicative|E2|CP True/coste0/orden0|134|134|válido|
|practice_0022|multiplicative|E3|donante0/coste0/orden0|134|134|válido|
|practice_0022|multiplicative|E3|donante0/coste1/orden0|134|134|válido|
|practice_0022|multiplicative|E3|donante0/coste0/orden1|134|134|válido|
|practice_0022|multiplicative|E3|donante0/coste1/orden1|134|134|válido|
|practice_0022|multiplicative|E3|donante1/coste0/orden0|143|143|válido|
|practice_0022|multiplicative|E3|donante1/coste1/orden0|143|143|válido|
|practice_0022|multiplicative|E3|donante1/coste0/orden1|143|143|válido|
|practice_0022|multiplicative|E3|donante1/coste1/orden1|143|143|válido|
|practice_0022|multiplicative|E3|donante2/coste0/orden0|136|136|válido|
|practice_0022|multiplicative|E3|donante2/coste1/orden0|136|136|válido|
|practice_0022|multiplicative|E3|donante2/coste0/orden1|136|136|válido|
|practice_0022|multiplicative|E3|donante2/coste1/orden1|136|136|válido|
|practice_0022|multiplicative|E3|donante3/coste0/orden0|135|135|válido|
|practice_0022|multiplicative|E3|donante3/coste1/orden0|135|135|válido|
|practice_0022|multiplicative|E3|donante3/coste0/orden1|135|135|válido|
|practice_0022|multiplicative|E3|donante3/coste1/orden1|135|135|válido|
|practice_0022|multiplicative|E3|donante4/coste0/orden0|150|150|válido|
|practice_0022|multiplicative|E3|donante4/coste1/orden0|150|150|válido|
|practice_0022|multiplicative|E3|donante4/coste0/orden1|150|150|válido|
|practice_0022|multiplicative|E3|donante4/coste1/orden1|150|150|válido|
|practice_0022|multiplicative|E3|donante5/coste0/orden0|146|146|válido|
|practice_0022|multiplicative|E3|donante5/coste1/orden0|146|146|válido|
|practice_0022|multiplicative|E3|donante5/coste0/orden1|146|146|válido|
|practice_0022|multiplicative|E3|donante5/coste1/orden1|146|146|válido|
|practice_0022|multiplicative|E3|donante6/coste0/orden0|145|145|válido|
|practice_0022|multiplicative|E3|donante6/coste1/orden0|145|145|válido|
|practice_0022|multiplicative|E3|donante6/coste0/orden1|145|145|válido|
|practice_0022|multiplicative|E3|donante6/coste1/orden1|145|145|válido|
|practice_0022|multiplicative|E3|donante7/coste0/orden0|153|136|válido|
|practice_0022|multiplicative|E3|donante7/coste1/orden0|146|136|válido|
|practice_0022|multiplicative|E3|donante7/coste0/orden1|149|136|válido|
|practice_0022|multiplicative|E3|donante7/coste1/orden1|149|136|válido|
|practice_0022|multiplicative|E3|donante8/coste0/orden0|135|135|válido|
|practice_0022|multiplicative|E3|donante8/coste1/orden0|135|135|válido|
|practice_0022|multiplicative|E3|donante8/coste0/orden1|135|135|válido|
|practice_0022|multiplicative|E3|donante8/coste1/orden1|135|135|válido|
|practice_0022|multiplicative|E3|donante9/coste0/orden0|150|137|válido|
|practice_0022|multiplicative|E3|donante9/coste1/orden0|139|137|válido|
|practice_0022|multiplicative|E3|donante9/coste0/orden1|137|137|válido|
|practice_0022|multiplicative|E3|donante9/coste1/orden1|137|137|válido|
|practice_0023|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0023|mixed|Cruce|CP True/coste0/orden0|235|172|válido|
|practice_0023|mixed|Cruce|CP True/coste1/orden0|207|172|válido|
|practice_0023|mixed|Cruce|CP True/coste0/orden1|189|172|válido|
|practice_0023|mixed|Cruce|CP True/coste1/orden1|187|172|válido|
|practice_0023|mixed|Cruce|mejor independiente/coste0/orden0|244|172|válido|
|practice_0023|mixed|Cruce|mejor independiente/coste1/orden0|216|172|válido|
|practice_0023|mixed|Cruce|mejor independiente/coste0/orden1|183|172|válido|
|practice_0023|mixed|Cruce|mejor independiente/coste1/orden1|181|172|válido|
|practice_0023|mixed|E1|cola10/pol1/inicio0|189|65|válido|
|practice_0023|mixed|E1|cola10/pol1/inicio1|189|65|válido|
|practice_0023|mixed|E1|cola10/pol2/inicio0|188|64|válido|
|practice_0023|mixed|E1|cola10/pol2/inicio1|188|64|válido|
|practice_0023|mixed|E1|cola8/pol1/inicio0|185|49|válido|
|practice_0023|mixed|E1|cola8/pol1/inicio1|185|49|válido|
|practice_0023|mixed|E1|cola8/pol2/inicio0|185|49|válido|
|practice_0023|mixed|E1|cola8/pol2/inicio1|185|49|válido|
|practice_0023|mixed|E1|cola6/pol1/inicio0|188|42|válido|
|practice_0023|mixed|E1|cola6/pol1/inicio1|188|42|válido|
|practice_0023|mixed|E1|cola6/pol2/inicio0|188|42|válido|
|practice_0023|mixed|E1|cola6/pol2/inicio1|185|39|válido|
|practice_0023|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0023|mixed|E2|CP False/coste0/orden0|208|176|válido|
|practice_0023|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0023|mixed|E2|CP True/coste0/orden0|196|174|válido|
|practice_0023|mixed|E3|donante0/coste0/orden0|190|176|válido|
|practice_0023|mixed|E3|donante0/coste1/orden0|192|176|válido|
|practice_0023|mixed|E3|donante0/coste0/orden1|181|176|válido|
|practice_0023|mixed|E3|donante0/coste1/orden1|181|176|válido|
|practice_0023|mixed|E3|donante1/coste0/orden0|193|193|válido|
|practice_0023|mixed|E3|donante1/coste1/orden0|193|193|válido|
|practice_0023|mixed|E3|donante1/coste0/orden1|193|193|válido|
|practice_0023|mixed|E3|donante1/coste1/orden1|193|193|válido|
|practice_0023|mixed|E3|donante2/coste0/orden0|183|183|válido|
|practice_0023|mixed|E3|donante2/coste1/orden0|183|183|válido|
|practice_0023|mixed|E3|donante2/coste0/orden1|183|183|válido|
|practice_0023|mixed|E3|donante2/coste1/orden1|183|183|válido|
|practice_0023|mixed|E3|donante3/coste0/orden0|181|181|válido|
|practice_0023|mixed|E3|donante3/coste1/orden0|181|181|válido|
|practice_0023|mixed|E3|donante3/coste0/orden1|181|181|válido|
|practice_0023|mixed|E3|donante3/coste1/orden1|181|181|válido|
|practice_0023|mixed|E3|donante4/coste0/orden0|200|200|válido|
|practice_0023|mixed|E3|donante4/coste1/orden0|200|200|válido|
|practice_0023|mixed|E3|donante4/coste0/orden1|200|200|válido|
|practice_0023|mixed|E3|donante4/coste1/orden1|200|200|válido|
|practice_0023|mixed|E3|donante5/coste0/orden0|195|195|válido|
|practice_0023|mixed|E3|donante5/coste1/orden0|195|195|válido|
|practice_0023|mixed|E3|donante5/coste0/orden1|195|195|válido|
|practice_0023|mixed|E3|donante5/coste1/orden1|195|195|válido|
|practice_0023|mixed|E3|donante6/coste0/orden0|193|193|válido|
|practice_0023|mixed|E3|donante6/coste1/orden0|193|193|válido|
|practice_0023|mixed|E3|donante6/coste0/orden1|193|193|válido|
|practice_0023|mixed|E3|donante6/coste1/orden1|193|193|válido|
|practice_0023|mixed|E3|donante7/coste0/orden0|234|183|válido|
|practice_0023|mixed|E3|donante7/coste1/orden0|193|183|válido|
|practice_0023|mixed|E3|donante7/coste0/orden1|213|183|válido|
|practice_0023|mixed|E3|donante7/coste1/orden1|195|183|válido|
|practice_0023|mixed|E3|donante8/coste0/orden0|190|176|válido|
|practice_0023|mixed|E3|donante8/coste1/orden0|192|176|válido|
|practice_0023|mixed|E3|donante8/coste0/orden1|181|176|válido|
|practice_0023|mixed|E3|donante8/coste1/orden1|181|176|válido|
|practice_0023|mixed|E3|donante9/coste0/orden0|223|171|válido|
|practice_0023|mixed|E3|donante9/coste1/orden0|189|171|válido|
|practice_0023|mixed|E3|donante9/coste0/orden1|190|171|válido|
|practice_0023|mixed|E3|donante9/coste1/orden1|188|171|válido|
|practice_0024|shared|Cruce|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0024|shared|Cruce|CP True/coste0/orden0|29|29|válido|
|practice_0024|shared|Cruce|CP True/coste1/orden0|29|29|válido|
|practice_0024|shared|Cruce|CP True/coste0/orden1|29|29|válido|
|practice_0024|shared|Cruce|CP True/coste1/orden1|29|29|válido|
|practice_0024|shared|Cruce|mejor independiente/coste0/orden0|29|29|válido|
|practice_0024|shared|Cruce|mejor independiente/coste1/orden0|29|29|válido|
|practice_0024|shared|Cruce|mejor independiente/coste0/orden1|29|29|válido|
|practice_0024|shared|Cruce|mejor independiente/coste1/orden1|29|29|válido|
|practice_0024|shared|E1|cola10/pol1/inicio0|44|33|válido|
|practice_0024|shared|E1|cola10/pol1/inicio1|44|33|válido|
|practice_0024|shared|E1|cola10/pol2/inicio0|42|31|válido|
|practice_0024|shared|E1|cola10/pol2/inicio1|42|31|válido|
|practice_0024|shared|E1|cola8/pol1/inicio0|41|23|válido|
|practice_0024|shared|E1|cola8/pol1/inicio1|41|23|válido|
|practice_0024|shared|E1|cola8/pol2/inicio0|39|21|válido|
|practice_0024|shared|E1|cola8/pol2/inicio1|39|21|válido|
|practice_0024|shared|E1|cola6/pol1/inicio0|36|13|válido|
|practice_0024|shared|E1|cola6/pol1/inicio1|36|13|válido|
|practice_0024|shared|E1|cola6/pol2/inicio0|39|16|válido|
|practice_0024|shared|E1|cola6/pol2/inicio1|39|16|válido|
|practice_0024|shared|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0024|shared|E2|CP False/coste0/orden0|36|36|válido|
|practice_0024|shared|E2|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0024|shared|E2|CP True/coste0/orden0|29|29|válido|
|practice_0024|shared|E3|donante0/coste0/orden0|37|37|válido|
|practice_0024|shared|E3|donante0/coste1/orden0|37|37|válido|
|practice_0024|shared|E3|donante0/coste0/orden1|37|37|válido|
|practice_0024|shared|E3|donante0/coste1/orden1|37|37|válido|
|practice_0024|shared|E3|donante1/coste0/orden0|43|43|válido|
|practice_0024|shared|E3|donante1/coste1/orden0|43|43|válido|
|practice_0024|shared|E3|donante1/coste0/orden1|43|43|válido|
|practice_0024|shared|E3|donante1/coste1/orden1|43|43|válido|
|practice_0024|shared|E3|donante2/coste0/orden0|44|44|válido|
|practice_0024|shared|E3|donante2/coste1/orden0|44|44|válido|
|practice_0024|shared|E3|donante2/coste0/orden1|44|44|válido|
|practice_0024|shared|E3|donante2/coste1/orden1|44|44|válido|
|practice_0024|shared|E3|donante3/coste0/orden0|38|38|válido|
|practice_0024|shared|E3|donante3/coste1/orden0|38|38|válido|
|practice_0024|shared|E3|donante3/coste0/orden1|38|38|válido|
|practice_0024|shared|E3|donante3/coste1/orden1|38|38|válido|
|practice_0024|shared|E3|donante4/coste0/orden0|45|45|válido|
|practice_0024|shared|E3|donante4/coste1/orden0|45|45|válido|
|practice_0024|shared|E3|donante4/coste0/orden1|45|45|válido|
|practice_0024|shared|E3|donante4/coste1/orden1|45|45|válido|
|practice_0024|shared|E3|donante5/coste0/orden0|43|43|válido|
|practice_0024|shared|E3|donante5/coste1/orden0|43|43|válido|
|practice_0024|shared|E3|donante5/coste0/orden1|43|43|válido|
|practice_0024|shared|E3|donante5/coste1/orden1|43|43|válido|
|practice_0024|shared|E3|donante6/coste0/orden0|38|38|válido|
|practice_0024|shared|E3|donante6/coste1/orden0|38|38|válido|
|practice_0024|shared|E3|donante6/coste0/orden1|38|38|válido|
|practice_0024|shared|E3|donante6/coste1/orden1|38|38|válido|
|practice_0024|shared|E3|donante7/coste0/orden0|44|44|válido|
|practice_0024|shared|E3|donante7/coste1/orden0|44|44|válido|
|practice_0024|shared|E3|donante7/coste0/orden1|44|44|válido|
|practice_0024|shared|E3|donante7/coste1/orden1|44|44|válido|
|practice_0024|shared|E3|donante8/coste0/orden0|43|43|válido|
|practice_0024|shared|E3|donante8/coste1/orden0|43|43|válido|
|practice_0024|shared|E3|donante8/coste0/orden1|43|43|válido|
|practice_0024|shared|E3|donante8/coste1/orden1|43|43|válido|
|practice_0025|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0025|mixed|Cruce|CP True/coste0/orden0|172|172|válido|
|practice_0025|mixed|Cruce|CP True/coste1/orden0|172|172|válido|
|practice_0025|mixed|Cruce|CP True/coste0/orden1|172|172|válido|
|practice_0025|mixed|Cruce|CP True/coste1/orden1|172|172|válido|
|practice_0025|mixed|Cruce|mejor independiente/coste0/orden0|173|173|válido|
|practice_0025|mixed|Cruce|mejor independiente/coste1/orden0|173|173|válido|
|practice_0025|mixed|Cruce|mejor independiente/coste0/orden1|173|173|válido|
|practice_0025|mixed|Cruce|mejor independiente/coste1/orden1|173|173|válido|
|practice_0025|mixed|E1|cola10/pol1/inicio0|179|45|válido|
|practice_0025|mixed|E1|cola10/pol1/inicio1|179|45|válido|
|practice_0025|mixed|E1|cola10/pol2/inicio0|179|45|válido|
|practice_0025|mixed|E1|cola10/pol2/inicio1|179|45|válido|
|practice_0025|mixed|E1|cola8/pol1/inicio0|180|35|válido|
|practice_0025|mixed|E1|cola8/pol1/inicio1|180|35|válido|
|practice_0025|mixed|E1|cola8/pol2/inicio0|180|35|válido|
|practice_0025|mixed|E1|cola8/pol2/inicio1|180|35|válido|
|practice_0025|mixed|E1|cola6/pol1/inicio0|182|30|válido|
|practice_0025|mixed|E1|cola6/pol1/inicio1|182|30|válido|
|practice_0025|mixed|E1|cola6/pol2/inicio0|182|30|válido|
|practice_0025|mixed|E1|cola6/pol2/inicio1|182|30|válido|
|practice_0025|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0025|mixed|E2|CP False/coste0/orden0|173|173|válido|
|practice_0025|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0025|mixed|E2|CP True/coste0/orden0|179|179|válido|
|practice_0025|mixed|E3|donante0/coste0/orden0|175|175|válido|
|practice_0025|mixed|E3|donante0/coste1/orden0|175|175|válido|
|practice_0025|mixed|E3|donante0/coste0/orden1|175|175|válido|
|practice_0025|mixed|E3|donante0/coste1/orden1|175|175|válido|
|practice_0025|mixed|E3|donante1/coste0/orden0|182|182|válido|
|practice_0025|mixed|E3|donante1/coste1/orden0|182|182|válido|
|practice_0025|mixed|E3|donante1/coste0/orden1|182|182|válido|
|practice_0025|mixed|E3|donante1/coste1/orden1|182|182|válido|
|practice_0025|mixed|E3|donante2/coste0/orden0|181|181|válido|
|practice_0025|mixed|E3|donante2/coste1/orden0|181|181|válido|
|practice_0025|mixed|E3|donante2/coste0/orden1|181|181|válido|
|practice_0025|mixed|E3|donante2/coste1/orden1|181|181|válido|
|practice_0025|mixed|E3|donante3/coste0/orden0|196|196|válido|
|practice_0025|mixed|E3|donante3/coste1/orden0|196|196|válido|
|practice_0025|mixed|E3|donante3/coste0/orden1|196|196|válido|
|practice_0025|mixed|E3|donante3/coste1/orden1|196|196|válido|
|practice_0025|mixed|E3|donante4/coste0/orden0|188|188|válido|
|practice_0025|mixed|E3|donante4/coste1/orden0|188|188|válido|
|practice_0025|mixed|E3|donante4/coste0/orden1|188|188|válido|
|practice_0025|mixed|E3|donante4/coste1/orden1|188|188|válido|
|practice_0025|mixed|E3|donante5/coste0/orden0|191|191|válido|
|practice_0025|mixed|E3|donante5/coste1/orden0|191|191|válido|
|practice_0025|mixed|E3|donante5/coste0/orden1|191|191|válido|
|practice_0025|mixed|E3|donante5/coste1/orden1|191|191|válido|
|practice_0025|mixed|E3|donante6/coste0/orden0|223|177|válido|
|practice_0025|mixed|E3|donante6/coste1/orden0|227|177|válido|
|practice_0025|mixed|E3|donante6/coste0/orden1|218|177|válido|
|practice_0025|mixed|E3|donante6/coste1/orden1|213|177|válido|
|practice_0025|mixed|E3|donante7/coste0/orden0|201|173|válido|
|practice_0025|mixed|E3|donante7/coste1/orden0|193|173|válido|
|practice_0025|mixed|E3|donante7/coste0/orden1|207|173|válido|
|practice_0025|mixed|E3|donante7/coste1/orden1|199|173|válido|
|practice_0025|mixed|E3|donante8/coste0/orden0|185|173|válido|
|practice_0025|mixed|E3|donante8/coste1/orden0|186|173|válido|
|practice_0025|mixed|E3|donante8/coste0/orden1|200|173|válido|
|practice_0025|mixed|E3|donante8/coste1/orden1|200|173|válido|
|practice_0026|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0026|shared|Cruce|CP True/coste0/orden0|145|145|válido|
|practice_0026|shared|Cruce|CP True/coste1/orden0|145|145|válido|
|practice_0026|shared|Cruce|CP True/coste0/orden1|145|145|válido|
|practice_0026|shared|Cruce|CP True/coste1/orden1|145|145|válido|
|practice_0026|shared|Cruce|mejor independiente/coste0/orden0|146|146|válido|
|practice_0026|shared|Cruce|mejor independiente/coste1/orden0|146|146|válido|
|practice_0026|shared|Cruce|mejor independiente/coste0/orden1|146|146|válido|
|practice_0026|shared|Cruce|mejor independiente/coste1/orden1|146|146|válido|
|practice_0026|shared|E1|cola10/pol1/inicio0|150|51|válido|
|practice_0026|shared|E1|cola10/pol1/inicio1|150|51|válido|
|practice_0026|shared|E1|cola10/pol2/inicio0|150|51|válido|
|practice_0026|shared|E1|cola10/pol2/inicio1|150|51|válido|
|practice_0026|shared|E1|cola8/pol1/inicio0|151|44|válido|
|practice_0026|shared|E1|cola8/pol1/inicio1|151|44|válido|
|practice_0026|shared|E1|cola8/pol2/inicio0|149|42|válido|
|practice_0026|shared|E1|cola8/pol2/inicio1|149|42|válido|
|practice_0026|shared|E1|cola6/pol1/inicio0|153|37|válido|
|practice_0026|shared|E1|cola6/pol1/inicio1|153|37|válido|
|practice_0026|shared|E1|cola6/pol2/inicio0|153|37|válido|
|practice_0026|shared|E1|cola6/pol2/inicio1|153|37|válido|
|practice_0026|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0026|shared|E2|CP False/coste0/orden0|147|147|válido|
|practice_0026|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0026|shared|E2|CP True/coste0/orden0|146|146|válido|
|practice_0026|shared|E3|donante0/coste0/orden0|147|147|válido|
|practice_0026|shared|E3|donante0/coste1/orden0|147|147|válido|
|practice_0026|shared|E3|donante0/coste0/orden1|147|147|válido|
|practice_0026|shared|E3|donante0/coste1/orden1|147|147|válido|
|practice_0026|shared|E3|donante1/coste0/orden0|153|153|válido|
|practice_0026|shared|E3|donante1/coste1/orden0|153|153|válido|
|practice_0026|shared|E3|donante1/coste0/orden1|153|153|válido|
|practice_0026|shared|E3|donante1/coste1/orden1|153|153|válido|
|practice_0026|shared|E3|donante2/coste0/orden0|154|154|válido|
|practice_0026|shared|E3|donante2/coste1/orden0|154|154|válido|
|practice_0026|shared|E3|donante2/coste0/orden1|154|154|válido|
|practice_0026|shared|E3|donante2/coste1/orden1|154|154|válido|
|practice_0026|shared|E3|donante3/coste0/orden0|161|161|válido|
|practice_0026|shared|E3|donante3/coste1/orden0|161|161|válido|
|practice_0026|shared|E3|donante3/coste0/orden1|161|161|válido|
|practice_0026|shared|E3|donante3/coste1/orden1|161|161|válido|
|practice_0026|shared|E3|donante4/coste0/orden0|156|156|válido|
|practice_0026|shared|E3|donante4/coste1/orden0|156|156|válido|
|practice_0026|shared|E3|donante4/coste0/orden1|156|156|válido|
|practice_0026|shared|E3|donante4/coste1/orden1|156|156|válido|
|practice_0026|shared|E3|donante5/coste0/orden0|157|157|válido|
|practice_0026|shared|E3|donante5/coste1/orden0|157|157|válido|
|practice_0026|shared|E3|donante5/coste0/orden1|157|157|válido|
|practice_0026|shared|E3|donante5/coste1/orden1|157|157|válido|
|practice_0026|shared|E3|donante6/coste0/orden0|175|148|válido|
|practice_0026|shared|E3|donante6/coste1/orden0|168|148|válido|
|practice_0026|shared|E3|donante6/coste0/orden1|164|148|válido|
|practice_0026|shared|E3|donante6/coste1/orden1|149|148|válido|
|practice_0026|shared|E3|donante7/coste0/orden0|164|146|válido|
|practice_0026|shared|E3|donante7/coste1/orden0|158|146|válido|
|practice_0026|shared|E3|donante7/coste0/orden1|165|146|válido|
|practice_0026|shared|E3|donante7/coste1/orden1|157|146|válido|
|practice_0026|shared|E3|donante8/coste0/orden0|165|145|válido|
|practice_0026|shared|E3|donante8/coste1/orden0|161|145|válido|
|practice_0026|shared|E3|donante8/coste0/orden1|168|145|válido|
|practice_0026|shared|E3|donante8/coste1/orden1|163|145|válido|
|practice_0027|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0027|shared|Cruce|CP True/coste0/orden0|183|181|válido|
|practice_0027|shared|Cruce|CP True/coste1/orden0|185|181|válido|
|practice_0027|shared|Cruce|CP True/coste0/orden1|181|181|válido|
|practice_0027|shared|Cruce|CP True/coste1/orden1|181|181|válido|
|practice_0027|shared|Cruce|mejor independiente/coste0/orden0|190|181|válido|
|practice_0027|shared|Cruce|mejor independiente/coste1/orden0|190|181|válido|
|practice_0027|shared|Cruce|mejor independiente/coste0/orden1|181|181|válido|
|practice_0027|shared|Cruce|mejor independiente/coste1/orden1|181|181|válido|
|practice_0027|shared|E1|cola10/pol1/inicio0|189|69|válido|
|practice_0027|shared|E1|cola10/pol1/inicio1|189|69|válido|
|practice_0027|shared|E1|cola10/pol2/inicio0|185|65|válido|
|practice_0027|shared|E1|cola10/pol2/inicio1|185|65|válido|
|practice_0027|shared|E1|cola8/pol1/inicio0|189|57|válido|
|practice_0027|shared|E1|cola8/pol1/inicio1|189|57|válido|
|practice_0027|shared|E1|cola8/pol2/inicio0|189|57|válido|
|practice_0027|shared|E1|cola8/pol2/inicio1|189|57|válido|
|practice_0027|shared|E1|cola6/pol1/inicio0|187|43|válido|
|practice_0027|shared|E1|cola6/pol1/inicio1|187|43|válido|
|practice_0027|shared|E1|cola6/pol2/inicio0|187|43|válido|
|practice_0027|shared|E1|cola6/pol2/inicio1|187|43|válido|
|practice_0027|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0027|shared|E2|CP False/coste0/orden0|221|189|válido|
|practice_0027|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0027|shared|E2|CP True/coste0/orden0|184|182|válido|
|practice_0027|shared|E3|donante0/coste0/orden0|184|182|válido|
|practice_0027|shared|E3|donante0/coste1/orden0|184|182|válido|
|practice_0027|shared|E3|donante0/coste0/orden1|182|182|válido|
|practice_0027|shared|E3|donante0/coste1/orden1|182|182|válido|
|practice_0027|shared|E3|donante1/coste0/orden0|205|205|válido|
|practice_0027|shared|E3|donante1/coste1/orden0|205|205|válido|
|practice_0027|shared|E3|donante1/coste0/orden1|205|205|válido|
|practice_0027|shared|E3|donante1/coste1/orden1|205|205|válido|
|practice_0027|shared|E3|donante2/coste0/orden0|197|197|válido|
|practice_0027|shared|E3|donante2/coste1/orden0|197|197|válido|
|practice_0027|shared|E3|donante2/coste0/orden1|197|197|válido|
|practice_0027|shared|E3|donante2/coste1/orden1|197|197|válido|
|practice_0027|shared|E3|donante3/coste0/orden0|190|190|válido|
|practice_0027|shared|E3|donante3/coste1/orden0|190|190|válido|
|practice_0027|shared|E3|donante3/coste0/orden1|190|190|válido|
|practice_0027|shared|E3|donante3/coste1/orden1|190|190|válido|
|practice_0027|shared|E3|donante4/coste0/orden0|220|220|válido|
|practice_0027|shared|E3|donante4/coste1/orden0|220|220|válido|
|practice_0027|shared|E3|donante4/coste0/orden1|220|220|válido|
|practice_0027|shared|E3|donante4/coste1/orden1|220|220|válido|
|practice_0027|shared|E3|donante5/coste0/orden0|216|216|válido|
|practice_0027|shared|E3|donante5/coste1/orden0|216|216|válido|
|practice_0027|shared|E3|donante5/coste0/orden1|216|216|válido|
|practice_0027|shared|E3|donante5/coste1/orden1|216|216|válido|
|practice_0027|shared|E3|donante6/coste0/orden0|204|204|válido|
|practice_0027|shared|E3|donante6/coste1/orden0|204|204|válido|
|practice_0027|shared|E3|donante6/coste0/orden1|204|204|válido|
|practice_0027|shared|E3|donante6/coste1/orden1|204|204|válido|
|practice_0027|shared|E3|donante7/coste0/orden0|187|184|válido|
|practice_0027|shared|E3|donante7/coste1/orden0|187|184|válido|
|practice_0027|shared|E3|donante7/coste0/orden1|187|184|válido|
|practice_0027|shared|E3|donante7/coste1/orden1|185|184|válido|
|practice_0027|shared|E3|donante8/coste0/orden0|199|186|válido|
|practice_0027|shared|E3|donante8/coste1/orden0|200|186|válido|
|practice_0027|shared|E3|donante8/coste0/orden1|195|186|válido|
|practice_0027|shared|E3|donante8/coste1/orden1|197|186|válido|
|practice_0027|shared|E3|donante9/coste0/orden0|190|181|válido|
|practice_0027|shared|E3|donante9/coste1/orden0|187|181|válido|
|practice_0027|shared|E3|donante9/coste0/orden1|181|181|válido|
|practice_0027|shared|E3|donante9/coste1/orden1|181|181|válido|
|practice_0028|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0028|mixed|Cruce|CP True/coste0/orden0|225|213|válido|
|practice_0028|mixed|Cruce|CP True/coste1/orden0|218|213|válido|
|practice_0028|mixed|Cruce|CP True/coste0/orden1|217|213|válido|
|practice_0028|mixed|Cruce|CP True/coste1/orden1|215|213|válido|
|practice_0028|mixed|Cruce|mejor independiente/coste0/orden0|249|214|válido|
|practice_0028|mixed|Cruce|mejor independiente/coste1/orden0|224|214|válido|
|practice_0028|mixed|Cruce|mejor independiente/coste0/orden1|218|214|válido|
|practice_0028|mixed|Cruce|mejor independiente/coste1/orden1|259|214|válido|
|practice_0028|mixed|E1|cola10/pol1/inicio0|227|80|válido|
|practice_0028|mixed|E1|cola10/pol1/inicio1|227|80|válido|
|practice_0028|mixed|E1|cola10/pol2/inicio0|222|75|válido|
|practice_0028|mixed|E1|cola10/pol2/inicio1|222|75|válido|
|practice_0028|mixed|E1|cola8/pol1/inicio0|226|65|válido|
|practice_0028|mixed|E1|cola8/pol1/inicio1|226|65|válido|
|practice_0028|mixed|E1|cola8/pol2/inicio0|226|65|válido|
|practice_0028|mixed|E1|cola8/pol2/inicio1|226|65|válido|
|practice_0028|mixed|E1|cola6/pol1/inicio0|224|49|válido|
|practice_0028|mixed|E1|cola6/pol1/inicio1|227|52|válido|
|practice_0028|mixed|E1|cola6/pol2/inicio0|223|48|válido|
|practice_0028|mixed|E1|cola6/pol2/inicio1|223|48|válido|
|practice_0028|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0028|mixed|E2|CP False/coste0/orden0|249|214|válido|
|practice_0028|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0028|mixed|E2|CP True/coste0/orden0|225|213|válido|
|practice_0028|mixed|E3|donante0/coste0/orden0|221|214|válido|
|practice_0028|mixed|E3|donante0/coste1/orden0|216|214|válido|
|practice_0028|mixed|E3|donante0/coste0/orden1|218|214|válido|
|practice_0028|mixed|E3|donante0/coste1/orden1|259|214|válido|
|practice_0028|mixed|E3|donante1/coste0/orden0|240|240|válido|
|practice_0028|mixed|E3|donante1/coste1/orden0|240|240|válido|
|practice_0028|mixed|E3|donante1/coste0/orden1|240|240|válido|
|practice_0028|mixed|E3|donante1/coste1/orden1|240|240|válido|
|practice_0028|mixed|E3|donante2/coste0/orden0|233|233|válido|
|practice_0028|mixed|E3|donante2/coste1/orden0|233|233|válido|
|practice_0028|mixed|E3|donante2/coste0/orden1|233|233|válido|
|practice_0028|mixed|E3|donante2/coste1/orden1|233|233|válido|
|practice_0028|mixed|E3|donante3/coste0/orden0|228|228|válido|
|practice_0028|mixed|E3|donante3/coste1/orden0|228|228|válido|
|practice_0028|mixed|E3|donante3/coste0/orden1|228|228|válido|
|practice_0028|mixed|E3|donante3/coste1/orden1|228|228|válido|
|practice_0028|mixed|E3|donante4/coste0/orden0|245|245|válido|
|practice_0028|mixed|E3|donante4/coste1/orden0|245|245|válido|
|practice_0028|mixed|E3|donante4/coste0/orden1|245|245|válido|
|practice_0028|mixed|E3|donante4/coste1/orden1|245|245|válido|
|practice_0028|mixed|E3|donante5/coste0/orden0|234|234|válido|
|practice_0028|mixed|E3|donante5/coste1/orden0|234|234|válido|
|practice_0028|mixed|E3|donante5/coste0/orden1|234|234|válido|
|practice_0028|mixed|E3|donante5/coste1/orden1|234|234|válido|
|practice_0028|mixed|E3|donante6/coste0/orden0|244|244|válido|
|practice_0028|mixed|E3|donante6/coste1/orden0|244|244|válido|
|practice_0028|mixed|E3|donante6/coste0/orden1|244|244|válido|
|practice_0028|mixed|E3|donante6/coste1/orden1|244|244|válido|
|practice_0028|mixed|E3|donante7/coste0/orden0|262|218|válido|
|practice_0028|mixed|E3|donante7/coste1/orden0|239|218|válido|
|practice_0028|mixed|E3|donante7/coste0/orden1|267|218|válido|
|practice_0028|mixed|E3|donante7/coste1/orden1|235|218|válido|
|practice_0028|mixed|E3|donante8/coste0/orden0|232|214|válido|
|practice_0028|mixed|E3|donante8/coste1/orden0|218|214|válido|
|practice_0028|mixed|E3|donante8/coste0/orden1|222|214|válido|
|practice_0028|mixed|E3|donante8/coste1/orden1|261|214|válido|
|practice_0028|mixed|E3|donante9/coste0/orden0|238|212|válido|
|practice_0028|mixed|E3|donante9/coste1/orden0|229|212|válido|
|practice_0028|mixed|E3|donante9/coste0/orden1|254|212|válido|
|practice_0028|mixed|E3|donante9/coste1/orden1|236|212|válido|
|practice_0029|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0029|mixed|Cruce|CP True/coste0/orden0|199|183|válido|
|practice_0029|mixed|Cruce|CP True/coste1/orden0|198|183|válido|
|practice_0029|mixed|Cruce|CP True/coste0/orden1|191|183|válido|
|practice_0029|mixed|Cruce|CP True/coste1/orden1|192|183|válido|
|practice_0029|mixed|Cruce|mejor independiente/coste0/orden0|194|185|válido|
|practice_0029|mixed|Cruce|mejor independiente/coste1/orden0|194|185|válido|
|practice_0029|mixed|Cruce|mejor independiente/coste0/orden1|195|185|válido|
|practice_0029|mixed|Cruce|mejor independiente/coste1/orden1|191|185|válido|
|practice_0029|mixed|E1|cola10/pol1/inicio0|195|69|válido|
|practice_0029|mixed|E1|cola10/pol1/inicio1|195|69|válido|
|practice_0029|mixed|E1|cola10/pol2/inicio0|196|70|válido|
|practice_0029|mixed|E1|cola10/pol2/inicio1|196|70|válido|
|practice_0029|mixed|E1|cola8/pol1/inicio0|195|56|válido|
|practice_0029|mixed|E1|cola8/pol1/inicio1|199|60|válido|
|practice_0029|mixed|E1|cola8/pol2/inicio0|199|60|válido|
|practice_0029|mixed|E1|cola8/pol2/inicio1|199|60|válido|
|practice_0029|mixed|E1|cola6/pol1/inicio0|201|47|válido|
|practice_0029|mixed|E1|cola6/pol1/inicio1|197|43|válido|
|practice_0029|mixed|E1|cola6/pol2/inicio0|201|47|válido|
|practice_0029|mixed|E1|cola6/pol2/inicio1|197|43|válido|
|practice_0029|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0029|mixed|E2|CP False/coste0/orden0|195|186|válido|
|practice_0029|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0029|mixed|E2|CP True/coste0/orden0|200|191|válido|
|practice_0029|mixed|E3|donante0/coste0/orden0|192|191|válido|
|practice_0029|mixed|E3|donante0/coste1/orden0|192|191|válido|
|practice_0029|mixed|E3|donante0/coste0/orden1|198|191|válido|
|practice_0029|mixed|E3|donante0/coste1/orden1|198|191|válido|
|practice_0029|mixed|E3|donante1/coste0/orden0|214|214|válido|
|practice_0029|mixed|E3|donante1/coste1/orden0|214|214|válido|
|practice_0029|mixed|E3|donante1/coste0/orden1|214|214|válido|
|practice_0029|mixed|E3|donante1/coste1/orden1|214|214|válido|
|practice_0029|mixed|E3|donante2/coste0/orden0|205|205|válido|
|practice_0029|mixed|E3|donante2/coste1/orden0|205|205|válido|
|practice_0029|mixed|E3|donante2/coste0/orden1|205|205|válido|
|practice_0029|mixed|E3|donante2/coste1/orden1|205|205|válido|
|practice_0029|mixed|E3|donante3/coste0/orden0|199|199|válido|
|practice_0029|mixed|E3|donante3/coste1/orden0|199|199|válido|
|practice_0029|mixed|E3|donante3/coste0/orden1|199|199|válido|
|practice_0029|mixed|E3|donante3/coste1/orden1|199|199|válido|
|practice_0029|mixed|E3|donante4/coste0/orden0|217|217|válido|
|practice_0029|mixed|E3|donante4/coste1/orden0|217|217|válido|
|practice_0029|mixed|E3|donante4/coste0/orden1|217|217|válido|
|practice_0029|mixed|E3|donante4/coste1/orden1|217|217|válido|
|practice_0029|mixed|E3|donante5/coste0/orden0|210|210|válido|
|practice_0029|mixed|E3|donante5/coste1/orden0|210|210|válido|
|practice_0029|mixed|E3|donante5/coste0/orden1|210|210|válido|
|practice_0029|mixed|E3|donante5/coste1/orden1|210|210|válido|
|practice_0029|mixed|E3|donante6/coste0/orden0|210|210|válido|
|practice_0029|mixed|E3|donante6/coste1/orden0|210|210|válido|
|practice_0029|mixed|E3|donante6/coste0/orden1|210|210|válido|
|practice_0029|mixed|E3|donante6/coste1/orden1|210|210|válido|
|practice_0029|mixed|E3|donante7/coste0/orden0|204|193|válido|
|practice_0029|mixed|E3|donante7/coste1/orden0|197|193|válido|
|practice_0029|mixed|E3|donante7/coste0/orden1|206|193|válido|
|practice_0029|mixed|E3|donante7/coste1/orden1|195|193|válido|
|practice_0029|mixed|E3|donante8/coste0/orden0|193|192|válido|
|practice_0029|mixed|E3|donante8/coste1/orden0|193|192|válido|
|practice_0029|mixed|E3|donante8/coste0/orden1|199|192|válido|
|practice_0029|mixed|E3|donante8/coste1/orden1|199|192|válido|
|practice_0029|mixed|E3|donante9/coste0/orden0|195|187|válido|
|practice_0029|mixed|E3|donante9/coste1/orden0|195|187|válido|
|practice_0029|mixed|E3|donante9/coste0/orden1|199|187|válido|
|practice_0029|mixed|E3|donante9/coste1/orden1|190|187|válido|
|practice_0030|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0030|multiplicative|Cruce|CP True/coste0/orden0|131|131|válido|
|practice_0030|multiplicative|Cruce|CP True/coste1/orden0|131|131|válido|
|practice_0030|multiplicative|Cruce|CP True/coste0/orden1|131|131|válido|
|practice_0030|multiplicative|Cruce|CP True/coste1/orden1|131|131|válido|
|practice_0030|multiplicative|Cruce|mejor independiente/coste0/orden0|132|132|válido|
|practice_0030|multiplicative|Cruce|mejor independiente/coste1/orden0|132|132|válido|
|practice_0030|multiplicative|Cruce|mejor independiente/coste0/orden1|132|132|válido|
|practice_0030|multiplicative|Cruce|mejor independiente/coste1/orden1|132|132|válido|
|practice_0030|multiplicative|E1|cola10/pol1/inicio0|139|53|válido|
|practice_0030|multiplicative|E1|cola10/pol1/inicio1|139|53|válido|
|practice_0030|multiplicative|E1|cola10/pol2/inicio0|136|50|válido|
|practice_0030|multiplicative|E1|cola10/pol2/inicio1|136|50|válido|
|practice_0030|multiplicative|E1|cola8/pol1/inicio0|136|43|válido|
|practice_0030|multiplicative|E1|cola8/pol1/inicio1|136|43|válido|
|practice_0030|multiplicative|E1|cola8/pol2/inicio0|136|43|válido|
|practice_0030|multiplicative|E1|cola8/pol2/inicio1|136|43|válido|
|practice_0030|multiplicative|E1|cola6/pol1/inicio0|138|36|válido|
|practice_0030|multiplicative|E1|cola6/pol1/inicio1|138|36|válido|
|practice_0030|multiplicative|E1|cola6/pol2/inicio0|138|36|válido|
|practice_0030|multiplicative|E1|cola6/pol2/inicio1|138|36|válido|
|practice_0030|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0030|multiplicative|E2|CP False/coste0/orden0|134|134|válido|
|practice_0030|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0030|multiplicative|E2|CP True/coste0/orden0|132|132|válido|
|practice_0030|multiplicative|E3|donante0/coste0/orden0|134|134|válido|
|practice_0030|multiplicative|E3|donante0/coste1/orden0|134|134|válido|
|practice_0030|multiplicative|E3|donante0/coste0/orden1|134|134|válido|
|practice_0030|multiplicative|E3|donante0/coste1/orden1|134|134|válido|
|practice_0030|multiplicative|E3|donante1/coste0/orden0|142|142|válido|
|practice_0030|multiplicative|E3|donante1/coste1/orden0|142|142|válido|
|practice_0030|multiplicative|E3|donante1/coste0/orden1|142|142|válido|
|practice_0030|multiplicative|E3|donante1/coste1/orden1|142|142|válido|
|practice_0030|multiplicative|E3|donante2/coste0/orden0|147|147|válido|
|practice_0030|multiplicative|E3|donante2/coste1/orden0|147|147|válido|
|practice_0030|multiplicative|E3|donante2/coste0/orden1|147|147|válido|
|practice_0030|multiplicative|E3|donante2/coste1/orden1|147|147|válido|
|practice_0030|multiplicative|E3|donante3/coste0/orden0|145|145|válido|
|practice_0030|multiplicative|E3|donante3/coste1/orden0|145|145|válido|
|practice_0030|multiplicative|E3|donante3/coste0/orden1|145|145|válido|
|practice_0030|multiplicative|E3|donante3/coste1/orden1|145|145|válido|
|practice_0030|multiplicative|E3|donante4/coste0/orden0|146|146|válido|
|practice_0030|multiplicative|E3|donante4/coste1/orden0|146|146|válido|
|practice_0030|multiplicative|E3|donante4/coste0/orden1|146|146|válido|
|practice_0030|multiplicative|E3|donante4/coste1/orden1|146|146|válido|
|practice_0030|multiplicative|E3|donante5/coste0/orden0|142|138|válido|
|practice_0030|multiplicative|E3|donante5/coste1/orden0|142|138|válido|
|practice_0030|multiplicative|E3|donante5/coste0/orden1|138|138|válido|
|practice_0030|multiplicative|E3|donante5/coste1/orden1|138|138|válido|
|practice_0030|multiplicative|E3|donante6/coste0/orden0|143|132|válido|
|practice_0030|multiplicative|E3|donante6/coste1/orden0|141|132|válido|
|practice_0030|multiplicative|E3|donante6/coste0/orden1|135|132|válido|
|practice_0030|multiplicative|E3|donante6/coste1/orden1|135|132|válido|
|practice_0031|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0031|mixed|Cruce|CP True/coste0/orden0|213|191|válido|
|practice_0031|mixed|Cruce|CP True/coste1/orden0|204|191|válido|
|practice_0031|mixed|Cruce|CP True/coste0/orden1|216|191|válido|
|practice_0031|mixed|Cruce|CP True/coste1/orden1|206|191|válido|
|practice_0031|mixed|Cruce|mejor independiente/coste0/orden0|225|194|válido|
|practice_0031|mixed|Cruce|mejor independiente/coste1/orden0|208|194|válido|
|practice_0031|mixed|Cruce|mejor independiente/coste0/orden1|208|194|válido|
|practice_0031|mixed|Cruce|mejor independiente/coste1/orden1|197|194|válido|
|practice_0031|mixed|E1|cola10/pol1/inicio0|205|72|válido|
|practice_0031|mixed|E1|cola10/pol1/inicio1|205|72|válido|
|practice_0031|mixed|E1|cola10/pol2/inicio0|203|70|válido|
|practice_0031|mixed|E1|cola10/pol2/inicio1|203|70|válido|
|practice_0031|mixed|E1|cola8/pol1/inicio0|208|60|válido|
|practice_0031|mixed|E1|cola8/pol1/inicio1|208|60|válido|
|practice_0031|mixed|E1|cola8/pol2/inicio0|206|58|válido|
|practice_0031|mixed|E1|cola8/pol2/inicio1|206|58|válido|
|practice_0031|mixed|E1|cola6/pol1/inicio0|207|45|válido|
|practice_0031|mixed|E1|cola6/pol1/inicio1|207|45|válido|
|practice_0031|mixed|E1|cola6/pol2/inicio0|207|45|válido|
|practice_0031|mixed|E1|cola6/pol2/inicio1|207|45|válido|
|practice_0031|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0031|mixed|E2|CP False/coste0/orden0|232|213|válido|
|practice_0031|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0031|mixed|E2|CP True/coste0/orden0|213|191|válido|
|practice_0031|mixed|E3|donante0/coste0/orden0|198|194|válido|
|practice_0031|mixed|E3|donante0/coste1/orden0|197|194|válido|
|practice_0031|mixed|E3|donante0/coste0/orden1|208|194|válido|
|practice_0031|mixed|E3|donante0/coste1/orden1|197|194|válido|
|practice_0031|mixed|E3|donante1/coste0/orden0|226|226|válido|
|practice_0031|mixed|E3|donante1/coste1/orden0|226|226|válido|
|practice_0031|mixed|E3|donante1/coste0/orden1|226|226|válido|
|practice_0031|mixed|E3|donante1/coste1/orden1|226|226|válido|
|practice_0031|mixed|E3|donante2/coste0/orden0|213|213|válido|
|practice_0031|mixed|E3|donante2/coste1/orden0|213|213|válido|
|practice_0031|mixed|E3|donante2/coste0/orden1|213|213|válido|
|practice_0031|mixed|E3|donante2/coste1/orden1|213|213|válido|
|practice_0031|mixed|E3|donante3/coste0/orden0|247|214|válido|
|practice_0031|mixed|E3|donante3/coste1/orden0|225|214|válido|
|practice_0031|mixed|E3|donante3/coste0/orden1|214|214|válido|
|practice_0031|mixed|E3|donante3/coste1/orden1|214|214|válido|
|practice_0031|mixed|E3|donante4/coste0/orden0|230|230|válido|
|practice_0031|mixed|E3|donante4/coste1/orden0|230|230|válido|
|practice_0031|mixed|E3|donante4/coste0/orden1|230|230|válido|
|practice_0031|mixed|E3|donante4/coste1/orden1|230|230|válido|
|practice_0031|mixed|E3|donante5/coste0/orden0|228|228|válido|
|practice_0031|mixed|E3|donante5/coste1/orden0|228|228|válido|
|practice_0031|mixed|E3|donante5/coste0/orden1|228|228|válido|
|practice_0031|mixed|E3|donante5/coste1/orden1|228|228|válido|
|practice_0031|mixed|E3|donante6/coste0/orden0|222|222|válido|
|practice_0031|mixed|E3|donante6/coste1/orden0|222|222|válido|
|practice_0031|mixed|E3|donante6/coste0/orden1|222|222|válido|
|practice_0031|mixed|E3|donante6/coste1/orden1|222|222|válido|
|practice_0031|mixed|E3|donante7/coste0/orden0|230|208|válido|
|practice_0031|mixed|E3|donante7/coste1/orden0|221|208|válido|
|practice_0031|mixed|E3|donante7/coste0/orden1|238|208|válido|
|practice_0031|mixed|E3|donante7/coste1/orden1|236|208|válido|
|practice_0031|mixed|E3|donante8/coste0/orden0|219|199|válido|
|practice_0031|mixed|E3|donante8/coste1/orden0|202|199|válido|
|practice_0031|mixed|E3|donante8/coste0/orden1|206|199|válido|
|practice_0031|mixed|E3|donante8/coste1/orden1|203|199|válido|
|practice_0032|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0032|multiplicative|Cruce|CP True/coste0/orden0|129|129|válido|
|practice_0032|multiplicative|Cruce|CP True/coste1/orden0|129|129|válido|
|practice_0032|multiplicative|Cruce|CP True/coste0/orden1|129|129|válido|
|practice_0032|multiplicative|Cruce|CP True/coste1/orden1|129|129|válido|
|practice_0032|multiplicative|Cruce|mejor independiente/coste0/orden0|138|132|válido|
|practice_0032|multiplicative|Cruce|mejor independiente/coste1/orden0|138|132|válido|
|practice_0032|multiplicative|Cruce|mejor independiente/coste0/orden1|138|132|válido|
|practice_0032|multiplicative|Cruce|mejor independiente/coste1/orden1|133|132|válido|
|practice_0032|multiplicative|E1|cola10/pol1/inicio0|136|46|válido|
|practice_0032|multiplicative|E1|cola10/pol1/inicio1|136|46|válido|
|practice_0032|multiplicative|E1|cola10/pol2/inicio0|136|46|válido|
|practice_0032|multiplicative|E1|cola10/pol2/inicio1|136|46|válido|
|practice_0032|multiplicative|E1|cola8/pol1/inicio0|138|40|válido|
|practice_0032|multiplicative|E1|cola8/pol1/inicio1|138|40|válido|
|practice_0032|multiplicative|E1|cola8/pol2/inicio0|138|40|válido|
|practice_0032|multiplicative|E1|cola8/pol2/inicio1|138|40|válido|
|practice_0032|multiplicative|E1|cola6/pol1/inicio0|138|32|válido|
|practice_0032|multiplicative|E1|cola6/pol1/inicio1|138|32|válido|
|practice_0032|multiplicative|E1|cola6/pol2/inicio0|138|32|válido|
|practice_0032|multiplicative|E1|cola6/pol2/inicio1|138|32|válido|
|practice_0032|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0032|multiplicative|E2|CP False/coste0/orden0|142|137|válido|
|practice_0032|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0032|multiplicative|E2|CP True/coste0/orden0|136|128|válido|
|practice_0032|multiplicative|E3|donante0/coste0/orden0|136|132|válido|
|practice_0032|multiplicative|E3|donante0/coste1/orden0|136|132|válido|
|practice_0032|multiplicative|E3|donante0/coste0/orden1|137|132|válido|
|practice_0032|multiplicative|E3|donante0/coste1/orden1|137|132|válido|
|practice_0032|multiplicative|E3|donante1/coste0/orden0|140|140|válido|
|practice_0032|multiplicative|E3|donante1/coste1/orden0|140|140|válido|
|practice_0032|multiplicative|E3|donante1/coste0/orden1|140|140|válido|
|practice_0032|multiplicative|E3|donante1/coste1/orden1|140|140|válido|
|practice_0032|multiplicative|E3|donante2/coste0/orden0|140|140|válido|
|practice_0032|multiplicative|E3|donante2/coste1/orden0|140|140|válido|
|practice_0032|multiplicative|E3|donante2/coste0/orden1|140|140|válido|
|practice_0032|multiplicative|E3|donante2/coste1/orden1|140|140|válido|
|practice_0032|multiplicative|E3|donante3/coste0/orden0|138|138|válido|
|practice_0032|multiplicative|E3|donante3/coste1/orden0|138|138|válido|
|practice_0032|multiplicative|E3|donante3/coste0/orden1|138|138|válido|
|practice_0032|multiplicative|E3|donante3/coste1/orden1|138|138|válido|
|practice_0032|multiplicative|E3|donante4/coste0/orden0|148|148|válido|
|practice_0032|multiplicative|E3|donante4/coste1/orden0|148|148|válido|
|practice_0032|multiplicative|E3|donante4/coste0/orden1|148|148|válido|
|practice_0032|multiplicative|E3|donante4/coste1/orden1|148|148|válido|
|practice_0032|multiplicative|E3|donante5/coste0/orden0|150|150|válido|
|practice_0032|multiplicative|E3|donante5/coste1/orden0|150|150|válido|
|practice_0032|multiplicative|E3|donante5/coste0/orden1|150|150|válido|
|practice_0032|multiplicative|E3|donante5/coste1/orden1|150|150|válido|
|practice_0032|multiplicative|E3|donante6/coste0/orden0|148|148|válido|
|practice_0032|multiplicative|E3|donante6/coste1/orden0|148|148|válido|
|practice_0032|multiplicative|E3|donante6/coste0/orden1|148|148|válido|
|practice_0032|multiplicative|E3|donante6/coste1/orden1|148|148|válido|
|practice_0032|multiplicative|E3|donante7/coste0/orden0|150|135|válido|
|practice_0032|multiplicative|E3|donante7/coste1/orden0|154|135|válido|
|practice_0032|multiplicative|E3|donante7/coste0/orden1|166|135|válido|
|practice_0032|multiplicative|E3|donante7/coste1/orden1|150|135|válido|
|practice_0032|multiplicative|E3|donante8/coste0/orden0|147|136|válido|
|practice_0032|multiplicative|E3|donante8/coste1/orden0|143|136|válido|
|practice_0032|multiplicative|E3|donante8/coste0/orden1|148|136|válido|
|practice_0032|multiplicative|E3|donante8/coste1/orden1|144|136|válido|
|practice_0033|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0033|shared|Cruce|CP True/coste0/orden0|31|31|válido|
|practice_0033|shared|Cruce|CP True/coste1/orden0|31|31|válido|
|practice_0033|shared|Cruce|CP True/coste0/orden1|31|31|válido|
|practice_0033|shared|Cruce|CP True/coste1/orden1|31|31|válido|
|practice_0033|shared|Cruce|mejor independiente/coste0/orden0|31|31|válido|
|practice_0033|shared|Cruce|mejor independiente/coste1/orden0|31|31|válido|
|practice_0033|shared|Cruce|mejor independiente/coste0/orden1|31|31|válido|
|practice_0033|shared|Cruce|mejor independiente/coste1/orden1|31|31|válido|
|practice_0033|shared|E1|cola10/pol1/inicio0|38|26|válido|
|practice_0033|shared|E1|cola10/pol1/inicio1|38|26|válido|
|practice_0033|shared|E1|cola10/pol2/inicio0|38|26|válido|
|practice_0033|shared|E1|cola10/pol2/inicio1|38|26|válido|
|practice_0033|shared|E1|cola8/pol1/inicio0|38|22|válido|
|practice_0033|shared|E1|cola8/pol1/inicio1|38|22|válido|
|practice_0033|shared|E1|cola8/pol2/inicio0|37|21|válido|
|practice_0033|shared|E1|cola8/pol2/inicio1|37|21|válido|
|practice_0033|shared|E1|cola6/pol1/inicio0|36|15|válido|
|practice_0033|shared|E1|cola6/pol1/inicio1|36|15|válido|
|practice_0033|shared|E1|cola6/pol2/inicio0|36|15|válido|
|practice_0033|shared|E1|cola6/pol2/inicio1|36|15|válido|
|practice_0033|shared|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0033|shared|E2|CP False/coste0/orden0|34|34|válido|
|practice_0033|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0033|shared|E2|CP True/coste0/orden0|31|31|válido|
|practice_0033|shared|E3|donante0/coste0/orden0|35|35|válido|
|practice_0033|shared|E3|donante0/coste1/orden0|35|35|válido|
|practice_0033|shared|E3|donante0/coste0/orden1|35|35|válido|
|practice_0033|shared|E3|donante0/coste1/orden1|35|35|válido|
|practice_0033|shared|E3|donante1/coste0/orden0|42|42|válido|
|practice_0033|shared|E3|donante1/coste1/orden0|42|42|válido|
|practice_0033|shared|E3|donante1/coste0/orden1|42|42|válido|
|practice_0033|shared|E3|donante1/coste1/orden1|42|42|válido|
|practice_0033|shared|E3|donante2/coste0/orden0|41|41|válido|
|practice_0033|shared|E3|donante2/coste1/orden0|41|41|válido|
|practice_0033|shared|E3|donante2/coste0/orden1|41|41|válido|
|practice_0033|shared|E3|donante2/coste1/orden1|41|41|válido|
|practice_0033|shared|E3|donante3/coste0/orden0|39|39|válido|
|practice_0033|shared|E3|donante3/coste1/orden0|39|39|válido|
|practice_0033|shared|E3|donante3/coste0/orden1|39|39|válido|
|practice_0033|shared|E3|donante3/coste1/orden1|39|39|válido|
|practice_0033|shared|E3|donante4/coste0/orden0|40|40|válido|
|practice_0033|shared|E3|donante4/coste1/orden0|40|40|válido|
|practice_0033|shared|E3|donante4/coste0/orden1|40|40|válido|
|practice_0033|shared|E3|donante4/coste1/orden1|40|40|válido|
|practice_0033|shared|E3|donante5/coste0/orden0|41|41|válido|
|practice_0033|shared|E3|donante5/coste1/orden0|41|41|válido|
|practice_0033|shared|E3|donante5/coste0/orden1|41|41|válido|
|practice_0033|shared|E3|donante5/coste1/orden1|41|41|válido|
|practice_0033|shared|E3|donante6/coste0/orden0|42|42|válido|
|practice_0033|shared|E3|donante6/coste1/orden0|42|42|válido|
|practice_0033|shared|E3|donante6/coste0/orden1|42|42|válido|
|practice_0033|shared|E3|donante6/coste1/orden1|42|42|válido|
|practice_0034|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0034|independent|Cruce|CP True/coste0/orden0|374|278|válido|
|practice_0034|independent|Cruce|CP True/coste1/orden0|317|278|válido|
|practice_0034|independent|Cruce|CP True/coste0/orden1|345|278|válido|
|practice_0034|independent|Cruce|CP True/coste1/orden1|296|278|válido|
|practice_0034|independent|Cruce|mejor independiente/coste0/orden0|381|280|válido|
|practice_0034|independent|Cruce|mejor independiente/coste1/orden0|333|280|válido|
|practice_0034|independent|Cruce|mejor independiente/coste0/orden1|294|280|válido|
|practice_0034|independent|Cruce|mejor independiente/coste1/orden1|303|280|válido|
|practice_0034|independent|E1|cola10/pol1/inicio0|294|96|válido|
|practice_0034|independent|E1|cola10/pol1/inicio1|294|96|válido|
|practice_0034|independent|E1|cola10/pol2/inicio0|287|89|válido|
|practice_0034|independent|E1|cola10/pol2/inicio1|291|93|válido|
|practice_0034|independent|E1|cola8/pol1/inicio0|292|76|válido|
|practice_0034|independent|E1|cola8/pol1/inicio1|292|76|válido|
|practice_0034|independent|E1|cola8/pol2/inicio0|294|78|válido|
|practice_0034|independent|E1|cola8/pol2/inicio1|294|78|válido|
|practice_0034|independent|E1|cola6/pol1/inicio0|288|54|válido|
|practice_0034|independent|E1|cola6/pol1/inicio1|288|54|válido|
|practice_0034|independent|E1|cola6/pol2/inicio0|288|54|válido|
|practice_0034|independent|E1|cola6/pol2/inicio1|288|54|válido|
|practice_0034|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0034|independent|E2|CP False/coste0/orden0|318|318|válido|
|practice_0034|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0034|independent|E2|CP True/coste0/orden0|374|278|válido|
|practice_0034|independent|E3|donante0/coste0/orden0|280|280|válido|
|practice_0034|independent|E3|donante0/coste1/orden0|280|280|válido|
|practice_0034|independent|E3|donante0/coste0/orden1|294|280|válido|
|practice_0034|independent|E3|donante0/coste1/orden1|303|280|válido|
|practice_0034|independent|E3|donante1/coste0/orden0|319|319|válido|
|practice_0034|independent|E3|donante1/coste1/orden0|319|319|válido|
|practice_0034|independent|E3|donante1/coste0/orden1|320|319|válido|
|practice_0034|independent|E3|donante1/coste1/orden1|320|319|válido|
|practice_0034|independent|E3|donante2/coste0/orden0|373|303|válido|
|practice_0034|independent|E3|donante2/coste1/orden0|330|303|válido|
|practice_0034|independent|E3|donante2/coste0/orden1|303|303|válido|
|practice_0034|independent|E3|donante2/coste1/orden1|303|303|válido|
|practice_0034|independent|E3|donante3/coste0/orden0|464|299|válido|
|practice_0034|independent|E3|donante3/coste1/orden0|371|299|válido|
|practice_0034|independent|E3|donante3/coste0/orden1|299|299|válido|
|practice_0034|independent|E3|donante3/coste1/orden1|299|299|válido|
|practice_0034|independent|E3|donante4/coste0/orden0|318|318|válido|
|practice_0034|independent|E3|donante4/coste1/orden0|318|318|válido|
|practice_0034|independent|E3|donante4/coste0/orden1|318|318|válido|
|practice_0034|independent|E3|donante4/coste1/orden1|318|318|válido|
|practice_0034|independent|E3|donante5/coste0/orden0|320|320|válido|
|practice_0034|independent|E3|donante5/coste1/orden0|320|320|válido|
|practice_0034|independent|E3|donante5/coste0/orden1|320|320|válido|
|practice_0034|independent|E3|donante5/coste1/orden1|320|320|válido|
|practice_0034|independent|E3|donante6/coste0/orden0|319|319|válido|
|practice_0034|independent|E3|donante6/coste1/orden0|319|319|válido|
|practice_0034|independent|E3|donante6/coste0/orden1|319|319|válido|
|practice_0034|independent|E3|donante6/coste1/orden1|319|319|válido|
|practice_0034|independent|E3|donante7/coste0/orden0|337|287|válido|
|practice_0034|independent|E3|donante7/coste1/orden0|314|287|válido|
|practice_0034|independent|E3|donante7/coste0/orden1|344|287|válido|
|practice_0034|independent|E3|donante7/coste1/orden1|341|287|válido|
|practice_0034|independent|E3|donante8/coste0/orden0|281|281|válido|
|practice_0034|independent|E3|donante8/coste1/orden0|281|281|válido|
|practice_0034|independent|E3|donante8/coste0/orden1|308|281|válido|
|practice_0034|independent|E3|donante8/coste1/orden1|305|281|válido|
|practice_0034|independent|E3|donante9/coste0/orden0|304|274|válido|
|practice_0034|independent|E3|donante9/coste1/orden0|289|274|válido|
|practice_0034|independent|E3|donante9/coste0/orden1|328|274|válido|
|practice_0034|independent|E3|donante9/coste1/orden1|307|274|válido|
|practice_0035|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0035|mixed|Cruce|CP True/coste0/orden0|309|233|válido|
|practice_0035|mixed|Cruce|CP True/coste1/orden0|288|233|válido|
|practice_0035|mixed|Cruce|CP True/coste0/orden1|278|233|válido|
|practice_0035|mixed|Cruce|CP True/coste1/orden1|265|233|válido|
|practice_0035|mixed|Cruce|mejor independiente/coste0/orden0|348|236|válido|
|practice_0035|mixed|Cruce|mejor independiente/coste1/orden0|301|236|válido|
|practice_0035|mixed|Cruce|mejor independiente/coste0/orden1|301|236|válido|
|practice_0035|mixed|Cruce|mejor independiente/coste1/orden1|287|236|válido|
|practice_0035|mixed|E1|cola10/pol1/inicio0|246|89|válido|
|practice_0035|mixed|E1|cola10/pol1/inicio1|246|89|válido|
|practice_0035|mixed|E1|cola10/pol2/inicio0|244|87|válido|
|practice_0035|mixed|E1|cola10/pol2/inicio1|244|87|válido|
|practice_0035|mixed|E1|cola8/pol1/inicio0|245|71|válido|
|practice_0035|mixed|E1|cola8/pol1/inicio1|245|71|válido|
|practice_0035|mixed|E1|cola8/pol2/inicio0|247|73|válido|
|practice_0035|mixed|E1|cola8/pol2/inicio1|247|73|válido|
|practice_0035|mixed|E1|cola6/pol1/inicio0|246|52|válido|
|practice_0035|mixed|E1|cola6/pol1/inicio1|246|52|válido|
|practice_0035|mixed|E1|cola6/pol2/inicio0|246|52|válido|
|practice_0035|mixed|E1|cola6/pol2/inicio1|246|52|válido|
|practice_0035|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0035|mixed|E2|CP False/coste0/orden0|338|259|válido|
|practice_0035|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0035|mixed|E2|CP True/coste0/orden0|331|237|válido|
|practice_0035|mixed|E3|donante0/coste0/orden0|265|235|válido|
|practice_0035|mixed|E3|donante0/coste1/orden0|255|235|válido|
|practice_0035|mixed|E3|donante0/coste0/orden1|246|235|válido|
|practice_0035|mixed|E3|donante0/coste1/orden1|246|235|válido|
|practice_0035|mixed|E3|donante1/coste0/orden0|265|265|válido|
|practice_0035|mixed|E3|donante1/coste1/orden0|265|265|válido|
|practice_0035|mixed|E3|donante1/coste0/orden1|265|265|válido|
|practice_0035|mixed|E3|donante1/coste1/orden1|265|265|válido|
|practice_0035|mixed|E3|donante2/coste0/orden0|267|249|válido|
|practice_0035|mixed|E3|donante2/coste1/orden0|259|249|válido|
|practice_0035|mixed|E3|donante2/coste0/orden1|249|249|válido|
|practice_0035|mixed|E3|donante2/coste1/orden1|249|249|válido|
|practice_0035|mixed|E3|donante3/coste0/orden0|246|246|válido|
|practice_0035|mixed|E3|donante3/coste1/orden0|246|246|válido|
|practice_0035|mixed|E3|donante3/coste0/orden1|246|246|válido|
|practice_0035|mixed|E3|donante3/coste1/orden1|246|246|válido|
|practice_0035|mixed|E3|donante4/coste0/orden0|267|267|válido|
|practice_0035|mixed|E3|donante4/coste1/orden0|267|267|válido|
|practice_0035|mixed|E3|donante4/coste0/orden1|267|267|válido|
|practice_0035|mixed|E3|donante4/coste1/orden1|267|267|válido|
|practice_0035|mixed|E3|donante5/coste0/orden0|254|254|válido|
|practice_0035|mixed|E3|donante5/coste1/orden0|254|254|válido|
|practice_0035|mixed|E3|donante5/coste0/orden1|254|254|válido|
|practice_0035|mixed|E3|donante5/coste1/orden1|254|254|válido|
|practice_0035|mixed|E3|donante6/coste0/orden0|279|279|válido|
|practice_0035|mixed|E3|donante6/coste1/orden0|279|279|válido|
|practice_0035|mixed|E3|donante6/coste0/orden1|279|279|válido|
|practice_0035|mixed|E3|donante6/coste1/orden1|279|279|válido|
|practice_0035|mixed|E3|donante7/coste0/orden0|243|243|válido|
|practice_0035|mixed|E3|donante7/coste1/orden0|243|243|válido|
|practice_0035|mixed|E3|donante7/coste0/orden1|243|243|válido|
|practice_0035|mixed|E3|donante7/coste1/orden1|243|243|válido|
|practice_0035|mixed|E3|donante8/coste0/orden0|246|234|válido|
|practice_0035|mixed|E3|donante8/coste1/orden0|246|234|válido|
|practice_0035|mixed|E3|donante8/coste0/orden1|260|234|válido|
|practice_0035|mixed|E3|donante8/coste1/orden1|260|234|válido|
|practice_0035|mixed|E3|donante9/coste0/orden0|265|235|válido|
|practice_0035|mixed|E3|donante9/coste1/orden0|255|235|válido|
|practice_0035|mixed|E3|donante9/coste0/orden1|246|235|válido|
|practice_0035|mixed|E3|donante9/coste1/orden1|246|235|válido|
|practice_0036|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0036|shared|Cruce|CP True/coste0/orden0|117|117|válido|
|practice_0036|shared|Cruce|CP True/coste1/orden0|117|117|válido|
|practice_0036|shared|Cruce|CP True/coste0/orden1|117|117|válido|
|practice_0036|shared|Cruce|CP True/coste1/orden1|117|117|válido|
|practice_0036|shared|Cruce|mejor independiente/coste0/orden0|117|117|válido|
|practice_0036|shared|Cruce|mejor independiente/coste1/orden0|117|117|válido|
|practice_0036|shared|Cruce|mejor independiente/coste0/orden1|117|117|válido|
|practice_0036|shared|Cruce|mejor independiente/coste1/orden1|117|117|válido|
|practice_0036|shared|E1|cola10/pol1/inicio0|129|40|válido|
|practice_0036|shared|E1|cola10/pol1/inicio1|129|40|válido|
|practice_0036|shared|E1|cola10/pol2/inicio0|129|40|válido|
|practice_0036|shared|E1|cola10/pol2/inicio1|129|40|válido|
|practice_0036|shared|E1|cola8/pol1/inicio0|127|31|válido|
|practice_0036|shared|E1|cola8/pol1/inicio1|127|31|válido|
|practice_0036|shared|E1|cola8/pol2/inicio0|127|31|válido|
|practice_0036|shared|E1|cola8/pol2/inicio1|127|31|válido|
|practice_0036|shared|E1|cola6/pol1/inicio0|127|23|válido|
|practice_0036|shared|E1|cola6/pol1/inicio1|127|23|válido|
|practice_0036|shared|E1|cola6/pol2/inicio0|127|23|válido|
|practice_0036|shared|E1|cola6/pol2/inicio1|127|23|válido|
|practice_0036|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0036|shared|E2|CP False/coste0/orden0|124|124|válido|
|practice_0036|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0036|shared|E2|CP True/coste0/orden0|117|117|válido|
|practice_0036|shared|E3|donante0/coste0/orden0|125|125|válido|
|practice_0036|shared|E3|donante0/coste1/orden0|125|125|válido|
|practice_0036|shared|E3|donante0/coste0/orden1|125|125|válido|
|practice_0036|shared|E3|donante0/coste1/orden1|125|125|válido|
|practice_0036|shared|E3|donante1/coste0/orden0|133|133|válido|
|practice_0036|shared|E3|donante1/coste1/orden0|133|133|válido|
|practice_0036|shared|E3|donante1/coste0/orden1|133|133|válido|
|practice_0036|shared|E3|donante1/coste1/orden1|133|133|válido|
|practice_0036|shared|E3|donante2/coste0/orden0|131|131|válido|
|practice_0036|shared|E3|donante2/coste1/orden0|131|131|válido|
|practice_0036|shared|E3|donante2/coste0/orden1|131|131|válido|
|practice_0036|shared|E3|donante2/coste1/orden1|131|131|válido|
|practice_0036|shared|E3|donante3/coste0/orden0|126|126|válido|
|practice_0036|shared|E3|donante3/coste1/orden0|126|126|válido|
|practice_0036|shared|E3|donante3/coste0/orden1|126|126|válido|
|practice_0036|shared|E3|donante3/coste1/orden1|126|126|válido|
|practice_0036|shared|E3|donante4/coste0/orden0|137|137|válido|
|practice_0036|shared|E3|donante4/coste1/orden0|137|137|válido|
|practice_0036|shared|E3|donante4/coste0/orden1|137|137|válido|
|practice_0036|shared|E3|donante4/coste1/orden1|137|137|válido|
|practice_0036|shared|E3|donante5/coste0/orden0|142|142|válido|
|practice_0036|shared|E3|donante5/coste1/orden0|142|142|válido|
|practice_0036|shared|E3|donante5/coste0/orden1|142|142|válido|
|practice_0036|shared|E3|donante5/coste1/orden1|142|142|válido|
|practice_0036|shared|E3|donante6/coste0/orden0|136|136|válido|
|practice_0036|shared|E3|donante6/coste1/orden0|136|136|válido|
|practice_0036|shared|E3|donante6/coste0/orden1|136|136|válido|
|practice_0036|shared|E3|donante6/coste1/orden1|136|136|válido|
|practice_0036|shared|E3|donante7/coste0/orden0|130|129|válido|
|practice_0036|shared|E3|donante7/coste1/orden0|130|129|válido|
|practice_0036|shared|E3|donante7/coste0/orden1|129|129|válido|
|practice_0036|shared|E3|donante7/coste1/orden1|129|129|válido|
|practice_0036|shared|E3|donante8/coste0/orden0|128|128|válido|
|practice_0036|shared|E3|donante8/coste1/orden0|128|128|válido|
|practice_0036|shared|E3|donante8/coste0/orden1|128|128|válido|
|practice_0036|shared|E3|donante8/coste1/orden1|128|128|válido|
|practice_0036|shared|E3|donante9/coste0/orden0|129|129|válido|
|practice_0036|shared|E3|donante9/coste1/orden0|129|129|válido|
|practice_0036|shared|E3|donante9/coste0/orden1|129|129|válido|
|practice_0036|shared|E3|donante9/coste1/orden1|129|129|válido|
|practice_0037|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0037|independent|Cruce|CP True/coste0/orden0|347|248|válido|
|practice_0037|independent|Cruce|CP True/coste1/orden0|313|248|válido|
|practice_0037|independent|Cruce|CP True/coste0/orden1|258|248|válido|
|practice_0037|independent|Cruce|CP True/coste1/orden1|256|248|válido|
|practice_0037|independent|Cruce|mejor independiente/coste0/orden0|347|248|válido|
|practice_0037|independent|Cruce|mejor independiente/coste1/orden0|313|248|válido|
|practice_0037|independent|Cruce|mejor independiente/coste0/orden1|258|248|válido|
|practice_0037|independent|Cruce|mejor independiente/coste1/orden1|256|248|válido|
|practice_0037|independent|E1|cola10/pol1/inicio0|265|75|válido|
|practice_0037|independent|E1|cola10/pol1/inicio1|265|75|válido|
|practice_0037|independent|E1|cola10/pol2/inicio0|268|78|válido|
|practice_0037|independent|E1|cola10/pol2/inicio1|268|78|válido|
|practice_0037|independent|E1|cola8/pol1/inicio0|262|58|válido|
|practice_0037|independent|E1|cola8/pol1/inicio1|262|58|válido|
|practice_0037|independent|E1|cola8/pol2/inicio0|265|61|válido|
|practice_0037|independent|E1|cola8/pol2/inicio1|265|61|válido|
|practice_0037|independent|E1|cola6/pol1/inicio0|271|52|válido|
|practice_0037|independent|E1|cola6/pol1/inicio1|271|52|válido|
|practice_0037|independent|E1|cola6/pol2/inicio0|271|52|válido|
|practice_0037|independent|E1|cola6/pol2/inicio1|271|52|válido|
|practice_0037|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0037|independent|E2|CP False/coste0/orden0|301|301|válido|
|practice_0037|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0037|independent|E2|CP True/coste0/orden0|367|258|válido|
|practice_0037|independent|E3|donante0/coste0/orden0|259|259|válido|
|practice_0037|independent|E3|donante0/coste1/orden0|259|259|válido|
|practice_0037|independent|E3|donante0/coste0/orden1|260|259|válido|
|practice_0037|independent|E3|donante0/coste1/orden1|260|259|válido|
|practice_0037|independent|E3|donante1/coste0/orden0|293|293|válido|
|practice_0037|independent|E3|donante1/coste1/orden0|293|293|válido|
|practice_0037|independent|E3|donante1/coste0/orden1|293|293|válido|
|practice_0037|independent|E3|donante1/coste1/orden1|293|293|válido|
|practice_0037|independent|E3|donante2/coste0/orden0|332|274|válido|
|practice_0037|independent|E3|donante2/coste1/orden0|289|274|válido|
|practice_0037|independent|E3|donante2/coste0/orden1|286|274|válido|
|practice_0037|independent|E3|donante2/coste1/orden1|286|274|válido|
|practice_0037|independent|E3|donante3/coste0/orden0|505|269|válido|
|practice_0037|independent|E3|donante3/coste1/orden0|395|269|válido|
|practice_0037|independent|E3|donante3/coste0/orden1|269|269|válido|
|practice_0037|independent|E3|donante3/coste1/orden1|269|269|válido|
|practice_0037|independent|E3|donante4/coste0/orden0|301|301|válido|
|practice_0037|independent|E3|donante4/coste1/orden0|301|301|válido|
|practice_0037|independent|E3|donante4/coste0/orden1|301|301|válido|
|practice_0037|independent|E3|donante4/coste1/orden1|301|301|válido|
|practice_0037|independent|E3|donante5/coste0/orden0|296|296|válido|
|practice_0037|independent|E3|donante5/coste1/orden0|296|296|válido|
|practice_0037|independent|E3|donante5/coste0/orden1|296|296|válido|
|practice_0037|independent|E3|donante5/coste1/orden1|296|296|válido|
|practice_0037|independent|E3|donante6/coste0/orden0|296|296|válido|
|practice_0037|independent|E3|donante6/coste1/orden0|296|296|válido|
|practice_0037|independent|E3|donante6/coste0/orden1|296|296|válido|
|practice_0037|independent|E3|donante6/coste1/orden1|296|296|válido|
|practice_0037|independent|E3|donante7/coste0/orden0|269|267|válido|
|practice_0037|independent|E3|donante7/coste1/orden0|269|267|válido|
|practice_0037|independent|E3|donante7/coste0/orden1|314|267|válido|
|practice_0037|independent|E3|donante7/coste1/orden1|274|267|válido|
|practice_0037|independent|E3|donante8/coste0/orden0|296|248|válido|
|practice_0037|independent|E3|donante8/coste1/orden0|269|248|válido|
|practice_0037|independent|E3|donante8/coste0/orden1|258|248|válido|
|practice_0037|independent|E3|donante8/coste1/orden1|256|248|válido|
|practice_0038|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0038|independent|Cruce|CP True/coste0/orden0|37|37|válido|
|practice_0038|independent|Cruce|CP True/coste1/orden0|37|37|válido|
|practice_0038|independent|Cruce|CP True/coste0/orden1|37|37|válido|
|practice_0038|independent|Cruce|CP True/coste1/orden1|37|37|válido|
|practice_0038|independent|Cruce|mejor independiente/coste0/orden0|39|39|válido|
|practice_0038|independent|Cruce|mejor independiente/coste1/orden0|39|39|válido|
|practice_0038|independent|Cruce|mejor independiente/coste0/orden1|39|39|válido|
|practice_0038|independent|Cruce|mejor independiente/coste1/orden1|39|39|válido|
|practice_0038|independent|E1|cola8/pol1/inicio0|41|31|válido|
|practice_0038|independent|E1|cola8/pol1/inicio1|44|34|válido|
|practice_0038|independent|E1|cola8/pol2/inicio0|40|30|válido|
|practice_0038|independent|E1|cola8/pol2/inicio1|40|30|válido|
|practice_0038|independent|E1|cola6/pol1/inicio0|44|22|válido|
|practice_0038|independent|E1|cola6/pol1/inicio1|48|26|válido|
|practice_0038|independent|E1|cola6/pol2/inicio0|44|22|válido|
|practice_0038|independent|E1|cola6/pol2/inicio1|47|25|válido|
|practice_0038|independent|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0038|independent|E2|CP False/coste0/orden0|43|43|válido|
|practice_0038|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0038|independent|E2|CP True/coste0/orden0|41|41|válido|
|practice_0038|independent|E3|donante0/coste0/orden0|43|43|válido|
|practice_0038|independent|E3|donante0/coste1/orden0|43|43|válido|
|practice_0038|independent|E3|donante0/coste0/orden1|43|43|válido|
|practice_0038|independent|E3|donante0/coste1/orden1|43|43|válido|
|practice_0038|independent|E3|donante1/coste0/orden0|45|45|válido|
|practice_0038|independent|E3|donante1/coste1/orden0|45|45|válido|
|practice_0038|independent|E3|donante1/coste0/orden1|45|45|válido|
|practice_0038|independent|E3|donante1/coste1/orden1|45|45|válido|
|practice_0038|independent|E3|donante2/coste0/orden0|46|46|válido|
|practice_0038|independent|E3|donante2/coste1/orden0|46|46|válido|
|practice_0038|independent|E3|donante2/coste0/orden1|46|46|válido|
|practice_0038|independent|E3|donante2/coste1/orden1|46|46|válido|
|practice_0038|independent|E3|donante3/coste0/orden0|45|45|válido|
|practice_0038|independent|E3|donante3/coste1/orden0|45|45|válido|
|practice_0038|independent|E3|donante3/coste0/orden1|45|45|válido|
|practice_0038|independent|E3|donante3/coste1/orden1|45|45|válido|
|practice_0038|independent|E3|donante4/coste0/orden0|45|45|válido|
|practice_0038|independent|E3|donante4/coste1/orden0|45|45|válido|
|practice_0038|independent|E3|donante4/coste0/orden1|45|45|válido|
|practice_0038|independent|E3|donante4/coste1/orden1|45|45|válido|
|practice_0038|independent|E3|donante5/coste0/orden0|47|47|válido|
|practice_0038|independent|E3|donante5/coste1/orden0|47|47|válido|
|practice_0038|independent|E3|donante5/coste0/orden1|47|47|válido|
|practice_0038|independent|E3|donante5/coste1/orden1|47|47|válido|
|practice_0038|independent|E3|donante6/coste0/orden0|46|46|válido|
|practice_0038|independent|E3|donante6/coste1/orden0|46|46|válido|
|practice_0038|independent|E3|donante6/coste0/orden1|46|46|válido|
|practice_0038|independent|E3|donante6/coste1/orden1|46|46|válido|
|practice_0038|independent|E3|donante7/coste0/orden0|46|46|válido|
|practice_0038|independent|E3|donante7/coste1/orden0|46|46|válido|
|practice_0038|independent|E3|donante7/coste0/orden1|46|46|válido|
|practice_0038|independent|E3|donante7/coste1/orden1|46|46|válido|
|practice_0039|tiny|Cruce|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0039|tiny|Cruce|CP True/coste0/orden0|6|6|válido|
|practice_0039|tiny|Cruce|CP True/coste1/orden0|6|6|válido|
|practice_0039|tiny|Cruce|CP True/coste0/orden1|6|6|válido|
|practice_0039|tiny|Cruce|CP True/coste1/orden1|6|6|válido|
|practice_0039|tiny|Cruce|mejor independiente/coste0/orden0|6|6|válido|
|practice_0039|tiny|Cruce|mejor independiente/coste1/orden0|6|6|válido|
|practice_0039|tiny|Cruce|mejor independiente/coste0/orden1|6|6|válido|
|practice_0039|tiny|Cruce|mejor independiente/coste1/orden1|6|6|válido|
|practice_0039|tiny|E1|cola4/pol1/inicio0|9|8|válido|
|practice_0039|tiny|E1|cola4/pol1/inicio1|9|8|válido|
|practice_0039|tiny|E1|cola4/pol2/inicio0|9|8|válido|
|practice_0039|tiny|E1|cola4/pol2/inicio1|9|8|válido|
|practice_0039|tiny|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0039|tiny|E2|CP False/coste0/orden0|6|6|válido|
|practice_0039|tiny|E2|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0039|tiny|E2|CP True/coste0/orden0|6|6|válido|
|practice_0039|tiny|E3|donante0/coste0/orden0|6|6|válido|
|practice_0039|tiny|E3|donante0/coste1/orden0|6|6|válido|
|practice_0039|tiny|E3|donante0/coste0/orden1|6|6|válido|
|practice_0039|tiny|E3|donante0/coste1/orden1|6|6|válido|
|practice_0039|tiny|E3|donante1/coste0/orden0|9|9|válido|
|practice_0039|tiny|E3|donante1/coste1/orden0|9|9|válido|
|practice_0039|tiny|E3|donante1/coste0/orden1|9|9|válido|
|practice_0039|tiny|E3|donante1/coste1/orden1|9|9|válido|
|practice_0039|tiny|E3|donante2/coste0/orden0|9|9|válido|
|practice_0039|tiny|E3|donante2/coste1/orden0|9|9|válido|
|practice_0039|tiny|E3|donante2/coste0/orden1|9|9|válido|
|practice_0039|tiny|E3|donante2/coste1/orden1|9|9|válido|
|practice_0040|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0040|mixed|Cruce|CP True/coste0/orden0|172|172|válido|
|practice_0040|mixed|Cruce|CP True/coste1/orden0|172|172|válido|
|practice_0040|mixed|Cruce|CP True/coste0/orden1|172|172|válido|
|practice_0040|mixed|Cruce|CP True/coste1/orden1|172|172|válido|
|practice_0040|mixed|Cruce|mejor independiente/coste0/orden0|173|173|válido|
|practice_0040|mixed|Cruce|mejor independiente/coste1/orden0|173|173|válido|
|practice_0040|mixed|Cruce|mejor independiente/coste0/orden1|173|173|válido|
|practice_0040|mixed|Cruce|mejor independiente/coste1/orden1|173|173|válido|
|practice_0040|mixed|E1|cola10/pol1/inicio0|179|54|válido|
|practice_0040|mixed|E1|cola10/pol1/inicio1|179|54|válido|
|practice_0040|mixed|E1|cola10/pol2/inicio0|182|57|válido|
|practice_0040|mixed|E1|cola10/pol2/inicio1|182|57|válido|
|practice_0040|mixed|E1|cola8/pol1/inicio0|181|46|válido|
|practice_0040|mixed|E1|cola8/pol1/inicio1|181|46|válido|
|practice_0040|mixed|E1|cola8/pol2/inicio0|181|46|válido|
|practice_0040|mixed|E1|cola8/pol2/inicio1|181|46|válido|
|practice_0040|mixed|E1|cola6/pol1/inicio0|181|36|válido|
|practice_0040|mixed|E1|cola6/pol1/inicio1|181|36|válido|
|practice_0040|mixed|E1|cola6/pol2/inicio0|181|36|válido|
|practice_0040|mixed|E1|cola6/pol2/inicio1|181|36|válido|
|practice_0040|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0040|mixed|E2|CP False/coste0/orden0|198|198|válido|
|practice_0040|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0040|mixed|E2|CP True/coste0/orden0|173|173|válido|
|practice_0040|mixed|E3|donante0/coste0/orden0|174|174|válido|
|practice_0040|mixed|E3|donante0/coste1/orden0|174|174|válido|
|practice_0040|mixed|E3|donante0/coste0/orden1|174|174|válido|
|practice_0040|mixed|E3|donante0/coste1/orden1|174|174|válido|
|practice_0040|mixed|E3|donante1/coste0/orden0|203|203|válido|
|practice_0040|mixed|E3|donante1/coste1/orden0|203|203|válido|
|practice_0040|mixed|E3|donante1/coste0/orden1|203|203|válido|
|practice_0040|mixed|E3|donante1/coste1/orden1|203|203|válido|
|practice_0040|mixed|E3|donante2/coste0/orden0|195|195|válido|
|practice_0040|mixed|E3|donante2/coste1/orden0|195|195|válido|
|practice_0040|mixed|E3|donante2/coste0/orden1|195|195|válido|
|practice_0040|mixed|E3|donante2/coste1/orden1|195|195|válido|
|practice_0040|mixed|E3|donante3/coste0/orden0|187|187|válido|
|practice_0040|mixed|E3|donante3/coste1/orden0|187|187|válido|
|practice_0040|mixed|E3|donante3/coste0/orden1|187|187|válido|
|practice_0040|mixed|E3|donante3/coste1/orden1|187|187|válido|
|practice_0040|mixed|E3|donante4/coste0/orden0|204|204|válido|
|practice_0040|mixed|E3|donante4/coste1/orden0|204|204|válido|
|practice_0040|mixed|E3|donante4/coste0/orden1|204|204|válido|
|practice_0040|mixed|E3|donante4/coste1/orden1|204|204|válido|
|practice_0040|mixed|E3|donante5/coste0/orden0|197|197|válido|
|practice_0040|mixed|E3|donante5/coste1/orden0|197|197|válido|
|practice_0040|mixed|E3|donante5/coste0/orden1|197|197|válido|
|practice_0040|mixed|E3|donante5/coste1/orden1|197|197|válido|
|practice_0040|mixed|E3|donante6/coste0/orden0|197|197|válido|
|practice_0040|mixed|E3|donante6/coste1/orden0|197|197|válido|
|practice_0040|mixed|E3|donante6/coste0/orden1|197|197|válido|
|practice_0040|mixed|E3|donante6/coste1/orden1|197|197|válido|
|practice_0040|mixed|E3|donante7/coste0/orden0|203|188|válido|
|practice_0040|mixed|E3|donante7/coste1/orden0|197|188|válido|
|practice_0040|mixed|E3|donante7/coste0/orden1|199|188|válido|
|practice_0040|mixed|E3|donante7/coste1/orden1|191|188|válido|
|practice_0040|mixed|E3|donante8/coste0/orden0|183|183|válido|
|practice_0040|mixed|E3|donante8/coste1/orden0|183|183|válido|
|practice_0040|mixed|E3|donante8/coste0/orden1|183|183|válido|
|practice_0040|mixed|E3|donante8/coste1/orden1|183|183|válido|
|practice_0041|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0041|mixed|Cruce|CP True/coste0/orden0|163|163|válido|
|practice_0041|mixed|Cruce|CP True/coste1/orden0|163|163|válido|
|practice_0041|mixed|Cruce|CP True/coste0/orden1|163|163|válido|
|practice_0041|mixed|Cruce|CP True/coste1/orden1|163|163|válido|
|practice_0041|mixed|Cruce|mejor independiente/coste0/orden0|164|164|válido|
|practice_0041|mixed|Cruce|mejor independiente/coste1/orden0|164|164|válido|
|practice_0041|mixed|Cruce|mejor independiente/coste0/orden1|164|164|válido|
|practice_0041|mixed|Cruce|mejor independiente/coste1/orden1|164|164|válido|
|practice_0041|mixed|E1|cola10/pol1/inicio0|180|66|válido|
|practice_0041|mixed|E1|cola10/pol1/inicio1|180|66|válido|
|practice_0041|mixed|E1|cola10/pol2/inicio0|178|64|válido|
|practice_0041|mixed|E1|cola10/pol2/inicio1|178|64|válido|
|practice_0041|mixed|E1|cola8/pol1/inicio0|182|57|válido|
|practice_0041|mixed|E1|cola8/pol1/inicio1|182|57|válido|
|practice_0041|mixed|E1|cola8/pol2/inicio0|180|55|válido|
|practice_0041|mixed|E1|cola8/pol2/inicio1|180|55|válido|
|practice_0041|mixed|E1|cola6/pol1/inicio0|164|28|válido|
|practice_0041|mixed|E1|cola6/pol1/inicio1|164|28|válido|
|practice_0041|mixed|E1|cola6/pol2/inicio0|164|28|válido|
|practice_0041|mixed|E1|cola6/pol2/inicio1|164|28|válido|
|practice_0041|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0041|mixed|E2|CP False/coste0/orden0|179|175|válido|
|practice_0041|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0041|mixed|E2|CP True/coste0/orden0|164|164|válido|
|practice_0041|mixed|E3|donante0/coste0/orden0|164|164|válido|
|practice_0041|mixed|E3|donante0/coste1/orden0|164|164|válido|
|practice_0041|mixed|E3|donante0/coste0/orden1|164|164|válido|
|practice_0041|mixed|E3|donante0/coste1/orden1|164|164|válido|
|practice_0041|mixed|E3|donante1/coste0/orden0|188|188|válido|
|practice_0041|mixed|E3|donante1/coste1/orden0|188|188|válido|
|practice_0041|mixed|E3|donante1/coste0/orden1|188|188|válido|
|practice_0041|mixed|E3|donante1/coste1/orden1|188|188|válido|
|practice_0041|mixed|E3|donante2/coste0/orden0|182|182|válido|
|practice_0041|mixed|E3|donante2/coste1/orden0|182|182|válido|
|practice_0041|mixed|E3|donante2/coste0/orden1|182|182|válido|
|practice_0041|mixed|E3|donante2/coste1/orden1|182|182|válido|
|practice_0041|mixed|E3|donante3/coste0/orden0|206|176|válido|
|practice_0041|mixed|E3|donante3/coste1/orden0|184|176|válido|
|practice_0041|mixed|E3|donante3/coste0/orden1|176|176|válido|
|practice_0041|mixed|E3|donante3/coste1/orden1|176|176|válido|
|practice_0041|mixed|E3|donante4/coste0/orden0|191|191|válido|
|practice_0041|mixed|E3|donante4/coste1/orden0|191|191|válido|
|practice_0041|mixed|E3|donante4/coste0/orden1|191|191|válido|
|practice_0041|mixed|E3|donante4/coste1/orden1|191|191|válido|
|practice_0041|mixed|E3|donante5/coste0/orden0|190|190|válido|
|practice_0041|mixed|E3|donante5/coste1/orden0|190|190|válido|
|practice_0041|mixed|E3|donante5/coste0/orden1|190|190|válido|
|practice_0041|mixed|E3|donante5/coste1/orden1|190|190|válido|
|practice_0041|mixed|E3|donante6/coste0/orden0|194|194|válido|
|practice_0041|mixed|E3|donante6/coste1/orden0|194|194|válido|
|practice_0041|mixed|E3|donante6/coste0/orden1|194|194|válido|
|practice_0041|mixed|E3|donante6/coste1/orden1|194|194|válido|
|practice_0041|mixed|E3|donante7/coste0/orden0|267|175|válido|
|practice_0041|mixed|E3|donante7/coste1/orden0|211|175|válido|
|practice_0041|mixed|E3|donante7/coste0/orden1|196|175|válido|
|practice_0041|mixed|E3|donante7/coste1/orden1|197|175|válido|
|practice_0041|mixed|E3|donante8/coste0/orden0|213|170|válido|
|practice_0041|mixed|E3|donante8/coste1/orden0|189|170|válido|
|practice_0041|mixed|E3|donante8/coste0/orden1|203|170|válido|
|practice_0041|mixed|E3|donante8/coste1/orden1|201|170|válido|
|practice_0041|mixed|E3|donante9/coste0/orden0|165|165|válido|
|practice_0041|mixed|E3|donante9/coste1/orden0|165|165|válido|
|practice_0041|mixed|E3|donante9/coste0/orden1|165|165|válido|
|practice_0041|mixed|E3|donante9/coste1/orden1|165|165|válido|
|practice_0042|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0042|shared|Cruce|CP True/coste0/orden0|154|154|válido|
|practice_0042|shared|Cruce|CP True/coste1/orden0|154|154|válido|
|practice_0042|shared|Cruce|CP True/coste0/orden1|154|154|válido|
|practice_0042|shared|Cruce|CP True/coste1/orden1|154|154|válido|
|practice_0042|shared|Cruce|mejor independiente/coste0/orden0|154|154|válido|
|practice_0042|shared|Cruce|mejor independiente/coste1/orden0|154|154|válido|
|practice_0042|shared|Cruce|mejor independiente/coste0/orden1|154|154|válido|
|practice_0042|shared|Cruce|mejor independiente/coste1/orden1|154|154|válido|
|practice_0042|shared|E1|cola10/pol1/inicio0|162|57|válido|
|practice_0042|shared|E1|cola10/pol1/inicio1|162|57|válido|
|practice_0042|shared|E1|cola10/pol2/inicio0|162|57|válido|
|practice_0042|shared|E1|cola10/pol2/inicio1|162|57|válido|
|practice_0042|shared|E1|cola8/pol1/inicio0|162|46|válido|
|practice_0042|shared|E1|cola8/pol1/inicio1|162|46|válido|
|practice_0042|shared|E1|cola8/pol2/inicio0|162|46|válido|
|practice_0042|shared|E1|cola8/pol2/inicio1|162|46|válido|
|practice_0042|shared|E1|cola6/pol1/inicio0|162|36|válido|
|practice_0042|shared|E1|cola6/pol1/inicio1|162|36|válido|
|practice_0042|shared|E1|cola6/pol2/inicio0|162|36|válido|
|practice_0042|shared|E1|cola6/pol2/inicio1|162|36|válido|
|practice_0042|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0042|shared|E2|CP False/coste0/orden0|166|166|válido|
|practice_0042|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0042|shared|E2|CP True/coste0/orden0|155|155|válido|
|practice_0042|shared|E3|donante0/coste0/orden0|156|156|válido|
|practice_0042|shared|E3|donante0/coste1/orden0|156|156|válido|
|practice_0042|shared|E3|donante0/coste0/orden1|156|156|válido|
|practice_0042|shared|E3|donante0/coste1/orden1|156|156|válido|
|practice_0042|shared|E3|donante1/coste0/orden0|185|185|válido|
|practice_0042|shared|E3|donante1/coste1/orden0|185|185|válido|
|practice_0042|shared|E3|donante1/coste0/orden1|185|185|válido|
|practice_0042|shared|E3|donante1/coste1/orden1|185|185|válido|
|practice_0042|shared|E3|donante2/coste0/orden0|170|170|válido|
|practice_0042|shared|E3|donante2/coste1/orden0|170|170|válido|
|practice_0042|shared|E3|donante2/coste0/orden1|170|170|válido|
|practice_0042|shared|E3|donante2/coste1/orden1|170|170|válido|
|practice_0042|shared|E3|donante3/coste0/orden0|170|170|válido|
|practice_0042|shared|E3|donante3/coste1/orden0|170|170|válido|
|practice_0042|shared|E3|donante3/coste0/orden1|170|170|válido|
|practice_0042|shared|E3|donante3/coste1/orden1|170|170|válido|
|practice_0042|shared|E3|donante4/coste0/orden0|200|200|válido|
|practice_0042|shared|E3|donante4/coste1/orden0|200|200|válido|
|practice_0042|shared|E3|donante4/coste0/orden1|200|200|válido|
|practice_0042|shared|E3|donante4/coste1/orden1|200|200|válido|
|practice_0042|shared|E3|donante5/coste0/orden0|187|187|válido|
|practice_0042|shared|E3|donante5/coste1/orden0|187|187|válido|
|practice_0042|shared|E3|donante5/coste0/orden1|187|187|válido|
|practice_0042|shared|E3|donante5/coste1/orden1|187|187|válido|
|practice_0042|shared|E3|donante6/coste0/orden0|181|181|válido|
|practice_0042|shared|E3|donante6/coste1/orden0|181|181|válido|
|practice_0042|shared|E3|donante6/coste0/orden1|181|181|válido|
|practice_0042|shared|E3|donante6/coste1/orden1|181|181|válido|
|practice_0042|shared|E3|donante7/coste0/orden0|192|169|válido|
|practice_0042|shared|E3|donante7/coste1/orden0|190|169|válido|
|practice_0042|shared|E3|donante7/coste0/orden1|169|169|válido|
|practice_0042|shared|E3|donante7/coste1/orden1|169|169|válido|
|practice_0042|shared|E3|donante8/coste0/orden0|198|172|válido|
|practice_0042|shared|E3|donante8/coste1/orden0|178|172|válido|
|practice_0042|shared|E3|donante8/coste0/orden1|172|172|válido|
|practice_0042|shared|E3|donante8/coste1/orden1|172|172|válido|
|practice_0043|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0043|shared|Cruce|CP True/coste0/orden0|116|116|válido|
|practice_0043|shared|Cruce|CP True/coste1/orden0|116|116|válido|
|practice_0043|shared|Cruce|CP True/coste0/orden1|116|116|válido|
|practice_0043|shared|Cruce|CP True/coste1/orden1|116|116|válido|
|practice_0043|shared|Cruce|mejor independiente/coste0/orden0|118|118|válido|
|practice_0043|shared|Cruce|mejor independiente/coste1/orden0|118|118|válido|
|practice_0043|shared|Cruce|mejor independiente/coste0/orden1|118|118|válido|
|practice_0043|shared|Cruce|mejor independiente/coste1/orden1|118|118|válido|
|practice_0043|shared|E1|cola10/pol1/inicio0|120|37|válido|
|practice_0043|shared|E1|cola10/pol1/inicio1|120|37|válido|
|practice_0043|shared|E1|cola10/pol2/inicio0|120|37|válido|
|practice_0043|shared|E1|cola10/pol2/inicio1|120|37|válido|
|practice_0043|shared|E1|cola8/pol1/inicio0|120|31|válido|
|practice_0043|shared|E1|cola8/pol1/inicio1|120|31|válido|
|practice_0043|shared|E1|cola8/pol2/inicio0|120|31|válido|
|practice_0043|shared|E1|cola8/pol2/inicio1|120|31|válido|
|practice_0043|shared|E1|cola6/pol1/inicio0|124|28|válido|
|practice_0043|shared|E1|cola6/pol1/inicio1|124|28|válido|
|practice_0043|shared|E1|cola6/pol2/inicio0|121|25|válido|
|practice_0043|shared|E1|cola6/pol2/inicio1|121|25|válido|
|practice_0043|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0043|shared|E2|CP False/coste0/orden0|124|124|válido|
|practice_0043|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0043|shared|E2|CP True/coste0/orden0|118|118|válido|
|practice_0043|shared|E3|donante0/coste0/orden0|122|122|válido|
|practice_0043|shared|E3|donante0/coste1/orden0|122|122|válido|
|practice_0043|shared|E3|donante0/coste0/orden1|122|122|válido|
|practice_0043|shared|E3|donante0/coste1/orden1|122|122|válido|
|practice_0043|shared|E3|donante1/coste0/orden0|127|127|válido|
|practice_0043|shared|E3|donante1/coste1/orden0|127|127|válido|
|practice_0043|shared|E3|donante1/coste0/orden1|127|127|válido|
|practice_0043|shared|E3|donante1/coste1/orden1|127|127|válido|
|practice_0043|shared|E3|donante2/coste0/orden0|123|123|válido|
|practice_0043|shared|E3|donante2/coste1/orden0|123|123|válido|
|practice_0043|shared|E3|donante2/coste0/orden1|123|123|válido|
|practice_0043|shared|E3|donante2/coste1/orden1|123|123|válido|
|practice_0043|shared|E3|donante3/coste0/orden0|125|125|válido|
|practice_0043|shared|E3|donante3/coste1/orden0|125|125|válido|
|practice_0043|shared|E3|donante3/coste0/orden1|125|125|válido|
|practice_0043|shared|E3|donante3/coste1/orden1|125|125|válido|
|practice_0043|shared|E3|donante4/coste0/orden0|130|130|válido|
|practice_0043|shared|E3|donante4/coste1/orden0|130|130|válido|
|practice_0043|shared|E3|donante4/coste0/orden1|130|130|válido|
|practice_0043|shared|E3|donante4/coste1/orden1|130|130|válido|
|practice_0043|shared|E3|donante5/coste0/orden0|137|137|válido|
|practice_0043|shared|E3|donante5/coste1/orden0|137|137|válido|
|practice_0043|shared|E3|donante5/coste0/orden1|137|137|válido|
|practice_0043|shared|E3|donante5/coste1/orden1|137|137|válido|
|practice_0043|shared|E3|donante6/coste0/orden0|129|129|válido|
|practice_0043|shared|E3|donante6/coste1/orden0|129|129|válido|
|practice_0043|shared|E3|donante6/coste0/orden1|129|129|válido|
|practice_0043|shared|E3|donante6/coste1/orden1|129|129|válido|
|practice_0043|shared|E3|donante7/coste0/orden0|126|126|válido|
|practice_0043|shared|E3|donante7/coste1/orden0|126|126|válido|
|practice_0043|shared|E3|donante7/coste0/orden1|126|126|válido|
|practice_0043|shared|E3|donante7/coste1/orden1|126|126|válido|
|practice_0043|shared|E3|donante8/coste0/orden0|127|127|válido|
|practice_0043|shared|E3|donante8/coste1/orden0|127|127|válido|
|practice_0043|shared|E3|donante8/coste0/orden1|127|127|válido|
|practice_0043|shared|E3|donante8/coste1/orden1|127|127|válido|
|practice_0043|shared|E3|donante9/coste0/orden0|122|122|válido|
|practice_0043|shared|E3|donante9/coste1/orden0|122|122|válido|
|practice_0043|shared|E3|donante9/coste0/orden1|122|122|válido|
|practice_0043|shared|E3|donante9/coste1/orden1|122|122|válido|
|practice_0044|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0044|mixed|Cruce|CP True/coste0/orden0|252|195|válido|
|practice_0044|mixed|Cruce|CP True/coste1/orden0|215|195|válido|
|practice_0044|mixed|Cruce|CP True/coste0/orden1|219|195|válido|
|practice_0044|mixed|Cruce|CP True/coste1/orden1|204|195|válido|
|practice_0044|mixed|Cruce|mejor independiente/coste0/orden0|256|195|válido|
|practice_0044|mixed|Cruce|mejor independiente/coste1/orden0|232|195|válido|
|practice_0044|mixed|Cruce|mejor independiente/coste0/orden1|242|195|válido|
|practice_0044|mixed|Cruce|mejor independiente/coste1/orden1|208|195|válido|
|practice_0044|mixed|E1|cola10/pol1/inicio0|208|74|válido|
|practice_0044|mixed|E1|cola10/pol1/inicio1|208|74|válido|
|practice_0044|mixed|E1|cola10/pol2/inicio0|207|73|válido|
|practice_0044|mixed|E1|cola10/pol2/inicio1|207|73|válido|
|practice_0044|mixed|E1|cola8/pol1/inicio0|212|64|válido|
|practice_0044|mixed|E1|cola8/pol1/inicio1|212|64|válido|
|practice_0044|mixed|E1|cola8/pol2/inicio0|211|63|válido|
|practice_0044|mixed|E1|cola8/pol2/inicio1|209|61|válido|
|practice_0044|mixed|E1|cola6/pol1/inicio0|207|45|válido|
|practice_0044|mixed|E1|cola6/pol1/inicio1|207|45|válido|
|practice_0044|mixed|E1|cola6/pol2/inicio0|207|45|válido|
|practice_0044|mixed|E1|cola6/pol2/inicio1|207|45|válido|
|practice_0044|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0044|mixed|E2|CP False/coste0/orden0|225|225|válido|
|practice_0044|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0044|mixed|E2|CP True/coste0/orden0|213|196|válido|
|practice_0044|mixed|E3|donante0/coste0/orden0|220|200|válido|
|practice_0044|mixed|E3|donante0/coste1/orden0|220|200|válido|
|practice_0044|mixed|E3|donante0/coste0/orden1|248|200|válido|
|practice_0044|mixed|E3|donante0/coste1/orden1|231|200|válido|
|practice_0044|mixed|E3|donante1/coste0/orden0|233|233|válido|
|practice_0044|mixed|E3|donante1/coste1/orden0|233|233|válido|
|practice_0044|mixed|E3|donante1/coste0/orden1|233|233|válido|
|practice_0044|mixed|E3|donante1/coste1/orden1|233|233|válido|
|practice_0044|mixed|E3|donante2/coste0/orden0|216|216|válido|
|practice_0044|mixed|E3|donante2/coste1/orden0|216|216|válido|
|practice_0044|mixed|E3|donante2/coste0/orden1|216|216|válido|
|practice_0044|mixed|E3|donante2/coste1/orden1|216|216|válido|
|practice_0044|mixed|E3|donante3/coste0/orden0|369|214|válido|
|practice_0044|mixed|E3|donante3/coste1/orden0|241|214|válido|
|practice_0044|mixed|E3|donante3/coste0/orden1|214|214|válido|
|practice_0044|mixed|E3|donante3/coste1/orden1|214|214|válido|
|practice_0044|mixed|E3|donante4/coste0/orden0|238|238|válido|
|practice_0044|mixed|E3|donante4/coste1/orden0|238|238|válido|
|practice_0044|mixed|E3|donante4/coste0/orden1|238|238|válido|
|practice_0044|mixed|E3|donante4/coste1/orden1|238|238|válido|
|practice_0044|mixed|E3|donante5/coste0/orden0|226|226|válido|
|practice_0044|mixed|E3|donante5/coste1/orden0|226|226|válido|
|practice_0044|mixed|E3|donante5/coste0/orden1|226|226|válido|
|practice_0044|mixed|E3|donante5/coste1/orden1|226|226|válido|
|practice_0044|mixed|E3|donante6/coste0/orden0|232|232|válido|
|practice_0044|mixed|E3|donante6/coste1/orden0|232|232|válido|
|practice_0044|mixed|E3|donante6/coste0/orden1|232|232|válido|
|practice_0044|mixed|E3|donante6/coste1/orden1|232|232|válido|
|practice_0044|mixed|E3|donante7/coste0/orden0|244|211|válido|
|practice_0044|mixed|E3|donante7/coste1/orden0|233|211|válido|
|practice_0044|mixed|E3|donante7/coste0/orden1|241|211|válido|
|practice_0044|mixed|E3|donante7/coste1/orden1|232|211|válido|
|practice_0044|mixed|E3|donante8/coste0/orden0|221|201|válido|
|practice_0044|mixed|E3|donante8/coste1/orden0|221|201|válido|
|practice_0044|mixed|E3|donante8/coste0/orden1|249|201|válido|
|practice_0044|mixed|E3|donante8/coste1/orden1|232|201|válido|
|practice_0044|mixed|E3|donante9/coste0/orden0|286|195|válido|
|practice_0044|mixed|E3|donante9/coste1/orden0|208|195|válido|
|practice_0044|mixed|E3|donante9/coste0/orden1|242|195|válido|
|practice_0044|mixed|E3|donante9/coste1/orden1|208|195|válido|
|practice_0045|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0045|multiplicative|Cruce|CP True/coste0/orden0|138|138|válido|
|practice_0045|multiplicative|Cruce|CP True/coste1/orden0|138|138|válido|
|practice_0045|multiplicative|Cruce|CP True/coste0/orden1|138|138|válido|
|practice_0045|multiplicative|Cruce|CP True/coste1/orden1|138|138|válido|
|practice_0045|multiplicative|Cruce|mejor independiente/coste0/orden0|138|138|válido|
|practice_0045|multiplicative|Cruce|mejor independiente/coste1/orden0|138|138|válido|
|practice_0045|multiplicative|Cruce|mejor independiente/coste0/orden1|138|138|válido|
|practice_0045|multiplicative|Cruce|mejor independiente/coste1/orden1|138|138|válido|
|practice_0045|multiplicative|E1|cola10/pol1/inicio0|141|50|válido|
|practice_0045|multiplicative|E1|cola10/pol1/inicio1|141|50|válido|
|practice_0045|multiplicative|E1|cola10/pol2/inicio0|141|50|válido|
|practice_0045|multiplicative|E1|cola10/pol2/inicio1|141|50|válido|
|practice_0045|multiplicative|E1|cola8/pol1/inicio0|139|39|válido|
|practice_0045|multiplicative|E1|cola8/pol1/inicio1|139|39|válido|
|practice_0045|multiplicative|E1|cola8/pol2/inicio0|139|39|válido|
|practice_0045|multiplicative|E1|cola8/pol2/inicio1|139|39|válido|
|practice_0045|multiplicative|E1|cola6/pol1/inicio0|141|31|válido|
|practice_0045|multiplicative|E1|cola6/pol1/inicio1|141|31|válido|
|practice_0045|multiplicative|E1|cola6/pol2/inicio0|141|31|válido|
|practice_0045|multiplicative|E1|cola6/pol2/inicio1|141|31|válido|
|practice_0045|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0045|multiplicative|E2|CP False/coste0/orden0|141|141|válido|
|practice_0045|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0045|multiplicative|E2|CP True/coste0/orden0|141|141|válido|
|practice_0045|multiplicative|E3|donante0/coste0/orden0|141|141|válido|
|practice_0045|multiplicative|E3|donante0/coste1/orden0|141|141|válido|
|practice_0045|multiplicative|E3|donante0/coste0/orden1|141|141|válido|
|practice_0045|multiplicative|E3|donante0/coste1/orden1|141|141|válido|
|practice_0045|multiplicative|E3|donante1/coste0/orden0|150|150|válido|
|practice_0045|multiplicative|E3|donante1/coste1/orden0|150|150|válido|
|practice_0045|multiplicative|E3|donante1/coste0/orden1|150|150|válido|
|practice_0045|multiplicative|E3|donante1/coste1/orden1|150|150|válido|
|practice_0045|multiplicative|E3|donante2/coste0/orden0|148|148|válido|
|practice_0045|multiplicative|E3|donante2/coste1/orden0|148|148|válido|
|practice_0045|multiplicative|E3|donante2/coste0/orden1|148|148|válido|
|practice_0045|multiplicative|E3|donante2/coste1/orden1|148|148|válido|
|practice_0045|multiplicative|E3|donante3/coste0/orden0|155|155|válido|
|practice_0045|multiplicative|E3|donante3/coste1/orden0|155|155|válido|
|practice_0045|multiplicative|E3|donante3/coste0/orden1|155|155|válido|
|practice_0045|multiplicative|E3|donante3/coste1/orden1|155|155|válido|
|practice_0045|multiplicative|E3|donante4/coste0/orden0|157|157|válido|
|practice_0045|multiplicative|E3|donante4/coste1/orden0|157|157|válido|
|practice_0045|multiplicative|E3|donante4/coste0/orden1|157|157|válido|
|practice_0045|multiplicative|E3|donante4/coste1/orden1|157|157|válido|
|practice_0045|multiplicative|E3|donante5/coste0/orden0|153|153|válido|
|practice_0045|multiplicative|E3|donante5/coste1/orden0|153|153|válido|
|practice_0045|multiplicative|E3|donante5/coste0/orden1|153|153|válido|
|practice_0045|multiplicative|E3|donante5/coste1/orden1|153|153|válido|
|practice_0045|multiplicative|E3|donante6/coste0/orden0|162|141|válido|
|practice_0045|multiplicative|E3|donante6/coste1/orden0|155|141|válido|
|practice_0045|multiplicative|E3|donante6/coste0/orden1|169|141|válido|
|practice_0045|multiplicative|E3|donante6/coste1/orden1|146|141|válido|
|practice_0045|multiplicative|E3|donante7/coste0/orden0|157|136|válido|
|practice_0045|multiplicative|E3|donante7/coste1/orden0|155|136|válido|
|practice_0045|multiplicative|E3|donante7/coste0/orden1|149|136|válido|
|practice_0045|multiplicative|E3|donante7/coste1/orden1|155|136|válido|
|practice_0045|multiplicative|E3|donante8/coste0/orden0|143|139|válido|
|practice_0045|multiplicative|E3|donante8/coste1/orden0|151|139|válido|
|practice_0045|multiplicative|E3|donante8/coste0/orden1|142|139|válido|
|practice_0045|multiplicative|E3|donante8/coste1/orden1|142|139|válido|
|practice_0046|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0046|independent|Cruce|CP True/coste0/orden0|263|240|válido|
|practice_0046|independent|Cruce|CP True/coste1/orden0|257|240|válido|
|practice_0046|independent|Cruce|CP True/coste0/orden1|240|240|válido|
|practice_0046|independent|Cruce|CP True/coste1/orden1|240|240|válido|
|practice_0046|independent|Cruce|mejor independiente/coste0/orden0|263|240|válido|
|practice_0046|independent|Cruce|mejor independiente/coste1/orden0|257|240|válido|
|practice_0046|independent|Cruce|mejor independiente/coste0/orden1|240|240|válido|
|practice_0046|independent|Cruce|mejor independiente/coste1/orden1|240|240|válido|
|practice_0046|independent|E1|cola10/pol1/inicio0|250|77|válido|
|practice_0046|independent|E1|cola10/pol1/inicio1|250|77|válido|
|practice_0046|independent|E1|cola10/pol2/inicio0|249|76|válido|
|practice_0046|independent|E1|cola10/pol2/inicio1|249|76|válido|
|practice_0046|independent|E1|cola8/pol1/inicio0|253|66|válido|
|practice_0046|independent|E1|cola8/pol1/inicio1|253|66|válido|
|practice_0046|independent|E1|cola8/pol2/inicio0|251|64|válido|
|practice_0046|independent|E1|cola8/pol2/inicio1|247|60|válido|
|practice_0046|independent|E1|cola6/pol1/inicio0|248|47|válido|
|practice_0046|independent|E1|cola6/pol1/inicio1|248|47|válido|
|practice_0046|independent|E1|cola6/pol2/inicio0|248|47|válido|
|practice_0046|independent|E1|cola6/pol2/inicio1|248|47|válido|
|practice_0046|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0046|independent|E2|CP False/coste0/orden0|406|273|válido|
|practice_0046|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0046|independent|E2|CP True/coste0/orden0|263|240|válido|
|practice_0046|independent|E3|donante0/coste0/orden0|240|240|válido|
|practice_0046|independent|E3|donante0/coste1/orden0|240|240|válido|
|practice_0046|independent|E3|donante0/coste0/orden1|240|240|válido|
|practice_0046|independent|E3|donante0/coste1/orden1|240|240|válido|
|practice_0046|independent|E3|donante1/coste0/orden0|271|271|válido|
|practice_0046|independent|E3|donante1/coste1/orden0|271|271|válido|
|practice_0046|independent|E3|donante1/coste0/orden1|271|271|válido|
|practice_0046|independent|E3|donante1/coste1/orden1|271|271|válido|
|practice_0046|independent|E3|donante2/coste0/orden0|304|264|válido|
|practice_0046|independent|E3|donante2/coste1/orden0|271|264|válido|
|practice_0046|independent|E3|donante2/coste0/orden1|264|264|válido|
|practice_0046|independent|E3|donante2/coste1/orden1|264|264|válido|
|practice_0046|independent|E3|donante3/coste0/orden0|265|253|válido|
|practice_0046|independent|E3|donante3/coste1/orden0|265|253|válido|
|practice_0046|independent|E3|donante3/coste0/orden1|253|253|válido|
|practice_0046|independent|E3|donante3/coste1/orden1|253|253|válido|
|practice_0046|independent|E3|donante4/coste0/orden0|287|287|válido|
|practice_0046|independent|E3|donante4/coste1/orden0|287|287|válido|
|practice_0046|independent|E3|donante4/coste0/orden1|287|287|válido|
|practice_0046|independent|E3|donante4/coste1/orden1|287|287|válido|
|practice_0046|independent|E3|donante5/coste0/orden0|273|273|válido|
|practice_0046|independent|E3|donante5/coste1/orden0|273|273|válido|
|practice_0046|independent|E3|donante5/coste0/orden1|273|273|válido|
|practice_0046|independent|E3|donante5/coste1/orden1|273|273|válido|
|practice_0046|independent|E3|donante6/coste0/orden0|281|281|válido|
|practice_0046|independent|E3|donante6/coste1/orden0|281|281|válido|
|practice_0046|independent|E3|donante6/coste0/orden1|281|281|válido|
|practice_0046|independent|E3|donante6/coste1/orden1|281|281|válido|
|practice_0046|independent|E3|donante7/coste0/orden0|270|246|válido|
|practice_0046|independent|E3|donante7/coste1/orden0|262|246|válido|
|practice_0046|independent|E3|donante7/coste0/orden1|273|246|válido|
|practice_0046|independent|E3|donante7/coste1/orden1|269|246|válido|
|practice_0046|independent|E3|donante8/coste0/orden0|243|243|válido|
|practice_0046|independent|E3|donante8/coste1/orden0|243|243|válido|
|practice_0046|independent|E3|donante8/coste0/orden1|243|243|válido|
|practice_0046|independent|E3|donante8/coste1/orden1|243|243|válido|
|practice_0046|independent|E3|donante9/coste0/orden0|244|239|válido|
|practice_0046|independent|E3|donante9/coste1/orden0|241|239|válido|
|practice_0046|independent|E3|donante9/coste0/orden1|245|239|válido|
|practice_0046|independent|E3|donante9/coste1/orden1|245|239|válido|
|practice_0047|tiny|Cruce|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0047|tiny|Cruce|CP True/coste0/orden0|2|2|válido|
|practice_0047|tiny|Cruce|CP True/coste1/orden0|2|2|válido|
|practice_0047|tiny|Cruce|CP True/coste0/orden1|2|2|válido|
|practice_0047|tiny|Cruce|CP True/coste1/orden1|2|2|válido|
|practice_0047|tiny|Cruce|mejor independiente/coste0/orden0|2|2|válido|
|practice_0047|tiny|Cruce|mejor independiente/coste1/orden0|2|2|válido|
|practice_0047|tiny|Cruce|mejor independiente/coste0/orden1|2|2|válido|
|practice_0047|tiny|Cruce|mejor independiente/coste1/orden1|2|2|válido|
|practice_0047|tiny|E1|cola2/pol1/inicio0|5|5|válido|
|practice_0047|tiny|E1|cola2/pol1/inicio1|5|5|válido|
|practice_0047|tiny|E1|cola2/pol2/inicio0|5|5|válido|
|practice_0047|tiny|E1|cola2/pol2/inicio1|5|5|válido|
|practice_0047|tiny|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0047|tiny|E2|CP False/coste0/orden0|2|2|válido|
|practice_0047|tiny|E2|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0047|tiny|E2|CP True/coste0/orden0|2|2|válido|
|practice_0047|tiny|E3|donante0/coste0/orden0|2|2|válido|
|practice_0047|tiny|E3|donante0/coste1/orden0|2|2|válido|
|practice_0047|tiny|E3|donante0/coste0/orden1|2|2|válido|
|practice_0047|tiny|E3|donante0/coste1/orden1|2|2|válido|
|practice_0047|tiny|E3|donante1/coste0/orden0|5|5|válido|
|practice_0047|tiny|E3|donante1/coste1/orden0|5|5|válido|
|practice_0047|tiny|E3|donante1/coste0/orden1|5|5|válido|
|practice_0047|tiny|E3|donante1/coste1/orden1|5|5|válido|
|practice_0048|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0048|multiplicative|Cruce|CP True/coste0/orden0|114|114|válido|
|practice_0048|multiplicative|Cruce|CP True/coste1/orden0|114|114|válido|
|practice_0048|multiplicative|Cruce|CP True/coste0/orden1|114|114|válido|
|practice_0048|multiplicative|Cruce|CP True/coste1/orden1|114|114|válido|
|practice_0048|multiplicative|Cruce|mejor independiente/coste0/orden0|116|116|válido|
|practice_0048|multiplicative|Cruce|mejor independiente/coste1/orden0|116|116|válido|
|practice_0048|multiplicative|Cruce|mejor independiente/coste0/orden1|116|116|válido|
|practice_0048|multiplicative|Cruce|mejor independiente/coste1/orden1|116|116|válido|
|practice_0048|multiplicative|E1|cola10/pol1/inicio0|116|32|válido|
|practice_0048|multiplicative|E1|cola10/pol1/inicio1|116|32|válido|
|practice_0048|multiplicative|E1|cola10/pol2/inicio0|116|32|válido|
|practice_0048|multiplicative|E1|cola10/pol2/inicio1|116|32|válido|
|practice_0048|multiplicative|E1|cola8/pol1/inicio0|116|26|válido|
|practice_0048|multiplicative|E1|cola8/pol1/inicio1|116|26|válido|
|practice_0048|multiplicative|E1|cola8/pol2/inicio0|116|26|válido|
|practice_0048|multiplicative|E1|cola8/pol2/inicio1|116|26|válido|
|practice_0048|multiplicative|E1|cola6/pol1/inicio0|116|20|válido|
|practice_0048|multiplicative|E1|cola6/pol1/inicio1|116|20|válido|
|practice_0048|multiplicative|E1|cola6/pol2/inicio0|116|20|válido|
|practice_0048|multiplicative|E1|cola6/pol2/inicio1|116|20|válido|
|practice_0048|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0048|multiplicative|E2|CP False/coste0/orden0|124|124|válido|
|practice_0048|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0048|multiplicative|E2|CP True/coste0/orden0|116|116|válido|
|practice_0048|multiplicative|E3|donante0/coste0/orden0|116|116|válido|
|practice_0048|multiplicative|E3|donante0/coste1/orden0|116|116|válido|
|practice_0048|multiplicative|E3|donante0/coste0/orden1|116|116|válido|
|practice_0048|multiplicative|E3|donante0/coste1/orden1|116|116|válido|
|practice_0048|multiplicative|E3|donante1/coste0/orden0|124|124|válido|
|practice_0048|multiplicative|E3|donante1/coste1/orden0|124|124|válido|
|practice_0048|multiplicative|E3|donante1/coste0/orden1|124|124|válido|
|practice_0048|multiplicative|E3|donante1/coste1/orden1|124|124|válido|
|practice_0048|multiplicative|E3|donante2/coste0/orden0|125|125|válido|
|practice_0048|multiplicative|E3|donante2/coste1/orden0|125|125|válido|
|practice_0048|multiplicative|E3|donante2/coste0/orden1|125|125|válido|
|practice_0048|multiplicative|E3|donante2/coste1/orden1|125|125|válido|
|practice_0048|multiplicative|E3|donante3/coste0/orden0|120|120|válido|
|practice_0048|multiplicative|E3|donante3/coste1/orden0|120|120|válido|
|practice_0048|multiplicative|E3|donante3/coste0/orden1|120|120|válido|
|practice_0048|multiplicative|E3|donante3/coste1/orden1|120|120|válido|
|practice_0048|multiplicative|E3|donante4/coste0/orden0|132|132|válido|
|practice_0048|multiplicative|E3|donante4/coste1/orden0|132|132|válido|
|practice_0048|multiplicative|E3|donante4/coste0/orden1|132|132|válido|
|practice_0048|multiplicative|E3|donante4/coste1/orden1|132|132|válido|
|practice_0048|multiplicative|E3|donante5/coste0/orden0|128|128|válido|
|practice_0048|multiplicative|E3|donante5/coste1/orden0|128|128|válido|
|practice_0048|multiplicative|E3|donante5/coste0/orden1|128|128|válido|
|practice_0048|multiplicative|E3|donante5/coste1/orden1|128|128|válido|
|practice_0048|multiplicative|E3|donante6/coste0/orden0|124|124|válido|
|practice_0048|multiplicative|E3|donante6/coste1/orden0|124|124|válido|
|practice_0048|multiplicative|E3|donante6/coste0/orden1|124|124|válido|
|practice_0048|multiplicative|E3|donante6/coste1/orden1|124|124|válido|
|practice_0048|multiplicative|E3|donante7/coste0/orden0|123|123|válido|
|practice_0048|multiplicative|E3|donante7/coste1/orden0|123|123|válido|
|practice_0048|multiplicative|E3|donante7/coste0/orden1|123|123|válido|
|practice_0048|multiplicative|E3|donante7/coste1/orden1|123|123|válido|
|practice_0048|multiplicative|E3|donante8/coste0/orden0|123|123|válido|
|practice_0048|multiplicative|E3|donante8/coste1/orden0|123|123|válido|
|practice_0048|multiplicative|E3|donante8/coste0/orden1|123|123|válido|
|practice_0048|multiplicative|E3|donante8/coste1/orden1|123|123|válido|
|practice_0048|multiplicative|E3|donante9/coste0/orden0|117|117|válido|
|practice_0048|multiplicative|E3|donante9/coste1/orden0|117|117|válido|
|practice_0048|multiplicative|E3|donante9/coste0/orden1|117|117|válido|
|practice_0048|multiplicative|E3|donante9/coste1/orden1|117|117|válido|
|practice_0049|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0049|multiplicative|Cruce|CP True/coste0/orden0|122|122|válido|
|practice_0049|multiplicative|Cruce|CP True/coste1/orden0|122|122|válido|
|practice_0049|multiplicative|Cruce|CP True/coste0/orden1|122|122|válido|
|practice_0049|multiplicative|Cruce|CP True/coste1/orden1|122|122|válido|
|practice_0049|multiplicative|Cruce|mejor independiente/coste0/orden0|123|123|válido|
|practice_0049|multiplicative|Cruce|mejor independiente/coste1/orden0|123|123|válido|
|practice_0049|multiplicative|Cruce|mejor independiente/coste0/orden1|123|123|válido|
|practice_0049|multiplicative|Cruce|mejor independiente/coste1/orden1|123|123|válido|
|practice_0049|multiplicative|E1|cola10/pol1/inicio0|129|42|válido|
|practice_0049|multiplicative|E1|cola10/pol1/inicio1|129|42|válido|
|practice_0049|multiplicative|E1|cola10/pol2/inicio0|129|42|válido|
|practice_0049|multiplicative|E1|cola10/pol2/inicio1|129|42|válido|
|practice_0049|multiplicative|E1|cola8/pol1/inicio0|129|35|válido|
|practice_0049|multiplicative|E1|cola8/pol1/inicio1|129|35|válido|
|practice_0049|multiplicative|E1|cola8/pol2/inicio0|129|35|válido|
|practice_0049|multiplicative|E1|cola8/pol2/inicio1|129|35|válido|
|practice_0049|multiplicative|E1|cola6/pol1/inicio0|129|28|válido|
|practice_0049|multiplicative|E1|cola6/pol1/inicio1|129|28|válido|
|practice_0049|multiplicative|E1|cola6/pol2/inicio0|129|28|válido|
|practice_0049|multiplicative|E1|cola6/pol2/inicio1|129|28|válido|
|practice_0049|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0049|multiplicative|E2|CP False/coste0/orden0|124|124|válido|
|practice_0049|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0049|multiplicative|E2|CP True/coste0/orden0|123|123|válido|
|practice_0049|multiplicative|E3|donante0/coste0/orden0|124|123|válido|
|practice_0049|multiplicative|E3|donante0/coste1/orden0|124|123|válido|
|practice_0049|multiplicative|E3|donante0/coste0/orden1|123|123|válido|
|practice_0049|multiplicative|E3|donante0/coste1/orden1|123|123|válido|
|practice_0049|multiplicative|E3|donante1/coste0/orden0|131|131|válido|
|practice_0049|multiplicative|E3|donante1/coste1/orden0|131|131|válido|
|practice_0049|multiplicative|E3|donante1/coste0/orden1|131|131|válido|
|practice_0049|multiplicative|E3|donante1/coste1/orden1|131|131|válido|
|practice_0049|multiplicative|E3|donante2/coste0/orden0|128|128|válido|
|practice_0049|multiplicative|E3|donante2/coste1/orden0|128|128|válido|
|practice_0049|multiplicative|E3|donante2/coste0/orden1|128|128|válido|
|practice_0049|multiplicative|E3|donante2/coste1/orden1|128|128|válido|
|practice_0049|multiplicative|E3|donante3/coste0/orden0|126|126|válido|
|practice_0049|multiplicative|E3|donante3/coste1/orden0|126|126|válido|
|practice_0049|multiplicative|E3|donante3/coste0/orden1|126|126|válido|
|practice_0049|multiplicative|E3|donante3/coste1/orden1|126|126|válido|
|practice_0049|multiplicative|E3|donante4/coste0/orden0|140|140|válido|
|practice_0049|multiplicative|E3|donante4/coste1/orden0|140|140|válido|
|practice_0049|multiplicative|E3|donante4/coste0/orden1|140|140|válido|
|practice_0049|multiplicative|E3|donante4/coste1/orden1|140|140|válido|
|practice_0049|multiplicative|E3|donante5/coste0/orden0|135|135|válido|
|practice_0049|multiplicative|E3|donante5/coste1/orden0|135|135|válido|
|practice_0049|multiplicative|E3|donante5/coste0/orden1|135|135|válido|
|practice_0049|multiplicative|E3|donante5/coste1/orden1|135|135|válido|
|practice_0049|multiplicative|E3|donante6/coste0/orden0|134|134|válido|
|practice_0049|multiplicative|E3|donante6/coste1/orden0|134|134|válido|
|practice_0049|multiplicative|E3|donante6/coste0/orden1|134|134|válido|
|practice_0049|multiplicative|E3|donante6/coste1/orden1|134|134|válido|
|practice_0049|multiplicative|E3|donante7/coste0/orden0|129|128|válido|
|practice_0049|multiplicative|E3|donante7/coste1/orden0|129|128|válido|
|practice_0049|multiplicative|E3|donante7/coste0/orden1|128|128|válido|
|practice_0049|multiplicative|E3|donante7/coste1/orden1|128|128|válido|
|practice_0049|multiplicative|E3|donante8/coste0/orden0|128|128|válido|
|practice_0049|multiplicative|E3|donante8/coste1/orden0|128|128|válido|
|practice_0049|multiplicative|E3|donante8/coste0/orden1|128|128|válido|
|practice_0049|multiplicative|E3|donante8/coste1/orden1|128|128|válido|
|practice_0049|multiplicative|E3|donante9/coste0/orden0|126|125|válido|
|practice_0049|multiplicative|E3|donante9/coste1/orden0|126|125|válido|
|practice_0049|multiplicative|E3|donante9/coste0/orden1|125|125|válido|
|practice_0049|multiplicative|E3|donante9/coste1/orden1|125|125|válido|
|practice_0050|tiny|Cruce|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0050|tiny|Cruce|CP True/coste0/orden0|3|3|válido|
|practice_0050|tiny|Cruce|CP True/coste1/orden0|3|3|válido|
|practice_0050|tiny|Cruce|CP True/coste0/orden1|3|3|válido|
|practice_0050|tiny|Cruce|CP True/coste1/orden1|3|3|válido|
|practice_0050|tiny|Cruce|mejor independiente/coste0/orden0|3|3|válido|
|practice_0050|tiny|Cruce|mejor independiente/coste1/orden0|3|3|válido|
|practice_0050|tiny|Cruce|mejor independiente/coste0/orden1|3|3|válido|
|practice_0050|tiny|Cruce|mejor independiente/coste1/orden1|3|3|válido|
|practice_0050|tiny|E1|cola2/pol1/inicio0|5|4|válido|
|practice_0050|tiny|E1|cola2/pol1/inicio1|5|4|válido|
|practice_0050|tiny|E1|cola2/pol2/inicio0|5|4|válido|
|practice_0050|tiny|E1|cola2/pol2/inicio1|5|4|válido|
|practice_0050|tiny|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0050|tiny|E2|CP False/coste0/orden0|3|3|válido|
|practice_0050|tiny|E2|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0050|tiny|E2|CP True/coste0/orden0|3|3|válido|
|practice_0050|tiny|E3|donante0/coste0/orden0|3|3|válido|
|practice_0050|tiny|E3|donante0/coste1/orden0|3|3|válido|
|practice_0050|tiny|E3|donante0/coste0/orden1|3|3|válido|
|practice_0050|tiny|E3|donante0/coste1/orden1|3|3|válido|
|practice_0050|tiny|E3|donante1/coste0/orden0|5|5|válido|
|practice_0050|tiny|E3|donante1/coste1/orden0|5|5|válido|
|practice_0050|tiny|E3|donante1/coste0/orden1|5|5|válido|
|practice_0050|tiny|E3|donante1/coste1/orden1|5|5|válido|
|practice_0051|tiny|Cruce|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0051|tiny|Cruce|CP True/coste0/orden0|5|5|válido|
|practice_0051|tiny|Cruce|CP True/coste1/orden0|5|5|válido|
|practice_0051|tiny|Cruce|CP True/coste0/orden1|5|5|válido|
|practice_0051|tiny|Cruce|CP True/coste1/orden1|5|5|válido|
|practice_0051|tiny|Cruce|mejor independiente/coste0/orden0|5|5|válido|
|practice_0051|tiny|Cruce|mejor independiente/coste1/orden0|5|5|válido|
|practice_0051|tiny|Cruce|mejor independiente/coste0/orden1|5|5|válido|
|practice_0051|tiny|Cruce|mejor independiente/coste1/orden1|5|5|válido|
|practice_0051|tiny|E1|cola4/pol1/inicio0|7|6|válido|
|practice_0051|tiny|E1|cola4/pol1/inicio1|7|6|válido|
|practice_0051|tiny|E1|cola4/pol2/inicio0|7|6|válido|
|practice_0051|tiny|E1|cola4/pol2/inicio1|7|6|válido|
|practice_0051|tiny|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0051|tiny|E2|CP False/coste0/orden0|5|5|válido|
|practice_0051|tiny|E2|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0051|tiny|E2|CP True/coste0/orden0|5|5|válido|
|practice_0051|tiny|E3|donante0/coste0/orden0|5|5|válido|
|practice_0051|tiny|E3|donante0/coste1/orden0|5|5|válido|
|practice_0051|tiny|E3|donante0/coste0/orden1|5|5|válido|
|practice_0051|tiny|E3|donante0/coste1/orden1|5|5|válido|
|practice_0051|tiny|E3|donante1/coste0/orden0|7|7|válido|
|practice_0051|tiny|E3|donante1/coste1/orden0|7|7|válido|
|practice_0051|tiny|E3|donante1/coste0/orden1|7|7|válido|
|practice_0051|tiny|E3|donante1/coste1/orden1|7|7|válido|
|practice_0052|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0052|independent|Cruce|CP True/coste0/orden0|223|184|válido|
|practice_0052|independent|Cruce|CP True/coste1/orden0|235|184|válido|
|practice_0052|independent|Cruce|CP True/coste0/orden1|212|184|válido|
|practice_0052|independent|Cruce|CP True/coste1/orden1|196|184|válido|
|practice_0052|independent|Cruce|mejor independiente/coste0/orden0|219|185|válido|
|practice_0052|independent|Cruce|mejor independiente/coste1/orden0|212|185|válido|
|practice_0052|independent|Cruce|mejor independiente/coste0/orden1|203|185|válido|
|practice_0052|independent|Cruce|mejor independiente/coste1/orden1|203|185|válido|
|practice_0052|independent|E1|cola10/pol1/inicio0|200|52|válido|
|practice_0052|independent|E1|cola10/pol1/inicio1|200|52|válido|
|practice_0052|independent|E1|cola10/pol2/inicio0|202|54|válido|
|practice_0052|independent|E1|cola10/pol2/inicio1|202|54|válido|
|practice_0052|independent|E1|cola8/pol1/inicio0|201|41|válido|
|practice_0052|independent|E1|cola8/pol1/inicio1|201|41|válido|
|practice_0052|independent|E1|cola8/pol2/inicio0|202|42|válido|
|practice_0052|independent|E1|cola8/pol2/inicio1|202|42|válido|
|practice_0052|independent|E1|cola6/pol1/inicio0|203|32|válido|
|practice_0052|independent|E1|cola6/pol1/inicio1|203|32|válido|
|practice_0052|independent|E1|cola6/pol2/inicio0|202|31|válido|
|practice_0052|independent|E1|cola6/pol2/inicio1|202|31|válido|
|practice_0052|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0052|independent|E2|CP False/coste0/orden0|213|213|válido|
|practice_0052|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0052|independent|E2|CP True/coste0/orden0|199|190|válido|
|practice_0052|independent|E3|donante0/coste0/orden0|195|191|válido|
|practice_0052|independent|E3|donante0/coste1/orden0|195|191|válido|
|practice_0052|independent|E3|donante0/coste0/orden1|191|191|válido|
|practice_0052|independent|E3|donante0/coste1/orden1|191|191|válido|
|practice_0052|independent|E3|donante1/coste0/orden0|214|214|válido|
|practice_0052|independent|E3|donante1/coste1/orden0|214|214|válido|
|practice_0052|independent|E3|donante1/coste0/orden1|214|214|válido|
|practice_0052|independent|E3|donante1/coste1/orden1|214|214|válido|
|practice_0052|independent|E3|donante2/coste0/orden0|234|201|válido|
|practice_0052|independent|E3|donante2/coste1/orden0|203|201|válido|
|practice_0052|independent|E3|donante2/coste0/orden1|201|201|válido|
|practice_0052|independent|E3|donante2/coste1/orden1|201|201|válido|
|practice_0052|independent|E3|donante3/coste0/orden0|201|192|válido|
|practice_0052|independent|E3|donante3/coste1/orden0|201|192|válido|
|practice_0052|independent|E3|donante3/coste0/orden1|192|192|válido|
|practice_0052|independent|E3|donante3/coste1/orden1|192|192|válido|
|practice_0052|independent|E3|donante4/coste0/orden0|209|209|válido|
|practice_0052|independent|E3|donante4/coste1/orden0|209|209|válido|
|practice_0052|independent|E3|donante4/coste0/orden1|209|209|válido|
|practice_0052|independent|E3|donante4/coste1/orden1|209|209|válido|
|practice_0052|independent|E3|donante5/coste0/orden0|207|207|válido|
|practice_0052|independent|E3|donante5/coste1/orden0|207|207|válido|
|practice_0052|independent|E3|donante5/coste0/orden1|207|207|válido|
|practice_0052|independent|E3|donante5/coste1/orden1|207|207|válido|
|practice_0052|independent|E3|donante6/coste0/orden0|213|213|válido|
|practice_0052|independent|E3|donante6/coste1/orden0|213|213|válido|
|practice_0052|independent|E3|donante6/coste0/orden1|213|213|válido|
|practice_0052|independent|E3|donante6/coste1/orden1|213|213|válido|
|practice_0052|independent|E3|donante7/coste0/orden0|230|198|válido|
|practice_0052|independent|E3|donante7/coste1/orden0|227|198|válido|
|practice_0052|independent|E3|donante7/coste0/orden1|226|198|válido|
|practice_0052|independent|E3|donante7/coste1/orden1|204|198|válido|
|practice_0052|independent|E3|donante8/coste0/orden0|198|189|válido|
|practice_0052|independent|E3|donante8/coste1/orden0|194|189|válido|
|practice_0052|independent|E3|donante8/coste0/orden1|208|189|válido|
|practice_0052|independent|E3|donante8/coste1/orden1|204|189|válido|
|practice_0052|independent|E3|donante9/coste0/orden0|203|186|válido|
|practice_0052|independent|E3|donante9/coste1/orden0|199|186|válido|
|practice_0052|independent|E3|donante9/coste0/orden1|196|186|válido|
|practice_0052|independent|E3|donante9/coste1/orden1|200|186|válido|
|practice_0053|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0053|mixed|Cruce|CP True/coste0/orden0|238|230|válido|
|practice_0053|mixed|Cruce|CP True/coste1/orden0|238|230|válido|
|practice_0053|mixed|Cruce|CP True/coste0/orden1|231|230|válido|
|practice_0053|mixed|Cruce|CP True/coste1/orden1|231|230|válido|
|practice_0053|mixed|Cruce|mejor independiente/coste0/orden0|240|231|válido|
|practice_0053|mixed|Cruce|mejor independiente/coste1/orden0|240|231|válido|
|practice_0053|mixed|Cruce|mejor independiente/coste0/orden1|232|231|válido|
|practice_0053|mixed|Cruce|mejor independiente/coste1/orden1|232|231|válido|
|practice_0053|mixed|E1|cola10/pol1/inicio0|237|86|válido|
|practice_0053|mixed|E1|cola10/pol1/inicio1|237|86|válido|
|practice_0053|mixed|E1|cola10/pol2/inicio0|238|87|válido|
|practice_0053|mixed|E1|cola10/pol2/inicio1|241|90|válido|
|practice_0053|mixed|E1|cola8/pol1/inicio0|241|73|válido|
|practice_0053|mixed|E1|cola8/pol1/inicio1|241|73|válido|
|practice_0053|mixed|E1|cola8/pol2/inicio0|239|71|válido|
|practice_0053|mixed|E1|cola8/pol2/inicio1|239|71|válido|
|practice_0053|mixed|E1|cola6/pol1/inicio0|237|51|válido|
|practice_0053|mixed|E1|cola6/pol1/inicio1|242|56|válido|
|practice_0053|mixed|E1|cola6/pol2/inicio0|237|51|válido|
|practice_0053|mixed|E1|cola6/pol2/inicio1|237|51|válido|
|practice_0053|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0053|mixed|E2|CP False/coste0/orden0|240|231|válido|
|practice_0053|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0053|mixed|E2|CP True/coste0/orden0|238|230|válido|
|practice_0053|mixed|E3|donante0/coste0/orden0|234|231|válido|
|practice_0053|mixed|E3|donante0/coste1/orden0|232|231|válido|
|practice_0053|mixed|E3|donante0/coste0/orden1|232|231|válido|
|practice_0053|mixed|E3|donante0/coste1/orden1|232|231|válido|
|practice_0053|mixed|E3|donante1/coste0/orden0|267|267|válido|
|practice_0053|mixed|E3|donante1/coste1/orden0|267|267|válido|
|practice_0053|mixed|E3|donante1/coste0/orden1|267|267|válido|
|practice_0053|mixed|E3|donante1/coste1/orden1|267|267|válido|
|practice_0053|mixed|E3|donante2/coste0/orden0|255|255|válido|
|practice_0053|mixed|E3|donante2/coste1/orden0|255|255|válido|
|practice_0053|mixed|E3|donante2/coste0/orden1|255|255|válido|
|practice_0053|mixed|E3|donante2/coste1/orden1|255|255|válido|
|practice_0053|mixed|E3|donante3/coste0/orden0|250|245|válido|
|practice_0053|mixed|E3|donante3/coste1/orden0|250|245|válido|
|practice_0053|mixed|E3|donante3/coste0/orden1|245|245|válido|
|practice_0053|mixed|E3|donante3/coste1/orden1|245|245|válido|
|practice_0053|mixed|E3|donante4/coste0/orden0|272|272|válido|
|practice_0053|mixed|E3|donante4/coste1/orden0|272|272|válido|
|practice_0053|mixed|E3|donante4/coste0/orden1|272|272|válido|
|practice_0053|mixed|E3|donante4/coste1/orden1|272|272|válido|
|practice_0053|mixed|E3|donante5/coste0/orden0|264|264|válido|
|practice_0053|mixed|E3|donante5/coste1/orden0|264|264|válido|
|practice_0053|mixed|E3|donante5/coste0/orden1|264|264|válido|
|practice_0053|mixed|E3|donante5/coste1/orden1|264|264|válido|
|practice_0053|mixed|E3|donante6/coste0/orden0|263|263|válido|
|practice_0053|mixed|E3|donante6/coste1/orden0|263|263|válido|
|practice_0053|mixed|E3|donante6/coste0/orden1|263|263|válido|
|practice_0053|mixed|E3|donante6/coste1/orden1|263|263|válido|
|practice_0053|mixed|E3|donante7/coste0/orden0|264|236|válido|
|practice_0053|mixed|E3|donante7/coste1/orden0|251|236|válido|
|practice_0053|mixed|E3|donante7/coste0/orden1|265|236|válido|
|practice_0053|mixed|E3|donante7/coste1/orden1|242|236|válido|
|practice_0053|mixed|E3|donante8/coste0/orden0|235|232|válido|
|practice_0053|mixed|E3|donante8/coste1/orden0|233|232|válido|
|practice_0053|mixed|E3|donante8/coste0/orden1|233|232|válido|
|practice_0053|mixed|E3|donante8/coste1/orden1|233|232|válido|
|practice_0053|mixed|E3|donante9/coste0/orden0|237|229|válido|
|practice_0053|mixed|E3|donante9/coste1/orden0|237|229|válido|
|practice_0053|mixed|E3|donante9/coste0/orden1|259|229|válido|
|practice_0053|mixed|E3|donante9/coste1/orden1|249|229|válido|
|practice_0054|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0054|independent|Cruce|CP True/coste0/orden0|412|301|válido|
|practice_0054|independent|Cruce|CP True/coste1/orden0|349|301|válido|
|practice_0054|independent|Cruce|CP True/coste0/orden1|433|301|válido|
|practice_0054|independent|Cruce|CP True/coste1/orden1|316|301|válido|
|practice_0054|independent|Cruce|mejor independiente/coste0/orden0|412|301|válido|
|practice_0054|independent|Cruce|mejor independiente/coste1/orden0|349|301|válido|
|practice_0054|independent|Cruce|mejor independiente/coste0/orden1|433|301|válido|
|practice_0054|independent|Cruce|mejor independiente/coste1/orden1|316|301|válido|
|practice_0054|independent|E1|cola10/pol1/inicio0|315|91|válido|
|practice_0054|independent|E1|cola10/pol1/inicio1|315|91|válido|
|practice_0054|independent|E1|cola10/pol2/inicio0|317|93|válido|
|practice_0054|independent|E1|cola10/pol2/inicio1|317|93|válido|
|practice_0054|independent|E1|cola8/pol1/inicio0|312|71|válido|
|practice_0054|independent|E1|cola8/pol1/inicio1|318|77|válido|
|practice_0054|independent|E1|cola8/pol2/inicio0|309|68|válido|
|practice_0054|independent|E1|cola8/pol2/inicio1|313|72|válido|
|practice_0054|independent|E1|cola6/pol1/inicio0|315|57|válido|
|practice_0054|independent|E1|cola6/pol1/inicio1|315|57|válido|
|practice_0054|independent|E1|cola6/pol2/inicio0|315|57|válido|
|practice_0054|independent|E1|cola6/pol2/inicio1|315|57|válido|
|practice_0054|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0054|independent|E2|CP False/coste0/orden0|449|302|válido|
|practice_0054|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0054|independent|E2|CP True/coste0/orden0|412|301|válido|
|practice_0054|independent|E3|donante0/coste0/orden0|301|301|válido|
|practice_0054|independent|E3|donante0/coste1/orden0|301|301|válido|
|practice_0054|independent|E3|donante0/coste0/orden1|433|301|válido|
|practice_0054|independent|E3|donante0/coste1/orden1|316|301|válido|
|practice_0054|independent|E3|donante1/coste0/orden0|346|346|válido|
|practice_0054|independent|E3|donante1/coste1/orden0|346|346|válido|
|practice_0054|independent|E3|donante1/coste0/orden1|346|346|válido|
|practice_0054|independent|E3|donante1/coste1/orden1|346|346|válido|
|practice_0054|independent|E3|donante2/coste0/orden0|472|330|válido|
|practice_0054|independent|E3|donante2/coste1/orden0|342|330|válido|
|practice_0054|independent|E3|donante2/coste0/orden1|330|330|válido|
|practice_0054|independent|E3|donante2/coste1/orden1|330|330|válido|
|practice_0054|independent|E3|donante3/coste0/orden0|420|325|válido|
|practice_0054|independent|E3|donante3/coste1/orden0|359|325|válido|
|practice_0054|independent|E3|donante3/coste0/orden1|357|325|válido|
|practice_0054|independent|E3|donante3/coste1/orden1|333|325|válido|
|practice_0054|independent|E3|donante4/coste0/orden0|353|353|válido|
|practice_0054|independent|E3|donante4/coste1/orden0|353|353|válido|
|practice_0054|independent|E3|donante4/coste0/orden1|353|353|válido|
|practice_0054|independent|E3|donante4/coste1/orden1|353|353|válido|
|practice_0054|independent|E3|donante5/coste0/orden0|368|368|válido|
|practice_0054|independent|E3|donante5/coste1/orden0|368|368|válido|
|practice_0054|independent|E3|donante5/coste0/orden1|368|368|válido|
|practice_0054|independent|E3|donante5/coste1/orden1|368|368|válido|
|practice_0054|independent|E3|donante6/coste0/orden0|355|355|válido|
|practice_0054|independent|E3|donante6/coste1/orden0|355|355|válido|
|practice_0054|independent|E3|donante6/coste0/orden1|355|355|válido|
|practice_0054|independent|E3|donante6/coste1/orden1|355|355|válido|
|practice_0054|independent|E3|donante7/coste0/orden0|401|313|válido|
|practice_0054|independent|E3|donante7/coste1/orden0|383|313|válido|
|practice_0054|independent|E3|donante7/coste0/orden1|388|313|válido|
|practice_0054|independent|E3|donante7/coste1/orden1|379|313|válido|
|practice_0054|independent|E3|donante8/coste0/orden0|303|302|válido|
|practice_0054|independent|E3|donante8/coste1/orden0|303|302|válido|
|practice_0054|independent|E3|donante8/coste0/orden1|327|302|válido|
|practice_0054|independent|E3|donante8/coste1/orden1|303|302|válido|
|practice_0055|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0055|independent|Cruce|CP True/coste0/orden0|452|297|válido|
|practice_0055|independent|Cruce|CP True/coste1/orden0|413|297|válido|
|practice_0055|independent|Cruce|CP True/coste0/orden1|346|297|válido|
|practice_0055|independent|Cruce|CP True/coste1/orden1|320|297|válido|
|practice_0055|independent|Cruce|mejor independiente/coste0/orden0|452|299|válido|
|practice_0055|independent|Cruce|mejor independiente/coste1/orden0|419|299|válido|
|practice_0055|independent|Cruce|mejor independiente/coste0/orden1|340|299|válido|
|practice_0055|independent|Cruce|mejor independiente/coste1/orden1|323|299|válido|
|practice_0055|independent|E1|cola10/pol1/inicio0|327|97|válido|
|practice_0055|independent|E1|cola10/pol1/inicio1|323|93|válido|
|practice_0055|independent|E1|cola10/pol2/inicio0|322|92|válido|
|practice_0055|independent|E1|cola10/pol2/inicio1|322|92|válido|
|practice_0055|independent|E1|cola8/pol1/inicio0|324|77|válido|
|practice_0055|independent|E1|cola8/pol1/inicio1|323|76|válido|
|practice_0055|independent|E1|cola8/pol2/inicio0|321|74|válido|
|practice_0055|independent|E1|cola8/pol2/inicio1|321|74|válido|
|practice_0055|independent|E1|cola6/pol1/inicio0|322|57|válido|
|practice_0055|independent|E1|cola6/pol1/inicio1|322|57|válido|
|practice_0055|independent|E1|cola6/pol2/inicio0|317|52|válido|
|practice_0055|independent|E1|cola6/pol2/inicio1|319|54|válido|
|practice_0055|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0055|independent|E2|CP False/coste0/orden0|509|301|válido|
|practice_0055|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0055|independent|E2|CP True/coste0/orden0|452|297|válido|
|practice_0055|independent|E3|donante0/coste0/orden0|315|299|válido|
|practice_0055|independent|E3|donante0/coste1/orden0|320|299|válido|
|practice_0055|independent|E3|donante0/coste0/orden1|340|299|válido|
|practice_0055|independent|E3|donante0/coste1/orden1|323|299|válido|
|practice_0055|independent|E3|donante1/coste0/orden0|349|349|válido|
|practice_0055|independent|E3|donante1/coste1/orden0|349|349|válido|
|practice_0055|independent|E3|donante1/coste0/orden1|349|349|válido|
|practice_0055|independent|E3|donante1/coste1/orden1|349|349|válido|
|practice_0055|independent|E3|donante2/coste0/orden0|763|324|válido|
|practice_0055|independent|E3|donante2/coste1/orden0|503|324|válido|
|practice_0055|independent|E3|donante2/coste0/orden1|423|324|válido|
|practice_0055|independent|E3|donante2/coste1/orden1|330|324|válido|
|practice_0055|independent|E3|donante3/coste0/orden0|585|328|válido|
|practice_0055|independent|E3|donante3/coste1/orden0|503|328|válido|
|practice_0055|independent|E3|donante3/coste0/orden1|328|328|válido|
|practice_0055|independent|E3|donante3/coste1/orden1|328|328|válido|
|practice_0055|independent|E3|donante4/coste0/orden0|352|352|válido|
|practice_0055|independent|E3|donante4/coste1/orden0|352|352|válido|
|practice_0055|independent|E3|donante4/coste0/orden1|352|352|válido|
|practice_0055|independent|E3|donante4/coste1/orden1|352|352|válido|
|practice_0055|independent|E3|donante5/coste0/orden0|357|357|válido|
|practice_0055|independent|E3|donante5/coste1/orden0|357|357|válido|
|practice_0055|independent|E3|donante5/coste0/orden1|357|357|válido|
|practice_0055|independent|E3|donante5/coste1/orden1|357|357|válido|
|practice_0055|independent|E3|donante6/coste0/orden0|348|348|válido|
|practice_0055|independent|E3|donante6/coste1/orden0|348|348|válido|
|practice_0055|independent|E3|donante6/coste0/orden1|348|348|válido|
|practice_0055|independent|E3|donante6/coste1/orden1|348|348|válido|
|practice_0055|independent|E3|donante7/coste0/orden0|350|307|válido|
|practice_0055|independent|E3|donante7/coste1/orden0|337|307|válido|
|practice_0055|independent|E3|donante7/coste0/orden1|338|307|válido|
|practice_0055|independent|E3|donante7/coste1/orden1|333|307|válido|
|practice_0055|independent|E3|donante8/coste0/orden0|315|299|válido|
|practice_0055|independent|E3|donante8/coste1/orden0|320|299|válido|
|practice_0055|independent|E3|donante8/coste0/orden1|340|299|válido|
|practice_0055|independent|E3|donante8/coste1/orden1|323|299|válido|
|practice_0055|independent|E3|donante9/coste0/orden0|335|301|válido|
|practice_0055|independent|E3|donante9/coste1/orden0|326|301|válido|
|practice_0055|independent|E3|donante9/coste0/orden1|355|301|válido|
|practice_0055|independent|E3|donante9/coste1/orden1|329|301|válido|
|practice_0056|tiny|Cruce|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0056|tiny|Cruce|CP True/coste0/orden0|4|4|válido|
|practice_0056|tiny|Cruce|CP True/coste1/orden0|4|4|válido|
|practice_0056|tiny|Cruce|CP True/coste0/orden1|4|4|válido|
|practice_0056|tiny|Cruce|CP True/coste1/orden1|4|4|válido|
|practice_0056|tiny|Cruce|mejor independiente/coste0/orden0|4|4|válido|
|practice_0056|tiny|Cruce|mejor independiente/coste1/orden0|4|4|válido|
|practice_0056|tiny|Cruce|mejor independiente/coste0/orden1|4|4|válido|
|practice_0056|tiny|Cruce|mejor independiente/coste1/orden1|4|4|válido|
|practice_0056|tiny|E1|cola4/pol1/inicio0|7|7|válido|
|practice_0056|tiny|E1|cola4/pol1/inicio1|7|7|válido|
|practice_0056|tiny|E1|cola4/pol2/inicio0|7|7|válido|
|practice_0056|tiny|E1|cola4/pol2/inicio1|7|7|válido|
|practice_0056|tiny|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0056|tiny|E2|CP False/coste0/orden0|4|4|válido|
|practice_0056|tiny|E2|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0056|tiny|E2|CP True/coste0/orden0|4|4|válido|
|practice_0056|tiny|E3|donante0/coste0/orden0|4|4|válido|
|practice_0056|tiny|E3|donante0/coste1/orden0|4|4|válido|
|practice_0056|tiny|E3|donante0/coste0/orden1|4|4|válido|
|practice_0056|tiny|E3|donante0/coste1/orden1|4|4|válido|
|practice_0056|tiny|E3|donante1/coste0/orden0|7|7|válido|
|practice_0056|tiny|E3|donante1/coste1/orden0|7|7|válido|
|practice_0056|tiny|E3|donante1/coste0/orden1|7|7|válido|
|practice_0056|tiny|E3|donante1/coste1/orden1|7|7|válido|
|practice_0056|tiny|E3|donante2/coste0/orden0|7|7|válido|
|practice_0056|tiny|E3|donante2/coste1/orden0|7|7|válido|
|practice_0056|tiny|E3|donante2/coste0/orden1|7|7|válido|
|practice_0056|tiny|E3|donante2/coste1/orden1|7|7|válido|
|practice_0057|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0057|independent|Cruce|CP True/coste0/orden0|240|226|válido|
|practice_0057|independent|Cruce|CP True/coste1/orden0|237|226|válido|
|practice_0057|independent|Cruce|CP True/coste0/orden1|226|226|válido|
|practice_0057|independent|Cruce|CP True/coste1/orden1|226|226|válido|
|practice_0057|independent|Cruce|mejor independiente/coste0/orden0|255|228|válido|
|practice_0057|independent|Cruce|mejor independiente/coste1/orden0|241|228|válido|
|practice_0057|independent|Cruce|mejor independiente/coste0/orden1|228|228|válido|
|practice_0057|independent|Cruce|mejor independiente/coste1/orden1|228|228|válido|
|practice_0057|independent|E1|cola10/pol1/inicio0|238|75|válido|
|practice_0057|independent|E1|cola10/pol1/inicio1|238|75|válido|
|practice_0057|independent|E1|cola10/pol2/inicio0|237|74|válido|
|practice_0057|independent|E1|cola10/pol2/inicio1|237|74|válido|
|practice_0057|independent|E1|cola8/pol1/inicio0|237|62|válido|
|practice_0057|independent|E1|cola8/pol1/inicio1|237|62|válido|
|practice_0057|independent|E1|cola8/pol2/inicio0|240|65|válido|
|practice_0057|independent|E1|cola8/pol2/inicio1|240|65|válido|
|practice_0057|independent|E1|cola6/pol1/inicio0|238|49|válido|
|practice_0057|independent|E1|cola6/pol1/inicio1|238|49|válido|
|practice_0057|independent|E1|cola6/pol2/inicio0|238|49|válido|
|practice_0057|independent|E1|cola6/pol2/inicio1|238|49|válido|
|practice_0057|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0057|independent|E2|CP False/coste0/orden0|255|228|válido|
|practice_0057|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0057|independent|E2|CP True/coste0/orden0|240|226|válido|
|practice_0057|independent|E3|donante0/coste0/orden0|228|228|válido|
|practice_0057|independent|E3|donante0/coste1/orden0|228|228|válido|
|practice_0057|independent|E3|donante0/coste0/orden1|228|228|válido|
|practice_0057|independent|E3|donante0/coste1/orden1|228|228|válido|
|practice_0057|independent|E3|donante1/coste0/orden0|256|256|válido|
|practice_0057|independent|E3|donante1/coste1/orden0|256|256|válido|
|practice_0057|independent|E3|donante1/coste0/orden1|256|256|válido|
|practice_0057|independent|E3|donante1/coste1/orden1|256|256|válido|
|practice_0057|independent|E3|donante2/coste0/orden0|260|243|válido|
|practice_0057|independent|E3|donante2/coste1/orden0|253|243|válido|
|practice_0057|independent|E3|donante2/coste0/orden1|243|243|válido|
|practice_0057|independent|E3|donante2/coste1/orden1|243|243|válido|
|practice_0057|independent|E3|donante3/coste0/orden0|304|244|válido|
|practice_0057|independent|E3|donante3/coste1/orden0|286|244|válido|
|practice_0057|independent|E3|donante3/coste0/orden1|289|244|válido|
|practice_0057|independent|E3|donante3/coste1/orden1|264|244|válido|
|practice_0057|independent|E3|donante4/coste0/orden0|260|260|válido|
|practice_0057|independent|E3|donante4/coste1/orden0|260|260|válido|
|practice_0057|independent|E3|donante4/coste0/orden1|260|260|válido|
|practice_0057|independent|E3|donante4/coste1/orden1|260|260|válido|
|practice_0057|independent|E3|donante5/coste0/orden0|257|257|válido|
|practice_0057|independent|E3|donante5/coste1/orden0|257|257|válido|
|practice_0057|independent|E3|donante5/coste0/orden1|257|257|válido|
|practice_0057|independent|E3|donante5/coste1/orden1|257|257|válido|
|practice_0057|independent|E3|donante6/coste0/orden0|262|262|válido|
|practice_0057|independent|E3|donante6/coste1/orden0|262|262|válido|
|practice_0057|independent|E3|donante6/coste0/orden1|262|262|válido|
|practice_0057|independent|E3|donante6/coste1/orden1|262|262|válido|
|practice_0057|independent|E3|donante7/coste0/orden0|248|235|válido|
|practice_0057|independent|E3|donante7/coste1/orden0|244|235|válido|
|practice_0057|independent|E3|donante7/coste0/orden1|253|235|válido|
|practice_0057|independent|E3|donante7/coste1/orden1|246|235|válido|
|practice_0057|independent|E3|donante8/coste0/orden0|229|229|válido|
|practice_0057|independent|E3|donante8/coste1/orden0|229|229|válido|
|practice_0057|independent|E3|donante8/coste0/orden1|229|229|válido|
|practice_0057|independent|E3|donante8/coste1/orden1|229|229|válido|
|practice_0057|independent|E3|donante9/coste0/orden0|254|225|válido|
|practice_0057|independent|E3|donante9/coste1/orden0|245|225|válido|
|practice_0057|independent|E3|donante9/coste0/orden1|288|225|válido|
|practice_0057|independent|E3|donante9/coste1/orden1|259|225|válido|
|practice_0058|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0058|independent|Cruce|CP True/coste0/orden0|281|236|válido|
|practice_0058|independent|Cruce|CP True/coste1/orden0|278|236|válido|
|practice_0058|independent|Cruce|CP True/coste0/orden1|256|236|válido|
|practice_0058|independent|Cruce|CP True/coste1/orden1|272|236|válido|
|practice_0058|independent|Cruce|mejor independiente/coste0/orden0|281|236|válido|
|practice_0058|independent|Cruce|mejor independiente/coste1/orden0|278|236|válido|
|practice_0058|independent|Cruce|mejor independiente/coste0/orden1|256|236|válido|
|practice_0058|independent|Cruce|mejor independiente/coste1/orden1|272|236|válido|
|practice_0058|independent|E1|cola10/pol1/inicio0|245|75|válido|
|practice_0058|independent|E1|cola10/pol1/inicio1|245|75|válido|
|practice_0058|independent|E1|cola10/pol2/inicio0|244|74|válido|
|practice_0058|independent|E1|cola10/pol2/inicio1|244|74|válido|
|practice_0058|independent|E1|cola8/pol1/inicio0|245|57|válido|
|practice_0058|independent|E1|cola8/pol1/inicio1|245|57|válido|
|practice_0058|independent|E1|cola8/pol2/inicio0|246|58|válido|
|practice_0058|independent|E1|cola8/pol2/inicio1|246|58|válido|
|practice_0058|independent|E1|cola6/pol1/inicio0|246|44|válido|
|practice_0058|independent|E1|cola6/pol1/inicio1|246|44|válido|
|practice_0058|independent|E1|cola6/pol2/inicio0|246|44|válido|
|practice_0058|independent|E1|cola6/pol2/inicio1|246|44|válido|
|practice_0058|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0058|independent|E2|CP False/coste0/orden0|271|271|válido|
|practice_0058|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0058|independent|E2|CP True/coste0/orden0|281|236|válido|
|practice_0058|independent|E3|donante0/coste0/orden0|262|236|válido|
|practice_0058|independent|E3|donante0/coste1/orden0|262|236|válido|
|practice_0058|independent|E3|donante0/coste0/orden1|256|236|válido|
|practice_0058|independent|E3|donante0/coste1/orden1|272|236|válido|
|practice_0058|independent|E3|donante1/coste0/orden0|267|267|válido|
|practice_0058|independent|E3|donante1/coste1/orden0|267|267|válido|
|practice_0058|independent|E3|donante1/coste0/orden1|267|267|válido|
|practice_0058|independent|E3|donante1/coste1/orden1|267|267|válido|
|practice_0058|independent|E3|donante2/coste0/orden0|260|260|válido|
|practice_0058|independent|E3|donante2/coste1/orden0|260|260|válido|
|practice_0058|independent|E3|donante2/coste0/orden1|260|260|válido|
|practice_0058|independent|E3|donante2/coste1/orden1|260|260|válido|
|practice_0058|independent|E3|donante3/coste0/orden0|307|253|válido|
|practice_0058|independent|E3|donante3/coste1/orden0|277|253|válido|
|practice_0058|independent|E3|donante3/coste0/orden1|258|253|válido|
|practice_0058|independent|E3|donante3/coste1/orden1|258|253|válido|
|practice_0058|independent|E3|donante4/coste0/orden0|271|271|válido|
|practice_0058|independent|E3|donante4/coste1/orden0|271|271|válido|
|practice_0058|independent|E3|donante4/coste0/orden1|271|271|válido|
|practice_0058|independent|E3|donante4/coste1/orden1|271|271|válido|
|practice_0058|independent|E3|donante5/coste0/orden0|272|272|válido|
|practice_0058|independent|E3|donante5/coste1/orden0|272|272|válido|
|practice_0058|independent|E3|donante5/coste0/orden1|272|272|válido|
|practice_0058|independent|E3|donante5/coste1/orden1|272|272|válido|
|practice_0058|independent|E3|donante6/coste0/orden0|272|272|válido|
|practice_0058|independent|E3|donante6/coste1/orden0|272|272|válido|
|practice_0058|independent|E3|donante6/coste0/orden1|272|272|válido|
|practice_0058|independent|E3|donante6/coste1/orden1|272|272|válido|
|practice_0058|independent|E3|donante7/coste0/orden0|248|248|válido|
|practice_0058|independent|E3|donante7/coste1/orden0|248|248|válido|
|practice_0058|independent|E3|donante7/coste0/orden1|269|248|válido|
|practice_0058|independent|E3|donante7/coste1/orden1|266|248|válido|
|practice_0058|independent|E3|donante8/coste0/orden0|261|237|válido|
|practice_0058|independent|E3|donante8/coste1/orden0|261|237|válido|
|practice_0058|independent|E3|donante8/coste0/orden1|257|237|válido|
|practice_0058|independent|E3|donante8/coste1/orden1|271|237|válido|
|practice_0058|independent|E3|donante9/coste0/orden0|286|234|válido|
|practice_0058|independent|E3|donante9/coste1/orden0|249|234|válido|
|practice_0058|independent|E3|donante9/coste0/orden1|267|234|válido|
|practice_0058|independent|E3|donante9/coste1/orden1|243|234|válido|
|practice_0059|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0059|multiplicative|Cruce|CP True/coste0/orden0|126|126|válido|
|practice_0059|multiplicative|Cruce|CP True/coste1/orden0|126|126|válido|
|practice_0059|multiplicative|Cruce|CP True/coste0/orden1|126|126|válido|
|practice_0059|multiplicative|Cruce|CP True/coste1/orden1|126|126|válido|
|practice_0059|multiplicative|Cruce|mejor independiente/coste0/orden0|129|129|válido|
|practice_0059|multiplicative|Cruce|mejor independiente/coste1/orden0|129|129|válido|
|practice_0059|multiplicative|Cruce|mejor independiente/coste0/orden1|129|129|válido|
|practice_0059|multiplicative|Cruce|mejor independiente/coste1/orden1|129|129|válido|
|practice_0059|multiplicative|E1|cola10/pol1/inicio0|132|45|válido|
|practice_0059|multiplicative|E1|cola10/pol1/inicio1|132|45|válido|
|practice_0059|multiplicative|E1|cola10/pol2/inicio0|131|44|válido|
|practice_0059|multiplicative|E1|cola10/pol2/inicio1|131|44|válido|
|practice_0059|multiplicative|E1|cola8/pol1/inicio0|132|36|válido|
|practice_0059|multiplicative|E1|cola8/pol1/inicio1|132|36|válido|
|practice_0059|multiplicative|E1|cola8/pol2/inicio0|132|36|válido|
|practice_0059|multiplicative|E1|cola8/pol2/inicio1|132|36|válido|
|practice_0059|multiplicative|E1|cola6/pol1/inicio0|132|27|válido|
|practice_0059|multiplicative|E1|cola6/pol1/inicio1|132|27|válido|
|practice_0059|multiplicative|E1|cola6/pol2/inicio0|132|27|válido|
|practice_0059|multiplicative|E1|cola6/pol2/inicio1|132|27|válido|
|practice_0059|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0059|multiplicative|E2|CP False/coste0/orden0|129|129|válido|
|practice_0059|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0059|multiplicative|E2|CP True/coste0/orden0|129|129|válido|
|practice_0059|multiplicative|E3|donante0/coste0/orden0|129|129|válido|
|practice_0059|multiplicative|E3|donante0/coste1/orden0|129|129|válido|
|practice_0059|multiplicative|E3|donante0/coste0/orden1|129|129|válido|
|practice_0059|multiplicative|E3|donante0/coste1/orden1|129|129|válido|
|practice_0059|multiplicative|E3|donante1/coste0/orden0|151|151|válido|
|practice_0059|multiplicative|E3|donante1/coste1/orden0|151|151|válido|
|practice_0059|multiplicative|E3|donante1/coste0/orden1|151|151|válido|
|practice_0059|multiplicative|E3|donante1/coste1/orden1|151|151|válido|
|practice_0059|multiplicative|E3|donante2/coste0/orden0|146|146|válido|
|practice_0059|multiplicative|E3|donante2/coste1/orden0|146|146|válido|
|practice_0059|multiplicative|E3|donante2/coste0/orden1|146|146|válido|
|practice_0059|multiplicative|E3|donante2/coste1/orden1|146|146|válido|
|practice_0059|multiplicative|E3|donante3/coste0/orden0|136|136|válido|
|practice_0059|multiplicative|E3|donante3/coste1/orden0|136|136|válido|
|practice_0059|multiplicative|E3|donante3/coste0/orden1|136|136|válido|
|practice_0059|multiplicative|E3|donante3/coste1/orden1|136|136|válido|
|practice_0059|multiplicative|E3|donante4/coste0/orden0|149|149|válido|
|practice_0059|multiplicative|E3|donante4/coste1/orden0|149|149|válido|
|practice_0059|multiplicative|E3|donante4/coste0/orden1|149|149|válido|
|practice_0059|multiplicative|E3|donante4/coste1/orden1|149|149|válido|
|practice_0059|multiplicative|E3|donante5/coste0/orden0|152|152|válido|
|practice_0059|multiplicative|E3|donante5/coste1/orden0|152|152|válido|
|practice_0059|multiplicative|E3|donante5/coste0/orden1|152|152|válido|
|practice_0059|multiplicative|E3|donante5/coste1/orden1|152|152|válido|
|practice_0059|multiplicative|E3|donante6/coste0/orden0|140|140|válido|
|practice_0059|multiplicative|E3|donante6/coste1/orden0|140|140|válido|
|practice_0059|multiplicative|E3|donante6/coste0/orden1|140|140|válido|
|practice_0059|multiplicative|E3|donante6/coste1/orden1|140|140|válido|
|practice_0059|multiplicative|E3|donante7/coste0/orden0|161|144|válido|
|practice_0059|multiplicative|E3|donante7/coste1/orden0|159|144|válido|
|practice_0059|multiplicative|E3|donante7/coste0/orden1|151|144|válido|
|practice_0059|multiplicative|E3|donante7/coste1/orden1|146|144|válido|
|practice_0059|multiplicative|E3|donante8/coste0/orden0|144|144|válido|
|practice_0059|multiplicative|E3|donante8/coste1/orden0|144|144|válido|
|practice_0059|multiplicative|E3|donante8/coste0/orden1|144|144|válido|
|practice_0059|multiplicative|E3|donante8/coste1/orden1|144|144|válido|
|practice_0059|multiplicative|E3|donante9/coste0/orden0|130|130|válido|
|practice_0059|multiplicative|E3|donante9/coste1/orden0|130|130|válido|
|practice_0059|multiplicative|E3|donante9/coste0/orden1|130|130|válido|
|practice_0059|multiplicative|E3|donante9/coste1/orden1|130|130|válido|
|practice_0060|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0060|mixed|Cruce|CP True/coste0/orden0|279|228|válido|
|practice_0060|mixed|Cruce|CP True/coste1/orden0|263|228|válido|
|practice_0060|mixed|Cruce|CP True/coste0/orden1|265|228|válido|
|practice_0060|mixed|Cruce|CP True/coste1/orden1|257|228|válido|
|practice_0060|mixed|Cruce|mejor independiente/coste0/orden0|301|228|válido|
|practice_0060|mixed|Cruce|mejor independiente/coste1/orden0|269|228|válido|
|practice_0060|mixed|Cruce|mejor independiente/coste0/orden1|276|228|válido|
|practice_0060|mixed|Cruce|mejor independiente/coste1/orden1|264|228|válido|
|practice_0060|mixed|E1|cola10/pol1/inicio0|243|92|válido|
|practice_0060|mixed|E1|cola10/pol1/inicio1|243|92|válido|
|practice_0060|mixed|E1|cola10/pol2/inicio0|240|89|válido|
|practice_0060|mixed|E1|cola10/pol2/inicio1|238|87|válido|
|practice_0060|mixed|E1|cola8/pol1/inicio0|248|79|válido|
|practice_0060|mixed|E1|cola8/pol1/inicio1|248|79|válido|
|practice_0060|mixed|E1|cola8/pol2/inicio0|247|78|válido|
|practice_0060|mixed|E1|cola8/pol2/inicio1|247|78|válido|
|practice_0060|mixed|E1|cola6/pol1/inicio0|242|50|válido|
|practice_0060|mixed|E1|cola6/pol1/inicio1|242|50|válido|
|practice_0060|mixed|E1|cola6/pol2/inicio0|242|50|válido|
|practice_0060|mixed|E1|cola6/pol2/inicio1|242|50|válido|
|practice_0060|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0060|mixed|E2|CP False/coste0/orden0|288|229|válido|
|practice_0060|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0060|mixed|E2|CP True/coste0/orden0|279|228|válido|
|practice_0060|mixed|E3|donante0/coste0/orden0|246|228|válido|
|practice_0060|mixed|E3|donante0/coste1/orden0|247|228|válido|
|practice_0060|mixed|E3|donante0/coste0/orden1|276|228|válido|
|practice_0060|mixed|E3|donante0/coste1/orden1|264|228|válido|
|practice_0060|mixed|E3|donante1/coste0/orden0|265|265|válido|
|practice_0060|mixed|E3|donante1/coste1/orden0|265|265|válido|
|practice_0060|mixed|E3|donante1/coste0/orden1|265|265|válido|
|practice_0060|mixed|E3|donante1/coste1/orden1|265|265|válido|
|practice_0060|mixed|E3|donante2/coste0/orden0|253|253|válido|
|practice_0060|mixed|E3|donante2/coste1/orden0|253|253|válido|
|practice_0060|mixed|E3|donante2/coste0/orden1|253|253|válido|
|practice_0060|mixed|E3|donante2/coste1/orden1|253|253|válido|
|practice_0060|mixed|E3|donante3/coste0/orden0|290|240|válido|
|practice_0060|mixed|E3|donante3/coste1/orden0|275|240|válido|
|practice_0060|mixed|E3|donante3/coste0/orden1|240|240|válido|
|practice_0060|mixed|E3|donante3/coste1/orden1|240|240|válido|
|practice_0060|mixed|E3|donante4/coste0/orden0|275|275|válido|
|practice_0060|mixed|E3|donante4/coste1/orden0|275|275|válido|
|practice_0060|mixed|E3|donante4/coste0/orden1|275|275|válido|
|practice_0060|mixed|E3|donante4/coste1/orden1|275|275|válido|
|practice_0060|mixed|E3|donante5/coste0/orden0|263|263|válido|
|practice_0060|mixed|E3|donante5/coste1/orden0|263|263|válido|
|practice_0060|mixed|E3|donante5/coste0/orden1|263|263|válido|
|practice_0060|mixed|E3|donante5/coste1/orden1|263|263|válido|
|practice_0060|mixed|E3|donante6/coste0/orden0|272|272|válido|
|practice_0060|mixed|E3|donante6/coste1/orden0|272|272|válido|
|practice_0060|mixed|E3|donante6/coste0/orden1|272|272|válido|
|practice_0060|mixed|E3|donante6/coste1/orden1|272|272|válido|
|practice_0060|mixed|E3|donante7/coste0/orden0|321|246|válido|
|practice_0060|mixed|E3|donante7/coste1/orden0|276|246|válido|
|practice_0060|mixed|E3|donante7/coste0/orden1|318|246|válido|
|practice_0060|mixed|E3|donante7/coste1/orden1|268|246|válido|
|practice_0060|mixed|E3|donante8/coste0/orden0|258|233|válido|
|practice_0060|mixed|E3|donante8/coste1/orden0|243|233|válido|
|practice_0060|mixed|E3|donante8/coste0/orden1|255|233|válido|
|practice_0060|mixed|E3|donante8/coste1/orden1|244|233|válido|
|practice_0060|mixed|E3|donante9/coste0/orden0|246|228|válido|
|practice_0060|mixed|E3|donante9/coste1/orden0|247|228|válido|
|practice_0060|mixed|E3|donante9/coste0/orden1|276|228|válido|
|practice_0060|mixed|E3|donante9/coste1/orden1|264|228|válido|
|practice_0061|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0061|mixed|Cruce|CP True/coste0/orden0|340|245|válido|
|practice_0061|mixed|Cruce|CP True/coste1/orden0|310|245|válido|
|practice_0061|mixed|Cruce|CP True/coste0/orden1|246|245|válido|
|practice_0061|mixed|Cruce|CP True/coste1/orden1|246|245|válido|
|practice_0061|mixed|Cruce|mejor independiente/coste0/orden0|345|245|válido|
|practice_0061|mixed|Cruce|mejor independiente/coste1/orden0|313|245|válido|
|practice_0061|mixed|Cruce|mejor independiente/coste0/orden1|246|245|válido|
|practice_0061|mixed|Cruce|mejor independiente/coste1/orden1|246|245|válido|
|practice_0061|mixed|E1|cola10/pol1/inicio0|257|91|válido|
|practice_0061|mixed|E1|cola10/pol1/inicio1|257|91|válido|
|practice_0061|mixed|E1|cola10/pol2/inicio0|257|91|válido|
|practice_0061|mixed|E1|cola10/pol2/inicio1|257|91|válido|
|practice_0061|mixed|E1|cola8/pol1/inicio0|262|80|válido|
|practice_0061|mixed|E1|cola8/pol1/inicio1|262|80|válido|
|practice_0061|mixed|E1|cola8/pol2/inicio0|260|78|válido|
|practice_0061|mixed|E1|cola8/pol2/inicio1|260|78|válido|
|practice_0061|mixed|E1|cola6/pol1/inicio0|255|54|válido|
|practice_0061|mixed|E1|cola6/pol1/inicio1|255|54|válido|
|practice_0061|mixed|E1|cola6/pol2/inicio0|255|54|válido|
|practice_0061|mixed|E1|cola6/pol2/inicio1|255|54|válido|
|practice_0061|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0061|mixed|E2|CP False/coste0/orden0|281|281|válido|
|practice_0061|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0061|mixed|E2|CP True/coste0/orden0|340|245|válido|
|practice_0061|mixed|E3|donante0/coste0/orden0|246|245|válido|
|practice_0061|mixed|E3|donante0/coste1/orden0|246|245|válido|
|practice_0061|mixed|E3|donante0/coste0/orden1|246|245|válido|
|practice_0061|mixed|E3|donante0/coste1/orden1|246|245|válido|
|practice_0061|mixed|E3|donante1/coste0/orden0|282|282|válido|
|practice_0061|mixed|E3|donante1/coste1/orden0|282|282|válido|
|practice_0061|mixed|E3|donante1/coste0/orden1|282|282|válido|
|practice_0061|mixed|E3|donante1/coste1/orden1|282|282|válido|
|practice_0061|mixed|E3|donante2/coste0/orden0|268|265|válido|
|practice_0061|mixed|E3|donante2/coste1/orden0|268|265|válido|
|practice_0061|mixed|E3|donante2/coste0/orden1|265|265|válido|
|practice_0061|mixed|E3|donante2/coste1/orden1|265|265|válido|
|practice_0061|mixed|E3|donante3/coste0/orden0|356|266|válido|
|practice_0061|mixed|E3|donante3/coste1/orden0|292|266|válido|
|practice_0061|mixed|E3|donante3/coste0/orden1|266|266|válido|
|practice_0061|mixed|E3|donante3/coste1/orden1|266|266|válido|
|practice_0061|mixed|E3|donante4/coste0/orden0|281|281|válido|
|practice_0061|mixed|E3|donante4/coste1/orden0|281|281|válido|
|practice_0061|mixed|E3|donante4/coste0/orden1|281|281|válido|
|practice_0061|mixed|E3|donante4/coste1/orden1|281|281|válido|
|practice_0061|mixed|E3|donante5/coste0/orden0|277|277|válido|
|practice_0061|mixed|E3|donante5/coste1/orden0|277|277|válido|
|practice_0061|mixed|E3|donante5/coste0/orden1|277|277|válido|
|practice_0061|mixed|E3|donante5/coste1/orden1|277|277|válido|
|practice_0061|mixed|E3|donante6/coste0/orden0|292|292|válido|
|practice_0061|mixed|E3|donante6/coste1/orden0|292|292|válido|
|practice_0061|mixed|E3|donante6/coste0/orden1|292|292|válido|
|practice_0061|mixed|E3|donante6/coste1/orden1|292|292|válido|
|practice_0061|mixed|E3|donante7/coste0/orden0|296|253|válido|
|practice_0061|mixed|E3|donante7/coste1/orden0|292|253|válido|
|practice_0061|mixed|E3|donante7/coste0/orden1|267|253|válido|
|practice_0061|mixed|E3|donante7/coste1/orden1|267|253|válido|
|practice_0061|mixed|E3|donante8/coste0/orden0|247|246|válido|
|practice_0061|mixed|E3|donante8/coste1/orden0|247|246|válido|
|practice_0061|mixed|E3|donante8/coste0/orden1|247|246|válido|
|practice_0061|mixed|E3|donante8/coste1/orden1|247|246|válido|
|practice_0061|mixed|E3|donante9/coste0/orden0|265|243|válido|
|practice_0061|mixed|E3|donante9/coste1/orden0|265|243|válido|
|practice_0061|mixed|E3|donante9/coste0/orden1|272|243|válido|
|practice_0061|mixed|E3|donante9/coste1/orden1|265|243|válido|
|practice_0062|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0062|independent|Cruce|CP True/coste0/orden0|374|279|válido|
|practice_0062|independent|Cruce|CP True/coste1/orden0|339|279|válido|
|practice_0062|independent|Cruce|CP True/coste0/orden1|284|279|válido|
|practice_0062|independent|Cruce|CP True/coste1/orden1|281|279|válido|
|practice_0062|independent|Cruce|mejor independiente/coste0/orden0|374|279|válido|
|practice_0062|independent|Cruce|mejor independiente/coste1/orden0|339|279|válido|
|practice_0062|independent|Cruce|mejor independiente/coste0/orden1|284|279|válido|
|practice_0062|independent|Cruce|mejor independiente/coste1/orden1|281|279|válido|
|practice_0062|independent|E1|cola10/pol1/inicio0|293|94|válido|
|practice_0062|independent|E1|cola10/pol1/inicio1|293|94|válido|
|practice_0062|independent|E1|cola10/pol2/inicio0|290|91|válido|
|practice_0062|independent|E1|cola10/pol2/inicio1|290|91|válido|
|practice_0062|independent|E1|cola8/pol1/inicio0|293|76|válido|
|practice_0062|independent|E1|cola8/pol1/inicio1|293|76|válido|
|practice_0062|independent|E1|cola8/pol2/inicio0|294|77|válido|
|practice_0062|independent|E1|cola8/pol2/inicio1|294|77|válido|
|practice_0062|independent|E1|cola6/pol1/inicio0|290|55|válido|
|practice_0062|independent|E1|cola6/pol1/inicio1|290|55|válido|
|practice_0062|independent|E1|cola6/pol2/inicio0|290|55|válido|
|practice_0062|independent|E1|cola6/pol2/inicio1|290|55|válido|
|practice_0062|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0062|independent|E2|CP False/coste0/orden0|374|279|válido|
|practice_0062|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0062|independent|E2|CP True/coste0/orden0|374|279|válido|
|practice_0062|independent|E3|donante0/coste0/orden0|279|279|válido|
|practice_0062|independent|E3|donante0/coste1/orden0|279|279|válido|
|practice_0062|independent|E3|donante0/coste0/orden1|284|279|válido|
|practice_0062|independent|E3|donante0/coste1/orden1|281|279|válido|
|practice_0062|independent|E3|donante1/coste0/orden0|313|313|válido|
|practice_0062|independent|E3|donante1/coste1/orden0|313|313|válido|
|practice_0062|independent|E3|donante1/coste0/orden1|313|313|válido|
|practice_0062|independent|E3|donante1/coste1/orden1|313|313|válido|
|practice_0062|independent|E3|donante2/coste0/orden0|501|299|válido|
|practice_0062|independent|E3|donante2/coste1/orden0|317|299|válido|
|practice_0062|independent|E3|donante2/coste0/orden1|301|299|válido|
|practice_0062|independent|E3|donante2/coste1/orden1|301|299|válido|
|practice_0062|independent|E3|donante3/coste0/orden0|365|298|válido|
|practice_0062|independent|E3|donante3/coste1/orden0|350|298|válido|
|practice_0062|independent|E3|donante3/coste0/orden1|298|298|válido|
|practice_0062|independent|E3|donante3/coste1/orden1|298|298|válido|
|practice_0062|independent|E3|donante4/coste0/orden0|324|324|válido|
|practice_0062|independent|E3|donante4/coste1/orden0|324|324|válido|
|practice_0062|independent|E3|donante4/coste0/orden1|324|324|válido|
|practice_0062|independent|E3|donante4/coste1/orden1|324|324|válido|
|practice_0062|independent|E3|donante5/coste0/orden0|334|334|válido|
|practice_0062|independent|E3|donante5/coste1/orden0|334|334|válido|
|practice_0062|independent|E3|donante5/coste0/orden1|334|334|válido|
|practice_0062|independent|E3|donante5/coste1/orden1|334|334|válido|
|practice_0062|independent|E3|donante6/coste0/orden0|326|326|válido|
|practice_0062|independent|E3|donante6/coste1/orden0|326|326|válido|
|practice_0062|independent|E3|donante6/coste0/orden1|326|326|válido|
|practice_0062|independent|E3|donante6/coste1/orden1|326|326|válido|
|practice_0062|independent|E3|donante7/coste0/orden0|295|286|válido|
|practice_0062|independent|E3|donante7/coste1/orden0|295|286|válido|
|practice_0062|independent|E3|donante7/coste0/orden1|304|286|válido|
|practice_0062|independent|E3|donante7/coste1/orden1|303|286|válido|
|practice_0062|independent|E3|donante8/coste0/orden0|279|279|válido|
|practice_0062|independent|E3|donante8/coste1/orden0|279|279|válido|
|practice_0062|independent|E3|donante8/coste0/orden1|280|279|válido|
|practice_0062|independent|E3|donante8/coste1/orden1|280|279|válido|
|practice_0063|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0063|multiplicative|Cruce|CP True/coste0/orden0|119|119|válido|
|practice_0063|multiplicative|Cruce|CP True/coste1/orden0|119|119|válido|
|practice_0063|multiplicative|Cruce|CP True/coste0/orden1|119|119|válido|
|practice_0063|multiplicative|Cruce|CP True/coste1/orden1|119|119|válido|
|practice_0063|multiplicative|Cruce|mejor independiente/coste0/orden0|120|120|válido|
|practice_0063|multiplicative|Cruce|mejor independiente/coste1/orden0|120|120|válido|
|practice_0063|multiplicative|Cruce|mejor independiente/coste0/orden1|120|120|válido|
|practice_0063|multiplicative|Cruce|mejor independiente/coste1/orden1|120|120|válido|
|practice_0063|multiplicative|E1|cola10/pol1/inicio0|128|45|válido|
|practice_0063|multiplicative|E1|cola10/pol1/inicio1|128|45|válido|
|practice_0063|multiplicative|E1|cola10/pol2/inicio0|124|41|válido|
|practice_0063|multiplicative|E1|cola10/pol2/inicio1|124|41|válido|
|practice_0063|multiplicative|E1|cola8/pol1/inicio0|127|36|válido|
|practice_0063|multiplicative|E1|cola8/pol1/inicio1|127|36|válido|
|practice_0063|multiplicative|E1|cola8/pol2/inicio0|121|30|válido|
|practice_0063|multiplicative|E1|cola8/pol2/inicio1|121|30|válido|
|practice_0063|multiplicative|E1|cola6/pol1/inicio0|127|30|válido|
|practice_0063|multiplicative|E1|cola6/pol1/inicio1|127|30|válido|
|practice_0063|multiplicative|E1|cola6/pol2/inicio0|126|29|válido|
|practice_0063|multiplicative|E1|cola6/pol2/inicio1|126|29|válido|
|practice_0063|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0063|multiplicative|E2|CP False/coste0/orden0|126|126|válido|
|practice_0063|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0063|multiplicative|E2|CP True/coste0/orden0|120|120|válido|
|practice_0063|multiplicative|E3|donante0/coste0/orden0|121|121|válido|
|practice_0063|multiplicative|E3|donante0/coste1/orden0|121|121|válido|
|practice_0063|multiplicative|E3|donante0/coste0/orden1|121|121|válido|
|practice_0063|multiplicative|E3|donante0/coste1/orden1|121|121|válido|
|practice_0063|multiplicative|E3|donante1/coste0/orden0|131|131|válido|
|practice_0063|multiplicative|E3|donante1/coste1/orden0|131|131|válido|
|practice_0063|multiplicative|E3|donante1/coste0/orden1|131|131|válido|
|practice_0063|multiplicative|E3|donante1/coste1/orden1|131|131|válido|
|practice_0063|multiplicative|E3|donante2/coste0/orden0|128|128|válido|
|practice_0063|multiplicative|E3|donante2/coste1/orden0|128|128|válido|
|practice_0063|multiplicative|E3|donante2/coste0/orden1|128|128|válido|
|practice_0063|multiplicative|E3|donante2/coste1/orden1|128|128|válido|
|practice_0063|multiplicative|E3|donante3/coste0/orden0|127|127|válido|
|practice_0063|multiplicative|E3|donante3/coste1/orden0|127|127|válido|
|practice_0063|multiplicative|E3|donante3/coste0/orden1|127|127|válido|
|practice_0063|multiplicative|E3|donante3/coste1/orden1|127|127|válido|
|practice_0063|multiplicative|E3|donante4/coste0/orden0|142|142|válido|
|practice_0063|multiplicative|E3|donante4/coste1/orden0|142|142|válido|
|practice_0063|multiplicative|E3|donante4/coste0/orden1|142|142|válido|
|practice_0063|multiplicative|E3|donante4/coste1/orden1|142|142|válido|
|practice_0063|multiplicative|E3|donante5/coste0/orden0|135|135|válido|
|practice_0063|multiplicative|E3|donante5/coste1/orden0|135|135|válido|
|practice_0063|multiplicative|E3|donante5/coste0/orden1|135|135|válido|
|practice_0063|multiplicative|E3|donante5/coste1/orden1|135|135|válido|
|practice_0063|multiplicative|E3|donante6/coste0/orden0|130|130|válido|
|practice_0063|multiplicative|E3|donante6/coste1/orden0|130|130|válido|
|practice_0063|multiplicative|E3|donante6/coste0/orden1|130|130|válido|
|practice_0063|multiplicative|E3|donante6/coste1/orden1|130|130|válido|
|practice_0063|multiplicative|E3|donante7/coste0/orden0|128|128|válido|
|practice_0063|multiplicative|E3|donante7/coste1/orden0|128|128|válido|
|practice_0063|multiplicative|E3|donante7/coste0/orden1|128|128|válido|
|practice_0063|multiplicative|E3|donante7/coste1/orden1|128|128|válido|
|practice_0063|multiplicative|E3|donante8/coste0/orden0|127|127|válido|
|practice_0063|multiplicative|E3|donante8/coste1/orden0|127|127|válido|
|practice_0063|multiplicative|E3|donante8/coste0/orden1|127|127|válido|
|practice_0063|multiplicative|E3|donante8/coste1/orden1|127|127|válido|
|practice_0064|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0064|mixed|Cruce|CP True/coste0/orden0|236|222|válido|
|practice_0064|mixed|Cruce|CP True/coste1/orden0|227|222|válido|
|practice_0064|mixed|Cruce|CP True/coste0/orden1|237|222|válido|
|practice_0064|mixed|Cruce|CP True/coste1/orden1|233|222|válido|
|practice_0064|mixed|Cruce|mejor independiente/coste0/orden0|241|223|válido|
|practice_0064|mixed|Cruce|mejor independiente/coste1/orden0|230|223|válido|
|practice_0064|mixed|Cruce|mejor independiente/coste0/orden1|242|223|válido|
|practice_0064|mixed|Cruce|mejor independiente/coste1/orden1|236|223|válido|
|practice_0064|mixed|E1|cola10/pol1/inicio0|240|89|válido|
|practice_0064|mixed|E1|cola10/pol1/inicio1|240|89|válido|
|practice_0064|mixed|E1|cola10/pol2/inicio0|234|83|válido|
|practice_0064|mixed|E1|cola10/pol2/inicio1|234|83|válido|
|practice_0064|mixed|E1|cola8/pol1/inicio0|240|73|válido|
|practice_0064|mixed|E1|cola8/pol1/inicio1|240|73|válido|
|practice_0064|mixed|E1|cola8/pol2/inicio0|234|67|válido|
|practice_0064|mixed|E1|cola8/pol2/inicio1|236|69|válido|
|practice_0064|mixed|E1|cola6/pol1/inicio0|225|41|válido|
|practice_0064|mixed|E1|cola6/pol1/inicio1|225|41|válido|
|practice_0064|mixed|E1|cola6/pol2/inicio0|225|41|válido|
|practice_0064|mixed|E1|cola6/pol2/inicio1|225|41|válido|
|practice_0064|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0064|mixed|E2|CP False/coste0/orden0|249|249|válido|
|practice_0064|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0064|mixed|E2|CP True/coste0/orden0|238|224|válido|
|practice_0064|mixed|E3|donante0/coste0/orden0|226|224|válido|
|practice_0064|mixed|E3|donante0/coste1/orden0|226|224|válido|
|practice_0064|mixed|E3|donante0/coste0/orden1|232|224|válido|
|practice_0064|mixed|E3|donante0/coste1/orden1|226|224|válido|
|practice_0064|mixed|E3|donante1/coste0/orden0|255|255|válido|
|practice_0064|mixed|E3|donante1/coste1/orden0|255|255|válido|
|practice_0064|mixed|E3|donante1/coste0/orden1|255|255|válido|
|practice_0064|mixed|E3|donante1/coste1/orden1|255|255|válido|
|practice_0064|mixed|E3|donante2/coste0/orden0|250|250|válido|
|practice_0064|mixed|E3|donante2/coste1/orden0|250|250|válido|
|practice_0064|mixed|E3|donante2/coste0/orden1|250|250|válido|
|practice_0064|mixed|E3|donante2/coste1/orden1|250|250|válido|
|practice_0064|mixed|E3|donante3/coste0/orden0|322|248|válido|
|practice_0064|mixed|E3|donante3/coste1/orden0|251|248|válido|
|practice_0064|mixed|E3|donante3/coste0/orden1|248|248|válido|
|practice_0064|mixed|E3|donante3/coste1/orden1|248|248|válido|
|practice_0064|mixed|E3|donante4/coste0/orden0|262|262|válido|
|practice_0064|mixed|E3|donante4/coste1/orden0|262|262|válido|
|practice_0064|mixed|E3|donante4/coste0/orden1|262|262|válido|
|practice_0064|mixed|E3|donante4/coste1/orden1|262|262|válido|
|practice_0064|mixed|E3|donante5/coste0/orden0|250|250|válido|
|practice_0064|mixed|E3|donante5/coste1/orden0|250|250|válido|
|practice_0064|mixed|E3|donante5/coste0/orden1|250|250|válido|
|practice_0064|mixed|E3|donante5/coste1/orden1|250|250|válido|
|practice_0064|mixed|E3|donante6/coste0/orden0|265|265|válido|
|practice_0064|mixed|E3|donante6/coste1/orden0|265|265|válido|
|practice_0064|mixed|E3|donante6/coste0/orden1|265|265|válido|
|practice_0064|mixed|E3|donante6/coste1/orden1|265|265|válido|
|practice_0064|mixed|E3|donante7/coste0/orden0|292|232|válido|
|practice_0064|mixed|E3|donante7/coste1/orden0|258|232|válido|
|practice_0064|mixed|E3|donante7/coste0/orden1|270|232|válido|
|practice_0064|mixed|E3|donante7/coste1/orden1|248|232|válido|
|practice_0064|mixed|E3|donante8/coste0/orden0|260|225|válido|
|practice_0064|mixed|E3|donante8/coste1/orden0|250|225|válido|
|practice_0064|mixed|E3|donante8/coste0/orden1|282|225|válido|
|practice_0064|mixed|E3|donante8/coste1/orden1|243|225|válido|
|practice_0064|mixed|E3|donante9/coste0/orden0|228|225|válido|
|practice_0064|mixed|E3|donante9/coste1/orden0|229|225|válido|
|practice_0064|mixed|E3|donante9/coste0/orden1|228|225|válido|
|practice_0064|mixed|E3|donante9/coste1/orden1|233|225|válido|
|practice_0065|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0065|mixed|Cruce|CP True/coste0/orden0|224|202|válido|
|practice_0065|mixed|Cruce|CP True/coste1/orden0|207|202|válido|
|practice_0065|mixed|Cruce|CP True/coste0/orden1|239|202|válido|
|practice_0065|mixed|Cruce|CP True/coste1/orden1|210|202|válido|
|practice_0065|mixed|Cruce|mejor independiente/coste0/orden0|237|206|válido|
|practice_0065|mixed|Cruce|mejor independiente/coste1/orden0|217|206|válido|
|practice_0065|mixed|Cruce|mejor independiente/coste0/orden1|225|206|válido|
|practice_0065|mixed|Cruce|mejor independiente/coste1/orden1|211|206|válido|
|practice_0065|mixed|E1|cola10/pol1/inicio0|226|78|válido|
|practice_0065|mixed|E1|cola10/pol1/inicio1|226|78|válido|
|practice_0065|mixed|E1|cola10/pol2/inicio0|225|77|válido|
|practice_0065|mixed|E1|cola10/pol2/inicio1|224|76|válido|
|practice_0065|mixed|E1|cola8/pol1/inicio0|226|64|válido|
|practice_0065|mixed|E1|cola8/pol1/inicio1|226|64|válido|
|practice_0065|mixed|E1|cola8/pol2/inicio0|223|61|válido|
|practice_0065|mixed|E1|cola8/pol2/inicio1|225|63|válido|
|practice_0065|mixed|E1|cola6/pol1/inicio0|228|49|válido|
|practice_0065|mixed|E1|cola6/pol1/inicio1|228|49|válido|
|practice_0065|mixed|E1|cola6/pol2/inicio0|228|49|válido|
|practice_0065|mixed|E1|cola6/pol2/inicio1|228|49|válido|
|practice_0065|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0065|mixed|E2|CP False/coste0/orden0|257|225|válido|
|practice_0065|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0065|mixed|E2|CP True/coste0/orden0|233|214|válido|
|practice_0065|mixed|E3|donante0/coste0/orden0|214|214|válido|
|practice_0065|mixed|E3|donante0/coste1/orden0|214|214|válido|
|practice_0065|mixed|E3|donante0/coste0/orden1|214|214|válido|
|practice_0065|mixed|E3|donante0/coste1/orden1|214|214|válido|
|practice_0065|mixed|E3|donante1/coste0/orden0|236|236|válido|
|practice_0065|mixed|E3|donante1/coste1/orden0|236|236|válido|
|practice_0065|mixed|E3|donante1/coste0/orden1|236|236|válido|
|practice_0065|mixed|E3|donante1/coste1/orden1|236|236|válido|
|practice_0065|mixed|E3|donante2/coste0/orden0|231|231|válido|
|practice_0065|mixed|E3|donante2/coste1/orden0|231|231|válido|
|practice_0065|mixed|E3|donante2/coste0/orden1|231|231|válido|
|practice_0065|mixed|E3|donante2/coste1/orden1|231|231|válido|
|practice_0065|mixed|E3|donante3/coste0/orden0|318|225|válido|
|practice_0065|mixed|E3|donante3/coste1/orden0|249|225|válido|
|practice_0065|mixed|E3|donante3/coste0/orden1|225|225|válido|
|practice_0065|mixed|E3|donante3/coste1/orden1|225|225|válido|
|practice_0065|mixed|E3|donante4/coste0/orden0|241|241|válido|
|practice_0065|mixed|E3|donante4/coste1/orden0|241|241|válido|
|practice_0065|mixed|E3|donante4/coste0/orden1|241|241|válido|
|practice_0065|mixed|E3|donante4/coste1/orden1|241|241|válido|
|practice_0065|mixed|E3|donante5/coste0/orden0|242|242|válido|
|practice_0065|mixed|E3|donante5/coste1/orden0|242|242|válido|
|practice_0065|mixed|E3|donante5/coste0/orden1|242|242|válido|
|practice_0065|mixed|E3|donante5/coste1/orden1|242|242|válido|
|practice_0065|mixed|E3|donante6/coste0/orden0|236|236|válido|
|practice_0065|mixed|E3|donante6/coste1/orden0|236|236|válido|
|practice_0065|mixed|E3|donante6/coste0/orden1|236|236|válido|
|practice_0065|mixed|E3|donante6/coste1/orden1|236|236|válido|
|practice_0065|mixed|E3|donante7/coste0/orden0|237|220|válido|
|practice_0065|mixed|E3|donante7/coste1/orden0|229|220|válido|
|practice_0065|mixed|E3|donante7/coste0/orden1|220|220|válido|
|practice_0065|mixed|E3|donante7/coste1/orden1|220|220|válido|
|practice_0065|mixed|E3|donante8/coste0/orden0|218|216|válido|
|practice_0065|mixed|E3|donante8/coste1/orden0|218|216|válido|
|practice_0065|mixed|E3|donante8/coste0/orden1|218|216|válido|
|practice_0065|mixed|E3|donante8/coste1/orden1|218|216|válido|
|practice_0065|mixed|E3|donante9/coste0/orden0|265|205|válido|
|practice_0065|mixed|E3|donante9/coste1/orden0|230|205|válido|
|practice_0065|mixed|E3|donante9/coste0/orden1|229|205|válido|
|practice_0065|mixed|E3|donante9/coste1/orden1|222|205|válido|
|practice_0066|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0066|multiplicative|Cruce|CP True/coste0/orden0|101|101|válido|
|practice_0066|multiplicative|Cruce|CP True/coste1/orden0|101|101|válido|
|practice_0066|multiplicative|Cruce|CP True/coste0/orden1|101|101|válido|
|practice_0066|multiplicative|Cruce|CP True/coste1/orden1|101|101|válido|
|practice_0066|multiplicative|Cruce|mejor independiente/coste0/orden0|101|101|válido|
|practice_0066|multiplicative|Cruce|mejor independiente/coste1/orden0|101|101|válido|
|practice_0066|multiplicative|Cruce|mejor independiente/coste0/orden1|101|101|válido|
|practice_0066|multiplicative|Cruce|mejor independiente/coste1/orden1|101|101|válido|
|practice_0066|multiplicative|E1|cola10/pol1/inicio0|108|35|válido|
|practice_0066|multiplicative|E1|cola10/pol1/inicio1|108|35|válido|
|practice_0066|multiplicative|E1|cola10/pol2/inicio0|108|35|válido|
|practice_0066|multiplicative|E1|cola10/pol2/inicio1|108|35|válido|
|practice_0066|multiplicative|E1|cola8/pol1/inicio0|108|28|válido|
|practice_0066|multiplicative|E1|cola8/pol1/inicio1|108|28|válido|
|practice_0066|multiplicative|E1|cola8/pol2/inicio0|108|28|válido|
|practice_0066|multiplicative|E1|cola8/pol2/inicio1|108|28|válido|
|practice_0066|multiplicative|E1|cola6/pol1/inicio0|108|23|válido|
|practice_0066|multiplicative|E1|cola6/pol1/inicio1|108|23|válido|
|practice_0066|multiplicative|E1|cola6/pol2/inicio0|108|23|válido|
|practice_0066|multiplicative|E1|cola6/pol2/inicio1|108|23|válido|
|practice_0066|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0066|multiplicative|E2|CP False/coste0/orden0|104|104|válido|
|practice_0066|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0066|multiplicative|E2|CP True/coste0/orden0|101|101|válido|
|practice_0066|multiplicative|E3|donante0/coste0/orden0|106|106|válido|
|practice_0066|multiplicative|E3|donante0/coste1/orden0|106|106|válido|
|practice_0066|multiplicative|E3|donante0/coste0/orden1|106|106|válido|
|practice_0066|multiplicative|E3|donante0/coste1/orden1|106|106|válido|
|practice_0066|multiplicative|E3|donante1/coste0/orden0|112|112|válido|
|practice_0066|multiplicative|E3|donante1/coste1/orden0|112|112|válido|
|practice_0066|multiplicative|E3|donante1/coste0/orden1|112|112|válido|
|practice_0066|multiplicative|E3|donante1/coste1/orden1|112|112|válido|
|practice_0066|multiplicative|E3|donante2/coste0/orden0|111|111|válido|
|practice_0066|multiplicative|E3|donante2/coste1/orden0|111|111|válido|
|practice_0066|multiplicative|E3|donante2/coste0/orden1|111|111|válido|
|practice_0066|multiplicative|E3|donante2/coste1/orden1|111|111|válido|
|practice_0066|multiplicative|E3|donante3/coste0/orden0|107|107|válido|
|practice_0066|multiplicative|E3|donante3/coste1/orden0|107|107|válido|
|practice_0066|multiplicative|E3|donante3/coste0/orden1|107|107|válido|
|practice_0066|multiplicative|E3|donante3/coste1/orden1|107|107|válido|
|practice_0066|multiplicative|E3|donante4/coste0/orden0|117|117|válido|
|practice_0066|multiplicative|E3|donante4/coste1/orden0|117|117|válido|
|practice_0066|multiplicative|E3|donante4/coste0/orden1|117|117|válido|
|practice_0066|multiplicative|E3|donante4/coste1/orden1|117|117|válido|
|practice_0066|multiplicative|E3|donante5/coste0/orden0|115|115|válido|
|practice_0066|multiplicative|E3|donante5/coste1/orden0|115|115|válido|
|practice_0066|multiplicative|E3|donante5/coste0/orden1|115|115|válido|
|practice_0066|multiplicative|E3|donante5/coste1/orden1|115|115|válido|
|practice_0066|multiplicative|E3|donante6/coste0/orden0|119|119|válido|
|practice_0066|multiplicative|E3|donante6/coste1/orden0|119|119|válido|
|practice_0066|multiplicative|E3|donante6/coste0/orden1|119|119|válido|
|practice_0066|multiplicative|E3|donante6/coste1/orden1|119|119|válido|
|practice_0066|multiplicative|E3|donante7/coste0/orden0|108|108|válido|
|practice_0066|multiplicative|E3|donante7/coste1/orden0|108|108|válido|
|practice_0066|multiplicative|E3|donante7/coste0/orden1|108|108|válido|
|practice_0066|multiplicative|E3|donante7/coste1/orden1|108|108|válido|
|practice_0066|multiplicative|E3|donante8/coste0/orden0|106|106|válido|
|practice_0066|multiplicative|E3|donante8/coste1/orden0|106|106|válido|
|practice_0066|multiplicative|E3|donante8/coste0/orden1|106|106|válido|
|practice_0066|multiplicative|E3|donante8/coste1/orden1|106|106|válido|
|practice_0066|multiplicative|E3|donante9/coste0/orden0|111|111|válido|
|practice_0066|multiplicative|E3|donante9/coste1/orden0|111|111|válido|
|practice_0066|multiplicative|E3|donante9/coste0/orden1|111|111|válido|
|practice_0066|multiplicative|E3|donante9/coste1/orden1|111|111|válido|
|practice_0067|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0067|shared|Cruce|CP True/coste0/orden0|147|147|válido|
|practice_0067|shared|Cruce|CP True/coste1/orden0|147|147|válido|
|practice_0067|shared|Cruce|CP True/coste0/orden1|147|147|válido|
|practice_0067|shared|Cruce|CP True/coste1/orden1|147|147|válido|
|practice_0067|shared|Cruce|mejor independiente/coste0/orden0|147|147|válido|
|practice_0067|shared|Cruce|mejor independiente/coste1/orden0|147|147|válido|
|practice_0067|shared|Cruce|mejor independiente/coste0/orden1|147|147|válido|
|practice_0067|shared|Cruce|mejor independiente/coste1/orden1|147|147|válido|
|practice_0067|shared|E1|cola10/pol1/inicio0|154|48|válido|
|practice_0067|shared|E1|cola10/pol1/inicio1|154|48|válido|
|practice_0067|shared|E1|cola10/pol2/inicio0|153|47|válido|
|practice_0067|shared|E1|cola10/pol2/inicio1|153|47|válido|
|practice_0067|shared|E1|cola8/pol1/inicio0|150|35|válido|
|practice_0067|shared|E1|cola8/pol1/inicio1|150|35|válido|
|practice_0067|shared|E1|cola8/pol2/inicio0|150|35|válido|
|practice_0067|shared|E1|cola8/pol2/inicio1|150|35|válido|
|practice_0067|shared|E1|cola6/pol1/inicio0|153|30|válido|
|practice_0067|shared|E1|cola6/pol1/inicio1|153|30|válido|
|practice_0067|shared|E1|cola6/pol2/inicio0|153|30|válido|
|practice_0067|shared|E1|cola6/pol2/inicio1|153|30|válido|
|practice_0067|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0067|shared|E2|CP False/coste0/orden0|150|150|válido|
|practice_0067|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0067|shared|E2|CP True/coste0/orden0|148|148|válido|
|practice_0067|shared|E3|donante0/coste0/orden0|150|150|válido|
|practice_0067|shared|E3|donante0/coste1/orden0|150|150|válido|
|practice_0067|shared|E3|donante0/coste0/orden1|150|150|válido|
|practice_0067|shared|E3|donante0/coste1/orden1|150|150|válido|
|practice_0067|shared|E3|donante1/coste0/orden0|173|173|válido|
|practice_0067|shared|E3|donante1/coste1/orden0|173|173|válido|
|practice_0067|shared|E3|donante1/coste0/orden1|173|173|válido|
|practice_0067|shared|E3|donante1/coste1/orden1|173|173|válido|
|practice_0067|shared|E3|donante2/coste0/orden0|165|165|válido|
|practice_0067|shared|E3|donante2/coste1/orden0|165|165|válido|
|practice_0067|shared|E3|donante2/coste0/orden1|165|165|válido|
|practice_0067|shared|E3|donante2/coste1/orden1|165|165|válido|
|practice_0067|shared|E3|donante3/coste0/orden0|160|160|válido|
|practice_0067|shared|E3|donante3/coste1/orden0|160|160|válido|
|practice_0067|shared|E3|donante3/coste0/orden1|160|160|válido|
|practice_0067|shared|E3|donante3/coste1/orden1|160|160|válido|
|practice_0067|shared|E3|donante4/coste0/orden0|194|194|válido|
|practice_0067|shared|E3|donante4/coste1/orden0|194|194|válido|
|practice_0067|shared|E3|donante4/coste0/orden1|194|194|válido|
|practice_0067|shared|E3|donante4/coste1/orden1|194|194|válido|
|practice_0067|shared|E3|donante5/coste0/orden0|172|172|válido|
|practice_0067|shared|E3|donante5/coste1/orden0|172|172|válido|
|practice_0067|shared|E3|donante5/coste0/orden1|172|172|válido|
|practice_0067|shared|E3|donante5/coste1/orden1|172|172|válido|
|practice_0067|shared|E3|donante6/coste0/orden0|192|192|válido|
|practice_0067|shared|E3|donante6/coste1/orden0|192|192|válido|
|practice_0067|shared|E3|donante6/coste0/orden1|192|192|válido|
|practice_0067|shared|E3|donante6/coste1/orden1|192|192|válido|
|practice_0067|shared|E3|donante7/coste0/orden0|186|163|válido|
|practice_0067|shared|E3|donante7/coste1/orden0|173|163|válido|
|practice_0067|shared|E3|donante7/coste0/orden1|181|163|válido|
|practice_0067|shared|E3|donante7/coste1/orden1|169|163|válido|
|practice_0067|shared|E3|donante8/coste0/orden0|156|156|válido|
|practice_0067|shared|E3|donante8/coste1/orden0|156|156|válido|
|practice_0067|shared|E3|donante8/coste0/orden1|156|156|válido|
|practice_0067|shared|E3|donante8/coste1/orden1|156|156|válido|
|practice_0068|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0068|mixed|Cruce|CP True/coste0/orden0|228|179|válido|
|practice_0068|mixed|Cruce|CP True/coste1/orden0|192|179|válido|
|practice_0068|mixed|Cruce|CP True/coste0/orden1|190|179|válido|
|practice_0068|mixed|Cruce|CP True/coste1/orden1|184|179|válido|
|practice_0068|mixed|Cruce|mejor independiente/coste0/orden0|223|180|válido|
|practice_0068|mixed|Cruce|mejor independiente/coste1/orden0|187|180|válido|
|practice_0068|mixed|Cruce|mejor independiente/coste0/orden1|181|180|válido|
|practice_0068|mixed|Cruce|mejor independiente/coste1/orden1|181|180|válido|
|practice_0068|mixed|E1|cola10/pol1/inicio0|189|62|válido|
|practice_0068|mixed|E1|cola10/pol1/inicio1|189|62|válido|
|practice_0068|mixed|E1|cola10/pol2/inicio0|189|62|válido|
|practice_0068|mixed|E1|cola10/pol2/inicio1|189|62|válido|
|practice_0068|mixed|E1|cola8/pol1/inicio0|189|51|válido|
|practice_0068|mixed|E1|cola8/pol1/inicio1|189|51|válido|
|practice_0068|mixed|E1|cola8/pol2/inicio0|189|51|válido|
|practice_0068|mixed|E1|cola8/pol2/inicio1|189|51|válido|
|practice_0068|mixed|E1|cola6/pol1/inicio0|189|39|válido|
|practice_0068|mixed|E1|cola6/pol1/inicio1|189|39|válido|
|practice_0068|mixed|E1|cola6/pol2/inicio0|189|39|válido|
|practice_0068|mixed|E1|cola6/pol2/inicio1|189|39|válido|
|practice_0068|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0068|mixed|E2|CP False/coste0/orden0|199|199|válido|
|practice_0068|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0068|mixed|E2|CP True/coste0/orden0|223|180|válido|
|practice_0068|mixed|E3|donante0/coste0/orden0|181|180|válido|
|practice_0068|mixed|E3|donante0/coste1/orden0|181|180|válido|
|practice_0068|mixed|E3|donante0/coste0/orden1|181|180|válido|
|practice_0068|mixed|E3|donante0/coste1/orden1|181|180|válido|
|practice_0068|mixed|E3|donante1/coste0/orden0|198|198|válido|
|practice_0068|mixed|E3|donante1/coste1/orden0|198|198|válido|
|practice_0068|mixed|E3|donante1/coste0/orden1|198|198|válido|
|practice_0068|mixed|E3|donante1/coste1/orden1|198|198|válido|
|practice_0068|mixed|E3|donante2/coste0/orden0|193|193|válido|
|practice_0068|mixed|E3|donante2/coste1/orden0|193|193|válido|
|practice_0068|mixed|E3|donante2/coste0/orden1|193|193|válido|
|practice_0068|mixed|E3|donante2/coste1/orden1|193|193|válido|
|practice_0068|mixed|E3|donante3/coste0/orden0|184|184|válido|
|practice_0068|mixed|E3|donante3/coste1/orden0|184|184|válido|
|practice_0068|mixed|E3|donante3/coste0/orden1|184|184|válido|
|practice_0068|mixed|E3|donante3/coste1/orden1|184|184|válido|
|practice_0068|mixed|E3|donante4/coste0/orden0|200|200|válido|
|practice_0068|mixed|E3|donante4/coste1/orden0|200|200|válido|
|practice_0068|mixed|E3|donante4/coste0/orden1|200|200|válido|
|practice_0068|mixed|E3|donante4/coste1/orden1|200|200|válido|
|practice_0068|mixed|E3|donante5/coste0/orden0|204|204|válido|
|practice_0068|mixed|E3|donante5/coste1/orden0|204|204|válido|
|practice_0068|mixed|E3|donante5/coste0/orden1|204|204|válido|
|practice_0068|mixed|E3|donante5/coste1/orden1|204|204|válido|
|practice_0068|mixed|E3|donante6/coste0/orden0|205|205|válido|
|practice_0068|mixed|E3|donante6/coste1/orden0|205|205|válido|
|practice_0068|mixed|E3|donante6/coste0/orden1|205|205|válido|
|practice_0068|mixed|E3|donante6/coste1/orden1|205|205|válido|
|practice_0068|mixed|E3|donante7/coste0/orden0|211|185|válido|
|practice_0068|mixed|E3|donante7/coste1/orden0|198|185|válido|
|practice_0068|mixed|E3|donante7/coste0/orden1|221|185|válido|
|practice_0068|mixed|E3|donante7/coste1/orden1|215|185|válido|
|practice_0068|mixed|E3|donante8/coste0/orden0|205|180|válido|
|practice_0068|mixed|E3|donante8/coste1/orden0|185|180|válido|
|practice_0068|mixed|E3|donante8/coste0/orden1|211|180|válido|
|practice_0068|mixed|E3|donante8/coste1/orden1|206|180|válido|
|practice_0069|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0069|independent|Cruce|CP True/coste0/orden0|205|192|válido|
|practice_0069|independent|Cruce|CP True/coste1/orden0|215|192|válido|
|practice_0069|independent|Cruce|CP True/coste0/orden1|192|192|válido|
|practice_0069|independent|Cruce|CP True/coste1/orden1|192|192|válido|
|practice_0069|independent|Cruce|mejor independiente/coste0/orden0|205|192|válido|
|practice_0069|independent|Cruce|mejor independiente/coste1/orden0|215|192|válido|
|practice_0069|independent|Cruce|mejor independiente/coste0/orden1|192|192|válido|
|practice_0069|independent|Cruce|mejor independiente/coste1/orden1|192|192|válido|
|practice_0069|independent|E1|cola10/pol1/inicio0|204|63|válido|
|practice_0069|independent|E1|cola10/pol1/inicio1|204|63|válido|
|practice_0069|independent|E1|cola10/pol2/inicio0|204|63|válido|
|practice_0069|independent|E1|cola10/pol2/inicio1|204|63|válido|
|practice_0069|independent|E1|cola8/pol1/inicio0|205|54|válido|
|practice_0069|independent|E1|cola8/pol1/inicio1|205|54|válido|
|practice_0069|independent|E1|cola8/pol2/inicio0|205|54|válido|
|practice_0069|independent|E1|cola8/pol2/inicio1|205|54|válido|
|practice_0069|independent|E1|cola6/pol1/inicio0|202|40|válido|
|practice_0069|independent|E1|cola6/pol1/inicio1|202|40|válido|
|practice_0069|independent|E1|cola6/pol2/inicio0|202|40|válido|
|practice_0069|independent|E1|cola6/pol2/inicio1|202|40|válido|
|practice_0069|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0069|independent|E2|CP False/coste0/orden0|217|217|válido|
|practice_0069|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0069|independent|E2|CP True/coste0/orden0|205|192|válido|
|practice_0069|independent|E3|donante0/coste0/orden0|192|192|válido|
|practice_0069|independent|E3|donante0/coste1/orden0|192|192|válido|
|practice_0069|independent|E3|donante0/coste0/orden1|192|192|válido|
|practice_0069|independent|E3|donante0/coste1/orden1|192|192|válido|
|practice_0069|independent|E3|donante1/coste0/orden0|220|220|válido|
|practice_0069|independent|E3|donante1/coste1/orden0|220|220|válido|
|practice_0069|independent|E3|donante1/coste0/orden1|220|220|válido|
|practice_0069|independent|E3|donante1/coste1/orden1|220|220|válido|
|practice_0069|independent|E3|donante2/coste0/orden0|209|209|válido|
|practice_0069|independent|E3|donante2/coste1/orden0|209|209|válido|
|practice_0069|independent|E3|donante2/coste0/orden1|209|209|válido|
|practice_0069|independent|E3|donante2/coste1/orden1|209|209|válido|
|practice_0069|independent|E3|donante3/coste0/orden0|262|204|válido|
|practice_0069|independent|E3|donante3/coste1/orden0|219|204|válido|
|practice_0069|independent|E3|donante3/coste0/orden1|204|204|válido|
|practice_0069|independent|E3|donante3/coste1/orden1|204|204|válido|
|practice_0069|independent|E3|donante4/coste0/orden0|217|217|válido|
|practice_0069|independent|E3|donante4/coste1/orden0|217|217|válido|
|practice_0069|independent|E3|donante4/coste0/orden1|217|217|válido|
|practice_0069|independent|E3|donante4/coste1/orden1|217|217|válido|
|practice_0069|independent|E3|donante5/coste0/orden0|218|218|válido|
|practice_0069|independent|E3|donante5/coste1/orden0|218|218|válido|
|practice_0069|independent|E3|donante5/coste0/orden1|218|218|válido|
|practice_0069|independent|E3|donante5/coste1/orden1|218|218|válido|
|practice_0069|independent|E3|donante6/coste0/orden0|221|221|válido|
|practice_0069|independent|E3|donante6/coste1/orden0|221|221|válido|
|practice_0069|independent|E3|donante6/coste0/orden1|221|221|válido|
|practice_0069|independent|E3|donante6/coste1/orden1|221|221|válido|
|practice_0069|independent|E3|donante7/coste0/orden0|211|204|válido|
|practice_0069|independent|E3|donante7/coste1/orden0|211|204|válido|
|practice_0069|independent|E3|donante7/coste0/orden1|231|204|válido|
|practice_0069|independent|E3|donante7/coste1/orden1|231|204|válido|
|practice_0069|independent|E3|donante8/coste0/orden0|200|191|válido|
|practice_0069|independent|E3|donante8/coste1/orden0|200|191|válido|
|practice_0069|independent|E3|donante8/coste0/orden1|204|191|válido|
|practice_0069|independent|E3|donante8/coste1/orden1|207|191|válido|
|practice_0070|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0070|independent|Cruce|CP True/coste0/orden0|277|236|válido|
|practice_0070|independent|Cruce|CP True/coste1/orden0|263|236|válido|
|practice_0070|independent|Cruce|CP True/coste0/orden1|236|236|válido|
|practice_0070|independent|Cruce|CP True/coste1/orden1|236|236|válido|
|practice_0070|independent|Cruce|mejor independiente/coste0/orden0|276|237|válido|
|practice_0070|independent|Cruce|mejor independiente/coste1/orden0|264|237|válido|
|practice_0070|independent|Cruce|mejor independiente/coste0/orden1|237|237|válido|
|practice_0070|independent|Cruce|mejor independiente/coste1/orden1|237|237|válido|
|practice_0070|independent|E1|cola10/pol1/inicio0|245|74|válido|
|practice_0070|independent|E1|cola10/pol1/inicio1|245|74|válido|
|practice_0070|independent|E1|cola10/pol2/inicio0|245|74|válido|
|practice_0070|independent|E1|cola10/pol2/inicio1|245|74|válido|
|practice_0070|independent|E1|cola8/pol1/inicio0|242|57|válido|
|practice_0070|independent|E1|cola8/pol1/inicio1|242|57|válido|
|practice_0070|independent|E1|cola8/pol2/inicio0|246|61|válido|
|practice_0070|independent|E1|cola8/pol2/inicio1|246|61|válido|
|practice_0070|independent|E1|cola6/pol1/inicio0|247|48|válido|
|practice_0070|independent|E1|cola6/pol1/inicio1|247|48|válido|
|practice_0070|independent|E1|cola6/pol2/inicio0|245|46|válido|
|practice_0070|independent|E1|cola6/pol2/inicio1|245|46|válido|
|practice_0070|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0070|independent|E2|CP False/coste0/orden0|276|237|válido|
|practice_0070|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0070|independent|E2|CP True/coste0/orden0|277|236|válido|
|practice_0070|independent|E3|donante0/coste0/orden0|237|237|válido|
|practice_0070|independent|E3|donante0/coste1/orden0|237|237|válido|
|practice_0070|independent|E3|donante0/coste0/orden1|237|237|válido|
|practice_0070|independent|E3|donante0/coste1/orden1|237|237|válido|
|practice_0070|independent|E3|donante1/coste0/orden0|271|271|válido|
|practice_0070|independent|E3|donante1/coste1/orden0|271|271|válido|
|practice_0070|independent|E3|donante1/coste0/orden1|271|271|válido|
|practice_0070|independent|E3|donante1/coste1/orden1|271|271|válido|
|practice_0070|independent|E3|donante2/coste0/orden0|374|255|válido|
|practice_0070|independent|E3|donante2/coste1/orden0|265|255|válido|
|practice_0070|independent|E3|donante2/coste0/orden1|255|255|válido|
|practice_0070|independent|E3|donante2/coste1/orden1|255|255|válido|
|practice_0070|independent|E3|donante3/coste0/orden0|358|253|válido|
|practice_0070|independent|E3|donante3/coste1/orden0|273|253|válido|
|practice_0070|independent|E3|donante3/coste0/orden1|253|253|válido|
|practice_0070|independent|E3|donante3/coste1/orden1|253|253|válido|
|practice_0070|independent|E3|donante4/coste0/orden0|272|272|válido|
|practice_0070|independent|E3|donante4/coste1/orden0|272|272|válido|
|practice_0070|independent|E3|donante4/coste0/orden1|272|272|válido|
|practice_0070|independent|E3|donante4/coste1/orden1|272|272|válido|
|practice_0070|independent|E3|donante5/coste0/orden0|267|267|válido|
|practice_0070|independent|E3|donante5/coste1/orden0|267|267|válido|
|practice_0070|independent|E3|donante5/coste0/orden1|267|267|válido|
|practice_0070|independent|E3|donante5/coste1/orden1|267|267|válido|
|practice_0070|independent|E3|donante6/coste0/orden0|283|283|válido|
|practice_0070|independent|E3|donante6/coste1/orden0|283|283|válido|
|practice_0070|independent|E3|donante6/coste0/orden1|283|283|válido|
|practice_0070|independent|E3|donante6/coste1/orden1|283|283|válido|
|practice_0070|independent|E3|donante7/coste0/orden0|269|252|válido|
|practice_0070|independent|E3|donante7/coste1/orden0|269|252|válido|
|practice_0070|independent|E3|donante7/coste0/orden1|299|252|válido|
|practice_0070|independent|E3|donante7/coste1/orden1|279|252|válido|
|practice_0070|independent|E3|donante8/coste0/orden0|256|238|válido|
|practice_0070|independent|E3|donante8/coste1/orden0|242|238|válido|
|practice_0070|independent|E3|donante8/coste0/orden1|238|238|válido|
|practice_0070|independent|E3|donante8/coste1/orden1|238|238|válido|
|practice_0071|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0071|shared|Cruce|CP True/coste0/orden0|156|156|válido|
|practice_0071|shared|Cruce|CP True/coste1/orden0|156|156|válido|
|practice_0071|shared|Cruce|CP True/coste0/orden1|156|156|válido|
|practice_0071|shared|Cruce|CP True/coste1/orden1|156|156|válido|
|practice_0071|shared|Cruce|mejor independiente/coste0/orden0|157|157|válido|
|practice_0071|shared|Cruce|mejor independiente/coste1/orden0|157|157|válido|
|practice_0071|shared|Cruce|mejor independiente/coste0/orden1|157|157|válido|
|practice_0071|shared|Cruce|mejor independiente/coste1/orden1|157|157|válido|
|practice_0071|shared|E1|cola10/pol1/inicio0|173|64|válido|
|practice_0071|shared|E1|cola10/pol1/inicio1|173|64|válido|
|practice_0071|shared|E1|cola10/pol2/inicio0|166|57|válido|
|practice_0071|shared|E1|cola10/pol2/inicio1|171|62|válido|
|practice_0071|shared|E1|cola8/pol1/inicio0|167|45|válido|
|practice_0071|shared|E1|cola8/pol1/inicio1|167|45|válido|
|practice_0071|shared|E1|cola8/pol2/inicio0|167|45|válido|
|practice_0071|shared|E1|cola8/pol2/inicio1|167|45|válido|
|practice_0071|shared|E1|cola6/pol1/inicio0|169|37|válido|
|practice_0071|shared|E1|cola6/pol1/inicio1|169|37|válido|
|practice_0071|shared|E1|cola6/pol2/inicio0|169|37|válido|
|practice_0071|shared|E1|cola6/pol2/inicio1|169|37|válido|
|practice_0071|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0071|shared|E2|CP False/coste0/orden0|168|168|válido|
|practice_0071|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0071|shared|E2|CP True/coste0/orden0|157|157|válido|
|practice_0071|shared|E3|donante0/coste0/orden0|161|161|válido|
|practice_0071|shared|E3|donante0/coste1/orden0|161|161|válido|
|practice_0071|shared|E3|donante0/coste0/orden1|161|161|válido|
|practice_0071|shared|E3|donante0/coste1/orden1|161|161|válido|
|practice_0071|shared|E3|donante1/coste0/orden0|169|169|válido|
|practice_0071|shared|E3|donante1/coste1/orden0|169|169|válido|
|practice_0071|shared|E3|donante1/coste0/orden1|169|169|válido|
|practice_0071|shared|E3|donante1/coste1/orden1|169|169|válido|
|practice_0071|shared|E3|donante2/coste0/orden0|168|168|válido|
|practice_0071|shared|E3|donante2/coste1/orden0|168|168|válido|
|practice_0071|shared|E3|donante2/coste0/orden1|168|168|válido|
|practice_0071|shared|E3|donante2/coste1/orden1|168|168|válido|
|practice_0071|shared|E3|donante3/coste0/orden0|168|168|válido|
|practice_0071|shared|E3|donante3/coste1/orden0|168|168|válido|
|practice_0071|shared|E3|donante3/coste0/orden1|168|168|válido|
|practice_0071|shared|E3|donante3/coste1/orden1|168|168|válido|
|practice_0071|shared|E3|donante4/coste0/orden0|202|202|válido|
|practice_0071|shared|E3|donante4/coste1/orden0|202|202|válido|
|practice_0071|shared|E3|donante4/coste0/orden1|202|202|válido|
|practice_0071|shared|E3|donante4/coste1/orden1|202|202|válido|
|practice_0071|shared|E3|donante5/coste0/orden0|173|173|válido|
|practice_0071|shared|E3|donante5/coste1/orden0|173|173|válido|
|practice_0071|shared|E3|donante5/coste0/orden1|173|173|válido|
|practice_0071|shared|E3|donante5/coste1/orden1|173|173|válido|
|practice_0071|shared|E3|donante6/coste0/orden0|186|186|válido|
|practice_0071|shared|E3|donante6/coste1/orden0|186|186|válido|
|practice_0071|shared|E3|donante6/coste0/orden1|186|186|válido|
|practice_0071|shared|E3|donante6/coste1/orden1|186|186|válido|
|practice_0071|shared|E3|donante7/coste0/orden0|165|165|válido|
|practice_0071|shared|E3|donante7/coste1/orden0|165|165|válido|
|practice_0071|shared|E3|donante7/coste0/orden1|165|165|válido|
|practice_0071|shared|E3|donante7/coste1/orden1|165|165|válido|
|practice_0071|shared|E3|donante8/coste0/orden0|168|166|válido|
|practice_0071|shared|E3|donante8/coste1/orden0|171|166|válido|
|practice_0071|shared|E3|donante8/coste0/orden1|166|166|válido|
|practice_0071|shared|E3|donante8/coste1/orden1|166|166|válido|
|practice_0072|tiny|Cruce|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0072|tiny|Cruce|CP True/coste0/orden0|5|5|válido|
|practice_0072|tiny|Cruce|CP True/coste1/orden0|5|5|válido|
|practice_0072|tiny|Cruce|CP True/coste0/orden1|5|5|válido|
|practice_0072|tiny|Cruce|CP True/coste1/orden1|5|5|válido|
|practice_0072|tiny|Cruce|mejor independiente/coste0/orden0|5|5|válido|
|practice_0072|tiny|Cruce|mejor independiente/coste1/orden0|5|5|válido|
|practice_0072|tiny|Cruce|mejor independiente/coste0/orden1|5|5|válido|
|practice_0072|tiny|Cruce|mejor independiente/coste1/orden1|5|5|válido|
|practice_0072|tiny|E1|cola4/pol1/inicio0|10|10|válido|
|practice_0072|tiny|E1|cola4/pol1/inicio1|10|10|válido|
|practice_0072|tiny|E1|cola4/pol2/inicio0|10|10|válido|
|practice_0072|tiny|E1|cola4/pol2/inicio1|10|10|válido|
|practice_0072|tiny|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0072|tiny|E2|CP False/coste0/orden0|5|5|válido|
|practice_0072|tiny|E2|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0072|tiny|E2|CP True/coste0/orden0|5|5|válido|
|practice_0072|tiny|E3|donante0/coste0/orden0|5|5|válido|
|practice_0072|tiny|E3|donante0/coste1/orden0|5|5|válido|
|practice_0072|tiny|E3|donante0/coste0/orden1|5|5|válido|
|practice_0072|tiny|E3|donante0/coste1/orden1|5|5|válido|
|practice_0072|tiny|E3|donante1/coste0/orden0|8|8|válido|
|practice_0072|tiny|E3|donante1/coste1/orden0|8|8|válido|
|practice_0072|tiny|E3|donante1/coste0/orden1|8|8|válido|
|practice_0072|tiny|E3|donante1/coste1/orden1|8|8|válido|
|practice_0072|tiny|E3|donante2/coste0/orden0|10|10|válido|
|practice_0072|tiny|E3|donante2/coste1/orden0|10|10|válido|
|practice_0072|tiny|E3|donante2/coste0/orden1|10|10|válido|
|practice_0072|tiny|E3|donante2/coste1/orden1|10|10|válido|
|practice_0073|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0073|shared|Cruce|CP True/coste0/orden0|141|141|válido|
|practice_0073|shared|Cruce|CP True/coste1/orden0|141|141|válido|
|practice_0073|shared|Cruce|CP True/coste0/orden1|141|141|válido|
|practice_0073|shared|Cruce|CP True/coste1/orden1|141|141|válido|
|practice_0073|shared|Cruce|mejor independiente/coste0/orden0|141|141|válido|
|practice_0073|shared|Cruce|mejor independiente/coste1/orden0|141|141|válido|
|practice_0073|shared|Cruce|mejor independiente/coste0/orden1|141|141|válido|
|practice_0073|shared|Cruce|mejor independiente/coste1/orden1|141|141|válido|
|practice_0073|shared|E1|cola10/pol1/inicio0|149|50|válido|
|practice_0073|shared|E1|cola10/pol1/inicio1|149|50|válido|
|practice_0073|shared|E1|cola10/pol2/inicio0|149|50|válido|
|practice_0073|shared|E1|cola10/pol2/inicio1|149|50|válido|
|practice_0073|shared|E1|cola8/pol1/inicio0|144|35|válido|
|practice_0073|shared|E1|cola8/pol1/inicio1|144|35|válido|
|practice_0073|shared|E1|cola8/pol2/inicio0|149|40|válido|
|practice_0073|shared|E1|cola8/pol2/inicio1|149|40|válido|
|practice_0073|shared|E1|cola6/pol1/inicio0|143|24|válido|
|practice_0073|shared|E1|cola6/pol1/inicio1|143|24|válido|
|practice_0073|shared|E1|cola6/pol2/inicio0|143|24|válido|
|practice_0073|shared|E1|cola6/pol2/inicio1|143|24|válido|
|practice_0073|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0073|shared|E2|CP False/coste0/orden0|155|155|válido|
|practice_0073|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0073|shared|E2|CP True/coste0/orden0|141|141|válido|
|practice_0073|shared|E3|donante0/coste0/orden0|144|144|válido|
|practice_0073|shared|E3|donante0/coste1/orden0|144|144|válido|
|practice_0073|shared|E3|donante0/coste0/orden1|144|144|válido|
|practice_0073|shared|E3|donante0/coste1/orden1|144|144|válido|
|practice_0073|shared|E3|donante1/coste0/orden0|159|159|válido|
|practice_0073|shared|E3|donante1/coste1/orden0|159|159|válido|
|practice_0073|shared|E3|donante1/coste0/orden1|159|159|válido|
|practice_0073|shared|E3|donante1/coste1/orden1|159|159|válido|
|practice_0073|shared|E3|donante2/coste0/orden0|147|147|válido|
|practice_0073|shared|E3|donante2/coste1/orden0|147|147|válido|
|practice_0073|shared|E3|donante2/coste0/orden1|147|147|válido|
|practice_0073|shared|E3|donante2/coste1/orden1|147|147|válido|
|practice_0073|shared|E3|donante3/coste0/orden0|145|145|válido|
|practice_0073|shared|E3|donante3/coste1/orden0|145|145|válido|
|practice_0073|shared|E3|donante3/coste0/orden1|145|145|válido|
|practice_0073|shared|E3|donante3/coste1/orden1|145|145|válido|
|practice_0073|shared|E3|donante4/coste0/orden0|172|172|válido|
|practice_0073|shared|E3|donante4/coste1/orden0|172|172|válido|
|practice_0073|shared|E3|donante4/coste0/orden1|172|172|válido|
|practice_0073|shared|E3|donante4/coste1/orden1|172|172|válido|
|practice_0073|shared|E3|donante5/coste0/orden0|165|165|válido|
|practice_0073|shared|E3|donante5/coste1/orden0|165|165|válido|
|practice_0073|shared|E3|donante5/coste0/orden1|165|165|válido|
|practice_0073|shared|E3|donante5/coste1/orden1|165|165|válido|
|practice_0073|shared|E3|donante6/coste0/orden0|165|165|válido|
|practice_0073|shared|E3|donante6/coste1/orden0|165|165|válido|
|practice_0073|shared|E3|donante6/coste0/orden1|165|165|válido|
|practice_0073|shared|E3|donante6/coste1/orden1|165|165|válido|
|practice_0073|shared|E3|donante7/coste0/orden0|155|151|válido|
|practice_0073|shared|E3|donante7/coste1/orden0|157|151|válido|
|practice_0073|shared|E3|donante7/coste0/orden1|151|151|válido|
|practice_0073|shared|E3|donante7/coste1/orden1|151|151|válido|
|practice_0073|shared|E3|donante8/coste0/orden0|145|145|válido|
|practice_0073|shared|E3|donante8/coste1/orden0|145|145|válido|
|practice_0073|shared|E3|donante8/coste0/orden1|145|145|válido|
|practice_0073|shared|E3|donante8/coste1/orden1|145|145|válido|
|practice_0073|shared|E3|donante9/coste0/orden0|145|145|válido|
|practice_0073|shared|E3|donante9/coste1/orden0|145|145|válido|
|practice_0073|shared|E3|donante9/coste0/orden1|145|145|válido|
|practice_0073|shared|E3|donante9/coste1/orden1|145|145|válido|
|practice_0074|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0074|shared|Cruce|CP True/coste0/orden0|185|185|válido|
|practice_0074|shared|Cruce|CP True/coste1/orden0|185|185|válido|
|practice_0074|shared|Cruce|CP True/coste0/orden1|186|185|válido|
|practice_0074|shared|Cruce|CP True/coste1/orden1|186|185|válido|
|practice_0074|shared|Cruce|mejor independiente/coste0/orden0|186|186|válido|
|practice_0074|shared|Cruce|mejor independiente/coste1/orden0|186|186|válido|
|practice_0074|shared|Cruce|mejor independiente/coste0/orden1|192|186|válido|
|practice_0074|shared|Cruce|mejor independiente/coste1/orden1|192|186|válido|
|practice_0074|shared|E1|cola10/pol1/inicio0|189|54|válido|
|practice_0074|shared|E1|cola10/pol1/inicio1|189|54|válido|
|practice_0074|shared|E1|cola10/pol2/inicio0|189|54|válido|
|practice_0074|shared|E1|cola10/pol2/inicio1|189|54|válido|
|practice_0074|shared|E1|cola8/pol1/inicio0|189|44|válido|
|practice_0074|shared|E1|cola8/pol1/inicio1|189|44|válido|
|practice_0074|shared|E1|cola8/pol2/inicio0|188|43|válido|
|practice_0074|shared|E1|cola8/pol2/inicio1|188|43|válido|
|practice_0074|shared|E1|cola6/pol1/inicio0|189|34|válido|
|practice_0074|shared|E1|cola6/pol1/inicio1|189|34|válido|
|practice_0074|shared|E1|cola6/pol2/inicio0|189|34|válido|
|practice_0074|shared|E1|cola6/pol2/inicio1|189|34|válido|
|practice_0074|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0074|shared|E2|CP False/coste0/orden0|186|186|válido|
|practice_0074|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0074|shared|E2|CP True/coste0/orden0|201|184|válido|
|practice_0074|shared|E3|donante0/coste0/orden0|188|188|válido|
|practice_0074|shared|E3|donante0/coste1/orden0|188|188|válido|
|practice_0074|shared|E3|donante0/coste0/orden1|198|188|válido|
|practice_0074|shared|E3|donante0/coste1/orden1|198|188|válido|
|practice_0074|shared|E3|donante1/coste0/orden0|226|226|válido|
|practice_0074|shared|E3|donante1/coste1/orden0|226|226|válido|
|practice_0074|shared|E3|donante1/coste0/orden1|226|226|válido|
|practice_0074|shared|E3|donante1/coste1/orden1|226|226|válido|
|practice_0074|shared|E3|donante2/coste0/orden0|210|205|válido|
|practice_0074|shared|E3|donante2/coste1/orden0|210|205|válido|
|practice_0074|shared|E3|donante2/coste0/orden1|205|205|válido|
|practice_0074|shared|E3|donante2/coste1/orden1|205|205|válido|
|practice_0074|shared|E3|donante3/coste0/orden0|220|209|válido|
|practice_0074|shared|E3|donante3/coste1/orden0|217|209|válido|
|practice_0074|shared|E3|donante3/coste0/orden1|209|209|válido|
|practice_0074|shared|E3|donante3/coste1/orden1|209|209|válido|
|practice_0074|shared|E3|donante4/coste0/orden0|253|253|válido|
|practice_0074|shared|E3|donante4/coste1/orden0|253|253|válido|
|practice_0074|shared|E3|donante4/coste0/orden1|253|253|válido|
|practice_0074|shared|E3|donante4/coste1/orden1|253|253|válido|
|practice_0074|shared|E3|donante5/coste0/orden0|216|216|válido|
|practice_0074|shared|E3|donante5/coste1/orden0|216|216|válido|
|practice_0074|shared|E3|donante5/coste0/orden1|216|216|válido|
|practice_0074|shared|E3|donante5/coste1/orden1|216|216|válido|
|practice_0074|shared|E3|donante6/coste0/orden0|228|228|válido|
|practice_0074|shared|E3|donante6/coste1/orden0|228|228|válido|
|practice_0074|shared|E3|donante6/coste0/orden1|228|228|válido|
|practice_0074|shared|E3|donante6/coste1/orden1|228|228|válido|
|practice_0074|shared|E3|donante7/coste0/orden0|209|202|válido|
|practice_0074|shared|E3|donante7/coste1/orden0|208|202|válido|
|practice_0074|shared|E3|donante7/coste0/orden1|247|202|válido|
|practice_0074|shared|E3|donante7/coste1/orden1|228|202|válido|
|practice_0074|shared|E3|donante8/coste0/orden0|196|196|válido|
|practice_0074|shared|E3|donante8/coste1/orden0|196|196|válido|
|practice_0074|shared|E3|donante8/coste0/orden1|196|196|válido|
|practice_0074|shared|E3|donante8/coste1/orden1|196|196|válido|
|practice_0075|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0075|mixed|Cruce|CP True/coste0/orden0|236|179|válido|
|practice_0075|mixed|Cruce|CP True/coste1/orden0|216|179|válido|
|practice_0075|mixed|Cruce|CP True/coste0/orden1|221|179|válido|
|practice_0075|mixed|Cruce|CP True/coste1/orden1|185|179|válido|
|practice_0075|mixed|Cruce|mejor independiente/coste0/orden0|228|180|válido|
|practice_0075|mixed|Cruce|mejor independiente/coste1/orden0|226|180|válido|
|practice_0075|mixed|Cruce|mejor independiente/coste0/orden1|216|180|válido|
|practice_0075|mixed|Cruce|mejor independiente/coste1/orden1|186|180|válido|
|practice_0075|mixed|E1|cola10/pol1/inicio0|196|58|válido|
|practice_0075|mixed|E1|cola10/pol1/inicio1|196|58|válido|
|practice_0075|mixed|E1|cola10/pol2/inicio0|191|53|válido|
|practice_0075|mixed|E1|cola10/pol2/inicio1|191|53|válido|
|practice_0075|mixed|E1|cola8/pol1/inicio0|192|45|válido|
|practice_0075|mixed|E1|cola8/pol1/inicio1|192|45|válido|
|practice_0075|mixed|E1|cola8/pol2/inicio0|194|47|válido|
|practice_0075|mixed|E1|cola8/pol2/inicio1|191|44|válido|
|practice_0075|mixed|E1|cola6/pol1/inicio0|191|33|válido|
|practice_0075|mixed|E1|cola6/pol1/inicio1|191|33|válido|
|practice_0075|mixed|E1|cola6/pol2/inicio0|191|33|válido|
|practice_0075|mixed|E1|cola6/pol2/inicio1|191|33|válido|
|practice_0075|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0075|mixed|E2|CP False/coste0/orden0|214|190|válido|
|practice_0075|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0075|mixed|E2|CP True/coste0/orden0|188|188|válido|
|practice_0075|mixed|E3|donante0/coste0/orden0|190|190|válido|
|practice_0075|mixed|E3|donante0/coste1/orden0|190|190|válido|
|practice_0075|mixed|E3|donante0/coste0/orden1|190|190|válido|
|practice_0075|mixed|E3|donante0/coste1/orden1|190|190|válido|
|practice_0075|mixed|E3|donante1/coste0/orden0|199|199|válido|
|practice_0075|mixed|E3|donante1/coste1/orden0|199|199|válido|
|practice_0075|mixed|E3|donante1/coste0/orden1|199|199|válido|
|practice_0075|mixed|E3|donante1/coste1/orden1|199|199|válido|
|practice_0075|mixed|E3|donante2/coste0/orden0|201|195|válido|
|practice_0075|mixed|E3|donante2/coste1/orden0|197|195|válido|
|practice_0075|mixed|E3|donante2/coste0/orden1|195|195|válido|
|practice_0075|mixed|E3|donante2/coste1/orden1|195|195|válido|
|practice_0075|mixed|E3|donante3/coste0/orden0|206|206|válido|
|practice_0075|mixed|E3|donante3/coste1/orden0|206|206|válido|
|practice_0075|mixed|E3|donante3/coste0/orden1|206|206|válido|
|practice_0075|mixed|E3|donante3/coste1/orden1|206|206|válido|
|practice_0075|mixed|E3|donante4/coste0/orden0|206|206|válido|
|practice_0075|mixed|E3|donante4/coste1/orden0|206|206|válido|
|practice_0075|mixed|E3|donante4/coste0/orden1|206|206|válido|
|practice_0075|mixed|E3|donante4/coste1/orden1|206|206|válido|
|practice_0075|mixed|E3|donante5/coste0/orden0|199|199|válido|
|practice_0075|mixed|E3|donante5/coste1/orden0|199|199|válido|
|practice_0075|mixed|E3|donante5/coste0/orden1|199|199|válido|
|practice_0075|mixed|E3|donante5/coste1/orden1|199|199|válido|
|practice_0075|mixed|E3|donante6/coste0/orden0|234|185|válido|
|practice_0075|mixed|E3|donante6/coste1/orden0|234|185|válido|
|practice_0075|mixed|E3|donante6/coste0/orden1|193|185|válido|
|practice_0075|mixed|E3|donante6/coste1/orden1|196|185|válido|
|practice_0075|mixed|E3|donante7/coste0/orden0|210|186|válido|
|practice_0075|mixed|E3|donante7/coste1/orden0|197|186|válido|
|practice_0075|mixed|E3|donante7/coste0/orden1|217|186|válido|
|practice_0075|mixed|E3|donante7/coste1/orden1|194|186|válido|
|practice_0075|mixed|E3|donante8/coste0/orden0|214|180|válido|
|practice_0075|mixed|E3|donante8/coste1/orden0|189|180|válido|
|practice_0075|mixed|E3|donante8/coste0/orden1|216|180|válido|
|practice_0075|mixed|E3|donante8/coste1/orden1|186|180|válido|
|practice_0076|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0076|independent|Cruce|CP True/coste0/orden0|463|291|válido|
|practice_0076|independent|Cruce|CP True/coste1/orden0|403|291|válido|
|practice_0076|independent|Cruce|CP True/coste0/orden1|296|291|válido|
|practice_0076|independent|Cruce|CP True/coste1/orden1|293|291|válido|
|practice_0076|independent|Cruce|mejor independiente/coste0/orden0|463|291|válido|
|practice_0076|independent|Cruce|mejor independiente/coste1/orden0|403|291|válido|
|practice_0076|independent|Cruce|mejor independiente/coste0/orden1|296|291|válido|
|practice_0076|independent|Cruce|mejor independiente/coste1/orden1|293|291|válido|
|practice_0076|independent|E1|cola10/pol1/inicio0|303|88|válido|
|practice_0076|independent|E1|cola10/pol1/inicio1|303|88|válido|
|practice_0076|independent|E1|cola10/pol2/inicio0|304|89|válido|
|practice_0076|independent|E1|cola10/pol2/inicio1|304|89|válido|
|practice_0076|independent|E1|cola8/pol1/inicio0|302|71|válido|
|practice_0076|independent|E1|cola8/pol1/inicio1|302|71|válido|
|practice_0076|independent|E1|cola8/pol2/inicio0|298|67|válido|
|practice_0076|independent|E1|cola8/pol2/inicio1|298|67|válido|
|practice_0076|independent|E1|cola6/pol1/inicio0|307|59|válido|
|practice_0076|independent|E1|cola6/pol1/inicio1|307|59|válido|
|practice_0076|independent|E1|cola6/pol2/inicio0|306|58|válido|
|practice_0076|independent|E1|cola6/pol2/inicio1|306|58|válido|
|practice_0076|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0076|independent|E2|CP False/coste0/orden0|354|347|válido|
|practice_0076|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0076|independent|E2|CP True/coste0/orden0|481|286|válido|
|practice_0076|independent|E3|donante0/coste0/orden0|313|286|válido|
|practice_0076|independent|E3|donante0/coste1/orden0|305|286|válido|
|practice_0076|independent|E3|donante0/coste0/orden1|459|286|válido|
|practice_0076|independent|E3|donante0/coste1/orden1|330|286|válido|
|practice_0076|independent|E3|donante1/coste0/orden0|337|337|válido|
|practice_0076|independent|E3|donante1/coste1/orden0|337|337|válido|
|practice_0076|independent|E3|donante1/coste0/orden1|337|337|válido|
|practice_0076|independent|E3|donante1/coste1/orden1|337|337|válido|
|practice_0076|independent|E3|donante2/coste0/orden0|622|318|válido|
|practice_0076|independent|E3|donante2/coste1/orden0|535|318|válido|
|practice_0076|independent|E3|donante2/coste0/orden1|318|318|válido|
|practice_0076|independent|E3|donante2/coste1/orden1|318|318|válido|
|practice_0076|independent|E3|donante3/coste0/orden0|611|306|válido|
|practice_0076|independent|E3|donante3/coste1/orden0|422|306|válido|
|practice_0076|independent|E3|donante3/coste0/orden1|306|306|válido|
|practice_0076|independent|E3|donante3/coste1/orden1|306|306|válido|
|practice_0076|independent|E3|donante4/coste0/orden0|348|348|válido|
|practice_0076|independent|E3|donante4/coste1/orden0|348|348|válido|
|practice_0076|independent|E3|donante4/coste0/orden1|348|348|válido|
|practice_0076|independent|E3|donante4/coste1/orden1|348|348|válido|
|practice_0076|independent|E3|donante5/coste0/orden0|348|348|válido|
|practice_0076|independent|E3|donante5/coste1/orden0|348|348|válido|
|practice_0076|independent|E3|donante5/coste0/orden1|348|348|válido|
|practice_0076|independent|E3|donante5/coste1/orden1|348|348|válido|
|practice_0076|independent|E3|donante6/coste0/orden0|340|340|válido|
|practice_0076|independent|E3|donante6/coste1/orden0|340|340|válido|
|practice_0076|independent|E3|donante6/coste0/orden1|340|340|válido|
|practice_0076|independent|E3|donante6/coste1/orden1|340|340|válido|
|practice_0076|independent|E3|donante7/coste0/orden0|314|301|válido|
|practice_0076|independent|E3|donante7/coste1/orden0|308|301|válido|
|practice_0076|independent|E3|donante7/coste0/orden1|344|301|válido|
|practice_0076|independent|E3|donante7/coste1/orden1|329|301|válido|
|practice_0076|independent|E3|donante8/coste0/orden0|310|292|válido|
|practice_0076|independent|E3|donante8/coste1/orden0|294|292|válido|
|practice_0076|independent|E3|donante8/coste0/orden1|297|292|válido|
|practice_0076|independent|E3|donante8/coste1/orden1|294|292|válido|
|practice_0076|independent|E3|donante9/coste0/orden0|314|287|válido|
|practice_0076|independent|E3|donante9/coste1/orden0|306|287|válido|
|practice_0076|independent|E3|donante9/coste0/orden1|456|287|válido|
|practice_0076|independent|E3|donante9/coste1/orden1|312|287|válido|
|practice_0077|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0077|multiplicative|Cruce|CP True/coste0/orden0|104|104|válido|
|practice_0077|multiplicative|Cruce|CP True/coste1/orden0|104|104|válido|
|practice_0077|multiplicative|Cruce|CP True/coste0/orden1|104|104|válido|
|practice_0077|multiplicative|Cruce|CP True/coste1/orden1|104|104|válido|
|practice_0077|multiplicative|Cruce|mejor independiente/coste0/orden0|104|104|válido|
|practice_0077|multiplicative|Cruce|mejor independiente/coste1/orden0|104|104|válido|
|practice_0077|multiplicative|Cruce|mejor independiente/coste0/orden1|104|104|válido|
|practice_0077|multiplicative|Cruce|mejor independiente/coste1/orden1|104|104|válido|
|practice_0077|multiplicative|E1|cola10/pol1/inicio0|109|40|válido|
|practice_0077|multiplicative|E1|cola10/pol1/inicio1|109|40|válido|
|practice_0077|multiplicative|E1|cola10/pol2/inicio0|107|38|válido|
|practice_0077|multiplicative|E1|cola10/pol2/inicio1|107|38|válido|
|practice_0077|multiplicative|E1|cola8/pol1/inicio0|108|29|válido|
|practice_0077|multiplicative|E1|cola8/pol1/inicio1|112|33|válido|
|practice_0077|multiplicative|E1|cola8/pol2/inicio0|108|29|válido|
|practice_0077|multiplicative|E1|cola8/pol2/inicio1|112|33|válido|
|practice_0077|multiplicative|E1|cola6/pol1/inicio0|108|23|válido|
|practice_0077|multiplicative|E1|cola6/pol1/inicio1|108|23|válido|
|practice_0077|multiplicative|E1|cola6/pol2/inicio0|108|23|válido|
|practice_0077|multiplicative|E1|cola6/pol2/inicio1|108|23|válido|
|practice_0077|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0077|multiplicative|E2|CP False/coste0/orden0|106|106|válido|
|practice_0077|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0077|multiplicative|E2|CP True/coste0/orden0|104|104|válido|
|practice_0077|multiplicative|E3|donante0/coste0/orden0|107|107|válido|
|practice_0077|multiplicative|E3|donante0/coste1/orden0|107|107|válido|
|practice_0077|multiplicative|E3|donante0/coste0/orden1|107|107|válido|
|practice_0077|multiplicative|E3|donante0/coste1/orden1|107|107|válido|
|practice_0077|multiplicative|E3|donante1/coste0/orden0|115|115|válido|
|practice_0077|multiplicative|E3|donante1/coste1/orden0|115|115|válido|
|practice_0077|multiplicative|E3|donante1/coste0/orden1|115|115|válido|
|practice_0077|multiplicative|E3|donante1/coste1/orden1|115|115|válido|
|practice_0077|multiplicative|E3|donante2/coste0/orden0|115|115|válido|
|practice_0077|multiplicative|E3|donante2/coste1/orden0|115|115|válido|
|practice_0077|multiplicative|E3|donante2/coste0/orden1|115|115|válido|
|practice_0077|multiplicative|E3|donante2/coste1/orden1|115|115|válido|
|practice_0077|multiplicative|E3|donante3/coste0/orden0|108|108|válido|
|practice_0077|multiplicative|E3|donante3/coste1/orden0|108|108|válido|
|practice_0077|multiplicative|E3|donante3/coste0/orden1|108|108|válido|
|practice_0077|multiplicative|E3|donante3/coste1/orden1|108|108|válido|
|practice_0077|multiplicative|E3|donante4/coste0/orden0|116|116|válido|
|practice_0077|multiplicative|E3|donante4/coste1/orden0|116|116|válido|
|practice_0077|multiplicative|E3|donante4/coste0/orden1|116|116|válido|
|practice_0077|multiplicative|E3|donante4/coste1/orden1|116|116|válido|
|practice_0077|multiplicative|E3|donante5/coste0/orden0|115|115|válido|
|practice_0077|multiplicative|E3|donante5/coste1/orden0|115|115|válido|
|practice_0077|multiplicative|E3|donante5/coste0/orden1|115|115|válido|
|practice_0077|multiplicative|E3|donante5/coste1/orden1|115|115|válido|
|practice_0077|multiplicative|E3|donante6/coste0/orden0|112|112|válido|
|practice_0077|multiplicative|E3|donante6/coste1/orden0|112|112|válido|
|practice_0077|multiplicative|E3|donante6/coste0/orden1|112|112|válido|
|practice_0077|multiplicative|E3|donante6/coste1/orden1|112|112|válido|
|practice_0077|multiplicative|E3|donante7/coste0/orden0|110|110|válido|
|practice_0077|multiplicative|E3|donante7/coste1/orden0|110|110|válido|
|practice_0077|multiplicative|E3|donante7/coste0/orden1|110|110|válido|
|practice_0077|multiplicative|E3|donante7/coste1/orden1|110|110|válido|
|practice_0077|multiplicative|E3|donante8/coste0/orden0|112|112|válido|
|practice_0077|multiplicative|E3|donante8/coste1/orden0|112|112|válido|
|practice_0077|multiplicative|E3|donante8/coste0/orden1|112|112|válido|
|practice_0077|multiplicative|E3|donante8/coste1/orden1|112|112|válido|
|practice_0077|multiplicative|E3|donante9/coste0/orden0|108|108|válido|
|practice_0077|multiplicative|E3|donante9/coste1/orden0|108|108|válido|
|practice_0077|multiplicative|E3|donante9/coste0/orden1|108|108|válido|
|practice_0077|multiplicative|E3|donante9/coste1/orden1|108|108|válido|
|practice_0078|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0078|mixed|Cruce|CP True/coste0/orden0|257|241|válido|
|practice_0078|mixed|Cruce|CP True/coste1/orden0|252|241|válido|
|practice_0078|mixed|Cruce|CP True/coste0/orden1|258|241|válido|
|practice_0078|mixed|Cruce|CP True/coste1/orden1|261|241|válido|
|practice_0078|mixed|Cruce|mejor independiente/coste0/orden0|262|245|válido|
|practice_0078|mixed|Cruce|mejor independiente/coste1/orden0|261|245|válido|
|practice_0078|mixed|Cruce|mejor independiente/coste0/orden1|285|245|válido|
|practice_0078|mixed|Cruce|mejor independiente/coste1/orden1|265|245|válido|
|practice_0078|mixed|E1|cola10/pol1/inicio0|267|96|válido|
|practice_0078|mixed|E1|cola10/pol1/inicio1|267|96|válido|
|practice_0078|mixed|E1|cola10/pol2/inicio0|264|93|válido|
|practice_0078|mixed|E1|cola10/pol2/inicio1|264|93|válido|
|practice_0078|mixed|E1|cola8/pol1/inicio0|270|73|válido|
|practice_0078|mixed|E1|cola8/pol1/inicio1|270|73|válido|
|practice_0078|mixed|E1|cola8/pol2/inicio0|271|74|válido|
|practice_0078|mixed|E1|cola8/pol2/inicio1|271|74|válido|
|practice_0078|mixed|E1|cola6/pol1/inicio0|270|56|válido|
|practice_0078|mixed|E1|cola6/pol1/inicio1|270|56|válido|
|practice_0078|mixed|E1|cola6/pol2/inicio0|269|55|válido|
|practice_0078|mixed|E1|cola6/pol2/inicio1|268|54|válido|
|practice_0078|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0078|mixed|E2|CP False/coste0/orden0|364|260|válido|
|practice_0078|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0078|mixed|E2|CP True/coste0/orden0|322|245|válido|
|practice_0078|mixed|E3|donante0/coste0/orden0|263|247|válido|
|practice_0078|mixed|E3|donante0/coste1/orden0|256|247|válido|
|practice_0078|mixed|E3|donante0/coste0/orden1|260|247|válido|
|practice_0078|mixed|E3|donante0/coste1/orden1|260|247|válido|
|practice_0078|mixed|E3|donante1/coste0/orden0|284|284|válido|
|practice_0078|mixed|E3|donante1/coste1/orden0|284|284|válido|
|practice_0078|mixed|E3|donante1/coste0/orden1|284|284|válido|
|practice_0078|mixed|E3|donante1/coste1/orden1|284|284|válido|
|practice_0078|mixed|E3|donante2/coste0/orden0|371|266|válido|
|practice_0078|mixed|E3|donante2/coste1/orden0|296|266|válido|
|practice_0078|mixed|E3|donante2/coste0/orden1|266|266|válido|
|practice_0078|mixed|E3|donante2/coste1/orden1|266|266|válido|
|practice_0078|mixed|E3|donante3/coste0/orden0|430|267|válido|
|practice_0078|mixed|E3|donante3/coste1/orden0|307|267|válido|
|practice_0078|mixed|E3|donante3/coste0/orden1|267|267|válido|
|practice_0078|mixed|E3|donante3/coste1/orden1|267|267|válido|
|practice_0078|mixed|E3|donante4/coste0/orden0|285|285|válido|
|practice_0078|mixed|E3|donante4/coste1/orden0|285|285|válido|
|practice_0078|mixed|E3|donante4/coste0/orden1|285|285|válido|
|practice_0078|mixed|E3|donante4/coste1/orden1|285|285|válido|
|practice_0078|mixed|E3|donante5/coste0/orden0|279|279|válido|
|practice_0078|mixed|E3|donante5/coste1/orden0|279|279|válido|
|practice_0078|mixed|E3|donante5/coste0/orden1|279|279|válido|
|practice_0078|mixed|E3|donante5/coste1/orden1|279|279|válido|
|practice_0078|mixed|E3|donante6/coste0/orden0|294|294|válido|
|practice_0078|mixed|E3|donante6/coste1/orden0|294|294|válido|
|practice_0078|mixed|E3|donante6/coste0/orden1|294|294|válido|
|practice_0078|mixed|E3|donante6/coste1/orden1|294|294|válido|
|practice_0078|mixed|E3|donante7/coste0/orden0|276|254|válido|
|practice_0078|mixed|E3|donante7/coste1/orden0|260|254|válido|
|practice_0078|mixed|E3|donante7/coste0/orden1|262|254|válido|
|practice_0078|mixed|E3|donante7/coste1/orden1|259|254|válido|
|practice_0078|mixed|E3|donante8/coste0/orden0|275|244|válido|
|practice_0078|mixed|E3|donante8/coste1/orden0|256|244|válido|
|practice_0078|mixed|E3|donante8/coste0/orden1|289|244|válido|
|practice_0078|mixed|E3|donante8/coste1/orden1|279|244|válido|
|practice_0078|mixed|E3|donante9/coste0/orden0|263|247|válido|
|practice_0078|mixed|E3|donante9/coste1/orden0|256|247|válido|
|practice_0078|mixed|E3|donante9/coste0/orden1|260|247|válido|
|practice_0078|mixed|E3|donante9/coste1/orden1|260|247|válido|
|practice_0079|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0079|multiplicative|Cruce|CP True/coste0/orden0|119|119|válido|
|practice_0079|multiplicative|Cruce|CP True/coste1/orden0|119|119|válido|
|practice_0079|multiplicative|Cruce|CP True/coste0/orden1|119|119|válido|
|practice_0079|multiplicative|Cruce|CP True/coste1/orden1|119|119|válido|
|practice_0079|multiplicative|Cruce|mejor independiente/coste0/orden0|119|119|válido|
|practice_0079|multiplicative|Cruce|mejor independiente/coste1/orden0|119|119|válido|
|practice_0079|multiplicative|Cruce|mejor independiente/coste0/orden1|119|119|válido|
|practice_0079|multiplicative|Cruce|mejor independiente/coste1/orden1|119|119|válido|
|practice_0079|multiplicative|E1|cola10/pol1/inicio0|127|47|válido|
|practice_0079|multiplicative|E1|cola10/pol1/inicio1|127|47|válido|
|practice_0079|multiplicative|E1|cola10/pol2/inicio0|128|48|válido|
|practice_0079|multiplicative|E1|cola10/pol2/inicio1|128|48|válido|
|practice_0079|multiplicative|E1|cola8/pol1/inicio0|125|38|válido|
|practice_0079|multiplicative|E1|cola8/pol1/inicio1|125|38|válido|
|practice_0079|multiplicative|E1|cola8/pol2/inicio0|124|37|válido|
|practice_0079|multiplicative|E1|cola8/pol2/inicio1|124|37|válido|
|practice_0079|multiplicative|E1|cola6/pol1/inicio0|122|26|válido|
|practice_0079|multiplicative|E1|cola6/pol1/inicio1|122|26|válido|
|practice_0079|multiplicative|E1|cola6/pol2/inicio0|120|24|válido|
|practice_0079|multiplicative|E1|cola6/pol2/inicio1|120|24|válido|
|practice_0079|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0079|multiplicative|E2|CP False/coste0/orden0|120|120|válido|
|practice_0079|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0079|multiplicative|E2|CP True/coste0/orden0|119|119|válido|
|practice_0079|multiplicative|E3|donante0/coste0/orden0|120|120|válido|
|practice_0079|multiplicative|E3|donante0/coste1/orden0|120|120|válido|
|practice_0079|multiplicative|E3|donante0/coste0/orden1|120|120|válido|
|practice_0079|multiplicative|E3|donante0/coste1/orden1|120|120|válido|
|practice_0079|multiplicative|E3|donante1/coste0/orden0|138|138|válido|
|practice_0079|multiplicative|E3|donante1/coste1/orden0|138|138|válido|
|practice_0079|multiplicative|E3|donante1/coste0/orden1|138|138|válido|
|practice_0079|multiplicative|E3|donante1/coste1/orden1|138|138|válido|
|practice_0079|multiplicative|E3|donante2/coste0/orden0|135|135|válido|
|practice_0079|multiplicative|E3|donante2/coste1/orden0|135|135|válido|
|practice_0079|multiplicative|E3|donante2/coste0/orden1|135|135|válido|
|practice_0079|multiplicative|E3|donante2/coste1/orden1|135|135|válido|
|practice_0079|multiplicative|E3|donante3/coste0/orden0|128|128|válido|
|practice_0079|multiplicative|E3|donante3/coste1/orden0|128|128|válido|
|practice_0079|multiplicative|E3|donante3/coste0/orden1|128|128|válido|
|practice_0079|multiplicative|E3|donante3/coste1/orden1|128|128|válido|
|practice_0079|multiplicative|E3|donante4/coste0/orden0|144|144|válido|
|practice_0079|multiplicative|E3|donante4/coste1/orden0|144|144|válido|
|practice_0079|multiplicative|E3|donante4/coste0/orden1|144|144|válido|
|practice_0079|multiplicative|E3|donante4/coste1/orden1|144|144|válido|
|practice_0079|multiplicative|E3|donante5/coste0/orden0|138|138|válido|
|practice_0079|multiplicative|E3|donante5/coste1/orden0|138|138|válido|
|practice_0079|multiplicative|E3|donante5/coste0/orden1|138|138|válido|
|practice_0079|multiplicative|E3|donante5/coste1/orden1|138|138|válido|
|practice_0079|multiplicative|E3|donante6/coste0/orden0|133|133|válido|
|practice_0079|multiplicative|E3|donante6/coste1/orden0|133|133|válido|
|practice_0079|multiplicative|E3|donante6/coste0/orden1|133|133|válido|
|practice_0079|multiplicative|E3|donante6/coste1/orden1|133|133|válido|
|practice_0079|multiplicative|E3|donante7/coste0/orden0|129|129|válido|
|practice_0079|multiplicative|E3|donante7/coste1/orden0|129|129|válido|
|practice_0079|multiplicative|E3|donante7/coste0/orden1|129|129|válido|
|practice_0079|multiplicative|E3|donante7/coste1/orden1|129|129|válido|
|practice_0079|multiplicative|E3|donante8/coste0/orden0|134|134|válido|
|practice_0079|multiplicative|E3|donante8/coste1/orden0|134|134|válido|
|practice_0079|multiplicative|E3|donante8/coste0/orden1|134|134|válido|
|practice_0079|multiplicative|E3|donante8/coste1/orden1|134|134|válido|
|practice_0079|multiplicative|E3|donante9/coste0/orden0|123|123|válido|
|practice_0079|multiplicative|E3|donante9/coste1/orden0|123|123|válido|
|practice_0079|multiplicative|E3|donante9/coste0/orden1|123|123|válido|
|practice_0079|multiplicative|E3|donante9/coste1/orden1|123|123|válido|
|practice_0080|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0080|multiplicative|Cruce|CP True/coste0/orden0|108|108|válido|
|practice_0080|multiplicative|Cruce|CP True/coste1/orden0|108|108|válido|
|practice_0080|multiplicative|Cruce|CP True/coste0/orden1|108|108|válido|
|practice_0080|multiplicative|Cruce|CP True/coste1/orden1|108|108|válido|
|practice_0080|multiplicative|Cruce|mejor independiente/coste0/orden0|108|108|válido|
|practice_0080|multiplicative|Cruce|mejor independiente/coste1/orden0|108|108|válido|
|practice_0080|multiplicative|Cruce|mejor independiente/coste0/orden1|108|108|válido|
|practice_0080|multiplicative|Cruce|mejor independiente/coste1/orden1|108|108|válido|
|practice_0080|multiplicative|E1|cola10/pol1/inicio0|116|40|válido|
|practice_0080|multiplicative|E1|cola10/pol1/inicio1|116|40|válido|
|practice_0080|multiplicative|E1|cola10/pol2/inicio0|113|37|válido|
|practice_0080|multiplicative|E1|cola10/pol2/inicio1|113|37|válido|
|practice_0080|multiplicative|E1|cola8/pol1/inicio0|117|35|válido|
|practice_0080|multiplicative|E1|cola8/pol1/inicio1|117|35|válido|
|practice_0080|multiplicative|E1|cola8/pol2/inicio0|113|31|válido|
|practice_0080|multiplicative|E1|cola8/pol2/inicio1|113|31|válido|
|practice_0080|multiplicative|E1|cola6/pol1/inicio0|113|22|válido|
|practice_0080|multiplicative|E1|cola6/pol1/inicio1|115|24|válido|
|practice_0080|multiplicative|E1|cola6/pol2/inicio0|113|22|válido|
|practice_0080|multiplicative|E1|cola6/pol2/inicio1|115|24|válido|
|practice_0080|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0080|multiplicative|E2|CP False/coste0/orden0|117|117|válido|
|practice_0080|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0080|multiplicative|E2|CP True/coste0/orden0|108|108|válido|
|practice_0080|multiplicative|E3|donante0/coste0/orden0|113|113|válido|
|practice_0080|multiplicative|E3|donante0/coste1/orden0|113|113|válido|
|practice_0080|multiplicative|E3|donante0/coste0/orden1|113|113|válido|
|practice_0080|multiplicative|E3|donante0/coste1/orden1|113|113|válido|
|practice_0080|multiplicative|E3|donante1/coste0/orden0|124|124|válido|
|practice_0080|multiplicative|E3|donante1/coste1/orden0|124|124|válido|
|practice_0080|multiplicative|E3|donante1/coste0/orden1|124|124|válido|
|practice_0080|multiplicative|E3|donante1/coste1/orden1|124|124|válido|
|practice_0080|multiplicative|E3|donante2/coste0/orden0|122|122|válido|
|practice_0080|multiplicative|E3|donante2/coste1/orden0|122|122|válido|
|practice_0080|multiplicative|E3|donante2/coste0/orden1|122|122|válido|
|practice_0080|multiplicative|E3|donante2/coste1/orden1|122|122|válido|
|practice_0080|multiplicative|E3|donante3/coste0/orden0|121|121|válido|
|practice_0080|multiplicative|E3|donante3/coste1/orden0|121|121|válido|
|practice_0080|multiplicative|E3|donante3/coste0/orden1|121|121|válido|
|practice_0080|multiplicative|E3|donante3/coste1/orden1|121|121|válido|
|practice_0080|multiplicative|E3|donante4/coste0/orden0|127|127|válido|
|practice_0080|multiplicative|E3|donante4/coste1/orden0|127|127|válido|
|practice_0080|multiplicative|E3|donante4/coste0/orden1|127|127|válido|
|practice_0080|multiplicative|E3|donante4/coste1/orden1|127|127|válido|
|practice_0080|multiplicative|E3|donante5/coste0/orden0|125|125|válido|
|practice_0080|multiplicative|E3|donante5/coste1/orden0|125|125|válido|
|practice_0080|multiplicative|E3|donante5/coste0/orden1|125|125|válido|
|practice_0080|multiplicative|E3|donante5/coste1/orden1|125|125|válido|
|practice_0080|multiplicative|E3|donante6/coste0/orden0|119|119|válido|
|practice_0080|multiplicative|E3|donante6/coste1/orden0|119|119|válido|
|practice_0080|multiplicative|E3|donante6/coste0/orden1|119|119|válido|
|practice_0080|multiplicative|E3|donante6/coste1/orden1|119|119|válido|
|practice_0080|multiplicative|E3|donante7/coste0/orden0|122|122|válido|
|practice_0080|multiplicative|E3|donante7/coste1/orden0|122|122|válido|
|practice_0080|multiplicative|E3|donante7/coste0/orden1|122|122|válido|
|practice_0080|multiplicative|E3|donante7/coste1/orden1|122|122|válido|
|practice_0080|multiplicative|E3|donante8/coste0/orden0|119|119|válido|
|practice_0080|multiplicative|E3|donante8/coste1/orden0|119|119|válido|
|practice_0080|multiplicative|E3|donante8/coste0/orden1|119|119|válido|
|practice_0080|multiplicative|E3|donante8/coste1/orden1|119|119|válido|
|practice_0081|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0081|shared|Cruce|CP True/coste0/orden0|160|160|válido|
|practice_0081|shared|Cruce|CP True/coste1/orden0|160|160|válido|
|practice_0081|shared|Cruce|CP True/coste0/orden1|160|160|válido|
|practice_0081|shared|Cruce|CP True/coste1/orden1|160|160|válido|
|practice_0081|shared|Cruce|mejor independiente/coste0/orden0|161|161|válido|
|practice_0081|shared|Cruce|mejor independiente/coste1/orden0|161|161|válido|
|practice_0081|shared|Cruce|mejor independiente/coste0/orden1|161|161|válido|
|practice_0081|shared|Cruce|mejor independiente/coste1/orden1|161|161|válido|
|practice_0081|shared|E1|cola10/pol1/inicio0|172|59|válido|
|practice_0081|shared|E1|cola10/pol1/inicio1|172|59|válido|
|practice_0081|shared|E1|cola10/pol2/inicio0|168|55|válido|
|practice_0081|shared|E1|cola10/pol2/inicio1|170|57|válido|
|practice_0081|shared|E1|cola8/pol1/inicio0|174|51|válido|
|practice_0081|shared|E1|cola8/pol1/inicio1|174|51|válido|
|practice_0081|shared|E1|cola8/pol2/inicio0|174|51|válido|
|practice_0081|shared|E1|cola8/pol2/inicio1|174|51|válido|
|practice_0081|shared|E1|cola6/pol1/inicio0|172|40|válido|
|practice_0081|shared|E1|cola6/pol1/inicio1|172|40|válido|
|practice_0081|shared|E1|cola6/pol2/inicio0|172|40|válido|
|practice_0081|shared|E1|cola6/pol2/inicio1|172|40|válido|
|practice_0081|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0081|shared|E2|CP False/coste0/orden0|166|166|válido|
|practice_0081|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0081|shared|E2|CP True/coste0/orden0|161|161|válido|
|practice_0081|shared|E3|donante0/coste0/orden0|162|162|válido|
|practice_0081|shared|E3|donante0/coste1/orden0|162|162|válido|
|practice_0081|shared|E3|donante0/coste0/orden1|162|162|válido|
|practice_0081|shared|E3|donante0/coste1/orden1|162|162|válido|
|practice_0081|shared|E3|donante1/coste0/orden0|203|203|válido|
|practice_0081|shared|E3|donante1/coste1/orden0|203|203|válido|
|practice_0081|shared|E3|donante1/coste0/orden1|203|203|válido|
|practice_0081|shared|E3|donante1/coste1/orden1|203|203|válido|
|practice_0081|shared|E3|donante2/coste0/orden0|189|189|válido|
|practice_0081|shared|E3|donante2/coste1/orden0|189|189|válido|
|practice_0081|shared|E3|donante2/coste0/orden1|189|189|válido|
|practice_0081|shared|E3|donante2/coste1/orden1|189|189|válido|
|practice_0081|shared|E3|donante3/coste0/orden0|175|175|válido|
|practice_0081|shared|E3|donante3/coste1/orden0|175|175|válido|
|practice_0081|shared|E3|donante3/coste0/orden1|175|175|válido|
|practice_0081|shared|E3|donante3/coste1/orden1|175|175|válido|
|practice_0081|shared|E3|donante4/coste0/orden0|204|204|válido|
|practice_0081|shared|E3|donante4/coste1/orden0|204|204|válido|
|practice_0081|shared|E3|donante4/coste0/orden1|204|204|válido|
|practice_0081|shared|E3|donante4/coste1/orden1|204|204|válido|
|practice_0081|shared|E3|donante5/coste0/orden0|199|199|válido|
|practice_0081|shared|E3|donante5/coste1/orden0|199|199|válido|
|practice_0081|shared|E3|donante5/coste0/orden1|199|199|válido|
|practice_0081|shared|E3|donante5/coste1/orden1|199|199|válido|
|practice_0081|shared|E3|donante6/coste0/orden0|199|199|válido|
|practice_0081|shared|E3|donante6/coste1/orden0|199|199|válido|
|practice_0081|shared|E3|donante6/coste0/orden1|199|199|válido|
|practice_0081|shared|E3|donante6/coste1/orden1|199|199|válido|
|practice_0081|shared|E3|donante7/coste0/orden0|212|179|válido|
|practice_0081|shared|E3|donante7/coste1/orden0|203|179|válido|
|practice_0081|shared|E3|donante7/coste0/orden1|186|179|válido|
|practice_0081|shared|E3|donante7/coste1/orden1|182|179|válido|
|practice_0081|shared|E3|donante8/coste0/orden0|173|171|válido|
|practice_0081|shared|E3|donante8/coste1/orden0|176|171|válido|
|practice_0081|shared|E3|donante8/coste0/orden1|171|171|válido|
|practice_0081|shared|E3|donante8/coste1/orden1|171|171|válido|
|practice_0082|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0082|mixed|Cruce|CP True/coste0/orden0|256|204|válido|
|practice_0082|mixed|Cruce|CP True/coste1/orden0|216|204|válido|
|practice_0082|mixed|Cruce|CP True/coste0/orden1|240|204|válido|
|practice_0082|mixed|Cruce|CP True/coste1/orden1|207|204|válido|
|practice_0082|mixed|Cruce|mejor independiente/coste0/orden0|259|204|válido|
|practice_0082|mixed|Cruce|mejor independiente/coste1/orden0|227|204|válido|
|practice_0082|mixed|Cruce|mejor independiente/coste0/orden1|246|204|válido|
|practice_0082|mixed|Cruce|mejor independiente/coste1/orden1|207|204|válido|
|practice_0082|mixed|E1|cola10/pol1/inicio0|220|75|válido|
|practice_0082|mixed|E1|cola10/pol1/inicio1|220|75|válido|
|practice_0082|mixed|E1|cola10/pol2/inicio0|224|79|válido|
|practice_0082|mixed|E1|cola10/pol2/inicio1|224|79|válido|
|practice_0082|mixed|E1|cola8/pol1/inicio0|225|66|válido|
|practice_0082|mixed|E1|cola8/pol1/inicio1|225|66|válido|
|practice_0082|mixed|E1|cola8/pol2/inicio0|224|65|válido|
|practice_0082|mixed|E1|cola8/pol2/inicio1|224|65|válido|
|practice_0082|mixed|E1|cola6/pol1/inicio0|214|41|válido|
|practice_0082|mixed|E1|cola6/pol1/inicio1|214|41|válido|
|practice_0082|mixed|E1|cola6/pol2/inicio0|214|41|válido|
|practice_0082|mixed|E1|cola6/pol2/inicio1|214|41|válido|
|practice_0082|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0082|mixed|E2|CP False/coste0/orden0|259|204|válido|
|practice_0082|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0082|mixed|E2|CP True/coste0/orden0|269|206|válido|
|practice_0082|mixed|E3|donante0/coste0/orden0|209|209|válido|
|practice_0082|mixed|E3|donante0/coste1/orden0|209|209|válido|
|practice_0082|mixed|E3|donante0/coste0/orden1|228|209|válido|
|practice_0082|mixed|E3|donante0/coste1/orden1|210|209|válido|
|practice_0082|mixed|E3|donante1/coste0/orden0|243|243|válido|
|practice_0082|mixed|E3|donante1/coste1/orden0|243|243|válido|
|practice_0082|mixed|E3|donante1/coste0/orden1|243|243|válido|
|practice_0082|mixed|E3|donante1/coste1/orden1|243|243|válido|
|practice_0082|mixed|E3|donante2/coste0/orden0|224|224|válido|
|practice_0082|mixed|E3|donante2/coste1/orden0|224|224|válido|
|practice_0082|mixed|E3|donante2/coste0/orden1|224|224|válido|
|practice_0082|mixed|E3|donante2/coste1/orden1|224|224|válido|
|practice_0082|mixed|E3|donante3/coste0/orden0|251|217|válido|
|practice_0082|mixed|E3|donante3/coste1/orden0|224|217|válido|
|practice_0082|mixed|E3|donante3/coste0/orden1|217|217|válido|
|practice_0082|mixed|E3|donante3/coste1/orden1|217|217|válido|
|practice_0082|mixed|E3|donante4/coste0/orden0|238|238|válido|
|practice_0082|mixed|E3|donante4/coste1/orden0|238|238|válido|
|practice_0082|mixed|E3|donante4/coste0/orden1|238|238|válido|
|practice_0082|mixed|E3|donante4/coste1/orden1|238|238|válido|
|practice_0082|mixed|E3|donante5/coste0/orden0|231|231|válido|
|practice_0082|mixed|E3|donante5/coste1/orden0|231|231|válido|
|practice_0082|mixed|E3|donante5/coste0/orden1|231|231|válido|
|practice_0082|mixed|E3|donante5/coste1/orden1|231|231|válido|
|practice_0082|mixed|E3|donante6/coste0/orden0|238|238|válido|
|practice_0082|mixed|E3|donante6/coste1/orden0|238|238|válido|
|practice_0082|mixed|E3|donante6/coste0/orden1|238|238|válido|
|practice_0082|mixed|E3|donante6/coste1/orden1|238|238|válido|
|practice_0082|mixed|E3|donante7/coste0/orden0|255|219|válido|
|practice_0082|mixed|E3|donante7/coste1/orden0|224|219|válido|
|practice_0082|mixed|E3|donante7/coste0/orden1|235|219|válido|
|practice_0082|mixed|E3|donante7/coste1/orden1|237|219|válido|
|practice_0082|mixed|E3|donante8/coste0/orden0|246|204|válido|
|practice_0082|mixed|E3|donante8/coste1/orden0|242|204|válido|
|practice_0082|mixed|E3|donante8/coste0/orden1|246|204|válido|
|practice_0082|mixed|E3|donante8/coste1/orden1|207|204|válido|
|practice_0083|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0083|shared|Cruce|CP True/coste0/orden0|136|136|válido|
|practice_0083|shared|Cruce|CP True/coste1/orden0|136|136|válido|
|practice_0083|shared|Cruce|CP True/coste0/orden1|136|136|válido|
|practice_0083|shared|Cruce|CP True/coste1/orden1|136|136|válido|
|practice_0083|shared|Cruce|mejor independiente/coste0/orden0|136|136|válido|
|practice_0083|shared|Cruce|mejor independiente/coste1/orden0|136|136|válido|
|practice_0083|shared|Cruce|mejor independiente/coste0/orden1|136|136|válido|
|practice_0083|shared|Cruce|mejor independiente/coste1/orden1|136|136|válido|
|practice_0083|shared|E1|cola10/pol1/inicio0|148|58|válido|
|practice_0083|shared|E1|cola10/pol1/inicio1|148|58|válido|
|practice_0083|shared|E1|cola10/pol2/inicio0|141|51|válido|
|practice_0083|shared|E1|cola10/pol2/inicio1|141|51|válido|
|practice_0083|shared|E1|cola8/pol1/inicio0|145|46|válido|
|practice_0083|shared|E1|cola8/pol1/inicio1|145|46|válido|
|practice_0083|shared|E1|cola8/pol2/inicio0|143|44|válido|
|practice_0083|shared|E1|cola8/pol2/inicio1|143|44|válido|
|practice_0083|shared|E1|cola6/pol1/inicio0|140|31|válido|
|practice_0083|shared|E1|cola6/pol1/inicio1|140|31|válido|
|practice_0083|shared|E1|cola6/pol2/inicio0|140|31|válido|
|practice_0083|shared|E1|cola6/pol2/inicio1|140|31|válido|
|practice_0083|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0083|shared|E2|CP False/coste0/orden0|136|136|válido|
|practice_0083|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0083|shared|E2|CP True/coste0/orden0|136|136|válido|
|practice_0083|shared|E3|donante0/coste0/orden0|136|136|válido|
|practice_0083|shared|E3|donante0/coste1/orden0|136|136|válido|
|practice_0083|shared|E3|donante0/coste0/orden1|136|136|válido|
|practice_0083|shared|E3|donante0/coste1/orden1|136|136|válido|
|practice_0083|shared|E3|donante1/coste0/orden0|155|155|válido|
|practice_0083|shared|E3|donante1/coste1/orden0|155|155|válido|
|practice_0083|shared|E3|donante1/coste0/orden1|155|155|válido|
|practice_0083|shared|E3|donante1/coste1/orden1|155|155|válido|
|practice_0083|shared|E3|donante2/coste0/orden0|147|147|válido|
|practice_0083|shared|E3|donante2/coste1/orden0|147|147|válido|
|practice_0083|shared|E3|donante2/coste0/orden1|147|147|válido|
|practice_0083|shared|E3|donante2/coste1/orden1|147|147|válido|
|practice_0083|shared|E3|donante3/coste0/orden0|141|141|válido|
|practice_0083|shared|E3|donante3/coste1/orden0|141|141|válido|
|practice_0083|shared|E3|donante3/coste0/orden1|141|141|válido|
|practice_0083|shared|E3|donante3/coste1/orden1|141|141|válido|
|practice_0083|shared|E3|donante4/coste0/orden0|161|161|válido|
|practice_0083|shared|E3|donante4/coste1/orden0|161|161|válido|
|practice_0083|shared|E3|donante4/coste0/orden1|161|161|válido|
|practice_0083|shared|E3|donante4/coste1/orden1|161|161|válido|
|practice_0083|shared|E3|donante5/coste0/orden0|153|153|válido|
|practice_0083|shared|E3|donante5/coste1/orden0|153|153|válido|
|practice_0083|shared|E3|donante5/coste0/orden1|153|153|válido|
|practice_0083|shared|E3|donante5/coste1/orden1|153|153|válido|
|practice_0083|shared|E3|donante6/coste0/orden0|153|153|válido|
|practice_0083|shared|E3|donante6/coste1/orden0|153|153|válido|
|practice_0083|shared|E3|donante6/coste0/orden1|153|153|válido|
|practice_0083|shared|E3|donante6/coste1/orden1|153|153|válido|
|practice_0083|shared|E3|donante7/coste0/orden0|141|141|válido|
|practice_0083|shared|E3|donante7/coste1/orden0|141|141|válido|
|practice_0083|shared|E3|donante7/coste0/orden1|141|141|válido|
|practice_0083|shared|E3|donante7/coste1/orden1|141|141|válido|
|practice_0083|shared|E3|donante8/coste0/orden0|145|139|válido|
|practice_0083|shared|E3|donante8/coste1/orden0|142|139|válido|
|practice_0083|shared|E3|donante8/coste0/orden1|139|139|válido|
|practice_0083|shared|E3|donante8/coste1/orden1|139|139|válido|
|practice_0083|shared|E3|donante9/coste0/orden0|137|137|válido|
|practice_0083|shared|E3|donante9/coste1/orden0|137|137|válido|
|practice_0083|shared|E3|donante9/coste0/orden1|137|137|válido|
|practice_0083|shared|E3|donante9/coste1/orden1|137|137|válido|
|practice_0084|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0084|shared|Cruce|CP True/coste0/orden0|142|142|válido|
|practice_0084|shared|Cruce|CP True/coste1/orden0|142|142|válido|
|practice_0084|shared|Cruce|CP True/coste0/orden1|142|142|válido|
|practice_0084|shared|Cruce|CP True/coste1/orden1|142|142|válido|
|practice_0084|shared|Cruce|mejor independiente/coste0/orden0|142|142|válido|
|practice_0084|shared|Cruce|mejor independiente/coste1/orden0|142|142|válido|
|practice_0084|shared|Cruce|mejor independiente/coste0/orden1|142|142|válido|
|practice_0084|shared|Cruce|mejor independiente/coste1/orden1|142|142|válido|
|practice_0084|shared|E1|cola10/pol1/inicio0|149|41|válido|
|practice_0084|shared|E1|cola10/pol1/inicio1|150|42|válido|
|practice_0084|shared|E1|cola10/pol2/inicio0|143|35|válido|
|practice_0084|shared|E1|cola10/pol2/inicio1|143|35|válido|
|practice_0084|shared|E1|cola8/pol1/inicio0|147|35|válido|
|practice_0084|shared|E1|cola8/pol1/inicio1|147|35|válido|
|practice_0084|shared|E1|cola8/pol2/inicio0|143|31|válido|
|practice_0084|shared|E1|cola8/pol2/inicio1|143|31|válido|
|practice_0084|shared|E1|cola6/pol1/inicio0|144|26|válido|
|practice_0084|shared|E1|cola6/pol1/inicio1|144|26|válido|
|practice_0084|shared|E1|cola6/pol2/inicio0|143|25|válido|
|practice_0084|shared|E1|cola6/pol2/inicio1|143|25|válido|
|practice_0084|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0084|shared|E2|CP False/coste0/orden0|146|146|válido|
|practice_0084|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0084|shared|E2|CP True/coste0/orden0|142|142|válido|
|practice_0084|shared|E3|donante0/coste0/orden0|144|144|válido|
|practice_0084|shared|E3|donante0/coste1/orden0|144|144|válido|
|practice_0084|shared|E3|donante0/coste0/orden1|144|144|válido|
|practice_0084|shared|E3|donante0/coste1/orden1|144|144|válido|
|practice_0084|shared|E3|donante1/coste0/orden0|168|168|válido|
|practice_0084|shared|E3|donante1/coste1/orden0|168|168|válido|
|practice_0084|shared|E3|donante1/coste0/orden1|168|168|válido|
|practice_0084|shared|E3|donante1/coste1/orden1|168|168|válido|
|practice_0084|shared|E3|donante2/coste0/orden0|150|150|válido|
|practice_0084|shared|E3|donante2/coste1/orden0|150|150|válido|
|practice_0084|shared|E3|donante2/coste0/orden1|150|150|válido|
|practice_0084|shared|E3|donante2/coste1/orden1|150|150|válido|
|practice_0084|shared|E3|donante3/coste0/orden0|152|152|válido|
|practice_0084|shared|E3|donante3/coste1/orden0|152|152|válido|
|practice_0084|shared|E3|donante3/coste0/orden1|152|152|válido|
|practice_0084|shared|E3|donante3/coste1/orden1|152|152|válido|
|practice_0084|shared|E3|donante4/coste0/orden0|206|206|válido|
|practice_0084|shared|E3|donante4/coste1/orden0|206|206|válido|
|practice_0084|shared|E3|donante4/coste0/orden1|206|206|válido|
|practice_0084|shared|E3|donante4/coste1/orden1|206|206|válido|
|practice_0084|shared|E3|donante5/coste0/orden0|157|157|válido|
|practice_0084|shared|E3|donante5/coste1/orden0|157|157|válido|
|practice_0084|shared|E3|donante5/coste0/orden1|157|157|válido|
|practice_0084|shared|E3|donante5/coste1/orden1|157|157|válido|
|practice_0084|shared|E3|donante6/coste0/orden0|152|152|válido|
|practice_0084|shared|E3|donante6/coste1/orden0|152|152|válido|
|practice_0084|shared|E3|donante6/coste0/orden1|152|152|válido|
|practice_0084|shared|E3|donante6/coste1/orden1|152|152|válido|
|practice_0084|shared|E3|donante7/coste0/orden0|160|158|válido|
|practice_0084|shared|E3|donante7/coste1/orden0|161|158|válido|
|practice_0084|shared|E3|donante7/coste0/orden1|158|158|válido|
|practice_0084|shared|E3|donante7/coste1/orden1|158|158|válido|
|practice_0084|shared|E3|donante8/coste0/orden0|145|145|válido|
|practice_0084|shared|E3|donante8/coste1/orden0|145|145|válido|
|practice_0084|shared|E3|donante8/coste0/orden1|145|145|válido|
|practice_0084|shared|E3|donante8/coste1/orden1|145|145|válido|
|practice_0084|shared|E3|donante9/coste0/orden0|160|145|válido|
|practice_0084|shared|E3|donante9/coste1/orden0|160|145|válido|
|practice_0084|shared|E3|donante9/coste0/orden1|145|145|válido|
|practice_0084|shared|E3|donante9/coste1/orden1|145|145|válido|
|practice_0085|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0085|mixed|Cruce|CP True/coste0/orden0|336|254|válido|
|practice_0085|mixed|Cruce|CP True/coste1/orden0|282|254|válido|
|practice_0085|mixed|Cruce|CP True/coste0/orden1|260|254|válido|
|practice_0085|mixed|Cruce|CP True/coste1/orden1|256|254|válido|
|practice_0085|mixed|Cruce|mejor independiente/coste0/orden0|402|257|válido|
|practice_0085|mixed|Cruce|mejor independiente/coste1/orden0|300|257|válido|
|practice_0085|mixed|Cruce|mejor independiente/coste0/orden1|264|257|válido|
|practice_0085|mixed|Cruce|mejor independiente/coste1/orden1|265|257|válido|
|practice_0085|mixed|E1|cola10/pol1/inicio0|273|91|válido|
|practice_0085|mixed|E1|cola10/pol1/inicio1|273|91|válido|
|practice_0085|mixed|E1|cola10/pol2/inicio0|274|92|válido|
|practice_0085|mixed|E1|cola10/pol2/inicio1|274|92|válido|
|practice_0085|mixed|E1|cola8/pol1/inicio0|272|73|válido|
|practice_0085|mixed|E1|cola8/pol1/inicio1|272|73|válido|
|practice_0085|mixed|E1|cola8/pol2/inicio0|266|67|válido|
|practice_0085|mixed|E1|cola8/pol2/inicio1|266|67|válido|
|practice_0085|mixed|E1|cola6/pol1/inicio0|268|51|válido|
|practice_0085|mixed|E1|cola6/pol1/inicio1|268|51|válido|
|practice_0085|mixed|E1|cola6/pol2/inicio0|268|51|válido|
|practice_0085|mixed|E1|cola6/pol2/inicio1|268|51|válido|
|practice_0085|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0085|mixed|E2|CP False/coste0/orden0|402|257|válido|
|practice_0085|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0085|mixed|E2|CP True/coste0/orden0|336|254|válido|
|practice_0085|mixed|E3|donante0/coste0/orden0|264|257|válido|
|practice_0085|mixed|E3|donante0/coste1/orden0|265|257|válido|
|practice_0085|mixed|E3|donante0/coste0/orden1|264|257|válido|
|practice_0085|mixed|E3|donante0/coste1/orden1|265|257|válido|
|practice_0085|mixed|E3|donante1/coste0/orden0|304|304|válido|
|practice_0085|mixed|E3|donante1/coste1/orden0|304|304|válido|
|practice_0085|mixed|E3|donante1/coste0/orden1|304|304|válido|
|practice_0085|mixed|E3|donante1/coste1/orden1|304|304|válido|
|practice_0085|mixed|E3|donante2/coste0/orden0|302|280|válido|
|practice_0085|mixed|E3|donante2/coste1/orden0|283|280|válido|
|practice_0085|mixed|E3|donante2/coste0/orden1|280|280|válido|
|practice_0085|mixed|E3|donante2/coste1/orden1|280|280|válido|
|practice_0085|mixed|E3|donante3/coste0/orden0|375|280|válido|
|practice_0085|mixed|E3|donante3/coste1/orden0|291|280|válido|
|practice_0085|mixed|E3|donante3/coste0/orden1|280|280|válido|
|practice_0085|mixed|E3|donante3/coste1/orden1|280|280|válido|
|practice_0085|mixed|E3|donante4/coste0/orden0|304|304|válido|
|practice_0085|mixed|E3|donante4/coste1/orden0|304|304|válido|
|practice_0085|mixed|E3|donante4/coste0/orden1|304|304|válido|
|practice_0085|mixed|E3|donante4/coste1/orden1|304|304|válido|
|practice_0085|mixed|E3|donante5/coste0/orden0|285|285|válido|
|practice_0085|mixed|E3|donante5/coste1/orden0|285|285|válido|
|practice_0085|mixed|E3|donante5/coste0/orden1|285|285|válido|
|practice_0085|mixed|E3|donante5/coste1/orden1|285|285|válido|
|practice_0085|mixed|E3|donante6/coste0/orden0|301|301|válido|
|practice_0085|mixed|E3|donante6/coste1/orden0|301|301|válido|
|practice_0085|mixed|E3|donante6/coste0/orden1|301|301|válido|
|practice_0085|mixed|E3|donante6/coste1/orden1|301|301|válido|
|practice_0085|mixed|E3|donante7/coste0/orden0|275|259|válido|
|practice_0085|mixed|E3|donante7/coste1/orden0|263|259|válido|
|practice_0085|mixed|E3|donante7/coste0/orden1|323|259|válido|
|practice_0085|mixed|E3|donante7/coste1/orden1|284|259|válido|
|practice_0085|mixed|E3|donante8/coste0/orden0|265|258|válido|
|practice_0085|mixed|E3|donante8/coste1/orden0|266|258|válido|
|practice_0085|mixed|E3|donante8/coste0/orden1|265|258|válido|
|practice_0085|mixed|E3|donante8/coste1/orden1|266|258|válido|
|practice_0085|mixed|E3|donante9/coste0/orden0|319|255|válido|
|practice_0085|mixed|E3|donante9/coste1/orden0|295|255|válido|
|practice_0085|mixed|E3|donante9/coste0/orden1|351|255|válido|
|practice_0085|mixed|E3|donante9/coste1/orden1|275|255|válido|
|practice_0086|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0086|shared|Cruce|CP True/coste0/orden0|135|135|válido|
|practice_0086|shared|Cruce|CP True/coste1/orden0|135|135|válido|
|practice_0086|shared|Cruce|CP True/coste0/orden1|135|135|válido|
|practice_0086|shared|Cruce|CP True/coste1/orden1|135|135|válido|
|practice_0086|shared|Cruce|mejor independiente/coste0/orden0|136|136|válido|
|practice_0086|shared|Cruce|mejor independiente/coste1/orden0|136|136|válido|
|practice_0086|shared|Cruce|mejor independiente/coste0/orden1|136|136|válido|
|practice_0086|shared|Cruce|mejor independiente/coste1/orden1|136|136|válido|
|practice_0086|shared|E1|cola10/pol1/inicio0|143|49|válido|
|practice_0086|shared|E1|cola10/pol1/inicio1|143|49|válido|
|practice_0086|shared|E1|cola10/pol2/inicio0|143|49|válido|
|practice_0086|shared|E1|cola10/pol2/inicio1|143|49|válido|
|practice_0086|shared|E1|cola8/pol1/inicio0|143|43|válido|
|practice_0086|shared|E1|cola8/pol1/inicio1|143|43|válido|
|practice_0086|shared|E1|cola8/pol2/inicio0|143|43|válido|
|practice_0086|shared|E1|cola8/pol2/inicio1|143|43|válido|
|practice_0086|shared|E1|cola6/pol1/inicio0|142|34|válido|
|practice_0086|shared|E1|cola6/pol1/inicio1|142|34|válido|
|practice_0086|shared|E1|cola6/pol2/inicio0|142|34|válido|
|practice_0086|shared|E1|cola6/pol2/inicio1|142|34|válido|
|practice_0086|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0086|shared|E2|CP False/coste0/orden0|137|137|válido|
|practice_0086|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0086|shared|E2|CP True/coste0/orden0|136|136|válido|
|practice_0086|shared|E3|donante0/coste0/orden0|138|138|válido|
|practice_0086|shared|E3|donante0/coste1/orden0|138|138|válido|
|practice_0086|shared|E3|donante0/coste0/orden1|138|138|válido|
|practice_0086|shared|E3|donante0/coste1/orden1|138|138|válido|
|practice_0086|shared|E3|donante1/coste0/orden0|153|153|válido|
|practice_0086|shared|E3|donante1/coste1/orden0|153|153|válido|
|practice_0086|shared|E3|donante1/coste0/orden1|153|153|válido|
|practice_0086|shared|E3|donante1/coste1/orden1|153|153|válido|
|practice_0086|shared|E3|donante2/coste0/orden0|146|146|válido|
|practice_0086|shared|E3|donante2/coste1/orden0|146|146|válido|
|practice_0086|shared|E3|donante2/coste0/orden1|146|146|válido|
|practice_0086|shared|E3|donante2/coste1/orden1|146|146|válido|
|practice_0086|shared|E3|donante3/coste0/orden0|146|146|válido|
|practice_0086|shared|E3|donante3/coste1/orden0|146|146|válido|
|practice_0086|shared|E3|donante3/coste0/orden1|146|146|válido|
|practice_0086|shared|E3|donante3/coste1/orden1|146|146|válido|
|practice_0086|shared|E3|donante4/coste0/orden0|156|156|válido|
|practice_0086|shared|E3|donante4/coste1/orden0|156|156|válido|
|practice_0086|shared|E3|donante4/coste0/orden1|156|156|válido|
|practice_0086|shared|E3|donante4/coste1/orden1|156|156|válido|
|practice_0086|shared|E3|donante5/coste0/orden0|148|148|válido|
|practice_0086|shared|E3|donante5/coste1/orden0|148|148|válido|
|practice_0086|shared|E3|donante5/coste0/orden1|148|148|válido|
|practice_0086|shared|E3|donante5/coste1/orden1|148|148|válido|
|practice_0086|shared|E3|donante6/coste0/orden0|150|150|válido|
|practice_0086|shared|E3|donante6/coste1/orden0|150|150|válido|
|practice_0086|shared|E3|donante6/coste0/orden1|150|150|válido|
|practice_0086|shared|E3|donante6/coste1/orden1|150|150|válido|
|practice_0086|shared|E3|donante7/coste0/orden0|167|146|válido|
|practice_0086|shared|E3|donante7/coste1/orden0|153|146|válido|
|practice_0086|shared|E3|donante7/coste0/orden1|149|146|válido|
|practice_0086|shared|E3|donante7/coste1/orden1|148|146|válido|
|practice_0086|shared|E3|donante8/coste0/orden0|140|140|válido|
|practice_0086|shared|E3|donante8/coste1/orden0|140|140|válido|
|practice_0086|shared|E3|donante8/coste0/orden1|140|140|válido|
|practice_0086|shared|E3|donante8/coste1/orden1|140|140|válido|
|practice_0086|shared|E3|donante9/coste0/orden0|155|141|válido|
|practice_0086|shared|E3|donante9/coste1/orden0|154|141|válido|
|practice_0086|shared|E3|donante9/coste0/orden1|147|141|válido|
|practice_0086|shared|E3|donante9/coste1/orden1|147|141|válido|
|practice_0087|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0087|mixed|Cruce|CP True/coste0/orden0|241|215|válido|
|practice_0087|mixed|Cruce|CP True/coste1/orden0|236|215|válido|
|practice_0087|mixed|Cruce|CP True/coste0/orden1|232|215|válido|
|practice_0087|mixed|Cruce|CP True/coste1/orden1|231|215|válido|
|practice_0087|mixed|Cruce|mejor independiente/coste0/orden0|281|221|válido|
|practice_0087|mixed|Cruce|mejor independiente/coste1/orden0|245|221|válido|
|practice_0087|mixed|Cruce|mejor independiente/coste0/orden1|234|221|válido|
|practice_0087|mixed|Cruce|mejor independiente/coste1/orden1|222|221|válido|
|practice_0087|mixed|E1|cola10/pol1/inicio0|235|89|válido|
|practice_0087|mixed|E1|cola10/pol1/inicio1|235|89|válido|
|practice_0087|mixed|E1|cola10/pol2/inicio0|234|88|válido|
|practice_0087|mixed|E1|cola10/pol2/inicio1|235|89|válido|
|practice_0087|mixed|E1|cola8/pol1/inicio0|235|71|válido|
|practice_0087|mixed|E1|cola8/pol1/inicio1|235|71|válido|
|practice_0087|mixed|E1|cola8/pol2/inicio0|231|67|válido|
|practice_0087|mixed|E1|cola8/pol2/inicio1|231|67|válido|
|practice_0087|mixed|E1|cola6/pol1/inicio0|232|51|válido|
|practice_0087|mixed|E1|cola6/pol1/inicio1|235|54|válido|
|practice_0087|mixed|E1|cola6/pol2/inicio0|232|51|válido|
|practice_0087|mixed|E1|cola6/pol2/inicio1|232|51|válido|
|practice_0087|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0087|mixed|E2|CP False/coste0/orden0|234|223|válido|
|practice_0087|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0087|mixed|E2|CP True/coste0/orden0|242|224|válido|
|practice_0087|mixed|E3|donante0/coste0/orden0|231|224|válido|
|practice_0087|mixed|E3|donante0/coste1/orden0|258|224|válido|
|practice_0087|mixed|E3|donante0/coste0/orden1|258|224|válido|
|practice_0087|mixed|E3|donante0/coste1/orden1|258|224|válido|
|practice_0087|mixed|E3|donante1/coste0/orden0|255|255|válido|
|practice_0087|mixed|E3|donante1/coste1/orden0|255|255|válido|
|practice_0087|mixed|E3|donante1/coste0/orden1|255|255|válido|
|practice_0087|mixed|E3|donante1/coste1/orden1|255|255|válido|
|practice_0087|mixed|E3|donante2/coste0/orden0|252|249|válido|
|practice_0087|mixed|E3|donante2/coste1/orden0|252|249|válido|
|practice_0087|mixed|E3|donante2/coste0/orden1|249|249|válido|
|practice_0087|mixed|E3|donante2/coste1/orden1|249|249|válido|
|practice_0087|mixed|E3|donante3/coste0/orden0|337|240|válido|
|practice_0087|mixed|E3|donante3/coste1/orden0|266|240|válido|
|practice_0087|mixed|E3|donante3/coste0/orden1|240|240|válido|
|practice_0087|mixed|E3|donante3/coste1/orden1|240|240|válido|
|practice_0087|mixed|E3|donante4/coste0/orden0|264|264|válido|
|practice_0087|mixed|E3|donante4/coste1/orden0|264|264|válido|
|practice_0087|mixed|E3|donante4/coste0/orden1|264|264|válido|
|practice_0087|mixed|E3|donante4/coste1/orden1|264|264|válido|
|practice_0087|mixed|E3|donante5/coste0/orden0|249|249|válido|
|practice_0087|mixed|E3|donante5/coste1/orden0|249|249|válido|
|practice_0087|mixed|E3|donante5/coste0/orden1|249|249|válido|
|practice_0087|mixed|E3|donante5/coste1/orden1|249|249|válido|
|practice_0087|mixed|E3|donante6/coste0/orden0|256|256|válido|
|practice_0087|mixed|E3|donante6/coste1/orden0|256|256|válido|
|practice_0087|mixed|E3|donante6/coste0/orden1|256|256|válido|
|practice_0087|mixed|E3|donante6/coste1/orden1|256|256|válido|
|practice_0087|mixed|E3|donante7/coste0/orden0|244|229|válido|
|practice_0087|mixed|E3|donante7/coste1/orden0|233|229|válido|
|practice_0087|mixed|E3|donante7/coste0/orden1|235|229|válido|
|practice_0087|mixed|E3|donante7/coste1/orden1|233|229|válido|
|practice_0087|mixed|E3|donante8/coste0/orden0|233|223|válido|
|practice_0087|mixed|E3|donante8/coste1/orden0|230|223|válido|
|practice_0087|mixed|E3|donante8/coste0/orden1|230|223|válido|
|practice_0087|mixed|E3|donante8/coste1/orden1|230|223|válido|
|practice_0087|mixed|E3|donante9/coste0/orden0|255|221|válido|
|practice_0087|mixed|E3|donante9/coste1/orden0|247|221|válido|
|practice_0087|mixed|E3|donante9/coste0/orden1|248|221|válido|
|practice_0087|mixed|E3|donante9/coste1/orden1|240|221|válido|
|practice_0088|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0088|shared|Cruce|CP True/coste0/orden0|195|193|válido|
|practice_0088|shared|Cruce|CP True/coste1/orden0|195|193|válido|
|practice_0088|shared|Cruce|CP True/coste0/orden1|193|193|válido|
|practice_0088|shared|Cruce|CP True/coste1/orden1|193|193|válido|
|practice_0088|shared|Cruce|mejor independiente/coste0/orden0|198|194|válido|
|practice_0088|shared|Cruce|mejor independiente/coste1/orden0|199|194|válido|
|practice_0088|shared|Cruce|mejor independiente/coste0/orden1|194|194|válido|
|practice_0088|shared|Cruce|mejor independiente/coste1/orden1|194|194|válido|
|practice_0088|shared|E1|cola10/pol1/inicio0|208|73|válido|
|practice_0088|shared|E1|cola10/pol1/inicio1|208|73|válido|
|practice_0088|shared|E1|cola10/pol2/inicio0|205|70|válido|
|practice_0088|shared|E1|cola10/pol2/inicio1|205|70|válido|
|practice_0088|shared|E1|cola8/pol1/inicio0|203|54|válido|
|practice_0088|shared|E1|cola8/pol1/inicio1|203|54|válido|
|practice_0088|shared|E1|cola8/pol2/inicio0|203|54|válido|
|practice_0088|shared|E1|cola8/pol2/inicio1|203|54|válido|
|practice_0088|shared|E1|cola6/pol1/inicio0|207|44|válido|
|practice_0088|shared|E1|cola6/pol1/inicio1|207|44|válido|
|practice_0088|shared|E1|cola6/pol2/inicio0|207|44|válido|
|practice_0088|shared|E1|cola6/pol2/inicio1|207|44|válido|
|practice_0088|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0088|shared|E2|CP False/coste0/orden0|236|227|válido|
|practice_0088|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0088|shared|E2|CP True/coste0/orden0|195|193|válido|
|practice_0088|shared|E3|donante0/coste0/orden0|204|194|válido|
|practice_0088|shared|E3|donante0/coste1/orden0|207|194|válido|
|practice_0088|shared|E3|donante0/coste0/orden1|194|194|válido|
|practice_0088|shared|E3|donante0/coste1/orden1|194|194|válido|
|practice_0088|shared|E3|donante1/coste0/orden0|233|233|válido|
|practice_0088|shared|E3|donante1/coste1/orden0|233|233|válido|
|practice_0088|shared|E3|donante1/coste0/orden1|233|233|válido|
|practice_0088|shared|E3|donante1/coste1/orden1|233|233|válido|
|practice_0088|shared|E3|donante2/coste0/orden0|228|228|válido|
|practice_0088|shared|E3|donante2/coste1/orden0|228|228|válido|
|practice_0088|shared|E3|donante2/coste0/orden1|228|228|válido|
|practice_0088|shared|E3|donante2/coste1/orden1|228|228|válido|
|practice_0088|shared|E3|donante3/coste0/orden0|215|208|válido|
|practice_0088|shared|E3|donante3/coste1/orden0|214|208|válido|
|practice_0088|shared|E3|donante3/coste0/orden1|208|208|válido|
|practice_0088|shared|E3|donante3/coste1/orden1|208|208|válido|
|practice_0088|shared|E3|donante4/coste0/orden0|245|245|válido|
|practice_0088|shared|E3|donante4/coste1/orden0|245|245|válido|
|practice_0088|shared|E3|donante4/coste0/orden1|245|245|válido|
|practice_0088|shared|E3|donante4/coste1/orden1|245|245|válido|
|practice_0088|shared|E3|donante5/coste0/orden0|221|221|válido|
|practice_0088|shared|E3|donante5/coste1/orden0|221|221|válido|
|practice_0088|shared|E3|donante5/coste0/orden1|221|221|válido|
|practice_0088|shared|E3|donante5/coste1/orden1|221|221|válido|
|practice_0088|shared|E3|donante6/coste0/orden0|245|245|válido|
|practice_0088|shared|E3|donante6/coste1/orden0|245|245|válido|
|practice_0088|shared|E3|donante6/coste0/orden1|245|245|válido|
|practice_0088|shared|E3|donante6/coste1/orden1|245|245|válido|
|practice_0088|shared|E3|donante7/coste0/orden0|211|211|válido|
|practice_0088|shared|E3|donante7/coste1/orden0|211|211|válido|
|practice_0088|shared|E3|donante7/coste0/orden1|211|211|válido|
|practice_0088|shared|E3|donante7/coste1/orden1|211|211|válido|
|practice_0088|shared|E3|donante8/coste0/orden0|213|208|válido|
|practice_0088|shared|E3|donante8/coste1/orden0|209|208|válido|
|practice_0088|shared|E3|donante8/coste0/orden1|213|208|válido|
|practice_0088|shared|E3|donante8/coste1/orden1|210|208|válido|
|practice_0088|shared|E3|donante9/coste0/orden0|205|195|válido|
|practice_0088|shared|E3|donante9/coste1/orden0|208|195|válido|
|practice_0088|shared|E3|donante9/coste0/orden1|195|195|válido|
|practice_0088|shared|E3|donante9/coste1/orden1|195|195|válido|
|practice_0089|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0089|shared|Cruce|CP True/coste0/orden0|148|148|válido|
|practice_0089|shared|Cruce|CP True/coste1/orden0|148|148|válido|
|practice_0089|shared|Cruce|CP True/coste0/orden1|148|148|válido|
|practice_0089|shared|Cruce|CP True/coste1/orden1|148|148|válido|
|practice_0089|shared|Cruce|mejor independiente/coste0/orden0|149|149|válido|
|practice_0089|shared|Cruce|mejor independiente/coste1/orden0|149|149|válido|
|practice_0089|shared|Cruce|mejor independiente/coste0/orden1|149|149|válido|
|practice_0089|shared|Cruce|mejor independiente/coste1/orden1|149|149|válido|
|practice_0089|shared|E1|cola10/pol1/inicio0|155|21|válido|
|practice_0089|shared|E1|cola10/pol1/inicio1|155|21|válido|
|practice_0089|shared|E1|cola10/pol2/inicio0|155|21|válido|
|practice_0089|shared|E1|cola10/pol2/inicio1|155|21|válido|
|practice_0089|shared|E1|cola8/pol1/inicio0|155|16|válido|
|practice_0089|shared|E1|cola8/pol1/inicio1|155|16|válido|
|practice_0089|shared|E1|cola8/pol2/inicio0|155|16|válido|
|practice_0089|shared|E1|cola8/pol2/inicio1|155|16|válido|
|practice_0089|shared|E1|cola6/pol1/inicio0|155|12|válido|
|practice_0089|shared|E1|cola6/pol1/inicio1|155|12|válido|
|practice_0089|shared|E1|cola6/pol2/inicio0|155|12|válido|
|practice_0089|shared|E1|cola6/pol2/inicio1|155|12|válido|
|practice_0089|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0089|shared|E2|CP False/coste0/orden0|151|151|válido|
|practice_0089|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0089|shared|E2|CP True/coste0/orden0|149|149|válido|
|practice_0089|shared|E3|donante0/coste0/orden0|154|154|válido|
|practice_0089|shared|E3|donante0/coste1/orden0|154|154|válido|
|practice_0089|shared|E3|donante0/coste0/orden1|154|154|válido|
|practice_0089|shared|E3|donante0/coste1/orden1|154|154|válido|
|practice_0089|shared|E3|donante1/coste0/orden0|165|165|válido|
|practice_0089|shared|E3|donante1/coste1/orden0|165|165|válido|
|practice_0089|shared|E3|donante1/coste0/orden1|165|165|válido|
|practice_0089|shared|E3|donante1/coste1/orden1|165|165|válido|
|practice_0089|shared|E3|donante2/coste0/orden0|155|155|válido|
|practice_0089|shared|E3|donante2/coste1/orden0|155|155|válido|
|practice_0089|shared|E3|donante2/coste0/orden1|155|155|válido|
|practice_0089|shared|E3|donante2/coste1/orden1|155|155|válido|
|practice_0089|shared|E3|donante3/coste0/orden0|155|155|válido|
|practice_0089|shared|E3|donante3/coste1/orden0|155|155|válido|
|practice_0089|shared|E3|donante3/coste0/orden1|155|155|válido|
|practice_0089|shared|E3|donante3/coste1/orden1|155|155|válido|
|practice_0089|shared|E3|donante4/coste0/orden0|193|193|válido|
|practice_0089|shared|E3|donante4/coste1/orden0|193|193|válido|
|practice_0089|shared|E3|donante4/coste0/orden1|193|193|válido|
|practice_0089|shared|E3|donante4/coste1/orden1|193|193|válido|
|practice_0089|shared|E3|donante5/coste0/orden0|181|181|válido|
|practice_0089|shared|E3|donante5/coste1/orden0|181|181|válido|
|practice_0089|shared|E3|donante5/coste0/orden1|181|181|válido|
|practice_0089|shared|E3|donante5/coste1/orden1|181|181|válido|
|practice_0089|shared|E3|donante6/coste0/orden0|195|195|válido|
|practice_0089|shared|E3|donante6/coste1/orden0|195|195|válido|
|practice_0089|shared|E3|donante6/coste0/orden1|195|195|válido|
|practice_0089|shared|E3|donante6/coste1/orden1|195|195|válido|
|practice_0089|shared|E3|donante7/coste0/orden0|209|178|válido|
|practice_0089|shared|E3|donante7/coste1/orden0|197|178|válido|
|practice_0089|shared|E3|donante7/coste0/orden1|178|178|válido|
|practice_0089|shared|E3|donante7/coste1/orden1|178|178|válido|
|practice_0089|shared|E3|donante8/coste0/orden0|217|166|válido|
|practice_0089|shared|E3|donante8/coste1/orden0|173|166|válido|
|practice_0089|shared|E3|donante8/coste0/orden1|215|166|válido|
|practice_0089|shared|E3|donante8/coste1/orden1|183|166|válido|
|practice_0089|shared|E3|donante9/coste0/orden0|165|158|válido|
|practice_0089|shared|E3|donante9/coste1/orden0|165|158|válido|
|practice_0089|shared|E3|donante9/coste0/orden1|158|158|válido|
|practice_0089|shared|E3|donante9/coste1/orden1|158|158|válido|
|practice_0090|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0090|shared|Cruce|CP True/coste0/orden0|171|143|válido|
|practice_0090|shared|Cruce|CP True/coste1/orden0|149|143|válido|
|practice_0090|shared|Cruce|CP True/coste0/orden1|143|143|válido|
|practice_0090|shared|Cruce|CP True/coste1/orden1|143|143|válido|
|practice_0090|shared|Cruce|mejor independiente/coste0/orden0|197|144|válido|
|practice_0090|shared|Cruce|mejor independiente/coste1/orden0|152|144|válido|
|practice_0090|shared|Cruce|mejor independiente/coste0/orden1|144|144|válido|
|practice_0090|shared|Cruce|mejor independiente/coste1/orden1|144|144|válido|
|practice_0090|shared|E1|cola10/pol1/inicio0|148|51|válido|
|practice_0090|shared|E1|cola10/pol1/inicio1|148|51|válido|
|practice_0090|shared|E1|cola10/pol2/inicio0|148|51|válido|
|practice_0090|shared|E1|cola10/pol2/inicio1|148|51|válido|
|practice_0090|shared|E1|cola8/pol1/inicio0|146|38|válido|
|practice_0090|shared|E1|cola8/pol1/inicio1|146|38|válido|
|practice_0090|shared|E1|cola8/pol2/inicio0|148|40|válido|
|practice_0090|shared|E1|cola8/pol2/inicio1|148|40|válido|
|practice_0090|shared|E1|cola6/pol1/inicio0|149|29|válido|
|practice_0090|shared|E1|cola6/pol1/inicio1|149|29|válido|
|practice_0090|shared|E1|cola6/pol2/inicio0|149|29|válido|
|practice_0090|shared|E1|cola6/pol2/inicio1|149|29|válido|
|practice_0090|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0090|shared|E2|CP False/coste0/orden0|152|152|válido|
|practice_0090|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0090|shared|E2|CP True/coste0/orden0|173|143|válido|
|practice_0090|shared|E3|donante0/coste0/orden0|154|144|válido|
|practice_0090|shared|E3|donante0/coste1/orden0|146|144|válido|
|practice_0090|shared|E3|donante0/coste0/orden1|144|144|válido|
|practice_0090|shared|E3|donante0/coste1/orden1|144|144|válido|
|practice_0090|shared|E3|donante1/coste0/orden0|155|155|válido|
|practice_0090|shared|E3|donante1/coste1/orden0|155|155|válido|
|practice_0090|shared|E3|donante1/coste0/orden1|155|155|válido|
|practice_0090|shared|E3|donante1/coste1/orden1|155|155|válido|
|practice_0090|shared|E3|donante2/coste0/orden0|158|158|válido|
|practice_0090|shared|E3|donante2/coste1/orden0|158|158|válido|
|practice_0090|shared|E3|donante2/coste0/orden1|158|158|válido|
|practice_0090|shared|E3|donante2/coste1/orden1|158|158|válido|
|practice_0090|shared|E3|donante3/coste0/orden0|154|154|válido|
|practice_0090|shared|E3|donante3/coste1/orden0|154|154|válido|
|practice_0090|shared|E3|donante3/coste0/orden1|154|154|válido|
|practice_0090|shared|E3|donante3/coste1/orden1|154|154|válido|
|practice_0090|shared|E3|donante4/coste0/orden0|156|156|válido|
|practice_0090|shared|E3|donante4/coste1/orden0|156|156|válido|
|practice_0090|shared|E3|donante4/coste0/orden1|156|156|válido|
|practice_0090|shared|E3|donante4/coste1/orden1|156|156|válido|
|practice_0090|shared|E3|donante5/coste0/orden0|153|153|válido|
|practice_0090|shared|E3|donante5/coste1/orden0|153|153|válido|
|practice_0090|shared|E3|donante5/coste0/orden1|153|153|válido|
|practice_0090|shared|E3|donante5/coste1/orden1|153|153|válido|
|practice_0090|shared|E3|donante6/coste0/orden0|167|167|válido|
|practice_0090|shared|E3|donante6/coste1/orden0|167|167|válido|
|practice_0090|shared|E3|donante6/coste0/orden1|167|167|válido|
|practice_0090|shared|E3|donante6/coste1/orden1|167|167|válido|
|practice_0090|shared|E3|donante7/coste0/orden0|193|150|válido|
|practice_0090|shared|E3|donante7/coste1/orden0|175|150|válido|
|practice_0090|shared|E3|donante7/coste0/orden1|152|150|válido|
|practice_0090|shared|E3|donante7/coste1/orden1|152|150|válido|
|practice_0090|shared|E3|donante8/coste0/orden0|153|145|válido|
|practice_0090|shared|E3|donante8/coste1/orden0|147|145|válido|
|practice_0090|shared|E3|donante8/coste0/orden1|145|145|válido|
|practice_0090|shared|E3|donante8/coste1/orden1|145|145|válido|
|practice_0090|shared|E3|donante9/coste0/orden0|159|149|válido|
|practice_0090|shared|E3|donante9/coste1/orden0|159|149|válido|
|practice_0090|shared|E3|donante9/coste0/orden1|149|149|válido|
|practice_0090|shared|E3|donante9/coste1/orden1|149|149|válido|
|practice_0091|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0091|shared|Cruce|CP True/coste0/orden0|225|195|válido|
|practice_0091|shared|Cruce|CP True/coste1/orden0|201|195|válido|
|practice_0091|shared|Cruce|CP True/coste0/orden1|205|195|válido|
|practice_0091|shared|Cruce|CP True/coste1/orden1|196|195|válido|
|practice_0091|shared|Cruce|mejor independiente/coste0/orden0|225|195|válido|
|practice_0091|shared|Cruce|mejor independiente/coste1/orden0|209|195|válido|
|practice_0091|shared|Cruce|mejor independiente/coste0/orden1|205|195|válido|
|practice_0091|shared|Cruce|mejor independiente/coste1/orden1|196|195|válido|
|practice_0091|shared|E1|cola10/pol1/inicio0|204|66|válido|
|practice_0091|shared|E1|cola10/pol1/inicio1|204|66|válido|
|practice_0091|shared|E1|cola10/pol2/inicio0|204|66|válido|
|practice_0091|shared|E1|cola10/pol2/inicio1|204|66|válido|
|practice_0091|shared|E1|cola8/pol1/inicio0|202|53|válido|
|practice_0091|shared|E1|cola8/pol1/inicio1|202|53|válido|
|practice_0091|shared|E1|cola8/pol2/inicio0|202|53|válido|
|practice_0091|shared|E1|cola8/pol2/inicio1|202|53|válido|
|practice_0091|shared|E1|cola6/pol1/inicio0|203|44|válido|
|practice_0091|shared|E1|cola6/pol1/inicio1|203|44|válido|
|practice_0091|shared|E1|cola6/pol2/inicio0|203|44|válido|
|practice_0091|shared|E1|cola6/pol2/inicio1|203|44|válido|
|practice_0091|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0091|shared|E2|CP False/coste0/orden0|250|204|válido|
|practice_0091|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0091|shared|E2|CP True/coste0/orden0|225|195|válido|
|practice_0091|shared|E3|donante0/coste0/orden0|205|195|válido|
|practice_0091|shared|E3|donante0/coste1/orden0|196|195|válido|
|practice_0091|shared|E3|donante0/coste0/orden1|205|195|válido|
|practice_0091|shared|E3|donante0/coste1/orden1|196|195|válido|
|practice_0091|shared|E3|donante1/coste0/orden0|230|230|válido|
|practice_0091|shared|E3|donante1/coste1/orden0|230|230|válido|
|practice_0091|shared|E3|donante1/coste0/orden1|230|230|válido|
|practice_0091|shared|E3|donante1/coste1/orden1|230|230|válido|
|practice_0091|shared|E3|donante2/coste0/orden0|205|205|válido|
|practice_0091|shared|E3|donante2/coste1/orden0|205|205|válido|
|practice_0091|shared|E3|donante2/coste0/orden1|205|205|válido|
|practice_0091|shared|E3|donante2/coste1/orden1|205|205|válido|
|practice_0091|shared|E3|donante3/coste0/orden0|234|206|válido|
|practice_0091|shared|E3|donante3/coste1/orden0|239|206|válido|
|practice_0091|shared|E3|donante3/coste0/orden1|206|206|válido|
|practice_0091|shared|E3|donante3/coste1/orden1|206|206|válido|
|practice_0091|shared|E3|donante4/coste0/orden0|262|262|válido|
|practice_0091|shared|E3|donante4/coste1/orden0|262|262|válido|
|practice_0091|shared|E3|donante4/coste0/orden1|262|262|válido|
|practice_0091|shared|E3|donante4/coste1/orden1|262|262|válido|
|practice_0091|shared|E3|donante5/coste0/orden0|239|239|válido|
|practice_0091|shared|E3|donante5/coste1/orden0|239|239|válido|
|practice_0091|shared|E3|donante5/coste0/orden1|239|239|válido|
|practice_0091|shared|E3|donante5/coste1/orden1|239|239|válido|
|practice_0091|shared|E3|donante6/coste0/orden0|244|244|válido|
|practice_0091|shared|E3|donante6/coste1/orden0|244|244|válido|
|practice_0091|shared|E3|donante6/coste0/orden1|244|244|válido|
|practice_0091|shared|E3|donante6/coste1/orden1|244|244|válido|
|practice_0091|shared|E3|donante7/coste0/orden0|245|211|válido|
|practice_0091|shared|E3|donante7/coste1/orden0|223|211|válido|
|practice_0091|shared|E3|donante7/coste0/orden1|214|211|válido|
|practice_0091|shared|E3|donante7/coste1/orden1|214|211|válido|
|practice_0091|shared|E3|donante8/coste0/orden0|205|195|válido|
|practice_0091|shared|E3|donante8/coste1/orden0|196|195|válido|
|practice_0091|shared|E3|donante8/coste0/orden1|205|195|válido|
|practice_0091|shared|E3|donante8/coste1/orden1|196|195|válido|
|practice_0091|shared|E3|donante9/coste0/orden0|217|199|válido|
|practice_0091|shared|E3|donante9/coste1/orden0|212|199|válido|
|practice_0091|shared|E3|donante9/coste0/orden1|224|199|válido|
|practice_0091|shared|E3|donante9/coste1/orden1|216|199|válido|
|practice_0092|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0092|shared|Cruce|CP True/coste0/orden0|155|151|válido|
|practice_0092|shared|Cruce|CP True/coste1/orden0|157|151|válido|
|practice_0092|shared|Cruce|CP True/coste0/orden1|151|151|válido|
|practice_0092|shared|Cruce|CP True/coste1/orden1|151|151|válido|
|practice_0092|shared|Cruce|mejor independiente/coste0/orden0|153|153|válido|
|practice_0092|shared|Cruce|mejor independiente/coste1/orden0|153|153|válido|
|practice_0092|shared|Cruce|mejor independiente/coste0/orden1|153|153|válido|
|practice_0092|shared|Cruce|mejor independiente/coste1/orden1|153|153|válido|
|practice_0092|shared|E1|cola10/pol1/inicio0|157|48|válido|
|practice_0092|shared|E1|cola10/pol1/inicio1|157|48|válido|
|practice_0092|shared|E1|cola10/pol2/inicio0|163|54|válido|
|practice_0092|shared|E1|cola10/pol2/inicio1|163|54|válido|
|practice_0092|shared|E1|cola8/pol1/inicio0|159|40|válido|
|practice_0092|shared|E1|cola8/pol1/inicio1|159|40|válido|
|practice_0092|shared|E1|cola8/pol2/inicio0|156|37|válido|
|practice_0092|shared|E1|cola8/pol2/inicio1|156|37|válido|
|practice_0092|shared|E1|cola6/pol1/inicio0|164|37|válido|
|practice_0092|shared|E1|cola6/pol1/inicio1|164|37|válido|
|practice_0092|shared|E1|cola6/pol2/inicio0|160|33|válido|
|practice_0092|shared|E1|cola6/pol2/inicio1|160|33|válido|
|practice_0092|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0092|shared|E2|CP False/coste0/orden0|166|156|válido|
|practice_0092|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0092|shared|E2|CP True/coste0/orden0|167|152|válido|
|practice_0092|shared|E3|donante0/coste0/orden0|156|156|válido|
|practice_0092|shared|E3|donante0/coste1/orden0|156|156|válido|
|practice_0092|shared|E3|donante0/coste0/orden1|156|156|válido|
|practice_0092|shared|E3|donante0/coste1/orden1|156|156|válido|
|practice_0092|shared|E3|donante1/coste0/orden0|183|183|válido|
|practice_0092|shared|E3|donante1/coste1/orden0|183|183|válido|
|practice_0092|shared|E3|donante1/coste0/orden1|183|183|válido|
|practice_0092|shared|E3|donante1/coste1/orden1|183|183|válido|
|practice_0092|shared|E3|donante2/coste0/orden0|174|174|válido|
|practice_0092|shared|E3|donante2/coste1/orden0|174|174|válido|
|practice_0092|shared|E3|donante2/coste0/orden1|174|174|válido|
|practice_0092|shared|E3|donante2/coste1/orden1|174|174|válido|
|practice_0092|shared|E3|donante3/coste0/orden0|165|165|válido|
|practice_0092|shared|E3|donante3/coste1/orden0|165|165|válido|
|practice_0092|shared|E3|donante3/coste0/orden1|165|165|válido|
|practice_0092|shared|E3|donante3/coste1/orden1|165|165|válido|
|practice_0092|shared|E3|donante4/coste0/orden0|195|195|válido|
|practice_0092|shared|E3|donante4/coste1/orden0|195|195|válido|
|practice_0092|shared|E3|donante4/coste0/orden1|195|195|válido|
|practice_0092|shared|E3|donante4/coste1/orden1|195|195|válido|
|practice_0092|shared|E3|donante5/coste0/orden0|176|176|válido|
|practice_0092|shared|E3|donante5/coste1/orden0|176|176|válido|
|practice_0092|shared|E3|donante5/coste0/orden1|176|176|válido|
|practice_0092|shared|E3|donante5/coste1/orden1|176|176|válido|
|practice_0092|shared|E3|donante6/coste0/orden0|184|184|válido|
|practice_0092|shared|E3|donante6/coste1/orden0|184|184|válido|
|practice_0092|shared|E3|donante6/coste0/orden1|184|184|válido|
|practice_0092|shared|E3|donante6/coste1/orden1|184|184|válido|
|practice_0092|shared|E3|donante7/coste0/orden0|171|171|válido|
|practice_0092|shared|E3|donante7/coste1/orden0|171|171|válido|
|practice_0092|shared|E3|donante7/coste0/orden1|190|171|válido|
|practice_0092|shared|E3|donante7/coste1/orden1|173|171|válido|
|practice_0092|shared|E3|donante8/coste0/orden0|157|157|válido|
|practice_0092|shared|E3|donante8/coste1/orden0|157|157|válido|
|practice_0092|shared|E3|donante8/coste0/orden1|157|157|válido|
|practice_0092|shared|E3|donante8/coste1/orden1|157|157|válido|
|practice_0093|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0093|mixed|Cruce|CP True/coste0/orden0|227|212|válido|
|practice_0093|mixed|Cruce|CP True/coste1/orden0|224|212|válido|
|practice_0093|mixed|Cruce|CP True/coste0/orden1|212|212|válido|
|practice_0093|mixed|Cruce|CP True/coste1/orden1|212|212|válido|
|practice_0093|mixed|Cruce|mejor independiente/coste0/orden0|227|212|válido|
|practice_0093|mixed|Cruce|mejor independiente/coste1/orden0|224|212|válido|
|practice_0093|mixed|Cruce|mejor independiente/coste0/orden1|212|212|válido|
|practice_0093|mixed|Cruce|mejor independiente/coste1/orden1|212|212|válido|
|practice_0093|mixed|E1|cola10/pol1/inicio0|217|68|válido|
|practice_0093|mixed|E1|cola10/pol1/inicio1|217|68|válido|
|practice_0093|mixed|E1|cola10/pol2/inicio0|219|70|válido|
|practice_0093|mixed|E1|cola10/pol2/inicio1|219|70|válido|
|practice_0093|mixed|E1|cola8/pol1/inicio0|220|58|válido|
|practice_0093|mixed|E1|cola8/pol1/inicio1|220|58|válido|
|practice_0093|mixed|E1|cola8/pol2/inicio0|217|55|válido|
|practice_0093|mixed|E1|cola8/pol2/inicio1|217|55|válido|
|practice_0093|mixed|E1|cola6/pol1/inicio0|216|40|válido|
|practice_0093|mixed|E1|cola6/pol1/inicio1|216|40|válido|
|practice_0093|mixed|E1|cola6/pol2/inicio0|216|40|válido|
|practice_0093|mixed|E1|cola6/pol2/inicio1|216|40|válido|
|practice_0093|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0093|mixed|E2|CP False/coste0/orden0|317|216|válido|
|practice_0093|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0093|mixed|E2|CP True/coste0/orden0|227|212|válido|
|practice_0093|mixed|E3|donante0/coste0/orden0|212|212|válido|
|practice_0093|mixed|E3|donante0/coste1/orden0|212|212|válido|
|practice_0093|mixed|E3|donante0/coste0/orden1|212|212|válido|
|practice_0093|mixed|E3|donante0/coste1/orden1|212|212|válido|
|practice_0093|mixed|E3|donante1/coste0/orden0|238|238|válido|
|practice_0093|mixed|E3|donante1/coste1/orden0|238|238|válido|
|practice_0093|mixed|E3|donante1/coste0/orden1|238|238|válido|
|practice_0093|mixed|E3|donante1/coste1/orden1|238|238|válido|
|practice_0093|mixed|E3|donante2/coste0/orden0|231|231|válido|
|practice_0093|mixed|E3|donante2/coste1/orden0|231|231|válido|
|practice_0093|mixed|E3|donante2/coste0/orden1|231|231|válido|
|practice_0093|mixed|E3|donante2/coste1/orden1|231|231|válido|
|practice_0093|mixed|E3|donante3/coste0/orden0|267|224|válido|
|practice_0093|mixed|E3|donante3/coste1/orden0|230|224|válido|
|practice_0093|mixed|E3|donante3/coste0/orden1|224|224|válido|
|practice_0093|mixed|E3|donante3/coste1/orden1|224|224|válido|
|practice_0093|mixed|E3|donante4/coste0/orden0|244|244|válido|
|practice_0093|mixed|E3|donante4/coste1/orden0|244|244|válido|
|practice_0093|mixed|E3|donante4/coste0/orden1|244|244|válido|
|practice_0093|mixed|E3|donante4/coste1/orden1|244|244|válido|
|practice_0093|mixed|E3|donante5/coste0/orden0|242|242|válido|
|practice_0093|mixed|E3|donante5/coste1/orden0|242|242|válido|
|practice_0093|mixed|E3|donante5/coste0/orden1|242|242|válido|
|practice_0093|mixed|E3|donante5/coste1/orden1|242|242|válido|
|practice_0093|mixed|E3|donante6/coste0/orden0|243|243|válido|
|practice_0093|mixed|E3|donante6/coste1/orden0|243|243|válido|
|practice_0093|mixed|E3|donante6/coste0/orden1|243|243|válido|
|practice_0093|mixed|E3|donante6/coste1/orden1|243|243|válido|
|practice_0093|mixed|E3|donante7/coste0/orden0|228|218|válido|
|practice_0093|mixed|E3|donante7/coste1/orden0|228|218|válido|
|practice_0093|mixed|E3|donante7/coste0/orden1|262|218|válido|
|practice_0093|mixed|E3|donante7/coste1/orden1|257|218|válido|
|practice_0093|mixed|E3|donante8/coste0/orden0|213|213|válido|
|practice_0093|mixed|E3|donante8/coste1/orden0|213|213|válido|
|practice_0093|mixed|E3|donante8/coste0/orden1|213|213|válido|
|practice_0093|mixed|E3|donante8/coste1/orden1|213|213|válido|
|practice_0093|mixed|E3|donante9/coste0/orden0|222|211|válido|
|practice_0093|mixed|E3|donante9/coste1/orden0|212|211|válido|
|practice_0093|mixed|E3|donante9/coste0/orden1|246|211|válido|
|practice_0093|mixed|E3|donante9/coste1/orden1|213|211|válido|
|practice_0094|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0094|multiplicative|Cruce|CP True/coste0/orden0|127|127|válido|
|practice_0094|multiplicative|Cruce|CP True/coste1/orden0|127|127|válido|
|practice_0094|multiplicative|Cruce|CP True/coste0/orden1|127|127|válido|
|practice_0094|multiplicative|Cruce|CP True/coste1/orden1|127|127|válido|
|practice_0094|multiplicative|Cruce|mejor independiente/coste0/orden0|128|128|válido|
|practice_0094|multiplicative|Cruce|mejor independiente/coste1/orden0|128|128|válido|
|practice_0094|multiplicative|Cruce|mejor independiente/coste0/orden1|128|128|válido|
|practice_0094|multiplicative|Cruce|mejor independiente/coste1/orden1|128|128|válido|
|practice_0094|multiplicative|E1|cola10/pol1/inicio0|133|48|válido|
|practice_0094|multiplicative|E1|cola10/pol1/inicio1|133|48|válido|
|practice_0094|multiplicative|E1|cola10/pol2/inicio0|133|48|válido|
|practice_0094|multiplicative|E1|cola10/pol2/inicio1|133|48|válido|
|practice_0094|multiplicative|E1|cola8/pol1/inicio0|134|40|válido|
|practice_0094|multiplicative|E1|cola8/pol1/inicio1|134|40|válido|
|practice_0094|multiplicative|E1|cola8/pol2/inicio0|133|39|válido|
|practice_0094|multiplicative|E1|cola8/pol2/inicio1|132|38|válido|
|practice_0094|multiplicative|E1|cola6/pol1/inicio0|133|30|válido|
|practice_0094|multiplicative|E1|cola6/pol1/inicio1|133|30|válido|
|practice_0094|multiplicative|E1|cola6/pol2/inicio0|133|30|válido|
|practice_0094|multiplicative|E1|cola6/pol2/inicio1|133|30|válido|
|practice_0094|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0094|multiplicative|E2|CP False/coste0/orden0|131|131|válido|
|practice_0094|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0094|multiplicative|E2|CP True/coste0/orden0|128|128|válido|
|practice_0094|multiplicative|E3|donante0/coste0/orden0|135|131|válido|
|practice_0094|multiplicative|E3|donante0/coste1/orden0|135|131|válido|
|practice_0094|multiplicative|E3|donante0/coste0/orden1|131|131|válido|
|practice_0094|multiplicative|E3|donante0/coste1/orden1|131|131|válido|
|practice_0094|multiplicative|E3|donante1/coste0/orden0|141|141|válido|
|practice_0094|multiplicative|E3|donante1/coste1/orden0|141|141|válido|
|practice_0094|multiplicative|E3|donante1/coste0/orden1|141|141|válido|
|practice_0094|multiplicative|E3|donante1/coste1/orden1|141|141|válido|
|practice_0094|multiplicative|E3|donante2/coste0/orden0|137|137|válido|
|practice_0094|multiplicative|E3|donante2/coste1/orden0|137|137|válido|
|practice_0094|multiplicative|E3|donante2/coste0/orden1|137|137|válido|
|practice_0094|multiplicative|E3|donante2/coste1/orden1|137|137|válido|
|practice_0094|multiplicative|E3|donante3/coste0/orden0|141|141|válido|
|practice_0094|multiplicative|E3|donante3/coste1/orden0|141|141|válido|
|practice_0094|multiplicative|E3|donante3/coste0/orden1|141|141|válido|
|practice_0094|multiplicative|E3|donante3/coste1/orden1|141|141|válido|
|practice_0094|multiplicative|E3|donante4/coste0/orden0|147|147|válido|
|practice_0094|multiplicative|E3|donante4/coste1/orden0|147|147|válido|
|practice_0094|multiplicative|E3|donante4/coste0/orden1|147|147|válido|
|practice_0094|multiplicative|E3|donante4/coste1/orden1|147|147|válido|
|practice_0094|multiplicative|E3|donante5/coste0/orden0|142|142|válido|
|practice_0094|multiplicative|E3|donante5/coste1/orden0|142|142|válido|
|practice_0094|multiplicative|E3|donante5/coste0/orden1|142|142|válido|
|practice_0094|multiplicative|E3|donante5/coste1/orden1|142|142|válido|
|practice_0094|multiplicative|E3|donante6/coste0/orden0|147|147|válido|
|practice_0094|multiplicative|E3|donante6/coste1/orden0|147|147|válido|
|practice_0094|multiplicative|E3|donante6/coste0/orden1|147|147|válido|
|practice_0094|multiplicative|E3|donante6/coste1/orden1|147|147|válido|
|practice_0094|multiplicative|E3|donante7/coste0/orden0|145|136|válido|
|practice_0094|multiplicative|E3|donante7/coste1/orden0|145|136|válido|
|practice_0094|multiplicative|E3|donante7/coste0/orden1|136|136|válido|
|practice_0094|multiplicative|E3|donante7/coste1/orden1|136|136|válido|
|practice_0094|multiplicative|E3|donante8/coste0/orden0|151|137|válido|
|practice_0094|multiplicative|E3|donante8/coste1/orden0|153|137|válido|
|practice_0094|multiplicative|E3|donante8/coste0/orden1|146|137|válido|
|practice_0094|multiplicative|E3|donante8/coste1/orden1|146|137|válido|
|practice_0094|multiplicative|E3|donante9/coste0/orden0|137|133|válido|
|practice_0094|multiplicative|E3|donante9/coste1/orden0|137|133|válido|
|practice_0094|multiplicative|E3|donante9/coste0/orden1|133|133|válido|
|practice_0094|multiplicative|E3|donante9/coste1/orden1|133|133|válido|
|practice_0095|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0095|mixed|Cruce|CP True/coste0/orden0|286|222|válido|
|practice_0095|mixed|Cruce|CP True/coste1/orden0|241|222|válido|
|practice_0095|mixed|Cruce|CP True/coste0/orden1|224|222|válido|
|practice_0095|mixed|Cruce|CP True/coste1/orden1|224|222|válido|
|practice_0095|mixed|Cruce|mejor independiente/coste0/orden0|297|224|válido|
|practice_0095|mixed|Cruce|mejor independiente/coste1/orden0|244|224|válido|
|practice_0095|mixed|Cruce|mejor independiente/coste0/orden1|229|224|válido|
|practice_0095|mixed|Cruce|mejor independiente/coste1/orden1|229|224|válido|
|practice_0095|mixed|E1|cola10/pol1/inicio0|232|71|válido|
|practice_0095|mixed|E1|cola10/pol1/inicio1|232|71|válido|
|practice_0095|mixed|E1|cola10/pol2/inicio0|232|71|válido|
|practice_0095|mixed|E1|cola10/pol2/inicio1|232|71|válido|
|practice_0095|mixed|E1|cola8/pol1/inicio0|237|62|válido|
|practice_0095|mixed|E1|cola8/pol1/inicio1|237|62|válido|
|practice_0095|mixed|E1|cola8/pol2/inicio0|235|60|válido|
|practice_0095|mixed|E1|cola8/pol2/inicio1|235|60|válido|
|practice_0095|mixed|E1|cola6/pol1/inicio0|231|42|válido|
|practice_0095|mixed|E1|cola6/pol1/inicio1|231|42|válido|
|practice_0095|mixed|E1|cola6/pol2/inicio0|231|42|válido|
|practice_0095|mixed|E1|cola6/pol2/inicio1|231|42|válido|
|practice_0095|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0095|mixed|E2|CP False/coste0/orden0|249|249|válido|
|practice_0095|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0095|mixed|E2|CP True/coste0/orden0|286|222|válido|
|practice_0095|mixed|E3|donante0/coste0/orden0|225|224|válido|
|practice_0095|mixed|E3|donante0/coste1/orden0|225|224|válido|
|practice_0095|mixed|E3|donante0/coste0/orden1|229|224|válido|
|practice_0095|mixed|E3|donante0/coste1/orden1|229|224|válido|
|practice_0095|mixed|E3|donante1/coste0/orden0|261|261|válido|
|practice_0095|mixed|E3|donante1/coste1/orden0|261|261|válido|
|practice_0095|mixed|E3|donante1/coste0/orden1|261|261|válido|
|practice_0095|mixed|E3|donante1/coste1/orden1|261|261|válido|
|practice_0095|mixed|E3|donante2/coste0/orden0|252|252|válido|
|practice_0095|mixed|E3|donante2/coste1/orden0|252|252|válido|
|practice_0095|mixed|E3|donante2/coste0/orden1|252|252|válido|
|practice_0095|mixed|E3|donante2/coste1/orden1|252|252|válido|
|practice_0095|mixed|E3|donante3/coste0/orden0|357|246|válido|
|practice_0095|mixed|E3|donante3/coste1/orden0|262|246|válido|
|practice_0095|mixed|E3|donante3/coste0/orden1|246|246|válido|
|practice_0095|mixed|E3|donante3/coste1/orden1|246|246|válido|
|practice_0095|mixed|E3|donante4/coste0/orden0|257|257|válido|
|practice_0095|mixed|E3|donante4/coste1/orden0|257|257|válido|
|practice_0095|mixed|E3|donante4/coste0/orden1|257|257|válido|
|practice_0095|mixed|E3|donante4/coste1/orden1|257|257|válido|
|practice_0095|mixed|E3|donante5/coste0/orden0|249|249|válido|
|practice_0095|mixed|E3|donante5/coste1/orden0|249|249|válido|
|practice_0095|mixed|E3|donante5/coste0/orden1|249|249|válido|
|practice_0095|mixed|E3|donante5/coste1/orden1|249|249|válido|
|practice_0095|mixed|E3|donante6/coste0/orden0|258|258|válido|
|practice_0095|mixed|E3|donante6/coste1/orden0|258|258|válido|
|practice_0095|mixed|E3|donante6/coste0/orden1|258|258|válido|
|practice_0095|mixed|E3|donante6/coste1/orden1|258|258|válido|
|practice_0095|mixed|E3|donante7/coste0/orden0|246|230|válido|
|practice_0095|mixed|E3|donante7/coste1/orden0|244|230|válido|
|practice_0095|mixed|E3|donante7/coste0/orden1|237|230|válido|
|practice_0095|mixed|E3|donante7/coste1/orden1|237|230|válido|
|practice_0095|mixed|E3|donante8/coste0/orden0|259|231|válido|
|practice_0095|mixed|E3|donante8/coste1/orden0|235|231|válido|
|practice_0095|mixed|E3|donante8/coste0/orden1|260|231|válido|
|practice_0095|mixed|E3|donante8/coste1/orden1|239|231|válido|
|practice_0096|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0096|independent|Cruce|CP True/coste0/orden0|216|198|válido|
|practice_0096|independent|Cruce|CP True/coste1/orden0|208|198|válido|
|practice_0096|independent|Cruce|CP True/coste0/orden1|201|198|válido|
|practice_0096|independent|Cruce|CP True/coste1/orden1|201|198|válido|
|practice_0096|independent|Cruce|mejor independiente/coste0/orden0|216|198|válido|
|practice_0096|independent|Cruce|mejor independiente/coste1/orden0|208|198|válido|
|practice_0096|independent|Cruce|mejor independiente/coste0/orden1|201|198|válido|
|practice_0096|independent|Cruce|mejor independiente/coste1/orden1|201|198|válido|
|practice_0096|independent|E1|cola10/pol1/inicio0|204|42|válido|
|practice_0096|independent|E1|cola10/pol1/inicio1|204|42|válido|
|practice_0096|independent|E1|cola10/pol2/inicio0|203|41|válido|
|practice_0096|independent|E1|cola10/pol2/inicio1|203|41|válido|
|practice_0096|independent|E1|cola8/pol1/inicio0|204|34|válido|
|practice_0096|independent|E1|cola8/pol1/inicio1|204|34|válido|
|practice_0096|independent|E1|cola8/pol2/inicio0|204|34|válido|
|practice_0096|independent|E1|cola8/pol2/inicio1|204|34|válido|
|practice_0096|independent|E1|cola6/pol1/inicio0|202|21|válido|
|practice_0096|independent|E1|cola6/pol1/inicio1|202|21|válido|
|practice_0096|independent|E1|cola6/pol2/inicio0|202|21|válido|
|practice_0096|independent|E1|cola6/pol2/inicio1|202|21|válido|
|practice_0096|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0096|independent|E2|CP False/coste0/orden0|230|230|válido|
|practice_0096|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0096|independent|E2|CP True/coste0/orden0|202|202|válido|
|practice_0096|independent|E3|donante0/coste0/orden0|202|202|válido|
|practice_0096|independent|E3|donante0/coste1/orden0|202|202|válido|
|practice_0096|independent|E3|donante0/coste0/orden1|202|202|válido|
|practice_0096|independent|E3|donante0/coste1/orden1|202|202|válido|
|practice_0096|independent|E3|donante1/coste0/orden0|214|214|válido|
|practice_0096|independent|E3|donante1/coste1/orden0|214|214|válido|
|practice_0096|independent|E3|donante1/coste0/orden1|214|214|válido|
|practice_0096|independent|E3|donante1/coste1/orden1|214|214|válido|
|practice_0096|independent|E3|donante2/coste0/orden0|206|206|válido|
|practice_0096|independent|E3|donante2/coste1/orden0|206|206|válido|
|practice_0096|independent|E3|donante2/coste0/orden1|206|206|válido|
|practice_0096|independent|E3|donante2/coste1/orden1|206|206|válido|
|practice_0096|independent|E3|donante3/coste0/orden0|220|220|válido|
|practice_0096|independent|E3|donante3/coste1/orden0|220|220|válido|
|practice_0096|independent|E3|donante3/coste0/orden1|220|220|válido|
|practice_0096|independent|E3|donante3/coste1/orden1|220|220|válido|
|practice_0096|independent|E3|donante4/coste0/orden0|213|213|válido|
|practice_0096|independent|E3|donante4/coste1/orden0|213|213|válido|
|practice_0096|independent|E3|donante4/coste0/orden1|213|213|válido|
|practice_0096|independent|E3|donante4/coste1/orden1|213|213|válido|
|practice_0096|independent|E3|donante5/coste0/orden0|219|219|válido|
|practice_0096|independent|E3|donante5/coste1/orden0|219|219|válido|
|practice_0096|independent|E3|donante5/coste0/orden1|219|219|válido|
|practice_0096|independent|E3|donante5/coste1/orden1|219|219|válido|
|practice_0096|independent|E3|donante6/coste0/orden0|217|199|válido|
|practice_0096|independent|E3|donante6/coste1/orden0|209|199|válido|
|practice_0096|independent|E3|donante6/coste0/orden1|202|199|válido|
|practice_0096|independent|E3|donante6/coste1/orden1|202|199|válido|
|practice_0096|independent|E3|donante7/coste0/orden0|258|202|válido|
|practice_0096|independent|E3|donante7/coste1/orden0|223|202|válido|
|practice_0096|independent|E3|donante7/coste0/orden1|213|202|válido|
|practice_0096|independent|E3|donante7/coste1/orden1|214|202|válido|
|practice_0096|independent|E3|donante8/coste0/orden0|227|190|válido|
|practice_0096|independent|E3|donante8/coste1/orden0|223|190|válido|
|practice_0096|independent|E3|donante8/coste0/orden1|261|190|válido|
|practice_0096|independent|E3|donante8/coste1/orden1|220|190|válido|
|practice_0097|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0097|multiplicative|Cruce|CP True/coste0/orden0|122|122|válido|
|practice_0097|multiplicative|Cruce|CP True/coste1/orden0|122|122|válido|
|practice_0097|multiplicative|Cruce|CP True/coste0/orden1|122|122|válido|
|practice_0097|multiplicative|Cruce|CP True/coste1/orden1|122|122|válido|
|practice_0097|multiplicative|Cruce|mejor independiente/coste0/orden0|122|122|válido|
|practice_0097|multiplicative|Cruce|mejor independiente/coste1/orden0|122|122|válido|
|practice_0097|multiplicative|Cruce|mejor independiente/coste0/orden1|122|122|válido|
|practice_0097|multiplicative|Cruce|mejor independiente/coste1/orden1|122|122|válido|
|practice_0097|multiplicative|E1|cola10/pol1/inicio0|127|41|válido|
|practice_0097|multiplicative|E1|cola10/pol1/inicio1|127|41|válido|
|practice_0097|multiplicative|E1|cola10/pol2/inicio0|127|41|válido|
|practice_0097|multiplicative|E1|cola10/pol2/inicio1|127|41|válido|
|practice_0097|multiplicative|E1|cola8/pol1/inicio0|126|30|válido|
|practice_0097|multiplicative|E1|cola8/pol1/inicio1|126|30|válido|
|practice_0097|multiplicative|E1|cola8/pol2/inicio0|126|30|válido|
|practice_0097|multiplicative|E1|cola8/pol2/inicio1|126|30|válido|
|practice_0097|multiplicative|E1|cola6/pol1/inicio0|127|23|válido|
|practice_0097|multiplicative|E1|cola6/pol1/inicio1|127|23|válido|
|practice_0097|multiplicative|E1|cola6/pol2/inicio0|127|23|válido|
|practice_0097|multiplicative|E1|cola6/pol2/inicio1|127|23|válido|
|practice_0097|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0097|multiplicative|E2|CP False/coste0/orden0|129|129|válido|
|practice_0097|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0097|multiplicative|E2|CP True/coste0/orden0|122|122|válido|
|practice_0097|multiplicative|E3|donante0/coste0/orden0|126|126|válido|
|practice_0097|multiplicative|E3|donante0/coste1/orden0|126|126|válido|
|practice_0097|multiplicative|E3|donante0/coste0/orden1|126|126|válido|
|practice_0097|multiplicative|E3|donante0/coste1/orden1|126|126|válido|
|practice_0097|multiplicative|E3|donante1/coste0/orden0|133|133|válido|
|practice_0097|multiplicative|E3|donante1/coste1/orden0|133|133|válido|
|practice_0097|multiplicative|E3|donante1/coste0/orden1|133|133|válido|
|practice_0097|multiplicative|E3|donante1/coste1/orden1|133|133|válido|
|practice_0097|multiplicative|E3|donante2/coste0/orden0|131|131|válido|
|practice_0097|multiplicative|E3|donante2/coste1/orden0|131|131|válido|
|practice_0097|multiplicative|E3|donante2/coste0/orden1|131|131|válido|
|practice_0097|multiplicative|E3|donante2/coste1/orden1|131|131|válido|
|practice_0097|multiplicative|E3|donante3/coste0/orden0|128|128|válido|
|practice_0097|multiplicative|E3|donante3/coste1/orden0|128|128|válido|
|practice_0097|multiplicative|E3|donante3/coste0/orden1|128|128|válido|
|practice_0097|multiplicative|E3|donante3/coste1/orden1|128|128|válido|
|practice_0097|multiplicative|E3|donante4/coste0/orden0|140|140|válido|
|practice_0097|multiplicative|E3|donante4/coste1/orden0|140|140|válido|
|practice_0097|multiplicative|E3|donante4/coste0/orden1|140|140|válido|
|practice_0097|multiplicative|E3|donante4/coste1/orden1|140|140|válido|
|practice_0097|multiplicative|E3|donante5/coste0/orden0|135|135|válido|
|practice_0097|multiplicative|E3|donante5/coste1/orden0|135|135|válido|
|practice_0097|multiplicative|E3|donante5/coste0/orden1|135|135|válido|
|practice_0097|multiplicative|E3|donante5/coste1/orden1|135|135|válido|
|practice_0097|multiplicative|E3|donante6/coste0/orden0|129|129|válido|
|practice_0097|multiplicative|E3|donante6/coste1/orden0|129|129|válido|
|practice_0097|multiplicative|E3|donante6/coste0/orden1|129|129|válido|
|practice_0097|multiplicative|E3|donante6/coste1/orden1|129|129|válido|
|practice_0097|multiplicative|E3|donante7/coste0/orden0|144|133|válido|
|practice_0097|multiplicative|E3|donante7/coste1/orden0|140|133|válido|
|practice_0097|multiplicative|E3|donante7/coste0/orden1|133|133|válido|
|practice_0097|multiplicative|E3|donante7/coste1/orden1|133|133|válido|
|practice_0097|multiplicative|E3|donante8/coste0/orden0|134|134|válido|
|practice_0097|multiplicative|E3|donante8/coste1/orden0|134|134|válido|
|practice_0097|multiplicative|E3|donante8/coste0/orden1|134|134|válido|
|practice_0097|multiplicative|E3|donante8/coste1/orden1|134|134|válido|
|practice_0098|tiny|Cruce|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0098|tiny|Cruce|CP True/coste0/orden0|1|1|válido|
|practice_0098|tiny|Cruce|CP True/coste1/orden0|1|1|válido|
|practice_0098|tiny|Cruce|CP True/coste0/orden1|1|1|válido|
|practice_0098|tiny|Cruce|CP True/coste1/orden1|1|1|válido|
|practice_0098|tiny|Cruce|mejor independiente/coste0/orden0|1|1|válido|
|practice_0098|tiny|Cruce|mejor independiente/coste1/orden0|1|1|válido|
|practice_0098|tiny|Cruce|mejor independiente/coste0/orden1|1|1|válido|
|practice_0098|tiny|Cruce|mejor independiente/coste1/orden1|1|1|válido|
|practice_0098|tiny|E1|cola1/pol1/inicio0|2|2|válido|
|practice_0098|tiny|E1|cola1/pol2/inicio0|2|2|válido|
|practice_0098|tiny|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0098|tiny|E2|CP False/coste0/orden0|1|1|válido|
|practice_0098|tiny|E2|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0098|tiny|E2|CP True/coste0/orden0|1|1|válido|
|practice_0098|tiny|E3|donante0/coste0/orden0|1|1|válido|
|practice_0098|tiny|E3|donante0/coste1/orden0|1|1|válido|
|practice_0098|tiny|E3|donante0/coste0/orden1|1|1|válido|
|practice_0098|tiny|E3|donante0/coste1/orden1|1|1|válido|
|practice_0098|tiny|E3|donante1/coste0/orden0|2|2|válido|
|practice_0098|tiny|E3|donante1/coste1/orden0|2|2|válido|
|practice_0098|tiny|E3|donante1/coste0/orden1|2|2|válido|
|practice_0098|tiny|E3|donante1/coste1/orden1|2|2|válido|
|practice_0099|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0099|multiplicative|Cruce|CP True/coste0/orden0|123|123|válido|
|practice_0099|multiplicative|Cruce|CP True/coste1/orden0|123|123|válido|
|practice_0099|multiplicative|Cruce|CP True/coste0/orden1|123|123|válido|
|practice_0099|multiplicative|Cruce|CP True/coste1/orden1|123|123|válido|
|practice_0099|multiplicative|Cruce|mejor independiente/coste0/orden0|123|123|válido|
|practice_0099|multiplicative|Cruce|mejor independiente/coste1/orden0|123|123|válido|
|practice_0099|multiplicative|Cruce|mejor independiente/coste0/orden1|123|123|válido|
|practice_0099|multiplicative|Cruce|mejor independiente/coste1/orden1|123|123|válido|
|practice_0099|multiplicative|E1|cola10/pol1/inicio0|127|43|válido|
|practice_0099|multiplicative|E1|cola10/pol1/inicio1|127|43|válido|
|practice_0099|multiplicative|E1|cola10/pol2/inicio0|127|43|válido|
|practice_0099|multiplicative|E1|cola10/pol2/inicio1|127|43|válido|
|practice_0099|multiplicative|E1|cola8/pol1/inicio0|130|38|válido|
|practice_0099|multiplicative|E1|cola8/pol1/inicio1|130|38|válido|
|practice_0099|multiplicative|E1|cola8/pol2/inicio0|130|38|válido|
|practice_0099|multiplicative|E1|cola8/pol2/inicio1|130|38|válido|
|practice_0099|multiplicative|E1|cola6/pol1/inicio0|129|29|válido|
|practice_0099|multiplicative|E1|cola6/pol1/inicio1|129|29|válido|
|practice_0099|multiplicative|E1|cola6/pol2/inicio0|129|29|válido|
|practice_0099|multiplicative|E1|cola6/pol2/inicio1|129|29|válido|
|practice_0099|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0099|multiplicative|E2|CP False/coste0/orden0|126|126|válido|
|practice_0099|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0099|multiplicative|E2|CP True/coste0/orden0|123|123|válido|
|practice_0099|multiplicative|E3|donante0/coste0/orden0|124|124|válido|
|practice_0099|multiplicative|E3|donante0/coste1/orden0|124|124|válido|
|practice_0099|multiplicative|E3|donante0/coste0/orden1|124|124|válido|
|practice_0099|multiplicative|E3|donante0/coste1/orden1|124|124|válido|
|practice_0099|multiplicative|E3|donante1/coste0/orden0|132|132|válido|
|practice_0099|multiplicative|E3|donante1/coste1/orden0|132|132|válido|
|practice_0099|multiplicative|E3|donante1/coste0/orden1|132|132|válido|
|practice_0099|multiplicative|E3|donante1/coste1/orden1|132|132|válido|
|practice_0099|multiplicative|E3|donante2/coste0/orden0|129|129|válido|
|practice_0099|multiplicative|E3|donante2/coste1/orden0|129|129|válido|
|practice_0099|multiplicative|E3|donante2/coste0/orden1|129|129|válido|
|practice_0099|multiplicative|E3|donante2/coste1/orden1|129|129|válido|
|practice_0099|multiplicative|E3|donante3/coste0/orden0|127|127|válido|
|practice_0099|multiplicative|E3|donante3/coste1/orden0|127|127|válido|
|practice_0099|multiplicative|E3|donante3/coste0/orden1|127|127|válido|
|practice_0099|multiplicative|E3|donante3/coste1/orden1|127|127|válido|
|practice_0099|multiplicative|E3|donante4/coste0/orden0|133|133|válido|
|practice_0099|multiplicative|E3|donante4/coste1/orden0|133|133|válido|
|practice_0099|multiplicative|E3|donante4/coste0/orden1|133|133|válido|
|practice_0099|multiplicative|E3|donante4/coste1/orden1|133|133|válido|
|practice_0099|multiplicative|E3|donante5/coste0/orden0|138|138|válido|
|practice_0099|multiplicative|E3|donante5/coste1/orden0|138|138|válido|
|practice_0099|multiplicative|E3|donante5/coste0/orden1|138|138|válido|
|practice_0099|multiplicative|E3|donante5/coste1/orden1|138|138|válido|
|practice_0099|multiplicative|E3|donante6/coste0/orden0|139|139|válido|
|practice_0099|multiplicative|E3|donante6/coste1/orden0|139|139|válido|
|practice_0099|multiplicative|E3|donante6/coste0/orden1|139|139|válido|
|practice_0099|multiplicative|E3|donante6/coste1/orden1|139|139|válido|
|practice_0099|multiplicative|E3|donante7/coste0/orden0|127|127|válido|
|practice_0099|multiplicative|E3|donante7/coste1/orden0|127|127|válido|
|practice_0099|multiplicative|E3|donante7/coste0/orden1|127|127|válido|
|practice_0099|multiplicative|E3|donante7/coste1/orden1|127|127|válido|
|practice_0099|multiplicative|E3|donante8/coste0/orden0|127|127|válido|
|practice_0099|multiplicative|E3|donante8/coste1/orden0|127|127|válido|
|practice_0099|multiplicative|E3|donante8/coste0/orden1|142|127|válido|
|practice_0099|multiplicative|E3|donante8/coste1/orden1|128|127|válido|
|practice_0099|multiplicative|E3|donante9/coste0/orden0|125|125|válido|
|practice_0099|multiplicative|E3|donante9/coste1/orden0|125|125|válido|
|practice_0099|multiplicative|E3|donante9/coste0/orden1|125|125|válido|
|practice_0099|multiplicative|E3|donante9/coste1/orden1|125|125|válido|
|practice_0100|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0100|independent|Cruce|CP True/coste0/orden0|345|296|válido|
|practice_0100|independent|Cruce|CP True/coste1/orden0|330|296|válido|
|practice_0100|independent|Cruce|CP True/coste0/orden1|296|296|válido|
|practice_0100|independent|Cruce|CP True/coste1/orden1|296|296|válido|
|practice_0100|independent|Cruce|mejor independiente/coste0/orden0|345|296|válido|
|practice_0100|independent|Cruce|mejor independiente/coste1/orden0|330|296|válido|
|practice_0100|independent|Cruce|mejor independiente/coste0/orden1|296|296|válido|
|practice_0100|independent|Cruce|mejor independiente/coste1/orden1|296|296|válido|
|practice_0100|independent|E1|cola10/pol1/inicio0|313|96|válido|
|practice_0100|independent|E1|cola10/pol1/inicio1|310|93|válido|
|practice_0100|independent|E1|cola10/pol2/inicio0|309|92|válido|
|practice_0100|independent|E1|cola10/pol2/inicio1|310|93|válido|
|practice_0100|independent|E1|cola8/pol1/inicio0|318|79|válido|
|practice_0100|independent|E1|cola8/pol1/inicio1|318|79|válido|
|practice_0100|independent|E1|cola8/pol2/inicio0|313|74|válido|
|practice_0100|independent|E1|cola8/pol2/inicio1|313|74|válido|
|practice_0100|independent|E1|cola6/pol1/inicio0|315|55|válido|
|practice_0100|independent|E1|cola6/pol1/inicio1|315|55|válido|
|practice_0100|independent|E1|cola6/pol2/inicio0|312|52|válido|
|practice_0100|independent|E1|cola6/pol2/inicio1|312|52|válido|
|practice_0100|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0100|independent|E2|CP False/coste0/orden0|594|339|válido|
|practice_0100|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0100|independent|E2|CP True/coste0/orden0|345|296|válido|
|practice_0100|independent|E3|donante0/coste0/orden0|308|296|válido|
|practice_0100|independent|E3|donante0/coste1/orden0|301|296|válido|
|practice_0100|independent|E3|donante0/coste0/orden1|296|296|válido|
|practice_0100|independent|E3|donante0/coste1/orden1|296|296|válido|
|practice_0100|independent|E3|donante1/coste0/orden0|342|342|válido|
|practice_0100|independent|E3|donante1/coste1/orden0|342|342|válido|
|practice_0100|independent|E3|donante1/coste0/orden1|342|342|válido|
|practice_0100|independent|E3|donante1/coste1/orden1|342|342|válido|
|practice_0100|independent|E3|donante2/coste0/orden0|623|318|válido|
|practice_0100|independent|E3|donante2/coste1/orden0|377|318|válido|
|practice_0100|independent|E3|donante2/coste0/orden1|395|318|válido|
|practice_0100|independent|E3|donante2/coste1/orden1|320|318|válido|
|practice_0100|independent|E3|donante3/coste0/orden0|598|312|válido|
|practice_0100|independent|E3|donante3/coste1/orden0|460|312|válido|
|practice_0100|independent|E3|donante3/coste0/orden1|312|312|válido|
|practice_0100|independent|E3|donante3/coste1/orden1|312|312|válido|
|practice_0100|independent|E3|donante4/coste0/orden0|354|354|válido|
|practice_0100|independent|E3|donante4/coste1/orden0|354|354|válido|
|practice_0100|independent|E3|donante4/coste0/orden1|354|354|válido|
|practice_0100|independent|E3|donante4/coste1/orden1|354|354|válido|
|practice_0100|independent|E3|donante5/coste0/orden0|346|346|válido|
|practice_0100|independent|E3|donante5/coste1/orden0|346|346|válido|
|practice_0100|independent|E3|donante5/coste0/orden1|346|346|válido|
|practice_0100|independent|E3|donante5/coste1/orden1|346|346|válido|
|practice_0100|independent|E3|donante6/coste0/orden0|339|339|válido|
|practice_0100|independent|E3|donante6/coste1/orden0|339|339|válido|
|practice_0100|independent|E3|donante6/coste0/orden1|339|339|válido|
|practice_0100|independent|E3|donante6/coste1/orden1|339|339|válido|
|practice_0100|independent|E3|donante7/coste0/orden0|326|308|válido|
|practice_0100|independent|E3|donante7/coste1/orden0|324|308|válido|
|practice_0100|independent|E3|donante7/coste0/orden1|326|308|válido|
|practice_0100|independent|E3|donante7/coste1/orden1|324|308|válido|
|practice_0100|independent|E3|donante8/coste0/orden0|309|297|válido|
|practice_0100|independent|E3|donante8/coste1/orden0|302|297|válido|
|practice_0100|independent|E3|donante8/coste0/orden1|297|297|válido|
|practice_0100|independent|E3|donante8/coste1/orden1|297|297|válido|
|practice_0100|independent|E3|donante9/coste0/orden0|327|284|válido|
|practice_0100|independent|E3|donante9/coste1/orden0|336|284|válido|
|practice_0100|independent|E3|donante9/coste0/orden1|375|284|válido|
|practice_0100|independent|E3|donante9/coste1/orden1|308|284|válido|
|practice_0101|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0101|shared|Cruce|CP True/coste0/orden0|148|148|válido|
|practice_0101|shared|Cruce|CP True/coste1/orden0|148|148|válido|
|practice_0101|shared|Cruce|CP True/coste0/orden1|148|148|válido|
|practice_0101|shared|Cruce|CP True/coste1/orden1|148|148|válido|
|practice_0101|shared|Cruce|mejor independiente/coste0/orden0|148|148|válido|
|practice_0101|shared|Cruce|mejor independiente/coste1/orden0|148|148|válido|
|practice_0101|shared|Cruce|mejor independiente/coste0/orden1|148|148|válido|
|practice_0101|shared|Cruce|mejor independiente/coste1/orden1|148|148|válido|
|practice_0101|shared|E1|cola10/pol1/inicio0|151|56|válido|
|practice_0101|shared|E1|cola10/pol1/inicio1|151|56|válido|
|practice_0101|shared|E1|cola10/pol2/inicio0|149|54|válido|
|practice_0101|shared|E1|cola10/pol2/inicio1|149|54|válido|
|practice_0101|shared|E1|cola8/pol1/inicio0|151|45|válido|
|practice_0101|shared|E1|cola8/pol1/inicio1|151|45|válido|
|practice_0101|shared|E1|cola8/pol2/inicio0|151|45|válido|
|practice_0101|shared|E1|cola8/pol2/inicio1|151|45|válido|
|practice_0101|shared|E1|cola6/pol1/inicio0|151|34|válido|
|practice_0101|shared|E1|cola6/pol1/inicio1|151|34|válido|
|practice_0101|shared|E1|cola6/pol2/inicio0|151|34|válido|
|practice_0101|shared|E1|cola6/pol2/inicio1|151|34|válido|
|practice_0101|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0101|shared|E2|CP False/coste0/orden0|165|158|válido|
|practice_0101|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0101|shared|E2|CP True/coste0/orden0|148|148|válido|
|practice_0101|shared|E3|donante0/coste0/orden0|149|148|válido|
|practice_0101|shared|E3|donante0/coste1/orden0|149|148|válido|
|practice_0101|shared|E3|donante0/coste0/orden1|148|148|válido|
|practice_0101|shared|E3|donante0/coste1/orden1|148|148|válido|
|practice_0101|shared|E3|donante1/coste0/orden0|168|168|válido|
|practice_0101|shared|E3|donante1/coste1/orden0|168|168|válido|
|practice_0101|shared|E3|donante1/coste0/orden1|168|168|válido|
|practice_0101|shared|E3|donante1/coste1/orden1|168|168|válido|
|practice_0101|shared|E3|donante2/coste0/orden0|161|161|válido|
|practice_0101|shared|E3|donante2/coste1/orden0|161|161|válido|
|practice_0101|shared|E3|donante2/coste0/orden1|161|161|válido|
|practice_0101|shared|E3|donante2/coste1/orden1|161|161|válido|
|practice_0101|shared|E3|donante3/coste0/orden0|155|155|válido|
|practice_0101|shared|E3|donante3/coste1/orden0|155|155|válido|
|practice_0101|shared|E3|donante3/coste0/orden1|155|155|válido|
|practice_0101|shared|E3|donante3/coste1/orden1|155|155|válido|
|practice_0101|shared|E3|donante4/coste0/orden0|175|175|válido|
|practice_0101|shared|E3|donante4/coste1/orden0|175|175|válido|
|practice_0101|shared|E3|donante4/coste0/orden1|175|175|válido|
|practice_0101|shared|E3|donante4/coste1/orden1|175|175|válido|
|practice_0101|shared|E3|donante5/coste0/orden0|163|163|válido|
|practice_0101|shared|E3|donante5/coste1/orden0|163|163|válido|
|practice_0101|shared|E3|donante5/coste0/orden1|163|163|válido|
|practice_0101|shared|E3|donante5/coste1/orden1|163|163|válido|
|practice_0101|shared|E3|donante6/coste0/orden0|166|166|válido|
|practice_0101|shared|E3|donante6/coste1/orden0|166|166|válido|
|practice_0101|shared|E3|donante6/coste0/orden1|166|166|válido|
|practice_0101|shared|E3|donante6/coste1/orden1|166|166|válido|
|practice_0101|shared|E3|donante7/coste0/orden0|156|153|válido|
|practice_0101|shared|E3|donante7/coste1/orden0|156|153|válido|
|practice_0101|shared|E3|donante7/coste0/orden1|153|153|válido|
|practice_0101|shared|E3|donante7/coste1/orden1|153|153|válido|
|practice_0101|shared|E3|donante8/coste0/orden0|167|149|válido|
|practice_0101|shared|E3|donante8/coste1/orden0|156|149|válido|
|practice_0101|shared|E3|donante8/coste0/orden1|149|149|válido|
|practice_0101|shared|E3|donante8/coste1/orden1|149|149|válido|
|practice_0102|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0102|independent|Cruce|CP True/coste0/orden0|370|268|válido|
|practice_0102|independent|Cruce|CP True/coste1/orden0|350|268|válido|
|practice_0102|independent|Cruce|CP True/coste0/orden1|274|268|válido|
|practice_0102|independent|Cruce|CP True/coste1/orden1|270|268|válido|
|practice_0102|independent|Cruce|mejor independiente/coste0/orden0|372|268|válido|
|practice_0102|independent|Cruce|mejor independiente/coste1/orden0|388|268|válido|
|practice_0102|independent|Cruce|mejor independiente/coste0/orden1|274|268|válido|
|practice_0102|independent|Cruce|mejor independiente/coste1/orden1|270|268|válido|
|practice_0102|independent|E1|cola10/pol1/inicio0|285|95|válido|
|practice_0102|independent|E1|cola10/pol1/inicio1|285|95|válido|
|practice_0102|independent|E1|cola10/pol2/inicio0|284|94|válido|
|practice_0102|independent|E1|cola10/pol2/inicio1|284|94|válido|
|practice_0102|independent|E1|cola8/pol1/inicio0|282|76|válido|
|practice_0102|independent|E1|cola8/pol1/inicio1|282|76|válido|
|practice_0102|independent|E1|cola8/pol2/inicio0|282|76|válido|
|practice_0102|independent|E1|cola8/pol2/inicio1|281|75|válido|
|practice_0102|independent|E1|cola6/pol1/inicio0|287|64|válido|
|practice_0102|independent|E1|cola6/pol1/inicio1|287|64|válido|
|practice_0102|independent|E1|cola6/pol2/inicio0|287|64|válido|
|practice_0102|independent|E1|cola6/pol2/inicio1|287|64|válido|
|practice_0102|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0102|independent|E2|CP False/coste0/orden0|371|267|válido|
|practice_0102|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0102|independent|E2|CP True/coste0/orden0|370|268|válido|
|practice_0102|independent|E3|donante0/coste0/orden0|286|268|válido|
|practice_0102|independent|E3|donante0/coste1/orden0|276|268|válido|
|practice_0102|independent|E3|donante0/coste0/orden1|274|268|válido|
|practice_0102|independent|E3|donante0/coste1/orden1|270|268|válido|
|practice_0102|independent|E3|donante1/coste0/orden0|307|307|válido|
|practice_0102|independent|E3|donante1/coste1/orden0|307|307|válido|
|practice_0102|independent|E3|donante1/coste0/orden1|307|307|válido|
|practice_0102|independent|E3|donante1/coste1/orden1|307|307|válido|
|practice_0102|independent|E3|donante2/coste0/orden0|340|288|válido|
|practice_0102|independent|E3|donante2/coste1/orden0|307|288|válido|
|practice_0102|independent|E3|donante2/coste0/orden1|288|288|válido|
|practice_0102|independent|E3|donante2/coste1/orden1|288|288|válido|
|practice_0102|independent|E3|donante3/coste0/orden0|418|285|válido|
|practice_0102|independent|E3|donante3/coste1/orden0|367|285|válido|
|practice_0102|independent|E3|donante3/coste0/orden1|285|285|válido|
|practice_0102|independent|E3|donante3/coste1/orden1|285|285|válido|
|practice_0102|independent|E3|donante4/coste0/orden0|309|309|válido|
|practice_0102|independent|E3|donante4/coste1/orden0|309|309|válido|
|practice_0102|independent|E3|donante4/coste0/orden1|309|309|válido|
|practice_0102|independent|E3|donante4/coste1/orden1|309|309|válido|
|practice_0102|independent|E3|donante5/coste0/orden0|310|310|válido|
|practice_0102|independent|E3|donante5/coste1/orden0|310|310|válido|
|practice_0102|independent|E3|donante5/coste0/orden1|310|310|válido|
|practice_0102|independent|E3|donante5/coste1/orden1|310|310|válido|
|practice_0102|independent|E3|donante6/coste0/orden0|311|311|válido|
|practice_0102|independent|E3|donante6/coste1/orden0|311|311|válido|
|practice_0102|independent|E3|donante6/coste0/orden1|311|311|válido|
|practice_0102|independent|E3|donante6/coste1/orden1|311|311|válido|
|practice_0102|independent|E3|donante7/coste0/orden0|295|275|válido|
|practice_0102|independent|E3|donante7/coste1/orden0|298|275|válido|
|practice_0102|independent|E3|donante7/coste0/orden1|300|275|válido|
|practice_0102|independent|E3|donante7/coste1/orden1|291|275|válido|
|practice_0102|independent|E3|donante8/coste0/orden0|286|268|válido|
|practice_0102|independent|E3|donante8/coste1/orden0|276|268|válido|
|practice_0102|independent|E3|donante8/coste0/orden1|274|268|válido|
|practice_0102|independent|E3|donante8/coste1/orden1|270|268|válido|
|practice_0102|independent|E3|donante9/coste0/orden0|357|263|válido|
|practice_0102|independent|E3|donante9/coste1/orden0|279|263|válido|
|practice_0102|independent|E3|donante9/coste0/orden1|330|263|válido|
|practice_0102|independent|E3|donante9/coste1/orden1|271|263|válido|
|practice_0103|tiny|Cruce|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0103|tiny|Cruce|CP True/coste0/orden0|5|5|válido|
|practice_0103|tiny|Cruce|CP True/coste1/orden0|5|5|válido|
|practice_0103|tiny|Cruce|CP True/coste0/orden1|5|5|válido|
|practice_0103|tiny|Cruce|CP True/coste1/orden1|5|5|válido|
|practice_0103|tiny|Cruce|mejor independiente/coste0/orden0|5|5|válido|
|practice_0103|tiny|Cruce|mejor independiente/coste1/orden0|5|5|válido|
|practice_0103|tiny|Cruce|mejor independiente/coste0/orden1|5|5|válido|
|practice_0103|tiny|Cruce|mejor independiente/coste1/orden1|5|5|válido|
|practice_0103|tiny|E1|cola2/pol1/inicio0|6|3|válido|
|practice_0103|tiny|E1|cola2/pol1/inicio1|6|3|válido|
|practice_0103|tiny|E1|cola2/pol2/inicio0|6|3|válido|
|practice_0103|tiny|E1|cola2/pol2/inicio1|6|3|válido|
|practice_0103|tiny|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0103|tiny|E2|CP False/coste0/orden0|5|5|válido|
|practice_0103|tiny|E2|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0103|tiny|E2|CP True/coste0/orden0|5|5|válido|
|practice_0103|tiny|E3|donante0/coste0/orden0|5|5|válido|
|practice_0103|tiny|E3|donante0/coste1/orden0|5|5|válido|
|practice_0103|tiny|E3|donante0/coste0/orden1|5|5|válido|
|practice_0103|tiny|E3|donante0/coste1/orden1|5|5|válido|
|practice_0103|tiny|E3|donante1/coste0/orden0|6|6|válido|
|practice_0103|tiny|E3|donante1/coste1/orden0|6|6|válido|
|practice_0103|tiny|E3|donante1/coste0/orden1|6|6|válido|
|practice_0103|tiny|E3|donante1/coste1/orden1|6|6|válido|
|practice_0104|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0104|independent|Cruce|CP True/coste0/orden0|257|202|válido|
|practice_0104|independent|Cruce|CP True/coste1/orden0|248|202|válido|
|practice_0104|independent|Cruce|CP True/coste0/orden1|293|202|válido|
|practice_0104|independent|Cruce|CP True/coste1/orden1|226|202|válido|
|practice_0104|independent|Cruce|mejor independiente/coste0/orden0|255|203|válido|
|practice_0104|independent|Cruce|mejor independiente/coste1/orden0|248|203|válido|
|practice_0104|independent|Cruce|mejor independiente/coste0/orden1|293|203|válido|
|practice_0104|independent|Cruce|mejor independiente/coste1/orden1|226|203|válido|
|practice_0104|independent|E1|cola10/pol1/inicio0|218|66|válido|
|practice_0104|independent|E1|cola10/pol1/inicio1|218|66|válido|
|practice_0104|independent|E1|cola10/pol2/inicio0|218|66|válido|
|practice_0104|independent|E1|cola10/pol2/inicio1|218|66|válido|
|practice_0104|independent|E1|cola8/pol1/inicio0|221|52|válido|
|practice_0104|independent|E1|cola8/pol1/inicio1|221|52|válido|
|practice_0104|independent|E1|cola8/pol2/inicio0|221|52|válido|
|practice_0104|independent|E1|cola8/pol2/inicio1|221|52|válido|
|practice_0104|independent|E1|cola6/pol1/inicio0|220|41|válido|
|practice_0104|independent|E1|cola6/pol1/inicio1|220|41|válido|
|practice_0104|independent|E1|cola6/pol2/inicio0|222|43|válido|
|practice_0104|independent|E1|cola6/pol2/inicio1|222|43|válido|
|practice_0104|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0104|independent|E2|CP False/coste0/orden0|230|230|válido|
|practice_0104|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0104|independent|E2|CP True/coste0/orden0|257|202|válido|
|practice_0104|independent|E3|donante0/coste0/orden0|205|203|válido|
|practice_0104|independent|E3|donante0/coste1/orden0|205|203|válido|
|practice_0104|independent|E3|donante0/coste0/orden1|293|203|válido|
|practice_0104|independent|E3|donante0/coste1/orden1|226|203|válido|
|practice_0104|independent|E3|donante1/coste0/orden0|230|230|válido|
|practice_0104|independent|E3|donante1/coste1/orden0|230|230|válido|
|practice_0104|independent|E3|donante1/coste0/orden1|230|230|válido|
|practice_0104|independent|E3|donante1/coste1/orden1|230|230|válido|
|practice_0104|independent|E3|donante2/coste0/orden0|219|219|válido|
|practice_0104|independent|E3|donante2/coste1/orden0|219|219|válido|
|practice_0104|independent|E3|donante2/coste0/orden1|219|219|válido|
|practice_0104|independent|E3|donante2/coste1/orden1|219|219|válido|
|practice_0104|independent|E3|donante3/coste0/orden0|238|219|válido|
|practice_0104|independent|E3|donante3/coste1/orden0|229|219|válido|
|practice_0104|independent|E3|donante3/coste0/orden1|219|219|válido|
|practice_0104|independent|E3|donante3/coste1/orden1|219|219|válido|
|practice_0104|independent|E3|donante4/coste0/orden0|235|235|válido|
|practice_0104|independent|E3|donante4/coste1/orden0|235|235|válido|
|practice_0104|independent|E3|donante4/coste0/orden1|235|235|válido|
|practice_0104|independent|E3|donante4/coste1/orden1|235|235|válido|
|practice_0104|independent|E3|donante5/coste0/orden0|230|230|válido|
|practice_0104|independent|E3|donante5/coste1/orden0|230|230|válido|
|practice_0104|independent|E3|donante5/coste0/orden1|230|230|válido|
|practice_0104|independent|E3|donante5/coste1/orden1|230|230|válido|
|practice_0104|independent|E3|donante6/coste0/orden0|231|231|válido|
|practice_0104|independent|E3|donante6/coste1/orden0|231|231|válido|
|practice_0104|independent|E3|donante6/coste0/orden1|231|231|válido|
|practice_0104|independent|E3|donante6/coste1/orden1|231|231|válido|
|practice_0104|independent|E3|donante7/coste0/orden0|234|209|válido|
|practice_0104|independent|E3|donante7/coste1/orden0|220|209|válido|
|practice_0104|independent|E3|donante7/coste0/orden1|291|209|válido|
|practice_0104|independent|E3|donante7/coste1/orden1|292|209|válido|
|practice_0104|independent|E3|donante8/coste0/orden0|216|202|válido|
|practice_0104|independent|E3|donante8/coste1/orden0|213|202|válido|
|practice_0104|independent|E3|donante8/coste0/orden1|286|202|válido|
|practice_0104|independent|E3|donante8/coste1/orden1|223|202|válido|
|practice_0104|independent|E3|donante9/coste0/orden0|219|201|válido|
|practice_0104|independent|E3|donante9/coste1/orden0|212|201|válido|
|practice_0104|independent|E3|donante9/coste0/orden1|245|201|válido|
|practice_0104|independent|E3|donante9/coste1/orden1|209|201|válido|
|practice_0105|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0105|mixed|Cruce|CP True/coste0/orden0|170|170|válido|
|practice_0105|mixed|Cruce|CP True/coste1/orden0|170|170|válido|
|practice_0105|mixed|Cruce|CP True/coste0/orden1|170|170|válido|
|practice_0105|mixed|Cruce|CP True/coste1/orden1|170|170|válido|
|practice_0105|mixed|Cruce|mejor independiente/coste0/orden0|170|170|válido|
|practice_0105|mixed|Cruce|mejor independiente/coste1/orden0|170|170|válido|
|practice_0105|mixed|Cruce|mejor independiente/coste0/orden1|170|170|válido|
|practice_0105|mixed|Cruce|mejor independiente/coste1/orden1|170|170|válido|
|practice_0105|mixed|E1|cola10/pol1/inicio0|—|—|registros ocupados|
|practice_0105|mixed|E1|cola10/pol1/inicio1|—|—|registros ocupados|
|practice_0105|mixed|E1|cola10/pol2/inicio0|—|—|registros ocupados|
|practice_0105|mixed|E1|cola10/pol2/inicio1|—|—|registros ocupados|
|practice_0105|mixed|E1|cola8/pol1/inicio0|—|—|registros ocupados|
|practice_0105|mixed|E1|cola8/pol1/inicio1|—|—|registros ocupados|
|practice_0105|mixed|E1|cola8/pol2/inicio0|—|—|registros ocupados|
|practice_0105|mixed|E1|cola8/pol2/inicio1|—|—|registros ocupados|
|practice_0105|mixed|E1|cola6/pol1/inicio0|—|—|registros ocupados|
|practice_0105|mixed|E1|cola6/pol1/inicio1|—|—|registros ocupados|
|practice_0105|mixed|E1|cola6/pol2/inicio0|—|—|registros ocupados|
|practice_0105|mixed|E1|cola6/pol2/inicio1|—|—|registros ocupados|
|practice_0105|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0105|mixed|E2|CP False/coste0/orden0|194|194|válido|
|practice_0105|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0105|mixed|E2|CP True/coste0/orden0|170|170|válido|
|practice_0105|mixed|E3|donante0/coste0/orden0|171|171|válido|
|practice_0105|mixed|E3|donante0/coste1/orden0|171|171|válido|
|practice_0105|mixed|E3|donante0/coste0/orden1|171|171|válido|
|practice_0105|mixed|E3|donante0/coste1/orden1|171|171|válido|
|practice_0105|mixed|E3|donante1/coste0/orden0|195|195|válido|
|practice_0105|mixed|E3|donante1/coste1/orden0|195|195|válido|
|practice_0105|mixed|E3|donante1/coste0/orden1|195|195|válido|
|practice_0105|mixed|E3|donante1/coste1/orden1|195|195|válido|
|practice_0105|mixed|E3|donante2/coste0/orden0|188|188|válido|
|practice_0105|mixed|E3|donante2/coste1/orden0|188|188|válido|
|practice_0105|mixed|E3|donante2/coste0/orden1|188|188|válido|
|practice_0105|mixed|E3|donante2/coste1/orden1|188|188|válido|
|practice_0105|mixed|E3|donante3/coste0/orden0|186|186|válido|
|practice_0105|mixed|E3|donante3/coste1/orden0|186|186|válido|
|practice_0105|mixed|E3|donante3/coste0/orden1|186|186|válido|
|practice_0105|mixed|E3|donante3/coste1/orden1|186|186|válido|
|practice_0105|mixed|E3|donante4/coste0/orden0|195|195|válido|
|practice_0105|mixed|E3|donante4/coste1/orden0|195|195|válido|
|practice_0105|mixed|E3|donante4/coste0/orden1|195|195|válido|
|practice_0105|mixed|E3|donante4/coste1/orden1|195|195|válido|
|practice_0105|mixed|E3|donante5/coste0/orden0|193|193|válido|
|practice_0105|mixed|E3|donante5/coste1/orden0|193|193|válido|
|practice_0105|mixed|E3|donante5/coste0/orden1|193|193|válido|
|practice_0105|mixed|E3|donante5/coste1/orden1|193|193|válido|
|practice_0105|mixed|E3|donante6/coste0/orden0|199|199|válido|
|practice_0105|mixed|E3|donante6/coste1/orden0|199|199|válido|
|practice_0105|mixed|E3|donante6/coste0/orden1|199|199|válido|
|practice_0105|mixed|E3|donante6/coste1/orden1|199|199|válido|
|practice_0105|mixed|E3|donante7/coste0/orden0|236|178|válido|
|practice_0105|mixed|E3|donante7/coste1/orden0|189|178|válido|
|practice_0105|mixed|E3|donante7/coste0/orden1|186|178|válido|
|practice_0105|mixed|E3|donante7/coste1/orden1|186|178|válido|
|practice_0105|mixed|E3|donante8/coste0/orden0|172|172|válido|
|practice_0105|mixed|E3|donante8/coste1/orden0|172|172|válido|
|practice_0105|mixed|E3|donante8/coste0/orden1|172|172|válido|
|practice_0105|mixed|E3|donante8/coste1/orden1|172|172|válido|
|practice_0105|mixed|E3|donante9/coste0/orden0|202|176|válido|
|practice_0105|mixed|E3|donante9/coste1/orden0|206|176|válido|
|practice_0105|mixed|E3|donante9/coste0/orden1|202|176|válido|
|practice_0105|mixed|E3|donante9/coste1/orden1|200|176|válido|
|practice_0106|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0106|shared|Cruce|CP True/coste0/orden0|173|159|válido|
|practice_0106|shared|Cruce|CP True/coste1/orden0|173|159|válido|
|practice_0106|shared|Cruce|CP True/coste0/orden1|159|159|válido|
|practice_0106|shared|Cruce|CP True/coste1/orden1|159|159|válido|
|practice_0106|shared|Cruce|mejor independiente/coste0/orden0|167|161|válido|
|practice_0106|shared|Cruce|mejor independiente/coste1/orden0|163|161|válido|
|practice_0106|shared|Cruce|mejor independiente/coste0/orden1|161|161|válido|
|practice_0106|shared|Cruce|mejor independiente/coste1/orden1|161|161|válido|
|practice_0106|shared|E1|cola10/pol1/inicio0|168|53|válido|
|practice_0106|shared|E1|cola10/pol1/inicio1|168|53|válido|
|practice_0106|shared|E1|cola10/pol2/inicio0|168|53|válido|
|practice_0106|shared|E1|cola10/pol2/inicio1|168|53|válido|
|practice_0106|shared|E1|cola8/pol1/inicio0|166|43|válido|
|practice_0106|shared|E1|cola8/pol1/inicio1|166|43|válido|
|practice_0106|shared|E1|cola8/pol2/inicio0|166|43|válido|
|practice_0106|shared|E1|cola8/pol2/inicio1|166|43|válido|
|practice_0106|shared|E1|cola6/pol1/inicio0|166|33|válido|
|practice_0106|shared|E1|cola6/pol1/inicio1|166|33|válido|
|practice_0106|shared|E1|cola6/pol2/inicio0|166|33|válido|
|practice_0106|shared|E1|cola6/pol2/inicio1|166|33|válido|
|practice_0106|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0106|shared|E2|CP False/coste0/orden0|161|161|válido|
|practice_0106|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0106|shared|E2|CP True/coste0/orden0|173|159|válido|
|practice_0106|shared|E3|donante0/coste0/orden0|161|161|válido|
|practice_0106|shared|E3|donante0/coste1/orden0|161|161|válido|
|practice_0106|shared|E3|donante0/coste0/orden1|161|161|válido|
|practice_0106|shared|E3|donante0/coste1/orden1|161|161|válido|
|practice_0106|shared|E3|donante1/coste0/orden0|180|180|válido|
|practice_0106|shared|E3|donante1/coste1/orden0|180|180|válido|
|practice_0106|shared|E3|donante1/coste0/orden1|180|180|válido|
|practice_0106|shared|E3|donante1/coste1/orden1|180|180|válido|
|practice_0106|shared|E3|donante2/coste0/orden0|176|176|válido|
|practice_0106|shared|E3|donante2/coste1/orden0|176|176|válido|
|practice_0106|shared|E3|donante2/coste0/orden1|176|176|válido|
|practice_0106|shared|E3|donante2/coste1/orden1|176|176|válido|
|practice_0106|shared|E3|donante3/coste0/orden0|167|167|válido|
|practice_0106|shared|E3|donante3/coste1/orden0|167|167|válido|
|practice_0106|shared|E3|donante3/coste0/orden1|167|167|válido|
|practice_0106|shared|E3|donante3/coste1/orden1|167|167|válido|
|practice_0106|shared|E3|donante4/coste0/orden0|195|195|válido|
|practice_0106|shared|E3|donante4/coste1/orden0|195|195|válido|
|practice_0106|shared|E3|donante4/coste0/orden1|195|195|válido|
|practice_0106|shared|E3|donante4/coste1/orden1|195|195|válido|
|practice_0106|shared|E3|donante5/coste0/orden0|181|181|válido|
|practice_0106|shared|E3|donante5/coste1/orden0|181|181|válido|
|practice_0106|shared|E3|donante5/coste0/orden1|181|181|válido|
|practice_0106|shared|E3|donante5/coste1/orden1|181|181|válido|
|practice_0106|shared|E3|donante6/coste0/orden0|192|192|válido|
|practice_0106|shared|E3|donante6/coste1/orden0|192|192|válido|
|practice_0106|shared|E3|donante6/coste0/orden1|192|192|válido|
|practice_0106|shared|E3|donante6/coste1/orden1|192|192|válido|
|practice_0106|shared|E3|donante7/coste0/orden0|210|174|válido|
|practice_0106|shared|E3|donante7/coste1/orden0|197|174|válido|
|practice_0106|shared|E3|donante7/coste0/orden1|184|174|válido|
|practice_0106|shared|E3|donante7/coste1/orden1|196|174|válido|
|practice_0106|shared|E3|donante8/coste0/orden0|166|166|válido|
|practice_0106|shared|E3|donante8/coste1/orden0|166|166|válido|
|practice_0106|shared|E3|donante8/coste0/orden1|166|166|válido|
|practice_0106|shared|E3|donante8/coste1/orden1|166|166|válido|
|practice_0107|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0107|shared|Cruce|CP True/coste0/orden0|108|108|válido|
|practice_0107|shared|Cruce|CP True/coste1/orden0|108|108|válido|
|practice_0107|shared|Cruce|CP True/coste0/orden1|108|108|válido|
|practice_0107|shared|Cruce|CP True/coste1/orden1|108|108|válido|
|practice_0107|shared|Cruce|mejor independiente/coste0/orden0|108|108|válido|
|practice_0107|shared|Cruce|mejor independiente/coste1/orden0|108|108|válido|
|practice_0107|shared|Cruce|mejor independiente/coste0/orden1|108|108|válido|
|practice_0107|shared|Cruce|mejor independiente/coste1/orden1|108|108|válido|
|practice_0107|shared|E1|cola10/pol1/inicio0|115|39|válido|
|practice_0107|shared|E1|cola10/pol1/inicio1|115|39|válido|
|practice_0107|shared|E1|cola10/pol2/inicio0|114|38|válido|
|practice_0107|shared|E1|cola10/pol2/inicio1|114|38|válido|
|practice_0107|shared|E1|cola8/pol1/inicio0|115|32|válido|
|practice_0107|shared|E1|cola8/pol1/inicio1|115|32|válido|
|practice_0107|shared|E1|cola8/pol2/inicio0|111|28|válido|
|practice_0107|shared|E1|cola8/pol2/inicio1|112|29|válido|
|practice_0107|shared|E1|cola6/pol1/inicio0|111|20|válido|
|practice_0107|shared|E1|cola6/pol1/inicio1|111|20|válido|
|practice_0107|shared|E1|cola6/pol2/inicio0|111|20|válido|
|practice_0107|shared|E1|cola6/pol2/inicio1|111|20|válido|
|practice_0107|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0107|shared|E2|CP False/coste0/orden0|115|115|válido|
|practice_0107|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0107|shared|E2|CP True/coste0/orden0|108|108|válido|
|practice_0107|shared|E3|donante0/coste0/orden0|111|111|válido|
|practice_0107|shared|E3|donante0/coste1/orden0|111|111|válido|
|practice_0107|shared|E3|donante0/coste0/orden1|111|111|válido|
|practice_0107|shared|E3|donante0/coste1/orden1|111|111|válido|
|practice_0107|shared|E3|donante1/coste0/orden0|125|125|válido|
|practice_0107|shared|E3|donante1/coste1/orden0|125|125|válido|
|practice_0107|shared|E3|donante1/coste0/orden1|125|125|válido|
|practice_0107|shared|E3|donante1/coste1/orden1|125|125|válido|
|practice_0107|shared|E3|donante2/coste0/orden0|120|120|válido|
|practice_0107|shared|E3|donante2/coste1/orden0|120|120|válido|
|practice_0107|shared|E3|donante2/coste0/orden1|120|120|válido|
|practice_0107|shared|E3|donante2/coste1/orden1|120|120|válido|
|practice_0107|shared|E3|donante3/coste0/orden0|119|119|válido|
|practice_0107|shared|E3|donante3/coste1/orden0|119|119|válido|
|practice_0107|shared|E3|donante3/coste0/orden1|119|119|válido|
|practice_0107|shared|E3|donante3/coste1/orden1|119|119|válido|
|practice_0107|shared|E3|donante4/coste0/orden0|135|135|válido|
|practice_0107|shared|E3|donante4/coste1/orden0|135|135|válido|
|practice_0107|shared|E3|donante4/coste0/orden1|135|135|válido|
|practice_0107|shared|E3|donante4/coste1/orden1|135|135|válido|
|practice_0107|shared|E3|donante5/coste0/orden0|126|126|válido|
|practice_0107|shared|E3|donante5/coste1/orden0|126|126|válido|
|practice_0107|shared|E3|donante5/coste0/orden1|126|126|válido|
|practice_0107|shared|E3|donante5/coste1/orden1|126|126|válido|
|practice_0107|shared|E3|donante6/coste0/orden0|123|123|válido|
|practice_0107|shared|E3|donante6/coste1/orden0|123|123|válido|
|practice_0107|shared|E3|donante6/coste0/orden1|123|123|válido|
|practice_0107|shared|E3|donante6/coste1/orden1|123|123|válido|
|practice_0107|shared|E3|donante7/coste0/orden0|124|124|válido|
|practice_0107|shared|E3|donante7/coste1/orden0|124|124|válido|
|practice_0107|shared|E3|donante7/coste0/orden1|124|124|válido|
|practice_0107|shared|E3|donante7/coste1/orden1|124|124|válido|
|practice_0107|shared|E3|donante8/coste0/orden0|125|125|válido|
|practice_0107|shared|E3|donante8/coste1/orden0|125|125|válido|
|practice_0107|shared|E3|donante8/coste0/orden1|125|125|válido|
|practice_0107|shared|E3|donante8/coste1/orden1|125|125|válido|
|practice_0108|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0108|multiplicative|Cruce|CP True/coste0/orden0|105|105|válido|
|practice_0108|multiplicative|Cruce|CP True/coste1/orden0|105|105|válido|
|practice_0108|multiplicative|Cruce|CP True/coste0/orden1|105|105|válido|
|practice_0108|multiplicative|Cruce|CP True/coste1/orden1|105|105|válido|
|practice_0108|multiplicative|Cruce|mejor independiente/coste0/orden0|105|105|válido|
|practice_0108|multiplicative|Cruce|mejor independiente/coste1/orden0|105|105|válido|
|practice_0108|multiplicative|Cruce|mejor independiente/coste0/orden1|105|105|válido|
|practice_0108|multiplicative|Cruce|mejor independiente/coste1/orden1|105|105|válido|
|practice_0108|multiplicative|E1|cola10/pol1/inicio0|110|32|válido|
|practice_0108|multiplicative|E1|cola10/pol1/inicio1|110|32|válido|
|practice_0108|multiplicative|E1|cola10/pol2/inicio0|110|32|válido|
|practice_0108|multiplicative|E1|cola10/pol2/inicio1|110|32|válido|
|practice_0108|multiplicative|E1|cola8/pol1/inicio0|109|25|válido|
|practice_0108|multiplicative|E1|cola8/pol1/inicio1|109|25|válido|
|practice_0108|multiplicative|E1|cola8/pol2/inicio0|109|25|válido|
|practice_0108|multiplicative|E1|cola8/pol2/inicio1|109|25|válido|
|practice_0108|multiplicative|E1|cola6/pol1/inicio0|110|20|válido|
|practice_0108|multiplicative|E1|cola6/pol1/inicio1|110|20|válido|
|practice_0108|multiplicative|E1|cola6/pol2/inicio0|110|20|válido|
|practice_0108|multiplicative|E1|cola6/pol2/inicio1|110|20|válido|
|practice_0108|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0108|multiplicative|E2|CP False/coste0/orden0|105|105|válido|
|practice_0108|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0108|multiplicative|E2|CP True/coste0/orden0|107|107|válido|
|practice_0108|multiplicative|E3|donante0/coste0/orden0|107|107|válido|
|practice_0108|multiplicative|E3|donante0/coste1/orden0|107|107|válido|
|practice_0108|multiplicative|E3|donante0/coste0/orden1|107|107|válido|
|practice_0108|multiplicative|E3|donante0/coste1/orden1|107|107|válido|
|practice_0108|multiplicative|E3|donante1/coste0/orden0|119|119|válido|
|practice_0108|multiplicative|E3|donante1/coste1/orden0|119|119|válido|
|practice_0108|multiplicative|E3|donante1/coste0/orden1|119|119|válido|
|practice_0108|multiplicative|E3|donante1/coste1/orden1|119|119|válido|
|practice_0108|multiplicative|E3|donante2/coste0/orden0|112|112|válido|
|practice_0108|multiplicative|E3|donante2/coste1/orden0|112|112|válido|
|practice_0108|multiplicative|E3|donante2/coste0/orden1|112|112|válido|
|practice_0108|multiplicative|E3|donante2/coste1/orden1|112|112|válido|
|practice_0108|multiplicative|E3|donante3/coste0/orden0|113|113|válido|
|practice_0108|multiplicative|E3|donante3/coste1/orden0|113|113|válido|
|practice_0108|multiplicative|E3|donante3/coste0/orden1|113|113|válido|
|practice_0108|multiplicative|E3|donante3/coste1/orden1|113|113|válido|
|practice_0108|multiplicative|E3|donante4/coste0/orden0|124|124|válido|
|practice_0108|multiplicative|E3|donante4/coste1/orden0|124|124|válido|
|practice_0108|multiplicative|E3|donante4/coste0/orden1|124|124|válido|
|practice_0108|multiplicative|E3|donante4/coste1/orden1|124|124|válido|
|practice_0108|multiplicative|E3|donante5/coste0/orden0|111|111|válido|
|practice_0108|multiplicative|E3|donante5/coste1/orden0|111|111|válido|
|practice_0108|multiplicative|E3|donante5/coste0/orden1|111|111|válido|
|practice_0108|multiplicative|E3|donante5/coste1/orden1|111|111|válido|
|practice_0108|multiplicative|E3|donante6/coste0/orden0|118|118|válido|
|practice_0108|multiplicative|E3|donante6/coste1/orden0|118|118|válido|
|practice_0108|multiplicative|E3|donante6/coste0/orden1|118|118|válido|
|practice_0108|multiplicative|E3|donante6/coste1/orden1|118|118|válido|
|practice_0108|multiplicative|E3|donante7/coste0/orden0|114|114|válido|
|practice_0108|multiplicative|E3|donante7/coste1/orden0|114|114|válido|
|practice_0108|multiplicative|E3|donante7/coste0/orden1|114|114|válido|
|practice_0108|multiplicative|E3|donante7/coste1/orden1|114|114|válido|
|practice_0108|multiplicative|E3|donante8/coste0/orden0|109|109|válido|
|practice_0108|multiplicative|E3|donante8/coste1/orden0|109|109|válido|
|practice_0108|multiplicative|E3|donante8/coste0/orden1|109|109|válido|
|practice_0108|multiplicative|E3|donante8/coste1/orden1|109|109|válido|
|practice_0108|multiplicative|E3|donante9/coste0/orden0|110|110|válido|
|practice_0108|multiplicative|E3|donante9/coste1/orden0|110|110|válido|
|practice_0108|multiplicative|E3|donante9/coste0/orden1|110|110|válido|
|practice_0108|multiplicative|E3|donante9/coste1/orden1|110|110|válido|
|practice_0109|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0109|multiplicative|Cruce|CP True/coste0/orden0|130|130|válido|
|practice_0109|multiplicative|Cruce|CP True/coste1/orden0|130|130|válido|
|practice_0109|multiplicative|Cruce|CP True/coste0/orden1|130|130|válido|
|practice_0109|multiplicative|Cruce|CP True/coste1/orden1|130|130|válido|
|practice_0109|multiplicative|Cruce|mejor independiente/coste0/orden0|131|131|válido|
|practice_0109|multiplicative|Cruce|mejor independiente/coste1/orden0|131|131|válido|
|practice_0109|multiplicative|Cruce|mejor independiente/coste0/orden1|131|131|válido|
|practice_0109|multiplicative|Cruce|mejor independiente/coste1/orden1|131|131|válido|
|practice_0109|multiplicative|E1|cola10/pol1/inicio0|140|46|válido|
|practice_0109|multiplicative|E1|cola10/pol1/inicio1|140|46|válido|
|practice_0109|multiplicative|E1|cola10/pol2/inicio0|140|46|válido|
|practice_0109|multiplicative|E1|cola10/pol2/inicio1|140|46|válido|
|practice_0109|multiplicative|E1|cola8/pol1/inicio0|137|35|válido|
|practice_0109|multiplicative|E1|cola8/pol1/inicio1|137|35|válido|
|practice_0109|multiplicative|E1|cola8/pol2/inicio0|141|39|válido|
|practice_0109|multiplicative|E1|cola8/pol2/inicio1|141|39|válido|
|practice_0109|multiplicative|E1|cola6/pol1/inicio0|141|31|válido|
|practice_0109|multiplicative|E1|cola6/pol1/inicio1|141|31|válido|
|practice_0109|multiplicative|E1|cola6/pol2/inicio0|141|31|válido|
|practice_0109|multiplicative|E1|cola6/pol2/inicio1|141|31|válido|
|practice_0109|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0109|multiplicative|E2|CP False/coste0/orden0|134|134|válido|
|practice_0109|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0109|multiplicative|E2|CP True/coste0/orden0|131|131|válido|
|practice_0109|multiplicative|E3|donante0/coste0/orden0|133|132|válido|
|practice_0109|multiplicative|E3|donante0/coste1/orden0|133|132|válido|
|practice_0109|multiplicative|E3|donante0/coste0/orden1|133|132|válido|
|practice_0109|multiplicative|E3|donante0/coste1/orden1|133|132|válido|
|practice_0109|multiplicative|E3|donante1/coste0/orden0|150|150|válido|
|practice_0109|multiplicative|E3|donante1/coste1/orden0|150|150|válido|
|practice_0109|multiplicative|E3|donante1/coste0/orden1|150|150|válido|
|practice_0109|multiplicative|E3|donante1/coste1/orden1|150|150|válido|
|practice_0109|multiplicative|E3|donante2/coste0/orden0|144|144|válido|
|practice_0109|multiplicative|E3|donante2/coste1/orden0|144|144|válido|
|practice_0109|multiplicative|E3|donante2/coste0/orden1|144|144|válido|
|practice_0109|multiplicative|E3|donante2/coste1/orden1|144|144|válido|
|practice_0109|multiplicative|E3|donante3/coste0/orden0|135|135|válido|
|practice_0109|multiplicative|E3|donante3/coste1/orden0|135|135|válido|
|practice_0109|multiplicative|E3|donante3/coste0/orden1|135|135|válido|
|practice_0109|multiplicative|E3|donante3/coste1/orden1|135|135|válido|
|practice_0109|multiplicative|E3|donante4/coste0/orden0|142|142|válido|
|practice_0109|multiplicative|E3|donante4/coste1/orden0|142|142|válido|
|practice_0109|multiplicative|E3|donante4/coste0/orden1|142|142|válido|
|practice_0109|multiplicative|E3|donante4/coste1/orden1|142|142|válido|
|practice_0109|multiplicative|E3|donante5/coste0/orden0|150|150|válido|
|practice_0109|multiplicative|E3|donante5/coste1/orden0|150|150|válido|
|practice_0109|multiplicative|E3|donante5/coste0/orden1|150|150|válido|
|practice_0109|multiplicative|E3|donante5/coste1/orden1|150|150|válido|
|practice_0109|multiplicative|E3|donante6/coste0/orden0|149|149|válido|
|practice_0109|multiplicative|E3|donante6/coste1/orden0|149|149|válido|
|practice_0109|multiplicative|E3|donante6/coste0/orden1|149|149|válido|
|practice_0109|multiplicative|E3|donante6/coste1/orden1|149|149|válido|
|practice_0109|multiplicative|E3|donante7/coste0/orden0|161|136|válido|
|practice_0109|multiplicative|E3|donante7/coste1/orden0|147|136|válido|
|practice_0109|multiplicative|E3|donante7/coste0/orden1|139|136|válido|
|practice_0109|multiplicative|E3|donante7/coste1/orden1|137|136|válido|
|practice_0109|multiplicative|E3|donante8/coste0/orden0|134|130|válido|
|practice_0109|multiplicative|E3|donante8/coste1/orden0|134|130|válido|
|practice_0109|multiplicative|E3|donante8/coste0/orden1|135|130|válido|
|practice_0109|multiplicative|E3|donante8/coste1/orden1|136|130|válido|
|practice_0110|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0110|independent|Cruce|CP True/coste0/orden0|293|231|válido|
|practice_0110|independent|Cruce|CP True/coste1/orden0|279|231|válido|
|practice_0110|independent|Cruce|CP True/coste0/orden1|237|231|válido|
|practice_0110|independent|Cruce|CP True/coste1/orden1|241|231|válido|
|practice_0110|independent|Cruce|mejor independiente/coste0/orden0|294|232|válido|
|practice_0110|independent|Cruce|mejor independiente/coste1/orden0|280|232|válido|
|practice_0110|independent|Cruce|mejor independiente/coste0/orden1|238|232|válido|
|practice_0110|independent|Cruce|mejor independiente/coste1/orden1|242|232|válido|
|practice_0110|independent|E1|cola10/pol1/inicio0|246|73|válido|
|practice_0110|independent|E1|cola10/pol1/inicio1|246|73|válido|
|practice_0110|independent|E1|cola10/pol2/inicio0|244|71|válido|
|practice_0110|independent|E1|cola10/pol2/inicio1|244|71|válido|
|practice_0110|independent|E1|cola8/pol1/inicio0|244|58|válido|
|practice_0110|independent|E1|cola8/pol1/inicio1|244|58|válido|
|practice_0110|independent|E1|cola8/pol2/inicio0|243|57|válido|
|practice_0110|independent|E1|cola8/pol2/inicio1|243|57|válido|
|practice_0110|independent|E1|cola6/pol1/inicio0|243|44|válido|
|practice_0110|independent|E1|cola6/pol1/inicio1|243|44|válido|
|practice_0110|independent|E1|cola6/pol2/inicio0|243|44|válido|
|practice_0110|independent|E1|cola6/pol2/inicio1|243|44|válido|
|practice_0110|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0110|independent|E2|CP False/coste0/orden0|334|259|válido|
|practice_0110|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0110|independent|E2|CP True/coste0/orden0|293|231|válido|
|practice_0110|independent|E3|donante0/coste0/orden0|238|232|válido|
|practice_0110|independent|E3|donante0/coste1/orden0|238|232|válido|
|practice_0110|independent|E3|donante0/coste0/orden1|238|232|válido|
|practice_0110|independent|E3|donante0/coste1/orden1|242|232|válido|
|practice_0110|independent|E3|donante1/coste0/orden0|268|268|válido|
|practice_0110|independent|E3|donante1/coste1/orden0|268|268|válido|
|practice_0110|independent|E3|donante1/coste0/orden1|292|268|válido|
|practice_0110|independent|E3|donante1/coste1/orden1|273|268|válido|
|practice_0110|independent|E3|donante2/coste0/orden0|317|252|válido|
|practice_0110|independent|E3|donante2/coste1/orden0|311|252|válido|
|practice_0110|independent|E3|donante2/coste0/orden1|252|252|válido|
|practice_0110|independent|E3|donante2/coste1/orden1|252|252|válido|
|practice_0110|independent|E3|donante3/coste0/orden0|413|249|válido|
|practice_0110|independent|E3|donante3/coste1/orden0|316|249|válido|
|practice_0110|independent|E3|donante3/coste0/orden1|249|249|válido|
|practice_0110|independent|E3|donante3/coste1/orden1|249|249|válido|
|practice_0110|independent|E3|donante4/coste0/orden0|267|267|válido|
|practice_0110|independent|E3|donante4/coste1/orden0|267|267|válido|
|practice_0110|independent|E3|donante4/coste0/orden1|267|267|válido|
|practice_0110|independent|E3|donante4/coste1/orden1|267|267|válido|
|practice_0110|independent|E3|donante5/coste0/orden0|272|272|válido|
|practice_0110|independent|E3|donante5/coste1/orden0|272|272|válido|
|practice_0110|independent|E3|donante5/coste0/orden1|272|272|válido|
|practice_0110|independent|E3|donante5/coste1/orden1|272|272|válido|
|practice_0110|independent|E3|donante6/coste0/orden0|269|269|válido|
|practice_0110|independent|E3|donante6/coste1/orden0|269|269|válido|
|practice_0110|independent|E3|donante6/coste0/orden1|269|269|válido|
|practice_0110|independent|E3|donante6/coste1/orden1|269|269|válido|
|practice_0110|independent|E3|donante7/coste0/orden0|252|244|válido|
|practice_0110|independent|E3|donante7/coste1/orden0|250|244|válido|
|practice_0110|independent|E3|donante7/coste0/orden1|252|244|válido|
|practice_0110|independent|E3|donante7/coste1/orden1|251|244|válido|
|practice_0110|independent|E3|donante8/coste0/orden0|238|232|válido|
|practice_0110|independent|E3|donante8/coste1/orden0|238|232|válido|
|practice_0110|independent|E3|donante8/coste0/orden1|238|232|válido|
|practice_0110|independent|E3|donante8/coste1/orden1|242|232|válido|
|practice_0110|independent|E3|donante9/coste0/orden0|283|235|válido|
|practice_0110|independent|E3|donante9/coste1/orden0|254|235|válido|
|practice_0110|independent|E3|donante9/coste0/orden1|270|235|válido|
|practice_0110|independent|E3|donante9/coste1/orden1|259|235|válido|
|practice_0111|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0111|shared|Cruce|CP True/coste0/orden0|158|158|válido|
|practice_0111|shared|Cruce|CP True/coste1/orden0|158|158|válido|
|practice_0111|shared|Cruce|CP True/coste0/orden1|158|158|válido|
|practice_0111|shared|Cruce|CP True/coste1/orden1|158|158|válido|
|practice_0111|shared|Cruce|mejor independiente/coste0/orden0|158|158|válido|
|practice_0111|shared|Cruce|mejor independiente/coste1/orden0|158|158|válido|
|practice_0111|shared|Cruce|mejor independiente/coste0/orden1|158|158|válido|
|practice_0111|shared|Cruce|mejor independiente/coste1/orden1|158|158|válido|
|practice_0111|shared|E1|cola10/pol1/inicio0|165|56|válido|
|practice_0111|shared|E1|cola10/pol1/inicio1|165|56|válido|
|practice_0111|shared|E1|cola10/pol2/inicio0|166|57|válido|
|practice_0111|shared|E1|cola10/pol2/inicio1|166|57|válido|
|practice_0111|shared|E1|cola8/pol1/inicio0|162|43|válido|
|practice_0111|shared|E1|cola8/pol1/inicio1|162|43|válido|
|practice_0111|shared|E1|cola8/pol2/inicio0|163|44|válido|
|practice_0111|shared|E1|cola8/pol2/inicio1|163|44|válido|
|practice_0111|shared|E1|cola6/pol1/inicio0|168|38|válido|
|practice_0111|shared|E1|cola6/pol1/inicio1|168|38|válido|
|practice_0111|shared|E1|cola6/pol2/inicio0|165|35|válido|
|practice_0111|shared|E1|cola6/pol2/inicio1|165|35|válido|
|practice_0111|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0111|shared|E2|CP False/coste0/orden0|176|170|válido|
|practice_0111|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0111|shared|E2|CP True/coste0/orden0|158|158|válido|
|practice_0111|shared|E3|donante0/coste0/orden0|161|160|válido|
|practice_0111|shared|E3|donante0/coste1/orden0|161|160|válido|
|practice_0111|shared|E3|donante0/coste0/orden1|160|160|válido|
|practice_0111|shared|E3|donante0/coste1/orden1|160|160|válido|
|practice_0111|shared|E3|donante1/coste0/orden0|183|183|válido|
|practice_0111|shared|E3|donante1/coste1/orden0|183|183|válido|
|practice_0111|shared|E3|donante1/coste0/orden1|183|183|válido|
|practice_0111|shared|E3|donante1/coste1/orden1|183|183|válido|
|practice_0111|shared|E3|donante2/coste0/orden0|175|175|válido|
|practice_0111|shared|E3|donante2/coste1/orden0|175|175|válido|
|practice_0111|shared|E3|donante2/coste0/orden1|175|175|válido|
|practice_0111|shared|E3|donante2/coste1/orden1|175|175|válido|
|practice_0111|shared|E3|donante3/coste0/orden0|171|171|válido|
|practice_0111|shared|E3|donante3/coste1/orden0|171|171|válido|
|practice_0111|shared|E3|donante3/coste0/orden1|171|171|válido|
|practice_0111|shared|E3|donante3/coste1/orden1|171|171|válido|
|practice_0111|shared|E3|donante4/coste0/orden0|186|186|válido|
|practice_0111|shared|E3|donante4/coste1/orden0|186|186|válido|
|practice_0111|shared|E3|donante4/coste0/orden1|186|186|válido|
|practice_0111|shared|E3|donante4/coste1/orden1|186|186|válido|
|practice_0111|shared|E3|donante5/coste0/orden0|184|184|válido|
|practice_0111|shared|E3|donante5/coste1/orden0|184|184|válido|
|practice_0111|shared|E3|donante5/coste0/orden1|184|184|válido|
|practice_0111|shared|E3|donante5/coste1/orden1|184|184|válido|
|practice_0111|shared|E3|donante6/coste0/orden0|173|173|válido|
|practice_0111|shared|E3|donante6/coste1/orden0|173|173|válido|
|practice_0111|shared|E3|donante6/coste0/orden1|173|173|válido|
|practice_0111|shared|E3|donante6/coste1/orden1|173|173|válido|
|practice_0111|shared|E3|donante7/coste0/orden0|182|165|válido|
|practice_0111|shared|E3|donante7/coste1/orden0|186|165|válido|
|practice_0111|shared|E3|donante7/coste0/orden1|177|165|válido|
|practice_0111|shared|E3|donante7/coste1/orden1|180|165|válido|
|practice_0111|shared|E3|donante8/coste0/orden0|167|162|válido|
|practice_0111|shared|E3|donante8/coste1/orden0|167|162|válido|
|practice_0111|shared|E3|donante8/coste0/orden1|162|162|válido|
|practice_0111|shared|E3|donante8/coste1/orden1|162|162|válido|
|practice_0111|shared|E3|donante9/coste0/orden0|176|163|válido|
|practice_0111|shared|E3|donante9/coste1/orden0|191|163|válido|
|practice_0111|shared|E3|donante9/coste0/orden1|174|163|válido|
|practice_0111|shared|E3|donante9/coste1/orden1|168|163|válido|
|practice_0112|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0112|multiplicative|Cruce|CP True/coste0/orden0|116|116|válido|
|practice_0112|multiplicative|Cruce|CP True/coste1/orden0|116|116|válido|
|practice_0112|multiplicative|Cruce|CP True/coste0/orden1|116|116|válido|
|practice_0112|multiplicative|Cruce|CP True/coste1/orden1|116|116|válido|
|practice_0112|multiplicative|Cruce|mejor independiente/coste0/orden0|116|116|válido|
|practice_0112|multiplicative|Cruce|mejor independiente/coste1/orden0|116|116|válido|
|practice_0112|multiplicative|Cruce|mejor independiente/coste0/orden1|116|116|válido|
|practice_0112|multiplicative|Cruce|mejor independiente/coste1/orden1|116|116|válido|
|practice_0112|multiplicative|E1|cola10/pol1/inicio0|124|51|válido|
|practice_0112|multiplicative|E1|cola10/pol1/inicio1|124|51|válido|
|practice_0112|multiplicative|E1|cola10/pol2/inicio0|126|53|válido|
|practice_0112|multiplicative|E1|cola10/pol2/inicio1|126|53|válido|
|practice_0112|multiplicative|E1|cola8/pol1/inicio0|124|43|válido|
|practice_0112|multiplicative|E1|cola8/pol1/inicio1|124|43|válido|
|practice_0112|multiplicative|E1|cola8/pol2/inicio0|127|46|válido|
|practice_0112|multiplicative|E1|cola8/pol2/inicio1|127|46|válido|
|practice_0112|multiplicative|E1|cola6/pol1/inicio0|125|36|válido|
|practice_0112|multiplicative|E1|cola6/pol1/inicio1|125|36|válido|
|practice_0112|multiplicative|E1|cola6/pol2/inicio0|125|36|válido|
|practice_0112|multiplicative|E1|cola6/pol2/inicio1|125|36|válido|
|practice_0112|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0112|multiplicative|E2|CP False/coste0/orden0|117|117|válido|
|practice_0112|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0112|multiplicative|E2|CP True/coste0/orden0|116|116|válido|
|practice_0112|multiplicative|E3|donante0/coste0/orden0|117|117|válido|
|practice_0112|multiplicative|E3|donante0/coste1/orden0|117|117|válido|
|practice_0112|multiplicative|E3|donante0/coste0/orden1|117|117|válido|
|practice_0112|multiplicative|E3|donante0/coste1/orden1|117|117|válido|
|practice_0112|multiplicative|E3|donante1/coste0/orden0|129|129|válido|
|practice_0112|multiplicative|E3|donante1/coste1/orden0|129|129|válido|
|practice_0112|multiplicative|E3|donante1/coste0/orden1|129|129|válido|
|practice_0112|multiplicative|E3|donante1/coste1/orden1|129|129|válido|
|practice_0112|multiplicative|E3|donante2/coste0/orden0|124|124|válido|
|practice_0112|multiplicative|E3|donante2/coste1/orden0|124|124|válido|
|practice_0112|multiplicative|E3|donante2/coste0/orden1|124|124|válido|
|practice_0112|multiplicative|E3|donante2/coste1/orden1|124|124|válido|
|practice_0112|multiplicative|E3|donante3/coste0/orden0|118|118|válido|
|practice_0112|multiplicative|E3|donante3/coste1/orden0|118|118|válido|
|practice_0112|multiplicative|E3|donante3/coste0/orden1|118|118|válido|
|practice_0112|multiplicative|E3|donante3/coste1/orden1|118|118|válido|
|practice_0112|multiplicative|E3|donante4/coste0/orden0|131|131|válido|
|practice_0112|multiplicative|E3|donante4/coste1/orden0|131|131|válido|
|practice_0112|multiplicative|E3|donante4/coste0/orden1|131|131|válido|
|practice_0112|multiplicative|E3|donante4/coste1/orden1|131|131|válido|
|practice_0112|multiplicative|E3|donante5/coste0/orden0|132|132|válido|
|practice_0112|multiplicative|E3|donante5/coste1/orden0|132|132|válido|
|practice_0112|multiplicative|E3|donante5/coste0/orden1|132|132|válido|
|practice_0112|multiplicative|E3|donante5/coste1/orden1|132|132|válido|
|practice_0112|multiplicative|E3|donante6/coste0/orden0|123|123|válido|
|practice_0112|multiplicative|E3|donante6/coste1/orden0|123|123|válido|
|practice_0112|multiplicative|E3|donante6/coste0/orden1|123|123|válido|
|practice_0112|multiplicative|E3|donante6/coste1/orden1|123|123|válido|
|practice_0112|multiplicative|E3|donante7/coste0/orden0|118|118|válido|
|practice_0112|multiplicative|E3|donante7/coste1/orden0|118|118|válido|
|practice_0112|multiplicative|E3|donante7/coste0/orden1|118|118|válido|
|practice_0112|multiplicative|E3|donante7/coste1/orden1|118|118|válido|
|practice_0112|multiplicative|E3|donante8/coste0/orden0|119|119|válido|
|practice_0112|multiplicative|E3|donante8/coste1/orden0|119|119|válido|
|practice_0112|multiplicative|E3|donante8/coste0/orden1|119|119|válido|
|practice_0112|multiplicative|E3|donante8/coste1/orden1|119|119|válido|
|practice_0113|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0113|independent|Cruce|CP True/coste0/orden0|436|266|válido|
|practice_0113|independent|Cruce|CP True/coste1/orden0|313|266|válido|
|practice_0113|independent|Cruce|CP True/coste0/orden1|404|266|válido|
|practice_0113|independent|Cruce|CP True/coste1/orden1|285|266|válido|
|practice_0113|independent|Cruce|mejor independiente/coste0/orden0|459|266|válido|
|practice_0113|independent|Cruce|mejor independiente/coste1/orden0|314|266|válido|
|practice_0113|independent|Cruce|mejor independiente/coste0/orden1|411|266|válido|
|practice_0113|independent|Cruce|mejor independiente/coste1/orden1|291|266|válido|
|practice_0113|independent|E1|cola10/pol1/inicio0|294|94|válido|
|practice_0113|independent|E1|cola10/pol1/inicio1|294|94|válido|
|practice_0113|independent|E1|cola10/pol2/inicio0|288|88|válido|
|practice_0113|independent|E1|cola10/pol2/inicio1|288|88|válido|
|practice_0113|independent|E1|cola8/pol1/inicio0|290|74|válido|
|practice_0113|independent|E1|cola8/pol1/inicio1|291|75|válido|
|practice_0113|independent|E1|cola8/pol2/inicio0|290|74|válido|
|practice_0113|independent|E1|cola8/pol2/inicio1|290|74|válido|
|practice_0113|independent|E1|cola6/pol1/inicio0|289|55|válido|
|practice_0113|independent|E1|cola6/pol1/inicio1|289|55|válido|
|practice_0113|independent|E1|cola6/pol2/inicio0|286|52|válido|
|practice_0113|independent|E1|cola6/pol2/inicio1|286|52|válido|
|practice_0113|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0113|independent|E2|CP False/coste0/orden0|405|271|válido|
|practice_0113|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0113|independent|E2|CP True/coste0/orden0|277|277|válido|
|practice_0113|independent|E3|donante0/coste0/orden0|277|277|válido|
|practice_0113|independent|E3|donante0/coste1/orden0|277|277|válido|
|practice_0113|independent|E3|donante0/coste0/orden1|315|277|válido|
|practice_0113|independent|E3|donante0/coste1/orden1|310|277|válido|
|practice_0113|independent|E3|donante1/coste0/orden0|306|306|válido|
|practice_0113|independent|E3|donante1/coste1/orden0|306|306|válido|
|practice_0113|independent|E3|donante1/coste0/orden1|306|306|válido|
|practice_0113|independent|E3|donante1/coste1/orden1|306|306|válido|
|practice_0113|independent|E3|donante2/coste0/orden0|607|285|válido|
|practice_0113|independent|E3|donante2/coste1/orden0|310|285|válido|
|practice_0113|independent|E3|donante2/coste0/orden1|285|285|válido|
|practice_0113|independent|E3|donante2/coste1/orden1|285|285|válido|
|practice_0113|independent|E3|donante3/coste0/orden0|443|288|válido|
|practice_0113|independent|E3|donante3/coste1/orden0|350|288|válido|
|practice_0113|independent|E3|donante3/coste0/orden1|288|288|válido|
|practice_0113|independent|E3|donante3/coste1/orden1|288|288|válido|
|practice_0113|independent|E3|donante4/coste0/orden0|318|318|válido|
|practice_0113|independent|E3|donante4/coste1/orden0|318|318|válido|
|practice_0113|independent|E3|donante4/coste0/orden1|318|318|válido|
|practice_0113|independent|E3|donante4/coste1/orden1|318|318|válido|
|practice_0113|independent|E3|donante5/coste0/orden0|317|317|válido|
|practice_0113|independent|E3|donante5/coste1/orden0|317|317|válido|
|practice_0113|independent|E3|donante5/coste0/orden1|317|317|válido|
|practice_0113|independent|E3|donante5/coste1/orden1|317|317|válido|
|practice_0113|independent|E3|donante6/coste0/orden0|308|308|válido|
|practice_0113|independent|E3|donante6/coste1/orden0|308|308|válido|
|practice_0113|independent|E3|donante6/coste0/orden1|308|308|válido|
|practice_0113|independent|E3|donante6/coste1/orden1|308|308|válido|
|practice_0113|independent|E3|donante7/coste0/orden0|278|278|válido|
|practice_0113|independent|E3|donante7/coste1/orden0|278|278|válido|
|practice_0113|independent|E3|donante7/coste0/orden1|317|278|válido|
|practice_0113|independent|E3|donante7/coste1/orden1|312|278|válido|
|practice_0113|independent|E3|donante8/coste0/orden0|296|271|válido|
|practice_0113|independent|E3|donante8/coste1/orden0|320|271|válido|
|practice_0113|independent|E3|donante8/coste0/orden1|358|271|válido|
|practice_0113|independent|E3|donante8/coste1/orden1|292|271|válido|
|practice_0113|independent|E3|donante9/coste0/orden0|316|266|válido|
|practice_0113|independent|E3|donante9/coste1/orden0|270|266|válido|
|practice_0113|independent|E3|donante9/coste0/orden1|411|266|válido|
|practice_0113|independent|E3|donante9/coste1/orden1|291|266|válido|
|practice_0114|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0114|mixed|Cruce|CP True/coste0/orden0|46|46|válido|
|practice_0114|mixed|Cruce|CP True/coste1/orden0|46|46|válido|
|practice_0114|mixed|Cruce|CP True/coste0/orden1|46|46|válido|
|practice_0114|mixed|Cruce|CP True/coste1/orden1|46|46|válido|
|practice_0114|mixed|Cruce|mejor independiente/coste0/orden0|46|46|válido|
|practice_0114|mixed|Cruce|mejor independiente/coste1/orden0|46|46|válido|
|practice_0114|mixed|Cruce|mejor independiente/coste0/orden1|46|46|válido|
|practice_0114|mixed|Cruce|mejor independiente/coste1/orden1|46|46|válido|
|practice_0114|mixed|E1|cola10/pol1/inicio0|50|24|válido|
|practice_0114|mixed|E1|cola10/pol1/inicio1|50|24|válido|
|practice_0114|mixed|E1|cola10/pol2/inicio0|50|24|válido|
|practice_0114|mixed|E1|cola10/pol2/inicio1|50|24|válido|
|practice_0114|mixed|E1|cola8/pol1/inicio0|50|20|válido|
|practice_0114|mixed|E1|cola8/pol1/inicio1|50|20|válido|
|practice_0114|mixed|E1|cola8/pol2/inicio0|49|19|válido|
|practice_0114|mixed|E1|cola8/pol2/inicio1|49|19|válido|
|practice_0114|mixed|E1|cola6/pol1/inicio0|50|17|válido|
|practice_0114|mixed|E1|cola6/pol1/inicio1|50|17|válido|
|practice_0114|mixed|E1|cola6/pol2/inicio0|49|16|válido|
|practice_0114|mixed|E1|cola6/pol2/inicio1|49|16|válido|
|practice_0114|mixed|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0114|mixed|E2|CP False/coste0/orden0|49|49|válido|
|practice_0114|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0114|mixed|E2|CP True/coste0/orden0|46|46|válido|
|practice_0114|mixed|E3|donante0/coste0/orden0|49|49|válido|
|practice_0114|mixed|E3|donante0/coste1/orden0|49|49|válido|
|practice_0114|mixed|E3|donante0/coste0/orden1|49|49|válido|
|practice_0114|mixed|E3|donante0/coste1/orden1|49|49|válido|
|practice_0114|mixed|E3|donante1/coste0/orden0|57|57|válido|
|practice_0114|mixed|E3|donante1/coste1/orden0|57|57|válido|
|practice_0114|mixed|E3|donante1/coste0/orden1|57|57|válido|
|practice_0114|mixed|E3|donante1/coste1/orden1|57|57|válido|
|practice_0114|mixed|E3|donante2/coste0/orden0|50|50|válido|
|practice_0114|mixed|E3|donante2/coste1/orden0|50|50|válido|
|practice_0114|mixed|E3|donante2/coste0/orden1|50|50|válido|
|practice_0114|mixed|E3|donante2/coste1/orden1|50|50|válido|
|practice_0114|mixed|E3|donante3/coste0/orden0|54|54|válido|
|practice_0114|mixed|E3|donante3/coste1/orden0|54|54|válido|
|practice_0114|mixed|E3|donante3/coste0/orden1|54|54|válido|
|practice_0114|mixed|E3|donante3/coste1/orden1|54|54|válido|
|practice_0114|mixed|E3|donante4/coste0/orden0|56|56|válido|
|practice_0114|mixed|E3|donante4/coste1/orden0|56|56|válido|
|practice_0114|mixed|E3|donante4/coste0/orden1|56|56|válido|
|practice_0114|mixed|E3|donante4/coste1/orden1|56|56|válido|
|practice_0114|mixed|E3|donante5/coste0/orden0|51|51|válido|
|practice_0114|mixed|E3|donante5/coste1/orden0|51|51|válido|
|practice_0114|mixed|E3|donante5/coste0/orden1|51|51|válido|
|practice_0114|mixed|E3|donante5/coste1/orden1|51|51|válido|
|practice_0114|mixed|E3|donante6/coste0/orden0|53|53|válido|
|practice_0114|mixed|E3|donante6/coste1/orden0|53|53|válido|
|practice_0114|mixed|E3|donante6/coste0/orden1|53|53|válido|
|practice_0114|mixed|E3|donante6/coste1/orden1|53|53|válido|
|practice_0114|mixed|E3|donante7/coste0/orden0|55|55|válido|
|practice_0114|mixed|E3|donante7/coste1/orden0|55|55|válido|
|practice_0114|mixed|E3|donante7/coste0/orden1|55|55|válido|
|practice_0114|mixed|E3|donante7/coste1/orden1|55|55|válido|
|practice_0114|mixed|E3|donante8/coste0/orden0|52|52|válido|
|practice_0114|mixed|E3|donante8/coste1/orden0|52|52|válido|
|practice_0114|mixed|E3|donante8/coste0/orden1|52|52|válido|
|practice_0114|mixed|E3|donante8/coste1/orden1|52|52|válido|
|practice_0114|mixed|E3|donante9/coste0/orden0|55|55|válido|
|practice_0114|mixed|E3|donante9/coste1/orden0|55|55|válido|
|practice_0114|mixed|E3|donante9/coste0/orden1|55|55|válido|
|practice_0114|mixed|E3|donante9/coste1/orden1|55|55|válido|
|practice_0115|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0115|shared|Cruce|CP True/coste0/orden0|184|174|válido|
|practice_0115|shared|Cruce|CP True/coste1/orden0|181|174|válido|
|practice_0115|shared|Cruce|CP True/coste0/orden1|182|174|válido|
|practice_0115|shared|Cruce|CP True/coste1/orden1|177|174|válido|
|practice_0115|shared|Cruce|mejor independiente/coste0/orden0|183|176|válido|
|practice_0115|shared|Cruce|mejor independiente/coste1/orden0|179|176|válido|
|practice_0115|shared|Cruce|mejor independiente/coste0/orden1|176|176|válido|
|practice_0115|shared|Cruce|mejor independiente/coste1/orden1|176|176|válido|
|practice_0115|shared|E1|cola10/pol1/inicio0|187|70|válido|
|practice_0115|shared|E1|cola10/pol1/inicio1|187|70|válido|
|practice_0115|shared|E1|cola10/pol2/inicio0|187|70|válido|
|practice_0115|shared|E1|cola10/pol2/inicio1|187|70|válido|
|practice_0115|shared|E1|cola8/pol1/inicio0|187|58|válido|
|practice_0115|shared|E1|cola8/pol1/inicio1|187|58|válido|
|practice_0115|shared|E1|cola8/pol2/inicio0|187|58|válido|
|practice_0115|shared|E1|cola8/pol2/inicio1|187|58|válido|
|practice_0115|shared|E1|cola6/pol1/inicio0|190|46|válido|
|practice_0115|shared|E1|cola6/pol1/inicio1|190|46|válido|
|practice_0115|shared|E1|cola6/pol2/inicio0|190|46|válido|
|practice_0115|shared|E1|cola6/pol2/inicio1|190|46|válido|
|practice_0115|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0115|shared|E2|CP False/coste0/orden0|218|186|válido|
|practice_0115|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0115|shared|E2|CP True/coste0/orden0|179|178|válido|
|practice_0115|shared|E3|donante0/coste0/orden0|190|180|válido|
|practice_0115|shared|E3|donante0/coste1/orden0|187|180|válido|
|practice_0115|shared|E3|donante0/coste0/orden1|185|180|válido|
|practice_0115|shared|E3|donante0/coste1/orden1|181|180|válido|
|practice_0115|shared|E3|donante1/coste0/orden0|189|189|válido|
|practice_0115|shared|E3|donante1/coste1/orden0|189|189|válido|
|practice_0115|shared|E3|donante1/coste0/orden1|189|189|válido|
|practice_0115|shared|E3|donante1/coste1/orden1|189|189|válido|
|practice_0115|shared|E3|donante2/coste0/orden0|184|184|válido|
|practice_0115|shared|E3|donante2/coste1/orden0|184|184|válido|
|practice_0115|shared|E3|donante2/coste0/orden1|184|184|válido|
|practice_0115|shared|E3|donante2/coste1/orden1|184|184|válido|
|practice_0115|shared|E3|donante3/coste0/orden0|184|181|válido|
|practice_0115|shared|E3|donante3/coste1/orden0|184|181|válido|
|practice_0115|shared|E3|donante3/coste0/orden1|181|181|válido|
|practice_0115|shared|E3|donante3/coste1/orden1|181|181|válido|
|practice_0115|shared|E3|donante4/coste0/orden0|197|197|válido|
|practice_0115|shared|E3|donante4/coste1/orden0|197|197|válido|
|practice_0115|shared|E3|donante4/coste0/orden1|197|197|válido|
|practice_0115|shared|E3|donante4/coste1/orden1|197|197|válido|
|practice_0115|shared|E3|donante5/coste0/orden0|197|197|válido|
|practice_0115|shared|E3|donante5/coste1/orden0|197|197|válido|
|practice_0115|shared|E3|donante5/coste0/orden1|197|197|válido|
|practice_0115|shared|E3|donante5/coste1/orden1|197|197|válido|
|practice_0115|shared|E3|donante6/coste0/orden0|203|203|válido|
|practice_0115|shared|E3|donante6/coste1/orden0|203|203|válido|
|practice_0115|shared|E3|donante6/coste0/orden1|203|203|válido|
|practice_0115|shared|E3|donante6/coste1/orden1|203|203|válido|
|practice_0115|shared|E3|donante7/coste0/orden0|190|180|válido|
|practice_0115|shared|E3|donante7/coste1/orden0|187|180|válido|
|practice_0115|shared|E3|donante7/coste0/orden1|185|180|válido|
|practice_0115|shared|E3|donante7/coste1/orden1|181|180|válido|
|practice_0115|shared|E3|donante8/coste0/orden0|192|180|válido|
|practice_0115|shared|E3|donante8/coste1/orden0|191|180|válido|
|practice_0115|shared|E3|donante8/coste0/orden1|195|180|válido|
|practice_0115|shared|E3|donante8/coste1/orden1|186|180|válido|
|practice_0115|shared|E3|donante9/coste0/orden0|204|177|válido|
|practice_0115|shared|E3|donante9/coste1/orden0|199|177|válido|
|practice_0115|shared|E3|donante9/coste0/orden1|177|177|válido|
|practice_0115|shared|E3|donante9/coste1/orden1|177|177|válido|
|practice_0116|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0116|independent|Cruce|CP True/coste0/orden0|60|60|válido|
|practice_0116|independent|Cruce|CP True/coste1/orden0|60|60|válido|
|practice_0116|independent|Cruce|CP True/coste0/orden1|60|60|válido|
|practice_0116|independent|Cruce|CP True/coste1/orden1|60|60|válido|
|practice_0116|independent|Cruce|mejor independiente/coste0/orden0|63|63|válido|
|practice_0116|independent|Cruce|mejor independiente/coste1/orden0|63|63|válido|
|practice_0116|independent|Cruce|mejor independiente/coste0/orden1|63|63|válido|
|practice_0116|independent|Cruce|mejor independiente/coste1/orden1|63|63|válido|
|practice_0116|independent|E1|cola10/pol1/inicio0|66|51|válido|
|practice_0116|independent|E1|cola10/pol1/inicio1|66|51|válido|
|practice_0116|independent|E1|cola10/pol2/inicio0|67|52|válido|
|practice_0116|independent|E1|cola10/pol2/inicio1|67|52|válido|
|practice_0116|independent|E1|cola8/pol1/inicio0|66|40|válido|
|practice_0116|independent|E1|cola8/pol1/inicio1|66|40|válido|
|practice_0116|independent|E1|cola8/pol2/inicio0|65|39|válido|
|practice_0116|independent|E1|cola8/pol2/inicio1|65|39|válido|
|practice_0116|independent|E1|cola6/pol1/inicio0|71|29|válido|
|practice_0116|independent|E1|cola6/pol1/inicio1|71|29|válido|
|practice_0116|independent|E1|cola6/pol2/inicio0|72|28|válido|
|practice_0116|independent|E1|cola6/pol2/inicio1|71|28|válido|
|practice_0116|independent|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0116|independent|E2|CP False/coste0/orden0|63|63|válido|
|practice_0116|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0116|independent|E2|CP True/coste0/orden0|64|64|válido|
|practice_0116|independent|E3|donante0/coste0/orden0|64|64|válido|
|practice_0116|independent|E3|donante0/coste1/orden0|64|64|válido|
|practice_0116|independent|E3|donante0/coste0/orden1|64|64|válido|
|practice_0116|independent|E3|donante0/coste1/orden1|64|64|válido|
|practice_0116|independent|E3|donante1/coste0/orden0|71|71|válido|
|practice_0116|independent|E3|donante1/coste1/orden0|71|71|válido|
|practice_0116|independent|E3|donante1/coste0/orden1|71|71|válido|
|practice_0116|independent|E3|donante1/coste1/orden1|71|71|válido|
|practice_0116|independent|E3|donante2/coste0/orden0|69|69|válido|
|practice_0116|independent|E3|donante2/coste1/orden0|69|69|válido|
|practice_0116|independent|E3|donante2/coste0/orden1|69|69|válido|
|practice_0116|independent|E3|donante2/coste1/orden1|69|69|válido|
|practice_0116|independent|E3|donante3/coste0/orden0|71|71|válido|
|practice_0116|independent|E3|donante3/coste1/orden0|71|71|válido|
|practice_0116|independent|E3|donante3/coste0/orden1|71|71|válido|
|practice_0116|independent|E3|donante3/coste1/orden1|71|71|válido|
|practice_0116|independent|E3|donante4/coste0/orden0|71|71|válido|
|practice_0116|independent|E3|donante4/coste1/orden0|71|71|válido|
|practice_0116|independent|E3|donante4/coste0/orden1|71|71|válido|
|practice_0116|independent|E3|donante4/coste1/orden1|71|71|válido|
|practice_0116|independent|E3|donante5/coste0/orden0|68|68|válido|
|practice_0116|independent|E3|donante5/coste1/orden0|68|68|válido|
|practice_0116|independent|E3|donante5/coste0/orden1|68|68|válido|
|practice_0116|independent|E3|donante5/coste1/orden1|68|68|válido|
|practice_0116|independent|E3|donante6/coste0/orden0|67|67|válido|
|practice_0116|independent|E3|donante6/coste1/orden0|67|67|válido|
|practice_0116|independent|E3|donante6/coste0/orden1|67|67|válido|
|practice_0116|independent|E3|donante6/coste1/orden1|67|67|válido|
|practice_0116|independent|E3|donante7/coste0/orden0|72|72|válido|
|practice_0116|independent|E3|donante7/coste1/orden0|72|72|válido|
|practice_0116|independent|E3|donante7/coste0/orden1|72|72|válido|
|practice_0116|independent|E3|donante7/coste1/orden1|72|72|válido|
|practice_0116|independent|E3|donante8/coste0/orden0|69|69|válido|
|practice_0116|independent|E3|donante8/coste1/orden0|69|69|válido|
|practice_0116|independent|E3|donante8/coste0/orden1|69|69|válido|
|practice_0116|independent|E3|donante8/coste1/orden1|69|69|válido|
|practice_0116|independent|E3|donante9/coste0/orden0|71|71|válido|
|practice_0116|independent|E3|donante9/coste1/orden0|71|71|válido|
|practice_0116|independent|E3|donante9/coste0/orden1|71|71|válido|
|practice_0116|independent|E3|donante9/coste1/orden1|71|71|válido|
|practice_0117|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0117|multiplicative|Cruce|CP True/coste0/orden0|137|137|válido|
|practice_0117|multiplicative|Cruce|CP True/coste1/orden0|137|137|válido|
|practice_0117|multiplicative|Cruce|CP True/coste0/orden1|137|137|válido|
|practice_0117|multiplicative|Cruce|CP True/coste1/orden1|137|137|válido|
|practice_0117|multiplicative|Cruce|mejor independiente/coste0/orden0|137|137|válido|
|practice_0117|multiplicative|Cruce|mejor independiente/coste1/orden0|137|137|válido|
|practice_0117|multiplicative|Cruce|mejor independiente/coste0/orden1|137|137|válido|
|practice_0117|multiplicative|Cruce|mejor independiente/coste1/orden1|137|137|válido|
|practice_0117|multiplicative|E1|cola10/pol1/inicio0|142|43|válido|
|practice_0117|multiplicative|E1|cola10/pol1/inicio1|142|43|válido|
|practice_0117|multiplicative|E1|cola10/pol2/inicio0|142|43|válido|
|practice_0117|multiplicative|E1|cola10/pol2/inicio1|142|43|válido|
|practice_0117|multiplicative|E1|cola8/pol1/inicio0|142|36|válido|
|practice_0117|multiplicative|E1|cola8/pol1/inicio1|142|36|válido|
|practice_0117|multiplicative|E1|cola8/pol2/inicio0|142|36|válido|
|practice_0117|multiplicative|E1|cola8/pol2/inicio1|142|36|válido|
|practice_0117|multiplicative|E1|cola6/pol1/inicio0|142|28|válido|
|practice_0117|multiplicative|E1|cola6/pol1/inicio1|142|28|válido|
|practice_0117|multiplicative|E1|cola6/pol2/inicio0|142|28|válido|
|practice_0117|multiplicative|E1|cola6/pol2/inicio1|142|28|válido|
|practice_0117|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0117|multiplicative|E2|CP False/coste0/orden0|142|142|válido|
|practice_0117|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0117|multiplicative|E2|CP True/coste0/orden0|137|137|válido|
|practice_0117|multiplicative|E3|donante0/coste0/orden0|139|139|válido|
|practice_0117|multiplicative|E3|donante0/coste1/orden0|139|139|válido|
|practice_0117|multiplicative|E3|donante0/coste0/orden1|139|139|válido|
|practice_0117|multiplicative|E3|donante0/coste1/orden1|139|139|válido|
|practice_0117|multiplicative|E3|donante1/coste0/orden0|143|143|válido|
|practice_0117|multiplicative|E3|donante1/coste1/orden0|143|143|válido|
|practice_0117|multiplicative|E3|donante1/coste0/orden1|143|143|válido|
|practice_0117|multiplicative|E3|donante1/coste1/orden1|143|143|válido|
|practice_0117|multiplicative|E3|donante2/coste0/orden0|145|145|válido|
|practice_0117|multiplicative|E3|donante2/coste1/orden0|145|145|válido|
|practice_0117|multiplicative|E3|donante2/coste0/orden1|145|145|válido|
|practice_0117|multiplicative|E3|donante2/coste1/orden1|145|145|válido|
|practice_0117|multiplicative|E3|donante3/coste0/orden0|142|142|válido|
|practice_0117|multiplicative|E3|donante3/coste1/orden0|142|142|válido|
|practice_0117|multiplicative|E3|donante3/coste0/orden1|142|142|válido|
|practice_0117|multiplicative|E3|donante3/coste1/orden1|142|142|válido|
|practice_0117|multiplicative|E3|donante4/coste0/orden0|151|151|válido|
|practice_0117|multiplicative|E3|donante4/coste1/orden0|151|151|válido|
|practice_0117|multiplicative|E3|donante4/coste0/orden1|151|151|válido|
|practice_0117|multiplicative|E3|donante4/coste1/orden1|151|151|válido|
|practice_0117|multiplicative|E3|donante5/coste0/orden0|154|154|válido|
|practice_0117|multiplicative|E3|donante5/coste1/orden0|154|154|válido|
|practice_0117|multiplicative|E3|donante5/coste0/orden1|154|154|válido|
|practice_0117|multiplicative|E3|donante5/coste1/orden1|154|154|válido|
|practice_0117|multiplicative|E3|donante6/coste0/orden0|159|159|válido|
|practice_0117|multiplicative|E3|donante6/coste1/orden0|159|159|válido|
|practice_0117|multiplicative|E3|donante6/coste0/orden1|159|159|válido|
|practice_0117|multiplicative|E3|donante6/coste1/orden1|159|159|válido|
|practice_0117|multiplicative|E3|donante7/coste0/orden0|149|143|válido|
|practice_0117|multiplicative|E3|donante7/coste1/orden0|152|143|válido|
|practice_0117|multiplicative|E3|donante7/coste0/orden1|156|143|válido|
|practice_0117|multiplicative|E3|donante7/coste1/orden1|167|143|válido|
|practice_0117|multiplicative|E3|donante8/coste0/orden0|169|144|válido|
|practice_0117|multiplicative|E3|donante8/coste1/orden0|162|144|válido|
|practice_0117|multiplicative|E3|donante8/coste0/orden1|146|144|válido|
|practice_0117|multiplicative|E3|donante8/coste1/orden1|146|144|válido|
|practice_0118|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0118|mixed|Cruce|CP True/coste0/orden0|35|35|válido|
|practice_0118|mixed|Cruce|CP True/coste1/orden0|35|35|válido|
|practice_0118|mixed|Cruce|CP True/coste0/orden1|35|35|válido|
|practice_0118|mixed|Cruce|CP True/coste1/orden1|35|35|válido|
|practice_0118|mixed|Cruce|mejor independiente/coste0/orden0|35|35|válido|
|practice_0118|mixed|Cruce|mejor independiente/coste1/orden0|35|35|válido|
|practice_0118|mixed|Cruce|mejor independiente/coste0/orden1|35|35|válido|
|practice_0118|mixed|Cruce|mejor independiente/coste1/orden1|35|35|válido|
|practice_0118|mixed|E1|cola8/pol1/inicio0|42|33|válido|
|practice_0118|mixed|E1|cola8/pol1/inicio1|41|32|válido|
|practice_0118|mixed|E1|cola8/pol2/inicio0|42|33|válido|
|practice_0118|mixed|E1|cola8/pol2/inicio1|41|32|válido|
|practice_0118|mixed|E1|cola6/pol1/inicio0|42|20|válido|
|practice_0118|mixed|E1|cola6/pol1/inicio1|41|19|válido|
|practice_0118|mixed|E1|cola6/pol2/inicio0|42|20|válido|
|practice_0118|mixed|E1|cola6/pol2/inicio1|41|19|válido|
|practice_0118|mixed|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0118|mixed|E2|CP False/coste0/orden0|40|40|válido|
|practice_0118|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0118|mixed|E2|CP True/coste0/orden0|35|35|válido|
|practice_0118|mixed|E3|donante0/coste0/orden0|40|40|válido|
|practice_0118|mixed|E3|donante0/coste1/orden0|40|40|válido|
|practice_0118|mixed|E3|donante0/coste0/orden1|40|40|válido|
|practice_0118|mixed|E3|donante0/coste1/orden1|40|40|válido|
|practice_0118|mixed|E3|donante1/coste0/orden0|43|43|válido|
|practice_0118|mixed|E3|donante1/coste1/orden0|43|43|válido|
|practice_0118|mixed|E3|donante1/coste0/orden1|43|43|válido|
|practice_0118|mixed|E3|donante1/coste1/orden1|43|43|válido|
|practice_0118|mixed|E3|donante2/coste0/orden0|44|44|válido|
|practice_0118|mixed|E3|donante2/coste1/orden0|44|44|válido|
|practice_0118|mixed|E3|donante2/coste0/orden1|44|44|válido|
|practice_0118|mixed|E3|donante2/coste1/orden1|44|44|válido|
|practice_0118|mixed|E3|donante3/coste0/orden0|43|43|válido|
|practice_0118|mixed|E3|donante3/coste1/orden0|43|43|válido|
|practice_0118|mixed|E3|donante3/coste0/orden1|43|43|válido|
|practice_0118|mixed|E3|donante3/coste1/orden1|43|43|válido|
|practice_0118|mixed|E3|donante4/coste0/orden0|44|44|válido|
|practice_0118|mixed|E3|donante4/coste1/orden0|44|44|válido|
|practice_0118|mixed|E3|donante4/coste0/orden1|44|44|válido|
|practice_0118|mixed|E3|donante4/coste1/orden1|44|44|válido|
|practice_0118|mixed|E3|donante5/coste0/orden0|43|43|válido|
|practice_0118|mixed|E3|donante5/coste1/orden0|43|43|válido|
|practice_0118|mixed|E3|donante5/coste0/orden1|43|43|válido|
|practice_0118|mixed|E3|donante5/coste1/orden1|43|43|válido|
|practice_0118|mixed|E3|donante6/coste0/orden0|41|41|válido|
|practice_0118|mixed|E3|donante6/coste1/orden0|41|41|válido|
|practice_0118|mixed|E3|donante6/coste0/orden1|41|41|válido|
|practice_0118|mixed|E3|donante6/coste1/orden1|41|41|válido|
|practice_0119|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0119|independent|Cruce|CP True/coste0/orden0|292|229|válido|
|practice_0119|independent|Cruce|CP True/coste1/orden0|273|229|válido|
|practice_0119|independent|Cruce|CP True/coste0/orden1|264|229|válido|
|practice_0119|independent|Cruce|CP True/coste1/orden1|253|229|válido|
|practice_0119|independent|Cruce|mejor independiente/coste0/orden0|275|231|válido|
|practice_0119|independent|Cruce|mejor independiente/coste1/orden0|272|231|válido|
|practice_0119|independent|Cruce|mejor independiente/coste0/orden1|254|231|válido|
|practice_0119|independent|Cruce|mejor independiente/coste1/orden1|252|231|válido|
|practice_0119|independent|E1|cola10/pol1/inicio0|250|78|válido|
|practice_0119|independent|E1|cola10/pol1/inicio1|250|78|válido|
|practice_0119|independent|E1|cola10/pol2/inicio0|247|75|válido|
|practice_0119|independent|E1|cola10/pol2/inicio1|247|75|válido|
|practice_0119|independent|E1|cola8/pol1/inicio0|245|59|válido|
|practice_0119|independent|E1|cola8/pol1/inicio1|245|59|válido|
|practice_0119|independent|E1|cola8/pol2/inicio0|241|55|válido|
|practice_0119|independent|E1|cola8/pol2/inicio1|244|58|válido|
|practice_0119|independent|E1|cola6/pol1/inicio0|247|46|válido|
|practice_0119|independent|E1|cola6/pol1/inicio1|247|46|válido|
|practice_0119|independent|E1|cola6/pol2/inicio0|247|46|válido|
|practice_0119|independent|E1|cola6/pol2/inicio1|247|46|válido|
|practice_0119|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0119|independent|E2|CP False/coste0/orden0|266|237|válido|
|practice_0119|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0119|independent|E2|CP True/coste0/orden0|293|230|válido|
|practice_0119|independent|E3|donante0/coste0/orden0|236|231|válido|
|practice_0119|independent|E3|donante0/coste1/orden0|236|231|válido|
|practice_0119|independent|E3|donante0/coste0/orden1|254|231|válido|
|practice_0119|independent|E3|donante0/coste1/orden1|252|231|válido|
|practice_0119|independent|E3|donante1/coste0/orden0|267|267|válido|
|practice_0119|independent|E3|donante1/coste1/orden0|267|267|válido|
|practice_0119|independent|E3|donante1/coste0/orden1|267|267|válido|
|practice_0119|independent|E3|donante1/coste1/orden1|267|267|válido|
|practice_0119|independent|E3|donante2/coste0/orden0|270|255|válido|
|practice_0119|independent|E3|donante2/coste1/orden0|263|255|válido|
|practice_0119|independent|E3|donante2/coste0/orden1|255|255|válido|
|practice_0119|independent|E3|donante2/coste1/orden1|255|255|válido|
|practice_0119|independent|E3|donante3/coste0/orden0|292|247|válido|
|practice_0119|independent|E3|donante3/coste1/orden0|276|247|válido|
|practice_0119|independent|E3|donante3/coste0/orden1|247|247|válido|
|practice_0119|independent|E3|donante3/coste1/orden1|247|247|válido|
|practice_0119|independent|E3|donante4/coste0/orden0|271|271|válido|
|practice_0119|independent|E3|donante4/coste1/orden0|271|271|válido|
|practice_0119|independent|E3|donante4/coste0/orden1|271|271|válido|
|practice_0119|independent|E3|donante4/coste1/orden1|271|271|válido|
|practice_0119|independent|E3|donante5/coste0/orden0|269|269|válido|
|practice_0119|independent|E3|donante5/coste1/orden0|269|269|válido|
|practice_0119|independent|E3|donante5/coste0/orden1|269|269|válido|
|practice_0119|independent|E3|donante5/coste1/orden1|269|269|válido|
|practice_0119|independent|E3|donante6/coste0/orden0|270|270|válido|
|practice_0119|independent|E3|donante6/coste1/orden0|270|270|válido|
|practice_0119|independent|E3|donante6/coste0/orden1|270|270|válido|
|practice_0119|independent|E3|donante6/coste1/orden1|270|270|válido|
|practice_0119|independent|E3|donante7/coste0/orden0|274|242|válido|
|practice_0119|independent|E3|donante7/coste1/orden0|266|242|válido|
|practice_0119|independent|E3|donante7/coste0/orden1|279|242|válido|
|practice_0119|independent|E3|donante7/coste1/orden1|275|242|válido|
|practice_0119|independent|E3|donante8/coste0/orden0|237|232|válido|
|practice_0119|independent|E3|donante8/coste1/orden0|237|232|válido|
|practice_0119|independent|E3|donante8/coste0/orden1|258|232|válido|
|practice_0119|independent|E3|donante8/coste1/orden1|256|232|válido|
|practice_0119|independent|E3|donante9/coste0/orden0|256|234|válido|
|practice_0119|independent|E3|donante9/coste1/orden0|238|234|válido|
|practice_0119|independent|E3|donante9/coste0/orden1|296|234|válido|
|practice_0119|independent|E3|donante9/coste1/orden1|250|234|válido|
|practice_0120|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0120|shared|Cruce|CP True/coste0/orden0|145|145|válido|
|practice_0120|shared|Cruce|CP True/coste1/orden0|145|145|válido|
|practice_0120|shared|Cruce|CP True/coste0/orden1|145|145|válido|
|practice_0120|shared|Cruce|CP True/coste1/orden1|145|145|válido|
|practice_0120|shared|Cruce|mejor independiente/coste0/orden0|146|146|válido|
|practice_0120|shared|Cruce|mejor independiente/coste1/orden0|146|146|válido|
|practice_0120|shared|Cruce|mejor independiente/coste0/orden1|146|146|válido|
|practice_0120|shared|Cruce|mejor independiente/coste1/orden1|146|146|válido|
|practice_0120|shared|E1|cola10/pol1/inicio0|154|45|válido|
|practice_0120|shared|E1|cola10/pol1/inicio1|154|45|válido|
|practice_0120|shared|E1|cola10/pol2/inicio0|154|45|válido|
|practice_0120|shared|E1|cola10/pol2/inicio1|154|45|válido|
|practice_0120|shared|E1|cola8/pol1/inicio0|155|36|válido|
|practice_0120|shared|E1|cola8/pol1/inicio1|155|36|válido|
|practice_0120|shared|E1|cola8/pol2/inicio0|155|36|válido|
|practice_0120|shared|E1|cola8/pol2/inicio1|155|36|válido|
|practice_0120|shared|E1|cola6/pol1/inicio0|156|30|válido|
|practice_0120|shared|E1|cola6/pol1/inicio1|156|30|válido|
|practice_0120|shared|E1|cola6/pol2/inicio0|156|30|válido|
|practice_0120|shared|E1|cola6/pol2/inicio1|156|30|válido|
|practice_0120|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0120|shared|E2|CP False/coste0/orden0|156|156|válido|
|practice_0120|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0120|shared|E2|CP True/coste0/orden0|146|146|válido|
|practice_0120|shared|E3|donante0/coste0/orden0|149|149|válido|
|practice_0120|shared|E3|donante0/coste1/orden0|149|149|válido|
|practice_0120|shared|E3|donante0/coste0/orden1|149|149|válido|
|practice_0120|shared|E3|donante0/coste1/orden1|149|149|válido|
|practice_0120|shared|E3|donante1/coste0/orden0|167|167|válido|
|practice_0120|shared|E3|donante1/coste1/orden0|167|167|válido|
|practice_0120|shared|E3|donante1/coste0/orden1|167|167|válido|
|practice_0120|shared|E3|donante1/coste1/orden1|167|167|válido|
|practice_0120|shared|E3|donante2/coste0/orden0|158|158|válido|
|practice_0120|shared|E3|donante2/coste1/orden0|158|158|válido|
|practice_0120|shared|E3|donante2/coste0/orden1|158|158|válido|
|practice_0120|shared|E3|donante2/coste1/orden1|158|158|válido|
|practice_0120|shared|E3|donante3/coste0/orden0|171|171|válido|
|practice_0120|shared|E3|donante3/coste1/orden0|171|171|válido|
|practice_0120|shared|E3|donante3/coste0/orden1|171|171|válido|
|practice_0120|shared|E3|donante3/coste1/orden1|171|171|válido|
|practice_0120|shared|E3|donante4/coste0/orden0|161|161|válido|
|practice_0120|shared|E3|donante4/coste1/orden0|161|161|válido|
|practice_0120|shared|E3|donante4/coste0/orden1|161|161|válido|
|practice_0120|shared|E3|donante4/coste1/orden1|161|161|válido|
|practice_0120|shared|E3|donante5/coste0/orden0|164|164|válido|
|practice_0120|shared|E3|donante5/coste1/orden0|164|164|válido|
|practice_0120|shared|E3|donante5/coste0/orden1|164|164|válido|
|practice_0120|shared|E3|donante5/coste1/orden1|164|164|válido|
|practice_0120|shared|E3|donante6/coste0/orden0|177|155|válido|
|practice_0120|shared|E3|donante6/coste1/orden0|177|155|válido|
|practice_0120|shared|E3|donante6/coste0/orden1|155|155|válido|
|practice_0120|shared|E3|donante6/coste1/orden1|155|155|válido|
|practice_0120|shared|E3|donante7/coste0/orden0|184|154|válido|
|practice_0120|shared|E3|donante7/coste1/orden0|174|154|válido|
|practice_0120|shared|E3|donante7/coste0/orden1|154|154|válido|
|practice_0120|shared|E3|donante7/coste1/orden1|154|154|válido|
|practice_0120|shared|E3|donante8/coste0/orden0|191|152|válido|
|practice_0120|shared|E3|donante8/coste1/orden0|179|152|válido|
|practice_0120|shared|E3|donante8/coste0/orden1|171|152|válido|
|practice_0120|shared|E3|donante8/coste1/orden1|163|152|válido|
|practice_0121|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0121|independent|Cruce|CP True/coste0/orden0|376|246|válido|
|practice_0121|independent|Cruce|CP True/coste1/orden0|340|246|válido|
|practice_0121|independent|Cruce|CP True/coste0/orden1|247|246|válido|
|practice_0121|independent|Cruce|CP True/coste1/orden1|247|246|válido|
|practice_0121|independent|Cruce|mejor independiente/coste0/orden0|378|246|válido|
|practice_0121|independent|Cruce|mejor independiente/coste1/orden0|341|246|válido|
|practice_0121|independent|Cruce|mejor independiente/coste0/orden1|247|246|válido|
|practice_0121|independent|Cruce|mejor independiente/coste1/orden1|247|246|válido|
|practice_0121|independent|E1|cola10/pol1/inicio0|259|76|válido|
|practice_0121|independent|E1|cola10/pol1/inicio1|261|78|válido|
|practice_0121|independent|E1|cola10/pol2/inicio0|256|73|válido|
|practice_0121|independent|E1|cola10/pol2/inicio1|260|77|válido|
|practice_0121|independent|E1|cola8/pol1/inicio0|255|60|válido|
|practice_0121|independent|E1|cola8/pol1/inicio1|258|63|válido|
|practice_0121|independent|E1|cola8/pol2/inicio0|255|60|válido|
|practice_0121|independent|E1|cola8/pol2/inicio1|258|63|válido|
|practice_0121|independent|E1|cola6/pol1/inicio0|260|51|válido|
|practice_0121|independent|E1|cola6/pol1/inicio1|260|51|válido|
|practice_0121|independent|E1|cola6/pol2/inicio0|257|48|válido|
|practice_0121|independent|E1|cola6/pol2/inicio1|257|48|válido|
|practice_0121|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0121|independent|E2|CP False/coste0/orden0|481|280|válido|
|practice_0121|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0121|independent|E2|CP True/coste0/orden0|376|246|válido|
|practice_0121|independent|E3|donante0/coste0/orden0|246|246|válido|
|practice_0121|independent|E3|donante0/coste1/orden0|246|246|válido|
|practice_0121|independent|E3|donante0/coste0/orden1|247|246|válido|
|practice_0121|independent|E3|donante0/coste1/orden1|247|246|válido|
|practice_0121|independent|E3|donante1/coste0/orden0|280|280|válido|
|practice_0121|independent|E3|donante1/coste1/orden0|280|280|válido|
|practice_0121|independent|E3|donante1/coste0/orden1|280|280|válido|
|practice_0121|independent|E3|donante1/coste1/orden1|280|280|válido|
|practice_0121|independent|E3|donante2/coste0/orden0|318|266|válido|
|practice_0121|independent|E3|donante2/coste1/orden0|288|266|válido|
|practice_0121|independent|E3|donante2/coste0/orden1|266|266|válido|
|practice_0121|independent|E3|donante2/coste1/orden1|266|266|válido|
|practice_0121|independent|E3|donante3/coste0/orden0|333|258|válido|
|practice_0121|independent|E3|donante3/coste1/orden0|307|258|válido|
|practice_0121|independent|E3|donante3/coste0/orden1|270|258|válido|
|practice_0121|independent|E3|donante3/coste1/orden1|270|258|válido|
|practice_0121|independent|E3|donante4/coste0/orden0|283|283|válido|
|practice_0121|independent|E3|donante4/coste1/orden0|283|283|válido|
|practice_0121|independent|E3|donante4/coste0/orden1|283|283|válido|
|practice_0121|independent|E3|donante4/coste1/orden1|283|283|válido|
|practice_0121|independent|E3|donante5/coste0/orden0|280|280|válido|
|practice_0121|independent|E3|donante5/coste1/orden0|280|280|válido|
|practice_0121|independent|E3|donante5/coste0/orden1|280|280|válido|
|practice_0121|independent|E3|donante5/coste1/orden1|280|280|válido|
|practice_0121|independent|E3|donante6/coste0/orden0|291|291|válido|
|practice_0121|independent|E3|donante6/coste1/orden0|291|291|válido|
|practice_0121|independent|E3|donante6/coste0/orden1|291|291|válido|
|practice_0121|independent|E3|donante6/coste1/orden1|291|291|válido|
|practice_0121|independent|E3|donante7/coste0/orden0|265|253|válido|
|practice_0121|independent|E3|donante7/coste1/orden0|263|253|válido|
|practice_0121|independent|E3|donante7/coste0/orden1|264|253|válido|
|practice_0121|independent|E3|donante7/coste1/orden1|265|253|válido|
|practice_0121|independent|E3|donante8/coste0/orden0|258|245|válido|
|practice_0121|independent|E3|donante8/coste1/orden0|257|245|válido|
|practice_0121|independent|E3|donante8/coste0/orden1|292|245|válido|
|practice_0121|independent|E3|donante8/coste1/orden1|266|245|válido|
|practice_0122|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0122|independent|Cruce|CP True/coste0/orden0|339|271|válido|
|practice_0122|independent|Cruce|CP True/coste1/orden0|338|271|válido|
|practice_0122|independent|Cruce|CP True/coste0/orden1|271|271|válido|
|practice_0122|independent|Cruce|CP True/coste1/orden1|271|271|válido|
|practice_0122|independent|Cruce|mejor independiente/coste0/orden0|339|271|válido|
|practice_0122|independent|Cruce|mejor independiente/coste1/orden0|338|271|válido|
|practice_0122|independent|Cruce|mejor independiente/coste0/orden1|271|271|válido|
|practice_0122|independent|Cruce|mejor independiente/coste1/orden1|271|271|válido|
|practice_0122|independent|E1|cola10/pol1/inicio0|292|95|válido|
|practice_0122|independent|E1|cola10/pol1/inicio1|290|93|válido|
|practice_0122|independent|E1|cola10/pol2/inicio0|285|88|válido|
|practice_0122|independent|E1|cola10/pol2/inicio1|283|86|válido|
|practice_0122|independent|E1|cola8/pol1/inicio0|280|65|válido|
|practice_0122|independent|E1|cola8/pol1/inicio1|280|65|válido|
|practice_0122|independent|E1|cola8/pol2/inicio0|280|65|válido|
|practice_0122|independent|E1|cola8/pol2/inicio1|280|65|válido|
|practice_0122|independent|E1|cola6/pol1/inicio0|283|52|válido|
|practice_0122|independent|E1|cola6/pol1/inicio1|283|52|válido|
|practice_0122|independent|E1|cola6/pol2/inicio0|283|52|válido|
|practice_0122|independent|E1|cola6/pol2/inicio1|283|52|válido|
|practice_0122|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0122|independent|E2|CP False/coste0/orden0|418|306|válido|
|practice_0122|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0122|independent|E2|CP True/coste0/orden0|339|271|válido|
|practice_0122|independent|E3|donante0/coste0/orden0|271|271|válido|
|practice_0122|independent|E3|donante0/coste1/orden0|271|271|válido|
|practice_0122|independent|E3|donante0/coste0/orden1|271|271|válido|
|practice_0122|independent|E3|donante0/coste1/orden1|271|271|válido|
|practice_0122|independent|E3|donante1/coste0/orden0|319|319|válido|
|practice_0122|independent|E3|donante1/coste1/orden0|319|319|válido|
|practice_0122|independent|E3|donante1/coste0/orden1|319|319|válido|
|practice_0122|independent|E3|donante1/coste1/orden1|319|319|válido|
|practice_0122|independent|E3|donante2/coste0/orden0|410|299|válido|
|practice_0122|independent|E3|donante2/coste1/orden0|311|299|válido|
|practice_0122|independent|E3|donante2/coste0/orden1|299|299|válido|
|practice_0122|independent|E3|donante2/coste1/orden1|299|299|válido|
|practice_0122|independent|E3|donante3/coste0/orden0|475|292|válido|
|practice_0122|independent|E3|donante3/coste1/orden0|396|292|válido|
|practice_0122|independent|E3|donante3/coste0/orden1|356|292|válido|
|practice_0122|independent|E3|donante3/coste1/orden1|300|292|válido|
|practice_0122|independent|E3|donante4/coste0/orden0|324|324|válido|
|practice_0122|independent|E3|donante4/coste1/orden0|324|324|válido|
|practice_0122|independent|E3|donante4/coste0/orden1|324|324|válido|
|practice_0122|independent|E3|donante4/coste1/orden1|324|324|válido|
|practice_0122|independent|E3|donante5/coste0/orden0|326|326|válido|
|practice_0122|independent|E3|donante5/coste1/orden0|326|326|válido|
|practice_0122|independent|E3|donante5/coste0/orden1|326|326|válido|
|practice_0122|independent|E3|donante5/coste1/orden1|326|326|válido|
|practice_0122|independent|E3|donante6/coste0/orden0|327|327|válido|
|practice_0122|independent|E3|donante6/coste1/orden0|327|327|válido|
|practice_0122|independent|E3|donante6/coste0/orden1|327|327|válido|
|practice_0122|independent|E3|donante6/coste1/orden1|327|327|válido|
|practice_0122|independent|E3|donante7/coste0/orden0|374|293|válido|
|practice_0122|independent|E3|donante7/coste1/orden0|345|293|válido|
|practice_0122|independent|E3|donante7/coste0/orden1|334|293|válido|
|practice_0122|independent|E3|donante7/coste1/orden1|351|293|válido|
|practice_0122|independent|E3|donante8/coste0/orden0|293|281|válido|
|practice_0122|independent|E3|donante8/coste1/orden0|293|281|válido|
|practice_0122|independent|E3|donante8/coste0/orden1|330|281|válido|
|practice_0122|independent|E3|donante8/coste1/orden1|308|281|válido|
|practice_0122|independent|E3|donante9/coste0/orden0|273|273|válido|
|practice_0122|independent|E3|donante9/coste1/orden0|273|273|válido|
|practice_0122|independent|E3|donante9/coste0/orden1|273|273|válido|
|practice_0122|independent|E3|donante9/coste1/orden1|273|273|válido|
|practice_0123|tiny|Cruce|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0123|tiny|Cruce|CP True/coste0/orden0|2|2|válido|
|practice_0123|tiny|Cruce|CP True/coste1/orden0|2|2|válido|
|practice_0123|tiny|Cruce|CP True/coste0/orden1|2|2|válido|
|practice_0123|tiny|Cruce|CP True/coste1/orden1|2|2|válido|
|practice_0123|tiny|Cruce|mejor independiente/coste0/orden0|2|2|válido|
|practice_0123|tiny|Cruce|mejor independiente/coste1/orden0|2|2|válido|
|practice_0123|tiny|Cruce|mejor independiente/coste0/orden1|2|2|válido|
|practice_0123|tiny|Cruce|mejor independiente/coste1/orden1|2|2|válido|
|practice_0123|tiny|E1|cola1/pol1/inicio0|3|2|válido|
|practice_0123|tiny|E1|cola1/pol2/inicio0|3|2|válido|
|practice_0123|tiny|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0123|tiny|E2|CP False/coste0/orden0|2|2|válido|
|practice_0123|tiny|E2|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0123|tiny|E2|CP True/coste0/orden0|2|2|válido|
|practice_0123|tiny|E3|donante0/coste0/orden0|2|2|válido|
|practice_0123|tiny|E3|donante0/coste1/orden0|2|2|válido|
|practice_0123|tiny|E3|donante0/coste0/orden1|2|2|válido|
|practice_0123|tiny|E3|donante0/coste1/orden1|2|2|válido|
|practice_0123|tiny|E3|donante1/coste0/orden0|3|3|válido|
|practice_0123|tiny|E3|donante1/coste1/orden0|3|3|válido|
|practice_0123|tiny|E3|donante1/coste0/orden1|3|3|válido|
|practice_0123|tiny|E3|donante1/coste1/orden1|3|3|válido|
|practice_0124|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0124|mixed|Cruce|CP True/coste0/orden0|212|196|válido|
|practice_0124|mixed|Cruce|CP True/coste1/orden0|198|196|válido|
|practice_0124|mixed|Cruce|CP True/coste0/orden1|196|196|válido|
|practice_0124|mixed|Cruce|CP True/coste1/orden1|196|196|válido|
|practice_0124|mixed|Cruce|mejor independiente/coste0/orden0|220|199|válido|
|practice_0124|mixed|Cruce|mejor independiente/coste1/orden0|228|199|válido|
|practice_0124|mixed|Cruce|mejor independiente/coste0/orden1|213|199|válido|
|practice_0124|mixed|Cruce|mejor independiente/coste1/orden1|201|199|válido|
|practice_0124|mixed|E1|cola10/pol1/inicio0|212|70|válido|
|practice_0124|mixed|E1|cola10/pol1/inicio1|212|70|válido|
|practice_0124|mixed|E1|cola10/pol2/inicio0|208|66|válido|
|practice_0124|mixed|E1|cola10/pol2/inicio1|208|66|válido|
|practice_0124|mixed|E1|cola8/pol1/inicio0|210|55|válido|
|practice_0124|mixed|E1|cola8/pol1/inicio1|210|55|válido|
|practice_0124|mixed|E1|cola8/pol2/inicio0|204|49|válido|
|practice_0124|mixed|E1|cola8/pol2/inicio1|204|49|válido|
|practice_0124|mixed|E1|cola6/pol1/inicio0|207|37|válido|
|practice_0124|mixed|E1|cola6/pol1/inicio1|207|37|válido|
|practice_0124|mixed|E1|cola6/pol2/inicio0|207|37|válido|
|practice_0124|mixed|E1|cola6/pol2/inicio1|207|37|válido|
|practice_0124|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0124|mixed|E2|CP False/coste0/orden0|223|202|válido|
|practice_0124|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0124|mixed|E2|CP True/coste0/orden0|217|199|válido|
|practice_0124|mixed|E3|donante0/coste0/orden0|207|202|válido|
|practice_0124|mixed|E3|donante0/coste1/orden0|207|202|válido|
|practice_0124|mixed|E3|donante0/coste0/orden1|218|202|válido|
|practice_0124|mixed|E3|donante0/coste1/orden1|211|202|válido|
|practice_0124|mixed|E3|donante1/coste0/orden0|226|226|válido|
|practice_0124|mixed|E3|donante1/coste1/orden0|226|226|válido|
|practice_0124|mixed|E3|donante1/coste0/orden1|226|226|válido|
|practice_0124|mixed|E3|donante1/coste1/orden1|226|226|válido|
|practice_0124|mixed|E3|donante2/coste0/orden0|216|216|válido|
|practice_0124|mixed|E3|donante2/coste1/orden0|216|216|válido|
|practice_0124|mixed|E3|donante2/coste0/orden1|216|216|válido|
|practice_0124|mixed|E3|donante2/coste1/orden1|216|216|válido|
|practice_0124|mixed|E3|donante3/coste0/orden0|211|211|válido|
|practice_0124|mixed|E3|donante3/coste1/orden0|211|211|válido|
|practice_0124|mixed|E3|donante3/coste0/orden1|211|211|válido|
|practice_0124|mixed|E3|donante3/coste1/orden1|211|211|válido|
|practice_0124|mixed|E3|donante4/coste0/orden0|227|227|válido|
|practice_0124|mixed|E3|donante4/coste1/orden0|227|227|válido|
|practice_0124|mixed|E3|donante4/coste0/orden1|227|227|válido|
|practice_0124|mixed|E3|donante4/coste1/orden1|227|227|válido|
|practice_0124|mixed|E3|donante5/coste0/orden0|223|223|válido|
|practice_0124|mixed|E3|donante5/coste1/orden0|223|223|válido|
|practice_0124|mixed|E3|donante5/coste0/orden1|223|223|válido|
|practice_0124|mixed|E3|donante5/coste1/orden1|223|223|válido|
|practice_0124|mixed|E3|donante6/coste0/orden0|222|222|válido|
|practice_0124|mixed|E3|donante6/coste1/orden0|222|222|válido|
|practice_0124|mixed|E3|donante6/coste0/orden1|222|222|válido|
|practice_0124|mixed|E3|donante6/coste1/orden1|222|222|válido|
|practice_0124|mixed|E3|donante7/coste0/orden0|224|205|válido|
|practice_0124|mixed|E3|donante7/coste1/orden0|233|205|válido|
|practice_0124|mixed|E3|donante7/coste0/orden1|280|205|válido|
|practice_0124|mixed|E3|donante7/coste1/orden1|249|205|válido|
|practice_0124|mixed|E3|donante8/coste0/orden0|206|202|válido|
|practice_0124|mixed|E3|donante8/coste1/orden0|206|202|válido|
|practice_0124|mixed|E3|donante8/coste0/orden1|218|202|válido|
|practice_0124|mixed|E3|donante8/coste1/orden1|212|202|válido|
|practice_0124|mixed|E3|donante9/coste0/orden0|260|199|válido|
|practice_0124|mixed|E3|donante9/coste1/orden0|209|199|válido|
|practice_0124|mixed|E3|donante9/coste0/orden1|213|199|válido|
|practice_0124|mixed|E3|donante9/coste1/orden1|201|199|válido|
|practice_0125|tiny|Cruce|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0125|tiny|Cruce|CP True/coste0/orden0|3|3|válido|
|practice_0125|tiny|Cruce|CP True/coste1/orden0|3|3|válido|
|practice_0125|tiny|Cruce|CP True/coste0/orden1|3|3|válido|
|practice_0125|tiny|Cruce|CP True/coste1/orden1|3|3|válido|
|practice_0125|tiny|Cruce|mejor independiente/coste0/orden0|3|3|válido|
|practice_0125|tiny|Cruce|mejor independiente/coste1/orden0|3|3|válido|
|practice_0125|tiny|Cruce|mejor independiente/coste0/orden1|3|3|válido|
|practice_0125|tiny|Cruce|mejor independiente/coste1/orden1|3|3|válido|
|practice_0125|tiny|E1|cola2/pol1/inicio0|4|3|válido|
|practice_0125|tiny|E1|cola2/pol1/inicio1|4|3|válido|
|practice_0125|tiny|E1|cola2/pol2/inicio0|4|3|válido|
|practice_0125|tiny|E1|cola2/pol2/inicio1|4|3|válido|
|practice_0125|tiny|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0125|tiny|E2|CP False/coste0/orden0|3|3|válido|
|practice_0125|tiny|E2|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0125|tiny|E2|CP True/coste0/orden0|3|3|válido|
|practice_0125|tiny|E3|donante0/coste0/orden0|3|3|válido|
|practice_0125|tiny|E3|donante0/coste1/orden0|3|3|válido|
|practice_0125|tiny|E3|donante0/coste0/orden1|3|3|válido|
|practice_0125|tiny|E3|donante0/coste1/orden1|3|3|válido|
|practice_0125|tiny|E3|donante1/coste0/orden0|4|4|válido|
|practice_0125|tiny|E3|donante1/coste1/orden0|4|4|válido|
|practice_0125|tiny|E3|donante1/coste0/orden1|4|4|válido|
|practice_0125|tiny|E3|donante1/coste1/orden1|4|4|válido|
|practice_0126|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0126|shared|Cruce|CP True/coste0/orden0|143|143|válido|
|practice_0126|shared|Cruce|CP True/coste1/orden0|143|143|válido|
|practice_0126|shared|Cruce|CP True/coste0/orden1|143|143|válido|
|practice_0126|shared|Cruce|CP True/coste1/orden1|143|143|válido|
|practice_0126|shared|Cruce|mejor independiente/coste0/orden0|144|144|válido|
|practice_0126|shared|Cruce|mejor independiente/coste1/orden0|144|144|válido|
|practice_0126|shared|Cruce|mejor independiente/coste0/orden1|144|144|válido|
|practice_0126|shared|Cruce|mejor independiente/coste1/orden1|144|144|válido|
|practice_0126|shared|E1|cola10/pol1/inicio0|157|52|válido|
|practice_0126|shared|E1|cola10/pol1/inicio1|157|52|válido|
|practice_0126|shared|E1|cola10/pol2/inicio0|157|52|válido|
|practice_0126|shared|E1|cola10/pol2/inicio1|157|52|válido|
|practice_0126|shared|E1|cola8/pol1/inicio0|153|41|válido|
|practice_0126|shared|E1|cola8/pol1/inicio1|153|41|válido|
|practice_0126|shared|E1|cola8/pol2/inicio0|153|41|válido|
|practice_0126|shared|E1|cola8/pol2/inicio1|153|41|válido|
|practice_0126|shared|E1|cola6/pol1/inicio0|152|31|válido|
|practice_0126|shared|E1|cola6/pol1/inicio1|152|31|válido|
|practice_0126|shared|E1|cola6/pol2/inicio0|152|31|válido|
|practice_0126|shared|E1|cola6/pol2/inicio1|152|31|válido|
|practice_0126|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0126|shared|E2|CP False/coste0/orden0|151|151|válido|
|practice_0126|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0126|shared|E2|CP True/coste0/orden0|144|144|válido|
|practice_0126|shared|E3|donante0/coste0/orden0|147|147|válido|
|practice_0126|shared|E3|donante0/coste1/orden0|147|147|válido|
|practice_0126|shared|E3|donante0/coste0/orden1|147|147|válido|
|practice_0126|shared|E3|donante0/coste1/orden1|147|147|válido|
|practice_0126|shared|E3|donante1/coste0/orden0|156|156|válido|
|practice_0126|shared|E3|donante1/coste1/orden0|156|156|válido|
|practice_0126|shared|E3|donante1/coste0/orden1|156|156|válido|
|practice_0126|shared|E3|donante1/coste1/orden1|156|156|válido|
|practice_0126|shared|E3|donante2/coste0/orden0|155|155|válido|
|practice_0126|shared|E3|donante2/coste1/orden0|155|155|válido|
|practice_0126|shared|E3|donante2/coste0/orden1|155|155|válido|
|practice_0126|shared|E3|donante2/coste1/orden1|155|155|válido|
|practice_0126|shared|E3|donante3/coste0/orden0|155|155|válido|
|practice_0126|shared|E3|donante3/coste1/orden0|155|155|válido|
|practice_0126|shared|E3|donante3/coste0/orden1|155|155|válido|
|practice_0126|shared|E3|donante3/coste1/orden1|155|155|válido|
|practice_0126|shared|E3|donante4/coste0/orden0|179|179|válido|
|practice_0126|shared|E3|donante4/coste1/orden0|179|179|válido|
|practice_0126|shared|E3|donante4/coste0/orden1|179|179|válido|
|practice_0126|shared|E3|donante4/coste1/orden1|179|179|válido|
|practice_0126|shared|E3|donante5/coste0/orden0|159|159|válido|
|practice_0126|shared|E3|donante5/coste1/orden0|159|159|válido|
|practice_0126|shared|E3|donante5/coste0/orden1|159|159|válido|
|practice_0126|shared|E3|donante5/coste1/orden1|159|159|válido|
|practice_0126|shared|E3|donante6/coste0/orden0|167|167|válido|
|practice_0126|shared|E3|donante6/coste1/orden0|167|167|válido|
|practice_0126|shared|E3|donante6/coste0/orden1|167|167|válido|
|practice_0126|shared|E3|donante6/coste1/orden1|167|167|válido|
|practice_0126|shared|E3|donante7/coste0/orden0|174|158|válido|
|practice_0126|shared|E3|donante7/coste1/orden0|161|158|válido|
|practice_0126|shared|E3|donante7/coste0/orden1|170|158|válido|
|practice_0126|shared|E3|donante7/coste1/orden1|161|158|válido|
|practice_0126|shared|E3|donante8/coste0/orden0|154|154|válido|
|practice_0126|shared|E3|donante8/coste1/orden0|154|154|válido|
|practice_0126|shared|E3|donante8/coste0/orden1|154|154|válido|
|practice_0126|shared|E3|donante8/coste1/orden1|154|154|válido|
|practice_0126|shared|E3|donante9/coste0/orden0|148|148|válido|
|practice_0126|shared|E3|donante9/coste1/orden0|148|148|válido|
|practice_0126|shared|E3|donante9/coste0/orden1|148|148|válido|
|practice_0126|shared|E3|donante9/coste1/orden1|148|148|válido|
|practice_0127|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0127|mixed|Cruce|CP True/coste0/orden0|174|174|válido|
|practice_0127|mixed|Cruce|CP True/coste1/orden0|174|174|válido|
|practice_0127|mixed|Cruce|CP True/coste0/orden1|176|174|válido|
|practice_0127|mixed|Cruce|CP True/coste1/orden1|176|174|válido|
|practice_0127|mixed|Cruce|mejor independiente/coste0/orden0|176|176|válido|
|practice_0127|mixed|Cruce|mejor independiente/coste1/orden0|176|176|válido|
|practice_0127|mixed|Cruce|mejor independiente/coste0/orden1|178|176|válido|
|practice_0127|mixed|Cruce|mejor independiente/coste1/orden1|184|176|válido|
|practice_0127|mixed|E1|cola10/pol1/inicio0|187|61|válido|
|practice_0127|mixed|E1|cola10/pol1/inicio1|187|61|válido|
|practice_0127|mixed|E1|cola10/pol2/inicio0|180|54|válido|
|practice_0127|mixed|E1|cola10/pol2/inicio1|180|54|válido|
|practice_0127|mixed|E1|cola8/pol1/inicio0|181|44|válido|
|practice_0127|mixed|E1|cola8/pol1/inicio1|181|44|válido|
|practice_0127|mixed|E1|cola8/pol2/inicio0|180|43|válido|
|practice_0127|mixed|E1|cola8/pol2/inicio1|180|43|válido|
|practice_0127|mixed|E1|cola6/pol1/inicio0|181|34|válido|
|practice_0127|mixed|E1|cola6/pol1/inicio1|181|34|válido|
|practice_0127|mixed|E1|cola6/pol2/inicio0|182|35|válido|
|practice_0127|mixed|E1|cola6/pol2/inicio1|182|35|válido|
|practice_0127|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0127|mixed|E2|CP False/coste0/orden0|197|197|válido|
|practice_0127|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0127|mixed|E2|CP True/coste0/orden0|189|173|válido|
|practice_0127|mixed|E3|donante0/coste0/orden0|179|177|válido|
|practice_0127|mixed|E3|donante0/coste1/orden0|179|177|válido|
|practice_0127|mixed|E3|donante0/coste0/orden1|183|177|válido|
|practice_0127|mixed|E3|donante0/coste1/orden1|189|177|válido|
|practice_0127|mixed|E3|donante1/coste0/orden0|196|196|válido|
|practice_0127|mixed|E3|donante1/coste1/orden0|196|196|válido|
|practice_0127|mixed|E3|donante1/coste0/orden1|196|196|válido|
|practice_0127|mixed|E3|donante1/coste1/orden1|196|196|válido|
|practice_0127|mixed|E3|donante2/coste0/orden0|196|196|válido|
|practice_0127|mixed|E3|donante2/coste1/orden0|196|196|válido|
|practice_0127|mixed|E3|donante2/coste0/orden1|196|196|válido|
|practice_0127|mixed|E3|donante2/coste1/orden1|196|196|válido|
|practice_0127|mixed|E3|donante3/coste0/orden0|199|183|válido|
|practice_0127|mixed|E3|donante3/coste1/orden0|189|183|válido|
|practice_0127|mixed|E3|donante3/coste0/orden1|183|183|válido|
|practice_0127|mixed|E3|donante3/coste1/orden1|183|183|válido|
|practice_0127|mixed|E3|donante4/coste0/orden0|200|200|válido|
|practice_0127|mixed|E3|donante4/coste1/orden0|200|200|válido|
|practice_0127|mixed|E3|donante4/coste0/orden1|200|200|válido|
|practice_0127|mixed|E3|donante4/coste1/orden1|200|200|válido|
|practice_0127|mixed|E3|donante5/coste0/orden0|199|199|válido|
|practice_0127|mixed|E3|donante5/coste1/orden0|199|199|válido|
|practice_0127|mixed|E3|donante5/coste0/orden1|199|199|válido|
|practice_0127|mixed|E3|donante5/coste1/orden1|199|199|válido|
|practice_0127|mixed|E3|donante6/coste0/orden0|197|197|válido|
|practice_0127|mixed|E3|donante6/coste1/orden0|197|197|válido|
|practice_0127|mixed|E3|donante6/coste0/orden1|197|197|válido|
|practice_0127|mixed|E3|donante6/coste1/orden1|197|197|válido|
|practice_0127|mixed|E3|donante7/coste0/orden0|193|181|válido|
|practice_0127|mixed|E3|donante7/coste1/orden0|193|181|válido|
|practice_0127|mixed|E3|donante7/coste0/orden1|190|181|válido|
|practice_0127|mixed|E3|donante7/coste1/orden1|190|181|válido|
|practice_0127|mixed|E3|donante8/coste0/orden0|183|180|válido|
|practice_0127|mixed|E3|donante8/coste1/orden0|185|180|válido|
|practice_0127|mixed|E3|donante8/coste0/orden1|180|180|válido|
|practice_0127|mixed|E3|donante8/coste1/orden1|180|180|válido|
|practice_0128|tiny|Cruce|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0128|tiny|Cruce|CP True/coste0/orden0|2|2|válido|
|practice_0128|tiny|Cruce|CP True/coste1/orden0|2|2|válido|
|practice_0128|tiny|Cruce|CP True/coste0/orden1|2|2|válido|
|practice_0128|tiny|Cruce|CP True/coste1/orden1|2|2|válido|
|practice_0128|tiny|Cruce|mejor independiente/coste0/orden0|2|2|válido|
|practice_0128|tiny|Cruce|mejor independiente/coste1/orden0|2|2|válido|
|practice_0128|tiny|Cruce|mejor independiente/coste0/orden1|2|2|válido|
|practice_0128|tiny|Cruce|mejor independiente/coste1/orden1|2|2|válido|
|practice_0128|tiny|E1|cola1/pol1/inicio0|4|3|válido|
|practice_0128|tiny|E1|cola1/pol2/inicio0|4|3|válido|
|practice_0128|tiny|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0128|tiny|E2|CP False/coste0/orden0|2|2|válido|
|practice_0128|tiny|E2|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0128|tiny|E2|CP True/coste0/orden0|2|2|válido|
|practice_0128|tiny|E3|donante0/coste0/orden0|2|2|válido|
|practice_0128|tiny|E3|donante0/coste1/orden0|2|2|válido|
|practice_0128|tiny|E3|donante0/coste0/orden1|2|2|válido|
|practice_0128|tiny|E3|donante0/coste1/orden1|2|2|válido|
|practice_0128|tiny|E3|donante1/coste0/orden0|4|4|válido|
|practice_0128|tiny|E3|donante1/coste1/orden0|4|4|válido|
|practice_0128|tiny|E3|donante1/coste0/orden1|4|4|válido|
|practice_0128|tiny|E3|donante1/coste1/orden1|4|4|válido|
|practice_0129|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0129|multiplicative|Cruce|CP True/coste0/orden0|116|116|válido|
|practice_0129|multiplicative|Cruce|CP True/coste1/orden0|116|116|válido|
|practice_0129|multiplicative|Cruce|CP True/coste0/orden1|116|116|válido|
|practice_0129|multiplicative|Cruce|CP True/coste1/orden1|116|116|válido|
|practice_0129|multiplicative|Cruce|mejor independiente/coste0/orden0|117|117|válido|
|practice_0129|multiplicative|Cruce|mejor independiente/coste1/orden0|117|117|válido|
|practice_0129|multiplicative|Cruce|mejor independiente/coste0/orden1|117|117|válido|
|practice_0129|multiplicative|Cruce|mejor independiente/coste1/orden1|117|117|válido|
|practice_0129|multiplicative|E1|cola10/pol1/inicio0|120|40|válido|
|practice_0129|multiplicative|E1|cola10/pol1/inicio1|120|40|válido|
|practice_0129|multiplicative|E1|cola10/pol2/inicio0|121|41|válido|
|practice_0129|multiplicative|E1|cola10/pol2/inicio1|121|41|válido|
|practice_0129|multiplicative|E1|cola8/pol1/inicio0|125|36|válido|
|practice_0129|multiplicative|E1|cola8/pol1/inicio1|125|36|válido|
|practice_0129|multiplicative|E1|cola8/pol2/inicio0|125|36|válido|
|practice_0129|multiplicative|E1|cola8/pol2/inicio1|125|36|válido|
|practice_0129|multiplicative|E1|cola6/pol1/inicio0|129|33|válido|
|practice_0129|multiplicative|E1|cola6/pol1/inicio1|129|33|válido|
|practice_0129|multiplicative|E1|cola6/pol2/inicio0|121|25|válido|
|practice_0129|multiplicative|E1|cola6/pol2/inicio1|121|25|válido|
|practice_0129|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0129|multiplicative|E2|CP False/coste0/orden0|123|123|válido|
|practice_0129|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0129|multiplicative|E2|CP True/coste0/orden0|117|117|válido|
|practice_0129|multiplicative|E3|donante0/coste0/orden0|119|119|válido|
|practice_0129|multiplicative|E3|donante0/coste1/orden0|119|119|válido|
|practice_0129|multiplicative|E3|donante0/coste0/orden1|119|119|válido|
|practice_0129|multiplicative|E3|donante0/coste1/orden1|119|119|válido|
|practice_0129|multiplicative|E3|donante1/coste0/orden0|125|125|válido|
|practice_0129|multiplicative|E3|donante1/coste1/orden0|125|125|válido|
|practice_0129|multiplicative|E3|donante1/coste0/orden1|125|125|válido|
|practice_0129|multiplicative|E3|donante1/coste1/orden1|125|125|válido|
|practice_0129|multiplicative|E3|donante2/coste0/orden0|124|124|válido|
|practice_0129|multiplicative|E3|donante2/coste1/orden0|124|124|válido|
|practice_0129|multiplicative|E3|donante2/coste0/orden1|124|124|válido|
|practice_0129|multiplicative|E3|donante2/coste1/orden1|124|124|válido|
|practice_0129|multiplicative|E3|donante3/coste0/orden0|122|122|válido|
|practice_0129|multiplicative|E3|donante3/coste1/orden0|122|122|válido|
|practice_0129|multiplicative|E3|donante3/coste0/orden1|122|122|válido|
|practice_0129|multiplicative|E3|donante3/coste1/orden1|122|122|válido|
|practice_0129|multiplicative|E3|donante4/coste0/orden0|132|132|válido|
|practice_0129|multiplicative|E3|donante4/coste1/orden0|132|132|válido|
|practice_0129|multiplicative|E3|donante4/coste0/orden1|132|132|válido|
|practice_0129|multiplicative|E3|donante4/coste1/orden1|132|132|válido|
|practice_0129|multiplicative|E3|donante5/coste0/orden0|128|128|válido|
|practice_0129|multiplicative|E3|donante5/coste1/orden0|128|128|válido|
|practice_0129|multiplicative|E3|donante5/coste0/orden1|128|128|válido|
|practice_0129|multiplicative|E3|donante5/coste1/orden1|128|128|válido|
|practice_0129|multiplicative|E3|donante6/coste0/orden0|128|128|válido|
|practice_0129|multiplicative|E3|donante6/coste1/orden0|128|128|válido|
|practice_0129|multiplicative|E3|donante6/coste0/orden1|128|128|válido|
|practice_0129|multiplicative|E3|donante6/coste1/orden1|128|128|válido|
|practice_0129|multiplicative|E3|donante7/coste0/orden0|124|124|válido|
|practice_0129|multiplicative|E3|donante7/coste1/orden0|124|124|válido|
|practice_0129|multiplicative|E3|donante7/coste0/orden1|124|124|válido|
|practice_0129|multiplicative|E3|donante7/coste1/orden1|124|124|válido|
|practice_0129|multiplicative|E3|donante8/coste0/orden0|120|120|válido|
|practice_0129|multiplicative|E3|donante8/coste1/orden0|120|120|válido|
|practice_0129|multiplicative|E3|donante8/coste0/orden1|120|120|válido|
|practice_0129|multiplicative|E3|donante8/coste1/orden1|120|120|válido|
|practice_0130|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0130|multiplicative|Cruce|CP True/coste0/orden0|104|104|válido|
|practice_0130|multiplicative|Cruce|CP True/coste1/orden0|104|104|válido|
|practice_0130|multiplicative|Cruce|CP True/coste0/orden1|104|104|válido|
|practice_0130|multiplicative|Cruce|CP True/coste1/orden1|104|104|válido|
|practice_0130|multiplicative|Cruce|mejor independiente/coste0/orden0|104|104|válido|
|practice_0130|multiplicative|Cruce|mejor independiente/coste1/orden0|104|104|válido|
|practice_0130|multiplicative|Cruce|mejor independiente/coste0/orden1|104|104|válido|
|practice_0130|multiplicative|Cruce|mejor independiente/coste1/orden1|104|104|válido|
|practice_0130|multiplicative|E1|cola10/pol1/inicio0|110|45|válido|
|practice_0130|multiplicative|E1|cola10/pol1/inicio1|110|45|válido|
|practice_0130|multiplicative|E1|cola10/pol2/inicio0|108|43|válido|
|practice_0130|multiplicative|E1|cola10/pol2/inicio1|108|43|válido|
|practice_0130|multiplicative|E1|cola8/pol1/inicio0|107|38|válido|
|practice_0130|multiplicative|E1|cola8/pol1/inicio1|107|38|válido|
|practice_0130|multiplicative|E1|cola8/pol2/inicio0|107|38|válido|
|practice_0130|multiplicative|E1|cola8/pol2/inicio1|107|38|válido|
|practice_0130|multiplicative|E1|cola6/pol1/inicio0|106|30|válido|
|practice_0130|multiplicative|E1|cola6/pol1/inicio1|106|30|válido|
|practice_0130|multiplicative|E1|cola6/pol2/inicio0|106|30|válido|
|practice_0130|multiplicative|E1|cola6/pol2/inicio1|106|30|válido|
|practice_0130|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0130|multiplicative|E2|CP False/coste0/orden0|106|106|válido|
|practice_0130|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0130|multiplicative|E2|CP True/coste0/orden0|104|104|válido|
|practice_0130|multiplicative|E3|donante0/coste0/orden0|106|106|válido|
|practice_0130|multiplicative|E3|donante0/coste1/orden0|106|106|válido|
|practice_0130|multiplicative|E3|donante0/coste0/orden1|106|106|válido|
|practice_0130|multiplicative|E3|donante0/coste1/orden1|106|106|válido|
|practice_0130|multiplicative|E3|donante1/coste0/orden0|111|111|válido|
|practice_0130|multiplicative|E3|donante1/coste1/orden0|111|111|válido|
|practice_0130|multiplicative|E3|donante1/coste0/orden1|111|111|válido|
|practice_0130|multiplicative|E3|donante1/coste1/orden1|111|111|válido|
|practice_0130|multiplicative|E3|donante2/coste0/orden0|112|112|válido|
|practice_0130|multiplicative|E3|donante2/coste1/orden0|112|112|válido|
|practice_0130|multiplicative|E3|donante2/coste0/orden1|112|112|válido|
|practice_0130|multiplicative|E3|donante2/coste1/orden1|112|112|válido|
|practice_0130|multiplicative|E3|donante3/coste0/orden0|107|107|válido|
|practice_0130|multiplicative|E3|donante3/coste1/orden0|107|107|válido|
|practice_0130|multiplicative|E3|donante3/coste0/orden1|107|107|válido|
|practice_0130|multiplicative|E3|donante3/coste1/orden1|107|107|válido|
|practice_0130|multiplicative|E3|donante4/coste0/orden0|118|118|válido|
|practice_0130|multiplicative|E3|donante4/coste1/orden0|118|118|válido|
|practice_0130|multiplicative|E3|donante4/coste0/orden1|118|118|válido|
|practice_0130|multiplicative|E3|donante4/coste1/orden1|118|118|válido|
|practice_0130|multiplicative|E3|donante5/coste0/orden0|115|115|válido|
|practice_0130|multiplicative|E3|donante5/coste1/orden0|115|115|válido|
|practice_0130|multiplicative|E3|donante5/coste0/orden1|115|115|válido|
|practice_0130|multiplicative|E3|donante5/coste1/orden1|115|115|válido|
|practice_0130|multiplicative|E3|donante6/coste0/orden0|108|108|válido|
|practice_0130|multiplicative|E3|donante6/coste1/orden0|108|108|válido|
|practice_0130|multiplicative|E3|donante6/coste0/orden1|108|108|válido|
|practice_0130|multiplicative|E3|donante6/coste1/orden1|108|108|válido|
|practice_0130|multiplicative|E3|donante7/coste0/orden0|111|111|válido|
|practice_0130|multiplicative|E3|donante7/coste1/orden0|111|111|válido|
|practice_0130|multiplicative|E3|donante7/coste0/orden1|111|111|válido|
|practice_0130|multiplicative|E3|donante7/coste1/orden1|111|111|válido|
|practice_0130|multiplicative|E3|donante8/coste0/orden0|108|108|válido|
|practice_0130|multiplicative|E3|donante8/coste1/orden0|108|108|válido|
|practice_0130|multiplicative|E3|donante8/coste0/orden1|108|108|válido|
|practice_0130|multiplicative|E3|donante8/coste1/orden1|108|108|válido|
|practice_0130|multiplicative|E3|donante9/coste0/orden0|109|109|válido|
|practice_0130|multiplicative|E3|donante9/coste1/orden0|109|109|válido|
|practice_0130|multiplicative|E3|donante9/coste0/orden1|109|109|válido|
|practice_0130|multiplicative|E3|donante9/coste1/orden1|109|109|válido|
|practice_0131|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0131|mixed|Cruce|CP True/coste0/orden0|201|201|válido|
|practice_0131|mixed|Cruce|CP True/coste1/orden0|201|201|válido|
|practice_0131|mixed|Cruce|CP True/coste0/orden1|201|201|válido|
|practice_0131|mixed|Cruce|CP True/coste1/orden1|201|201|válido|
|practice_0131|mixed|Cruce|mejor independiente/coste0/orden0|202|202|válido|
|practice_0131|mixed|Cruce|mejor independiente/coste1/orden0|202|202|válido|
|practice_0131|mixed|Cruce|mejor independiente/coste0/orden1|202|202|válido|
|practice_0131|mixed|Cruce|mejor independiente/coste1/orden1|202|202|válido|
|practice_0131|mixed|E1|cola10/pol1/inicio0|217|78|válido|
|practice_0131|mixed|E1|cola10/pol1/inicio1|217|78|válido|
|practice_0131|mixed|E1|cola10/pol2/inicio0|215|76|válido|
|practice_0131|mixed|E1|cola10/pol2/inicio1|215|76|válido|
|practice_0131|mixed|E1|cola8/pol1/inicio0|213|60|válido|
|practice_0131|mixed|E1|cola8/pol1/inicio1|213|60|válido|
|practice_0131|mixed|E1|cola8/pol2/inicio0|213|60|válido|
|practice_0131|mixed|E1|cola8/pol2/inicio1|213|60|válido|
|practice_0131|mixed|E1|cola6/pol1/inicio0|214|48|válido|
|practice_0131|mixed|E1|cola6/pol1/inicio1|214|48|válido|
|practice_0131|mixed|E1|cola6/pol2/inicio0|214|48|válido|
|practice_0131|mixed|E1|cola6/pol2/inicio1|214|48|válido|
|practice_0131|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0131|mixed|E2|CP False/coste0/orden0|224|224|válido|
|practice_0131|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0131|mixed|E2|CP True/coste0/orden0|202|202|válido|
|practice_0131|mixed|E3|donante0/coste0/orden0|202|202|válido|
|practice_0131|mixed|E3|donante0/coste1/orden0|202|202|válido|
|practice_0131|mixed|E3|donante0/coste0/orden1|202|202|válido|
|practice_0131|mixed|E3|donante0/coste1/orden1|202|202|válido|
|practice_0131|mixed|E3|donante1/coste0/orden0|228|228|válido|
|practice_0131|mixed|E3|donante1/coste1/orden0|228|228|válido|
|practice_0131|mixed|E3|donante1/coste0/orden1|228|228|válido|
|practice_0131|mixed|E3|donante1/coste1/orden1|228|228|válido|
|practice_0131|mixed|E3|donante2/coste0/orden0|214|214|válido|
|practice_0131|mixed|E3|donante2/coste1/orden0|214|214|válido|
|practice_0131|mixed|E3|donante2/coste0/orden1|214|214|válido|
|practice_0131|mixed|E3|donante2/coste1/orden1|214|214|válido|
|practice_0131|mixed|E3|donante3/coste0/orden0|212|212|válido|
|practice_0131|mixed|E3|donante3/coste1/orden0|212|212|válido|
|practice_0131|mixed|E3|donante3/coste0/orden1|212|212|válido|
|practice_0131|mixed|E3|donante3/coste1/orden1|212|212|válido|
|practice_0131|mixed|E3|donante4/coste0/orden0|231|231|válido|
|practice_0131|mixed|E3|donante4/coste1/orden0|231|231|válido|
|practice_0131|mixed|E3|donante4/coste0/orden1|231|231|válido|
|practice_0131|mixed|E3|donante4/coste1/orden1|231|231|válido|
|practice_0131|mixed|E3|donante5/coste0/orden0|224|224|válido|
|practice_0131|mixed|E3|donante5/coste1/orden0|224|224|válido|
|practice_0131|mixed|E3|donante5/coste0/orden1|224|224|válido|
|practice_0131|mixed|E3|donante5/coste1/orden1|224|224|válido|
|practice_0131|mixed|E3|donante6/coste0/orden0|229|229|válido|
|practice_0131|mixed|E3|donante6/coste1/orden0|229|229|válido|
|practice_0131|mixed|E3|donante6/coste0/orden1|229|229|válido|
|practice_0131|mixed|E3|donante6/coste1/orden1|229|229|válido|
|practice_0131|mixed|E3|donante7/coste0/orden0|234|208|válido|
|practice_0131|mixed|E3|donante7/coste1/orden0|234|208|válido|
|practice_0131|mixed|E3|donante7/coste0/orden1|208|208|válido|
|practice_0131|mixed|E3|donante7/coste1/orden1|208|208|válido|
|practice_0131|mixed|E3|donante8/coste0/orden0|225|198|válido|
|practice_0131|mixed|E3|donante8/coste1/orden0|225|198|válido|
|practice_0131|mixed|E3|donante8/coste0/orden1|220|198|válido|
|practice_0131|mixed|E3|donante8/coste1/orden1|220|198|válido|
|practice_0132|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0132|mixed|Cruce|CP True/coste0/orden0|163|162|válido|
|practice_0132|mixed|Cruce|CP True/coste1/orden0|163|162|válido|
|practice_0132|mixed|Cruce|CP True/coste0/orden1|167|162|válido|
|practice_0132|mixed|Cruce|CP True/coste1/orden1|165|162|válido|
|practice_0132|mixed|Cruce|mejor independiente/coste0/orden0|163|163|válido|
|practice_0132|mixed|Cruce|mejor independiente/coste1/orden0|163|163|válido|
|practice_0132|mixed|Cruce|mejor independiente/coste0/orden1|163|163|válido|
|practice_0132|mixed|Cruce|mejor independiente/coste1/orden1|163|163|válido|
|practice_0132|mixed|E1|cola10/pol1/inicio0|178|63|válido|
|practice_0132|mixed|E1|cola10/pol1/inicio1|178|63|válido|
|practice_0132|mixed|E1|cola10/pol2/inicio0|177|62|válido|
|practice_0132|mixed|E1|cola10/pol2/inicio1|177|62|válido|
|practice_0132|mixed|E1|cola8/pol1/inicio0|177|51|válido|
|practice_0132|mixed|E1|cola8/pol1/inicio1|177|51|válido|
|practice_0132|mixed|E1|cola8/pol2/inicio0|175|49|válido|
|practice_0132|mixed|E1|cola8/pol2/inicio1|175|49|válido|
|practice_0132|mixed|E1|cola6/pol1/inicio0|176|39|válido|
|practice_0132|mixed|E1|cola6/pol1/inicio1|176|39|válido|
|practice_0132|mixed|E1|cola6/pol2/inicio0|177|40|válido|
|practice_0132|mixed|E1|cola6/pol2/inicio1|177|40|válido|
|practice_0132|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0132|mixed|E2|CP False/coste0/orden0|209|176|válido|
|practice_0132|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0132|mixed|E2|CP True/coste0/orden0|163|163|válido|
|practice_0132|mixed|E3|donante0/coste0/orden0|169|166|válido|
|practice_0132|mixed|E3|donante0/coste1/orden0|169|166|válido|
|practice_0132|mixed|E3|donante0/coste0/orden1|166|166|válido|
|practice_0132|mixed|E3|donante0/coste1/orden1|166|166|válido|
|practice_0132|mixed|E3|donante1/coste0/orden0|179|179|válido|
|practice_0132|mixed|E3|donante1/coste1/orden0|179|179|válido|
|practice_0132|mixed|E3|donante1/coste0/orden1|179|179|válido|
|practice_0132|mixed|E3|donante1/coste1/orden1|179|179|válido|
|practice_0132|mixed|E3|donante2/coste0/orden0|182|175|válido|
|practice_0132|mixed|E3|donante2/coste1/orden0|182|175|válido|
|practice_0132|mixed|E3|donante2/coste0/orden1|175|175|válido|
|practice_0132|mixed|E3|donante2/coste1/orden1|175|175|válido|
|practice_0132|mixed|E3|donante3/coste0/orden0|170|170|válido|
|practice_0132|mixed|E3|donante3/coste1/orden0|170|170|válido|
|practice_0132|mixed|E3|donante3/coste0/orden1|170|170|válido|
|practice_0132|mixed|E3|donante3/coste1/orden1|170|170|válido|
|practice_0132|mixed|E3|donante4/coste0/orden0|186|186|válido|
|practice_0132|mixed|E3|donante4/coste1/orden0|186|186|válido|
|practice_0132|mixed|E3|donante4/coste0/orden1|186|186|válido|
|practice_0132|mixed|E3|donante4/coste1/orden1|186|186|válido|
|practice_0132|mixed|E3|donante5/coste0/orden0|183|183|válido|
|practice_0132|mixed|E3|donante5/coste1/orden0|183|183|válido|
|practice_0132|mixed|E3|donante5/coste0/orden1|183|183|válido|
|practice_0132|mixed|E3|donante5/coste1/orden1|183|183|válido|
|practice_0132|mixed|E3|donante6/coste0/orden0|180|180|válido|
|practice_0132|mixed|E3|donante6/coste1/orden0|180|180|válido|
|practice_0132|mixed|E3|donante6/coste0/orden1|180|180|válido|
|practice_0132|mixed|E3|donante6/coste1/orden1|180|180|válido|
|practice_0132|mixed|E3|donante7/coste0/orden0|191|166|válido|
|practice_0132|mixed|E3|donante7/coste1/orden0|203|166|válido|
|practice_0132|mixed|E3|donante7/coste0/orden1|166|166|válido|
|practice_0132|mixed|E3|donante7/coste1/orden1|166|166|válido|
|practice_0132|mixed|E3|donante8/coste0/orden0|182|163|válido|
|practice_0132|mixed|E3|donante8/coste1/orden0|166|163|válido|
|practice_0132|mixed|E3|donante8/coste0/orden1|163|163|válido|
|practice_0132|mixed|E3|donante8/coste1/orden1|163|163|válido|
|practice_0133|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0133|shared|Cruce|CP True/coste0/orden0|190|162|válido|
|practice_0133|shared|Cruce|CP True/coste1/orden0|180|162|válido|
|practice_0133|shared|Cruce|CP True/coste0/orden1|162|162|válido|
|practice_0133|shared|Cruce|CP True/coste1/orden1|162|162|válido|
|practice_0133|shared|Cruce|mejor independiente/coste0/orden0|196|162|válido|
|practice_0133|shared|Cruce|mejor independiente/coste1/orden0|182|162|válido|
|practice_0133|shared|Cruce|mejor independiente/coste0/orden1|162|162|válido|
|practice_0133|shared|Cruce|mejor independiente/coste1/orden1|162|162|válido|
|practice_0133|shared|E1|cola10/pol1/inicio0|173|66|válido|
|practice_0133|shared|E1|cola10/pol1/inicio1|173|66|válido|
|practice_0133|shared|E1|cola10/pol2/inicio0|171|64|válido|
|practice_0133|shared|E1|cola10/pol2/inicio1|171|64|válido|
|practice_0133|shared|E1|cola8/pol1/inicio0|173|57|válido|
|practice_0133|shared|E1|cola8/pol1/inicio1|173|57|válido|
|practice_0133|shared|E1|cola8/pol2/inicio0|173|57|válido|
|practice_0133|shared|E1|cola8/pol2/inicio1|173|57|válido|
|practice_0133|shared|E1|cola6/pol1/inicio0|172|44|válido|
|practice_0133|shared|E1|cola6/pol1/inicio1|172|44|válido|
|practice_0133|shared|E1|cola6/pol2/inicio0|172|44|válido|
|practice_0133|shared|E1|cola6/pol2/inicio1|172|44|válido|
|practice_0133|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0133|shared|E2|CP False/coste0/orden0|183|183|válido|
|practice_0133|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0133|shared|E2|CP True/coste0/orden0|190|162|válido|
|practice_0133|shared|E3|donante0/coste0/orden0|162|162|válido|
|practice_0133|shared|E3|donante0/coste1/orden0|162|162|válido|
|practice_0133|shared|E3|donante0/coste0/orden1|162|162|válido|
|practice_0133|shared|E3|donante0/coste1/orden1|162|162|válido|
|practice_0133|shared|E3|donante1/coste0/orden0|189|189|válido|
|practice_0133|shared|E3|donante1/coste1/orden0|189|189|válido|
|practice_0133|shared|E3|donante1/coste0/orden1|189|189|válido|
|practice_0133|shared|E3|donante1/coste1/orden1|189|189|válido|
|practice_0133|shared|E3|donante2/coste0/orden0|206|178|válido|
|practice_0133|shared|E3|donante2/coste1/orden0|199|178|válido|
|practice_0133|shared|E3|donante2/coste0/orden1|178|178|válido|
|practice_0133|shared|E3|donante2/coste1/orden1|178|178|válido|
|practice_0133|shared|E3|donante3/coste0/orden0|180|174|válido|
|practice_0133|shared|E3|donante3/coste1/orden0|178|174|válido|
|practice_0133|shared|E3|donante3/coste0/orden1|174|174|válido|
|practice_0133|shared|E3|donante3/coste1/orden1|174|174|válido|
|practice_0133|shared|E3|donante4/coste0/orden0|215|215|válido|
|practice_0133|shared|E3|donante4/coste1/orden0|215|215|válido|
|practice_0133|shared|E3|donante4/coste0/orden1|215|215|válido|
|practice_0133|shared|E3|donante4/coste1/orden1|215|215|válido|
|practice_0133|shared|E3|donante5/coste0/orden0|184|184|válido|
|practice_0133|shared|E3|donante5/coste1/orden0|184|184|válido|
|practice_0133|shared|E3|donante5/coste0/orden1|184|184|válido|
|practice_0133|shared|E3|donante5/coste1/orden1|184|184|válido|
|practice_0133|shared|E3|donante6/coste0/orden0|198|198|válido|
|practice_0133|shared|E3|donante6/coste1/orden0|198|198|válido|
|practice_0133|shared|E3|donante6/coste0/orden1|198|198|válido|
|practice_0133|shared|E3|donante6/coste1/orden1|198|198|válido|
|practice_0133|shared|E3|donante7/coste0/orden0|202|183|válido|
|practice_0133|shared|E3|donante7/coste1/orden0|196|183|válido|
|practice_0133|shared|E3|donante7/coste0/orden1|183|183|válido|
|practice_0133|shared|E3|donante7/coste1/orden1|183|183|válido|
|practice_0133|shared|E3|donante8/coste0/orden0|167|164|válido|
|practice_0133|shared|E3|donante8/coste1/orden0|170|164|válido|
|practice_0133|shared|E3|donante8/coste0/orden1|164|164|válido|
|practice_0133|shared|E3|donante8/coste1/orden1|164|164|válido|
|practice_0134|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0134|independent|Cruce|CP True/coste0/orden0|226|187|válido|
|practice_0134|independent|Cruce|CP True/coste1/orden0|196|187|válido|
|practice_0134|independent|Cruce|CP True/coste0/orden1|201|187|válido|
|practice_0134|independent|Cruce|CP True/coste1/orden1|203|187|válido|
|practice_0134|independent|Cruce|mejor independiente/coste0/orden0|228|187|válido|
|practice_0134|independent|Cruce|mejor independiente/coste1/orden0|200|187|válido|
|practice_0134|independent|Cruce|mejor independiente/coste0/orden1|201|187|válido|
|practice_0134|independent|Cruce|mejor independiente/coste1/orden1|203|187|válido|
|practice_0134|independent|E1|cola10/pol1/inicio0|214|56|válido|
|practice_0134|independent|E1|cola10/pol1/inicio1|214|56|válido|
|practice_0134|independent|E1|cola10/pol2/inicio0|213|55|válido|
|practice_0134|independent|E1|cola10/pol2/inicio1|213|55|válido|
|practice_0134|independent|E1|cola8/pol1/inicio0|214|47|válido|
|practice_0134|independent|E1|cola8/pol1/inicio1|214|47|válido|
|practice_0134|independent|E1|cola8/pol2/inicio0|213|46|válido|
|practice_0134|independent|E1|cola8/pol2/inicio1|213|46|válido|
|practice_0134|independent|E1|cola6/pol1/inicio0|217|34|válido|
|practice_0134|independent|E1|cola6/pol1/inicio1|217|34|válido|
|practice_0134|independent|E1|cola6/pol2/inicio0|217|34|válido|
|practice_0134|independent|E1|cola6/pol2/inicio1|217|34|válido|
|practice_0134|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0134|independent|E2|CP False/coste0/orden0|240|213|válido|
|practice_0134|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0134|independent|E2|CP True/coste0/orden0|208|208|válido|
|practice_0134|independent|E3|donante0/coste0/orden0|207|207|válido|
|practice_0134|independent|E3|donante0/coste1/orden0|207|207|válido|
|practice_0134|independent|E3|donante0/coste0/orden1|207|207|válido|
|practice_0134|independent|E3|donante0/coste1/orden1|207|207|válido|
|practice_0134|independent|E3|donante1/coste0/orden0|215|215|válido|
|practice_0134|independent|E3|donante1/coste1/orden0|215|215|válido|
|practice_0134|independent|E3|donante1/coste0/orden1|215|215|válido|
|practice_0134|independent|E3|donante1/coste1/orden1|215|215|válido|
|practice_0134|independent|E3|donante2/coste0/orden0|219|200|válido|
|practice_0134|independent|E3|donante2/coste1/orden0|218|200|válido|
|practice_0134|independent|E3|donante2/coste0/orden1|200|200|válido|
|practice_0134|independent|E3|donante2/coste1/orden1|200|200|válido|
|practice_0134|independent|E3|donante3/coste0/orden0|219|219|válido|
|practice_0134|independent|E3|donante3/coste1/orden0|219|219|válido|
|practice_0134|independent|E3|donante3/coste0/orden1|219|219|válido|
|practice_0134|independent|E3|donante3/coste1/orden1|219|219|válido|
|practice_0134|independent|E3|donante4/coste0/orden0|219|219|válido|
|practice_0134|independent|E3|donante4/coste1/orden0|219|219|válido|
|practice_0134|independent|E3|donante4/coste0/orden1|219|219|válido|
|practice_0134|independent|E3|donante4/coste1/orden1|219|219|válido|
|practice_0134|independent|E3|donante5/coste0/orden0|219|219|válido|
|practice_0134|independent|E3|donante5/coste1/orden0|219|219|válido|
|practice_0134|independent|E3|donante5/coste0/orden1|219|219|válido|
|practice_0134|independent|E3|donante5/coste1/orden1|219|219|válido|
|practice_0134|independent|E3|donante6/coste0/orden0|244|201|válido|
|practice_0134|independent|E3|donante6/coste1/orden0|225|201|válido|
|practice_0134|independent|E3|donante6/coste0/orden1|231|201|válido|
|practice_0134|independent|E3|donante6/coste1/orden1|226|201|válido|
|practice_0134|independent|E3|donante7/coste0/orden0|224|195|válido|
|practice_0134|independent|E3|donante7/coste1/orden0|220|195|válido|
|practice_0134|independent|E3|donante7/coste0/orden1|211|195|válido|
|practice_0134|independent|E3|donante7/coste1/orden1|210|195|válido|
|practice_0134|independent|E3|donante8/coste0/orden0|226|188|válido|
|practice_0134|independent|E3|donante8/coste1/orden0|215|188|válido|
|practice_0134|independent|E3|donante8/coste0/orden1|202|188|válido|
|practice_0134|independent|E3|donante8/coste1/orden1|204|188|válido|
|practice_0135|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0135|independent|Cruce|CP True/coste0/orden0|676|296|válido|
|practice_0135|independent|Cruce|CP True/coste1/orden0|574|296|válido|
|practice_0135|independent|Cruce|CP True/coste0/orden1|369|296|válido|
|practice_0135|independent|Cruce|CP True/coste1/orden1|360|296|válido|
|practice_0135|independent|Cruce|mejor independiente/coste0/orden0|676|296|válido|
|practice_0135|independent|Cruce|mejor independiente/coste1/orden0|574|296|válido|
|practice_0135|independent|Cruce|mejor independiente/coste0/orden1|369|296|válido|
|practice_0135|independent|Cruce|mejor independiente/coste1/orden1|360|296|válido|
|practice_0135|independent|E1|cola10/pol1/inicio0|328|95|válido|
|practice_0135|independent|E1|cola10/pol1/inicio1|328|95|válido|
|practice_0135|independent|E1|cola10/pol2/inicio0|323|90|válido|
|practice_0135|independent|E1|cola10/pol2/inicio1|323|90|válido|
|practice_0135|independent|E1|cola8/pol1/inicio0|326|77|válido|
|practice_0135|independent|E1|cola8/pol1/inicio1|329|80|válido|
|practice_0135|independent|E1|cola8/pol2/inicio0|328|79|válido|
|practice_0135|independent|E1|cola8/pol2/inicio1|328|79|válido|
|practice_0135|independent|E1|cola6/pol1/inicio0|323|56|válido|
|practice_0135|independent|E1|cola6/pol1/inicio1|326|59|válido|
|practice_0135|independent|E1|cola6/pol2/inicio0|323|56|válido|
|practice_0135|independent|E1|cola6/pol2/inicio1|323|56|válido|
|practice_0135|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0135|independent|E2|CP False/coste0/orden0|492|349|válido|
|practice_0135|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0135|independent|E2|CP True/coste0/orden0|676|296|válido|
|practice_0135|independent|E3|donante0/coste0/orden0|320|296|válido|
|practice_0135|independent|E3|donante0/coste1/orden0|316|296|válido|
|practice_0135|independent|E3|donante0/coste0/orden1|369|296|válido|
|practice_0135|independent|E3|donante0/coste1/orden1|360|296|válido|
|practice_0135|independent|E3|donante1/coste0/orden0|344|344|válido|
|practice_0135|independent|E3|donante1/coste1/orden0|344|344|válido|
|practice_0135|independent|E3|donante1/coste0/orden1|344|344|válido|
|practice_0135|independent|E3|donante1/coste1/orden1|344|344|válido|
|practice_0135|independent|E3|donante2/coste0/orden0|479|330|válido|
|practice_0135|independent|E3|donante2/coste1/orden0|367|330|válido|
|practice_0135|independent|E3|donante2/coste0/orden1|330|330|válido|
|practice_0135|independent|E3|donante2/coste1/orden1|330|330|válido|
|practice_0135|independent|E3|donante3/coste0/orden0|472|325|válido|
|practice_0135|independent|E3|donante3/coste1/orden0|429|325|válido|
|practice_0135|independent|E3|donante3/coste0/orden1|325|325|válido|
|practice_0135|independent|E3|donante3/coste1/orden1|325|325|válido|
|practice_0135|independent|E3|donante4/coste0/orden0|352|352|válido|
|practice_0135|independent|E3|donante4/coste1/orden0|352|352|válido|
|practice_0135|independent|E3|donante4/coste0/orden1|352|352|válido|
|practice_0135|independent|E3|donante4/coste1/orden1|352|352|válido|
|practice_0135|independent|E3|donante5/coste0/orden0|347|347|válido|
|practice_0135|independent|E3|donante5/coste1/orden0|347|347|válido|
|practice_0135|independent|E3|donante5/coste0/orden1|347|347|válido|
|practice_0135|independent|E3|donante5/coste1/orden1|347|347|válido|
|practice_0135|independent|E3|donante6/coste0/orden0|349|349|válido|
|practice_0135|independent|E3|donante6/coste1/orden0|349|349|válido|
|practice_0135|independent|E3|donante6/coste0/orden1|349|349|válido|
|practice_0135|independent|E3|donante6/coste1/orden1|349|349|válido|
|practice_0135|independent|E3|donante7/coste0/orden0|389|314|válido|
|practice_0135|independent|E3|donante7/coste1/orden0|383|314|válido|
|practice_0135|independent|E3|donante7/coste0/orden1|390|314|válido|
|practice_0135|independent|E3|donante7/coste1/orden1|378|314|válido|
|practice_0135|independent|E3|donante8/coste0/orden0|337|305|válido|
|practice_0135|independent|E3|donante8/coste1/orden0|308|305|válido|
|practice_0135|independent|E3|donante8/coste0/orden1|337|305|válido|
|practice_0135|independent|E3|donante8/coste1/orden1|308|305|válido|
|practice_0135|independent|E3|donante9/coste0/orden0|320|296|válido|
|practice_0135|independent|E3|donante9/coste1/orden0|316|296|válido|
|practice_0135|independent|E3|donante9/coste0/orden1|369|296|válido|
|practice_0135|independent|E3|donante9/coste1/orden1|360|296|válido|
|practice_0136|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0136|mixed|Cruce|CP True/coste0/orden0|50|50|válido|
|practice_0136|mixed|Cruce|CP True/coste1/orden0|50|50|válido|
|practice_0136|mixed|Cruce|CP True/coste0/orden1|50|50|válido|
|practice_0136|mixed|Cruce|CP True/coste1/orden1|50|50|válido|
|practice_0136|mixed|Cruce|mejor independiente/coste0/orden0|50|50|válido|
|practice_0136|mixed|Cruce|mejor independiente/coste1/orden0|50|50|válido|
|practice_0136|mixed|Cruce|mejor independiente/coste0/orden1|50|50|válido|
|practice_0136|mixed|Cruce|mejor independiente/coste1/orden1|50|50|válido|
|practice_0136|mixed|E1|cola10/pol1/inicio0|51|37|válido|
|practice_0136|mixed|E1|cola10/pol1/inicio1|51|37|válido|
|practice_0136|mixed|E1|cola10/pol2/inicio0|52|38|válido|
|practice_0136|mixed|E1|cola10/pol2/inicio1|52|38|válido|
|practice_0136|mixed|E1|cola8/pol1/inicio0|50|29|válido|
|practice_0136|mixed|E1|cola8/pol1/inicio1|50|29|válido|
|practice_0136|mixed|E1|cola8/pol2/inicio0|51|30|válido|
|practice_0136|mixed|E1|cola8/pol2/inicio1|51|30|válido|
|practice_0136|mixed|E1|cola6/pol1/inicio0|51|19|válido|
|practice_0136|mixed|E1|cola6/pol1/inicio1|51|19|válido|
|practice_0136|mixed|E1|cola6/pol2/inicio0|51|19|válido|
|practice_0136|mixed|E1|cola6/pol2/inicio1|51|19|válido|
|practice_0136|mixed|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0136|mixed|E2|CP False/coste0/orden0|50|50|válido|
|practice_0136|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0136|mixed|E2|CP True/coste0/orden0|50|50|válido|
|practice_0136|mixed|E3|donante0/coste0/orden0|50|50|válido|
|practice_0136|mixed|E3|donante0/coste1/orden0|50|50|válido|
|practice_0136|mixed|E3|donante0/coste0/orden1|50|50|válido|
|practice_0136|mixed|E3|donante0/coste1/orden1|50|50|válido|
|practice_0136|mixed|E3|donante1/coste0/orden0|56|56|válido|
|practice_0136|mixed|E3|donante1/coste1/orden0|56|56|válido|
|practice_0136|mixed|E3|donante1/coste0/orden1|56|56|válido|
|practice_0136|mixed|E3|donante1/coste1/orden1|56|56|válido|
|practice_0136|mixed|E3|donante2/coste0/orden0|55|55|válido|
|practice_0136|mixed|E3|donante2/coste1/orden0|55|55|válido|
|practice_0136|mixed|E3|donante2/coste0/orden1|55|55|válido|
|practice_0136|mixed|E3|donante2/coste1/orden1|55|55|válido|
|practice_0136|mixed|E3|donante3/coste0/orden0|52|52|válido|
|practice_0136|mixed|E3|donante3/coste1/orden0|52|52|válido|
|practice_0136|mixed|E3|donante3/coste0/orden1|52|52|válido|
|practice_0136|mixed|E3|donante3/coste1/orden1|52|52|válido|
|practice_0136|mixed|E3|donante4/coste0/orden0|57|57|válido|
|practice_0136|mixed|E3|donante4/coste1/orden0|57|57|válido|
|practice_0136|mixed|E3|donante4/coste0/orden1|57|57|válido|
|practice_0136|mixed|E3|donante4/coste1/orden1|57|57|válido|
|practice_0136|mixed|E3|donante5/coste0/orden0|55|55|válido|
|practice_0136|mixed|E3|donante5/coste1/orden0|55|55|válido|
|practice_0136|mixed|E3|donante5/coste0/orden1|55|55|válido|
|practice_0136|mixed|E3|donante5/coste1/orden1|55|55|válido|
|practice_0136|mixed|E3|donante6/coste0/orden0|51|51|válido|
|practice_0136|mixed|E3|donante6/coste1/orden0|51|51|válido|
|practice_0136|mixed|E3|donante6/coste0/orden1|51|51|válido|
|practice_0136|mixed|E3|donante6/coste1/orden1|51|51|válido|
|practice_0136|mixed|E3|donante7/coste0/orden0|58|58|válido|
|practice_0136|mixed|E3|donante7/coste1/orden0|58|58|válido|
|practice_0136|mixed|E3|donante7/coste0/orden1|58|58|válido|
|practice_0136|mixed|E3|donante7/coste1/orden1|58|58|válido|
|practice_0136|mixed|E3|donante8/coste0/orden0|55|55|válido|
|practice_0136|mixed|E3|donante8/coste1/orden0|55|55|válido|
|practice_0136|mixed|E3|donante8/coste0/orden1|55|55|válido|
|practice_0136|mixed|E3|donante8/coste1/orden1|55|55|válido|
|practice_0136|mixed|E3|donante9/coste0/orden0|52|52|válido|
|practice_0136|mixed|E3|donante9/coste1/orden0|52|52|válido|
|practice_0136|mixed|E3|donante9/coste0/orden1|52|52|válido|
|practice_0136|mixed|E3|donante9/coste1/orden1|52|52|válido|
|practice_0137|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0137|multiplicative|Cruce|CP True/coste0/orden0|97|97|válido|
|practice_0137|multiplicative|Cruce|CP True/coste1/orden0|97|97|válido|
|practice_0137|multiplicative|Cruce|CP True/coste0/orden1|97|97|válido|
|practice_0137|multiplicative|Cruce|CP True/coste1/orden1|97|97|válido|
|practice_0137|multiplicative|Cruce|mejor independiente/coste0/orden0|97|97|válido|
|practice_0137|multiplicative|Cruce|mejor independiente/coste1/orden0|97|97|válido|
|practice_0137|multiplicative|Cruce|mejor independiente/coste0/orden1|97|97|válido|
|practice_0137|multiplicative|Cruce|mejor independiente/coste1/orden1|97|97|válido|
|practice_0137|multiplicative|E1|cola10/pol1/inicio0|100|35|válido|
|practice_0137|multiplicative|E1|cola10/pol1/inicio1|100|35|válido|
|practice_0137|multiplicative|E1|cola10/pol2/inicio0|99|34|válido|
|practice_0137|multiplicative|E1|cola10/pol2/inicio1|99|34|válido|
|practice_0137|multiplicative|E1|cola8/pol1/inicio0|103|30|válido|
|practice_0137|multiplicative|E1|cola8/pol1/inicio1|103|30|válido|
|practice_0137|multiplicative|E1|cola8/pol2/inicio0|99|26|válido|
|practice_0137|multiplicative|E1|cola8/pol2/inicio1|99|26|válido|
|practice_0137|multiplicative|E1|cola6/pol1/inicio0|103|24|válido|
|practice_0137|multiplicative|E1|cola6/pol1/inicio1|103|24|válido|
|practice_0137|multiplicative|E1|cola6/pol2/inicio0|99|20|válido|
|practice_0137|multiplicative|E1|cola6/pol2/inicio1|99|20|válido|
|practice_0137|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0137|multiplicative|E2|CP False/coste0/orden0|99|99|válido|
|practice_0137|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0137|multiplicative|E2|CP True/coste0/orden0|97|97|válido|
|practice_0137|multiplicative|E3|donante0/coste0/orden0|99|99|válido|
|practice_0137|multiplicative|E3|donante0/coste1/orden0|99|99|válido|
|practice_0137|multiplicative|E3|donante0/coste0/orden1|99|99|válido|
|practice_0137|multiplicative|E3|donante0/coste1/orden1|99|99|válido|
|practice_0137|multiplicative|E3|donante1/coste0/orden0|106|106|válido|
|practice_0137|multiplicative|E3|donante1/coste1/orden0|106|106|válido|
|practice_0137|multiplicative|E3|donante1/coste0/orden1|106|106|válido|
|practice_0137|multiplicative|E3|donante1/coste1/orden1|106|106|válido|
|practice_0137|multiplicative|E3|donante2/coste0/orden0|107|107|válido|
|practice_0137|multiplicative|E3|donante2/coste1/orden0|107|107|válido|
|practice_0137|multiplicative|E3|donante2/coste0/orden1|107|107|válido|
|practice_0137|multiplicative|E3|donante2/coste1/orden1|107|107|válido|
|practice_0137|multiplicative|E3|donante3/coste0/orden0|103|103|válido|
|practice_0137|multiplicative|E3|donante3/coste1/orden0|103|103|válido|
|practice_0137|multiplicative|E3|donante3/coste0/orden1|103|103|válido|
|practice_0137|multiplicative|E3|donante3/coste1/orden1|103|103|válido|
|practice_0137|multiplicative|E3|donante4/coste0/orden0|111|111|válido|
|practice_0137|multiplicative|E3|donante4/coste1/orden0|111|111|válido|
|practice_0137|multiplicative|E3|donante4/coste0/orden1|111|111|válido|
|practice_0137|multiplicative|E3|donante4/coste1/orden1|111|111|válido|
|practice_0137|multiplicative|E3|donante5/coste0/orden0|106|106|válido|
|practice_0137|multiplicative|E3|donante5/coste1/orden0|106|106|válido|
|practice_0137|multiplicative|E3|donante5/coste0/orden1|106|106|válido|
|practice_0137|multiplicative|E3|donante5/coste1/orden1|106|106|válido|
|practice_0137|multiplicative|E3|donante6/coste0/orden0|102|102|válido|
|practice_0137|multiplicative|E3|donante6/coste1/orden0|102|102|válido|
|practice_0137|multiplicative|E3|donante6/coste0/orden1|102|102|válido|
|practice_0137|multiplicative|E3|donante6/coste1/orden1|102|102|válido|
|practice_0137|multiplicative|E3|donante7/coste0/orden0|104|104|válido|
|practice_0137|multiplicative|E3|donante7/coste1/orden0|104|104|válido|
|practice_0137|multiplicative|E3|donante7/coste0/orden1|104|104|válido|
|practice_0137|multiplicative|E3|donante7/coste1/orden1|104|104|válido|
|practice_0137|multiplicative|E3|donante8/coste0/orden0|106|106|válido|
|practice_0137|multiplicative|E3|donante8/coste1/orden0|106|106|válido|
|practice_0137|multiplicative|E3|donante8/coste0/orden1|106|106|válido|
|practice_0137|multiplicative|E3|donante8/coste1/orden1|106|106|válido|
|practice_0137|multiplicative|E3|donante9/coste0/orden0|100|100|válido|
|practice_0137|multiplicative|E3|donante9/coste1/orden0|100|100|válido|
|practice_0137|multiplicative|E3|donante9/coste0/orden1|100|100|válido|
|practice_0137|multiplicative|E3|donante9/coste1/orden1|100|100|válido|
|practice_0138|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0138|independent|Cruce|CP True/coste0/orden0|432|305|válido|
|practice_0138|independent|Cruce|CP True/coste1/orden0|428|305|válido|
|practice_0138|independent|Cruce|CP True/coste0/orden1|349|305|válido|
|practice_0138|independent|Cruce|CP True/coste1/orden1|336|305|válido|
|practice_0138|independent|Cruce|mejor independiente/coste0/orden0|432|305|válido|
|practice_0138|independent|Cruce|mejor independiente/coste1/orden0|428|305|válido|
|practice_0138|independent|Cruce|mejor independiente/coste0/orden1|349|305|válido|
|practice_0138|independent|Cruce|mejor independiente/coste1/orden1|336|305|válido|
|practice_0138|independent|E1|cola10/pol1/inicio0|318|93|válido|
|practice_0138|independent|E1|cola10/pol1/inicio1|315|90|válido|
|practice_0138|independent|E1|cola10/pol2/inicio0|318|93|válido|
|practice_0138|independent|E1|cola10/pol2/inicio1|316|91|válido|
|practice_0138|independent|E1|cola8/pol1/inicio0|335|78|válido|
|practice_0138|independent|E1|cola8/pol1/inicio1|335|78|válido|
|practice_0138|independent|E1|cola8/pol2/inicio0|331|74|válido|
|practice_0138|independent|E1|cola8/pol2/inicio1|331|74|válido|
|practice_0138|independent|E1|cola6/pol1/inicio0|326|51|válido|
|practice_0138|independent|E1|cola6/pol1/inicio1|326|51|válido|
|practice_0138|independent|E1|cola6/pol2/inicio0|330|55|válido|
|practice_0138|independent|E1|cola6/pol2/inicio1|330|55|válido|
|practice_0138|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0138|independent|E2|CP False/coste0/orden0|551|393|válido|
|practice_0138|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0138|independent|E2|CP True/coste0/orden0|432|305|válido|
|practice_0138|independent|E3|donante0/coste0/orden0|335|305|válido|
|practice_0138|independent|E3|donante0/coste1/orden0|307|305|válido|
|practice_0138|independent|E3|donante0/coste0/orden1|349|305|válido|
|practice_0138|independent|E3|donante0/coste1/orden1|336|305|válido|
|practice_0138|independent|E3|donante1/coste0/orden0|346|346|válido|
|practice_0138|independent|E3|donante1/coste1/orden0|346|346|válido|
|practice_0138|independent|E3|donante1/coste0/orden1|346|346|válido|
|practice_0138|independent|E3|donante1/coste1/orden1|346|346|válido|
|practice_0138|independent|E3|donante2/coste0/orden0|559|325|válido|
|practice_0138|independent|E3|donante2/coste1/orden0|353|325|válido|
|practice_0138|independent|E3|donante2/coste0/orden1|330|325|válido|
|practice_0138|independent|E3|donante2/coste1/orden1|330|325|válido|
|practice_0138|independent|E3|donante3/coste0/orden0|409|321|válido|
|practice_0138|independent|E3|donante3/coste1/orden0|388|321|válido|
|practice_0138|independent|E3|donante3/coste0/orden1|356|321|válido|
|practice_0138|independent|E3|donante3/coste1/orden1|325|321|válido|
|practice_0138|independent|E3|donante4/coste0/orden0|363|363|válido|
|practice_0138|independent|E3|donante4/coste1/orden0|363|363|válido|
|practice_0138|independent|E3|donante4/coste0/orden1|363|363|válido|
|practice_0138|independent|E3|donante4/coste1/orden1|363|363|válido|
|practice_0138|independent|E3|donante5/coste0/orden0|366|366|válido|
|practice_0138|independent|E3|donante5/coste1/orden0|366|366|válido|
|practice_0138|independent|E3|donante5/coste0/orden1|366|366|válido|
|practice_0138|independent|E3|donante5/coste1/orden1|366|366|válido|
|practice_0138|independent|E3|donante6/coste0/orden0|359|359|válido|
|practice_0138|independent|E3|donante6/coste1/orden0|359|359|válido|
|practice_0138|independent|E3|donante6/coste0/orden1|359|359|válido|
|practice_0138|independent|E3|donante6/coste1/orden1|359|359|válido|
|practice_0138|independent|E3|donante7/coste0/orden0|346|314|válido|
|practice_0138|independent|E3|donante7/coste1/orden0|365|314|válido|
|practice_0138|independent|E3|donante7/coste0/orden1|338|314|válido|
|practice_0138|independent|E3|donante7/coste1/orden1|338|314|válido|
|practice_0138|independent|E3|donante8/coste0/orden0|335|305|válido|
|practice_0138|independent|E3|donante8/coste1/orden0|307|305|válido|
|practice_0138|independent|E3|donante8/coste0/orden1|349|305|válido|
|practice_0138|independent|E3|donante8/coste1/orden1|336|305|válido|
|practice_0138|independent|E3|donante9/coste0/orden0|363|294|válido|
|practice_0138|independent|E3|donante9/coste1/orden0|382|294|válido|
|practice_0138|independent|E3|donante9/coste0/orden1|429|294|válido|
|practice_0138|independent|E3|donante9/coste1/orden1|340|294|válido|
|practice_0139|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0139|mixed|Cruce|CP True/coste0/orden0|206|190|válido|
|practice_0139|mixed|Cruce|CP True/coste1/orden0|191|190|válido|
|practice_0139|mixed|Cruce|CP True/coste0/orden1|190|190|válido|
|practice_0139|mixed|Cruce|CP True/coste1/orden1|190|190|válido|
|practice_0139|mixed|Cruce|mejor independiente/coste0/orden0|208|190|válido|
|practice_0139|mixed|Cruce|mejor independiente/coste1/orden0|191|190|válido|
|practice_0139|mixed|Cruce|mejor independiente/coste0/orden1|190|190|válido|
|practice_0139|mixed|Cruce|mejor independiente/coste1/orden1|190|190|válido|
|practice_0139|mixed|E1|cola10/pol1/inicio0|197|71|válido|
|practice_0139|mixed|E1|cola10/pol1/inicio1|197|71|válido|
|practice_0139|mixed|E1|cola10/pol2/inicio0|194|68|válido|
|practice_0139|mixed|E1|cola10/pol2/inicio1|194|68|válido|
|practice_0139|mixed|E1|cola8/pol1/inicio0|200|60|válido|
|practice_0139|mixed|E1|cola8/pol1/inicio1|200|60|válido|
|practice_0139|mixed|E1|cola8/pol2/inicio0|197|57|válido|
|practice_0139|mixed|E1|cola8/pol2/inicio1|197|57|válido|
|practice_0139|mixed|E1|cola6/pol1/inicio0|198|44|válido|
|practice_0139|mixed|E1|cola6/pol1/inicio1|198|44|válido|
|practice_0139|mixed|E1|cola6/pol2/inicio0|197|43|válido|
|practice_0139|mixed|E1|cola6/pol2/inicio1|197|43|válido|
|practice_0139|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0139|mixed|E2|CP False/coste0/orden0|208|190|válido|
|practice_0139|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0139|mixed|E2|CP True/coste0/orden0|206|190|válido|
|practice_0139|mixed|E3|donante0/coste0/orden0|191|190|válido|
|practice_0139|mixed|E3|donante0/coste1/orden0|191|190|válido|
|practice_0139|mixed|E3|donante0/coste0/orden1|190|190|válido|
|practice_0139|mixed|E3|donante0/coste1/orden1|190|190|válido|
|practice_0139|mixed|E3|donante1/coste0/orden0|218|218|válido|
|practice_0139|mixed|E3|donante1/coste1/orden0|218|218|válido|
|practice_0139|mixed|E3|donante1/coste0/orden1|218|218|válido|
|practice_0139|mixed|E3|donante1/coste1/orden1|218|218|válido|
|practice_0139|mixed|E3|donante2/coste0/orden0|208|202|válido|
|practice_0139|mixed|E3|donante2/coste1/orden0|208|202|válido|
|practice_0139|mixed|E3|donante2/coste0/orden1|202|202|válido|
|practice_0139|mixed|E3|donante2/coste1/orden1|202|202|válido|
|practice_0139|mixed|E3|donante3/coste0/orden0|234|205|válido|
|practice_0139|mixed|E3|donante3/coste1/orden0|230|205|válido|
|practice_0139|mixed|E3|donante3/coste0/orden1|205|205|válido|
|practice_0139|mixed|E3|donante3/coste1/orden1|205|205|válido|
|practice_0139|mixed|E3|donante4/coste0/orden0|221|221|válido|
|practice_0139|mixed|E3|donante4/coste1/orden0|221|221|válido|
|practice_0139|mixed|E3|donante4/coste0/orden1|221|221|válido|
|practice_0139|mixed|E3|donante4/coste1/orden1|221|221|válido|
|practice_0139|mixed|E3|donante5/coste0/orden0|218|218|válido|
|practice_0139|mixed|E3|donante5/coste1/orden0|218|218|válido|
|practice_0139|mixed|E3|donante5/coste0/orden1|218|218|válido|
|practice_0139|mixed|E3|donante5/coste1/orden1|218|218|válido|
|practice_0139|mixed|E3|donante6/coste0/orden0|218|218|válido|
|practice_0139|mixed|E3|donante6/coste1/orden0|218|218|válido|
|practice_0139|mixed|E3|donante6/coste0/orden1|218|218|válido|
|practice_0139|mixed|E3|donante6/coste1/orden1|218|218|válido|
|practice_0139|mixed|E3|donante7/coste0/orden0|200|195|válido|
|practice_0139|mixed|E3|donante7/coste1/orden0|196|195|válido|
|practice_0139|mixed|E3|donante7/coste0/orden1|200|195|válido|
|practice_0139|mixed|E3|donante7/coste1/orden1|196|195|válido|
|practice_0139|mixed|E3|donante8/coste0/orden0|192|191|válido|
|practice_0139|mixed|E3|donante8/coste1/orden0|192|191|válido|
|practice_0139|mixed|E3|donante8/coste0/orden1|191|191|válido|
|practice_0139|mixed|E3|donante8/coste1/orden1|191|191|válido|
|practice_0139|mixed|E3|donante9/coste0/orden0|193|191|válido|
|practice_0139|mixed|E3|donante9/coste1/orden0|193|191|válido|
|practice_0139|mixed|E3|donante9/coste0/orden1|193|191|válido|
|practice_0139|mixed|E3|donante9/coste1/orden1|193|191|válido|
|practice_0140|tiny|Cruce|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0140|tiny|Cruce|CP True/coste0/orden0|1|1|válido|
|practice_0140|tiny|Cruce|CP True/coste1/orden0|1|1|válido|
|practice_0140|tiny|Cruce|CP True/coste0/orden1|1|1|válido|
|practice_0140|tiny|Cruce|CP True/coste1/orden1|1|1|válido|
|practice_0140|tiny|Cruce|mejor independiente/coste0/orden0|1|1|válido|
|practice_0140|tiny|Cruce|mejor independiente/coste1/orden0|1|1|válido|
|practice_0140|tiny|Cruce|mejor independiente/coste0/orden1|1|1|válido|
|practice_0140|tiny|Cruce|mejor independiente/coste1/orden1|1|1|válido|
|practice_0140|tiny|E1|cola1/pol1/inicio0|1|1|válido|
|practice_0140|tiny|E1|cola1/pol2/inicio0|1|1|válido|
|practice_0140|tiny|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0140|tiny|E2|CP False/coste0/orden0|1|1|válido|
|practice_0140|tiny|E2|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0140|tiny|E2|CP True/coste0/orden0|1|1|válido|
|practice_0140|tiny|E3|donante0/coste0/orden0|1|1|válido|
|practice_0140|tiny|E3|donante0/coste1/orden0|1|1|válido|
|practice_0140|tiny|E3|donante0/coste0/orden1|1|1|válido|
|practice_0140|tiny|E3|donante0/coste1/orden1|1|1|válido|
|practice_0140|tiny|E3|donante1/coste0/orden0|1|1|válido|
|practice_0140|tiny|E3|donante1/coste1/orden0|1|1|válido|
|practice_0140|tiny|E3|donante1/coste0/orden1|1|1|válido|
|practice_0140|tiny|E3|donante1/coste1/orden1|1|1|válido|
|practice_0141|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0141|shared|Cruce|CP True/coste0/orden0|145|145|válido|
|practice_0141|shared|Cruce|CP True/coste1/orden0|145|145|válido|
|practice_0141|shared|Cruce|CP True/coste0/orden1|145|145|válido|
|practice_0141|shared|Cruce|CP True/coste1/orden1|145|145|válido|
|practice_0141|shared|Cruce|mejor independiente/coste0/orden0|146|146|válido|
|practice_0141|shared|Cruce|mejor independiente/coste1/orden0|146|146|válido|
|practice_0141|shared|Cruce|mejor independiente/coste0/orden1|146|146|válido|
|practice_0141|shared|Cruce|mejor independiente/coste1/orden1|146|146|válido|
|practice_0141|shared|E1|cola10/pol1/inicio0|153|57|válido|
|practice_0141|shared|E1|cola10/pol1/inicio1|153|57|válido|
|practice_0141|shared|E1|cola10/pol2/inicio0|153|57|válido|
|practice_0141|shared|E1|cola10/pol2/inicio1|153|57|válido|
|practice_0141|shared|E1|cola8/pol1/inicio0|149|43|válido|
|practice_0141|shared|E1|cola8/pol1/inicio1|150|44|válido|
|practice_0141|shared|E1|cola8/pol2/inicio0|149|43|válido|
|practice_0141|shared|E1|cola8/pol2/inicio1|150|44|válido|
|practice_0141|shared|E1|cola6/pol1/inicio0|148|30|válido|
|practice_0141|shared|E1|cola6/pol1/inicio1|148|30|válido|
|practice_0141|shared|E1|cola6/pol2/inicio0|148|30|válido|
|practice_0141|shared|E1|cola6/pol2/inicio1|148|30|válido|
|practice_0141|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0141|shared|E2|CP False/coste0/orden0|156|146|válido|
|practice_0141|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0141|shared|E2|CP True/coste0/orden0|146|146|válido|
|practice_0141|shared|E3|donante0/coste0/orden0|148|148|válido|
|practice_0141|shared|E3|donante0/coste1/orden0|148|148|válido|
|practice_0141|shared|E3|donante0/coste0/orden1|148|148|válido|
|practice_0141|shared|E3|donante0/coste1/orden1|148|148|válido|
|practice_0141|shared|E3|donante1/coste0/orden0|157|157|válido|
|practice_0141|shared|E3|donante1/coste1/orden0|157|157|válido|
|practice_0141|shared|E3|donante1/coste0/orden1|157|157|válido|
|practice_0141|shared|E3|donante1/coste1/orden1|157|157|válido|
|practice_0141|shared|E3|donante2/coste0/orden0|155|155|válido|
|practice_0141|shared|E3|donante2/coste1/orden0|155|155|válido|
|practice_0141|shared|E3|donante2/coste0/orden1|155|155|válido|
|practice_0141|shared|E3|donante2/coste1/orden1|155|155|válido|
|practice_0141|shared|E3|donante3/coste0/orden0|152|152|válido|
|practice_0141|shared|E3|donante3/coste1/orden0|152|152|válido|
|practice_0141|shared|E3|donante3/coste0/orden1|152|152|válido|
|practice_0141|shared|E3|donante3/coste1/orden1|152|152|válido|
|practice_0141|shared|E3|donante4/coste0/orden0|166|166|válido|
|practice_0141|shared|E3|donante4/coste1/orden0|166|166|válido|
|practice_0141|shared|E3|donante4/coste0/orden1|166|166|válido|
|practice_0141|shared|E3|donante4/coste1/orden1|166|166|válido|
|practice_0141|shared|E3|donante5/coste0/orden0|162|162|válido|
|practice_0141|shared|E3|donante5/coste1/orden0|162|162|válido|
|practice_0141|shared|E3|donante5/coste0/orden1|162|162|válido|
|practice_0141|shared|E3|donante5/coste1/orden1|162|162|válido|
|practice_0141|shared|E3|donante6/coste0/orden0|162|162|válido|
|practice_0141|shared|E3|donante6/coste1/orden0|162|162|válido|
|practice_0141|shared|E3|donante6/coste0/orden1|162|162|válido|
|practice_0141|shared|E3|donante6/coste1/orden1|162|162|válido|
|practice_0141|shared|E3|donante7/coste0/orden0|205|153|válido|
|practice_0141|shared|E3|donante7/coste1/orden0|177|153|válido|
|practice_0141|shared|E3|donante7/coste0/orden1|244|153|válido|
|practice_0141|shared|E3|donante7/coste1/orden1|174|153|válido|
|practice_0141|shared|E3|donante8/coste0/orden0|159|145|válido|
|practice_0141|shared|E3|donante8/coste1/orden0|155|145|válido|
|practice_0141|shared|E3|donante8/coste0/orden1|159|145|válido|
|practice_0141|shared|E3|donante8/coste1/orden1|161|145|válido|
|practice_0142|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0142|independent|Cruce|CP True/coste0/orden0|307|234|válido|
|practice_0142|independent|Cruce|CP True/coste1/orden0|303|234|válido|
|practice_0142|independent|Cruce|CP True/coste0/orden1|256|234|válido|
|practice_0142|independent|Cruce|CP True/coste1/orden1|251|234|válido|
|practice_0142|independent|Cruce|mejor independiente/coste0/orden0|321|236|válido|
|practice_0142|independent|Cruce|mejor independiente/coste1/orden0|313|236|válido|
|practice_0142|independent|Cruce|mejor independiente/coste0/orden1|250|236|válido|
|practice_0142|independent|Cruce|mejor independiente/coste1/orden1|244|236|válido|
|practice_0142|independent|E1|cola10/pol1/inicio0|248|75|válido|
|practice_0142|independent|E1|cola10/pol1/inicio1|248|75|válido|
|practice_0142|independent|E1|cola10/pol2/inicio0|252|79|válido|
|practice_0142|independent|E1|cola10/pol2/inicio1|252|79|válido|
|practice_0142|independent|E1|cola8/pol1/inicio0|253|57|válido|
|practice_0142|independent|E1|cola8/pol1/inicio1|253|57|válido|
|practice_0142|independent|E1|cola8/pol2/inicio0|256|60|válido|
|practice_0142|independent|E1|cola8/pol2/inicio1|256|60|válido|
|practice_0142|independent|E1|cola6/pol1/inicio0|254|44|válido|
|practice_0142|independent|E1|cola6/pol1/inicio1|254|44|válido|
|practice_0142|independent|E1|cola6/pol2/inicio0|258|48|válido|
|practice_0142|independent|E1|cola6/pol2/inicio1|258|48|válido|
|practice_0142|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0142|independent|E2|CP False/coste0/orden0|297|247|válido|
|practice_0142|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0142|independent|E2|CP True/coste0/orden0|279|238|válido|
|practice_0142|independent|E3|donante0/coste0/orden0|251|238|válido|
|practice_0142|independent|E3|donante0/coste1/orden0|247|238|válido|
|practice_0142|independent|E3|donante0/coste0/orden1|274|238|válido|
|practice_0142|independent|E3|donante0/coste1/orden1|263|238|válido|
|practice_0142|independent|E3|donante1/coste0/orden0|264|264|válido|
|practice_0142|independent|E3|donante1/coste1/orden0|264|264|válido|
|practice_0142|independent|E3|donante1/coste0/orden1|264|264|válido|
|practice_0142|independent|E3|donante1/coste1/orden1|264|264|válido|
|practice_0142|independent|E3|donante2/coste0/orden0|280|255|válido|
|practice_0142|independent|E3|donante2/coste1/orden0|267|255|válido|
|practice_0142|independent|E3|donante2/coste0/orden1|255|255|válido|
|practice_0142|independent|E3|donante2/coste1/orden1|255|255|válido|
|practice_0142|independent|E3|donante3/coste0/orden0|312|249|válido|
|practice_0142|independent|E3|donante3/coste1/orden0|295|249|válido|
|practice_0142|independent|E3|donante3/coste0/orden1|249|249|válido|
|practice_0142|independent|E3|donante3/coste1/orden1|249|249|válido|
|practice_0142|independent|E3|donante4/coste0/orden0|271|271|válido|
|practice_0142|independent|E3|donante4/coste1/orden0|271|271|válido|
|practice_0142|independent|E3|donante4/coste0/orden1|271|271|válido|
|practice_0142|independent|E3|donante4/coste1/orden1|271|271|válido|
|practice_0142|independent|E3|donante5/coste0/orden0|270|270|válido|
|practice_0142|independent|E3|donante5/coste1/orden0|270|270|válido|
|practice_0142|independent|E3|donante5/coste0/orden1|270|270|válido|
|practice_0142|independent|E3|donante5/coste1/orden1|270|270|válido|
|practice_0142|independent|E3|donante6/coste0/orden0|272|272|válido|
|practice_0142|independent|E3|donante6/coste1/orden0|272|272|válido|
|practice_0142|independent|E3|donante6/coste0/orden1|272|272|válido|
|practice_0142|independent|E3|donante6/coste1/orden1|272|272|válido|
|practice_0142|independent|E3|donante7/coste0/orden0|270|248|válido|
|practice_0142|independent|E3|donante7/coste1/orden0|269|248|válido|
|practice_0142|independent|E3|donante7/coste0/orden1|295|248|válido|
|practice_0142|independent|E3|donante7/coste1/orden1|315|248|válido|
|practice_0142|independent|E3|donante8/coste0/orden0|253|239|válido|
|practice_0142|independent|E3|donante8/coste1/orden0|248|239|válido|
|practice_0142|independent|E3|donante8/coste0/orden1|276|239|válido|
|practice_0142|independent|E3|donante8/coste1/orden1|264|239|válido|
|practice_0142|independent|E3|donante9/coste0/orden0|280|236|válido|
|practice_0142|independent|E3|donante9/coste1/orden0|263|236|válido|
|practice_0142|independent|E3|donante9/coste0/orden1|250|236|válido|
|practice_0142|independent|E3|donante9/coste1/orden1|244|236|válido|
|practice_0143|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0143|shared|Cruce|CP True/coste0/orden0|140|140|válido|
|practice_0143|shared|Cruce|CP True/coste1/orden0|140|140|válido|
|practice_0143|shared|Cruce|CP True/coste0/orden1|140|140|válido|
|practice_0143|shared|Cruce|CP True/coste1/orden1|140|140|válido|
|practice_0143|shared|Cruce|mejor independiente/coste0/orden0|142|142|válido|
|practice_0143|shared|Cruce|mejor independiente/coste1/orden0|142|142|válido|
|practice_0143|shared|Cruce|mejor independiente/coste0/orden1|142|142|válido|
|practice_0143|shared|Cruce|mejor independiente/coste1/orden1|142|142|válido|
|practice_0143|shared|E1|cola10/pol1/inicio0|144|45|válido|
|practice_0143|shared|E1|cola10/pol1/inicio1|144|45|válido|
|practice_0143|shared|E1|cola10/pol2/inicio0|144|45|válido|
|practice_0143|shared|E1|cola10/pol2/inicio1|144|45|válido|
|practice_0143|shared|E1|cola8/pol1/inicio0|146|39|válido|
|practice_0143|shared|E1|cola8/pol1/inicio1|146|39|válido|
|practice_0143|shared|E1|cola8/pol2/inicio0|146|39|válido|
|practice_0143|shared|E1|cola8/pol2/inicio1|146|39|válido|
|practice_0143|shared|E1|cola6/pol1/inicio0|148|32|válido|
|practice_0143|shared|E1|cola6/pol1/inicio1|148|32|válido|
|practice_0143|shared|E1|cola6/pol2/inicio0|148|32|válido|
|practice_0143|shared|E1|cola6/pol2/inicio1|148|32|válido|
|practice_0143|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0143|shared|E2|CP False/coste0/orden0|147|147|válido|
|practice_0143|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0143|shared|E2|CP True/coste0/orden0|145|145|válido|
|practice_0143|shared|E3|donante0/coste0/orden0|145|145|válido|
|practice_0143|shared|E3|donante0/coste1/orden0|145|145|válido|
|practice_0143|shared|E3|donante0/coste0/orden1|145|145|válido|
|practice_0143|shared|E3|donante0/coste1/orden1|145|145|válido|
|practice_0143|shared|E3|donante1/coste0/orden0|156|156|válido|
|practice_0143|shared|E3|donante1/coste1/orden0|156|156|válido|
|practice_0143|shared|E3|donante1/coste0/orden1|156|156|válido|
|practice_0143|shared|E3|donante1/coste1/orden1|156|156|válido|
|practice_0143|shared|E3|donante2/coste0/orden0|160|160|válido|
|practice_0143|shared|E3|donante2/coste1/orden0|160|160|válido|
|practice_0143|shared|E3|donante2/coste0/orden1|160|160|válido|
|practice_0143|shared|E3|donante2/coste1/orden1|160|160|válido|
|practice_0143|shared|E3|donante3/coste0/orden0|152|152|válido|
|practice_0143|shared|E3|donante3/coste1/orden0|152|152|válido|
|practice_0143|shared|E3|donante3/coste0/orden1|152|152|válido|
|practice_0143|shared|E3|donante3/coste1/orden1|152|152|válido|
|practice_0143|shared|E3|donante4/coste0/orden0|165|165|válido|
|practice_0143|shared|E3|donante4/coste1/orden0|165|165|válido|
|practice_0143|shared|E3|donante4/coste0/orden1|165|165|válido|
|practice_0143|shared|E3|donante4/coste1/orden1|165|165|válido|
|practice_0143|shared|E3|donante5/coste0/orden0|167|167|válido|
|practice_0143|shared|E3|donante5/coste1/orden0|167|167|válido|
|practice_0143|shared|E3|donante5/coste0/orden1|167|167|válido|
|practice_0143|shared|E3|donante5/coste1/orden1|167|167|válido|
|practice_0143|shared|E3|donante6/coste0/orden0|151|151|válido|
|practice_0143|shared|E3|donante6/coste1/orden0|151|151|válido|
|practice_0143|shared|E3|donante6/coste0/orden1|151|151|válido|
|practice_0143|shared|E3|donante6/coste1/orden1|151|151|válido|
|practice_0143|shared|E3|donante7/coste0/orden0|150|150|válido|
|practice_0143|shared|E3|donante7/coste1/orden0|150|150|válido|
|practice_0143|shared|E3|donante7/coste0/orden1|150|150|válido|
|practice_0143|shared|E3|donante7/coste1/orden1|150|150|válido|
|practice_0143|shared|E3|donante8/coste0/orden0|155|155|válido|
|practice_0143|shared|E3|donante8/coste1/orden0|155|155|válido|
|practice_0143|shared|E3|donante8/coste0/orden1|155|155|válido|
|practice_0143|shared|E3|donante8/coste1/orden1|155|155|válido|
|practice_0144|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0144|multiplicative|Cruce|CP True/coste0/orden0|111|111|válido|
|practice_0144|multiplicative|Cruce|CP True/coste1/orden0|111|111|válido|
|practice_0144|multiplicative|Cruce|CP True/coste0/orden1|111|111|válido|
|practice_0144|multiplicative|Cruce|CP True/coste1/orden1|111|111|válido|
|practice_0144|multiplicative|Cruce|mejor independiente/coste0/orden0|108|108|válido|
|practice_0144|multiplicative|Cruce|mejor independiente/coste1/orden0|108|108|válido|
|practice_0144|multiplicative|Cruce|mejor independiente/coste0/orden1|108|108|válido|
|practice_0144|multiplicative|Cruce|mejor independiente/coste1/orden1|108|108|válido|
|practice_0144|multiplicative|E1|cola10/pol1/inicio0|115|36|válido|
|practice_0144|multiplicative|E1|cola10/pol1/inicio1|115|36|válido|
|practice_0144|multiplicative|E1|cola10/pol2/inicio0|115|36|válido|
|practice_0144|multiplicative|E1|cola10/pol2/inicio1|115|36|válido|
|practice_0144|multiplicative|E1|cola8/pol1/inicio0|115|32|válido|
|practice_0144|multiplicative|E1|cola8/pol1/inicio1|115|32|válido|
|practice_0144|multiplicative|E1|cola8/pol2/inicio0|115|32|válido|
|practice_0144|multiplicative|E1|cola8/pol2/inicio1|115|32|válido|
|practice_0144|multiplicative|E1|cola6/pol1/inicio0|116|26|válido|
|practice_0144|multiplicative|E1|cola6/pol1/inicio1|116|26|válido|
|practice_0144|multiplicative|E1|cola6/pol2/inicio0|116|26|válido|
|practice_0144|multiplicative|E1|cola6/pol2/inicio1|116|26|válido|
|practice_0144|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0144|multiplicative|E2|CP False/coste0/orden0|113|113|válido|
|practice_0144|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0144|multiplicative|E2|CP True/coste0/orden0|108|108|válido|
|practice_0144|multiplicative|E3|donante0/coste0/orden0|111|111|válido|
|practice_0144|multiplicative|E3|donante0/coste1/orden0|111|111|válido|
|practice_0144|multiplicative|E3|donante0/coste0/orden1|111|111|válido|
|practice_0144|multiplicative|E3|donante0/coste1/orden1|111|111|válido|
|practice_0144|multiplicative|E3|donante1/coste0/orden0|116|116|válido|
|practice_0144|multiplicative|E3|donante1/coste1/orden0|116|116|válido|
|practice_0144|multiplicative|E3|donante1/coste0/orden1|116|116|válido|
|practice_0144|multiplicative|E3|donante1/coste1/orden1|116|116|válido|
|practice_0144|multiplicative|E3|donante2/coste0/orden0|119|119|válido|
|practice_0144|multiplicative|E3|donante2/coste1/orden0|119|119|válido|
|practice_0144|multiplicative|E3|donante2/coste0/orden1|119|119|válido|
|practice_0144|multiplicative|E3|donante2/coste1/orden1|119|119|válido|
|practice_0144|multiplicative|E3|donante3/coste0/orden0|125|125|válido|
|practice_0144|multiplicative|E3|donante3/coste1/orden0|125|125|válido|
|practice_0144|multiplicative|E3|donante3/coste0/orden1|125|125|válido|
|practice_0144|multiplicative|E3|donante3/coste1/orden1|125|125|válido|
|practice_0144|multiplicative|E3|donante4/coste0/orden0|118|118|válido|
|practice_0144|multiplicative|E3|donante4/coste1/orden0|118|118|válido|
|practice_0144|multiplicative|E3|donante4/coste0/orden1|118|118|válido|
|practice_0144|multiplicative|E3|donante4/coste1/orden1|118|118|válido|
|practice_0144|multiplicative|E3|donante5/coste0/orden0|114|114|válido|
|practice_0144|multiplicative|E3|donante5/coste1/orden0|114|114|válido|
|practice_0144|multiplicative|E3|donante5/coste0/orden1|114|114|válido|
|practice_0144|multiplicative|E3|donante5/coste1/orden1|114|114|válido|
|practice_0144|multiplicative|E3|donante6/coste0/orden0|114|114|válido|
|practice_0144|multiplicative|E3|donante6/coste1/orden0|114|114|válido|
|practice_0144|multiplicative|E3|donante6/coste0/orden1|114|114|válido|
|practice_0144|multiplicative|E3|donante6/coste1/orden1|114|114|válido|
|practice_0144|multiplicative|E3|donante7/coste0/orden0|115|115|válido|
|practice_0144|multiplicative|E3|donante7/coste1/orden0|115|115|válido|
|practice_0144|multiplicative|E3|donante7/coste0/orden1|115|115|válido|
|practice_0144|multiplicative|E3|donante7/coste1/orden1|115|115|válido|
|practice_0144|multiplicative|E3|donante8/coste0/orden0|111|111|válido|
|practice_0144|multiplicative|E3|donante8/coste1/orden0|111|111|válido|
|practice_0144|multiplicative|E3|donante8/coste0/orden1|111|111|válido|
|practice_0144|multiplicative|E3|donante8/coste1/orden1|111|111|válido|
|practice_0145|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0145|shared|Cruce|CP True/coste0/orden0|177|169|válido|
|practice_0145|shared|Cruce|CP True/coste1/orden0|181|169|válido|
|practice_0145|shared|Cruce|CP True/coste0/orden1|169|169|válido|
|practice_0145|shared|Cruce|CP True/coste1/orden1|169|169|válido|
|practice_0145|shared|Cruce|mejor independiente/coste0/orden0|190|174|válido|
|practice_0145|shared|Cruce|mejor independiente/coste1/orden0|183|174|válido|
|practice_0145|shared|Cruce|mejor independiente/coste0/orden1|188|174|válido|
|practice_0145|shared|Cruce|mejor independiente/coste1/orden1|188|174|válido|
|practice_0145|shared|E1|cola10/pol1/inicio0|186|51|válido|
|practice_0145|shared|E1|cola10/pol1/inicio1|183|48|válido|
|practice_0145|shared|E1|cola10/pol2/inicio0|183|48|válido|
|practice_0145|shared|E1|cola10/pol2/inicio1|183|48|válido|
|practice_0145|shared|E1|cola8/pol1/inicio0|182|38|válido|
|practice_0145|shared|E1|cola8/pol1/inicio1|182|38|válido|
|practice_0145|shared|E1|cola8/pol2/inicio0|182|38|válido|
|practice_0145|shared|E1|cola8/pol2/inicio1|182|38|válido|
|practice_0145|shared|E1|cola6/pol1/inicio0|180|30|válido|
|practice_0145|shared|E1|cola6/pol1/inicio1|180|30|válido|
|practice_0145|shared|E1|cola6/pol2/inicio0|180|30|válido|
|practice_0145|shared|E1|cola6/pol2/inicio1|180|30|válido|
|practice_0145|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0145|shared|E2|CP False/coste0/orden0|212|179|válido|
|practice_0145|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0145|shared|E2|CP True/coste0/orden0|211|177|válido|
|practice_0145|shared|E3|donante0/coste0/orden0|179|179|válido|
|practice_0145|shared|E3|donante0/coste1/orden0|179|179|válido|
|practice_0145|shared|E3|donante0/coste0/orden1|179|179|válido|
|practice_0145|shared|E3|donante0/coste1/orden1|179|179|válido|
|practice_0145|shared|E3|donante1/coste0/orden0|207|207|válido|
|practice_0145|shared|E3|donante1/coste1/orden0|207|207|válido|
|practice_0145|shared|E3|donante1/coste0/orden1|207|207|válido|
|practice_0145|shared|E3|donante1/coste1/orden1|207|207|válido|
|practice_0145|shared|E3|donante2/coste0/orden0|189|189|válido|
|practice_0145|shared|E3|donante2/coste1/orden0|189|189|válido|
|practice_0145|shared|E3|donante2/coste0/orden1|189|189|válido|
|practice_0145|shared|E3|donante2/coste1/orden1|189|189|válido|
|practice_0145|shared|E3|donante3/coste0/orden0|180|180|válido|
|practice_0145|shared|E3|donante3/coste1/orden0|180|180|válido|
|practice_0145|shared|E3|donante3/coste0/orden1|180|180|válido|
|practice_0145|shared|E3|donante3/coste1/orden1|180|180|válido|
|practice_0145|shared|E3|donante4/coste0/orden0|234|234|válido|
|practice_0145|shared|E3|donante4/coste1/orden0|234|234|válido|
|practice_0145|shared|E3|donante4/coste0/orden1|234|234|válido|
|practice_0145|shared|E3|donante4/coste1/orden1|234|234|válido|
|practice_0145|shared|E3|donante5/coste0/orden0|191|191|válido|
|practice_0145|shared|E3|donante5/coste1/orden0|191|191|válido|
|practice_0145|shared|E3|donante5/coste0/orden1|191|191|válido|
|practice_0145|shared|E3|donante5/coste1/orden1|191|191|válido|
|practice_0145|shared|E3|donante6/coste0/orden0|200|200|válido|
|practice_0145|shared|E3|donante6/coste1/orden0|200|200|válido|
|practice_0145|shared|E3|donante6/coste0/orden1|200|200|válido|
|practice_0145|shared|E3|donante6/coste1/orden1|200|200|válido|
|practice_0145|shared|E3|donante7/coste0/orden0|189|189|válido|
|practice_0145|shared|E3|donante7/coste1/orden0|189|189|válido|
|practice_0145|shared|E3|donante7/coste0/orden1|202|189|válido|
|practice_0145|shared|E3|donante7/coste1/orden1|202|189|válido|
|practice_0145|shared|E3|donante8/coste0/orden0|180|180|válido|
|practice_0145|shared|E3|donante8/coste1/orden0|180|180|válido|
|practice_0145|shared|E3|donante8/coste0/orden1|180|180|válido|
|practice_0145|shared|E3|donante8/coste1/orden1|180|180|válido|
|practice_0145|shared|E3|donante9/coste0/orden0|198|174|válido|
|practice_0145|shared|E3|donante9/coste1/orden0|193|174|válido|
|practice_0145|shared|E3|donante9/coste0/orden1|207|174|válido|
|practice_0145|shared|E3|donante9/coste1/orden1|213|174|válido|
|practice_0146|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0146|mixed|Cruce|CP True/coste0/orden0|38|38|válido|
|practice_0146|mixed|Cruce|CP True/coste1/orden0|38|38|válido|
|practice_0146|mixed|Cruce|CP True/coste0/orden1|38|38|válido|
|practice_0146|mixed|Cruce|CP True/coste1/orden1|38|38|válido|
|practice_0146|mixed|Cruce|mejor independiente/coste0/orden0|40|40|válido|
|practice_0146|mixed|Cruce|mejor independiente/coste1/orden0|40|40|válido|
|practice_0146|mixed|Cruce|mejor independiente/coste0/orden1|40|40|válido|
|practice_0146|mixed|Cruce|mejor independiente/coste1/orden1|40|40|válido|
|practice_0146|mixed|E1|cola8/pol1/inicio0|47|38|válido|
|practice_0146|mixed|E1|cola8/pol1/inicio1|42|33|válido|
|practice_0146|mixed|E1|cola8/pol2/inicio0|44|35|válido|
|practice_0146|mixed|E1|cola8/pol2/inicio1|42|33|válido|
|practice_0146|mixed|E1|cola6/pol1/inicio0|42|17|válido|
|practice_0146|mixed|E1|cola6/pol1/inicio1|42|17|válido|
|practice_0146|mixed|E1|cola6/pol2/inicio0|42|17|válido|
|practice_0146|mixed|E1|cola6/pol2/inicio1|42|17|válido|
|practice_0146|mixed|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0146|mixed|E2|CP False/coste0/orden0|42|42|válido|
|practice_0146|mixed|E2|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0146|mixed|E2|CP True/coste0/orden0|40|40|válido|
|practice_0146|mixed|E3|donante0/coste0/orden0|42|42|válido|
|practice_0146|mixed|E3|donante0/coste1/orden0|42|42|válido|
|practice_0146|mixed|E3|donante0/coste0/orden1|42|42|válido|
|practice_0146|mixed|E3|donante0/coste1/orden1|42|42|válido|
|practice_0146|mixed|E3|donante1/coste0/orden0|47|47|válido|
|practice_0146|mixed|E3|donante1/coste1/orden0|47|47|válido|
|practice_0146|mixed|E3|donante1/coste0/orden1|47|47|válido|
|practice_0146|mixed|E3|donante1/coste1/orden1|47|47|válido|
|practice_0146|mixed|E3|donante2/coste0/orden0|43|43|válido|
|practice_0146|mixed|E3|donante2/coste1/orden0|43|43|válido|
|practice_0146|mixed|E3|donante2/coste0/orden1|43|43|válido|
|practice_0146|mixed|E3|donante2/coste1/orden1|43|43|válido|
|practice_0146|mixed|E3|donante3/coste0/orden0|47|47|válido|
|practice_0146|mixed|E3|donante3/coste1/orden0|47|47|válido|
|practice_0146|mixed|E3|donante3/coste0/orden1|47|47|válido|
|practice_0146|mixed|E3|donante3/coste1/orden1|47|47|válido|
|practice_0146|mixed|E3|donante4/coste0/orden0|47|47|válido|
|practice_0146|mixed|E3|donante4/coste1/orden0|47|47|válido|
|practice_0146|mixed|E3|donante4/coste0/orden1|47|47|válido|
|practice_0146|mixed|E3|donante4/coste1/orden1|47|47|válido|
|practice_0146|mixed|E3|donante5/coste0/orden0|45|45|válido|
|practice_0146|mixed|E3|donante5/coste1/orden0|45|45|válido|
|practice_0146|mixed|E3|donante5/coste0/orden1|45|45|válido|
|practice_0146|mixed|E3|donante5/coste1/orden1|45|45|válido|
|practice_0146|mixed|E3|donante6/coste0/orden0|45|45|válido|
|practice_0146|mixed|E3|donante6/coste1/orden0|45|45|válido|
|practice_0146|mixed|E3|donante6/coste0/orden1|45|45|válido|
|practice_0146|mixed|E3|donante6/coste1/orden1|45|45|válido|
|practice_0147|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0147|shared|Cruce|CP True/coste0/orden0|171|166|válido|
|practice_0147|shared|Cruce|CP True/coste1/orden0|183|166|válido|
|practice_0147|shared|Cruce|CP True/coste0/orden1|166|166|válido|
|practice_0147|shared|Cruce|CP True/coste1/orden1|166|166|válido|
|practice_0147|shared|Cruce|mejor independiente/coste0/orden0|189|167|válido|
|practice_0147|shared|Cruce|mejor independiente/coste1/orden0|187|167|válido|
|practice_0147|shared|Cruce|mejor independiente/coste0/orden1|167|167|válido|
|practice_0147|shared|Cruce|mejor independiente/coste1/orden1|167|167|válido|
|practice_0147|shared|E1|cola10/pol1/inicio0|171|56|válido|
|practice_0147|shared|E1|cola10/pol1/inicio1|171|56|válido|
|practice_0147|shared|E1|cola10/pol2/inicio0|173|58|válido|
|practice_0147|shared|E1|cola10/pol2/inicio1|173|58|válido|
|practice_0147|shared|E1|cola8/pol1/inicio0|173|48|válido|
|practice_0147|shared|E1|cola8/pol1/inicio1|173|48|válido|
|practice_0147|shared|E1|cola8/pol2/inicio0|170|45|válido|
|practice_0147|shared|E1|cola8/pol2/inicio1|170|45|válido|
|practice_0147|shared|E1|cola6/pol1/inicio0|172|38|válido|
|practice_0147|shared|E1|cola6/pol1/inicio1|172|38|válido|
|practice_0147|shared|E1|cola6/pol2/inicio0|172|38|válido|
|practice_0147|shared|E1|cola6/pol2/inicio1|172|38|válido|
|practice_0147|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0147|shared|E2|CP False/coste0/orden0|194|173|válido|
|practice_0147|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0147|shared|E2|CP True/coste0/orden0|171|166|válido|
|practice_0147|shared|E3|donante0/coste0/orden0|167|167|válido|
|practice_0147|shared|E3|donante0/coste1/orden0|167|167|válido|
|practice_0147|shared|E3|donante0/coste0/orden1|167|167|válido|
|practice_0147|shared|E3|donante0/coste1/orden1|167|167|válido|
|practice_0147|shared|E3|donante1/coste0/orden0|188|188|válido|
|practice_0147|shared|E3|donante1/coste1/orden0|188|188|válido|
|practice_0147|shared|E3|donante1/coste0/orden1|188|188|válido|
|practice_0147|shared|E3|donante1/coste1/orden1|188|188|válido|
|practice_0147|shared|E3|donante2/coste0/orden0|175|175|válido|
|practice_0147|shared|E3|donante2/coste1/orden0|175|175|válido|
|practice_0147|shared|E3|donante2/coste0/orden1|175|175|válido|
|practice_0147|shared|E3|donante2/coste1/orden1|175|175|válido|
|practice_0147|shared|E3|donante3/coste0/orden0|175|175|válido|
|practice_0147|shared|E3|donante3/coste1/orden0|175|175|válido|
|practice_0147|shared|E3|donante3/coste0/orden1|175|175|válido|
|practice_0147|shared|E3|donante3/coste1/orden1|175|175|válido|
|practice_0147|shared|E3|donante4/coste0/orden0|207|207|válido|
|practice_0147|shared|E3|donante4/coste1/orden0|207|207|válido|
|practice_0147|shared|E3|donante4/coste0/orden1|207|207|válido|
|practice_0147|shared|E3|donante4/coste1/orden1|207|207|válido|
|practice_0147|shared|E3|donante5/coste0/orden0|186|186|válido|
|practice_0147|shared|E3|donante5/coste1/orden0|186|186|válido|
|practice_0147|shared|E3|donante5/coste0/orden1|186|186|válido|
|practice_0147|shared|E3|donante5/coste1/orden1|186|186|válido|
|practice_0147|shared|E3|donante6/coste0/orden0|196|196|válido|
|practice_0147|shared|E3|donante6/coste1/orden0|196|196|válido|
|practice_0147|shared|E3|donante6/coste0/orden1|196|196|válido|
|practice_0147|shared|E3|donante6/coste1/orden1|196|196|válido|
|practice_0147|shared|E3|donante7/coste0/orden0|191|179|válido|
|practice_0147|shared|E3|donante7/coste1/orden0|191|179|válido|
|practice_0147|shared|E3|donante7/coste0/orden1|179|179|válido|
|practice_0147|shared|E3|donante7/coste1/orden1|179|179|válido|
|practice_0147|shared|E3|donante8/coste0/orden0|186|170|válido|
|practice_0147|shared|E3|donante8/coste1/orden0|181|170|válido|
|practice_0147|shared|E3|donante8/coste0/orden1|186|170|válido|
|practice_0147|shared|E3|donante8/coste1/orden1|181|170|válido|
|practice_0147|shared|E3|donante9/coste0/orden0|168|168|válido|
|practice_0147|shared|E3|donante9/coste1/orden0|168|168|válido|
|practice_0147|shared|E3|donante9/coste0/orden1|168|168|válido|
|practice_0147|shared|E3|donante9/coste1/orden1|168|168|válido|
|practice_0148|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0148|shared|Cruce|CP True/coste0/orden0|157|157|válido|
|practice_0148|shared|Cruce|CP True/coste1/orden0|157|157|válido|
|practice_0148|shared|Cruce|CP True/coste0/orden1|157|157|válido|
|practice_0148|shared|Cruce|CP True/coste1/orden1|157|157|válido|
|practice_0148|shared|Cruce|mejor independiente/coste0/orden0|159|159|válido|
|practice_0148|shared|Cruce|mejor independiente/coste1/orden0|159|159|válido|
|practice_0148|shared|Cruce|mejor independiente/coste0/orden1|159|159|válido|
|practice_0148|shared|Cruce|mejor independiente/coste1/orden1|159|159|válido|
|practice_0148|shared|E1|cola10/pol1/inicio0|160|51|válido|
|practice_0148|shared|E1|cola10/pol1/inicio1|160|51|válido|
|practice_0148|shared|E1|cola10/pol2/inicio0|162|53|válido|
|practice_0148|shared|E1|cola10/pol2/inicio1|162|53|válido|
|practice_0148|shared|E1|cola8/pol1/inicio0|162|42|válido|
|practice_0148|shared|E1|cola8/pol1/inicio1|162|42|válido|
|practice_0148|shared|E1|cola8/pol2/inicio0|162|42|válido|
|practice_0148|shared|E1|cola8/pol2/inicio1|162|42|válido|
|practice_0148|shared|E1|cola6/pol1/inicio0|162|34|válido|
|practice_0148|shared|E1|cola6/pol1/inicio1|162|34|válido|
|practice_0148|shared|E1|cola6/pol2/inicio0|161|33|válido|
|practice_0148|shared|E1|cola6/pol2/inicio1|161|33|válido|
|practice_0148|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0148|shared|E2|CP False/coste0/orden0|170|170|válido|
|practice_0148|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0148|shared|E2|CP True/coste0/orden0|161|161|válido|
|practice_0148|shared|E3|donante0/coste0/orden0|164|164|válido|
|practice_0148|shared|E3|donante0/coste1/orden0|164|164|válido|
|practice_0148|shared|E3|donante0/coste0/orden1|164|164|válido|
|practice_0148|shared|E3|donante0/coste1/orden1|164|164|válido|
|practice_0148|shared|E3|donante1/coste0/orden0|176|176|válido|
|practice_0148|shared|E3|donante1/coste1/orden0|176|176|válido|
|practice_0148|shared|E3|donante1/coste0/orden1|176|176|válido|
|practice_0148|shared|E3|donante1/coste1/orden1|176|176|válido|
|practice_0148|shared|E3|donante2/coste0/orden0|168|168|válido|
|practice_0148|shared|E3|donante2/coste1/orden0|168|168|válido|
|practice_0148|shared|E3|donante2/coste0/orden1|168|168|válido|
|practice_0148|shared|E3|donante2/coste1/orden1|168|168|válido|
|practice_0148|shared|E3|donante3/coste0/orden0|179|179|válido|
|practice_0148|shared|E3|donante3/coste1/orden0|179|179|válido|
|practice_0148|shared|E3|donante3/coste0/orden1|179|179|válido|
|practice_0148|shared|E3|donante3/coste1/orden1|179|179|válido|
|practice_0148|shared|E3|donante4/coste0/orden0|175|175|válido|
|practice_0148|shared|E3|donante4/coste1/orden0|175|175|válido|
|practice_0148|shared|E3|donante4/coste0/orden1|175|175|válido|
|practice_0148|shared|E3|donante4/coste1/orden1|175|175|válido|
|practice_0148|shared|E3|donante5/coste0/orden0|176|176|válido|
|practice_0148|shared|E3|donante5/coste1/orden0|176|176|válido|
|practice_0148|shared|E3|donante5/coste0/orden1|176|176|válido|
|practice_0148|shared|E3|donante5/coste1/orden1|176|176|válido|
|practice_0148|shared|E3|donante6/coste0/orden0|184|163|válido|
|practice_0148|shared|E3|donante6/coste1/orden0|197|163|válido|
|practice_0148|shared|E3|donante6/coste0/orden1|205|163|válido|
|practice_0148|shared|E3|donante6/coste1/orden1|195|163|válido|
|practice_0148|shared|E3|donante7/coste0/orden0|201|160|válido|
|practice_0148|shared|E3|donante7/coste1/orden0|189|160|válido|
|practice_0148|shared|E3|donante7/coste0/orden1|179|160|válido|
|practice_0148|shared|E3|donante7/coste1/orden1|171|160|válido|
|practice_0148|shared|E3|donante8/coste0/orden0|184|156|válido|
|practice_0148|shared|E3|donante8/coste1/orden0|165|156|válido|
|practice_0148|shared|E3|donante8/coste0/orden1|187|156|válido|
|practice_0148|shared|E3|donante8/coste1/orden1|186|156|válido|
|practice_0149|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0149|shared|Cruce|CP True/coste0/orden0|196|168|válido|
|practice_0149|shared|Cruce|CP True/coste1/orden0|189|168|válido|
|practice_0149|shared|Cruce|CP True/coste0/orden1|168|168|válido|
|practice_0149|shared|Cruce|CP True/coste1/orden1|168|168|válido|
|practice_0149|shared|Cruce|mejor independiente/coste0/orden0|226|170|válido|
|practice_0149|shared|Cruce|mejor independiente/coste1/orden0|192|170|válido|
|practice_0149|shared|Cruce|mejor independiente/coste0/orden1|170|170|válido|
|practice_0149|shared|Cruce|mejor independiente/coste1/orden1|170|170|válido|
|practice_0149|shared|E1|cola10/pol1/inicio0|179|62|válido|
|practice_0149|shared|E1|cola10/pol1/inicio1|179|62|válido|
|practice_0149|shared|E1|cola10/pol2/inicio0|175|58|válido|
|practice_0149|shared|E1|cola10/pol2/inicio1|175|58|válido|
|practice_0149|shared|E1|cola8/pol1/inicio0|179|51|válido|
|practice_0149|shared|E1|cola8/pol1/inicio1|179|51|válido|
|practice_0149|shared|E1|cola8/pol2/inicio0|175|47|válido|
|practice_0149|shared|E1|cola8/pol2/inicio1|175|47|válido|
|practice_0149|shared|E1|cola6/pol1/inicio0|176|38|válido|
|practice_0149|shared|E1|cola6/pol1/inicio1|176|38|válido|
|practice_0149|shared|E1|cola6/pol2/inicio0|174|36|válido|
|practice_0149|shared|E1|cola6/pol2/inicio1|174|36|válido|
|practice_0149|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0149|shared|E2|CP False/coste0/orden0|215|172|válido|
|practice_0149|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0149|shared|E2|CP True/coste0/orden0|200|171|válido|
|practice_0149|shared|E3|donante0/coste0/orden0|172|172|válido|
|practice_0149|shared|E3|donante0/coste1/orden0|172|172|válido|
|practice_0149|shared|E3|donante0/coste0/orden1|172|172|válido|
|practice_0149|shared|E3|donante0/coste1/orden1|172|172|válido|
|practice_0149|shared|E3|donante1/coste0/orden0|203|203|válido|
|practice_0149|shared|E3|donante1/coste1/orden0|203|203|válido|
|practice_0149|shared|E3|donante1/coste0/orden1|203|203|válido|
|practice_0149|shared|E3|donante1/coste1/orden1|203|203|válido|
|practice_0149|shared|E3|donante2/coste0/orden0|194|194|válido|
|practice_0149|shared|E3|donante2/coste1/orden0|194|194|válido|
|practice_0149|shared|E3|donante2/coste0/orden1|194|194|válido|
|practice_0149|shared|E3|donante2/coste1/orden1|194|194|válido|
|practice_0149|shared|E3|donante3/coste0/orden0|203|193|válido|
|practice_0149|shared|E3|donante3/coste1/orden0|218|193|válido|
|practice_0149|shared|E3|donante3/coste0/orden1|193|193|válido|
|practice_0149|shared|E3|donante3/coste1/orden1|193|193|válido|
|practice_0149|shared|E3|donante4/coste0/orden0|213|213|válido|
|practice_0149|shared|E3|donante4/coste1/orden0|213|213|válido|
|practice_0149|shared|E3|donante4/coste0/orden1|213|213|válido|
|practice_0149|shared|E3|donante4/coste1/orden1|213|213|válido|
|practice_0149|shared|E3|donante5/coste0/orden0|197|197|válido|
|practice_0149|shared|E3|donante5/coste1/orden0|197|197|válido|
|practice_0149|shared|E3|donante5/coste0/orden1|197|197|válido|
|practice_0149|shared|E3|donante5/coste1/orden1|197|197|válido|
|practice_0149|shared|E3|donante6/coste0/orden0|215|215|válido|
|practice_0149|shared|E3|donante6/coste1/orden0|215|215|válido|
|practice_0149|shared|E3|donante6/coste0/orden1|215|215|válido|
|practice_0149|shared|E3|donante6/coste1/orden1|215|215|válido|
|practice_0149|shared|E3|donante7/coste0/orden0|202|187|válido|
|practice_0149|shared|E3|donante7/coste1/orden0|204|187|válido|
|practice_0149|shared|E3|donante7/coste0/orden1|219|187|válido|
|practice_0149|shared|E3|donante7/coste1/orden1|233|187|válido|
|practice_0149|shared|E3|donante8/coste0/orden0|192|181|válido|
|practice_0149|shared|E3|donante8/coste1/orden0|192|181|válido|
|practice_0149|shared|E3|donante8/coste0/orden1|204|181|válido|
|practice_0149|shared|E3|donante8/coste1/orden1|204|181|válido|
|practice_0150|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0150|independent|Cruce|CP True/coste0/orden0|209|185|válido|
|practice_0150|independent|Cruce|CP True/coste1/orden0|216|185|válido|
|practice_0150|independent|Cruce|CP True/coste0/orden1|185|185|válido|
|practice_0150|independent|Cruce|CP True/coste1/orden1|185|185|válido|
|practice_0150|independent|Cruce|mejor independiente/coste0/orden0|209|185|válido|
|practice_0150|independent|Cruce|mejor independiente/coste1/orden0|216|185|válido|
|practice_0150|independent|Cruce|mejor independiente/coste0/orden1|185|185|válido|
|practice_0150|independent|Cruce|mejor independiente/coste1/orden1|185|185|válido|
|practice_0150|independent|E1|cola10/pol1/inicio0|198|62|válido|
|practice_0150|independent|E1|cola10/pol1/inicio1|199|63|válido|
|practice_0150|independent|E1|cola10/pol2/inicio0|194|58|válido|
|practice_0150|independent|E1|cola10/pol2/inicio1|194|58|válido|
|practice_0150|independent|E1|cola8/pol1/inicio0|196|49|válido|
|practice_0150|independent|E1|cola8/pol1/inicio1|196|49|válido|
|practice_0150|independent|E1|cola8/pol2/inicio0|191|44|válido|
|practice_0150|independent|E1|cola8/pol2/inicio1|191|44|válido|
|practice_0150|independent|E1|cola6/pol1/inicio0|198|35|válido|
|practice_0150|independent|E1|cola6/pol1/inicio1|198|35|válido|
|practice_0150|independent|E1|cola6/pol2/inicio0|191|28|válido|
|practice_0150|independent|E1|cola6/pol2/inicio1|191|28|válido|
|practice_0150|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0150|independent|E2|CP False/coste0/orden0|197|186|válido|
|practice_0150|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0150|independent|E2|CP True/coste0/orden0|218|183|válido|
|practice_0150|independent|E3|donante0/coste0/orden0|197|185|válido|
|practice_0150|independent|E3|donante0/coste1/orden0|188|185|válido|
|practice_0150|independent|E3|donante0/coste0/orden1|214|185|válido|
|practice_0150|independent|E3|donante0/coste1/orden1|206|185|válido|
|practice_0150|independent|E3|donante1/coste0/orden0|203|203|válido|
|practice_0150|independent|E3|donante1/coste1/orden0|203|203|válido|
|practice_0150|independent|E3|donante1/coste0/orden1|203|203|válido|
|practice_0150|independent|E3|donante1/coste1/orden1|203|203|válido|
|practice_0150|independent|E3|donante2/coste0/orden0|200|194|válido|
|practice_0150|independent|E3|donante2/coste1/orden0|200|194|válido|
|practice_0150|independent|E3|donante2/coste0/orden1|194|194|válido|
|practice_0150|independent|E3|donante2/coste1/orden1|194|194|válido|
|practice_0150|independent|E3|donante3/coste0/orden0|210|186|válido|
|practice_0150|independent|E3|donante3/coste1/orden0|217|186|válido|
|practice_0150|independent|E3|donante3/coste0/orden1|186|186|válido|
|practice_0150|independent|E3|donante3/coste1/orden1|186|186|válido|
|practice_0150|independent|E3|donante4/coste0/orden0|213|213|válido|
|practice_0150|independent|E3|donante4/coste1/orden0|213|213|válido|
|practice_0150|independent|E3|donante4/coste0/orden1|213|213|válido|
|practice_0150|independent|E3|donante4/coste1/orden1|213|213|válido|
|practice_0150|independent|E3|donante5/coste0/orden0|196|196|válido|
|practice_0150|independent|E3|donante5/coste1/orden0|196|196|válido|
|practice_0150|independent|E3|donante5/coste0/orden1|196|196|válido|
|practice_0150|independent|E3|donante5/coste1/orden1|196|196|válido|
|practice_0150|independent|E3|donante6/coste0/orden0|205|205|válido|
|practice_0150|independent|E3|donante6/coste1/orden0|205|205|válido|
|practice_0150|independent|E3|donante6/coste0/orden1|205|205|válido|
|practice_0150|independent|E3|donante6/coste1/orden1|205|205|válido|
|practice_0150|independent|E3|donante7/coste0/orden0|226|189|válido|
|practice_0150|independent|E3|donante7/coste1/orden0|212|189|válido|
|practice_0150|independent|E3|donante7/coste0/orden1|228|189|válido|
|practice_0150|independent|E3|donante7/coste1/orden1|221|189|válido|
|practice_0150|independent|E3|donante8/coste0/orden0|197|185|válido|
|practice_0150|independent|E3|donante8/coste1/orden0|188|185|válido|
|practice_0150|independent|E3|donante8/coste0/orden1|214|185|válido|
|practice_0150|independent|E3|donante8/coste1/orden1|206|185|válido|
|practice_0150|independent|E3|donante9/coste0/orden0|227|181|válido|
|practice_0150|independent|E3|donante9/coste1/orden0|203|181|válido|
|practice_0150|independent|E3|donante9/coste0/orden1|211|181|válido|
|practice_0150|independent|E3|donante9/coste1/orden1|206|181|válido|
|practice_0151|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0151|independent|Cruce|CP True/coste0/orden0|291|290|válido|
|practice_0151|independent|Cruce|CP True/coste1/orden0|291|290|válido|
|practice_0151|independent|Cruce|CP True/coste0/orden1|294|290|válido|
|practice_0151|independent|Cruce|CP True/coste1/orden1|291|290|válido|
|practice_0151|independent|Cruce|mejor independiente/coste0/orden0|291|290|válido|
|practice_0151|independent|Cruce|mejor independiente/coste1/orden0|291|290|válido|
|practice_0151|independent|Cruce|mejor independiente/coste0/orden1|294|290|válido|
|practice_0151|independent|Cruce|mejor independiente/coste1/orden1|291|290|válido|
|practice_0151|independent|E1|cola10/pol1/inicio0|299|84|válido|
|practice_0151|independent|E1|cola10/pol1/inicio1|299|84|válido|
|practice_0151|independent|E1|cola10/pol2/inicio0|300|85|válido|
|practice_0151|independent|E1|cola10/pol2/inicio1|300|85|válido|
|practice_0151|independent|E1|cola8/pol1/inicio0|304|74|válido|
|practice_0151|independent|E1|cola8/pol1/inicio1|304|74|válido|
|practice_0151|independent|E1|cola8/pol2/inicio0|306|76|válido|
|practice_0151|independent|E1|cola8/pol2/inicio1|305|75|válido|
|practice_0151|independent|E1|cola6/pol1/inicio0|308|60|válido|
|practice_0151|independent|E1|cola6/pol1/inicio1|308|60|válido|
|practice_0151|independent|E1|cola6/pol2/inicio0|303|55|válido|
|practice_0151|independent|E1|cola6/pol2/inicio1|303|55|válido|
|practice_0151|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0151|independent|E2|CP False/coste0/orden0|497|333|válido|
|practice_0151|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0151|independent|E2|CP True/coste0/orden0|291|290|válido|
|practice_0151|independent|E3|donante0/coste0/orden0|293|292|válido|
|practice_0151|independent|E3|donante0/coste1/orden0|293|292|válido|
|practice_0151|independent|E3|donante0/coste0/orden1|295|292|válido|
|practice_0151|independent|E3|donante0/coste1/orden1|293|292|válido|
|practice_0151|independent|E3|donante1/coste0/orden0|333|333|válido|
|practice_0151|independent|E3|donante1/coste1/orden0|333|333|válido|
|practice_0151|independent|E3|donante1/coste0/orden1|333|333|válido|
|practice_0151|independent|E3|donante1/coste1/orden1|333|333|válido|
|practice_0151|independent|E3|donante2/coste0/orden0|412|312|válido|
|practice_0151|independent|E3|donante2/coste1/orden0|356|312|válido|
|practice_0151|independent|E3|donante2/coste0/orden1|312|312|válido|
|practice_0151|independent|E3|donante2/coste1/orden1|312|312|válido|
|practice_0151|independent|E3|donante3/coste0/orden0|474|307|válido|
|practice_0151|independent|E3|donante3/coste1/orden0|455|307|válido|
|practice_0151|independent|E3|donante3/coste0/orden1|309|307|válido|
|practice_0151|independent|E3|donante3/coste1/orden1|309|307|válido|
|practice_0151|independent|E3|donante4/coste0/orden0|336|336|válido|
|practice_0151|independent|E3|donante4/coste1/orden0|336|336|válido|
|practice_0151|independent|E3|donante4/coste0/orden1|348|336|válido|
|practice_0151|independent|E3|donante4/coste1/orden1|345|336|válido|
|practice_0151|independent|E3|donante5/coste0/orden0|336|336|válido|
|practice_0151|independent|E3|donante5/coste1/orden0|336|336|válido|
|practice_0151|independent|E3|donante5/coste0/orden1|336|336|válido|
|practice_0151|independent|E3|donante5/coste1/orden1|336|336|válido|
|practice_0151|independent|E3|donante6/coste0/orden0|347|347|válido|
|practice_0151|independent|E3|donante6/coste1/orden0|347|347|válido|
|practice_0151|independent|E3|donante6/coste0/orden1|347|347|válido|
|practice_0151|independent|E3|donante6/coste1/orden1|347|347|válido|
|practice_0151|independent|E3|donante7/coste0/orden0|295|294|válido|
|practice_0151|independent|E3|donante7/coste1/orden0|295|294|válido|
|practice_0151|independent|E3|donante7/coste0/orden1|297|294|válido|
|practice_0151|independent|E3|donante7/coste1/orden1|295|294|válido|
|practice_0151|independent|E3|donante8/coste0/orden0|345|284|válido|
|practice_0151|independent|E3|donante8/coste1/orden0|306|284|válido|
|practice_0151|independent|E3|donante8/coste0/orden1|364|284|válido|
|practice_0151|independent|E3|donante8/coste1/orden1|353|284|válido|
|practice_0151|independent|E3|donante9/coste0/orden0|337|284|válido|
|practice_0151|independent|E3|donante9/coste1/orden0|340|284|válido|
|practice_0151|independent|E3|donante9/coste0/orden1|339|284|válido|
|practice_0151|independent|E3|donante9/coste1/orden1|378|284|válido|
|practice_0152|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0152|shared|Cruce|CP True/coste0/orden0|144|144|válido|
|practice_0152|shared|Cruce|CP True/coste1/orden0|144|144|válido|
|practice_0152|shared|Cruce|CP True/coste0/orden1|144|144|válido|
|practice_0152|shared|Cruce|CP True/coste1/orden1|144|144|válido|
|practice_0152|shared|Cruce|mejor independiente/coste0/orden0|146|146|válido|
|practice_0152|shared|Cruce|mejor independiente/coste1/orden0|146|146|válido|
|practice_0152|shared|Cruce|mejor independiente/coste0/orden1|146|146|válido|
|practice_0152|shared|Cruce|mejor independiente/coste1/orden1|146|146|válido|
|practice_0152|shared|E1|cola10/pol1/inicio0|151|36|válido|
|practice_0152|shared|E1|cola10/pol1/inicio1|151|36|válido|
|practice_0152|shared|E1|cola10/pol2/inicio0|150|35|válido|
|practice_0152|shared|E1|cola10/pol2/inicio1|150|35|válido|
|practice_0152|shared|E1|cola8/pol1/inicio0|147|23|válido|
|practice_0152|shared|E1|cola8/pol1/inicio1|147|23|válido|
|practice_0152|shared|E1|cola8/pol2/inicio0|147|23|válido|
|practice_0152|shared|E1|cola8/pol2/inicio1|147|23|válido|
|practice_0152|shared|E1|cola6/pol1/inicio0|147|16|válido|
|practice_0152|shared|E1|cola6/pol1/inicio1|147|16|válido|
|practice_0152|shared|E1|cola6/pol2/inicio0|147|16|válido|
|practice_0152|shared|E1|cola6/pol2/inicio1|147|16|válido|
|practice_0152|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0152|shared|E2|CP False/coste0/orden0|148|148|válido|
|practice_0152|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0152|shared|E2|CP True/coste0/orden0|147|147|válido|
|practice_0152|shared|E3|donante0/coste0/orden0|148|148|válido|
|practice_0152|shared|E3|donante0/coste1/orden0|148|148|válido|
|practice_0152|shared|E3|donante0/coste0/orden1|148|148|válido|
|practice_0152|shared|E3|donante0/coste1/orden1|148|148|válido|
|practice_0152|shared|E3|donante1/coste0/orden0|162|162|válido|
|practice_0152|shared|E3|donante1/coste1/orden0|162|162|válido|
|practice_0152|shared|E3|donante1/coste0/orden1|162|162|válido|
|practice_0152|shared|E3|donante1/coste1/orden1|162|162|válido|
|practice_0152|shared|E3|donante2/coste0/orden0|158|158|válido|
|practice_0152|shared|E3|donante2/coste1/orden0|158|158|válido|
|practice_0152|shared|E3|donante2/coste0/orden1|158|158|válido|
|practice_0152|shared|E3|donante2/coste1/orden1|158|158|válido|
|practice_0152|shared|E3|donante3/coste0/orden0|178|178|válido|
|practice_0152|shared|E3|donante3/coste1/orden0|178|178|válido|
|practice_0152|shared|E3|donante3/coste0/orden1|178|178|válido|
|practice_0152|shared|E3|donante3/coste1/orden1|178|178|válido|
|practice_0152|shared|E3|donante4/coste0/orden0|156|156|válido|
|practice_0152|shared|E3|donante4/coste1/orden0|156|156|válido|
|practice_0152|shared|E3|donante4/coste0/orden1|156|156|válido|
|practice_0152|shared|E3|donante4/coste1/orden1|156|156|válido|
|practice_0152|shared|E3|donante5/coste0/orden0|167|167|válido|
|practice_0152|shared|E3|donante5/coste1/orden0|167|167|válido|
|practice_0152|shared|E3|donante5/coste0/orden1|167|167|válido|
|practice_0152|shared|E3|donante5/coste1/orden1|167|167|válido|
|practice_0152|shared|E3|donante6/coste0/orden0|187|158|válido|
|practice_0152|shared|E3|donante6/coste1/orden0|178|158|válido|
|practice_0152|shared|E3|donante6/coste0/orden1|158|158|válido|
|practice_0152|shared|E3|donante6/coste1/orden1|158|158|válido|
|practice_0152|shared|E3|donante7/coste0/orden0|151|151|válido|
|practice_0152|shared|E3|donante7/coste1/orden0|151|151|válido|
|practice_0152|shared|E3|donante7/coste0/orden1|151|151|válido|
|practice_0152|shared|E3|donante7/coste1/orden1|151|151|válido|
|practice_0152|shared|E3|donante8/coste0/orden0|152|152|válido|
|practice_0152|shared|E3|donante8/coste1/orden0|152|152|válido|
|practice_0152|shared|E3|donante8/coste0/orden1|152|152|válido|
|practice_0152|shared|E3|donante8/coste1/orden1|152|152|válido|
|practice_0153|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0153|shared|Cruce|CP True/coste0/orden0|131|131|válido|
|practice_0153|shared|Cruce|CP True/coste1/orden0|131|131|válido|
|practice_0153|shared|Cruce|CP True/coste0/orden1|131|131|válido|
|practice_0153|shared|Cruce|CP True/coste1/orden1|131|131|válido|
|practice_0153|shared|Cruce|mejor independiente/coste0/orden0|133|133|válido|
|practice_0153|shared|Cruce|mejor independiente/coste1/orden0|133|133|válido|
|practice_0153|shared|Cruce|mejor independiente/coste0/orden1|133|133|válido|
|practice_0153|shared|Cruce|mejor independiente/coste1/orden1|133|133|válido|
|practice_0153|shared|E1|cola10/pol1/inicio0|143|39|válido|
|practice_0153|shared|E1|cola10/pol1/inicio1|142|38|válido|
|practice_0153|shared|E1|cola10/pol2/inicio0|141|37|válido|
|practice_0153|shared|E1|cola10/pol2/inicio1|145|41|válido|
|practice_0153|shared|E1|cola8/pol1/inicio0|138|27|válido|
|practice_0153|shared|E1|cola8/pol1/inicio1|138|27|válido|
|practice_0153|shared|E1|cola8/pol2/inicio0|144|33|válido|
|practice_0153|shared|E1|cola8/pol2/inicio1|142|31|válido|
|practice_0153|shared|E1|cola6/pol1/inicio0|143|25|válido|
|practice_0153|shared|E1|cola6/pol1/inicio1|143|25|válido|
|practice_0153|shared|E1|cola6/pol2/inicio0|142|24|válido|
|practice_0153|shared|E1|cola6/pol2/inicio1|138|20|válido|
|practice_0153|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0153|shared|E2|CP False/coste0/orden0|146|146|válido|
|practice_0153|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0153|shared|E2|CP True/coste0/orden0|133|133|válido|
|practice_0153|shared|E3|donante0/coste0/orden0|138|138|válido|
|practice_0153|shared|E3|donante0/coste1/orden0|138|138|válido|
|practice_0153|shared|E3|donante0/coste0/orden1|138|138|válido|
|practice_0153|shared|E3|donante0/coste1/orden1|138|138|válido|
|practice_0153|shared|E3|donante1/coste0/orden0|149|149|válido|
|practice_0153|shared|E3|donante1/coste1/orden0|149|149|válido|
|practice_0153|shared|E3|donante1/coste0/orden1|149|149|válido|
|practice_0153|shared|E3|donante1/coste1/orden1|149|149|válido|
|practice_0153|shared|E3|donante2/coste0/orden0|148|148|válido|
|practice_0153|shared|E3|donante2/coste1/orden0|148|148|válido|
|practice_0153|shared|E3|donante2/coste0/orden1|148|148|válido|
|practice_0153|shared|E3|donante2/coste1/orden1|148|148|válido|
|practice_0153|shared|E3|donante3/coste0/orden0|150|150|válido|
|practice_0153|shared|E3|donante3/coste1/orden0|150|150|válido|
|practice_0153|shared|E3|donante3/coste0/orden1|150|150|válido|
|practice_0153|shared|E3|donante3/coste1/orden1|150|150|válido|
|practice_0153|shared|E3|donante4/coste0/orden0|185|185|válido|
|practice_0153|shared|E3|donante4/coste1/orden0|185|185|válido|
|practice_0153|shared|E3|donante4/coste0/orden1|185|185|válido|
|practice_0153|shared|E3|donante4/coste1/orden1|185|185|válido|
|practice_0153|shared|E3|donante5/coste0/orden0|146|146|válido|
|practice_0153|shared|E3|donante5/coste1/orden0|146|146|válido|
|practice_0153|shared|E3|donante5/coste0/orden1|146|146|válido|
|practice_0153|shared|E3|donante5/coste1/orden1|146|146|válido|
|practice_0153|shared|E3|donante6/coste0/orden0|160|160|válido|
|practice_0153|shared|E3|donante6/coste1/orden0|160|160|válido|
|practice_0153|shared|E3|donante6/coste0/orden1|160|160|válido|
|practice_0153|shared|E3|donante6/coste1/orden1|160|160|válido|
|practice_0153|shared|E3|donante7/coste0/orden0|141|141|válido|
|practice_0153|shared|E3|donante7/coste1/orden0|141|141|válido|
|practice_0153|shared|E3|donante7/coste0/orden1|141|141|válido|
|practice_0153|shared|E3|donante7/coste1/orden1|141|141|válido|
|practice_0153|shared|E3|donante8/coste0/orden0|138|138|válido|
|practice_0153|shared|E3|donante8/coste1/orden0|138|138|válido|
|practice_0153|shared|E3|donante8/coste0/orden1|138|138|válido|
|practice_0153|shared|E3|donante8/coste1/orden1|138|138|válido|
|practice_0154|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0154|independent|Cruce|CP True/coste0/orden0|508|286|válido|
|practice_0154|independent|Cruce|CP True/coste1/orden0|487|286|válido|
|practice_0154|independent|Cruce|CP True/coste0/orden1|372|286|válido|
|practice_0154|independent|Cruce|CP True/coste1/orden1|359|286|válido|
|practice_0154|independent|Cruce|mejor independiente/coste0/orden0|508|286|válido|
|practice_0154|independent|Cruce|mejor independiente/coste1/orden0|487|286|válido|
|practice_0154|independent|Cruce|mejor independiente/coste0/orden1|372|286|válido|
|practice_0154|independent|Cruce|mejor independiente/coste1/orden1|359|286|válido|
|practice_0154|independent|E1|cola10/pol1/inicio0|308|87|válido|
|practice_0154|independent|E1|cola10/pol1/inicio1|308|87|válido|
|practice_0154|independent|E1|cola10/pol2/inicio0|309|88|válido|
|practice_0154|independent|E1|cola10/pol2/inicio1|312|91|válido|
|practice_0154|independent|E1|cola8/pol1/inicio0|314|77|válido|
|practice_0154|independent|E1|cola8/pol1/inicio1|315|78|válido|
|practice_0154|independent|E1|cola8/pol2/inicio0|314|77|válido|
|practice_0154|independent|E1|cola8/pol2/inicio1|314|77|válido|
|practice_0154|independent|E1|cola6/pol1/inicio0|312|59|válido|
|practice_0154|independent|E1|cola6/pol1/inicio1|312|59|válido|
|practice_0154|independent|E1|cola6/pol2/inicio0|310|57|válido|
|practice_0154|independent|E1|cola6/pol2/inicio1|310|57|válido|
|practice_0154|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0154|independent|E2|CP False/coste0/orden0|527|299|válido|
|practice_0154|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0154|independent|E2|CP True/coste0/orden0|508|286|válido|
|practice_0154|independent|E3|donante0/coste0/orden0|298|286|válido|
|practice_0154|independent|E3|donante0/coste1/orden0|303|286|válido|
|practice_0154|independent|E3|donante0/coste0/orden1|372|286|válido|
|practice_0154|independent|E3|donante0/coste1/orden1|359|286|válido|
|practice_0154|independent|E3|donante1/coste0/orden0|336|336|válido|
|practice_0154|independent|E3|donante1/coste1/orden0|336|336|válido|
|practice_0154|independent|E3|donante1/coste0/orden1|336|336|válido|
|practice_0154|independent|E3|donante1/coste1/orden1|336|336|válido|
|practice_0154|independent|E3|donante2/coste0/orden0|429|314|válido|
|practice_0154|independent|E3|donante2/coste1/orden0|430|314|válido|
|practice_0154|independent|E3|donante2/coste0/orden1|320|314|válido|
|practice_0154|independent|E3|donante2/coste1/orden1|320|314|válido|
|practice_0154|independent|E3|donante3/coste0/orden0|598|314|válido|
|practice_0154|independent|E3|donante3/coste1/orden0|430|314|válido|
|practice_0154|independent|E3|donante3/coste0/orden1|451|314|válido|
|practice_0154|independent|E3|donante3/coste1/orden1|317|314|válido|
|practice_0154|independent|E3|donante4/coste0/orden0|351|351|válido|
|practice_0154|independent|E3|donante4/coste1/orden0|351|351|válido|
|practice_0154|independent|E3|donante4/coste0/orden1|351|351|válido|
|practice_0154|independent|E3|donante4/coste1/orden1|351|351|válido|
|practice_0154|independent|E3|donante5/coste0/orden0|336|336|válido|
|practice_0154|independent|E3|donante5/coste1/orden0|336|336|válido|
|practice_0154|independent|E3|donante5/coste0/orden1|336|336|válido|
|practice_0154|independent|E3|donante5/coste1/orden1|336|336|válido|
|practice_0154|independent|E3|donante6/coste0/orden0|343|343|válido|
|practice_0154|independent|E3|donante6/coste1/orden0|343|343|válido|
|practice_0154|independent|E3|donante6/coste0/orden1|343|343|válido|
|practice_0154|independent|E3|donante6/coste1/orden1|343|343|válido|
|practice_0154|independent|E3|donante7/coste0/orden0|341|306|válido|
|practice_0154|independent|E3|donante7/coste1/orden0|329|306|válido|
|practice_0154|independent|E3|donante7/coste0/orden1|332|306|válido|
|practice_0154|independent|E3|donante7/coste1/orden1|326|306|válido|
|practice_0154|independent|E3|donante8/coste0/orden0|306|295|válido|
|practice_0154|independent|E3|donante8/coste1/orden0|298|295|válido|
|practice_0154|independent|E3|donante8/coste0/orden1|317|295|válido|
|practice_0154|independent|E3|donante8/coste1/orden1|316|295|válido|
|practice_0154|independent|E3|donante9/coste0/orden0|299|287|válido|
|practice_0154|independent|E3|donante9/coste1/orden0|304|287|válido|
|practice_0154|independent|E3|donante9/coste0/orden1|383|287|válido|
|practice_0154|independent|E3|donante9/coste1/orden1|360|287|válido|
|practice_0155|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0155|multiplicative|Cruce|CP True/coste0/orden0|150|149|válido|
|practice_0155|multiplicative|Cruce|CP True/coste1/orden0|150|149|válido|
|practice_0155|multiplicative|Cruce|CP True/coste0/orden1|149|149|válido|
|practice_0155|multiplicative|Cruce|CP True/coste1/orden1|149|149|válido|
|practice_0155|multiplicative|Cruce|mejor independiente/coste0/orden0|150|149|válido|
|practice_0155|multiplicative|Cruce|mejor independiente/coste1/orden0|150|149|válido|
|practice_0155|multiplicative|Cruce|mejor independiente/coste0/orden1|149|149|válido|
|practice_0155|multiplicative|Cruce|mejor independiente/coste1/orden1|149|149|válido|
|practice_0155|multiplicative|E1|cola10/pol1/inicio0|159|59|válido|
|practice_0155|multiplicative|E1|cola10/pol1/inicio1|159|59|válido|
|practice_0155|multiplicative|E1|cola10/pol2/inicio0|160|60|válido|
|practice_0155|multiplicative|E1|cola10/pol2/inicio1|156|56|válido|
|practice_0155|multiplicative|E1|cola8/pol1/inicio0|156|45|válido|
|practice_0155|multiplicative|E1|cola8/pol1/inicio1|156|45|válido|
|practice_0155|multiplicative|E1|cola8/pol2/inicio0|158|47|válido|
|practice_0155|multiplicative|E1|cola8/pol2/inicio1|158|47|válido|
|practice_0155|multiplicative|E1|cola6/pol1/inicio0|162|39|válido|
|practice_0155|multiplicative|E1|cola6/pol1/inicio1|162|39|válido|
|practice_0155|multiplicative|E1|cola6/pol2/inicio0|162|39|válido|
|practice_0155|multiplicative|E1|cola6/pol2/inicio1|162|39|válido|
|practice_0155|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0155|multiplicative|E2|CP False/coste0/orden0|159|156|válido|
|practice_0155|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0155|multiplicative|E2|CP True/coste0/orden0|150|149|válido|
|practice_0155|multiplicative|E3|donante0/coste0/orden0|154|152|válido|
|practice_0155|multiplicative|E3|donante0/coste1/orden0|154|152|válido|
|practice_0155|multiplicative|E3|donante0/coste0/orden1|152|152|válido|
|practice_0155|multiplicative|E3|donante0/coste1/orden1|152|152|válido|
|practice_0155|multiplicative|E3|donante1/coste0/orden0|160|160|válido|
|practice_0155|multiplicative|E3|donante1/coste1/orden0|160|160|válido|
|practice_0155|multiplicative|E3|donante1/coste0/orden1|160|160|válido|
|practice_0155|multiplicative|E3|donante1/coste1/orden1|160|160|válido|
|practice_0155|multiplicative|E3|donante2/coste0/orden0|159|159|válido|
|practice_0155|multiplicative|E3|donante2/coste1/orden0|159|159|válido|
|practice_0155|multiplicative|E3|donante2/coste0/orden1|159|159|válido|
|practice_0155|multiplicative|E3|donante2/coste1/orden1|159|159|válido|
|practice_0155|multiplicative|E3|donante3/coste0/orden0|155|155|válido|
|practice_0155|multiplicative|E3|donante3/coste1/orden0|155|155|válido|
|practice_0155|multiplicative|E3|donante3/coste0/orden1|155|155|válido|
|practice_0155|multiplicative|E3|donante3/coste1/orden1|155|155|válido|
|practice_0155|multiplicative|E3|donante4/coste0/orden0|168|168|válido|
|practice_0155|multiplicative|E3|donante4/coste1/orden0|168|168|válido|
|practice_0155|multiplicative|E3|donante4/coste0/orden1|168|168|válido|
|practice_0155|multiplicative|E3|donante4/coste1/orden1|168|168|válido|
|practice_0155|multiplicative|E3|donante5/coste0/orden0|162|162|válido|
|practice_0155|multiplicative|E3|donante5/coste1/orden0|162|162|válido|
|practice_0155|multiplicative|E3|donante5/coste0/orden1|162|162|válido|
|practice_0155|multiplicative|E3|donante5/coste1/orden1|162|162|válido|
|practice_0155|multiplicative|E3|donante6/coste0/orden0|163|163|válido|
|practice_0155|multiplicative|E3|donante6/coste1/orden0|163|163|válido|
|practice_0155|multiplicative|E3|donante6/coste0/orden1|163|163|válido|
|practice_0155|multiplicative|E3|donante6/coste1/orden1|163|163|válido|
|practice_0155|multiplicative|E3|donante7/coste0/orden0|212|155|válido|
|practice_0155|multiplicative|E3|donante7/coste1/orden0|166|155|válido|
|practice_0155|multiplicative|E3|donante7/coste0/orden1|230|155|válido|
|practice_0155|multiplicative|E3|donante7/coste1/orden1|186|155|válido|
|practice_0155|multiplicative|E3|donante8/coste0/orden0|179|151|válido|
|practice_0155|multiplicative|E3|donante8/coste1/orden0|171|151|válido|
|practice_0155|multiplicative|E3|donante8/coste0/orden1|169|151|válido|
|practice_0155|multiplicative|E3|donante8/coste1/orden1|158|151|válido|
|practice_0156|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0156|multiplicative|Cruce|CP True/coste0/orden0|98|98|válido|
|practice_0156|multiplicative|Cruce|CP True/coste1/orden0|98|98|válido|
|practice_0156|multiplicative|Cruce|CP True/coste0/orden1|98|98|válido|
|practice_0156|multiplicative|Cruce|CP True/coste1/orden1|98|98|válido|
|practice_0156|multiplicative|Cruce|mejor independiente/coste0/orden0|98|98|válido|
|practice_0156|multiplicative|Cruce|mejor independiente/coste1/orden0|98|98|válido|
|practice_0156|multiplicative|Cruce|mejor independiente/coste0/orden1|98|98|válido|
|practice_0156|multiplicative|Cruce|mejor independiente/coste1/orden1|98|98|válido|
|practice_0156|multiplicative|E1|cola10/pol1/inicio0|—|—|registros ocupados|
|practice_0156|multiplicative|E1|cola10/pol1/inicio1|—|—|registros ocupados|
|practice_0156|multiplicative|E1|cola10/pol2/inicio0|—|—|registros ocupados|
|practice_0156|multiplicative|E1|cola10/pol2/inicio1|—|—|registros ocupados|
|practice_0156|multiplicative|E1|cola8/pol1/inicio0|—|—|registros ocupados|
|practice_0156|multiplicative|E1|cola8/pol1/inicio1|—|—|registros ocupados|
|practice_0156|multiplicative|E1|cola8/pol2/inicio0|—|—|registros ocupados|
|practice_0156|multiplicative|E1|cola8/pol2/inicio1|—|—|registros ocupados|
|practice_0156|multiplicative|E1|cola6/pol1/inicio0|—|—|registros ocupados|
|practice_0156|multiplicative|E1|cola6/pol1/inicio1|—|—|registros ocupados|
|practice_0156|multiplicative|E1|cola6/pol2/inicio0|—|—|registros ocupados|
|practice_0156|multiplicative|E1|cola6/pol2/inicio1|—|—|registros ocupados|
|practice_0156|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0156|multiplicative|E2|CP False/coste0/orden0|105|105|válido|
|practice_0156|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0156|multiplicative|E2|CP True/coste0/orden0|98|98|válido|
|practice_0156|multiplicative|E3|donante0/coste0/orden0|106|106|válido|
|practice_0156|multiplicative|E3|donante0/coste1/orden0|106|106|válido|
|practice_0156|multiplicative|E3|donante0/coste0/orden1|106|106|válido|
|practice_0156|multiplicative|E3|donante0/coste1/orden1|106|106|válido|
|practice_0156|multiplicative|E3|donante1/coste0/orden0|111|111|válido|
|practice_0156|multiplicative|E3|donante1/coste1/orden0|111|111|válido|
|practice_0156|multiplicative|E3|donante1/coste0/orden1|111|111|válido|
|practice_0156|multiplicative|E3|donante1/coste1/orden1|111|111|válido|
|practice_0156|multiplicative|E3|donante2/coste0/orden0|110|110|válido|
|practice_0156|multiplicative|E3|donante2/coste1/orden0|110|110|válido|
|practice_0156|multiplicative|E3|donante2/coste0/orden1|110|110|válido|
|practice_0156|multiplicative|E3|donante2/coste1/orden1|110|110|válido|
|practice_0156|multiplicative|E3|donante3/coste0/orden0|109|109|válido|
|practice_0156|multiplicative|E3|donante3/coste1/orden0|109|109|válido|
|practice_0156|multiplicative|E3|donante3/coste0/orden1|109|109|válido|
|practice_0156|multiplicative|E3|donante3/coste1/orden1|109|109|válido|
|practice_0156|multiplicative|E3|donante4/coste0/orden0|114|114|válido|
|practice_0156|multiplicative|E3|donante4/coste1/orden0|114|114|válido|
|practice_0156|multiplicative|E3|donante4/coste0/orden1|114|114|válido|
|practice_0156|multiplicative|E3|donante4/coste1/orden1|114|114|válido|
|practice_0156|multiplicative|E3|donante5/coste0/orden0|110|110|válido|
|practice_0156|multiplicative|E3|donante5/coste1/orden0|110|110|válido|
|practice_0156|multiplicative|E3|donante5/coste0/orden1|110|110|válido|
|practice_0156|multiplicative|E3|donante5/coste1/orden1|110|110|válido|
|practice_0156|multiplicative|E3|donante6/coste0/orden0|107|107|válido|
|practice_0156|multiplicative|E3|donante6/coste1/orden0|107|107|válido|
|practice_0156|multiplicative|E3|donante6/coste0/orden1|107|107|válido|
|practice_0156|multiplicative|E3|donante6/coste1/orden1|107|107|válido|
|practice_0156|multiplicative|E3|donante7/coste0/orden0|108|108|válido|
|practice_0156|multiplicative|E3|donante7/coste1/orden0|108|108|válido|
|practice_0156|multiplicative|E3|donante7/coste0/orden1|108|108|válido|
|practice_0156|multiplicative|E3|donante7/coste1/orden1|108|108|válido|
|practice_0156|multiplicative|E3|donante8/coste0/orden0|107|107|válido|
|practice_0156|multiplicative|E3|donante8/coste1/orden0|107|107|válido|
|practice_0156|multiplicative|E3|donante8/coste0/orden1|107|107|válido|
|practice_0156|multiplicative|E3|donante8/coste1/orden1|107|107|válido|
|practice_0156|multiplicative|E3|donante9/coste0/orden0|106|106|válido|
|practice_0156|multiplicative|E3|donante9/coste1/orden0|106|106|válido|
|practice_0156|multiplicative|E3|donante9/coste0/orden1|106|106|válido|
|practice_0156|multiplicative|E3|donante9/coste1/orden1|106|106|válido|
|practice_0157|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0157|shared|Cruce|CP True/coste0/orden0|121|121|válido|
|practice_0157|shared|Cruce|CP True/coste1/orden0|121|121|válido|
|practice_0157|shared|Cruce|CP True/coste0/orden1|121|121|válido|
|practice_0157|shared|Cruce|CP True/coste1/orden1|121|121|válido|
|practice_0157|shared|Cruce|mejor independiente/coste0/orden0|124|124|válido|
|practice_0157|shared|Cruce|mejor independiente/coste1/orden0|124|124|válido|
|practice_0157|shared|Cruce|mejor independiente/coste0/orden1|124|124|válido|
|practice_0157|shared|Cruce|mejor independiente/coste1/orden1|124|124|válido|
|practice_0157|shared|E1|cola10/pol1/inicio0|125|35|válido|
|practice_0157|shared|E1|cola10/pol1/inicio1|125|35|válido|
|practice_0157|shared|E1|cola10/pol2/inicio0|125|35|válido|
|practice_0157|shared|E1|cola10/pol2/inicio1|125|35|válido|
|practice_0157|shared|E1|cola8/pol1/inicio0|127|30|válido|
|practice_0157|shared|E1|cola8/pol1/inicio1|127|30|válido|
|practice_0157|shared|E1|cola8/pol2/inicio0|126|29|válido|
|practice_0157|shared|E1|cola8/pol2/inicio1|126|29|válido|
|practice_0157|shared|E1|cola6/pol1/inicio0|127|22|válido|
|practice_0157|shared|E1|cola6/pol1/inicio1|127|22|válido|
|practice_0157|shared|E1|cola6/pol2/inicio0|127|22|válido|
|practice_0157|shared|E1|cola6/pol2/inicio1|127|22|válido|
|practice_0157|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0157|shared|E2|CP False/coste0/orden0|124|124|válido|
|practice_0157|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0157|shared|E2|CP True/coste0/orden0|125|125|válido|
|practice_0157|shared|E3|donante0/coste0/orden0|127|127|válido|
|practice_0157|shared|E3|donante0/coste1/orden0|127|127|válido|
|practice_0157|shared|E3|donante0/coste0/orden1|127|127|válido|
|practice_0157|shared|E3|donante0/coste1/orden1|127|127|válido|
|practice_0157|shared|E3|donante1/coste0/orden0|146|146|válido|
|practice_0157|shared|E3|donante1/coste1/orden0|146|146|válido|
|practice_0157|shared|E3|donante1/coste0/orden1|146|146|válido|
|practice_0157|shared|E3|donante1/coste1/orden1|146|146|válido|
|practice_0157|shared|E3|donante2/coste0/orden0|130|130|válido|
|practice_0157|shared|E3|donante2/coste1/orden0|130|130|válido|
|practice_0157|shared|E3|donante2/coste0/orden1|130|130|válido|
|practice_0157|shared|E3|donante2/coste1/orden1|130|130|válido|
|practice_0157|shared|E3|donante3/coste0/orden0|128|128|válido|
|practice_0157|shared|E3|donante3/coste1/orden0|128|128|válido|
|practice_0157|shared|E3|donante3/coste0/orden1|128|128|válido|
|practice_0157|shared|E3|donante3/coste1/orden1|128|128|válido|
|practice_0157|shared|E3|donante4/coste0/orden0|154|154|válido|
|practice_0157|shared|E3|donante4/coste1/orden0|154|154|válido|
|practice_0157|shared|E3|donante4/coste0/orden1|154|154|válido|
|practice_0157|shared|E3|donante4/coste1/orden1|154|154|válido|
|practice_0157|shared|E3|donante5/coste0/orden0|144|144|válido|
|practice_0157|shared|E3|donante5/coste1/orden0|144|144|válido|
|practice_0157|shared|E3|donante5/coste0/orden1|144|144|válido|
|practice_0157|shared|E3|donante5/coste1/orden1|144|144|válido|
|practice_0157|shared|E3|donante6/coste0/orden0|137|137|válido|
|practice_0157|shared|E3|donante6/coste1/orden0|137|137|válido|
|practice_0157|shared|E3|donante6/coste0/orden1|137|137|válido|
|practice_0157|shared|E3|donante6/coste1/orden1|137|137|válido|
|practice_0157|shared|E3|donante7/coste0/orden0|132|132|válido|
|practice_0157|shared|E3|donante7/coste1/orden0|132|132|válido|
|practice_0157|shared|E3|donante7/coste0/orden1|132|132|válido|
|practice_0157|shared|E3|donante7/coste1/orden1|132|132|válido|
|practice_0157|shared|E3|donante8/coste0/orden0|130|130|válido|
|practice_0157|shared|E3|donante8/coste1/orden0|130|130|válido|
|practice_0157|shared|E3|donante8/coste0/orden1|130|130|válido|
|practice_0157|shared|E3|donante8/coste1/orden1|130|130|válido|
|practice_0158|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0158|shared|Cruce|CP True/coste0/orden0|216|203|válido|
|practice_0158|shared|Cruce|CP True/coste1/orden0|212|203|válido|
|practice_0158|shared|Cruce|CP True/coste0/orden1|203|203|válido|
|practice_0158|shared|Cruce|CP True/coste1/orden1|203|203|válido|
|practice_0158|shared|Cruce|mejor independiente/coste0/orden0|217|203|válido|
|practice_0158|shared|Cruce|mejor independiente/coste1/orden0|212|203|válido|
|practice_0158|shared|Cruce|mejor independiente/coste0/orden1|203|203|válido|
|practice_0158|shared|Cruce|mejor independiente/coste1/orden1|203|203|válido|
|practice_0158|shared|E1|cola10/pol1/inicio0|211|69|válido|
|practice_0158|shared|E1|cola10/pol1/inicio1|211|69|válido|
|practice_0158|shared|E1|cola10/pol2/inicio0|211|69|válido|
|practice_0158|shared|E1|cola10/pol2/inicio1|211|69|válido|
|practice_0158|shared|E1|cola8/pol1/inicio0|208|56|válido|
|practice_0158|shared|E1|cola8/pol1/inicio1|208|56|válido|
|practice_0158|shared|E1|cola8/pol2/inicio0|209|57|válido|
|practice_0158|shared|E1|cola8/pol2/inicio1|209|57|válido|
|practice_0158|shared|E1|cola6/pol1/inicio0|208|44|válido|
|practice_0158|shared|E1|cola6/pol1/inicio1|208|44|válido|
|practice_0158|shared|E1|cola6/pol2/inicio0|208|44|válido|
|practice_0158|shared|E1|cola6/pol2/inicio1|208|44|válido|
|practice_0158|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0158|shared|E2|CP False/coste0/orden0|229|213|válido|
|practice_0158|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0158|shared|E2|CP True/coste0/orden0|216|203|válido|
|practice_0158|shared|E3|donante0/coste0/orden0|207|203|válido|
|practice_0158|shared|E3|donante0/coste1/orden0|209|203|válido|
|practice_0158|shared|E3|donante0/coste0/orden1|203|203|válido|
|practice_0158|shared|E3|donante0/coste1/orden1|203|203|válido|
|practice_0158|shared|E3|donante1/coste0/orden0|239|239|válido|
|practice_0158|shared|E3|donante1/coste1/orden0|239|239|válido|
|practice_0158|shared|E3|donante1/coste0/orden1|239|239|válido|
|practice_0158|shared|E3|donante1/coste1/orden1|239|239|válido|
|practice_0158|shared|E3|donante2/coste0/orden0|226|226|válido|
|practice_0158|shared|E3|donante2/coste1/orden0|226|226|válido|
|practice_0158|shared|E3|donante2/coste0/orden1|226|226|válido|
|practice_0158|shared|E3|donante2/coste1/orden1|226|226|válido|
|practice_0158|shared|E3|donante3/coste0/orden0|240|218|válido|
|practice_0158|shared|E3|donante3/coste1/orden0|231|218|válido|
|practice_0158|shared|E3|donante3/coste0/orden1|218|218|válido|
|practice_0158|shared|E3|donante3/coste1/orden1|218|218|válido|
|practice_0158|shared|E3|donante4/coste0/orden0|251|251|válido|
|practice_0158|shared|E3|donante4/coste1/orden0|251|251|válido|
|practice_0158|shared|E3|donante4/coste0/orden1|251|251|válido|
|practice_0158|shared|E3|donante4/coste1/orden1|251|251|válido|
|practice_0158|shared|E3|donante5/coste0/orden0|251|251|válido|
|practice_0158|shared|E3|donante5/coste1/orden0|251|251|válido|
|practice_0158|shared|E3|donante5/coste0/orden1|251|251|válido|
|practice_0158|shared|E3|donante5/coste1/orden1|251|251|válido|
|practice_0158|shared|E3|donante6/coste0/orden0|244|244|válido|
|practice_0158|shared|E3|donante6/coste1/orden0|244|244|válido|
|practice_0158|shared|E3|donante6/coste0/orden1|244|244|válido|
|practice_0158|shared|E3|donante6/coste1/orden1|244|244|válido|
|practice_0158|shared|E3|donante7/coste0/orden0|283|215|válido|
|practice_0158|shared|E3|donante7/coste1/orden0|251|215|válido|
|practice_0158|shared|E3|donante7/coste0/orden1|247|215|válido|
|practice_0158|shared|E3|donante7/coste1/orden1|234|215|válido|
|practice_0158|shared|E3|donante8/coste0/orden0|210|210|válido|
|practice_0158|shared|E3|donante8/coste1/orden0|210|210|válido|
|practice_0158|shared|E3|donante8/coste0/orden1|210|210|válido|
|practice_0158|shared|E3|donante8/coste1/orden1|210|210|válido|
|practice_0158|shared|E3|donante9/coste0/orden0|207|203|válido|
|practice_0158|shared|E3|donante9/coste1/orden0|209|203|válido|
|practice_0158|shared|E3|donante9/coste0/orden1|203|203|válido|
|practice_0158|shared|E3|donante9/coste1/orden1|203|203|válido|
|practice_0159|shared|Cruce|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0159|shared|Cruce|CP True/coste0/orden0|29|29|válido|
|practice_0159|shared|Cruce|CP True/coste1/orden0|29|29|válido|
|practice_0159|shared|Cruce|CP True/coste0/orden1|29|29|válido|
|practice_0159|shared|Cruce|CP True/coste1/orden1|29|29|válido|
|practice_0159|shared|Cruce|mejor independiente/coste0/orden0|29|29|válido|
|practice_0159|shared|Cruce|mejor independiente/coste1/orden0|29|29|válido|
|practice_0159|shared|Cruce|mejor independiente/coste0/orden1|29|29|válido|
|practice_0159|shared|Cruce|mejor independiente/coste1/orden1|29|29|válido|
|practice_0159|shared|E1|cola7/pol1/inicio0|32|18|válido|
|practice_0159|shared|E1|cola7/pol1/inicio1|32|18|válido|
|practice_0159|shared|E1|cola7/pol2/inicio0|32|18|válido|
|practice_0159|shared|E1|cola7/pol2/inicio1|33|19|válido|
|practice_0159|shared|E1|cola6/pol1/inicio0|32|16|válido|
|practice_0159|shared|E1|cola6/pol1/inicio1|33|17|válido|
|practice_0159|shared|E1|cola6/pol2/inicio0|32|16|válido|
|practice_0159|shared|E1|cola6/pol2/inicio1|33|17|válido|
|practice_0159|shared|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0159|shared|E2|CP False/coste0/orden0|32|32|válido|
|practice_0159|shared|E2|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0159|shared|E2|CP True/coste0/orden0|29|29|válido|
|practice_0159|shared|E3|donante0/coste0/orden0|32|32|válido|
|practice_0159|shared|E3|donante0/coste1/orden0|32|32|válido|
|practice_0159|shared|E3|donante0/coste0/orden1|32|32|válido|
|practice_0159|shared|E3|donante0/coste1/orden1|32|32|válido|
|practice_0159|shared|E3|donante1/coste0/orden0|39|39|válido|
|practice_0159|shared|E3|donante1/coste1/orden0|39|39|válido|
|practice_0159|shared|E3|donante1/coste0/orden1|39|39|válido|
|practice_0159|shared|E3|donante1/coste1/orden1|39|39|válido|
|practice_0159|shared|E3|donante2/coste0/orden0|34|34|válido|
|practice_0159|shared|E3|donante2/coste1/orden0|34|34|válido|
|practice_0159|shared|E3|donante2/coste0/orden1|34|34|válido|
|practice_0159|shared|E3|donante2/coste1/orden1|34|34|válido|
|practice_0159|shared|E3|donante3/coste0/orden0|38|38|válido|
|practice_0159|shared|E3|donante3/coste1/orden0|38|38|válido|
|practice_0159|shared|E3|donante3/coste0/orden1|38|38|válido|
|practice_0159|shared|E3|donante3/coste1/orden1|38|38|válido|
|practice_0159|shared|E3|donante4/coste0/orden0|39|39|válido|
|practice_0159|shared|E3|donante4/coste1/orden0|39|39|válido|
|practice_0159|shared|E3|donante4/coste0/orden1|39|39|válido|
|practice_0159|shared|E3|donante4/coste1/orden1|39|39|válido|
|practice_0159|shared|E3|donante5/coste0/orden0|33|33|válido|
|practice_0159|shared|E3|donante5/coste1/orden0|33|33|válido|
|practice_0159|shared|E3|donante5/coste0/orden1|33|33|válido|
|practice_0159|shared|E3|donante5/coste1/orden1|33|33|válido|
|practice_0159|shared|E3|donante6/coste0/orden0|39|39|válido|
|practice_0159|shared|E3|donante6/coste1/orden0|39|39|válido|
|practice_0159|shared|E3|donante6/coste0/orden1|39|39|válido|
|practice_0159|shared|E3|donante6/coste1/orden1|39|39|válido|
|practice_0160|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0160|mixed|Cruce|CP True/coste0/orden0|162|161|válido|
|practice_0160|mixed|Cruce|CP True/coste1/orden0|162|161|válido|
|practice_0160|mixed|Cruce|CP True/coste0/orden1|163|161|válido|
|practice_0160|mixed|Cruce|CP True/coste1/orden1|163|161|válido|
|practice_0160|mixed|Cruce|mejor independiente/coste0/orden0|162|161|válido|
|practice_0160|mixed|Cruce|mejor independiente/coste1/orden0|162|161|válido|
|practice_0160|mixed|Cruce|mejor independiente/coste0/orden1|163|161|válido|
|practice_0160|mixed|Cruce|mejor independiente/coste1/orden1|163|161|válido|
|practice_0160|mixed|E1|cola10/pol1/inicio0|170|61|válido|
|practice_0160|mixed|E1|cola10/pol1/inicio1|170|61|válido|
|practice_0160|mixed|E1|cola10/pol2/inicio0|170|61|válido|
|practice_0160|mixed|E1|cola10/pol2/inicio1|170|61|válido|
|practice_0160|mixed|E1|cola8/pol1/inicio0|165|44|válido|
|practice_0160|mixed|E1|cola8/pol1/inicio1|165|44|válido|
|practice_0160|mixed|E1|cola8/pol2/inicio0|169|48|válido|
|practice_0160|mixed|E1|cola8/pol2/inicio1|169|48|válido|
|practice_0160|mixed|E1|cola6/pol1/inicio0|168|36|válido|
|practice_0160|mixed|E1|cola6/pol1/inicio1|168|36|válido|
|practice_0160|mixed|E1|cola6/pol2/inicio0|168|36|válido|
|practice_0160|mixed|E1|cola6/pol2/inicio1|168|36|válido|
|practice_0160|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0160|mixed|E2|CP False/coste0/orden0|182|182|válido|
|practice_0160|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0160|mixed|E2|CP True/coste0/orden0|162|161|válido|
|practice_0160|mixed|E3|donante0/coste0/orden0|163|163|válido|
|practice_0160|mixed|E3|donante0/coste1/orden0|163|163|válido|
|practice_0160|mixed|E3|donante0/coste0/orden1|163|163|válido|
|practice_0160|mixed|E3|donante0/coste1/orden1|163|163|válido|
|practice_0160|mixed|E3|donante1/coste0/orden0|187|187|válido|
|practice_0160|mixed|E3|donante1/coste1/orden0|187|187|válido|
|practice_0160|mixed|E3|donante1/coste0/orden1|187|187|válido|
|practice_0160|mixed|E3|donante1/coste1/orden1|187|187|válido|
|practice_0160|mixed|E3|donante2/coste0/orden0|180|180|válido|
|practice_0160|mixed|E3|donante2/coste1/orden0|180|180|válido|
|practice_0160|mixed|E3|donante2/coste0/orden1|180|180|válido|
|practice_0160|mixed|E3|donante2/coste1/orden1|180|180|válido|
|practice_0160|mixed|E3|donante3/coste0/orden0|178|178|válido|
|practice_0160|mixed|E3|donante3/coste1/orden0|178|178|válido|
|practice_0160|mixed|E3|donante3/coste0/orden1|178|178|válido|
|practice_0160|mixed|E3|donante3/coste1/orden1|178|178|válido|
|practice_0160|mixed|E3|donante4/coste0/orden0|184|184|válido|
|practice_0160|mixed|E3|donante4/coste1/orden0|184|184|válido|
|practice_0160|mixed|E3|donante4/coste0/orden1|184|184|válido|
|practice_0160|mixed|E3|donante4/coste1/orden1|184|184|válido|
|practice_0160|mixed|E3|donante5/coste0/orden0|182|182|válido|
|practice_0160|mixed|E3|donante5/coste1/orden0|182|182|válido|
|practice_0160|mixed|E3|donante5/coste0/orden1|182|182|válido|
|practice_0160|mixed|E3|donante5/coste1/orden1|182|182|válido|
|practice_0160|mixed|E3|donante6/coste0/orden0|183|183|válido|
|practice_0160|mixed|E3|donante6/coste1/orden0|183|183|válido|
|practice_0160|mixed|E3|donante6/coste0/orden1|183|183|válido|
|practice_0160|mixed|E3|donante6/coste1/orden1|183|183|válido|
|practice_0160|mixed|E3|donante7/coste0/orden0|182|171|válido|
|practice_0160|mixed|E3|donante7/coste1/orden0|184|171|válido|
|practice_0160|mixed|E3|donante7/coste0/orden1|190|171|válido|
|practice_0160|mixed|E3|donante7/coste1/orden1|176|171|válido|
|practice_0160|mixed|E3|donante8/coste0/orden0|174|167|válido|
|practice_0160|mixed|E3|donante8/coste1/orden0|173|167|válido|
|practice_0160|mixed|E3|donante8/coste0/orden1|173|167|válido|
|practice_0160|mixed|E3|donante8/coste1/orden1|168|167|válido|
|practice_0161|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0161|mixed|Cruce|CP True/coste0/orden0|357|235|válido|
|practice_0161|mixed|Cruce|CP True/coste1/orden0|311|235|válido|
|practice_0161|mixed|Cruce|CP True/coste0/orden1|235|235|válido|
|practice_0161|mixed|Cruce|CP True/coste1/orden1|235|235|válido|
|practice_0161|mixed|Cruce|mejor independiente/coste0/orden0|357|235|válido|
|practice_0161|mixed|Cruce|mejor independiente/coste1/orden0|311|235|válido|
|practice_0161|mixed|Cruce|mejor independiente/coste0/orden1|235|235|válido|
|practice_0161|mixed|Cruce|mejor independiente/coste1/orden1|235|235|válido|
|practice_0161|mixed|E1|cola10/pol1/inicio0|248|91|válido|
|practice_0161|mixed|E1|cola10/pol1/inicio1|248|91|válido|
|practice_0161|mixed|E1|cola10/pol2/inicio0|248|91|válido|
|practice_0161|mixed|E1|cola10/pol2/inicio1|248|91|válido|
|practice_0161|mixed|E1|cola8/pol1/inicio0|245|72|válido|
|practice_0161|mixed|E1|cola8/pol1/inicio1|245|72|válido|
|practice_0161|mixed|E1|cola8/pol2/inicio0|243|70|válido|
|practice_0161|mixed|E1|cola8/pol2/inicio1|243|70|válido|
|practice_0161|mixed|E1|cola6/pol1/inicio0|244|54|válido|
|practice_0161|mixed|E1|cola6/pol1/inicio1|244|54|válido|
|practice_0161|mixed|E1|cola6/pol2/inicio0|244|54|válido|
|practice_0161|mixed|E1|cola6/pol2/inicio1|244|54|válido|
|practice_0161|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0161|mixed|E2|CP False/coste0/orden0|358|236|válido|
|practice_0161|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0161|mixed|E2|CP True/coste0/orden0|357|235|válido|
|practice_0161|mixed|E3|donante0/coste0/orden0|235|235|válido|
|practice_0161|mixed|E3|donante0/coste1/orden0|235|235|válido|
|practice_0161|mixed|E3|donante0/coste0/orden1|235|235|válido|
|practice_0161|mixed|E3|donante0/coste1/orden1|235|235|válido|
|practice_0161|mixed|E3|donante1/coste0/orden0|274|274|válido|
|practice_0161|mixed|E3|donante1/coste1/orden0|274|274|válido|
|practice_0161|mixed|E3|donante1/coste0/orden1|274|274|válido|
|practice_0161|mixed|E3|donante1/coste1/orden1|274|274|válido|
|practice_0161|mixed|E3|donante2/coste0/orden0|325|259|válido|
|practice_0161|mixed|E3|donante2/coste1/orden0|261|259|válido|
|practice_0161|mixed|E3|donante2/coste0/orden1|259|259|válido|
|practice_0161|mixed|E3|donante2/coste1/orden1|259|259|válido|
|practice_0161|mixed|E3|donante3/coste0/orden0|493|261|válido|
|practice_0161|mixed|E3|donante3/coste1/orden0|299|261|válido|
|practice_0161|mixed|E3|donante3/coste0/orden1|261|261|válido|
|practice_0161|mixed|E3|donante3/coste1/orden1|261|261|válido|
|practice_0161|mixed|E3|donante4/coste0/orden0|274|274|válido|
|practice_0161|mixed|E3|donante4/coste1/orden0|274|274|válido|
|practice_0161|mixed|E3|donante4/coste0/orden1|274|274|válido|
|practice_0161|mixed|E3|donante4/coste1/orden1|274|274|válido|
|practice_0161|mixed|E3|donante5/coste0/orden0|265|265|válido|
|practice_0161|mixed|E3|donante5/coste1/orden0|265|265|válido|
|practice_0161|mixed|E3|donante5/coste0/orden1|265|265|válido|
|practice_0161|mixed|E3|donante5/coste1/orden1|265|265|válido|
|practice_0161|mixed|E3|donante6/coste0/orden0|267|267|válido|
|practice_0161|mixed|E3|donante6/coste1/orden0|267|267|válido|
|practice_0161|mixed|E3|donante6/coste0/orden1|267|267|válido|
|practice_0161|mixed|E3|donante6/coste1/orden1|267|267|válido|
|practice_0161|mixed|E3|donante7/coste0/orden0|293|243|válido|
|practice_0161|mixed|E3|donante7/coste1/orden0|265|243|válido|
|practice_0161|mixed|E3|donante7/coste0/orden1|280|243|válido|
|practice_0161|mixed|E3|donante7/coste1/orden1|259|243|válido|
|practice_0161|mixed|E3|donante8/coste0/orden0|237|237|válido|
|practice_0161|mixed|E3|donante8/coste1/orden0|237|237|válido|
|practice_0161|mixed|E3|donante8/coste0/orden1|237|237|válido|
|practice_0161|mixed|E3|donante8/coste1/orden1|237|237|válido|
|practice_0161|mixed|E3|donante9/coste0/orden0|272|235|válido|
|practice_0161|mixed|E3|donante9/coste1/orden0|260|235|válido|
|practice_0161|mixed|E3|donante9/coste0/orden1|262|235|válido|
|practice_0161|mixed|E3|donante9/coste1/orden1|250|235|válido|
|practice_0162|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0162|independent|Cruce|CP True/coste0/orden0|383|239|válido|
|practice_0162|independent|Cruce|CP True/coste1/orden0|306|239|válido|
|practice_0162|independent|Cruce|CP True/coste0/orden1|273|239|válido|
|practice_0162|independent|Cruce|CP True/coste1/orden1|269|239|válido|
|practice_0162|independent|Cruce|mejor independiente/coste0/orden0|407|240|válido|
|practice_0162|independent|Cruce|mejor independiente/coste1/orden0|298|240|válido|
|practice_0162|independent|Cruce|mejor independiente/coste0/orden1|261|240|válido|
|practice_0162|independent|Cruce|mejor independiente/coste1/orden1|257|240|válido|
|practice_0162|independent|E1|cola10/pol1/inicio0|260|71|válido|
|practice_0162|independent|E1|cola10/pol1/inicio1|260|71|válido|
|practice_0162|independent|E1|cola10/pol2/inicio0|267|78|válido|
|practice_0162|independent|E1|cola10/pol2/inicio1|260|71|válido|
|practice_0162|independent|E1|cola8/pol1/inicio0|266|63|válido|
|practice_0162|independent|E1|cola8/pol1/inicio1|265|62|válido|
|practice_0162|independent|E1|cola8/pol2/inicio0|268|65|válido|
|practice_0162|independent|E1|cola8/pol2/inicio1|268|65|válido|
|practice_0162|independent|E1|cola6/pol1/inicio0|268|51|válido|
|practice_0162|independent|E1|cola6/pol1/inicio1|266|49|válido|
|practice_0162|independent|E1|cola6/pol2/inicio0|268|51|válido|
|practice_0162|independent|E1|cola6/pol2/inicio1|261|44|válido|
|practice_0162|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0162|independent|E2|CP False/coste0/orden0|365|304|válido|
|practice_0162|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0162|independent|E2|CP True/coste0/orden0|288|253|válido|
|practice_0162|independent|E3|donante0/coste0/orden0|255|254|válido|
|practice_0162|independent|E3|donante0/coste1/orden0|255|254|válido|
|practice_0162|independent|E3|donante0/coste0/orden1|262|254|válido|
|practice_0162|independent|E3|donante0/coste1/orden1|268|254|válido|
|practice_0162|independent|E3|donante1/coste0/orden0|285|285|válido|
|practice_0162|independent|E3|donante1/coste1/orden0|285|285|válido|
|practice_0162|independent|E3|donante1/coste0/orden1|285|285|válido|
|practice_0162|independent|E3|donante1/coste1/orden1|285|285|válido|
|practice_0162|independent|E3|donante2/coste0/orden0|290|268|válido|
|practice_0162|independent|E3|donante2/coste1/orden0|300|268|válido|
|practice_0162|independent|E3|donante2/coste0/orden1|268|268|válido|
|practice_0162|independent|E3|donante2/coste1/orden1|268|268|válido|
|practice_0162|independent|E3|donante3/coste0/orden0|406|258|válido|
|practice_0162|independent|E3|donante3/coste1/orden0|333|258|válido|
|practice_0162|independent|E3|donante3/coste0/orden1|259|258|válido|
|practice_0162|independent|E3|donante3/coste1/orden1|259|258|válido|
|practice_0162|independent|E3|donante4/coste0/orden0|289|289|válido|
|practice_0162|independent|E3|donante4/coste1/orden0|289|289|válido|
|practice_0162|independent|E3|donante4/coste0/orden1|289|289|válido|
|practice_0162|independent|E3|donante4/coste1/orden1|289|289|válido|
|practice_0162|independent|E3|donante5/coste0/orden0|283|283|válido|
|practice_0162|independent|E3|donante5/coste1/orden0|283|283|válido|
|practice_0162|independent|E3|donante5/coste0/orden1|283|283|válido|
|practice_0162|independent|E3|donante5/coste1/orden1|283|283|válido|
|practice_0162|independent|E3|donante6/coste0/orden0|285|285|válido|
|practice_0162|independent|E3|donante6/coste1/orden0|285|285|válido|
|practice_0162|independent|E3|donante6/coste0/orden1|285|285|válido|
|practice_0162|independent|E3|donante6/coste1/orden1|285|285|válido|
|practice_0162|independent|E3|donante7/coste0/orden0|296|261|válido|
|practice_0162|independent|E3|donante7/coste1/orden0|281|261|válido|
|practice_0162|independent|E3|donante7/coste0/orden1|278|261|válido|
|practice_0162|independent|E3|donante7/coste1/orden1|269|261|válido|
|practice_0162|independent|E3|donante8/coste0/orden0|327|241|válido|
|practice_0162|independent|E3|donante8/coste1/orden0|260|241|válido|
|practice_0162|independent|E3|donante8/coste0/orden1|262|241|válido|
|practice_0162|independent|E3|donante8/coste1/orden1|258|241|válido|
|practice_0163|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0163|independent|Cruce|CP True/coste0/orden0|308|229|válido|
|practice_0163|independent|Cruce|CP True/coste1/orden0|280|229|válido|
|practice_0163|independent|Cruce|CP True/coste0/orden1|237|229|válido|
|practice_0163|independent|Cruce|CP True/coste1/orden1|237|229|válido|
|practice_0163|independent|Cruce|mejor independiente/coste0/orden0|292|231|válido|
|practice_0163|independent|Cruce|mejor independiente/coste1/orden0|270|231|válido|
|practice_0163|independent|Cruce|mejor independiente/coste0/orden1|253|231|válido|
|practice_0163|independent|Cruce|mejor independiente/coste1/orden1|244|231|válido|
|practice_0163|independent|E1|cola10/pol1/inicio0|246|79|válido|
|practice_0163|independent|E1|cola10/pol1/inicio1|246|79|válido|
|practice_0163|independent|E1|cola10/pol2/inicio0|243|76|válido|
|practice_0163|independent|E1|cola10/pol2/inicio1|243|76|válido|
|practice_0163|independent|E1|cola8/pol1/inicio0|237|56|válido|
|practice_0163|independent|E1|cola8/pol1/inicio1|241|60|válido|
|practice_0163|independent|E1|cola8/pol2/inicio0|241|60|válido|
|practice_0163|independent|E1|cola8/pol2/inicio1|249|68|válido|
|practice_0163|independent|E1|cola6/pol1/inicio0|242|47|válido|
|practice_0163|independent|E1|cola6/pol1/inicio1|242|47|válido|
|practice_0163|independent|E1|cola6/pol2/inicio0|242|47|válido|
|practice_0163|independent|E1|cola6/pol2/inicio1|242|47|válido|
|practice_0163|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0163|independent|E2|CP False/coste0/orden0|396|290|válido|
|practice_0163|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0163|independent|E2|CP True/coste0/orden0|308|229|válido|
|practice_0163|independent|E3|donante0/coste0/orden0|231|231|válido|
|practice_0163|independent|E3|donante0/coste1/orden0|231|231|válido|
|practice_0163|independent|E3|donante0/coste0/orden1|253|231|válido|
|practice_0163|independent|E3|donante0/coste1/orden1|244|231|válido|
|practice_0163|independent|E3|donante1/coste0/orden0|263|263|válido|
|practice_0163|independent|E3|donante1/coste1/orden0|263|263|válido|
|practice_0163|independent|E3|donante1/coste0/orden1|279|263|válido|
|practice_0163|independent|E3|donante1/coste1/orden1|269|263|válido|
|practice_0163|independent|E3|donante2/coste0/orden0|391|247|válido|
|practice_0163|independent|E3|donante2/coste1/orden0|296|247|válido|
|practice_0163|independent|E3|donante2/coste0/orden1|247|247|válido|
|practice_0163|independent|E3|donante2/coste1/orden1|247|247|válido|
|practice_0163|independent|E3|donante3/coste0/orden0|377|244|válido|
|practice_0163|independent|E3|donante3/coste1/orden0|344|244|válido|
|practice_0163|independent|E3|donante3/coste0/orden1|244|244|válido|
|practice_0163|independent|E3|donante3/coste1/orden1|244|244|válido|
|practice_0163|independent|E3|donante4/coste0/orden0|267|267|válido|
|practice_0163|independent|E3|donante4/coste1/orden0|267|267|válido|
|practice_0163|independent|E3|donante4/coste0/orden1|267|267|válido|
|practice_0163|independent|E3|donante4/coste1/orden1|267|267|válido|
|practice_0163|independent|E3|donante5/coste0/orden0|257|257|válido|
|practice_0163|independent|E3|donante5/coste1/orden0|257|257|válido|
|practice_0163|independent|E3|donante5/coste0/orden1|257|257|válido|
|practice_0163|independent|E3|donante5/coste1/orden1|257|257|válido|
|practice_0163|independent|E3|donante6/coste0/orden0|261|261|válido|
|practice_0163|independent|E3|donante6/coste1/orden0|261|261|válido|
|practice_0163|independent|E3|donante6/coste0/orden1|261|261|válido|
|practice_0163|independent|E3|donante6/coste1/orden1|261|261|válido|
|practice_0163|independent|E3|donante7/coste0/orden0|241|239|válido|
|practice_0163|independent|E3|donante7/coste1/orden0|241|239|válido|
|practice_0163|independent|E3|donante7/coste0/orden1|261|239|válido|
|practice_0163|independent|E3|donante7/coste1/orden1|253|239|válido|
|practice_0163|independent|E3|donante8/coste0/orden0|258|228|válido|
|practice_0163|independent|E3|donante8/coste1/orden0|243|228|válido|
|practice_0163|independent|E3|donante8/coste0/orden1|282|228|válido|
|practice_0163|independent|E3|donante8/coste1/orden1|242|228|válido|
|practice_0164|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0164|mixed|Cruce|CP True/coste0/orden0|299|219|válido|
|practice_0164|mixed|Cruce|CP True/coste1/orden0|257|219|válido|
|practice_0164|mixed|Cruce|CP True/coste0/orden1|219|219|válido|
|practice_0164|mixed|Cruce|CP True/coste1/orden1|219|219|válido|
|practice_0164|mixed|Cruce|mejor independiente/coste0/orden0|297|220|válido|
|practice_0164|mixed|Cruce|mejor independiente/coste1/orden0|259|220|válido|
|practice_0164|mixed|Cruce|mejor independiente/coste0/orden1|220|220|válido|
|practice_0164|mixed|Cruce|mejor independiente/coste1/orden1|220|220|válido|
|practice_0164|mixed|E1|cola10/pol1/inicio0|236|92|válido|
|practice_0164|mixed|E1|cola10/pol1/inicio1|236|92|válido|
|practice_0164|mixed|E1|cola10/pol2/inicio0|235|91|válido|
|practice_0164|mixed|E1|cola10/pol2/inicio1|235|91|válido|
|practice_0164|mixed|E1|cola8/pol1/inicio0|231|71|válido|
|practice_0164|mixed|E1|cola8/pol1/inicio1|231|71|válido|
|practice_0164|mixed|E1|cola8/pol2/inicio0|231|71|válido|
|practice_0164|mixed|E1|cola8/pol2/inicio1|232|72|válido|
|practice_0164|mixed|E1|cola6/pol1/inicio0|229|53|válido|
|practice_0164|mixed|E1|cola6/pol1/inicio1|229|53|válido|
|practice_0164|mixed|E1|cola6/pol2/inicio0|234|58|válido|
|practice_0164|mixed|E1|cola6/pol2/inicio1|234|58|válido|
|practice_0164|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0164|mixed|E2|CP False/coste0/orden0|314|219|válido|
|practice_0164|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0164|mixed|E2|CP True/coste0/orden0|297|220|válido|
|practice_0164|mixed|E3|donante0/coste0/orden0|220|220|válido|
|practice_0164|mixed|E3|donante0/coste1/orden0|220|220|válido|
|practice_0164|mixed|E3|donante0/coste0/orden1|220|220|válido|
|practice_0164|mixed|E3|donante0/coste1/orden1|220|220|válido|
|practice_0164|mixed|E3|donante1/coste0/orden0|256|256|válido|
|practice_0164|mixed|E3|donante1/coste1/orden0|256|256|válido|
|practice_0164|mixed|E3|donante1/coste0/orden1|256|256|válido|
|practice_0164|mixed|E3|donante1/coste1/orden1|256|256|válido|
|practice_0164|mixed|E3|donante2/coste0/orden0|243|243|válido|
|practice_0164|mixed|E3|donante2/coste1/orden0|243|243|válido|
|practice_0164|mixed|E3|donante2/coste0/orden1|243|243|válido|
|practice_0164|mixed|E3|donante2/coste1/orden1|243|243|válido|
|practice_0164|mixed|E3|donante3/coste0/orden0|243|243|válido|
|practice_0164|mixed|E3|donante3/coste1/orden0|243|243|válido|
|practice_0164|mixed|E3|donante3/coste0/orden1|243|243|válido|
|practice_0164|mixed|E3|donante3/coste1/orden1|243|243|válido|
|practice_0164|mixed|E3|donante4/coste0/orden0|281|281|válido|
|practice_0164|mixed|E3|donante4/coste1/orden0|281|281|válido|
|practice_0164|mixed|E3|donante4/coste0/orden1|281|281|válido|
|practice_0164|mixed|E3|donante4/coste1/orden1|281|281|válido|
|practice_0164|mixed|E3|donante5/coste0/orden0|259|259|válido|
|practice_0164|mixed|E3|donante5/coste1/orden0|259|259|válido|
|practice_0164|mixed|E3|donante5/coste0/orden1|259|259|válido|
|practice_0164|mixed|E3|donante5/coste1/orden1|259|259|válido|
|practice_0164|mixed|E3|donante6/coste0/orden0|274|274|válido|
|practice_0164|mixed|E3|donante6/coste1/orden0|274|274|válido|
|practice_0164|mixed|E3|donante6/coste0/orden1|274|274|válido|
|practice_0164|mixed|E3|donante6/coste1/orden1|274|274|válido|
|practice_0164|mixed|E3|donante7/coste0/orden0|247|236|válido|
|practice_0164|mixed|E3|donante7/coste1/orden0|253|236|válido|
|practice_0164|mixed|E3|donante7/coste0/orden1|240|236|válido|
|practice_0164|mixed|E3|donante7/coste1/orden1|240|236|válido|
|practice_0164|mixed|E3|donante8/coste0/orden0|267|230|válido|
|practice_0164|mixed|E3|donante8/coste1/orden0|254|230|válido|
|practice_0164|mixed|E3|donante8/coste0/orden1|252|230|válido|
|practice_0164|mixed|E3|donante8/coste1/orden1|242|230|válido|
|practice_0165|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0165|mixed|Cruce|CP True/coste0/orden0|259|202|válido|
|practice_0165|mixed|Cruce|CP True/coste1/orden0|228|202|válido|
|practice_0165|mixed|Cruce|CP True/coste0/orden1|220|202|válido|
|practice_0165|mixed|Cruce|CP True/coste1/orden1|221|202|válido|
|practice_0165|mixed|Cruce|mejor independiente/coste0/orden0|261|204|válido|
|practice_0165|mixed|Cruce|mejor independiente/coste1/orden0|222|204|válido|
|practice_0165|mixed|Cruce|mejor independiente/coste0/orden1|205|204|válido|
|practice_0165|mixed|Cruce|mejor independiente/coste1/orden1|205|204|válido|
|practice_0165|mixed|E1|cola10/pol1/inicio0|219|76|válido|
|practice_0165|mixed|E1|cola10/pol1/inicio1|219|76|válido|
|practice_0165|mixed|E1|cola10/pol2/inicio0|214|71|válido|
|practice_0165|mixed|E1|cola10/pol2/inicio1|214|71|válido|
|practice_0165|mixed|E1|cola8/pol1/inicio0|214|58|válido|
|practice_0165|mixed|E1|cola8/pol1/inicio1|214|58|válido|
|practice_0165|mixed|E1|cola8/pol2/inicio0|213|57|válido|
|practice_0165|mixed|E1|cola8/pol2/inicio1|213|57|válido|
|practice_0165|mixed|E1|cola6/pol1/inicio0|211|41|válido|
|practice_0165|mixed|E1|cola6/pol1/inicio1|211|41|válido|
|practice_0165|mixed|E1|cola6/pol2/inicio0|211|41|válido|
|practice_0165|mixed|E1|cola6/pol2/inicio1|211|41|válido|
|practice_0165|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0165|mixed|E2|CP False/coste0/orden0|276|217|válido|
|practice_0165|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0165|mixed|E2|CP True/coste0/orden0|259|202|válido|
|practice_0165|mixed|E3|donante0/coste0/orden0|205|204|válido|
|practice_0165|mixed|E3|donante0/coste1/orden0|205|204|válido|
|practice_0165|mixed|E3|donante0/coste0/orden1|205|204|válido|
|practice_0165|mixed|E3|donante0/coste1/orden1|205|204|válido|
|practice_0165|mixed|E3|donante1/coste0/orden0|244|244|válido|
|practice_0165|mixed|E3|donante1/coste1/orden0|244|244|válido|
|practice_0165|mixed|E3|donante1/coste0/orden1|244|244|válido|
|practice_0165|mixed|E3|donante1/coste1/orden1|244|244|válido|
|practice_0165|mixed|E3|donante2/coste0/orden0|224|224|válido|
|practice_0165|mixed|E3|donante2/coste1/orden0|224|224|válido|
|practice_0165|mixed|E3|donante2/coste0/orden1|224|224|válido|
|practice_0165|mixed|E3|donante2/coste1/orden1|224|224|válido|
|practice_0165|mixed|E3|donante3/coste0/orden0|278|219|válido|
|practice_0165|mixed|E3|donante3/coste1/orden0|250|219|válido|
|practice_0165|mixed|E3|donante3/coste0/orden1|229|219|válido|
|practice_0165|mixed|E3|donante3/coste1/orden1|225|219|válido|
|practice_0165|mixed|E3|donante4/coste0/orden0|238|238|válido|
|practice_0165|mixed|E3|donante4/coste1/orden0|238|238|válido|
|practice_0165|mixed|E3|donante4/coste0/orden1|238|238|válido|
|practice_0165|mixed|E3|donante4/coste1/orden1|238|238|válido|
|practice_0165|mixed|E3|donante5/coste0/orden0|230|230|válido|
|practice_0165|mixed|E3|donante5/coste1/orden0|230|230|válido|
|practice_0165|mixed|E3|donante5/coste0/orden1|230|230|válido|
|practice_0165|mixed|E3|donante5/coste1/orden1|230|230|válido|
|practice_0165|mixed|E3|donante6/coste0/orden0|242|242|válido|
|practice_0165|mixed|E3|donante6/coste1/orden0|242|242|válido|
|practice_0165|mixed|E3|donante6/coste0/orden1|242|242|válido|
|practice_0165|mixed|E3|donante6/coste1/orden1|242|242|válido|
|practice_0165|mixed|E3|donante7/coste0/orden0|257|219|válido|
|practice_0165|mixed|E3|donante7/coste1/orden0|290|219|válido|
|practice_0165|mixed|E3|donante7/coste0/orden1|243|219|válido|
|practice_0165|mixed|E3|donante7/coste1/orden1|248|219|válido|
|practice_0165|mixed|E3|donante8/coste0/orden0|226|208|válido|
|practice_0165|mixed|E3|donante8/coste1/orden0|219|208|válido|
|practice_0165|mixed|E3|donante8/coste0/orden1|214|208|válido|
|practice_0165|mixed|E3|donante8/coste1/orden1|214|208|válido|
|practice_0166|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0166|shared|Cruce|CP True/coste0/orden0|39|39|válido|
|practice_0166|shared|Cruce|CP True/coste1/orden0|39|39|válido|
|practice_0166|shared|Cruce|CP True/coste0/orden1|39|39|válido|
|practice_0166|shared|Cruce|CP True/coste1/orden1|39|39|válido|
|practice_0166|shared|Cruce|mejor independiente/coste0/orden0|39|39|válido|
|practice_0166|shared|Cruce|mejor independiente/coste1/orden0|39|39|válido|
|practice_0166|shared|Cruce|mejor independiente/coste0/orden1|39|39|válido|
|practice_0166|shared|Cruce|mejor independiente/coste1/orden1|39|39|válido|
|practice_0166|shared|E1|cola10/pol1/inicio0|47|32|válido|
|practice_0166|shared|E1|cola10/pol1/inicio1|47|32|válido|
|practice_0166|shared|E1|cola10/pol2/inicio0|46|31|válido|
|practice_0166|shared|E1|cola10/pol2/inicio1|47|32|válido|
|practice_0166|shared|E1|cola8/pol1/inicio0|44|21|válido|
|practice_0166|shared|E1|cola8/pol1/inicio1|44|21|válido|
|practice_0166|shared|E1|cola8/pol2/inicio0|44|21|válido|
|practice_0166|shared|E1|cola8/pol2/inicio1|44|21|válido|
|practice_0166|shared|E1|cola6/pol1/inicio0|44|17|válido|
|practice_0166|shared|E1|cola6/pol1/inicio1|44|17|válido|
|practice_0166|shared|E1|cola6/pol2/inicio0|44|17|válido|
|practice_0166|shared|E1|cola6/pol2/inicio1|44|17|válido|
|practice_0166|shared|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0166|shared|E2|CP False/coste0/orden0|43|43|válido|
|practice_0166|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0166|shared|E2|CP True/coste0/orden0|39|39|válido|
|practice_0166|shared|E3|donante0/coste0/orden0|43|43|válido|
|practice_0166|shared|E3|donante0/coste1/orden0|43|43|válido|
|practice_0166|shared|E3|donante0/coste0/orden1|43|43|válido|
|practice_0166|shared|E3|donante0/coste1/orden1|43|43|válido|
|practice_0166|shared|E3|donante1/coste0/orden0|54|54|válido|
|practice_0166|shared|E3|donante1/coste1/orden0|54|54|válido|
|practice_0166|shared|E3|donante1/coste0/orden1|54|54|válido|
|practice_0166|shared|E3|donante1/coste1/orden1|54|54|válido|
|practice_0166|shared|E3|donante2/coste0/orden0|49|49|válido|
|practice_0166|shared|E3|donante2/coste1/orden0|49|49|válido|
|practice_0166|shared|E3|donante2/coste0/orden1|49|49|válido|
|practice_0166|shared|E3|donante2/coste1/orden1|49|49|válido|
|practice_0166|shared|E3|donante3/coste0/orden0|54|54|válido|
|practice_0166|shared|E3|donante3/coste1/orden0|54|54|válido|
|practice_0166|shared|E3|donante3/coste0/orden1|54|54|válido|
|practice_0166|shared|E3|donante3/coste1/orden1|54|54|válido|
|practice_0166|shared|E3|donante4/coste0/orden0|49|49|válido|
|practice_0166|shared|E3|donante4/coste1/orden0|49|49|válido|
|practice_0166|shared|E3|donante4/coste0/orden1|49|49|válido|
|practice_0166|shared|E3|donante4/coste1/orden1|49|49|válido|
|practice_0166|shared|E3|donante5/coste0/orden0|47|47|válido|
|practice_0166|shared|E3|donante5/coste1/orden0|47|47|válido|
|practice_0166|shared|E3|donante5/coste0/orden1|47|47|válido|
|practice_0166|shared|E3|donante5/coste1/orden1|47|47|válido|
|practice_0166|shared|E3|donante6/coste0/orden0|53|53|válido|
|practice_0166|shared|E3|donante6/coste1/orden0|53|53|válido|
|practice_0166|shared|E3|donante6/coste0/orden1|53|53|válido|
|practice_0166|shared|E3|donante6/coste1/orden1|53|53|válido|
|practice_0167|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0167|mixed|Cruce|CP True/coste0/orden0|294|262|válido|
|practice_0167|mixed|Cruce|CP True/coste1/orden0|283|262|válido|
|practice_0167|mixed|Cruce|CP True/coste0/orden1|273|262|válido|
|practice_0167|mixed|Cruce|CP True/coste1/orden1|266|262|válido|
|practice_0167|mixed|Cruce|mejor independiente/coste0/orden0|295|263|válido|
|practice_0167|mixed|Cruce|mejor independiente/coste1/orden0|284|263|válido|
|practice_0167|mixed|Cruce|mejor independiente/coste0/orden1|274|263|válido|
|practice_0167|mixed|Cruce|mejor independiente/coste1/orden1|267|263|válido|
|practice_0167|mixed|E1|cola10/pol1/inicio0|280|92|válido|
|practice_0167|mixed|E1|cola10/pol1/inicio1|280|92|válido|
|practice_0167|mixed|E1|cola10/pol2/inicio0|279|91|válido|
|practice_0167|mixed|E1|cola10/pol2/inicio1|279|91|válido|
|practice_0167|mixed|E1|cola8/pol1/inicio0|278|73|válido|
|practice_0167|mixed|E1|cola8/pol1/inicio1|278|73|válido|
|practice_0167|mixed|E1|cola8/pol2/inicio0|274|69|válido|
|practice_0167|mixed|E1|cola8/pol2/inicio1|274|69|válido|
|practice_0167|mixed|E1|cola6/pol1/inicio0|279|59|válido|
|practice_0167|mixed|E1|cola6/pol1/inicio1|279|59|válido|
|practice_0167|mixed|E1|cola6/pol2/inicio0|276|56|válido|
|practice_0167|mixed|E1|cola6/pol2/inicio1|276|56|válido|
|practice_0167|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0167|mixed|E2|CP False/coste0/orden0|329|301|válido|
|practice_0167|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0167|mixed|E2|CP True/coste0/orden0|295|263|válido|
|practice_0167|mixed|E3|donante0/coste0/orden0|274|263|válido|
|practice_0167|mixed|E3|donante0/coste1/orden0|275|263|válido|
|practice_0167|mixed|E3|donante0/coste0/orden1|274|263|válido|
|practice_0167|mixed|E3|donante0/coste1/orden1|267|263|válido|
|practice_0167|mixed|E3|donante1/coste0/orden0|300|300|válido|
|practice_0167|mixed|E3|donante1/coste1/orden0|300|300|válido|
|practice_0167|mixed|E3|donante1/coste0/orden1|300|300|válido|
|practice_0167|mixed|E3|donante1/coste1/orden1|300|300|válido|
|practice_0167|mixed|E3|donante2/coste0/orden0|407|290|válido|
|practice_0167|mixed|E3|donante2/coste1/orden0|295|290|válido|
|practice_0167|mixed|E3|donante2/coste0/orden1|290|290|válido|
|practice_0167|mixed|E3|donante2/coste1/orden1|290|290|válido|
|practice_0167|mixed|E3|donante3/coste0/orden0|418|280|válido|
|practice_0167|mixed|E3|donante3/coste1/orden0|303|280|válido|
|practice_0167|mixed|E3|donante3/coste0/orden1|280|280|válido|
|practice_0167|mixed|E3|donante3/coste1/orden1|280|280|válido|
|practice_0167|mixed|E3|donante4/coste0/orden0|304|304|válido|
|practice_0167|mixed|E3|donante4/coste1/orden0|304|304|válido|
|practice_0167|mixed|E3|donante4/coste0/orden1|304|304|válido|
|practice_0167|mixed|E3|donante4/coste1/orden1|304|304|válido|
|practice_0167|mixed|E3|donante5/coste0/orden0|312|312|válido|
|practice_0167|mixed|E3|donante5/coste1/orden0|312|312|válido|
|practice_0167|mixed|E3|donante5/coste0/orden1|312|312|válido|
|practice_0167|mixed|E3|donante5/coste1/orden1|312|312|válido|
|practice_0167|mixed|E3|donante6/coste0/orden0|304|304|válido|
|practice_0167|mixed|E3|donante6/coste1/orden0|304|304|válido|
|practice_0167|mixed|E3|donante6/coste0/orden1|304|304|válido|
|practice_0167|mixed|E3|donante6/coste1/orden1|304|304|válido|
|practice_0167|mixed|E3|donante7/coste0/orden0|322|272|válido|
|practice_0167|mixed|E3|donante7/coste1/orden0|309|272|válido|
|practice_0167|mixed|E3|donante7/coste0/orden1|303|272|válido|
|practice_0167|mixed|E3|donante7/coste1/orden1|299|272|válido|
|practice_0167|mixed|E3|donante8/coste0/orden0|275|264|válido|
|practice_0167|mixed|E3|donante8/coste1/orden0|268|264|válido|
|practice_0167|mixed|E3|donante8/coste0/orden1|275|264|válido|
|practice_0167|mixed|E3|donante8/coste1/orden1|268|264|válido|
|practice_0167|mixed|E3|donante9/coste0/orden0|311|264|válido|
|practice_0167|mixed|E3|donante9/coste1/orden0|283|264|válido|
|practice_0167|mixed|E3|donante9/coste0/orden1|285|264|válido|
|practice_0167|mixed|E3|donante9/coste1/orden1|352|264|válido|
|practice_0168|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0168|multiplicative|Cruce|CP True/coste0/orden0|116|116|válido|
|practice_0168|multiplicative|Cruce|CP True/coste1/orden0|116|116|válido|
|practice_0168|multiplicative|Cruce|CP True/coste0/orden1|116|116|válido|
|practice_0168|multiplicative|Cruce|CP True/coste1/orden1|116|116|válido|
|practice_0168|multiplicative|Cruce|mejor independiente/coste0/orden0|116|116|válido|
|practice_0168|multiplicative|Cruce|mejor independiente/coste1/orden0|116|116|válido|
|practice_0168|multiplicative|Cruce|mejor independiente/coste0/orden1|116|116|válido|
|practice_0168|multiplicative|Cruce|mejor independiente/coste1/orden1|116|116|válido|
|practice_0168|multiplicative|E1|cola10/pol1/inicio0|—|—|registros ocupados|
|practice_0168|multiplicative|E1|cola10/pol1/inicio1|—|—|registros ocupados|
|practice_0168|multiplicative|E1|cola10/pol2/inicio0|—|—|registros ocupados|
|practice_0168|multiplicative|E1|cola10/pol2/inicio1|—|—|registros ocupados|
|practice_0168|multiplicative|E1|cola8/pol1/inicio0|—|—|registros ocupados|
|practice_0168|multiplicative|E1|cola8/pol1/inicio1|—|—|registros ocupados|
|practice_0168|multiplicative|E1|cola8/pol2/inicio0|—|—|registros ocupados|
|practice_0168|multiplicative|E1|cola8/pol2/inicio1|—|—|registros ocupados|
|practice_0168|multiplicative|E1|cola6/pol1/inicio0|—|—|registros ocupados|
|practice_0168|multiplicative|E1|cola6/pol1/inicio1|—|—|registros ocupados|
|practice_0168|multiplicative|E1|cola6/pol2/inicio0|—|—|registros ocupados|
|practice_0168|multiplicative|E1|cola6/pol2/inicio1|—|—|registros ocupados|
|practice_0168|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0168|multiplicative|E2|CP False/coste0/orden0|118|118|válido|
|practice_0168|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0168|multiplicative|E2|CP True/coste0/orden0|116|116|válido|
|practice_0168|multiplicative|E3|donante0/coste0/orden0|117|117|válido|
|practice_0168|multiplicative|E3|donante0/coste1/orden0|117|117|válido|
|practice_0168|multiplicative|E3|donante0/coste0/orden1|117|117|válido|
|practice_0168|multiplicative|E3|donante0/coste1/orden1|117|117|válido|
|practice_0168|multiplicative|E3|donante1/coste0/orden0|130|130|válido|
|practice_0168|multiplicative|E3|donante1/coste1/orden0|130|130|válido|
|practice_0168|multiplicative|E3|donante1/coste0/orden1|130|130|válido|
|practice_0168|multiplicative|E3|donante1/coste1/orden1|130|130|válido|
|practice_0168|multiplicative|E3|donante2/coste0/orden0|127|127|válido|
|practice_0168|multiplicative|E3|donante2/coste1/orden0|127|127|válido|
|practice_0168|multiplicative|E3|donante2/coste0/orden1|127|127|válido|
|practice_0168|multiplicative|E3|donante2/coste1/orden1|127|127|válido|
|practice_0168|multiplicative|E3|donante3/coste0/orden0|122|122|válido|
|practice_0168|multiplicative|E3|donante3/coste1/orden0|122|122|válido|
|practice_0168|multiplicative|E3|donante3/coste0/orden1|122|122|válido|
|practice_0168|multiplicative|E3|donante3/coste1/orden1|122|122|válido|
|practice_0168|multiplicative|E3|donante4/coste0/orden0|132|132|válido|
|practice_0168|multiplicative|E3|donante4/coste1/orden0|132|132|válido|
|practice_0168|multiplicative|E3|donante4/coste0/orden1|132|132|válido|
|practice_0168|multiplicative|E3|donante4/coste1/orden1|132|132|válido|
|practice_0168|multiplicative|E3|donante5/coste0/orden0|129|129|válido|
|practice_0168|multiplicative|E3|donante5/coste1/orden0|129|129|válido|
|practice_0168|multiplicative|E3|donante5/coste0/orden1|129|129|válido|
|practice_0168|multiplicative|E3|donante5/coste1/orden1|129|129|válido|
|practice_0168|multiplicative|E3|donante6/coste0/orden0|121|121|válido|
|practice_0168|multiplicative|E3|donante6/coste1/orden0|121|121|válido|
|practice_0168|multiplicative|E3|donante6/coste0/orden1|121|121|válido|
|practice_0168|multiplicative|E3|donante6/coste1/orden1|121|121|válido|
|practice_0168|multiplicative|E3|donante7/coste0/orden0|128|128|válido|
|practice_0168|multiplicative|E3|donante7/coste1/orden0|128|128|válido|
|practice_0168|multiplicative|E3|donante7/coste0/orden1|128|128|válido|
|practice_0168|multiplicative|E3|donante7/coste1/orden1|128|128|válido|
|practice_0168|multiplicative|E3|donante8/coste0/orden0|123|123|válido|
|practice_0168|multiplicative|E3|donante8/coste1/orden0|123|123|válido|
|practice_0168|multiplicative|E3|donante8/coste0/orden1|123|123|válido|
|practice_0168|multiplicative|E3|donante8/coste1/orden1|123|123|válido|
|practice_0168|multiplicative|E3|donante9/coste0/orden0|119|119|válido|
|practice_0168|multiplicative|E3|donante9/coste1/orden0|119|119|válido|
|practice_0168|multiplicative|E3|donante9/coste0/orden1|119|119|válido|
|practice_0168|multiplicative|E3|donante9/coste1/orden1|119|119|válido|
|practice_0169|tiny|Cruce|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0169|tiny|Cruce|CP True/coste0/orden0|3|3|válido|
|practice_0169|tiny|Cruce|CP True/coste1/orden0|3|3|válido|
|practice_0169|tiny|Cruce|CP True/coste0/orden1|3|3|válido|
|practice_0169|tiny|Cruce|CP True/coste1/orden1|3|3|válido|
|practice_0169|tiny|Cruce|mejor independiente/coste0/orden0|3|3|válido|
|practice_0169|tiny|Cruce|mejor independiente/coste1/orden0|3|3|válido|
|practice_0169|tiny|Cruce|mejor independiente/coste0/orden1|3|3|válido|
|practice_0169|tiny|Cruce|mejor independiente/coste1/orden1|3|3|válido|
|practice_0169|tiny|E1|cola3/pol1/inicio0|6|5|válido|
|practice_0169|tiny|E1|cola3/pol1/inicio1|6|5|válido|
|practice_0169|tiny|E1|cola3/pol2/inicio0|6|5|válido|
|practice_0169|tiny|E1|cola3/pol2/inicio1|6|5|válido|
|practice_0169|tiny|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0169|tiny|E2|CP False/coste0/orden0|4|4|válido|
|practice_0169|tiny|E2|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0169|tiny|E2|CP True/coste0/orden0|3|3|válido|
|practice_0169|tiny|E3|donante0/coste0/orden0|4|4|válido|
|practice_0169|tiny|E3|donante0/coste1/orden0|4|4|válido|
|practice_0169|tiny|E3|donante0/coste0/orden1|4|4|válido|
|practice_0169|tiny|E3|donante0/coste1/orden1|4|4|válido|
|practice_0169|tiny|E3|donante1/coste0/orden0|6|6|válido|
|practice_0169|tiny|E3|donante1/coste1/orden0|6|6|válido|
|practice_0169|tiny|E3|donante1/coste0/orden1|6|6|válido|
|practice_0169|tiny|E3|donante1/coste1/orden1|6|6|válido|
|practice_0170|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0170|multiplicative|Cruce|CP True/coste0/orden0|28|28|válido|
|practice_0170|multiplicative|Cruce|CP True/coste1/orden0|28|28|válido|
|practice_0170|multiplicative|Cruce|CP True/coste0/orden1|28|28|válido|
|practice_0170|multiplicative|Cruce|CP True/coste1/orden1|28|28|válido|
|practice_0170|multiplicative|Cruce|mejor independiente/coste0/orden0|28|28|válido|
|practice_0170|multiplicative|Cruce|mejor independiente/coste1/orden0|28|28|válido|
|practice_0170|multiplicative|Cruce|mejor independiente/coste0/orden1|28|28|válido|
|practice_0170|multiplicative|Cruce|mejor independiente/coste1/orden1|28|28|válido|
|practice_0170|multiplicative|E1|cola10/pol1/inicio0|37|24|válido|
|practice_0170|multiplicative|E1|cola10/pol1/inicio1|37|24|válido|
|practice_0170|multiplicative|E1|cola10/pol2/inicio0|38|25|válido|
|practice_0170|multiplicative|E1|cola10/pol2/inicio1|38|25|válido|
|practice_0170|multiplicative|E1|cola8/pol1/inicio0|34|17|válido|
|practice_0170|multiplicative|E1|cola8/pol1/inicio1|34|17|válido|
|practice_0170|multiplicative|E1|cola8/pol2/inicio0|34|17|válido|
|practice_0170|multiplicative|E1|cola8/pol2/inicio1|34|17|válido|
|practice_0170|multiplicative|E1|cola6/pol1/inicio0|35|13|válido|
|practice_0170|multiplicative|E1|cola6/pol1/inicio1|35|13|válido|
|practice_0170|multiplicative|E1|cola6/pol2/inicio0|35|13|válido|
|practice_0170|multiplicative|E1|cola6/pol2/inicio1|35|13|válido|
|practice_0170|multiplicative|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0170|multiplicative|E2|CP False/coste0/orden0|34|34|válido|
|practice_0170|multiplicative|E2|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0170|multiplicative|E2|CP True/coste0/orden0|28|28|válido|
|practice_0170|multiplicative|E3|donante0/coste0/orden0|34|34|válido|
|practice_0170|multiplicative|E3|donante0/coste1/orden0|34|34|válido|
|practice_0170|multiplicative|E3|donante0/coste0/orden1|34|34|válido|
|practice_0170|multiplicative|E3|donante0/coste1/orden1|34|34|válido|
|practice_0170|multiplicative|E3|donante1/coste0/orden0|41|41|válido|
|practice_0170|multiplicative|E3|donante1/coste1/orden0|41|41|válido|
|practice_0170|multiplicative|E3|donante1/coste0/orden1|41|41|válido|
|practice_0170|multiplicative|E3|donante1/coste1/orden1|41|41|válido|
|practice_0170|multiplicative|E3|donante2/coste0/orden0|41|41|válido|
|practice_0170|multiplicative|E3|donante2/coste1/orden0|41|41|válido|
|practice_0170|multiplicative|E3|donante2/coste0/orden1|41|41|válido|
|practice_0170|multiplicative|E3|donante2/coste1/orden1|41|41|válido|
|practice_0170|multiplicative|E3|donante3/coste0/orden0|39|39|válido|
|practice_0170|multiplicative|E3|donante3/coste1/orden0|39|39|válido|
|practice_0170|multiplicative|E3|donante3/coste0/orden1|39|39|válido|
|practice_0170|multiplicative|E3|donante3/coste1/orden1|39|39|válido|
|practice_0170|multiplicative|E3|donante4/coste0/orden0|39|39|válido|
|practice_0170|multiplicative|E3|donante4/coste1/orden0|39|39|válido|
|practice_0170|multiplicative|E3|donante4/coste0/orden1|39|39|válido|
|practice_0170|multiplicative|E3|donante4/coste1/orden1|39|39|válido|
|practice_0170|multiplicative|E3|donante5/coste0/orden0|37|37|válido|
|practice_0170|multiplicative|E3|donante5/coste1/orden0|37|37|válido|
|practice_0170|multiplicative|E3|donante5/coste0/orden1|37|37|válido|
|practice_0170|multiplicative|E3|donante5/coste1/orden1|37|37|válido|
|practice_0171|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0171|mixed|Cruce|CP True/coste0/orden0|243|243|válido|
|practice_0171|mixed|Cruce|CP True/coste1/orden0|243|243|válido|
|practice_0171|mixed|Cruce|CP True/coste0/orden1|243|243|válido|
|practice_0171|mixed|Cruce|CP True/coste1/orden1|243|243|válido|
|practice_0171|mixed|Cruce|mejor independiente/coste0/orden0|244|244|válido|
|practice_0171|mixed|Cruce|mejor independiente/coste1/orden0|244|244|válido|
|practice_0171|mixed|Cruce|mejor independiente/coste0/orden1|244|244|válido|
|practice_0171|mixed|Cruce|mejor independiente/coste1/orden1|244|244|válido|
|practice_0171|mixed|E1|cola10/pol1/inicio0|260|93|válido|
|practice_0171|mixed|E1|cola10/pol1/inicio1|260|93|válido|
|practice_0171|mixed|E1|cola10/pol2/inicio0|259|92|válido|
|practice_0171|mixed|E1|cola10/pol2/inicio1|259|92|válido|
|practice_0171|mixed|E1|cola8/pol1/inicio0|260|77|válido|
|practice_0171|mixed|E1|cola8/pol1/inicio1|260|77|válido|
|practice_0171|mixed|E1|cola8/pol2/inicio0|256|73|válido|
|practice_0171|mixed|E1|cola8/pol2/inicio1|256|73|válido|
|practice_0171|mixed|E1|cola6/pol1/inicio0|260|60|válido|
|practice_0171|mixed|E1|cola6/pol1/inicio1|261|61|válido|
|practice_0171|mixed|E1|cola6/pol2/inicio0|260|60|válido|
|practice_0171|mixed|E1|cola6/pol2/inicio1|260|60|válido|
|practice_0171|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0171|mixed|E2|CP False/coste0/orden0|245|245|válido|
|practice_0171|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0171|mixed|E2|CP True/coste0/orden0|244|244|válido|
|practice_0171|mixed|E3|donante0/coste0/orden0|245|245|válido|
|practice_0171|mixed|E3|donante0/coste1/orden0|245|245|válido|
|practice_0171|mixed|E3|donante0/coste0/orden1|245|245|válido|
|practice_0171|mixed|E3|donante0/coste1/orden1|245|245|válido|
|practice_0171|mixed|E3|donante1/coste0/orden0|292|292|válido|
|practice_0171|mixed|E3|donante1/coste1/orden0|292|292|válido|
|practice_0171|mixed|E3|donante1/coste0/orden1|292|292|válido|
|practice_0171|mixed|E3|donante1/coste1/orden1|292|292|válido|
|practice_0171|mixed|E3|donante2/coste0/orden0|282|273|válido|
|practice_0171|mixed|E3|donante2/coste1/orden0|282|273|válido|
|practice_0171|mixed|E3|donante2/coste0/orden1|273|273|válido|
|practice_0171|mixed|E3|donante2/coste1/orden1|273|273|válido|
|practice_0171|mixed|E3|donante3/coste0/orden0|290|271|válido|
|practice_0171|mixed|E3|donante3/coste1/orden0|285|271|válido|
|practice_0171|mixed|E3|donante3/coste0/orden1|271|271|válido|
|practice_0171|mixed|E3|donante3/coste1/orden1|271|271|válido|
|practice_0171|mixed|E3|donante4/coste0/orden0|293|293|válido|
|practice_0171|mixed|E3|donante4/coste1/orden0|293|293|válido|
|practice_0171|mixed|E3|donante4/coste0/orden1|293|293|válido|
|practice_0171|mixed|E3|donante4/coste1/orden1|293|293|válido|
|practice_0171|mixed|E3|donante5/coste0/orden0|291|291|válido|
|practice_0171|mixed|E3|donante5/coste1/orden0|291|291|válido|
|practice_0171|mixed|E3|donante5/coste0/orden1|291|291|válido|
|practice_0171|mixed|E3|donante5/coste1/orden1|291|291|válido|
|practice_0171|mixed|E3|donante6/coste0/orden0|289|289|válido|
|practice_0171|mixed|E3|donante6/coste1/orden0|289|289|válido|
|practice_0171|mixed|E3|donante6/coste0/orden1|289|289|válido|
|practice_0171|mixed|E3|donante6/coste1/orden1|289|289|válido|
|practice_0171|mixed|E3|donante7/coste0/orden0|275|260|válido|
|practice_0171|mixed|E3|donante7/coste1/orden0|266|260|válido|
|practice_0171|mixed|E3|donante7/coste0/orden1|273|260|válido|
|practice_0171|mixed|E3|donante7/coste1/orden1|273|260|válido|
|practice_0171|mixed|E3|donante8/coste0/orden0|246|246|válido|
|practice_0171|mixed|E3|donante8/coste1/orden0|246|246|válido|
|practice_0171|mixed|E3|donante8/coste0/orden1|246|246|válido|
|practice_0171|mixed|E3|donante8/coste1/orden1|246|246|válido|
|practice_0171|mixed|E3|donante9/coste0/orden0|343|261|válido|
|practice_0171|mixed|E3|donante9/coste1/orden0|295|261|válido|
|practice_0171|mixed|E3|donante9/coste0/orden1|273|261|válido|
|practice_0171|mixed|E3|donante9/coste1/orden1|265|261|válido|
|practice_0172|tiny|Cruce|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0172|tiny|Cruce|CP True/coste0/orden0|2|2|válido|
|practice_0172|tiny|Cruce|CP True/coste1/orden0|2|2|válido|
|practice_0172|tiny|Cruce|CP True/coste0/orden1|2|2|válido|
|practice_0172|tiny|Cruce|CP True/coste1/orden1|2|2|válido|
|practice_0172|tiny|Cruce|mejor independiente/coste0/orden0|2|2|válido|
|practice_0172|tiny|Cruce|mejor independiente/coste1/orden0|2|2|válido|
|practice_0172|tiny|Cruce|mejor independiente/coste0/orden1|2|2|válido|
|practice_0172|tiny|Cruce|mejor independiente/coste1/orden1|2|2|válido|
|practice_0172|tiny|E1|cola1/pol1/inicio0|2|1|válido|
|practice_0172|tiny|E1|cola1/pol2/inicio0|2|1|válido|
|practice_0172|tiny|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0172|tiny|E2|CP False/coste0/orden0|2|2|válido|
|practice_0172|tiny|E2|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0172|tiny|E2|CP True/coste0/orden0|2|2|válido|
|practice_0172|tiny|E3|donante0/coste0/orden0|2|2|válido|
|practice_0172|tiny|E3|donante0/coste1/orden0|2|2|válido|
|practice_0172|tiny|E3|donante0/coste0/orden1|2|2|válido|
|practice_0172|tiny|E3|donante0/coste1/orden1|2|2|válido|
|practice_0172|tiny|E3|donante1/coste0/orden0|2|2|válido|
|practice_0172|tiny|E3|donante1/coste1/orden0|2|2|válido|
|practice_0172|tiny|E3|donante1/coste0/orden1|2|2|válido|
|practice_0172|tiny|E3|donante1/coste1/orden1|2|2|válido|
|practice_0173|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0173|shared|Cruce|CP True/coste0/orden0|128|128|válido|
|practice_0173|shared|Cruce|CP True/coste1/orden0|128|128|válido|
|practice_0173|shared|Cruce|CP True/coste0/orden1|128|128|válido|
|practice_0173|shared|Cruce|CP True/coste1/orden1|128|128|válido|
|practice_0173|shared|Cruce|mejor independiente/coste0/orden0|130|130|válido|
|practice_0173|shared|Cruce|mejor independiente/coste1/orden0|130|130|válido|
|practice_0173|shared|Cruce|mejor independiente/coste0/orden1|130|130|válido|
|practice_0173|shared|Cruce|mejor independiente/coste1/orden1|130|130|válido|
|practice_0173|shared|E1|cola10/pol1/inicio0|134|29|válido|
|practice_0173|shared|E1|cola10/pol1/inicio1|134|29|válido|
|practice_0173|shared|E1|cola10/pol2/inicio0|134|29|válido|
|practice_0173|shared|E1|cola10/pol2/inicio1|134|29|válido|
|practice_0173|shared|E1|cola8/pol1/inicio0|134|23|válido|
|practice_0173|shared|E1|cola8/pol1/inicio1|134|23|válido|
|practice_0173|shared|E1|cola8/pol2/inicio0|134|23|válido|
|practice_0173|shared|E1|cola8/pol2/inicio1|134|23|válido|
|practice_0173|shared|E1|cola6/pol1/inicio0|134|19|válido|
|practice_0173|shared|E1|cola6/pol1/inicio1|134|19|válido|
|practice_0173|shared|E1|cola6/pol2/inicio0|134|19|válido|
|practice_0173|shared|E1|cola6/pol2/inicio1|134|19|válido|
|practice_0173|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0173|shared|E2|CP False/coste0/orden0|138|138|válido|
|practice_0173|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0173|shared|E2|CP True/coste0/orden0|131|131|válido|
|practice_0173|shared|E3|donante0/coste0/orden0|135|135|válido|
|practice_0173|shared|E3|donante0/coste1/orden0|135|135|válido|
|practice_0173|shared|E3|donante0/coste0/orden1|135|135|válido|
|practice_0173|shared|E3|donante0/coste1/orden1|135|135|válido|
|practice_0173|shared|E3|donante1/coste0/orden0|147|147|válido|
|practice_0173|shared|E3|donante1/coste1/orden0|147|147|válido|
|practice_0173|shared|E3|donante1/coste0/orden1|147|147|válido|
|practice_0173|shared|E3|donante1/coste1/orden1|147|147|válido|
|practice_0173|shared|E3|donante2/coste0/orden0|152|152|válido|
|practice_0173|shared|E3|donante2/coste1/orden0|152|152|válido|
|practice_0173|shared|E3|donante2/coste0/orden1|152|152|válido|
|practice_0173|shared|E3|donante2/coste1/orden1|152|152|válido|
|practice_0173|shared|E3|donante3/coste0/orden0|136|136|válido|
|practice_0173|shared|E3|donante3/coste1/orden0|136|136|válido|
|practice_0173|shared|E3|donante3/coste0/orden1|136|136|válido|
|practice_0173|shared|E3|donante3/coste1/orden1|136|136|válido|
|practice_0173|shared|E3|donante4/coste0/orden0|169|169|válido|
|practice_0173|shared|E3|donante4/coste1/orden0|169|169|válido|
|practice_0173|shared|E3|donante4/coste0/orden1|169|169|válido|
|practice_0173|shared|E3|donante4/coste1/orden1|169|169|válido|
|practice_0173|shared|E3|donante5/coste0/orden0|154|154|válido|
|practice_0173|shared|E3|donante5/coste1/orden0|154|154|válido|
|practice_0173|shared|E3|donante5/coste0/orden1|154|154|válido|
|practice_0173|shared|E3|donante5/coste1/orden1|154|154|válido|
|practice_0173|shared|E3|donante6/coste0/orden0|161|161|válido|
|practice_0173|shared|E3|donante6/coste1/orden0|161|161|válido|
|practice_0173|shared|E3|donante6/coste0/orden1|161|161|válido|
|practice_0173|shared|E3|donante6/coste1/orden1|161|161|válido|
|practice_0173|shared|E3|donante7/coste0/orden0|167|150|válido|
|practice_0173|shared|E3|donante7/coste1/orden0|155|150|válido|
|practice_0173|shared|E3|donante7/coste0/orden1|150|150|válido|
|practice_0173|shared|E3|donante7/coste1/orden1|150|150|válido|
|practice_0173|shared|E3|donante8/coste0/orden0|148|143|válido|
|practice_0173|shared|E3|donante8/coste1/orden0|153|143|válido|
|practice_0173|shared|E3|donante8/coste0/orden1|151|143|válido|
|practice_0173|shared|E3|donante8/coste1/orden1|152|143|válido|
|practice_0173|shared|E3|donante9/coste0/orden0|153|143|válido|
|practice_0173|shared|E3|donante9/coste1/orden0|153|143|válido|
|practice_0173|shared|E3|donante9/coste0/orden1|143|143|válido|
|practice_0173|shared|E3|donante9/coste1/orden1|143|143|válido|
|practice_0174|tiny|Cruce|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0174|tiny|Cruce|CP True/coste0/orden0|2|2|válido|
|practice_0174|tiny|Cruce|CP True/coste1/orden0|2|2|válido|
|practice_0174|tiny|Cruce|CP True/coste0/orden1|2|2|válido|
|practice_0174|tiny|Cruce|CP True/coste1/orden1|2|2|válido|
|practice_0174|tiny|Cruce|mejor independiente/coste0/orden0|2|2|válido|
|practice_0174|tiny|Cruce|mejor independiente/coste1/orden0|2|2|válido|
|practice_0174|tiny|Cruce|mejor independiente/coste0/orden1|2|2|válido|
|practice_0174|tiny|Cruce|mejor independiente/coste1/orden1|2|2|válido|
|practice_0174|tiny|E1|cola1/pol1/inicio0|2|1|válido|
|practice_0174|tiny|E1|cola1/pol2/inicio0|2|1|válido|
|practice_0174|tiny|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0174|tiny|E2|CP False/coste0/orden0|2|2|válido|
|practice_0174|tiny|E2|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0174|tiny|E2|CP True/coste0/orden0|2|2|válido|
|practice_0174|tiny|E3|donante0/coste0/orden0|2|2|válido|
|practice_0174|tiny|E3|donante0/coste1/orden0|2|2|válido|
|practice_0174|tiny|E3|donante0/coste0/orden1|2|2|válido|
|practice_0174|tiny|E3|donante0/coste1/orden1|2|2|válido|
|practice_0174|tiny|E3|donante1/coste0/orden0|2|2|válido|
|practice_0174|tiny|E3|donante1/coste1/orden0|2|2|válido|
|practice_0174|tiny|E3|donante1/coste0/orden1|2|2|válido|
|practice_0174|tiny|E3|donante1/coste1/orden1|2|2|válido|
|practice_0175|tiny|Cruce|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0175|tiny|Cruce|CP True/coste0/orden0|4|4|válido|
|practice_0175|tiny|Cruce|CP True/coste1/orden0|4|4|válido|
|practice_0175|tiny|Cruce|CP True/coste0/orden1|4|4|válido|
|practice_0175|tiny|Cruce|CP True/coste1/orden1|4|4|válido|
|practice_0175|tiny|Cruce|mejor independiente/coste0/orden0|4|4|válido|
|practice_0175|tiny|Cruce|mejor independiente/coste1/orden0|4|4|válido|
|practice_0175|tiny|Cruce|mejor independiente/coste0/orden1|4|4|válido|
|practice_0175|tiny|Cruce|mejor independiente/coste1/orden1|4|4|válido|
|practice_0175|tiny|E1|cola3/pol1/inicio0|7|6|válido|
|practice_0175|tiny|E1|cola3/pol1/inicio1|7|6|válido|
|practice_0175|tiny|E1|cola3/pol2/inicio0|7|6|válido|
|practice_0175|tiny|E1|cola3/pol2/inicio1|7|6|válido|
|practice_0175|tiny|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0175|tiny|E2|CP False/coste0/orden0|4|4|válido|
|practice_0175|tiny|E2|CP-SAT cruzado|—|—|OPTIMAL|
|practice_0175|tiny|E2|CP True/coste0/orden0|4|4|válido|
|practice_0175|tiny|E3|donante0/coste0/orden0|4|4|válido|
|practice_0175|tiny|E3|donante0/coste1/orden0|4|4|válido|
|practice_0175|tiny|E3|donante0/coste0/orden1|4|4|válido|
|practice_0175|tiny|E3|donante0/coste1/orden1|4|4|válido|
|practice_0175|tiny|E3|donante1/coste0/orden0|6|6|válido|
|practice_0175|tiny|E3|donante1/coste1/orden0|6|6|válido|
|practice_0175|tiny|E3|donante1/coste0/orden1|6|6|válido|
|practice_0175|tiny|E3|donante1/coste1/orden1|6|6|válido|
|practice_0175|tiny|E3|donante2/coste0/orden0|7|7|válido|
|practice_0175|tiny|E3|donante2/coste1/orden0|7|7|válido|
|practice_0175|tiny|E3|donante2/coste0/orden1|7|7|válido|
|practice_0175|tiny|E3|donante2/coste1/orden1|7|7|válido|
|practice_0176|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0176|independent|Cruce|CP True/coste0/orden0|399|275|válido|
|practice_0176|independent|Cruce|CP True/coste1/orden0|369|275|válido|
|practice_0176|independent|Cruce|CP True/coste0/orden1|330|275|válido|
|practice_0176|independent|Cruce|CP True/coste1/orden1|298|275|válido|
|practice_0176|independent|Cruce|mejor independiente/coste0/orden0|399|275|válido|
|practice_0176|independent|Cruce|mejor independiente/coste1/orden0|369|275|válido|
|practice_0176|independent|Cruce|mejor independiente/coste0/orden1|330|275|válido|
|practice_0176|independent|Cruce|mejor independiente/coste1/orden1|298|275|válido|
|practice_0176|independent|E1|cola10/pol1/inicio0|300|89|válido|
|practice_0176|independent|E1|cola10/pol1/inicio1|300|89|válido|
|practice_0176|independent|E1|cola10/pol2/inicio0|300|89|válido|
|practice_0176|independent|E1|cola10/pol2/inicio1|300|89|válido|
|practice_0176|independent|E1|cola8/pol1/inicio0|307|80|válido|
|practice_0176|independent|E1|cola8/pol1/inicio1|310|83|válido|
|practice_0176|independent|E1|cola8/pol2/inicio0|303|76|válido|
|practice_0176|independent|E1|cola8/pol2/inicio1|306|79|válido|
|practice_0176|independent|E1|cola6/pol1/inicio0|300|56|válido|
|practice_0176|independent|E1|cola6/pol1/inicio1|300|56|válido|
|practice_0176|independent|E1|cola6/pol2/inicio0|300|56|válido|
|practice_0176|independent|E1|cola6/pol2/inicio1|300|56|válido|
|practice_0176|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0176|independent|E2|CP False/coste0/orden0|552|351|válido|
|practice_0176|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0176|independent|E2|CP True/coste0/orden0|392|274|válido|
|practice_0176|independent|E3|donante0/coste0/orden0|300|275|válido|
|practice_0176|independent|E3|donante0/coste1/orden0|286|275|válido|
|practice_0176|independent|E3|donante0/coste0/orden1|330|275|válido|
|practice_0176|independent|E3|donante0/coste1/orden1|298|275|válido|
|practice_0176|independent|E3|donante1/coste0/orden0|318|318|válido|
|practice_0176|independent|E3|donante1/coste1/orden0|318|318|válido|
|practice_0176|independent|E3|donante1/coste0/orden1|318|318|válido|
|practice_0176|independent|E3|donante1/coste1/orden1|318|318|válido|
|practice_0176|independent|E3|donante2/coste0/orden0|372|306|válido|
|practice_0176|independent|E3|donante2/coste1/orden0|324|306|válido|
|practice_0176|independent|E3|donante2/coste0/orden1|353|306|válido|
|practice_0176|independent|E3|donante2/coste1/orden1|337|306|válido|
|practice_0176|independent|E3|donante3/coste0/orden0|499|301|válido|
|practice_0176|independent|E3|donante3/coste1/orden0|362|301|válido|
|practice_0176|independent|E3|donante3/coste0/orden1|366|301|válido|
|practice_0176|independent|E3|donante3/coste1/orden1|305|301|válido|
|practice_0176|independent|E3|donante4/coste0/orden0|334|334|válido|
|practice_0176|independent|E3|donante4/coste1/orden0|334|334|válido|
|practice_0176|independent|E3|donante4/coste0/orden1|334|334|válido|
|practice_0176|independent|E3|donante4/coste1/orden1|334|334|válido|
|practice_0176|independent|E3|donante5/coste0/orden0|326|326|válido|
|practice_0176|independent|E3|donante5/coste1/orden0|326|326|válido|
|practice_0176|independent|E3|donante5/coste0/orden1|326|326|válido|
|practice_0176|independent|E3|donante5/coste1/orden1|326|326|válido|
|practice_0176|independent|E3|donante6/coste0/orden0|330|330|válido|
|practice_0176|independent|E3|donante6/coste1/orden0|330|330|válido|
|practice_0176|independent|E3|donante6/coste0/orden1|330|330|válido|
|practice_0176|independent|E3|donante6/coste1/orden1|330|330|válido|
|practice_0176|independent|E3|donante7/coste0/orden0|362|297|válido|
|practice_0176|independent|E3|donante7/coste1/orden0|343|297|válido|
|practice_0176|independent|E3|donante7/coste0/orden1|376|297|válido|
|practice_0176|independent|E3|donante7/coste1/orden1|381|297|válido|
|practice_0176|independent|E3|donante8/coste0/orden0|317|287|válido|
|practice_0176|independent|E3|donante8/coste1/orden0|290|287|válido|
|practice_0176|independent|E3|donante8/coste0/orden1|506|287|válido|
|practice_0176|independent|E3|donante8/coste1/orden1|334|287|válido|
|practice_0176|independent|E3|donante9/coste0/orden0|300|275|válido|
|practice_0176|independent|E3|donante9/coste1/orden0|286|275|válido|
|practice_0176|independent|E3|donante9/coste0/orden1|330|275|válido|
|practice_0176|independent|E3|donante9/coste1/orden1|298|275|válido|
|practice_0177|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0177|mixed|Cruce|CP True/coste0/orden0|162|162|válido|
|practice_0177|mixed|Cruce|CP True/coste1/orden0|162|162|válido|
|practice_0177|mixed|Cruce|CP True/coste0/orden1|165|162|válido|
|practice_0177|mixed|Cruce|CP True/coste1/orden1|165|162|válido|
|practice_0177|mixed|Cruce|mejor independiente/coste0/orden0|164|164|válido|
|practice_0177|mixed|Cruce|mejor independiente/coste1/orden0|164|164|válido|
|practice_0177|mixed|Cruce|mejor independiente/coste0/orden1|164|164|válido|
|practice_0177|mixed|Cruce|mejor independiente/coste1/orden1|164|164|válido|
|practice_0177|mixed|E1|cola10/pol1/inicio0|170|57|válido|
|practice_0177|mixed|E1|cola10/pol1/inicio1|170|57|válido|
|practice_0177|mixed|E1|cola10/pol2/inicio0|170|57|válido|
|practice_0177|mixed|E1|cola10/pol2/inicio1|170|57|válido|
|practice_0177|mixed|E1|cola8/pol1/inicio0|171|48|válido|
|practice_0177|mixed|E1|cola8/pol1/inicio1|171|48|válido|
|practice_0177|mixed|E1|cola8/pol2/inicio0|169|46|válido|
|practice_0177|mixed|E1|cola8/pol2/inicio1|171|48|válido|
|practice_0177|mixed|E1|cola6/pol1/inicio0|166|32|válido|
|practice_0177|mixed|E1|cola6/pol1/inicio1|166|32|válido|
|practice_0177|mixed|E1|cola6/pol2/inicio0|166|32|válido|
|practice_0177|mixed|E1|cola6/pol2/inicio1|166|32|válido|
|practice_0177|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0177|mixed|E2|CP False/coste0/orden0|165|165|válido|
|practice_0177|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0177|mixed|E2|CP True/coste0/orden0|164|164|válido|
|practice_0177|mixed|E3|donante0/coste0/orden0|165|165|válido|
|practice_0177|mixed|E3|donante0/coste1/orden0|165|165|válido|
|practice_0177|mixed|E3|donante0/coste0/orden1|165|165|válido|
|practice_0177|mixed|E3|donante0/coste1/orden1|165|165|válido|
|practice_0177|mixed|E3|donante1/coste0/orden0|179|179|válido|
|practice_0177|mixed|E3|donante1/coste1/orden0|179|179|válido|
|practice_0177|mixed|E3|donante1/coste0/orden1|179|179|válido|
|practice_0177|mixed|E3|donante1/coste1/orden1|179|179|válido|
|practice_0177|mixed|E3|donante2/coste0/orden0|177|177|válido|
|practice_0177|mixed|E3|donante2/coste1/orden0|177|177|válido|
|practice_0177|mixed|E3|donante2/coste0/orden1|177|177|válido|
|practice_0177|mixed|E3|donante2/coste1/orden1|177|177|válido|
|practice_0177|mixed|E3|donante3/coste0/orden0|171|171|válido|
|practice_0177|mixed|E3|donante3/coste1/orden0|171|171|válido|
|practice_0177|mixed|E3|donante3/coste0/orden1|171|171|válido|
|practice_0177|mixed|E3|donante3/coste1/orden1|171|171|válido|
|practice_0177|mixed|E3|donante4/coste0/orden0|188|188|válido|
|practice_0177|mixed|E3|donante4/coste1/orden0|188|188|válido|
|practice_0177|mixed|E3|donante4/coste0/orden1|188|188|válido|
|practice_0177|mixed|E3|donante4/coste1/orden1|188|188|válido|
|practice_0177|mixed|E3|donante5/coste0/orden0|187|187|válido|
|practice_0177|mixed|E3|donante5/coste1/orden0|187|187|válido|
|practice_0177|mixed|E3|donante5/coste0/orden1|187|187|válido|
|practice_0177|mixed|E3|donante5/coste1/orden1|187|187|válido|
|practice_0177|mixed|E3|donante6/coste0/orden0|189|189|válido|
|practice_0177|mixed|E3|donante6/coste1/orden0|189|189|válido|
|practice_0177|mixed|E3|donante6/coste0/orden1|189|189|válido|
|practice_0177|mixed|E3|donante6/coste1/orden1|189|189|válido|
|practice_0177|mixed|E3|donante7/coste0/orden0|215|172|válido|
|practice_0177|mixed|E3|donante7/coste1/orden0|197|172|válido|
|practice_0177|mixed|E3|donante7/coste0/orden1|189|172|válido|
|practice_0177|mixed|E3|donante7/coste1/orden1|192|172|válido|
|practice_0177|mixed|E3|donante8/coste0/orden0|166|166|válido|
|practice_0177|mixed|E3|donante8/coste1/orden0|166|166|válido|
|practice_0177|mixed|E3|donante8/coste0/orden1|166|166|válido|
|practice_0177|mixed|E3|donante8/coste1/orden1|166|166|válido|
|practice_0177|mixed|E3|donante9/coste0/orden0|197|168|válido|
|practice_0177|mixed|E3|donante9/coste1/orden0|196|168|válido|
|practice_0177|mixed|E3|donante9/coste0/orden1|182|168|válido|
|practice_0177|mixed|E3|donante9/coste1/orden1|189|168|válido|
|practice_0178|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0178|multiplicative|Cruce|CP True/coste0/orden0|89|89|válido|
|practice_0178|multiplicative|Cruce|CP True/coste1/orden0|89|89|válido|
|practice_0178|multiplicative|Cruce|CP True/coste0/orden1|89|89|válido|
|practice_0178|multiplicative|Cruce|CP True/coste1/orden1|89|89|válido|
|practice_0178|multiplicative|Cruce|mejor independiente/coste0/orden0|89|89|válido|
|practice_0178|multiplicative|Cruce|mejor independiente/coste1/orden0|89|89|válido|
|practice_0178|multiplicative|Cruce|mejor independiente/coste0/orden1|89|89|válido|
|practice_0178|multiplicative|Cruce|mejor independiente/coste1/orden1|89|89|válido|
|practice_0178|multiplicative|E1|cola10/pol1/inicio0|101|40|válido|
|practice_0178|multiplicative|E1|cola10/pol1/inicio1|101|40|válido|
|practice_0178|multiplicative|E1|cola10/pol2/inicio0|101|40|válido|
|practice_0178|multiplicative|E1|cola10/pol2/inicio1|101|40|válido|
|practice_0178|multiplicative|E1|cola8/pol1/inicio0|96|29|válido|
|practice_0178|multiplicative|E1|cola8/pol1/inicio1|96|29|válido|
|practice_0178|multiplicative|E1|cola8/pol2/inicio0|96|29|válido|
|practice_0178|multiplicative|E1|cola8/pol2/inicio1|96|29|válido|
|practice_0178|multiplicative|E1|cola6/pol1/inicio0|95|22|válido|
|practice_0178|multiplicative|E1|cola6/pol1/inicio1|95|22|válido|
|practice_0178|multiplicative|E1|cola6/pol2/inicio0|92|19|válido|
|practice_0178|multiplicative|E1|cola6/pol2/inicio1|92|19|válido|
|practice_0178|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0178|multiplicative|E2|CP False/coste0/orden0|92|92|válido|
|practice_0178|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0178|multiplicative|E2|CP True/coste0/orden0|89|89|válido|
|practice_0178|multiplicative|E3|donante0/coste0/orden0|92|92|válido|
|practice_0178|multiplicative|E3|donante0/coste1/orden0|92|92|válido|
|practice_0178|multiplicative|E3|donante0/coste0/orden1|92|92|válido|
|practice_0178|multiplicative|E3|donante0/coste1/orden1|92|92|válido|
|practice_0178|multiplicative|E3|donante1/coste0/orden0|93|93|válido|
|practice_0178|multiplicative|E3|donante1/coste1/orden0|93|93|válido|
|practice_0178|multiplicative|E3|donante1/coste0/orden1|93|93|válido|
|practice_0178|multiplicative|E3|donante1/coste1/orden1|93|93|válido|
|practice_0178|multiplicative|E3|donante2/coste0/orden0|94|94|válido|
|practice_0178|multiplicative|E3|donante2/coste1/orden0|94|94|válido|
|practice_0178|multiplicative|E3|donante2/coste0/orden1|94|94|válido|
|practice_0178|multiplicative|E3|donante2/coste1/orden1|94|94|válido|
|practice_0178|multiplicative|E3|donante3/coste0/orden0|96|96|válido|
|practice_0178|multiplicative|E3|donante3/coste1/orden0|96|96|válido|
|practice_0178|multiplicative|E3|donante3/coste0/orden1|96|96|válido|
|practice_0178|multiplicative|E3|donante3/coste1/orden1|96|96|válido|
|practice_0178|multiplicative|E3|donante4/coste0/orden0|94|94|válido|
|practice_0178|multiplicative|E3|donante4/coste1/orden0|94|94|válido|
|practice_0178|multiplicative|E3|donante4/coste0/orden1|94|94|válido|
|practice_0178|multiplicative|E3|donante4/coste1/orden1|94|94|válido|
|practice_0178|multiplicative|E3|donante5/coste0/orden0|93|93|válido|
|practice_0178|multiplicative|E3|donante5/coste1/orden0|93|93|válido|
|practice_0178|multiplicative|E3|donante5/coste0/orden1|93|93|válido|
|practice_0178|multiplicative|E3|donante5/coste1/orden1|93|93|válido|
|practice_0178|multiplicative|E3|donante6/coste0/orden0|94|94|válido|
|practice_0178|multiplicative|E3|donante6/coste1/orden0|94|94|válido|
|practice_0178|multiplicative|E3|donante6/coste0/orden1|94|94|válido|
|practice_0178|multiplicative|E3|donante6/coste1/orden1|94|94|válido|
|practice_0178|multiplicative|E3|donante7/coste0/orden0|95|95|válido|
|practice_0178|multiplicative|E3|donante7/coste1/orden0|95|95|válido|
|practice_0178|multiplicative|E3|donante7/coste0/orden1|95|95|válido|
|practice_0178|multiplicative|E3|donante7/coste1/orden1|95|95|válido|
|practice_0178|multiplicative|E3|donante8/coste0/orden0|93|93|válido|
|practice_0178|multiplicative|E3|donante8/coste1/orden0|93|93|válido|
|practice_0178|multiplicative|E3|donante8/coste0/orden1|93|93|válido|
|practice_0178|multiplicative|E3|donante8/coste1/orden1|93|93|válido|
|practice_0178|multiplicative|E3|donante9/coste0/orden0|93|93|válido|
|practice_0178|multiplicative|E3|donante9/coste1/orden0|93|93|válido|
|practice_0178|multiplicative|E3|donante9/coste0/orden1|93|93|válido|
|practice_0178|multiplicative|E3|donante9/coste1/orden1|93|93|válido|
|practice_0179|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0179|independent|Cruce|CP True/coste0/orden0|424|293|válido|
|practice_0179|independent|Cruce|CP True/coste1/orden0|429|293|válido|
|practice_0179|independent|Cruce|CP True/coste0/orden1|405|293|válido|
|practice_0179|independent|Cruce|CP True/coste1/orden1|338|293|válido|
|practice_0179|independent|Cruce|mejor independiente/coste0/orden0|424|293|válido|
|practice_0179|independent|Cruce|mejor independiente/coste1/orden0|429|293|válido|
|practice_0179|independent|Cruce|mejor independiente/coste0/orden1|405|293|válido|
|practice_0179|independent|Cruce|mejor independiente/coste1/orden1|338|293|válido|
|practice_0179|independent|E1|cola10/pol1/inicio0|307|90|válido|
|practice_0179|independent|E1|cola10/pol1/inicio1|307|90|válido|
|practice_0179|independent|E1|cola10/pol2/inicio0|306|89|válido|
|practice_0179|independent|E1|cola10/pol2/inicio1|306|89|válido|
|practice_0179|independent|E1|cola8/pol1/inicio0|305|72|válido|
|practice_0179|independent|E1|cola8/pol1/inicio1|305|72|válido|
|practice_0179|independent|E1|cola8/pol2/inicio0|305|72|válido|
|practice_0179|independent|E1|cola8/pol2/inicio1|305|72|válido|
|practice_0179|independent|E1|cola6/pol1/inicio0|304|54|válido|
|practice_0179|independent|E1|cola6/pol1/inicio1|304|54|válido|
|practice_0179|independent|E1|cola6/pol2/inicio0|308|58|válido|
|practice_0179|independent|E1|cola6/pol2/inicio1|308|58|válido|
|practice_0179|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0179|independent|E2|CP False/coste0/orden0|529|345|válido|
|practice_0179|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0179|independent|E2|CP True/coste0/orden0|424|293|válido|
|practice_0179|independent|E3|donante0/coste0/orden0|319|293|válido|
|practice_0179|independent|E3|donante0/coste1/orden0|303|293|válido|
|practice_0179|independent|E3|donante0/coste0/orden1|405|293|válido|
|practice_0179|independent|E3|donante0/coste1/orden1|338|293|válido|
|practice_0179|independent|E3|donante1/coste0/orden0|342|342|válido|
|practice_0179|independent|E3|donante1/coste1/orden0|342|342|válido|
|practice_0179|independent|E3|donante1/coste0/orden1|342|342|válido|
|practice_0179|independent|E3|donante1/coste1/orden1|342|342|válido|
|practice_0179|independent|E3|donante2/coste0/orden0|570|320|válido|
|practice_0179|independent|E3|donante2/coste1/orden0|360|320|válido|
|practice_0179|independent|E3|donante2/coste0/orden1|324|320|válido|
|practice_0179|independent|E3|donante2/coste1/orden1|324|320|válido|
|practice_0179|independent|E3|donante3/coste0/orden0|512|315|válido|
|practice_0179|independent|E3|donante3/coste1/orden0|419|315|válido|
|practice_0179|independent|E3|donante3/coste0/orden1|317|315|válido|
|practice_0179|independent|E3|donante3/coste1/orden1|317|315|válido|
|practice_0179|independent|E3|donante4/coste0/orden0|358|358|válido|
|practice_0179|independent|E3|donante4/coste1/orden0|358|358|válido|
|practice_0179|independent|E3|donante4/coste0/orden1|358|358|válido|
|practice_0179|independent|E3|donante4/coste1/orden1|358|358|válido|
|practice_0179|independent|E3|donante5/coste0/orden0|346|346|válido|
|practice_0179|independent|E3|donante5/coste1/orden0|346|346|válido|
|practice_0179|independent|E3|donante5/coste0/orden1|346|346|válido|
|practice_0179|independent|E3|donante5/coste1/orden1|346|346|válido|
|practice_0179|independent|E3|donante6/coste0/orden0|345|345|válido|
|practice_0179|independent|E3|donante6/coste1/orden0|345|345|válido|
|practice_0179|independent|E3|donante6/coste0/orden1|345|345|válido|
|practice_0179|independent|E3|donante6/coste1/orden1|345|345|válido|
|practice_0179|independent|E3|donante7/coste0/orden0|339|308|válido|
|practice_0179|independent|E3|donante7/coste1/orden0|365|308|válido|
|practice_0179|independent|E3|donante7/coste0/orden1|356|308|válido|
|practice_0179|independent|E3|donante7/coste1/orden1|356|308|válido|
|practice_0179|independent|E3|donante8/coste0/orden0|326|290|válido|
|practice_0179|independent|E3|donante8/coste1/orden0|307|290|válido|
|practice_0179|independent|E3|donante8/coste0/orden1|320|290|válido|
|practice_0179|independent|E3|donante8/coste1/orden1|312|290|válido|
|practice_0179|independent|E3|donante9/coste0/orden0|319|293|válido|
|practice_0179|independent|E3|donante9/coste1/orden0|303|293|válido|
|practice_0179|independent|E3|donante9/coste0/orden1|405|293|válido|
|practice_0179|independent|E3|donante9/coste1/orden1|338|293|válido|
|practice_0180|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0180|multiplicative|Cruce|CP True/coste0/orden0|139|138|válido|
|practice_0180|multiplicative|Cruce|CP True/coste1/orden0|139|138|válido|
|practice_0180|multiplicative|Cruce|CP True/coste0/orden1|138|138|válido|
|practice_0180|multiplicative|Cruce|CP True/coste1/orden1|138|138|válido|
|practice_0180|multiplicative|Cruce|mejor independiente/coste0/orden0|150|143|válido|
|practice_0180|multiplicative|Cruce|mejor independiente/coste1/orden0|149|143|válido|
|practice_0180|multiplicative|Cruce|mejor independiente/coste0/orden1|143|143|válido|
|practice_0180|multiplicative|Cruce|mejor independiente/coste1/orden1|143|143|válido|
|practice_0180|multiplicative|E1|cola10/pol1/inicio0|148|50|válido|
|practice_0180|multiplicative|E1|cola10/pol1/inicio1|148|50|válido|
|practice_0180|multiplicative|E1|cola10/pol2/inicio0|148|50|válido|
|practice_0180|multiplicative|E1|cola10/pol2/inicio1|148|50|válido|
|practice_0180|multiplicative|E1|cola8/pol1/inicio0|148|39|válido|
|practice_0180|multiplicative|E1|cola8/pol1/inicio1|148|39|válido|
|practice_0180|multiplicative|E1|cola8/pol2/inicio0|148|39|válido|
|practice_0180|multiplicative|E1|cola8/pol2/inicio1|148|39|válido|
|practice_0180|multiplicative|E1|cola6/pol1/inicio0|148|31|válido|
|practice_0180|multiplicative|E1|cola6/pol1/inicio1|148|31|válido|
|practice_0180|multiplicative|E1|cola6/pol2/inicio0|148|31|válido|
|practice_0180|multiplicative|E1|cola6/pol2/inicio1|148|31|válido|
|practice_0180|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0180|multiplicative|E2|CP False/coste0/orden0|153|153|válido|
|practice_0180|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0180|multiplicative|E2|CP True/coste0/orden0|145|145|válido|
|practice_0180|multiplicative|E3|donante0/coste0/orden0|147|147|válido|
|practice_0180|multiplicative|E3|donante0/coste1/orden0|147|147|válido|
|practice_0180|multiplicative|E3|donante0/coste0/orden1|147|147|válido|
|practice_0180|multiplicative|E3|donante0/coste1/orden1|147|147|válido|
|practice_0180|multiplicative|E3|donante1/coste0/orden0|155|155|válido|
|practice_0180|multiplicative|E3|donante1/coste1/orden0|155|155|válido|
|practice_0180|multiplicative|E3|donante1/coste0/orden1|155|155|válido|
|practice_0180|multiplicative|E3|donante1/coste1/orden1|155|155|válido|
|practice_0180|multiplicative|E3|donante2/coste0/orden0|151|151|válido|
|practice_0180|multiplicative|E3|donante2/coste1/orden0|151|151|válido|
|practice_0180|multiplicative|E3|donante2/coste0/orden1|151|151|válido|
|practice_0180|multiplicative|E3|donante2/coste1/orden1|151|151|válido|
|practice_0180|multiplicative|E3|donante3/coste0/orden0|148|148|válido|
|practice_0180|multiplicative|E3|donante3/coste1/orden0|148|148|válido|
|practice_0180|multiplicative|E3|donante3/coste0/orden1|148|148|válido|
|practice_0180|multiplicative|E3|donante3/coste1/orden1|148|148|válido|
|practice_0180|multiplicative|E3|donante4/coste0/orden0|160|160|válido|
|practice_0180|multiplicative|E3|donante4/coste1/orden0|160|160|válido|
|practice_0180|multiplicative|E3|donante4/coste0/orden1|160|160|válido|
|practice_0180|multiplicative|E3|donante4/coste1/orden1|160|160|válido|
|practice_0180|multiplicative|E3|donante5/coste0/orden0|156|156|válido|
|practice_0180|multiplicative|E3|donante5/coste1/orden0|156|156|válido|
|practice_0180|multiplicative|E3|donante5/coste0/orden1|156|156|válido|
|practice_0180|multiplicative|E3|donante5/coste1/orden1|156|156|válido|
|practice_0180|multiplicative|E3|donante6/coste0/orden0|155|155|válido|
|practice_0180|multiplicative|E3|donante6/coste1/orden0|155|155|válido|
|practice_0180|multiplicative|E3|donante6/coste0/orden1|155|155|válido|
|practice_0180|multiplicative|E3|donante6/coste1/orden1|155|155|válido|
|practice_0180|multiplicative|E3|donante7/coste0/orden0|185|145|válido|
|practice_0180|multiplicative|E3|donante7/coste1/orden0|158|145|válido|
|practice_0180|multiplicative|E3|donante7/coste0/orden1|159|145|válido|
|practice_0180|multiplicative|E3|donante7/coste1/orden1|156|145|válido|
|practice_0180|multiplicative|E3|donante8/coste0/orden0|185|146|válido|
|practice_0180|multiplicative|E3|donante8/coste1/orden0|158|146|válido|
|practice_0180|multiplicative|E3|donante8/coste0/orden1|169|146|válido|
|practice_0180|multiplicative|E3|donante8/coste1/orden1|164|146|válido|
|practice_0180|multiplicative|E3|donante9/coste0/orden0|169|143|válido|
|practice_0180|multiplicative|E3|donante9/coste1/orden0|154|143|válido|
|practice_0180|multiplicative|E3|donante9/coste0/orden1|143|143|válido|
|practice_0180|multiplicative|E3|donante9/coste1/orden1|143|143|válido|
|practice_0181|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0181|independent|Cruce|CP True/coste0/orden0|304|237|válido|
|practice_0181|independent|Cruce|CP True/coste1/orden0|294|237|válido|
|practice_0181|independent|Cruce|CP True/coste0/orden1|280|237|válido|
|practice_0181|independent|Cruce|CP True/coste1/orden1|280|237|válido|
|practice_0181|independent|Cruce|mejor independiente/coste0/orden0|304|239|válido|
|practice_0181|independent|Cruce|mejor independiente/coste1/orden0|292|239|válido|
|practice_0181|independent|Cruce|mejor independiente/coste0/orden1|240|239|válido|
|practice_0181|independent|Cruce|mejor independiente/coste1/orden1|240|239|válido|
|practice_0181|independent|E1|cola10/pol1/inicio0|253|77|válido|
|practice_0181|independent|E1|cola10/pol1/inicio1|253|77|válido|
|practice_0181|independent|E1|cola10/pol2/inicio0|249|73|válido|
|practice_0181|independent|E1|cola10/pol2/inicio1|249|73|válido|
|practice_0181|independent|E1|cola8/pol1/inicio0|249|59|válido|
|practice_0181|independent|E1|cola8/pol1/inicio1|249|59|válido|
|practice_0181|independent|E1|cola8/pol2/inicio0|241|51|válido|
|practice_0181|independent|E1|cola8/pol2/inicio1|241|51|válido|
|practice_0181|independent|E1|cola6/pol1/inicio0|247|43|válido|
|practice_0181|independent|E1|cola6/pol1/inicio1|247|43|válido|
|practice_0181|independent|E1|cola6/pol2/inicio0|247|43|válido|
|practice_0181|independent|E1|cola6/pol2/inicio1|247|43|válido|
|practice_0181|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0181|independent|E2|CP False/coste0/orden0|415|298|válido|
|practice_0181|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0181|independent|E2|CP True/coste0/orden0|304|239|válido|
|practice_0181|independent|E3|donante0/coste0/orden0|257|239|válido|
|practice_0181|independent|E3|donante0/coste1/orden0|257|239|válido|
|practice_0181|independent|E3|donante0/coste0/orden1|240|239|válido|
|practice_0181|independent|E3|donante0/coste1/orden1|240|239|válido|
|practice_0181|independent|E3|donante1/coste0/orden0|281|281|válido|
|practice_0181|independent|E3|donante1/coste1/orden0|281|281|válido|
|practice_0181|independent|E3|donante1/coste0/orden1|281|281|válido|
|practice_0181|independent|E3|donante1/coste1/orden1|281|281|válido|
|practice_0181|independent|E3|donante2/coste0/orden0|376|264|válido|
|practice_0181|independent|E3|donante2/coste1/orden0|332|264|válido|
|practice_0181|independent|E3|donante2/coste0/orden1|264|264|válido|
|practice_0181|independent|E3|donante2/coste1/orden1|264|264|válido|
|practice_0181|independent|E3|donante3/coste0/orden0|399|255|válido|
|practice_0181|independent|E3|donante3/coste1/orden0|322|255|válido|
|practice_0181|independent|E3|donante3/coste0/orden1|255|255|válido|
|practice_0181|independent|E3|donante3/coste1/orden1|255|255|válido|
|practice_0181|independent|E3|donante4/coste0/orden0|293|293|válido|
|practice_0181|independent|E3|donante4/coste1/orden0|293|293|válido|
|practice_0181|independent|E3|donante4/coste0/orden1|293|293|válido|
|practice_0181|independent|E3|donante4/coste1/orden1|293|293|válido|
|practice_0181|independent|E3|donante5/coste0/orden0|285|285|válido|
|practice_0181|independent|E3|donante5/coste1/orden0|285|285|válido|
|practice_0181|independent|E3|donante5/coste0/orden1|285|285|válido|
|practice_0181|independent|E3|donante5/coste1/orden1|285|285|válido|
|practice_0181|independent|E3|donante6/coste0/orden0|287|287|válido|
|practice_0181|independent|E3|donante6/coste1/orden0|287|287|válido|
|practice_0181|independent|E3|donante6/coste0/orden1|287|287|válido|
|practice_0181|independent|E3|donante6/coste1/orden1|287|287|válido|
|practice_0181|independent|E3|donante7/coste0/orden0|260|260|válido|
|practice_0181|independent|E3|donante7/coste1/orden0|260|260|válido|
|practice_0181|independent|E3|donante7/coste0/orden1|299|260|válido|
|practice_0181|independent|E3|donante7/coste1/orden1|269|260|válido|
|practice_0181|independent|E3|donante8/coste0/orden0|264|246|válido|
|practice_0181|independent|E3|donante8/coste1/orden0|248|246|válido|
|practice_0181|independent|E3|donante8/coste0/orden1|246|246|válido|
|practice_0181|independent|E3|donante8/coste1/orden1|246|246|válido|
|practice_0181|independent|E3|donante9/coste0/orden0|257|239|válido|
|practice_0181|independent|E3|donante9/coste1/orden0|257|239|válido|
|practice_0181|independent|E3|donante9/coste0/orden1|240|239|válido|
|practice_0181|independent|E3|donante9/coste1/orden1|240|239|válido|
|practice_0182|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0182|mixed|Cruce|CP True/coste0/orden0|227|196|válido|
|practice_0182|mixed|Cruce|CP True/coste1/orden0|217|196|válido|
|practice_0182|mixed|Cruce|CP True/coste0/orden1|222|196|válido|
|practice_0182|mixed|Cruce|CP True/coste1/orden1|228|196|válido|
|practice_0182|mixed|Cruce|mejor independiente/coste0/orden0|219|198|válido|
|practice_0182|mixed|Cruce|mejor independiente/coste1/orden0|216|198|válido|
|practice_0182|mixed|Cruce|mejor independiente/coste0/orden1|235|198|válido|
|practice_0182|mixed|Cruce|mejor independiente/coste1/orden1|221|198|válido|
|practice_0182|mixed|E1|cola10/pol1/inicio0|205|74|válido|
|practice_0182|mixed|E1|cola10/pol1/inicio1|205|74|válido|
|practice_0182|mixed|E1|cola10/pol2/inicio0|207|76|válido|
|practice_0182|mixed|E1|cola10/pol2/inicio1|207|76|válido|
|practice_0182|mixed|E1|cola8/pol1/inicio0|204|59|válido|
|practice_0182|mixed|E1|cola8/pol1/inicio1|204|59|válido|
|practice_0182|mixed|E1|cola8/pol2/inicio0|204|59|válido|
|practice_0182|mixed|E1|cola8/pol2/inicio1|204|59|válido|
|practice_0182|mixed|E1|cola6/pol1/inicio0|207|48|válido|
|practice_0182|mixed|E1|cola6/pol1/inicio1|207|48|válido|
|practice_0182|mixed|E1|cola6/pol2/inicio0|208|49|válido|
|practice_0182|mixed|E1|cola6/pol2/inicio1|208|49|válido|
|practice_0182|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0182|mixed|E2|CP False/coste0/orden0|219|198|válido|
|practice_0182|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0182|mixed|E2|CP True/coste0/orden0|227|196|válido|
|practice_0182|mixed|E3|donante0/coste0/orden0|198|198|válido|
|practice_0182|mixed|E3|donante0/coste1/orden0|198|198|válido|
|practice_0182|mixed|E3|donante0/coste0/orden1|235|198|válido|
|practice_0182|mixed|E3|donante0/coste1/orden1|221|198|válido|
|practice_0182|mixed|E3|donante1/coste0/orden0|220|220|válido|
|practice_0182|mixed|E3|donante1/coste1/orden0|220|220|válido|
|practice_0182|mixed|E3|donante1/coste0/orden1|220|220|válido|
|practice_0182|mixed|E3|donante1/coste1/orden1|220|220|válido|
|practice_0182|mixed|E3|donante2/coste0/orden0|212|212|válido|
|practice_0182|mixed|E3|donante2/coste1/orden0|212|212|válido|
|practice_0182|mixed|E3|donante2/coste0/orden1|212|212|válido|
|practice_0182|mixed|E3|donante2/coste1/orden1|212|212|válido|
|practice_0182|mixed|E3|donante3/coste0/orden0|211|211|válido|
|practice_0182|mixed|E3|donante3/coste1/orden0|211|211|válido|
|practice_0182|mixed|E3|donante3/coste0/orden1|211|211|válido|
|practice_0182|mixed|E3|donante3/coste1/orden1|211|211|válido|
|practice_0182|mixed|E3|donante4/coste0/orden0|222|222|válido|
|practice_0182|mixed|E3|donante4/coste1/orden0|222|222|válido|
|practice_0182|mixed|E3|donante4/coste0/orden1|222|222|válido|
|practice_0182|mixed|E3|donante4/coste1/orden1|222|222|válido|
|practice_0182|mixed|E3|donante5/coste0/orden0|227|227|válido|
|practice_0182|mixed|E3|donante5/coste1/orden0|227|227|válido|
|practice_0182|mixed|E3|donante5/coste0/orden1|227|227|válido|
|practice_0182|mixed|E3|donante5/coste1/orden1|227|227|válido|
|practice_0182|mixed|E3|donante6/coste0/orden0|223|223|válido|
|practice_0182|mixed|E3|donante6/coste1/orden0|223|223|válido|
|practice_0182|mixed|E3|donante6/coste0/orden1|223|223|válido|
|practice_0182|mixed|E3|donante6/coste1/orden1|223|223|válido|
|practice_0182|mixed|E3|donante7/coste0/orden0|211|204|válido|
|practice_0182|mixed|E3|donante7/coste1/orden0|211|204|válido|
|practice_0182|mixed|E3|donante7/coste0/orden1|226|204|válido|
|practice_0182|mixed|E3|donante7/coste1/orden1|206|204|válido|
|practice_0182|mixed|E3|donante8/coste0/orden0|200|200|válido|
|practice_0182|mixed|E3|donante8/coste1/orden0|200|200|válido|
|practice_0182|mixed|E3|donante8/coste0/orden1|236|200|válido|
|practice_0182|mixed|E3|donante8/coste1/orden1|223|200|válido|
|practice_0182|mixed|E3|donante9/coste0/orden0|208|194|válido|
|practice_0182|mixed|E3|donante9/coste1/orden0|219|194|válido|
|practice_0182|mixed|E3|donante9/coste0/orden1|238|194|válido|
|practice_0182|mixed|E3|donante9/coste1/orden1|210|194|válido|
|practice_0183|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0183|mixed|Cruce|CP True/coste0/orden0|249|242|válido|
|practice_0183|mixed|Cruce|CP True/coste1/orden0|266|242|válido|
|practice_0183|mixed|Cruce|CP True/coste0/orden1|265|242|válido|
|practice_0183|mixed|Cruce|CP True/coste1/orden1|262|242|válido|
|practice_0183|mixed|Cruce|mejor independiente/coste0/orden0|249|242|válido|
|practice_0183|mixed|Cruce|mejor independiente/coste1/orden0|266|242|válido|
|practice_0183|mixed|Cruce|mejor independiente/coste0/orden1|265|242|válido|
|practice_0183|mixed|Cruce|mejor independiente/coste1/orden1|262|242|válido|
|practice_0183|mixed|E1|cola10/pol1/inicio0|260|91|válido|
|practice_0183|mixed|E1|cola10/pol1/inicio1|260|91|válido|
|practice_0183|mixed|E1|cola10/pol2/inicio0|257|88|válido|
|practice_0183|mixed|E1|cola10/pol2/inicio1|257|88|válido|
|practice_0183|mixed|E1|cola8/pol1/inicio0|259|70|válido|
|practice_0183|mixed|E1|cola8/pol1/inicio1|259|70|válido|
|practice_0183|mixed|E1|cola8/pol2/inicio0|260|71|válido|
|practice_0183|mixed|E1|cola8/pol2/inicio1|260|71|válido|
|practice_0183|mixed|E1|cola6/pol1/inicio0|261|55|válido|
|practice_0183|mixed|E1|cola6/pol1/inicio1|261|55|válido|
|practice_0183|mixed|E1|cola6/pol2/inicio0|261|55|válido|
|practice_0183|mixed|E1|cola6/pol2/inicio1|261|55|válido|
|practice_0183|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0183|mixed|E2|CP False/coste0/orden0|266|244|válido|
|practice_0183|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0183|mixed|E2|CP True/coste0/orden0|249|242|válido|
|practice_0183|mixed|E3|donante0/coste0/orden0|272|244|válido|
|practice_0183|mixed|E3|donante0/coste1/orden0|296|244|válido|
|practice_0183|mixed|E3|donante0/coste0/orden1|277|244|válido|
|practice_0183|mixed|E3|donante0/coste1/orden1|275|244|válido|
|practice_0183|mixed|E3|donante1/coste0/orden0|287|287|válido|
|practice_0183|mixed|E3|donante1/coste1/orden0|287|287|válido|
|practice_0183|mixed|E3|donante1/coste0/orden1|287|287|válido|
|practice_0183|mixed|E3|donante1/coste1/orden1|287|287|válido|
|practice_0183|mixed|E3|donante2/coste0/orden0|353|276|válido|
|practice_0183|mixed|E3|donante2/coste1/orden0|287|276|válido|
|practice_0183|mixed|E3|donante2/coste0/orden1|276|276|válido|
|practice_0183|mixed|E3|donante2/coste1/orden1|276|276|válido|
|practice_0183|mixed|E3|donante3/coste0/orden0|310|268|válido|
|practice_0183|mixed|E3|donante3/coste1/orden0|285|268|válido|
|practice_0183|mixed|E3|donante3/coste0/orden1|268|268|válido|
|practice_0183|mixed|E3|donante3/coste1/orden1|268|268|válido|
|practice_0183|mixed|E3|donante4/coste0/orden0|290|290|válido|
|practice_0183|mixed|E3|donante4/coste1/orden0|290|290|válido|
|practice_0183|mixed|E3|donante4/coste0/orden1|290|290|válido|
|practice_0183|mixed|E3|donante4/coste1/orden1|290|290|válido|
|practice_0183|mixed|E3|donante5/coste0/orden0|283|283|válido|
|practice_0183|mixed|E3|donante5/coste1/orden0|283|283|válido|
|practice_0183|mixed|E3|donante5/coste0/orden1|283|283|válido|
|practice_0183|mixed|E3|donante5/coste1/orden1|283|283|válido|
|practice_0183|mixed|E3|donante6/coste0/orden0|286|286|válido|
|practice_0183|mixed|E3|donante6/coste1/orden0|286|286|válido|
|practice_0183|mixed|E3|donante6/coste0/orden1|286|286|válido|
|practice_0183|mixed|E3|donante6/coste1/orden1|286|286|válido|
|practice_0183|mixed|E3|donante7/coste0/orden0|294|252|válido|
|practice_0183|mixed|E3|donante7/coste1/orden0|284|252|válido|
|practice_0183|mixed|E3|donante7/coste0/orden1|312|252|válido|
|practice_0183|mixed|E3|donante7/coste1/orden1|279|252|válido|
|practice_0183|mixed|E3|donante8/coste0/orden0|272|244|válido|
|practice_0183|mixed|E3|donante8/coste1/orden0|296|244|válido|
|practice_0183|mixed|E3|donante8/coste0/orden1|277|244|válido|
|practice_0183|mixed|E3|donante8/coste1/orden1|275|244|válido|
|practice_0183|mixed|E3|donante9/coste0/orden0|288|247|válido|
|practice_0183|mixed|E3|donante9/coste1/orden0|269|247|válido|
|practice_0183|mixed|E3|donante9/coste0/orden1|267|247|válido|
|practice_0183|mixed|E3|donante9/coste1/orden1|267|247|válido|
|practice_0184|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0184|mixed|Cruce|CP True/coste0/orden0|227|218|válido|
|practice_0184|mixed|Cruce|CP True/coste1/orden0|227|218|válido|
|practice_0184|mixed|Cruce|CP True/coste0/orden1|218|218|válido|
|practice_0184|mixed|Cruce|CP True/coste1/orden1|218|218|válido|
|practice_0184|mixed|Cruce|mejor independiente/coste0/orden0|223|219|válido|
|practice_0184|mixed|Cruce|mejor independiente/coste1/orden0|223|219|válido|
|practice_0184|mixed|Cruce|mejor independiente/coste0/orden1|219|219|válido|
|practice_0184|mixed|Cruce|mejor independiente/coste1/orden1|219|219|válido|
|practice_0184|mixed|E1|cola10/pol1/inicio0|232|74|válido|
|practice_0184|mixed|E1|cola10/pol1/inicio1|232|74|válido|
|practice_0184|mixed|E1|cola10/pol2/inicio0|233|75|válido|
|practice_0184|mixed|E1|cola10/pol2/inicio1|233|75|válido|
|practice_0184|mixed|E1|cola8/pol1/inicio0|223|51|válido|
|practice_0184|mixed|E1|cola8/pol1/inicio1|223|51|válido|
|practice_0184|mixed|E1|cola8/pol2/inicio0|223|51|válido|
|practice_0184|mixed|E1|cola8/pol2/inicio1|223|51|válido|
|practice_0184|mixed|E1|cola6/pol1/inicio0|228|42|válido|
|practice_0184|mixed|E1|cola6/pol1/inicio1|228|42|válido|
|practice_0184|mixed|E1|cola6/pol2/inicio0|234|48|válido|
|practice_0184|mixed|E1|cola6/pol2/inicio1|234|48|válido|
|practice_0184|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0184|mixed|E2|CP False/coste0/orden0|318|249|válido|
|practice_0184|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0184|mixed|E2|CP True/coste0/orden0|227|218|válido|
|practice_0184|mixed|E3|donante0/coste0/orden0|222|222|válido|
|practice_0184|mixed|E3|donante0/coste1/orden0|222|222|válido|
|practice_0184|mixed|E3|donante0/coste0/orden1|222|222|válido|
|practice_0184|mixed|E3|donante0/coste1/orden1|222|222|válido|
|practice_0184|mixed|E3|donante1/coste0/orden0|256|256|válido|
|practice_0184|mixed|E3|donante1/coste1/orden0|256|256|válido|
|practice_0184|mixed|E3|donante1/coste0/orden1|256|256|válido|
|practice_0184|mixed|E3|donante1/coste1/orden1|256|256|válido|
|practice_0184|mixed|E3|donante2/coste0/orden0|291|238|válido|
|practice_0184|mixed|E3|donante2/coste1/orden0|272|238|válido|
|practice_0184|mixed|E3|donante2/coste0/orden1|238|238|válido|
|practice_0184|mixed|E3|donante2/coste1/orden1|238|238|válido|
|practice_0184|mixed|E3|donante3/coste0/orden0|275|240|válido|
|practice_0184|mixed|E3|donante3/coste1/orden0|258|240|válido|
|practice_0184|mixed|E3|donante3/coste0/orden1|240|240|válido|
|practice_0184|mixed|E3|donante3/coste1/orden1|240|240|válido|
|practice_0184|mixed|E3|donante4/coste0/orden0|249|249|válido|
|practice_0184|mixed|E3|donante4/coste1/orden0|249|249|válido|
|practice_0184|mixed|E3|donante4/coste0/orden1|249|249|válido|
|practice_0184|mixed|E3|donante4/coste1/orden1|249|249|válido|
|practice_0184|mixed|E3|donante5/coste0/orden0|249|249|válido|
|practice_0184|mixed|E3|donante5/coste1/orden0|249|249|válido|
|practice_0184|mixed|E3|donante5/coste0/orden1|249|249|válido|
|practice_0184|mixed|E3|donante5/coste1/orden1|249|249|válido|
|practice_0184|mixed|E3|donante6/coste0/orden0|250|250|válido|
|practice_0184|mixed|E3|donante6/coste1/orden0|250|250|válido|
|practice_0184|mixed|E3|donante6/coste0/orden1|250|250|válido|
|practice_0184|mixed|E3|donante6/coste1/orden1|250|250|válido|
|practice_0184|mixed|E3|donante7/coste0/orden0|225|219|válido|
|practice_0184|mixed|E3|donante7/coste1/orden0|240|219|válido|
|practice_0184|mixed|E3|donante7/coste0/orden1|243|219|válido|
|practice_0184|mixed|E3|donante7/coste1/orden1|232|219|válido|
|practice_0184|mixed|E3|donante8/coste0/orden0|230|222|válido|
|practice_0184|mixed|E3|donante8/coste1/orden0|230|222|válido|
|practice_0184|mixed|E3|donante8/coste0/orden1|225|222|válido|
|practice_0184|mixed|E3|donante8/coste1/orden1|227|222|válido|
|practice_0185|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0185|shared|Cruce|CP True/coste0/orden0|160|160|válido|
|practice_0185|shared|Cruce|CP True/coste1/orden0|160|160|válido|
|practice_0185|shared|Cruce|CP True/coste0/orden1|160|160|válido|
|practice_0185|shared|Cruce|CP True/coste1/orden1|160|160|válido|
|practice_0185|shared|Cruce|mejor independiente/coste0/orden0|161|161|válido|
|practice_0185|shared|Cruce|mejor independiente/coste1/orden0|161|161|válido|
|practice_0185|shared|Cruce|mejor independiente/coste0/orden1|161|161|válido|
|practice_0185|shared|Cruce|mejor independiente/coste1/orden1|161|161|válido|
|practice_0185|shared|E1|cola10/pol1/inicio0|169|57|válido|
|practice_0185|shared|E1|cola10/pol1/inicio1|169|57|válido|
|practice_0185|shared|E1|cola10/pol2/inicio0|167|55|válido|
|practice_0185|shared|E1|cola10/pol2/inicio1|167|55|válido|
|practice_0185|shared|E1|cola8/pol1/inicio0|170|47|válido|
|practice_0185|shared|E1|cola8/pol1/inicio1|170|47|válido|
|practice_0185|shared|E1|cola8/pol2/inicio0|164|41|válido|
|practice_0185|shared|E1|cola8/pol2/inicio1|164|41|válido|
|practice_0185|shared|E1|cola6/pol1/inicio0|164|32|válido|
|practice_0185|shared|E1|cola6/pol1/inicio1|164|32|válido|
|practice_0185|shared|E1|cola6/pol2/inicio0|164|32|válido|
|practice_0185|shared|E1|cola6/pol2/inicio1|164|32|válido|
|practice_0185|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0185|shared|E2|CP False/coste0/orden0|179|179|válido|
|practice_0185|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0185|shared|E2|CP True/coste0/orden0|161|161|válido|
|practice_0185|shared|E3|donante0/coste0/orden0|164|164|válido|
|practice_0185|shared|E3|donante0/coste1/orden0|164|164|válido|
|practice_0185|shared|E3|donante0/coste0/orden1|164|164|válido|
|practice_0185|shared|E3|donante0/coste1/orden1|164|164|válido|
|practice_0185|shared|E3|donante1/coste0/orden0|177|177|válido|
|practice_0185|shared|E3|donante1/coste1/orden0|177|177|válido|
|practice_0185|shared|E3|donante1/coste0/orden1|177|177|válido|
|practice_0185|shared|E3|donante1/coste1/orden1|177|177|válido|
|practice_0185|shared|E3|donante2/coste0/orden0|171|171|válido|
|practice_0185|shared|E3|donante2/coste1/orden0|171|171|válido|
|practice_0185|shared|E3|donante2/coste0/orden1|171|171|válido|
|practice_0185|shared|E3|donante2/coste1/orden1|171|171|válido|
|practice_0185|shared|E3|donante3/coste0/orden0|168|168|válido|
|practice_0185|shared|E3|donante3/coste1/orden0|168|168|válido|
|practice_0185|shared|E3|donante3/coste0/orden1|168|168|válido|
|practice_0185|shared|E3|donante3/coste1/orden1|168|168|válido|
|practice_0185|shared|E3|donante4/coste0/orden0|189|189|válido|
|practice_0185|shared|E3|donante4/coste1/orden0|189|189|válido|
|practice_0185|shared|E3|donante4/coste0/orden1|189|189|válido|
|practice_0185|shared|E3|donante4/coste1/orden1|189|189|válido|
|practice_0185|shared|E3|donante5/coste0/orden0|181|181|válido|
|practice_0185|shared|E3|donante5/coste1/orden0|181|181|válido|
|practice_0185|shared|E3|donante5/coste0/orden1|181|181|válido|
|practice_0185|shared|E3|donante5/coste1/orden1|181|181|válido|
|practice_0185|shared|E3|donante6/coste0/orden0|179|179|válido|
|practice_0185|shared|E3|donante6/coste1/orden0|179|179|válido|
|practice_0185|shared|E3|donante6/coste0/orden1|179|179|válido|
|practice_0185|shared|E3|donante6/coste1/orden1|179|179|válido|
|practice_0185|shared|E3|donante7/coste0/orden0|238|167|válido|
|practice_0185|shared|E3|donante7/coste1/orden0|207|167|válido|
|practice_0185|shared|E3|donante7/coste0/orden1|257|167|válido|
|practice_0185|shared|E3|donante7/coste1/orden1|233|167|válido|
|practice_0185|shared|E3|donante8/coste0/orden0|178|163|válido|
|practice_0185|shared|E3|donante8/coste1/orden0|167|163|válido|
|practice_0185|shared|E3|donante8/coste0/orden1|186|163|válido|
|practice_0185|shared|E3|donante8/coste1/orden1|166|163|válido|
|practice_0186|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0186|independent|Cruce|CP True/coste0/orden0|101|101|válido|
|practice_0186|independent|Cruce|CP True/coste1/orden0|101|101|válido|
|practice_0186|independent|Cruce|CP True/coste0/orden1|101|101|válido|
|practice_0186|independent|Cruce|CP True/coste1/orden1|101|101|válido|
|practice_0186|independent|Cruce|mejor independiente/coste0/orden0|104|104|válido|
|practice_0186|independent|Cruce|mejor independiente/coste1/orden0|104|104|válido|
|practice_0186|independent|Cruce|mejor independiente/coste0/orden1|104|104|válido|
|practice_0186|independent|Cruce|mejor independiente/coste1/orden1|104|104|válido|
|practice_0186|independent|E1|cola10/pol1/inicio0|109|57|válido|
|practice_0186|independent|E1|cola10/pol1/inicio1|108|56|válido|
|practice_0186|independent|E1|cola10/pol2/inicio0|107|55|válido|
|practice_0186|independent|E1|cola10/pol2/inicio1|107|55|válido|
|practice_0186|independent|E1|cola8/pol1/inicio0|110|46|válido|
|practice_0186|independent|E1|cola8/pol1/inicio1|110|46|válido|
|practice_0186|independent|E1|cola8/pol2/inicio0|108|44|válido|
|practice_0186|independent|E1|cola8/pol2/inicio1|108|44|válido|
|practice_0186|independent|E1|cola6/pol1/inicio0|107|31|válido|
|practice_0186|independent|E1|cola6/pol1/inicio1|107|31|válido|
|practice_0186|independent|E1|cola6/pol2/inicio0|107|31|válido|
|practice_0186|independent|E1|cola6/pol2/inicio1|107|31|válido|
|practice_0186|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0186|independent|E2|CP False/coste0/orden0|106|106|válido|
|practice_0186|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0186|independent|E2|CP True/coste0/orden0|104|104|válido|
|practice_0186|independent|E3|donante0/coste0/orden0|106|106|válido|
|practice_0186|independent|E3|donante0/coste1/orden0|106|106|válido|
|practice_0186|independent|E3|donante0/coste0/orden1|106|106|válido|
|practice_0186|independent|E3|donante0/coste1/orden1|106|106|válido|
|practice_0186|independent|E3|donante1/coste0/orden0|118|118|válido|
|practice_0186|independent|E3|donante1/coste1/orden0|118|118|válido|
|practice_0186|independent|E3|donante1/coste0/orden1|118|118|válido|
|practice_0186|independent|E3|donante1/coste1/orden1|118|118|válido|
|practice_0186|independent|E3|donante2/coste0/orden0|118|118|válido|
|practice_0186|independent|E3|donante2/coste1/orden0|118|118|válido|
|practice_0186|independent|E3|donante2/coste0/orden1|118|118|válido|
|practice_0186|independent|E3|donante2/coste1/orden1|118|118|válido|
|practice_0186|independent|E3|donante3/coste0/orden0|111|111|válido|
|practice_0186|independent|E3|donante3/coste1/orden0|111|111|válido|
|practice_0186|independent|E3|donante3/coste0/orden1|111|111|válido|
|practice_0186|independent|E3|donante3/coste1/orden1|111|111|válido|
|practice_0186|independent|E3|donante4/coste0/orden0|118|118|válido|
|practice_0186|independent|E3|donante4/coste1/orden0|118|118|válido|
|practice_0186|independent|E3|donante4/coste0/orden1|118|118|válido|
|practice_0186|independent|E3|donante4/coste1/orden1|118|118|válido|
|practice_0186|independent|E3|donante5/coste0/orden0|109|109|válido|
|practice_0186|independent|E3|donante5/coste1/orden0|109|109|válido|
|practice_0186|independent|E3|donante5/coste0/orden1|109|109|válido|
|practice_0186|independent|E3|donante5/coste1/orden1|109|109|válido|
|practice_0186|independent|E3|donante6/coste0/orden0|111|111|válido|
|practice_0186|independent|E3|donante6/coste1/orden0|111|111|válido|
|practice_0186|independent|E3|donante6/coste0/orden1|111|111|válido|
|practice_0186|independent|E3|donante6/coste1/orden1|111|111|válido|
|practice_0186|independent|E3|donante7/coste0/orden0|112|112|válido|
|practice_0186|independent|E3|donante7/coste1/orden0|112|112|válido|
|practice_0186|independent|E3|donante7/coste0/orden1|112|112|válido|
|practice_0186|independent|E3|donante7/coste1/orden1|112|112|válido|
|practice_0186|independent|E3|donante8/coste0/orden0|110|110|válido|
|practice_0186|independent|E3|donante8/coste1/orden0|110|110|válido|
|practice_0186|independent|E3|donante8/coste0/orden1|110|110|válido|
|practice_0186|independent|E3|donante8/coste1/orden1|110|110|válido|
|practice_0186|independent|E3|donante9/coste0/orden0|107|107|válido|
|practice_0186|independent|E3|donante9/coste1/orden0|107|107|válido|
|practice_0186|independent|E3|donante9/coste0/orden1|107|107|válido|
|practice_0186|independent|E3|donante9/coste1/orden1|107|107|válido|
|practice_0187|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0187|independent|Cruce|CP True/coste0/orden0|219|189|válido|
|practice_0187|independent|Cruce|CP True/coste1/orden0|227|189|válido|
|practice_0187|independent|Cruce|CP True/coste0/orden1|212|189|válido|
|practice_0187|independent|Cruce|CP True/coste1/orden1|211|189|válido|
|practice_0187|independent|Cruce|mejor independiente/coste0/orden0|219|189|válido|
|practice_0187|independent|Cruce|mejor independiente/coste1/orden0|227|189|válido|
|practice_0187|independent|Cruce|mejor independiente/coste0/orden1|212|189|válido|
|practice_0187|independent|Cruce|mejor independiente/coste1/orden1|211|189|válido|
|practice_0187|independent|E1|cola10/pol1/inicio0|198|61|válido|
|practice_0187|independent|E1|cola10/pol1/inicio1|198|61|válido|
|practice_0187|independent|E1|cola10/pol2/inicio0|195|58|válido|
|practice_0187|independent|E1|cola10/pol2/inicio1|197|60|válido|
|practice_0187|independent|E1|cola8/pol1/inicio0|200|51|válido|
|practice_0187|independent|E1|cola8/pol1/inicio1|200|51|válido|
|practice_0187|independent|E1|cola8/pol2/inicio0|197|48|válido|
|practice_0187|independent|E1|cola8/pol2/inicio1|198|49|válido|
|practice_0187|independent|E1|cola6/pol1/inicio0|198|37|válido|
|practice_0187|independent|E1|cola6/pol1/inicio1|198|37|válido|
|practice_0187|independent|E1|cola6/pol2/inicio0|198|37|válido|
|practice_0187|independent|E1|cola6/pol2/inicio1|198|37|válido|
|practice_0187|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0187|independent|E2|CP False/coste0/orden0|219|189|válido|
|practice_0187|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0187|independent|E2|CP True/coste0/orden0|219|189|válido|
|practice_0187|independent|E3|donante0/coste0/orden0|190|189|válido|
|practice_0187|independent|E3|donante0/coste1/orden0|190|189|válido|
|practice_0187|independent|E3|donante0/coste0/orden1|212|189|válido|
|practice_0187|independent|E3|donante0/coste1/orden1|211|189|válido|
|practice_0187|independent|E3|donante1/coste0/orden0|218|218|válido|
|practice_0187|independent|E3|donante1/coste1/orden0|218|218|válido|
|practice_0187|independent|E3|donante1/coste0/orden1|218|218|válido|
|practice_0187|independent|E3|donante1/coste1/orden1|218|218|válido|
|practice_0187|independent|E3|donante2/coste0/orden0|278|201|válido|
|practice_0187|independent|E3|donante2/coste1/orden0|221|201|válido|
|practice_0187|independent|E3|donante2/coste0/orden1|202|201|válido|
|practice_0187|independent|E3|donante2/coste1/orden1|202|201|válido|
|practice_0187|independent|E3|donante3/coste0/orden0|209|200|válido|
|practice_0187|independent|E3|donante3/coste1/orden0|209|200|válido|
|practice_0187|independent|E3|donante3/coste0/orden1|200|200|válido|
|practice_0187|independent|E3|donante3/coste1/orden1|200|200|válido|
|practice_0187|independent|E3|donante4/coste0/orden0|212|212|válido|
|practice_0187|independent|E3|donante4/coste1/orden0|212|212|válido|
|practice_0187|independent|E3|donante4/coste0/orden1|212|212|válido|
|practice_0187|independent|E3|donante4/coste1/orden1|212|212|válido|
|practice_0187|independent|E3|donante5/coste0/orden0|212|212|válido|
|practice_0187|independent|E3|donante5/coste1/orden0|212|212|válido|
|practice_0187|independent|E3|donante5/coste0/orden1|212|212|válido|
|practice_0187|independent|E3|donante5/coste1/orden1|212|212|válido|
|practice_0187|independent|E3|donante6/coste0/orden0|217|217|válido|
|practice_0187|independent|E3|donante6/coste1/orden0|217|217|válido|
|practice_0187|independent|E3|donante6/coste0/orden1|217|217|válido|
|practice_0187|independent|E3|donante6/coste1/orden1|217|217|válido|
|practice_0187|independent|E3|donante7/coste0/orden0|248|201|válido|
|practice_0187|independent|E3|donante7/coste1/orden0|233|201|válido|
|practice_0187|independent|E3|donante7/coste0/orden1|228|201|válido|
|practice_0187|independent|E3|donante7/coste1/orden1|218|201|válido|
|practice_0187|independent|E3|donante8/coste0/orden0|212|185|válido|
|practice_0187|independent|E3|donante8/coste1/orden0|207|185|válido|
|practice_0187|independent|E3|donante8/coste0/orden1|227|185|válido|
|practice_0187|independent|E3|donante8/coste1/orden1|214|185|válido|
|practice_0188|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0188|independent|Cruce|CP True/coste0/orden0|215|191|válido|
|practice_0188|independent|Cruce|CP True/coste1/orden0|212|191|válido|
|practice_0188|independent|Cruce|CP True/coste0/orden1|233|191|válido|
|practice_0188|independent|Cruce|CP True/coste1/orden1|216|191|válido|
|practice_0188|independent|Cruce|mejor independiente/coste0/orden0|221|193|válido|
|practice_0188|independent|Cruce|mejor independiente/coste1/orden0|221|193|válido|
|practice_0188|independent|Cruce|mejor independiente/coste0/orden1|220|193|válido|
|practice_0188|independent|Cruce|mejor independiente/coste1/orden1|202|193|válido|
|practice_0188|independent|E1|cola10/pol1/inicio0|219|62|válido|
|practice_0188|independent|E1|cola10/pol1/inicio1|219|62|válido|
|practice_0188|independent|E1|cola10/pol2/inicio0|218|61|válido|
|practice_0188|independent|E1|cola10/pol2/inicio1|218|61|válido|
|practice_0188|independent|E1|cola8/pol1/inicio0|221|49|válido|
|practice_0188|independent|E1|cola8/pol1/inicio1|221|49|válido|
|practice_0188|independent|E1|cola8/pol2/inicio0|223|51|válido|
|practice_0188|independent|E1|cola8/pol2/inicio1|223|51|válido|
|practice_0188|independent|E1|cola6/pol1/inicio0|222|38|válido|
|practice_0188|independent|E1|cola6/pol1/inicio1|222|38|válido|
|practice_0188|independent|E1|cola6/pol2/inicio0|222|38|válido|
|practice_0188|independent|E1|cola6/pol2/inicio1|222|38|válido|
|practice_0188|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0188|independent|E2|CP False/coste0/orden0|232|210|válido|
|practice_0188|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0188|independent|E2|CP True/coste0/orden0|225|206|válido|
|practice_0188|independent|E3|donante0/coste0/orden0|217|206|válido|
|practice_0188|independent|E3|donante0/coste1/orden0|229|206|válido|
|practice_0188|independent|E3|donante0/coste0/orden1|216|206|válido|
|practice_0188|independent|E3|donante0/coste1/orden1|216|206|válido|
|practice_0188|independent|E3|donante1/coste0/orden0|224|224|válido|
|practice_0188|independent|E3|donante1/coste1/orden0|224|224|válido|
|practice_0188|independent|E3|donante1/coste0/orden1|224|224|válido|
|practice_0188|independent|E3|donante1/coste1/orden1|224|224|válido|
|practice_0188|independent|E3|donante2/coste0/orden0|220|220|válido|
|practice_0188|independent|E3|donante2/coste1/orden0|220|220|válido|
|practice_0188|independent|E3|donante2/coste0/orden1|220|220|válido|
|practice_0188|independent|E3|donante2/coste1/orden1|220|220|válido|
|practice_0188|independent|E3|donante3/coste0/orden0|279|210|válido|
|practice_0188|independent|E3|donante3/coste1/orden0|239|210|válido|
|practice_0188|independent|E3|donante3/coste0/orden1|215|210|válido|
|practice_0188|independent|E3|donante3/coste1/orden1|215|210|válido|
|practice_0188|independent|E3|donante4/coste0/orden0|233|233|válido|
|practice_0188|independent|E3|donante4/coste1/orden0|233|233|válido|
|practice_0188|independent|E3|donante4/coste0/orden1|233|233|válido|
|practice_0188|independent|E3|donante4/coste1/orden1|233|233|válido|
|practice_0188|independent|E3|donante5/coste0/orden0|234|234|válido|
|practice_0188|independent|E3|donante5/coste1/orden0|234|234|válido|
|practice_0188|independent|E3|donante5/coste0/orden1|234|234|válido|
|practice_0188|independent|E3|donante5/coste1/orden1|234|234|válido|
|practice_0188|independent|E3|donante6/coste0/orden0|230|230|válido|
|practice_0188|independent|E3|donante6/coste1/orden0|230|230|válido|
|practice_0188|independent|E3|donante6/coste0/orden1|230|230|válido|
|practice_0188|independent|E3|donante6/coste1/orden1|230|230|válido|
|practice_0188|independent|E3|donante7/coste0/orden0|259|212|válido|
|practice_0188|independent|E3|donante7/coste1/orden0|270|212|válido|
|practice_0188|independent|E3|donante7/coste0/orden1|227|212|válido|
|practice_0188|independent|E3|donante7/coste1/orden1|227|212|válido|
|practice_0188|independent|E3|donante8/coste0/orden0|218|207|válido|
|practice_0188|independent|E3|donante8/coste1/orden0|230|207|válido|
|practice_0188|independent|E3|donante8/coste0/orden1|217|207|válido|
|practice_0188|independent|E3|donante8/coste1/orden1|217|207|válido|
|practice_0188|independent|E3|donante9/coste0/orden0|227|193|válido|
|practice_0188|independent|E3|donante9/coste1/orden0|212|193|válido|
|practice_0188|independent|E3|donante9/coste0/orden1|220|193|válido|
|practice_0188|independent|E3|donante9/coste1/orden1|200|193|válido|
|practice_0189|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0189|shared|Cruce|CP True/coste0/orden0|168|168|válido|
|practice_0189|shared|Cruce|CP True/coste1/orden0|168|168|válido|
|practice_0189|shared|Cruce|CP True/coste0/orden1|168|168|válido|
|practice_0189|shared|Cruce|CP True/coste1/orden1|168|168|válido|
|practice_0189|shared|Cruce|mejor independiente/coste0/orden0|168|168|válido|
|practice_0189|shared|Cruce|mejor independiente/coste1/orden0|168|168|válido|
|practice_0189|shared|Cruce|mejor independiente/coste0/orden1|168|168|válido|
|practice_0189|shared|Cruce|mejor independiente/coste1/orden1|168|168|válido|
|practice_0189|shared|E1|cola10/pol1/inicio0|175|67|válido|
|practice_0189|shared|E1|cola10/pol1/inicio1|175|67|válido|
|practice_0189|shared|E1|cola10/pol2/inicio0|175|67|válido|
|practice_0189|shared|E1|cola10/pol2/inicio1|175|67|válido|
|practice_0189|shared|E1|cola8/pol1/inicio0|182|61|válido|
|practice_0189|shared|E1|cola8/pol1/inicio1|182|61|válido|
|practice_0189|shared|E1|cola8/pol2/inicio0|182|61|válido|
|practice_0189|shared|E1|cola8/pol2/inicio1|182|61|válido|
|practice_0189|shared|E1|cola6/pol1/inicio0|172|39|válido|
|practice_0189|shared|E1|cola6/pol1/inicio1|172|39|válido|
|practice_0189|shared|E1|cola6/pol2/inicio0|172|39|válido|
|practice_0189|shared|E1|cola6/pol2/inicio1|172|39|válido|
|practice_0189|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0189|shared|E2|CP False/coste0/orden0|184|184|válido|
|practice_0189|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0189|shared|E2|CP True/coste0/orden0|168|168|válido|
|practice_0189|shared|E3|donante0/coste0/orden0|170|170|válido|
|practice_0189|shared|E3|donante0/coste1/orden0|170|170|válido|
|practice_0189|shared|E3|donante0/coste0/orden1|170|170|válido|
|practice_0189|shared|E3|donante0/coste1/orden1|170|170|válido|
|practice_0189|shared|E3|donante1/coste0/orden0|187|187|válido|
|practice_0189|shared|E3|donante1/coste1/orden0|187|187|válido|
|practice_0189|shared|E3|donante1/coste0/orden1|187|187|válido|
|practice_0189|shared|E3|donante1/coste1/orden1|187|187|válido|
|practice_0189|shared|E3|donante2/coste0/orden0|186|186|válido|
|practice_0189|shared|E3|donante2/coste1/orden0|186|186|válido|
|practice_0189|shared|E3|donante2/coste0/orden1|186|186|válido|
|practice_0189|shared|E3|donante2/coste1/orden1|186|186|válido|
|practice_0189|shared|E3|donante3/coste0/orden0|182|182|válido|
|practice_0189|shared|E3|donante3/coste1/orden0|182|182|válido|
|practice_0189|shared|E3|donante3/coste0/orden1|182|182|válido|
|practice_0189|shared|E3|donante3/coste1/orden1|182|182|válido|
|practice_0189|shared|E3|donante4/coste0/orden0|208|208|válido|
|practice_0189|shared|E3|donante4/coste1/orden0|208|208|válido|
|practice_0189|shared|E3|donante4/coste0/orden1|208|208|válido|
|practice_0189|shared|E3|donante4/coste1/orden1|208|208|válido|
|practice_0189|shared|E3|donante5/coste0/orden0|191|191|válido|
|practice_0189|shared|E3|donante5/coste1/orden0|191|191|válido|
|practice_0189|shared|E3|donante5/coste0/orden1|191|191|válido|
|practice_0189|shared|E3|donante5/coste1/orden1|191|191|válido|
|practice_0189|shared|E3|donante6/coste0/orden0|193|193|válido|
|practice_0189|shared|E3|donante6/coste1/orden0|193|193|válido|
|practice_0189|shared|E3|donante6/coste0/orden1|193|193|válido|
|practice_0189|shared|E3|donante6/coste1/orden1|193|193|válido|
|practice_0189|shared|E3|donante7/coste0/orden0|226|182|válido|
|practice_0189|shared|E3|donante7/coste1/orden0|222|182|válido|
|practice_0189|shared|E3|donante7/coste0/orden1|182|182|válido|
|practice_0189|shared|E3|donante7/coste1/orden1|182|182|válido|
|practice_0189|shared|E3|donante8/coste0/orden0|180|174|válido|
|practice_0189|shared|E3|donante8/coste1/orden0|175|174|válido|
|practice_0189|shared|E3|donante8/coste0/orden1|180|174|válido|
|practice_0189|shared|E3|donante8/coste1/orden1|175|174|válido|
|practice_0189|shared|E3|donante9/coste0/orden0|171|171|válido|
|practice_0189|shared|E3|donante9/coste1/orden0|171|171|válido|
|practice_0189|shared|E3|donante9/coste0/orden1|171|171|válido|
|practice_0189|shared|E3|donante9/coste1/orden1|171|171|válido|
|practice_0190|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0190|mixed|Cruce|CP True/coste0/orden0|326|259|válido|
|practice_0190|mixed|Cruce|CP True/coste1/orden0|301|259|válido|
|practice_0190|mixed|Cruce|CP True/coste0/orden1|272|259|válido|
|practice_0190|mixed|Cruce|CP True/coste1/orden1|272|259|válido|
|practice_0190|mixed|Cruce|mejor independiente/coste0/orden0|339|259|válido|
|practice_0190|mixed|Cruce|mejor independiente/coste1/orden0|290|259|válido|
|practice_0190|mixed|Cruce|mejor independiente/coste0/orden1|274|259|válido|
|practice_0190|mixed|Cruce|mejor independiente/coste1/orden1|274|259|válido|
|practice_0190|mixed|E1|cola10/pol1/inicio0|275|94|válido|
|practice_0190|mixed|E1|cola10/pol1/inicio1|275|94|válido|
|practice_0190|mixed|E1|cola10/pol2/inicio0|272|91|válido|
|practice_0190|mixed|E1|cola10/pol2/inicio1|267|86|válido|
|practice_0190|mixed|E1|cola8/pol1/inicio0|281|74|válido|
|practice_0190|mixed|E1|cola8/pol1/inicio1|281|74|válido|
|practice_0190|mixed|E1|cola8/pol2/inicio0|279|72|válido|
|practice_0190|mixed|E1|cola8/pol2/inicio1|279|72|válido|
|practice_0190|mixed|E1|cola6/pol1/inicio0|276|53|válido|
|practice_0190|mixed|E1|cola6/pol1/inicio1|276|53|válido|
|practice_0190|mixed|E1|cola6/pol2/inicio0|279|56|válido|
|practice_0190|mixed|E1|cola6/pol2/inicio1|279|56|válido|
|practice_0190|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0190|mixed|E2|CP False/coste0/orden0|365|254|válido|
|practice_0190|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0190|mixed|E2|CP True/coste0/orden0|339|259|válido|
|practice_0190|mixed|E3|donante0/coste0/orden0|279|259|válido|
|practice_0190|mixed|E3|donante0/coste1/orden0|271|259|válido|
|practice_0190|mixed|E3|donante0/coste0/orden1|273|259|válido|
|practice_0190|mixed|E3|donante0/coste1/orden1|269|259|válido|
|practice_0190|mixed|E3|donante1/coste0/orden0|295|295|válido|
|practice_0190|mixed|E3|donante1/coste1/orden0|295|295|válido|
|practice_0190|mixed|E3|donante1/coste0/orden1|295|295|válido|
|practice_0190|mixed|E3|donante1/coste1/orden1|295|295|válido|
|practice_0190|mixed|E3|donante2/coste0/orden0|359|284|válido|
|practice_0190|mixed|E3|donante2/coste1/orden0|346|284|válido|
|practice_0190|mixed|E3|donante2/coste0/orden1|284|284|válido|
|practice_0190|mixed|E3|donante2/coste1/orden1|284|284|válido|
|practice_0190|mixed|E3|donante3/coste0/orden0|289|278|válido|
|practice_0190|mixed|E3|donante3/coste1/orden0|285|278|válido|
|practice_0190|mixed|E3|donante3/coste0/orden1|278|278|válido|
|practice_0190|mixed|E3|donante3/coste1/orden1|278|278|válido|
|practice_0190|mixed|E3|donante4/coste0/orden0|297|297|válido|
|practice_0190|mixed|E3|donante4/coste1/orden0|297|297|válido|
|practice_0190|mixed|E3|donante4/coste0/orden1|297|297|válido|
|practice_0190|mixed|E3|donante4/coste1/orden1|297|297|válido|
|practice_0190|mixed|E3|donante5/coste0/orden0|303|303|válido|
|practice_0190|mixed|E3|donante5/coste1/orden0|303|303|válido|
|practice_0190|mixed|E3|donante5/coste0/orden1|303|303|válido|
|practice_0190|mixed|E3|donante5/coste1/orden1|303|303|válido|
|practice_0190|mixed|E3|donante6/coste0/orden0|297|297|válido|
|practice_0190|mixed|E3|donante6/coste1/orden0|297|297|válido|
|practice_0190|mixed|E3|donante6/coste0/orden1|297|297|válido|
|practice_0190|mixed|E3|donante6/coste1/orden1|297|297|válido|
|practice_0190|mixed|E3|donante7/coste0/orden0|330|266|válido|
|practice_0190|mixed|E3|donante7/coste1/orden0|293|266|válido|
|practice_0190|mixed|E3|donante7/coste0/orden1|277|266|válido|
|practice_0190|mixed|E3|donante7/coste1/orden1|274|266|válido|
|practice_0190|mixed|E3|donante8/coste0/orden0|282|260|válido|
|practice_0190|mixed|E3|donante8/coste1/orden0|272|260|válido|
|practice_0190|mixed|E3|donante8/coste0/orden1|282|260|válido|
|practice_0190|mixed|E3|donante8/coste1/orden1|278|260|válido|
|practice_0190|mixed|E3|donante9/coste0/orden0|335|250|válido|
|practice_0190|mixed|E3|donante9/coste1/orden0|320|250|válido|
|practice_0190|mixed|E3|donante9/coste0/orden1|312|250|válido|
|practice_0190|mixed|E3|donante9/coste1/orden1|293|250|válido|
|practice_0191|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0191|independent|Cruce|CP True/coste0/orden0|325|264|válido|
|practice_0191|independent|Cruce|CP True/coste1/orden0|325|264|válido|
|practice_0191|independent|Cruce|CP True/coste0/orden1|274|264|válido|
|practice_0191|independent|Cruce|CP True/coste1/orden1|269|264|válido|
|practice_0191|independent|Cruce|mejor independiente/coste0/orden0|326|265|válido|
|practice_0191|independent|Cruce|mejor independiente/coste1/orden0|316|265|válido|
|practice_0191|independent|Cruce|mejor independiente/coste0/orden1|275|265|válido|
|practice_0191|independent|Cruce|mejor independiente/coste1/orden1|270|265|válido|
|practice_0191|independent|E1|cola10/pol1/inicio0|284|96|válido|
|practice_0191|independent|E1|cola10/pol1/inicio1|284|96|válido|
|practice_0191|independent|E1|cola10/pol2/inicio0|280|92|válido|
|practice_0191|independent|E1|cola10/pol2/inicio1|280|92|válido|
|practice_0191|independent|E1|cola8/pol1/inicio0|284|76|válido|
|practice_0191|independent|E1|cola8/pol1/inicio1|283|75|válido|
|practice_0191|independent|E1|cola8/pol2/inicio0|283|75|válido|
|practice_0191|independent|E1|cola8/pol2/inicio1|283|75|válido|
|practice_0191|independent|E1|cola6/pol1/inicio0|286|60|válido|
|practice_0191|independent|E1|cola6/pol1/inicio1|286|60|válido|
|practice_0191|independent|E1|cola6/pol2/inicio0|285|59|válido|
|practice_0191|independent|E1|cola6/pol2/inicio1|285|59|válido|
|practice_0191|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0191|independent|E2|CP False/coste0/orden0|348|275|válido|
|practice_0191|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0191|independent|E2|CP True/coste0/orden0|326|265|válido|
|practice_0191|independent|E3|donante0/coste0/orden0|275|265|válido|
|practice_0191|independent|E3|donante0/coste1/orden0|276|265|válido|
|practice_0191|independent|E3|donante0/coste0/orden1|275|265|válido|
|practice_0191|independent|E3|donante0/coste1/orden1|270|265|válido|
|practice_0191|independent|E3|donante1/coste0/orden0|313|313|válido|
|practice_0191|independent|E3|donante1/coste1/orden0|313|313|válido|
|practice_0191|independent|E3|donante1/coste0/orden1|313|313|válido|
|practice_0191|independent|E3|donante1/coste1/orden1|313|313|válido|
|practice_0191|independent|E3|donante2/coste0/orden0|565|288|válido|
|practice_0191|independent|E3|donante2/coste1/orden0|374|288|válido|
|practice_0191|independent|E3|donante2/coste0/orden1|288|288|válido|
|practice_0191|independent|E3|donante2/coste1/orden1|288|288|válido|
|practice_0191|independent|E3|donante3/coste0/orden0|575|293|válido|
|practice_0191|independent|E3|donante3/coste1/orden0|419|293|válido|
|practice_0191|independent|E3|donante3/coste0/orden1|293|293|válido|
|practice_0191|independent|E3|donante3/coste1/orden1|293|293|válido|
|practice_0191|independent|E3|donante4/coste0/orden0|322|322|válido|
|practice_0191|independent|E3|donante4/coste1/orden0|322|322|válido|
|practice_0191|independent|E3|donante4/coste0/orden1|322|322|válido|
|practice_0191|independent|E3|donante4/coste1/orden1|322|322|válido|
|practice_0191|independent|E3|donante5/coste0/orden0|328|328|válido|
|practice_0191|independent|E3|donante5/coste1/orden0|328|328|válido|
|practice_0191|independent|E3|donante5/coste0/orden1|328|328|válido|
|practice_0191|independent|E3|donante5/coste1/orden1|328|328|válido|
|practice_0191|independent|E3|donante6/coste0/orden0|317|317|válido|
|practice_0191|independent|E3|donante6/coste1/orden0|317|317|válido|
|practice_0191|independent|E3|donante6/coste0/orden1|317|317|válido|
|practice_0191|independent|E3|donante6/coste1/orden1|317|317|válido|
|practice_0191|independent|E3|donante7/coste0/orden0|321|280|válido|
|practice_0191|independent|E3|donante7/coste1/orden0|315|280|válido|
|practice_0191|independent|E3|donante7/coste0/orden1|309|280|válido|
|practice_0191|independent|E3|donante7/coste1/orden1|307|280|válido|
|practice_0191|independent|E3|donante8/coste0/orden0|275|265|válido|
|practice_0191|independent|E3|donante8/coste1/orden0|276|265|válido|
|practice_0191|independent|E3|donante8/coste0/orden1|275|265|válido|
|practice_0191|independent|E3|donante8/coste1/orden1|270|265|válido|
|practice_0191|independent|E3|donante9/coste0/orden0|276|269|válido|
|practice_0191|independent|E3|donante9/coste1/orden0|276|269|válido|
|practice_0191|independent|E3|donante9/coste0/orden1|287|269|válido|
|practice_0191|independent|E3|donante9/coste1/orden1|289|269|válido|
|practice_0192|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0192|multiplicative|Cruce|CP True/coste0/orden0|113|113|válido|
|practice_0192|multiplicative|Cruce|CP True/coste1/orden0|113|113|válido|
|practice_0192|multiplicative|Cruce|CP True/coste0/orden1|113|113|válido|
|practice_0192|multiplicative|Cruce|CP True/coste1/orden1|113|113|válido|
|practice_0192|multiplicative|Cruce|mejor independiente/coste0/orden0|114|114|válido|
|practice_0192|multiplicative|Cruce|mejor independiente/coste1/orden0|114|114|válido|
|practice_0192|multiplicative|Cruce|mejor independiente/coste0/orden1|114|114|válido|
|practice_0192|multiplicative|Cruce|mejor independiente/coste1/orden1|114|114|válido|
|practice_0192|multiplicative|E1|cola10/pol1/inicio0|—|—|registros ocupados|
|practice_0192|multiplicative|E1|cola10/pol1/inicio1|—|—|registros ocupados|
|practice_0192|multiplicative|E1|cola10/pol2/inicio0|—|—|registros ocupados|
|practice_0192|multiplicative|E1|cola10/pol2/inicio1|—|—|registros ocupados|
|practice_0192|multiplicative|E1|cola8/pol1/inicio0|—|—|registros ocupados|
|practice_0192|multiplicative|E1|cola8/pol1/inicio1|—|—|registros ocupados|
|practice_0192|multiplicative|E1|cola8/pol2/inicio0|—|—|registros ocupados|
|practice_0192|multiplicative|E1|cola8/pol2/inicio1|—|—|registros ocupados|
|practice_0192|multiplicative|E1|cola6/pol1/inicio0|—|—|registros ocupados|
|practice_0192|multiplicative|E1|cola6/pol1/inicio1|—|—|registros ocupados|
|practice_0192|multiplicative|E1|cola6/pol2/inicio0|—|—|registros ocupados|
|practice_0192|multiplicative|E1|cola6/pol2/inicio1|—|—|registros ocupados|
|practice_0192|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0192|multiplicative|E2|CP False/coste0/orden0|115|115|válido|
|practice_0192|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0192|multiplicative|E2|CP True/coste0/orden0|114|114|válido|
|practice_0192|multiplicative|E3|donante0/coste0/orden0|116|116|válido|
|practice_0192|multiplicative|E3|donante0/coste1/orden0|116|116|válido|
|practice_0192|multiplicative|E3|donante0/coste0/orden1|116|116|válido|
|practice_0192|multiplicative|E3|donante0/coste1/orden1|116|116|válido|
|practice_0192|multiplicative|E3|donante1/coste0/orden0|125|125|válido|
|practice_0192|multiplicative|E3|donante1/coste1/orden0|125|125|válido|
|practice_0192|multiplicative|E3|donante1/coste0/orden1|125|125|válido|
|practice_0192|multiplicative|E3|donante1/coste1/orden1|125|125|válido|
|practice_0192|multiplicative|E3|donante2/coste0/orden0|123|123|válido|
|practice_0192|multiplicative|E3|donante2/coste1/orden0|123|123|válido|
|practice_0192|multiplicative|E3|donante2/coste0/orden1|123|123|válido|
|practice_0192|multiplicative|E3|donante2/coste1/orden1|123|123|válido|
|practice_0192|multiplicative|E3|donante3/coste0/orden0|118|118|válido|
|practice_0192|multiplicative|E3|donante3/coste1/orden0|118|118|válido|
|practice_0192|multiplicative|E3|donante3/coste0/orden1|118|118|válido|
|practice_0192|multiplicative|E3|donante3/coste1/orden1|118|118|válido|
|practice_0192|multiplicative|E3|donante4/coste0/orden0|122|122|válido|
|practice_0192|multiplicative|E3|donante4/coste1/orden0|122|122|válido|
|practice_0192|multiplicative|E3|donante4/coste0/orden1|122|122|válido|
|practice_0192|multiplicative|E3|donante4/coste1/orden1|122|122|válido|
|practice_0192|multiplicative|E3|donante5/coste0/orden0|125|125|válido|
|practice_0192|multiplicative|E3|donante5/coste1/orden0|125|125|válido|
|practice_0192|multiplicative|E3|donante5/coste0/orden1|125|125|válido|
|practice_0192|multiplicative|E3|donante5/coste1/orden1|125|125|válido|
|practice_0192|multiplicative|E3|donante6/coste0/orden0|123|123|válido|
|practice_0192|multiplicative|E3|donante6/coste1/orden0|123|123|válido|
|practice_0192|multiplicative|E3|donante6/coste0/orden1|123|123|válido|
|practice_0192|multiplicative|E3|donante6/coste1/orden1|123|123|válido|
|practice_0192|multiplicative|E3|donante7/coste0/orden0|123|123|válido|
|practice_0192|multiplicative|E3|donante7/coste1/orden0|123|123|válido|
|practice_0192|multiplicative|E3|donante7/coste0/orden1|123|123|válido|
|practice_0192|multiplicative|E3|donante7/coste1/orden1|123|123|válido|
|practice_0192|multiplicative|E3|donante8/coste0/orden0|123|123|válido|
|practice_0192|multiplicative|E3|donante8/coste1/orden0|123|123|válido|
|practice_0192|multiplicative|E3|donante8/coste0/orden1|123|123|válido|
|practice_0192|multiplicative|E3|donante8/coste1/orden1|123|123|válido|
|practice_0192|multiplicative|E3|donante9/coste0/orden0|117|117|válido|
|practice_0192|multiplicative|E3|donante9/coste1/orden0|117|117|válido|
|practice_0192|multiplicative|E3|donante9/coste0/orden1|117|117|válido|
|practice_0192|multiplicative|E3|donante9/coste1/orden1|117|117|válido|
|practice_0193|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0193|multiplicative|Cruce|CP True/coste0/orden0|124|124|válido|
|practice_0193|multiplicative|Cruce|CP True/coste1/orden0|124|124|válido|
|practice_0193|multiplicative|Cruce|CP True/coste0/orden1|124|124|válido|
|practice_0193|multiplicative|Cruce|CP True/coste1/orden1|124|124|válido|
|practice_0193|multiplicative|Cruce|mejor independiente/coste0/orden0|124|124|válido|
|practice_0193|multiplicative|Cruce|mejor independiente/coste1/orden0|124|124|válido|
|practice_0193|multiplicative|Cruce|mejor independiente/coste0/orden1|124|124|válido|
|practice_0193|multiplicative|Cruce|mejor independiente/coste1/orden1|124|124|válido|
|practice_0193|multiplicative|E1|cola10/pol1/inicio0|130|45|válido|
|practice_0193|multiplicative|E1|cola10/pol1/inicio1|130|45|válido|
|practice_0193|multiplicative|E1|cola10/pol2/inicio0|129|44|válido|
|practice_0193|multiplicative|E1|cola10/pol2/inicio1|129|44|válido|
|practice_0193|multiplicative|E1|cola8/pol1/inicio0|130|38|válido|
|practice_0193|multiplicative|E1|cola8/pol1/inicio1|130|38|válido|
|practice_0193|multiplicative|E1|cola8/pol2/inicio0|130|38|válido|
|practice_0193|multiplicative|E1|cola8/pol2/inicio1|130|38|válido|
|practice_0193|multiplicative|E1|cola6/pol1/inicio0|136|35|válido|
|practice_0193|multiplicative|E1|cola6/pol1/inicio1|136|35|válido|
|practice_0193|multiplicative|E1|cola6/pol2/inicio0|136|35|válido|
|practice_0193|multiplicative|E1|cola6/pol2/inicio1|136|35|válido|
|practice_0193|multiplicative|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0193|multiplicative|E2|CP False/coste0/orden0|131|131|válido|
|practice_0193|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0193|multiplicative|E2|CP True/coste0/orden0|124|124|válido|
|practice_0193|multiplicative|E3|donante0/coste0/orden0|126|126|válido|
|practice_0193|multiplicative|E3|donante0/coste1/orden0|126|126|válido|
|practice_0193|multiplicative|E3|donante0/coste0/orden1|126|126|válido|
|practice_0193|multiplicative|E3|donante0/coste1/orden1|126|126|válido|
|practice_0193|multiplicative|E3|donante1/coste0/orden0|141|141|válido|
|practice_0193|multiplicative|E3|donante1/coste1/orden0|141|141|válido|
|practice_0193|multiplicative|E3|donante1/coste0/orden1|141|141|válido|
|practice_0193|multiplicative|E3|donante1/coste1/orden1|141|141|válido|
|practice_0193|multiplicative|E3|donante2/coste0/orden0|134|134|válido|
|practice_0193|multiplicative|E3|donante2/coste1/orden0|134|134|válido|
|practice_0193|multiplicative|E3|donante2/coste0/orden1|134|134|válido|
|practice_0193|multiplicative|E3|donante2/coste1/orden1|134|134|válido|
|practice_0193|multiplicative|E3|donante3/coste0/orden0|128|128|válido|
|practice_0193|multiplicative|E3|donante3/coste1/orden0|128|128|válido|
|practice_0193|multiplicative|E3|donante3/coste0/orden1|128|128|válido|
|practice_0193|multiplicative|E3|donante3/coste1/orden1|128|128|válido|
|practice_0193|multiplicative|E3|donante4/coste0/orden0|144|144|válido|
|practice_0193|multiplicative|E3|donante4/coste1/orden0|144|144|válido|
|practice_0193|multiplicative|E3|donante4/coste0/orden1|144|144|válido|
|practice_0193|multiplicative|E3|donante4/coste1/orden1|144|144|válido|
|practice_0193|multiplicative|E3|donante5/coste0/orden0|136|136|válido|
|practice_0193|multiplicative|E3|donante5/coste1/orden0|136|136|válido|
|practice_0193|multiplicative|E3|donante5/coste0/orden1|136|136|válido|
|practice_0193|multiplicative|E3|donante5/coste1/orden1|136|136|válido|
|practice_0193|multiplicative|E3|donante6/coste0/orden0|135|135|válido|
|practice_0193|multiplicative|E3|donante6/coste1/orden0|135|135|válido|
|practice_0193|multiplicative|E3|donante6/coste0/orden1|135|135|válido|
|practice_0193|multiplicative|E3|donante6/coste1/orden1|135|135|válido|
|practice_0193|multiplicative|E3|donante7/coste0/orden0|134|133|válido|
|practice_0193|multiplicative|E3|donante7/coste1/orden0|134|133|válido|
|practice_0193|multiplicative|E3|donante7/coste0/orden1|133|133|válido|
|practice_0193|multiplicative|E3|donante7/coste1/orden1|133|133|válido|
|practice_0193|multiplicative|E3|donante8/coste0/orden0|128|128|válido|
|practice_0193|multiplicative|E3|donante8/coste1/orden0|128|128|válido|
|practice_0193|multiplicative|E3|donante8/coste0/orden1|128|128|válido|
|practice_0193|multiplicative|E3|donante8/coste1/orden1|128|128|válido|
|practice_0194|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0194|mixed|Cruce|CP True/coste0/orden0|221|214|válido|
|practice_0194|mixed|Cruce|CP True/coste1/orden0|221|214|válido|
|practice_0194|mixed|Cruce|CP True/coste0/orden1|214|214|válido|
|practice_0194|mixed|Cruce|CP True/coste1/orden1|214|214|válido|
|practice_0194|mixed|Cruce|mejor independiente/coste0/orden0|226|215|válido|
|practice_0194|mixed|Cruce|mejor independiente/coste1/orden0|226|215|válido|
|practice_0194|mixed|Cruce|mejor independiente/coste0/orden1|215|215|válido|
|practice_0194|mixed|Cruce|mejor independiente/coste1/orden1|215|215|válido|
|practice_0194|mixed|E1|cola10/pol1/inicio0|228|87|válido|
|practice_0194|mixed|E1|cola10/pol1/inicio1|228|87|válido|
|practice_0194|mixed|E1|cola10/pol2/inicio0|229|88|válido|
|practice_0194|mixed|E1|cola10/pol2/inicio1|227|86|válido|
|practice_0194|mixed|E1|cola8/pol1/inicio0|232|74|válido|
|practice_0194|mixed|E1|cola8/pol1/inicio1|232|74|válido|
|practice_0194|mixed|E1|cola8/pol2/inicio0|228|70|válido|
|practice_0194|mixed|E1|cola8/pol2/inicio1|228|70|válido|
|practice_0194|mixed|E1|cola6/pol1/inicio0|233|59|válido|
|practice_0194|mixed|E1|cola6/pol1/inicio1|233|59|válido|
|practice_0194|mixed|E1|cola6/pol2/inicio0|233|59|válido|
|practice_0194|mixed|E1|cola6/pol2/inicio1|233|59|válido|
|practice_0194|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0194|mixed|E2|CP False/coste0/orden0|220|220|válido|
|practice_0194|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0194|mixed|E2|CP True/coste0/orden0|226|215|válido|
|practice_0194|mixed|E3|donante0/coste0/orden0|220|220|válido|
|practice_0194|mixed|E3|donante0/coste1/orden0|220|220|válido|
|practice_0194|mixed|E3|donante0/coste0/orden1|220|220|válido|
|practice_0194|mixed|E3|donante0/coste1/orden1|220|220|válido|
|practice_0194|mixed|E3|donante1/coste0/orden0|249|249|válido|
|practice_0194|mixed|E3|donante1/coste1/orden0|249|249|válido|
|practice_0194|mixed|E3|donante1/coste0/orden1|249|249|válido|
|practice_0194|mixed|E3|donante1/coste1/orden1|249|249|válido|
|practice_0194|mixed|E3|donante2/coste0/orden0|248|242|válido|
|practice_0194|mixed|E3|donante2/coste1/orden0|245|242|válido|
|practice_0194|mixed|E3|donante2/coste0/orden1|242|242|válido|
|practice_0194|mixed|E3|donante2/coste1/orden1|242|242|válido|
|practice_0194|mixed|E3|donante3/coste0/orden0|289|242|válido|
|practice_0194|mixed|E3|donante3/coste1/orden0|295|242|válido|
|practice_0194|mixed|E3|donante3/coste0/orden1|242|242|válido|
|practice_0194|mixed|E3|donante3/coste1/orden1|242|242|válido|
|practice_0194|mixed|E3|donante4/coste0/orden0|250|250|válido|
|practice_0194|mixed|E3|donante4/coste1/orden0|250|250|válido|
|practice_0194|mixed|E3|donante4/coste0/orden1|250|250|válido|
|practice_0194|mixed|E3|donante4/coste1/orden1|250|250|válido|
|practice_0194|mixed|E3|donante5/coste0/orden0|246|246|válido|
|practice_0194|mixed|E3|donante5/coste1/orden0|246|246|válido|
|practice_0194|mixed|E3|donante5/coste0/orden1|246|246|válido|
|practice_0194|mixed|E3|donante5/coste1/orden1|246|246|válido|
|practice_0194|mixed|E3|donante6/coste0/orden0|256|256|válido|
|practice_0194|mixed|E3|donante6/coste1/orden0|256|256|válido|
|practice_0194|mixed|E3|donante6/coste0/orden1|256|256|válido|
|practice_0194|mixed|E3|donante6/coste1/orden1|256|256|válido|
|practice_0194|mixed|E3|donante7/coste0/orden0|243|224|válido|
|practice_0194|mixed|E3|donante7/coste1/orden0|234|224|válido|
|practice_0194|mixed|E3|donante7/coste0/orden1|265|224|válido|
|practice_0194|mixed|E3|donante7/coste1/orden1|254|224|válido|
|practice_0194|mixed|E3|donante8/coste0/orden0|231|218|válido|
|practice_0194|mixed|E3|donante8/coste1/orden0|226|218|válido|
|practice_0194|mixed|E3|donante8/coste0/orden1|244|218|válido|
|practice_0194|mixed|E3|donante8/coste1/orden1|235|218|válido|
|practice_0195|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0195|independent|Cruce|CP True/coste0/orden0|276|228|válido|
|practice_0195|independent|Cruce|CP True/coste1/orden0|262|228|válido|
|practice_0195|independent|Cruce|CP True/coste0/orden1|250|228|válido|
|practice_0195|independent|Cruce|CP True/coste1/orden1|252|228|válido|
|practice_0195|independent|Cruce|mejor independiente/coste0/orden0|276|228|válido|
|practice_0195|independent|Cruce|mejor independiente/coste1/orden0|262|228|válido|
|practice_0195|independent|Cruce|mejor independiente/coste0/orden1|250|228|válido|
|practice_0195|independent|Cruce|mejor independiente/coste1/orden1|252|228|válido|
|practice_0195|independent|E1|cola10/pol1/inicio0|238|72|válido|
|practice_0195|independent|E1|cola10/pol1/inicio1|238|72|válido|
|practice_0195|independent|E1|cola10/pol2/inicio0|237|71|válido|
|practice_0195|independent|E1|cola10/pol2/inicio1|239|73|válido|
|practice_0195|independent|E1|cola8/pol1/inicio0|243|63|válido|
|practice_0195|independent|E1|cola8/pol1/inicio1|243|63|válido|
|practice_0195|independent|E1|cola8/pol2/inicio0|241|61|válido|
|practice_0195|independent|E1|cola8/pol2/inicio1|244|64|válido|
|practice_0195|independent|E1|cola6/pol1/inicio0|249|48|válido|
|practice_0195|independent|E1|cola6/pol1/inicio1|249|48|válido|
|practice_0195|independent|E1|cola6/pol2/inicio0|247|46|válido|
|practice_0195|independent|E1|cola6/pol2/inicio1|247|46|válido|
|practice_0195|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0195|independent|E2|CP False/coste0/orden0|275|231|válido|
|practice_0195|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0195|independent|E2|CP True/coste0/orden0|275|231|válido|
|practice_0195|independent|E3|donante0/coste0/orden0|244|232|válido|
|practice_0195|independent|E3|donante0/coste1/orden0|241|232|válido|
|practice_0195|independent|E3|donante0/coste0/orden1|287|232|válido|
|practice_0195|independent|E3|donante0/coste1/orden1|243|232|válido|
|practice_0195|independent|E3|donante1/coste0/orden0|262|262|válido|
|practice_0195|independent|E3|donante1/coste1/orden0|262|262|válido|
|practice_0195|independent|E3|donante1/coste0/orden1|262|262|válido|
|practice_0195|independent|E3|donante1/coste1/orden1|262|262|válido|
|practice_0195|independent|E3|donante2/coste0/orden0|352|250|válido|
|practice_0195|independent|E3|donante2/coste1/orden0|281|250|válido|
|practice_0195|independent|E3|donante2/coste0/orden1|250|250|válido|
|practice_0195|independent|E3|donante2/coste1/orden1|250|250|válido|
|practice_0195|independent|E3|donante3/coste0/orden0|333|244|válido|
|practice_0195|independent|E3|donante3/coste1/orden0|279|244|válido|
|practice_0195|independent|E3|donante3/coste0/orden1|244|244|válido|
|practice_0195|independent|E3|donante3/coste1/orden1|244|244|válido|
|practice_0195|independent|E3|donante4/coste0/orden0|271|271|válido|
|practice_0195|independent|E3|donante4/coste1/orden0|271|271|válido|
|practice_0195|independent|E3|donante4/coste0/orden1|271|271|válido|
|practice_0195|independent|E3|donante4/coste1/orden1|271|271|válido|
|practice_0195|independent|E3|donante5/coste0/orden0|269|269|válido|
|practice_0195|independent|E3|donante5/coste1/orden0|269|269|válido|
|practice_0195|independent|E3|donante5/coste0/orden1|269|269|válido|
|practice_0195|independent|E3|donante5/coste1/orden1|269|269|válido|
|practice_0195|independent|E3|donante6/coste0/orden0|261|261|válido|
|practice_0195|independent|E3|donante6/coste1/orden0|261|261|válido|
|practice_0195|independent|E3|donante6/coste0/orden1|261|261|válido|
|practice_0195|independent|E3|donante6/coste1/orden1|261|261|válido|
|practice_0195|independent|E3|donante7/coste0/orden0|251|236|válido|
|practice_0195|independent|E3|donante7/coste1/orden0|244|236|válido|
|practice_0195|independent|E3|donante7/coste0/orden1|264|236|válido|
|practice_0195|independent|E3|donante7/coste1/orden1|241|236|válido|
|practice_0195|independent|E3|donante8/coste0/orden0|254|231|válido|
|practice_0195|independent|E3|donante8/coste1/orden0|251|231|válido|
|practice_0195|independent|E3|donante8/coste0/orden1|295|231|válido|
|practice_0195|independent|E3|donante8/coste1/orden1|249|231|válido|
|practice_0195|independent|E3|donante9/coste0/orden0|258|229|válido|
|practice_0195|independent|E3|donante9/coste1/orden0|244|229|válido|
|practice_0195|independent|E3|donante9/coste0/orden1|286|229|válido|
|practice_0195|independent|E3|donante9/coste1/orden1|295|229|válido|
|practice_0196|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0196|shared|Cruce|CP True/coste0/orden0|126|126|válido|
|practice_0196|shared|Cruce|CP True/coste1/orden0|126|126|válido|
|practice_0196|shared|Cruce|CP True/coste0/orden1|126|126|válido|
|practice_0196|shared|Cruce|CP True/coste1/orden1|126|126|válido|
|practice_0196|shared|Cruce|mejor independiente/coste0/orden0|128|128|válido|
|practice_0196|shared|Cruce|mejor independiente/coste1/orden0|128|128|válido|
|practice_0196|shared|Cruce|mejor independiente/coste0/orden1|128|128|válido|
|practice_0196|shared|Cruce|mejor independiente/coste1/orden1|128|128|válido|
|practice_0196|shared|E1|cola10/pol1/inicio0|131|32|válido|
|practice_0196|shared|E1|cola10/pol1/inicio1|131|32|válido|
|practice_0196|shared|E1|cola10/pol2/inicio0|130|31|válido|
|practice_0196|shared|E1|cola10/pol2/inicio1|130|31|válido|
|practice_0196|shared|E1|cola8/pol1/inicio0|131|30|válido|
|practice_0196|shared|E1|cola8/pol1/inicio1|131|30|válido|
|practice_0196|shared|E1|cola8/pol2/inicio0|130|29|válido|
|practice_0196|shared|E1|cola8/pol2/inicio1|130|29|válido|
|practice_0196|shared|E1|cola6/pol1/inicio0|128|18|válido|
|practice_0196|shared|E1|cola6/pol1/inicio1|128|18|válido|
|practice_0196|shared|E1|cola6/pol2/inicio0|128|18|válido|
|practice_0196|shared|E1|cola6/pol2/inicio1|128|18|válido|
|practice_0196|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0196|shared|E2|CP False/coste0/orden0|131|131|válido|
|practice_0196|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0196|shared|E2|CP True/coste0/orden0|132|132|válido|
|practice_0196|shared|E3|donante0/coste0/orden0|128|128|válido|
|practice_0196|shared|E3|donante0/coste1/orden0|128|128|válido|
|practice_0196|shared|E3|donante0/coste0/orden1|128|128|válido|
|practice_0196|shared|E3|donante0/coste1/orden1|128|128|válido|
|practice_0196|shared|E3|donante1/coste0/orden0|137|137|válido|
|practice_0196|shared|E3|donante1/coste1/orden0|137|137|válido|
|practice_0196|shared|E3|donante1/coste0/orden1|137|137|válido|
|practice_0196|shared|E3|donante1/coste1/orden1|137|137|válido|
|practice_0196|shared|E3|donante2/coste0/orden0|129|129|válido|
|practice_0196|shared|E3|donante2/coste1/orden0|129|129|válido|
|practice_0196|shared|E3|donante2/coste0/orden1|129|129|válido|
|practice_0196|shared|E3|donante2/coste1/orden1|129|129|válido|
|practice_0196|shared|E3|donante3/coste0/orden0|135|135|válido|
|practice_0196|shared|E3|donante3/coste1/orden0|135|135|válido|
|practice_0196|shared|E3|donante3/coste0/orden1|135|135|válido|
|practice_0196|shared|E3|donante3/coste1/orden1|135|135|válido|
|practice_0196|shared|E3|donante4/coste0/orden0|160|160|válido|
|practice_0196|shared|E3|donante4/coste1/orden0|160|160|válido|
|practice_0196|shared|E3|donante4/coste0/orden1|160|160|válido|
|practice_0196|shared|E3|donante4/coste1/orden1|160|160|válido|
|practice_0196|shared|E3|donante5/coste0/orden0|140|140|válido|
|practice_0196|shared|E3|donante5/coste1/orden0|140|140|válido|
|practice_0196|shared|E3|donante5/coste0/orden1|140|140|válido|
|practice_0196|shared|E3|donante5/coste1/orden1|140|140|válido|
|practice_0196|shared|E3|donante6/coste0/orden0|145|145|válido|
|practice_0196|shared|E3|donante6/coste1/orden0|145|145|válido|
|practice_0196|shared|E3|donante6/coste0/orden1|145|145|válido|
|practice_0196|shared|E3|donante6/coste1/orden1|145|145|válido|
|practice_0196|shared|E3|donante7/coste0/orden0|133|133|válido|
|practice_0196|shared|E3|donante7/coste1/orden0|133|133|válido|
|practice_0196|shared|E3|donante7/coste0/orden1|133|133|válido|
|practice_0196|shared|E3|donante7/coste1/orden1|133|133|válido|
|practice_0196|shared|E3|donante8/coste0/orden0|133|133|válido|
|practice_0196|shared|E3|donante8/coste1/orden0|133|133|válido|
|practice_0196|shared|E3|donante8/coste0/orden1|133|133|válido|
|practice_0196|shared|E3|donante8/coste1/orden1|133|133|válido|
|practice_0196|shared|E3|donante9/coste0/orden0|129|129|válido|
|practice_0196|shared|E3|donante9/coste1/orden0|129|129|válido|
|practice_0196|shared|E3|donante9/coste0/orden1|129|129|válido|
|practice_0196|shared|E3|donante9/coste1/orden1|129|129|válido|
|practice_0197|mixed|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0197|mixed|Cruce|CP True/coste0/orden0|180|180|válido|
|practice_0197|mixed|Cruce|CP True/coste1/orden0|180|180|válido|
|practice_0197|mixed|Cruce|CP True/coste0/orden1|180|180|válido|
|practice_0197|mixed|Cruce|CP True/coste1/orden1|180|180|válido|
|practice_0197|mixed|Cruce|mejor independiente/coste0/orden0|181|181|válido|
|practice_0197|mixed|Cruce|mejor independiente/coste1/orden0|181|181|válido|
|practice_0197|mixed|Cruce|mejor independiente/coste0/orden1|181|181|válido|
|practice_0197|mixed|Cruce|mejor independiente/coste1/orden1|181|181|válido|
|practice_0197|mixed|E1|cola10/pol1/inicio0|194|75|válido|
|practice_0197|mixed|E1|cola10/pol1/inicio1|194|75|válido|
|practice_0197|mixed|E1|cola10/pol2/inicio0|197|78|válido|
|practice_0197|mixed|E1|cola10/pol2/inicio1|197|78|válido|
|practice_0197|mixed|E1|cola8/pol1/inicio0|198|65|válido|
|practice_0197|mixed|E1|cola8/pol1/inicio1|198|65|válido|
|practice_0197|mixed|E1|cola8/pol2/inicio0|199|66|válido|
|practice_0197|mixed|E1|cola8/pol2/inicio1|199|66|válido|
|practice_0197|mixed|E1|cola6/pol1/inicio0|192|45|válido|
|practice_0197|mixed|E1|cola6/pol1/inicio1|192|45|válido|
|practice_0197|mixed|E1|cola6/pol2/inicio0|191|44|válido|
|practice_0197|mixed|E1|cola6/pol2/inicio1|191|44|válido|
|practice_0197|mixed|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0197|mixed|E2|CP False/coste0/orden0|190|190|válido|
|practice_0197|mixed|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0197|mixed|E2|CP True/coste0/orden0|181|181|válido|
|practice_0197|mixed|E3|donante0/coste0/orden0|182|182|válido|
|practice_0197|mixed|E3|donante0/coste1/orden0|182|182|válido|
|practice_0197|mixed|E3|donante0/coste0/orden1|182|182|válido|
|practice_0197|mixed|E3|donante0/coste1/orden1|182|182|válido|
|practice_0197|mixed|E3|donante1/coste0/orden0|209|209|válido|
|practice_0197|mixed|E3|donante1/coste1/orden0|209|209|válido|
|practice_0197|mixed|E3|donante1/coste0/orden1|209|209|válido|
|practice_0197|mixed|E3|donante1/coste1/orden1|209|209|válido|
|practice_0197|mixed|E3|donante2/coste0/orden0|199|199|válido|
|practice_0197|mixed|E3|donante2/coste1/orden0|199|199|válido|
|practice_0197|mixed|E3|donante2/coste0/orden1|199|199|válido|
|practice_0197|mixed|E3|donante2/coste1/orden1|199|199|válido|
|practice_0197|mixed|E3|donante3/coste0/orden0|199|198|válido|
|practice_0197|mixed|E3|donante3/coste1/orden0|199|198|válido|
|practice_0197|mixed|E3|donante3/coste0/orden1|198|198|válido|
|practice_0197|mixed|E3|donante3/coste1/orden1|198|198|válido|
|practice_0197|mixed|E3|donante4/coste0/orden0|214|214|válido|
|practice_0197|mixed|E3|donante4/coste1/orden0|214|214|válido|
|practice_0197|mixed|E3|donante4/coste0/orden1|214|214|válido|
|practice_0197|mixed|E3|donante4/coste1/orden1|214|214|válido|
|practice_0197|mixed|E3|donante5/coste0/orden0|209|209|válido|
|practice_0197|mixed|E3|donante5/coste1/orden0|209|209|válido|
|practice_0197|mixed|E3|donante5/coste0/orden1|209|209|válido|
|practice_0197|mixed|E3|donante5/coste1/orden1|209|209|válido|
|practice_0197|mixed|E3|donante6/coste0/orden0|215|215|válido|
|practice_0197|mixed|E3|donante6/coste1/orden0|215|215|válido|
|practice_0197|mixed|E3|donante6/coste0/orden1|215|215|válido|
|practice_0197|mixed|E3|donante6/coste1/orden1|215|215|válido|
|practice_0197|mixed|E3|donante7/coste0/orden0|198|190|válido|
|practice_0197|mixed|E3|donante7/coste1/orden0|198|190|válido|
|practice_0197|mixed|E3|donante7/coste0/orden1|190|190|válido|
|practice_0197|mixed|E3|donante7/coste1/orden1|190|190|válido|
|practice_0197|mixed|E3|donante8/coste0/orden0|297|183|válido|
|practice_0197|mixed|E3|donante8/coste1/orden0|202|183|válido|
|practice_0197|mixed|E3|donante8/coste0/orden1|188|183|válido|
|practice_0197|mixed|E3|donante8/coste1/orden1|188|183|válido|
|practice_0198|multiplicative|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0198|multiplicative|Cruce|CP True/coste0/orden0|34|34|válido|
|practice_0198|multiplicative|Cruce|CP True/coste1/orden0|34|34|válido|
|practice_0198|multiplicative|Cruce|CP True/coste0/orden1|34|34|válido|
|practice_0198|multiplicative|Cruce|CP True/coste1/orden1|34|34|válido|
|practice_0198|multiplicative|Cruce|mejor independiente/coste0/orden0|38|38|válido|
|practice_0198|multiplicative|Cruce|mejor independiente/coste1/orden0|38|38|válido|
|practice_0198|multiplicative|Cruce|mejor independiente/coste0/orden1|38|38|válido|
|practice_0198|multiplicative|Cruce|mejor independiente/coste1/orden1|38|38|válido|
|practice_0198|multiplicative|E1|cola10/pol1/inicio0|45|33|válido|
|practice_0198|multiplicative|E1|cola10/pol1/inicio1|45|33|válido|
|practice_0198|multiplicative|E1|cola10/pol2/inicio0|45|33|válido|
|practice_0198|multiplicative|E1|cola10/pol2/inicio1|42|30|válido|
|practice_0198|multiplicative|E1|cola8/pol1/inicio0|46|27|válido|
|practice_0198|multiplicative|E1|cola8/pol1/inicio1|46|27|válido|
|practice_0198|multiplicative|E1|cola8/pol2/inicio0|46|27|válido|
|practice_0198|multiplicative|E1|cola8/pol2/inicio1|46|27|válido|
|practice_0198|multiplicative|E1|cola6/pol1/inicio0|45|17|válido|
|practice_0198|multiplicative|E1|cola6/pol1/inicio1|45|17|válido|
|practice_0198|multiplicative|E1|cola6/pol2/inicio0|45|17|válido|
|practice_0198|multiplicative|E1|cola6/pol2/inicio1|45|17|válido|
|practice_0198|multiplicative|E2|CP-SAT observado|—|—|OPTIMAL|
|practice_0198|multiplicative|E2|CP False/coste0/orden0|41|41|válido|
|practice_0198|multiplicative|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0198|multiplicative|E2|CP True/coste0/orden0|39|39|válido|
|practice_0198|multiplicative|E3|donante0/coste0/orden0|41|41|válido|
|practice_0198|multiplicative|E3|donante0/coste1/orden0|41|41|válido|
|practice_0198|multiplicative|E3|donante0/coste0/orden1|41|41|válido|
|practice_0198|multiplicative|E3|donante0/coste1/orden1|41|41|válido|
|practice_0198|multiplicative|E3|donante1/coste0/orden0|47|47|válido|
|practice_0198|multiplicative|E3|donante1/coste1/orden0|47|47|válido|
|practice_0198|multiplicative|E3|donante1/coste0/orden1|47|47|válido|
|practice_0198|multiplicative|E3|donante1/coste1/orden1|47|47|válido|
|practice_0198|multiplicative|E3|donante2/coste0/orden0|43|43|válido|
|practice_0198|multiplicative|E3|donante2/coste1/orden0|43|43|válido|
|practice_0198|multiplicative|E3|donante2/coste0/orden1|43|43|válido|
|practice_0198|multiplicative|E3|donante2/coste1/orden1|43|43|válido|
|practice_0198|multiplicative|E3|donante3/coste0/orden0|50|50|válido|
|practice_0198|multiplicative|E3|donante3/coste1/orden0|50|50|válido|
|practice_0198|multiplicative|E3|donante3/coste0/orden1|50|50|válido|
|practice_0198|multiplicative|E3|donante3/coste1/orden1|50|50|válido|
|practice_0198|multiplicative|E3|donante4/coste0/orden0|49|49|válido|
|practice_0198|multiplicative|E3|donante4/coste1/orden0|49|49|válido|
|practice_0198|multiplicative|E3|donante4/coste0/orden1|49|49|válido|
|practice_0198|multiplicative|E3|donante4/coste1/orden1|49|49|válido|
|practice_0198|multiplicative|E3|donante5/coste0/orden0|43|43|válido|
|practice_0198|multiplicative|E3|donante5/coste1/orden0|43|43|válido|
|practice_0198|multiplicative|E3|donante5/coste0/orden1|43|43|válido|
|practice_0198|multiplicative|E3|donante5/coste1/orden1|43|43|válido|
|practice_0198|multiplicative|E3|donante6/coste0/orden0|48|48|válido|
|practice_0198|multiplicative|E3|donante6/coste1/orden0|48|48|válido|
|practice_0198|multiplicative|E3|donante6/coste0/orden1|48|48|válido|
|practice_0198|multiplicative|E3|donante6/coste1/orden1|48|48|válido|
|practice_0198|multiplicative|E3|donante7/coste0/orden0|48|48|válido|
|practice_0198|multiplicative|E3|donante7/coste1/orden0|48|48|válido|
|practice_0198|multiplicative|E3|donante7/coste0/orden1|48|48|válido|
|practice_0198|multiplicative|E3|donante7/coste1/orden1|48|48|válido|
|practice_0198|multiplicative|E3|donante8/coste0/orden0|44|44|válido|
|practice_0198|multiplicative|E3|donante8/coste1/orden0|44|44|válido|
|practice_0198|multiplicative|E3|donante8/coste0/orden1|44|44|válido|
|practice_0198|multiplicative|E3|donante8/coste1/orden1|44|44|válido|
|practice_0199|independent|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0199|independent|Cruce|CP True/coste0/orden0|382|249|válido|
|practice_0199|independent|Cruce|CP True/coste1/orden0|347|249|válido|
|practice_0199|independent|Cruce|CP True/coste0/orden1|275|249|válido|
|practice_0199|independent|Cruce|CP True/coste1/orden1|269|249|válido|
|practice_0199|independent|Cruce|mejor independiente/coste0/orden0|390|250|válido|
|practice_0199|independent|Cruce|mejor independiente/coste1/orden0|358|250|válido|
|practice_0199|independent|Cruce|mejor independiente/coste0/orden1|282|250|válido|
|practice_0199|independent|Cruce|mejor independiente/coste1/orden1|265|250|válido|
|practice_0199|independent|E1|cola10/pol1/inicio0|273|77|válido|
|practice_0199|independent|E1|cola10/pol1/inicio1|273|77|válido|
|practice_0199|independent|E1|cola10/pol2/inicio0|270|74|válido|
|practice_0199|independent|E1|cola10/pol2/inicio1|270|74|válido|
|practice_0199|independent|E1|cola8/pol1/inicio0|267|58|válido|
|practice_0199|independent|E1|cola8/pol1/inicio1|267|58|válido|
|practice_0199|independent|E1|cola8/pol2/inicio0|265|56|válido|
|practice_0199|independent|E1|cola8/pol2/inicio1|265|56|válido|
|practice_0199|independent|E1|cola6/pol1/inicio0|268|45|válido|
|practice_0199|independent|E1|cola6/pol1/inicio1|268|45|válido|
|practice_0199|independent|E1|cola6/pol2/inicio0|268|45|válido|
|practice_0199|independent|E1|cola6/pol2/inicio1|268|45|válido|
|practice_0199|independent|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0199|independent|E2|CP False/coste0/orden0|417|274|válido|
|practice_0199|independent|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0199|independent|E2|CP True/coste0/orden0|390|250|válido|
|practice_0199|independent|E3|donante0/coste0/orden0|267|250|válido|
|practice_0199|independent|E3|donante0/coste1/orden0|263|250|válido|
|practice_0199|independent|E3|donante0/coste0/orden1|282|250|válido|
|practice_0199|independent|E3|donante0/coste1/orden1|265|250|válido|
|practice_0199|independent|E3|donante1/coste0/orden0|291|291|válido|
|practice_0199|independent|E3|donante1/coste1/orden0|291|291|válido|
|practice_0199|independent|E3|donante1/coste0/orden1|291|291|válido|
|practice_0199|independent|E3|donante1/coste1/orden1|291|291|válido|
|practice_0199|independent|E3|donante2/coste0/orden0|415|272|válido|
|practice_0199|independent|E3|donante2/coste1/orden0|352|272|válido|
|practice_0199|independent|E3|donante2/coste0/orden1|272|272|válido|
|practice_0199|independent|E3|donante2/coste1/orden1|272|272|válido|
|practice_0199|independent|E3|donante3/coste0/orden0|372|271|válido|
|practice_0199|independent|E3|donante3/coste1/orden0|339|271|válido|
|practice_0199|independent|E3|donante3/coste0/orden1|271|271|válido|
|practice_0199|independent|E3|donante3/coste1/orden1|271|271|válido|
|practice_0199|independent|E3|donante4/coste0/orden0|291|291|válido|
|practice_0199|independent|E3|donante4/coste1/orden0|291|291|válido|
|practice_0199|independent|E3|donante4/coste0/orden1|291|291|válido|
|practice_0199|independent|E3|donante4/coste1/orden1|291|291|válido|
|practice_0199|independent|E3|donante5/coste0/orden0|296|296|válido|
|practice_0199|independent|E3|donante5/coste1/orden0|296|296|válido|
|practice_0199|independent|E3|donante5/coste0/orden1|296|296|válido|
|practice_0199|independent|E3|donante5/coste1/orden1|296|296|válido|
|practice_0199|independent|E3|donante6/coste0/orden0|289|289|válido|
|practice_0199|independent|E3|donante6/coste1/orden0|289|289|válido|
|practice_0199|independent|E3|donante6/coste0/orden1|289|289|válido|
|practice_0199|independent|E3|donante6/coste1/orden1|289|289|válido|
|practice_0199|independent|E3|donante7/coste0/orden0|286|260|válido|
|practice_0199|independent|E3|donante7/coste1/orden0|263|260|válido|
|practice_0199|independent|E3|donante7/coste0/orden1|270|260|válido|
|practice_0199|independent|E3|donante7/coste1/orden1|265|260|válido|
|practice_0199|independent|E3|donante8/coste0/orden0|279|256|válido|
|practice_0199|independent|E3|donante8/coste1/orden0|263|256|válido|
|practice_0199|independent|E3|donante8/coste0/orden1|263|256|válido|
|practice_0199|independent|E3|donante8/coste1/orden1|275|256|válido|
|practice_0199|independent|E3|donante9/coste0/orden0|267|250|válido|
|practice_0199|independent|E3|donante9/coste1/orden0|263|250|válido|
|practice_0199|independent|E3|donante9/coste0/orden1|282|250|válido|
|practice_0199|independent|E3|donante9/coste1/orden1|265|250|válido|
|practice_0200|shared|Cruce|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0200|shared|Cruce|CP True/coste0/orden0|122|122|válido|
|practice_0200|shared|Cruce|CP True/coste1/orden0|122|122|válido|
|practice_0200|shared|Cruce|CP True/coste0/orden1|122|122|válido|
|practice_0200|shared|Cruce|CP True/coste1/orden1|122|122|válido|
|practice_0200|shared|Cruce|mejor independiente/coste0/orden0|125|125|válido|
|practice_0200|shared|Cruce|mejor independiente/coste1/orden0|125|125|válido|
|practice_0200|shared|Cruce|mejor independiente/coste0/orden1|125|125|válido|
|practice_0200|shared|Cruce|mejor independiente/coste1/orden1|125|125|válido|
|practice_0200|shared|E1|cola10/pol1/inicio0|132|47|válido|
|practice_0200|shared|E1|cola10/pol1/inicio1|132|47|válido|
|practice_0200|shared|E1|cola10/pol2/inicio0|129|44|válido|
|practice_0200|shared|E1|cola10/pol2/inicio1|129|44|válido|
|practice_0200|shared|E1|cola8/pol1/inicio0|128|35|válido|
|practice_0200|shared|E1|cola8/pol1/inicio1|128|35|válido|
|practice_0200|shared|E1|cola8/pol2/inicio0|126|33|válido|
|practice_0200|shared|E1|cola8/pol2/inicio1|126|33|válido|
|practice_0200|shared|E1|cola6/pol1/inicio0|134|32|válido|
|practice_0200|shared|E1|cola6/pol1/inicio1|134|32|válido|
|practice_0200|shared|E1|cola6/pol2/inicio0|129|27|válido|
|practice_0200|shared|E1|cola6/pol2/inicio1|129|27|válido|
|practice_0200|shared|E2|CP-SAT observado|—|—|FEASIBLE|
|practice_0200|shared|E2|CP False/coste0/orden0|129|129|válido|
|practice_0200|shared|E2|CP-SAT cruzado|—|—|FEASIBLE|
|practice_0200|shared|E2|CP True/coste0/orden0|125|125|válido|
|practice_0200|shared|E3|donante0/coste0/orden0|129|129|válido|
|practice_0200|shared|E3|donante0/coste1/orden0|129|129|válido|
|practice_0200|shared|E3|donante0/coste0/orden1|129|129|válido|
|practice_0200|shared|E3|donante0/coste1/orden1|129|129|válido|
|practice_0200|shared|E3|donante1/coste0/orden0|145|145|válido|
|practice_0200|shared|E3|donante1/coste1/orden0|145|145|válido|
|practice_0200|shared|E3|donante1/coste0/orden1|145|145|válido|
|practice_0200|shared|E3|donante1/coste1/orden1|145|145|válido|
|practice_0200|shared|E3|donante2/coste0/orden0|142|142|válido|
|practice_0200|shared|E3|donante2/coste1/orden0|142|142|válido|
|practice_0200|shared|E3|donante2/coste0/orden1|142|142|válido|
|practice_0200|shared|E3|donante2/coste1/orden1|142|142|válido|
|practice_0200|shared|E3|donante3/coste0/orden0|135|135|válido|
|practice_0200|shared|E3|donante3/coste1/orden0|135|135|válido|
|practice_0200|shared|E3|donante3/coste0/orden1|135|135|válido|
|practice_0200|shared|E3|donante3/coste1/orden1|135|135|válido|
|practice_0200|shared|E3|donante4/coste0/orden0|148|148|válido|
|practice_0200|shared|E3|donante4/coste1/orden0|148|148|válido|
|practice_0200|shared|E3|donante4/coste0/orden1|148|148|válido|
|practice_0200|shared|E3|donante4/coste1/orden1|148|148|válido|
|practice_0200|shared|E3|donante5/coste0/orden0|142|142|válido|
|practice_0200|shared|E3|donante5/coste1/orden0|142|142|válido|
|practice_0200|shared|E3|donante5/coste0/orden1|142|142|válido|
|practice_0200|shared|E3|donante5/coste1/orden1|142|142|válido|
|practice_0200|shared|E3|donante6/coste0/orden0|150|150|válido|
|practice_0200|shared|E3|donante6/coste1/orden0|150|150|válido|
|practice_0200|shared|E3|donante6/coste0/orden1|150|150|válido|
|practice_0200|shared|E3|donante6/coste1/orden1|150|150|válido|
|practice_0200|shared|E3|donante7/coste0/orden0|131|131|válido|
|practice_0200|shared|E3|donante7/coste1/orden0|131|131|válido|
|practice_0200|shared|E3|donante7/coste0/orden1|131|131|válido|
|practice_0200|shared|E3|donante7/coste1/orden1|131|131|válido|
|practice_0200|shared|E3|donante8/coste0/orden0|132|131|válido|
|practice_0200|shared|E3|donante8/coste1/orden0|132|131|válido|
|practice_0200|shared|E3|donante8/coste0/orden1|131|131|válido|
|practice_0200|shared|E3|donante8/coste1/orden1|131|131|válido|
|practice_0200|shared|E3|donante9/coste0/orden0|130|130|válido|
|practice_0200|shared|E3|donante9/coste1/orden0|130|130|válido|
|practice_0200|shared|E3|donante9/coste0/orden1|130|130|válido|
|practice_0200|shared|E3|donante9/coste1/orden1|130|130|válido|